# Biohub L2 K-DN: DaughterNet training with honest folds + OOF predictions

Trains `tools/daughternet.py` (DivNet UNet-GAP body, 3-logit head, classes cont / parent / daughter) **from random init**
on the 199 train videos with folds grouped by video, and predicts every labelled sample with the fold model that never
saw its video.

* Folds: `experiments/l2_daughternet/splits.json` (tools/make_splits.py, 5 folds) embedded verbatim.
* Labels (annotated GT nodes only): parent = >= 2 GT children; daughter = child of a parent at its first frame;
  cont = in/out-degree 1 and >= 2 frames (lineage) from any parent/daughter node, <= 60 per video; hard negatives =
  cont nodes inside the L2 insertion gate of an annotated one-child node (as u or v), <= 40 per video.
* **v2 pool rows** (KDN_EXTRA_SAMPLES = the embedded `pool_samples.csv`, a bit-exact CSV copy of
  `experiments/l2_daughternet/pool_samples.parquet` (sha256 8b791ff8dbc15f33, 11238 rows in
  197 videos, {'cont': 11032, 'daughter': 133, 'parent': 73}, tools/make_pool_samples.py): nodes of OUR predicted graphs
  (exp39 candidate triples) labelled by the nearest annotated GT node within 7 um, cropped at their PREDICTED
  coordinates exactly like GT samples; source=pool, fold = the video's fold; a pool row within 1 voxel of a GT sample
  (same t) is dropped (GT label kept). The class-balanced sampler is unchanged; KDN_POOL_WEIGHT scales the loss of pool
  rows. The readout has a separate source=pool block (`by_source.pool` in oof_readout.json, `[readout pool]` lines):
  the block that matters for insertion precision. `oof.csv` carries `source` (gt / pool).
* Input = the DivNet contract (`prepare_sample`): lags -1..+2, XY mean-pool 4, 16x32x32 crop, percentile 50/99.5
  clipped to [-0.5, 6], gaussian marker -> (5, 16, 32, 32). An 18x34x34 crop is cached so the +-1 voxel jitter has a
  margin; its inner crop equals the contract sample exactly (asserted per video and per source).
* Training: AdamW 1e-3 + cosine, <= 30 epochs, early stopping on the fold's val loss (class-macro CE), batch 64,
  class-balanced sampler, flips z/y/x + rot90 yx (all 5 channels), +-1 voxel jitter, gain 0.8-1.2. Two folds run
  concurrently, one per T4 (one subprocess per fold).
* Outputs in /kaggle/working: `oof.csv`, `oof_readout.json`, `folds/fold_k/{best,last}.pt`
  ({'model_state','config','fold','epoch','val_loss'}), `folds/fold_k/{history,summary}.json`,
  `labels_summary.csv`, `splits_used.json`, `contract_check.json`.
* Knobs (env, set in the config cell): KDN_SMOKE=1 (5 videos, 1 epoch, 2 folds), KDN_MAX_VIDEOS, KDN_EPOCHS,
  KDN_EPOCH_SAMPLES, KDN_PATIENCE, KDN_INNER_VAL, KDN_CACHE_DTYPE, KDN_EXTRA_SAMPLES, KDN_POOL_WEIGHT, KDN_ES_SOURCE, ...


In [ ]:
import os, sys, platform, importlib
from pathlib import Path

KDN_SRC = Path("/kaggle/working/kdn_src")
(KDN_SRC / "tools").mkdir(parents=True, exist_ok=True)
print("python", sys.version.split()[0], platform.platform(), "cpus", os.cpu_count())
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
for _name in ("numpy", "pandas", "blosc2", "numcodecs", "zstandard", "pyarrow", "imagecodecs", "zarr"):
    try:
        _mod = importlib.import_module(_name)
        print(f"  {_name:12s} {getattr(_mod, '__version__', '?')}")
    except Exception as _exc:
        print(f"  {_name:12s} MISSING ({type(_exc).__name__})")
for _p in sorted(Path("/kaggle/input").glob("*")) if Path("/kaggle/input").exists() else []:
    print("input:", _p)


In [ ]:
%%writefile /kaggle/working/kdn_src/tools/__init__.py
# tools package materialised by the K-DN kernel (divnet_infer + daughternet, verbatim repo copies)


In [ ]:
%%writefile /kaggle/working/kdn_src/tools/divnet_infer.py
"""Inference for the public ``giorgosi/biohub-divnet-v2`` checkpoint.

The dataset manifest is the contract: four image channels at lags -1, 0, +1,
+2 plus a Gaussian centre marker, XY mean pooling by four, a 16x32x32 crop,
percentile normalisation, and a small 3-D U-Net followed by global pooling.

``read_test_frame`` is supplied by the caller (the public pipeline notebook or
the local B20 scorer). This module raises on an incompatible checkpoint or
tensor shape; a silent geometric fallback would invalidate B20.
"""

from __future__ import annotations

import os
from pathlib import Path
from typing import Callable

import numpy as np
import torch


class _DivNetConvBlock(torch.nn.Module):
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.block = torch.nn.Sequential(
            torch.nn.Conv3d(in_channels, out_channels, 3, padding=1, bias=False),
            torch.nn.GroupNorm(min(8, out_channels), out_channels),
            torch.nn.SiLU(inplace=True),
            torch.nn.Conv3d(out_channels, out_channels, 3, padding=1, bias=False),
            torch.nn.GroupNorm(min(8, out_channels), out_channels),
            torch.nn.SiLU(inplace=True),
        )

    def forward(self, value: torch.Tensor) -> torch.Tensor:
        return self.block(value)


class DivNetMitosisClassifier(torch.nn.Module):
    """Architecture reconstructed exactly from the public checkpoint shapes."""

    def __init__(self, in_channels: int = 5, base_channels: int = 16) -> None:
        super().__init__()
        c = int(base_channels)
        self.enc1 = _DivNetConvBlock(in_channels, c)
        self.pool1 = torch.nn.MaxPool3d(2, 2)
        self.enc2 = _DivNetConvBlock(c, c * 2)
        self.pool2 = torch.nn.MaxPool3d(2, 2)
        self.enc3 = _DivNetConvBlock(c * 2, c * 4)
        self.pool3 = torch.nn.MaxPool3d(2, 2)
        self.bottleneck = _DivNetConvBlock(c * 4, c * 8)
        self.up3 = torch.nn.ConvTranspose3d(c * 8, c * 4, 2, 2)
        self.dec3 = _DivNetConvBlock(c * 8, c * 4)
        self.up2 = torch.nn.ConvTranspose3d(c * 4, c * 2, 2, 2)
        self.dec2 = _DivNetConvBlock(c * 4, c * 2)
        self.up1 = torch.nn.ConvTranspose3d(c * 2, c, 2, 2)
        self.dec1 = _DivNetConvBlock(c * 2, c)
        self.gap = torch.nn.AdaptiveAvgPool3d((1, 1, 1))
        self.head = torch.nn.Linear(c, 1)

    def forward(self, value: torch.Tensor) -> torch.Tensor:
        e1 = self.enc1(value)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        bottleneck = self.bottleneck(self.pool3(e3))
        d3 = self.dec3(torch.cat([self.up3(bottleneck), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(self.gap(d1).flatten(1))


def _checkpoint_candidates() -> list[Path]:
    root = Path(__file__).resolve().parent.parent
    explicit = os.environ.get("BIOHUB_DIVNET_CHECKPOINT", "").strip()
    candidates = [Path(explicit)] if explicit else []
    candidates.extend(
        [
            root / "external" / "giorgosi__biohub-divnet-v2" / "best_overall.pt",
            root / "external" / "giorgosi__biohub-divnet-v2" / "weights" / "divnet" / "best_overall.pt",
            Path("/kaggle/input/biohub-divnet-v2/best_overall.pt"),
            Path("/kaggle/input/biohub-divnet-v2/weights/divnet/best_overall.pt"),
            Path("/kaggle/input/datasets/giorgosi/biohub-divnet-v2/best_overall.pt"),
            Path("/kaggle/input/datasets/giorgosi/biohub-divnet-v2/weights/divnet/best_overall.pt"),
        ]
    )
    return candidates


def load_divnet_mitosis_model(checkpoint: Path | None = None) -> dict[str, object]:
    if not globals().get("DIVNET_VERIFY", True):
        raise RuntimeError("DivNet is disabled by configuration")
    path = Path(checkpoint) if checkpoint is not None else next(
        (item for item in _checkpoint_candidates() if item.is_file()), None
    )
    if path is None:
        checked = "\n".join(str(item) for item in _checkpoint_candidates())
        raise FileNotFoundError(f"DivNet checkpoint not found. Checked:\n{checked}")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    payload = torch.load(path, map_location="cpu", weights_only=False)
    state = payload.get("model_state", payload) if isinstance(payload, dict) else payload
    config = payload.get("config", {}) if isinstance(payload, dict) else {}
    model_cfg = config.get("model", {}) if isinstance(config, dict) else {}
    model = DivNetMitosisClassifier(
        in_channels=int(model_cfg.get("in_channels", 5)),
        base_channels=int(model_cfg.get("base_channels", 16)),
    )
    model.load_state_dict(state, strict=True)
    model.to(device).eval()
    return {
        "model": model,
        "device": device,
        "torch": torch,
        "path": path.resolve(),
        "config": config,
        "prepared_frames": {},
    }


def _pool_xy(frame: np.ndarray, factor: int) -> np.ndarray:
    frame = np.asarray(frame)
    if factor <= 1:
        return frame.astype(np.float32, copy=False)
    z, y, x = frame.shape
    y2, x2 = (y // factor) * factor, (x // factor) * factor
    value = frame[:, :y2, :x2].astype(np.float32, copy=False)
    return value.reshape(z, y2 // factor, factor, x2 // factor, factor).mean(axis=(2, 4))


def _normalise(frame: np.ndarray, lo_pct: float, hi_pct: float, clip: tuple[float, float]) -> np.ndarray:
    value = np.asarray(frame, dtype=np.float32)
    lo, hi = np.percentile(value, [lo_pct, hi_pct])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(value)
    return np.clip((value - float(lo)) / (float(hi) - float(lo)), clip[0], clip[1]).astype(np.float32)


def _crop_with_padding(frame: np.ndarray, centre: tuple[int, int, int], shape: tuple[int, int, int]) -> np.ndarray:
    result = np.zeros(shape, dtype=np.float32)
    source_slices: list[slice] = []
    target_slices: list[slice] = []
    for coordinate, size, limit in zip(centre, shape, frame.shape):
        start = coordinate - size // 2
        stop = start + size
        source_start, source_stop = max(0, start), min(limit, stop)
        target_start = source_start - start
        target_stop = target_start + max(0, source_stop - source_start)
        source_slices.append(slice(source_start, source_stop))
        target_slices.append(slice(target_start, target_stop))
    result[tuple(target_slices)] = frame[tuple(source_slices)]
    return result


def _marker(shape: tuple[int, int, int], sigma: tuple[float, float, float]) -> np.ndarray:
    axes = [np.arange(size, dtype=np.float32) - size // 2 for size in shape]
    zz, yy, xx = np.meshgrid(*axes, indexing="ij")
    distance = (zz / sigma[0]) ** 2 + (yy / sigma[1]) ** 2 + (xx / sigma[2]) ** 2
    return np.exp(-0.5 * distance).astype(np.float32)


def prepare_divnet_input(
    dataset: str,
    t: int,
    parent_node: dict[str, object],
    divnet_bundle: dict[str, object],
    frame_cache: dict[object, np.ndarray],
    read_frame: Callable[[str, int, dict], np.ndarray] | None = None,
) -> np.ndarray:
    """Build one ``(5, 16, 32, 32)`` sample from the manifest recipe."""
    cfg = divnet_bundle.get("config", {})
    input_cfg = cfg.get("input", {}) if isinstance(cfg, dict) else {}
    norm_cfg = cfg.get("normalization", {}) if isinstance(cfg, dict) else {}
    lags = tuple(int(value) for value in input_cfg.get("image_lags", (-1, 0, 1, 2)))
    pool_xy = int(input_cfg.get("pool_xy", 4))
    shape = (
        int(input_cfg.get("crop_z", 16)),
        int(input_cfg.get("crop_yx", 32)),
        int(input_cfg.get("crop_yx", 32)),
    )
    sigma = tuple(float(value) for value in input_cfg.get("marker_sigma", (1.5, 2.0, 2.0)))
    clip = tuple(float(value) for value in norm_cfg.get("clip", (-0.5, 6.0)))
    reader = read_frame or globals().get("read_test_frame")
    if reader is None:
        raise RuntimeError("DivNet needs a read_frame callback or a global read_test_frame")
    prepared = divnet_bundle.setdefault("prepared_frames", {})
    channels: list[np.ndarray] = []
    for lag in lags:
        frame_t = max(0, int(t) + lag)
        key = (dataset, frame_t)
        if key not in prepared:
            raw = reader(dataset, frame_t, frame_cache)
            pooled = _pool_xy(raw, pool_xy)
            prepared[key] = _normalise(
                pooled,
                float(norm_cfg.get("lo_pct", 50.0)),
                float(norm_cfg.get("hi_pct", 99.5)),
                clip,
            )
            frame_cache.pop(frame_t, None)
        centre = (
            int(round(float(parent_node["z"]))),
            int(round(float(parent_node["y"]) / pool_xy)),
            int(round(float(parent_node["x"]) / pool_xy)),
        )
        channels.append(_crop_with_padding(prepared[key], centre, shape))
    channels.append(_marker(shape, sigma))
    sample = np.stack(channels, axis=0)
    if sample.shape != (5, *shape) or not np.isfinite(sample).all():
        raise RuntimeError(f"Invalid DivNet input: shape={sample.shape}, finite={np.isfinite(sample).all()}")
    return sample


def divnet_score_division(
    dataset: str,
    t: int,
    parent_node: dict[str, object],
    daughter1: dict[str, object],
    daughter2: dict[str, object],
    divnet_bundle: dict[str, object],
    frame_cache: dict[object, np.ndarray],
    read_frame: Callable[[str, int, dict], np.ndarray] | None = None,
) -> float:
    """Return the parent division probability; daughter arguments preserve the public API."""
    del daughter1, daughter2
    sample = prepare_divnet_input(dataset, t, parent_node, divnet_bundle, frame_cache, read_frame)
    torch_mod = divnet_bundle["torch"]
    tensor = torch_mod.from_numpy(sample).unsqueeze(0).to(
        device=divnet_bundle["device"], dtype=torch_mod.float32
    )
    with torch_mod.inference_mode():
        logit = divnet_bundle["model"](tensor)
        probability = float(torch_mod.sigmoid(logit).item())
    if not np.isfinite(probability):
        raise RuntimeError("DivNet produced a non-finite probability")
    return probability


DIVNET_BUNDLE = None


In [ ]:
%%writefile /kaggle/working/kdn_src/tools/daughternet.py
"""DaughterNet: 3-class node classifier (continuation / parent / daughter) for the L2 division stage.

Shared contract between training (Kaggle K-DN kernels), the submission kernel patch (--patch divinsert) and the laptop
replay. The input sample is EXACTLY the DivNet manifest contract of tools/divnet_infer.prepare_divnet_input: image
channels at lags -1, 0, +1, +2 around the node's frame t, XY mean-pool by 4, a 16x32x32 crop centred on the node,
percentile 50/99.5 normalisation clipped to [-0.5, 6], plus one gaussian centre marker -> (5, 16, 32, 32).
The network is the DivNet UNet-GAP body with a 3-logit head. Class order is fixed: CLASSES = (cont, parent, daughter).
  parent   = GT dividing node at the last frame before the split (label at its own t)
  daughter = GT daughter node at its first frame (label at its own t)
  cont     = annotated node with in-degree = out-degree = 1, >= 2 frames from any GT division
Scoring averages softmax over the loaded fold models (ensemble) or uses one fold model per video (OOF).
"""
from __future__ import annotations

from pathlib import Path
from typing import Callable, Sequence

import numpy as np
import torch

try:  # repo layout
    from tools.divnet_infer import _DivNetConvBlock, prepare_divnet_input
except Exception:  # embedded in a notebook cell after divnet_infer's code (names already global)
    _DivNetConvBlock = globals().get("_DivNetConvBlock")
    prepare_divnet_input = globals().get("prepare_divnet_input")

CLASSES = ("cont", "parent", "daughter")
DEFAULT_CFG = {
    "input": {"image_lags": [-1, 0, 1, 2], "pool_xy": 4, "crop_z": 16, "crop_yx": 32, "marker_sigma": [1.5, 2.0, 2.0]},
    "normalization": {"lo_pct": 50.0, "hi_pct": 99.5, "clip": [-0.5, 6.0]},
    "model": {"in_channels": 5, "base_channels": 16, "n_classes": 3},
}


class DaughterNet(torch.nn.Module):
    """DivNet UNet-GAP body (same layer names as DivNetMitosisClassifier) with an n_classes head."""

    def __init__(self, in_channels: int = 5, base_channels: int = 16, n_classes: int = 3) -> None:
        super().__init__()
        c = int(base_channels)
        self.enc1 = _DivNetConvBlock(in_channels, c)
        self.pool1 = torch.nn.MaxPool3d(2, 2)
        self.enc2 = _DivNetConvBlock(c, c * 2)
        self.pool2 = torch.nn.MaxPool3d(2, 2)
        self.enc3 = _DivNetConvBlock(c * 2, c * 4)
        self.pool3 = torch.nn.MaxPool3d(2, 2)
        self.bottleneck = _DivNetConvBlock(c * 4, c * 8)
        self.up3 = torch.nn.ConvTranspose3d(c * 8, c * 4, 2, 2)
        self.dec3 = _DivNetConvBlock(c * 8, c * 4)
        self.up2 = torch.nn.ConvTranspose3d(c * 4, c * 2, 2, 2)
        self.dec2 = _DivNetConvBlock(c * 4, c * 2)
        self.up1 = torch.nn.ConvTranspose3d(c * 2, c, 2, 2)
        self.dec1 = _DivNetConvBlock(c * 2, c)
        self.gap = torch.nn.AdaptiveAvgPool3d((1, 1, 1))
        self.head = torch.nn.Linear(c, int(n_classes))

    def forward(self, value: torch.Tensor) -> torch.Tensor:
        e1 = self.enc1(value)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b = self.bottleneck(self.pool3(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(self.gap(d1).flatten(1))


def build_model(cfg: dict | None = None) -> DaughterNet:
    m = (cfg or DEFAULT_CFG)["model"]
    return DaughterNet(int(m["in_channels"]), int(m["base_channels"]), int(m["n_classes"]))


def load_daughternet(paths: Sequence[Path | str], device: str | None = None) -> dict:
    """Load one or more fold checkpoints ({'model_state', 'config'}) -> bundle usable by prepare_sample/score_nodes."""
    dev = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    models, cfg = [], None
    for p in paths:
        payload = torch.load(Path(p), map_location="cpu", weights_only=False)
        cfg = payload.get("config", DEFAULT_CFG) if isinstance(payload, dict) else DEFAULT_CFG
        model = build_model(cfg)
        model.load_state_dict(payload["model_state"] if "model_state" in payload else payload, strict=True)
        models.append(model.to(dev).eval())
    if not models:
        raise FileNotFoundError("no DaughterNet checkpoints given")
    return {"models": models, "device": dev, "torch": torch, "config": cfg, "prepared_frames": {}}


def prepare_sample(dataset: str, t: int, node: dict, bundle: dict, frame_cache: dict,
                   read_frame: Callable[[str, int, dict], np.ndarray] | None = None) -> np.ndarray:
    """(5, 16, 32, 32) float32 sample, identical to the DivNet contract (bundle['config'] carries input/normalization)."""
    return prepare_divnet_input(dataset, int(t), node, bundle, frame_cache, read_frame)


@torch.no_grad()
def score_nodes(dataset: str, nodes: Sequence[dict], bundle: dict, frame_cache: dict,
                read_frame: Callable[[str, int, dict], np.ndarray] | None = None, batch: int = 64,
                model_index: int | None = None) -> np.ndarray:
    """Softmax probabilities (N, 3) in CLASSES order; mean over fold models unless model_index selects one (OOF)."""
    if not nodes:
        return np.zeros((0, 3), dtype=np.float32)
    order = sorted(range(len(nodes)), key=lambda i: int(nodes[i]["t"]))
    out = np.zeros((len(nodes), 3), dtype=np.float32)
    models = bundle["models"] if model_index is None else [bundle["models"][model_index]]
    dev = bundle["device"]
    for s in range(0, len(order), batch):
        idx = order[s:s + batch]
        x = np.stack([prepare_sample(dataset, int(nodes[i]["t"]), nodes[i], bundle, frame_cache, read_frame) for i in idx])
        xt = torch.from_numpy(x).to(dev, dtype=torch.float32)
        p = torch.stack([torch.softmax(m(xt), dim=1) for m in models]).mean(0)
        out[idx] = p.float().cpu().numpy()
    return out


In [ ]:
%%writefile /kaggle/working/kdn_src/kdn_core.py
"""K-DN: train DaughterNet (tools/daughternet.py) from scratch with honest video folds and write out-of-fold predictions.

One module shared by the Kaggle kernel (kdn-train.ipynb materialises it with %%writefile next to tools/divnet_infer.py
and tools/daughternet.py) and by the laptop runner run_local.py. Steps (run_all runs them in order):

  step_prepare      locate the data, folds (embedded splits.json, else the inline seed-0 rule), GT labels per video
  step_build_cache  DivNet-contract crops for every labelled node -> one memmap (N, 4, 18, 34, 34) + meta.csv
  step_train        one subprocess per fold (two at once on 2 GPUs): balanced sampler, flips/rot90/jitter/gain,
                    AdamW + cosine, early stopping on the fold's val loss, best.pt/last.pt every epoch, fold OOF
  step_oof          oof.csv + oof_readout.json (+ in-sample DivNet comparison)

Labels (annotated GT nodes only, never unannotated space):
  parent   node with >= 2 GT children (at its own t)
  daughter each child of a parent (at its own, first, frame)
  cont     in-degree 1, out-degree 1, >= cont_min_dist frames (lineage) from every parent AND daughter node,
           random subsample of cont_per_video per video
  hardneg  cont nodes that sit in the L2 insertion gate of an annotated one-child GT node, as u or as v
           (u: exactly one child c1 at t+1 within 10 um; v at t+1, v != c1, within 15 um of u; 3 <= |c1-v| <= 20 um;
           |midpoint(c1, v) - u| <= 6.5 um; cos(c1-u, v-u) <= 0.25), up to hardneg_per_video, label cont
Extra samples (v2, env KDN_EXTRA_SAMPLES = parquet/csv of tools/make_pool_samples.py): candidate nodes of the REAL
predicted pool labelled by GT matching (parent / daughter / cont), at their PREDICTED coordinates. They join their
video's sample list with source=pool (GT rows: source=gt), kind pool_<label>, fold = the video's fold (OOF routing and
the 44b6 leak groups unchanged), and are cropped exactly like GT samples. A pool row at the same stem and t as a GT
sample and within KDN_DEDUP_VOX (1) voxel of it is the same node: the GT row is kept, the pool row dropped. The sampler
stays class-balanced (a class is drawn uniformly, then a row of that class, gt or pool); KDN_POOL_WEIGHT (1.0) scales
the loss of pool rows (batch loss = sum(w * CE) / sum(w)). GT rows keep their v1 row order and indices (pool rows are
appended), so a run without KDN_EXTRA_SAMPLES is v1. The OOF readout reports source=gt and source=pool separately.
Crops: prepare_sample (DivNet contract) called with an enlarged 18x34x34 crop; the inner 16x32x32 of it equals the
contract sample exactly (asserted on the first nodes of every video, per source), the 1-voxel margin feeds the +-1
voxel jitter.
Node coordinates are voxel indices of the full-resolution (64, 256, 256) frame, exactly as in the GT .geff.
Frame index t+lag is clamped to [0, T-1] (the contract clamps the low end with max(0, .); the high end is clamped here,
as in friend A's FrameReader, so nodes at t >= T-2 get a repeated last frame).
"""
from __future__ import annotations

import copy
import json
import math
import os
import struct
import subprocess
import sys
import threading
import time
import zlib
from pathlib import Path

import numpy as np

try:  # repo layout (laptop) and the Kaggle kernel (tools/ materialised under kdn_src)
    from tools.daughternet import CLASSES, DEFAULT_CFG, build_model, load_daughternet, prepare_sample, score_nodes
    from tools.divnet_infer import _marker, load_divnet_mitosis_model, _checkpoint_candidates
except Exception:  # pragma: no cover - notebook fallback when the cells were executed into globals
    CLASSES = globals()["CLASSES"]
    DEFAULT_CFG = globals()["DEFAULT_CFG"]
    build_model = globals()["build_model"]
    load_daughternet = globals()["load_daughternet"]
    prepare_sample = globals()["prepare_sample"]
    score_nodes = globals()["score_nodes"]
    _marker = globals()["_marker"]
    load_divnet_mitosis_model = globals()["load_divnet_mitosis_model"]
    _checkpoint_candidates = globals().get("_checkpoint_candidates", lambda: [])

COMPETITION = "biohub-cell-tracking-during-development"
VOXEL_SCALE_UM = np.array([1.625, 0.40625, 0.40625], dtype=np.float64)
CLASS_INDEX = {name: i for i, name in enumerate(CLASSES)}
CROP = (16, 32, 32)          # DivNet contract crop (z, y, x) in pooled voxels
MARGIN = 1                   # +-1 voxel jitter margin stored around it
BIG = (CROP[0] + 2 * MARGIN, CROP[1] + 2 * MARGIN, CROP[2] + 2 * MARGIN)
OOF_COLUMNS = ["stem", "t", "z", "y", "x", "label", "fold", "p_cont", "p_par", "p_dau"]


def log(*args) -> None:
    print(time.strftime("%H:%M:%S"), *args, flush=True)


# ----------------------------------------------------------------------------------------------------------------------
# configuration
# ----------------------------------------------------------------------------------------------------------------------
def _env_flag(name: str, default: str = "0") -> bool:
    return os.environ.get(name, default).strip() not in ("", "0", "false", "False", "no")


def config_from_env(overrides: dict | None = None) -> dict:
    env = os.environ.get
    smoke = _env_flag("KDN_SMOKE")
    on_kaggle = Path("/kaggle/working").exists()
    cfg = {
        "smoke": smoke,
        "data_dir": env("KDN_DATA_DIR", ""),
        "work_dir": env("KDN_WORK_DIR", "/kaggle/working" if on_kaggle else str(Path.cwd() / "kdn_out")),
        "cache_dir": env("KDN_CACHE_DIR", ""),
        "splits_path": env("KDN_SPLITS", ""),
        "stems": [s.strip() for s in env("KDN_STEMS", "").split(",") if s.strip()],
        "max_videos": int(env("KDN_MAX_VIDEOS", "0") or 0),
        "smoke_videos": int(env("KDN_SMOKE_VIDEOS", "5")),
        "n_folds": int(env("KDN_FOLDS", "2" if smoke else "5")),
        "epochs": int(env("KDN_EPOCHS", "1" if smoke else "30")),
        "patience": int(env("KDN_PATIENCE", "6")),
        "batch": int(env("KDN_BATCH", "64")),
        "lr": float(env("KDN_LR", "1e-3")),
        "weight_decay": float(env("KDN_WD", "0.01")),
        "warmup_frac": float(env("KDN_WARMUP_FRAC", "0.03")),
        "epoch_samples": int(env("KDN_EPOCH_SAMPLES", "0")),   # 0 = number of training samples
        "eval_batch": int(env("KDN_EVAL_BATCH", "256")),
        "cont_per_video": int(env("KDN_CONT_PER_VIDEO", "60")),
        "hardneg_per_video": int(env("KDN_HARDNEG_PER_VIDEO", "40")),
        "cont_min_dist": int(env("KDN_CONT_MIN_DIST", "2")),
        "seed": int(env("KDN_SEED", "0")),
        "build_workers": int(env("KDN_BUILD_WORKERS", "0")),  # 0 = min(4, cpu count)
        "contract_checks": int(env("KDN_CONTRACT_CHECKS", "2")),  # exact contract checks per video
        "cache_dtype": env("KDN_CACHE_DTYPE", "float32"),
        "jitter": int(env("KDN_JITTER", "1")),
        "gain": [float(env("KDN_GAIN_LO", "0.8")), float(env("KDN_GAIN_HI", "1.2"))],
        "amp": _env_flag("KDN_AMP", "1"),
        "gpus": int(env("KDN_GPUS", "-1")),                   # -1 = all visible, 0 = CPU
        "inner_val": _env_flag("KDN_INNER_VAL"),              # 1 = early-stop on fold k+1, train on the other K-2
        "divnet_compare": _env_flag("KDN_DIVNET", "1"),
        "divnet_checkpoint": env("KDN_DIVNET_CKPT", ""),
        "keep_cache": _env_flag("KDN_KEEP_CACHE", "1"),
        "extra_samples": env("KDN_EXTRA_SAMPLES", ""),        # pool rows (tools/make_pool_samples.py), "" = off
        "pool_weight": float(env("KDN_POOL_WEIGHT", "1.0")),  # loss weight of source=pool rows
        "dedup_vox": float(env("KDN_DEDUP_VOX", "1.0")),      # pool row within this many voxels of a GT sample = same node
        "es_source": env("KDN_ES_SOURCE", "all"),             # early-stop loss over all / gt / pool rows of the val fold
        "gate": {"child_um": 10.0, "v_um": 15.0, "sister_min_um": 3.0, "sister_max_um": 20.0,
                 "mid_um": 6.5, "cos_max": 0.25},
    }
    if overrides:
        cfg.update(overrides)
    return cfg


def resolve_data_dir(cfg: dict) -> Path:
    if cfg.get("data_dir"):
        return Path(cfg["data_dir"])
    candidates = [Path(f"/kaggle/input/competitions/{COMPETITION}"), Path(f"/kaggle/input/{COMPETITION}")]
    comp_dir = next((p for p in candidates if p.exists()), candidates[0])
    return comp_dir / "train"


def resolve_cache_dir(cfg: dict) -> Path:
    if cfg.get("cache_dir"):
        return Path(cfg["cache_dir"])
    for base in (Path("/kaggle/temp"), Path("/tmp")):
        if base.exists() and os.access(base, os.W_OK):
            return base / "kdn_cache"
    return Path(cfg["work_dir"]) / "_kdn_cache"


# ----------------------------------------------------------------------------------------------------------------------
# codecs: zstd (GT .geff arrays) and blosc (image chunks); several decoders so the kernel needs no extra wheels
# ----------------------------------------------------------------------------------------------------------------------
_ZSTD_FN = None


def _zstd_candidates():
    def via_zstandard(buf, size):
        import zstandard
        return zstandard.ZstdDecompressor().decompress(buf, max_output_size=max(size, 1))

    def via_numcodecs(buf, size):
        from numcodecs import Zstd
        return bytes(Zstd().decode(buf))

    def via_pyarrow(buf, size):
        import pyarrow as pa
        return pa.Codec("zstd").decompress(buf, decompressed_size=size, asbytes=True)

    def via_imagecodecs(buf, size):
        import imagecodecs
        return bytes(imagecodecs.zstd_decode(buf))

    return [via_zstandard, via_numcodecs, via_pyarrow, via_imagecodecs]


def zstd_decompress(buf: bytes, size: int) -> bytes:
    global _ZSTD_FN
    if _ZSTD_FN is not None:
        return _ZSTD_FN(buf, size)
    errors = []
    for fn in _zstd_candidates():
        try:
            out = fn(buf, size)
            if len(out) != size:
                raise ValueError(f"size {len(out)} != {size}")
            _ZSTD_FN = fn
            return out
        except Exception as exc:  # try the next decoder
            errors.append(f"{fn.__name__}: {type(exc).__name__}: {exc}")
    raise RuntimeError("no zstd decoder works: " + " | ".join(errors))


def _bitunshuffle(block: bytes, typesize: int) -> bytes:
    n = len(block) // typesize
    n8 = n - (n % 8)
    if n8 == 0:
        return block
    head = np.frombuffer(block, dtype=np.uint8, count=n8 * typesize).reshape(typesize * 8, n8 // 8)
    bits = np.unpackbits(head, axis=1, bitorder="little").reshape(typesize, 8, n8)
    elems = np.packbits(bits, axis=1, bitorder="little").reshape(typesize, n8)
    return elems.T.tobytes() + block[n8 * typesize:]


def blosc1_decompress_pure(raw: bytes) -> bytes:
    """Blosc-1 frame decoder (numpy + a zstd decoder); last-resort fallback, tested against blosc2 locally."""
    flags, typesize = raw[2], raw[3]
    nbytes, blocksize, _cbytes = struct.unpack("<iii", raw[4:16])
    if flags & 0x2:  # memcpyed
        return bytes(raw[16:16 + nbytes])
    compcode = flags >> 5
    if compcode != 4:
        raise RuntimeError(f"pure blosc decoder supports zstd only (compcode {compcode})")
    nblocks = (nbytes + blocksize - 1) // blocksize
    bstarts = struct.unpack(f"<{nblocks}i", raw[16:16 + 4 * nblocks])
    out = bytearray(nbytes)
    dont_split = bool(flags & 0x10)
    for i, start in enumerate(bstarts):
        bsize = min(blocksize, nbytes - i * blocksize)
        nstreams = 1 if (dont_split or typesize <= 1 or bsize < typesize) else typesize
        neb = bsize // nstreams
        pos, parts = start, []
        for _ in range(nstreams):
            csize = struct.unpack("<i", raw[pos:pos + 4])[0]
            pos += 4
            chunk = raw[pos:pos + csize]
            pos += csize
            parts.append(bytes(chunk) if csize == neb else zstd_decompress(bytes(chunk), neb))
        block = b"".join(parts)
        if flags & 0x4:
            block = _bitunshuffle(block, typesize)
        elif (flags & 0x1) and typesize > 1:
            n = bsize // typesize
            block = np.frombuffer(block, np.uint8, count=n * typesize).reshape(typesize, n).T.tobytes() + block[n * typesize:]
        out[i * blocksize:i * blocksize + bsize] = block
    return bytes(out)


_BLOSC_FN = None


def _blosc_candidates():
    def via_blosc2(raw):
        import blosc2
        return blosc2.decompress(raw)

    def via_numcodecs(raw):
        from numcodecs import Blosc
        return bytes(Blosc().decode(raw))

    def via_blosc(raw):
        import blosc
        return blosc.decompress(raw)

    return [via_blosc2, via_numcodecs, via_blosc, blosc1_decompress_pure]


def blosc_decompress(raw: bytes) -> bytes:
    global _BLOSC_FN
    nbytes = struct.unpack("<i", raw[4:8])[0]
    if _BLOSC_FN is not None:
        return _BLOSC_FN(raw)
    errors = []
    for fn in _blosc_candidates():
        try:
            out = fn(raw)
            if len(out) != nbytes:
                raise ValueError(f"size {len(out)} != {nbytes}")
            _BLOSC_FN = fn
            return out
        except Exception as exc:
            errors.append(f"{fn.__name__}: {type(exc).__name__}: {exc}")
    raise RuntimeError("no blosc decoder works: " + " | ".join(errors))


def decoder_names() -> dict:
    return {"zstd": getattr(_ZSTD_FN, "__name__", None), "blosc": getattr(_BLOSC_FN, "__name__", None)}


def _zarr_dtype(meta: dict) -> np.dtype:
    dtype = np.dtype(meta["data_type"])
    for codec in meta.get("codecs", []):
        if codec.get("name") == "bytes":
            endian = (codec.get("configuration") or {}).get("endian", "little")
            dtype = dtype.newbyteorder("<" if endian == "little" else ">")
    return dtype


def _decode_chunk(raw: bytes, meta: dict, chunk_shape: tuple) -> np.ndarray:
    dtype = _zarr_dtype(meta)
    size = int(np.prod(chunk_shape)) * dtype.itemsize
    data = raw
    codecs = meta.get("codecs", [])
    for codec in reversed(codecs):
        name = codec.get("name")
        if name == "bytes":
            continue
        if name == "zstd":
            data = zstd_decompress(data, size)
        elif name == "blosc":
            data = blosc_decompress(data)
        elif name == "gzip":
            data = zlib.decompress(data, 16 + zlib.MAX_WBITS)
        else:
            raise RuntimeError(f"unsupported zarr codec {name}")
    return np.frombuffer(data, dtype=dtype, count=int(np.prod(chunk_shape))).reshape(chunk_shape)


def read_zarr_array(path: Path) -> np.ndarray:
    """Whole zarr v3 array with a regular chunk grid, decoded with numpy (+ zstd/blosc); zarr is the fallback."""
    path = Path(path)
    try:
        meta = json.loads((path / "zarr.json").read_text())
        shape = tuple(int(v) for v in meta["shape"])
        chunks = tuple(int(v) for v in meta["chunk_grid"]["configuration"]["chunk_shape"])
        sep = ((meta.get("chunk_key_encoding") or {}).get("configuration") or {}).get("separator", "/")
        dtype = _zarr_dtype(meta)
        out = np.full(shape, meta.get("fill_value", 0) or 0, dtype=dtype.newbyteorder("="))
        grid = [range(math.ceil(s / c)) if s else range(0) for s, c in zip(shape, chunks)]
        for idx in np.ndindex(*[len(g) for g in grid]):
            key = sep.join(str(i) for i in idx)
            chunk_path = path / "c" / key if sep == "/" else path / f"c{sep}{key}"
            if not chunk_path.exists():
                continue
            chunk = _decode_chunk(chunk_path.read_bytes(), meta, chunks)
            sl = tuple(slice(i * c, min((i + 1) * c, s)) for i, c, s in zip(idx, chunks, shape))
            out[sl] = chunk[tuple(slice(0, x.stop - x.start) for x in sl)]
        return out
    except Exception as first:
        try:
            import zarr
            return np.asarray(zarr.open(str(path), mode="r")[...])
        except Exception:
            raise first


def read_gt_geff(path: Path) -> dict:
    """GT graph arrays: ids, t, z, y, x (voxel units of the full-resolution frame) and edges (E, 2) source->target."""
    path = Path(path)
    ids = read_zarr_array(path / "nodes" / "ids").astype(np.int64)
    props = path / "nodes" / "props"
    cols = {k: read_zarr_array(props / k / "values") for k in ("t", "z", "y", "x")}
    edges_path = path / "edges" / "ids"
    edges = read_zarr_array(edges_path).astype(np.int64).reshape(-1, 2) if edges_path.exists() else np.zeros((0, 2), np.int64)
    return {"ids": ids, "t": cols["t"].astype(np.int64), "z": cols["z"].astype(np.float64),
            "y": cols["y"].astype(np.float64), "x": cols["x"].astype(np.float64), "edges": edges}


class FrameReader:
    """read_test_frame of the exp39 notebook, parameterised by directory: one blosc chunk per frame, zarr fallback.

    Callable as read_frame(dataset, t, frame_cache); t is clamped to [0, T-1]; the frame is cached under the requested
    t so prepare_divnet_input's frame_cache.pop(frame_t) releases it."""

    def __init__(self, data_dir: Path) -> None:
        self.data_dir = Path(data_dir)
        self.meta: dict[str, tuple] = {}
        self.frames_read = 0

    def _meta(self, dataset: str) -> tuple:
        if dataset not in self.meta:
            meta = json.loads((self.data_dir / f"{dataset}.zarr" / "0" / "zarr.json").read_text())
            shape = tuple(int(v) for v in meta["shape"])
            chunks = tuple(int(v) for v in meta["chunk_grid"]["configuration"]["chunk_shape"])
            self.meta[dataset] = (meta, shape, chunks)
        return self.meta[dataset]

    def __call__(self, dataset: str, t: int, frame_cache: dict) -> np.ndarray:
        if t in frame_cache:
            return frame_cache[t]
        meta, shape, chunks = self._meta(dataset)
        index = max(0, min(int(t), shape[0] - 1))
        zarr_path = self.data_dir / f"{dataset}.zarr"
        frame = None
        if chunks[0] == 1 and tuple(chunks[1:]) == tuple(shape[1:]):
            try:
                raw = (zarr_path / "0" / "c" / str(index) / "0" / "0" / "0").read_bytes()
                frame = _decode_chunk(raw, meta, (1, *shape[1:]))[0].copy()
            except Exception:
                frame = None
        if frame is None:
            import zarr
            frame = np.asarray(zarr.open(str(zarr_path / "0"), mode="r")[index])
        self.frames_read += 1
        frame_cache[t] = frame
        return frame


# ----------------------------------------------------------------------------------------------------------------------
# labels
# ----------------------------------------------------------------------------------------------------------------------
def _stem_seed(stem: str, seed: int) -> list[int]:
    return [int(seed), zlib.crc32(stem.encode("utf-8"))]


def extract_labels(stem: str, geff_path: Path, cfg: dict) -> tuple[list[dict], dict]:
    g = read_gt_geff(geff_path)
    ids = g["ids"]
    n = len(ids)
    pos_of = {int(v): i for i, v in enumerate(ids)}
    t = g["t"]
    pts_um = np.stack([g["z"], g["y"], g["x"]], axis=1) * VOXEL_SCALE_UM
    children: list[list[int]] = [[] for _ in range(n)]
    parents: list[list[int]] = [[] for _ in range(n)]
    dropped_edges = 0
    for s, d in g["edges"]:
        a, b = pos_of.get(int(s)), pos_of.get(int(d))
        if a is None or b is None:
            dropped_edges += 1
            continue
        children[a].append(b)
        parents[b].append(a)
    outdeg = np.array([len(c) for c in children])
    indeg = np.array([len(p) for p in parents])
    is_parent = outdeg >= 2
    is_daughter = np.zeros(n, bool)
    for i in np.flatnonzero(is_parent):
        is_daughter[children[i]] = True
    ambiguous = is_parent & is_daughter
    division_node = is_parent | is_daughter

    # lineage distance (frames) to the nearest parent or daughter node among ancestors and descendants
    order = np.argsort(t, kind="stable")
    up = np.full(n, -np.inf)
    for i in order:
        if division_node[i]:
            up[i] = t[i]
        elif parents[i]:
            up[i] = max(up[p] for p in parents[i])
    down = np.full(n, np.inf)
    for i in order[::-1]:
        if division_node[i]:
            down[i] = t[i]
        elif children[i]:
            down[i] = min(down[c] for c in children[i])
    dist = np.minimum(t - up, down - t)
    cont_ok = (indeg == 1) & (outdeg == 1) & ~division_node & (dist >= int(cfg["cont_min_dist"]))

    # hard negatives: cont nodes in the insertion gate of an annotated one-child node (as u or as v)
    gate = cfg["gate"]
    by_t: dict[int, np.ndarray] = {}
    for tt in np.unique(t):
        by_t[int(tt)] = np.flatnonzero(t == tt)
    hard_role: dict[int, set] = {}
    gate_triples = 0
    for u in np.flatnonzero(outdeg == 1):
        c1 = children[u][0]
        if t[c1] != t[u] + 1:
            continue
        vec_c = pts_um[c1] - pts_um[u]
        n_c = float(np.linalg.norm(vec_c))
        if n_c > gate["child_um"]:
            continue
        cand = by_t.get(int(t[u]) + 1)
        if cand is None:
            continue
        cand = cand[cand != c1]
        vec_v = pts_um[cand] - pts_um[u]
        n_v = np.linalg.norm(vec_v, axis=1)
        sister = np.linalg.norm(pts_um[cand] - pts_um[c1], axis=1)
        mid = np.linalg.norm(0.5 * (pts_um[cand] + pts_um[c1]) - pts_um[u], axis=1)
        cos = (vec_v @ vec_c) / np.maximum(n_c * n_v, 1e-9)  # zero-length vector -> cos 0 (passes)
        ok = ((n_v <= gate["v_um"]) & (sister >= gate["sister_min_um"]) & (sister <= gate["sister_max_um"])
              & (mid <= gate["mid_um"]) & (cos <= gate["cos_max"]))
        for v in cand[ok]:
            gate_triples += 1
            if cont_ok[u]:
                hard_role.setdefault(int(u), set()).add("u")
            if cont_ok[v]:
                hard_role.setdefault(int(v), set()).add("v")

    rng = np.random.default_rng(_stem_seed(stem, cfg["seed"]))
    hard_nodes = sorted(hard_role)
    if len(hard_nodes) > cfg["hardneg_per_video"]:
        hard_nodes = sorted(rng.choice(hard_nodes, size=cfg["hardneg_per_video"], replace=False).tolist())
    hard_set = set(hard_nodes)
    cont_pool = [i for i in np.flatnonzero(cont_ok) if int(i) not in hard_set]
    if len(cont_pool) > cfg["cont_per_video"]:
        cont_pool = sorted(rng.choice(cont_pool, size=cfg["cont_per_video"], replace=False).tolist())

    rows: list[dict] = []

    def add(i: int, label: str, kind: str) -> None:
        rows.append({"stem": stem, "node_id": int(ids[i]), "t": int(t[i]), "z": float(g["z"][i]),
                     "y": float(g["y"][i]), "x": float(g["x"][i]), "label": label, "kind": kind})

    for i in np.flatnonzero(is_parent & ~ambiguous):
        add(int(i), "parent", "parent")
    for i in np.flatnonzero(is_daughter & ~ambiguous):
        add(int(i), "daughter", "daughter")
    for i in cont_pool:
        add(int(i), "cont", "cont")
    for i in hard_nodes:
        add(int(i), "cont", "hardneg_" + "".join(sorted(hard_role[i])))
    stats = {
        "stem": stem, "embryo": stem[:4], "gt_nodes": int(n), "gt_edges": int(len(g["edges"])),
        "dropped_edges": int(dropped_edges), "divisions": int(is_parent.sum()),
        "parent": int((is_parent & ~ambiguous).sum()), "daughter": int((is_daughter & ~ambiguous).sum()),
        "ambiguous_dropped": int(ambiguous.sum()), "cont_eligible": int(cont_ok.sum()),
        "cont": len(cont_pool), "gate_triples": int(gate_triples), "hardneg_candidates": len(hard_role),
        "hardneg": len(hard_nodes),
        "hardneg_u": sum(1 for i in hard_nodes if "u" in hard_role[i]),
        "hardneg_v": sum(1 for i in hard_nodes if "v" in hard_role[i]),
        "edge_dt_not_1": int(sum(1 for a in range(n) for b in children[a] if t[b] - t[a] != 1)),
    }
    return rows, stats


# ----------------------------------------------------------------------------------------------------------------------
# folds
# ----------------------------------------------------------------------------------------------------------------------
def make_splits_inline(n_div: dict[str, int], n_folds: int = 5, seed: int = 0) -> dict[str, int]:
    """Fallback only (used when no splits.json is available): 5 folds grouped by video, stratified by embryo prefix
    and division count, seed 0. Per embryo: videos in seeded random order, stable-sorted by division count (desc);
    division videos go greedily to the fold with the fewest divisions (then fewest videos), zero-division videos to
    the fold with the fewest videos of that embryo."""
    rng = np.random.default_rng(seed)
    folds: dict[str, int] = {}
    for emb in sorted({s[:4] for s in n_div}):
        vids = sorted(s for s in n_div if s[:4] == emb)
        vids = [vids[i] for i in rng.permutation(len(vids))]
        vids.sort(key=lambda s: -n_div[s])
        div_sum, cnt = [0] * n_folds, [0] * n_folds
        for s in vids:
            if n_div[s] > 0:
                f = min(range(n_folds), key=lambda k: (div_sum[k], cnt[k], k))
            else:
                f = min(range(n_folds), key=lambda k: (cnt[k], k))
            folds[s] = f
            div_sum[f] += n_div[s]
            cnt[f] += 1
    return folds


def load_splits(path: Path) -> dict[str, int] | None:
    if not path or not Path(path).is_file():
        return None
    d = json.loads(Path(path).read_text())
    if isinstance(d, dict) and isinstance(d.get("folds"), dict):
        return {str(k): int(v) for k, v in d["folds"].items()}
    if isinstance(d, dict) and isinstance(d.get("folds"), list):
        return {str(s): i for i, group in enumerate(d["folds"]) for s in group}
    if isinstance(d, dict) and all(isinstance(v, int) for v in d.values()):
        return {str(k): int(v) for k, v in d.items()}
    if isinstance(d, list):
        return {str(r["stem"]): int(r["fold"]) for r in d}
    raise ValueError(f"unrecognised splits format in {path}")


def _default_splits_path() -> Path | None:
    here = Path(__file__).resolve().parent
    for p in (here / "splits.json", here.parent / "splits.json"):
        if p.is_file():
            return p
    return None


# ----------------------------------------------------------------------------------------------------------------------
# extra samples (v2): labelled nodes of the real candidate pool at their predicted coordinates
# ----------------------------------------------------------------------------------------------------------------------
EXTRA_REQUIRED = ["stem", "node_id", "t", "z", "y", "x", "label"]
EXTRA_OPTIONAL = ["dist_um", "gt_id", "n_triples", "n_u", "n_c1", "n_v"]


def sha256_file(path: Path) -> str:
    import hashlib
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def load_extra_samples(path: Path):
    """tools/make_pool_samples.py output (.parquet, or the .csv text copy the notebook embeds; coordinates are read
    back bit-exactly with float_precision='round_trip')."""
    import pandas as pd

    path = Path(path)
    if path.suffix.lower() in (".parquet", ".pq"):
        df = pd.read_parquet(path)
    else:
        df = pd.read_csv(path, dtype={"stem": str, "label": str}, float_precision="round_trip")
    missing = [c for c in EXTRA_REQUIRED if c not in df.columns]
    if missing:
        raise ValueError(f"extra samples {path}: missing columns {missing}")
    df = df[EXTRA_REQUIRED + [c for c in EXTRA_OPTIONAL if c in df.columns]].copy()
    df["stem"] = df["stem"].astype(str)
    df["label"] = df["label"].astype(str)
    df["node_id"] = df["node_id"].astype(np.int64)
    df["t"] = df["t"].astype(np.int64)
    for col in ("z", "y", "x"):
        df[col] = df[col].astype(np.float64)
    dup = int(df.duplicated(["stem", "node_id"]).sum())
    if dup:
        raise ValueError(f"extra samples {path}: {dup} duplicated (stem, node_id) rows")
    return df


def merge_extra_samples(meta, extra, folds: dict, dedup_vox: float):
    """GT sample table (v1 order, indices 0..n_gt-1) + pool rows appended (sorted by stem, t, node_id).

    Pool rows of videos outside the selection or with an unknown label are dropped; a pool row at the same stem and t
    as a GT sample and within dedup_vox voxels (euclidean, full-resolution voxel units) of it is dropped (GT label
    kept). fold = the video's fold (the same map the GT rows use). Returns (meta, info, per-video counts)."""
    import pandas as pd

    info = {"rows_file": int(len(extra))}
    ex = extra[extra["stem"].isin(set(folds))].copy()
    info["rows_other_videos"] = int(len(extra) - len(ex))
    bad = ~ex["label"].isin(set(CLASS_INDEX))
    info["rows_unknown_label"] = int(bad.sum())
    ex = ex[~bad].reset_index(drop=True)
    ex["_eid"] = np.arange(len(ex))
    gt = meta[["stem", "t", "z", "y", "x", "label"]].rename(
        columns={"z": "z_gt", "y": "y_gt", "x": "x_gt", "label": "label_gt"})
    pairs = ex[["_eid", "stem", "t", "z", "y", "x", "label"]].merge(gt, on=["stem", "t"], how="inner")
    d = np.sqrt((pairs["z"] - pairs["z_gt"]) ** 2 + (pairs["y"] - pairs["y_gt"]) ** 2 + (pairs["x"] - pairs["x_gt"]) ** 2)
    close = pairs[d.to_numpy() <= float(dedup_vox)]
    dup_ids = set(close["_eid"].tolist())
    same = close.drop_duplicates("_eid")
    info["dedup_dropped"] = len(dup_ids)
    info["dedup_same_label"] = int((same["label"] == same["label_gt"]).sum())
    info["dedup_dropped_by_label"] = {k: int(v) for k, v in same["label"].value_counts().items()}
    ex = ex[~ex["_eid"].isin(dup_ids)].drop(columns="_eid")
    ex["kind"] = "pool_" + ex["label"]
    ex["label_idx"] = ex["label"].map(CLASS_INDEX).astype(int)
    ex["fold"] = ex["stem"].map(folds).astype(int)
    ex["embryo"] = ex["stem"].str[:4]
    ex["source"] = "pool"
    ex = ex.sort_values(["stem", "t", "node_id"]).reset_index(drop=True)
    meta = meta.copy()
    meta["source"] = "gt"
    if "gt_id" in ex.columns:
        meta["gt_id"] = meta["node_id"].astype(np.int64)   # a GT sample IS its GT node
    if "dist_um" in ex.columns:
        meta["dist_um"] = 0.0
    out = pd.concat([meta, ex[[c for c in ex.columns if c in meta.columns or c in EXTRA_OPTIONAL]]],
                    ignore_index=True)
    if "gt_id" in out.columns:
        out["gt_id"] = out["gt_id"].astype(np.int64)
    info["rows_used"] = int(len(ex))
    info["by_label"] = {k: int(v) for k, v in ex["label"].value_counts().items()}
    per_video = {s: {f"pool_{k}": int(v) for k, v in g["label"].value_counts().items()}
                 for s, g in ex.groupby("stem")}
    for stem, n in same["stem"].value_counts().items():
        per_video.setdefault(stem, {})["pool_dedup_dropped"] = int(n)
    return out, info, per_video


# ----------------------------------------------------------------------------------------------------------------------
# step 1: prepare
# ----------------------------------------------------------------------------------------------------------------------
def step_prepare(cfg: dict) -> dict:
    import pandas as pd

    t0 = time.time()
    if cfg.get("es_source", "all") not in ("all", "gt", "pool"):
        raise ValueError(f"KDN_ES_SOURCE must be all, gt or pool (got {cfg.get('es_source')!r})")
    if cfg.get("es_source", "all") == "pool" and not cfg.get("extra_samples"):
        raise ValueError("KDN_ES_SOURCE=pool needs KDN_EXTRA_SAMPLES")
    if not float(cfg.get("pool_weight", 1.0)) > 0:
        raise ValueError(f"KDN_POOL_WEIGHT must be > 0 (got {cfg.get('pool_weight')})")
    data_dir = resolve_data_dir(cfg)
    work = Path(cfg["work_dir"])
    work.mkdir(parents=True, exist_ok=True)
    cache = resolve_cache_dir(cfg)
    cache.mkdir(parents=True, exist_ok=True)
    all_stems = sorted(p.name[:-5] for p in data_dir.iterdir() if p.name.endswith(".geff")
                       and (data_dir / f"{p.name[:-5]}.zarr").exists())
    log(f"[prepare] data_dir={data_dir} videos with .zarr+.geff: {len(all_stems)}; cache={cache}; work={work}")
    if not all_stems:
        raise FileNotFoundError(f"no <stem>.zarr + <stem>.geff pairs under {data_dir}")

    # GT labels for every video (cheap: the .geff files are tiny); per-video try/except
    label_rows, stats_rows, n_div = [], [], {}
    for i, stem in enumerate(all_stems):
        ts = time.time()
        try:
            rows, st = extract_labels(stem, data_dir / f"{stem}.geff", cfg)
            st["label_secs"] = round(time.time() - ts, 3)
            label_rows.append(rows)
            stats_rows.append(st)
            n_div[stem] = st["divisions"]
        except Exception as exc:
            log(f"[prepare] {stem}: label extraction FAILED {type(exc).__name__}: {exc}")
            stats_rows.append({"stem": stem, "embryo": stem[:4], "error": f"{type(exc).__name__}: {exc}"})
            label_rows.append([])
            n_div[stem] = 0
    log(f"[prepare] GT parsed for {len(all_stems)} videos in {time.time() - t0:.1f} s; "
        f"divisions {sum(n_div.values())} in {sum(1 for v in n_div.values() if v)} videos; zstd decoder {decoder_names()['zstd']}")

    # folds
    splits_path = Path(cfg["splits_path"]) if cfg.get("splits_path") else _default_splits_path()
    splits = load_splits(splits_path) if splits_path else None
    splits_source = str(splits_path) if splits else "inline make_splits_inline(seed=0)"
    if splits is None:
        splits = make_splits_inline(n_div, 5, 0)
    missing = [s for s in all_stems if s not in splits]
    if missing:
        log(f"[prepare] WARNING {len(missing)} stems missing from splits -> inline fold rule for them")
        extra = make_splits_inline({s: n_div[s] for s in missing}, 5, 0)
        splits.update(extra)

    # video selection
    if cfg["stems"]:
        stems = [s for s in cfg["stems"] if s in set(all_stems)]
    else:
        limit = cfg["max_videos"] or 0
        if cfg["smoke"]:
            limit = min(limit, cfg["smoke_videos"]) if limit else cfg["smoke_videos"]
        if limit and limit < len(all_stems):
            by_emb = {e: sorted((s for s in all_stems if s[:4] == e), key=lambda s: (-n_div[s], s))
                      for e in sorted({s[:4] for s in all_stems})}
            stems, k = [], 0
            while len(stems) < limit:  # alternate embryos, division-rich videos first
                emb = sorted(by_emb, reverse=True)[k % len(by_emb)]
                if by_emb[emb]:
                    stems.append(by_emb[emb].pop(0))
                k += 1
                if not any(by_emb.values()):
                    break
            stems = sorted(stems)
        else:
            stems = list(all_stems)
    n_folds = int(cfg["n_folds"])
    orig_folds = sorted({splits[s] for s in splits})
    if n_folds == len(orig_folds) and len(stems) == len(all_stems):
        folds = {s: int(splits[s]) for s in stems}
    else:
        folds = {s: int(splits[s]) % n_folds for s in stems}
        if len({folds[s] for s in stems}) < min(n_folds, len(stems)):  # small subsets: deal round-robin
            ordered = sorted(stems, key=lambda s: (splits[s], s))
            folds = {s: i % n_folds for i, s in enumerate(ordered)}

    chosen = set(stems)
    rows = [r for stem, rs in zip(all_stems, label_rows) if stem in chosen for r in rs]
    meta = pd.DataFrame(rows, columns=["stem", "node_id", "t", "z", "y", "x", "label", "kind"])
    meta["label_idx"] = meta["label"].map(CLASS_INDEX).astype(int)
    meta["fold"] = meta["stem"].map(folds).astype(int)
    meta["embryo"] = meta["stem"].str[:4]
    meta = meta.sort_values(["stem", "t", "node_id", "kind"]).reset_index(drop=True)
    stats = pd.DataFrame(stats_rows)
    extra_info = None
    if cfg.get("extra_samples"):
        extra_path = Path(cfg["extra_samples"])
        if not extra_path.is_file():
            raise FileNotFoundError(f"KDN_EXTRA_SAMPLES={extra_path} not found")
        extra = load_extra_samples(extra_path)
        n_gt = len(meta)
        meta, extra_info, per_video = merge_extra_samples(meta, extra, folds, cfg["dedup_vox"])
        if not (meta.iloc[:n_gt]["source"] == "gt").all() or (meta.iloc[n_gt:]["source"] != "pool").any():
            raise RuntimeError("extra-sample merge broke the GT-first row order")
        extra_info.update({"path": str(extra_path), "sha256": sha256_file(extra_path), "dedup_vox": cfg["dedup_vox"],
                           "pool_weight": cfg["pool_weight"]})
        pv = pd.DataFrame.from_dict(per_video, orient="index").rename_axis("stem").reset_index()
        if not pv.empty:
            stats = stats.merge(pv, on="stem", how="left")
            for col in [c for c in pv.columns if c != "stem"]:
                stats[col] = stats[col].fillna(0).astype(int)
        log(f"[prepare] extra samples {extra_path.name} (sha256 {extra_info['sha256'][:12]}): "
            f"{extra_info['rows_file']} rows in file, {extra_info['rows_other_videos']} outside the selected videos, "
            f"{extra_info['rows_unknown_label']} unknown label, {extra_info['dedup_dropped']} dropped as duplicates of "
            f"GT samples (<= {cfg['dedup_vox']} voxel; same label {extra_info['dedup_same_label']}), "
            f"{extra_info['rows_used']} used {extra_info['by_label']}; pool loss weight {cfg['pool_weight']}")
    else:
        meta["source"] = "gt"
    stats["selected"] = stats["stem"].isin(stems)
    stats["fold"] = stats["stem"].map(folds)
    (work / "splits_used.json").write_text(json.dumps(
        {"source": splits_source, "n_folds": n_folds, "folds": {s: folds[s] for s in stems}}, indent=1))
    log(f"[prepare] folds from {splits_source}; selected {len(stems)} videos, {len(meta)} samples; "
        f"per fold videos {dict(sorted(pd.Series(folds).value_counts().items()))}")
    _print_label_counts(meta)
    return {"data_dir": str(data_dir), "cache_dir": str(cache), "work_dir": str(work), "stems": stems,
            "folds": folds, "meta": meta, "stats": stats, "splits_source": splits_source, "extra_info": extra_info,
            "timing": {"prepare_s": round(time.time() - t0, 1)}}


def _print_label_counts(meta) -> None:
    if meta.empty:
        log("[prepare] no labelled samples")
        return
    tab = meta.groupby(["embryo", "kind"]).size().unstack(fill_value=0)
    log("[prepare] samples by embryo x kind:\n" + tab.to_string())
    tab = meta.groupby(["fold", "label"]).size().unstack(fill_value=0)
    log("[prepare] samples by fold x label:\n" + tab.to_string())
    if "source" in meta.columns and (meta["source"] != "gt").any():
        tab = meta.groupby(["source", "label"]).size().unstack(fill_value=0)
        log("[prepare] samples by source x label:\n" + tab.to_string())
        tab = meta.groupby(["fold", "source"]).size().unstack(fill_value=0)
        log("[prepare] samples by fold x source:\n" + tab.to_string())


# ----------------------------------------------------------------------------------------------------------------------
# step 2: crops
# ----------------------------------------------------------------------------------------------------------------------
def _big_cfg() -> dict:
    cfg = copy.deepcopy(DEFAULT_CFG)
    cfg["input"]["crop_z"] = BIG[0]
    cfg["input"]["crop_yx"] = BIG[1]
    return cfg


def _build_worker_init(n_threads: int) -> None:
    try:
        import blosc2
        blosc2.set_nthreads(max(1, n_threads))
    except Exception:
        pass
    try:
        import torch
        torch.set_num_threads(1)
    except Exception:
        pass


def crop_video(job: dict) -> dict:
    """All crops of one video: frames are read once (sorted by t), prepared frames kept for lags -1..+2 only."""
    t0 = time.time()
    stem = job["stem"]
    try:
        reader = FrameReader(Path(job["data_dir"]))
        prepared: dict = {}
        bundle = {"config": _big_cfg(), "prepared_frames": prepared}
        ref_bundle = {"config": copy.deepcopy(DEFAULT_CFG), "prepared_frames": prepared}
        frame_cache: dict = {}
        nodes = job["nodes"]
        sources = job.get("sources") or ["gt"] * len(nodes)
        out = np.empty((len(nodes), 4, *BIG), dtype=np.dtype(job["dtype"]))
        checked: dict[str, int] = {}   # exact contract checks per source (gt, pool): the first nodes of each, by t
        inner = (slice(None), slice(MARGIN, MARGIN + CROP[0]), slice(MARGIN, MARGIN + CROP[1]),
                 slice(MARGIN, MARGIN + CROP[2]))
        for i in sorted(range(len(nodes)), key=lambda k: nodes[k][0]):
            tt, z, y, x = nodes[i]
            node = {"t": int(tt), "z": float(z), "y": float(y), "x": float(x)}
            sample = prepare_sample(stem, int(tt), node, bundle, frame_cache, reader)
            if sample.shape != (5, *BIG):
                raise RuntimeError(f"unexpected enlarged sample shape {sample.shape}")
            out[i] = sample[:4]
            if checked.get(sources[i], 0) < job["checks"]:
                ref = prepare_sample(stem, int(tt), node, ref_bundle, frame_cache, reader)
                if ref.shape != (5, *CROP) or not np.array_equal(sample[inner], ref):
                    raise RuntimeError(f"inner crop != DivNet-contract sample (contract drift, {sources[i]} node)")
                checked[sources[i]] = checked.get(sources[i], 0) + 1
            for key in [k for k in prepared if k[1] < int(tt) - 1]:
                prepared.pop(key)
        prepared.clear()
        frame_cache.clear()
        return {"stem": stem, "crops": out, "secs": time.time() - t0, "error": None,
                "checked": sum(checked.values()), "checked_pool": checked.get("pool", 0),
                "frames_read": reader.frames_read, "decoders": decoder_names()}
    except Exception as exc:
        return {"stem": stem, "crops": None, "secs": time.time() - t0, "error": f"{type(exc).__name__}: {exc}",
                "checked": 0, "checked_pool": 0, "frames_read": 0, "decoders": decoder_names()}


def step_build_cache(cfg: dict, state: dict) -> dict:
    import multiprocessing as mp

    t0 = time.time()
    meta = state["meta"]
    cache = Path(state["cache_dir"])
    crops_path = cache / "crops.npy"
    dtype = np.dtype(cfg["cache_dtype"])
    n = len(meta)
    need = n * 4 * int(np.prod(BIG)) * dtype.itemsize
    try:
        import shutil
        free = shutil.disk_usage(cache).free
        log(f"[build] cache disk free {free / 2**30:.1f} GiB, needed {need / 2**30:.2f} GiB")
        if free < 1.2 * need and dtype == np.float32:
            dtype = np.dtype(np.float16)
            log("[build] WARNING not enough disk for float32 -> caching float16 (OOF inputs rounded to fp16)")
    except Exception:
        pass
    state["cache_dtype"] = dtype.name
    log(f"[build] {n} samples -> {crops_path} ({n * 4 * int(np.prod(BIG)) * dtype.itemsize / 2**30:.2f} GiB {dtype})")
    crops = np.lib.format.open_memmap(crops_path, mode="w+", dtype=dtype, shape=(max(n, 1), 4, *BIG))
    jobs = []
    has_source = "source" in meta.columns
    for stem, grp in meta.groupby("stem", sort=True):
        jobs.append({"stem": stem, "data_dir": state["data_dir"], "dtype": dtype.name, "checks": cfg["contract_checks"],
                     "index": grp.index.to_numpy(),
                     "sources": grp["source"].tolist() if has_source else None,
                     "nodes": list(zip(grp["t"].tolist(), grp["z"].tolist(), grp["y"].tolist(), grp["x"].tolist()))})
    n_workers = cfg["build_workers"] or min(4, os.cpu_count() or 1)
    n_workers = max(1, min(n_workers, len(jobs) or 1))
    per_worker_threads = max(1, (os.cpu_count() or 1) // n_workers)
    valid = np.zeros(n, bool)
    video_rows = []
    lite = [{k: v for k, v in j.items() if k != "index"} for j in jobs]
    index_of = {j["stem"]: j["index"] for j in jobs}
    done = 0
    ctx = mp.get_context("spawn")
    with ctx.Pool(n_workers, initializer=_build_worker_init, initargs=(per_worker_threads,)) as pool:
        for res in pool.imap_unordered(crop_video, lite):
            done += 1
            stem = res["stem"]
            idx = index_of[stem]
            grp = meta.loc[idx]
            counts = grp["kind"].value_counts().to_dict()
            if res["error"] is None:
                crops[idx] = res["crops"]
                valid[idx] = True
                status = "ok"
            else:
                status = "FAILED " + res["error"]
            video_rows.append({"stem": stem, "crop_secs": round(res["secs"], 2), "frames_read": res["frames_read"],
                               "contract_checks": res["checked"], "contract_checks_pool": res.get("checked_pool", 0),
                               "crop_error": res["error"], "n_samples": len(idx)})
            log(f"[build] {done}/{len(jobs)} {stem} n={len(idx)} "
                + " ".join(f"{k}={v}" for k, v in sorted(counts.items()))
                + f" frames={res['frames_read']} {res['secs']:.1f}s decoders={res['decoders']} {status}")
    crops.flush()
    del crops
    meta = meta.copy()
    meta["valid"] = valid
    meta.to_csv(cache / "meta.csv", index_label="row")
    import pandas as pd
    vid = pd.DataFrame(video_rows)
    stats = state["stats"].merge(vid, on="stem", how="left") if not vid.empty else state["stats"]
    stats.to_csv(Path(state["work_dir"]) / "labels_summary.csv", index=False)
    state["meta"] = meta
    state["stats"] = stats
    secs = time.time() - t0
    state["timing"]["build_s"] = round(secs, 1)
    ok = vid[vid["crop_error"].isna()] if not vid.empty else vid
    per_video = float(ok["crop_secs"].mean()) if len(ok) else float("nan")
    state["timing"]["crop_s_per_video_mean"] = round(per_video, 2)
    log(f"[build] done in {secs:.1f} s with {n_workers} workers; valid samples {int(valid.sum())}/{n}; "
        f"videos failed {int(vid['crop_error'].notna().sum()) if not vid.empty else 0}; mean crop time per video "
        f"{per_video:.2f} s (single worker)")
    if has_source and (meta["source"] == "pool").any():
        n_chk = int(vid["contract_checks"].sum()) if not vid.empty else 0
        n_chk_pool = int(vid["contract_checks_pool"].sum()) if not vid.empty else 0
        state["contract_checks"] = {"gt": n_chk - n_chk_pool, "pool": n_chk_pool}
        log(f"[build] exact contract checks (inner crop == prepare_sample): gt {n_chk - n_chk_pool}, pool {n_chk_pool}; "
            f"valid by source {meta[meta['valid']].groupby('source').size().to_dict()}")
    return state


# ----------------------------------------------------------------------------------------------------------------------
# step 3: training (subprocess per fold)
# ----------------------------------------------------------------------------------------------------------------------
def _src_paths() -> list[str]:
    import tools.daughternet as dn  # noqa: F401  (the tools package the kernel materialised or the repo copy)
    paths = [str(Path(__file__).resolve().parent), str(Path(dn.__file__).resolve().parent.parent)]
    return list(dict.fromkeys(paths))


def _fold_dir(work: Path, fold: int) -> Path:
    return work / "folds" / f"fold_{fold}"


def _pump(stream, prefix: str, sink) -> None:
    for line in iter(stream.readline, ""):
        print(prefix + line.rstrip(), flush=True)
        sink.write(line)
        sink.flush()
    stream.close()


def step_train(cfg: dict, state: dict) -> dict:
    import torch

    t0 = time.time()
    work = Path(state["work_dir"])
    cache = Path(state["cache_dir"])
    meta = state["meta"]
    folds = sorted(int(f) for f in meta.loc[meta["valid"], "fold"].unique())
    (work / "folds").mkdir(parents=True, exist_ok=True)
    cfg_path = cache / "kdn_cfg.json"
    cfg_path.write_text(json.dumps({**cfg, "_cache_dir": str(cache), "_work_dir": str(work),
                                    "_extra_info": state.get("extra_info")}, indent=1, default=_json_default))
    n_gpu = torch.cuda.device_count() if cfg["gpus"] != 0 else 0
    if cfg["gpus"] > 0:
        n_gpu = min(n_gpu, cfg["gpus"])
    slots = [str(i) for i in range(n_gpu)] or ["cpu"]
    log(f"[train] folds {folds} on slots {slots} (subprocess per fold)")
    env_base = os.environ.copy()
    env_base["PYTHONPATH"] = os.pathsep.join(_src_paths() + [p for p in env_base.get("PYTHONPATH", "").split(os.pathsep) if p])
    env_base["PYTHONUNBUFFERED"] = "1"
    queue = list(folds)
    running: dict[str, tuple] = {}
    results: dict[int, int] = {}
    while queue or running:
        for slot in slots:
            if slot not in running and queue:
                fold = queue.pop(0)
                fdir = _fold_dir(work, fold)
                fdir.mkdir(parents=True, exist_ok=True)
                # a rerun into the same work dir must never pick up a previous run's fold model / OOF rows
                # (a fold that fails or never saves best.pt would otherwise load or merge stale files silently)
                for stale in ("best.pt", "last.pt", "oof.csv", "summary.json", "history.json", "train.log"):
                    (fdir / stale).unlink(missing_ok=True)
                env = dict(env_base)
                if slot != "cpu":
                    env["CUDA_VISIBLE_DEVICES"] = slot
                device = "cuda:0" if slot != "cpu" else "cpu"
                cmd = [sys.executable, str(Path(__file__).resolve()), "worker", "--cfg", str(cfg_path),
                       "--fold", str(fold), "--device", device, "--threads",
                       str(max(1, (os.cpu_count() or 1) // len(slots)))]
                sink = open(fdir / "train.log", "a", encoding="utf-8")
                proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                                        encoding="utf-8", errors="replace")
                th = threading.Thread(target=_pump, args=(proc.stdout, f"[fold {fold}@{slot}] ", sink), daemon=True)
                th.start()
                running[slot] = (fold, proc, th, sink, time.time())
                log(f"[train] started fold {fold} on {slot} (pid {proc.pid})")
        time.sleep(1.0)
        for slot in list(running):
            fold, proc, th, sink, ts = running[slot]
            if proc.poll() is not None:
                th.join(timeout=10)
                sink.close()
                results[fold] = proc.returncode
                log(f"[train] fold {fold} finished rc={proc.returncode} in {time.time() - ts:.0f} s")
                del running[slot]
    state["train_rc"] = results
    state["timing"]["train_s"] = round(time.time() - t0, 1)
    failed = [f for f, rc in results.items() if rc != 0]
    if failed:
        log(f"[train] WARNING folds failed: {failed} (their videos get no OOF prediction)")
    return state


def _load_split(crops, rows: np.ndarray, inner_only: bool, dtype, device, torch, chunk: int = 1024):
    """Rows of the memmap -> one tensor on device; inner_only keeps the contract 16x32x32 crop."""
    if len(rows) == 0:
        shape = (0, 4, *CROP) if inner_only else (0, 4, *BIG)
        return torch.zeros(shape, dtype=dtype, device=device)
    parts = []
    for s in range(0, len(rows), chunk):
        block = np.asarray(crops[rows[s:s + chunk]], dtype=np.float32)
        if inner_only:
            block = block[:, :, MARGIN:MARGIN + CROP[0], MARGIN:MARGIN + CROP[1], MARGIN:MARGIN + CROP[2]]
        parts.append(torch.from_numpy(np.ascontiguousarray(block)).to(device=device, dtype=dtype))
    return torch.cat(parts)


def augment_batch(xb, rng: np.random.Generator, cfg: dict, marker, torch):
    """(B, 4, 18, 34, 34) image crops -> (B, 5, 16, 32, 32): +-jitter crop (image shifts, marker stays centred), then
    random flips on z/y/x and a random k*90 degree rotation in yx applied to all 5 channels, then a per-sample
    intensity gain on the 4 image channels (clipped back to the contract range [-0.5, 6])."""
    b = xb.shape[0]
    dev = xb.device
    j = int(cfg["jitter"])
    offs = torch.from_numpy(rng.integers(MARGIN - j, MARGIN + j + 1, size=(b, 3))).to(dev)
    zi = offs[:, 0, None] + torch.arange(CROP[0], device=dev)
    yi = offs[:, 1, None] + torch.arange(CROP[1], device=dev)
    xi = offs[:, 2, None] + torch.arange(CROP[2], device=dev)
    bi = torch.arange(b, device=dev)[:, None, None, None]
    img = xb[bi, :, zi[:, :, None, None], yi[:, None, :, None], xi[:, None, None, :]]  # (B, 16, 32, 32, 4)
    img = img.permute(0, 4, 1, 2, 3).float()
    x = torch.cat([img, marker.to(dev).expand(b, 1, *CROP)], dim=1)
    flips = torch.from_numpy(rng.random((b, 3)) < 0.5).to(dev)
    for axis, dim in enumerate((2, 3, 4)):
        m = flips[:, axis].view(b, 1, 1, 1, 1)
        x = torch.where(m, x.flip(dim), x)
    rot = rng.integers(0, 4, size=b)
    for k in (1, 2, 3):
        sel = np.flatnonzero(rot == k)
        if len(sel):
            idx = torch.from_numpy(sel).to(dev)
            x[idx] = torch.rot90(x[idx], k, dims=(3, 4))
    gain = torch.from_numpy(rng.uniform(cfg["gain"][0], cfg["gain"][1], size=(b, 1, 1, 1, 1)).astype(np.float32)).to(dev)
    x[:, :4] = (x[:, :4] * gain).clamp(-0.5, 6.0)
    return x


def contract_marker(torch):
    sigma = tuple(float(v) for v in DEFAULT_CFG["input"]["marker_sigma"])
    return torch.from_numpy(_marker(CROP, sigma)).view(1, 1, *CROP)


def predict_inner(model, x_inner, marker, batch: int, torch) -> np.ndarray:
    """Softmax (N, 3) for contract samples (image inner crops + centred marker), fp32 like score_nodes."""
    out = []
    model.eval()
    with torch.no_grad():
        for s in range(0, x_inner.shape[0], batch):
            xb = x_inner[s:s + batch].float()
            x = torch.cat([xb, marker.to(xb.device).expand(xb.shape[0], 1, *CROP)], dim=1)
            out.append(torch.softmax(model(x).float(), dim=1).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, len(CLASSES)), np.float32)


def auc_score(scores, positive) -> float | None:
    import pandas as pd
    s = np.asarray(scores, dtype=np.float64)
    y = np.asarray(positive, dtype=bool)
    n1, n0 = int(y.sum()), int((~y).sum())
    if n1 == 0 or n0 == 0:
        return None
    ranks = pd.Series(s).rank(method="average").to_numpy()
    return float((ranks[y].sum() - n1 * (n1 + 1) / 2.0) / (n1 * n0))


def _val_metrics(probs: np.ndarray, y: np.ndarray) -> dict:
    if len(y) == 0:
        return {"n": 0}
    p_true = np.clip(probs[np.arange(len(y)), y], 1e-7, 1.0)
    ce = -np.log(p_true)
    present = [c for c in range(len(CLASSES)) if (y == c).any()]
    macro = float(np.mean([ce[y == c].mean() for c in present]))
    res = {"n": int(len(y)), "macro_ce": macro, "ce": float(ce.mean()),
           "acc": float((probs.argmax(1) == y).mean())}
    for c, name in enumerate(CLASSES):
        res[f"n_{name}"] = int((y == c).sum())
        res[f"ce_{name}"] = float(ce[y == c].mean()) if (y == c).any() else None
        res[f"auc_{name}"] = auc_score(probs[:, c], y == c)
    return res


def _atomic_save(obj, path: Path, torch) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def train_fold(cfg: dict, fold: int, device_str: str, threads: int = 0) -> dict:
    import pandas as pd
    import torch

    t_start = time.time()
    if threads:
        torch.set_num_threads(int(threads))
    cache = Path(cfg["_cache_dir"])
    work = Path(cfg["_work_dir"])
    fdir = _fold_dir(work, fold)
    fdir.mkdir(parents=True, exist_ok=True)
    device = torch.device(device_str)
    use_cuda = device.type == "cuda"
    if use_cuda:
        torch.backends.cudnn.benchmark = True
    seed = int(cfg["seed"]) * 1000 + int(fold)
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    meta = pd.read_csv(cache / "meta.csv", index_col="row")
    crops = np.load(cache / "crops.npy", mmap_mode="r")
    meta = meta[meta["valid"].astype(bool)]
    if "source" not in meta.columns:  # cache written before v2
        meta = meta.assign(source="gt")
    meta = meta.assign(source=meta["source"].fillna("gt").astype(str))
    has_pool = bool((meta["source"] == "pool").any())
    n_folds = int(cfg["n_folds"])
    if cfg.get("inner_val") and n_folds < 3:
        log(f"fold {fold}: KDN_INNER_VAL needs >= 3 folds (have {n_folds}) -> early stopping on the OOF fold")
        cfg = {**cfg, "inner_val": False}
    if cfg.get("inner_val"):
        es_fold = (fold + 1) % n_folds
        train_mask = ~meta["fold"].isin([fold, es_fold])
        es_mask = meta["fold"] == es_fold
    else:
        es_fold = fold
        train_mask = meta["fold"] != fold
        es_mask = meta["fold"] == fold
    oof_mask = meta["fold"] == fold
    tr_rows = meta.index[train_mask].to_numpy()
    es_rows = meta.index[es_mask].to_numpy()
    oof_rows = meta.index[oof_mask].to_numpy()
    y_tr = meta.loc[tr_rows, "label_idx"].to_numpy().astype(np.int64)
    y_es = meta.loc[es_rows, "label_idx"].to_numpy().astype(np.int64)
    src_tr = meta.loc[tr_rows, "source"].to_numpy()
    src_es = meta.loc[es_rows, "source"].to_numpy()
    es_source = cfg.get("es_source", "all") if has_pool else "all"
    es_sel = np.ones(len(es_rows), bool) if es_source == "all" else (src_es == es_source)
    log(f"fold {fold}: device {device} train {len(tr_rows)} {np.bincount(y_tr, minlength=3).tolist()} | "
        f"early-stop fold {es_fold}: {len(es_rows)} {np.bincount(y_es, minlength=3).tolist()} | oof {len(oof_rows)}")
    if has_pool:
        comp = {s: np.bincount(y_tr[src_tr == s], minlength=3).tolist() for s in ("gt", "pool")}
        comp_es = {s: np.bincount(y_es[src_es == s], minlength=3).tolist() for s in ("gt", "pool")}
        log(f"fold {fold}: train by source [cont, parent, daughter] {comp} | early-stop by source {comp_es}; "
            f"early-stop loss on {es_source} rows ({int(es_sel.sum())}); pool loss weight {cfg.get('pool_weight', 1.0)}")
    if len(tr_rows) == 0:
        raise RuntimeError("no training samples")
    store_dtype = torch.float16 if use_cuda else torch.float32
    t_load = time.time()
    x_tr = _load_split(crops, tr_rows, False, store_dtype, device, torch)
    x_es = _load_split(crops, es_rows, True, torch.float32, device, torch)
    y_tr_t = torch.from_numpy(y_tr).to(device)
    pool_w = float(cfg.get("pool_weight", 1.0))
    weighted = bool((src_tr == "pool").any()) and pool_w != 1.0
    w_tr_t = (torch.from_numpy(np.where(src_tr == "pool", pool_w, 1.0).astype(np.float32)).to(device)
              if weighted else None)
    log(f"fold {fold}: loaded crops in {time.time() - t_load:.1f} s (train {tuple(x_tr.shape)} {x_tr.dtype})")
    marker = contract_marker(torch).to(device)

    ckpt_cfg = copy.deepcopy(DEFAULT_CFG)
    ckpt_cfg["classes"] = list(CLASSES)
    ckpt_cfg["train"] = {k: cfg[k] for k in ("epochs", "patience", "batch", "lr", "weight_decay", "warmup_frac",
                                               "epoch_samples", "jitter", "gain", "seed", "cont_per_video",
                                               "hardneg_per_video", "cont_min_dist", "inner_val", "n_folds")}
    ckpt_cfg["train"].update({"fold": fold, "early_stop_fold": es_fold, "kernel": "biohub-l2-kdn-train"})
    if cfg.get("extra_samples"):
        extra_info = cfg.get("_extra_info") or {}
        ckpt_cfg["train"].update({"extra_samples": Path(cfg["extra_samples"]).name,
                                  "extra_samples_sha256": extra_info.get("sha256"), "pool_weight": pool_w,
                                  "dedup_vox": cfg.get("dedup_vox"), "es_source": es_source,
                                  "train_counts_by_source": {s: np.bincount(y_tr[src_tr == s], minlength=3).tolist()
                                                             for s in ("gt", "pool")}})
    model = build_model(ckpt_cfg).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    classes_present = [c for c in range(len(CLASSES)) if (y_tr == c).any()]
    pools = {c: np.flatnonzero(y_tr == c) for c in classes_present}
    epoch_samples = int(cfg["epoch_samples"]) or len(tr_rows)
    steps_per_epoch = max(1, math.ceil(epoch_samples / cfg["batch"]))
    total_steps = steps_per_epoch * int(cfg["epochs"])
    warmup = max(1, int(cfg["warmup_frac"] * total_steps))

    def lr_at(step: int) -> float:
        if step < warmup:
            return cfg["lr"] * (step + 1) / warmup
        prog = (step - warmup) / max(1, total_steps - warmup)
        return cfg["lr"] * 0.5 * (1.0 + math.cos(math.pi * min(1.0, prog)))

    use_amp = bool(cfg["amp"]) and use_cuda
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    except Exception:  # older torch
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    best, best_epoch, bad, step = float("inf"), -1, 0, 0
    history = []
    for epoch in range(int(cfg["epochs"])):
        te = time.time()
        model.train()
        cls = rng.choice(classes_present, size=epoch_samples)
        order = np.empty(epoch_samples, dtype=np.int64)
        for c in classes_present:
            sel = np.flatnonzero(cls == c)
            order[sel] = rng.choice(pools[c], size=len(sel), replace=True)
        rng.shuffle(order)
        run_loss, seen = torch.zeros((), device=device), 0
        for s in range(0, epoch_samples, cfg["batch"]):
            idx = order[s:s + cfg["batch"]]
            idx_t = torch.from_numpy(idx).to(device)
            xb = augment_batch(x_tr[idx_t], rng, cfg, marker, torch)
            yb = y_tr_t[idx_t]
            for group in opt.param_groups:
                group["lr"] = lr_at(step)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(xb)
            if weighted:  # pool rows up/down-weighted: weighted mean of the per-sample CE over the batch
                wb = w_tr_t[idx_t]
                ce = torch.nn.functional.cross_entropy(logits.float(), yb, reduction="none")
                loss = (ce * wb).sum() / wb.sum()
            else:         # v1 path, bit-identical when there are no pool rows or KDN_POOL_WEIGHT = 1
                loss = torch.nn.functional.cross_entropy(logits.float(), yb)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            run_loss += loss.detach() * len(idx)
            seen += len(idx)
            step += 1
        train_loss = float(run_loss) / max(1, seen)
        if use_cuda:
            torch.cuda.synchronize()
        train_secs = time.time() - te
        probs = predict_inner(model, x_es, marker, cfg["eval_batch"], torch)
        vm = _val_metrics(probs[es_sel], y_es[es_sel]) if es_source != "all" else _val_metrics(probs, y_es)
        vm_src = ({s: _val_metrics(probs[src_es == s], y_es[src_es == s]) for s in ("gt", "pool")} if has_pool else {})
        val_loss = vm.get("macro_ce", float("nan")) if vm["n"] else float("nan")
        improved = vm["n"] > 0 and val_loss < best
        state_cpu = {k: v.detach().cpu() for k, v in model.state_dict().items()}
        payload = {"model_state": state_cpu, "config": ckpt_cfg, "fold": int(fold), "epoch": int(epoch),
                   "val_loss": float(val_loss)}
        _atomic_save(payload, fdir / "last.pt", torch)
        if improved or (vm["n"] == 0):
            best, best_epoch, bad = (val_loss if vm["n"] else best), epoch, 0
            _atomic_save(payload, fdir / "best.pt", torch)
        else:
            bad += 1
        rec = {"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "lr": lr_at(step - 1),
               "secs": round(time.time() - te, 1), "train_secs": round(train_secs, 1),
               "samples_per_s": round(seen / max(1e-9, train_secs), 1),
               **{f"val_{k}": v for k, v in vm.items()},
               **{f"val_{s}_{k}": v for s, m in vm_src.items() for k, v in m.items()
                  if k in ("n", "macro_ce", "auc_parent", "auc_daughter")}}
        history.append(rec)
        (fdir / "history.json").write_text(json.dumps(history, indent=1))
        auc_txt = " ".join(f"auc_{n}={vm.get('auc_' + n) if vm.get('auc_' + n) is None else round(vm['auc_' + n], 3)}"
                           for n in CLASSES)
        src_txt = "".join(
            f" | {s}: CE {_r4(m.get('macro_ce'))} auc_par {_r4(m.get('auc_parent'))} auc_dau {_r4(m.get('auc_daughter'))}"
            for s, m in vm_src.items() if m["n"])
        log(f"fold {fold} epoch {epoch}: train {train_loss:.4f} val(macro CE) {val_loss:.4f} "
            f"{'*' if improved else ' '} {auc_txt} {rec['secs']}s (train {rec['samples_per_s']} samples/s){src_txt}")
        if vm["n"] and bad >= int(cfg["patience"]):
            log(f"fold {fold}: early stop at epoch {epoch} (best {best_epoch}, val {best:.4f})")
            break
    del x_tr
    if not (fdir / "best.pt").is_file():
        raise RuntimeError(f"fold {fold}: no best.pt written in this run (epochs={cfg['epochs']}, history "
                           f"{[h['val_loss'] for h in history]}) -> no OOF for this fold")
    # OOF prediction with best.pt through the shared loader (same path the submission patch uses)
    bundle = load_daughternet([fdir / "best.pt"], device=str(device))
    x_oof = _load_split(crops, oof_rows, True, torch.float32, device, torch)
    p_oof = predict_inner(bundle["models"][0], x_oof, marker, cfg["eval_batch"], torch)
    oof = pd.DataFrame({"row": oof_rows, "p_cont": p_oof[:, 0], "p_par": p_oof[:, 1], "p_dau": p_oof[:, 2]})
    oof.to_csv(fdir / "oof.csv", index=False)
    y_oof = meta.loc[oof_rows, "label_idx"].to_numpy().astype(np.int64)
    om = _val_metrics(p_oof, y_oof)
    summary = {"fold": fold, "best_epoch": best_epoch, "best_val_loss": best, "epochs_run": len(history),
               "early_stop_fold": es_fold, "n_train": int(len(tr_rows)), "n_oof": int(len(oof_rows)),
               "train_counts": np.bincount(y_tr, minlength=3).tolist(), "oof_metrics": om,
               "secs": round(time.time() - t_start, 1), "device": str(device)}
    if has_pool:
        src_oof = meta.loc[oof_rows, "source"].to_numpy()
        summary["es_source"] = es_source
        summary["pool_weight"] = pool_w
        summary["train_counts_by_source"] = {s: np.bincount(y_tr[src_tr == s], minlength=3).tolist()
                                             for s in ("gt", "pool")}
        summary["oof_metrics_by_source"] = {s: _val_metrics(p_oof[src_oof == s], y_oof[src_oof == s])
                                            for s in ("gt", "pool")}
    (fdir / "summary.json").write_text(json.dumps(summary, indent=1))
    log(f"fold {fold}: OOF {len(oof_rows)} samples macro CE {om.get('macro_ce')} "
        f"auc_parent {om.get('auc_parent')} auc_daughter {om.get('auc_daughter')}; total {summary['secs']} s")
    for s, m in summary.get("oof_metrics_by_source", {}).items():
        log(f"fold {fold}: OOF source={s} n={m['n']} macro CE {_r4(m.get('macro_ce'))} "
            f"auc_parent {_r4(m.get('auc_parent'))} auc_daughter {_r4(m.get('auc_daughter'))}")
    return summary


def _r4(value):
    return None if value is None else round(float(value), 4)


def worker_main(argv: list[str]) -> int:
    import argparse

    ap = argparse.ArgumentParser()
    ap.add_argument("--cfg", required=True)
    ap.add_argument("--fold", type=int, required=True)
    ap.add_argument("--device", default="cpu")
    ap.add_argument("--threads", type=int, default=0)
    a = ap.parse_args(argv)
    cfg = json.loads(Path(a.cfg).read_text())
    train_fold(cfg, a.fold, a.device, a.threads)
    return 0


# ----------------------------------------------------------------------------------------------------------------------
# step 4: OOF merge, readout, DivNet comparison
# ----------------------------------------------------------------------------------------------------------------------
TOP_FRACS = (("top0.1%", 0.001), ("top1%", 0.01), ("top5%", 0.05))


def _rank_block(scores: np.ndarray, positive: np.ndarray) -> dict:
    n = len(scores)
    n_pos = int(positive.sum())
    base = n_pos / n if n else None
    out = {"n": int(n), "n_pos": n_pos, "base_rate": base, "auc": auc_score(scores, positive)}
    order = np.argsort(-scores, kind="stable")
    for name, frac in TOP_FRACS:
        if n == 0:
            out[name] = None
            continue
        k = max(1, int(math.ceil(frac * n)))
        tp = int(positive[order[:k]].sum())
        fp = k - tp
        prec = tp / k
        tpr = tp / n_pos if n_pos else None
        fpr = fp / (n - n_pos) if n - n_pos else None
        lr_plus = (tpr / fpr) if (tpr is not None and fpr) else None
        out[name] = {"k": k, "tp": tp, "precision": prec, "lift": (prec / base) if base else None,
                     "lr_plus": lr_plus, "lr_plus_infinite": bool(fpr == 0 and tp > 0)}
    return out


def _find_divnet_checkpoint(cfg: dict) -> Path | None:
    if cfg.get("divnet_checkpoint"):
        p = Path(cfg["divnet_checkpoint"])
        return p if p.is_file() else None
    for p in _checkpoint_candidates():
        if Path(p).is_file():
            return Path(p)
    root = Path("/kaggle/input")
    if root.exists():  # pruned walk: a plain rglob would descend into every competition .zarr chunk tree
        found = []
        for dirpath, dirnames, files in os.walk(root, followlinks=True):
            depth = Path(dirpath).relative_to(root).parts
            dirnames[:] = [] if len(depth) >= 6 else sorted(
                d for d in dirnames if not d.endswith((".zarr", ".geff")) and not d.startswith("."))
            if "best_overall.pt" in files and "divnet" in dirpath.lower():
                found.append(Path(dirpath) / "best_overall.pt")
        if found:
            return sorted(found)[0]
    return None


def step_oof(cfg: dict, state: dict) -> dict:
    import pandas as pd
    import torch

    t0 = time.time()
    work = Path(state["work_dir"])
    cache = Path(state["cache_dir"])
    meta = pd.read_csv(cache / "meta.csv", index_col="row", float_precision="round_trip")  # exact node coordinates
    meta = meta[meta["valid"].astype(bool)].copy()
    for col in ("p_cont", "p_par", "p_dau"):
        meta[col] = np.nan
    fold_summaries = {}
    for fold in sorted(meta["fold"].unique()):
        fdir = _fold_dir(work, int(fold))
        f_oof = fdir / "oof.csv"
        if f_oof.is_file():
            o = pd.read_csv(f_oof).set_index("row")
            meta.loc[o.index, ["p_cont", "p_par", "p_dau"]] = o[["p_cont", "p_par", "p_dau"]].to_numpy()
        if (fdir / "summary.json").is_file():
            fold_summaries[int(fold)] = json.loads((fdir / "summary.json").read_text())
    has_p = meta["p_par"].notna()
    log(f"[oof] predictions for {int(has_p.sum())}/{len(meta)} samples")

    divnet_info = {"enabled": bool(cfg["divnet_compare"])}
    if cfg["divnet_compare"]:
        try:
            ckpt = _find_divnet_checkpoint(cfg)
            if ckpt is None:
                raise FileNotFoundError("best_overall.pt not found")
            bundle = load_divnet_mitosis_model(ckpt)
            dev = bundle["device"]
            crops = np.load(cache / "crops.npy", mmap_mode="r")
            marker = contract_marker(torch).to(dev)
            rows = meta.index.to_numpy()
            ps = []
            for s in range(0, len(rows), 1024):
                x_in = _load_split(crops, rows[s:s + 1024], True, torch.float32, dev, torch)
                with torch.no_grad():
                    for b in range(0, x_in.shape[0], cfg["eval_batch"]):
                        xb = x_in[b:b + cfg["eval_batch"]]
                        x = torch.cat([xb, marker.expand(xb.shape[0], 1, *CROP)], dim=1)
                        ps.append(torch.sigmoid(bundle["model"](x)).flatten().float().cpu().numpy())
            meta["p_divnet"] = np.concatenate(ps) if ps else np.zeros(0)
            del crops, bundle
            divnet_info.update({"checkpoint": str(ckpt), "in_sample": True,
                                "note": "public giorgosi DivNet best_overall.pt was trained on these 199 train videos "
                                        "(4 folds, best fold model): its numbers are IN-SAMPLE, DaughterNet's are OOF"})
            log(f"[oof] DivNet scored {len(meta)} samples with {ckpt}")
        except Exception as exc:
            divnet_info["error"] = f"{type(exc).__name__}: {exc}"
            log(f"[oof] DivNet comparison skipped: {divnet_info['error']}")

    out = meta.reset_index(drop=True)
    if "source" not in out.columns:  # cache written before v2
        out["source"] = "gt"
    out["source"] = out["source"].fillna("gt").astype(str)
    extra = [c for c in ("node_id", "kind", "embryo", "label_idx", "source", "p_divnet", "gt_id", "dist_um",
                         "n_triples", "n_u", "n_c1", "n_v") if c in out.columns]
    out[OOF_COLUMNS + extra].to_csv(work / "oof.csv", index=False)

    readout = {
        "kernel": "biohub-l2-kdn-train", "classes": list(CLASSES), "n_samples": int(len(out)),
        "n_with_oof": int(out["p_par"].notna().sum()), "splits_source": state.get("splits_source"),
        "definitions": {
            "base_rate": "fraction of the embryo's labelled samples that carry the class (sampling-dependent: cont is "
                         "subsampled to <= cont_per_video + hardneg_per_video per video)",
            "precision": "fraction of the class among the top-k samples ranked by that class probability",
            "lift": "precision / base_rate",
            "lr_plus": "likelihood ratio at the cutoff = TPR / FPR = (tp/P) / (fp/N); null when fp = 0",
            "auc": "one-vs-rest ROC AUC of the class probability",
            "source": "gt = annotated GT nodes at GT coordinates (per_embryo / hardneg / divnet_comparison are on "
                      "these rows only, as in v1); pool = exported candidate nodes of our predicted graphs labelled "
                      "by the nearest GT node within 7 um, at PREDICTED coordinates (by_source.pool; cont "
                      "subsampled per video, preferring nodes in many candidate triples -> base rates and AUCs are "
                      "on that sample)",
        },
        "counts": {e: g.groupby("kind").size().to_dict() for e, g in out.groupby("embryo")},
        "counts_by_source": {s: g.groupby("label").size().to_dict() for s, g in out.groupby("source")},
        "per_embryo": {}, "hardneg": {}, "by_source": {}, "folds": fold_summaries, "divnet_comparison": divnet_info,
        "extra_samples": state.get("extra_info"),
        "timing": state.get("timing", {}), "train_rc": state.get("train_rc"),
        "config": {k: v for k, v in cfg.items() if not k.startswith("_")},
    }
    scored_all = out[out["p_par"].notna()]
    scored = scored_all[scored_all["source"] == "gt"]
    groups = [(e, g) for e, g in scored.groupby("embryo")] + [("all", scored)]
    prob_col = {"cont": "p_cont", "parent": "p_par", "daughter": "p_dau"}
    for emb, g in groups:
        readout["per_embryo"][emb] = {
            name: _rank_block(g[prob_col[name]].to_numpy(), (g["label"] == name).to_numpy()) for name in CLASSES}
        hu = g[g["kind"].str.startswith("hardneg") & g["kind"].str.contains("u")]
        hv = g[g["kind"].str.startswith("hardneg") & g["kind"].str.contains("v")]
        par, dau = g[g["label"] == "parent"], g[g["label"] == "daughter"]
        readout["hardneg"][emb] = {
            "n_hardneg_u": int(len(hu)), "n_hardneg_v": int(len(hv)),
            "auc_p_par_parent_vs_hardneg_u": auc_score(np.r_[par["p_par"], hu["p_par"]],
                                                       np.r_[np.ones(len(par)), np.zeros(len(hu))]),
            "auc_p_dau_daughter_vs_hardneg_v": auc_score(np.r_[dau["p_dau"], hv["p_dau"]],
                                                         np.r_[np.ones(len(dau)), np.zeros(len(hv))]),
        }
        if "p_divnet" in g.columns and divnet_info.get("in_sample"):
            sub = g[g["label"].isin(["parent", "cont"])]
            sub_rand = g[g["kind"].isin(["parent", "cont"])]
            y = (sub["label"] == "parent").to_numpy()
            y_r = (sub_rand["kind"] == "parent").to_numpy()
            divnet_info.setdefault("per_embryo", {})[emb] = {
                "subset": "parent vs cont (random cont + hard negatives)",
                "divnet_in_sample": _rank_block(sub["p_divnet"].to_numpy(), y),
                "daughternet_oof_p_par": _rank_block(sub["p_par"].to_numpy(), y),
                "random_cont_only": {"divnet_in_sample_auc": auc_score(sub_rand["p_divnet"].to_numpy(), y_r),
                                     "daughternet_oof_auc": auc_score(sub_rand["p_par"].to_numpy(), y_r)},
            }
    readout["by_source"]["gt"] = {"per_embryo": readout["per_embryo"], "hardneg": readout["hardneg"]}
    pool = scored_all[scored_all["source"] == "pool"]
    if len(pool):
        readout["by_source"]["pool"] = _pool_block(pool, prob_col, bool(divnet_info.get("in_sample")))
    state["timing"]["oof_s"] = round(time.time() - t0, 1)
    readout["timing"] = state["timing"]
    (work / "oof_readout.json").write_text(json.dumps(readout, indent=1, default=_json_default))
    log(f"[oof] wrote {work / 'oof.csv'} ({len(out)} rows) and {work / 'oof_readout.json'}")
    _print_readout(readout)
    if not cfg.get("keep_cache"):
        import shutil
        shutil.rmtree(cache, ignore_errors=True)
    state["readout"] = readout
    return state


def _vs_auc(pos_scores, neg_scores) -> float | None:
    return auc_score(np.r_[np.asarray(pos_scores, float), np.asarray(neg_scores, float)],
                     np.r_[np.ones(len(pos_scores)), np.zeros(len(neg_scores))])


def _pool_block(pool, prob_col: dict, divnet_in_sample: bool) -> dict:
    """OOF readout on source=pool rows (the real candidate pool at predicted coordinates), per embryo + all:
    one-vs-rest rank blocks per class, parent-vs-cont and daughter-vs-cont rank blocks (the question the insertion
    asks), the same against cont nodes that occur as u / c1 / v of candidate triples, and DivNet (in-sample) vs
    DaughterNet on parent-vs-cont."""
    import pandas as pd

    block = {"per_embryo": {}, "vs_cont": {}, "gate_roles": {}, "divnet_parent_vs_cont": {}}
    groups = [(e, g) for e, g in pool.groupby("embryo")] + [("all", pool)]
    for emb, g in groups:
        block["per_embryo"][emb] = {
            name: _rank_block(g[prob_col[name]].to_numpy(), (g["label"] == name).to_numpy()) for name in CLASSES}
        par, dau, cont = g[g["label"] == "parent"], g[g["label"] == "daughter"], g[g["label"] == "cont"]
        pc = pd.concat([par, cont], ignore_index=True)
        dc = pd.concat([dau, cont], ignore_index=True)
        block["vs_cont"][emb] = {
            "parent_vs_cont_p_par": _rank_block(pc["p_par"].to_numpy(), (pc["label"] == "parent").to_numpy()),
            "daughter_vs_cont_p_dau": _rank_block(dc["p_dau"].to_numpy(), (dc["label"] == "daughter").to_numpy()),
        }
        if all(c in g.columns for c in ("n_u", "n_c1", "n_v")):
            cu, cc1, cv = cont[cont["n_u"] > 0], cont[cont["n_c1"] > 0], cont[cont["n_v"] > 0]
            block["gate_roles"][emb] = {
                "n_cont_as_u": int(len(cu)), "n_cont_as_c1": int(len(cc1)), "n_cont_as_v": int(len(cv)),
                "auc_p_par_parent_vs_cont_as_u": _vs_auc(par["p_par"], cu["p_par"]),
                "auc_p_dau_daughter_vs_cont_as_c1": _vs_auc(dau["p_dau"], cc1["p_dau"]),
                "auc_p_dau_daughter_vs_cont_as_v": _vs_auc(dau["p_dau"], cv["p_dau"]),
            }
        if divnet_in_sample and "p_divnet" in g.columns:
            block["divnet_parent_vs_cont"][emb] = {
                "divnet_in_sample_auc": _vs_auc(par["p_divnet"], cont["p_divnet"]),
                "daughternet_oof_auc": _vs_auc(par["p_par"], cont["p_par"])}
    return block


def _json_default(value):
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    return str(value)


def _print_rank_line(tag: str, emb: str, name: str, r: dict) -> None:
    tops = " ".join(f"{k}:{r[k]['tp']}/{r[k]['k']}" for k, _ in TOP_FRACS if r.get(k))
    auc = None if r["auc"] is None else round(r["auc"], 4)
    base = "n/a" if r["base_rate"] is None else f"{r['base_rate']:.4f}"
    log(f"{tag} {emb:5s} {name:8s} n={r['n']:6d} n_pos={r['n_pos']:5d} base={base} AUC={auc} {tops}")


def _print_readout(readout: dict) -> None:
    pool = readout.get("by_source", {}).get("pool")
    if pool:
        log("[readout] ---- source=gt (GT nodes at GT coordinates; v1 readout) ----")
    for emb, block in readout["per_embryo"].items():
        for name, r in block.items():
            _print_rank_line("[readout]", emb, name, r)
    for emb, r in readout.get("divnet_comparison", {}).get("per_embryo", {}).items():
        a, b = r["divnet_in_sample"]["auc"], r["daughternet_oof_p_par"]["auc"]
        log(f"[readout] {emb:5s} parent-vs-cont AUC DivNet(in-sample)={a} DaughterNet(OOF)={b}")
    if not pool:
        return
    log("[readout pool] ---- source=pool (real candidate pool at PREDICTED coordinates; the block that matters) ----")
    for emb, block in pool["per_embryo"].items():
        for name, r in block.items():
            _print_rank_line("[readout pool]", emb, name, r)
    for emb, r in pool["vs_cont"].items():
        _print_rank_line("[readout pool]", emb, "par|cont", r["parent_vs_cont_p_par"])
        _print_rank_line("[readout pool]", emb, "dau|cont", r["daughter_vs_cont_p_dau"])
    for emb, r in pool["gate_roles"].items():
        log(f"[readout pool] {emb:5s} gate roles: AUC p_par parent vs cont-as-u ({r['n_cont_as_u']}) "
            f"{_r4(r['auc_p_par_parent_vs_cont_as_u'])}; p_dau daughter vs cont-as-c1 ({r['n_cont_as_c1']}) "
            f"{_r4(r['auc_p_dau_daughter_vs_cont_as_c1'])}; vs cont-as-v ({r['n_cont_as_v']}) "
            f"{_r4(r['auc_p_dau_daughter_vs_cont_as_v'])}")
    for emb, r in pool["divnet_parent_vs_cont"].items():
        log(f"[readout pool] {emb:5s} parent-vs-cont AUC DivNet(in-sample)={_r4(r['divnet_in_sample_auc'])} "
            f"DaughterNet(OOF)={_r4(r['daughternet_oof_auc'])}")


def step_verify(cfg: dict, state: dict, per_fold: int = 2) -> dict:
    """End-to-end contract check on a few OOF samples per fold: raw frames -> tools.daughternet.score_nodes with the
    fold's best.pt (the submission-patch path) must reproduce oof.csv, and DivNet's divnet_score_division must
    reproduce p_divnet. Differences are only fp rounding (float32 cache) or fp16 rounding (float16 cache)."""
    import pandas as pd
    import torch
    from tools.divnet_infer import divnet_score_division

    work = Path(state["work_dir"])
    oof = pd.read_csv(work / "oof.csv", float_precision="round_trip")
    oof = oof[oof["p_par"].notna()]
    if "source" not in oof.columns:
        oof = oof.assign(source="gt")
    reader = FrameReader(Path(state["data_dir"]))
    report = {"checked": 0, "checked_pool": 0, "max_abs_diff_daughternet": 0.0, "max_abs_diff_divnet": None,
              "rows": []}
    divnet_bundle = None
    if "p_divnet" in oof.columns and oof["p_divnet"].notna().any():
        ckpt = _find_divnet_checkpoint(cfg)
        divnet_bundle = load_divnet_mitosis_model(ckpt) if ckpt else None
        report["max_abs_diff_divnet"] = 0.0
    for fold, g in oof.groupby("fold"):
        best = _fold_dir(work, int(fold)) / "best.pt"
        if not best.is_file():
            continue
        bundle = load_daughternet([best], device="cpu")
        # prefer parents/daughters so the check covers the classes that matter; pool rows (predicted coordinates)
        # get one division-class and one cont row per fold on top
        gt, gp = g[g["source"] == "gt"], g[g["source"] == "pool"]
        pick = pd.concat([gt[gt["label"] != "cont"].head(per_fold), gt[gt["label"] == "cont"].head(1),
                          gp[gp["label"] != "cont"].head(1), gp[gp["label"] == "cont"].head(1)])
        for _, r in pick.iterrows():
            node = {"t": int(r["t"]), "z": float(r["z"]), "y": float(r["y"]), "x": float(r["x"])}
            bundle["prepared_frames"] = {}
            p = score_nodes(r["stem"], [node], bundle, {}, reader, batch=1)[0]
            diff = float(np.abs(p - r[["p_cont", "p_par", "p_dau"]].to_numpy(dtype=np.float64)).max())
            report["max_abs_diff_daughternet"] = max(report["max_abs_diff_daughternet"], diff)
            row = {"fold": int(fold), "stem": r["stem"], "t": int(r["t"]), "label": r["label"], "diff": diff,
                   "source": r["source"]}
            report["checked_pool"] += int(r["source"] == "pool")
            if divnet_bundle is not None:
                divnet_bundle["prepared_frames"] = {}
                q = divnet_score_division(r["stem"], int(r["t"]), node, None, None, divnet_bundle, {}, reader)
                d2 = abs(q - float(r["p_divnet"]))
                report["max_abs_diff_divnet"] = max(report["max_abs_diff_divnet"], d2)
                row["diff_divnet"] = d2
            report["rows"].append(row)
            report["checked"] += 1
    # OOF p come from the fold worker (cuDNN on GPU, batch 256); the re-score runs on CPU one node at a time, so
    # only fp rounding separates them; a contract drift (crop shift, normalisation) moves p far more than this.
    tol = 1e-3 if state.get("cache_dtype", cfg["cache_dtype"]) == "float32" else 2e-2
    report["tolerance"] = tol
    report["pass"] = bool(report["checked"] > 0 and report["max_abs_diff_daughternet"] <= tol
                          and (report["max_abs_diff_divnet"] is None or report["max_abs_diff_divnet"] <= tol))
    (work / "contract_check.json").write_text(json.dumps(report, indent=1, default=_json_default))
    log(f"[verify] {report['checked']} samples ({report['checked_pool']} pool) re-scored from raw frames via "
        f"score_nodes: max |dp| DaughterNet "
        f"{report['max_abs_diff_daughternet']:.2e}, DivNet {report['max_abs_diff_divnet']} -> "
        f"{'PASS' if report['pass'] else 'FAIL'} (tol {tol})")
    state["verify"] = report
    return state


def run_all(cfg: dict) -> dict:
    state = step_prepare(cfg)
    state = step_build_cache(cfg, state)
    state = step_train(cfg, state)
    state = step_oof(cfg, state)
    state = step_verify(cfg, state)
    return state


if __name__ == "__main__":
    if len(sys.argv) > 1 and sys.argv[1] == "worker":
        sys.exit(worker_main(sys.argv[2:]))
    run_all(config_from_env())


In [ ]:
%%writefile /kaggle/working/kdn_src/splits.json
{
 "folds": {
  "44b6_0113de3b": 4,
  "44b6_0b24845f": 1,
  "44b6_0c582fdc": 2,
  "44b6_0db75fae": 2,
  "44b6_12dfb391": 4,
  "44b6_144b256d": 3,
  "44b6_1574802b": 0,
  "44b6_18ced818": 1,
  "44b6_1d530831": 4,
  "44b6_24264f12": 4,
  "44b6_267148e4": 2,
  "44b6_2a2eff9f": 3,
  "44b6_2f31fc2f": 1,
  "44b6_33b596bf": 2,
  "44b6_341df25f": 1,
  "44b6_3a861e03": 4,
  "44b6_3bb3690f": 2,
  "44b6_40c45f5a": 4,
  "44b6_415c0a3a": 3,
  "44b6_53f95252": 0,
  "44b6_551a5dba": 4,
  "44b6_5740d24b": 0,
  "44b6_587a1e22": 2,
  "44b6_5f15d135": 0,
  "44b6_668e0cc7": 3,
  "44b6_66f9292d": 3,
  "44b6_706092f0": 0,
  "44b6_71a4179f": 4,
  "44b6_74d0c52e": 0,
  "44b6_7a302da0": 4,
  "44b6_7e557709": 4,
  "44b6_808952d6": 1,
  "44b6_81c256f0": 1,
  "44b6_87bba6c4": 1,
  "44b6_8cc6506c": 2,
  "44b6_8f5ab931": 4,
  "44b6_8f9ecab4": 3,
  "44b6_90724892": 4,
  "44b6_949adeb1": 4,
  "44b6_95029e92": 1,
  "44b6_97725144": 0,
  "44b6_996155de": 1,
  "44b6_9be80b04": 2,
  "44b6_9bfa6a0a": 4,
  "44b6_a21120c2": 4,
  "44b6_a2bb48bb": 4,
  "44b6_aaf8b0ea": 1,
  "44b6_abf82518": 0,
  "44b6_b2c44266": 4,
  "44b6_c15fded2": 4,
  "44b6_c50204e0": 4,
  "44b6_c771cb04": 0,
  "44b6_c8e2a523": 4,
  "44b6_c96cfa10": 4,
  "44b6_cf2536e8": 3,
  "44b6_cf8fed6b": 3,
  "44b6_d29c9ab2": 4,
  "44b6_d2f34f90": 2,
  "44b6_d5e7d891": 4,
  "44b6_d754aa59": 1,
  "44b6_d78e09d9": 4,
  "44b6_db3c847b": 1,
  "44b6_ddf577ad": 3,
  "44b6_deabac95": 0,
  "44b6_e28840c6": 3,
  "44b6_e29f0176": 2,
  "44b6_e31261b4": 3,
  "44b6_e35b117d": 0,
  "44b6_e57ff5c6": 0,
  "44b6_eb2880fc": 3,
  "44b6_f28707c6": 2,
  "6bba_05b6850b": 2,
  "6bba_05db0fb1": 3,
  "6bba_062c8d37": 0,
  "6bba_07477033": 3,
  "6bba_07e24132": 0,
  "6bba_085bf656": 0,
  "6bba_09961292": 2,
  "6bba_0c7fa718": 2,
  "6bba_0e7c0d07": 1,
  "6bba_12665c0e": 1,
  "6bba_13f19531": 1,
  "6bba_15403b9a": 0,
  "6bba_1d0d8384": 2,
  "6bba_1ebfb80d": 2,
  "6bba_1f58c2f6": 2,
  "6bba_207c6aaf": 1,
  "6bba_20852818": 1,
  "6bba_2312ac41": 4,
  "6bba_23af9eeb": 1,
  "6bba_2540cd90": 2,
  "6bba_2646afc7": 3,
  "6bba_268e1230": 3,
  "6bba_2819ca14": 3,
  "6bba_283bf9f1": 4,
  "6bba_312f0dc3": 4,
  "6bba_32db13fc": 3,
  "6bba_337b1b3a": 1,
  "6bba_372c8cb8": 0,
  "6bba_3a1849c2": 1,
  "6bba_3abfe10a": 4,
  "6bba_3c5691b6": 0,
  "6bba_3db54e20": 0,
  "6bba_3fda6b25": 1,
  "6bba_43fea39d": 2,
  "6bba_474be664": 0,
  "6bba_48816121": 3,
  "6bba_4aa14c89": 2,
  "6bba_4f99ce20": 4,
  "6bba_4ffd3da3": 2,
  "6bba_55b7eebe": 0,
  "6bba_55c70843": 0,
  "6bba_57b7cc1e": 3,
  "6bba_5a4d9360": 4,
  "6bba_5b28472a": 1,
  "6bba_5c039895": 4,
  "6bba_5c824876": 2,
  "6bba_5dfe9ad1": 4,
  "6bba_5f89039d": 3,
  "6bba_61dd1e0d": 0,
  "6bba_61ecbe65": 2,
  "6bba_6321a359": 4,
  "6bba_6479435d": 3,
  "6bba_67ebd073": 2,
  "6bba_6ca87370": 3,
  "6bba_6db0e9b4": 3,
  "6bba_6feb10f0": 1,
  "6bba_6feeb0b1": 0,
  "6bba_705ec2c9": 1,
  "6bba_718b21f9": 2,
  "6bba_74686d6a": 3,
  "6bba_767a1e17": 0,
  "6bba_76db78c1": 3,
  "6bba_784a78c9": 1,
  "6bba_786893ac": 3,
  "6bba_789f8168": 2,
  "6bba_78a7bd97": 0,
  "6bba_7af54fde": 3,
  "6bba_7b5d3b2c": 3,
  "6bba_7d3058ae": 1,
  "6bba_7f87b3d8": 2,
  "6bba_80d12824": 2,
  "6bba_825bd1c6": 3,
  "6bba_87289e13": 3,
  "6bba_8b7818bf": 1,
  "6bba_907271db": 0,
  "6bba_91951b3a": 4,
  "6bba_96833384": 0,
  "6bba_969618f6": 0,
  "6bba_971fa5e0": 4,
  "6bba_9a41d029": 4,
  "6bba_9e23430b": 4,
  "6bba_a5e926bb": 2,
  "6bba_a90a0b9c": 1,
  "6bba_ab78413d": 0,
  "6bba_acd782a8": 2,
  "6bba_ae82a791": 3,
  "6bba_aeee7805": 4,
  "6bba_af149c94": 1,
  "6bba_afb141ff": 4,
  "6bba_b1ae37b9": 3,
  "6bba_b204cac7": 1,
  "6bba_b329af44": 1,
  "6bba_b693381b": 3,
  "6bba_bb9f20c3": 0,
  "6bba_bbb708ca": 0,
  "6bba_c27cba08": 0,
  "6bba_c328f2fd": 2,
  "6bba_c73a1d11": 0,
  "6bba_cdcfe533": 4,
  "6bba_cf35214c": 1,
  "6bba_cff5865f": 3,
  "6bba_d0fc38b5": 4,
  "6bba_d1acb6ff": 4,
  "6bba_d2b9fc0c": 3,
  "6bba_d3da753b": 2,
  "6bba_d5eae175": 1,
  "6bba_d6ecebbb": 1,
  "6bba_d82a4fc6": 1,
  "6bba_debd7bfa": 0,
  "6bba_df673a83": 1,
  "6bba_e16ffc58": 3,
  "6bba_e5e44988": 3,
  "6bba_ebdf3b34": 2,
  "6bba_ebff6e76": 4,
  "6bba_ed9377fd": 0,
  "6bba_edf14583": 4,
  "6bba_eebc57a5": 0,
  "6bba_ef7b4f7e": 2,
  "6bba_f17befbc": 1,
  "6bba_f1fde7e0": 3,
  "6bba_f20478e9": 2,
  "6bba_f4ae811c": 1,
  "6bba_f8ffd5e7": 0,
  "6bba_fbc898dc": 0,
  "6bba_fc516dc6": 2,
  "6bba_fc5f39dc": 1,
  "6bba_fc83837d": 2,
  "6bba_fe670320": 2
 },
 "selection": [
  "6bba_05db0fb1",
  "6bba_07477033",
  "6bba_085bf656",
  "6bba_0c7fa718",
  "6bba_12665c0e",
  "6bba_15403b9a",
  "6bba_1ebfb80d",
  "6bba_207c6aaf",
  "6bba_2312ac41",
  "6bba_2540cd90",
  "6bba_268e1230",
  "6bba_283bf9f1",
  "6bba_32db13fc",
  "6bba_372c8cb8",
  "6bba_3abfe10a",
  "6bba_3db54e20",
  "6bba_43fea39d",
  "6bba_48816121",
  "6bba_4f99ce20",
  "6bba_55b7eebe",
  "6bba_57b7cc1e",
  "6bba_5b28472a",
  "6bba_5c824876",
  "6bba_5f89039d",
  "6bba_61ecbe65",
  "6bba_6479435d",
  "6bba_6ca87370",
  "6bba_6feb10f0",
  "6bba_705ec2c9",
  "6bba_74686d6a",
  "6bba_76db78c1",
  "6bba_786893ac",
  "6bba_78a7bd97",
  "6bba_7b5d3b2c",
  "6bba_7f87b3d8",
  "6bba_825bd1c6",
  "6bba_8b7818bf",
  "6bba_91951b3a",
  "6bba_969618f6",
  "6bba_9a41d029",
  "6bba_a5e926bb",
  "6bba_ab78413d",
  "6bba_ae82a791",
  "6bba_af149c94",
  "6bba_b1ae37b9",
  "6bba_b329af44",
  "6bba_bb9f20c3",
  "6bba_c27cba08",
  "6bba_c73a1d11",
  "6bba_cf35214c",
  "6bba_d0fc38b5",
  "6bba_d2b9fc0c",
  "6bba_d5eae175",
  "6bba_d82a4fc6",
  "6bba_df673a83",
  "6bba_e5e44988",
  "6bba_ebff6e76",
  "6bba_edf14583",
  "6bba_ef7b4f7e",
  "6bba_f1fde7e0",
  "6bba_f4ae811c",
  "6bba_fbc898dc",
  "6bba_fc5f39dc",
  "6bba_fe670320"
 ],
 "confirmation": [
  "44b6_0113de3b",
  "44b6_0b24845f",
  "44b6_0c582fdc",
  "44b6_0db75fae",
  "44b6_12dfb391",
  "44b6_144b256d",
  "44b6_1574802b",
  "44b6_18ced818",
  "44b6_1d530831",
  "44b6_24264f12",
  "44b6_267148e4",
  "44b6_2a2eff9f",
  "44b6_2f31fc2f",
  "44b6_33b596bf",
  "44b6_341df25f",
  "44b6_3a861e03",
  "44b6_3bb3690f",
  "44b6_40c45f5a",
  "44b6_415c0a3a",
  "44b6_53f95252",
  "44b6_551a5dba",
  "44b6_5740d24b",
  "44b6_587a1e22",
  "44b6_5f15d135",
  "44b6_668e0cc7",
  "44b6_66f9292d",
  "44b6_706092f0",
  "44b6_71a4179f",
  "44b6_74d0c52e",
  "44b6_7a302da0",
  "44b6_7e557709",
  "44b6_808952d6",
  "44b6_81c256f0",
  "44b6_87bba6c4",
  "44b6_8cc6506c",
  "44b6_8f5ab931",
  "44b6_8f9ecab4",
  "44b6_90724892",
  "44b6_949adeb1",
  "44b6_95029e92",
  "44b6_97725144",
  "44b6_996155de",
  "44b6_9be80b04",
  "44b6_9bfa6a0a",
  "44b6_a21120c2",
  "44b6_a2bb48bb",
  "44b6_aaf8b0ea",
  "44b6_abf82518",
  "44b6_b2c44266",
  "44b6_c15fded2",
  "44b6_c50204e0",
  "44b6_c771cb04",
  "44b6_c8e2a523",
  "44b6_c96cfa10",
  "44b6_cf2536e8",
  "44b6_cf8fed6b",
  "44b6_d29c9ab2",
  "44b6_d2f34f90",
  "44b6_d5e7d891",
  "44b6_d754aa59",
  "44b6_d78e09d9",
  "44b6_db3c847b",
  "44b6_ddf577ad",
  "44b6_deabac95",
  "44b6_e28840c6",
  "44b6_e29f0176",
  "44b6_e31261b4",
  "44b6_e35b117d",
  "44b6_e57ff5c6",
  "44b6_eb2880fc",
  "44b6_f28707c6",
  "6bba_05b6850b",
  "6bba_062c8d37",
  "6bba_07e24132",
  "6bba_09961292",
  "6bba_0e7c0d07",
  "6bba_13f19531",
  "6bba_1d0d8384",
  "6bba_1f58c2f6",
  "6bba_20852818",
  "6bba_23af9eeb",
  "6bba_2646afc7",
  "6bba_2819ca14",
  "6bba_312f0dc3",
  "6bba_337b1b3a",
  "6bba_3a1849c2",
  "6bba_3c5691b6",
  "6bba_3fda6b25",
  "6bba_474be664",
  "6bba_4aa14c89",
  "6bba_4ffd3da3",
  "6bba_55c70843",
  "6bba_5a4d9360",
  "6bba_5c039895",
  "6bba_5dfe9ad1",
  "6bba_61dd1e0d",
  "6bba_6321a359",
  "6bba_67ebd073",
  "6bba_6db0e9b4",
  "6bba_6feeb0b1",
  "6bba_718b21f9",
  "6bba_767a1e17",
  "6bba_784a78c9",
  "6bba_789f8168",
  "6bba_7af54fde",
  "6bba_7d3058ae",
  "6bba_80d12824",
  "6bba_87289e13",
  "6bba_907271db",
  "6bba_96833384",
  "6bba_971fa5e0",
  "6bba_9e23430b",
  "6bba_a90a0b9c",
  "6bba_acd782a8",
  "6bba_aeee7805",
  "6bba_afb141ff",
  "6bba_b204cac7",
  "6bba_b693381b",
  "6bba_bbb708ca",
  "6bba_c328f2fd",
  "6bba_cdcfe533",
  "6bba_cff5865f",
  "6bba_d1acb6ff",
  "6bba_d3da753b",
  "6bba_d6ecebbb",
  "6bba_debd7bfa",
  "6bba_e16ffc58",
  "6bba_ebdf3b34",
  "6bba_ed9377fd",
  "6bba_eebc57a5",
  "6bba_f17befbc",
  "6bba_f20478e9",
  "6bba_f8ffd5e7",
  "6bba_fc516dc6",
  "6bba_fc83837d"
 ],
 "divisions_per_video": {
  "44b6_0113de3b": 0,
  "44b6_0b24845f": 0,
  "44b6_0c582fdc": 0,
  "44b6_0db75fae": 0,
  "44b6_12dfb391": 1,
  "44b6_144b256d": 0,
  "44b6_1574802b": 0,
  "44b6_18ced818": 0,
  "44b6_1d530831": 0,
  "44b6_24264f12": 0,
  "44b6_267148e4": 1,
  "44b6_2a2eff9f": 1,
  "44b6_2f31fc2f": 0,
  "44b6_33b596bf": 0,
  "44b6_341df25f": 1,
  "44b6_3a861e03": 0,
  "44b6_3bb3690f": 0,
  "44b6_40c45f5a": 0,
  "44b6_415c0a3a": 0,
  "44b6_53f95252": 0,
  "44b6_551a5dba": 0,
  "44b6_5740d24b": 0,
  "44b6_587a1e22": 1,
  "44b6_5f15d135": 1,
  "44b6_668e0cc7": 0,
  "44b6_66f9292d": 0,
  "44b6_706092f0": 1,
  "44b6_71a4179f": 0,
  "44b6_74d0c52e": 1,
  "44b6_7a302da0": 1,
  "44b6_7e557709": 0,
  "44b6_808952d6": 0,
  "44b6_81c256f0": 0,
  "44b6_87bba6c4": 0,
  "44b6_8cc6506c": 0,
  "44b6_8f5ab931": 0,
  "44b6_8f9ecab4": 0,
  "44b6_90724892": 0,
  "44b6_949adeb1": 0,
  "44b6_95029e92": 0,
  "44b6_97725144": 0,
  "44b6_996155de": 1,
  "44b6_9be80b04": 1,
  "44b6_9bfa6a0a": 0,
  "44b6_a21120c2": 2,
  "44b6_a2bb48bb": 0,
  "44b6_aaf8b0ea": 1,
  "44b6_abf82518": 0,
  "44b6_b2c44266": 0,
  "44b6_c15fded2": 0,
  "44b6_c50204e0": 2,
  "44b6_c771cb04": 0,
  "44b6_c8e2a523": 1,
  "44b6_c96cfa10": 0,
  "44b6_cf2536e8": 0,
  "44b6_cf8fed6b": 0,
  "44b6_d29c9ab2": 0,
  "44b6_d2f34f90": 2,
  "44b6_d5e7d891": 2,
  "44b6_d754aa59": 1,
  "44b6_d78e09d9": 0,
  "44b6_db3c847b": 0,
  "44b6_ddf577ad": 0,
  "44b6_deabac95": 1,
  "44b6_e28840c6": 2,
  "44b6_e29f0176": 0,
  "44b6_e31261b4": 0,
  "44b6_e35b117d": 0,
  "44b6_e57ff5c6": 0,
  "44b6_eb2880fc": 1,
  "44b6_f28707c6": 0,
  "6bba_05b6850b": 0,
  "6bba_05db0fb1": 3,
  "6bba_062c8d37": 1,
  "6bba_07477033": 0,
  "6bba_07e24132": 2,
  "6bba_085bf656": 1,
  "6bba_09961292": 4,
  "6bba_0c7fa718": 0,
  "6bba_0e7c0d07": 1,
  "6bba_12665c0e": 3,
  "6bba_13f19531": 0,
  "6bba_15403b9a": 0,
  "6bba_1d0d8384": 1,
  "6bba_1ebfb80d": 0,
  "6bba_1f58c2f6": 0,
  "6bba_207c6aaf": 1,
  "6bba_20852818": 3,
  "6bba_2312ac41": 2,
  "6bba_23af9eeb": 0,
  "6bba_2540cd90": 0,
  "6bba_2646afc7": 0,
  "6bba_268e1230": 1,
  "6bba_2819ca14": 1,
  "6bba_283bf9f1": 0,
  "6bba_312f0dc3": 0,
  "6bba_32db13fc": 1,
  "6bba_337b1b3a": 2,
  "6bba_372c8cb8": 0,
  "6bba_3a1849c2": 0,
  "6bba_3abfe10a": 1,
  "6bba_3c5691b6": 2,
  "6bba_3db54e20": 1,
  "6bba_3fda6b25": 1,
  "6bba_43fea39d": 0,
  "6bba_474be664": 1,
  "6bba_48816121": 5,
  "6bba_4aa14c89": 0,
  "6bba_4f99ce20": 2,
  "6bba_4ffd3da3": 3,
  "6bba_55b7eebe": 1,
  "6bba_55c70843": 0,
  "6bba_57b7cc1e": 3,
  "6bba_5a4d9360": 0,
  "6bba_5b28472a": 0,
  "6bba_5c039895": 3,
  "6bba_5c824876": 1,
  "6bba_5dfe9ad1": 0,
  "6bba_5f89039d": 0,
  "6bba_61dd1e0d": 0,
  "6bba_61ecbe65": 1,
  "6bba_6321a359": 1,
  "6bba_6479435d": 0,
  "6bba_67ebd073": 1,
  "6bba_6ca87370": 1,
  "6bba_6db0e9b4": 0,
  "6bba_6feb10f0": 0,
  "6bba_6feeb0b1": 0,
  "6bba_705ec2c9": 1,
  "6bba_718b21f9": 0,
  "6bba_74686d6a": 2,
  "6bba_767a1e17": 0,
  "6bba_76db78c1": 0,
  "6bba_784a78c9": 1,
  "6bba_786893ac": 3,
  "6bba_789f8168": 0,
  "6bba_78a7bd97": 2,
  "6bba_7af54fde": 2,
  "6bba_7b5d3b2c": 0,
  "6bba_7d3058ae": 2,
  "6bba_7f87b3d8": 3,
  "6bba_80d12824": 1,
  "6bba_825bd1c6": 0,
  "6bba_87289e13": 1,
  "6bba_8b7818bf": 2,
  "6bba_907271db": 1,
  "6bba_91951b3a": 0,
  "6bba_96833384": 0,
  "6bba_969618f6": 3,
  "6bba_971fa5e0": 0,
  "6bba_9a41d029": 2,
  "6bba_9e23430b": 1,
  "6bba_a5e926bb": 0,
  "6bba_a90a0b9c": 0,
  "6bba_ab78413d": 2,
  "6bba_acd782a8": 0,
  "6bba_ae82a791": 0,
  "6bba_aeee7805": 1,
  "6bba_af149c94": 0,
  "6bba_afb141ff": 4,
  "6bba_b1ae37b9": 0,
  "6bba_b204cac7": 2,
  "6bba_b329af44": 1,
  "6bba_b693381b": 0,
  "6bba_bb9f20c3": 3,
  "6bba_bbb708ca": 0,
  "6bba_c27cba08": 0,
  "6bba_c328f2fd": 2,
  "6bba_c73a1d11": 0,
  "6bba_cdcfe533": 4,
  "6bba_cf35214c": 0,
  "6bba_cff5865f": 0,
  "6bba_d0fc38b5": 0,
  "6bba_d1acb6ff": 1,
  "6bba_d2b9fc0c": 2,
  "6bba_d3da753b": 1,
  "6bba_d5eae175": 0,
  "6bba_d6ecebbb": 0,
  "6bba_d82a4fc6": 0,
  "6bba_debd7bfa": 4,
  "6bba_df673a83": 4,
  "6bba_e16ffc58": 1,
  "6bba_e5e44988": 0,
  "6bba_ebdf3b34": 2,
  "6bba_ebff6e76": 0,
  "6bba_ed9377fd": 0,
  "6bba_edf14583": 0,
  "6bba_eebc57a5": 0,
  "6bba_ef7b4f7e": 2,
  "6bba_f17befbc": 0,
  "6bba_f1fde7e0": 0,
  "6bba_f20478e9": 1,
  "6bba_f4ae811c": 0,
  "6bba_f8ffd5e7": 2,
  "6bba_fbc898dc": 0,
  "6bba_fc516dc6": 0,
  "6bba_fc5f39dc": 2,
  "6bba_fc83837d": 0,
  "6bba_fe670320": 2
 },
 "summary": {
  "total": {
   "videos": 199,
   "divisions": 151,
   "videos_with_divisions": 87,
   "44b6": {
    "videos": 71,
    "divisions": 26,
    "videos_with_divisions": 21
   },
   "6bba": {
    "videos": 128,
    "divisions": 125,
    "videos_with_divisions": 66
   }
  },
  "per_embryo": {
   "44b6": {
    "videos": 71,
    "divisions": 26,
    "videos_with_divisions": 21
   },
   "6bba": {
    "videos": 128,
    "divisions": 125,
    "videos_with_divisions": 66
   }
  },
  "per_fold": {
   "0": {
    "videos": 39,
    "divisions": 30,
    "videos_with_divisions": 18,
    "44b6": {
     "videos": 12,
     "divisions": 4,
     "videos_with_divisions": 4
    },
    "6bba": {
     "videos": 27,
     "divisions": 26,
     "videos_with_divisions": 14
    },
    "selection_videos": 12,
    "confirmation_videos": 27,
    "division_count_histogram": {
     "0": 21,
     "1": 10,
     "2": 5,
     "3": 2,
     "4": 1
    }
   },
   "1": {
    "videos": 39,
    "divisions": 30,
    "videos_with_divisions": 18,
    "44b6": {
     "videos": 12,
     "divisions": 4,
     "videos_with_divisions": 4
    },
    "6bba": {
     "videos": 27,
     "divisions": 26,
     "videos_with_divisions": 14
    },
    "selection_videos": 14,
    "confirmation_videos": 25,
    "division_count_histogram": {
     "0": 21,
     "1": 10,
     "2": 5,
     "3": 2,
     "4": 1
    }
   },
   "2": {
    "videos": 38,
    "divisions": 30,
    "videos_with_divisions": 18,
    "44b6": {
     "videos": 11,
     "divisions": 5,
     "videos_with_divisions": 4
    },
    "6bba": {
     "videos": 27,
     "divisions": 25,
     "videos_with_divisions": 14
    },
    "selection_videos": 10,
    "confirmation_videos": 28,
    "division_count_histogram": {
     "0": 20,
     "1": 10,
     "2": 5,
     "3": 2,
     "4": 1
    }
   },
   "3": {
    "videos": 39,
    "divisions": 30,
    "videos_with_divisions": 16,
    "44b6": {
     "videos": 12,
     "divisions": 4,
     "videos_with_divisions": 3
    },
    "6bba": {
     "videos": 27,
     "divisions": 26,
     "videos_with_divisions": 13
    },
    "selection_videos": 19,
    "confirmation_videos": 20,
    "division_count_histogram": {
     "0": 23,
     "1": 8,
     "2": 4,
     "3": 3,
     "5": 1
    }
   },
   "4": {
    "videos": 44,
    "divisions": 31,
    "videos_with_divisions": 17,
    "44b6": {
     "videos": 24,
     "divisions": 9,
     "videos_with_divisions": 6
    },
    "6bba": {
     "videos": 20,
     "divisions": 22,
     "videos_with_divisions": 11
    },
    "selection_videos": 9,
    "confirmation_videos": 35,
    "division_count_histogram": {
     "0": 27,
     "1": 8,
     "2": 6,
     "3": 1,
     "4": 2
    }
   }
  },
  "per_split": {
   "selection": {
    "videos": 64,
    "divisions": 64,
    "videos_with_divisions": 32,
    "44b6": {
     "videos": 0,
     "divisions": 0,
     "videos_with_divisions": 0
    },
    "6bba": {
     "videos": 64,
     "divisions": 64,
     "videos_with_divisions": 32
    }
   },
   "confirmation": {
    "videos": 135,
    "divisions": 87,
    "videos_with_divisions": 55,
    "44b6": {
     "videos": 71,
     "divisions": 26,
     "videos_with_divisions": 21
    },
    "6bba": {
     "videos": 64,
     "divisions": 61,
     "videos_with_divisions": 34
    }
   }
  },
  "division_count_histogram": {
   "44b6": {
    "0": 50,
    "1": 16,
    "2": 5
   },
   "6bba": {
    "0": 62,
    "1": 30,
    "2": 20,
    "3": 10,
    "4": 5,
    "5": 1
   }
  },
  "shared_annotation": {
   "min_shared_nodes": 8,
   "min_shared_ids": 4,
   "links": 62,
   "links_per_embryo": {
    "44b6": 62,
    "6bba": 0
   },
   "shared_gt_nodes": 2429,
   "shared_gt_divisions": 2,
   "groups": 11,
   "grouped_videos": 64,
   "group_sizes": [
    24,
    11,
    6,
    4,
    3,
    3,
    3,
    3,
    3,
    2,
    2
   ],
   "cross_fold_links": 0,
   "selection_confirmation_links": 0
  }
 },
 "meta": {
  "generator": "tools/make_splits.py",
  "seed": 20260923,
  "n_folds": 5,
  "division_definition": "GT node with >= 2 outgoing GT edges (== tracksdata dividing_nodes())",
  "reader": "tracksdata IndexedRXGraph.from_geff (same as tools/score_geffs.py)",
  "fold_method": "videos that share GT annotations (overlapping space-time crops: >= 8 identical GT nodes or >= 4 shared node ids at one (dt,dz,dy,dx) offset) are merged into one unit (leak_groups) that sits in one fold; units placed greedily then improved by best moves/swaps on a weighted squared deviation of videos + divisions per fold (w=2.0), per (embryo, fold) (w=1.0) and video counts per (embryo, min(n_div,3)) stratum (w=1.0); sha256(seed:...) ordering, no RNG",
  "selection_rule": "6bba stems with odd index (0-based) in sorted 6bba order",
  "confirmation_rule": "6bba even-index stems + all 44b6 stems",
  "leak_groups": [
   [
    "44b6_0113de3b",
    "44b6_12dfb391",
    "44b6_1d530831",
    "44b6_24264f12",
    "44b6_3a861e03",
    "44b6_40c45f5a",
    "44b6_551a5dba",
    "44b6_71a4179f",
    "44b6_7a302da0",
    "44b6_7e557709",
    "44b6_8f5ab931",
    "44b6_90724892",
    "44b6_949adeb1",
    "44b6_9bfa6a0a",
    "44b6_a21120c2",
    "44b6_a2bb48bb",
    "44b6_b2c44266",
    "44b6_c15fded2",
    "44b6_c50204e0",
    "44b6_c8e2a523",
    "44b6_c96cfa10",
    "44b6_d29c9ab2",
    "44b6_d5e7d891",
    "44b6_d78e09d9"
   ],
   [
    "44b6_144b256d",
    "44b6_2a2eff9f",
    "44b6_415c0a3a",
    "44b6_668e0cc7",
    "44b6_66f9292d",
    "44b6_8f9ecab4",
    "44b6_cf2536e8",
    "44b6_cf8fed6b",
    "44b6_ddf577ad",
    "44b6_e31261b4",
    "44b6_eb2880fc"
   ],
   [
    "44b6_341df25f",
    "44b6_808952d6",
    "44b6_81c256f0",
    "44b6_996155de",
    "44b6_aaf8b0ea",
    "44b6_db3c847b"
   ],
   [
    "44b6_53f95252",
    "44b6_5f15d135",
    "44b6_97725144",
    "44b6_e57ff5c6"
   ],
   [
    "44b6_0b24845f",
    "44b6_2f31fc2f",
    "44b6_87bba6c4"
   ],
   [
    "44b6_0c582fdc",
    "44b6_33b596bf",
    "44b6_f28707c6"
   ],
   [
    "44b6_0db75fae",
    "44b6_267148e4",
    "44b6_587a1e22"
   ],
   [
    "44b6_1574802b",
    "44b6_abf82518",
    "44b6_deabac95"
   ],
   [
    "44b6_3bb3690f",
    "44b6_8cc6506c",
    "44b6_d2f34f90"
   ],
   [
    "44b6_5740d24b",
    "44b6_74d0c52e"
   ],
   [
    "44b6_95029e92",
    "44b6_d754aa59"
   ]
  ],
  "shared_annotation_links": [
   {
    "a": "44b6_0113de3b",
    "b": "44b6_c8e2a523",
    "shared_nodes": 14,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     45,
     50,
     11,
     148
    ]
   },
   {
    "a": "44b6_0b24845f",
    "b": "44b6_2f31fc2f",
    "shared_nodes": 10,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -49,
     12,
     88,
     209
    ]
   },
   {
    "a": "44b6_0c582fdc",
    "b": "44b6_33b596bf",
    "shared_nodes": 7,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -44,
     20,
     221,
     88
    ]
   },
   {
    "a": "44b6_0db75fae",
    "b": "44b6_267148e4",
    "shared_nodes": 9,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -28,
     -22,
     -51,
     240
    ]
   },
   {
    "a": "44b6_0db75fae",
    "b": "44b6_587a1e22",
    "shared_nodes": 25,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -39,
     10,
     53,
     -227
    ]
   },
   {
    "a": "44b6_12dfb391",
    "b": "44b6_3a861e03",
    "shared_nodes": 52,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -86,
     6,
     -114,
     75
    ]
   },
   {
    "a": "44b6_12dfb391",
    "b": "44b6_7e557709",
    "shared_nodes": 99,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -15,
     -34,
     227,
     -152
    ]
   },
   {
    "a": "44b6_12dfb391",
    "b": "44b6_9bfa6a0a",
    "shared_nodes": 62,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     74,
     -26,
     132,
     -89
    ]
   },
   {
    "a": "44b6_12dfb391",
    "b": "44b6_b2c44266",
    "shared_nodes": 12,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     88,
     -4,
     -87,
     87
    ]
   },
   {
    "a": "44b6_12dfb391",
    "b": "44b6_d29c9ab2",
    "shared_nodes": 16,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -98,
     -18,
     104,
     -83
    ]
   },
   {
    "a": "44b6_144b256d",
    "b": "44b6_66f9292d",
    "shared_nodes": 18,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -91,
     12,
     -78,
     -9
    ]
   },
   {
    "a": "44b6_1574802b",
    "b": "44b6_abf82518",
    "shared_nodes": 37,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     61,
     -44,
     -19,
     -143
    ]
   },
   {
    "a": "44b6_1574802b",
    "b": "44b6_deabac95",
    "shared_nodes": 51,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -47,
     38,
     -45,
     188
    ]
   },
   {
    "a": "44b6_1d530831",
    "b": "44b6_24264f12",
    "shared_nodes": 10,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -8,
     -34,
     -208,
     -155
    ]
   },
   {
    "a": "44b6_1d530831",
    "b": "44b6_7a302da0",
    "shared_nodes": 9,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     97,
     13,
     98,
     -113
    ]
   },
   {
    "a": "44b6_1d530831",
    "b": "44b6_c96cfa10",
    "shared_nodes": 94,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     7,
     23,
     184,
     -180
    ]
   },
   {
    "a": "44b6_24264f12",
    "b": "44b6_8f5ab931",
    "shared_nodes": 28,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     58,
     -42,
     -160,
     206
    ]
   },
   {
    "a": "44b6_24264f12",
    "b": "44b6_d78e09d9",
    "shared_nodes": 73,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -2,
     -58,
     -130,
     -3
    ]
   },
   {
    "a": "44b6_2a2eff9f",
    "b": "44b6_cf2536e8",
    "shared_nodes": 8,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     92,
     15,
     -62,
     1
    ]
   },
   {
    "a": "44b6_2a2eff9f",
    "b": "44b6_e31261b4",
    "shared_nodes": 11,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -32,
     -55,
     -78,
     71
    ]
   },
   {
    "a": "44b6_2a2eff9f",
    "b": "44b6_eb2880fc",
    "shared_nodes": 54,
    "shared_divisions": 1,
    "offset_tzyx_a_minus_b": [
     -61,
     51,
     3,
     -102
    ]
   },
   {
    "a": "44b6_2f31fc2f",
    "b": "44b6_87bba6c4",
    "shared_nodes": 11,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -24,
     60,
     47,
     -20
    ]
   },
   {
    "a": "44b6_33b596bf",
    "b": "44b6_f28707c6",
    "shared_nodes": 9,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -43,
     -14,
     209,
     120
    ]
   },
   {
    "a": "44b6_341df25f",
    "b": "44b6_81c256f0",
    "shared_nodes": 11,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     94,
     4,
     -178,
     -70
    ]
   },
   {
    "a": "44b6_341df25f",
    "b": "44b6_db3c847b",
    "shared_nodes": 26,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     2,
     -24,
     -235,
     -47
    ]
   },
   {
    "a": "44b6_3a861e03",
    "b": "44b6_c8e2a523",
    "shared_nodes": 77,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     36,
     -16,
     -177,
     -177
    ]
   },
   {
    "a": "44b6_3a861e03",
    "b": "44b6_c96cfa10",
    "shared_nodes": 90,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     63,
     12,
     -200,
     80
    ]
   },
   {
    "a": "44b6_3a861e03",
    "b": "44b6_d29c9ab2",
    "shared_nodes": 134,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -12,
     -24,
     218,
     -158
    ]
   },
   {
    "a": "44b6_3bb3690f",
    "b": "44b6_8cc6506c",
    "shared_nodes": 36,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -91,
     28,
     -77,
     79
    ]
   },
   {
    "a": "44b6_40c45f5a",
    "b": "44b6_90724892",
    "shared_nodes": 33,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -67,
     3,
     185,
     123
    ]
   },
   {
    "a": "44b6_40c45f5a",
    "b": "44b6_d5e7d891",
    "shared_nodes": 21,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     20,
     -35,
     -214,
     -110
    ]
   },
   {
    "a": "44b6_415c0a3a",
    "b": "44b6_66f9292d",
    "shared_nodes": 14,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     92,
     -20,
     -25,
     27
    ]
   },
   {
    "a": "44b6_415c0a3a",
    "b": "44b6_ddf577ad",
    "shared_nodes": 12,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -6,
     58,
     -101,
     -40
    ]
   },
   {
    "a": "44b6_415c0a3a",
    "b": "44b6_eb2880fc",
    "shared_nodes": 10,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -58,
     -52,
     72,
     -29
    ]
   },
   {
    "a": "44b6_53f95252",
    "b": "44b6_97725144",
    "shared_nodes": 35,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -79,
     27,
     -51,
     128
    ]
   },
   {
    "a": "44b6_551a5dba",
    "b": "44b6_7a302da0",
    "shared_nodes": 31,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -91,
     -17,
     42,
     29
    ]
   },
   {
    "a": "44b6_5740d24b",
    "b": "44b6_74d0c52e",
    "shared_nodes": 9,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     91,
     -30,
     -40,
     75
    ]
   },
   {
    "a": "44b6_5f15d135",
    "b": "44b6_97725144",
    "shared_nodes": 15,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     0,
     -39,
     214,
     -150
    ]
   },
   {
    "a": "44b6_5f15d135",
    "b": "44b6_e57ff5c6",
    "shared_nodes": 57,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -65,
     50,
     -47,
     67
    ]
   },
   {
    "a": "44b6_668e0cc7",
    "b": "44b6_eb2880fc",
    "shared_nodes": 62,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     50,
     -53,
     90,
     46
    ]
   },
   {
    "a": "44b6_71a4179f",
    "b": "44b6_9bfa6a0a",
    "shared_nodes": 18,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -91,
     -11,
     57,
     -46
    ]
   },
   {
    "a": "44b6_7a302da0",
    "b": "44b6_c96cfa10",
    "shared_nodes": 44,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -90,
     10,
     86,
     -67
    ]
   },
   {
    "a": "44b6_7e557709",
    "b": "44b6_9bfa6a0a",
    "shared_nodes": 22,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     89,
     8,
     -95,
     63
    ]
   },
   {
    "a": "44b6_7e557709",
    "b": "44b6_d29c9ab2",
    "shared_nodes": 74,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -83,
     16,
     -123,
     69
    ]
   },
   {
    "a": "44b6_808952d6",
    "b": "44b6_81c256f0",
    "shared_nodes": 37,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -72,
     4,
     175,
     103
    ]
   },
   {
    "a": "44b6_808952d6",
    "b": "44b6_aaf8b0ea",
    "shared_nodes": 12,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     88,
     -16,
     -157,
     -36
    ]
   },
   {
    "a": "44b6_8cc6506c",
    "b": "44b6_d2f34f90",
    "shared_nodes": 114,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -71,
     39,
     -104,
     76
    ]
   },
   {
    "a": "44b6_8f5ab931",
    "b": "44b6_949adeb1",
    "shared_nodes": 118,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -61,
     35,
     170,
     52
    ]
   },
   {
    "a": "44b6_8f5ab931",
    "b": "44b6_c50204e0",
    "shared_nodes": 4,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     17,
     13,
     50,
     254
    ]
   },
   {
    "a": "44b6_8f5ab931",
    "b": "44b6_d5e7d891",
    "shared_nodes": 102,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     35,
     56,
     101,
     56
    ]
   },
   {
    "a": "44b6_8f5ab931",
    "b": "44b6_d78e09d9",
    "shared_nodes": 57,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -60,
     -16,
     30,
     -209
    ]
   },
   {
    "a": "44b6_8f9ecab4",
    "b": "44b6_ddf577ad",
    "shared_nodes": 28,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -93,
     -6,
     45,
     -31
    ]
   },
   {
    "a": "44b6_90724892",
    "b": "44b6_c15fded2",
    "shared_nodes": 7,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -39,
     2,
     239,
     50
    ]
   },
   {
    "a": "44b6_949adeb1",
    "b": "44b6_c50204e0",
    "shared_nodes": 28,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     78,
     -22,
     -120,
     202
    ]
   },
   {
    "a": "44b6_949adeb1",
    "b": "44b6_d5e7d891",
    "shared_nodes": 28,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     96,
     21,
     -69,
     4
    ]
   },
   {
    "a": "44b6_95029e92",
    "b": "44b6_d754aa59",
    "shared_nodes": 17,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -64,
     19,
     -218,
     78
    ]
   },
   {
    "a": "44b6_996155de",
    "b": "44b6_aaf8b0ea",
    "shared_nodes": 30,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -90,
     22,
     -8,
     -115
    ]
   },
   {
    "a": "44b6_a21120c2",
    "b": "44b6_d78e09d9",
    "shared_nodes": 22,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     50,
     9,
     -54,
     222
    ]
   },
   {
    "a": "44b6_a2bb48bb",
    "b": "44b6_c50204e0",
    "shared_nodes": 4,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -99,
     -25,
     23,
     72
    ]
   },
   {
    "a": "44b6_a2bb48bb",
    "b": "44b6_d5e7d891",
    "shared_nodes": 78,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -81,
     18,
     74,
     -126
    ]
   },
   {
    "a": "44b6_c50204e0",
    "b": "44b6_d5e7d891",
    "shared_nodes": 164,
    "shared_divisions": 1,
    "offset_tzyx_a_minus_b": [
     18,
     43,
     51,
     -198
    ]
   },
   {
    "a": "44b6_cf8fed6b",
    "b": "44b6_eb2880fc",
    "shared_nodes": 29,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     87,
     9,
     90,
     -96
    ]
   }
  ],
  "data_dir": "biohub-cell-tracking-during-development/train"
 }
}


In [ ]:
%%writefile /kaggle/working/kdn_src/pool_samples.csv
stem,node_id,t,z,y,x,label,dist_um,gt_id,n_triples,n_u,n_c1,n_v
44b6_0113de3b,6476,28,3.16,77.28,74.07999999999998,cont,0.5411518386737739,39000000003,1,0,1,0
44b6_0113de3b,6989,30,6.84,85.28,76.63999999999999,cont,0.677061666319992,41000000003,1,1,0,0
44b6_0113de3b,7245,31,8.519999999999998,91.36,76.63999999999999,cont,0.8069560861657881,42000000003,1,0,1,0
44b6_0113de3b,11243,46,27.880000000000003,116.0,106.72,cont,1.9963705348707128,57000000003,1,1,0,0
44b6_0113de3b,11247,46,30.0,130.71999999999997,114.72,cont,6.259372597353498,57000000003,1,1,0,0
44b6_0113de3b,11523,47,29.24,111.84,112.16,cont,3.7903974066052775,58000000003,3,2,1,0
44b6_0113de3b,11786,48,29.279999999999994,109.91999999999999,117.27999999999997,cont,6.753766657392015,59000000003,4,0,2,2
44b6_0113de3b,11801,48,31.799999999999994,125.92,119.84,cont,2.0647733683143006,59000000003,5,0,1,4
44b6_0113de3b,12357,50,32.52,129.28,127.36,cont,1.0256826994738735,61000000003,3,3,0,0
44b6_0113de3b,12634,51,35.13333333333333,127.46666666666667,113.06666666666665,cont,0.28917086932584724,62000000003,2,0,0,2
44b6_0113de3b,17404,67,55.16000000000001,127.36,173.92000000000002,cont,1.4389280949720689,78000000003,1,1,0,0
44b6_0113de3b,17700,68,56.68,128.0,179.35999999999999,cont,0.581377674149944,79000000003,1,0,1,0
44b6_0b24845f,6131,14,61.160000000000004,175.2,229.28000000000003,cont,2.785346701938548,93000000036,1,1,0,0
44b6_0b24845f,6474,15,60.0,173.06666666666666,229.06666666666666,cont,1.5981461699592887,94000000036,5,1,0,4
44b6_0b24845f,6495,15,61.52000000000001,178.55999999999997,225.28,cont,4.586819772456734,94000000036,5,1,1,3
44b6_0b24845f,6823,16,58.08,173.44,225.75999999999996,cont,2.0902579087280233,95000000036,1,0,1,0
44b6_0b24845f,6857,16,61.99999999999999,179.99999999999997,236.5333333333333,cont,6.103912821798067,95000000036,3,0,0,3
44b6_0b24845f,6858,16,61.68,183.51999999999995,221.28000000000006,cont,6.386105278258395,95000000036,1,0,1,0
44b6_0b24845f,7139,17,56.160000000000004,171.36,220.0,cont,3.467652529089378,96000000036,1,1,0,0
44b6_0b24845f,7477,18,55.16,169.28,216.64000000000004,cont,3.042744700513014,97000000036,1,0,1,0
44b6_0b24845f,9680,25,50.16000000000001,161.44000000000005,185.92000000000002,cont,1.426487162578043,104000000036,1,1,0,0
44b6_0b24845f,9984,26,50.16,159.35999999999999,181.27999999999997,cont,1.3941893926221156,105000000036,1,0,1,0
44b6_0b24845f,11465,31,48.000000000000014,152.63999999999996,157.92000000000002,cont,1.7956066175529335,110000000036,1,1,0,0
44b6_0b24845f,11729,32,46.68,152.0,153.92000000000002,cont,2.189412378356342,111000000036,1,0,1,0
44b6_0b24845f,14223,40,39.04,158.07999999999998,140.64,cont,6.3223986745538285,119000000036,1,1,0,0
44b6_0b24845f,14521,41,38.86666666666667,150.93333333333334,136.00000000000003,cont,3.734162156881906,120000000036,2,0,0,2
44b6_0b24845f,16546,48,33.31999999999999,135.36000000000004,123.35999999999999,cont,1.4858756887775162,127000000038,1,1,0,0
44b6_0b24845f,16835,49,32.839999999999996,132.8,124.63999999999999,cont,1.0451921175076004,128000000038,1,0,1,0
44b6_0c582fdc,8681,22,43.0,244.8,195.35999999999999,cont,1.3257450735341223,110000000085,1,1,0,0
44b6_0c582fdc,9054,23,42.160000000000004,238.72000000000003,194.08000000000004,cont,1.238522935798909,111000000085,1,0,1,0
44b6_0c582fdc,9432,24,41.519999999999996,232.64,192.0,cont,1.088264817266463,112000000085,2,2,0,0
44b6_0c582fdc,9806,25,40.64000000000001,225.92000000000002,189.28000000000003,cont,1.581685216786203,113000000085,2,0,2,0
44b6_0c582fdc,18762,51,30.840000000000007,115.36000000000004,114.88000000000002,cont,3.6152506828710895,139000000082,3,0,0,3
44b6_0c582fdc,31868,62,29.137765808105467,72.81306488037112,85.74210998535155,cont,6.55288807787337,150000000077,1,1,0,0
44b6_0c582fdc,22236,63,28.901229248046874,75.59691772460937,87.36982666015625,cont,4.307271328798951,151000000077,1,0,1,0
44b6_0c582fdc,24142,70,28.850902404785153,47.83533264160155,81.40513671875001,cont,1.3940834144863576,158000000077,1,1,0,0
44b6_0c582fdc,24386,71,25.999999999999996,55.466666666666654,73.06666666666666,cont,6.247391816901587,159000000077,4,0,0,4
44b6_0c582fdc,24401,71,28.170902404785156,45.75533264160156,81.40513671875001,cont,0.4453925486412092,159000000077,1,0,1,0
44b6_0c582fdc,25214,74,25.480000000000004,45.920000000000016,71.36,cont,6.030348507756415,162000000076,1,1,0,0
44b6_0c582fdc,25492,75,25.519999999999996,43.36,69.28,cont,6.258022476190066,163000000076,1,0,1,0
44b6_0c582fdc,28357,86,23.319999999999997,22.72,80.64,cont,6.944456084712466,174000000075,1,0,1,0
44b6_0db75fae,614,2,22.320000000000004,96.63999999999999,29.44000000000001,cont,1.437551086048763,43000000019,5,2,1,2
44b6_0db75fae,19718,2,22.601316528320314,96.34963958740235,28.389332122802735,cont,1.3583757108716172,43000000019,3,3,0,0
44b6_0db75fae,872,3,21.96,93.92000000000002,25.120000000000005,cont,2.4173206370897566,44000000019,2,0,2,0
44b6_0db75fae,878,3,23.053918457031244,104.03095092773438,27.533036499023435,cont,6.486295468235425,44000000019,3,0,3,0
44b6_0db75fae,1100,4,6.5200000000000005,75.19999999999999,245.28000000000003,cont,1.6353669313031876,45000000002,1,1,0,0
44b6_0db75fae,1380,5,6.880000000000001,76.80000000000003,246.08000000000004,cont,1.5285368821196148,46000000002,1,0,1,0
44b6_0db75fae,4493,16,28.16,93.92000000000002,24.0,cont,1.4337805140606417,57000000020,5,5,0,0
44b6_0db75fae,4785,17,28.880000000000003,88.16,25.28000000000001,cont,5.401359203941165,58000000020,5,0,5,0
44b6_0db75fae,5070,18,31.000000000000007,104.80000000000001,14.560000000000002,cont,6.984116913576123,59000000020,1,0,1,0
44b6_0db75fae,5887,21,29.839999999999996,123.52000000000001,23.200000000000003,cont,6.365044309547268,62000000020,2,2,0,0
44b6_0db75fae,8314,30,19.160000000000004,152.0,37.92,cont,6.688409984817017,71000000026,1,0,1,0
44b6_0db75fae,12103,45,31.68,139.35999999999999,74.72,cont,0.7497926796788571,86000000029,1,1,0,0
44b6_0db75fae,12329,46,32.68000000000001,137.92000000000002,74.72,cont,0.5332872701462016,87000000030,1,0,1,0
44b6_0db75fae,13990,54,33.000000000000014,132.00000000000006,144.64000000000001,cont,1.8026516926461196,95000000025,1,1,0,0
44b6_0db75fae,14181,55,32.839999999999996,132.00000000000006,148.64000000000001,cont,0.6106195624118155,96000000025,1,0,1,0
44b6_0db75fae,14196,55,37.160000000000004,152.8,115.83999999999999,cont,6.364463474048693,96000000030,2,0,0,2
44b6_0db75fae,15038,60,34.84,128.64000000000004,173.28000000000003,cont,1.9131571583380058,101000000025,1,1,0,0
44b6_0db75fae,15058,60,42.160000000000004,138.72000000000003,129.92000000000002,cont,0.8300235312929465,101000000030,1,1,0,0
44b6_0db75fae,15195,61,35.00000000000001,128.0,178.08,cont,3.2965479103146533,102000000025,1,0,1,0
44b6_0db75fae,15217,61,42.36,137.28000000000003,134.72,cont,1.3973110337358756,102000000030,1,0,1,0
44b6_0db75fae,16541,70,49.519999999999996,134.08,179.36000000000004,cont,0.9354694677005768,111000000030,1,1,0,0
44b6_0db75fae,16671,71,49.120000000000005,133.27999999999997,184.64,cont,0.43724278152990087,112000000030,1,0,1,0
44b6_0db75fae,16802,72,48.839999999999996,132.64000000000001,189.27999999999997,cont,0.5994906692351285,113000000030,1,1,0,0
44b6_0db75fae,16924,73,48.640000000000015,132.00000000000006,194.07999999999998,cont,2.247263961465125,114000000030,1,0,1,0
44b6_0db75fae,16939,73,53.733333333333334,128.53333333333333,191.46666666666658,cont,6.234404500765625,114000000030,7,0,0,7
44b6_0db75fae,17980,82,54.360000000000014,132.00000000000006,228.64000000000004,cont,0.7581979045737457,123000000031,1,1,0,0
44b6_0db75fae,18088,83,55.00000000000001,132.00000000000006,233.27999999999997,cont,1.232110105672351,124000000031,1,0,1,0
44b6_12dfb391,446,1,7.24,240.0000000000001,56.0,cont,0.9885824763266188,107000000041,1,1,0,0
44b6_12dfb391,4214,9,39.480000000000004,196.00000000000003,93.27999999999999,cont,1.6393987083684183,115000000063,1,1,0,0
44b6_12dfb391,4895,11,20.160000000000007,235.99999999999994,96.64000000000001,cont,5.098384088365675,117000000080,1,1,0,0
44b6_12dfb391,5089,11,37.36,46.08000000000001,193.27999999999997,cont,1.2213472325673707,117000000037,1,1,0,0
44b6_12dfb391,5533,12,37.03999999999999,44.00000000000001,194.71999999999997,cont,1.6383514007990008,118000000037,1,0,1,0
44b6_12dfb391,6430,14,41.519999999999996,187.99999999999997,94.56,cont,1.7235727791422049,120000000063,1,0,1,0
44b6_12dfb391,6444,14,42.999999999999986,106.56,141.28000000000003,cont,2.383825706716001,120000000056,5,0,0,5
44b6_12dfb391,8065,18,30.20930114746094,146.953974609375,24.281386413574218,cont,6.101521368772342,124000000043,1,1,0,0
44b6_12dfb391,8077,18,31.000000000000007,154.08,12.0,cont,0.37374999999999403,124000000043,1,1,0,0
44b6_12dfb391,8515,19,31.159999999999997,152.64,12.640000000000004,cont,0.3322320348792354,125000000043,1,0,1,0
44b6_12dfb391,8947,20,30.569301147460934,144.873974609375,27.641386413574217,cont,6.340811067896502,126000000044,1,1,0,0
44b6_12dfb391,8950,20,31.320000000000004,151.2,13.280000000000001,cont,0.7218000502216667,126000000044,1,1,0,0
44b6_12dfb391,9078,20,42.0,177.92,100.16000000000001,cont,1.218208212909438,126000000064,5,0,1,4
44b6_12dfb391,9308,21,21.999999999999993,213.06666666666666,116.0,cont,6.37442944069856,127000000045,3,0,0,3
44b6_12dfb391,9337,21,24.680000000000003,201.28000000000006,128.64000000000001,cont,1.8953380305370242,127000000045,1,0,1,0
44b6_12dfb391,9397,21,30.56930114746094,144.23397460937503,28.92138641357422,cont,6.4139964648033025,127000000044,2,1,1,0
44b6_12dfb391,9410,21,31.680000000000003,151.36,15.36,cont,1.100450191966912,127000000044,2,1,1,0
44b6_12dfb391,10504,23,44.48,87.36000000000001,150.07999999999998,cont,5.217464811812334,129000000058,5,0,0,5
44b6_12dfb391,10749,24,24.0,200.0,131.36000000000004,cont,2.4900705938788223,130000000045,1,1,0,0
44b6_12dfb391,10967,24,44.0,86.08,151.35999999999999,cont,5.88566385911564,130000000057,1,1,0,0
44b6_12dfb391,10986,24,47.00000000000001,110.71999999999998,152.0,cont,5.449180127551306,130000000057,1,1,0,0
44b6_12dfb391,11193,25,24.0,200.64,132.00000000000006,cont,2.447284528308892,131000000045,1,0,1,0
44b6_12dfb391,11422,25,45.13333333333332,95.46666666666664,151.99999999999997,cont,1.8543679666008746,131000000057,5,0,0,5
44b6_12dfb391,14080,31,31.159999999999997,101.28,115.36000000000001,cont,6.617980833494456,137000000079,1,0,1,0
44b6_12dfb391,14081,31,30.679999999999996,132.00000000000006,28.640000000000004,cont,5.997218756432006,137000000044,1,0,1,0
44b6_12dfb391,14107,31,32.84000000000001,142.08,22.080000000000005,cont,0.45644858691860374,137000000044,1,0,1,0
44b6_12dfb391,15031,33,33.480000000000004,141.92000000000002,25.28,cont,0.938006529828024,139000000044,1,0,1,0
44b6_12dfb391,16324,36,16.16,15.359999999999996,136.0,cont,6.9971677305892825,142000000079,1,0,1,0
44b6_12dfb391,18351,40,17.000000000000004,183.99999999999997,60.0,cont,1.9902104160113323,146000000042,1,1,0,0
44b6_12dfb391,18838,41,14.133333333333331,179.99999999999997,63.999999999999986,cont,1.5210131172273953,147000000042,5,0,0,5
44b6_12dfb391,19735,42,50.86666666666666,72.53333333333333,167.99999999999994,cont,6.6328183688299935,148000000056,5,0,0,5
44b6_12dfb391,19908,43,19.866666666666664,204.53333333333325,131.99999999999997,cont,1.0305911194822037,149000000045,4,0,0,4
44b6_12dfb391,21054,45,32.99999999999999,123.99999999999999,123.99999999999999,cont,1.7235727791421929,151000000073,5,0,0,5
44b6_12dfb391,21131,45,38.0,155.35999999999999,44.00000000000001,cont,0.48232671758881424,151000000044,1,0,1,0
44b6_12dfb391,21467,46,24.680000000000003,203.36,72.64,cont,0.8020325741514535,152000000074,1,1,0,0
44b6_12dfb391,21940,47,24.320000000000007,204.00000000000003,74.72,cont,1.1827946250300552,153000000074,1,0,1,0
44b6_12dfb391,22034,47,33.000000000000014,125.28000000000002,125.28000000000002,cont,1.9413819980879803,153000000073,1,1,0,0
44b6_12dfb391,22214,47,46.67999999999999,76.0,160.00000000000006,cont,1.2612345142755825,153000000054,1,0,1,0
44b6_12dfb391,22532,48,33.03292419433594,126.95699707031251,127.09324218749998,cont,1.9213702949736622,154000000073,1,0,1,0
44b6_12dfb391,22871,49,17.32,7.360000000000001,152.0,cont,2.2519592164379887,155000000076,1,0,1,0
44b6_12dfb391,23084,49,38.16,154.56,48.0,cont,1.3838284756428538,155000000044,1,1,0,0
44b6_12dfb391,24121,51,40.31999999999999,154.08000000000004,64.64,cont,6.062447937920776,157000000044,1,0,1,0
44b6_12dfb391,24441,52,23.840000000000003,211.36,129.28,cont,2.0593950112836503,158000000045,1,1,0,0
44b6_12dfb391,24452,52,25.0,201.28000000000006,72.64,cont,1.1919128533580081,158000000074,1,0,1,0
44b6_12dfb391,24888,53,15.679999999999998,192.0,65.28,cont,2.207130490025457,159000000042,1,1,0,0
44b6_12dfb391,24944,53,22.866666666666656,203.99999999999991,131.99999999999997,cont,5.936688906588534,159000000045,3,0,0,3
44b6_12dfb391,24953,53,24.0,212.64,128.00000000000006,cont,1.8299116269645583,159000000045,5,0,1,4
44b6_12dfb391,25372,54,15.799999999999997,192.64,66.08,cont,0.7420042536940153,160000000042,1,0,1,0
44b6_12dfb391,25833,55,8.0,239.35999999999996,51.360000000000014,cont,2.494361767065885,161000000043,1,1,0,0
44b6_12dfb391,25954,55,25.0,200.0,71.99999999999999,cont,7.105427357601002e-15,161000000073,1,0,1,0
44b6_12dfb391,26576,56,34.0,53.280000000000015,205.28,cont,6.330683834507935,162000000039,1,0,1,0
44b6_12dfb391,26591,56,35.13333333333333,27.999999999999996,219.46666666666658,cont,5.733292832342403,162000000039,4,0,0,4
44b6_12dfb391,26940,57,17.160000000000004,196.64000000000001,66.08000000000001,cont,1.2419295722785537,163000000042,1,1,0,0
44b6_12dfb391,27477,58,18.0,190.93333333333328,79.99999999999999,cont,6.226870047646555,164000000042,5,1,0,4
44b6_12dfb391,27478,58,17.84,198.72,64.64000000000001,cont,1.6935119758950594,164000000042,2,1,1,0
44b6_12dfb391,27998,59,18.080000000000002,190.55999999999997,76.95999999999998,cont,5.05218223270895,165000000042,1,0,1,0
44b6_12dfb391,27999,59,17.679999999999996,199.36,64.0,cont,1.435621054631067,165000000042,1,0,1,0
44b6_12dfb391,29489,62,9.2,242.71999999999997,41.92000000000001,cont,1.5203088707562065,168000000043,2,0,0,2
44b6_12dfb391,29789,62,41.68000000000001,160.00000000000006,68.0,cont,1.6451068544322918,168000000044,1,1,0,0
44b6_12dfb391,32207,67,32.839999999999996,34.72,221.27999999999997,cont,5.97348320182621,173000000037,1,1,0,0
44b6_12dfb391,32636,68,23.999999999999993,211.46666666666658,107.99999999999997,cont,6.390117220386315,174000000043,3,0,0,3
44b6_12dfb391,33122,69,14.84,8.640000000000002,154.72000000000003,cont,1.4836525873667274,175000000073,1,0,1,0
44b6_12dfb391,33208,69,24.680000000000003,213.44,90.72,cont,1.2554636842617162,175000000042,1,1,0,0
44b6_12dfb391,33393,69,41.47999999999999,44.64000000000001,164.64,cont,6.098428204258875,175000000050,1,0,1,0
44b6_12dfb391,33478,69,46.640000000000015,36.64000000000001,156.0,cont,5.239888029576594,175000000050,6,1,0,5
44b6_12dfb391,33657,70,6.32,245.92,41.92000000000001,cont,1.2462807167729122,176000000041,3,0,0,3
44b6_12dfb391,50890,70,48.39956649780274,74.83383941650389,178.5130554199219,cont,6.534754210856173,176000000051,1,1,0,0
44b6_12dfb391,34590,71,44.84,89.27999999999999,174.55999999999995,cont,5.600972153340878,177000000051,1,0,1,0
44b6_12dfb391,37141,76,31.680000000000003,154.08000000000004,146.07999999999998,cont,0.6412061388508401,182000000070,1,1,0,0
44b6_12dfb391,37715,77,31.32,152.64,144.64000000000001,cont,1.298577346560459,183000000070,1,0,1,0
44b6_12dfb391,37920,77,44.52,80.0,176.00000000000003,cont,6.431162049933116,183000000051,1,1,0,0
44b6_12dfb391,38472,78,43.360000000000014,30.72,170.72000000000003,cont,1.1989813384702717,184000000050,1,1,0,0
44b6_12dfb391,39619,80,41.32,27.36,168.64000000000004,cont,0.599490669235135,186000000050,1,1,0,0
44b6_12dfb391,40149,81,40.477917480468754,26.027406005859376,167.6114990234375,cont,0.8154565929582352,187000000049,1,0,1,0
44b6_12dfb391,50723,83,40.15531433105469,29.521663513183597,165.8458728027344,cont,1.492855457853129,189000000049,1,1,0,0
44b6_12dfb391,41460,84,9.52,248.00000000000006,32.0,cont,2.2000003551135863,190000000040,1,0,1,0
44b6_12dfb391,41733,84,40.000000000000014,24.639999999999993,163.36,cont,1.1035652450127176,190000000048,3,0,0,3
44b6_12dfb391,47137,94,22.959999999999997,216.0,97.44,cont,4.298178756752676,200000000039,1,1,0,0
44b6_12dfb391,47761,95,23.199999999999996,216.0,96.63999999999999,cont,4.221279611977858,201000000039,1,0,1,0
44b6_12dfb391,48352,96,29.839999999999996,49.760000000000005,214.08,cont,1.5022243923595382,202000000035,2,0,0,2
44b6_144b256d,1102,2,31.680000000000003,112.64,118.72000000000003,cont,1.072253759611024,331000000014,1,1,0,0
44b6_144b256d,1524,3,31.32,113.28000000000003,118.72,cont,0.7914267180225851,332000000014,1,0,1,0
44b6_144b256d,2053,4,45.519999999999996,60.640000000000015,121.92000000000002,cont,2.945869990342402,333000000009,1,1,0,0
44b6_144b256d,2505,5,45.16,60.160000000000004,122.88000000000002,cont,2.612967661491423,334000000009,1,0,1,0
44b6_144b256d,2524,5,47.86666666666666,67.46666666666667,135.99999999999994,cont,5.852005828963428,334000000009,3,0,0,3
44b6_144b256d,2857,6,31.680000000000003,116.0,118.72,cont,0.8401422870561861,335000000014,1,0,0,1
44b6_144b256d,4850,11,31.680000000000003,119.35999999999999,119.36000000000001,cont,4.243272984972811,340000000014,1,1,0,0
44b6_144b256d,5231,12,31.32,119.36000000000001,117.92000000000002,cont,3.4465326779243015,341000000014,1,0,1,0
44b6_144b256d,5258,12,34.999999999999986,116.0,103.99999999999999,cont,5.510643111289265,341000000014,2,0,0,2
44b6_144b256d,7671,18,27.866666666666667,119.99999999999997,111.46666666666665,cont,5.1606366020956065,347000000013,1,0,0,1
44b6_144b256d,7769,18,45.0,63.36,116.0,cont,2.8136911366566166,347000000008,1,1,0,0
44b6_144b256d,8176,19,45.0,63.36000000000001,115.35999999999999,cont,1.999278385443112,348000000008,1,0,1,0
44b6_144b256d,14456,35,29.840000000000003,128.0,118.72000000000003,cont,6.765349445889683,364000000012,1,0,1,0
44b6_144b256d,23337,57,32.68,125.92000000000002,130.08,cont,1.1059554692662779,386000000011,1,1,0,0
44b6_144b256d,23747,58,32.32,126.72000000000001,129.28,cont,0.6073675267085048,387000000011,1,0,1,0
44b6_144b256d,25767,63,32.0,128.0,128.0,cont,1.1490485194281397,392000000012,1,1,0,0
44b6_144b256d,26114,64,28.320000000000004,135.36000000000004,122.55999999999997,cont,6.345848667633049,393000000012,1,0,1,0
44b6_144b256d,26127,64,30.999999999999996,111.99999999999999,135.99999999999994,cont,6.358818311015024,393000000012,3,0,0,3
44b6_144b256d,26133,64,31.84,128.0,128.0,cont,1.639640299730403,393000000012,1,0,1,0
44b6_144b256d,28284,70,31.000000000000007,124.63999999999999,131.35999999999999,cont,1.762357263723798,399000000012,1,1,0,0
44b6_144b256d,28618,71,28.84000000000001,109.92000000000002,136.0,cont,6.990522602066303,400000000012,3,0,0,3
44b6_144b256d,28620,71,30.320000000000007,125.28000000000002,130.72,cont,0.7914267180225925,400000000012,1,0,1,0
44b6_1574802b,3532,11,58.0,108.0,231.36,cont,0.4823267175888096,150000000021,1,1,0,0
44b6_1574802b,3838,12,57.84,108.0,228.00000000000003,cont,0.9448784644069163,151000000021,1,0,1,0
44b6_1574802b,4728,15,57.84,108.0,226.72,cont,0.9620494075150132,154000000021,1,1,0,0
44b6_1574802b,5004,16,57.32000000000001,108.64000000000001,226.72,cont,1.4492595091977036,155000000021,1,0,1,0
44b6_1574802b,6175,20,54.36000000000001,112.00000000000001,217.92000000000002,cont,6.599280855706925,159000000021,2,2,0,0
44b6_1574802b,6471,21,55.99999999999999,108.53333333333333,203.99999999999994,cont,1.7778305384966189,160000000021,4,0,0,4
44b6_1574802b,6503,21,61.99999999999999,100.0,244.53333333333342,cont,2.188393666660032,160000000020,2,0,0,2
44b6_1574802b,6756,22,56.0,110.72000000000003,200.00000000000003,cont,1.7688630705908288,161000000021,1,1,0,0
44b6_1574802b,6792,22,62.0,100.00000000000001,246.72000000000006,cont,2.414314731968461,161000000020,1,1,0,0
44b6_1574802b,7025,23,51.73333333333333,100.0,200.53333333333333,cont,6.826397013106958,162000000021,5,0,0,5
44b6_1574802b,7034,23,55.68,112.0,196.64000000000001,cont,1.7748987647187138,162000000021,1,0,1,0
44b6_1574802b,7067,23,61.99999999999999,83.46666666666664,244.53333333333327,cont,6.462255865257565,162000000020,5,0,0,5
44b6_1574802b,7072,23,61.84,100.0,246.72,cont,1.9367518716913588,162000000020,1,0,1,0
44b6_1574802b,8755,29,51.68,116.0,171.35999999999996,cont,0.5401750295043174,168000000021,1,1,0,0
44b6_1574802b,9031,30,50.160000000000004,117.28,168.63999999999996,cont,0.6770616663200085,169000000021,1,0,1,0
44b6_1574802b,9317,31,49.0,119.36000000000001,165.28,cont,1.6355283932111977,170000000021,1,1,0,0
44b6_1574802b,9577,32,47.800000000000004,120.64000000000001,163.36,cont,1.4680865522849935,171000000021,1,0,1,0
44b6_1574802b,9903,33,59.546143798828126,103.24,212.66781982421878,cont,2.559645452291378,172000000020,1,1,0,0
44b6_1574802b,10169,34,59.02614379882812,104.52000000000002,208.82781982421875,cont,1.9090768205517352,173000000020,1,0,1,0
44b6_1574802b,10190,34,61.73333333333334,101.06666666666666,207.99999999999997,cont,3.6654667781046655,173000000020,7,0,0,7
44b6_1574802b,10380,35,45.36,124.00000000000003,156.79999999999995,cont,1.515350619493711,174000000022,1,1,0,0
44b6_1574802b,21775,35,58.538823547363286,106.44999999999999,204.06259460449223,cont,2.3815747179115685,174000000021,1,1,0,0
44b6_1574802b,10625,36,44.160000000000004,123.36,152.64,cont,2.004620444124026,175000000022,1,0,1,0
44b6_1574802b,10677,36,57.86614379882812,107.23999999999998,199.22781982421878,cont,2.2824779073420114,175000000021,1,0,1,0
44b6_1574802b,10702,36,61.520000000000024,100.64,200.0,cont,5.324448328230855,175000000021,6,0,0,6
44b6_1574802b,14305,52,26.84,136.64,115.35999999999999,cont,3.515149578965306,191000000022,1,0,0,1
44b6_1574802b,14548,53,42.599999999999994,125.06666666666666,151.46666666666664,cont,6.21153755399758,192000000021,3,0,0,3
44b6_1574802b,14897,55,23.680000000000007,133.92000000000004,104.64000000000001,cont,1.9657799024814655,194000000022,1,1,0,0
44b6_1574802b,15099,56,22.039999999999996,134.56000000000003,101.43999999999998,cont,0.8649214198411369,195000000022,3,0,1,2
44b6_1574802b,15106,56,24.866666666666667,137.06666666666666,109.06666666666663,cont,6.164121448281706,195000000022,2,0,0,2
44b6_1574802b,16170,62,15.160000000000004,132.00000000000006,77.28,cont,1.4287068016216635,201000000022,1,1,0,0
44b6_1574802b,16347,63,14.520000000000003,132.00000000000006,75.36,cont,0.9170818188689648,202000000022,1,0,1,0
44b6_1574802b,16877,66,11.840000000000003,132.00000000000006,67.35999999999999,cont,1.0824256614197756,205000000022,1,1,0,0
44b6_1574802b,17055,67,11.000000000000002,132.00000000000006,63.36,cont,1.0507356054212862,206000000022,1,0,1,0
44b6_1574802b,17546,70,8.0,132.00000000000006,49.92,cont,2.6014722394252234,209000000022,1,1,0,0
44b6_1574802b,17705,71,7.159999999999999,132.00000000000006,46.72,cont,2.304121646636749,210000000022,1,0,1,0
44b6_1574802b,18329,75,4.520000000000001,129.28,35.35999999999999,cont,1.3797197405632786,214000000022,1,1,0,0
44b6_1574802b,18473,76,3.4800000000000004,128.64000000000004,32.64,cont,1.0582481100857264,215000000022,2,1,1,0
44b6_1574802b,18622,77,2.4799999999999995,128.0,29.28,cont,0.7882506343162681,216000000022,1,0,1,0
44b6_1574802b,18681,77,24.480000000000004,124.00000000000003,76.0,cont,0.7800000000000082,216000000021,1,1,0,0
44b6_1574802b,18830,78,23.68,124.00000000000003,73.44,cont,0.5675880988886244,217000000021,1,0,1,0
44b6_1574802b,20494,90,9.68,116.63999999999999,17.28,cont,0.9198131603755243,229000000021,1,1,0,0
44b6_1574802b,20620,91,8.0,116.0,13.280000000000001,cont,0.9646534351776291,230000000021,1,0,1,0
44b6_1574802b,20745,92,6.640000000000001,116.0,9.28,cont,1.1627553482998914,231000000021,1,1,0,0
44b6_1574802b,20869,93,5.319999999999999,116.0,7.360000000000001,cont,1.0906885726457396,232000000021,1,0,1,0
44b6_1574802b,20871,93,5.999999999999998,104.53333333333332,11.999999999999996,cont,5.694524432026301,232000000021,4,0,0,4
44b6_18ced818,1547,1,50.999999999999986,246.08000000000004,7.040000000000002,cont,2.421468110981408,222000000006,1,1,0,0
44b6_18ced818,2391,2,50.839999999999996,241.28000000000003,8.64,cont,3.546671855415999,223000000006,3,2,1,0
44b6_18ced818,3207,3,46.86666666666666,252.0,7.46666666666667,cont,4.8733446263549105,224000000006,5,1,0,4
44b6_18ced818,3261,3,50.68000000000001,238.07999999999998,10.720000000000002,cont,4.567435344369985,224000000006,4,2,2,0
44b6_18ced818,4121,4,50.160000000000004,233.27999999999997,12.64,cont,2.01565513543365,225000000006,2,0,2,0
44b6_18ced818,5026,5,49.68,229.92,14.560000000000002,cont,4.439819992128062,226000000006,1,1,0,0
44b6_18ced818,5866,6,44.03999999999999,227.99999999999997,14.719999999999999,cont,3.71855251576472,227000000006,1,1,0,0
44b6_18ced818,5922,6,48.84000000000001,226.72,18.080000000000002,cont,4.818035939311807,227000000006,1,0,1,0
44b6_18ced818,6797,7,43.51999999999999,224.64000000000001,16.640000000000004,cont,5.951605481926718,228000000006,1,0,1,0
44b6_18ced818,9570,10,42.32,200.0,18.720000000000002,cont,6.092623229159999,231000000006,1,1,0,0
44b6_18ced818,9617,10,45.48000000000001,213.92000000000002,21.919999999999998,cont,2.4340851078999015,231000000006,1,1,0,0
44b6_18ced818,10491,11,40.999999999999986,215.46666666666664,32.00000000000001,cont,4.993479254932839,232000000006,7,0,0,7
44b6_18ced818,10505,11,41.84000000000001,198.08,21.280000000000005,cont,6.385939877966275,232000000006,1,0,1,0
44b6_18ced818,10548,11,45.16000000000001,212.0,23.360000000000007,cont,3.5364570017179817,232000000006,1,0,1,0
44b6_18ced818,21007,22,43.16,188.64000000000001,63.36000000000001,cont,1.8963130345488755,243000000006,1,1,0,0
44b6_18ced818,22015,23,42.84,184.0,61.92,cont,1.623780792625663,244000000006,2,1,1,0
44b6_18ced818,22999,24,42.16000000000001,180.0,61.44000000000001,cont,2.0150655231282375,245000000006,1,0,1,0
44b6_18ced818,25942,27,41.03999999999999,169.28,60.0,cont,0.4268525799383166,248000000006,1,1,0,0
44b6_18ced818,26894,28,40.160000000000004,167.36000000000004,61.27999999999999,cont,0.5994906692351478,249000000006,1,0,1,0
44b6_18ced818,28788,30,35.84,164.64,77.91999999999999,cont,5.3505700630867254,251000000006,1,1,0,0
44b6_18ced818,28847,30,38.52,164.0,66.08,cont,1.1499673908420238,251000000006,1,1,0,0
44b6_18ced818,29765,31,35.0,163.36,79.36,cont,5.008902699194706,252000000006,2,1,1,0
44b6_18ced818,29802,31,37.480000000000004,162.07999999999998,67.35999999999999,cont,1.5385240492108097,252000000006,1,0,1,0
44b6_18ced818,30716,32,34.32,161.28,81.44,cont,6.491239408810923,253000000006,1,0,1,0
44b6_18ced818,31658,33,34.0,160.00000000000003,82.56000000000002,cont,4.6588497775738675,254000000006,1,1,0,0
44b6_18ced818,31690,33,36.480000000000004,157.28000000000003,71.19999999999999,cont,2.444423500541597,254000000006,1,1,0,0
44b6_18ced818,32569,34,33.800000000000004,158.71999999999997,85.28,cont,3.8513871283603742,255000000006,1,0,1,0
44b6_18ced818,32622,34,36.399999999999984,155.03999999999996,75.35999999999999,cont,2.51518295805691,255000000006,1,0,1,0
44b6_18ced818,38360,40,36.28000000000001,145.44,94.72,cont,4.011117557801079,261000000006,1,1,0,0
44b6_18ced818,39204,41,30.26666666666666,147.46666666666667,103.46666666666663,cont,5.164969899476667,262000000006,9,0,0,9
44b6_18ced818,39331,41,36.52000000000001,141.92000000000002,97.28,cont,6.231551898403809,262000000006,1,0,1,0
44b6_18ced818,40180,42,31.360000000000003,144.95999999999995,104.96000000000002,cont,1.9616785701281507,263000000006,1,1,0,0
44b6_18ced818,41139,43,31.68,143.36,107.35999999999999,cont,1.5468258830585853,264000000006,1,0,1,0
44b6_18ced818,44898,47,31.840000000000003,133.92000000000002,122.08000000000003,cont,1.4341488111420022,268000000006,1,1,0,0
44b6_18ced818,45784,48,32.0,131.20000000000002,125.43999999999998,cont,1.746100118836262,269000000006,1,0,1,0
44b6_18ced818,48436,51,35.28,118.39999999999999,135.20000000000002,cont,6.39162305776084,272000000006,7,1,0,6
44b6_18ced818,49236,52,28.086361389160157,119.53295898437503,135.62507812500002,cont,6.12540828036412,273000000006,1,1,0,0
44b6_18ced818,49263,52,29.0,138.08000000000004,147.35999999999999,cont,6.471704035646257,273000000006,2,2,0,0
44b6_18ced818,49340,52,32.68,127.36,135.36000000000004,cont,2.888023220993896,273000000006,2,2,0,0
44b6_18ced818,50190,53,28.48,134.56,149.28000000000003,cont,4.685075372926251,274000000006,2,0,2,0
44b6_18ced818,50226,53,28.880000000000003,114.56000000000003,137.28,cont,6.066366808065583,274000000006,1,0,1,0
44b6_18ced818,50261,53,31.84,126.08000000000001,138.72000000000003,cont,3.3651235118491525,274000000006,2,0,2,0
44b6_18ced818,52042,55,29.64,108.0,144.0,cont,4.606122623747657,276000000007,1,1,0,0
44b6_18ced818,52976,56,30.519999999999992,105.28000000000002,148.0,cont,6.3323520758482665,277000000007,1,0,1,0
44b6_18ced818,53795,57,28.36,114.72,143.99999999999997,cont,1.363935596903325,278000000007,1,1,0,0
44b6_18ced818,54647,58,28.040000000000006,111.36,143.99999999999997,cont,1.8614586014735996,279000000007,1,0,1,0
44b6_18ced818,57174,61,26.52,101.28,144.64000000000001,cont,0.8457808891787509,282000000007,1,1,0,0
44b6_18ced818,58053,62,26.160000000000007,99.36000000000001,146.08,cont,0.5724522359987835,283000000007,1,0,1,0
44b6_18ced818,62253,67,23.64,94.72,154.08,cont,0.7833781015576097,288000000008,1,1,0,0
44b6_18ced818,63108,68,23.519999999999996,92.63999999999999,156.00000000000006,cont,0.9170818188689739,289000000008,1,0,1,0
44b6_18ced818,68783,75,21.68,69.92,160.64,cont,0.7594158939606237,296000000008,1,1,0,0
44b6_18ced818,69596,76,21.68,68.00000000000001,162.07999999999998,cont,1.034526612755813,297000000008,1,0,1,0
44b6_18ced818,70393,77,21.480000000000004,64.64,161.92,cont,0.9354694677005693,298000000008,1,1,0,0
44b6_18ced818,71209,78,21.360000000000003,64.0,164.64,cont,1.3002031091333275,299000000008,1,0,1,0
44b6_18ced818,78958,88,23.8,58.72,201.28000000000006,cont,1.5499808466558298,309000000008,1,1,0,0
44b6_18ced818,79699,89,23.319999999999997,56.63999999999999,205.44000000000003,cont,3.3024302131157888,310000000008,1,0,1,0
44b6_18ced818,80427,90,23.840000000000003,59.839999999999996,228.15999999999997,cont,6.371015544047885,311000000008,1,1,0,0
44b6_18ced818,81196,91,22.36,53.92000000000001,209.92000000000004,cont,4.5304871771698,312000000008,1,1,0,0
44b6_18ced818,81972,92,21.72,53.28,211.36,cont,2.7149660449073694,313000000008,1,0,1,0
44b6_1d530831,1098,3,23.519999999999996,15.36,23.360000000000007,cont,1.3341833878818958,79000000052,1,1,0,0
44b6_1d530831,1443,4,22.84,12.64,29.279999999999994,cont,0.9998703040894849,80000000053,1,0,1,0
44b6_1d530831,4793,13,46.68,210.56,4.640000000000001,cont,1.4375510860487766,89000000088,1,1,0,0
44b6_1d530831,5133,14,45.12000000000001,207.36000000000004,5.279999999999999,cont,0.6132087735836759,90000000088,1,0,1,0
44b6_1d530831,5817,16,43.8,203.2,10.559999999999999,cont,0.8630876621178164,92000000088,1,0,1,0
44b6_1d530831,6534,18,43.199999999999996,198.72000000000003,18.08,cont,1.5965573314165689,94000000087,2,2,0,0
44b6_1d530831,6896,19,42.64,196.00000000000003,21.280000000000005,cont,2.5406707406706475,95000000087,2,0,2,0
44b6_1d530831,9832,27,53.000000000000014,242.71999999999997,23.360000000000007,cont,1.6714608131811182,103000000075,2,2,0,0
44b6_1d530831,10115,28,38.8,171.35999999999996,52.640000000000015,cont,0.49073923829261357,104000000087,1,1,0,0
44b6_1d530831,10193,28,52.64000000000001,240.0,26.720000000000002,cont,2.6674264868033446,104000000075,3,1,2,0
44b6_1d530831,10555,29,52.16,237.28000000000003,28.640000000000008,cont,1.9608707408954766,105000000075,1,0,1,0
44b6_1d530831,11551,32,37.52000000000001,162.07999999999998,63.2,cont,1.5614381391845091,108000000087,1,0,1,0
44b6_1d530831,11645,32,51.68,231.36,34.72,cont,1.2486092262994068,108000000075,1,1,0,0
44b6_1d530831,12004,33,51.00000000000001,228.64000000000001,36.00000000000001,cont,1.7562784695201537,109000000075,1,0,1,0
44b6_1d530831,12274,34,37.32,156.0,72.0,cont,1.6945251916097301,110000000086,1,1,0,0
44b6_1d530831,12647,35,37.2,152.0,77.28,cont,1.6471120559937602,111000000086,1,0,1,0
44b6_1d530831,13425,37,46.52,212.0,38.08,cont,0.8649214198411337,113000000074,1,1,0,0
44b6_1d530831,13772,38,46.32,208.64000000000004,40.00000000000001,cont,1.2056799171007293,114000000074,1,0,1,0
44b6_1d530831,14129,39,47.160000000000004,187.35999999999999,35.99999999999999,cont,6.931763168559648,115000000074,1,0,1,0
44b6_1d530831,15845,44,47.00000000000001,192.0,12.0,cont,0.8125,120000000065,1,1,0,0
44b6_1d530831,16186,45,47.16000000000001,192.0,16.64,cont,0.7151846352516319,121000000066,2,1,1,0
44b6_1d530831,16507,46,47.32000000000001,192.0,21.28,cont,1.2131035868795284,122000000066,1,0,1,0
44b6_1d530831,17189,48,41.73333333333333,179.46666666666664,54.933333333333316,cont,6.697090086373345,124000000075,4,0,0,4
44b6_1d530831,17190,48,42.16000000000001,184.64,70.08,cont,0.8069560861657832,124000000075,1,0,1,0
44b6_1d530831,17606,49,49.00000000000001,201.28000000000006,29.919999999999998,cont,6.86075307910876,125000000065,1,1,0,0
44b6_1d530831,17975,50,47.84,185.28,43.36,cont,0.3192618439463036,126000000066,1,1,0,0
44b6_1d530831,17976,50,47.68000000000001,182.72,60.000000000000014,cont,6.987462209200138,126000000066,1,1,0,0
44b6_1d530831,17984,50,49.00000000000001,199.36000000000007,35.36,cont,6.804890015643186,126000000066,2,1,1,0
44b6_1d530831,18292,51,41.32,182.55999999999997,81.92,cont,1.9818339044935047,127000000075,1,0,1,0
44b6_1d530831,18294,51,40.68,195.20000000000005,81.92,cont,6.719291489807548,127000000075,1,0,1,0
44b6_1d530831,18331,51,47.52000000000001,178.72000000000003,63.36,cont,6.921698897308373,127000000066,1,0,1,0
44b6_1d530831,18332,51,47.68000000000001,182.72,47.36,cont,1.1204254214360003,127000000066,1,0,1,0
44b6_1d530831,18621,52,36.68000000000001,174.72,92.0,cont,5.726205795507167,128000000075,1,1,0,0
44b6_1d530831,20060,56,28.133333333333333,119.99999999999997,136.0,cont,1.9062511384331773,132000000084,4,0,0,4
44b6_1d530831,20227,56,48.68000000000001,184.64,60.64,cont,6.549294332216258,132000000065,1,1,0,0
44b6_1d530831,20456,57,28.680000000000007,119.68,140.00000000000003,cont,0.8502959043768069,133000000084,1,1,0,0
44b6_1d530831,20607,57,47.68000000000001,182.72,64.0,cont,5.682227913635645,133000000065,1,0,1,0
44b6_1d530831,20841,58,28.520000000000003,118.72,143.36,cont,1.13564133862766,134000000084,2,1,1,0
44b6_1d530831,20988,58,47.2,181.28000000000006,68.0,cont,6.62173044320745,134000000065,1,1,0,0
44b6_1d530831,21004,58,50.00000000000001,235.35999999999999,15.040000000000003,cont,6.942611630359274,134000000019,2,2,0,0
44b6_1d530831,21006,58,52.480000000000004,221.28000000000003,7.359999999999999,cont,0.8726719028363386,134000000019,1,1,0,0
44b6_1d530831,21211,59,28.519999999999996,117.28,148.0,cont,2.520217314538561,135000000084,1,0,1,0
44b6_1d530831,21388,59,50.16,233.92000000000007,16.640000000000004,cont,6.565502765401931,135000000019,2,0,2,0
44b6_1d530831,21393,59,52.48,220.64000000000004,10.719999999999999,cont,1.238522935798935,135000000019,1,0,1,0
44b6_1d530831,21736,60,44.640000000000015,184.16000000000003,112.64,cont,6.191932377093617,136000000074,1,0,1,0
44b6_1d530831,21749,60,45.68,162.72,86.72,cont,5.973328479583217,136000000065,1,1,0,0
44b6_1d530831,21782,60,50.16000000000001,234.72,18.72,cont,6.406478776402832,136000000019,2,2,0,0
44b6_1d530831,21791,60,52.67999999999999,220.0,13.280000000000001,cont,1.3762704358155695,136000000019,1,1,0,0
44b6_1d530831,22130,61,45.320000000000014,161.28,85.28,cont,3.1707077285678613,137000000065,1,0,1,0
44b6_1d530831,22181,61,50.35999999999999,233.92000000000007,20.000000000000004,cont,6.9709469631105625,137000000019,3,1,2,0
44b6_1d530831,22188,61,52.84,220.0,16.640000000000004,cont,0.5479407472528358,137000000019,1,0,1,0
44b6_1d530831,22399,62,27.000000000000004,115.36000000000004,158.72,cont,3.031615957290766,138000000084,1,1,0,0
44b6_1d530831,22508,62,39.999999999999986,199.4666666666667,119.99999999999997,cont,1.8595030884919617,138000000074,5,1,0,4
44b6_1d530831,22809,63,26.839999999999996,114.71999999999997,161.27999999999997,cont,3.9647668988857196,139000000083,1,0,1,0
44b6_1d530831,22925,63,38.36,203.35999999999996,120.80000000000001,cont,3.5556321958970822,139000000073,4,1,0,3
44b6_1d530831,22933,63,40.08000000000001,196.96,120.32000000000002,cont,1.1750671523364036,139000000073,5,1,1,3
44b6_1d530831,22968,63,45.160000000000004,175.36,77.28,cont,5.846862005597543,139000000064,1,1,0,0
44b6_1d530831,23348,64,38.52,204.00000000000003,124.8,cont,3.164747472153193,140000000072,2,1,1,0
44b6_1d530831,23355,64,40.16,195.36,121.28000000000003,cont,1.8642227770574915,140000000072,2,1,1,0
44b6_1d530831,23394,64,45.0,156.00000000000006,84.00000000000001,cont,2.2980970388562847,140000000064,1,1,0,0
44b6_1d530831,23570,65,19.999999999999993,113.06666666666665,148.0,cont,5.450482983823046,141000000082,3,0,0,3
44b6_1d530831,23773,65,40.36,193.27999999999997,123.36000000000003,cont,2.8841346730865407,141000000072,1,0,1,0
44b6_1d530831,23803,65,45.0,153.91999999999996,83.35999999999999,cont,2.1224764427432374,141000000064,1,0,1,0
44b6_1d530831,24460,67,24.840000000000003,114.07999999999998,141.28000000000003,cont,6.981999288348564,143000000081,1,1,0,0
44b6_1d530831,24600,67,40.160000000000004,193.27999999999997,127.36000000000001,cont,3.633101722357909,143000000071,1,1,0,0
44b6_1d530831,24691,67,53.16,212.00000000000006,32.0,cont,1.7842470400703987,143000000019,1,1,0,0
44b6_1d530831,25129,68,53.36,209.92000000000002,35.36,cont,1.4809804607083823,144000000019,2,1,1,0
44b6_1d530831,25561,69,53.0,208.64,39.36,cont,1.106552218605167,145000000019,1,0,1,0
44b6_1d530831,26602,72,26.32,101.28,176.0,cont,6.5882277823782,148000000080,2,0,2,0
44b6_1d530831,28151,75,38.0,179.99999999999997,144.00000000000003,cont,5.859020822628994,151000000068,1,0,1,0
44b6_1d530831,28470,76,25.96,111.99999999999999,163.20000000000005,cont,4.902439443787148,152000000078,1,1,0,0
44b6_1d530831,29030,77,33.68000000000001,181.92000000000002,133.92000000000002,cont,2.5217361331035275,153000000068,1,1,0,0
44b6_1d530831,29499,78,33.31999999999999,184.16000000000003,137.44,cont,1.638029017600103,154000000068,1,0,1,0
44b6_1d530831,32645,84,42.00000000000001,132.00000000000006,126.72000000000001,cont,0.8204238614886603,160000000059,1,1,0,0
44b6_1d530831,32864,85,17.133333333333333,104.5333333333333,175.99999999999994,cont,6.822527660666219,161000000077,3,0,0,3
44b6_1d530831,33167,85,41.47999999999999,132.00000000000006,127.36,cont,1.1357575940754019,161000000060,1,0,1,0
44b6_1d530831,35252,89,38.316926879882814,127.09175048828128,114.64901977539063,cont,5.925983415803959,165000000060,1,1,0,0
44b6_1d530831,35587,90,22.86666666666666,109.06666666666663,179.99999999999997,cont,3.2976113250940884,166000000077,2,0,0,2
44b6_1d530831,35786,90,38.596926879882815,129.17175048828125,116.4090197753906,cont,4.733218716021313,166000000060,2,1,1,0
44b6_1d530831,41507,91,39.223085479736326,130.4764385986328,120.48029449462891,cont,3.9954984653093053,167000000060,1,0,1,0
44b6_24264f12,1696,6,58.0,235.35999999999999,152.64,cont,0.682112985509006,74000000050,2,2,0,0
44b6_24264f12,1944,7,58.0,232.63999999999993,157.28000000000003,cont,0.32702494170934554,75000000050,2,0,2,0
44b6_24264f12,5614,22,0.0,50.72,153.27999999999997,cont,0.9332085109984781,90000000009,1,1,0,0
44b6_24264f12,5615,22,0.32000000000000006,54.720000000000006,136.64,cont,6.703080681671684,90000000009,1,1,0,0
44b6_24264f12,5880,23,0.0,53.28,159.36000000000004,cont,1.7599582736530912,91000000009,1,0,1,0
44b6_24264f12,6144,24,0.0,55.35999999999999,164.64,cont,1.6521543700574723,92000000009,1,1,0,0
44b6_24264f12,6411,25,0.0,57.44,170.72000000000003,cont,1.0462022091832888,93000000009,1,0,1,0
44b6_24264f12,7793,30,2.2666666666666666,51.99999999999999,191.46666666666664,cont,3.9252150728406647,98000000009,4,0,0,4
44b6_24264f12,8739,33,11.000000000000002,138.07999999999998,24.640000000000008,cont,6.785907386820138,101000000021,2,1,1,0
44b6_24264f12,8755,33,11.880000000000003,149.28,10.720000000000002,cont,1.5258568289652894,101000000021,2,1,1,0
44b6_24264f12,9019,34,11.000000000000002,124.00000000000003,18.720000000000002,cont,6.915783126298851,102000000022,3,1,2,0
44b6_24264f12,9026,34,12.0,148.00000000000006,15.36,cont,1.6950705184445816,102000000021,1,0,1,0
44b6_24264f12,9035,34,13.000000000000004,131.68000000000004,6.720000000000001,cont,0.32008787855836907,102000000022,2,1,1,0
44b6_24264f12,9313,35,12.16,146.72000000000003,20.640000000000008,cont,1.913157158338021,103000000021,1,1,0,0
44b6_24264f12,9320,35,13.16,130.72,12.0,cont,0.9620494075150229,103000000022,2,1,1,0
44b6_24264f12,9604,36,12.0,145.27999999999997,25.28,cont,0.31383962863220083,104000000021,1,0,1,0
44b6_24264f12,9609,36,13.639999999999999,129.28,17.28,cont,1.2580900703049818,104000000022,1,0,1,0
44b6_24264f12,9884,37,11.84,144.64,30.720000000000006,cont,0.38488837667562215,105000000021,1,1,0,0
44b6_24264f12,9896,37,14.320000000000002,129.28,22.08,cont,1.395703787520837,105000000022,2,2,0,0
44b6_24264f12,10084,38,1.5199999999999998,22.560000000000002,178.72,cont,1.0623573433642748,106000000014,2,2,0,0
44b6_24264f12,10170,38,12.0,143.36,35.36,cont,0.5715289253397442,106000000021,1,0,1,0
44b6_24264f12,10188,38,15.040000000000001,127.36,24.159999999999997,cont,2.916546813013637,106000000022,2,0,2,0
44b6_24264f12,10353,39,1.44,23.36,188.79999999999995,cont,2.088425318870647,107000000014,3,1,2,0
44b6_24264f12,10456,39,12.120000000000003,142.72,40.63999999999999,cont,2.1480754467662484,107000000021,1,1,0,0
44b6_24264f12,10485,39,15.520000000000003,126.88000000000002,26.08,cont,2.518330610245596,107000000022,2,2,0,0
44b6_24264f12,10687,40,2.3599999999999994,23.36,196.64000000000001,cont,1.8264450614513477,108000000014,1,0,1,0
44b6_24264f12,10772,40,12.840000000000002,141.28000000000003,45.28,cont,1.0935899882954345,108000000021,1,0,1,0
44b6_24264f12,10811,40,18.999999999999996,127.99999999999997,44.533333333333324,cont,1.0291666666666615,108000000022,6,0,0,6
44b6_24264f12,10813,40,20.16,142.72,48.64,cont,6.825696393042981,108000000022,2,0,2,0
44b6_24264f12,11303,42,3.3200000000000003,24.0,213.27999999999997,cont,0.7353910524339968,110000000014,1,1,0,0
44b6_24264f12,11616,43,3.6799999999999997,24.0,222.72000000000003,cont,1.1430578506794875,111000000014,2,1,1,0
44b6_24264f12,11745,43,21.84,139.35999999999999,67.36,cont,5.168624091574076,111000000022,1,1,0,0
44b6_24264f12,11753,43,23.159999999999997,127.36,66.56,cont,1.1030865786510107,111000000022,1,1,0,0
44b6_24264f12,11903,44,1.1333333333333326,15.999999999999996,223.99999999999997,cont,5.070011574060872,112000000014,1,0,0,1
44b6_24264f12,11926,44,3.84,24.639999999999993,232.0,cont,1.6096525246151727,112000000014,1,0,1,0
44b6_24264f12,12048,44,22.319999999999997,138.08,72.64,cont,6.410187326630953,112000000022,1,0,1,0
44b6_24264f12,12062,44,24.840000000000003,126.72000000000001,76.0,cont,0.28379404944430103,112000000022,1,0,1,0
44b6_24264f12,12207,45,0.7599999999999999,16.32,232.0,cont,6.756268501362273,113000000014,1,1,0,0
44b6_24264f12,12232,45,4.8,24.0,238.72000000000003,cont,1.4578885631281906,113000000014,1,1,0,0
44b6_24264f12,12682,46,28.000000000000004,124.63999999999999,96.0,cont,1.7637801202247498,114000000022,1,1,0,0
44b6_24264f12,13001,47,29.36,124.00000000000003,105.43999999999998,cont,1.026711984930526,115000000022,1,0,1,0
44b6_24264f12,14720,53,1.6800000000000002,44.00000000000001,180.0,cont,2.8771679326205475,121000000013,1,0,1,0
44b6_24264f12,16623,59,4.840000000000001,43.36,206.72,cont,0.46984704958103396,127000000014,1,1,0,0
44b6_24264f12,16937,60,5.520000000000001,43.36,210.71999999999997,cont,0.9728309205612287,128000000014,1,0,1,0
44b6_24264f12,16938,60,6.68,32.0,223.84000000000006,cont,6.937056292117017,128000000014,4,0,1,3
44b6_24264f12,17119,60,39.16000000000001,146.72000000000003,191.36,cont,6.880124113161025,128000000024,1,1,0,0
44b6_24264f12,17244,61,5.84,44.00000000000001,215.36000000000007,cont,0.5040120286263056,129000000015,1,1,0,0
44b6_24264f12,17435,61,39.68,147.35999999999999,196.00000000000006,cont,6.412164343066986,129000000025,1,0,1,0
44b6_24264f12,17560,62,6.159999999999998,44.00000000000001,218.56,cont,2.148259833562969,130000000015,1,0,1,0
44b6_24264f12,17755,62,40.36000000000001,148.00000000000003,202.08,cont,5.097425820941391,130000000025,1,1,0,0
44b6_24264f12,18067,63,42.13333333333333,144.0,215.99999999999994,cont,1.6758872546438852,131000000025,2,0,0,2
44b6_24264f12,18499,65,6.840000000000001,44.00000000000001,232.0,cont,1.480980460708379,133000000015,1,1,0,0
44b6_24264f12,18817,66,7.32,44.00000000000001,235.99999999999994,cont,0.7749052361418264,134000000015,1,0,1,0
44b6_24264f12,18827,66,8.84,32.0,243.35999999999993,cont,6.596018922425847,134000000015,1,0,1,0
44b6_24264f12,19017,66,41.0,148.00000000000003,219.36,cont,6.8400919447402115,134000000025,1,1,0,0
44b6_24264f12,19340,67,44.0,145.92,234.72000000000006,cont,1.8707272148819472,135000000025,1,1,0,0
44b6_24264f12,19655,68,44.52,148.00000000000006,239.36000000000007,cont,0.917081818868945,136000000025,1,0,1,0
44b6_24264f12,20203,70,30.799999999999997,114.72,204.00000000000003,cont,6.895229442339105,138000000024,1,1,0,0
44b6_24264f12,20527,71,32.2,114.72,207.36,cont,6.76095696351486,139000000024,1,0,1,0
44b6_24264f12,21142,73,32.0,128.64000000000004,208.0,cont,0.9825540697590025,141000000024,1,1,0,0
44b6_24264f12,21454,74,32.160000000000004,128.0,210.71999999999997,cont,0.9018384001582527,142000000024,1,0,1,0
44b6_24264f12,22903,79,3.1599999999999993,18.72,228.64000000000007,cont,1.9608707408954795,147000000067,1,1,0,0
44b6_24264f12,23051,79,34.12000000000001,118.56000000000003,241.76000000000005,cont,6.60280113379919,147000000024,4,0,0,4
44b6_24264f12,23216,80,3.839999999999999,19.36,232.0,cont,1.372812464432049,148000000067,2,1,1,0
44b6_24264f12,23527,81,4.680000000000001,20.000000000000007,235.99999999999994,cont,1.177312219634208,149000000067,1,0,1,0
44b6_24264f12,24143,83,5.520000000000001,20.640000000000004,244.0,cont,1.038602576782859,151000000065,2,2,0,0
44b6_24264f12,24440,84,5.48,21.439999999999998,247.36,cont,1.3649999999999982,152000000065,3,1,2,0
44b6_24264f12,24747,85,6.000000000000002,20.639999999999997,250.08000000000004,cont,1.2359617965374106,153000000065,1,0,1,0
44b6_267148e4,647,3,35.32000000000001,85.91999999999999,20.799999999999997,parent,3.1099969855290914,16000000001,1,1,0,0
44b6_267148e4,837,4,32.879999999999995,86.08,44.480000000000004,daughter,2.894005928207475,17000000001,3,1,0,2
44b6_267148e4,846,4,35.32000000000001,88.79999999999998,15.2,daughter,2.2071903095338246,17000000002,1,0,1,0
44b6_267148e4,1797,8,61.039999999999985,51.99999999999999,77.28,cont,1.8591874905452896,21000000077,1,1,0,0
44b6_267148e4,1998,9,60.47999999999999,49.28,78.72,cont,1.3573371219045107,22000000078,1,0,1,0
44b6_267148e4,2110,10,28.959999999999997,90.72,31.200000000000006,cont,3.5626063580614655,23000000001,5,5,0,0
44b6_267148e4,2343,11,24.200000000000003,91.2,10.56,cont,1.1428268132136248,24000000001,6,0,1,5
44b6_267148e4,2455,11,55.84,48.480000000000004,63.2,cont,3.1045581408632095,24000000078,8,0,3,5
44b6_267148e4,2577,12,23.72,94.08000000000001,13.919999999999998,cont,0.5897205376277768,25000000001,2,2,0,0
44b6_267148e4,2796,13,23.160000000000004,96.0,18.56,cont,1.8439991357102057,26000000001,4,2,2,0
44b6_267148e4,2910,13,55.16000000000001,62.56,61.60000000000001,cont,4.034845876858254,26000000078,7,7,0,0
44b6_267148e4,3147,14,50.199999999999996,57.92000000000001,83.35999999999999,cont,1.3774211638057676,27000000078,8,0,2,6
44b6_267148e4,3568,16,28.16,98.08000000000001,7.359999999999999,cont,1.3341833878818852,29000000002,1,1,0,0
44b6_267148e4,4075,18,22.319999999999997,108.8,40.000000000000014,cont,1.9600625787203834,31000000001,1,0,1,0
44b6_267148e4,5464,23,25.200000000000006,110.71999999999998,48.16000000000001,cont,2.0772051535657234,36000000001,1,0,1,0
44b6_267148e4,6612,27,21.64,116.63999999999999,60.640000000000015,cont,5.170284234208024,40000000001,1,1,0,0
44b6_267148e4,7010,28,44.36000000000001,84.0,114.56000000000003,cont,1.1849136308187302,41000000078,1,1,0,0
44b6_267148e4,7203,29,20.16,123.19999999999999,57.279999999999994,cont,6.81202635968182,42000000001,3,3,0,0
44b6_267148e4,7249,29,30.519999999999992,117.28,15.360000000000005,cont,2.884500877534956,42000000002,3,3,0,0
44b6_267148e4,7524,30,21.84,127.68,36.160000000000004,cont,5.371380903687612,43000000001,6,4,2,0
44b6_267148e4,7561,30,30.520000000000007,121.60000000000001,12.159999999999998,cont,4.481141909435583,43000000002,7,4,3,0
44b6_267148e4,7599,30,39.480000000000004,93.27999999999999,116.16000000000003,cont,5.762887866556141,43000000078,14,7,0,7
44b6_267148e4,7625,30,43.99999999999999,89.6,119.20000000000005,cont,3.0237665357133654,43000000078,6,4,2,0
44b6_267148e4,7810,31,16.52,131.83999999999997,38.72,cont,6.509519591529009,44000000001,7,0,2,5
44b6_267148e4,7829,31,21.32,128.64000000000001,32.48000000000001,cont,2.2951650948243376,44000000001,4,0,4,0
44b6_267148e4,7864,31,29.640000000000008,124.0,8.48,cont,3.044002812909356,44000000002,4,0,4,0
44b6_267148e4,7901,31,38.28000000000001,89.76,112.48,cont,5.460435250508875,44000000078,7,0,7,0
44b6_267148e4,7926,31,41.31999999999999,86.88000000000001,101.28,cont,2.7231237806974398,44000000078,10,0,3,7
44b6_267148e4,8219,32,39.24000000000001,75.20000000000002,89.27999999999999,cont,6.967631636718456,45000000078,2,1,1,0
44b6_267148e4,8849,34,41.04,99.52,106.08000000000001,cont,1.8732662817923091,47000000078,1,1,0,0
44b6_267148e4,9169,35,40.68,101.44000000000001,108.0,cont,1.314644082061764,48000000078,1,0,1,0
44b6_267148e4,9400,36,20.68,135.20000000000002,32.48,cont,3.07541714650224,49000000001,1,1,0,0
44b6_267148e4,9717,37,20.36,138.72000000000006,36.64000000000001,cont,1.4783035040207189,50000000001,1,0,1,0
44b6_267148e4,9742,37,26.2,134.07999999999998,12.0,cont,1.9854946644350344,50000000002,1,1,0,0
44b6_267148e4,10351,39,14.520000000000003,148.16000000000003,53.60000000000001,cont,5.851015536853405,52000000001,1,0,1,0
44b6_267148e4,10457,39,38.0,107.99999999999997,126.93333333333332,cont,1.6817855920948368,52000000078,7,1,0,6
44b6_267148e4,10758,40,32.36000000000001,103.36,124.64000000000003,cont,6.841578080019256,53000000078,1,0,1,0
44b6_267148e4,10777,40,37.00000000000001,107.68,118.56000000000003,cont,1.765949638296649,53000000078,1,0,1,0
44b6_267148e4,11957,44,12.0,140.48000000000002,43.36000000000001,cont,6.902425411585414,57000000001,2,2,0,0
44b6_267148e4,11970,44,15.320000000000004,129.91999999999996,28.0,cont,2.607757957038968,57000000001,3,3,0,0
44b6_267148e4,11998,44,21.68,134.88,5.279999999999999,cont,5.355897450474571,57000000002,3,3,0,0
44b6_267148e4,12294,45,15.36,121.60000000000002,28.000000000000004,cont,2.7742324388918873,58000000001,8,5,3,0
44b6_267148e4,12329,45,23.4,102.72,4.96,cont,2.670147467837685,58000000002,6,3,1,2
44b6_267148e4,12549,46,12.680000000000001,137.28,41.43999999999999,cont,6.9109131315984555,59000000001,11,3,4,4
44b6_267148e4,12562,46,15.200000000000001,119.52000000000001,27.200000000000003,cont,4.683525148058879,59000000001,8,3,5,0
44b6_267148e4,12601,46,22.599999999999994,125.60000000000001,-0.5333333333333313,cont,3.1556441937863915,59000000002,5,1,0,4
44b6_267148e4,12874,47,11.160000000000002,138.88,51.36000000000001,cont,5.960029493425348,60000000001,5,0,2,3
44b6_267148e4,12885,47,13.2,136.48,37.92,cont,0.5797857901846128,60000000001,9,0,3,6
44b6_267148e4,13114,48,10.960000000000003,146.08,56.48000000000001,cont,6.952721589420939,61000000001,3,3,0,0
44b6_267148e4,13159,48,20.64,144.00000000000003,7.2,cont,5.782374755452998,61000000002,1,0,1,0
44b6_267148e4,13208,48,31.133333333333336,102.40000000000002,131.99999999999997,cont,4.866717501161433,61000000081,7,4,0,3
44b6_267148e4,13221,48,34.00000000000001,102.93333333333334,122.39999999999999,cont,1.8452477438303927,61000000081,7,4,0,3
44b6_267148e4,13471,49,14.040000000000003,156.00000000000003,31.359999999999996,cont,3.008313481005591,62000000001,4,1,2,1
44b6_267148e4,13496,49,20.32,152.0,14.080000000000002,cont,2.792353398121379,62000000002,4,2,0,2
44b6_267148e4,13558,49,33.160000000000004,125.28,121.28000000000003,cont,0.9687508064512829,62000000081,8,0,4,4
44b6_267148e4,13802,50,13.640000000000004,160.64,38.879999999999995,cont,3.8919004435494062,63000000001,2,1,1,0
44b6_267148e4,13840,50,20.52,158.08000000000004,21.28000000000001,cont,2.919668752615607,63000000002,2,0,2,0
44b6_267148e4,14142,51,14.0,164.64,43.84,cont,4.740350890229541,64000000001,3,2,1,0
44b6_267148e4,14173,51,20.520000000000003,163.36,29.279999999999994,cont,2.907297123188483,64000000002,3,3,0,0
44b6_267148e4,14226,51,32.800000000000004,132.64000000000001,133.44,cont,1.898678816967208,64000000081,1,1,0,0
44b6_267148e4,14438,52,7.4799999999999995,180.0,59.2,cont,6.808614258790111,65000000001,5,0,2,3
44b6_267148e4,14494,52,20.16,167.36000000000004,36.8,cont,5.2382499463083985,65000000002,3,0,3,0
44b6_267148e4,14552,52,33.120000000000005,138.08000000000004,139.04,cont,4.871586304788208,65000000081,1,0,1,0
44b6_267148e4,14562,52,33.52,150.71999999999997,140.79999999999998,cont,6.221394789353911,65000000081,2,1,1,0
44b6_267148e4,14875,53,32.64,153.27999999999997,146.72000000000003,cont,5.101801305666451,66000000081,4,1,1,2
44b6_267148e4,15073,54,5.84,188.00000000000003,64.0,cont,6.522426935198284,67000000001,1,1,0,0
44b6_267148e4,15132,54,18.84,173.28000000000003,48.0,cont,4.185124326110274,67000000002,1,1,0,0
44b6_267148e4,15402,55,8.866666666666665,180.53333333333327,67.46666666666667,cont,1.7639547778595068,68000000001,3,0,0,3
44b6_267148e4,16690,59,17.2,183.35999999999999,60.640000000000015,cont,1.5532994117361896,72000000002,1,0,1,0
44b6_267148e4,17657,62,27.64,179.36000000000004,175.36,cont,1.1030865786510056,75000000083,1,1,0,0
44b6_267148e4,17959,63,27.84,183.36,177.28,cont,0.3848883766756192,76000000084,1,0,1,0
44b6_267148e4,18811,66,28.319999999999997,189.28000000000003,189.92000000000002,cont,1.7025316627011735,79000000085,1,0,1,0
44b6_267148e4,20179,71,28.84000000000001,203.35999999999999,214.08,cont,1.4412201861617266,84000000086,1,0,1,0
44b6_267148e4,21156,75,16.8,217.44,110.72,cont,2.079111138203052,88000000003,1,1,0,0
44b6_267148e4,21395,76,10.733333333333327,216.53333333333325,116.0,cont,2.85404881143534,89000000002,1,0,1,0
44b6_267148e4,21403,76,13.133333333333328,217.06666666666666,111.99999999999996,cont,6.8400833658215605,89000000002,7,0,0,7
44b6_267148e4,22419,80,9.133333333333333,228.53333333333333,131.99999999999997,cont,1.9554466756700184,93000000002,7,0,0,7
44b6_267148e4,22937,82,24.160000000000004,236.64000000000001,232.0,cont,3.0676367634548853,95000000084,1,1,0,0
44b6_267148e4,24239,88,19.159999999999997,227.36,183.99999999999997,cont,6.539226946856929,101000000003,1,1,0,0
44b6_267148e4,24255,88,24.32,244.0,248.00000000000006,cont,2.9619609214167455,101000000084,1,0,1,0
44b6_267148e4,24468,89,19.160000000000004,225.92,188.00000000000003,cont,6.59047193397409,102000000003,1,0,1,0
44b6_267148e4,24670,90,15.0,232.64,168.0,cont,6.1874102266214726,103000000003,1,1,0,0
44b6_267148e4,24892,91,18.48,232.64,176.00000000000003,cont,1.4543523042921787,104000000003,1,0,1,0
44b6_2a2eff9f,2961,5,30.799999999999997,124.47999999999999,60.0,cont,1.547508582367153,154000000036,1,1,0,0
44b6_2a2eff9f,3498,6,30.0,126.08,61.280000000000015,cont,2.57848126326332,155000000036,1,0,1,0
44b6_2a2eff9f,6853,12,56.68,162.07999999999998,63.36,daughter,6.813072911689698,161000000048,1,1,0,0
44b6_2a2eff9f,8839,16,61.160000000000004,152.0,80.64,cont,2.1210452169861873,165000000049,1,1,0,0
44b6_2a2eff9f,8845,16,62.13333333333335,151.99999999999997,32.00000000000001,cont,6.310416666666661,165000000048,3,0,0,3
44b6_2a2eff9f,9295,17,60.68000000000001,151.36,84.0,cont,2.8762959035884843,166000000049,3,2,1,0
44b6_2a2eff9f,9501,18,22.48,136.64,88.64,cont,3.572820358274406,167000000036,1,1,0,0
44b6_2a2eff9f,9779,18,60.0,150.71999999999997,86.56,cont,2.344734793105614,167000000049,2,0,2,0
44b6_2a2eff9f,9997,19,21.84,139.35999999999999,91.36000000000001,cont,2.790697997365536,168000000036,1,0,1,0
44b6_2a2eff9f,10239,19,58.680000000000014,159.36,102.72000000000003,cont,6.5451199616584645,168000000049,1,1,0,0
44b6_2a2eff9f,10250,19,59.36,149.28,90.72,cont,1.2866265240931445,168000000049,1,1,0,0
44b6_2a2eff9f,10737,20,57.32000000000001,160.0,103.36,cont,5.939702880826612,169000000046,2,1,1,0
44b6_2a2eff9f,10741,20,58.31999999999999,149.27999999999997,94.08000000000001,cont,1.3077969309109332,169000000046,3,2,1,0
44b6_2a2eff9f,10753,20,58.839999999999996,142.72,58.72,cont,2.318345989062029,169000000045,2,2,0,0
44b6_2a2eff9f,11213,21,56.32,161.27999999999997,104.0,cont,6.178163663257869,170000000045,1,0,1,0
44b6_2a2eff9f,11222,21,57.64,149.28000000000003,96.64000000000001,cont,1.6472723666716387,170000000045,2,0,2,0
44b6_2a2eff9f,11229,21,58.16,140.64000000000001,60.0,cont,1.0163722251222678,170000000044,2,0,2,0
44b6_2a2eff9f,13559,26,62.000000000000014,130.72,43.36,cont,1.3330943796295904,175000000066,1,1,0,0
44b6_2a2eff9f,14025,27,61.32000000000001,131.36000000000004,44.64000000000001,cont,1.316249999999989,176000000066,1,0,1,0
44b6_2a2eff9f,14432,28,53.0,130.72,54.08,cont,5.891112479829256,177000000045,1,1,0,0
44b6_2a2eff9f,14433,28,53.0,130.72,67.35999999999999,cont,1.671460813181093,177000000045,1,1,0,0
44b6_2a2eff9f,14484,28,60.839999999999996,132.64000000000001,46.72,cont,1.316250000000004,177000000066,1,1,0,0
44b6_2a2eff9f,14893,29,52.0,128.0,68.0,cont,1.6750116604071745,178000000045,2,1,1,0
44b6_2a2eff9f,14894,29,52.16000000000001,132.00000000000003,54.72,cont,6.312555074017186,178000000045,2,1,1,0
44b6_2a2eff9f,14946,29,60.16000000000001,133.92000000000002,47.360000000000014,cont,0.715922700087113,178000000066,2,1,1,0
44b6_2a2eff9f,15340,30,51.00000000000001,125.28000000000002,68.0,cont,1.371561609990595,179000000045,1,0,1,0
44b6_2a2eff9f,15341,30,51.160000000000004,132.64000000000001,55.36000000000001,cont,4.722799090581771,179000000045,2,1,1,0
44b6_2a2eff9f,15401,30,59.68000000000001,135.36000000000004,48.64,cont,1.0097253648888895,179000000066,2,1,1,0
44b6_2a2eff9f,16754,33,56.99999999999999,136.64,49.280000000000015,cont,1.7124292868612017,182000000066,2,2,0,0
44b6_2a2eff9f,16884,34,8.079999999999998,145.27999999999997,121.60000000000002,cont,6.4015100757555565,183000000035,1,0,1,0
44b6_2a2eff9f,17215,34,55.84000000000001,136.0,50.080000000000005,cont,1.864151951960997,183000000066,2,0,2,0
44b6_2a2eff9f,18052,36,43.0,143.36,100.0,cont,1.9470185540975156,185000000046,1,1,0,0
44b6_2a2eff9f,18473,37,42.16000000000001,143.36,104.0,cont,1.5189187149087127,186000000046,1,0,1,0
44b6_2a2eff9f,18476,37,43.30113037109375,109.16894042968751,54.37464660644531,cont,6.9447599672056395,186000000045,1,1,0,0
44b6_2a2eff9f,18491,37,45.0,112.64,72.64,cont,1.775419404675977,186000000045,1,1,0,0
44b6_2a2eff9f,18507,37,45.86666666666667,148.0,103.46666666666663,cont,4.819996416117846,186000000046,4,0,0,4
44b6_2a2eff9f,18924,38,42.50113037109375,109.96894042968748,56.29464660644532,cont,6.939520713897973,187000000045,1,0,1,0
44b6_2a2eff9f,18933,38,43.84,113.28000000000003,73.92,cont,0.8300235312929384,187000000045,1,0,1,0
44b6_2a2eff9f,20612,42,39.64,116.63999999999999,83.35999999999999,cont,1.3530503732308083,191000000044,1,1,0,0
44b6_2a2eff9f,20613,42,39.32000000000001,141.43999999999997,117.28000000000003,cont,1.255463684261701,191000000045,1,1,0,0
44b6_2a2eff9f,21025,43,39.160000000000004,117.28,84.64000000000001,cont,0.6211242327425309,192000000044,1,0,1,0
44b6_2a2eff9f,21026,43,38.68000000000001,140.0,118.07999999999998,cont,1.0345266127558028,192000000045,1,0,1,0
44b6_2a2eff9f,22308,46,35.68,134.72,118.56,cont,2.401703822289503,195000000044,1,1,0,0
44b6_2a2eff9f,24414,51,31.840000000000003,128.0,129.92000000000002,cont,0.8537051598766372,200000000043,1,0,1,0
44b6_2a2eff9f,25018,52,63.0,65.28,124.64000000000001,cont,2.2940718439055092,201000000034,1,1,0,0
44b6_2a2eff9f,25236,53,30.840000000000007,128.64000000000001,135.36000000000004,cont,0.7609790158079106,202000000043,1,1,0,0
44b6_2a2eff9f,25434,53,63.0,66.72,126.72000000000001,cont,3.450897075399383,202000000034,1,0,1,0
44b6_2a2eff9f,25641,54,30.320000000000007,129.92000000000002,137.44,cont,1.3275364872575,203000000043,1,0,1,0
44b6_2a2eff9f,25803,54,60.640000000000015,92.0,126.72000000000001,cont,4.75489550358364,203000000034,1,1,0,0
44b6_2a2eff9f,26219,55,60.67999999999999,92.0,126.72,cont,3.905007202298101,204000000034,1,0,1,0
44b6_2a2eff9f,26639,56,60.519999999999996,91.36000000000001,126.72000000000001,cont,4.508723488971138,205000000034,1,1,0,0
44b6_2a2eff9f,27079,57,60.16000000000001,90.08,125.28,cont,5.747746977077181,206000000034,1,0,1,0
44b6_2a2eff9f,27515,58,59.84,88.64,124.63999999999999,cont,5.570053438926768,207000000034,1,1,0,0
44b6_2a2eff9f,27972,59,59.31999999999999,86.56,124.00000000000003,cont,5.631979003867125,208000000034,1,0,1,0
44b6_2a2eff9f,28445,60,63.0,78.56,121.44,cont,2.79183323508049,209000000035,1,1,0,0
44b6_2a2eff9f,28857,61,58.67999999999999,84.64,124.0,cont,4.57690704788726,210000000035,1,1,0,0
44b6_2a2eff9f,28895,61,63.0,82.08,118.56,cont,4.887172303592334,210000000035,2,1,1,0
44b6_2a2eff9f,29333,62,58.133333333333326,83.46666666666664,123.99999999999999,cont,3.4416086556854415,211000000035,1,0,1,0
44b6_2a2eff9f,29359,62,60.99999999999999,76.5333333333333,127.99999999999997,cont,2.2987353105083117,211000000035,4,0,0,4
44b6_2a2eff9f,29387,62,63.0,85.44,115.84000000000002,cont,6.75914057036839,211000000035,1,0,1,0
44b6_2a2eff9f,30278,64,58.839999999999996,80.0,128.0,cont,1.4241713599493517,213000000035,1,0,1,0
44b6_2a2eff9f,31672,67,56.48,76.63999999999999,128.0,cont,1.1033259377899147,216000000035,1,1,0,0
44b6_2a2eff9f,32130,68,56.52,73.92,128.0,cont,0.8456247690317485,217000000035,1,0,1,0
44b6_2a2eff9f,33081,70,56.00000000000001,70.72,129.28,cont,1.9413819980879725,219000000035,1,1,0,0
44b6_2a2eff9f,33539,71,55.68000000000001,68.64000000000001,130.72,cont,1.4673669019710207,220000000035,1,0,1,0
44b6_2a2eff9f,34004,72,55.32000000000001,67.36,131.36000000000004,cont,2.1762869755618253,221000000035,1,1,0,0
44b6_2a2eff9f,34450,73,54.84,65.28,132.00000000000006,cont,0.4955583971642271,222000000035,1,0,1,0
44b6_2a2eff9f,35343,75,53.31999999999999,64.63999999999999,132.00000000000006,cont,3.131236277415051,224000000035,1,1,0,0
44b6_2a2eff9f,35787,76,52.48,66.08000000000001,132.64000000000001,cont,5.095197800380277,225000000035,1,0,1,0
44b6_2a2eff9f,35801,76,54.160000000000004,50.88,132.8,cont,3.680185542197563,225000000035,5,1,0,4
44b6_2a2eff9f,36234,77,51.84000000000001,65.92,133.92000000000002,cont,4.907096423802967,226000000035,1,1,0,0
44b6_2a2eff9f,36249,77,54.36000000000001,53.440000000000005,132.00000000000003,cont,1.9846965290693732,226000000035,1,0,1,0
44b6_2a2eff9f,36723,78,50.599999999999994,51.46666666666666,132.53333333333333,cont,5.8278903177355375,227000000035,5,1,0,4
44b6_2a2eff9f,36725,78,51.160000000000004,66.72,135.36000000000004,cont,6.490568156948974,227000000035,1,0,1,0
44b6_2a2eff9f,36766,78,58.0,55.99999999999999,135.35999999999999,cont,6.641778752713763,227000000035,1,1,0,0
44b6_2a2eff9f,37221,79,49.800000000000004,48.959999999999994,135.04,cont,5.723415173652868,228000000035,1,0,1,0
44b6_2a2eff9f,39083,83,55.31999999999999,58.239999999999995,136.00000000000003,cont,5.411152401522231,232000000034,1,1,0,0
44b6_2a2eff9f,39541,84,54.86666666666667,58.93333333333334,135.99999999999997,cont,6.346983644920542,233000000033,1,0,1,0
44b6_2a2eff9f,41399,88,49.160000000000004,52.64,129.91999999999996,cont,6.085424312445274,237000000033,1,1,0,0
44b6_2a2eff9f,41866,89,48.84,51.36000000000001,127.36000000000001,cont,6.383023994354075,238000000033,1,0,1,0
44b6_2a2eff9f,41887,89,52.999999999999986,44.533333333333324,136.53333333333336,cont,1.8394747986102302,238000000033,3,0,0,3
44b6_2f31fc2f,1325,7,10.680000000000001,74.72,242.72000000000003,cont,6.232928936302726,37000000020,1,0,1,0
44b6_2f31fc2f,6264,25,62.31999999999999,108.0,187.84,cont,4.9990874167191794,55000000022,1,0,1,0
44b6_2f31fc2f,6265,25,61.99999999999999,115.73333333333332,203.46666666666664,cont,6.717540265768571,55000000022,2,0,0,2
44b6_2f31fc2f,6557,26,60.480000000000004,120.32000000000001,183.36,cont,3.7848548931894324,56000000022,6,6,0,0
44b6_2f31fc2f,6573,26,62.32000000000001,103.84000000000002,189.12,cont,4.041287117058126,56000000022,4,4,0,0
44b6_2f31fc2f,6575,26,61.839999999999996,120.64,200.96000000000004,cont,6.80781914878033,56000000022,3,3,0,0
44b6_2f31fc2f,6839,27,55.88000000000001,89.92000000000002,184.95999999999998,cont,6.740303336089536,57000000022,12,4,5,3
44b6_2f31fc2f,6871,27,60.84000000000001,78.08000000000001,174.88000000000005,cont,5.91662923990509,57000000022,5,3,0,2
44b6_2f31fc2f,7122,28,57.73333333333332,63.999999999999986,179.99999999999991,cont,2.6537520934728005,58000000022,7,0,3,4
44b6_2f31fc2f,7407,29,56.8365771484375,88.18615600585937,170.0272314453125,cont,3.7510876828929813,59000000022,11,1,6,4
44b6_2f31fc2f,7440,29,60.31999999999999,76.64000000000001,169.43999999999994,cont,6.8995747151545475,59000000022,5,0,1,4
44b6_2f31fc2f,7454,29,61.20000000000001,88.47999999999999,185.60000000000002,cont,6.050676098586026,59000000022,9,0,3,6
44b6_2f31fc2f,7710,30,56.52000000000001,85.6,172.64,cont,1.7471584072430255,60000000022,3,2,1,0
44b6_2f31fc2f,8031,31,54.32000000000001,103.19999999999999,179.2,cont,5.083783347567036,61000000022,11,5,3,3
44b6_2f31fc2f,8053,31,58.466666666666654,82.93333333333331,175.46666666666658,cont,6.587098738569722,61000000022,8,0,2,6
44b6_2f31fc2f,8382,32,54.00000000000001,120.48000000000002,164.64,cont,3.564829282378053,62000000022,10,0,5,5
44b6_2f31fc2f,8403,32,54.76,100.47999999999999,179.35999999999999,cont,6.9306011869461965,62000000022,8,0,5,3
44b6_2f31fc2f,9057,34,54.84,127.03999999999999,184.31999999999996,cont,6.304518345004624,64000000022,1,1,0,0
44b6_2f31fc2f,12032,43,39.839999999999996,120.00000000000003,146.56000000000003,cont,5.23169248427314,73000000023,1,1,0,0
44b6_2f31fc2f,12048,43,44.0,108.0,144.64000000000001,cont,4.583421893083811,73000000023,1,1,0,0
44b6_2f31fc2f,12367,44,38.839999999999996,120.00000000000003,143.36,cont,3.894477861344192,74000000023,1,0,1,0
44b6_2f31fc2f,12380,44,41.999999999999986,120.53333333333336,153.59999999999997,cont,3.2998349337925403,74000000023,2,0,0,2
44b6_2f31fc2f,12384,44,42.84,108.64000000000001,141.28,cont,5.366462551113163,74000000023,2,1,1,0
44b6_2f31fc2f,12694,45,41.31999999999999,109.28,138.07999999999998,cont,5.295655353683798,75000000023,1,0,1,0
44b6_2f31fc2f,14064,49,33.64000000000001,118.72000000000003,122.56,cont,2.572739906208933,79000000025,1,1,0,0
44b6_2f31fc2f,14400,50,33.04,116.63999999999999,120.0,cont,5.892098522597877,80000000026,1,0,1,0
44b6_2f31fc2f,14401,50,33.040000000000006,126.56000000000002,144.64000000000001,cont,6.992562119852787,80000000026,1,0,1,0
44b6_2f31fc2f,14741,51,28.30235168457031,121.75983642578124,120.1679052734375,cont,4.128754050118336,81000000026,1,0,1,0
44b6_2f31fc2f,14748,51,31.680000000000003,127.84000000000002,142.56,cont,6.981564339386985,81000000026,1,1,0,0
44b6_2f31fc2f,15094,52,30.68000000000001,130.72,140.64000000000004,cont,6.216639214640038,82000000026,2,1,1,0
44b6_2f31fc2f,15393,53,28.919999999999998,132.32,138.24000000000004,cont,6.665749207703529,83000000026,1,0,1,0
44b6_2f31fc2f,15681,54,26.68,128.64000000000004,112.00000000000001,cont,3.142810495798311,84000000026,1,1,0,0
44b6_2f31fc2f,15691,54,29.36,113.91999999999999,107.36,cont,6.656175351694097,84000000026,1,1,0,0
44b6_2f31fc2f,15956,55,26.36,127.03999999999999,107.99999999999997,cont,2.7766585696660773,85000000026,1,0,1,0
44b6_2f31fc2f,15960,55,27.360000000000003,126.56,127.2,cont,5.431415129825749,85000000026,2,0,0,2
44b6_2f31fc2f,16083,55,60.31999999999999,89.28000000000002,62.08,cont,4.356606513388622,85000000035,1,0,1,0
44b6_2f31fc2f,16363,56,59.36,86.08,58.72,cont,4.404739280025556,86000000035,2,2,0,0
44b6_2f31fc2f,16676,57,58.08000000000001,80.64,54.07999999999999,cont,1.5177882757486063,87000000036,2,0,2,0
44b6_2f31fc2f,16687,57,62.266666666666666,83.46666666666664,53.86666666666665,cont,5.4922435837582215,87000000036,3,0,0,3
44b6_2f31fc2f,16953,58,55.64,79.19999999999999,48.0,cont,0.7168442125036598,88000000036,2,2,0,0
44b6_2f31fc2f,17117,59,21.32,107.19999999999999,78.56,cont,5.324696294156132,89000000027,1,1,0,0
44b6_2f31fc2f,17251,59,53.480000000000004,77.76,41.92000000000001,cont,1.9119835904107587,89000000036,2,0,2,0
44b6_2f31fc2f,19475,69,9.32,132.00000000000003,72.64,cont,2.2120303173781255,99000000027,1,1,0,0
44b6_2f31fc2f,19697,70,8.520000000000001,132.8,69.92,cont,3.2947050323207963,100000000027,1,0,1,0
44b6_2f31fc2f,20433,73,9.319999999999997,152.64,72.0,cont,4.615686568648259,103000000027,1,1,0,0
44b6_2f31fc2f,20665,74,8.32,155.35999999999999,70.56,cont,4.664655835375211,104000000027,1,0,1,0
44b6_33b596bf,8002,24,35.0,239.35999999999996,178.08,cont,1.1357575940754445,68000000083,1,1,0,0
44b6_33b596bf,8325,25,35.32000000000001,232.0,173.92000000000002,cont,0.6606779188833319,69000000084,3,2,1,0
44b6_33b596bf,8641,26,35.84,224.64000000000001,172.0,cont,0.8918274777107947,70000000085,2,0,2,0
44b6_33b596bf,13712,42,33.36,154.08,142.72,cont,1.3049670733393963,86000000087,1,1,0,0
44b6_33b596bf,14011,43,32.72,147.2,137.28,cont,2.74008924352839,87000000088,1,0,1,0
44b6_33b596bf,16223,51,27.866666666666664,119.99999999999994,119.46666666666667,cont,6.6003922573881155,95000000086,6,0,0,6
44b6_33b596bf,16506,52,30.0,119.36000000000001,130.72,cont,1.0785153278929238,96000000086,1,1,0,0
44b6_33b596bf,16784,53,29.480000000000004,114.71999999999997,131.36000000000004,cont,1.2804546311759728,97000000086,1,0,1,0
44b6_33b596bf,16786,53,29.32,128.0,120.64000000000001,cont,6.518356891310873,97000000086,1,1,0,0
44b6_33b596bf,17057,54,29.159999999999997,121.28000000000003,119.36000000000001,cont,6.878684694401977,98000000086,1,0,1,0
44b6_341df25f,580,5,30.96,234.08000000000004,148.00000000000003,daughter,2.3573704232682524,80000000001,1,1,0,0
44b6_341df25f,667,6,14.520000000000003,4.640000000000001,148.00000000000003,cont,3.082578314090327,81000000027,3,2,1,0
44b6_341df25f,771,7,13.64,6.720000000000001,148.00000000000003,cont,2.3573704232682613,82000000027,2,0,2,0
44b6_341df25f,1559,14,33.68000000000001,202.40000000000003,139.35999999999999,cont,3.0716368803945544,89000000000,1,1,0,0
44b6_341df25f,1560,14,35.84,201.76000000000002,123.36,cont,6.780321019317596,89000000000,1,1,0,0
44b6_341df25f,1659,15,33.31999999999999,206.88,137.28000000000003,cont,2.0719227935664084,90000000000,1,0,1,0
44b6_341df25f,1662,15,35.120000000000005,206.24,122.71999999999998,cont,6.0131367906027275,90000000000,1,0,1,0
44b6_341df25f,2999,28,29.68,228.00000000000003,133.27999999999997,cont,2.202639396837351,103000000001,1,1,0,0
44b6_341df25f,3098,29,29.32,225.28,133.28000000000003,cont,1.2557791406135008,104000000001,1,0,1,0
44b6_341df25f,3643,35,26.320000000000004,174.08,132.00000000000006,cont,3.0217136073757764,110000000000,1,1,0,0
44b6_341df25f,3730,36,25.68,171.35999999999996,132.00000000000006,cont,4.107908721296026,111000000000,1,0,1,0
44b6_341df25f,3907,38,24.799999999999997,164.64,133.28,cont,2.484709490061159,113000000000,1,1,0,0
44b6_341df25f,4001,39,24.84,161.91999999999996,134.72,cont,1.0320988930330248,114000000000,1,0,1,0
44b6_341df25f,5645,57,28.48,180.00000000000006,129.91999999999996,cont,2.4702138065357815,132000000001,1,1,0,0
44b6_341df25f,5734,58,28.680000000000007,177.92000000000002,131.36,cont,2.215251114433748,133000000001,1,0,1,0
44b6_3a861e03,465,1,22.92,209.28,105.28000000000003,cont,0.9799976084154417,21000000030,1,0,1,0
44b6_3a861e03,2238,6,30.040000000000003,119.36000000000001,164.64,cont,1.7161261790730797,26000000034,1,1,0,0
44b6_3a861e03,4144,12,16.639999999999997,244.00000000000003,65.44,cont,0.8624755431315163,32000000076,1,1,0,0
44b6_3a861e03,4504,13,17.2,244.00000000000003,64.64000000000001,cont,2.9566963193740436,33000000076,1,0,1,0
44b6_3a861e03,5891,17,18.839999999999996,42.56,7.359999999999999,cont,1.535861423599143,37000000016,1,1,0,0
44b6_3a861e03,7666,22,20.840000000000003,46.72,21.280000000000005,cont,1.4006142625648303,42000000016,1,0,1,0
44b6_3a861e03,8703,25,19.32,84.0,1.92,cont,0.6606779188833236,45000000015,1,1,0,0
44b6_3a861e03,11820,34,9.799999999999999,131.36,141.28000000000003,cont,1.6172628303711216,54000000025,1,1,0,0
44b6_3a861e03,11876,34,16.8,246.72,54.56,cont,1.5368926645995797,54000000077,1,1,0,0
44b6_3a861e03,12252,35,17.0,243.51999999999998,54.72,cont,1.8235512228890163,55000000077,2,1,1,0
44b6_3a861e03,12275,35,20.48,98.07999999999998,6.560000000000001,cont,6.486355992581651,55000000015,1,1,0,0
44b6_3a861e03,12455,35,40.000000000000014,170.72000000000003,157.92000000000002,cont,0.9374433316206281,55000000041,2,1,1,0
44b6_3a861e03,12619,36,17.52,239.36,50.72000000000001,cont,1.911292919727368,56000000078,3,2,1,0
44b6_3a861e03,12758,36,31.520000000000003,85.92,186.07999999999998,cont,2.6719763869652735,56000000035,2,2,0,0
44b6_3a861e03,13044,37,22.68,73.75999999999998,16.000000000000004,cont,2.53103857141687,57000000015,5,3,2,0
44b6_3a861e03,13064,37,25.28,185.91999999999996,104.64000000000003,cont,3.7645677457710818,57000000031,3,3,0,0
44b6_3a861e03,13086,37,27.079999999999995,36.480000000000004,49.92,cont,2.546743631581312,57000000016,3,3,0,0
44b6_3a861e03,13136,37,32.4,67.46666666666667,196.5333333333333,cont,1.367373102481129,57000000035,4,1,0,3
44b6_3a861e03,13139,37,31.68,82.56000000000002,184.0,cont,6.8026195551787305,57000000035,9,3,2,4
44b6_3a861e03,13210,37,39.68,164.32,162.71999999999997,cont,3.832968211777911,57000000041,7,7,0,0
44b6_3a861e03,13323,38,11.52,111.52000000000002,155.04,cont,5.976134960197943,58000000025,5,3,2,0
44b6_3a861e03,13359,38,16.32,47.360000000000014,226.88,cont,3.4419325966671677,58000000073,10,3,3,4
44b6_3a861e03,13388,38,18.8,218.56,71.2,cont,2.560135799620796,58000000078,8,5,0,3
44b6_3a861e03,13415,38,19.880000000000003,47.360000000000014,236.32000000000005,cont,3.5327963428423272,58000000073,3,1,2,0
44b6_3a861e03,13433,38,23.159999999999997,56.0,26.720000000000002,cont,0.2837940494443024,58000000015,6,1,2,3
44b6_3a861e03,13474,38,27.64,20.64,58.56,cont,1.388210403721279,58000000016,8,2,3,3
44b6_3a861e03,13478,38,26.2,180.48,108.48000000000002,cont,5.068541456869016,58000000031,5,2,3,0
44b6_3a861e03,13481,38,28.0,33.86666666666665,59.46666666666667,cont,6.334142629213693,58000000016,7,0,3,4
44b6_3a861e03,13529,38,31.599999999999994,51.466666666666654,204.5333333333333,cont,0.7031242283946454,58000000035,4,0,1,3
44b6_3a861e03,13598,38,39.4,134.71999999999997,169.28000000000003,cont,2.8471372683803025,58000000041,13,5,2,6
44b6_3a861e03,13690,39,9.84,111.51999999999998,5.279999999999999,cont,2.2705850017121194,59000000082,2,0,2,0
44b6_3a861e03,13723,39,13.839999999999998,129.92,141.28,cont,6.071696761408619,59000000026,7,0,2,5
44b6_3a861e03,13727,39,15.800000000000004,55.68,224.48000000000002,cont,4.835105512033836,59000000074,3,0,3,0
44b6_3a861e03,13801,39,22.319999999999997,71.35999999999999,17.919999999999998,cont,0.5822853681829886,59000000015,8,0,3,5
44b6_3a861e03,13842,39,25.2,32.0,40.959999999999994,cont,3.2449585898744573,59000000016,7,0,4,3
44b6_3a861e03,13846,39,24.880000000000003,199.04000000000002,109.28,cont,3.683914782537744,59000000032,7,0,3,4
44b6_3a861e03,13853,39,26.360000000000003,177.28,105.12000000000002,cont,6.808614258790118,59000000032,5,0,2,3
44b6_3a861e03,13891,39,30.999999999999996,64.64,195.20000000000005,cont,0.7368259631690576,59000000036,4,0,3,1
44b6_3a861e03,13894,39,30.520000000000007,137.92000000000002,153.27999999999997,cont,1.219508097554084,59000000077,5,0,3,2
44b6_3a861e03,13907,39,31.52,79.36,183.36,cont,6.989691516798144,59000000036,5,0,4,1
44b6_3a861e03,13978,39,40.32,159.68,157.92000000000002,cont,2.394711947708955,59000000042,13,0,6,7
44b6_3a861e03,13985,39,39.88,166.88,166.72,cont,6.558178696292138,59000000042,11,0,4,7
44b6_3a861e03,14532,41,17.800000000000004,232.0,42.879999999999995,cont,3.1712073804782985,61000000079,4,4,0,0
44b6_3a861e03,14594,41,23.96,59.36,22.879999999999995,cont,6.436846326035757,61000000015,4,4,0,0
44b6_3a861e03,14624,41,25.520000000000007,36.0,53.75999999999999,cont,2.873310644970363,61000000016,3,3,0,0
44b6_3a861e03,14663,41,29.159999999999997,62.08,194.24000000000004,cont,4.98800890260833,61000000036,3,3,0,0
44b6_3a861e03,14675,41,30.0,137.92000000000002,153.91999999999996,cont,1.2437355526798997,61000000077,4,4,0,0
44b6_3a861e03,14681,41,31.64,78.56,196.47999999999996,cont,5.119678487219691,61000000036,5,5,0,0
44b6_3a861e03,14854,42,11.866666666666664,119.46666666666667,127.99999999999997,cont,2.2485691824036786,62000000026,4,0,0,4
44b6_3a861e03,14915,42,18.520000000000003,232.0,35.04000000000001,cont,2.9921629404997345,62000000079,4,0,4,0
44b6_3a861e03,14984,42,27.440000000000005,196.64000000000001,92.63999999999999,cont,4.028984246990807,62000000032,1,0,1,0
44b6_3a861e03,14997,42,27.999999999999996,36.0,23.46666666666666,cont,2.0880529873012703,62000000016,2,0,0,2
44b6_3a861e03,14998,42,28.640000000000008,59.36,191.84,cont,6.803551117798699,62000000036,5,0,3,2
44b6_3a861e03,15038,42,30.36,136.16000000000003,130.72,cont,1.5574588678356713,62000000077,5,0,2,3
44b6_3a861e03,15136,42,40.999999999999986,148.0,135.99999999999994,cont,1.1490485194281548,62000000042,2,0,0,2
44b6_3a861e03,18447,51,27.680000000000007,64.64000000000001,5.920000000000001,cont,0.5822853681829822,71000000016,1,1,0,0
44b6_3a861e03,19784,54,41.68,128.64000000000004,136.0,cont,1.3793369104392166,74000000043,1,0,1,0
44b6_3a861e03,20453,56,29.32,63.36,12.0,cont,1.259243944793862,76000000017,1,1,0,0
44b6_3a861e03,20586,56,42.32,131.99999999999997,123.99999999999999,cont,6.931515549466796,76000000044,1,0,1,0
44b6_3a861e03,20858,57,29.840000000000003,62.08,13.280000000000001,cont,1.8120025179342405,77000000017,1,0,1,0
44b6_3a861e03,22131,60,34.88,39.36000000000001,192.64000000000004,cont,1.6715398028464639,80000000040,1,1,0,0
44b6_3a861e03,23514,64,8.999999999999998,119.35999999999999,4.0,cont,0.8255575767322341,84000000087,1,1,0,0
44b6_3a861e03,25176,68,8.266666666666664,87.99999999999997,157.06666666666658,cont,6.397746049499699,88000000031,1,0,0,1
44b6_3a861e03,25799,69,32.84,34.080000000000005,194.72,cont,1.3701168973850457,89000000041,2,1,1,0
44b6_3a861e03,26615,71,32.32,68.0,49.27999999999999,cont,6.554070419403501,91000000019,1,1,0,0
44b6_3a861e03,26911,72,18.68,235.99999999999994,11.360000000000003,cont,1.1863612961488494,92000000083,1,0,1,0
44b6_3a861e03,27277,73,9.680000000000001,123.35999999999999,12.0,cont,1.2592439447938628,93000000085,1,0,1,0
44b6_3a861e03,28335,75,32.839999999999996,241.91999999999996,53.279999999999994,cont,0.5822853681829893,95000000056,1,1,0,0
44b6_3a861e03,28710,76,29.999999999999993,252.53333333333336,47.466666666666654,cont,5.993193477233637,96000000056,6,0,0,6
44b6_3a861e03,28748,76,32.68,241.28000000000003,53.280000000000015,cont,1.1798887659436303,96000000056,2,1,1,0
44b6_3a861e03,29149,77,32.480000000000004,241.28000000000003,55.36000000000001,cont,2.010605086161869,97000000056,2,1,1,0
44b6_3a861e03,29553,78,32.480000000000004,242.71999999999997,56.0,cont,2.788804907572429,98000000056,1,0,1,0
44b6_3a861e03,30615,81,15.32,13.279999999999998,232.0,cont,3.2731953520222405,101000000077,1,1,0,0
44b6_3a861e03,32266,85,17.839999999999996,238.72000000000003,10.079999999999998,cont,0.469284695041293,105000000082,1,0,1,0
44b6_3a861e03,34748,91,12.16,117.28,35.36,cont,0.3848883766756297,111000000084,1,1,0,0
44b6_3a861e03,34880,91,28.48,178.08,98.72,cont,6.154267828101082,111000000037,2,0,0,2
44b6_3a861e03,34893,91,30.000000000000007,168.64000000000004,112.64,cont,1.2064462275625774,111000000037,1,0,1,0
44b6_3a861e03,35212,92,12.32,116.63999999999999,38.08,cont,0.972830920561223,112000000084,1,0,1,0
44b6_3a861e03,35498,92,40.36,43.20000000000001,153.27999999999997,cont,3.0647517232232806,112000000046,1,1,0,0
44b6_3a861e03,35920,93,39.44000000000001,42.080000000000005,156.00000000000003,cont,6.05931101693913,113000000046,1,0,1,0
44b6_3bb3690f,326,1,48.31999999999999,84.96000000000001,167.68,cont,2.330274930131644,387000000002,2,2,0,0
44b6_3bb3690f,344,1,51.52,96.64000000000001,223.68000000000004,cont,1.6945251916097404,387000000005,2,2,0,0
44b6_3bb3690f,540,2,48.133333333333326,75.99999999999999,168.53333333333333,cont,6.584537073494409,388000000002,2,0,0,2
44b6_3bb3690f,543,2,48.0,87.36,166.08,cont,1.8852801515955109,388000000002,2,0,2,0
44b6_3bb3690f,553,2,50.48000000000002,100.80000000000001,222.72000000000003,cont,0.926820000053969,388000000005,2,0,2,0
44b6_3bb3690f,575,2,53.67999999999999,77.91999999999999,143.99999999999997,cont,6.97384421427665,388000000003,1,0,1,0
44b6_3bb3690f,1012,4,50.00000000000001,99.36000000000001,195.83999999999997,cont,1.8720677071089171,390000000004,2,0,2,0
44b6_3bb3690f,1027,4,52.86666666666666,100.53333333333332,219.99999999999997,cont,6.590382894899862,390000000005,3,0,0,3
44b6_3bb3690f,1440,6,48.133333333333326,108.5333333333333,211.99999999999994,cont,2.051551326977248,392000000006,1,0,1,0
44b6_3bb3690f,1863,8,48.480000000000004,94.07999999999998,167.20000000000002,cont,5.900741743628519,394000000003,1,1,0,0
44b6_3bb3690f,2053,9,45.68000000000001,111.99999999999999,177.92,cont,2.4938853346735854,395000000006,1,1,0,0
44b6_3bb3690f,2066,9,48.16,95.36000000000001,167.36,cont,4.733968340620792,395000000003,1,0,1,0
44b6_3bb3690f,2948,13,49.32000000000001,69.44000000000001,132.00000000000006,cont,0.6979937768347408,399000000004,1,1,0,0
44b6_3bb3690f,3190,14,47.32000000000001,93.28,150.56,cont,6.430443462545962,400000000003,1,0,1,0
44b6_3bb3690f,3200,14,49.00000000000001,68.64000000000001,132.00000000000006,cont,0.2600000000000051,400000000004,1,0,1,0
44b6_3bb3690f,3603,16,48.0,68.64000000000001,132.00000000000006,cont,0.48232671758883516,402000000004,1,0,1,0
44b6_3bb3690f,4456,20,42.64000000000001,98.72000000000001,156.64,cont,1.0780255388904325,406000000003,1,1,0,0
44b6_3bb3690f,4457,20,43.120000000000005,116.0,190.4,cont,2.0079766713037337,406000000005,1,1,0,0
44b6_3bb3690f,4677,21,43.36,115.83999999999999,189.92000000000002,cont,2.3627410591302564,407000000005,1,0,1,0
44b6_3bb3690f,5044,23,39.160000000000004,127.36000000000001,140.00000000000003,cont,0.5040120286262956,409000000006,1,1,0,0
44b6_3bb3690f,5237,24,38.68000000000001,128.64000000000001,140.00000000000003,cont,0.5813776741499377,410000000006,1,0,1,0
44b6_3bb3690f,5443,25,43.31999999999999,80.64,131.36000000000004,cont,0.938569390082588,411000000004,1,1,0,0
44b6_3bb3690f,5817,27,37.160000000000004,132.00000000000006,140.64000000000001,cont,0.5040120286262857,413000000006,1,1,0,0
44b6_3bb3690f,6444,30,41.08058715820312,145.47531982421876,207.98352539062503,cont,5.085510706768491,416000000005,2,2,0,0
44b6_3bb3690f,6624,31,40.99999999999999,131.46666666666664,207.46666666666664,cont,2.2485691824036715,417000000005,4,0,0,4
44b6_3bb3690f,6627,31,40.920587158203126,146.91531982421878,207.34352539062502,cont,5.09580021400326,417000000005,2,0,2,0
44b6_3bb3690f,7434,35,39.0,136.00000000000003,205.28,cont,2.656464991299517,421000000005,1,1,0,0
44b6_3bb3690f,7436,35,39.16000000000001,151.35999999999999,204.00000000000003,cont,5.050901528440249,421000000005,2,1,1,0
44b6_3bb3690f,7597,36,32.0,150.72,132.00000000000006,cont,0.9262500000000031,422000000006,1,1,0,0
44b6_3bb3690f,7633,36,38.999999999999986,135.99999999999997,207.46666666666658,cont,3.195833333333309,422000000005,1,0,1,0
44b6_3bb3690f,7778,37,31.84,152.0,132.00000000000006,cont,0.6306172571377837,423000000006,1,0,1,0
44b6_3bb3690f,8172,39,37.84,152.0,199.36000000000007,cont,2.872023731186773,425000000005,1,1,0,0
44b6_3bb3690f,8530,41,35.99999999999999,121.28000000000003,133.44,cont,2.6325501538432214,427000000003,1,0,1,0
44b6_3bb3690f,9095,44,38.0,97.92000000000002,103.36,cont,2.5153404342156196,430000000004,1,1,0,0
44b6_3bb3690f,9640,47,33.160000000000004,115.36000000000004,123.36,cont,5.274821087013268,433000000003,1,1,0,0
44b6_3bb3690f,9644,47,34.68000000000001,164.0,187.99999999999997,cont,3.619922305796092,433000000005,1,1,0,0
44b6_3bb3690f,9655,47,36.839999999999996,104.64000000000001,92.63999999999999,cont,3.9477133412901275,433000000004,1,1,0,0
44b6_3bb3690f,9822,48,32.84,118.72,123.35999999999999,cont,4.584285999476914,434000000003,1,0,1,0
44b6_3bb3690f,9827,48,34.32000000000001,166.08,186.55999999999997,cont,5.116144177991864,434000000005,1,0,1,0
44b6_3bb3690f,9838,48,36.0,100.0,100.53333333333332,cont,0.18958333333333854,434000000004,4,0,0,4
44b6_3bb3690f,9839,48,36.160000000000004,107.36,89.28000000000002,cont,5.628250310931452,434000000004,1,0,1,0
44b6_3bb3690f,9842,48,36.0,148.0,191.99999999999994,cont,4.667457150312154,434000000005,5,0,0,5
44b6_3bb3690f,10371,51,35.16000000000001,110.08,82.56000000000002,cont,5.471715481683962,437000000004,1,1,0,0
44b6_3bb3690f,10560,52,35.32000000000001,107.36000000000001,83.36,cont,4.700184338140369,438000000004,1,0,1,0
44b6_3bb3690f,10693,53,24.480000000000004,199.36,121.28000000000003,cont,1.0256826994738637,439000000007,1,1,0,0
44b6_3bb3690f,10743,53,35.00000000000001,148.64000000000001,180.64000000000001,cont,2.587885840797468,439000000006,1,1,0,0
44b6_3bb3690f,10744,53,34.839999999999996,155.35999999999999,196.64000000000001,cont,5.7868711861419575,439000000006,1,1,0,0
44b6_3bb3690f,10933,54,34.86666666666667,148.0,195.99999999999997,cont,6.846460967374136,440000000006,5,0,0,5
44b6_3bb3690f,10934,54,34.84,156.64,196.00000000000003,cont,6.904089367903647,440000000006,1,0,1,0
44b6_3bb3690f,11100,55,34.32,104.0,96.0,cont,1.2454329066633791,441000000005,1,1,0,0
44b6_3bb3690f,11264,56,31.84,127.36000000000001,119.36000000000001,cont,1.8963130345488841,442000000004,1,0,1,0
44b6_3bb3690f,11272,56,34.519999999999996,104.0,95.36000000000001,cont,0.9558536760404402,442000000005,1,0,1,0
44b6_3bb3690f,11450,57,31.680000000000003,127.36,118.72,cont,2.4173206370897464,443000000004,1,1,0,0
44b6_3bb3690f,11459,57,33.0,150.72,158.56,cont,1.9068667035217717,443000000006,1,0,1,0
44b6_3bb3690f,11460,57,34.13333333333332,152.0,173.0666666666666,cont,6.017743584369627,443000000006,1,0,1,0
44b6_3bb3690f,11651,58,31.32,128.0,117.28,cont,3.3371106982687873,444000000004,1,0,1,0
44b6_3bb3690f,11853,59,31.64000000000001,152.64,154.08,cont,1.3257450735341347,445000000006,1,0,1,0
44b6_3bb3690f,12820,64,31.64,156.64,148.8,cont,3.1247782421317543,450000000006,1,1,0,0
44b6_3bb3690f,12984,65,28.133333333333333,167.46666666666664,148.53333333333336,cont,1.5408964855455538,451000000006,6,0,0,6
44b6_3bb3690f,12995,65,32.28,156.16,147.35999999999999,cont,6.695729632571793,451000000006,1,0,1,0
44b6_3bb3690f,14141,71,29.839999999999996,127.36,112.64,cont,2.1307338084566076,457000000004,1,1,0,0
44b6_3bb3690f,14335,72,33.32000000000001,107.35999999999999,95.36000000000001,cont,4.597946042799995,458000000005,5,1,0,4
44b6_3bb3690f,14522,73,32.480000000000004,112.00000000000001,96.64000000000001,cont,4.635809159413283,459000000005,1,0,1,0
44b6_3bb3690f,14855,75,19.32,226.08,111.99999999999999,cont,0.7583720228225763,461000000007,1,1,0,0
44b6_3bb3690f,15048,76,19.32,226.08,111.99999999999999,cont,1.3574343906797104,462000000007,1,0,1,0
44b6_3bb3690f,15280,77,27.360000000000003,165.91999999999996,141.44,cont,1.0526187581456217,463000000006,1,0,1,0
44b6_3bb3690f,15440,78,19.52,224.64000000000004,111.99999999999999,cont,1.623780792625641,464000000007,1,1,0,0
44b6_3bb3690f,15635,79,19.32,223.99999999999997,111.36,cont,2.3269296427911303,465000000007,1,0,1,0
44b6_3bb3690f,16036,81,26.0,173.28000000000003,135.36000000000004,cont,1.1146362915767556,467000000006,2,2,0,0
44b6_3bb3690f,16214,82,25.0,163.99999999999994,128.53333333333333,cont,6.318257825100584,468000000006,2,0,0,2
44b6_3bb3690f,16219,82,26.0,174.72,134.72000000000003,cont,1.3642259710179967,468000000006,2,0,2,0
44b6_3bb3690f,16589,84,27.680000000000007,131.36000000000004,91.36000000000001,cont,1.2199410846430039,470000000004,1,1,0,0
44b6_3bb3690f,16778,85,27.32,132.64000000000001,91.36,cont,1.2624900989710717,471000000004,1,0,1,0
44b6_3bb3690f,16798,85,30.866666666666664,143.46666666666664,87.46666666666664,cont,6.315238679838893,471000000004,4,0,0,4
44b6_3bb3690f,16954,86,24.36,201.12,129.92000000000002,cont,4.825566028975255,472000000006,1,0,1,0
44b6_3bb3690f,17532,89,21.039999999999996,208.48,119.20000000000002,cont,3.1878590484053717,475000000006,1,1,0,0
44b6_3bb3690f,17717,90,20.320000000000004,212.0,118.72,cont,4.949532332705785,476000000006,3,0,1,2
44b6_3bb3690f,17730,90,23.0,198.72,117.92000000000002,cont,2.863552481184868,476000000006,2,0,0,2
44b6_3bb3690f,18139,92,26.320000000000004,139.35999999999999,88.64,cont,1.5616917909754162,478000000004,1,1,0,0
44b6_3bb3690f,18150,92,28.16,153.27999999999997,100.00000000000001,cont,6.95112625766501,478000000004,1,1,0,0
44b6_40c45f5a,432,1,24.866666666666667,251.46666666666664,208.53333333333327,cont,6.852511514948375,142000000026,7,2,0,5
44b6_40c45f5a,460,1,29.68,243.68,222.72000000000003,cont,3.8706735928646947,142000000026,4,2,2,0
44b6_40c45f5a,466,1,34.32,244.00000000000003,176.32,cont,3.9164859887404178,142000000027,1,1,0,0
44b6_40c45f5a,695,2,29.68,238.72000000000003,220.00000000000003,cont,3.943697899814343,143000000026,2,0,2,0
44b6_40c45f5a,706,2,33.84000000000001,240.00000000000006,173.28000000000003,cont,3.004272998580537,143000000027,2,1,1,0
44b6_40c45f5a,940,3,29.68,232.0,217.28000000000003,cont,3.758565633988588,144000000026,1,1,0,0
44b6_40c45f5a,952,3,33.36,235.36000000000004,170.72000000000003,cont,0.8561760917007888,144000000027,1,0,1,0
44b6_40c45f5a,1191,4,29.480000000000004,226.72000000000006,215.36000000000004,cont,3.3187019770537045,145000000026,2,1,1,0
44b6_40c45f5a,1440,5,29.32,219.84000000000003,212.64,cont,4.184051565169822,146000000026,1,0,1,0
44b6_40c45f5a,1449,5,32.32,226.55999999999997,165.28,cont,2.4841249087958523,146000000027,1,1,0,0
44b6_40c45f5a,1693,6,32.0,223.35999999999999,163.36,cont,1.8411328883326108,147000000027,1,0,1,0
44b6_40c45f5a,3828,14,28.16,210.07999999999998,147.35999999999996,cont,6.45957428937852,155000000027,1,1,0,0
44b6_40c45f5a,4091,15,28.320000000000007,205.92000000000002,145.27999999999997,cont,6.68430274037614,156000000027,1,0,1,0
44b6_40c45f5a,5293,19,28.16,178.72000000000003,156.0,cont,4.90752690389976,160000000027,1,1,0,0
44b6_40c45f5a,5593,20,27.999999999999996,176.64000000000001,155.36,cont,5.45554878082857,161000000027,1,0,1,0
44b6_40c45f5a,5620,20,31.866666666666667,175.99999999999994,131.99999999999997,cont,6.891311457996323,161000000027,2,0,0,2
44b6_40c45f5a,6848,24,26.639999999999997,164.00000000000003,189.28,cont,1.5217845198976265,165000000026,1,1,0,0
44b6_40c45f5a,6858,24,27.999999999999996,176.48000000000002,136.64,cont,5.7544276431283885,165000000027,1,1,0,0
44b6_40c45f5a,7194,25,26.84,161.91999999999996,188.64000000000001,cont,2.045241810275736,166000000026,1,0,1,0
44b6_40c45f5a,7195,25,27.999999999999996,174.07999999999996,135.36000000000004,cont,6.438979198987977,166000000027,1,0,1,0
44b6_40c45f5a,9798,33,27.0,145.44,179.36,cont,1.6613191294871659,174000000027,1,1,0,0
44b6_40c45f5a,10117,34,24.133333333333326,148.00000000000003,171.99999999999994,cont,5.907892555255621,175000000027,4,0,0,4
44b6_40c45f5a,10131,34,26.84,143.36,179.36,cont,0.45033320996789644,175000000027,1,0,1,0
44b6_40c45f5a,11069,37,23.639999999999997,142.08,166.08,cont,6.3630526528153215,178000000027,2,2,0,0
44b6_40c45f5a,11086,37,26.320000000000004,136.64,176.64,cont,0.5994906692351443,178000000027,1,1,0,0
44b6_40c45f5a,11405,38,24.0,140.64000000000001,164.64,cont,6.405097823218636,179000000027,2,0,2,0
44b6_40c45f5a,11412,38,26.320000000000004,134.72,176.64000000000004,cont,0.5520218519225469,179000000027,1,0,1,0
44b6_40c45f5a,12537,41,29.159999999999997,146.72000000000003,135.36000000000004,cont,3.038402386534747,182000000028,1,1,0,0
44b6_40c45f5a,12898,42,29.159999999999997,144.0,133.92000000000002,cont,3.0985984735037957,183000000028,1,0,1,0
44b6_40c45f5a,13280,43,29.0,141.43999999999997,134.07999999999998,cont,3.279320861474219,184000000028,1,1,0,0
44b6_40c45f5a,13658,44,29.0,136.64,132.0,cont,1.8353150274544137,185000000028,1,0,1,0
44b6_40c45f5a,16818,52,28.320000000000007,109.91999999999999,155.2,cont,4.4725302192942245,193000000027,1,1,0,0
44b6_40c45f5a,16834,52,29.840000000000003,113.28000000000002,140.64,cont,6.982982551352956,193000000028,1,1,0,0
44b6_40c45f5a,17216,53,26.999999999999993,100.53333333333332,155.46666666666664,cont,0.6511274943426049,194000000027,5,0,0,5
44b6_40c45f5a,17235,53,28.68,110.08,155.35999999999999,cont,5.2664542744620855,194000000027,1,0,1,0
44b6_40c45f5a,17243,53,30.000000000000007,112.0,140.00000000000003,cont,6.080193253507662,194000000028,1,0,1,0
44b6_40c45f5a,19466,58,31.68,129.28,117.92000000000002,cont,6.739382621946906,199000000028,1,1,0,0
44b6_40c45f5a,19916,59,31.480000000000004,128.8,118.72000000000003,cont,6.970056716053313,200000000028,1,0,1,0
44b6_40c45f5a,20278,60,24.840000000000003,86.72,149.44,cont,0.6243046131497012,201000000027,1,1,0,0
44b6_40c45f5a,20675,61,24.680000000000003,84.64000000000001,149.92000000000002,cont,0.7594158939606082,202000000027,2,1,1,0
44b6_40c45f5a,20717,61,30.480000000000004,104.16000000000001,134.07999999999998,cont,3.8126600376246396,202000000028,4,1,0,3
44b6_40c45f5a,37069,61,29.938085784912115,108.42412139892579,136.22140747070316,cont,3.9067496584875796,202000000028,1,1,0,0
44b6_40c45f5a,21077,62,23.0,95.36000000000001,140.64,cont,6.453869115693326,203000000027,2,1,1,0
44b6_40c45f5a,21089,62,24.320000000000007,83.36000000000001,151.36,cont,0.8842449109268278,203000000027,2,1,1,0
44b6_40c45f5a,21138,62,29.541371459960942,104.54405395507813,138.1784033203125,cont,4.981607754356944,203000000028,1,0,1,0
44b6_40c45f5a,21161,62,31.2,102.71999999999998,131.36000000000004,cont,1.8673364218854684,203000000028,1,0,1,0
44b6_40c45f5a,21529,63,24.159999999999997,80.64,151.35999999999999,cont,1.6882021687286246,204000000027,1,0,1,0
44b6_40c45f5a,21530,63,23.680000000000007,92.95999999999998,139.99999999999997,cont,6.688706082830971,204000000027,1,0,1,0
44b6_40c45f5a,22856,66,23.840000000000003,77.28,151.36,cont,0.7597635322914613,207000000027,1,1,0,0
44b6_40c45f5a,23248,67,20.48,80.64,160.64000000000001,cont,5.9914038682515836,208000000027,1,0,1,0
44b6_40c45f5a,23289,67,23.68,77.44000000000001,150.71999999999997,cont,1.306989025393856,208000000027,1,0,1,0
44b6_40c45f5a,23652,68,20.84,77.91999999999999,160.00000000000006,cont,6.356430061363083,209000000027,1,1,0,0
44b6_40c45f5a,23669,68,23.320000000000007,76.63999999999999,149.28,cont,1.4309229975788451,209000000027,1,1,0,0
44b6_40c45f5a,24075,69,22.133333333333333,91.99999999999999,139.99999999999994,cont,6.870524427056337,210000000027,5,0,0,5
44b6_40c45f5a,24083,69,23.0,76.0,149.27999999999997,cont,1.011423810773693,210000000027,1,0,1,0
44b6_40c45f5a,24149,69,28.839999999999996,89.27999999999999,136.0,cont,6.897737400771358,210000000028,1,0,1,0
44b6_40c45f5a,24502,70,22.84,74.72,148.64000000000001,cont,0.9731023199026897,211000000027,1,1,0,0
44b6_40c45f5a,24923,71,22.48,73.27999999999999,149.44,cont,1.0623573433642786,212000000027,2,1,1,0
44b6_40c45f5a,25357,72,20.999999999999993,72.0,159.99999999999994,cont,6.254139254325874,213000000027,6,0,0,6
44b6_40c45f5a,25376,72,22.519999999999996,72.64,148.64000000000001,cont,1.1651374650658255,213000000027,1,0,1,0
44b6_40c45f5a,28515,79,27.84,65.28,113.92,cont,2.05766326812722,220000000028,1,1,0,0
44b6_40c45f5a,28534,79,31.159999999999997,58.08000000000001,104.00000000000001,cont,6.73953934813054,220000000028,1,0,1,0
44b6_40c45f5a,28906,80,27.44,64.64,113.92,cont,1.3222548213184895,221000000028,1,0,1,0
44b6_40c45f5a,32112,88,19.159999999999997,54.07999999999999,116.63999999999999,cont,6.910855817118462,229000000026,1,1,0,0
44b6_40c45f5a,33837,92,24.6,27.99999999999999,124.53333333333335,cont,4.472031623353953,233000000025,4,0,0,4
44b6_40c45f5a,34680,94,25.680000000000003,20.640000000000008,121.92000000000002,cont,5.733947838313492,235000000025,1,1,0,0
44b6_40c45f5a,35048,95,21.959999999999997,20.320000000000004,146.07999999999998,cont,6.911199696868837,236000000025,4,0,0,4
44b6_40c45f5a,35102,95,26.000000000000007,17.28,121.28000000000003,cont,6.61572605614834,236000000025,1,0,1,0
44b6_415c0a3a,586,1,22.760000000000005,152.0,140.0,cont,0.9885824763265809,147000000011,1,1,0,0
44b6_415c0a3a,758,1,47.24000000000001,145.76,108.00000000000001,cont,0.906511169263801,147000000051,1,0,1,0
44b6_415c0a3a,771,1,49.919999999999995,134.4,108.00000000000001,cont,6.736678521348626,147000000051,1,0,1,0
44b6_415c0a3a,1018,2,22.319999999999997,152.64,139.35999999999999,cont,1.1241899861678228,148000000011,1,0,1,0
44b6_415c0a3a,1101,2,34.32000000000001,111.99999999999999,152.0,cont,1.2454329066633751,148000000007,1,1,0,0
44b6_415c0a3a,1138,2,39.32,106.72,121.28000000000003,cont,2.865580493111298,148000000065,1,1,0,0
44b6_415c0a3a,1525,3,34.16,111.99999999999999,152.0,cont,0.2599999999999909,149000000007,1,0,1,0
44b6_415c0a3a,1562,3,39.16000000000001,105.92000000000002,120.0,cont,1.5721402609182045,149000000065,1,0,1,0
44b6_415c0a3a,3926,9,19.96,152.0,137.28,cont,3.187030602692735,155000000011,1,1,0,0
44b6_415c0a3a,4345,10,20.16,152.0,139.35999999999999,cont,1.9338182761055887,156000000011,2,1,1,0
44b6_415c0a3a,4754,11,20.360000000000003,152.0,140.64,cont,4.002814049203391,157000000011,1,0,1,0
44b6_415c0a3a,8154,19,18.839999999999996,148.64000000000001,149.28000000000003,cont,4.9625329029135905,165000000011,1,1,0,0
44b6_415c0a3a,8520,20,15.866666666666665,153.0666666666666,146.93333333333334,cont,0.4852378068191368,166000000011,4,0,0,4
44b6_415c0a3a,8533,20,19.0,148.00000000000003,150.72,cont,5.356316510672231,166000000011,1,0,1,0
44b6_415c0a3a,9415,22,27.52,120.95999999999998,162.71999999999997,cont,2.4965839160741266,168000000007,1,1,0,0
44b6_415c0a3a,9809,23,24.866666666666667,123.99999999999999,156.53333333333333,cont,6.829780898246702,169000000007,3,0,0,3
44b6_415c0a3a,9830,23,28.266666666666662,119.46666666666664,164.53333333333327,cont,1.1130558068518912,169000000007,1,0,1,0
44b6_415c0a3a,15937,38,8.999999999999998,154.08000000000004,143.36,cont,0.5534550568926179,184000000012,1,1,0,0
44b6_415c0a3a,16353,39,8.16,153.27999999999997,143.36,cont,0.5994906692351358,185000000012,1,0,1,0
44b6_415c0a3a,18165,43,37.00000000000001,123.36,136.0,cont,5.522753219636029,189000000057,1,1,0,0
44b6_415c0a3a,18406,44,6.32,156.16000000000003,136.64000000000001,cont,1.5604231197018419,190000000012,1,1,0,0
44b6_415c0a3a,18538,44,32.64,127.52000000000001,139.2,cont,2.4585342432026427,190000000056,4,0,0,4
44b6_415c0a3a,18562,44,37.160000000000004,122.71999999999998,135.36000000000004,cont,5.772433683031798,190000000056,1,0,1,0
44b6_415c0a3a,18819,45,6.0,154.72,133.92000000000002,cont,1.8079174829067841,191000000012,1,0,1,0
44b6_415c0a3a,20660,49,24.72,123.36000000000003,156.48,cont,4.1392549148367355,195000000007,1,1,0,0
44b6_415c0a3a,21104,50,21.4,117.59999999999997,159.46666666666664,cont,0.7041666666666739,196000000007,5,0,0,5
44b6_415c0a3a,21142,50,24.680000000000007,124.16,155.2,cont,6.769446524642921,196000000007,1,0,1,0
44b6_415c0a3a,22151,52,46.0,84.64000000000001,132.0,cont,1.8299116269645335,198000000035,1,1,0,0
44b6_415c0a3a,22441,53,27.84,129.28,124.00000000000003,cont,6.539085610771577,199000000053,1,1,0,0
44b6_415c0a3a,22833,54,28.16,128.00000000000006,124.00000000000003,cont,6.954468437630589,200000000052,1,0,1,0
44b6_415c0a3a,26943,64,19.160000000000004,103.36000000000003,164.64,cont,2.0291689339480716,210000000007,1,1,0,0
44b6_415c0a3a,27377,65,19.0,104.00000000000001,164.64,cont,0.8530863086464251,211000000007,1,0,1,0
44b6_415c0a3a,29356,70,1.8399999999999999,149.44,139.35999999999999,cont,1.7843950305355563,216000000012,1,1,0,0
44b6_415c0a3a,29752,71,1.5200000000000002,150.07999999999998,138.72000000000003,cont,1.1119085506461497,217000000012,1,0,1,0
44b6_415c0a3a,32944,80,0.0,152.0,144.00000000000003,cont,2.3688242072809156,226000000012,1,1,0,0
44b6_415c0a3a,33286,81,0.0,152.0,145.27999999999997,cont,1.7855785126955246,227000000012,1,0,1,0
44b6_415c0a3a,34451,84,14.319999999999999,102.56,164.64,cont,4.0989316443434305,230000000007,2,0,0,2
44b6_53f95252,1051,2,45.959999999999994,117.92000000000002,168.64000000000004,cont,1.1969975459456947,170000000046,1,1,0,0
44b6_53f95252,1429,3,45.480000000000004,118.56,167.36000000000004,cont,1.2488206936946415,171000000046,2,1,1,0
44b6_53f95252,1790,4,45.20000000000001,120.0,164.64,cont,1.4125353977865354,172000000047,1,0,1,0
44b6_53f95252,2771,7,26.000000000000004,71.36,240.0,cont,0.4317732333065582,175000000057,1,1,0,0
44b6_53f95252,3146,8,25.64,71.20000000000002,238.07999999999998,cont,1.0900831390311523,176000000057,1,0,1,0
44b6_53f95252,3897,10,25.680000000000007,74.72,232.64,cont,0.8527767146211114,178000000057,1,1,0,0
44b6_53f95252,4300,11,26.32,77.28,229.27999999999997,cont,1.1017691738744417,179000000057,1,0,1,0
44b6_53f95252,5074,13,27.72,81.44000000000001,218.23999999999998,cont,2.7823113143571816,181000000057,1,1,0,0
44b6_53f95252,5422,14,27.84,82.08000000000001,209.11999999999995,cont,2.4017038222894893,182000000057,2,0,1,1
44b6_53f95252,8355,22,31.159999999999997,88.00000000000001,188.64000000000004,cont,0.8655318090630627,190000000053,1,1,0,0
44b6_53f95252,8727,23,32.0,89.92,187.35999999999999,cont,0.2620233768197078,191000000053,1,0,1,0
44b6_53f95252,9903,26,33.68,97.28000000000003,180.79999999999998,cont,1.0402538752631618,194000000053,1,1,0,0
44b6_53f95252,10277,27,33.32000000000001,98.72,175.36,cont,0.6508119928212787,195000000050,1,0,1,0
44b6_53f95252,12676,33,32.52,104.64,153.92000000000002,cont,1.4637633432355213,201000000049,1,1,0,0
44b6_53f95252,13070,34,31.68,105.28000000000002,155.52,cont,1.3612224515118712,202000000049,1,0,1,0
44b6_53f95252,14208,37,29.480000000000004,108.64000000000001,152.0,cont,0.9170818188689684,205000000049,1,1,0,0
44b6_53f95252,14585,38,29.159999999999997,109.28,149.28,cont,0.48858213229711095,206000000047,1,0,1,0
44b6_53f95252,18158,47,34.0,116.0,135.46666666666664,cont,6.794031094051924,215000000048,3,0,0,3
44b6_551a5dba,507,1,5.133333333333332,59.999999999999986,228.0,cont,0.614021635974204,265000000020,4,0,0,4
44b6_551a5dba,1097,2,28.000000000000004,159.35999999999999,74.07999999999998,cont,2.515340434215618,266000000004,1,1,0,0
44b6_551a5dba,1098,2,28.320000000000004,175.36,69.28,cont,6.574807031388837,266000000004,1,1,0,0
44b6_551a5dba,1519,3,28.16,173.27999999999997,68.64000000000001,cont,5.624143545909883,267000000004,1,0,1,0
44b6_551a5dba,2836,6,28.559999999999995,157.28,74.55999999999999,cont,4.068378247225783,270000000004,1,1,0,0
44b6_551a5dba,3320,7,31.533333333333328,160.53333333333333,64.00000000000001,cont,6.956674577738665,271000000004,5,0,0,5
44b6_551a5dba,3839,8,34.839999999999996,176.80000000000007,126.08,cont,0.667042211932628,272000000005,1,1,0,0
44b6_551a5dba,3849,8,35.90031921386719,174.63152343750002,115.02747436523441,cont,5.260226897497064,272000000005,1,1,0,0
44b6_551a5dba,39915,8,7.585399894714356,50.47051971435546,220.4223498535156,cont,4.445414503895258,272000000020,1,1,0,0
44b6_551a5dba,5736,12,27.72,154.08,83.36000000000003,cont,3.2273334581353597,276000000005,5,0,0,5
44b6_551a5dba,5998,13,5.160000000000001,47.360000000000014,220.0,cont,2.168751080691372,277000000021,1,1,0,0
44b6_551a5dba,6247,13,29.040000000000006,164.0,72.64,cont,6.134713980700329,277000000005,5,1,0,4
44b6_551a5dba,6435,14,2.466666666666667,57.06666666666666,213.06666666666666,cont,5.492043250239381,278000000021,5,1,0,4
44b6_551a5dba,6680,14,28.36,142.56,66.24,cont,6.759941405996064,278000000005,4,0,0,4
44b6_551a5dba,6914,15,1.2800000000000002,54.08000000000001,213.76000000000002,cont,6.443365774189761,279000000021,2,1,1,0
44b6_551a5dba,7357,16,1.64,50.56,212.0,cont,6.750931417219404,280000000021,1,0,1,0
44b6_551a5dba,7397,16,6.5200000000000005,41.28000000000001,221.27999999999997,cont,4.383044033545641,280000000021,1,1,0,0
44b6_551a5dba,7565,16,28.000000000000004,136.64,68.64000000000001,cont,6.805860066883545,280000000005,1,1,0,0
44b6_551a5dba,7617,16,33.160000000000004,163.36,122.08000000000003,cont,0.3000755113300754,280000000006,1,1,0,0
44b6_551a5dba,7831,17,6.879999999999999,40.64000000000001,222.08000000000004,cont,5.6274995835184125,281000000021,1,0,1,0
44b6_551a5dba,8051,17,33.000000000000014,161.28,121.28000000000003,cont,1.1430578506794744,281000000006,1,0,1,0
44b6_551a5dba,8683,19,7.000000000000001,40.000000000000014,222.56,cont,6.6251590782787995,283000000021,1,1,0,0
44b6_551a5dba,9106,20,6.680000000000001,39.36,222.72,cont,5.992792024590876,284000000021,2,1,1,0
44b6_551a5dba,9318,20,30.679999999999996,171.99999999999994,128.0,cont,6.054602588114252,284000000006,1,0,1,0
44b6_551a5dba,9545,21,6.0,39.36000000000001,222.07999999999996,cont,6.559849464736212,285000000021,2,1,1,0
44b6_551a5dba,9892,22,0.13333333333333344,43.999999999999986,215.46666666666664,cont,2.478975832991423,286000000021,2,0,0,2
44b6_551a5dba,9906,22,2.4799999999999995,48.0,204.64,cont,6.071718506732676,286000000021,1,0,1,0
44b6_551a5dba,10125,22,31.119999999999997,163.36,108.32000000000002,cont,5.772502300779096,286000000006,2,0,0,2
44b6_551a5dba,11461,25,30.360000000000003,161.28,124.00000000000003,cont,4.79636581173704,289000000007,1,1,0,0
44b6_551a5dba,11868,26,29.68,143.36,122.08000000000003,cont,6.956898105477753,290000000007,2,2,0,0
44b6_551a5dba,12317,27,28.840000000000003,141.92000000000002,118.72,cont,6.957372550934716,291000000007,2,0,2,0
44b6_551a5dba,13572,30,25.16,133.27999999999997,72.64,cont,4.82493668999916,294000000006,2,2,0,0
44b6_551a5dba,13615,30,28.733333333333334,135.46666666666664,119.46666666666667,cont,6.821775031715233,294000000007,6,0,0,6
44b6_551a5dba,13995,31,25.160000000000004,132.64000000000001,73.27999999999999,cont,5.3070868009200645,295000000006,2,0,2,0
44b6_551a5dba,14054,31,30.44935760498047,144.53109619140628,78.45940795898437,cont,6.207759261741664,295000000006,2,1,1,0
44b6_551a5dba,14877,33,30.480000000000004,147.36000000000004,123.35999999999999,cont,2.748892958992763,297000000007,2,1,1,0
44b6_551a5dba,15324,34,30.64,146.72000000000003,124.00000000000003,cont,2.317833376776687,298000000007,1,0,1,0
44b6_551a5dba,16147,36,25.133333333333326,127.46666666666663,76.5333333333333,cont,2.4550410510824694,300000000006,2,0,2,0
44b6_551a5dba,16184,36,28.160000000000004,131.36000000000004,77.92000000000002,cont,3.613241489646108,300000000006,4,0,1,3
44b6_551a5dba,16224,36,31.680000000000003,144.64,124.00000000000003,cont,0.8451562355564702,300000000007,1,1,0,0
44b6_551a5dba,16641,37,32.32,143.99999999999997,124.63999999999999,cont,1.395987553669432,301000000007,1,0,1,0
44b6_551a5dba,17920,40,27.679999999999996,123.36,83.35999999999999,cont,1.3654835453054759,304000000006,1,0,1,0
44b6_551a5dba,17968,40,31.999999999999993,149.28,143.52,cont,6.405736808712966,304000000007,1,1,0,0
44b6_551a5dba,17992,40,33.68000000000001,142.08,128.16000000000003,cont,1.486586379091357,304000000007,2,1,1,0
44b6_551a5dba,18428,41,32.52,149.27999999999997,140.48,cont,6.862080815977018,305000000007,9,0,1,8
44b6_551a5dba,18438,41,33.68000000000001,140.0,127.36,cont,0.6758906161502681,305000000007,1,0,1,0
44b6_551a5dba,20905,47,32.0,128.64000000000004,124.63999999999999,cont,1.638109619347862,311000000007,2,2,0,0
44b6_551a5dba,21357,48,32.160000000000004,128.0,125.28000000000002,cont,1.6152205731725882,312000000007,3,1,2,0
44b6_551a5dba,21735,49,26.840000000000003,108.64000000000001,84.64000000000001,cont,4.042724367613996,313000000006,1,1,0,0
44b6_551a5dba,22153,50,27.0,105.92,85.28,cont,4.794521124418996,314000000006,1,0,1,0
44b6_551a5dba,22171,50,27.733333333333327,116.53333333333333,87.99999999999997,cont,2.9376366693974947,314000000006,5,0,0,5
44b6_551a5dba,23750,54,28.866666666666667,115.46666666666667,118.93333333333334,cont,2.8871392271012275,318000000007,7,1,0,6
44b6_551a5dba,23751,54,29.32,115.36000000000004,132.64000000000001,cont,4.843372404637081,318000000007,1,0,1,0
44b6_551a5dba,24101,55,28.320000000000007,112.96000000000001,114.56000000000002,cont,2.4494415817691904,319000000007,2,1,1,0
44b6_551a5dba,24813,57,26.84,94.72,90.72,cont,6.422944831811967,321000000006,2,2,0,0
44b6_551a5dba,24814,57,26.84,111.36,92.64,cont,1.4798210407005328,321000000006,1,1,0,0
44b6_551a5dba,24819,57,27.999999999999996,109.28,112.64,cont,2.421468110981441,321000000007,2,1,1,0
44b6_551a5dba,25513,59,27.32,109.28,95.36000000000001,cont,3.4910658293707293,323000000006,1,0,1,0
44b6_551a5dba,25875,60,27.479999999999997,108.0,96.0,cont,3.7305696615932544,324000000006,1,1,0,0
44b6_551a5dba,25901,60,30.320000000000007,114.07999999999998,107.36000000000001,cont,4.097578553243378,324000000006,1,0,1,0
44b6_551a5dba,26611,62,29.480000000000004,89.27999999999999,109.28,cont,6.999808925963627,326000000007,1,1,0,0
44b6_551a5dba,26612,62,29.480000000000004,114.56000000000003,109.92,cont,2.876250000000012,326000000006,1,1,0,0
44b6_551a5dba,28437,67,28.839999999999996,86.08,111.36000000000001,cont,6.741615626094385,331000000007,1,1,0,0
44b6_551a5dba,29881,71,27.479999999999997,92.63999999999999,109.91999999999999,cont,0.9632837717931398,335000000007,1,1,0,0
44b6_551a5dba,30280,72,27.679999999999996,91.36000000000001,110.71999999999998,cont,1.3004061865432612,336000000007,1,0,1,0
44b6_551a5dba,30293,72,28.999999999999993,108.5333333333333,108.5333333333333,cont,5.711180859117996,336000000006,7,0,0,7
44b6_551a5dba,30333,72,31.2,111.36,122.08000000000003,cont,1.060615976685257,336000000006,1,0,1,0
44b6_551a5dba,30669,73,28.08,106.72000000000001,107.04,cont,5.891919254368651,337000000007,1,1,0,0
44b6_551a5dba,31032,74,27.32,104.64000000000001,105.28000000000002,cont,6.462394341496039,338000000007,1,0,1,0
44b6_551a5dba,31041,74,27.866666666666664,91.99999999999999,127.46666666666667,cont,5.8950905009074175,338000000007,7,1,0,6
44b6_551a5dba,31430,75,28.0,88.32,112.0,cont,1.9560167592584707,339000000007,1,1,0,0
44b6_551a5dba,31819,76,26.320000000000004,100.8,103.36,cont,6.826547443620379,340000000007,1,0,1,0
44b6_551a5dba,33017,79,27.160000000000004,84.64000000000001,117.92000000000002,cont,2.2364274664070747,343000000007,1,1,0,0
44b6_551a5dba,33393,80,27.160000000000004,83.36000000000001,117.92000000000002,cont,2.6930912767672672,344000000007,1,0,1,0
44b6_551a5dba,33394,80,26.52,96.63999999999999,104.0,cont,6.580949067193876,344000000007,2,0,2,0
44b6_551a5dba,33790,81,31.000000000000007,104.64,127.36,cont,1.7225766492380095,345000000006,1,1,0,0
44b6_551a5dba,35169,85,32.68,110.08,134.07999999999998,cont,0.6412061388508407,349000000006,1,1,0,0
44b6_551a5dba,35492,86,32.13333333333333,83.99999999999997,135.99999999999994,cont,5.5188895949376535,350000000007,7,0,0,7
44b6_551a5dba,36127,88,31.320000000000004,95.36000000000001,132.00000000000003,cont,2.1499800144419985,352000000007,2,1,1,0
44b6_551a5dba,36445,89,31.48,96.63999999999999,135.36000000000004,cont,1.0361844068504382,353000000007,1,0,1,0
44b6_5740d24b,285,1,29.92,143.04,15.680000000000001,cont,1.9451867680251125,148000000028,3,3,0,0
44b6_5740d24b,472,2,30.520000000000007,141.28000000000003,18.720000000000002,cont,1.524558337683408,149000000028,3,0,3,0
44b6_5740d24b,2134,11,32.32,136.0,45.28,cont,1.2870369312882954,158000000028,1,1,0,0
44b6_5740d24b,2308,12,32.68,136.0,48.0,cont,0.5200000000000031,159000000028,1,0,1,0
44b6_5740d24b,3342,18,34.839999999999996,136.0,64.64,cont,0.36769552621700946,165000000028,1,1,0,0
44b6_5740d24b,3500,19,35.68,136.0,67.36000000000001,cont,1.1146362915767614,166000000028,1,0,1,0
44b6_5740d24b,3793,21,36.040000000000006,136.0,72.64,cont,0.26800186566515233,168000000028,1,1,0,0
44b6_5740d24b,3941,22,35.480000000000004,136.0,75.36,cont,0.891531337082445,169000000028,1,0,1,0
44b6_5740d24b,4496,26,35.32000000000001,136.0,84.64000000000001,cont,0.9377249730064874,173000000028,2,2,0,0
44b6_5740d24b,4633,27,35.68,136.0,87.36,cont,0.9756768483980769,174000000029,3,1,2,0
44b6_5740d24b,4657,27,48.16000000000001,140.00000000000003,10.079999999999998,cont,0.48342043037091265,174000000032,1,1,0,0
44b6_5740d24b,4781,28,35.84,136.0,89.27999999999999,cont,0.8606365739962452,175000000029,1,0,1,0
44b6_5740d24b,4812,28,48.67999999999999,140.00000000000003,12.0,cont,1.0467068894872087,175000000032,1,0,1,0
44b6_5740d24b,5447,32,50.00000000000001,138.72000000000003,18.56,cont,0.6438774145751631,179000000032,1,1,0,0
44b6_5740d24b,5603,33,50.68000000000001,137.92000000000002,21.440000000000005,cont,2.1526803786210165,180000000032,1,0,1,0
44b6_5740d24b,5903,35,52.0,138.72000000000003,25.92,cont,1.295217766439293,182000000032,1,1,0,0
44b6_5740d24b,6084,36,52.359999999999985,138.72000000000003,28.000000000000004,cont,2.7454806614689704,183000000032,1,0,1,0
44b6_5740d24b,6230,37,52.67999999999999,138.72000000000003,30.080000000000005,cont,2.366147316842301,184000000032,1,1,0,0
44b6_5740d24b,6381,38,52.64000000000001,137.28000000000003,32.0,cont,0.6540498834186783,185000000032,1,0,1,0
44b6_5740d24b,7138,43,54.15999999999999,133.27999999999997,43.36,cont,0.6770616663199835,190000000031,1,1,0,0
44b6_5740d24b,7285,44,54.84,132.64000000000001,44.64000000000001,cont,0.39571335901128557,191000000031,1,0,1,0
44b6_5740d24b,7542,46,56.16000000000001,131.35999999999999,50.56,cont,0.7004485170232052,193000000031,1,1,0,0
44b6_5740d24b,7688,47,56.640000000000015,131.2,54.720000000000006,cont,1.0335051100502497,194000000031,1,0,1,0
44b6_5740d24b,7952,49,57.84,132.16,60.0,cont,2.3546805117679916,196000000031,1,1,0,0
44b6_5740d24b,8082,50,58.31999999999999,131.36,61.280000000000015,cont,1.4492595091977334,197000000031,1,0,1,0
44b6_5740d24b,8464,53,31.320000000000007,124.63999999999999,133.44,cont,0.6082364363469244,200000000029,1,1,0,0
44b6_5740d24b,8595,54,31.48,125.28000000000002,133.92000000000002,cont,1.219508097554099,201000000029,1,0,1,0
44b6_5740d24b,9030,57,60.16000000000001,125.28,80.80000000000001,cont,1.4333200096279781,204000000031,1,1,0,0
44b6_5740d24b,9173,58,60.000000000000014,124.63999999999999,83.36,cont,1.72257664923797,205000000031,1,0,1,0
44b6_5740d24b,9193,58,61.99999999999999,136.5333333333333,75.99999999999999,cont,5.52586288196492,205000000031,3,0,0,3
44b6_5740d24b,9263,59,32.16,123.36,151.36,cont,0.7299850169695155,206000000029,2,2,0,0
44b6_5740d24b,9383,60,31.840000000000003,121.92000000000002,155.35999999999999,cont,0.806956086165789,207000000029,2,0,2,0
44b6_5740d24b,9384,60,31.866666666666667,127.99999999999997,144.53333333333333,cont,5.812388738481118,207000000029,3,0,0,3
44b6_5740d24b,10068,65,60.68000000000001,120.00000000000003,94.72,cont,2.1860931190138975,212000000031,1,1,0,0
44b6_5740d24b,10186,66,60.31999999999999,120.00000000000003,95.2,cont,1.1079831056925196,213000000031,1,0,1,0
44b6_5740d24b,10273,67,25.64,116.0,173.91999999999996,cont,0.7129658564756113,214000000029,1,1,0,0
44b6_5740d24b,10397,68,25.0,116.0,176.64000000000004,cont,0.8530863086464406,215000000029,1,0,1,0
44b6_5740d24b,10831,71,61.000000000000014,121.28000000000003,106.08,cont,2.0652848623131908,218000000031,1,1,0,0
44b6_5740d24b,10948,72,60.133333333333326,119.99999999999997,116.0,cont,1.5210131172273962,219000000031,3,0,0,3
44b6_5740d24b,10957,72,61.52000000000001,123.36000000000003,104.64000000000003,cont,5.39113189645736,219000000031,2,0,1,1
44b6_5740d24b,11254,75,24.480000000000004,130.08,193.44,cont,6.81394491557571,222000000029,2,2,0,0
44b6_5740d24b,11256,75,25.399999999999995,116.32,201.12000000000003,cont,1.0745907651287583,222000000029,1,1,0,0
44b6_5740d24b,11390,76,25.866666666666656,116.0,201.0666666666666,cont,0.21835281401643197,223000000029,1,0,1,0
44b6_5740d24b,11514,77,23.0,112.64,196.80000000000004,cont,3.9154070669599514,224000000029,1,0,0,1
44b6_5740d24b,11877,80,24.480000000000004,108.64,202.07999999999998,cont,0.8846927432730445,227000000029,1,1,0,0
44b6_5740d24b,12005,81,24.64,107.35999999999999,201.44,cont,0.6289400806118157,228000000029,1,0,1,0
44b6_5740d24b,12572,85,60.84000000000001,122.71999999999998,135.36000000000004,cont,6.727303741098651,232000000030,1,1,0,0
44b6_587a1e22,1293,6,2.6800000000000006,143.99999999999997,45.28,cont,1.2131035868795341,8000000061,1,0,1,0
44b6_587a1e22,1515,7,3.84,143.99999999999997,44.63999999999999,cont,0.5479407472528489,9000000061,2,2,0,0
44b6_587a1e22,1740,8,5.160000000000002,144.64000000000001,43.36,cont,0.6636687803415171,10000000061,4,2,2,0
44b6_587a1e22,1966,9,6.32,145.27999999999997,42.07999999999999,cont,0.6504061231261592,11000000061,2,0,2,0
44b6_587a1e22,3320,15,2.5200000000000005,25.44,244.64000000000001,cont,1.3650000000000009,17000000018,1,1,0,0
44b6_587a1e22,3347,15,11.600000000000001,158.07999999999998,28.640000000000004,cont,6.35227444503148,17000000062,4,0,0,4
44b6_587a1e22,3367,15,15.73333333333333,155.46666666666664,19.46666666666666,cont,1.4819311021996506,17000000062,4,0,0,4
44b6_587a1e22,3555,16,3.3200000000000003,26.720000000000002,241.28000000000003,cont,0.9006664199358221,18000000018,1,0,1,0
44b6_587a1e22,4267,19,2.8666666666666663,115.46666666666667,89.59999999999998,parent,4.234797667506938,21000000052,4,0,0,4
44b6_587a1e22,4505,20,3.16,113.6,91.52000000000004,daughter,1.457979123650253,22000000052,1,1,0,0
44b6_587a1e22,4525,20,7.040000000000001,124.00000000000003,72.0,daughter,3.3120115205868474,22000000053,1,1,0,0
44b6_587a1e22,4755,21,5.199999999999999,30.720000000000006,224.15999999999997,cont,1.814259921428029,23000000019,1,1,0,0
44b6_587a1e22,4974,22,0.4800000000000001,129.92000000000002,72.48,cont,6.726636418002691,24000000052,5,0,2,3
44b6_587a1e22,4989,22,3.8,116.0,87.04,cont,3.979624668231917,24000000052,1,0,1,0
44b6_587a1e22,4993,22,4.64,32.64,217.6,cont,3.2683109169722435,24000000019,1,0,1,0
44b6_587a1e22,5011,22,9.320000000000002,125.28,59.68,cont,3.16508120946367,24000000053,1,0,1,0
44b6_587a1e22,5485,24,4.879999999999999,117.44000000000003,93.28,cont,2.7571897490923587,26000000052,1,1,0,0
44b6_587a1e22,5503,24,11.04,132.64000000000001,57.44,cont,1.892828571212937,26000000053,1,1,0,0
44b6_587a1e22,5729,25,5.280000000000001,115.83999999999999,100.80000000000001,cont,2.2894629555858725,27000000052,1,0,1,0
44b6_587a1e22,5748,25,9.866666666666665,123.99999999999999,72.53333333333333,cont,0.6595222650188197,27000000053,7,0,0,7
44b6_587a1e22,5803,25,24.96,165.28,20.160000000000004,cont,6.282700578771839,27000000065,1,0,1,0
44b6_587a1e22,5819,25,29.119999999999997,169.28,9.440000000000001,cont,1.9961059747668635,27000000065,2,1,1,0
44b6_587a1e22,6086,26,30.680000000000003,168.64000000000004,12.0,cont,0.6758906161502689,28000000065,1,0,1,0
44b6_587a1e22,6251,27,8.0,113.44,110.72000000000003,cont,1.162755348299885,29000000052,1,1,0,0
44b6_587a1e22,6480,28,5.320000000000001,38.71999999999999,242.71999999999997,cont,1.1826829932403795,30000000019,1,1,0,0
44b6_587a1e22,6507,28,10.000000000000004,111.36,113.28000000000002,cont,1.950338512284475,30000000052,1,0,1,0
44b6_587a1e22,6725,29,6.16,36.64000000000001,246.08000000000004,cont,1.2630128908684872,31000000019,1,0,1,0
44b6_587a1e22,6955,30,1.6800000000000002,123.36,111.99999999999999,cont,1.3527576002373833,32000000032,2,1,1,0
44b6_587a1e22,6997,30,13.52,105.92,118.08000000000001,cont,0.8655318090630773,32000000052,1,1,0,0
44b6_587a1e22,7197,31,2.5200000000000005,123.35999999999999,112.64,cont,1.0859572390292394,33000000032,1,0,1,0
44b6_587a1e22,7239,31,15.0,104.64,118.72,cont,0.5813776741499472,33000000052,1,0,1,0
44b6_587a1e22,7679,33,2.5199999999999996,52.64,235.99999999999994,cont,1.088264817266447,35000000023,1,0,1,0
44b6_587a1e22,7917,34,3.5199999999999996,51.36000000000001,238.07999999999998,cont,1.1391238409409208,36000000023,1,1,0,0
44b6_587a1e22,8149,35,4.160000000000001,50.72,238.72000000000003,cont,0.779999999999988,37000000023,2,1,1,0
44b6_587a1e22,8151,35,4.160000000000001,139.35999999999999,90.07999999999998,cont,1.3851635733732053,37000000031,2,2,0,0
44b6_587a1e22,8155,35,5.480000000000001,123.35999999999999,116.80000000000001,cont,1.3605433197807435,37000000032,1,1,0,0
44b6_587a1e22,8383,36,4.68,49.28,238.72000000000003,cont,1.2714521717312157,38000000023,1,0,1,0
44b6_587a1e22,8385,36,5.520000000000001,138.56,89.44,cont,1.1881406166780066,38000000031,2,0,2,0
44b6_587a1e22,8390,36,6.680000000000001,122.55999999999997,114.72,cont,0.7697767533512553,38000000032,1,0,1,0
44b6_587a1e22,8616,37,6.48,139.35999999999999,87.36,cont,1.3513904876089664,39000000031,1,1,0,0
44b6_587a1e22,8851,38,5.48,48.64,234.72000000000006,cont,0.9625982092752905,40000000023,1,1,0,0
44b6_587a1e22,8860,38,7.839999999999999,140.0,84.0,cont,0.9448784644069315,40000000031,1,0,1,0
44b6_587a1e22,9092,39,6.138199539184569,50.35486694335938,231.44937499999997,cont,0.7182955354923268,41000000023,1,0,1,0
44b6_587a1e22,9326,40,5.800000000000001,68.0,227.04000000000002,cont,6.154139105106414,42000000023,2,2,0,0
44b6_587a1e22,9327,40,5.93819953918457,53.71486694335939,228.089375,cont,0.48091856960084456,42000000023,2,2,0,0
44b6_587a1e22,9355,40,12.679999999999998,156.79999999999998,78.72,cont,6.818671172779923,42000000031,1,1,0,0
44b6_587a1e22,9564,41,5.733333333333331,67.46666666666667,246.9333333333333,cont,6.795758289489601,43000000023,6,1,0,5
44b6_587a1e22,9567,41,7.4799999999999995,66.72,230.08000000000004,cont,1.0092021910895699,43000000023,10,1,2,7
44b6_587a1e22,9582,41,12.480000000000004,155.36,72.0,cont,1.1559222508456215,43000000031,7,2,1,4
44b6_587a1e22,19938,41,6.380948963165284,56.9784506225586,224.20109375000004,cont,5.356206569934891,43000000023,3,1,2,0
44b6_587a1e22,9794,42,6.058199539184571,61.23486694335937,221.36937500000002,cont,1.2571021224478482,44000000023,2,1,1,0
44b6_587a1e22,9814,42,12.28,154.56,65.12,cont,3.9224147536944627,44000000031,2,0,2,0
44b6_587a1e22,10037,43,6.77819953918457,62.99486694335938,219.289375,cont,0.9974567944117726,45000000023,1,0,1,0
44b6_587a1e22,10527,45,15.16,135.35999999999999,88.8,cont,0.9422197859310782,47000000032,1,1,0,0
44b6_587a1e22,10528,45,15.000000000000004,156.64,57.44,cont,1.9331354065351984,47000000031,1,1,0,0
44b6_587a1e22,10532,45,15.84,172.64,70.08,cont,6.928981707292918,47000000031,1,1,0,0
44b6_587a1e22,10767,46,16.16,137.28,88.00000000000001,cont,0.7455545335520377,48000000032,1,0,1,0
44b6_587a1e22,10768,46,16.0,158.72,56.64,cont,1.3405018882866182,48000000031,1,0,1,0
44b6_587a1e22,10831,46,36.64000000000001,138.72,65.91999999999999,cont,1.6906248844732048,48000000053,1,1,0,0
44b6_587a1e22,11074,47,37.84,138.56,67.2,cont,1.4694349466716776,49000000053,1,0,1,0
44b6_587a1e22,11858,50,61.839999999999996,189.92000000000002,20.959999999999997,cont,3.1166974708816424,52000000065,1,1,0,0
44b6_587a1e22,12086,51,62.000000000000014,190.72000000000003,18.559999999999995,cont,2.140131786245854,53000000065,1,0,1,0
44b6_587a1e22,12349,53,9.800000000000002,77.44000000000001,211.84,cont,1.628003474351332,55000000024,1,1,0,0
44b6_587a1e22,12374,53,16.8,50.08,249.27999999999997,cont,2.171245682206422,55000000020,1,1,0,0
44b6_587a1e22,12582,54,10.160000000000002,74.72,209.44000000000005,cont,1.7198151245119195,56000000024,1,0,1,0
44b6_587a1e22,12606,54,17.64,47.36,247.36,cont,2.4052195845701836,56000000020,1,0,1,0
44b6_587a1e22,12696,54,45.000000000000014,142.07999999999996,64.64,cont,1.6135849063498486,56000000054,1,1,0,0
44b6_587a1e22,12699,54,46.160000000000004,158.88,64.64,cont,5.567753698755005,56000000054,1,1,0,0
44b6_587a1e22,12812,55,10.560000000000002,71.67999999999999,205.44000000000003,cont,3.864152963457832,57000000024,1,1,0,0
44b6_587a1e22,12836,55,18.84,45.76000000000001,245.43999999999997,cont,2.739125373636636,57000000020,1,1,0,0
44b6_587a1e22,12888,55,35.160000000000004,120.48000000000002,96.0,cont,1.0405076885828464,57000000053,1,1,0,0
44b6_587a1e22,12925,55,46.31999999999999,138.72000000000006,64.0,cont,0.6696104277861683,57000000054,1,0,1,0
44b6_587a1e22,12928,55,47.32,153.91999999999996,64.0,cont,6.442423117507871,57000000054,1,0,1,0
44b6_587a1e22,13063,56,18.04,50.239999999999995,205.28000000000003,cont,6.857827302615598,58000000024,1,0,1,0
44b6_587a1e22,13125,56,37.266666666666666,98.9333333333333,97.06666666666663,cont,0.945204453191668,58000000053,5,0,3,2
44b6_587a1e22,13351,57,37.68,118.56,97.28,cont,1.6224792949372278,59000000054,2,0,1,1
44b6_587a1e22,13931,60,14.64,84.64000000000001,209.92000000000002,cont,1.2355344238425756,62000000025,4,0,1,3
44b6_587a1e22,14606,63,29.159999999999997,141.28000000000003,112.16,cont,1.5835206029603852,65000000034,2,2,0,0
44b6_587a1e22,14616,63,32.0,164.64,72.80000000000001,cont,1.8603233925852778,65000000033,1,1,0,0
44b6_587a1e22,14809,64,30.200000000000003,140.64,116.63999999999999,cont,1.42008637501386,66000000034,2,0,2,0
44b6_587a1e22,14816,64,33.160000000000004,164.0,75.99999999999999,cont,0.48232671758882323,66000000033,1,0,1,0
44b6_587a1e22,15081,65,55.480000000000004,157.28000000000003,104.80000000000001,cont,6.8047153917338905,67000000055,1,1,0,0
44b6_587a1e22,17622,80,54.599999999999994,114.71999999999997,205.28000000000003,cont,6.826895570096279,82000000059,2,0,0,2
44b6_5f15d135,492,1,17.24,158.07999999999998,60.64,cont,1.9591866935032007,90000000066,5,0,5,0
44b6_5f15d135,779,1,59.60000000000001,172.48000000000005,201.2800000000001,cont,1.399576945008702,90000000011,1,1,0,0
44b6_5f15d135,1144,2,58.680000000000014,174.72,200.64,cont,0.7497926796788427,91000000011,1,0,1,0
44b6_5f15d135,1188,2,62.36,144.64000000000001,143.36,cont,2.2226293550657457,91000000026,1,1,0,0
44b6_5f15d135,1553,3,62.160000000000004,146.72000000000003,143.99999999999997,cont,1.2592439447938613,92000000026,1,0,1,0
44b6_5f15d135,2489,6,55.800000000000004,173.27999999999997,196.64000000000001,cont,0.8133120941557557,95000000011,1,1,0,0
44b6_5f15d135,2605,7,13.68,190.07999999999998,63.36,cont,1.1500821981493337,96000000065,1,1,0,0
44b6_5f15d135,2770,7,52.13333333333332,164.5333333333333,192.5333333333333,cont,4.515044296571216,96000000011,3,0,0,3
44b6_5f15d135,2791,7,55.839999999999996,171.36,195.35999999999999,cont,3.012523858826676,96000000011,2,1,1,0
44b6_5f15d135,2899,8,12.96,192.64000000000004,65.91999999999999,cont,0.4065748701038833,97000000065,1,0,1,0
44b6_5f15d135,3079,8,55.84000000000001,169.44,194.72000000000003,cont,3.1677498421592807,97000000011,1,0,1,0
44b6_5f15d135,3399,9,58.639999999999986,143.36,136.64,cont,4.150372159517751,98000000026,2,2,0,0
44b6_5f15d135,3426,9,60.0,154.72000000000003,148.16000000000003,cont,4.666466978614563,98000000026,4,1,0,3
44b6_5f15d135,3728,10,57.13333333333333,136.5333333333333,139.46666666666664,cont,5.822097810640661,99000000026,4,0,0,4
44b6_5f15d135,3762,10,59.03999999999999,148.15999999999997,135.36,cont,1.8661340385942173,99000000026,6,0,2,4
44b6_5f15d135,3763,10,59.68000000000001,153.44,150.07999999999998,cont,6.4903240481504465,99000000026,1,0,1,0
44b6_5f15d135,4419,12,58.8,135.84,135.36,cont,6.8404007786459395,101000000026,1,1,0,0
44b6_5f15d135,4794,13,60.0,139.2,133.27999999999997,cont,6.481163331146656,102000000026,2,1,1,0
44b6_5f15d135,5093,14,54.0,151.35999999999999,184.0,cont,1.9351150644083175,103000000011,1,1,0,0
44b6_5f15d135,5116,14,57.67999999999999,150.07999999999998,146.72,cont,6.0026541004792175,103000000026,1,1,0,0
44b6_5f15d135,5159,14,61.04000000000001,142.72,132.00000000000003,cont,5.757638947520092,103000000026,1,0,1,0
44b6_5f15d135,5452,15,53.519999999999996,148.00000000000003,181.28000000000003,cont,0.9821508603569925,104000000011,1,0,1,0
44b6_5f15d135,5468,15,57.16,150.72,143.36,cont,5.846862005597542,104000000026,1,0,1,0
44b6_5f15d135,6260,18,9.160000000000002,214.55999999999995,78.72000000000001,cont,0.4526726190084773,107000000065,1,1,0,0
44b6_5f15d135,6585,19,8.519999999999998,218.08000000000004,78.56,cont,0.8943407138781027,108000000065,2,1,1,0
44b6_5f15d135,6812,19,57.0,148.00000000000006,135.35999999999999,cont,6.788261237238884,108000000026,1,1,0,0
44b6_5f15d135,6894,20,7.360000000000001,220.64000000000004,79.36000000000001,cont,1.2437355526799,109000000065,1,0,1,0
44b6_5f15d135,7128,20,57.03999999999999,148.64000000000004,132.64000000000001,cont,6.624302085880444,109000000026,1,0,1,0
44b6_5f15d135,7742,22,55.839999999999996,155.99999999999997,127.20000000000003,cont,3.9996462734221963,111000000026,2,2,0,0
44b6_5f15d135,8050,23,55.04,160.16000000000003,125.92,cont,2.097509311659901,112000000026,2,0,2,0
44b6_5f15d135,9216,27,51.86666666666666,136.5333333333333,112.5333333333333,cont,6.335242653534789,116000000026,2,0,0,2
44b6_5f15d135,9222,27,54.061574096679685,136.56923583984374,96.468447265625,cont,5.32713039673093,116000000026,1,0,1,0
44b6_5f15d135,10975,33,44.68000000000001,110.72000000000003,143.36,cont,2.1529870238809896,122000000011,1,1,0,0
44b6_5f15d135,11833,36,43.16,113.92,139.35999999999999,parent,2.8200188385895624,125000000011,4,0,0,4
44b6_5f15d135,11858,36,50.20000000000001,126.08000000000001,98.72000000000001,cont,6.4378103080550595,125000000027,1,1,0,0
44b6_5f15d135,12131,37,46.0,125.28,133.27999999999997,daughter,5.519572701985187,126000000011,2,1,1,0
44b6_5f15d135,12144,37,49.0,124.00000000000003,95.99999999999997,cont,6.625736987309392,126000000028,1,0,1,0
44b6_5f15d135,13384,41,45.64000000000001,88.00000000000001,128.0,cont,5.970542793582513,130000000013,2,1,1,0
44b6_5f15d135,13697,42,44.96,88.00000000000001,128.0,cont,4.8271253414946695,131000000013,1,0,1,0
44b6_5f15d135,14007,43,37.00000000000001,138.08,146.72,cont,6.633762719038112,132000000012,1,0,1,0
44b6_5f15d135,14050,43,45.84,128.0,68.64000000000001,cont,1.7754194046759775,132000000029,1,1,0,0
44b6_5f15d135,14376,44,45.0,128.64000000000001,68.0,cont,1.4241713599493517,133000000029,2,1,1,0
44b6_5f15d135,14697,45,44.160000000000004,129.28,68.0,cont,1.9772317295906392,134000000029,1,0,1,0
44b6_5f15d135,15362,47,42.32,148.64000000000001,57.919999999999995,cont,6.166634135612399,136000000029,1,1,0,0
44b6_5f15d135,15363,47,42.84,131.36000000000004,66.72,cont,1.8642227770574922,136000000029,1,1,0,0
44b6_5f15d135,15655,48,38.533333333333324,88.53333333333332,131.46666666666664,cont,1.208778653572814,137000000013,3,0,0,3
44b6_5f15d135,15670,48,42.0,132.00000000000006,65.28,cont,1.8704448835771477,137000000029,1,0,1,0
44b6_5f15d135,15673,48,41.67999999999999,151.36,56.64,cont,6.773755536074513,137000000029,2,1,1,0
44b6_5f15d135,16237,50,29.840000000000003,128.0,133.27999999999997,cont,4.113529506397139,139000000012,1,1,0,0
44b6_5f15d135,16265,50,35.68,82.71999999999998,126.08000000000001,cont,1.1113146775328733,139000000013,1,1,0,0
44b6_5f15d135,16561,51,28.839999999999996,124.63999999999999,132.00000000000003,cont,4.829750414100103,140000000012,1,0,1,0
44b6_5f15d135,16577,51,34.120000000000005,79.19999999999999,121.91999999999999,cont,0.8081005893451589,140000000013,1,0,1,0
44b6_5f15d135,16618,51,39.84,128.0,57.92000000000001,cont,1.6639396961729143,140000000029,2,2,0,0
44b6_5f15d135,16939,52,38.839999999999996,125.92,56.0,cont,1.5143918952833841,141000000029,2,0,2,0
44b6_5f15d135,16940,52,38.52,144.48000000000002,53.28,cont,6.343684482420298,141000000029,2,0,1,1
44b6_5f15d135,17218,53,26.68,119.36000000000001,129.28,cont,5.5328565463510815,142000000012,1,1,0,0
44b6_5f15d135,17531,54,25.64,117.28,128.00000000000006,cont,5.509924284869998,143000000012,1,0,1,0
44b6_5f15d135,18201,56,24.36,114.71999999999997,124.64000000000001,cont,6.566709245314584,145000000012,1,1,0,0
44b6_5f15d135,18539,57,26.13333333333333,115.46666666666667,116.0,cont,1.6369181005950082,146000000012,8,1,0,7
44b6_5f15d135,18546,57,26.999999999999993,55.99999999999999,112.5333333333333,cont,6.554166666666677,146000000013,2,0,0,2
44b6_5f15d135,18830,58,22.36,111.36,121.28000000000003,cont,6.6466472986386185,147000000012,1,0,1,0
44b6_5f15d135,18838,58,24.840000000000003,113.28000000000003,116.0,cont,2.055801622846905,147000000012,1,0,1,0
44b6_5f15d135,18903,58,34.48,124.0,49.28,cont,2.736231726754887,147000000029,1,1,0,0
44b6_5f15d135,18904,58,34.32,134.56000000000003,49.91999999999999,cont,4.821870908941891,147000000029,1,1,0,0
44b6_5f15d135,19189,59,33.64000000000001,135.35999999999999,49.599999999999994,cont,5.487882674584057,148000000029,1,0,1,0
44b6_5f15d135,19193,59,33.73333333333331,123.99999999999999,47.466666666666654,cont,4.197479815979563,148000000029,1,0,1,0
44b6_5f15d135,19204,59,35.99999999999999,123.46666666666665,55.99999999999999,cont,1.9234887646611742,148000000029,2,0,0,2
44b6_5f15d135,20760,65,21.800000000000004,60.0,116.80000000000001,cont,4.869580320725802,154000000013,1,1,0,0
44b6_5f15d135,21034,66,21.2,59.2,119.20000000000002,cont,6.6047804656930165,155000000013,1,0,1,0
44b6_5f15d135,21035,66,21.999999999999993,63.999999999999986,103.99999999999999,cont,1.7404671430534633e-14,155000000013,3,0,0,3
44b6_5f15d135,23209,75,8.36,104.64,112.16,cont,5.462345106499588,164000000012,5,0,0,5
44b6_5f15d135,23615,77,12.160000000000004,75.36,108.0,cont,1.4477099372802507,166000000013,1,1,0,0
44b6_5f15d135,23832,78,11.68,76.64000000000001,107.36,cont,1.2780808366453145,167000000013,1,0,1,0
44b6_5f15d135,24505,81,8.999999999999998,85.91999999999999,102.72,cont,2.146784295754933,170000000013,1,1,0,0
44b6_5f15d135,24757,82,8.16,90.72,99.36000000000001,cont,1.5889309495695452,171000000013,1,0,1,0
44b6_5f15d135,25603,86,5.48,99.36000000000001,94.08000000000001,cont,4.003209844924434,175000000014,1,1,0,0
44b6_5f15d135,25604,86,5.160000000000002,104.0,114.72,cont,6.108465565303611,175000000014,1,1,0,0
44b6_5f15d135,25810,87,4.32,103.36,115.35999999999999,cont,5.954266983642901,176000000014,1,0,1,0
44b6_5f15d135,25814,87,5.0,98.56,93.91999999999999,cont,4.477604863372832,176000000014,1,0,1,0
44b6_668e0cc7,493,3,41.36,16.0,188.0,cont,1.3808675941233515,41000000044,1,0,1,0
44b6_668e0cc7,913,6,19.0,18.720000000000002,145.44000000000003,cont,1.717433586779988,44000000026,1,1,0,0
44b6_668e0cc7,1194,7,57.00000000000001,9.28,195.36,cont,1.7580817849292432,45000000042,4,3,1,0
44b6_668e0cc7,1393,8,56.480000000000004,15.36,194.56,cont,1.3082006965676234,46000000042,4,1,3,0
44b6_668e0cc7,1488,9,17.32,33.28000000000001,137.28000000000003,cont,0.9006664199358236,47000000026,1,1,0,0
44b6_668e0cc7,1693,10,16.48,38.080000000000005,135.36000000000004,cont,0.8228342785761866,48000000026,1,0,1,0
44b6_668e0cc7,2502,13,51.00000000000001,20.639999999999997,176.64,cont,1.6219095612887824,51000000070,1,1,0,0
44b6_668e0cc7,2754,14,50.839999999999996,25.28,175.83999999999997,cont,2.1371066889605554,52000000070,1,0,1,0
44b6_668e0cc7,3847,19,12.16,52.64000000000001,148.00000000000006,cont,0.822513297764836,57000000027,2,2,0,0
44b6_668e0cc7,3915,19,29.799999999999997,81.12,185.60000000000002,cont,3.6781759236066933,57000000045,4,4,0,0
44b6_668e0cc7,3976,19,49.480000000000004,32.64,189.44,cont,1.663622271580908,57000000072,2,2,0,0
44b6_668e0cc7,3979,19,50.2,13.44,193.28000000000003,cont,2.0129677251511024,57000000070,3,3,0,0
44b6_668e0cc7,3982,19,52.872191772460944,57.03667419433594,211.02567382812498,cont,4.399276517830017,57000000043,3,3,0,0
44b6_668e0cc7,4050,20,11.680000000000003,52.32000000000001,153.76000000000002,cont,6.1680256666943185,58000000027,7,5,2,0
44b6_668e0cc7,4105,20,28.66666666666666,68.53333333333333,211.46666666666658,cont,1.372459774759637,58000000045,10,5,0,5
44b6_668e0cc7,4155,20,48.99999999999998,29.600000000000005,206.39999999999998,cont,2.2072501274209784,58000000072,9,3,0,6
44b6_668e0cc7,4158,20,49.56,11.36,202.24000000000004,cont,3.319457787048958,58000000070,5,2,3,0
44b6_668e0cc7,4160,20,50.59999999999997,30.93333333333332,243.99999999999997,cont,6.815212890006528,58000000043,6,0,4,2
44b6_668e0cc7,4161,20,51.00000000000001,45.28,229.6,cont,2.1966970945034814,58000000043,8,2,4,2
44b6_668e0cc7,4239,21,11.36,58.72,152.64,cont,0.796250000000002,59000000028,5,0,5,0
44b6_668e0cc7,4314,21,26.560000000000006,87.99999999999999,202.56,cont,2.984519433677719,59000000046,8,0,2,6
44b6_668e0cc7,4378,21,47.999999999999986,40.16000000000001,199.51999999999995,cont,3.1961316951120438,59000000073,3,0,3,0
44b6_668e0cc7,4383,21,50.0,31.839999999999996,192.64000000000004,cont,1.5668404712988462,59000000071,5,0,2,3
44b6_668e0cc7,4385,21,48.39999999999999,65.28,222.56000000000003,cont,3.11601959637613,59000000044,10,0,5,5
44b6_668e0cc7,4708,23,10.639999999999999,70.08,147.35999999999999,cont,1.9076281903190706,61000000028,3,3,0,0
44b6_668e0cc7,4789,23,27.64,103.19999999999999,198.72000000000003,cont,1.1542076557101808,61000000046,3,3,0,0
44b6_668e0cc7,4856,23,47.120000000000005,52.640000000000015,196.48000000000002,cont,2.887245930380726,61000000073,4,4,0,0
44b6_668e0cc7,4864,23,47.88,74.24,225.28,cont,2.6998483753722176,61000000044,2,2,0,0
44b6_668e0cc7,4867,23,50.800000000000004,36.96,192.0,cont,3.141591953134593,61000000071,6,5,1,0
44b6_668e0cc7,4873,23,50.83999999999999,56.16,181.92000000000002,cont,6.560171491660863,61000000071,5,5,0,0
44b6_668e0cc7,5090,24,28.16,105.28,172.64,cont,1.5296593779335224,62000000046,8,0,3,5
44b6_668e0cc7,5183,24,47.999999999999986,59.999999999999986,163.99999999999997,cont,1.8168052317185632,62000000073,3,0,0,3
44b6_668e0cc7,5202,24,51.86666666666666,47.999999999999986,159.99999999999997,cont,0.9338842310181983,62000000071,3,0,0,3
44b6_668e0cc7,10182,38,55.160000000000004,77.44,239.36,cont,0.700448517023203,76000000009,1,0,1,0
44b6_668e0cc7,11546,42,29.0,69.28,225.28,cont,1.986757723025131,80000000033,2,1,1,0
44b6_668e0cc7,11590,42,35.52,118.72,156.64000000000004,cont,1.879950132317355,80000000048,1,1,0,0
44b6_668e0cc7,11935,43,28.32,72.00000000000001,222.72000000000003,cont,2.8954198745605018,81000000033,1,0,1,0
44b6_668e0cc7,11978,43,34.84,121.28000000000003,155.36,cont,0.41778620429591967,81000000048,1,0,1,0
44b6_668e0cc7,12752,45,27.32,78.72,216.64000000000001,cont,1.2299650655608034,83000000033,1,1,0,0
44b6_668e0cc7,13168,46,27.160000000000004,81.43999999999998,213.27999999999997,cont,1.740799026309473,84000000033,1,0,1,0
44b6_668e0cc7,14784,50,27.84,130.08,125.44000000000003,cont,6.8915328483581915,88000000077,1,0,1,0
44b6_668e0cc7,15170,51,24.36,97.44,208.15999999999997,cont,2.455471250595303,89000000034,2,2,0,0
44b6_668e0cc7,15382,51,62.68000000000001,64.64000000000001,185.43999999999997,cont,1.5070506170995002,89000000006,4,3,1,0
44b6_668e0cc7,15566,52,23.319999999999997,100.64000000000003,205.92,cont,1.152948665379328,90000000034,2,0,2,0
44b6_668e0cc7,15605,52,26.04,153.44000000000003,138.72000000000003,cont,1.8975658750093491,90000000049,5,0,0,5
44b6_668e0cc7,15755,52,50.320000000000014,110.72000000000003,209.92000000000004,cont,0.5332872701462336,90000000010,1,0,1,0
44b6_668e0cc7,15822,52,61.96,68.64,181.92000000000004,cont,0.1633104788432091,90000000006,3,0,3,0
44b6_668e0cc7,16182,53,49.84000000000002,112.64,206.55999999999995,cont,0.732693020643679,91000000010,1,1,0,0
44b6_668e0cc7,17189,55,63.0,84.53333333333333,163.99999999999994,cont,6.9152188494733435,93000000006,5,0,0,5
44b6_668e0cc7,17969,57,47.00000000000001,120.0,204.00000000000003,cont,1.6750116604071574,95000000010,2,2,0,0
44b6_668e0cc7,18041,57,58.839999999999996,78.08000000000001,180.64000000000004,cont,0.990583792518317,95000000006,1,1,0,0
44b6_668e0cc7,18046,57,60.31999999999999,68.64,197.28000000000003,cont,2.857829432016563,95000000069,2,1,1,0
44b6_668e0cc7,18189,58,15.64,108.63999999999999,208.64000000000004,cont,2.7771815749064706,96000000034,1,1,0,0
44b6_668e0cc7,18495,58,62.86666666666666,63.999999999999986,190.93333333333328,cont,4.093975290662557,96000000069,5,0,0,5
44b6_668e0cc7,18578,59,9.799999999999999,110.56,223.35999999999999,cont,6.770168133990468,97000000034,2,0,0,2
44b6_668e0cc7,18616,59,14.480000000000002,108.64000000000001,209.28000000000003,cont,2.8522800638261305,97000000034,1,0,1,0
44b6_668e0cc7,20311,63,11.32,108.0,214.72,cont,1.8791774297548354,101000000034,1,1,0,0
44b6_668e0cc7,21057,64,60.000000000000014,64.0,192.0,cont,6.423376497217028,102000000069,1,1,0,0
44b6_668e0cc7,26445,76,62.86666666666666,108.5333333333333,128.53333333333336,cont,6.156441409883891,114000000049,3,0,0,3
44b6_668e0cc7,27338,78,62.160000000000004,105.28000000000002,124.64000000000001,cont,3.3602941184515407,116000000049,1,1,0,0
44b6_668e0cc7,27430,79,8.519999999999998,199.36000000000004,146.56,cont,1.0380939564894844,117000000075,1,1,0,0
44b6_668e0cc7,27783,79,60.133333333333326,103.99999999999999,131.99999999999997,cont,1.6758872546438903,117000000049,4,0,0,4
44b6_668e0cc7,29114,82,59.160000000000004,53.92,108.0,cont,1.2144089354496719,120000000050,1,1,0,0
44b6_668e0cc7,29122,82,60.16000000000001,116.0,141.28000000000003,cont,0.5640880361255715,120000000049,1,1,0,0
44b6_668e0cc7,29561,83,58.84000000000001,56.00000000000001,108.64000000000001,cont,0.7334134662657791,121000000051,1,0,1,0
44b6_668e0cc7,29675,84,7.839999999999999,180.64000000000001,125.92,cont,3.572820358274399,122000000074,1,1,0,0
44b6_668e0cc7,29702,84,11.840000000000003,184.0,124.63999999999999,cont,3.1018777075345887,122000000074,1,1,0,0
44b6_668e0cc7,30070,85,3.3200000000000003,174.55999999999995,134.72,cont,6.928562486187167,123000000074,1,1,0,0
44b6_668e0cc7,30489,86,1.3199999999999998,199.36,148.00000000000003,cont,2.2150126974805375,124000000076,1,0,1,0
44b6_668e0cc7,30511,86,5.440000000000001,182.56000000000003,130.08,cont,0.7377213650830512,124000000074,4,0,0,4
44b6_668e0cc7,31294,87,58.519999999999996,64.0,111.99999999999999,cont,0.9375841628888536,125000000051,1,1,0,0
44b6_668e0cc7,31715,88,58.15999999999998,66.56000000000002,111.36,cont,1.9940544438404684,126000000052,2,1,1,0
44b6_668e0cc7,31733,88,60.16,68.0,204.64,cont,3.0901501783894,126000000037,1,1,0,0
44b6_668e0cc7,32080,89,45.68,101.28,210.08000000000004,cont,6.018579696448328,127000000006,1,1,0,0
44b6_668e0cc7,32995,91,58.16000000000001,132.00000000000003,168.64000000000004,cont,2.0058714596155056,129000000052,1,1,0,0
44b6_668e0cc7,33247,92,31.360000000000003,153.44000000000005,215.36,cont,1.4780355416903683,130000000011,1,0,1,0
44b6_668e0cc7,34533,95,29.840000000000003,158.72,212.64,cont,1.8940837164708342,133000000011,1,1,0,0
44b6_668e0cc7,34667,95,52.0,88.64,105.92000000000002,cont,4.589294608106996,133000000052,1,1,0,0
44b6_668e0cc7,34957,96,29.160000000000004,159.36,212.0,cont,1.7359381469395638,134000000011,1,0,1,0
44b6_668e0cc7,35093,96,51.00000000000001,89.28000000000003,106.72,cont,4.764687325785402,134000000052,1,0,1,0
44b6_66f9292d,1277,4,20.000000000000007,178.72000000000003,121.28000000000003,cont,0.5322960290101801,242000000012,1,1,0,0
44b6_66f9292d,1590,5,19.84,178.72000000000003,122.71999999999998,cont,0.7541820900816927,243000000012,1,0,1,0
44b6_66f9292d,2191,7,32.839999999999996,120.00000000000003,140.00000000000003,cont,0.2600000000000051,245000000007,1,1,0,0
44b6_66f9292d,2458,8,32.839999999999996,120.00000000000003,140.00000000000003,cont,0.4823267175888292,246000000007,1,0,1,0
44b6_66f9292d,8008,28,32.16000000000001,117.28,140.00000000000003,cont,1.453898315735985,266000000007,1,1,0,0
44b6_66f9292d,8296,29,31.480000000000004,116.63999999999999,139.35999999999999,cont,1.3899213107222985,267000000007,1,0,1,0
44b6_66f9292d,9177,32,31.159999999999997,119.35999999999999,136.64,cont,1.82174023944141,270000000007,1,1,0,0
44b6_66f9292d,9484,33,31.000000000000007,119.36000000000001,136.0,cont,2.488744663479948,271000000007,1,0,1,0
44b6_66f9292d,9728,34,16.565921325683597,179.72883300781254,111.31871215820313,cont,6.588986957148814,272000000012,1,1,0,0
44b6_66f9292d,9740,34,19.84,176.00000000000003,126.72,cont,1.8025784483622305,272000000012,1,1,0,0
44b6_66f9292d,10029,35,20.000000000000007,176.00000000000003,130.08,cont,2.0907631680800316,273000000012,1,0,1,0
44b6_66f9292d,17901,60,21.000000000000004,187.19999999999996,128.16000000000003,cont,2.12750918035623,298000000014,1,1,0,0
44b6_66f9292d,18235,61,21.32,188.64000000000001,127.36,cont,1.9726188177141568,299000000014,1,0,1,0
44b6_66f9292d,18243,61,22.866666666666656,176.53333333333336,135.99999999999994,cont,5.791528775336892,299000000014,2,0,0,2
44b6_66f9292d,18957,63,32.68,138.07999999999998,128.64000000000004,cont,1.3797197405632757,301000000009,1,1,0,0
44b6_66f9292d,19257,64,32.84000000000001,136.64,128.0,cont,2.206472368397103,302000000009,1,0,1,0
44b6_66f9292d,22822,75,21.160000000000004,190.72000000000003,124.63999999999999,cont,1.9608707408954837,313000000014,1,1,0,0
44b6_66f9292d,23147,76,21.0,191.36,125.28000000000002,cont,1.6714608131810909,314000000014,1,0,1,0
44b6_66f9292d,24127,79,35.32000000000001,141.92000000000002,127.36,cont,4.017038126841715,317000000009,1,1,0,0
44b6_66f9292d,24433,80,35.16,143.36,128.64000000000001,cont,2.9883215397610647,318000000009,1,0,1,0
44b6_706092f0,2790,7,29.159999999999997,117.44,112.16,cont,6.617122911432138,404000000013,1,1,0,0
44b6_706092f0,2818,7,34.68000000000001,111.99999999999999,116.63999999999999,cont,2.99326587943672,404000000013,1,1,0,0
44b6_706092f0,3203,8,34.32000000000001,111.99999999999999,116.0,cont,2.5002831089698745,405000000013,1,0,1,0
44b6_706092f0,4278,11,34.0,110.56,117.28,cont,2.1631429679981857,408000000013,1,1,0,0
44b6_706092f0,4634,12,33.84000000000001,111.36000000000001,118.72000000000003,cont,2.0335886537104853,409000000013,1,0,1,0
44b6_706092f0,13939,38,30.159999999999997,128.64000000000001,128.0,cont,5.609475521383444,435000000014,1,1,0,0
44b6_706092f0,13943,38,32.0,116.0,111.99999999999999,cont,3.3003906018833633,435000000014,1,1,0,0
44b6_706092f0,14323,39,29.800000000000004,129.44000000000003,128.8,cont,6.98789679105953,436000000014,1,0,1,0
44b6_706092f0,14332,39,32.0,116.0,111.99999999999999,cont,2.601269221457099,436000000014,1,0,1,0
44b6_706092f0,14733,40,29.839999999999996,128.64000000000001,129.28,cont,6.706664595758467,437000000014,3,0,0,3
44b6_706092f0,15455,42,30.320000000000007,128.0,130.72,cont,4.990046655593508,439000000015,1,1,0,0
44b6_706092f0,15462,42,32.0,116.0,114.72,cont,3.6435529535056874,439000000015,1,1,0,0
44b6_706092f0,15838,43,30.679999999999996,128.0,131.36000000000004,cont,5.342890281720198,440000000015,1,0,1,0
44b6_706092f0,15841,43,32.160000000000004,116.63999999999999,115.35999999999999,cont,3.2659266009051806,440000000015,1,0,1,0
44b6_706092f0,19848,54,31.32,130.72,130.08,cont,0.5332872701462171,451000000015,1,1,0,0
44b6_706092f0,20212,55,31.159999999999997,130.72000000000003,129.44000000000003,cont,1.995245912162203,452000000015,2,1,1,0
44b6_706092f0,20229,55,33.13333333333333,139.99999999999994,135.46666666666664,cont,2.032693931229074,452000000016,4,1,0,3
44b6_706092f0,20571,56,31.000000000000007,131.35999999999999,128.64000000000004,cont,0.2983103459486589,453000000015,1,0,1,0
44b6_706092f0,20592,56,33.68,140.0,133.28000000000006,cont,1.4869415968692479,453000000016,1,0,1,0
44b6_706092f0,22488,61,33.00000000000001,140.64000000000004,129.28,cont,1.3405018882866233,458000000015,1,1,0,0
44b6_706092f0,22887,62,32.68,141.28000000000003,129.92000000000002,cont,1.3002031091333481,459000000012,1,0,1,0
44b6_706092f0,24412,66,29.43999999999999,130.23999999999995,122.24000000000001,cont,1.238416327411745,463000000011,1,1,0,0
44b6_706092f0,24426,66,31.52,143.99999999999997,133.28000000000003,cont,2.0716678818044203,463000000012,1,1,0,0
44b6_706092f0,24772,67,31.159999999999997,143.99999999999997,134.07999999999998,cont,2.1853078301465874,464000000012,1,0,1,0
44b6_706092f0,25134,68,31.320000000000007,143.99999999999997,132.0,cont,2.774375211196203,465000000010,1,0,0,1
44b6_706092f0,26199,71,29.68,131.36000000000004,119.36000000000001,cont,6.7243591702109295,468000000010,1,1,0,0
44b6_706092f0,26586,72,29.160000000000004,131.35999999999999,119.36000000000001,cont,6.103881577529167,469000000010,1,0,1,0
44b6_706092f0,26601,72,30.999999999999996,144.0,128.53333333333336,cont,1.0238073176194447,469000000010,3,0,0,3
44b6_706092f0,31851,86,29.0,152.0,128.64000000000001,cont,1.2461747720524607,483000000011,2,2,0,0
44b6_706092f0,32224,87,29.0,152.8,128.16,cont,1.7757168446855498,484000000011,2,0,2,0
44b6_71a4179f,926,2,21.68,182.56000000000003,90.07999999999998,cont,3.7638662386567248,273000000031,2,2,0,0
44b6_71a4179f,1356,3,21.12,181.28,91.36,cont,4.162792031797894,274000000031,2,0,2,0
44b6_71a4179f,1372,3,22.59999999999999,191.99999999999994,85.06666666666666,cont,2.750258826203636,274000000031,1,0,0,1
44b6_71a4179f,1788,4,21.880000000000003,192.0,86.08,cont,1.6787909525905902,275000000031,1,1,0,0
44b6_71a4179f,2219,5,20.88,181.28000000000003,93.91999999999999,cont,4.6275994384237595,276000000032,2,0,0,2
44b6_71a4179f,2220,5,22.16,192.0,85.28,cont,2.222451138383021,276000000032,1,0,1,0
44b6_71a4179f,2650,6,22.16,192.0,86.72,cont,2.1002771394508866,277000000032,2,2,0,0
44b6_71a4179f,3059,7,20.840000000000003,182.07999999999998,95.36000000000001,cont,4.973296504834602,278000000032,2,0,0,2
44b6_71a4179f,3071,7,22.68,192.0,85.92,cont,2.7947165553951945,278000000032,2,0,2,0
44b6_71a4179f,4359,10,23.32,192.0,88.00000000000001,cont,2.1831317098379595,281000000027,1,1,0,0
44b6_71a4179f,4458,10,36.2,136.48,131.35999999999999,cont,1.6651294994684291,281000000028,1,1,0,0
44b6_71a4179f,4801,11,23.72,192.0,87.36000000000001,cont,2.8490842827126084,282000000027,2,0,1,1
44b6_71a4179f,4872,11,36.00000000000001,137.28000000000003,130.56,cont,2.1813771452914743,282000000028,1,0,1,0
44b6_71a4179f,6011,14,23.0,191.36,84.64000000000001,cont,1.666080730336918,285000000027,1,1,0,0
44b6_71a4179f,6429,15,23.16,191.36,84.64000000000001,cont,2.0046204441240247,286000000027,1,0,1,0
44b6_71a4179f,13193,32,36.36,114.71999999999997,132.64000000000001,cont,6.976172504497015,303000000025,1,1,0,0
44b6_71a4179f,13199,32,38.32000000000001,129.92000000000002,129.92000000000002,cont,5.523972330895596,303000000025,1,1,0,0
44b6_71a4179f,13585,33,36.160000000000004,113.28000000000002,132.00000000000006,cont,6.840555190370155,304000000025,1,0,1,0
44b6_71a4179f,13586,33,35.86666666666666,123.99999999999999,125.06666666666666,cont,2.077664161517831,304000000025,3,0,0,3
44b6_71a4179f,13608,33,38.52,130.72,130.72,cont,4.575377887399032,304000000025,1,0,1,0
44b6_71a4179f,14399,35,35.16,124.00000000000003,124.63999999999999,cont,1.6521543700574568,306000000025,1,1,0,0
44b6_71a4179f,14779,36,35.00000000000001,124.00000000000003,124.63999999999999,cont,2.0365081941892464,307000000025,1,0,1,0
44b6_71a4179f,16187,40,30.000000000000007,132.64000000000001,120.64000000000001,cont,6.688942951804853,311000000025,1,1,0,0
44b6_71a4179f,16578,41,30.000000000000007,132.64000000000001,120.63999999999999,cont,6.8044049381632075,312000000024,2,1,1,0
44b6_71a4179f,16606,41,34.480000000000004,125.92,121.92000000000002,cont,1.6994268335235843,312000000024,1,1,0,0
44b6_71a4179f,16960,42,30.000000000000007,132.64000000000001,120.64000000000001,cont,6.835380498735957,313000000024,1,0,1,0
44b6_71a4179f,16990,42,34.480000000000004,125.92,121.92000000000002,cont,1.2308235302024428,313000000024,1,0,1,0
44b6_71a4179f,17733,44,30.000000000000007,133.44,121.44000000000001,cont,5.395440494065322,315000000024,1,1,0,0
44b6_71a4179f,18124,45,30.000000000000007,132.64000000000001,120.64000000000001,cont,5.393482467293266,316000000024,1,0,1,0
44b6_71a4179f,18527,46,34.68000000000001,128.0,125.28000000000002,cont,2.8633219343971885,317000000023,1,1,0,0
44b6_71a4179f,18886,47,30.000000000000007,132.00000000000006,120.00000000000003,cont,5.701990469344877,318000000023,1,1,0,0
44b6_71a4179f,18915,47,34.0,128.0,127.36000000000001,cont,1.822682175394274,318000000023,1,0,1,0
44b6_71a4179f,19263,48,30.000000000000007,132.00000000000006,120.64000000000001,cont,5.996690363233695,319000000023,1,0,1,0
44b6_71a4179f,20744,52,32.68,122.71999999999998,127.36,cont,1.737230644589256,323000000021,1,1,0,0
44b6_71a4179f,21138,53,32.0,121.28000000000003,127.36000000000004,cont,2.2239358410259893,324000000021,1,0,1,0
44b6_71a4179f,22745,57,29.0,131.35999999999999,123.99999999999999,cont,3.6147028318383225,328000000019,1,1,0,0
44b6_71a4179f,22760,57,30.840000000000007,121.28000000000003,129.28,cont,1.9854946644350389,328000000019,1,1,0,0
44b6_71a4179f,23160,58,29.0,131.35999999999999,124.79999999999998,cont,3.806837803545087,329000000019,2,1,1,0
44b6_71a4179f,23176,58,31.199999999999996,122.71999999999998,130.72,cont,2.135685267660004,329000000019,3,2,1,0
44b6_71a4179f,23567,59,28.84000000000001,130.56,123.84,cont,3.6124009501991785,330000000019,3,0,1,2
44b6_71a4179f,23581,59,31.000000000000007,123.35999999999999,132.00000000000003,cont,2.137724403425308,330000000019,2,0,2,0
44b6_71a4179f,25157,63,30.679999999999996,121.28000000000003,133.27999999999997,cont,3.087371707860901,334000000019,1,1,0,0
44b6_71a4179f,25515,64,27.679999999999996,128.0,128.0,cont,4.452440693035678,335000000019,2,0,0,2
44b6_71a4179f,25528,64,30.839999999999996,121.28,133.28000000000003,cont,1.7914841961345906,335000000019,1,0,1,0
44b6_71a4179f,28127,71,31.48,120.64000000000001,132.64000000000001,cont,2.6473539265651636,342000000017,1,1,0,0
44b6_71a4179f,28481,72,29.999999999999993,116.0,132.53333333333333,cont,2.042772872089417,343000000017,2,0,0,2
44b6_71a4179f,28490,72,31.999999999999993,121.28000000000003,133.28,cont,3.2933021517164067,343000000017,1,0,1,0
44b6_71a4179f,31165,79,29.159999999999997,128.0,131.36000000000004,cont,3.7704552526850152,350000000013,1,1,0,0
44b6_71a4179f,31564,80,29.0,128.0,132.00000000000006,cont,5.281249999999995,351000000013,2,1,1,0
44b6_71a4179f,31592,80,34.0,124.00000000000003,136.0,cont,3.2752922102462856,351000000013,3,0,0,3
44b6_71a4179f,31974,81,29.0,128.0,132.00000000000006,cont,3.656249999999998,352000000013,1,0,1,0
44b6_71a4179f,34455,87,34.0,120.00000000000003,136.0,cont,3.300390601883357,358000000013,1,1,0,0
44b6_71a4179f,34839,88,34.0,120.00000000000003,136.0,cont,3.2752922102462856,359000000013,1,0,1,0
44b6_71a4179f,35572,90,29.0,124.00000000000003,135.35999999999999,cont,5.031756806772366,361000000013,2,2,0,0
44b6_71a4179f,35933,91,29.0,124.00000000000003,136.0,cont,5.138701197773621,362000000013,2,0,2,0
44b6_71a4179f,35956,91,33.48000000000001,117.28,136.0,cont,2.6467054992953156,362000000013,3,0,0,3
44b6_74d0c52e,1198,12,54.15999999999999,144.64000000000001,126.56000000000002,cont,2.97459345961762,250000000026,1,1,0,0
44b6_74d0c52e,1295,13,51.26666666666666,148.53333333333336,115.46666666666667,cont,1.5859367302953877,251000000026,4,0,0,4
44b6_74d0c52e,1298,13,54.680000000000014,142.56,127.19999999999996,cont,6.307114658264906,251000000026,1,0,1,0
44b6_74d0c52e,2229,22,46.79999999999999,140.64,89.28000000000002,cont,0.651014832780346,260000000024,1,1,0,0
44b6_74d0c52e,2326,23,46.839999999999996,140.00000000000003,89.92,cont,0.931367442258951,261000000024,2,1,1,0
44b6_74d0c52e,2439,24,47.0,139.35999999999999,92.63999999999999,cont,1.7623572637237914,262000000022,1,0,1,0
44b6_74d0c52e,4943,46,37.31999999999999,124.00000000000003,125.92,cont,0.6606779188833064,284000000022,2,2,0,0
44b6_74d0c52e,5061,47,35.32,123.35999999999999,128.64000000000004,cont,1.6034065298607387,285000000022,2,0,2,0
44b6_74d0c52e,5535,51,32.84000000000001,127.36,129.92000000000002,cont,1.0582481100857217,289000000021,1,1,0,0
44b6_74d0c52e,5652,52,33.000000000000014,128.0,130.08,cont,0.8794538433027572,290000000021,1,0,1,0
44b6_74d0c52e,5884,54,32.36,127.36,132.64000000000001,cont,1.103086578651014,292000000021,1,1,0,0
44b6_74d0c52e,5996,55,31.68000000000001,126.72000000000001,133.92000000000002,cont,4.3040112104872446,293000000021,3,2,1,0
44b6_74d0c52e,6001,55,36.0,131.46666666666664,127.46666666666667,cont,3.4607371164087115,293000000021,6,2,0,4
44b6_74d0c52e,7761,71,17.0,120.53333333333336,94.93333333333331,cont,3.154365499017007,309000000022,1,0,0,1
44b6_74d0c52e,7773,71,29.32,107.36000000000001,138.72000000000003,cont,1.1973284062862541,309000000021,1,1,0,0
44b6_74d0c52e,7861,72,16.08,122.72,90.23999999999998,cont,2.1831317098379497,310000000022,1,1,0,0
44b6_74d0c52e,7877,72,29.480000000000004,108.0,138.72000000000003,cont,1.3525623830345266,310000000021,1,0,1,0
44b6_74d0c52e,7963,73,15.319999999999997,123.35999999999999,88.0,cont,1.1723673965528054,311000000022,1,0,1,0
44b6_74d0c52e,8102,74,31.189821166992193,99.0002978515625,145.74816406250002,cont,6.744949555729407,312000000021,1,1,0,0
44b6_74d0c52e,8103,74,30.360000000000003,106.72,135.36,cont,1.4894257492403018,312000000021,1,1,0,0
44b6_74d0c52e,8199,75,30.82982116699219,96.44029785156249,144.46816406249997,cont,6.621720408747751,313000000021,1,0,1,0
44b6_74d0c52e,8200,75,30.320000000000007,105.28000000000002,133.27999999999997,cont,1.0683061944030798,313000000021,1,0,1,0
44b6_74d0c52e,9299,87,10.800000000000002,125.28,65.28,cont,2.941788624714567,325000000020,1,1,0,0
44b6_74d0c52e,9321,87,31.48,88.00000000000001,162.72,cont,1.5115994219699818,325000000019,1,1,0,0
44b6_74d0c52e,9383,88,10.320000000000002,124.63999999999999,66.08000000000001,cont,0.5411518386737739,326000000020,1,0,1,0
44b6_74d0c52e,9407,88,32.68,86.08,161.92000000000002,cont,1.8479329668578373,326000000019,1,0,1,0
44b6_74d0c52e,9487,89,33.68000000000001,84.00000000000001,161.92,cont,1.537579774353179,327000000019,1,1,0,0
44b6_74d0c52e,9575,90,34.52,81.92,162.07999999999998,cont,1.4748161961749633,328000000017,1,0,1,0
44b6_7a302da0,1386,2,34.519999999999996,169.28,98.72000000000001,cont,3.2040533410821954,175000000024,1,0,1,0
44b6_7a302da0,1414,2,38.480000000000004,89.27999999999999,191.36,cont,2.6816931451976496,175000000020,1,0,1,0
44b6_7a302da0,1578,2,49.68,206.72,20.000000000000007,cont,1.1108393504463225,175000000005,2,0,2,0
44b6_7a302da0,2438,4,34.00000000000001,166.08,96.64000000000001,cont,3.3385742653114816,177000000024,1,1,0,0
44b6_7a302da0,2523,4,39.36000000000001,190.72000000000003,6.720000000000001,cont,0.7909260790364611,177000000004,1,1,0,0
44b6_7a302da0,2963,5,33.519999999999996,164.00000000000003,94.72,cont,5.8087329190624635,178000000024,1,0,1,0
44b6_7a302da0,2964,5,33.86666666666666,166.93333333333328,103.46666666666663,cont,6.995368631578068,178000000024,5,0,0,5
44b6_7a302da0,3010,5,38.0,88.00000000000001,193.27999999999997,cont,1.7061726172928637,178000000020,1,1,0,0
44b6_7a302da0,3038,5,39.32,189.28,8.0,cont,1.1911371929798888,178000000004,2,1,1,0
44b6_7a302da0,3556,6,37.67999999999999,88.00000000000001,194.71999999999997,cont,1.221239124823631,179000000020,1,0,1,0
44b6_7a302da0,3587,6,39.32000000000001,188.64000000000001,9.28,cont,0.7497926796788621,179000000004,1,0,1,0
44b6_7a302da0,3588,6,39.00000000000001,223.99999999999997,91.36,cont,1.763780120224751,179000000023,1,0,1,0
44b6_7a302da0,3954,7,24.68,40.000000000000014,247.36,cont,1.3959875536694362,180000000058,1,0,1,0
44b6_7a302da0,4686,8,36.83999999999999,88.00000000000001,196.00000000000003,cont,1.6396402997303903,181000000020,1,1,0,0
44b6_7a302da0,4708,8,38.52000000000001,88.00000000000001,180.15999999999997,cont,6.967536889927478,181000000020,1,1,0,0
44b6_7a302da0,5237,9,36.160000000000004,88.00000000000001,196.00000000000003,cont,1.9282800788526626,182000000020,1,0,1,0
44b6_7a302da0,5252,9,37.96,188.16000000000003,23.36,cont,4.826359426368907,182000000004,1,1,0,0
44b6_7a302da0,5320,9,41.16,171.99999999999994,8.0,cont,6.632329553972744,182000000004,1,1,0,0
44b6_7a302da0,5431,9,49.00000000000001,197.28000000000003,27.36,cont,0.9377249730064882,182000000005,1,1,0,0
44b6_7a302da0,5864,10,38.360000000000014,180.48000000000002,21.920000000000005,cont,4.692678139932448,183000000004,4,0,1,3
44b6_7a302da0,6002,10,49.160000000000004,196.00000000000003,28.640000000000008,cont,0.5479407472528539,183000000005,1,0,1,0
44b6_7a302da0,6321,11,31.000000000000007,158.71999999999997,104.00000000000003,cont,0.9646534351776225,184000000024,1,0,1,0
44b6_7a302da0,7020,12,39.16,173.27999999999997,21.92,cont,5.695248293533848,185000000004,1,1,0,0
44b6_7a302da0,7048,12,40.31999999999999,185.28,17.28,cont,1.0683061944030752,185000000004,1,1,0,0
44b6_7a302da0,7463,13,30.480000000000004,155.99999999999997,99.19999999999999,cont,1.3792411863049983,186000000024,1,1,0,0
44b6_7a302da0,8215,14,42.84,220.0,104.0,cont,6.305774759099798,187000000023,1,0,1,0
44b6_7a302da0,8568,15,30.680000000000003,153.27999999999997,99.36000000000001,cont,1.9587149007703901,188000000024,1,1,0,0
44b6_7a302da0,8658,15,36.839999999999996,85.44,180.0,cont,6.4280407444648295,188000000020,1,1,0,0
44b6_7a302da0,9316,16,42.84,185.28,25.28,cont,1.9298542691094498,189000000004,1,1,0,0
44b6_7a302da0,9720,17,33.480000000000004,81.28000000000002,189.28,cont,0.8407706658179803,190000000020,1,0,1,0
44b6_7a302da0,11073,19,44.160000000000004,188.64000000000001,17.28,cont,4.460203190438753,192000000004,1,1,0,0
44b6_7a302da0,11091,19,44.04000000000001,192.16,33.440000000000005,cont,5.11624740410391,192000000004,2,2,0,0
44b6_7a302da0,11426,20,30.799999999999997,149.27999999999997,100.63999999999999,cont,1.6891404063901798,193000000024,2,2,0,0
44b6_7a302da0,11587,20,41.39999999999999,207.46666666666664,100.53333333333332,cont,6.394993837413416,193000000023,5,0,0,5
44b6_7a302da0,11632,20,44.32,188.00000000000003,18.72,cont,4.849828766307126,193000000004,1,0,1,0
44b6_7a302da0,11633,20,44.0,192.64000000000004,35.36,cont,5.825959282598891,193000000004,2,0,2,0
44b6_7a302da0,11647,20,44.86666666666667,179.99999999999997,27.999999999999996,cont,0.4604166666666801,193000000004,9,0,0,9
44b6_7a302da0,11721,20,51.32000000000001,188.64000000000001,52.0,cont,2.744759210750552,193000000005,2,1,1,0
44b6_7a302da0,11728,20,51.84000000000001,169.28,42.720000000000006,cont,6.623823716328811,193000000005,2,1,1,0
44b6_7a302da0,11992,21,30.679999999999996,148.00000000000003,102.08,cont,0.6403819660952392,194000000024,2,0,2,0
44b6_7a302da0,12149,21,40.360000000000014,216.0,106.07999999999998,cont,0.7312500000000128,194000000023,1,1,0,0
44b6_7a302da0,12307,21,52.16,188.00000000000003,54.08,cont,3.133554541172064,194000000005,1,0,1,0
44b6_7a302da0,12729,22,39.839999999999996,214.72,104.64000000000001,cont,0.5994906692351408,195000000023,1,0,1,0
44b6_7a302da0,13805,24,30.680000000000003,145.28,103.36000000000003,cont,0.852776714621116,197000000024,1,0,1,0
44b6_7a302da0,14931,26,30.320000000000007,143.36,104.0,cont,1.1863612961488503,199000000024,1,1,0,0
44b6_7a302da0,15502,27,31.359999999999996,57.279999999999994,187.35999999999999,cont,6.717778301827182,200000000020,1,1,0,0
44b6_7a302da0,15776,27,53.52,188.00000000000003,60.640000000000015,cont,4.590013786743999,200000000005,1,1,0,0
44b6_7a302da0,16302,28,52.84,189.28000000000003,62.71999999999999,cont,5.816705602185819,201000000005,5,0,1,4
44b6_7a302da0,16330,28,55.99999999999999,187.99999999999997,55.466666666666654,cont,0.4604166666666818,201000000005,2,0,0,2
44b6_7a302da0,16771,29,45.31999999999999,184.0,21.919999999999998,cont,5.98339916978636,202000000004,1,1,0,0
44b6_7a302da0,19980,35,33.160000000000004,64.64,197.28,cont,0.9731023199026887,208000000020,1,1,0,0
44b6_7a302da0,20548,36,32.84,63.36,198.72,cont,0.4698470495810295,209000000020,1,0,1,0
44b6_7a302da0,20766,36,48.73333333333332,170.93333333333328,36.0,cont,1.1919743920860242,209000000004,5,0,0,5
44b6_7a302da0,22858,40,38.64,208.8,113.28000000000003,cont,5.333986695005895,213000000023,1,1,0,0
44b6_7a302da0,22953,40,46.16,163.36,34.72,cont,2.0156551354336307,213000000004,1,0,1,0
44b6_7a302da0,23051,40,52.866666666666646,163.4666666666666,51.99999999999999,cont,6.647730920313277,213000000005,9,0,0,9
44b6_7a302da0,23446,41,40.39999999999999,200.53333333333333,116.53333333333333,cont,6.584871260236511,214000000023,4,0,0,4
44b6_7a302da0,23983,42,37.67999999999999,210.71999999999997,114.07999999999998,cont,6.100830886035113,215000000023,1,1,0,0
44b6_7a302da0,24512,43,38.28,196.64000000000001,119.36000000000001,cont,4.565093269857685,216000000023,2,1,1,0
44b6_7a302da0,24514,43,37.32,208.64000000000001,113.28000000000002,cont,6.2921082764443925,216000000023,2,1,1,0
44b6_7a302da0,24595,43,45.48,160.00000000000003,32.63999999999999,cont,1.7205059212336373,216000000004,1,0,1,0
44b6_7a302da0,24968,44,30.159999999999997,129.28,120.63999999999999,cont,0.772687072817956,217000000024,1,1,0,0
44b6_7a302da0,25647,45,45.00000000000001,156.00000000000006,32.64,cont,1.9889495500137913,218000000004,1,1,0,0
44b6_7a302da0,29200,52,34.839999999999996,171.99999999999994,115.35999999999999,cont,5.927063591062636,225000000023,1,1,0,0
44b6_7a302da0,29696,53,34.68000000000001,171.99999999999994,116.63999999999999,cont,6.23045004895315,226000000023,1,0,1,0
44b6_7a302da0,30206,54,31.73333333333333,43.999999999999986,232.5333333333333,cont,6.019571667116814,227000000020,5,0,0,5
44b6_7a302da0,31888,57,32.84,114.71999999999997,147.36000000000004,cont,6.508850224886131,230000000023,2,0,2,0
44b6_7a302da0,34104,61,46.47999999999999,141.28000000000003,33.92,cont,4.106365675691816,234000000004,1,1,0,0
44b6_7a302da0,34520,62,33.93276794433594,130.68073486328126,150.58410888671875,cont,1.7826538507991108,235000000022,1,1,0,0
44b6_7a302da0,38996,70,47.2,135.36000000000004,35.36,cont,2.38476708768383,243000000004,3,3,0,0
44b6_7a302da0,42265,76,46.36,132.8,40.8,cont,0.7658216584166251,249000000004,1,1,0,0
44b6_7a302da0,42309,76,50.0,134.07999999999998,47.360000000000014,cont,6.994770926377791,249000000004,2,1,1,0
44b6_7a302da0,42348,76,53.16,148.64000000000001,79.36,cont,1.6162828689619946,249000000005,2,2,0,0
44b6_7a302da0,42918,77,53.0,148.00000000000006,80.00000000000003,cont,1.7235727791422013,250000000005,2,0,2,0
44b6_7a302da0,43353,78,44.160000000000004,116.63999999999999,45.919999999999995,cont,6.8533014398171685,251000000004,1,1,0,0
44b6_7a302da0,44918,81,45.480000000000004,143.99999999999997,35.36,cont,6.882982865371657,254000000004,1,1,0,0
44b6_7a302da0,46805,85,22.000000000000004,35.36,202.07999999999998,cont,5.935366722031585,258000000019,1,0,1,0
44b6_7a302da0,47019,85,44.64000000000001,125.92,45.919999999999995,cont,2.2416166738539336,258000000004,1,1,0,0
44b6_7a302da0,49242,89,51.84000000000002,130.56,76.48,cont,0.9114497380547032,262000000005,5,0,0,5
44b6_7a302da0,49507,90,25.28,22.720000000000006,214.55999999999995,parent,2.2357189140184777,263000000019,2,2,0,0
44b6_7a302da0,50023,91,23.133333333333326,18.93333333333333,198.93333333333328,daughter,1.8950593658247352,264000000020,5,1,0,4
44b6_7a302da0,50036,91,25.480000000000004,23.520000000000003,218.08000000000004,daughter,2.003566351534178,264000000019,3,1,2,0
44b6_7a302da0,50784,92,46.0,133.92000000000002,40.000000000000014,cont,5.7871221528925805,265000000004,1,1,0,0
44b6_7e557709,770,1,42.08,12.320000000000004,225.28,cont,0.5633854142591905,92000000042,1,1,0,0
44b6_7e557709,1211,2,42.160000000000004,12.64,223.36,cont,1.6162828689619817,93000000042,1,0,1,0
44b6_7e557709,2159,4,51.86666666666666,31.999999999999993,247.99999999999997,cont,0.6140216359742268,95000000081,2,0,0,2
44b6_7e557709,2243,5,6.0,242.56000000000006,33.92,cont,0.7357500424736834,96000000015,1,1,0,0
44b6_7e557709,2675,6,5.160000000000002,242.71999999999997,32.0,cont,0.7092524673908526,97000000015,1,0,1,0
44b6_7e557709,2719,6,13.640000000000002,236.0,64.64,cont,0.640175757116745,97000000007,1,1,0,0
44b6_7e557709,3171,7,13.16,237.27999999999997,64.0,cont,1.4287068016216593,98000000007,1,0,1,0
44b6_7e557709,4665,10,30.680000000000003,179.35999999999999,160.64000000000001,cont,0.7726870728179608,101000000070,4,0,0,4
44b6_7e557709,4984,11,16.68,231.36,74.08000000000001,cont,6.886799079579712,102000000007,1,0,1,0
44b6_7e557709,5628,12,40.84,12.640000000000004,208.64000000000007,cont,1.9697386406576707,103000000041,1,1,0,0
44b6_7e557709,6064,13,40.839999999999996,12.64,208.64,cont,0.6278895703067583,104000000041,1,0,1,0
44b6_7e557709,9205,20,51.32,32.0,233.27999999999997,cont,6.791994598422487,111000000080,1,1,0,0
44b6_7e557709,9633,21,51.68,32.0,235.35999999999999,cont,6.8670431227712605,112000000080,1,0,1,0
44b6_7e557709,10020,22,42.83999999999999,9.28,214.08,cont,0.5411518386737785,113000000041,1,1,0,0
44b6_7e557709,10477,23,42.999999999999986,8.64,215.35999999999996,cont,1.8411328883326477,114000000041,1,0,1,0
44b6_7e557709,14647,32,61.00000000000001,47.360000000000014,164.0,cont,1.6456685571523662,123000000012,1,1,0,0
44b6_7e557709,15081,33,61.00000000000001,47.360000000000014,164.64,cont,0.6821129855089914,124000000012,1,0,1,0
44b6_7e557709,17397,38,61.00000000000001,61.44000000000001,162.07999999999998,cont,6.780846762204557,129000000014,1,0,1,0
44b6_7e557709,27071,58,58.48000000000002,51.36000000000001,144.64000000000001,cont,6.545866300574725,149000000014,2,2,0,0
44b6_7e557709,27085,58,59.68000000000001,39.36,154.72000000000003,cont,0.8831990857105756,149000000014,1,1,0,0
44b6_7e557709,27596,59,59.83999999999999,39.36,156.00000000000006,cont,2.0578557559994595,150000000014,1,0,1,0
44b6_7e557709,29467,63,44.00000000000001,12.0,199.36,cont,1.9889495500137822,154000000043,1,1,0,0
44b6_7e557709,29949,64,43.84,12.0,199.36000000000004,cont,1.518918714908732,155000000043,1,0,1,0
44b6_7e557709,31465,67,42.999999999999986,10.559999999999999,200.64000000000001,cont,0.2704539008777616,158000000043,1,1,0,0
44b6_7e557709,31951,68,42.68000000000001,11.36,202.07999999999998,cont,0.5822853681829734,159000000043,1,0,1,0
44b6_7e557709,32602,69,60.31999999999999,40.000000000000014,152.0,cont,5.667710213348944,160000000014,1,1,0,0
44b6_7e557709,32842,70,31.160000000000004,192.0,156.00000000000003,cont,1.4809804607083774,161000000058,1,1,0,0
44b6_7e557709,32875,70,34.480000000000004,179.36,140.64000000000001,cont,5.92046622425127,161000000061,1,1,0,0
44b6_7e557709,32888,70,35.32,162.07999999999998,142.72,cont,1.327834821240957,161000000061,1,1,0,0
44b6_7e557709,33103,70,60.16000000000001,40.000000000000014,152.0,cont,4.898802308983285,161000000014,1,0,1,0
44b6_7e557709,33104,70,59.86666666666666,47.466666666666654,163.46666666666664,cont,1.0478835316166155,161000000014,4,0,0,4
44b6_7e557709,33111,70,60.68000000000001,60.000000000000014,152.0,cont,6.193126356090283,161000000014,1,0,1,0
44b6_7e557709,33360,71,30.840000000000007,192.0,154.56,cont,1.9886840026761328,162000000058,1,0,1,0
44b6_7e557709,33396,71,34.16,179.36,140.0,cont,6.2077764990776,162000000061,1,0,1,0
44b6_7e557709,33406,71,34.839999999999996,164.00000000000003,143.36,cont,0.5479407472528269,162000000061,1,0,1,0
44b6_7e557709,33415,71,36.26666666666666,157.06666666666658,136.00000000000003,cont,5.018608168744498,162000000061,3,0,0,3
44b6_7e557709,34407,73,34.32000000000001,167.36000000000004,144.64000000000001,cont,1.1445584574411158,164000000061,2,2,0,0
44b6_7e557709,34408,73,34.0,179.35999999999999,138.07999999999998,cont,5.799793234461035,164000000061,1,1,0,0
44b6_7e557709,34924,74,34.16,168.64000000000004,144.64000000000001,cont,1.3805807202043703,165000000061,2,0,2,0
44b6_7e557709,34925,74,34.0,179.99999999999997,136.64,cont,5.843360815917492,165000000061,1,0,1,0
44b6_7e557709,35356,75,24.199999999999996,197.28,120.64000000000001,cont,6.249493104444555,166000000063,1,1,0,0
44b6_7e557709,35415,75,30.679999999999996,198.72,125.44000000000001,cont,5.112142560854502,166000000063,1,1,0,0
44b6_7e557709,35884,76,23.840000000000003,198.72,120.64000000000001,cont,5.360924010839915,167000000063,1,0,1,0
44b6_7e557709,35917,76,26.999999999999993,191.46666666666664,116.0,cont,2.01293128574007,167000000063,2,0,0,2
44b6_7e557709,35955,76,30.839999999999996,199.36,124.63999999999999,cont,6.9337818865320475,167000000063,1,0,1,0
44b6_7e557709,36201,76,58.999999999999986,54.080000000000005,168.0,cont,1.8477186102055931,167000000014,1,1,0,0
44b6_7e557709,36741,77,55.99999999999999,55.466666666666654,168.53333333333333,cont,6.507218214328537,168000000014,8,0,0,8
44b6_7e557709,36761,77,58.999999999999986,54.08000000000001,168.0,cont,1.8025052011020892,168000000014,1,0,1,0
44b6_7e557709,36799,77,62.52000000000001,62.71999999999999,156.0,cont,6.927304670649335,168000000014,2,1,1,0
44b6_7e557709,37303,78,60.16,41.28000000000001,156.64,cont,6.350736177798606,169000000014,1,0,1,0
44b6_7e557709,37502,79,25.800000000000004,188.64000000000004,116.0,cont,1.4607837151679726,170000000059,1,1,0,0
44b6_7e557709,38005,80,25.64,188.64000000000004,116.63999999999999,cont,5.295804943537842,171000000059,1,0,1,0
44b6_7e557709,38012,80,26.266666666666662,203.46666666666664,119.46666666666667,cont,1.8575297216105713,171000000059,4,0,0,4
44b6_7e557709,38871,81,61.839999999999996,64.64,157.92000000000002,cont,6.486600250709147,172000000014,1,1,0,0
44b6_7e557709,48783,81,59.998218231201164,49.323596801757816,161.99559448242186,cont,2.2084183007905764,172000000014,1,1,0,0
44b6_7e557709,39381,82,59.32000000000001,52.160000000000004,162.07999999999998,cont,1.3949467955803827,173000000014,4,0,0,4
44b6_7e557709,39390,82,60.070319213867194,46.863820800781255,159.51804199218748,cont,3.0890967754904803,173000000014,1,0,1,0
44b6_7e557709,39406,82,62.000000000000014,65.28,160.64000000000001,cont,6.108465565303628,173000000014,1,0,1,0
44b6_7e557709,42330,88,29.32,207.99999999999997,162.08,cont,1.6752481159517854,179000000056,1,1,0,0
44b6_7e557709,42874,89,29.519999999999996,208.64000000000004,161.28,cont,1.7881646491584615,180000000056,1,0,1,0
44b6_7e557709,44196,91,58.680000000000014,53.28,161.91999999999996,cont,3.0993227586845653,182000000015,1,1,0,0
44b6_7e557709,44704,92,58.31999999999999,52.64,160.64000000000001,cont,3.327720691404245,183000000015,1,0,1,0
44b6_7e557709,44705,92,58.0,55.99999999999999,167.46666666666664,cont,0.21666666666668277,183000000015,6,0,0,6
44b6_7e557709,44893,93,22.68,195.36000000000004,121.28,cont,0.5924010993406341,184000000059,1,1,0,0
44b6_7e557709,45127,93,45.84,22.72,186.71999999999997,cont,6.4475650684968,184000000041,1,0,1,0
44b6_7e557709,45438,94,22.319999999999997,196.00000000000003,121.28000000000003,cont,1.4418613187474125,185000000059,1,0,1,0
44b6_808952d6,15362,88,34.0,40.64000000000001,76.0,cont,0.4317732333065582,329000000000,1,1,0,0
44b6_808952d6,15513,89,33.68000000000001,38.72,75.36,cont,0.5520218519225375,330000000000,1,0,1,0
44b6_81c256f0,2750,22,19.84,170.71999999999997,153.27999999999997,cont,1.910947490775187,191000000000,1,1,0,0
44b6_81c256f0,2873,23,20.52,169.44000000000005,152.63999999999996,cont,2.4201591502419832,192000000000,1,0,1,0
44b6_81c256f0,3469,28,22.319999999999997,156.64,145.92,cont,0.9241093685273294,197000000000,1,1,0,0
44b6_81c256f0,3593,29,22.840000000000003,153.91999999999996,144.64,cont,0.47820301651075786,198000000000,1,0,1,0
44b6_81c256f0,4374,35,23.16,190.56000000000003,156.0,cont,3.1067688198512635,204000000001,1,1,0,0
44b6_81c256f0,4512,36,23.68,190.72000000000003,155.36,cont,2.2752321310143286,205000000001,1,0,1,0
44b6_81c256f0,6458,52,30.800000000000004,121.28000000000003,124.64000000000001,cont,1.9705428344748024,221000000000,1,1,0,0
44b6_81c256f0,6576,53,31.160000000000007,118.72000000000003,123.36000000000003,cont,3.1335966755471114,222000000000,1,0,1,0
44b6_81c256f0,10388,86,26.48,89.28000000000003,115.36000000000001,cont,2.700092880346901,255000000001,1,1,0,0
44b6_81c256f0,10495,87,27.0,87.36,113.28000000000002,cont,1.6495951207796413,256000000001,1,0,1,0
44b6_87bba6c4,1126,5,59.160000000000004,175.36,215.36,cont,3.1349446765453366,11000000019,1,0,1,0
44b6_87bba6c4,1127,5,58.99999999999999,187.99999999999994,211.46666666666658,cont,5.965346251708928,11000000019,10,1,0,9
44b6_87bba6c4,1362,6,58.680000000000014,174.72,218.72000000000006,cont,3.84105454270046,12000000019,1,1,0,0
44b6_87bba6c4,1363,6,58.999999999999986,187.68,213.28000000000003,cont,6.366350995664641,12000000019,3,2,1,0
44b6_87bba6c4,1594,7,58.35999999999999,172.95999999999998,220.96,cont,4.290061552588734,13000000019,2,1,1,0
44b6_87bba6c4,1596,7,58.999999999999986,185.44,217.28000000000003,cont,6.195705417868098,13000000019,3,1,2,0
44b6_87bba6c4,1829,8,60.120000000000005,180.64000000000001,241.6,cont,6.890459890312106,14000000019,1,0,1,0
44b6_87bba6c4,1830,8,58.84000000000001,182.07999999999998,220.0,cont,4.502159204204123,14000000019,2,1,1,0
44b6_87bba6c4,1873,8,61.99999999999999,175.46666666666667,212.5333333333333,cont,6.39120761063371,14000000019,8,1,0,7
44b6_87bba6c4,2077,9,57.879999999999995,177.28,223.52,cont,2.819831555252895,15000000019,1,0,1,0
44b6_87bba6c4,3930,16,48.839999999999996,148.00000000000006,223.36,cont,2.783307665799791,22000000021,1,1,0,0
44b6_87bba6c4,3943,16,50.36,165.28,233.44,cont,5.886919955503039,22000000021,2,2,0,0
44b6_87bba6c4,4218,17,47.52000000000001,143.84000000000003,223.20000000000005,cont,1.4933210681229876,23000000021,6,5,1,0
44b6_87bba6c4,4512,18,46.16,143.20000000000002,224.64000000000004,cont,6.997167730589293,24000000021,7,0,5,2
44b6_87bba6c4,4518,18,46.040000000000006,127.84000000000002,211.36,cont,3.183839074136755,24000000021,10,0,5,5
44b6_87bba6c4,5103,20,40.68,134.07999999999998,223.35999999999999,cont,5.664447926541473,26000000021,5,5,0,0
44b6_87bba6c4,5121,20,43.35999999999999,147.36,231.20000000000002,cont,4.678109427161804,26000000021,3,3,0,0
44b6_87bba6c4,5127,20,45.0,137.76,216.16000000000003,cont,3.942994777133228,26000000021,6,6,0,0
44b6_87bba6c4,5410,21,40.84,135.20000000000002,226.72,cont,4.252411080786986,27000000021,7,0,5,2
44b6_87bba6c4,5411,21,41.8,150.56,236.15999999999997,cont,4.483115546135298,27000000021,3,0,3,0
44b6_87bba6c4,7490,27,59.68000000000001,192.0,167.36,cont,5.462030872303799,33000000067,1,0,1,0
44b6_87bba6c4,8321,30,30.320000000000007,126.72000000000004,233.27999999999997,cont,4.396428415486831,36000000021,2,1,1,0
44b6_87bba6c4,8830,31,55.84000000000001,178.08,168.0,cont,5.2198935872774035,37000000067,2,2,0,0
44b6_87bba6c4,9159,32,54.160000000000004,174.55999999999995,167.36000000000004,cont,1.8408460181394612,38000000067,2,0,2,0
44b6_87bba6c4,9943,34,61.4,139.99999999999997,161.28,cont,1.700358877560857,40000000029,2,1,1,0
44b6_87bba6c4,9963,34,62.133333333333326,150.93333333333334,151.99999999999997,cont,4.719421401830488,40000000029,6,0,0,6
44b6_87bba6c4,10050,35,21.52,124.00000000000003,226.71999999999997,cont,6.767749464371459,41000000021,2,2,0,0
44b6_87bba6c4,10272,35,58.0,136.5333333333333,155.46666666666664,cont,3.2644124875525287,41000000029,7,0,0,7
44b6_87bba6c4,10311,35,61.866666666666646,139.99999999999994,163.46666666666658,cont,4.587248327943647,41000000029,1,0,1,0
44b6_87bba6c4,10324,35,62.44,145.92,151.36,cont,5.959475648075084,41000000029,1,1,0,0
44b6_87bba6c4,10427,36,22.733333333333327,115.99999999999997,227.46666666666664,cont,5.187006335278792,42000000021,9,0,0,9
44b6_87bba6c4,10632,36,55.73333333333333,134.1333333333333,152.5333333333333,cont,3.6886068254152584,42000000029,7,0,0,7
44b6_87bba6c4,10643,36,57.83999999999999,134.08,156.32,cont,1.3739660885553173,42000000029,6,0,0,6
44b6_87bba6c4,10781,37,18.32,124.64000000000001,222.56000000000006,cont,6.864081557280609,43000000021,5,5,0,0
44b6_87bba6c4,10809,37,23.32,112.32,228.63999999999996,cont,6.928562486187175,43000000021,7,7,0,0
44b6_87bba6c4,10994,37,53.56,128.16,147.36,cont,2.7890416140674543,43000000029,3,3,0,0
44b6_87bba6c4,11032,37,57.359999999999985,128.8,156.16000000000003,cont,4.3712802043566015,43000000029,1,1,0,0
44b6_87bba6c4,11192,38,20.840000000000003,96.0,223.20000000000005,cont,5.636337740501366,44000000021,12,0,2,10
44b6_87bba6c4,11336,38,43.68,150.07999999999998,153.91999999999996,cont,2.5722779909061084,44000000067,9,1,3,5
44b6_87bba6c4,11397,38,52.160000000000004,121.11999999999998,143.2,cont,3.1200423174213414,44000000029,3,0,3,0
44b6_87bba6c4,11714,39,43.639999999999986,152.8,153.28000000000003,cont,1.475532213304735,45000000067,2,1,1,0
44b6_87bba6c4,11759,39,50.6769091796875,118.07782470703125,141.39192138671876,cont,2.983850669543967,45000000029,1,1,0,0
44b6_87bba6c4,12114,40,44.00000000000001,154.08000000000007,153.27999999999997,cont,2.659147537275798,46000000067,2,1,1,0
44b6_87bba6c4,12157,40,49.8769091796875,114.23782470703125,140.11192138671873,cont,3.482627236032647,46000000029,1,0,1,0
44b6_87bba6c4,12477,41,41.466666666666654,159.99999999999997,151.46666666666664,cont,1.8513989174105534,47000000067,5,2,0,3
44b6_87bba6c4,12497,41,44.20000000000002,152.8,154.72000000000003,cont,5.980551944636902,47000000067,3,2,1,0
44b6_87bba6c4,12855,42,39.96,156.96000000000004,148.95999999999998,cont,1.8682553693218753,48000000067,2,0,2,0
44b6_87bba6c4,12889,42,43.84,149.92000000000004,156.00000000000006,cont,5.8003623270447635,48000000067,2,0,2,0
44b6_87bba6c4,14489,46,36.959999999999994,161.44,146.08,cont,1.8751683257777245,52000000067,5,5,0,0
44b6_87bba6c4,14559,46,46.75363647460938,93.79644775390625,139.42831542968753,cont,1.3300875082228125,52000000029,2,2,0,0
44b6_87bba6c4,14893,47,31.879999999999992,146.07999999999998,142.56,cont,3.920057796130067,53000000067,9,0,4,5
44b6_87bba6c4,14981,47,46.43999999999999,74.07999999999998,119.67999999999999,cont,4.279461487734174,53000000029,6,1,2,3
44b6_87bba6c4,15414,48,44.133333333333326,69.06666666666663,124.53333333333329,cont,2.15886616653279,54000000030,6,1,0,5
44b6_87bba6c4,15437,48,46.2,78.24000000000001,119.52000000000001,cont,4.2040103621423235,54000000030,5,1,1,3
44b6_87bba6c4,15867,49,44.68,65.75999999999999,119.03999999999999,cont,4.961867717906234,55000000030,4,1,1,2
44b6_87bba6c4,16205,50,30.0,127.36,127.36,cont,3.270733862606372,56000000069,2,2,0,0
44b6_87bba6c4,16633,51,29.680000000000007,116.16000000000001,125.28,cont,5.042451195847115,57000000069,5,3,2,0
44b6_87bba6c4,16655,51,32.52,94.55999999999999,134.56000000000003,cont,6.518680967994999,57000000069,14,4,4,6
44b6_87bba6c4,17052,52,30.32,72.96000000000001,130.56,cont,3.0550864349474582,58000000069,12,4,4,4
44b6_87bba6c4,17084,52,33.24,90.72,133.44,cont,6.539408660192447,58000000069,16,6,4,6
44b6_87bba6c4,17508,53,28.200000000000003,102.88000000000001,120.80000000000001,cont,2.610591707352948,59000000070,9,2,3,4
44b6_87bba6c4,17590,53,39.999999999999986,29.279999999999994,110.39999999999999,cont,1.7463269489130857,59000000031,6,2,4,0
44b6_87bba6c4,17608,53,42.519999999999996,41.76,103.51999999999998,cont,5.820789921694479,59000000031,6,1,2,3
44b6_87bba6c4,17952,54,27.32,99.19999999999999,120.0,cont,1.7538711647381588,60000000070,6,4,2,0
44b6_87bba6c4,18047,54,39.99999999999999,10.933333333333316,101.06666666666663,cont,2.3174852420721477,60000000031,2,0,2,0
44b6_87bba6c4,18417,55,28.119999999999997,115.2,134.56,cont,2.490547756016732,61000000070,14,6,5,3
44b6_87bba6c4,18475,55,35.999999999999986,33.066666666666656,119.73333333333332,cont,5.590068232787717,61000000031,4,0,2,2
44b6_87bba6c4,18498,55,39.31015523274739,38.25415166219075,98.10815429687499,cont,6.435086153524963,61000000031,1,0,1,0
44b6_87bba6c4,18893,56,26.13333333333333,127.99999999999997,116.53333333333333,cont,1.8750196758227025,62000000070,4,0,0,4
44b6_87bba6c4,18919,56,28.4,146.72,120.0,cont,6.971401513505011,62000000070,3,0,2,1
44b6_87bba6c4,18964,56,35.52,44.80000000000001,96.80000000000001,cont,4.777306543702208,62000000031,3,0,2,1
44b6_87bba6c4,18988,56,40.480000000000004,54.08,90.56,cont,5.190571530428996,62000000031,2,0,0,2
44b6_87bba6c4,20656,60,24.520000000000003,132.64000000000001,123.36,cont,3.8869782608087733,66000000070,1,1,0,0
44b6_87bba6c4,21070,61,22.680000000000003,134.08,122.07999999999997,cont,3.826762869907153,67000000070,1,0,1,0
44b6_87bba6c4,22290,64,14.16,128.16000000000003,108.32,cont,4.1994854595533475,70000000072,3,0,1,2
44b6_87bba6c4,22742,65,14.840000000000003,129.44000000000003,109.44000000000001,cont,1.8791774297548303,71000000072,2,2,0,0
44b6_87bba6c4,22783,65,20.2,43.68000000000001,100.00000000000001,cont,4.814746391296227,71000000032,2,0,0,2
44b6_87bba6c4,23190,66,15.040000000000003,132.00000000000006,108.64000000000001,cont,1.9850291244714835,72000000072,2,0,2,0
44b6_87bba6c4,24034,68,8.84,131.35999999999999,108.0,cont,5.392258339508596,74000000072,1,1,0,0
44b6_87bba6c4,24456,69,8.680000000000001,130.56,104.0,cont,2.3049810194446265,75000000072,1,0,1,0
44b6_8cc6506c,734,2,51.73333333333333,68.53333333333333,148.0,cont,5.414296356381151,297000000003,3,0,0,3
44b6_8cc6506c,1071,3,58.31999999999999,48.0,180.64000000000001,cont,0.6758906161502611,298000000005,1,0,1,0
44b6_8cc6506c,1912,6,57.0,56.0,181.28000000000003,cont,0.42187453703678574,301000000005,1,1,0,0
44b6_8cc6506c,2179,7,56.68,57.28000000000001,182.72,cont,0.7441364542205939,302000000005,2,1,1,0
44b6_8cc6506c,3504,12,60.31999999999999,65.28,220.0,cont,0.7218000502216534,307000000004,1,1,0,0
44b6_8cc6506c,3753,13,60.0,67.36,220.0,cont,0.14624999999999844,308000000004,1,0,1,0
44b6_8cc6506c,3974,14,55.16000000000001,67.35999999999999,188.00000000000003,cont,2.362517526390031,309000000005,1,1,0,0
44b6_8cc6506c,4222,15,54.84,68.0,188.00000000000003,cont,1.6396402997304054,310000000005,3,2,1,0
44b6_8cc6506c,4224,15,56.60000000000001,74.88,212.0,cont,5.573394667973902,310000000004,2,1,1,0
44b6_8cc6506c,4497,16,58.400000000000006,71.20000000000002,214.08,cont,1.355098013060314,311000000004,5,0,1,4
44b6_8cc6506c,4749,17,57.68000000000001,75.2,216.63999999999996,cont,1.5052097029982265,312000000004,1,1,0,0
44b6_8cc6506c,4924,18,40.32,96.64000000000001,132.64000000000001,cont,1.1645707363659856,313000000002,1,1,0,0
44b6_8cc6506c,5227,19,53.053983764648436,71.92326538085939,186.958369140625,cont,1.7960201215768037,314000000005,1,1,0,0
44b6_8cc6506c,5512,20,59.48,72.64,217.92,cont,4.0678265080998735,315000000004,2,2,0,0
44b6_8cc6506c,5706,21,48.67999999999999,76.64000000000001,179.99999999999997,cont,2.729080946490968,316000000005,1,0,0,1
44b6_8cc6506c,5727,21,56.36000000000001,78.56,216.0,cont,1.61211138728067,316000000004,1,1,0,0
44b6_8cc6506c,5747,21,59.800000000000004,74.55999999999999,219.83999999999997,cont,6.284402567070327,316000000004,2,0,2,0
44b6_8cc6506c,5920,22,48.84000000000001,76.0,177.28,cont,0.9990776996810528,317000000005,1,1,0,0
44b6_8cc6506c,5944,22,56.160000000000004,77.28,216.0,cont,1.9971640049079642,317000000004,1,0,1,0
44b6_8cc6506c,6139,23,48.839999999999996,74.56,175.36,cont,0.8800541531633219,318000000005,1,0,1,0
44b6_8cc6506c,6691,25,54.32000000000001,78.72000000000001,212.00000000000006,cont,2.4843374997169723,320000000004,1,1,0,0
44b6_8cc6506c,6916,26,53.480000000000004,79.36,209.92000000000002,cont,1.4972062524916212,321000000004,1,0,1,0
44b6_8cc6506c,7121,27,44.84,87.36000000000001,140.0,cont,3.040617741512416,322000000003,2,2,0,0
44b6_8cc6506c,7132,27,47.32,79.36000000000001,172.64,cont,0.6368673331236157,322000000005,1,1,0,0
44b6_8cc6506c,7343,28,38.15999999999999,103.36,129.28,cont,0.4177862042959178,323000000002,1,0,1,0
44b6_8cc6506c,7372,28,44.13333333333334,79.99999999999997,135.99999999999994,cont,0.21666666666668277,323000000003,4,0,0,4
44b6_8cc6506c,7380,28,44.52,90.72,141.44,cont,4.9562233605841435,323000000003,2,0,2,0
44b6_8cc6506c,7395,28,47.03999999999999,80.64,173.44000000000005,cont,0.3221436985259913,323000000005,2,1,1,0
44b6_8cc6506c,7403,28,49.866666666666674,87.46666666666665,162.9333333333333,cont,6.901150077764826,323000000005,7,2,0,5
44b6_8cc6506c,7630,29,46.16,82.72000000000001,172.64,cont,0.6368673331236157,324000000005,1,0,1,0
44b6_8cc6506c,7636,29,47.13333333333332,76.5333333333333,135.99999999999997,cont,6.9693158091179885,324000000003,2,0,2,0
44b6_8cc6506c,7645,29,49.24,88.96,162.24000000000007,cont,6.892145892245739,324000000005,2,0,2,0
44b6_8cc6506c,8142,31,48.411475219726555,91.42143798828127,162.3807373046875,cont,6.916927197608149,326000000005,1,0,1,0
44b6_8cc6506c,9354,36,45.36,87.36,178.72000000000003,cont,1.1605958868184942,331000000005,1,1,0,0
44b6_8cc6506c,9592,37,44.63999999999999,88.64,177.28,cont,1.0780255388904205,332000000005,2,1,1,0
44b6_8cc6506c,9799,38,36.480000000000004,110.08,136.0,cont,4.345409215482481,333000000002,5,0,0,5
44b6_8cc6506c,9839,38,44.00000000000001,91.36,175.36,cont,0.206828733497073,333000000005,2,1,1,0
44b6_8cc6506c,9849,38,46.68000000000001,88.00000000000001,199.36,cont,0.9756768483980606,333000000004,5,0,0,5
44b6_8cc6506c,9862,38,48.73333333333332,100.0,189.59999999999994,cont,6.246628604584331,333000000004,1,0,1,0
44b6_8cc6506c,11049,43,44.32,95.36000000000001,187.99999999999997,cont,0.5813776741499377,338000000004,2,2,0,0
44b6_8cc6506c,11285,44,44.00000000000001,96.64000000000001,185.28,cont,0.7587201394453624,339000000004,2,0,2,0
44b6_8cc6506c,11764,46,43.0,100.63999999999999,181.92,cont,2.086401282711457,341000000004,1,1,0,0
44b6_8cc6506c,11982,47,42.64000000000001,103.36,181.91999999999996,cont,1.325745073534121,342000000004,1,0,1,0
44b6_8cc6506c,12180,48,32.68,128.0,130.72000000000003,cont,1.6490347555464047,343000000002,1,1,0,0
44b6_8cc6506c,12192,48,35.64,109.28,111.36,cont,1.078025538890428,343000000003,1,1,0,0
44b6_8cc6506c,12433,49,34.733333333333334,123.99999999999999,138.93333333333328,cont,6.05583575581346,344000000002,5,0,0,5
44b6_8cc6506c,12442,49,36.04,112.64,110.08,cont,2.27650842849307,344000000003,1,0,1,0
44b6_8cc6506c,12455,49,39.52000000000001,107.35999999999999,160.00000000000006,cont,1.1559222508455904,344000000005,1,1,0,0
44b6_8cc6506c,12664,50,35.66519409179688,108.82069580078124,109.94583251953128,cont,1.364979640808899,345000000003,1,1,0,0
44b6_8cc6506c,12669,50,35.84,115.36000000000004,108.0,cont,3.6827677261809675,345000000003,5,1,0,4
44b6_8cc6506c,12883,51,34.93932800292969,110.18152832031248,109.76376708984375,cont,0.33381544411311226,346000000003,1,0,1,0
44b6_8cc6506c,12884,51,35.8,118.08000000000001,105.28000000000002,cont,3.8404013868995546,346000000003,1,0,1,0
44b6_8cc6506c,12900,51,38.47999999999999,110.71999999999998,160.00000000000006,cont,1.4582507714724307,346000000005,1,0,1,0
44b6_8cc6506c,14087,57,35.120000000000005,110.71999999999998,108.64000000000001,cont,2.1755588362533524,352000000003,1,0,1,0
44b6_8cc6506c,14098,57,37.86666666666666,108.5333333333333,112.0,cont,6.885135206047552,352000000003,1,0,1,0
44b6_8cc6506c,14304,58,35.480000000000004,121.28000000000003,160.64,cont,1.079004894567209,353000000005,1,0,1,0
44b6_8cc6506c,14722,60,32.99999999999999,131.46666666666664,115.46666666666667,cont,4.756807986688358,355000000002,6,0,0,6
44b6_8cc6506c,14935,61,33.839999999999996,131.36,159.99999999999997,cont,0.5040120286262956,356000000005,1,1,0,0
44b6_8cc6506c,14947,61,39.36,115.36000000000001,170.72000000000003,cont,0.6136392466262238,356000000004,1,1,0,0
44b6_8cc6506c,15127,62,32.68,134.72,112.64,cont,4.358939449567979,357000000002,1,0,1,0
44b6_8cc6506c,15322,63,32.16000000000001,140.00000000000003,158.55999999999997,cont,0.5332872701462082,358000000005,1,1,0,0
44b6_8cc6506c,15729,65,30.999999999999996,161.44,162.07999999999998,cont,5.926796273071981,360000000005,1,0,1,0
44b6_8cc6506c,15739,65,31.399999999999995,160.16,151.36,cont,6.428451529139805,360000000005,2,0,0,2
44b6_8cc6506c,15743,65,34.0,116.63999999999999,98.72000000000001,cont,0.9377249730064755,360000000003,1,1,0,0
44b6_8cc6506c,15953,66,33.160000000000004,116.0,95.99999999999997,cont,0.8530863086464362,361000000003,1,0,1,0
44b6_8cc6506c,16165,67,33.800000000000004,123.36000000000003,157.28000000000003,cont,6.884804940047308,362000000004,1,1,0,0
44b6_8cc6506c,16174,67,35.480000000000004,126.72000000000001,170.72000000000003,cont,0.9986811615826033,362000000004,1,1,0,0
44b6_8cc6506c,16755,70,33.160000000000004,132.64000000000001,168.64000000000004,cont,2.0291689339480947,365000000004,1,0,1,0
44b6_8cc6506c,17169,72,31.000000000000007,120.0,88.64,cont,0.6857808049953004,367000000003,1,1,0,0
44b6_8cc6506c,18047,76,31.840000000000003,148.00000000000003,149.92000000000004,cont,1.5721402609182327,371000000004,2,1,1,0
44b6_8cc6506c,18260,77,34.999999999999986,139.46666666666667,144.0,cont,6.278954113301374,372000000004,5,0,0,5
44b6_8cc6506c,18642,79,26.320000000000004,155.36,108.0,cont,0.8451562355564746,374000000002,1,0,1,0
44b6_8cc6506c,19097,81,24.680000000000003,160.64000000000001,108.0,cont,0.7587201394453624,376000000002,1,1,0,0
44b6_8cc6506c,19960,85,24.680000000000003,171.35999999999996,126.72000000000001,cont,1.6910153200074964,380000000004,1,1,0,0
44b6_8cc6506c,20183,86,23.36,185.92000000000002,152.0,cont,2.8714720201492487,381000000005,3,2,1,0
44b6_8cc6506c,20203,86,27.08,173.91999999999996,152.63999999999996,cont,6.948637568797209,381000000005,2,1,1,0
44b6_8cc6506c,20403,87,23.16,173.28000000000003,124.63999999999999,cont,0.46984704958101875,382000000004,1,0,1,0
44b6_8cc6506c,20404,87,22.999999999999996,188.64000000000001,152.0,cont,4.2594222759665445,382000000005,2,0,2,0
44b6_8cc6506c,20407,87,25.439999999999998,174.07999999999998,151.36,cont,5.938391248898979,382000000005,6,0,1,5
44b6_8cc6506c,21607,93,20.320000000000004,167.36000000000004,86.71999999999998,cont,0.8527767146211094,388000000002,2,0,1,1
44b6_8f5ab931,572,1,31.08,171.04000000000002,200.96000000000004,cont,5.993056398866939,127000000046,2,0,2,0
44b6_8f5ab931,1019,2,39.960000000000015,193.27999999999997,160.00000000000006,cont,1.073607382845362,128000000034,1,0,1,0
44b6_8f5ab931,1823,4,47.800000000000004,204.00000000000003,13.280000000000001,cont,1.7911156613965518,130000000015,2,1,1,0
44b6_8f5ab931,2597,6,50.48000000000002,217.28000000000003,145.92,cont,2.7711848774486545,132000000010,1,1,0,0
44b6_8f5ab931,2905,7,38.999999999999986,183.99999999999997,232.0,cont,6.712351604504915,133000000046,6,0,0,6
44b6_8f5ab931,2927,7,40.84,200.00000000000003,177.44000000000003,cont,1.9886840026761305,133000000035,1,1,0,0
44b6_8f5ab931,2980,7,50.160000000000004,216.64000000000001,152.16000000000003,cont,1.8224648487693738,133000000010,1,0,1,0
44b6_8f5ab931,3725,9,49.84,216.64000000000004,161.44000000000005,cont,2.0170956843937766,135000000010,1,1,0,0
44b6_8f5ab931,4083,10,46.733333333333334,219.99999999999997,155.99999999999997,cont,1.006465642870035,136000000010,9,0,0,9
44b6_8f5ab931,5166,13,51.36000000000001,216.0,168.64,cont,6.568739652703557,139000000010,5,1,0,4
44b6_8f5ab931,5194,13,58.8,200.64,151.36,cont,2.1778637755837784,139000000016,1,1,0,0
44b6_8f5ab931,5502,14,48.239999999999995,220.64000000000004,164.79999999999998,cont,1.6215024861220635,140000000010,6,0,0,6
44b6_8f5ab931,5586,15,0.9999999999999998,98.93333333333334,47.999999999999986,cont,5.239487656515453,141000000060,2,0,0,2
44b6_8f5ab931,5613,15,7.381472473144531,116.9764929199219,48.20024902343751,cont,6.458004962766554,141000000061,1,0,1,0
44b6_8f5ab931,5811,15,40.84,204.00000000000003,184.64000000000001,cont,0.7151846352516266,141000000035,1,1,0,0
44b6_8f5ab931,6174,16,44.46666666666666,180.53333333333333,11.466666666666665,cont,4.058164353087071,142000000068,4,0,0,4
44b6_8f5ab931,6500,17,43.2,179.04000000000005,13.280000000000001,cont,2.0554804791094465,143000000068,1,1,0,0
44b6_8f5ab931,7547,20,43.32000000000001,176.64000000000004,20.639999999999997,cont,1.100450191966894,146000000067,1,1,0,0
44b6_8f5ab931,7703,21,7.160000000000002,116.0,63.36,cont,0.8225132977648469,147000000059,1,1,0,0
44b6_8f5ab931,7876,21,38.52000000000001,200.64,192.0,cont,5.792959762720597,147000000035,1,1,0,0
44b6_8f5ab931,7995,21,58.16000000000001,208.64000000000001,73.92,cont,1.8909442912206442,147000000015,1,1,0,0
44b6_8f5ab931,8217,22,37.160000000000004,156.64,12.0,cont,0.29831034594865347,148000000068,1,0,1,0
44b6_8f5ab931,8224,22,37.86666666666666,168.53333333333333,0.0,cont,6.906674823535241,148000000068,5,0,0,5
44b6_8f5ab931,8270,22,44.64000000000001,177.92,25.92,cont,0.7319718659210829,148000000067,1,0,1,0
44b6_8f5ab931,8437,23,15.160000000000004,124.63999999999999,50.72,cont,0.38488837667562503,149000000060,1,1,0,0
44b6_8f5ab931,8988,24,46.36,179.99999999999997,33.279999999999994,cont,1.0462022091832976,150000000065,1,1,0,0
44b6_8f5ab931,9290,25,39.52000000000001,157.92000000000002,23.36,cont,0.9564060330215299,151000000066,1,0,1,0
44b6_8f5ab931,9340,25,47.16000000000001,180.64000000000001,37.92,cont,1.5192663731222193,151000000065,1,0,1,0
44b6_8f5ab931,9878,27,18.48,139.35999999999999,55.36,cont,6.51857969576195,153000000058,1,1,0,0
44b6_8f5ab931,10577,29,19.520000000000003,137.28000000000003,63.36,cont,6.820820148633165,155000000058,1,0,1,0
44b6_8f5ab931,10748,29,46.32,196.64,227.36000000000004,cont,1.3533430828877224,155000000035,1,0,1,0
44b6_8f5ab931,10785,29,52.32000000000001,227.03999999999994,209.91999999999996,cont,6.346431206788556,155000000010,1,0,1,0
44b6_8f5ab931,12475,34,55.16,223.36000000000004,223.35999999999999,cont,1.5728119722331864,160000000010,1,1,0,0
44b6_8f5ab931,12742,35,44.86666666666667,155.99999999999997,72.0,cont,0.9338842310182011,161000000065,6,0,0,6
44b6_8f5ab931,12771,35,49.64,176.00000000000003,74.72,cont,2.6883806696597,161000000064,1,0,1,0
44b6_8f5ab931,12809,35,55.32000000000001,223.99999999999997,226.72,cont,1.4418613187473925,161000000010,1,0,1,0
44b6_8f5ab931,13134,36,56.0,223.99999999999997,229.28000000000003,cont,0.42187453703679945,162000000010,1,1,0,0
44b6_8f5ab931,13230,37,13.999999999999998,124.64000000000001,12.0,cont,1.8867502650059444,163000000040,1,1,0,0
44b6_8f5ab931,13758,38,55.26666666666666,232.0,240.00000000000006,cont,5.1473298262572955,164000000010,1,0,0,1
44b6_8f5ab931,14462,41,2.12,106.08,63.36,cont,3.5352247541705184,167000000041,1,1,0,0
44b6_8f5ab931,14864,42,24.680000000000007,120.64000000000001,107.36000000000001,cont,1.0097253648888744,168000000057,1,1,0,0
44b6_8f5ab931,15405,43,61.73333333333333,225.59999999999994,241.0666666666666,cont,3.201451629296062,169000000010,4,0,0,4
44b6_8f5ab931,15697,44,58.88000000000001,233.92000000000002,245.28,cont,3.6552387490422373,170000000010,3,0,0,3
44b6_8f5ab931,18462,53,57.800000000000004,204.64,91.52000000000001,cont,6.483729636559487,179000000062,7,1,1,5
44b6_8f5ab931,18486,53,59.21050292968749,220.18449707031255,247.21049072265626,cont,3.044358044356581,179000000011,1,1,0,0
44b6_8f5ab931,18744,54,54.0,166.08,116.63999999999999,cont,1.6318916362614286,180000000063,1,0,1,0
44b6_8f5ab931,18905,55,28.04,124.00000000000003,136.0,cont,5.152582635193728,181000000054,4,0,0,4
44b6_8f5ab931,19424,57,4.840000000000001,110.71999999999998,81.28,cont,0.5924010993406439,183000000040,1,1,0,0
44b6_8f5ab931,20183,59,53.0,174.72,132.64000000000001,cont,2.0749792920653487,185000000061,4,0,0,4
44b6_8f5ab931,20382,60,33.20572448730469,143.28692138671877,161.5968115234375,cont,5.062533436808172,186000000053,3,2,1,0
44b6_8f5ab931,20392,60,35.16000000000001,137.28000000000003,152.0,cont,1.9554091643438933,186000000053,3,2,1,0
44b6_8f5ab931,20776,61,51.96000000000001,178.72,140.0,cont,6.783085581061157,187000000060,1,0,1,0
44b6_8f5ab931,20810,61,56.80918762207032,176.15204589843754,138.411298828125,cont,3.0540945895586455,187000000060,1,0,1,0
44b6_8f5ab931,20818,61,58.16000000000001,195.35999999999999,118.72,cont,0.5994906692351532,187000000059,1,0,1,0
44b6_8f5ab931,20950,62,25.000000000000004,128.64000000000001,154.72000000000003,cont,3.5127824619523373,188000000051,1,0,1,0
44b6_8f5ab931,22103,66,21.679999999999996,143.99999999999997,85.28,cont,6.582212892523294,192000000038,1,0,1,0
44b6_8f5ab931,22292,66,58.86666666666666,228.0,243.46666666666664,cont,3.6162082082694993,192000000011,4,0,0,4
44b6_8f5ab931,22457,67,36.64,136.64000000000001,150.56,cont,2.8191759655970396,193000000051,2,2,0,0
44b6_8f5ab931,22898,68,60.133333333333326,200.5333333333333,107.99999999999997,cont,1.6402754257204109,194000000058,2,0,0,2
44b6_8f5ab931,23028,69,26.36,134.72,91.36,cont,1.5818521580729337,195000000037,1,1,0,0
44b6_8f5ab931,24081,72,61.0,171.99999999999994,141.28000000000003,cont,6.650082941212091,198000000056,3,0,0,3
44b6_8f5ab931,24809,75,39.84000000000001,140.00000000000003,163.36,cont,0.7151846352516081,201000000047,1,0,1,0
44b6_8f5ab931,25105,76,40.160000000000004,139.35999999999999,164.0,cont,0.5479407472528443,202000000047,1,1,0,0
44b6_8f5ab931,25368,77,32.2,138.72000000000003,183.36,cont,6.3625546412192,203000000046,8,1,0,7
44b6_8f5ab931,25369,77,33.00000000000001,145.28,171.99999999999994,cont,2.0066611728191606,203000000046,1,1,0,0
44b6_8f5ab931,25403,77,40.839999999999996,139.35999999999999,164.64,cont,0.663668780341531,203000000047,1,0,1,0
44b6_8f5ab931,25647,78,33.36,140.64000000000001,186.07999999999998,cont,6.025244937967249,204000000046,2,1,1,0
44b6_8f5ab931,25648,78,33.67999999999999,146.55999999999997,176.00000000000003,cont,2.2601524423144705,204000000046,2,1,1,0
44b6_8f5ab931,25924,79,34.16,142.56,187.36,cont,5.150455380352311,205000000046,1,0,1,0
44b6_8f5ab931,26213,80,37.480000000000004,153.27999999999997,172.0,cont,6.587826960576611,206000000044,1,0,1,0
44b6_8f5ab931,27605,85,36.839999999999996,147.35999999999999,166.72,cont,5.985230389258206,211000000045,1,1,0,0
44b6_8f5ab931,27883,86,37.0,136.5333333333333,190.93333333333334,cont,4.729900824071853,212000000045,5,0,0,5
44b6_8f5ab931,27885,86,37.00000000000001,148.00000000000003,165.92,cont,6.731580887874707,212000000045,1,0,1,0
44b6_8f5ab931,28155,87,34.68,141.28000000000003,181.92,cont,3.8609398143457274,213000000045,1,1,0,0
44b6_8f5ab931,28442,88,36.0,143.46666666666664,182.93333333333325,cont,1.6396045106637438,214000000045,7,0,0,7
44b6_8f5ab931,29024,90,46.0,142.08,173.28000000000003,cont,0.4746001606615716,216000000045,1,1,0,0
44b6_8f5ab931,29549,92,36.839999999999996,145.27999999999997,194.07999999999998,cont,2.044789873067655,218000000044,2,2,0,0
44b6_8f5ab931,29812,93,34.480000000000004,140.64000000000001,135.36,cont,1.0361844068504404,219000000037,1,1,0,0
44b6_8f5ab931,29822,93,37.160000000000004,144.00000000000003,194.07999999999998,cont,1.9825665972420614,219000000044,3,1,2,0
44b6_8f5ab931,30084,94,34.68,140.0,136.64,cont,1.3005077133565963,220000000037,1,0,1,0
44b6_8f9ecab4,580,1,14.240000000000004,167.68,106.08000000000001,cont,3.9267882073012315,234000000018,1,1,0,0
44b6_8f9ecab4,1060,2,14.52,166.72,105.92000000000003,cont,4.232243790237053,235000000018,2,1,1,0
44b6_8f9ecab4,1572,3,15.999999999999996,166.93333333333328,93.06666666666666,cont,0.5757995648564759,236000000018,7,2,0,5
44b6_8f9ecab4,2147,4,22.000000000000004,138.56,111.99999999999999,cont,1.658853613945482,237000000017,2,2,0,0
44b6_8f9ecab4,2654,5,21.999999999999993,136.5333333333333,124.53333333333335,cont,5.9341554907913325,238000000017,7,0,0,7
44b6_8f9ecab4,2742,5,32.0,113.28000000000002,120.64000000000001,cont,1.2903154895218456,238000000008,1,0,1,0
44b6_8f9ecab4,3113,6,16.16,170.07999999999998,95.36000000000001,cont,0.4782030165107748,239000000018,2,1,1,0
44b6_8f9ecab4,3775,7,32.160000000000004,111.99999999999999,120.80000000000001,cont,0.9128971738372379,240000000008,1,0,1,0
44b6_8f9ecab4,4370,8,40.67999999999999,117.28,160.00000000000006,cont,0.8401422870562087,241000000009,1,1,0,0
44b6_8f9ecab4,4899,9,40.52,117.44,160.00000000000006,cont,1.0562499999999904,242000000009,1,0,1,0
44b6_8f9ecab4,5254,10,22.319999999999997,147.35999999999999,109.28,cont,0.5520218519225439,243000000017,1,1,0,0
44b6_8f9ecab4,5270,10,24.480000000000004,148.00000000000003,96.63999999999999,cont,6.451270465962512,243000000017,1,1,0,0
44b6_8f9ecab4,6319,12,24.320000000000007,148.00000000000003,96.63999999999999,cont,6.9577710601168965,245000000017,1,1,0,0
44b6_8f9ecab4,6373,12,30.639999999999993,113.28000000000003,117.28,cont,1.2866265240931314,245000000008,1,1,0,0
44b6_8f9ecab4,6775,13,15.266666666666666,168.53333333333333,95.46666666666667,cont,0.7821411584021553,246000000018,4,0,0,4
44b6_8f9ecab4,8104,15,44.839999999999996,104.0,165.44000000000005,cont,6.1579138614469136,248000000009,2,0,2,0
44b6_8f9ecab4,8447,16,25.0,145.28,97.92000000000002,cont,6.906785273374286,249000000017,1,1,0,0
44b6_8f9ecab4,11043,21,17.999999999999996,184.0,103.36,cont,6.752671809180418,254000000018,2,2,0,0
44b6_8f9ecab4,11061,21,20.48,145.28,101.28000000000002,cont,0.8601762174112928,254000000017,2,2,0,0
44b6_8f9ecab4,12664,24,21.0,145.92,100.0,cont,0.779999999999994,257000000017,1,0,1,0
44b6_8f9ecab4,14524,28,11.159999999999998,174.72,94.56,cont,1.4073850974413533,261000000018,1,0,1,0
44b6_8f9ecab4,14568,28,19.48,152.0,86.71999999999998,cont,5.358879459597876,261000000017,1,1,0,0
44b6_8f9ecab4,15018,29,19.680000000000003,152.0,87.36,cont,4.75170117563173,262000000017,1,0,1,0
44b6_8f9ecab4,15163,29,40.84,110.72000000000003,149.27999999999997,cont,0.8008793994728652,262000000009,2,2,0,0
44b6_8f9ecab4,15654,30,41.16,109.28,149.28000000000003,cont,1.400614262564829,263000000009,2,0,2,0
44b6_8f9ecab4,16408,32,8.0,186.55999999999997,103.36,cont,6.8168120527486975,265000000018,1,1,0,0
44b6_8f9ecab4,16433,32,10.360000000000001,171.99999999999994,99.36000000000001,cont,0.8866307362707369,265000000018,1,1,0,0
44b6_8f9ecab4,16677,32,41.519999999999996,107.36,151.35999999999999,cont,0.862322445492406,265000000009,1,0,1,0
44b6_8f9ecab4,16945,33,12.866666666666665,172.0,103.46666666666665,cont,4.764896688427658,266000000018,5,0,0,5
44b6_8f9ecab4,16990,33,19.04,156.0,91.35999999999997,cont,5.237871854341238,266000000017,1,0,1,0
44b6_8f9ecab4,17490,34,20.080000000000002,151.99999999999997,120.0,cont,6.955094917576612,267000000017,1,1,0,0
44b6_8f9ecab4,17666,34,41.48,105.28000000000002,152.0,cont,0.7882506343162704,267000000009,1,1,0,0
44b6_8f9ecab4,18080,35,30.320000000000007,120.00000000000003,120.00000000000003,cont,1.0467068894872031,268000000008,1,1,0,0
44b6_8f9ecab4,18186,35,43.266666666666666,116.0,143.46666666666664,cont,6.326147238951127,268000000009,6,0,0,6
44b6_8f9ecab4,18613,36,30.200000000000003,120.47999999999999,108.8,cont,4.736784411655235,269000000008,7,0,0,7
44b6_8f9ecab4,18986,37,9.84,172.64,101.28000000000003,cont,0.9279589497924978,270000000018,4,1,0,3
44b6_8f9ecab4,19518,38,9.84,173.28000000000003,101.92000000000002,cont,1.1356413386276394,271000000018,2,1,1,0
44b6_8f9ecab4,49373,38,11.777253990173339,169.70316345214846,94.26488586425782,cont,4.975861862318824,271000000018,2,1,1,0
44b6_8f9ecab4,21067,41,9.159999999999998,177.28,104.64000000000001,cont,1.5616917909754138,274000000018,2,2,0,0
44b6_8f9ecab4,21070,41,10.159999999999998,167.36,91.99999999999999,cont,6.6884494653095805,274000000018,1,1,0,0
44b6_8f9ecab4,21598,42,8.799999999999999,178.72,106.07999999999998,cont,0.4384489708050442,275000000018,2,0,2,0
44b6_8f9ecab4,22458,43,43.52,123.35999999999999,152.64,cont,6.298651765457426,276000000010,1,0,1,0
44b6_8f9ecab4,23927,46,21.36,164.0,101.92000000000002,cont,3.0509347952717656,279000000018,1,1,0,0
44b6_8f9ecab4,25005,48,21.320000000000004,164.0,106.07999999999998,cont,1.5289687088034285,281000000018,8,0,0,8
44b6_8f9ecab4,25016,48,21.999999999999993,163.46666666666664,99.46666666666667,cont,4.324267861120796,281000000018,5,0,0,5
44b6_8f9ecab4,25171,48,43.16000000000001,111.36,158.08,cont,0.4782030165107823,281000000010,1,0,1,0
44b6_8f9ecab4,27941,54,10.84,187.36,106.72,cont,0.6368673331236258,287000000020,1,0,1,0
44b6_8f9ecab4,27997,54,20.000000000000007,154.72000000000003,120.00000000000003,cont,6.563149491669376,287000000019,1,0,1,0
44b6_8f9ecab4,28085,54,32.0,128.0,128.64000000000001,cont,0.5524999999999949,287000000010,1,1,0,0
44b6_8f9ecab4,28199,54,45.86666666666665,111.99999999999999,163.99999999999994,cont,5.457745276846853,287000000011,4,0,0,4
44b6_8f9ecab4,29522,57,26.480000000000004,165.44000000000005,118.07999999999998,cont,6.894846467652496,290000000019,1,1,0,0
44b6_8f9ecab4,31080,60,32.160000000000004,125.28,125.28000000000002,cont,1.8901062304801775,293000000010,2,2,0,0
44b6_8f9ecab4,31567,61,32.99999999999999,123.46666666666665,137.06666666666666,cont,3.92941740981478,294000000010,7,0,0,7
44b6_8f9ecab4,31651,61,43.68,106.72,158.71999999999997,cont,1.0683061944030914,294000000011,1,1,0,0
44b6_8f9ecab4,31652,61,43.84,117.92000000000002,148.00000000000006,cont,6.90213848202134,294000000011,2,2,0,0
44b6_8f9ecab4,32131,62,43.84,105.28000000000002,157.28,cont,0.48858213229711095,295000000011,1,0,1,0
44b6_8f9ecab4,33015,64,33.480000000000004,118.72,136.0,cont,6.663550222103835,297000000010,2,1,1,0
44b6_8f9ecab4,33030,64,33.999999999999986,120.53333333333336,111.46666666666667,cont,6.765035017257636,297000000010,8,0,0,8
44b6_8f9ecab4,33517,65,32.52,121.44000000000001,121.28000000000003,cont,1.8149147707261488,298000000010,1,0,1,0
44b6_8f9ecab4,33528,65,34.16,116.64000000000001,134.56000000000003,cont,5.031153259939519,298000000010,2,1,1,0
44b6_8f9ecab4,33971,66,23.16,143.99999999999997,106.72,cont,0.49555839716426076,299000000019,1,1,0,0
44b6_8f9ecab4,34035,66,32.12,118.56000000000003,120.63999999999999,cont,0.9945743436264631,299000000010,5,0,0,5
44b6_8f9ecab4,34564,67,34.32,124.00000000000003,116.63999999999999,cont,4.6922842251935375,300000000010,1,0,1,0
44b6_8f9ecab4,36575,71,47.0,103.36,156.64,cont,1.8411328883326177,304000000011,1,0,1,0
44b6_8f9ecab4,37290,73,25.159999999999997,160.64000000000004,97.28000000000003,cont,6.715517710869363,306000000019,4,0,0,4
44b6_8f9ecab4,37645,74,13.999999999999998,187.36,104.64000000000001,cont,1.1675147215345845,307000000020,1,0,1,0
44b6_8f9ecab4,37871,74,47.00000000000001,120.00000000000003,149.44,cont,6.846651667968812,307000000011,1,1,0,0
44b6_8f9ecab4,38292,75,47.00000000000001,120.00000000000003,149.27999999999994,cont,6.353604537976869,308000000011,1,0,1,0
44b6_8f9ecab4,38907,77,14.16,189.28000000000003,105.28000000000002,cont,1.021813644947053,310000000020,1,1,0,0
44b6_8f9ecab4,39379,78,23.159999999999997,161.27999999999997,97.44,cont,6.624760490953613,311000000019,1,0,1,0
44b6_8f9ecab4,39832,79,25.680000000000007,153.28000000000003,111.99999999999999,cont,1.376270435815583,312000000019,1,1,0,0
44b6_8f9ecab4,39984,79,47.00000000000001,120.00000000000003,153.27999999999997,cont,6.846709519908094,312000000011,1,0,1,0
44b6_8f9ecab4,39990,79,49.28,107.36000000000001,164.64,cont,1.1936838986934415,312000000011,1,1,0,0
44b6_8f9ecab4,40231,80,20.999999999999993,155.99999999999997,108.53333333333333,cont,6.640057900621896,313000000019,4,0,0,4
44b6_8f9ecab4,40425,80,49.16,108.0,165.28000000000003,cont,0.2837940494443067,313000000011,1,0,1,0
44b6_8f9ecab4,40759,81,35.32000000000001,116.63999999999999,122.71999999999998,cont,1.4492595091977094,314000000010,1,1,0,0
44b6_8f9ecab4,41194,82,35.68,116.0,123.35999999999999,cont,0.9377249730064834,315000000010,2,1,1,0
44b6_8f9ecab4,41653,83,35.73333333333333,127.46666666666667,119.46666666666667,cont,5.031162007512115,316000000010,6,0,0,6
44b6_8f9ecab4,49622,84,28.876782150268557,151.75678771972656,109.8625115966797,cont,6.44723290800566,317000000019,1,1,0,0
44b6_8f9ecab4,42454,85,29.10523651123046,151.53635009765625,110.97889404296875,cont,6.723987443438096,318000000019,2,1,1,0
44b6_90724892,8313,30,29.480000000000004,247.35999999999999,156.64000000000004,cont,1.0859572390292296,104000000023,2,2,0,0
44b6_90724892,8590,31,29.519999999999996,243.36000000000004,155.36,cont,0.8069560861657976,105000000023,2,0,2,0
44b6_90724892,9146,33,30.0,231.36,153.27999999999997,cont,0.6758906161502654,107000000023,1,1,0,0
44b6_90724892,9432,34,30.199999999999992,224.64000000000001,152.63999999999996,cont,1.5167440456451582,108000000023,2,1,1,0
44b6_90724892,9458,34,34.760000000000005,247.68000000000004,126.72,cont,0.6727578501957264,108000000024,2,1,1,0
44b6_90724892,9696,35,30.320000000000007,217.28000000000003,152.0,cont,0.7218000502216695,109000000023,1,0,1,0
44b6_90724892,9724,35,34.32000000000001,242.71999999999997,124.00000000000003,cont,1.1108393504463099,109000000024,3,2,1,0
44b6_90724892,9725,35,33.86666666666666,252.53333333333336,134.93333333333334,cont,6.174109036600626,109000000024,7,1,0,6
44b6_90724892,10005,36,34.16,237.27999999999997,121.92000000000002,cont,1.4610548415442883,110000000024,3,1,2,0
44b6_90724892,10287,37,32.84,222.07999999999996,156.64000000000004,cont,6.975510061995461,111000000023,1,0,1,0
44b6_90724892,10295,37,33.84000000000001,232.0,120.0,cont,1.9282800788526389,111000000024,1,0,1,0
44b6_90724892,10313,37,37.84,230.72000000000003,129.91999999999996,cont,6.19653645696529,111000000024,1,0,1,0
44b6_90724892,10562,38,32.16000000000001,215.35999999999996,155.36,cont,6.155576359082866,112000000023,2,2,0,0
44b6_90724892,10574,38,33.68000000000001,226.07999999999998,118.08000000000001,cont,0.6412061388508299,112000000024,1,1,0,0
44b6_90724892,10842,39,32.00000000000001,208.0,154.08,cont,6.317760926546686,113000000023,2,0,2,0
44b6_90724892,10850,39,33.31999999999999,220.00000000000003,115.35999999999999,cont,1.3793369104392321,113000000024,1,0,1,0
44b6_90724892,10870,39,37.00000000000001,212.0,124.64000000000001,cont,6.711486073329824,113000000024,2,0,2,0
44b6_90724892,15553,56,35.28,125.44000000000001,82.72000000000001,cont,3.720824211448311,130000000027,1,1,0,0
44b6_90724892,15782,57,29.840000000000003,93.91999999999999,109.92,cont,0.5890484912127284,131000000026,1,1,0,0
44b6_90724892,15815,57,35.32000000000001,118.72,79.36,cont,5.665496723589213,131000000027,1,0,1,0
44b6_90724892,16028,58,29.68,89.92,108.0,cont,1.376750022698385,132000000026,1,0,1,0
44b6_90724892,16060,58,35.0,112.64,76.63999999999999,cont,5.0089026991947065,132000000027,1,1,0,0
44b6_90724892,16302,59,30.400000000000002,111.52000000000001,78.39999999999999,cont,4.866027159038462,133000000027,6,1,0,5
44b6_90724892,16327,59,35.35999999999999,105.28000000000002,73.92,cont,5.234139882779202,133000000027,1,0,1,0
44b6_90724892,16544,60,30.520000000000007,107.36000000000001,76.0,cont,2.708685393784212,134000000027,1,0,1,0
44b6_90724892,17780,65,23.680000000000007,62.72000000000001,92.0,cont,6.969962002227837,139000000026,1,1,0,0
44b6_90724892,17794,65,24.999999999999993,71.36,100.32000000000001,cont,1.9665185766984374,139000000026,1,1,0,0
44b6_90724892,18017,66,23.6,59.2,89.75999999999999,cont,5.677299781806489,140000000026,2,1,1,0
44b6_90724892,18032,66,23.999999999999993,68.53333333333333,100.0,cont,1.8266699456327147,140000000026,1,0,1,0
44b6_90724892,18306,67,24.040000000000003,55.36,88.80000000000001,cont,4.006770247031888,141000000026,5,0,1,4
44b6_90724892,18574,68,26.639999999999997,61.27999999999999,100.64000000000001,cont,4.3024464334259855,142000000026,1,1,0,0
44b6_90724892,18783,69,23.64,45.28,83.20000000000002,cont,5.00647705477614,143000000026,6,0,0,6
44b6_90724892,18809,69,26.160000000000007,54.720000000000006,96.64000000000001,cont,3.3038292177411464,143000000026,1,0,1,0
44b6_90724892,19700,73,26.52,30.72,88.00000000000001,cont,5.2919391826153905,147000000026,2,2,0,0
44b6_90724892,19707,73,28.48,37.28,41.280000000000015,cont,0.8407706658179731,147000000027,1,1,0,0
44b6_90724892,19917,74,26.68,26.720000000000002,86.72,cont,6.021211594230845,148000000026,2,0,2,0
44b6_90724892,19919,74,28.48,33.279999999999994,37.92,cont,1.503190876934796,148000000027,1,0,1,0
44b6_90724892,20124,75,25.32,14.560000000000002,72.64,cont,1.4375510860487695,149000000026,1,1,0,0
44b6_90724892,20342,76,25.680000000000003,12.000000000000004,71.36,cont,0.8606365739962458,150000000026,1,0,1,0
44b6_90724892,21075,79,25.84,7.360000000000001,30.720000000000006,cont,5.3738383919224795,153000000027,1,0,1,0
44b6_90724892,21556,81,26.32,8.0,28.000000000000004,cont,2.0473710276840387,155000000027,1,1,0,0
44b6_90724892,21786,82,24.866666666666667,4.5333333333333306,31.46666666666666,cont,4.644771926585848,156000000027,2,0,0,2
44b6_90724892,21808,82,26.84,7.359999999999999,25.92,cont,0.9905837925183313,156000000027,2,0,1,1
44b6_949adeb1,9515,26,45.760000000000005,150.24000000000004,24.799999999999997,cont,5.547941341164294,91000000056,1,1,0,0
44b6_949adeb1,10260,28,48.519999999999996,154.71999999999997,20.640000000000008,cont,4.5567851194674525,93000000055,1,1,0,0
44b6_949adeb1,14406,39,49.160000000000004,174.72,29.280000000000005,cont,0.6508119928212738,104000000053,1,1,0,0
44b6_949adeb1,14799,40,48.480000000000004,173.27999999999997,33.279999999999994,cont,0.882900050968406,105000000053,1,0,1,0
44b6_949adeb1,17552,47,32.12,128.00000000000003,115.36000000000001,cont,3.131025441049635,112000000055,1,1,0,0
44b6_949adeb1,19112,51,14.840000000000003,16.640000000000004,11.36,cont,1.4798210407005346,116000000014,2,0,2,0
44b6_949adeb1,19939,53,7.579139938354492,40.78682983398438,30.224537963867192,cont,3.146920980378721,118000000009,1,0,1,0
44b6_949adeb1,19998,53,16.52,10.720000000000002,8.640000000000002,cont,6.900206178984797,118000000014,1,0,1,0
44b6_949adeb1,22133,58,3.0,24.64,83.36000000000001,cont,1.8924100044652066,123000000033,1,0,1,0
44b6_949adeb1,22677,59,14.533333333333331,51.99999999999999,73.06666666666666,cont,1.5710667929962554,124000000009,5,0,0,5
44b6_949adeb1,23753,61,27.0,121.91999999999999,167.36000000000004,cont,1.9958413795940686,126000000056,1,1,0,0
44b6_949adeb1,46164,61,14.137173423767093,37.725571136474606,78.84847381591797,cont,6.316149517268663,126000000009,1,1,0,0
44b6_949adeb1,24260,62,27.040000000000003,123.35999999999999,170.72000000000003,cont,1.70012591445457,127000000056,1,0,1,0
44b6_949adeb1,25136,64,13.200000000000001,48.64,82.72000000000001,cont,1.1380802091680509,129000000010,1,1,0,0
44b6_949adeb1,25628,65,13.680000000000003,48.0,86.72000000000003,cont,1.4668269325315777,130000000010,2,1,1,0
44b6_949adeb1,26116,66,14.320000000000002,48.0,91.36,cont,2.2150126974805393,131000000010,2,1,1,0
44b6_949adeb1,26188,66,22.48,31.200000000000006,81.92,cont,2.4715496783394815,131000000016,2,1,1,0
44b6_949adeb1,26247,66,26.36,125.28000000000002,182.72,cont,1.4323063787123218,131000000056,3,1,2,0
44b6_949adeb1,26633,67,14.52,48.64,94.56,cont,1.6813842883172172,132000000010,1,0,1,0
44b6_949adeb1,26652,67,17.266666666666666,43.999999999999986,83.46666666666664,cont,6.455271451603638,132000000010,6,0,0,6
44b6_949adeb1,27080,68,6.5200000000000005,24.159999999999997,128.64000000000001,cont,2.5847718105279656,133000000035,1,1,0,0
44b6_949adeb1,27590,69,5.159999999999999,28.960000000000004,119.51999999999998,cont,4.570296249971117,134000000035,5,0,0,5
44b6_949adeb1,27612,69,7.000000000000001,49.44000000000001,128.64000000000001,cont,6.481224445465537,134000000035,2,0,2,0
44b6_949adeb1,28654,71,3.6799999999999997,1.9200000000000004,209.28000000000003,cont,5.290042680593419,136000000046,4,3,0,1
44b6_949adeb1,28669,71,5.160000000000001,33.28,125.92,cont,1.59911889801853,136000000035,2,1,1,0
44b6_949adeb1,28702,71,8.68,52.64,100.63999999999999,cont,6.002852056522802,136000000010,1,0,1,0
44b6_949adeb1,29166,72,1.9385070800781254,16.143243713378908,212.0416943359375,cont,0.9732314430848141,137000000046,4,1,1,2
44b6_949adeb1,29203,72,5.12,17.919999999999998,210.71999999999997,cont,5.315439333912109,137000000046,5,1,0,4
44b6_949adeb1,29826,73,15.160000000000004,48.0,112.64,cont,3.8303838743656007,138000000010,1,1,0,0
44b6_949adeb1,30278,74,5.0,19.36,219.36,cont,6.018250628297235,139000000046,2,2,0,0
44b6_949adeb1,30343,74,12.6,51.99999999999999,111.46666666666667,cont,0.7896080690936376,139000000010,7,0,0,7
44b6_949adeb1,30806,75,3.6800000000000006,6.080000000000001,205.76000000000002,cont,5.572328535944377,140000000046,4,0,0,4
44b6_949adeb1,30819,75,5.160000000000002,34.720000000000006,129.92000000000002,cont,1.6094064317318906,140000000035,1,1,0,0
44b6_949adeb1,31353,76,4.680000000000001,8.0,209.28,cont,6.4953874258969355,141000000046,1,1,0,0
44b6_949adeb1,32395,78,0.4800000000000002,9.920000000000002,215.36,cont,1.4272274258155235,143000000046,2,0,2,0
44b6_949adeb1,33180,79,20.520000000000003,132.00000000000006,187.35999999999999,cont,0.9558536760404205,144000000055,1,1,0,0
44b6_949adeb1,33200,79,21.88,35.99999999999999,12.799999999999999,cont,1.8841592952826471,144000000015,1,1,0,0
44b6_949adeb1,33487,79,52.84,139.35999999999999,9.280000000000001,cont,3.5652366681610252,144000000025,1,0,1,0
44b6_949adeb1,33722,80,17.866666666666664,122.93333333333337,183.99999999999997,cont,6.643426288235779,145000000055,5,1,0,4
44b6_949adeb1,33749,80,19.866666666666667,135.99999999999994,200.0,cont,5.7243942243796475,145000000055,4,0,0,4
44b6_949adeb1,33761,80,22.16,36.63999999999999,15.360000000000003,cont,0.45033320996790516,145000000015,1,0,1,0
44b6_949adeb1,34405,81,27.480000000000008,31.359999999999996,120.80000000000001,cont,1.1211322401929085,146000000016,1,1,0,0
44b6_949adeb1,34698,82,0.32000000000000006,7.360000000000001,211.20000000000002,cont,1.0662031056510857,147000000046,1,1,0,0
44b6_949adeb1,35009,82,28.000000000000004,31.360000000000003,123.36000000000003,cont,2.2228669550829965,147000000016,1,0,1,0
44b6_949adeb1,35290,83,0.8,8.0,212.64,cont,2.129990096807958,148000000046,1,0,1,0
44b6_949adeb1,35475,83,17.27404266357422,125.23394165039066,186.73395996093748,cont,4.924129010212048,148000000055,1,0,1,0
44b6_949adeb1,35476,83,17.48,131.35999999999999,184.0,cont,2.8127055480444456,148000000055,1,0,1,0
44b6_949adeb1,35566,83,24.679999999999996,40.64000000000001,26.079999999999995,cont,0.8764461264105164,148000000015,1,1,0,0
44b6_949adeb1,36170,84,26.000000000000004,42.720000000000006,30.720000000000006,cont,1.6550287950667133,149000000015,4,3,1,0
44b6_949adeb1,36629,85,12.680000000000003,29.28,148.64000000000004,cont,4.1534249797606675,150000000035,1,1,0,0
44b6_949adeb1,36793,85,27.64,44.00000000000001,35.2,cont,0.7614993434008999,150000000015,3,0,3,0
44b6_949adeb1,37245,86,10.999999999999996,27.999999999999996,163.46666666666664,cont,1.4083333333333172,151000000035,6,1,0,5
44b6_949adeb1,37277,86,13.839999999999998,30.72,149.92000000000002,cont,6.268035976284749,151000000035,1,0,1,0
44b6_949adeb1,37387,86,23.639999999999997,136.64,180.64000000000001,cont,4.403510141069276,151000000053,1,1,0,0
44b6_949adeb1,37473,86,30.866666666666664,31.999999999999993,44.533333333333324,cont,6.491982903080452,151000000015,2,0,0,2
44b6_949adeb1,37707,86,55.480000000000004,140.00000000000003,28.000000000000004,cont,1.0218136449470658,151000000025,1,1,0,0
44b6_949adeb1,38018,87,23.159999999999997,138.72000000000003,179.36,cont,3.551322156957874,152000000053,1,0,1,0
44b6_949adeb1,38353,87,55.68,140.00000000000003,29.920000000000005,cont,1.5778406922119976,152000000025,2,1,1,0
44b6_949adeb1,38441,88,2.6800000000000006,12.640000000000004,222.07999999999998,cont,1.0256826994738664,153000000046,1,1,0,0
44b6_949adeb1,38650,88,23.480000000000004,47.360000000000014,142.08,cont,5.694204976333403,153000000010,1,0,1,0
44b6_949adeb1,38972,88,55.68000000000001,139.35999999999999,32.0,cont,1.1638902976655365,153000000025,2,1,1,0
44b6_949adeb1,39063,89,2.4799999999999995,13.280000000000001,221.92000000000002,cont,1.5823528762573793,154000000046,1,0,1,0
44b6_949adeb1,39604,89,55.64,138.72000000000003,34.720000000000006,cont,0.6638676920140026,154000000025,1,0,1,0
44b6_949adeb1,39796,90,17.32,56.64,157.92000000000002,cont,6.27487811136599,155000000010,1,1,0,0
44b6_949adeb1,39821,90,18.86666666666666,137.33333333333331,177.5999999999999,cont,2.741978515699151,155000000053,4,0,0,4
44b6_949adeb1,39866,90,22.199999999999996,141.43999999999997,177.92,cont,3.791860114112856,155000000053,1,0,1,0
44b6_949adeb1,39989,90,34.12000000000001,168.63999999999996,145.92,cont,6.728147613756721,155000000052,3,2,1,0
44b6_949adeb1,40404,91,17.68,55.99999999999999,160.64000000000001,cont,6.212942640166574,156000000010,1,0,1,0
44b6_949adeb1,40443,91,21.64,140.64000000000001,180.00000000000006,cont,1.6231301742312387,156000000053,6,0,0,6
44b6_949adeb1,41223,92,34.52000000000001,159.36,157.44000000000003,cont,2.828900515217875,157000000052,2,1,1,0
44b6_949adeb1,41303,92,42.519999999999996,112.64,16.64,cont,6.300747599491667,157000000024,2,2,0,0
44b6_949adeb1,41344,92,48.480000000000004,104.0,12.0,cont,4.998796886501792,157000000024,1,1,0,0
44b6_949adeb1,41577,93,14.84,8.64,9.280000000000001,cont,0.7241740985426096,158000000065,1,0,1,0
44b6_949adeb1,41631,93,20.8,50.72,164.64,cont,1.988883166503246,158000000010,3,0,0,3
44b6_949adeb1,41795,93,34.16,46.56000000000001,72.64,cont,1.2725939307178868,158000000015,2,0,2,0
44b6_949adeb1,41820,93,35.13333333333335,159.46666666666664,168.53333333333333,cont,0.6862298873798508,158000000052,7,1,0,6
44b6_949adeb1,41955,93,49.00000000000001,104.64,16.0,cont,2.7170077751085024,158000000024,2,1,1,0
44b6_949adeb1,42299,94,21.160000000000004,138.72000000000003,188.64,cont,0.7241740985426005,159000000053,2,0,2,0
44b6_949adeb1,42619,94,49.52,105.28000000000002,20.000000000000007,cont,3.703895120342904,159000000024,2,1,1,0
44b6_949adeb1,43262,95,50.0,106.72,24.000000000000007,cont,4.412974655773588,160000000024,1,0,1,0
44b6_95029e92,245,4,53.16,17.919999999999998,189.28,cont,0.5822853681829823,74000000055,1,1,0,0
44b6_95029e92,295,5,53.16,20.0,190.07999999999998,cont,0.26202337681969373,75000000055,1,0,1,0
44b6_97725144,393,1,55.28000000000001,27.840000000000003,201.11999999999992,cont,2.017161139448221,90000000053,7,1,3,3
44b6_97725144,617,2,55.199999999999996,33.92000000000001,201.43999999999997,cont,3.549053571024266,91000000053,2,1,1,0
44b6_97725144,862,3,54.84,38.720000000000006,200.00000000000003,cont,2.839335859844679,92000000053,1,0,1,0
44b6_97725144,2642,10,29.133333333333333,182.39999999999998,136.53333333333336,cont,1.534217645812416,99000000063,1,0,0,1
44b6_97725144,3345,12,48.68,73.27999999999999,182.56000000000003,cont,1.2369228452090277,101000000052,1,1,0,0
44b6_97725144,3629,13,47.480000000000004,76.16,181.28,cont,2.2023996146249214,102000000052,1,0,1,0
44b6_97725144,6078,21,45.320000000000014,10.719999999999999,229.44,cont,1.4865863790913865,110000000064,1,1,0,0
44b6_97725144,6402,22,44.16000000000001,12.64,227.36000000000004,cont,0.6278895703067825,111000000064,1,0,1,0
44b6_97725144,6632,23,22.36,193.27999999999997,128.48,cont,2.099271078850935,112000000062,1,1,0,0
44b6_97725144,6980,24,21.480000000000004,195.36,133.44,cont,2.015327594337956,113000000062,1,0,1,0
44b6_97725144,8209,27,40.000000000000014,16.0,215.99999999999997,cont,1.4647552056572377,116000000064,1,1,0,0
44b6_97725144,8571,28,37.86666666666666,11.999999999999996,223.99999999999997,cont,6.365966161833647,117000000064,1,0,0,1
44b6_97725144,8576,28,39.0,18.080000000000002,213.27999999999997,cont,1.2952177664392885,117000000064,1,0,1,0
44b6_97725144,10025,32,33.279999999999994,23.360000000000007,207.36,cont,0.7614993434008994,121000000065,1,1,0,0
44b6_97725144,10427,33,32.84,25.28,206.55999999999997,cont,0.8600227104559418,122000000065,1,0,1,0
44b6_97725144,13006,39,30.84,40.8,202.72000000000003,cont,2.106554132345049,128000000065,1,1,0,0
44b6_97725144,13430,40,29.519999999999996,40.64000000000001,199.36,cont,2.4138771975806974,129000000065,1,0,1,0
44b6_97725144,14740,43,24.96,41.28,189.92000000000002,cont,0.4790305966219703,132000000064,1,1,0,0
44b6_97725144,15236,44,24.520000000000003,43.36,188.64000000000001,cont,2.4138771975806823,133000000064,2,1,1,0
44b6_97725144,15690,45,23.680000000000007,44.64000000000001,187.35999999999999,cont,0.5994906692351332,134000000064,1,0,1,0
44b6_97725144,15821,45,38.040000000000006,128.64000000000001,142.72,cont,1.8345954935625328,134000000053,1,1,0,0
44b6_97725144,16269,46,36.68,130.08,140.79999999999998,cont,2.452512104965025,135000000053,1,0,1,0
44b6_97725144,17527,49,22.16,53.28000000000001,184.16,cont,1.5835206029603806,138000000064,1,1,0,0
44b6_97725144,18000,50,22.320000000000004,56.8,180.0,cont,1.7538711647381615,139000000063,1,0,1,0
44b6_97725144,22238,59,33.480000000000004,165.92000000000002,93.27999999999999,cont,1.328331895649569,148000000053,1,1,0,0
44b6_97725144,22737,60,33.68000000000001,167.36,94.07999999999998,cont,0.6568699452707494,149000000053,1,0,1,0
44b6_97725144,23500,62,17.160000000000004,70.56,160.00000000000003,cont,0.7964157990647788,151000000060,1,1,0,0
44b6_97725144,23985,63,17.52,74.08000000000001,153.92000000000002,cont,2.988056433536693,152000000060,1,0,1,0
44b6_97725144,31991,81,16.16,164.00000000000003,53.44,cont,6.891935164560094,170000000046,1,1,0,0
44b6_97725144,32012,81,18.8,167.20000000000002,42.720000000000006,cont,0.9118842100288919,170000000046,1,1,0,0
44b6_97725144,32512,82,18.520000000000003,168.0,40.00000000000001,cont,0.8794538433027557,171000000046,1,0,1,0
44b6_996155de,1568,17,34.64,69.28,8.0,cont,2.456116408580832,436000000001,1,1,0,0
44b6_996155de,1659,18,34.48,68.64000000000001,8.0,cont,2.8091827659481368,437000000001,1,0,1,0
44b6_996155de,1753,19,34.32,68.0,8.640000000000002,cont,1.300507713356592,438000000001,1,1,0,0
44b6_996155de,1849,20,34.519999999999996,68.0,9.920000000000002,cont,2.5491273016465925,439000000001,3,2,1,0
44b6_996155de,1954,21,34.68,68.0,12.0,cont,2.2937264985172083,440000000001,2,0,2,0
44b6_996155de,1960,21,38.99999999999999,72.0,4.533333333333333,cont,6.41846430447172,440000000001,3,0,0,3
44b6_996155de,6829,69,29.159999999999997,68.0,80.64,cont,0.5040120286262976,488000000000,1,1,0,0
44b6_996155de,6837,69,33.68,60.000000000000014,60.000000000000014,cont,1.0467068894872078,488000000001,1,1,0,0
44b6_996155de,6950,70,29.159999999999997,68.0,81.28,cont,0.28379404944430386,489000000000,1,0,1,0
44b6_996155de,6958,70,34.0,60.000000000000014,60.640000000000015,cont,1.2567250345640366,489000000001,1,0,1,0
44b6_996155de,7621,76,34.84,60.000000000000014,63.36000000000001,cont,1.8906649789161394,495000000001,1,1,0,0
44b6_996155de,7740,77,34.160000000000004,60.000000000000014,64.0,cont,1.6396402997303987,496000000001,1,0,1,0
44b6_996155de,8345,82,32.99999999999999,128.53333333333333,144.53333333333333,cont,2.7583813251112073,501000000002,1,0,0,1
44b6_996155de,9100,88,33.000000000000014,140.64,155.35999999999999,cont,1.638109619347842,507000000002,1,1,0,0
44b6_996155de,9225,89,33.000000000000014,143.36,156.64,cont,1.7623572637237663,508000000002,1,0,1,0
44b6_996155de,9767,93,32.68,62.71999999999999,70.72,cont,0.6073675267085007,512000000001,1,1,0,0
44b6_996155de,9900,94,32.32,63.36,71.35999999999999,cont,0.5994906692351455,513000000001,1,0,1,0
44b6_9be80b04,584,1,35.92000000000001,157.28,153.27999999999997,cont,1.6660807303369265,400000000008,1,0,1,0
44b6_9be80b04,1256,3,31.159999999999997,152.0,130.08,cont,1.3653868499440096,402000000007,1,1,0,0
44b6_9be80b04,1626,4,31.000000000000007,151.36,129.44000000000003,cont,1.733274087529138,403000000007,1,0,1,0
44b6_9be80b04,1633,4,32.13333333333334,159.99999999999994,119.46666666666667,cont,5.043102842965037,403000000007,5,0,0,5
44b6_9be80b04,2024,5,35.00000000000001,154.71999999999997,151.36,cont,0.3913518238107359,404000000008,1,1,0,0
44b6_9be80b04,2395,6,34.839999999999996,153.27999999999997,150.56,cont,0.3637242258909898,405000000008,1,0,1,0
44b6_9be80b04,3765,10,29.519999999999996,146.08000000000007,126.72000000000001,cont,4.2586782720698695,409000000007,1,1,0,0
44b6_9be80b04,4099,11,29.519999999999996,145.28,125.92,cont,2.6231043750678387,410000000007,1,0,1,0
44b6_9be80b04,4130,11,33.84000000000001,134.08,147.36000000000004,cont,6.679145365426627,410000000008,1,0,1,0
44b6_9be80b04,5959,16,34.839999999999996,132.64000000000001,151.35999999999996,cont,6.7904392807093,415000000008,1,0,1,0
44b6_9be80b04,7230,20,33.160000000000004,142.72,143.99999999999997,cont,0.8606365739962369,419000000008,1,1,0,0
44b6_9be80b04,7552,21,33.32,141.92000000000002,143.99999999999997,cont,0.521014635111148,420000000008,1,0,1,0
44b6_9be80b04,10014,29,32.000000000000014,139.35999999999999,124.00000000000003,cont,1.9351150644083295,428000000007,1,1,0,0
44b6_9be80b04,10331,30,32.00000000000001,138.72000000000003,123.36,cont,1.7599582736531116,429000000007,2,1,1,0
44b6_9be80b04,10335,30,33.00000000000001,138.08,140.00000000000003,cont,0.03250000000000597,429000000008,1,1,0,0
44b6_9be80b04,10645,31,30.133333333333326,139.99999999999994,131.46666666666664,cont,3.089518711394302,430000000007,6,0,0,6
44b6_9be80b04,10671,31,33.00000000000001,136.0,139.35999999999999,cont,0.7803384682302974,430000000008,1,0,1,0
44b6_9be80b04,12083,35,31.0,136.64000000000001,121.92000000000002,cont,0.6670422119326486,434000000007,1,1,0,0
44b6_9be80b04,12425,36,30.520000000000007,137.28,124.47999999999999,cont,1.3470848154440784,435000000007,1,0,1,0
44b6_9be80b04,13461,39,29.840000000000003,137.28000000000003,132.64000000000001,cont,4.251044724535375,438000000008,1,1,0,0
44b6_9be80b04,13468,39,31.48,125.92,133.28,cont,1.1865838571293637,438000000008,1,1,0,0
44b6_9be80b04,13469,39,31.160000000000004,132.00000000000006,106.72,cont,6.8891184577564095,438000000007,1,1,0,0
44b6_9be80b04,13817,40,30.160000000000004,136.64,132.64000000000001,cont,4.045988947402105,439000000008,1,0,1,0
44b6_9be80b04,13834,40,31.680000000000003,125.28,134.72,cont,0.6073675267084953,439000000009,1,0,1,0
44b6_9be80b04,13835,40,31.360000000000003,132.00000000000006,107.36000000000001,cont,6.797415984217817,439000000008,1,0,1,0
44b6_9be80b04,14935,43,32.32,125.28,137.28,cont,0.7914267180225878,442000000009,1,1,0,0
44b6_9be80b04,14937,43,31.360000000000003,136.0,130.56,cont,1.758982749346908,442000000008,1,1,0,0
44b6_9be80b04,15282,44,31.159999999999997,136.0,131.36,cont,2.0944226531433547,443000000008,2,1,1,0
44b6_9be80b04,15299,44,32.52,124.8,139.35999999999999,cont,1.5381807436059016,443000000009,1,0,1,0
44b6_9be80b04,16597,48,31.320000000000004,130.71999999999997,130.72,cont,1.3397137054236539,447000000008,1,1,0,0
44b6_9be80b04,16936,49,31.680000000000003,127.2,128.64000000000004,cont,1.138080209168034,448000000008,1,0,1,0
44b6_9be80b04,20608,61,30.160000000000004,126.72000000000001,128.0,cont,0.391351823810756,460000000008,1,1,0,0
44b6_9be80b04,20621,61,32.64,121.92000000000002,145.28,cont,1.1344781289209553,460000000009,1,1,0,0
44b6_9be80b04,20907,62,30.32,124.64000000000001,128.0,cont,1.2056799171007186,461000000008,2,1,1,0
44b6_9be80b04,20926,62,32.839999999999996,122.08000000000003,146.72,cont,1.396365818831149,461000000009,1,0,1,0
44b6_9be80b04,21216,63,30.320000000000007,124.00000000000003,128.0,cont,1.1773122196341925,462000000008,1,0,1,0
44b6_9be80b04,22448,67,31.320000000000004,127.99999999999997,132.00000000000003,cont,1.3715616099905865,466000000008,2,1,1,0
44b6_9be80b04,22455,67,32.0,116.63999999999999,142.72,cont,0.7455545335520339,466000000009,1,1,0,0
44b6_9be80b04,22754,68,28.240000000000002,126.56,130.56,cont,6.447339809952628,467000000008,5,0,0,5
44b6_9be80b04,22766,68,32.0,116.0,141.28000000000003,cont,1.0114238107737126,467000000009,1,0,1,0
44b6_9be80b04,22768,68,31.680000000000003,129.44,133.44,cont,0.8390414918226602,467000000008,7,0,1,6
44b6_9be80b04,24003,72,31.680000000000003,112.0,140.79999999999998,cont,1.2742528448075023,471000000009,1,1,0,0
44b6_9be80b04,24318,73,31.32,109.92,138.56,cont,0.5508246658420477,472000000009,1,0,1,0
44b6_9be80b04,25577,77,31.000000000000007,108.64000000000001,138.72000000000003,cont,0.9620494075150101,476000000009,1,0,1,0
44b6_9be80b04,25874,78,31.000000000000007,108.64000000000001,140.0,cont,0.8255575767322341,477000000010,1,1,0,0
44b6_9be80b04,25914,78,35.68,132.64000000000001,162.08,daughter,6.892548172664443,477000000009,1,0,1,0
44b6_9be80b04,26207,79,31.000000000000007,110.07999999999998,140.64000000000001,cont,0.40134539364493105,478000000010,1,0,1,0
44b6_9be80b04,26800,81,32.32,116.63999999999999,125.92,cont,0.8457808891787465,480000000008,1,1,0,0
44b6_9be80b04,26809,81,33.800000000000004,132.64000000000001,154.56,cont,1.5819356221098217,480000000009,2,2,0,0
44b6_9be80b04,27115,82,32.52,117.44,125.28,cont,1.3049670733393963,481000000008,1,0,1,0
44b6_9be80b04,27122,82,33.68,133.44,153.91999999999996,cont,2.412947173582547,481000000009,3,1,2,0
44b6_9be80b04,27453,83,34.0,133.28000000000003,156.64000000000004,cont,0.675890616150297,482000000009,1,0,1,0
44b6_9be80b04,27474,83,36.84,116.64000000000001,131.35999999999999,cont,6.510006360403966,482000000008,1,0,1,0
44b6_9be80b04,28078,85,34.68000000000001,134.72,162.72,cont,2.8202997402758574,484000000009,1,1,0,0
44b6_9be80b04,28382,86,34.84,135.35999999999999,164.64,cont,1.5410107275746057,485000000009,1,0,1,0
44b6_9be80b04,28672,87,30.480000000000004,115.36000000000001,121.28,cont,0.9060741208642848,486000000008,1,1,0,0
44b6_9be80b04,28680,87,31.000000000000007,119.36000000000001,108.64000000000001,cont,5.709210349295249,486000000008,1,1,0,0
44b6_9be80b04,28975,88,30.000000000000007,105.28000000000002,136.0,cont,6.791528040507521,487000000008,1,1,0,0
44b6_9be80b04,28977,88,30.480000000000004,115.35999999999999,120.64000000000001,cont,1.2863186327656047,487000000008,2,1,1,0
44b6_9be80b04,28982,88,31.159999999999997,119.20000000000002,109.44000000000001,cont,5.772799634925841,487000000008,2,1,1,0
44b6_9be80b04,29265,89,30.160000000000004,106.72,136.0,cont,6.80814883980954,488000000008,1,0,1,0
44b6_9be80b04,29276,89,30.520000000000007,115.36000000000004,121.44000000000001,cont,2.884363556315308,488000000008,1,0,1,0
44b6_9be80b04,29596,90,30.599999999999994,114.71999999999997,120.63999999999999,cont,2.8854619430864203,489000000008,1,1,0,0
44b6_9be80b04,29617,90,35.480000000000004,148.64000000000001,176.00000000000003,cont,5.5769706438621425,489000000009,1,1,0,0
44b6_9be80b04,29909,91,30.133333333333326,112.53333333333332,119.46666666666664,cont,2.019842970381624,490000000008,1,0,1,0
44b6_9be80b04,29914,91,31.159999999999997,116.16,108.0,cont,6.232928936302739,490000000008,5,0,0,5
44b6_9be80b04,29943,91,35.84,148.64000000000001,176.00000000000003,cont,5.14157806514693,490000000009,1,0,1,0
44b6_9be80b04,30243,92,32.68,124.00000000000003,151.36,cont,4.701420164429035,491000000010,1,1,0,0
44b6_9be80b04,30272,92,37.00000000000001,116.0,164.0,cont,6.512682938889324,491000000010,1,1,0,0
44b6_9be80b04,30560,93,32.52,116.63999999999999,111.36,cont,6.835457761737684,492000000008,1,1,0,0
44b6_9be80b04,30561,93,32.35999999999999,124.00000000000003,150.71999999999997,cont,5.839721873728248,492000000010,2,1,1,0
44b6_9be80b04,30579,93,35.160000000000004,139.2,184.0,cont,1.6009342975275376,492000000009,1,1,0,0
44b6_9be80b04,30585,93,35.84,148.64000000000001,174.72,cont,5.209791562529157,492000000009,2,2,0,0
44b6_9be80b04,30596,93,37.00000000000001,116.0,164.0,cont,5.759587794495374,492000000010,1,0,1,0
44b6_9be80b04,30608,93,37.99999999999999,127.99999999999997,148.0,cont,6.906249999999989,492000000010,6,1,0,5
44b6_9be80b04,30884,94,30.839999999999996,110.08,128.64000000000004,cont,2.336047008302721,493000000008,1,1,0,0
44b6_9be80b04,30898,94,32.84000000000001,116.0,111.99999999999999,cont,5.747678063792383,493000000008,2,1,1,0
44b6_9be80b04,30908,94,33.999999999999986,129.06666666666666,173.06666666666666,cont,5.312546159930188,493000000010,6,0,0,6
44b6_9be80b04,30921,94,35.8,148.64000000000001,173.92000000000002,cont,5.609428446820587,493000000009,2,0,2,0
44b6_9be80b04,30932,94,35.35999999999999,143.52,186.07999999999998,cont,1.2099431804841234,493000000009,5,0,1,4
44b6_9be80b04,31254,95,31.0,124.00000000000003,118.07999999999998,cont,6.615845798157648,494000000008,4,0,0,4
44b6_9bfa6a0a,1305,2,48.68000000000001,88.00000000000001,152.0,cont,0.5199999999999818,182000000071,1,1,0,0
44b6_9bfa6a0a,1752,3,47.160000000000004,155.35999999999999,116.0,cont,6.663728545266527,183000000015,1,1,0,0
44b6_9bfa6a0a,1778,3,50.48000000000001,148.64000000000001,98.56,cont,2.8935040392230347,183000000015,1,1,0,0
44b6_9bfa6a0a,2230,4,46.839999999999996,152.8,115.36000000000004,cont,6.679560473938401,184000000015,1,0,1,0
44b6_9bfa6a0a,2246,4,49.0,82.72,181.44,cont,0.2543527324406081,184000000042,1,1,0,0
44b6_9bfa6a0a,2259,4,50.16,147.35999999999996,96.63999999999999,cont,3.7133294527957137,184000000015,1,0,1,0
44b6_9bfa6a0a,2260,4,49.86666666666667,151.99999999999997,103.99999999999999,cont,0.6140216359742052,184000000015,6,0,0,6
44b6_9bfa6a0a,2341,4,57.84,20.0,232.0,cont,0.2599999999999909,184000000068,1,1,0,0
44b6_9bfa6a0a,2728,5,48.56,80.96,177.92,cont,0.9392724910801995,185000000042,2,1,1,0
44b6_9bfa6a0a,2831,5,57.160000000000004,19.36,231.36,cont,0.6278895703067583,185000000068,1,0,1,0
44b6_9bfa6a0a,3235,6,48.73333333333332,79.46666666666665,174.93333333333328,cont,2.193750000000021,186000000042,1,0,1,0
44b6_9bfa6a0a,3236,6,48.99999999999999,83.46666666666665,187.46666666666664,cont,3.194570725160965,186000000042,3,0,0,3
44b6_9bfa6a0a,3267,6,51.00000000000001,151.36,111.36,cont,0.6106195624118252,186000000015,2,2,0,0
44b6_9bfa6a0a,3768,7,51.52,151.36,114.07999999999998,cont,0.9564060330215464,187000000015,2,0,2,0
44b6_9bfa6a0a,4550,9,37.64,118.72,120.64000000000001,cont,6.14071567999203,189000000041,2,0,0,2
44b6_9bfa6a0a,5988,12,32.52,121.76000000000002,127.99999999999999,cont,2.9188998890849196,192000000040,1,1,0,0
44b6_9bfa6a0a,6013,12,35.64,118.07999999999998,122.08000000000003,cont,3.0195718343003524,192000000040,1,1,0,0
44b6_9bfa6a0a,6503,13,32.266666666666666,121.06666666666663,127.99999999999997,cont,1.9000845193189464,193000000040,1,0,1,0
44b6_9bfa6a0a,6541,13,35.04,117.91999999999999,121.91999999999999,cont,3.570343558398828,193000000040,1,0,1,0
44b6_9bfa6a0a,7202,14,51.120000000000005,151.2,121.92000000000002,cont,1.5285368821196328,194000000015,1,1,0,0
44b6_9bfa6a0a,7729,15,49.32,81.44,178.56000000000003,cont,1.6624313895616905,195000000040,1,1,0,0
44b6_9bfa6a0a,7770,15,51.52,154.08,124.63999999999999,cont,1.6721715858427852,195000000015,1,0,1,0
44b6_9bfa6a0a,8253,16,49.68,81.28,178.56000000000003,cont,1.6040651483029065,196000000040,1,0,1,0
44b6_9bfa6a0a,10651,21,50.64,146.56000000000003,116.63999999999999,cont,2.9496323923499284,201000000015,1,1,0,0
44b6_9bfa6a0a,10952,22,34.160000000000004,124.00000000000003,128.0,cont,0.48232671758881723,202000000038,1,1,0,0
44b6_9bfa6a0a,11126,22,51.32,147.35999999999999,115.2,cont,2.304121646636744,202000000015,1,0,1,0
44b6_9bfa6a0a,11426,23,34.480000000000004,124.00000000000003,128.0,cont,0.9375841628888503,203000000038,1,0,1,0
44b6_9bfa6a0a,11906,24,34.519999999999996,124.00000000000003,128.0,cont,1.3887809402494102,204000000038,1,1,0,0
44b6_9bfa6a0a,12360,25,34.68,124.00000000000003,127.36,cont,2.6468052676764966,205000000038,1,0,1,0
44b6_9bfa6a0a,17652,36,50.64,144.00000000000003,109.91999999999999,cont,1.5670932486613554,216000000015,1,1,0,0
44b6_9bfa6a0a,18131,37,50.519999999999996,142.72,110.71999999999998,cont,2.6291375130449177,217000000015,1,0,1,0
44b6_9bfa6a0a,18913,39,35.00000000000001,124.00000000000003,120.00000000000003,cont,3.5648292823780383,219000000039,1,1,0,0
44b6_9bfa6a0a,19350,40,32.99999999999999,127.99999999999997,127.99999999999997,cont,1.8616713760758192,220000000039,1,0,0,1
44b6_9bfa6a0a,19365,40,35.00000000000001,124.00000000000003,120.00000000000003,cont,3.5648292823780383,220000000039,1,0,1,0
44b6_9bfa6a0a,19810,41,32.92000000000001,128.00000000000003,128.00000000000003,cont,0.5890484912127328,221000000039,1,1,0,0
44b6_9bfa6a0a,20256,42,32.68,128.0,128.0,cont,2.3294248887869324,222000000039,1,0,1,0
44b6_9bfa6a0a,21255,44,47.00000000000001,77.28,184.64000000000001,cont,1.1863612961488548,224000000040,1,1,0,0
44b6_9bfa6a0a,21284,44,50.0,136.0,112.64,cont,1.6813842883172188,224000000015,1,1,0,0
44b6_9bfa6a0a,21716,45,47.00000000000001,77.28,185.28,cont,1.3073930405581977,225000000040,1,0,1,0
44b6_9bfa6a0a,21743,45,50.0,136.0,113.28000000000003,cont,1.6511151534644681,225000000015,2,1,1,0
44b6_9bfa6a0a,22200,46,50.0,136.0,114.72,cont,0.42187453703678385,226000000015,1,0,1,0
44b6_9bfa6a0a,23105,48,47.00000000000001,78.72,187.35999999999999,cont,0.8451562355564836,228000000040,1,1,0,0
44b6_9bfa6a0a,23547,49,46.839999999999996,77.28,186.56,cont,1.8004530679803865,229000000039,1,0,1,0
44b6_9bfa6a0a,23548,49,46.86666666666666,84.53333333333329,197.06666666666666,cont,3.417654836775254,229000000039,1,0,0,1
44b6_9bfa6a0a,24463,51,46.48,81.28000000000002,196.64000000000001,cont,2.4366331792044593,231000000039,1,1,0,0
44b6_9bfa6a0a,24500,51,50.320000000000014,136.64,120.64000000000001,cont,0.5994906692351593,231000000015,1,1,0,0
44b6_9bfa6a0a,24930,52,46.839999999999996,80.64,197.28000000000003,cont,3.540151170854158,232000000039,1,0,1,0
44b6_9bfa6a0a,29156,61,49.36,82.72000000000001,192.00000000000006,cont,3.1654149115874373,241000000037,1,1,0,0
44b6_9bfa6a0a,29605,62,50.0,75.46666666666667,191.46666666666664,cont,2.7583813251111877,242000000037,5,0,0,5
44b6_9bfa6a0a,29608,62,49.519999999999996,84.80000000000001,191.36,cont,5.135745594604552,242000000037,1,0,1,0
44b6_9bfa6a0a,33291,70,48.68000000000001,81.28000000000002,184.64,cont,4.343008210042894,250000000036,1,1,0,0
44b6_9bfa6a0a,33298,70,50.0,67.36000000000001,182.07999999999993,cont,2.197358143885512,250000000036,1,1,0,0
44b6_9bfa6a0a,33739,71,48.31999999999999,80.64,183.36,cont,4.180578982031083,251000000036,1,0,1,0
44b6_9bfa6a0a,33748,71,49.48,66.72,179.99999999999997,cont,1.9485098472935702,251000000036,1,0,1,0
44b6_9bfa6a0a,35507,75,49.00000000000001,64.8,176.00000000000003,cont,2.1125000000000007,255000000036,1,1,0,0
44b6_9bfa6a0a,35946,76,48.839999999999996,63.36,175.36,cont,2.722444903023752,256000000036,1,0,1,0
44b6_9bfa6a0a,35947,76,48.68,75.2,176.64000000000004,cont,2.1910399927888147,256000000036,3,0,0,3
44b6_9bfa6a0a,39980,85,48.16,78.72000000000001,174.72,cont,3.3353694042039823,265000000034,1,1,0,0
44b6_9bfa6a0a,39987,85,48.68000000000001,64.0,170.72000000000003,cont,3.094889184203535,265000000034,1,1,0,0
44b6_9bfa6a0a,40413,86,48.31999999999999,64.64,170.56,cont,2.699701661758196,266000000032,1,0,1,0
44b6_9bfa6a0a,40415,86,48.0,79.36,173.92000000000002,cont,3.4846684293487686,266000000032,1,0,1,0
44b6_9bfa6a0a,42986,92,33.68,132.00000000000006,132.00000000000006,cont,2.7897989398879535,272000000031,1,1,0,0
44b6_9bfa6a0a,43407,93,33.32,122.71999999999998,136.00000000000003,cont,4.9754729674675255,273000000031,1,1,0,0
44b6_9bfa6a0a,43424,93,33.88000000000001,132.00000000000006,132.00000000000006,cont,3.161199337276933,273000000031,2,1,1,0
44b6_9bfa6a0a,43824,94,32.48,123.35999999999999,138.08,cont,4.695350259831534,274000000031,1,0,1,0
44b6_9bfa6a0a,43830,94,33.32000000000001,132.00000000000006,132.00000000000006,cont,2.1831317098379626,274000000031,1,0,1,0
44b6_a21120c2,2521,10,35.720000000000006,41.92,196.00000000000003,cont,3.8079474868359102,26000000060,2,2,0,0
44b6_a21120c2,2552,10,46.2,83.36000000000001,155.52000000000004,cont,2.5910980418540563,26000000038,1,1,0,0
44b6_a21120c2,2554,10,46.2,122.55999999999997,112.80000000000001,cont,1.8895473168195527,26000000025,2,2,0,0
44b6_a21120c2,2615,10,58.55999999999999,180.48000000000005,92.79999999999998,cont,4.3033362784007725,26000000007,3,3,0,0
44b6_a21120c2,2630,10,61.52,164.64,93.92000000000002,cont,5.110721879049972,26000000007,2,2,0,0
44b6_a21120c2,2755,11,34.44,44.00000000000001,200.0,cont,6.0668456074800545,27000000060,6,0,2,4
44b6_a21120c2,2767,11,34.2,39.199999999999996,216.48000000000002,cont,2.118366853262204,27000000060,4,0,2,2
44b6_a21120c2,2795,11,44.32,85.12,179.84,cont,3.2121609743286523,27000000038,5,0,3,2
44b6_a21120c2,2796,11,44.133333333333326,116.53333333333333,131.99999999999997,cont,1.0305911194821762,27000000025,2,0,0,2
44b6_a21120c2,2857,11,58.0,166.88000000000005,109.44000000000001,cont,3.0737853291991444,27000000007,6,0,3,3
44b6_a21120c2,4261,17,49.480000000000004,75.36,208.64000000000004,cont,0.835098235239422,33000000038,1,1,0,0
44b6_a21120c2,4512,18,50.32,74.56,214.56,cont,0.6114838509723689,34000000038,1,0,1,0
44b6_a21120c2,4832,20,3.3200000000000003,114.72,61.44000000000001,cont,0.5788741767430977,36000000011,2,2,0,0
44b6_a21120c2,4841,20,6.16,120.0,71.36,cont,6.704715341645757,36000000011,2,2,0,0
44b6_a21120c2,5087,21,4.68,114.07999999999998,64.0,cont,1.235213797283695,37000000011,3,1,2,0
44b6_a21120c2,5336,22,6.0,111.84,66.72,cont,0.7776265974103523,38000000011,1,0,1,0
44b6_a21120c2,5588,23,7.32,112.64,66.72,cont,0.6142843905716637,39000000011,3,3,0,0
44b6_a21120c2,5839,24,8.52,112.64,67.36000000000001,cont,1.3538307916796737,40000000011,3,0,3,0
44b6_a21120c2,7735,31,56.68,112.64,206.72,cont,1.4538075010124223,47000000025,1,1,0,0
44b6_a21120c2,7745,31,59.519999999999996,98.72000000000001,206.08000000000004,cont,6.957809012181917,47000000025,1,1,0,0
44b6_a21120c2,7982,32,56.99999999999999,111.36,210.56000000000003,cont,0.9195260328016849,48000000025,1,0,1,0
44b6_a21120c2,8081,33,16.160000000000004,125.92000000000002,75.84,cont,2.073133196396216,49000000011,1,1,0,0
44b6_a21120c2,8228,33,55.16000000000001,116.0,198.56,cont,6.685290289134782,49000000025,1,0,1,0
44b6_a21120c2,8258,33,62.000000000000014,160.64000000000001,190.56,cont,1.7634806668914513,49000000007,1,0,1,0
44b6_a21120c2,8340,34,17.16,126.72000000000001,80.64,cont,1.5122980237043184,50000000011,1,0,1,0
44b6_a21120c2,9984,40,56.52,93.44,230.08000000000004,cont,3.428095316644501,56000000026,3,3,0,0
44b6_a21120c2,10119,41,22.199999999999996,124.96000000000001,81.44,cont,3.7734305512358395,57000000011,4,4,0,0
44b6_a21120c2,10127,41,24.040000000000003,117.76,98.24,cont,5.646448275243472,57000000011,1,1,0,0
44b6_a21120c2,10256,41,57.68000000000001,72.47999999999999,235.2,cont,5.526624761099673,57000000026,6,4,2,0
44b6_a21120c2,10257,41,56.08,88.47999999999999,230.24,cont,4.0280337945454265,57000000026,7,4,3,0
44b6_a21120c2,10393,42,24.680000000000003,114.24,95.19999999999999,cont,4.155999052574478,58000000011,5,2,1,2
44b6_a21120c2,10479,42,45.16,235.19999999999993,21.919999999999998,cont,1.8940140079999113,58000000017,7,2,2,3
44b6_a21120c2,10531,42,58.040000000000006,70.08,238.87999999999997,cont,2.1560510777808575,58000000026,8,2,4,2
44b6_a21120c2,10673,43,25.16,117.44,95.84,cont,2.082157084612015,59000000011,2,0,2,0
44b6_a21120c2,10754,43,45.31999999999999,240.79999999999998,22.560000000000002,cont,2.5716619747742944,59000000017,2,0,2,0
44b6_a21120c2,10799,43,59.11999999999999,88.64,245.28,cont,2.0586255275061354,59000000027,6,0,3,3
44b6_a21120c2,11214,45,26.640000000000004,126.72000000000004,98.24,cont,2.938780040254804,61000000011,5,5,0,0
44b6_a21120c2,11289,45,45.800000000000004,249.44,22.240000000000002,cont,2.219597725377282,61000000017,1,1,0,0
44b6_a21120c2,11478,46,28.04,129.44000000000003,99.84000000000003,cont,2.439016195108192,62000000011,5,0,5,0
44b6_a21120c2,11553,46,46.68000000000001,250.08000000000004,21.119999999999997,cont,2.670740768026716,62000000017,1,0,1,0
44b6_a21120c2,13275,53,34.480000000000004,120.64000000000001,146.71999999999997,daughter,1.2701014969284923,69000000012,1,1,0,0
44b6_a21120c2,13393,54,0.0,115.99999999999997,115.46666666666664,cont,6.669817289372393,70000000069,3,0,0,3
44b6_a21120c2,13598,54,52.52000000000001,244.63999999999996,55.36000000000001,cont,2.602081198291087,70000000018,1,1,0,0
44b6_a21120c2,13854,55,53.52,244.64000000000001,59.2,cont,1.072992311249245,71000000018,1,0,1,0
44b6_a21120c2,13913,56,4.32,103.36,112.0,cont,1.2354275575686342,72000000069,1,1,0,0
44b6_a21120c2,14163,57,3.3600000000000008,90.72,163.36,cont,6.665986892426365,73000000070,1,0,1,0
44b6_a21120c2,14169,57,5.6800000000000015,101.92000000000002,116.0,cont,0.7583720228225748,73000000069,1,0,1,0
44b6_a21120c2,14653,59,3.0,91.36000000000001,154.71999999999997,cont,1.6495951207796407,75000000070,2,2,0,0
44b6_a21120c2,14918,60,4.0,90.72,157.28,cont,0.5966206918972903,76000000071,3,1,2,0
44b6_a21120c2,15184,61,5.160000000000002,88.64,160.64000000000001,cont,1.0040869608754035,77000000072,1,0,1,0
44b6_a21120c2,15384,61,60.760000000000005,238.72000000000003,106.56,daughter,3.322161382293164,77000000019,5,0,0,5
44b6_a21120c2,15449,62,6.48,87.36,164.64,cont,0.8961105191325452,78000000072,3,3,0,0
44b6_a21120c2,15469,62,10.439999999999998,96.48,141.28000000000003,cont,4.424329221757807,78000000071,1,1,0,0
44b6_a21120c2,15470,62,12.520000000000001,107.35999999999999,149.28000000000003,cont,6.989559289576133,78000000071,2,2,0,0
44b6_a21120c2,15703,63,8.0,85.28,169.28000000000003,cont,1.632943086883313,79000000072,3,0,3,0
44b6_a21120c2,15727,63,16.8,95.68,155.84000000000003,cont,3.831865774332408,79000000071,7,0,1,6
44b6_a21120c2,16072,64,42.36,136.8,180.64000000000001,cont,1.3790497181030155,80000000013,1,1,0,0
44b6_a21120c2,16310,65,42.32000000000001,135.35999999999999,188.00000000000003,cont,0.8451562355564901,81000000013,1,0,1,0
44b6_a21120c2,16780,67,43.32000000000001,132.00000000000003,202.72000000000003,cont,0.669610427786183,83000000013,1,1,0,0
44b6_a21120c2,17014,68,44.32,130.08,208.64000000000001,cont,1.3034485509600986,84000000013,1,0,1,0
44b6_a21120c2,18691,76,29.279999999999994,103.36000000000003,210.72000000000003,cont,3.7591276414881354,92000000072,1,1,0,0
44b6_a21120c2,18899,77,30.640000000000008,102.72,217.44000000000005,cont,4.1122775395758415,93000000071,1,0,1,0
44b6_a2bb48bb,1648,2,33.86666666666666,161.59999999999997,128.53333333333333,cont,4.969920808948781,244000000030,7,0,0,7
44b6_a2bb48bb,1900,2,59.32000000000001,171.84,49.28,cont,5.588299159851775,244000000015,1,1,0,0
44b6_a2bb48bb,2434,3,57.64000000000001,162.56000000000003,20.640000000000008,cont,6.521758893312443,245000000015,1,1,0,0
44b6_a2bb48bb,2450,3,59.160000000000004,171.83999999999997,48.480000000000004,cont,5.755368282091076,245000000015,2,1,1,0
44b6_a2bb48bb,2790,4,31.68000000000001,153.27999999999997,134.56,cont,1.1338960765872848,246000000030,2,0,2,0
44b6_a2bb48bb,3011,4,55.35999999999999,167.36,76.0,cont,1.746553749530769,246000000016,7,1,2,4
44b6_a2bb48bb,3028,4,58.640000000000015,172.79999999999998,50.72000000000002,cont,2.2114333587065262,246000000015,5,0,1,4
44b6_a2bb48bb,3038,4,59.000000000000014,163.36,41.440000000000005,cont,6.8895400789893,246000000015,4,0,1,3
44b6_a2bb48bb,6106,9,55.84000000000001,166.08,81.27999999999999,cont,2.2463237327909917,251000000016,1,0,1,0
44b6_a2bb48bb,6494,10,32.36,152.0,79.03999999999999,cont,4.607584270797443,252000000011,4,0,0,4
44b6_a2bb48bb,6773,10,60.519999999999996,172.0,62.71999999999999,cont,0.93744333162064,252000000015,1,0,1,0
44b6_a2bb48bb,7083,11,32.32,151.2,76.64000000000001,cont,4.365779569561432,253000000011,3,3,0,0
44b6_a2bb48bb,7099,11,33.64,173.92000000000002,94.56,cont,5.082406694913345,253000000010,1,1,0,0
44b6_a2bb48bb,7699,12,31.133333333333336,169.06666666666666,92.53333333333332,cont,1.3457446436205236,254000000010,4,0,0,4
44b6_a2bb48bb,7709,12,32.266666666666666,140.53333333333333,77.06666666666666,cont,4.9870117764382025,254000000011,9,0,0,9
44b6_a2bb48bb,9307,14,58.31999999999999,174.72,64.16,cont,6.988709197698832,256000000015,2,1,1,0
44b6_a2bb48bb,9911,15,57.0,157.06666666666658,87.99999999999997,cont,4.063944187745602,257000000016,6,0,0,6
44b6_a2bb48bb,10547,16,60.133333333333326,155.99999999999997,55.99999999999998,cont,6.359049012256312,258000000015,4,0,0,4
44b6_a2bb48bb,12149,19,31.48,148.00000000000006,92.63999999999999,cont,1.1813425889639153,261000000011,2,1,1,0
44b6_a2bb48bb,13465,21,30.200000000000003,169.28000000000003,100.0,cont,3.315318612139716,263000000010,1,1,0,0
44b6_a2bb48bb,13770,21,58.31999999999999,145.92,63.84,cont,1.7852087588010623,263000000015,6,0,0,6
44b6_a2bb48bb,14171,22,29.840000000000003,169.28,100.64,cont,5.975847743207645,264000000010,5,0,1,4
44b6_a2bb48bb,14209,22,33.00000000000001,152.0,101.28,cont,1.3930525160596094,264000000011,1,0,1,0
44b6_a2bb48bb,14210,22,32.99999999999999,165.0666666666666,108.53333333333333,cont,1.116674440271442,264000000010,7,0,0,7
44b6_a2bb48bb,18924,29,33.68000000000001,146.72,108.64,cont,0.8527767146211174,271000000011,1,1,0,0
44b6_a2bb48bb,20316,31,33.31999999999999,145.92000000000002,113.28000000000002,cont,0.7405793762318659,273000000011,2,0,2,0
44b6_a2bb48bb,21983,33,55.99999999999999,128.64000000000004,68.64,cont,5.193317972404547,275000000015,1,0,1,0
44b6_a2bb48bb,24460,37,30.959999999999994,136.95999999999998,102.08000000000001,cont,2.8389173222374673,279000000012,5,0,0,5
44b6_a2bb48bb,26496,40,28.520000000000003,152.0,107.36,cont,5.104802426392228,282000000011,1,1,0,0
44b6_a2bb48bb,27485,41,57.68000000000001,124.64000000000001,88.64,cont,5.234442574190672,283000000016,2,1,1,0
44b6_a2bb48bb,28621,43,32.160000000000004,126.08000000000001,112.64,cont,0.3691290424770205,285000000012,1,1,0,0
44b6_a2bb48bb,29289,44,28.866666666666667,120.5333333333333,103.99999999999999,cont,6.664261284810367,286000000012,5,0,0,5
44b6_a2bb48bb,31031,46,56.68,117.28,88.64,cont,6.154203466940297,288000000017,1,1,0,0
44b6_a2bb48bb,31771,47,58.640000000000015,109.92,86.56,cont,2.24849741494179,289000000017,2,1,1,0
44b6_a2bb48bb,32159,48,32.32,118.08000000000001,121.28000000000003,cont,1.2216714983988144,290000000013,2,1,1,0
44b6_a2bb48bb,32185,48,32.88,134.07999999999998,132.16,cont,1.889058131715375,290000000012,3,2,1,0
44b6_a2bb48bb,32464,48,56.879999999999995,119.20000000000002,110.56,cont,3.463957274563301,290000000018,6,1,0,5
44b6_a2bb48bb,32824,49,28.24,130.72,124.0,cont,6.7593945041327474,291000000012,1,0,1,0
44b6_a2bb48bb,32865,49,32.4,131.68,132.32000000000002,cont,0.8526218754524231,291000000012,2,0,2,0
44b6_a2bb48bb,33121,49,52.999999999999986,128.53333333333336,119.99999999999997,cont,6.948551008024009,291000000018,3,0,0,3
44b6_a2bb48bb,36975,55,28.519999999999996,112.00000000000001,112.00000000000001,cont,1.1973284062862746,297000000012,1,1,0,0
44b6_a2bb48bb,37673,56,27.679999999999996,108.0,108.64000000000001,cont,1.9554091643438778,298000000012,2,1,1,0
44b6_a2bb48bb,37696,56,30.639999999999993,100.0,117.28000000000003,cont,1.4507164178432677,298000000013,7,2,0,5
44b6_a2bb48bb,38408,57,31.680000000000003,99.36,119.36000000000001,cont,2.8639213174422284,299000000013,2,0,2,0
44b6_a2bb48bb,38673,57,56.8,92.63999999999999,88.00000000000001,cont,3.640181357089235,299000000017,1,0,1,0
44b6_a2bb48bb,40015,59,57.52000000000001,95.36000000000001,122.71999999999998,cont,1.0511125058717694,301000000018,1,1,0,0
44b6_a2bb48bb,40678,60,58.0,95.46666666666664,102.93333333333334,cont,5.921905691066762,302000000017,4,0,0,4
44b6_a2bb48bb,42314,63,25.160000000000007,91.99999999999999,78.72,cont,6.030151454358338,305000000012,2,2,0,0
44b6_a2bb48bb,42951,64,22.399999999999995,84.53333333333329,97.06666666666666,cont,5.951004014846768,306000000012,6,0,0,6
44b6_a2bb48bb,43000,64,26.999999999999993,79.99999999999997,83.99999999999997,cont,5.873088119762905,306000000012,4,0,0,4
44b6_a2bb48bb,44255,66,22.680000000000003,81.92,98.56,cont,5.511816992834572,308000000012,1,1,0,0
44b6_a2bb48bb,44683,66,59.2,90.72,111.36,cont,1.8884289932375002,308000000017,1,1,0,0
44b6_a2bb48bb,44954,67,29.64,86.08,113.28,cont,2.2720382506683277,309000000013,8,2,1,5
44b6_a2bb48bb,45175,67,55.26666666666666,84.53333333333329,151.99999999999997,cont,6.644254318686865,309000000018,5,0,0,5
44b6_a2bb48bb,45210,67,57.36,95.36000000000001,142.72,cont,1.0278686990564425,309000000018,1,1,0,0
44b6_a2bb48bb,45418,68,22.798026123046874,80.013017578125,101.12368164062501,cont,5.8218269433011525,310000000012,2,0,2,0
44b6_a2bb48bb,45458,68,29.119999999999997,84.00000000000001,111.20000000000002,cont,2.1963965517410537,310000000013,2,0,2,0
44b6_a2bb48bb,47597,71,57.866666666666674,75.46666666666667,143.46666666666664,cont,6.156441409883908,313000000018,3,0,0,3
44b6_a2bb48bb,48291,72,59.160000000000004,83.99999999999999,119.36000000000001,cont,3.0210144198596565,314000000017,2,1,1,0
44b6_a2bb48bb,48597,73,27.8,76.0,109.92000000000002,cont,0.9515620381772296,315000000013,2,0,2,0
44b6_a2bb48bb,49635,74,59.16000000000001,81.28000000000002,117.28,cont,1.489691662727542,316000000017,1,1,0,0
44b6_a2bb48bb,50527,76,24.52,77.43999999999998,105.92,cont,2.9409357566937775,318000000012,1,1,0,0
44b6_a2bb48bb,51878,78,26.000000000000004,56.64,125.43999999999998,cont,6.221373567187547,320000000013,2,1,1,0
44b6_a2bb48bb,51890,78,26.519999999999996,67.36000000000001,118.08000000000001,cont,2.447716091073479,320000000013,1,1,0,0
44b6_a2bb48bb,52893,79,56.99999999999999,87.99999999999999,160.0,cont,0.5745242597140748,321000000018,3,2,1,0
44b6_a2bb48bb,53535,80,56.99999999999999,88.00000000000001,161.27999999999997,cont,1.0114238107736837,322000000018,3,1,2,0
44b6_a2bb48bb,54822,82,58.239999999999995,92.95999999999998,148.0,cont,6.663530408124504,324000000018,1,1,0,0
44b6_a2bb48bb,55491,83,57.160000000000004,80.64,162.56,cont,2.3566982332280126,325000000018,1,0,1,0
44b6_a2bb48bb,55509,83,57.866666666666646,91.46666666666664,147.99999999999997,cont,6.693091185547635,325000000018,1,0,1,0
44b6_a2bb48bb,56232,84,59.64,72.64,125.44000000000001,cont,1.3149453410693444,326000000017,3,3,0,0
44b6_a2bb48bb,56872,85,57.400000000000006,74.72,164.8,cont,4.204607034254218,327000000018,2,1,1,0
44b6_a2bb48bb,56883,85,59.83999999999999,71.36,125.28000000000002,cont,1.1645707363659814,327000000017,3,0,3,0
44b6_a2bb48bb,57404,86,52.00000000000001,57.919999999999995,161.44,cont,5.915089285040414,328000000016,1,1,0,0
44b6_a2bb48bb,57471,86,56.72,71.35999999999999,164.79999999999998,cont,4.862525706872917,328000000016,1,0,1,0
44b6_a2bb48bb,58031,87,55.800000000000004,68.64,164.64,cont,1.6916398294258768,329000000016,5,0,0,5
44b6_a2bb48bb,58878,89,26.0,55.36000000000001,124.63999999999999,cont,2.057855755999428,331000000013,2,1,1,0
44b6_a2bb48bb,59208,89,59.0,64.0,128.64000000000001,cont,0.431773233306557,331000000015,2,1,1,0
44b6_a2bb48bb,62015,94,50.16,51.99999999999999,160.64,cont,5.203477352934298,336000000016,2,2,0,0
44b6_a2bb48bb,62562,95,49.959999999999994,51.99999999999999,161.91999999999996,cont,5.351285622399553,337000000016,2,0,2,0
44b6_a2bb48bb,62622,95,55.32000000000001,55.99999999999999,168.64000000000004,cont,3.940583118334153,337000000016,2,0,2,0
44b6_aaf8b0ea,342,1,49.919999999999995,195.0400000000001,111.36000000000001,cont,0.4363359514181388,330000000000,1,1,0,0
44b6_aaf8b0ea,554,2,49.52,192.64000000000004,110.56,cont,1.0663269315739856,331000000000,1,0,1,0
44b6_aaf8b0ea,6104,32,40.16,144.00000000000003,113.28000000000003,cont,1.2513555150315847,361000000000,1,1,0,0
44b6_aaf8b0ea,6272,33,39.480000000000004,142.72,114.07999999999998,cont,1.219508097554099,362000000000,1,0,1,0
44b6_aaf8b0ea,11911,64,24.866666666666667,127.99999999999997,144.53333333333333,cont,1.8954463890832962,393000000000,2,0,0,2
44b6_aaf8b0ea,11923,64,27.840000000000003,124.00000000000003,133.27999999999997,cont,0.28379404944429815,393000000001,1,0,1,0
44b6_aaf8b0ea,13632,73,23.64,107.35999999999999,124.63999999999999,cont,0.8456247690317578,402000000001,1,1,0,0
44b6_aaf8b0ea,13827,74,23.36,105.28000000000002,123.36,cont,0.7038332544004994,403000000001,1,0,1,0
44b6_aaf8b0ea,14608,78,19.840000000000003,108.64,145.27999999999997,cont,1.779059618169105,407000000000,1,1,0,0
44b6_aaf8b0ea,14801,79,19.359999999999996,107.35999999999999,144.64,cont,2.4134395838719405,408000000000,1,0,1,0
44b6_aaf8b0ea,14809,79,21.86666666666666,111.46666666666665,144.53333333333333,cont,6.813059992274632,408000000000,1,0,0,1
44b6_aaf8b0ea,15190,81,21.68,99.36000000000001,123.36,cont,1.485875688777494,410000000001,1,1,0,0
44b6_aaf8b0ea,15403,82,21.36,97.92000000000002,123.35999999999999,cont,1.205789419840788,411000000001,1,0,1,0
44b6_abf82518,810,1,58.839999999999996,148.0,220.96000000000004,cont,2.089373410857913,201000000022,1,0,1,0
44b6_abf82518,2827,6,54.84,149.27999999999997,206.72,cont,0.5924010993406313,206000000022,1,1,0,0
44b6_abf82518,3209,7,53.84000000000002,150.07999999999998,201.28000000000006,cont,0.8650740575233697,207000000022,1,0,1,0
44b6_abf82518,3959,9,51.84000000000002,149.28000000000003,193.27999999999997,cont,1.910947490775214,209000000022,1,1,0,0
44b6_abf82518,4340,10,51.160000000000004,150.08,190.72000000000003,cont,2.0688617794574933,210000000022,1,0,1,0
44b6_abf82518,5500,13,49.16000000000001,148.64000000000001,181.28000000000003,cont,1.880090589572761,213000000022,1,1,0,0
44b6_abf82518,5876,14,48.52,148.64000000000001,178.72,cont,1.542380971906751,214000000022,2,1,1,0
44b6_abf82518,6238,15,47.16,147.36000000000004,175.36,cont,0.6278895703067618,215000000022,1,0,1,0
44b6_abf82518,6617,16,46.16,147.36,172.64,cont,0.39571335901128385,216000000022,1,1,0,0
44b6_abf82518,6988,17,44.84,146.56000000000003,169.92000000000002,cont,0.5089649423093915,217000000022,1,0,1,0
44b6_abf82518,8433,21,38.47999999999999,144.64,166.72,cont,0.9625982092752867,221000000022,1,1,0,0
44b6_abf82518,8551,21,61.160000000000004,143.99999999999997,204.00000000000003,cont,3.0446967213500833,221000000021,1,0,1,0
44b6_abf82518,8804,22,37.31999999999999,143.99999999999997,164.64,cont,0.9756768483980821,222000000022,1,0,1,0
44b6_abf82518,9919,25,58.31999999999999,140.64,181.28,cont,1.4538075010124158,225000000021,1,1,0,0
44b6_abf82518,10161,26,34.266666666666666,128.53333333333336,155.99999999999997,cont,5.26595006002926,226000000022,6,0,0,6
44b6_abf82518,10264,26,56.99999999999997,127.46666666666667,179.46666666666667,cont,5.2807638665148655,226000000021,4,0,0,4
44b6_abf82518,10268,26,57.68000000000001,139.35999999999999,176.00000000000003,cont,0.6758906161502697,226000000021,1,0,1,0
44b6_abf82518,10300,26,61.99999999999999,140.53333333333333,176.53333333333336,cont,6.559424543907953,226000000021,2,0,0,2
44b6_abf82518,30039,30,51.4774038696289,121.64456512451176,158.59740051269534,cont,6.463083561424911,230000000021,1,1,0,0
44b6_abf82518,11976,31,51.06995727539063,121.08647338867186,156.94106689453128,cont,6.387297012538329,231000000021,1,0,1,0
44b6_abf82518,15915,43,38.67999999999999,130.72,142.08,cont,1.1435197855743362,243000000020,1,1,0,0
44b6_abf82518,16231,44,37.32,129.28,140.0,cont,0.9713368687021017,244000000020,1,0,1,0
44b6_abf82518,17994,50,32.16000000000001,128.0,126.72000000000001,cont,0.5813776741499505,250000000020,1,1,0,0
44b6_abf82518,18267,51,31.160000000000004,128.0,123.35999999999999,cont,0.8225132977648504,251000000020,1,0,1,0
44b6_abf82518,18715,53,6.840000000000001,117.92000000000002,107.2,cont,0.5534550568926104,253000000021,1,1,0,0
44b6_abf82518,18977,54,5.640000000000002,116.0,102.56,cont,1.1394715057867875,254000000021,1,0,1,0
44b6_b2c44266,6464,15,19.360000000000003,8.0,216.63999999999996,cont,1.7005918087536294,209000000032,1,1,0,0
44b6_b2c44266,6545,15,32.52,132.00000000000006,127.36,cont,1.490223347354335,209000000036,1,1,0,0
44b6_b2c44266,6901,16,19.32,8.0,216.0,cont,1.2612345142755963,210000000032,1,0,1,0
44b6_b2c44266,6982,16,32.160000000000004,132.64000000000001,127.36000000000004,cont,1.0040869608754004,210000000036,1,0,1,0
44b6_b2c44266,9321,21,32.52,133.28000000000003,129.28,cont,1.3980667455454272,215000000036,1,1,0,0
44b6_b2c44266,9334,21,32.62673645019531,147.9000659179687,124.25699584960941,cont,5.392040772190294,215000000036,1,1,0,0
44b6_b2c44266,9794,22,32.32,132.64000000000001,130.08,cont,3.1712073804782936,216000000036,1,0,1,0
44b6_b2c44266,9795,22,32.22673645019532,147.90006591796876,124.89699584960941,cont,4.66246980112823,216000000036,1,0,1,0
44b6_b2c44266,13602,30,18.68,8.0,216.0,cont,4.358787998113695,224000000032,1,1,0,0
44b6_b2c44266,14103,31,18.319999999999997,8.0,216.0,cont,4.805413776408857,225000000032,1,0,1,0
44b6_b2c44266,15977,35,22.866666666666656,19.999999999999993,209.06666666666663,cont,5.686855122170374,229000000031,1,0,0,1
44b6_b2c44266,19148,42,32.0,128.0,132.00000000000006,cont,0.4062500000000213,236000000034,1,1,0,0
44b6_b2c44266,19600,43,32.36000000000001,128.0,132.00000000000006,cont,1.116529920109615,237000000034,1,0,1,0
44b6_b2c44266,22612,50,31.840000000000003,127.36000000000001,131.36,cont,1.4136566061105558,244000000033,1,1,0,0
44b6_b2c44266,23062,51,31.48,127.36,130.56,cont,1.6636222715809013,245000000033,1,0,1,0
44b6_b2c44266,24891,55,32.0,123.36000000000003,132.00000000000003,cont,2.6630175553308035,249000000034,1,1,0,0
44b6_b2c44266,25337,56,32.0,121.92000000000002,132.8,cont,4.121513867803917,250000000032,3,0,1,2
44b6_b2c44266,25338,56,31.866666666666667,123.99999999999999,123.46666666666665,cont,1.4816835980472427,250000000032,4,0,0,4
44b6_b2c44266,25774,57,31.24,123.67999999999999,120.96000000000001,cont,0.41141713928809664,251000000032,1,1,0,0
44b6_b2c44266,25779,57,31.84,121.28000000000003,132.64000000000001,cont,5.044336087385537,251000000032,1,1,0,0
44b6_b2c44266,26228,58,31.000000000000007,122.71999999999998,119.36000000000001,cont,1.7933993559717978,252000000032,1,0,1,0
44b6_b2c44266,26235,58,31.680000000000003,121.44000000000001,132.64000000000001,cont,6.626255069796216,252000000032,1,0,1,0
44b6_b2c44266,30640,68,22.16,6.720000000000001,211.36,cont,3.8509757286303476,262000000026,1,1,0,0
44b6_c15fded2,4608,22,14.880000000000003,228.64000000000001,240.16,cont,2.159844467780028,57000000018,7,2,2,3
44b6_c15fded2,4826,23,14.479999999999999,231.84,239.84000000000003,cont,2.628333894789638,58000000019,4,1,2,1
44b6_c15fded2,4843,23,17.2,198.72000000000003,249.28000000000003,cont,2.0793016405755016,58000000018,4,1,2,1
44b6_c15fded2,5057,24,16.16,248.16000000000008,214.72000000000003,cont,2.7861050321191865,59000000019,1,1,0,0
44b6_c15fded2,5082,24,18.767227478027348,225.21783447265628,227.40409179687498,cont,2.162902364872855,59000000018,2,1,1,0
44b6_c15fded2,5285,25,16.479999999999997,242.40000000000003,212.0,cont,2.350864535123206,60000000019,1,0,1,0
44b6_c15fded2,5314,25,20.45402923583985,207.4798266601563,226.272646484375,cont,0.968507484289844,60000000018,7,1,3,3
44b6_c15fded2,5499,26,17.64,240.16,208.64000000000004,cont,2.0566363618782986,61000000019,2,2,0,0
44b6_c15fded2,5533,26,21.36,210.88,223.35999999999999,cont,2.390241881065593,61000000018,6,5,1,0
44b6_c15fded2,5770,27,23.959999999999997,247.52000000000004,191.36,cont,5.973748430633815,62000000019,4,2,1,1
44b6_c15fded2,5788,27,25.13333333333333,215.99999999999994,195.46666666666664,cont,1.0464826171779538,62000000018,4,0,0,4
44b6_c15fded2,6026,28,25.32,247.36,188.00000000000003,cont,6.658634549402765,63000000019,2,0,2,0
44b6_c15fded2,6251,29,25.84,208.0,185.28,cont,0.4955583971642475,64000000018,1,1,0,0
44b6_c15fded2,6253,29,25.680000000000003,218.07999999999998,172.64,cont,6.252217856489014,64000000018,1,1,0,0
44b6_c15fded2,6266,29,26.36,247.36,183.36,cont,6.42097113468828,64000000019,1,1,0,0
44b6_c15fded2,6464,30,23.733333333333327,240.0,174.93333333333328,cont,0.9958420501710523,65000000019,7,2,0,5
44b6_c15fded2,6488,30,26.0,203.35999999999999,182.07999999999993,cont,1.1952314106481927,65000000018,1,0,1,0
44b6_c15fded2,6512,30,27.52,244.0,179.36,cont,6.587967250981142,65000000019,2,1,1,0
44b6_c15fded2,6699,31,23.559999999999995,232.0,169.92000000000002,cont,1.0643439880978243,66000000019,2,0,2,0
44b6_c15fded2,6959,32,24.0,224.00000000000003,166.07999999999998,cont,0.03249999999999886,67000000019,1,1,0,0
44b6_c15fded2,7197,33,24.32,216.64000000000001,161.28,cont,1.2406531797807099,68000000019,1,0,1,0
44b6_c15fded2,7450,34,25.000000000000004,209.28000000000003,158.56,cont,0.6438774145751738,69000000020,1,1,0,0
44b6_c15fded2,7718,35,25.680000000000003,203.35999999999999,155.99999999999997,cont,0.860636573996236,70000000020,1,0,1,0
44b6_c15fded2,12452,55,30.639999999999993,110.08,115.35999999999999,cont,1.3479666399803871,90000000024,1,1,0,0
44b6_c15fded2,12477,55,34.120000000000005,112.80000000000001,127.36,cont,6.856363946181685,90000000024,1,1,0,0
44b6_c15fded2,12683,56,31.0,107.35999999999999,112.00000000000001,cont,1.1468632492586102,91000000024,1,0,1,0
44b6_c15fded2,14027,62,29.159999999999997,63.36,104.0,cont,6.71062043051907,97000000023,1,1,0,0
44b6_c15fded2,14068,62,33.040000000000006,77.28,98.72000000000001,cont,2.040264995043548,97000000024,1,1,0,0
44b6_c15fded2,14253,63,29.0,56.0,101.28000000000002,cont,6.118054863475806,98000000023,1,0,1,0
44b6_c15fded2,14281,63,32.32,72.00000000000001,94.56,cont,1.418504934781684,98000000024,1,0,1,0
44b6_c15fded2,15308,68,29.0,27.36,97.28000000000003,cont,6.842485042913862,103000000023,1,1,0,0
44b6_c15fded2,15315,68,30.480000000000004,40.0,88.00000000000001,cont,1.0218136449470578,103000000024,2,2,0,0
44b6_c15fded2,15343,68,34.16,39.36,98.08,cont,6.861003252076178,103000000024,2,2,0,0
44b6_c15fded2,15506,69,29.159999999999997,22.72,96.0,cont,6.869926537089028,104000000024,1,0,1,0
44b6_c15fded2,15514,69,30.480000000000004,34.720000000000006,86.08,cont,0.9309420631811555,104000000024,2,0,2,0
44b6_c15fded2,15541,69,34.16,32.64,95.99999999999997,cont,6.376173666471123,104000000024,2,0,2,0
44b6_c15fded2,15943,71,34.52000000000001,21.279999999999998,92.63999999999999,cont,5.995104904211438,106000000024,2,2,0,0
44b6_c15fded2,16123,72,32.04,16.639999999999997,79.36000000000001,cont,1.66139860132962,107000000024,1,1,0,0
44b6_c15fded2,16140,72,34.68000000000001,17.440000000000005,91.36000000000001,cont,6.0780430907077445,107000000024,3,1,2,0
44b6_c15fded2,16306,73,31.84,11.36,76.64000000000001,cont,1.9697386406576867,108000000024,1,0,1,0
44b6_c15fded2,16322,73,35.16000000000001,12.0,90.08,cont,6.419984058586136,108000000024,1,0,1,0
44b6_c50204e0,461,1,22.96,128.00000000000003,13.76,cont,1.0842537756448085,144000000046,2,1,1,0
44b6_c50204e0,765,2,22.480000000000004,128.0,12.0,cont,0.9687508064512839,145000000046,2,1,1,0
44b6_c50204e0,51079,6,20.8947908782959,124.60699981689456,6.204738807678222,cont,6.21896606187786,149000000046,2,2,0,0
44b6_c50204e0,2523,7,25.0,130.72,14.880000000000003,cont,1.006057434990668,150000000046,5,0,1,4
44b6_c50204e0,7011,18,38.52000000000001,155.35999999999999,15.36,cont,1.3041574147701747,161000000035,3,2,1,0
44b6_c50204e0,7440,19,38.480000000000004,156.0,20.000000000000007,cont,1.5029273518703494,162000000035,3,1,2,0
44b6_c50204e0,8333,21,38.68000000000001,155.36,26.080000000000002,cont,0.5822853681829759,164000000035,1,0,1,0
44b6_c50204e0,10594,26,39.31999999999999,149.92000000000002,25.28,cont,1.770951809762205,169000000033,1,0,1,0
44b6_c50204e0,11083,27,39.397071533203125,149.093173828125,25.029893493652338,cont,1.281170596707118,170000000033,1,1,0,0
44b6_c50204e0,11455,28,24.040000000000003,134.4,69.44,parent,2.765461647718151,171000000043,1,1,0,0
44b6_c50204e0,11564,28,41.760000000000005,161.27999999999997,28.640000000000008,cont,5.10663844911698,171000000033,1,0,1,0
44b6_c50204e0,11924,29,22.96,139.52,67.35999999999999,daughter,5.085523110998516,172000000043,4,1,1,2
44b6_c50204e0,11964,29,26.160000000000004,133.92000000000004,86.56,daughter,2.547728461296461,172000000044,3,0,0,3
44b6_c50204e0,14026,33,23.13333333333333,142.93333333333334,60.000000000000014,cont,5.5229418143524125,176000000043,7,0,0,7
44b6_c50204e0,14037,33,23.36,137.92000000000002,50.080000000000005,cont,2.80184119152032,176000000043,1,0,1,0
44b6_c50204e0,15352,35,56.32,196.64000000000001,13.280000000000001,cont,1.1204254214359863,178000000010,2,0,2,0
44b6_c50204e0,15358,35,56.84,207.36000000000004,0.6400000000000001,cont,6.556990782744795,178000000010,2,0,2,0
44b6_c50204e0,16387,37,49.000000000000014,160.00000000000006,53.280000000000015,cont,6.598260424157271,180000000034,1,0,1,0
44b6_c50204e0,16889,38,49.84,158.72,32.0,cont,6.780944117525233,181000000034,1,1,0,0
44b6_c50204e0,17204,39,19.133333333333326,135.99999999999994,28.53333333333333,cont,4.755651334786379,182000000043,5,1,0,4
44b6_c50204e0,17455,39,50.52000000000001,159.36,34.080000000000005,cont,5.8570825181996575,182000000034,3,2,1,0
44b6_c50204e0,18012,40,51.160000000000004,160.64,36.64000000000001,cont,6.230725529583221,183000000034,2,0,2,0
44b6_c50204e0,18088,40,61.99999999999999,197.59999999999997,32.00000000000001,cont,6.585261123334442,183000000010,2,0,0,2
44b6_c50204e0,18642,41,62.0,200.48,35.68,cont,6.694783066873789,184000000010,1,1,0,0
44b6_c50204e0,18965,42,30.479999999999997,131.36,130.72000000000003,cont,0.8300235312929287,185000000044,1,1,0,0
44b6_c50204e0,19209,42,59.160000000000004,203.35999999999999,21.92,cont,0.5242986386593105,185000000010,2,1,1,0
44b6_c50204e0,19568,43,30.839999999999996,129.92000000000002,134.72000000000003,cont,1.6499152440352702,186000000044,2,1,1,0
44b6_c50204e0,20992,45,61.133333333333326,198.93333333333328,29.599999999999984,cont,3.927176723027498,188000000010,4,0,0,4
44b6_c50204e0,21011,45,61.72,202.07999999999998,24.64,cont,2.1183668532621995,188000000010,3,0,0,3
44b6_c50204e0,21250,46,23.160000000000004,120.63999999999999,35.52,cont,2.360784430014739,189000000043,2,1,1,0
44b6_c50204e0,22729,48,46.86666666666666,155.46666666666664,35.466666666666654,cont,3.4786008771487587,191000000033,6,0,0,6
44b6_c50204e0,23173,49,31.84,126.56000000000002,128.64000000000004,cont,0.6362450883896671,192000000043,1,0,1,0
44b6_c50204e0,23335,49,49.16,156.96000000000004,37.27999999999999,cont,1.6950705184445942,192000000033,1,1,0,0
44b6_c50204e0,23963,50,50.4,144.0,37.440000000000005,cont,5.774949133975122,193000000033,7,0,0,7
44b6_c50204e0,23964,50,50.87999999999999,159.36000000000004,40.0,cont,1.0017172754824757,193000000033,1,0,1,0
44b6_c50204e0,24789,52,22.840000000000003,115.36000000000001,47.360000000000014,cont,5.596066838637642,195000000041,1,0,1,0
44b6_c50204e0,25429,53,30.999999999999996,132.53333333333333,115.46666666666667,cont,0.6595222650188232,196000000042,7,1,0,6
44b6_c50204e0,25755,53,62.754302978515625,199.82735351562502,24.151198120117186,cont,0.7778631295976147,196000000010,1,0,1,0
44b6_c50204e0,25962,54,22.480000000000004,128.00000000000003,34.239999999999995,cont,3.4810670957337186,197000000041,1,1,0,0
44b6_c50204e0,26225,54,52.800000000000004,160.64000000000001,42.56,cont,0.9013990861433053,197000000033,1,0,1,0
44b6_c50204e0,26502,55,21.28000000000001,128.0,35.35999999999999,cont,6.0527705433131915,198000000041,5,0,1,4
44b6_c50204e0,26544,55,25.39999999999999,123.99999999999999,27.999999999999996,cont,2.415736171439242,198000000041,3,0,0,3
44b6_c50204e0,27990,57,54.239999999999995,155.2,46.08000000000001,cont,0.399697150978074,200000000033,5,0,0,5
44b6_c50204e0,29492,60,29.0,132.00000000000006,118.07999999999998,cont,2.0119180059087944,203000000041,1,0,1,0
44b6_c50204e0,30580,62,26.040000000000003,127.99999999999997,36.8,cont,5.294134283572709,205000000040,2,2,0,0
44b6_c50204e0,30836,62,52.52000000000001,152.63999999999996,44.00000000000001,cont,1.5488730782410824,205000000033,1,1,0,0
44b6_c50204e0,31717,64,24.981279602050787,128.178408203125,105.5276318359375,cont,6.971191018338041,207000000041,1,0,1,0
44b6_c50204e0,31777,64,31.6,131.46666666666664,111.46666666666665,cont,4.711955188813759,207000000041,8,0,0,8
44b6_c50204e0,32847,66,28.519999999999996,116.0,96.63999999999999,cont,6.148966503608555,209000000042,3,3,0,0
44b6_c50204e0,32858,66,29.760000000000005,132.0,104.31999999999998,cont,3.621417410697079,209000000042,2,2,0,0
44b6_c50204e0,33056,66,50.04,149.27999999999994,59.2,daughter,5.590684914435793,209000000034,1,1,0,0
44b6_c50204e0,33397,67,26.866666666666664,129.06666666666666,99.46666666666664,cont,3.4924900829016936,210000000042,9,0,0,9
44b6_c50204e0,33937,68,26.8,126.08000000000003,37.28,cont,5.858525038352242,211000000041,8,2,2,4
44b6_c50204e0,34196,68,55.26666666666666,142.93333333333334,32.53333333333333,cont,5.212468774886481,211000000033,7,0,0,7
44b6_c50204e0,34465,69,25.32,107.35999999999999,56.64,cont,6.72877554332882,212000000041,2,0,2,0
44b6_c50204e0,34486,69,26.866666666666664,112.5333333333333,43.999999999999986,cont,3.29627643987205,212000000041,5,1,0,4
44b6_c50204e0,35032,70,27.319999999999997,111.04000000000002,44.00000000000001,cont,2.492402518153914,213000000041,2,1,1,0
44b6_c50204e0,36017,72,24.480000000000004,119.36000000000001,66.72,cont,6.473723450418623,215000000041,1,1,0,0
44b6_c50204e0,36027,72,25.320000000000004,101.92000000000002,58.08000000000001,cont,5.4372461147625035,215000000041,1,1,0,0
44b6_c50204e0,36057,72,28.040000000000006,107.35999999999999,43.36,cont,5.796787499339623,215000000041,1,1,0,0
44b6_c50204e0,36546,73,25.000000000000004,118.72,66.72,cont,3.332161460673831,216000000041,2,1,1,0
44b6_c50204e0,37376,74,54.24,142.72,68.0,cont,2.9753479313350897,217000000034,3,0,0,3
44b6_c50204e0,37948,75,56.00000000000001,159.35999999999999,10.080000000000002,cont,6.9947709263778,218000000011,1,1,0,0
44b6_c50204e0,39500,78,43.640000000000015,136.0,31.359999999999996,cont,6.859290346858032,221000000033,1,1,0,0
44b6_c50204e0,40729,80,57.88000000000001,142.08,82.72,cont,1.807479252992978,223000000034,5,0,0,5
44b6_c50204e0,41900,82,61.68,173.91999999999996,37.919999999999995,cont,2.1454923677328672,225000000010,2,1,1,0
44b6_c50204e0,42419,83,58.0,128.16,80.0,cont,6.3570739387315625,226000000034,4,0,0,4
44b6_c50204e0,42460,83,61.48,172.64000000000004,38.720000000000006,cont,0.8913832298736654,226000000010,1,0,1,0
44b6_c50204e0,43502,85,56.480000000000004,144.64000000000001,93.27999999999999,cont,0.8650740575233983,228000000034,2,1,1,0
44b6_c50204e0,45046,88,48.99999999999999,135.46666666666664,39.999999999999986,cont,3.8969895788644116,231000000032,2,0,2,0
44b6_c50204e0,45047,88,49.133333333333326,139.99999999999994,28.53333333333333,cont,5.381888350962157,231000000032,7,1,0,6
44b6_c50204e0,46195,90,55.40596130371094,144.77671630859376,11.359597015380857,cont,1.702416781011162,233000000011,1,1,0,0
44b6_c50204e0,46208,90,56.8,143.36,94.72,cont,1.4538983157360037,233000000032,2,1,1,0
44b6_c50204e0,46286,90,60.2,167.36000000000004,36.00000000000001,cont,4.575493313567398,233000000010,1,1,0,0
44b6_c50204e0,46750,91,57.6,138.08000000000004,48.0,cont,2.6579556288621586,234000000031,1,1,0,0
44b6_c50204e0,46765,91,56.60000000000001,160.53333333333333,92.5333333333333,cont,6.346636932046944,234000000032,11,0,0,11
44b6_c50204e0,47295,92,57.84,137.92000000000002,49.28,cont,3.035054571173318,235000000031,1,0,1,0
44b6_c50204e0,51569,92,61.63754653930664,142.95649169921873,86.58314575195311,cont,6.004589088791551,235000000032,1,1,0,0
44b6_c50204e0,47816,93,57.36,156.64000000000001,88.64,cont,6.461965282520487,236000000032,1,1,0,0
44b6_c50204e0,47828,93,58.99999999999999,128.53333333333336,100.0,cont,6.69287199864734,236000000032,6,1,0,5
44b6_c50204e0,48371,94,57.36,155.35999999999999,87.36,cont,6.7321496613266065,237000000032,1,0,1,0
44b6_c50204e0,48411,94,59.24000000000001,142.72,93.27999999999999,cont,3.3083419657737974,237000000032,9,0,1,8
44b6_c50204e0,49342,96,58.31999999999999,135.84,77.44,cont,4.449830404071589,239000000032,6,0,0,6
44b6_c50204e0,49358,96,58.839999999999996,145.27999999999997,97.44,cont,6.6610333751603354,239000000032,1,0,1,0
44b6_c771cb04,691,1,46.68000000000002,90.71999999999998,187.68,cont,6.6170031925336135,192000000014,1,1,0,0
44b6_c771cb04,708,1,47.559999999999995,113.76000000000002,180.32000000000002,cont,5.170360843306793,192000000014,1,1,0,0
44b6_c771cb04,1078,2,47.16000000000001,116.0,181.28000000000003,cont,6.72963885082254,193000000014,1,0,1,0
44b6_c771cb04,1094,2,48.86666666666666,95.99999999999997,175.99999999999994,cont,5.081933294224193,193000000014,7,0,0,7
44b6_c771cb04,2249,5,47.86666666666666,101.59999999999997,156.53333333333333,cont,6.171732505387394,196000000014,7,0,0,7
44b6_c771cb04,3025,7,38.84,120.00000000000003,182.72,cont,1.8884289932374914,198000000013,1,1,0,0
44b6_c771cb04,6595,16,42.32000000000001,112.0,164.0,cont,2.1831317098379666,207000000014,1,1,0,0
44b6_c771cb04,6990,17,42.040000000000006,113.28000000000003,164.0,cont,3.700756890488763,208000000014,1,0,1,0
44b6_c771cb04,7288,18,31.01773681640625,129.59314941406254,165.97233642578124,cont,6.312214100445181,209000000013,2,2,0,0
44b6_c771cb04,7748,19,31.399999999999995,121.28000000000003,171.36,cont,4.264812422604314,210000000013,5,0,1,4
44b6_c771cb04,7769,19,35.519999999999996,110.71999999999998,182.72000000000003,cont,6.267361885195409,210000000013,1,0,1,0
44b6_c771cb04,8654,21,40.00000000000001,121.44000000000001,162.72,cont,0.6438774145751573,212000000014,1,0,1,0
44b6_c771cb04,8667,21,41.999999999999986,111.99999999999999,156.53333333333327,cont,6.118316222849044,212000000014,5,0,0,5
44b6_c771cb04,9390,23,30.679999999999996,124.00000000000003,168.0,cont,0.6598780663880197,214000000013,2,0,2,0
44b6_c771cb04,14855,36,26.160000000000004,115.36000000000004,168.0,cont,6.239597968819774,227000000013,1,1,0,0
44b6_c771cb04,14857,36,25.680000000000003,127.36,156.00000000000003,cont,0.9756768483980612,227000000013,1,1,0,0
44b6_c771cb04,15303,37,25.320000000000004,126.08000000000001,153.92000000000002,cont,1.0350369860541153,228000000013,1,0,1,0
44b6_c771cb04,17777,43,25.64,112.64,162.07999999999998,cont,5.998891694513246,234000000013,1,1,0,0
44b6_c771cb04,17813,43,28.840000000000003,122.71999999999998,156.64000000000004,cont,6.4338714288133065,234000000013,1,0,1,0
44b6_c771cb04,17839,43,31.999999999999993,136.0,150.72,cont,0.9262500000000031,234000000014,1,1,0,0
44b6_c771cb04,18173,44,23.680000000000007,128.0,151.36,cont,2.4323487234563905,235000000013,3,0,0,3
44b6_c771cb04,18192,44,25.840000000000007,112.64,161.92,cont,6.80060073173687,235000000013,2,1,1,0
44b6_c771cb04,18201,44,26.52,125.92,143.83999999999997,cont,4.830078447861907,235000000013,8,1,1,6
44b6_c771cb04,18222,44,28.839999999999996,123.35999999999999,156.0,cont,6.464089867297635,235000000013,1,1,0,0
44b6_c771cb04,18259,44,32.52,136.0,151.35999999999999,cont,1.0882648172664613,235000000014,2,1,1,0
44b6_c771cb04,18619,45,24.720000000000006,125.92,152.8,cont,2.180953418232485,236000000013,7,0,0,7
44b6_c771cb04,18631,45,27.32,127.99999999999997,141.92000000000002,cont,5.371184256055635,236000000013,1,0,1,0
44b6_c771cb04,19108,46,32.804543151855476,133.23484497070314,150.46119140625,cont,1.1823224846335298,237000000014,1,1,0,0
44b6_c771cb04,19467,47,24.907638854980473,123.18953735351565,152.2337036132813,cont,0.90259652773014,238000000013,1,1,0,0
44b6_c771cb04,19502,47,28.839999999999996,124.63999999999999,152.64,cont,6.277171188122875,238000000013,1,0,1,0
44b6_c771cb04,19918,48,24.387638854980473,122.3895373535156,150.95370361328128,cont,1.3777357557552807,239000000013,1,0,1,0
44b6_c771cb04,19964,48,29.64,129.28,132.64000000000001,cont,3.963834357348954,239000000014,6,0,0,6
44b6_c771cb04,20388,49,32.12454315185547,119.63484497070311,145.02119140624998,cont,6.45149632342236,240000000014,1,1,0,0
44b6_c771cb04,22215,53,31.360000000000003,131.36000000000004,120.0,cont,3.0833063624946666,244000000014,1,1,0,0
44b6_c771cb04,22538,54,20.640000000000004,131.52,141.28,cont,6.236465470320509,245000000013,5,1,0,4
44b6_c771cb04,22625,54,31.159999999999997,132.64000000000001,118.72,cont,2.969173390861502,245000000014,1,0,1,0
44b6_c771cb04,22955,55,21.866666666666664,119.46666666666667,139.99999999999994,cont,1.986152146040743,246000000013,8,0,0,8
44b6_c771cb04,23056,55,32.68,112.00000000000001,114.71999999999997,cont,6.880834115497908,246000000014,1,0,1,0
44b6_c771cb04,25410,61,21.800000000000004,123.35999999999999,136.0,cont,1.3082006965676214,252000000013,1,1,0,0
44b6_c771cb04,25865,62,22.319999999999997,124.00000000000003,135.36,cont,2.1880249370151095,253000000013,1,0,1,0
44b6_c771cb04,28086,67,20.266666666666662,119.99999999999997,127.99999999999997,cont,1.2590109240766967,258000000013,8,0,0,8
44b6_c771cb04,28114,67,23.92,127.67999999999999,130.72,cont,6.014190640684748,258000000013,1,0,1,0
44b6_c771cb04,29341,70,21.68,122.71999999999998,124.64000000000001,cont,4.804864234554392,261000000013,1,1,0,0
44b6_c771cb04,29728,71,22.000000000000004,123.35999999999999,123.84,cont,5.416322697411968,262000000013,1,0,1,0
44b6_c771cb04,30464,73,22.839999999999993,124.00000000000003,124.79999999999998,cont,5.094990493121645,264000000013,1,0,1,0
44b6_c771cb04,31503,76,16.840000000000003,130.72,110.72000000000003,cont,6.769231977299927,267000000013,1,1,0,0
44b6_c771cb04,31873,77,17.26666666666666,116.53333333333333,116.53333333333333,cont,1.345744643620536,268000000013,6,0,0,6
44b6_c771cb04,32774,79,29.520000000000003,140.64000000000004,128.0,cont,2.9780092448647633,270000000014,1,1,0,0
44b6_c771cb04,33204,80,28.68,141.28000000000003,128.0,cont,2.740811923500046,271000000014,1,0,1,0
44b6_c771cb04,33377,80,61.52000000000001,54.08,73.76,cont,3.3061462649132682,271000000022,1,1,0,0
44b6_c771cb04,33567,81,27.680000000000007,142.72,128.64000000000001,cont,3.9539292166147755,272000000014,1,1,0,0
44b6_c771cb04,33744,81,60.67999999999999,56.0,78.72,cont,5.422437758978896,272000000022,3,0,1,2
44b6_c771cb04,33745,81,59.71999999999999,63.2,64.64,cont,5.902576238389482,272000000022,3,0,1,2
44b6_c771cb04,33919,82,26.640000000000008,143.36,128.64000000000001,cont,4.281127188019519,273000000014,1,0,1,0
44b6_c771cb04,33953,82,30.999999999999996,148.0,127.99999999999997,cont,3.9804208320226526,273000000014,4,0,0,4
44b6_c771cb04,34080,82,59.160000000000004,63.36000000000001,64.64,cont,6.337354164988725,273000000022,2,2,0,0
44b6_c771cb04,34218,83,15.679999999999998,116.63999999999999,113.44,cont,2.5694019853849315,274000000013,1,1,0,0
44b6_c771cb04,34283,83,25.28,145.92000000000002,113.28000000000002,cont,6.543263831032644,274000000014,1,1,0,0
44b6_c771cb04,34341,83,30.919999999999998,148.0,127.68,cont,5.132350978109349,274000000014,1,1,0,0
44b6_c771cb04,34467,83,58.79999999999998,63.52000000000001,65.28,cont,6.89080478790398,274000000022,2,0,2,0
44b6_c771cb04,34648,84,25.639999999999997,146.72000000000003,114.55999999999999,cont,5.4683362700642455,275000000014,1,0,1,0
44b6_c771cb04,34652,84,25.519999999999996,143.99999999999997,128.64000000000001,cont,4.467568025503372,275000000014,1,1,0,0
44b6_c771cb04,34805,84,58.36000000000001,61.920000000000016,66.72,cont,4.382200503171885,275000000022,3,0,0,3
44b6_c771cb04,35013,85,26.40000000000001,147.35999999999999,118.24,cont,4.287567965933589,276000000015,3,0,0,3
44b6_c771cb04,35346,86,24.840000000000003,143.99999999999997,128.64000000000001,cont,6.953633043237184,277000000015,1,1,0,0
44b6_c771cb04,35383,86,30.159999999999997,150.72,124.63999999999999,cont,6.980618704670799,277000000015,1,1,0,0
44b6_c771cb04,35708,87,26.8,148.00000000000003,117.92000000000002,cont,2.944300903864968,278000000015,1,0,1,0
44b6_c771cb04,35990,88,14.84,119.36000000000001,114.72,cont,0.3848883766756237,279000000014,1,1,0,0
44b6_c771cb04,36396,89,26.320000000000004,132.00000000000006,114.71999999999997,cont,6.347346517443311,280000000015,2,2,0,0
44b6_c771cb04,36422,89,30.000000000000007,155.35999999999999,123.35999999999999,cont,6.537207569941466,280000000015,2,2,0,0
44b6_c771cb04,36719,90,27.2,150.72,118.72000000000003,cont,1.8437127114873455,281000000015,6,1,1,4
44b6_c771cb04,36727,90,30.000000000000007,156.00000000000003,123.20000000000002,cont,5.241097362432814,281000000015,3,1,2,0
44b6_c771cb04,37035,91,26.679999999999996,152.0,119.36000000000001,cont,4.4711130116001385,282000000015,1,0,1,0
44b6_c771cb04,37048,91,30.000000000000007,158.08,123.99999999999999,cont,5.72519118130565,282000000015,1,0,1,0
44b6_c771cb04,37051,91,31.26666666666666,148.0,113.6,cont,3.726985512038431,282000000015,3,0,0,3
44b6_c771cb04,37174,91,60.16000000000001,70.72,78.72,cont,3.2643495715072985,282000000023,1,1,0,0
44b6_c771cb04,37309,92,14.84,124.64000000000001,112.64,cont,0.663668780341515,283000000014,1,0,1,0
44b6_c771cb04,37556,92,59.84,71.35999999999999,77.28,cont,1.9062434426116641,283000000023,2,1,1,0
44b6_c771cb04,37901,93,59.31999999999999,71.99999999999999,76.0,cont,1.59415730089601,284000000023,1,0,1,0
44b6_c771cb04,37929,93,61.99999999999999,76.5333333333333,63.999999999999986,cont,5.303356510529697,284000000023,3,0,0,3
44b6_c8e2a523,494,1,41.96000000000001,143.2,225.11999999999998,cont,2.636559370183049,57000000013,1,1,0,0
44b6_c8e2a523,497,1,42.84,214.56,224.64000000000001,cont,2.315724778552074,57000000016,2,2,0,0
44b6_c8e2a523,729,2,39.266666666666666,122.66666666666663,225.6,cont,6.422634197486443,58000000013,5,1,0,4
44b6_c8e2a523,732,2,39.0,237.76,200.16000000000003,cont,2.484550072447721,58000000015,1,1,0,0
44b6_c8e2a523,744,2,42.2,124.0,241.76,cont,2.1225386480580313,58000000013,6,3,2,1
44b6_c8e2a523,745,2,43.639999999999986,196.64,234.56000000000006,cont,1.0958814945056599,58000000016,7,3,2,2
44b6_c8e2a523,981,3,38.8,134.56,219.84,cont,5.491730715357409,59000000013,7,0,4,3
44b6_c8e2a523,986,3,37.99999999999999,248.5333333333333,194.93333333333325,cont,0.19150808657137336,59000000015,2,0,0,2
44b6_c8e2a523,989,3,39.0,235.19999999999993,203.35999999999999,cont,6.753160602636393,59000000015,4,0,1,3
44b6_c8e2a523,991,3,40.28,130.24000000000004,229.12,cont,2.6405124976034333,59000000013,4,0,1,3
44b6_c8e2a523,998,3,42.68000000000001,212.0,225.92000000000004,cont,1.2352137972837138,59000000016,7,0,4,3
44b6_c8e2a523,1494,5,38.47999999999999,250.72000000000003,188.95999999999998,cont,3.76579506877367,61000000015,2,2,0,0
44b6_c8e2a523,1499,5,38.040000000000006,237.28000000000003,208.0,cont,6.573521696929272,61000000015,2,2,0,0
44b6_c8e2a523,1500,5,41.120000000000005,136.0,228.32000000000008,cont,3.6348093571052376,61000000013,1,1,0,0
44b6_c8e2a523,1503,5,41.31999999999999,213.28000000000003,230.40000000000006,cont,2.653928985956501,61000000016,3,3,0,0
44b6_c8e2a523,1771,6,39.16000000000001,250.08000000000004,181.76000000000002,cont,1.5939088156165053,62000000015,2,0,2,0
44b6_c8e2a523,1779,6,42.0,136.16000000000003,223.04000000000002,cont,2.5650819309527035,62000000013,1,0,1,0
44b6_c8e2a523,1784,6,44.00000000000001,215.46666666666664,202.93333333333328,cont,1.6362458089242775,62000000016,3,0,0,3
44b6_c8e2a523,2617,9,46.0,211.36,208.0,cont,0.1462500000000091,65000000016,1,1,0,0
44b6_c8e2a523,2889,10,46.52,210.71999999999997,210.71999999999997,cont,0.90139908614331,66000000016,1,0,1,0
44b6_c8e2a523,4000,14,42.999999999999986,241.43999999999997,182.72,cont,1.0462022091832974,70000000016,1,1,0,0
44b6_c8e2a523,4292,15,43.68,241.28000000000003,183.36,cont,0.5520218519225493,71000000016,1,0,1,0
44b6_c8e2a523,4906,18,0.4800000000000001,112.64,8.64,cont,0.8623224454924036,74000000003,1,0,1,0
44b6_c8e2a523,5503,20,2.5200000000000005,114.72,16.640000000000004,cont,0.830023531292939,76000000003,1,1,0,0
44b6_c8e2a523,5791,21,3.8400000000000007,115.35999999999999,22.080000000000005,cont,0.5242986386593048,77000000003,5,4,1,0
44b6_c8e2a523,6085,22,5.32,116.0,25.28,cont,1.1827946250300603,78000000003,4,0,4,0
44b6_c8e2a523,6322,22,46.839999999999996,232.0,207.36000000000004,cont,6.991976763584121,78000000017,1,0,1,0
44b6_c8e2a523,6390,23,6.679999999999999,116.0,30.72,cont,0.8401422870561872,79000000003,1,1,0,0
44b6_c8e2a523,6685,24,7.840000000000001,116.0,33.92,cont,0.8537051598766398,80000000004,1,0,1,0
44b6_c8e2a523,7495,26,47.00000000000001,232.64000000000004,203.35999999999999,cont,1.6381096193478748,82000000018,1,1,0,0
44b6_c8e2a523,7788,27,47.16000000000001,232.0,206.08,cont,1.9216956217101795,83000000018,1,0,1,0
44b6_c8e2a523,9867,34,48.31999999999999,245.27999999999994,224.64000000000001,cont,6.59986103073389,90000000019,1,0,1,0
44b6_c96cfa10,831,2,24.36,136.0,18.720000000000002,cont,1.448439354098058,85000000016,1,0,1,0
44b6_c96cfa10,2122,6,21.000000000000004,235.35999999999999,115.35999999999999,cont,1.7225766492380001,89000000041,1,1,0,0
44b6_c96cfa10,2501,7,25.319999999999997,227.36,1.2799999999999998,cont,0.6508119928212688,90000000020,2,2,0,0
44b6_c96cfa10,2861,8,26.13333333333333,244.53333333333327,0.0,cont,6.91383978376381,91000000020,5,0,0,5
44b6_c96cfa10,3808,11,20.48,231.36,117.28,cont,0.8726719028363382,94000000041,1,1,0,0
44b6_c96cfa10,4121,12,20.16,230.08000000000004,119.36000000000001,cont,1.8909442912206578,95000000041,1,0,1,0
44b6_c96cfa10,4811,14,20.000000000000007,223.99999999999997,122.08000000000003,cont,1.8477186102055794,97000000041,1,1,0,0
44b6_c96cfa10,5541,16,24.10776611328125,221.7833422851563,16.602846069335943,cont,5.432642168482874,99000000020,1,0,1,0
44b6_c96cfa10,5587,16,26.199999999999996,217.92,18.720000000000002,cont,1.6027476407719374,99000000020,1,0,1,0
44b6_c96cfa10,5911,17,24.52,221.43999999999997,19.360000000000003,cont,4.543640645176507,100000000020,1,1,0,0
44b6_c96cfa10,5972,17,32.279999999999994,85.28,191.84,cont,6.390383522332598,100000000074,1,1,0,0
44b6_c96cfa10,5982,17,31.520000000000003,71.36,196.64000000000001,cont,0.8961105191325522,100000000074,1,1,0,0
44b6_c96cfa10,6239,18,24.520000000000007,218.72000000000006,22.080000000000005,cont,5.0100887280067194,101000000020,3,2,1,0
44b6_c96cfa10,6302,18,31.000000000000007,67.36000000000001,199.36000000000007,cont,1.8924100044652226,101000000075,1,0,1,0
44b6_c96cfa10,6593,19,24.200000000000003,216.0,23.36,cont,6.724751854157889,102000000020,4,2,2,0
44b6_c96cfa10,7355,21,26.999999999999996,235.35999999999999,3.3600000000000008,cont,1.7225766492379884,104000000019,2,2,0,0
44b6_c96cfa10,43557,21,25.485106658935546,229.30014099121092,18.12346778869629,cont,6.997607000958312,104000000019,2,2,0,0
44b6_c96cfa10,7721,22,27.16,232.63999999999993,5.280000000000001,cont,0.5994906692351485,105000000019,2,0,2,0
44b6_c96cfa10,8056,23,26.23338073730468,224.29339599609378,23.734874572753903,cont,6.840946206835111,106000000019,1,1,0,0
44b6_c96cfa10,8078,23,27.84,230.72000000000003,8.64,cont,0.6211242327425284,106000000019,1,1,0,0
44b6_c96cfa10,8423,24,28.16,227.36000000000004,12.0,cont,1.082425661419743,107000000019,4,3,1,0
44b6_c96cfa10,8802,25,28.519999999999996,223.99999999999997,15.359999999999996,cont,1.155922250845634,108000000019,3,0,3,0
44b6_c96cfa10,9522,27,28.640000000000008,217.28000000000003,21.920000000000005,cont,0.9849698599449489,110000000019,2,2,0,0
44b6_c96cfa10,9877,28,28.520000000000003,218.72000000000006,12.0,cont,6.183076954478248,111000000019,3,2,1,0
44b6_c96cfa10,10238,29,28.479999999999997,216.0,15.359999999999996,cont,5.962000765263287,112000000019,2,0,2,0
44b6_c96cfa10,10252,29,28.999999999999993,211.36,29.28,cont,0.18527850657861686,112000000019,1,0,1,0
44b6_c96cfa10,10590,30,27.64,242.08,77.44000000000001,cont,6.029954394520743,113000000046,1,0,1,0
44b6_c96cfa10,11002,31,30.000000000000007,171.36,62.71999999999999,cont,0.18527850657861902,114000000020,2,1,1,0
44b6_c96cfa10,11370,32,28.36,160.64000000000001,48.64,cont,6.471663232894614,115000000020,1,0,1,0
44b6_c96cfa10,11383,32,30.000000000000007,166.55999999999997,65.28,cont,1.998155399362129,115000000020,1,0,1,0
44b6_c96cfa10,11415,32,33.733333333333334,223.99999999999997,75.99999999999999,cont,6.397688723823974,115000000046,4,1,0,3
44b6_c96cfa10,11774,33,31.320000000000004,200.64,45.920000000000016,cont,0.9487829375046849,116000000019,1,0,1,0
44b6_c96cfa10,12157,34,30.360000000000003,218.72000000000006,73.44,cont,1.564226046644155,117000000046,2,2,0,0
44b6_c96cfa10,13648,38,25.359999999999996,47.360000000000014,124.63999999999999,cont,6.829022624065615,121000000017,1,1,0,0
44b6_c96cfa10,13675,38,27.679999999999996,59.36,128.64000000000001,cont,0.6368673331236316,121000000017,1,1,0,0
44b6_c96cfa10,15364,42,31.680000000000003,140.64000000000004,102.72000000000001,cont,0.5520218519225387,125000000021,1,1,0,0
44b6_c96cfa10,15766,43,31.84,122.72000000000001,106.87999999999998,cont,6.259710082144376,126000000022,6,0,0,6
44b6_c96cfa10,18481,50,26.866666666666664,51.46666666666666,192.0,cont,6.9448025254902355,133000000019,7,1,0,6
44b6_c96cfa10,19014,51,38.36000000000001,175.36,111.36,cont,2.344058979633415,134000000022,1,1,0,0
44b6_c96cfa10,19395,52,33.13333333333333,120.53333333333336,123.99999999999999,cont,4.500073301872123,135000000023,7,0,0,7
44b6_c96cfa10,19823,53,32.52,160.64000000000007,14.080000000000002,cont,0.9598510626654675,136000000005,3,0,3,0
44b6_c96cfa10,20996,56,24.64,132.00000000000003,8.0,cont,1.1165299201096164,139000000004,3,2,1,0
44b6_c96cfa10,21421,57,22.840000000000003,146.72000000000003,14.719999999999999,cont,6.827978516735396,140000000004,2,0,2,0
44b6_c96cfa10,21497,57,27.133333333333326,127.99999999999997,0.5333333333333338,cont,6.744674468093732,140000000004,6,0,0,6
44b6_c96cfa10,23212,61,22.68,114.71999999999997,9.280000000000001,cont,6.403757807920291,144000000004,2,2,0,0
44b6_c96cfa10,23232,61,23.840000000000003,124.64000000000001,21.280000000000005,cont,0.5994906692351393,144000000004,2,2,0,0
44b6_c96cfa10,23280,61,27.16,128.0,10.559999999999999,cont,6.523864005518508,144000000004,2,2,0,0
44b6_c96cfa10,23384,61,33.839999999999996,157.44,32.0,cont,0.6850102645216359,144000000005,1,1,0,0
44b6_c96cfa10,23688,62,23.64,124.00000000000003,24.64,cont,1.0720074626605909,145000000004,2,0,2,0
44b6_c96cfa10,23736,62,25.599999999999998,107.99999999999997,147.46666666666667,cont,5.935206062719689,145000000023,4,0,0,4
44b6_c96cfa10,23842,62,32.68,118.07999999999998,144.64,cont,6.9668546749950195,145000000023,1,0,1,0
44b6_c96cfa10,24188,63,23.68,121.92000000000002,27.36,cont,0.6568699452707587,146000000004,1,1,0,0
44b6_c96cfa10,24473,64,3.52,252.0,145.92000000000002,cont,4.451966208878947,147000000034,1,0,1,0
44b6_c96cfa10,24681,64,23.68,113.44,18.720000000000002,cont,6.180748968571687,147000000004,1,0,1,0
44b6_c96cfa10,25195,65,23.999999999999993,113.28,20.639999999999997,cont,5.602834104272583,148000000004,1,1,0,0
44b6_c96cfa10,25358,65,34.480000000000004,139.35999999999999,150.72,cont,5.677834644474951,148000000022,1,1,0,0
44b6_c96cfa10,25375,65,36.96,149.28,144.64000000000001,cont,1.6083395941156233,148000000022,1,1,0,0
44b6_c96cfa10,25901,66,37.36,148.64000000000001,146.72000000000003,cont,0.8126624837532448,149000000022,1,0,1,0
44b6_c96cfa10,26168,67,22.520000000000003,136.0,35.36,cont,6.58496037383673,150000000004,2,1,1,0
44b6_c96cfa10,26207,67,24.200000000000003,113.44,24.0,cont,6.117990121355215,150000000004,2,2,0,0
44b6_c96cfa10,26684,68,23.320000000000007,120.00000000000003,37.279999999999994,cont,0.971336868702093,151000000004,1,0,1,0
44b6_c96cfa10,26731,68,25.266666666666666,165.0666666666666,131.99999999999997,cont,3.969533496590683,151000000048,8,0,0,8
44b6_c96cfa10,26917,68,37.32,146.72000000000003,148.64000000000001,cont,1.0722537596110266,151000000022,1,1,0,0
44b6_c96cfa10,27312,69,30.000000000000007,16.640000000000004,237.27999999999997,cont,1.8761538116582872,152000000019,1,1,0,0
44b6_c96cfa10,27816,70,29.840000000000003,15.359999999999996,239.36,cont,2.074215589686853,153000000019,1,0,1,0
44b6_c96cfa10,29810,74,25.040000000000003,96.63999999999999,154.72000000000003,cont,1.7347207988607225,157000000023,1,0,1,0
44b6_c96cfa10,30290,75,23.360000000000007,156.64,134.72,cont,0.8561760917007771,158000000048,1,1,0,0
44b6_c96cfa10,30323,75,25.44069793701172,114.15245239257814,37.05813537597656,cont,4.926251756542163,158000000004,2,2,0,0
44b6_c96cfa10,30830,76,25.0,112.64,52.0,cont,0.146250000000002,159000000004,2,0,2,0
44b6_c96cfa10,30850,76,25.360000000000003,114.07999999999998,38.72,cont,5.444332058434717,159000000004,2,0,2,0
44b6_c96cfa10,32938,80,24.159999999999997,109.28,57.28000000000001,cont,1.9587149007703928,163000000004,2,2,0,0
44b6_c96cfa10,33178,80,39.68,136.00000000000003,76.0,cont,1.105000000000004,163000000005,1,1,0,0
44b6_c96cfa10,33974,82,23.519999999999996,120.64000000000001,51.35999999999999,cont,6.8301438948970965,165000000004,1,0,1,0
44b6_c96cfa10,34139,82,32.52,145.27999999999997,155.35999999999999,cont,1.1356413386276465,165000000022,3,3,0,0
44b6_c96cfa10,34664,83,32.32,145.28,154.56,cont,0.8276340525860466,166000000022,3,0,3,0
44b6_c96cfa10,35030,84,22.84,90.08000000000001,165.44000000000003,cont,3.537166282845631,167000000023,1,1,0,0
44b6_c96cfa10,36730,87,25.840000000000007,104.64,63.36,cont,0.3957133590112882,170000000004,1,1,0,0
44b6_c96cfa10,37271,88,25.84,104.64,63.36,cont,0.3957133590112929,171000000004,1,0,1,0
44b6_c96cfa10,39122,91,39.36,120.00000000000003,86.72000000000003,cont,1.4185049347816747,174000000005,1,1,0,0
44b6_c96cfa10,40517,94,24.0,81.28000000000002,166.55999999999995,cont,3.742618777340262,177000000024,1,0,1,0
44b6_cf2536e8,616,1,37.760000000000005,112.0,138.72,cont,0.5745242597140661,242000000032,1,0,1,0
44b6_cf2536e8,5156,13,29.68,116.63999999999999,134.71999999999997,cont,2.1529870238810123,254000000031,1,1,0,0
44b6_cf2536e8,5475,14,29.319999999999993,116.63999999999999,132.64000000000001,cont,4.612281329504963,255000000031,1,0,1,0
44b6_cf2536e8,5501,14,33.999999999999986,111.99999999999999,135.99999999999994,cont,3.4710015215352295,255000000031,4,0,0,4
44b6_cf2536e8,8917,24,32.68,120.64000000000001,139.35999999999999,cont,2.0008627045602028,265000000027,1,1,0,0
44b6_cf2536e8,9232,25,32.32,122.71999999999998,138.72000000000003,cont,1.0683061944030872,266000000025,1,0,1,0
44b6_cf2536e8,9554,26,29.0,127.99999999999997,139.99999999999994,cont,3.656249999999989,267000000025,2,0,0,2
44b6_cf2536e8,13698,40,32.16000000000001,120.64000000000001,136.0,cont,2.0058714596155127,281000000025,1,1,0,0
44b6_cf2536e8,13968,41,32.0,121.28000000000003,136.0,cont,1.8149147707261588,282000000025,1,0,1,0
44b6_cf2536e8,18474,59,34.00000000000001,132.64000000000001,128.64000000000001,cont,0.682112985509003,300000000023,1,1,0,0
44b6_cf2536e8,18493,59,37.68,128.0,124.00000000000003,cont,6.432085830039889,300000000023,1,1,0,0
44b6_cf2536e8,18719,60,34.68000000000001,132.00000000000006,130.08,cont,1.4491684037750934,301000000023,1,0,1,0
44b6_cf2536e8,18732,60,38.0,128.0,124.00000000000003,cont,6.809990937035082,301000000023,1,0,1,0
44b6_cf2536e8,19646,64,35.99999999999999,132.64000000000001,132.00000000000006,cont,1.9351150644083146,305000000023,2,2,0,0
44b6_cf2536e8,19862,65,35.99999999999999,132.64000000000001,132.00000000000006,cont,1.6813842883171994,306000000023,2,0,2,0
44b6_cf2536e8,21252,71,39.160000000000004,133.92000000000002,125.92000000000004,cont,5.897518810906835,312000000023,1,1,0,0
44b6_cf2536e8,21476,72,35.68,129.43999999999997,130.72,cont,2.1354379673734516,313000000023,1,1,0,0
44b6_cf2536e8,21484,72,38.67999999999999,134.72000000000003,128.64000000000001,cont,4.710510355842549,313000000023,2,1,1,0
44b6_cf2536e8,21709,73,34.88,126.56,128.0,cont,3.8279012689723304,314000000023,2,1,1,0
44b6_cf2536e8,21721,73,38.040000000000006,134.08,129.92000000000002,cont,4.100477487439733,314000000023,1,0,1,0
44b6_cf2536e8,21938,74,33.800000000000004,125.28,125.92,cont,6.082972135395653,315000000023,2,0,1,1
44b6_cf2536e8,21946,74,36.96,133.27999999999997,133.44,cont,1.945661841122448,315000000023,3,0,0,3
44b6_cf8fed6b,354,7,51.133333333333326,13.066666666666663,139.46666666666664,cont,1.4096348152979203,8000000036,2,1,0,1
44b6_cf8fed6b,357,7,52.84,22.080000000000005,143.36,cont,5.82604993220107,8000000036,3,1,2,0
44b6_cf8fed6b,441,8,49.76000000000001,13.76,137.28,cont,0.719601669328242,9000000036,1,0,1,0
44b6_cf8fed6b,637,10,47.32000000000001,13.280000000000001,135.36000000000004,cont,2.426860111337275,11000000036,1,1,0,0
44b6_cf8fed6b,760,11,46.48000000000001,14.560000000000002,134.72000000000003,cont,2.414314731968468,12000000036,1,0,1,0
44b6_cf8fed6b,1034,13,45.68,24.0,132.00000000000003,cont,1.3859259089143285,14000000036,2,2,0,0
44b6_cf8fed6b,1183,14,45.160000000000004,31.200000000000006,130.08,cont,2.072113956808358,15000000036,2,0,2,0
44b6_cf8fed6b,1424,15,60.67999999999999,40.0,86.08,cont,2.2267836951531783,16000000003,3,2,1,0
44b6_cf8fed6b,1610,16,59.160000000000004,62.71999999999999,83.35999999999999,cont,6.532136183898798,17000000004,3,1,2,0
44b6_cf8fed6b,1618,16,59.519999999999996,44.80000000000001,85.28,cont,1.5050342604073843,17000000004,3,1,2,0
44b6_cf8fed6b,1816,17,58.16000000000001,48.0,83.36000000000001,cont,0.5040120286263143,18000000004,1,0,1,0
44b6_cf8fed6b,1819,17,59.32000000000001,64.64,78.56,cont,6.944399047073841,18000000004,1,0,1,0
44b6_cf8fed6b,2544,20,56.68,49.44000000000001,73.27999999999999,cont,0.6228224566439459,21000000004,1,1,0,0
44b6_cf8fed6b,2809,21,56.52,46.56,70.56,cont,2.4762462140304216,22000000004,1,0,1,0
44b6_cf8fed6b,2824,21,58.839999999999996,46.4,51.2,cont,6.855516597055248,22000000004,1,0,1,0
44b6_cf8fed6b,3119,22,58.519999999999996,51.36000000000001,51.36000000000001,cont,6.539307708962776,23000000004,1,1,0,0
44b6_cf8fed6b,3404,23,55.64,45.92,66.88000000000001,cont,1.1298133142249576,24000000004,2,2,0,0
44b6_cf8fed6b,3738,24,55.640000000000015,47.36,63.19999999999999,cont,1.5245583376834222,25000000004,2,0,2,0
44b6_cf8fed6b,3992,25,41.32000000000001,90.07999999999998,120.8,cont,1.449168403775085,26000000039,2,2,0,0
44b6_cf8fed6b,4088,25,55.32000000000001,48.800000000000004,61.280000000000015,cont,1.6787909525905886,26000000004,2,2,0,0
44b6_cf8fed6b,4340,26,37.13333333333334,83.46666666666665,136.5333333333333,cont,2.3426691952414633,27000000039,5,0,0,5
44b6_cf8fed6b,4437,26,49.88000000000002,49.28,60.800000000000004,cont,2.7906979973655175,27000000004,6,0,3,3
44b6_cf8fed6b,20952,57,30.359999999999996,129.92,160.79999999999998,cont,1.3716578700609106,58000000040,3,3,0,0
44b6_cf8fed6b,20982,57,33.36,159.20000000000002,102.08000000000003,cont,2.4804548474019894,58000000004,3,3,0,0
44b6_cf8fed6b,21609,58,30.360000000000003,147.2,145.44,cont,2.76331238054983,59000000041,6,0,3,3
44b6_cf8fed6b,21630,58,31.080000000000002,173.76000000000002,85.6,cont,4.687216658103187,59000000004,6,0,2,4
44b6_cf8fed6b,22921,60,30.480000000000004,152.0,144.8,cont,2.237194811588835,61000000041,2,2,0,0
44b6_cf8fed6b,22933,60,32.64,176.32,88.95999999999998,cont,4.07745497369377,61000000004,2,2,0,0
44b6_cf8fed6b,23612,61,31.520000000000003,162.72,110.88,cont,2.224351380178047,62000000041,6,0,2,4
44b6_cf8fed6b,23624,61,33.32000000000001,183.99999999999997,63.36,cont,1.3959875536694448,62000000004,8,0,5,3
44b6_cf8fed6b,28228,68,29.0,176.64000000000004,92.0,cont,1.2567250345640302,69000000042,1,1,0,0
44b6_cf8fed6b,28865,69,28.84000000000001,180.80000000000004,91.36000000000001,cont,1.9992783854431049,70000000042,1,0,1,0
44b6_cf8fed6b,29531,70,31.320000000000004,209.92000000000002,47.360000000000014,cont,0.8457808891787578,71000000004,1,1,0,0
44b6_cf8fed6b,30140,71,27.599999999999998,199.46666666666667,39.99999999999999,cont,0.6851601597031548,72000000004,5,0,0,5
44b6_cf8fed6b,41864,92,12.68,203.36,88.64,cont,2.3638584057637626,93000000046,1,1,0,0
44b6_cf8fed6b,42319,93,11.840000000000003,204.00000000000003,90.72,cont,0.7092524673908522,94000000046,1,0,1,0
44b6_d29c9ab2,957,3,38.99999999999999,11.360000000000003,210.71999999999997,cont,1.140976829300235,11000000072,2,1,1,0
44b6_d29c9ab2,2290,8,34.32000000000001,107.36,230.72000000000003,cont,1.1973284062862533,16000000071,1,1,0,0
44b6_d29c9ab2,5506,19,21.800000000000004,25.28,146.88,cont,1.8886387293497944,27000000033,1,0,1,0
44b6_d29c9ab2,5545,19,27.8,27.199999999999996,158.07999999999998,cont,2.451219721893574,27000000077,2,0,2,0
44b6_d29c9ab2,5589,19,34.040000000000006,119.84,154.71999999999997,cont,2.008765556878149,27000000010,4,0,1,3
44b6_d29c9ab2,7007,24,23.840000000000003,102.72000000000003,232.63999999999993,cont,2.0848819660834534,32000000035,2,1,1,0
44b6_d29c9ab2,8677,30,0.4800000000000001,236.64000000000004,81.28000000000002,cont,6.411731922811494,38000000012,1,0,1,0
44b6_d29c9ab2,8684,30,1.32,240.64000000000004,66.72,cont,0.6508119928212834,38000000012,1,0,1,0
44b6_d29c9ab2,8943,30,40.32,153.27999999999997,246.72,cont,0.9006664199358052,38000000054,1,0,1,0
44b6_d29c9ab2,8957,30,43.86666666666667,146.4,252.0,cont,6.877247107511678,38000000054,3,0,0,3
44b6_d29c9ab2,10021,34,13.000000000000002,17.919999999999998,26.560000000000002,cont,0.7708051796660436,42000000036,1,1,0,0
44b6_d29c9ab2,10373,35,15.16,205.92,244.00000000000003,cont,0.8221921916437802,43000000064,1,0,1,0
44b6_d29c9ab2,11037,37,15.160000000000004,205.28,242.72000000000003,cont,0.5924010993406266,45000000064,1,1,0,0
44b6_d29c9ab2,11374,38,15.320000000000004,206.72,241.27999999999997,cont,1.2265251984773755,46000000064,1,0,1,0
44b6_d29c9ab2,15383,49,15.16,201.12,227.51999999999998,cont,3.370024341529303,57000000065,2,2,0,0
44b6_d29c9ab2,15422,49,19.36,40.8,40.8,cont,2.1122499852053505,57000000037,2,2,0,0
44b6_d29c9ab2,15478,49,25.680000000000003,109.92,225.92,cont,2.1775606334841706,57000000036,2,2,0,0
44b6_d29c9ab2,15487,49,26.840000000000003,34.4,167.36000000000004,cont,6.428759600420607,57000000034,1,1,0,0
44b6_d29c9ab2,15545,49,34.52,129.59999999999997,226.08,cont,3.184668350079784,57000000077,3,3,0,0
44b6_d29c9ab2,15732,50,13.599999999999998,226.72000000000003,207.19999999999996,cont,3.2448772126229826,58000000064,8,1,3,4
44b6_d29c9ab2,15739,50,11.719999999999999,140.64000000000001,120.0,cont,3.8330715490321867,58000000062,11,4,1,6
44b6_d29c9ab2,15772,50,16.64,176.16000000000003,243.36000000000004,cont,1.301826841404025,58000000065,9,3,3,3
44b6_d29c9ab2,15787,50,18.96,213.27999999999997,200.64000000000001,cont,1.8345954935625564,58000000056,11,4,3,4
44b6_d29c9ab2,15798,50,19.84,33.44,43.68,cont,4.32738445830735,58000000037,4,2,2,0
44b6_d29c9ab2,15843,50,26.440000000000005,33.92,168.8,cont,2.702829824924248,58000000034,4,2,1,1
44b6_d29c9ab2,15858,50,25.4,105.76,232.16000000000003,cont,4.09090320864476,58000000036,3,1,2,0
44b6_d29c9ab2,15894,50,32.160000000000004,131.20000000000002,122.24000000000004,cont,2.689460912525023,58000000010,4,4,0,0
44b6_d29c9ab2,15915,50,35.480000000000004,102.40000000000003,241.76000000000005,cont,3.5318992765932564,58000000077,4,2,2,0
44b6_d29c9ab2,15920,50,36.16,107.36,235.36000000000004,cont,2.419449834156499,58000000077,12,5,3,4
44b6_d29c9ab2,15976,50,48.04000000000001,106.72000000000001,243.2,cont,1.2887797038283892,58000000058,4,2,1,1
44b6_d29c9ab2,16091,51,12.680000000000003,161.92000000000004,111.84,cont,2.178288103649292,59000000063,7,0,4,3
44b6_d29c9ab2,16113,51,14.68,198.56000000000003,227.99999999999997,cont,2.152435030959122,59000000066,7,0,2,5
44b6_d29c9ab2,16134,51,17.0,229.92,181.92,cont,1.7732614725414877,59000000057,6,0,1,5
44b6_d29c9ab2,16154,51,19.359999999999996,57.12,43.2,cont,3.021800994440234,59000000038,6,1,3,2
44b6_d29c9ab2,16191,51,24.388064270019534,51.658980102539054,161.16028320312503,cont,1.1091441252660412,59000000035,2,0,2,0
44b6_d29c9ab2,16213,51,25.040000000000003,114.72,228.48000000000002,cont,4.1080372822675315,59000000037,8,0,4,4
44b6_d29c9ab2,16252,51,30.360000000000003,57.44,168.0,cont,0.7476766095712831,59000000080,4,0,2,2
44b6_d29c9ab2,16261,51,31.866666666666667,147.46666666666667,115.46666666666667,cont,1.4374592385525216,59000000010,3,0,0,3
44b6_d29c9ab2,16270,51,33.32,139.68,228.00000000000006,cont,5.097762530022379,59000000078,3,0,3,0
44b6_d29c9ab2,16280,51,33.879999999999995,133.76,216.16000000000003,cont,3.06694804528866,59000000078,7,0,3,4
44b6_d29c9ab2,16317,51,41.16,162.72,222.88000000000005,cont,2.490494742516024,59000000056,8,0,3,5
44b6_d29c9ab2,16344,51,44.94405151367188,163.8823120117187,226.71492431640624,cont,4.825814373665749,59000000056,4,0,4,0
44b6_d29c9ab2,16536,52,19.52,57.92000000000001,40.800000000000004,cont,1.2490321252874172,60000000038,1,0,1,0
44b6_d29c9ab2,16843,53,12.96,166.08,99.52,cont,3.0601817756466705,61000000063,4,4,0,0
44b6_d29c9ab2,16894,53,17.0,233.44000000000003,178.56000000000003,cont,1.0645920580203416,61000000057,3,3,0,0
44b6_d29c9ab2,16915,53,20.48,59.36,36.32000000000001,cont,3.28430952598868,61000000038,4,4,0,0
44b6_d29c9ab2,16936,53,20.88000000000001,49.92,51.68000000000001,cont,5.660437400281012,61000000038,3,3,0,0
44b6_d29c9ab2,16963,53,26.120000000000008,50.879999999999995,156.16,cont,2.0422054561184724,61000000035,3,3,0,0
44b6_d29c9ab2,16966,53,25.320000000000004,115.52000000000001,218.72000000000006,cont,2.90197885071548,61000000037,1,1,0,0
44b6_d29c9ab2,17004,53,29.840000000000003,60.640000000000015,173.12,cont,3.945304566823714,61000000080,4,4,0,0
44b6_d29c9ab2,17015,53,32.44,152.0,113.28,cont,4.152026049111442,61000000010,8,8,0,0
44b6_d29c9ab2,17071,53,40.96,97.92000000000002,249.27999999999997,cont,1.7497258713867265,61000000039,2,2,0,0
44b6_d29c9ab2,17072,53,39.52000000000001,173.12,226.72,cont,3.1817234421771996,61000000056,3,3,0,0
44b6_d29c9ab2,17074,53,42.44,10.079999999999998,208.64000000000004,cont,2.8424032525312146,61000000079,4,4,0,0
44b6_d29c9ab2,17224,54,13.680000000000001,167.36,91.04,cont,3.317587837947925,62000000063,4,0,4,0
44b6_d29c9ab2,17243,54,15.520000000000003,197.60000000000002,198.08000000000004,cont,2.3285178359849352,62000000066,5,0,4,1
44b6_d29c9ab2,17274,54,18.866666666666667,239.99999999999994,148.53333333333336,cont,1.9356532417306065,62000000057,3,0,0,3
44b6_d29c9ab2,17287,54,20.999999999999996,62.08,29.11999999999999,cont,2.7547464720006425,62000000038,4,0,4,0
44b6_d29c9ab2,17307,54,21.88,230.24000000000004,160.00000000000006,cont,6.446561583511023,62000000057,8,0,4,4
44b6_d29c9ab2,17367,54,27.880000000000003,47.2,150.56,cont,6.861465085898201,62000000035,6,0,3,3
44b6_d29c9ab2,17399,54,33.96,60.800000000000004,142.72,cont,1.8014794267490268,62000000080,8,0,5,3
44b6_d29c9ab2,17425,54,35.199999999999996,129.44000000000003,196.16000000000003,cont,1.8500751673918476,62000000078,5,0,4,1
44b6_d29c9ab2,17469,54,43.76,98.08000000000001,227.84,cont,3.66782334975391,62000000039,8,0,5,3
44b6_d29c9ab2,17473,54,42.0,169.44000000000005,202.08,cont,1.7838029950081293,62000000056,6,0,4,2
44b6_d29c9ab2,17474,54,43.80000000000001,8.799999999999997,182.72000000000003,cont,3.1320373261664662,62000000079,7,0,2,5
44b6_d29c9ab2,17510,54,47.86666666666666,147.46666666666664,202.93333333333325,cont,6.595445082277118,62000000059,3,0,0,3
44b6_d29c9ab2,17517,54,50.480000000000004,68.64000000000001,223.51999999999998,cont,1.997097894445846,62000000052,8,0,5,3
44b6_d29c9ab2,18735,57,50.0,129.28,204.79999999999995,cont,1.8580508940822964,65000000059,1,1,0,0
44b6_d29c9ab2,20279,61,33.84000000000001,164.64,77.92000000000002,cont,1.3731971134910117,69000000010,1,0,1,0
44b6_d29c9ab2,25593,73,52.0,124.63999999999999,194.71999999999997,cont,6.453725910278502,81000000064,1,1,0,0
44b6_d29c9ab2,26484,75,52.16000000000001,108.64000000000001,196.00000000000003,cont,6.567935606033278,83000000064,1,1,0,0
44b6_d29c9ab2,30045,83,42.0,13.280000000000001,168.0,cont,0.29249999999999954,91000000083,1,1,0,0
44b6_d29c9ab2,31358,86,37.266666666666666,161.06666666666663,163.99999999999997,cont,6.931005468585038,94000000081,3,0,0,3
44b6_d29c9ab2,32394,88,48.31999999999999,175.36,153.27999999999997,cont,0.7799999999999775,96000000061,2,1,1,0
44b6_d29c9ab2,32442,88,56.32000000000001,24.639999999999993,212.0,cont,0.5813776741499536,96000000056,1,0,1,0
44b6_d29c9ab2,35241,94,54.0,89.28000000000002,176.64000000000004,cont,6.828481255008316,102000000064,2,1,1,0
44b6_d29c9ab2,35256,94,56.99999999999999,106.08,179.35999999999999,cont,1.913088144858973,102000000064,2,1,1,0
44b6_d29c9ab2,35706,95,54.32000000000001,88.64,176.00000000000003,cont,6.75753861439207,103000000064,1,0,1,0
44b6_d29c9ab2,35721,95,56.99999999999999,104.64,179.36000000000004,cont,1.4859645436550633,103000000064,2,1,1,0
44b6_d29c9ab2,36174,96,54.68000000000001,88.8,176.00000000000003,cont,5.733141863324146,104000000064,3,0,0,3
44b6_d2f34f90,1350,7,46.82673217773437,142.63879882812503,105.46918457031252,cont,6.781546246924131,231000000003,1,0,1,0
44b6_d2f34f90,1634,9,28.48,160.00000000000006,88.00000000000001,cont,1.502927351870322,233000000002,1,1,0,0
44b6_d2f34f90,1796,10,28.0,160.00000000000006,90.72,cont,1.2240468230422934,234000000002,1,0,1,0
44b6_d2f34f90,2139,12,27.480000000000004,160.00000000000006,94.72,cont,1.2824122777016536,236000000002,1,1,0,0
44b6_d2f34f90,2323,13,27.52,159.35999999999999,95.36000000000001,cont,1.6919519976937916,237000000002,1,0,1,0
44b6_d2f34f90,2691,15,43.16,130.72,109.28,cont,0.4075479266294875,239000000003,1,1,0,0
44b6_d2f34f90,2868,16,43.32000000000001,129.28,111.36,cont,1.2486092262993973,240000000003,1,0,1,0
44b6_d2f34f90,3434,19,25.160000000000007,157.28,94.72,cont,0.7541820900817009,243000000002,1,1,0,0
44b6_d2f34f90,3639,20,24.520000000000007,157.92000000000002,93.28000000000003,cont,0.9588877019755877,244000000002,2,1,1,0
44b6_d2f34f90,3640,20,25.64,172.64000000000004,93.28000000000003,cont,6.149846796872278,244000000002,2,1,1,0
44b6_d2f34f90,3692,20,42.0,124.63999999999999,116.0,cont,1.6456685571523797,244000000003,2,1,1,0
44b6_d2f34f90,3826,21,23.480000000000004,158.72,92.63999999999999,cont,2.761735188246698,245000000002,1,0,1,0
44b6_d2f34f90,3890,21,41.16,124.00000000000003,115.36000000000004,cont,0.5040120286262902,245000000003,1,0,1,0
44b6_d2f34f90,4853,26,39.000000000000014,124.00000000000003,116.0,cont,0.8125,250000000003,1,1,0,0
44b6_d2f34f90,5094,27,42.52000000000001,132.00000000000006,120.64000000000001,cont,6.742966821993142,251000000003,3,1,2,0
44b6_d2f34f90,5303,28,42.32,132.00000000000006,117.28,cont,6.099662132036501,252000000003,1,0,1,0
44b6_d2f34f90,5519,29,37.67999999999999,125.28000000000002,115.36000000000004,cont,0.5924010993406434,253000000003,1,1,0,0
44b6_d2f34f90,5741,30,37.32,126.72000000000001,114.71999999999997,cont,1.2557791406135101,254000000003,1,0,1,0
44b6_d2f34f90,6118,32,19.84,158.71999999999997,92.63999999999999,cont,0.8020325741514616,256000000002,1,1,0,0
44b6_d2f34f90,6316,33,19.520000000000003,157.28,92.0,cont,1.1636633963479255,257000000002,2,1,1,0
44b6_d2f34f90,6508,34,18.48,172.64000000000004,84.8,cont,6.145207736521218,258000000002,1,0,1,0
44b6_d2f34f90,6511,34,18.84,156.64000000000004,92.0,cont,1.073238382187277,258000000002,1,0,1,0
44b6_d2f34f90,7644,40,17.52,153.27999999999997,96.0,cont,2.407688531039678,264000000002,1,1,0,0
44b6_d2f34f90,7704,40,35.199999999999996,121.91999999999999,116.63999999999999,cont,1.308604337643741,264000000003,1,1,0,0
44b6_d2f34f90,7843,41,17.679999999999996,152.63999999999996,96.0,cont,2.2460886141245644,265000000002,1,0,1,0
44b6_d2f34f90,7904,41,34.32,123.35999999999999,116.63999999999999,cont,1.1645707363659756,265000000003,1,0,1,0
44b6_d2f34f90,9157,47,32.68,126.72000000000001,126.08000000000001,cont,2.1968774004254334,271000000003,1,0,1,0
44b6_d2f34f90,9276,48,15.84,156.0,96.0,cont,0.26000000000000156,272000000002,1,1,0,0
44b6_d2f34f90,9497,49,32.000000000000014,128.0,127.36,cont,3.3175878379479053,273000000003,1,1,0,0
44b6_d2f34f90,9694,50,31.680000000000003,128.0,126.08000000000001,cont,0.9374433316206301,274000000003,1,0,1,0
44b6_d2f34f90,9834,51,15.319999999999997,154.71999999999997,91.99999999999999,parent,0.7353910524340138,275000000002,1,1,0,0
44b6_d2f34f90,10022,52,11.999999999999996,164.53333333333333,95.99999999999997,daughter,0.4483089367587551,276000000002,5,0,0,5
44b6_d2f34f90,10030,52,15.160000000000004,153.92000000000002,89.92,daughter,1.4637633432355086,276000000003,1,0,1,0
44b6_d2f34f90,10463,54,29.68,131.36000000000004,124.00000000000003,cont,1.9638311568207496,278000000004,1,1,0,0
44b6_d2f34f90,12191,62,26.0,141.92000000000002,127.2,parent,0.8943407138781098,286000000004,1,1,0,0
44b6_d2f34f90,12397,63,22.133333333333333,144.0,115.46666666666667,daughter,1.4249025307796381,287000000005,5,0,0,5
44b6_d2f34f90,12410,63,26.16,143.36,128.64000000000001,daughter,0.39571335901128324,287000000004,1,0,1,0
44b6_d2f34f90,12609,64,15.0,150.56000000000003,79.36,cont,0.5806959725191809,288000000003,1,1,0,0
44b6_d2f34f90,12870,65,18.16,160.64000000000001,78.72,cont,6.022614802766324,289000000003,1,0,1,0
44b6_d2f34f90,13615,68,22.48,145.44,110.72000000000003,cont,1.0623573433642723,292000000005,1,1,0,0
44b6_d2f34f90,13842,69,22.320000000000004,144.64000000000001,109.28,cont,0.7671996562173359,293000000005,1,0,1,0
44b6_d2f34f90,14053,70,15.64,166.72,82.56,cont,6.0046113789736655,294000000003,2,2,0,0
44b6_d2f34f90,14096,70,28.720413208007805,163.24824218749998,134.79289306640624,cont,6.575291637355823,294000000004,1,0,1,0
44b6_d2f34f90,14272,71,12.840000000000002,159.36,77.28,cont,0.4177862042959316,295000000003,1,1,0,0
44b6_d2f34f90,14275,71,14.733333333333334,168.53333333333333,82.93333333333331,cont,5.191317621359083,295000000003,2,0,2,0
44b6_d2f34f90,14473,72,14.640000000000002,167.20000000000002,70.08,cont,5.7222154800391865,296000000003,3,0,0,3
44b6_d2f34f90,14725,73,26.000000000000004,152.0,156.64000000000004,cont,6.870906622309771,297000000004,1,1,0,0
44b6_d2f34f90,15485,77,18.000000000000004,160.0,105.92000000000003,cont,0.5520218519225588,301000000005,1,1,0,0
44b6_d2f34f90,15494,77,21.000000000000004,167.67999999999998,113.27999999999999,cont,6.893103664170442,301000000005,1,1,0,0
44b6_d2f34f90,15645,78,5.68,192.0,62.08000000000001,cont,0.758372022822569,302000000002,1,1,0,0
44b6_d2f34f90,15676,78,17.68,161.27999999999997,107.36,cont,0.7497926796788438,302000000005,1,0,1,0
44b6_d2f34f90,15686,78,20.999999999999993,167.99999999999994,115.46666666666664,cont,6.7940310940519,302000000005,1,0,1,0
44b6_d2f34f90,15835,79,5.200000000000001,193.27999999999997,60.64,cont,0.7499687493489303,303000000002,1,0,1,0
44b6_d2f34f90,15857,79,15.0,168.64000000000004,75.35999999999999,cont,5.402043594788926,303000000003,1,1,0,0
44b6_d2f34f90,15864,79,17.160000000000004,163.35999999999999,107.36000000000001,cont,1.1366872096139793,303000000005,1,1,0,0
44b6_d2f34f90,16000,80,4.4799999999999995,194.71999999999997,61.27999999999999,cont,1.2589293566360251,304000000002,1,1,0,0
44b6_d2f34f90,16035,80,14.68,170.72000000000003,73.92,cont,4.759946493921971,304000000003,1,0,1,0
44b6_d2f34f90,16044,80,17.16,164.64,108.64000000000001,cont,2.4098810266276667,304000000005,1,0,1,0
44b6_d2f34f90,16197,81,3.959999999999999,195.36000000000004,60.640000000000015,cont,0.7181323433044718,305000000002,1,0,1,0
44b6_d2f34f90,16215,81,7.866666666666666,200.5333333333333,55.99999999999998,cont,6.646903322955902,305000000002,6,0,0,6
44b6_d2f34f90,16619,83,17.359999999999996,168.63999999999996,111.36,cont,2.4994380618450913,307000000005,1,1,0,0
44b6_d2f34f90,16743,84,3.839999999999999,196.00000000000003,63.36,cont,1.681855374281621,308000000002,1,1,0,0
44b6_d2f34f90,16774,84,17.000000000000004,169.28,111.99999999999999,cont,3.4757150793038294,308000000005,1,0,1,0
44b6_d2f34f90,16913,85,3.5200000000000005,196.64000000000001,62.720000000000006,cont,1.0159824371021353,309000000002,1,0,1,0
44b6_d2f34f90,17114,86,11.32,176.0,64.0,cont,2.597408865099986,310000000003,1,1,0,0
44b6_d2f34f90,17142,86,23.16,166.24,145.28,cont,6.248669108298183,310000000004,5,1,0,4
44b6_d2f34f90,17288,87,9.133333333333333,172.0,59.999999999999986,cont,1.9062511384331813,311000000003,3,0,0,3
44b6_d2f34f90,17292,87,11.36,178.08,62.71999999999999,cont,3.0259053397289213,311000000003,1,0,1,0
44b6_d2f34f90,17315,87,22.48,169.44,142.56,cont,6.122304764139077,311000000004,1,0,1,0
44b6_d2f34f90,17460,88,11.599999999999998,179.67999999999998,60.95999999999998,cont,4.061362340705386,312000000003,1,1,0,0
44b6_d2f34f90,17465,88,14.84,172.64,111.99999999999999,cont,1.7225766492379941,312000000005,1,1,0,0
44b6_d2f34f90,17475,88,18.16,181.27999999999997,140.64,cont,2.6018274827512924,312000000004,1,1,0,0
44b6_d2f34f90,17619,89,1.8399999999999999,200.64,56.63999999999999,cont,0.4503332099679018,313000000002,1,1,0,0
44b6_d2f34f90,17651,89,14.520000000000003,173.28000000000003,111.36,cont,1.0820596679481265,313000000005,2,1,1,0
44b6_d2f34f90,17805,90,0.0,211.99999999999994,63.466666666666654,cont,6.236757150710951,314000000002,1,0,1,0
44b6_d2f34f90,17815,90,1.68,200.64000000000004,54.720000000000006,cont,0.7671996562173224,314000000002,1,0,1,0
44b6_d2f34f90,17847,90,13.52,175.36,110.08,cont,0.8581784852232085,314000000005,1,0,1,0
44b6_d2f34f90,18060,91,20.68,175.36,139.35999999999999,cont,4.611565623516591,315000000004,2,2,0,0
44b6_d2f34f90,18239,92,19.68,176.64000000000001,139.36,cont,4.508782055611025,316000000004,2,0,2,0
44b6_d2f34f90,18377,93,0.9999999999999999,200.32,48.31999999999999,cont,0.603223113781292,317000000002,2,2,0,0
44b6_d2f34f90,18573,94,0.9999999999999998,200.0,47.999999999999986,cont,0.5745242597140748,318000000002,2,0,2,0
44b6_d2f34f90,18575,94,2.3200000000000003,206.72,55.99999999999999,cont,4.252969918186584,318000000002,2,0,0,2
44b6_d2f34f90,18766,95,9.84,179.35999999999999,98.56,cont,1.9940544438404877,319000000005,1,1,0,0
44b6_d2f34f90,18772,95,14.64,186.71999999999997,133.92,cont,3.1624520747356755,319000000004,1,1,0,0
44b6_d2f34f90,18947,96,9.68,179.99999999999997,98.08000000000001,cont,1.6109643850811954,320000000005,1,0,1,0
44b6_d2f34f90,18957,96,14.680000000000007,185.28,134.08,cont,2.644160133671936,320000000004,4,0,1,3
44b6_d5e7d891,344,1,0.0,125.27999999999997,173.59999999999997,cont,0.5742944040298359,162000000010,1,0,1,0
44b6_d5e7d891,620,1,30.080000000000005,176.00000000000003,27.040000000000003,cont,5.980353250435964,162000000024,1,1,0,0
44b6_d5e7d891,934,2,25.999999999999996,179.99999999999997,23.999999999999993,cont,3.8325548348980085,163000000024,7,0,0,7
44b6_d5e7d891,1284,3,26.08000000000001,180.00000000000006,24.0,cont,2.9726398200925406,164000000024,1,1,0,0
44b6_d5e7d891,1426,4,3.3200000000000003,202.56,196.64000000000001,cont,1.310217658444584,165000000052,4,3,1,0
44b6_d5e7d891,1794,5,3.866666666666668,195.46666666666664,207.99999999999997,cont,6.375695087378486,166000000052,8,0,0,8
44b6_d5e7d891,1795,5,3.6799999999999997,204.64000000000001,195.36,cont,2.250844980224096,166000000052,3,0,3,0
44b6_d5e7d891,2511,7,4.160000000000001,206.08000000000004,189.92000000000002,cont,1.6159560830975501,168000000052,3,2,1,0
44b6_d5e7d891,3606,10,3.84,205.28,181.28000000000003,cont,1.093589988295432,171000000048,2,0,2,0
44b6_d5e7d891,3616,10,3.5600000000000005,190.56000000000003,188.64000000000001,cont,6.19449062171378,171000000048,1,0,1,0
44b6_d5e7d891,4185,11,28.639999999999997,181.28,27.199999999999996,cont,4.791546103555728,172000000024,1,1,0,0
44b6_d5e7d891,4416,12,10.133333333333333,131.46666666666667,192.5333333333333,parent,3.4786008771487418,173000000010,4,0,1,3
44b6_d5e7d891,4602,12,30.839999999999996,192.0,12.639999999999997,cont,6.597520012284922,173000000024,2,1,1,0
44b6_d5e7d891,4783,13,6.640000000000001,129.28,189.92000000000002,daughter,2.300910485546971,174000000011,1,1,0,0
44b6_d5e7d891,4831,13,11.000000000000002,136.8,200.79999999999998,daughter,2.709416450086621,174000000010,1,1,0,0
44b6_d5e7d891,6082,16,16.840000000000003,233.92000000000007,164.0,cont,0.6101869590543698,177000000048,1,0,1,0
44b6_d5e7d891,7647,20,4.0,116.63999999999999,240.00000000000006,cont,0.4823267175888335,181000000034,2,0,2,0
44b6_d5e7d891,7793,20,17.4,239.99999999999994,164.53333333333333,cont,1.073129024291955,181000000048,3,0,0,3
44b6_d5e7d891,7834,20,19.68,242.72000000000003,168.0,cont,3.070261084175095,181000000048,1,0,1,0
44b6_d5e7d891,8133,21,6.2,93.27999999999999,48.64,cont,3.1814329515172903,182000000062,1,1,0,0
44b6_d5e7d891,8148,21,7.040000000000001,223.36,179.36,cont,0.5752132756639166,182000000049,2,0,2,0
44b6_d5e7d891,8795,22,29.840000000000003,193.27999999999997,41.28000000000001,cont,6.7843312124335355,183000000025,1,1,0,0
44b6_d5e7d891,10528,26,22.519999999999996,127.36,114.72,cont,2.142536464100437,187000000016,1,0,1,0
44b6_d5e7d891,10773,27,3.3600000000000003,124.00000000000003,199.36,cont,1.0720074626605887,188000000011,1,0,1,0
44b6_d5e7d891,10994,27,25.680000000000007,132.00000000000003,160.00000000000006,cont,0.7749052361418286,188000000017,1,0,1,0
44b6_d5e7d891,11447,28,26.000000000000004,132.00000000000003,160.00000000000006,cont,0.4062500000000284,189000000017,1,1,0,0
44b6_d5e7d891,11524,28,35.68,175.36,36.00000000000001,cont,5.780296542998456,189000000025,8,0,0,8
44b6_d5e7d891,11607,29,0.0,107.99999999999997,232.53333333333336,cont,6.553327253409352,190000000033,3,1,0,2
44b6_d5e7d891,13260,32,16.999999999999993,231.4666666666667,178.93333333333334,cont,6.882577894381012,193000000046,6,1,0,5
44b6_d5e7d891,13542,33,0.0,76.0,85.92000000000002,cont,2.175862257244243,194000000059,1,1,0,0
44b6_d5e7d891,14169,34,8.720000000000002,112.0,240.0000000000001,cont,1.0159824371021025,195000000033,3,3,0,0
44b6_d5e7d891,14876,35,28.000000000000004,128.0,164.0,cont,1.6750116604071676,196000000017,1,1,0,0
44b6_d5e7d891,14977,35,41.13333333333333,183.99999999999997,32.53333333333333,cont,4.994874543108067,196000000025,8,1,0,7
44b6_d5e7d891,14980,35,40.199999999999996,194.08000000000004,39.36,cont,1.461145205823165,196000000025,13,1,2,10
44b6_d5e7d891,16109,38,5.68,103.36,49.28,cont,2.248967124037165,199000000054,1,1,0,0
44b6_d5e7d891,16466,38,44.86666666666667,187.99999999999994,39.466666666666654,cont,5.173412934310301,199000000025,5,0,0,5
44b6_d5e7d891,17434,40,41.16,197.92000000000002,53.280000000000015,cont,5.807164352117139,201000000025,1,1,0,0
44b6_d5e7d891,18133,42,9.000000000000002,211.36,182.07999999999998,cont,1.0729923112492374,203000000045,3,0,3,0
44b6_d5e7d891,52768,44,40.284784698486334,186.08702270507814,36.3006103515625,cont,2.40698506727677,205000000025,2,1,1,0
44b6_d5e7d891,19674,45,6.36,108.64000000000001,194.72,cont,1.2134300556686384,206000000011,1,0,1,0
44b6_d5e7d891,21313,48,12.200000000000001,91.36,228.00000000000003,daughter,5.430539942077574,209000000033,1,0,1,0
44b6_d5e7d891,23576,52,20.866666666666664,135.99999999999994,228.53333333333333,cont,1.8543679666009016,213000000010,6,0,0,6
44b6_d5e7d891,24495,54,4.5200000000000005,101.28000000000002,228.64000000000004,cont,6.2055854377326884,215000000033,2,1,1,0
44b6_d5e7d891,24502,54,5.520000000000001,95.36000000000001,212.0,cont,6.966399827924028,215000000011,2,2,0,0
44b6_d5e7d891,25006,55,4.92,100.32000000000001,229.28000000000006,cont,5.649113315379683,216000000033,2,1,1,0
44b6_d5e7d891,25026,55,7.4799999999999995,103.36,201.92000000000004,cont,0.9598510626654594,216000000011,5,3,2,0
44b6_d5e7d891,25149,55,20.680000000000003,138.07999999999998,232.0,cont,2.4846032127887145,216000000010,1,1,0,0
44b6_d5e7d891,28458,61,29.64,119.20000000000002,138.56,cont,3.872549224993791,222000000016,8,1,0,7
44b6_d5e7d891,28468,61,30.434348144531256,117.71791015625001,122.72282714843752,cont,6.708562072939,222000000016,3,1,2,0
44b6_d5e7d891,28571,61,39.84000000000001,190.56000000000003,59.519999999999996,cont,6.862061575248336,222000000025,7,1,0,6
44b6_d5e7d891,29044,62,31.2,119.35999999999999,141.28000000000003,cont,2.2402026275540434,223000000016,6,0,1,5
44b6_d5e7d891,29406,63,12.866666666666665,100.0,219.99999999999997,cont,1.737137752005986,224000000011,6,0,0,6
44b6_d5e7d891,30208,64,33.13333333333334,116.0,160.53333333333325,cont,6.027973707133137,225000000016,1,0,1,0
44b6_d5e7d891,30325,64,45.28325134277345,189.50136230468746,63.83752563476562,cont,1.0676043295247235,225000000025,1,1,0,0
44b6_d5e7d891,30815,65,34.839999999999996,117.91999999999999,150.07999999999998,cont,0.9869785015389168,226000000016,2,0,2,0
44b6_d5e7d891,31744,67,14.0,84.8,216.80000000000004,cont,6.957201754297495,228000000011,2,2,0,0
44b6_d5e7d891,31804,67,19.0,120.64000000000001,235.99999999999994,cont,1.0412687573340358,228000000010,2,2,0,0
44b6_d5e7d891,34016,71,4.159999999999999,207.36,173.27999999999997,cont,3.0546542357523916,232000000040,2,1,1,0
44b6_d5e7d891,34606,72,3.8000000000000007,206.72,172.64000000000004,cont,2.0234349662146265,233000000039,2,1,1,0
44b6_d5e7d891,35297,73,14.0,87.36,246.08000000000004,cont,1.6875763871599974,234000000031,2,2,0,0
44b6_d5e7d891,35570,73,35.13333333333335,112.5333333333333,200.0,cont,0.7529946002831496,234000000016,7,0,0,7
44b6_d5e7d891,36399,74,57.866666666666674,235.46666666666664,235.99999999999994,cont,2.178988400930354,235000000025,7,1,0,6
44b6_d5e7d891,36907,75,40.519999999999996,109.92000000000003,161.92,cont,0.8655318090630764,236000000015,1,1,0,0
44b6_d5e7d891,37069,75,60.48000000000002,231.36,231.36,cont,5.986443539364597,236000000025,2,1,1,0
44b6_d5e7d891,37167,76,5.84,204.00000000000003,176.64000000000001,cont,0.29831034594864186,237000000039,1,1,0,0
44b6_d5e7d891,38350,77,56.68,189.28,102.72000000000001,cont,6.2809350667317085,238000000024,1,0,1,0
44b6_d5e7d891,38552,78,15.200000000000001,89.92,212.0,cont,0.6805627267195882,239000000011,2,2,0,0
44b6_d5e7d891,38913,78,45.68000000000001,185.28,55.2,cont,6.155683603386376,239000000023,1,0,1,0
44b6_d5e7d891,39030,78,60.67999999999999,234.56,211.36,cont,4.335309965850202,239000000026,1,0,1,0
44b6_d5e7d891,39212,79,15.200000000000001,91.36,215.36,cont,1.4200863750138593,240000000011,3,1,2,0
44b6_d5e7d891,39600,79,47.0,175.99999999999994,68.53333333333333,cont,3.329047374792238,240000000022,2,0,2,0
44b6_d5e7d891,39681,79,56.13333333333331,183.4666666666666,101.06666666666666,cont,4.724392322034627,240000000023,6,1,0,5
44b6_d5e7d891,41935,83,15.680000000000001,105.28000000000002,229.92000000000004,cont,1.3002031091333512,244000000010,1,0,1,0
44b6_d5e7d891,42893,84,48.480000000000004,176.00000000000003,79.36,cont,2.0673295727822296,245000000022,1,1,0,0
44b6_d5e7d891,44688,87,49.48,176.00000000000003,84.8,cont,2.5359373267097842,248000000022,2,1,1,0
44b6_d5e7d891,44784,87,59.83999999999999,207.20000000000002,236.0,cont,6.120902838838382,248000000024,2,2,0,0
44b6_d5e7d891,45398,88,59.999999999999986,215.99999999999994,219.99999999999997,cont,4.02166981799853,249000000025,3,0,0,3
44b6_d5e7d891,46006,89,57.31999999999999,197.28000000000003,109.28,cont,6.186151115596841,250000000023,1,0,1,0
44b6_d5e7d891,47239,91,58.680000000000014,220.0,212.0,cont,5.732566097525598,252000000025,1,1,0,0
44b6_d5e7d891,47737,92,43.84,96.63999999999999,191.36,cont,1.5259433557639075,253000000015,2,1,1,0
44b6_d5e7d891,48545,93,56.32,211.36,235.99999999999994,cont,1.3527576002373891,254000000024,2,1,1,0
44b6_d5e7d891,48583,93,60.000000000000014,204.64000000000001,218.72,cont,1.442502166376184,254000000025,2,1,1,0
44b6_d5e7d891,49222,94,58.0,208.64000000000004,208.64,cont,6.271257972887112,255000000025,1,1,0,0
44b6_d5e7d891,49245,94,60.000000000000014,202.08,217.44000000000003,cont,1.6431795588127243,255000000025,1,0,1,0
44b6_d754aa59,1407,21,49.480000000000004,15.84,165.44000000000003,cont,1.4111326346236903,27000000051,1,1,0,0
44b6_d754aa59,1468,22,48.64,23.36,166.71999999999997,cont,1.73700262665316,28000000051,1,0,1,0
44b6_d754aa59,3829,63,33.28000000000001,225.44000000000003,110.56,parent,2.611451344559195,69000000054,1,1,0,0
44b6_d754aa59,3888,64,27.72,242.4,100.64,daughter,3.3751527743200067,70000000054,1,0,0,1
44b6_d754aa59,3895,64,34.0,226.56000000000003,112.16,daughter,2.3002217963709546,70000000055,1,0,1,0
44b6_d78e09d9,533,1,48.56,146.71999999999997,70.71999999999998,cont,1.3987276584453519,67000000027,1,1,0,0
44b6_d78e09d9,834,2,48.83999999999999,143.36,75.36,cont,1.7553009001307807,68000000027,1,0,1,0
44b6_d78e09d9,1114,3,41.16,91.35999999999999,113.92,cont,5.700045093023394,69000000041,1,1,0,0
44b6_d78e09d9,1124,3,45.160000000000004,112.0,121.92000000000002,cont,6.191761789870478,69000000041,1,1,0,0
44b6_d78e09d9,1408,4,40.47999999999999,88.64,119.36000000000001,cont,6.1437250762465,70000000041,1,0,1,0
44b6_d78e09d9,1417,4,41.73333333333332,103.46666666666665,123.46666666666665,cont,0.7607476448643659,70000000041,4,0,0,4
44b6_d78e09d9,1427,4,45.32,110.72000000000003,125.43999999999998,cont,6.529770885145973,70000000041,1,0,1,0
44b6_d78e09d9,3335,10,58.0,173.28000000000003,85.92,cont,1.7691616128833414,76000000008,1,1,0,0
44b6_d78e09d9,3940,12,57.48,173.92000000000002,95.99999999999997,cont,0.7806767897151815,78000000008,1,1,0,0
44b6_d78e09d9,4266,13,58.0,187.35999999999999,112.64,cont,6.323150421269443,79000000008,2,1,1,0
44b6_d78e09d9,4586,14,58.0,187.36,120.00000000000003,cont,5.849571164837658,80000000009,2,1,1,0
44b6_d78e09d9,4917,15,57.84,188.00000000000003,128.0,cont,6.618379829119823,81000000009,1,0,1,0
44b6_d78e09d9,5507,17,42.999999999999986,155.36,21.279999999999998,cont,1.9688670600118963,83000000014,2,2,0,0
44b6_d78e09d9,5842,18,43.84,154.71999999999997,30.080000000000002,cont,0.9728309205612273,84000000014,2,0,2,0
44b6_d78e09d9,6975,21,62.86666666666666,168.53333333333333,151.46666666666664,cont,6.690844179424079,87000000009,5,0,0,5
44b6_d78e09d9,7206,22,45.84,151.36,56.0,cont,1.6096525246151792,88000000014,1,0,1,0
44b6_d78e09d9,7432,23,22.680000000000003,61.28,248.0,cont,1.0958814945056767,89000000068,1,1,0,0
44b6_d78e09d9,7529,23,40.68,178.56000000000003,10.560000000000002,cont,0.83904149182266,89000000013,2,2,0,0
44b6_d78e09d9,7877,24,41.84,179.99999999999997,17.44,cont,1.604723496431721,90000000013,3,1,2,0
44b6_d78e09d9,8221,25,42.999999999999986,180.64000000000004,23.36,cont,1.8411328883325848,91000000013,2,1,1,0
44b6_d78e09d9,8557,26,43.84,182.72,29.28,cont,0.968750806451271,92000000013,1,0,1,0
44b6_d78e09d9,8748,27,14.68,136.64,6.080000000000001,cont,1.1529486653793466,93000000072,1,0,1,0
44b6_d78e09d9,8876,27,42.64000000000001,120.64000000000001,239.36,cont,5.301411516049295,93000000045,1,1,0,0
44b6_d78e09d9,8892,27,44.52,184.0,34.72,cont,1.2772541348533584,93000000013,1,1,0,0
44b6_d78e09d9,9195,28,38.72,110.72000000000003,247.2,cont,4.037495549533145,94000000045,1,0,0,1
44b6_d78e09d9,9208,28,42.84,123.36000000000003,244.0,cont,4.854291432588295,94000000045,1,0,1,0
44b6_d78e09d9,9221,28,44.99999999999999,185.28,39.36,cont,1.2800421135650206,94000000013,2,1,1,0
44b6_d78e09d9,9419,29,16.840000000000003,136.64,14.560000000000002,cont,1.1783211043684212,95000000072,2,0,2,0
44b6_d78e09d9,9459,29,25.040000000000003,153.91999999999996,8.640000000000002,cont,3.718943062215383,95000000071,1,0,1,0
44b6_d78e09d9,9551,29,45.480000000000004,186.72000000000006,44.64000000000001,cont,1.219724610311675,95000000013,1,0,1,0
44b6_d78e09d9,9884,30,46.16,154.08000000000004,199.36,cont,6.653397745701063,96000000032,1,1,0,0
44b6_d78e09d9,9906,30,51.00000000000001,153.27999999999997,208.64000000000004,cont,3.5640884616546877,96000000032,1,1,0,0
44b6_d78e09d9,10212,31,46.64,152.0,202.72000000000003,cont,6.311300009704798,97000000032,2,1,1,0
44b6_d78e09d9,10242,31,51.0,154.08,213.28000000000003,cont,3.4720664326161708,97000000032,2,1,1,0
44b6_d78e09d9,10249,31,52.0,156.0,109.28,cont,0.42187453703678385,97000000014,1,1,0,0
44b6_d78e09d9,10419,32,19.48,136.0,34.07999999999999,cont,1.2196163587374527,98000000072,1,1,0,0
44b6_d78e09d9,10557,32,47.64,150.72,204.64000000000001,cont,6.476109219662677,98000000032,1,0,1,0
44b6_d78e09d9,10570,32,50.79999999999999,154.08000000000004,218.08,cont,2.9723733152146097,98000000032,6,0,1,5
44b6_d78e09d9,10591,32,52.67999999999999,156.0,114.07999999999998,cont,0.9652007563196425,98000000014,3,2,1,0
44b6_d78e09d9,10757,33,20.52,136.0,39.36,cont,2.452888921761441,99000000072,1,0,1,0
44b6_d78e09d9,10921,33,53.32000000000001,156.0,119.36000000000001,cont,1.1863612961488503,99000000014,2,0,2,0
44b6_d78e09d9,11637,35,60.48000000000002,192.0,196.00000000000003,cont,5.260684960154897,101000000009,1,1,0,0
44b6_d78e09d9,11955,36,58.84000000000001,181.28000000000006,214.08,cont,0.3926989941417898,102000000009,1,0,1,0
44b6_d78e09d9,11975,36,60.67999999999999,189.92000000000002,200.64,cont,6.874806360909378,102000000009,1,0,1,0
44b6_d78e09d9,12577,38,51.00000000000001,184.64000000000001,84.64000000000001,cont,1.666080730336934,104000000013,1,0,1,0
44b6_d78e09d9,12915,39,53.86666666666666,191.46666666666664,87.99999999999997,cont,6.162455276488584,105000000013,7,0,0,7
44b6_d78e09d9,12938,39,57.67999999999999,152.64,170.72000000000003,cont,1.2538852120509136,105000000014,1,1,0,0
44b6_d78e09d9,13249,40,58.239999999999995,153.27999999999997,181.44000000000003,cont,1.5604231197018357,106000000014,1,0,1,0
44b6_d78e09d9,13752,42,36.00000000000001,151.35999999999999,68.64,cont,0.2068287334970604,108000000071,1,1,0,0
44b6_d78e09d9,13871,42,57.519999999999996,154.71999999999997,202.56,cont,6.25833893697042,108000000014,6,0,0,6
44b6_d78e09d9,14065,43,36.68000000000001,152.0,74.08000000000001,cont,1.2952177664392865,109000000071,1,0,1,0
44b6_d78e09d9,14884,46,27.52,128.64000000000004,100.0,cont,4.246352294911479,112000000071,1,1,0,0
44b6_d78e09d9,25557,46,3.3478910160064697,90.5892352294922,123.10350097656251,cont,3.412059936703653,112000000060,1,1,0,0
44b6_d78e09d9,15154,47,27.52,127.36000000000001,104.0,cont,4.2626447922621935,113000000071,1,0,1,0
44b6_d78e09d9,25558,47,2.419202690124511,86.20810729980468,126.80683959960942,cont,3.4126611836594316,113000000060,1,0,1,0
44b6_d78e09d9,16091,51,5.4799999999999995,85.28,152.64,cont,1.2701014969284992,117000000060,1,1,0,0
44b6_d78e09d9,16331,52,6.16,84.64000000000001,157.28000000000003,cont,0.7597635322914519,118000000060,1,0,1,0
44b6_d78e09d9,16782,54,3.0,71.99999999999999,195.36,cont,1.7637801202247416,120000000059,1,1,0,0
44b6_d78e09d9,16789,54,5.160000000000002,71.99999999999999,204.00000000000003,cont,6.560272121833068,120000000059,1,1,0,0
44b6_d78e09d9,17006,55,0.9999999999999998,63.999999999999986,187.99999999999997,cont,5.629165124598864,121000000060,3,0,0,3
44b6_d78e09d9,17014,55,3.6399999999999997,71.99999999999999,198.56000000000003,cont,1.4707821048680227,121000000060,1,0,1,0
44b6_d78e09d9,17265,56,11.04,87.36,176.64000000000001,cont,0.3053097812059043,122000000061,1,0,1,0
44b6_d78e09d9,17656,57,61.04000000000001,160.64000000000001,194.08000000000004,cont,6.584779917544706,123000000013,3,0,0,3
44b6_d78e09d9,17916,59,13.68,86.72,189.92000000000002,cont,0.5332872701462131,125000000060,1,1,0,0
44b6_d78e09d9,18103,60,2.0,88.64,116.0,cont,1.6315679766715208,126000000040,1,0,1,0
44b6_d78e09d9,18129,60,14.16,85.28,195.35999999999996,cont,1.4036275547665729,126000000061,1,0,1,0
44b6_d78e09d9,18446,61,48.99999999999999,131.46666666666664,172.0,cont,5.6127880555873935,127000000071,2,0,0,2
44b6_d78e09d9,19154,65,4.640000000000001,88.00000000000001,132.64000000000001,cont,1.5481909806286842,131000000040,1,1,0,0
44b6_d78e09d9,19354,66,5.84,88.00000000000001,137.28000000000003,cont,0.7092524673908471,132000000040,1,0,1,0
44b6_d78e09d9,19755,68,14.480000000000002,74.72,245.44000000000003,cont,1.4048498585258076,134000000060,1,1,0,0
44b6_d78e09d9,19767,68,20.160000000000004,88.00000000000001,226.72,cont,0.7092524673908387,134000000061,1,1,0,0
44b6_d78e09d9,19817,68,42.64,128.0,198.72000000000003,cont,6.5211920162267925,134000000071,1,1,0,0
44b6_d78e09d9,19960,69,21.0,88.00000000000001,229.28000000000003,cont,0.6598780663880296,135000000061,1,0,1,0
44b6_d78e09d9,20126,70,9.679999999999998,88.00000000000001,153.27999999999997,cont,0.721800050221669,136000000040,1,0,1,0
44b6_d78e09d9,20849,74,2.32,71.36,216.0,cont,2.28253217392439,140000000041,1,0,1,0
44b6_d78e09d9,21567,78,4.0,71.99999999999999,224.0,cont,7.105427357601002e-15,144000000041,1,1,0,0
44b6_d78e09d9,21708,78,58.520000000000024,148.00000000000003,224.0,cont,0.9687508064512378,144000000068,1,1,0,0
44b6_d78e09d9,21760,79,4.5200000000000005,72.64,226.07999999999998,cont,0.794257711640245,145000000041,1,0,1,0
44b6_d78e09d9,21871,79,51.13333333333331,123.46666666666665,212.53333333333325,cont,0.3447145558284377,145000000068,3,0,0,3
44b6_d78e09d9,21896,79,58.680000000000014,148.00000000000003,226.72,cont,0.8401422870561661,145000000067,1,0,1,0
44b6_db3c847b,890,3,39.63999999999999,236.0,196.00000000000003,cont,0.7122247275263681,80000000027,1,1,0,0
44b6_db3c847b,1135,4,38.84,236.64000000000001,196.00000000000003,cont,1.9992783854431009,81000000027,1,0,1,0
44b6_db3c847b,2443,9,45.99999999999999,226.08,175.36,cont,6.551793647544174,86000000028,1,1,0,0
44b6_db3c847b,2690,10,45.68,221.28000000000003,172.64,cont,3.7745500559669325,87000000029,1,0,1,0
44b6_db3c847b,3151,12,32.68,231.84,181.28000000000003,cont,2.43825821540706,89000000028,2,2,0,0
44b6_db3c847b,3392,13,31.8,236.15999999999997,179.35999999999999,cont,2.8238086536626454,90000000028,2,0,2,0
44b6_db3c847b,4382,17,30.680000000000003,224.8,172.79999999999998,cont,6.8104562301287865,94000000028,5,0,1,4
44b6_db3c847b,5271,21,28.16,224.64000000000001,171.99999999999994,cont,6.839725186365311,98000000028,1,1,0,0
44b6_db3c847b,5483,22,23.999999999999993,224.53333333333327,171.46666666666664,cont,0.6340045784973435,99000000028,4,0,0,4
44b6_db3c847b,5497,22,28.119999999999997,225.76,172.0,cont,6.695709913817945,99000000028,1,0,1,0
44b6_db3c847b,6815,28,18.13333333333333,212.5333333333333,164.5333333333333,cont,3.1024310024559805,105000000028,4,0,0,4
44b6_db3c847b,7735,32,34.36,180.64000000000001,150.71999999999997,cont,1.1719168325866847,109000000029,1,1,0,0
44b6_db3c847b,7941,33,33.480000000000004,178.08,148.64000000000001,cont,0.963283771793122,110000000029,1,0,1,0
44b6_db3c847b,8151,34,33.00000000000001,174.55999999999995,147.35999999999996,cont,0.6504061231261328,111000000029,2,2,0,0
44b6_db3c847b,8347,35,32.480000000000004,172.64,145.27999999999997,cont,1.0159824371021349,112000000029,2,0,2,0
44b6_db3c847b,9268,40,13.360000000000001,185.28000000000003,154.71999999999997,cont,1.1192463982519605,117000000028,1,1,0,0
44b6_db3c847b,9455,41,13.320000000000002,181.91999999999996,153.27999999999997,cont,1.1113146775328722,118000000028,1,0,1,0
44b6_db3c847b,14998,77,29.0,19.36,67.36000000000001,cont,1.5014331695083811,154000000032,1,1,0,0
44b6_db3c847b,15126,78,26.266666666666655,20.533333333333328,63.999999999999986,cont,6.556740197223703,155000000032,2,0,0,2
44b6_db3c847b,15130,78,29.0,16.64,66.72,cont,1.8761538116583083,155000000032,1,0,1,0
44b6_ddf577ad,535,1,17.920000000000005,52.000000000000014,120.00000000000006,cont,0.917657513727222,141000000075,2,1,1,0
44b6_ddf577ad,544,1,19.266666666666662,55.99999999999999,147.46666666666667,cont,1.71504018822366,141000000083,4,0,0,4
44b6_ddf577ad,874,2,10.160000000000002,77.28,193.27999999999997,cont,1.5610998726859235,142000000073,1,1,0,0
44b6_ddf577ad,1136,2,48.49636657714845,16.804892272949218,112.41889038085938,cont,5.0514473827552555,142000000008,1,0,1,0
44b6_ddf577ad,1275,3,9.84,78.72000000000001,195.36,cont,1.3823010706788823,143000000073,1,0,1,0
44b6_ddf577ad,1572,3,53.8,46.08000000000001,183.36,cont,0.5593872205368948,143000000009,7,0,1,6
44b6_ddf577ad,1920,4,40.199999999999996,59.36,116.63999999999999,cont,5.978630957418263,144000000017,1,0,1,0
44b6_ddf577ad,1986,4,53.0,48.0,183.36,cont,1.9470185540975156,144000000009,1,1,0,0
44b6_ddf577ad,2998,7,15.0,55.99999999999999,156.0,cont,2.071476677397071,147000000081,1,1,0,0
44b6_ddf577ad,3627,8,52.67999999999999,68.0,181.27999999999997,cont,6.634658289995055,148000000009,1,1,0,0
44b6_ddf577ad,3956,9,38.316270751953134,64.48321228027345,119.78990234374999,cont,6.428888467177057,149000000017,1,1,0,0
44b6_ddf577ad,4405,10,41.0,75.36,115.35999999999999,cont,6.122649804210593,150000000017,2,2,0,0
44b6_ddf577ad,4429,10,45.16000000000001,60.000000000000014,120.0,cont,5.416542081900609,150000000017,3,3,0,0
44b6_ddf577ad,4808,11,38.133333333333326,72.0,119.99999999999997,cont,5.5029174711642375,151000000017,8,0,0,8
44b6_ddf577ad,4825,11,40.04,68.0,123.36000000000003,cont,1.7780203176566916,151000000017,8,0,1,7
44b6_ddf577ad,4826,11,41.0,76.95999999999998,116.0,cont,5.512463718701461,151000000017,2,0,2,0
44b6_ddf577ad,4858,11,46.999999999999986,47.999999999999986,131.46666666666664,cont,0.8408927960474272,151000000008,4,0,0,4
44b6_ddf577ad,5265,12,51.86666666666666,47.999999999999986,175.99999999999994,cont,6.626788608137786,152000000009,8,0,0,8
44b6_ddf577ad,5399,13,10.160000000000002,99.36000000000001,204.64000000000001,cont,0.7609790158079205,153000000070,1,1,0,0
44b6_ddf577ad,5569,13,33.0,100.0,127.46666666666667,cont,6.679707622614263,153000000018,7,0,0,7
44b6_ddf577ad,5610,13,38.839999999999996,80.0,120.64000000000001,cont,5.516318547047479,153000000017,5,0,0,5
44b6_ddf577ad,5623,13,41.0,71.36,126.08,cont,0.14981759743100764,153000000017,1,0,1,0
44b6_ddf577ad,5658,13,46.839999999999996,44.64000000000001,141.28000000000003,cont,5.939280522504393,153000000008,1,0,1,0
44b6_ddf577ad,5823,14,10.0,100.00000000000001,206.08,cont,1.219183256323668,154000000070,1,0,1,0
44b6_ddf577ad,6139,14,53.67999999999999,60.000000000000014,180.0,cont,2.597408865099997,154000000009,1,1,0,0
44b6_ddf577ad,6530,15,53.120000000000005,60.000000000000014,179.36,cont,1.735025215955088,155000000009,1,0,1,0
44b6_ddf577ad,6543,15,55.133333333333326,59.999999999999986,180.53333333333327,cont,2.2607689817601395,155000000009,5,0,0,5
44b6_ddf577ad,7467,18,14.000000000000002,63.2,161.44,cont,1.6368195150962694,158000000079,1,1,0,0
44b6_ddf577ad,8871,21,38.31999999999999,95.36000000000001,124.8,cont,6.027961538737634,161000000017,1,0,1,0
44b6_ddf577ad,9248,22,34.99999999999999,111.35999999999997,130.72,cont,5.41895472508305,162000000018,1,1,0,0
44b6_ddf577ad,9635,23,33.84000000000001,112.0,120.64000000000001,cont,1.175965693589742,163000000018,1,0,1,0
44b6_ddf577ad,9641,23,34.999999999999986,112.5333333333333,128.53333333333327,cont,4.581248105198909,163000000018,1,0,1,0
44b6_ddf577ad,10526,25,38.15999999999999,103.36,131.36,cont,6.337354164988741,165000000017,1,0,1,0
44b6_ddf577ad,11073,26,53.31999999999999,62.56,180.80000000000004,cont,1.7441330224498173,166000000009,1,1,0,0
44b6_ddf577ad,11414,27,38.67999999999999,106.08000000000001,132.64000000000004,cont,5.568061966474522,167000000017,1,0,1,0
44b6_ddf577ad,11495,27,52.48,65.44,179.20000000000005,cont,1.1170028256454911,167000000009,1,0,1,0
44b6_ddf577ad,11511,27,55.99999999999999,63.999999999999986,179.99999999999997,cont,6.66299540841053,167000000009,7,1,0,6
44b6_ddf577ad,13576,32,35.0,131.36,129.92000000000002,cont,0.1498175974310165,172000000018,2,0,2,0
44b6_ddf577ad,14528,34,43.839999999999996,98.08000000000001,138.72000000000003,cont,4.422986123649939,174000000018,2,1,1,0
44b6_ddf577ad,14540,34,45.36,78.08,144.16000000000003,cont,2.2245887962722435,174000000008,5,1,0,4
44b6_ddf577ad,14950,35,41.133333333333326,108.53333333333333,140.5333333333334,cont,0.3752776749732641,175000000018,4,0,0,4
44b6_ddf577ad,14973,35,43.68,98.08000000000001,138.72000000000003,cont,5.95628449622749,175000000018,1,0,1,0
44b6_ddf577ad,15006,35,49.160000000000004,58.08000000000001,180.64000000000001,cont,6.3612264196851145,175000000009,2,2,0,0
44b6_ddf577ad,15142,36,7.000000000000001,110.72000000000001,204.64,cont,2.398182905868517,176000000067,1,0,1,0
44b6_ddf577ad,15989,38,5.866666666666664,114.4,203.99999999999991,cont,1.0628267471439037,178000000067,1,0,1,0
44b6_ddf577ad,15992,38,6.999999999999999,100.0,203.99999999999994,cont,6.749131888620939,178000000067,4,0,0,4
44b6_ddf577ad,16695,39,45.35999999999999,82.72000000000001,150.07999999999998,cont,2.927210969933674,179000000008,1,0,1,0
44b6_ddf577ad,19745,46,48.16,73.92,171.99999999999994,cont,0.8537051598766104,186000000009,1,1,0,0
44b6_ddf577ad,20657,48,46.519999999999996,101.43999999999998,150.07999999999998,cont,6.75556495139377,188000000008,1,1,0,0
44b6_ddf577ad,21450,50,31.320000000000004,135.36000000000004,116.63999999999999,cont,5.6088635212492175,190000000019,1,1,0,0
44b6_ddf577ad,21957,51,40.68,92.16,132.64000000000001,cont,5.343285652339769,191000000018,1,0,1,0
44b6_ddf577ad,21972,51,43.52,65.76000000000002,149.92000000000002,cont,6.050676098586006,191000000008,6,1,0,5
44b6_ddf577ad,21989,51,46.16,78.72000000000001,188.64000000000001,cont,6.575951570875513,191000000009,1,1,0,0
44b6_ddf577ad,21994,51,47.160000000000004,66.72,178.72000000000003,cont,1.46511571641969,191000000009,1,0,1,0
44b6_ddf577ad,23428,54,46.32,61.920000000000016,181.44000000000003,cont,0.9298067675597863,194000000009,2,2,0,0
44b6_ddf577ad,23891,55,38.0,101.92000000000002,141.76000000000002,cont,3.5663104621583472,195000000018,5,3,2,0
44b6_ddf577ad,23905,55,39.919999999999995,84.00000000000001,126.87999999999998,cont,6.7427318508524365,195000000018,7,5,2,0
44b6_ddf577ad,23928,55,41.24000000000001,80.0,147.35999999999999,cont,2.941070436507754,195000000008,5,4,1,0
44b6_ddf577ad,23932,55,43.52,74.55999999999999,132.64000000000001,cont,6.084122394602197,195000000008,4,4,0,0
44b6_ddf577ad,24406,56,35.68,96.63999999999999,157.6,cont,6.221437233469448,196000000018,9,0,2,7
44b6_ddf577ad,24433,56,40.12,84.0,145.92,cont,5.4612572957149705,196000000018,9,0,4,5
44b6_ddf577ad,24442,56,39.040000000000006,104.64000000000001,141.92000000000002,cont,4.806155558239869,196000000018,8,0,3,5
44b6_ddf577ad,24448,56,41.00000000000001,74.07999999999998,167.04000000000002,cont,4.75794894098287,196000000008,9,0,1,8
44b6_ddf577ad,24482,56,45.84,62.56,180.79999999999995,cont,0.8407706658179775,196000000009,1,0,1,0
44b6_ddf577ad,25266,58,30.133333333333326,108.53333333333333,127.99999999999997,cont,5.760097188247397,198000000019,4,0,0,4
44b6_ddf577ad,25354,58,41.0,76.0,158.71999999999997,cont,2.0344325068677023,198000000008,1,0,1,0
44b6_ddf577ad,27359,63,20.000000000000007,41.92000000000001,152.0,cont,1.1862500000000047,203000000055,1,1,0,0
44b6_ddf577ad,27436,63,29.0,116.0,128.0,cont,4.958913218891817,203000000019,2,0,2,0
44b6_ddf577ad,27886,64,29.479999999999997,115.83999999999999,128.0,cont,5.761765137525125,204000000019,1,1,0,0
44b6_ddf577ad,28322,65,30.0,108.0,143.99999999999997,cont,6.918188211157605,205000000018,3,1,2,0
44b6_ddf577ad,28709,66,23.662328186035158,120.47310546875002,139.060498046875,cont,5.243140379971987,206000000019,1,1,0,0
44b6_ddf577ad,30548,70,32.52,92.0,143.99999999999997,cont,0.779999999999994,210000000018,1,0,1,0
44b6_ddf577ad,35443,81,20.320000000000004,121.92000000000002,133.27999999999997,cont,6.369751270654128,221000000019,1,0,1,0
44b6_ddf577ad,36920,84,28.679999999999993,95.36000000000001,143.36,cont,1.2199410846430192,224000000018,1,0,1,0
44b6_ddf577ad,39245,89,27.679999999999996,98.08000000000001,143.36,cont,1.0881434877809102,229000000018,1,1,0,0
44b6_ddf577ad,39647,90,20.32,124.63999999999999,134.72000000000003,cont,3.5880500449408514,230000000019,7,0,1,6
44b6_ddf577ad,39706,90,27.679999999999996,98.08000000000001,143.36,cont,1.3034485509601028,230000000018,1,0,1,0
44b6_ddf577ad,41229,93,39.266666666666666,80.53333333333332,135.99999999999994,cont,6.926400342269359,233000000008,8,0,0,8
44b6_ddf577ad,41519,94,18.266666666666666,127.46666666666667,123.99999999999999,cont,4.934818188703167,234000000018,5,1,0,4
44b6_ddf577ad,41984,95,20.52,120.64000000000001,136.64,cont,4.3075068920432305,235000000018,6,0,0,6
44b6_deabac95,4045,16,22.519999999999996,112.64,233.92000000000007,cont,2.5063485018049696,108000000018,1,1,0,0
44b6_deabac95,4292,17,23.360000000000007,115.36000000000001,228.00000000000003,cont,1.1464026615897083,109000000018,1,0,1,0
44b6_deabac95,7898,33,28.36,132.64000000000001,162.72,cont,1.0902042526517677,125000000019,1,1,0,0
44b6_deabac95,8110,34,29.0,133.28,156.00000000000006,cont,0.9646534351776487,126000000020,1,0,1,0
44b6_deabac95,8511,36,27.64,124.64000000000001,185.44,cont,6.822542712398368,128000000019,1,0,1,0
44b6_deabac95,8520,36,30.133333333333333,119.99999999999997,193.0666666666666,cont,1.912206086711235,128000000019,1,0,0,1
44b6_deabac95,10659,47,28.0,128.0,80.0,cont,5.450415695155737,139000000021,1,1,0,0
44b6_deabac95,13068,62,20.16,152.64,38.72,cont,0.9731023199026918,154000000021,1,1,0,0
44b6_deabac95,13211,63,20.16,154.08,38.720000000000006,cont,0.9625982092752902,155000000021,1,0,1,0
44b6_deabac95,13492,65,25.519999999999996,142.72,76.47999999999999,cont,1.3072920484727215,157000000020,1,1,0,0
44b6_deabac95,13618,66,25.16,143.36,70.71999999999998,cont,1.1212500000000103,158000000020,1,0,1,0
44b6_deabac95,17186,67,18.9996656036377,155.2315808105469,24.084419631958006,cont,1.6868084161493637,159000000021,1,1,0,0
44b6_deabac95,13868,68,18.319851379394528,155.3029248046875,17.95751983642578,cont,1.1781419511655904,160000000021,1,0,1,0
44b6_deabac95,14887,76,24.520000000000003,144.64000000000001,40.000000000000014,cont,1.0386025767828573,168000000020,1,1,0,0
44b6_deabac95,15001,77,23.840000000000003,145.27999999999997,35.99999999999999,cont,0.8490527810448582,169000000020,1,0,1,0
44b6_e28840c6,1532,3,47.32000000000001,122.08000000000003,170.56000000000003,cont,0.5508246658420529,361000000009,1,1,0,0
44b6_e28840c6,1842,4,32.0,143.36,124.00000000000003,cont,0.7803384682302779,362000000008,1,1,0,0
44b6_e28840c6,2257,5,32.160000000000004,142.72,124.63999999999999,cont,0.31926184394632257,363000000008,1,0,1,0
44b6_e28840c6,3950,9,32.000000000000014,142.72,124.63999999999999,cont,0.18527850657861902,367000000008,1,1,0,0
44b6_e28840c6,3951,9,32.32,156.00000000000006,119.36000000000001,cont,5.780296542998481,367000000008,5,0,0,5
44b6_e28840c6,34193,9,32.9099787902832,156.04304992675782,118.90711334228514,cont,6.032414465769364,367000000008,1,1,0,0
44b6_e28840c6,4360,10,32.839999999999996,153.91999999999996,117.28,cont,6.628904627651526,368000000008,5,0,0,5
44b6_e28840c6,4436,10,47.00000000000001,120.00000000000003,173.28000000000003,cont,0.8635464666131274,368000000009,1,1,0,0
44b6_e28840c6,4815,11,47.16000000000001,120.00000000000003,174.08,cont,0.48342043037093657,369000000009,1,0,1,0
44b6_e28840c6,10554,26,47.480000000000004,97.28,168.0,cont,1.1777607195436597,384000000010,2,1,1,0
44b6_e28840c6,10878,27,48.0,98.56,168.0,cont,1.8255773915394549,385000000010,1,0,1,0
44b6_e28840c6,13198,34,48.99999999999999,111.99999999999999,184.5333333333333,cont,1.1064461533566639,392000000010,1,0,0,1
44b6_e28840c6,13213,34,52.16000000000002,96.0,186.07999999999998,cont,6.424095875101829,392000000011,1,0,1,0
44b6_e28840c6,13479,35,33.160000000000004,127.36,128.0,cont,2.6879877464378468,393000000009,1,1,0,0
44b6_e28840c6,13815,36,33.32,126.72000000000001,128.0,cont,2.6526849605635423,394000000009,1,0,1,0
44b6_e28840c6,13816,36,33.0,143.46666666666664,135.46666666666664,cont,6.881938417981428,394000000009,5,0,0,5
44b6_e28840c6,14496,38,52.06705810546875,95.99493530273438,187.0715869140625,cont,5.391318029710572,396000000011,1,1,0,0
44b6_e28840c6,14815,39,51.74705810546876,95.9949353027344,187.0715869140625,cont,5.0309369984643375,397000000011,1,0,1,0
44b6_e28840c6,15767,42,47.480000000000004,108.8,180.0,cont,1.186250000000004,400000000010,6,2,1,3
44b6_e28840c6,15775,42,48.96000000000001,98.72,181.92000000000002,cont,1.646711211020332,400000000011,5,1,0,4
44b6_e28840c6,16100,43,47.160000000000004,110.08,180.0,cont,1.4611452058231602,401000000010,2,0,2,0
44b6_e28840c6,16104,43,48.16,99.36000000000001,180.64000000000001,cont,0.823475864855795,401000000011,1,0,1,0
44b6_e28840c6,16452,44,47.00000000000001,111.36,180.0,cont,0.259999999999998,402000000010,1,1,0,0
44b6_e28840c6,16456,44,47.84,99.36000000000001,180.64000000000001,cont,0.627889570306757,402000000011,1,1,0,0
44b6_e28840c6,16780,45,47.64000000000001,98.72000000000001,181.28000000000003,cont,0.7164757497640719,403000000011,1,0,1,0
44b6_e28840c6,16781,45,46.839999999999996,110.72000000000003,180.0,cont,1.4606933285258663,403000000010,1,0,1,0
44b6_e28840c6,17492,47,47.52000000000001,97.28,182.07999999999998,cont,0.8532410635336434,405000000011,1,1,0,0
44b6_e28840c6,17820,48,47.480000000000004,97.44,181.28000000000006,cont,1.06235734336426,406000000011,4,0,1,3
44b6_e28840c6,17840,48,49.86666666666667,91.99999999999999,187.46666666666664,cont,5.396180008168326,406000000011,5,0,0,5
44b6_e28840c6,20024,55,30.000000000000007,132.00000000000006,140.00000000000003,cont,5.859020822628986,413000000009,1,1,0,0
44b6_e28840c6,20033,55,32.64,131.35999999999999,129.92000000000002,cont,1.3257450735341292,413000000009,1,1,0,0
44b6_e28840c6,20315,56,28.16,146.72000000000003,116.0,cont,6.671015899021684,414000000009,1,0,1,0
44b6_e28840c6,20328,56,30.000000000000007,132.00000000000006,140.00000000000003,cont,6.9181882111576005,414000000009,1,0,1,0
44b6_e28840c6,20338,56,30.999999999999996,139.99999999999994,123.99999999999999,cont,0.5745242597140598,414000000009,6,0,0,6
44b6_e28840c6,20341,56,32.64,130.72,132.00000000000003,cont,5.148199017617724,414000000009,1,0,1,0
44b6_e28840c6,21590,60,51.00000000000001,91.36,196.64000000000001,cont,6.5493144917999,418000000011,2,2,0,0
44b6_e28840c6,21878,61,46.16,105.43999999999998,183.36,cont,0.8800541531633228,419000000010,3,0,0,3
44b6_e28840c6,21882,61,47.64,102.40000000000003,186.56000000000006,cont,3.4668909432660397,419000000011,4,0,0,4
44b6_e28840c6,21901,61,51.00000000000001,90.72,196.64,cont,6.393605814601028,419000000011,2,0,2,0
44b6_e28840c6,22108,62,27.999999999999996,150.07999999999998,136.0,daughter,6.433214714472071,420000000010,1,0,1,0
44b6_e28840c6,22180,62,46.0,105.28,182.55999999999997,cont,0.7348522385486704,420000000011,1,1,0,0
44b6_e28840c6,22182,62,47.48,98.56,185.27999999999997,cont,2.6527347327805004,420000000012,1,1,0,0
44b6_e28840c6,22462,63,46.0,105.28000000000002,183.36,cont,0.39135182381074124,421000000011,1,0,1,0
44b6_e28840c6,22467,63,47.32000000000001,94.72,184.64,cont,1.6212581919607962,421000000012,1,0,1,0
44b6_e28840c6,22757,64,46.559999999999995,113.76000000000002,168.96,cont,6.395526024104667,422000000011,1,1,0,0
44b6_e28840c6,22764,64,47.68000000000001,93.28000000000003,185.28,cont,1.5668404712988655,422000000012,1,1,0,0
44b6_e28840c6,22999,65,33.400000000000006,122.56,134.56000000000003,cont,1.1987610791980114,423000000009,3,0,0,3
44b6_e28840c6,23050,65,47.84,92.63999999999999,186.72000000000006,cont,1.512298023704301,423000000012,1,0,1,0
44b6_e28840c6,23908,68,45.68,104.0,181.91999999999996,cont,1.188918232049631,426000000011,1,1,0,0
44b6_e28840c6,23914,68,47.32,90.72,188.00000000000003,cont,1.6490347555463998,426000000012,1,1,0,0
44b6_e28840c6,24148,69,30.320000000000007,128.64000000000004,128.64000000000004,cont,0.8020325741514489,427000000010,1,0,1,0
44b6_e28840c6,24205,69,45.320000000000014,104.0,182.72,cont,0.8401422870561993,427000000011,1,0,1,0
44b6_e28840c6,24213,69,47.160000000000004,89.27999999999999,188.00000000000003,cont,0.9018384001582445,427000000012,1,0,1,0
44b6_e28840c6,24227,69,50.0,91.46666666666664,200.0,cont,6.373738982034886,427000000012,3,0,0,3
44b6_e28840c6,24438,70,30.0,128.0,128.0,cont,1.8168052317185792,428000000010,1,1,0,0
44b6_e28840c6,24455,70,33.000000000000014,116.0,136.0,cont,1.7235727791422297,428000000009,1,1,0,0
44b6_e28840c6,24739,71,29.68,128.0,128.0,cont,1.4304615732343156,429000000010,1,0,1,0
44b6_e28840c6,24747,71,30.999999999999996,130.93333333333334,139.99999999999994,cont,5.734763932465843,429000000009,4,0,0,4
44b6_e28840c6,24754,71,33.000000000000014,116.0,136.0,cont,1.816805231718598,429000000009,2,1,1,0
44b6_e28840c6,25041,72,33.000000000000014,116.0,136.0,cont,1.6750116604071952,430000000009,1,0,1,0
44b6_e28840c6,25338,73,31.000000000000007,131.36,140.00000000000003,cont,6.573843054104674,431000000010,1,1,0,0
44b6_e28840c6,25347,73,33.000000000000014,115.36000000000004,136.0,cont,0.9825540697590224,431000000009,1,1,0,0
44b6_e28840c6,25956,75,28.320000000000004,126.56000000000002,125.92,cont,0.568517809747422,433000000010,1,1,0,0
44b6_e28840c6,25977,75,32.839999999999996,113.92,137.28,cont,1.5251644378557967,433000000009,3,3,0,0
44b6_e28840c6,26266,76,28.16,127.36000000000001,126.72000000000001,cont,0.3848883766756192,434000000010,1,0,1,0
44b6_e28840c6,26286,76,32.68,113.92,138.72000000000003,cont,1.379049718103016,434000000009,3,0,3,0
44b6_e28840c6,27652,80,32.32,112.64,141.28000000000003,cont,2.152987023881012,438000000009,1,1,0,0
44b6_e28840c6,27708,80,46.32,88.00000000000001,188.00000000000003,cont,0.659878066388014,438000000012,1,1,0,0
44b6_e28840c6,27988,81,28.16,124.63999999999999,128.64000000000001,cont,2.0079109168486595,439000000011,1,0,1,0
44b6_e28840c6,28007,81,33.999999999999986,123.99999999999999,135.99999999999994,cont,6.345827861674149,439000000010,10,0,0,10
44b6_e28840c6,28063,81,44.68,101.27999999999999,184.64000000000004,cont,0.813149740207778,439000000012,1,0,1,0
44b6_e28840c6,28071,81,45.99999999999999,116.0,191.99999999999994,cont,6.397631397634582,439000000012,2,0,0,2
44b6_e28840c6,29395,85,45.0,102.72000000000001,186.72000000000003,cont,0.31383962863220155,443000000012,1,1,0,0
44b6_e28840c6,29719,86,45.0,102.56,186.55999999999997,cont,0.673346354040768,444000000012,1,0,1,0
44b6_e28840c6,30374,88,45.160000000000004,102.72,187.36,cont,0.4177862042959335,446000000012,1,1,0,0
44b6_e28840c6,30711,89,45.32000000000001,102.72000000000001,188.00000000000003,cont,0.5966206918972999,447000000012,1,0,1,0
44b6_e28840c6,30716,89,47.480000000000004,88.00000000000001,191.36,cont,1.0258114166356258,447000000013,1,0,1,0
44b6_e28840c6,31073,90,47.68000000000001,88.00000000000001,192.64,cont,1.5161345792837824,448000000013,1,1,0,0
44b6_e28840c6,31392,91,45.68000000000001,101.28000000000002,189.28000000000003,cont,0.7914267180225711,449000000012,1,1,0,0
44b6_e28840c6,31399,91,47.52000000000001,88.00000000000001,193.27999999999997,cont,0.9268200000539458,449000000013,2,1,1,0
44b6_e28840c6,31705,92,45.68,100.64000000000003,190.72000000000003,cont,0.5520218519225377,450000000012,1,0,1,0
44b6_e29f0176,6742,32,2.3600000000000003,48.0,244.00000000000006,cont,0.7122247275263457,53000000014,1,1,0,0
44b6_e29f0176,6957,33,3.4800000000000013,50.080000000000005,242.72000000000003,cont,1.8063832130807638,54000000014,1,0,1,0
44b6_e29f0176,7857,37,10.266666666666667,17.86666666666666,251.9999999999999,cont,1.7405124764148223,58000000014,1,0,1,0
44b6_e29f0176,8082,38,12.040000000000001,45.120000000000005,234.72000000000006,cont,2.2908465956933908,59000000014,5,0,1,4
44b6_e29f0176,8535,40,14.719999999999999,55.52000000000001,225.92,cont,4.294952160676531,61000000014,4,4,0,0
44b6_e29f0176,8762,41,20.160000000000004,84.00000000000001,186.55999999999997,cont,0.796415799064784,62000000014,3,0,3,0
44b6_e29f0176,8967,42,21.479999999999997,90.55999999999999,179.35999999999999,cont,1.1920236207810626,63000000014,1,1,0,0
44b6_e29f0176,9177,43,22.199999999999996,94.07999999999998,173.91999999999996,cont,3.5353741403845795,64000000014,1,0,1,0
44b6_e29f0176,9375,44,22.879999999999995,98.88000000000001,169.76,cont,4.4647300100342076,65000000014,1,1,0,0
44b6_e29f0176,9567,45,25.32,106.72,148.16000000000003,cont,2.619679429338627,66000000014,4,0,1,3
44b6_e29f0176,10129,48,28.68,122.71999999999998,137.28,cont,0.5443143622944416,69000000015,1,1,0,0
44b6_e29f0176,10306,49,30.0,124.64000000000001,132.64000000000001,cont,1.638109619347862,70000000015,1,0,1,0
44b6_e29f0176,12565,64,45.320000000000014,166.07999999999998,71.36,cont,0.5822853681830122,85000000017,1,1,0,0
44b6_e29f0176,12571,64,48.64,166.07999999999998,79.36,cont,6.627848915749369,85000000017,1,1,0,0
44b6_e29f0176,12689,65,46.16,167.35999999999999,68.00000000000001,cont,1.7828405179656508,86000000017,1,0,1,0
44b6_e29f0176,12698,65,48.84,167.35999999999999,78.72,cont,5.06783808073029,86000000017,1,0,1,0
44b6_e29f0176,13038,68,49.48,173.28000000000003,63.2,cont,1.3353703840133682,89000000018,1,1,0,0
44b6_e29f0176,13045,68,50.32,170.07999999999998,76.32000000000001,cont,6.293954564699369,89000000018,2,2,0,0
44b6_e29f0176,13143,69,50.0,181.44,47.36,cont,2.1908592064530343,90000000018,4,1,0,3
44b6_e29f0176,13145,69,51.16,179.04000000000002,62.56,cont,6.396661360623987,90000000018,4,0,1,3
44b6_e29f0176,13241,70,52.2,186.72000000000003,44.80000000000001,cont,1.7311398881950573,91000000018,1,0,1,0
44b6_e29f0176,13878,77,59.84,201.28000000000003,14.080000000000002,cont,0.728355382007991,98000000018,1,1,0,0
44b6_e29f0176,13961,78,60.16000000000001,201.28000000000006,11.360000000000003,cont,1.47696320621062,99000000018,1,0,1,0
44b6_e31261b4,14700,18,58.999999999999986,167.36,40.000000000000014,cont,0.4317732333065685,135000000006,1,1,0,0
44b6_e31261b4,15481,19,58.0,167.36000000000004,40.63999999999999,cont,1.1035652450127038,136000000006,2,1,1,0
44b6_e31261b4,16272,20,56.99999999999999,167.36000000000004,41.28,cont,0.28379404944428943,137000000006,1,0,1,0
44b6_e31261b4,17915,22,57.84,193.27999999999997,129.28,cont,6.208138056212982,139000000047,1,1,0,0
44b6_e31261b4,17991,22,61.99999999999999,194.56000000000006,106.71999999999998,cont,6.195705417868094,139000000047,1,0,1,0
44b6_e31261b4,19579,24,60.13333333333334,186.93333333333334,127.46666666666667,cont,2.0810153771656816,141000000047,5,1,0,4
44b6_e31261b4,20400,25,58.839999999999996,182.23999999999998,128.95999999999998,cont,1.9562867408690443,142000000047,2,1,1,0
44b6_e31261b4,20417,25,60.16000000000001,193.92000000000002,119.36000000000001,cont,6.1993483780555465,142000000047,1,1,0,0
44b6_e31261b4,20456,25,61.99999999999999,180.53333333333327,132.53333333333333,cont,3.5532879189170963,142000000047,6,1,0,5
44b6_e31261b4,21140,26,51.32,156.64000000000004,57.44000000000001,cont,0.7794920220887293,143000000006,1,1,0,0
44b6_e31261b4,21225,26,57.32000000000001,179.99999999999997,131.36,cont,2.8393358598446827,143000000047,1,0,1,0
44b6_e31261b4,21250,26,58.999999999999986,194.08,123.35999999999999,cont,6.600981224408991,143000000047,1,0,1,0
44b6_e31261b4,21307,26,61.760000000000005,182.07999999999998,134.72,cont,4.560260751864976,143000000047,1,0,1,0
44b6_e31261b4,21985,27,50.519999999999996,155.35999999999999,59.36,cont,0.9669774105427689,144000000006,1,0,1,0
44b6_e31261b4,22064,27,56.2,177.92000000000002,133.28,cont,1.5546588251124398,144000000047,1,1,0,0
44b6_e31261b4,22077,27,58.0,192.64000000000004,125.28000000000002,cont,6.232568817309615,144000000047,1,1,0,0
44b6_e31261b4,22145,27,61.04000000000001,180.79999999999995,136.0,cont,6.685073040924858,144000000047,1,1,0,0
44b6_e31261b4,22856,28,54.64,177.28,136.64,cont,2.366091516087228,145000000047,1,0,1,0
44b6_e31261b4,22919,28,59.800000000000004,178.55999999999997,137.92000000000002,cont,6.373336179937485,145000000047,1,0,1,0
44b6_e31261b4,23699,29,55.99999999999999,190.72000000000003,128.64000000000001,cont,6.827030947820588,146000000047,1,1,0,0
44b6_e31261b4,24422,30,47.84,150.72,66.08,cont,1.41980742532219,147000000006,2,2,0,0
44b6_e31261b4,24550,30,57.84,174.72,142.08,cont,6.391623057760846,147000000047,1,1,0,0
44b6_e31261b4,25240,31,46.99999999999999,148.64000000000001,68.0,cont,1.695070518444574,148000000006,2,0,2,0
44b6_e31261b4,26099,32,48.480000000000004,179.99999999999997,155.35999999999999,cont,6.1107346632790405,149000000047,1,1,0,0
44b6_e31261b4,26927,33,48.0,178.56000000000003,156.0,cont,6.888945968905549,150000000047,1,0,1,0
44b6_e31261b4,29346,36,43.47999999999999,142.72,76.64000000000001,cont,0.8017032649303475,153000000006,1,1,0,0
44b6_e31261b4,30158,37,43.16000000000001,141.92000000000002,80.0,cont,0.6520280860515157,154000000006,1,0,1,0
44b6_e31261b4,30973,38,40.99999999999999,179.46666666666664,172.0,cont,6.966736754531819,155000000047,7,0,0,7
44b6_e31261b4,35049,43,39.99999999999999,133.27999999999997,106.72,cont,0.7575010313524422,160000000006,1,1,0,0
44b6_e31261b4,35843,44,38.839999999999996,132.64000000000001,110.08,cont,0.9939103644192432,161000000006,1,0,1,0
44b6_e31261b4,40519,50,33.0,128.64000000000004,126.72000000000001,cont,1.7258693461557297,167000000006,1,1,0,0
44b6_e31261b4,41280,51,31.840000000000003,127.36000000000001,128.64000000000001,cont,1.3972165410200448,168000000006,1,0,1,0
44b6_e31261b4,41292,51,32.86666666666666,141.06666666666666,128.53333333333333,cont,6.116817451025212,168000000006,6,0,0,6
44b6_e31261b4,42808,53,29.840000000000003,141.92000000000002,131.36000000000004,cont,6.643468244260682,170000000006,1,1,0,0
44b6_e31261b4,43549,54,28.8,124.00000000000003,136.64,cont,1.5549135828077338,171000000006,1,1,0,0
44b6_e31261b4,44314,55,28.840000000000003,124.00000000000003,140.64,cont,1.572308215649853,172000000006,1,0,1,0
44b6_e31261b4,46604,58,27.880000000000003,122.71999999999998,154.08000000000004,cont,1.5001135373697658,175000000006,1,1,0,0
44b6_e31261b4,47324,59,26.639999999999997,121.28000000000003,157.28,cont,1.397311033735865,176000000006,1,0,1,0
44b6_e31261b4,54549,68,61.92000000000001,207.36000000000007,50.72000000000002,cont,1.5728959159778877,185000000035,1,0,1,0
44b6_e31261b4,54555,68,62.000000000000014,228.16000000000003,51.36000000000001,cont,6.976096799966308,185000000035,1,0,1,0
44b6_e31261b4,56173,71,14.68,101.92000000000002,183.36,cont,1.2491378276635394,188000000006,1,1,0,0
44b6_e31261b4,56895,72,14.200000000000003,100.00000000000001,184.0,cont,2.1512078729169843,189000000006,1,0,1,0
44b6_e31261b4,59009,75,11.840000000000003,92.64,185.92000000000002,cont,1.578928137218416,192000000006,2,2,0,0
44b6_e31261b4,59503,75,58.839999999999996,197.28,54.56,cont,1.6592515255379428,192000000034,1,1,0,0
44b6_e31261b4,59676,76,11.479999999999999,89.92,189.28000000000003,cont,1.2113609340324687,193000000006,2,0,2,0
44b6_e31261b4,60173,76,58.31999999999999,195.36,55.2,cont,2.1515147280462488,193000000034,1,0,1,0
44b6_e31261b4,61017,78,12.48,94.4,196.80000000000004,cont,3.3668886323280676,195000000006,4,4,0,0
44b6_e31261b4,61018,78,11.360000000000003,98.08000000000001,189.27999999999997,cont,6.34892719776343,195000000006,5,5,0,0
44b6_e31261b4,61643,79,8.879999999999999,71.36,201.43999999999997,cont,1.2609204227468171,196000000006,7,0,3,4
44b6_e31261b4,65437,85,5.320000000000001,62.71999999999999,207.36000000000004,cont,0.5520218519225566,202000000006,1,1,0,0
44b6_e31261b4,66073,86,5.32,60.640000000000015,207.20000000000005,cont,0.7276299282602449,203000000006,1,0,1,0
44b6_e31261b4,67881,89,5.160000000000001,55.36,216.64000000000001,cont,1.3805807202043616,206000000006,1,1,0,0
44b6_e31261b4,68493,90,5.200000000000001,52.64000000000001,220.0,cont,0.5404193973202667,207000000006,1,0,1,0
44b6_e31261b4,71077,94,0.16000000000000003,32.00000000000001,216.64,cont,6.0809966545378105,211000000006,1,1,0,0
44b6_e31261b4,71110,94,3.3200000000000003,39.36,222.72,cont,2.570840390319865,211000000006,1,1,0,0
44b6_e31261b4,71715,95,0.0,30.4,220.64,cont,3.9128771783049916,212000000006,1,0,1,0
44b6_e31261b4,71743,95,3.1599999999999993,34.720000000000006,221.44000000000005,cont,4.0233437974525526,212000000006,2,1,1,0
44b6_e35b117d,429,1,61.68000000000001,101.28000000000003,15.680000000000001,cont,6.400458113486876,122000000083,1,1,0,0
44b6_e35b117d,2244,9,60.8,98.72000000000001,43.839999999999996,cont,2.0060689170614205,130000000082,1,1,0,0
44b6_e35b117d,2460,10,57.0,103.99999999999999,44.533333333333324,cont,3.3340012351694024,131000000081,4,0,0,4
44b6_e35b117d,2476,10,60.84000000000001,97.92000000000002,49.28,cont,4.439373900112503,131000000081,1,0,1,0
44b6_e35b117d,3161,13,56.160000000000004,104.0,55.36,cont,3.0012830589599457,134000000081,1,1,0,0
44b6_e35b117d,3403,14,55.68000000000001,104.0,58.72,cont,2.207130490025436,135000000081,1,0,1,0
44b6_e35b117d,3414,14,60.26666666666668,108.53333333333333,55.99999999999998,cont,5.848996827567024,135000000081,3,0,0,3
44b6_e35b117d,4312,18,53.31999999999999,106.56000000000002,75.84,cont,2.792731637662325,139000000080,1,1,0,0
44b6_e35b117d,4549,19,53.36000000000001,109.91999999999999,82.07999999999997,cont,2.4390703275018355,140000000079,1,0,1,0
44b6_e35b117d,4797,20,55.99999999999999,108.64000000000001,96.0,cont,5.034380082244471,141000000079,1,1,0,0
44b6_e35b117d,5040,21,55.0,108.64000000000001,96.64000000000001,cont,6.643388748409954,142000000078,1,0,1,0
44b6_e35b117d,5314,22,49.199999999999996,126.4,99.84000000000003,cont,4.822336376177014,143000000078,2,0,0,2
44b6_e35b117d,5543,23,38.68000000000001,74.72,4.000000000000001,cont,1.2212391248236456,144000000002,1,1,0,0
44b6_e35b117d,5591,23,49.36000000000001,128.0,99.36,cont,4.516564651369446,144000000078,1,1,0,0
44b6_e35b117d,5813,24,38.0,75.36,6.7200000000000015,cont,1.7258693461557282,145000000002,1,0,1,0
44b6_e35b117d,5854,24,47.73333333333332,115.46666666666664,103.46666666666663,cont,0.7679450392660491,145000000077,4,0,0,4
44b6_e35b117d,5856,24,49.000000000000014,129.44000000000003,100.00000000000001,cont,5.754335865241111,145000000077,1,0,1,0
44b6_e35b117d,6380,26,37.160000000000004,76.63999999999999,11.36,cont,1.9084585566629464,147000000002,1,1,0,0
44b6_e35b117d,6423,26,46.0,118.72000000000003,108.64000000000001,cont,0.3913518238107613,147000000077,1,1,0,0
44b6_e35b117d,6425,26,46.67999999999999,132.64000000000001,104.64000000000001,cont,6.201371320119447,147000000077,1,1,0,0
44b6_e35b117d,6666,27,36.84000000000001,77.44000000000001,13.280000000000001,cont,1.4073850974413669,148000000002,2,1,1,0
44b6_e35b117d,6707,27,45.00000000000001,118.07999999999998,108.16,cont,2.8131749190194313,148000000077,1,0,1,0
44b6_e35b117d,6709,27,45.31999999999999,133.28,107.36,cont,5.59986411665498,148000000077,1,0,1,0
44b6_e35b117d,6976,28,36.84,77.28,16.0,cont,1.428706801621671,149000000002,1,0,1,0
44b6_e35b117d,7002,28,44.120000000000005,134.72,109.44000000000003,cont,6.282343312809313,149000000077,1,1,0,0
44b6_e35b117d,7317,29,48.35999999999999,125.28,101.28,cont,4.694084708438905,150000000075,1,1,0,0
44b6_e35b117d,7613,30,47.64,124.63999999999999,104.8,cont,4.10057408328395,151000000075,1,0,1,0
44b6_e35b117d,7615,30,48.64,53.28,31.360000000000003,cont,6.021299304344534,151000000003,1,0,1,0
44b6_e35b117d,8655,34,42.0,120.0,108.0,cont,3.2752922102462856,155000000075,2,2,0,0
44b6_e35b117d,8965,35,42.32000000000001,123.35999999999999,110.72000000000001,cont,1.465115716419687,156000000075,2,0,2,0
44b6_e35b117d,9883,38,39.637968139648436,123.96525146484377,114.11322631835938,cont,1.8768699148408865,159000000075,1,1,0,0
44b6_e35b117d,9894,38,42.52000000000001,129.44000000000003,114.72,cont,4.527426145173451,159000000075,1,1,0,0
44b6_e35b117d,10200,39,38.32000000000001,121.92000000000002,113.28000000000002,cont,2.5423331434923995,160000000073,1,0,1,0
44b6_e35b117d,10212,39,41.480000000000004,129.28,116.0,cont,4.46366528903098,160000000073,1,0,1,0
44b6_e35b117d,10866,41,38.52000000000001,58.72,53.92,cont,5.017330590064799,162000000003,1,1,0,0
44b6_e35b117d,11195,42,38.16,58.55999999999998,56.64,cont,5.511433712066223,163000000003,1,0,1,0
44b6_e35b117d,11202,42,38.86666666666666,67.99999999999997,47.466666666666654,cont,1.4816835980472376,163000000003,5,0,0,5
44b6_e35b117d,11831,44,36.84000000000001,127.36,117.28,cont,0.3848883766756165,165000000074,1,1,0,0
44b6_e35b117d,11915,44,60.000000000000014,240.64000000000004,240.00000000000006,cont,3.8576555380178803,165000000030,1,1,0,0
44b6_e35b117d,12146,45,36.519999999999996,128.0,120.0,cont,0.8794538433027699,166000000074,1,0,1,0
44b6_e35b117d,12209,45,59.84000000000002,241.28000000000003,238.72000000000003,cont,3.5944838381191837,166000000030,1,0,1,0
44b6_e35b117d,12753,47,35.160000000000004,128.64000000000001,124.64000000000001,cont,0.4503332099679169,168000000074,1,1,0,0
44b6_e35b117d,13038,48,34.36,129.44000000000003,126.72000000000001,cont,0.8350982352394294,169000000071,1,0,1,0
44b6_e35b117d,13374,49,56.48,243.36000000000004,236.0,cont,3.7005785020723225,170000000030,1,1,0,0
44b6_e35b117d,13648,50,55.68000000000001,242.72000000000003,237.27999999999997,cont,3.6552387490422302,171000000030,1,0,1,0
44b6_e35b117d,14802,54,32.11822875976563,131.928681640625,140.09111816406252,cont,2.4083334515978,175000000072,2,2,0,0
44b6_e35b117d,15111,55,32.07822875976562,133.04868164062503,144.09111816406246,cont,2.669376385024897,176000000072,2,0,2,0
44b6_e35b117d,27710,55,32.009369354248044,122.5669631958008,156.9705322265625,cont,5.008944024716951,176000000072,2,1,1,0
44b6_e35b117d,15376,56,30.395275268554684,122.01198364257813,152.27134765625004,cont,3.053402495640589,177000000072,1,0,1,0
44b6_e35b117d,15673,57,31.52,132.8,147.52000000000004,cont,4.976136333793518,178000000072,1,1,0,0
44b6_e35b117d,16843,61,18.000000000000004,79.36000000000001,100.64000000000003,cont,2.2959128086667504,182000000002,1,1,0,0
44b6_e35b117d,17187,62,14.999999999999993,79.99999999999997,99.46666666666667,cont,0.8343249308739372,183000000002,5,0,0,5
44b6_e35b117d,17215,62,18.04,76.64000000000001,105.43999999999998,cont,5.999155799985521,183000000002,4,0,1,3
44b6_e35b117d,21225,76,4.159999999999999,91.35999999999997,117.92000000000002,cont,6.302842736614955,197000000002,1,0,1,0
44b6_e35b117d,22354,80,4.4799999999999995,83.52,126.72000000000001,cont,1.7724422592851943,201000000002,1,1,0,0
44b6_e35b117d,22660,81,4.160000000000001,82.72,128.64000000000001,cont,0.7597635322914591,202000000002,1,0,1,0
44b6_e35b117d,25648,92,6.680000000000001,57.279999999999994,161.27999999999997,cont,1.4070097947420224,213000000003,1,1,0,0
44b6_e35b117d,25919,93,6.32,56.64,162.72,cont,0.9198131603755154,214000000003,1,0,1,0
44b6_e35b117d,25935,93,9.133333333333333,56.533333333333324,175.4666666666667,cont,6.92677098293282,214000000003,4,0,0,4
44b6_e57ff5c6,383,3,44.00000000000001,53.44,145.76,cont,6.918321802467719,27000000056,2,1,0,1
44b6_e57ff5c6,515,4,43.120000000000005,62.24000000000001,142.08,cont,6.395753107531593,28000000056,1,0,1,0
44b6_e57ff5c6,1992,13,26.88,110.88,99.36000000000001,cont,4.207275320798481,37000000056,1,1,0,0
44b6_e57ff5c6,2206,14,25.59999999999999,110.66666666666666,96.5333333333333,cont,1.499399474697196,38000000056,1,0,1,0
44b6_e57ff5c6,2408,15,22.36,121.43999999999998,83.52,cont,3.4158257603396605,39000000056,2,0,1,1
44b6_e57ff5c6,3592,20,11.374180145263672,131.5317138671875,98.42032958984376,cont,6.81640752945727,44000000056,2,0,1,1
44b6_e57ff5c6,4035,21,59.52000000000001,133.92000000000004,222.72000000000003,cont,1.7972235531508147,45000000009,1,1,0,0
44b6_e57ff5c6,4303,22,59.160000000000004,138.72,221.28000000000003,cont,0.40754792662949146,46000000009,1,0,1,0
44b6_e57ff5c6,4304,22,59.31999999999999,143.36,234.71999999999997,cont,5.487329296889326,46000000009,1,0,1,0
44b6_e57ff5c6,4327,22,61.99999999999999,128.53333333333333,227.46666666666664,cont,6.839439916254498,46000000009,5,0,0,5
44b6_e57ff5c6,4456,23,29.36,10.400000000000002,17.44,cont,1.591504791070388,47000000046,2,0,2,0
44b6_e57ff5c6,4575,23,58.2,140.16,220.0,cont,2.240850842871969,47000000009,3,3,0,0
44b6_e57ff5c6,4839,24,56.52,142.72,219.36,cont,2.4915547982133557,48000000009,3,0,3,0
44b6_e57ff5c6,4976,25,23.68,20.48,9.120000000000001,cont,2.093792163754558,49000000046,2,2,0,0
44b6_e57ff5c6,5117,25,55.0,143.68,219.20000000000002,cont,4.5315070616738575,49000000009,2,2,0,0
44b6_e57ff5c6,5253,26,20.48,28.639999999999997,8.48,cont,2.657955628862154,50000000046,3,1,2,0
44b6_e57ff5c6,5410,26,53.120000000000005,148.64000000000001,220.64000000000004,cont,1.8567713914211517,50000000009,2,0,2,0
44b6_e57ff5c6,5440,26,59.47999999999998,68.64,172.64,cont,4.241094345802769,50000000020,2,1,1,0
44b6_e57ff5c6,5464,26,62.266666666666666,72.53333333333332,183.99999999999997,cont,5.567585066455838,50000000020,7,1,0,6
44b6_e57ff5c6,5734,27,52.000000000000014,151.51999999999995,222.72000000000003,cont,1.5063495817704609,51000000009,1,1,0,0
44b6_e57ff5c6,5752,27,58.480000000000004,76.63999999999999,174.72,cont,4.150403971302069,51000000020,3,2,1,0
44b6_e57ff5c6,5787,27,62.44,77.76,188.64000000000001,cont,6.118680668657262,51000000020,1,0,1,0
44b6_e57ff5c6,5859,28,15.0,42.879999999999995,10.239999999999998,cont,5.620832483938657,52000000046,3,1,2,0
44b6_e57ff5c6,6048,28,50.84,154.88,224.0,cont,2.532603033442072,52000000009,2,1,1,0
44b6_e57ff5c6,6074,28,57.31999999999999,85.28,176.00000000000003,cont,2.9113361408466893,52000000020,7,5,2,0
44b6_e57ff5c6,6188,29,12.359999999999998,45.28000000000001,8.000000000000004,cont,4.025607485212633,53000000046,1,0,1,0
44b6_e57ff5c6,6385,29,47.60000000000001,144.95999999999998,208.16000000000003,cont,3.512293810674151,53000000009,4,0,1,3
44b6_e57ff5c6,6386,29,50.0,152.0,224.80000000000004,cont,5.203172860140256,53000000009,3,0,1,2
44b6_e57ff5c6,6405,29,53.109892578125,81.69800781250001,167.7963720703125,cont,5.986070952281367,53000000020,7,0,7,0
44b6_e57ff5c6,6738,30,52.64000000000001,81.12,165.11999999999998,cont,5.086769142835154,54000000021,2,2,0,0
44b6_e57ff5c6,7066,31,53.31999999999999,54.24000000000001,159.68000000000006,cont,4.427908824998554,55000000021,6,0,1,5
44b6_e57ff5c6,7347,32,45.36,131.52,200.0,cont,2.0772051535657154,56000000009,5,5,0,0
44b6_e57ff5c6,7358,32,47.68000000000001,137.12,215.36,cont,6.9735791563586815,56000000009,4,4,0,0
44b6_e57ff5c6,7373,32,52.63999999999999,55.99999999999999,158.56,cont,2.1057391813802675,56000000021,2,2,0,0
44b6_e57ff5c6,7462,33,4.639999999999999,32.47999999999999,3.68,cont,3.4853882495354838,57000000047,4,1,3,0
44b6_e57ff5c6,7648,33,44.160000000000004,124.96000000000001,195.51999999999998,cont,4.834231615520717,57000000009,9,4,5,0
44b6_e57ff5c6,7957,34,44.599999999999994,91.99999999999996,200.5333333333333,cont,1.3981401957402049,58000000009,6,0,3,3
44b6_e57ff5c6,7983,34,51.13333333333333,10.933333333333334,153.6,cont,2.323648987703512,58000000021,2,0,0,2
44b6_e57ff5c6,8253,35,42.519999999999996,120.96000000000002,196.64000000000001,cont,2.936757618956669,59000000009,7,1,2,4
44b6_e57ff5c6,8287,35,48.24,43.68,151.84,cont,4.258647269086748,59000000021,4,0,3,1
44b6_e57ff5c6,9076,38,10.866666666666664,32.00000000000001,16.53333333333333,cont,1.4249025307796344,62000000060,2,1,0,1
44b6_e57ff5c6,9224,38,40.839999999999996,161.11999999999995,192.64000000000004,cont,2.112249985205337,62000000009,6,0,3,3
44b6_e57ff5c6,9262,38,48.26666666666665,81.06666666666669,138.93333333333334,cont,0.9962102668780783,62000000021,2,0,0,2
44b6_e57ff5c6,9391,39,9.319999999999997,40.31999999999999,19.04,cont,0.5362499999999933,63000000060,1,0,1,0
44b6_e57ff5c6,9555,39,40.480000000000004,164.48,193.27999999999997,cont,0.9913831877735215,63000000009,1,1,0,0
44b6_e57ff5c6,9723,40,7.840000000000001,48.64,20.0,cont,0.5040120286263002,64000000060,2,2,0,0
44b6_e57ff5c6,9892,40,40.360000000000014,168.0,192.64000000000004,cont,2.4430187524863705,64000000009,2,1,1,0
44b6_e57ff5c6,10058,41,6.039999999999999,58.08000000000001,22.080000000000005,cont,2.3617908988731378,65000000060,2,0,2,0
44b6_e57ff5c6,10615,42,41.88000000000001,113.28000000000003,143.36,cont,1.1518029345334924,66000000021,2,2,0,0
44b6_e57ff5c6,10976,43,40.84,118.07999999999998,142.72,cont,1.802871407643935,67000000021,2,0,2,0
44b6_e57ff5c6,12663,48,31.840000000000003,178.72000000000003,197.44,cont,6.930124908686691,72000000009,2,2,0,0
44b6_e57ff5c6,12688,48,37.00000000000001,192.64,182.72,cont,5.432946369144472,72000000009,1,1,0,0
44b6_e57ff5c6,13014,49,33.133333333333326,188.53333333333325,183.46666666666667,cont,1.2087786535728453,73000000009,7,0,0,7
44b6_e57ff5c6,13034,49,36.84000000000001,193.27999999999997,179.36000000000004,cont,6.92745714530231,73000000009,4,0,1,3
44b6_e57ff5c6,13338,50,29.5687548828125,133.45458740234378,127.14116455078127,cont,4.5432005183548885,74000000022,1,1,0,0
44b6_e57ff5c6,13687,51,28.688754882812496,133.45458740234378,124.42116455078124,cont,3.1403693961132193,75000000022,1,0,1,0
44b6_e57ff5c6,15050,55,24.840000000000003,141.28000000000003,114.07999999999998,cont,5.690029794517773,79000000024,1,1,0,0
44b6_e57ff5c6,15378,56,23.319999999999997,146.08,110.71999999999998,cont,6.581791644377693,80000000025,3,2,1,0
44b6_e57ff5c6,15385,56,24.160000000000004,201.43999999999997,167.36000000000004,cont,0.7004485170232123,80000000011,1,0,1,0
44b6_e57ff5c6,15735,57,26.84,161.92000000000002,106.08000000000001,cont,2.162776716399544,81000000025,1,1,0,0
44b6_e57ff5c6,16341,59,20.319999999999997,211.36,160.0,cont,1.25924394479385,83000000011,1,1,0,0
44b6_e57ff5c6,17941,65,17.119999999999997,182.71999999999997,87.84,cont,3.783459271949414,89000000026,2,2,0,0
44b6_e57ff5c6,18178,66,10.160000000000002,219.36,136.0,cont,1.1035652450127185,90000000011,1,0,1,0
44b6_e57ff5c6,18205,66,16.48,183.84,88.79999999999998,cont,6.581651222337747,90000000026,4,0,2,2
44b6_e57ff5c6,18835,69,15.239999999999995,191.36,86.71999999999998,cont,5.290142513439492,93000000026,2,1,1,0
44b6_e57ff5c6,19023,70,12.999999999999998,195.99999999999997,75.46666666666667,cont,1.6889666979974602,94000000026,3,1,0,2
44b6_e57ff5c6,19029,70,14.866666666666667,191.99999999999994,84.53333333333332,cont,6.151733396752684,94000000026,1,0,1,0
44b6_e57ff5c6,19193,71,3.8666666666666663,216.5333333333333,128.53333333333336,cont,4.121385727216037,95000000011,3,0,0,3
44b6_e57ff5c6,19213,71,12.079999999999998,199.68,73.28,cont,0.6245160626437034,95000000026,1,0,1,0
44b6_e57ff5c6,19567,73,2.0,232.0,130.72,cont,6.298819457842874,97000000011,2,1,1,0
44b6_e57ff5c6,19590,73,5.840000000000002,215.36000000000004,127.36000000000004,cont,3.3458035750025634,97000000011,1,1,0,0
44b6_e57ff5c6,19776,74,5.68,212.64,125.92000000000002,cont,1.115110000179356,98000000011,1,0,1,0
44b6_e57ff5c6,20516,78,3.84,201.28000000000006,120.64000000000001,cont,1.5404108624000172,102000000011,1,1,0,0
44b6_e57ff5c6,20663,79,3.3200000000000003,198.72,118.72000000000003,cont,0.744136454220591,103000000011,1,0,1,0
44b6_e57ff5c6,24004,79,7.47451738357544,196.92934692382818,80.54404571533203,cont,6.372306252654527,103000000026,2,1,1,0
44b6_e57ff5c6,20847,80,6.868674392700196,196.76859863281254,78.32179809570312,cont,6.6348297072955384,104000000026,1,0,1,0
44b6_e57ff5c6,21714,85,1.8399999999999999,180.0,104.00000000000001,cont,0.6306172571378029,109000000011,1,1,0,0
44b6_e57ff5c6,21872,86,1.4800000000000002,178.72000000000003,102.08000000000001,cont,0.9380065298280081,110000000011,1,0,1,0
44b6_e57ff5c6,22988,94,2.999999999999999,179.46666666666664,39.999999999999986,cont,6.778628845701593,118000000026,2,0,0,2
44b6_e57ff5c6,22990,94,3.133333333333333,195.99999999999997,19.46666666666666,cont,6.255487868441762,118000000026,1,0,0,1
44b6_eb2880fc,534,1,3.3600000000000003,120.32000000000002,128.48,cont,1.0966041332221923,89000000005,2,2,0,0
44b6_eb2880fc,876,1,50.44000000000001,68.96000000000001,48.800000000000004,cont,1.6340746616969384,89000000033,4,4,0,0
44b6_eb2880fc,978,2,1.44,117.92000000000002,124.64000000000001,cont,1.7366225302292997,90000000005,2,0,2,0
44b6_eb2880fc,1400,2,49.160000000000004,75.36,45.92,cont,2.0899420596035685,90000000033,4,0,4,0
44b6_eb2880fc,5442,10,42.480000000000004,100.64000000000001,51.36000000000001,cont,1.0361844068504453,98000000033,1,1,0,0
44b6_eb2880fc,5974,11,42.13333333333333,96.5333333333333,44.533333333333324,cont,4.446288246016756,99000000033,6,0,0,6
44b6_eb2880fc,5975,11,41.84000000000001,103.36,52.64000000000001,cont,0.8234758648557882,99000000033,1,0,1,0
44b6_eb2880fc,7155,13,61.160000000000004,24.0,179.99999999999997,cont,4.581260914038828,101000000034,1,1,0,0
44b6_eb2880fc,7665,14,60.52,25.92,181.92000000000002,cont,5.917588719233538,102000000034,1,0,1,0
44b6_eb2880fc,7683,14,63.0,18.93333333333333,171.4666666666667,cont,1.2645335657422805,102000000034,3,0,0,3
44b6_eb2880fc,8681,16,58.47999999999999,29.439999999999998,186.07999999999998,cont,6.948257537865169,104000000034,1,0,1,0
44b6_eb2880fc,9718,18,62.52000000000001,100.63999999999999,111.36,cont,1.5642260466441544,106000000048,3,2,1,0
44b6_eb2880fc,10220,19,62.000000000000014,103.36,111.36,cont,0.6821129855089976,107000000048,2,0,2,0
44b6_eb2880fc,10696,20,60.68000000000001,25.92,178.72000000000003,cont,2.9160488036039403,108000000034,1,1,0,0
44b6_eb2880fc,11172,21,60.16000000000001,28.000000000000004,178.72000000000003,cont,1.582185987802938,109000000034,1,0,1,0
44b6_eb2880fc,11180,21,60.35999999999999,108.64000000000001,109.92000000000002,cont,1.1206610772218333,109000000048,1,1,0,0
44b6_eb2880fc,11650,22,59.16000000000001,111.36,109.91999999999999,cont,2.1228496472901663,110000000048,1,0,1,0
44b6_eb2880fc,12364,24,27.36,114.72,72.47999999999999,cont,1.0642199314521399,112000000033,1,1,0,0
44b6_eb2880fc,12537,24,57.96000000000001,31.360000000000003,179.35999999999999,cont,1.5882660553257595,112000000034,1,1,0,0
44b6_eb2880fc,12542,24,56.99999999999999,116.0,112.0,cont,1.21875,112000000048,1,1,0,0
44b6_eb2880fc,12972,25,55.99999999999999,117.28,113.92,cont,1.0472113265716703,113000000048,1,0,1,0
44b6_eb2880fc,12982,25,57.84,32.0,180.64000000000004,cont,1.5275864337247838,113000000034,4,0,1,3
44b6_eb2880fc,13215,26,24.64,114.56000000000003,76.8,cont,0.6170722202465415,114000000033,1,0,1,0
44b6_eb2880fc,13237,26,26.999999999999993,128.53333333333333,70.93333333333331,cont,6.845710969691573,114000000033,3,0,0,3
44b6_eb2880fc,13484,26,62.92,107.52000000000001,104.96000000000002,cont,2.2032986633681677,114000000075,1,1,0,0
44b6_eb2880fc,13669,27,23.680000000000007,116.0,78.71999999999998,cont,0.7218000502216556,115000000033,1,1,0,0
44b6_eb2880fc,14104,28,22.48,116.63999999999999,79.19999999999999,cont,1.0128587327460863,116000000033,1,0,1,0
44b6_eb2880fc,14314,28,53.16,121.28000000000003,120.0,cont,0.49555839716424704,116000000048,1,0,1,0
44b6_eb2880fc,14845,29,61.52000000000001,108.0,100.64000000000001,cont,1.3086043376437304,117000000075,3,3,0,0
44b6_eb2880fc,15291,30,60.519999999999996,108.0,100.0,cont,0.8794538433027635,118000000075,4,1,3,0
44b6_eb2880fc,16385,33,17.679999999999996,119.36000000000001,82.08,cont,0.6911505353394463,121000000034,1,0,1,0
44b6_eb2880fc,16394,33,19.999999999999993,120.53333333333336,92.0,cont,4.896693630087996,121000000034,6,0,0,6
44b6_eb2880fc,17096,34,55.68000000000001,109.28,101.28,cont,1.8878695830220886,122000000076,2,2,0,0
44b6_eb2880fc,17134,34,61.95999999999999,92.0,79.36,cont,1.6963163214742965,122000000074,2,2,0,0
44b6_eb2880fc,17500,35,46.68000000000001,52.00000000000001,183.36,cont,6.19218824911677,123000000035,1,1,0,0
44b6_eb2880fc,17541,35,54.480000000000004,110.71999999999998,102.72000000000003,cont,1.0343989800845632,123000000076,3,1,2,0
44b6_eb2880fc,17605,35,62.160000000000004,92.0,79.36,cont,3.8303838743656007,123000000074,3,1,2,0
44b6_eb2880fc,17946,36,45.800000000000004,53.92000000000001,183.20000000000002,cont,6.3801481663829644,124000000035,1,0,1,0
44b6_eb2880fc,17974,36,50.0,43.466666666666654,187.46666666666664,cont,2.008188532124295,124000000035,3,0,0,3
44b6_eb2880fc,17997,36,53.480000000000004,111.36,103.36,cont,1.666397686628258,124000000076,1,0,1,0
44b6_eb2880fc,18010,36,55.233099975585944,86.85093505859375,89.004873046875,cont,5.530844605431111,124000000074,2,0,2,0
44b6_eb2880fc,18037,36,58.999999999999986,91.46666666666663,84.53333333333333,cont,1.7537373286942044,124000000074,7,0,0,7
44b6_eb2880fc,18364,37,41.18814636230468,125.95695068359376,134.89777832031248,cont,6.425186591438251,125000000049,2,1,1,0
44b6_eb2880fc,18388,37,44.99999999999999,130.08,128.64000000000004,cont,1.4337805140606257,125000000049,1,1,0,0
44b6_eb2880fc,18414,37,49.919999999999995,44.959999999999994,188.64000000000004,cont,2.2159662113850067,125000000035,1,1,0,0
44b6_eb2880fc,18594,38,11.266666666666667,117.06666666666666,95.46666666666667,cont,1.0920999458230343,126000000035,6,0,0,6
44b6_eb2880fc,18839,38,49.52,46.72,190.07999999999998,cont,2.6741990623362337,126000000036,1,0,1,0
44b6_eb2880fc,18887,38,57.16,95.36000000000001,87.36,cont,0.7609790158079145,126000000074,2,2,0,0
44b6_eb2880fc,19258,39,48.84000000000001,47.360000000000014,189.92000000000002,cont,1.9031240763544697,127000000036,1,1,0,0
44b6_eb2880fc,19309,39,56.52,96.64000000000001,88.00000000000001,cont,1.254516839265224,127000000074,2,0,2,0
44b6_eb2880fc,19337,39,61.00000000000001,95.36000000000001,82.72000000000001,cont,6.7212954480516744,127000000074,2,0,2,0
44b6_eb2880fc,19705,40,48.16,48.0,191.36,cont,2.311673162992549,128000000036,1,0,1,0
44b6_eb2880fc,20186,41,54.84,99.36000000000001,89.28000000000003,cont,1.4036275547665846,129000000074,1,1,0,0
44b6_eb2880fc,20535,42,40.31999999999999,124.00000000000003,125.28000000000002,cont,1.7780945735534035,130000000050,1,1,0,0
44b6_eb2880fc,20959,43,39.68,124.00000000000003,123.36000000000003,cont,1.3600580180639488,131000000050,2,1,1,0
44b6_eb2880fc,21441,44,46.32,116.0,109.28,cont,3.0530113208109806,132000000075,7,1,0,6
44b6_eb2880fc,21867,45,47.133333333333326,124.53333333333335,109.06666666666665,cont,1.8898267943121034,133000000075,5,0,0,5
44b6_eb2880fc,23170,48,49.99999999999999,107.99999999999997,95.99999999999997,cont,3.656249999999989,136000000073,7,0,0,7
44b6_eb2880fc,23917,50,32.32,128.64000000000001,127.99999999999997,cont,0.5813776741499505,138000000050,1,1,0,0
44b6_eb2880fc,24017,50,48.16,109.92,97.28000000000003,cont,3.0696589794307862,138000000073,1,0,1,0
44b6_eb2880fc,24040,50,52.0,98.72000000000001,92.64,cont,5.354615415695135,138000000073,1,0,1,0
44b6_eb2880fc,25281,53,40.160000000000004,55.35999999999999,201.44000000000003,cont,0.7326930206437028,141000000037,1,1,0,0
44b6_eb2880fc,25690,54,39.68,55.99999999999999,201.92000000000007,cont,1.2352137972837065,142000000037,1,0,1,0
44b6_eb2880fc,25706,54,42.866666666666674,51.99999999999998,204.5333333333333,cont,6.757712286508088,142000000037,1,0,1,0
44b6_eb2880fc,27624,59,33.160000000000004,134.72000000000003,118.72,cont,1.5610998726859213,147000000071,1,1,0,0
44b6_eb2880fc,28027,60,32.16000000000001,135.36000000000004,117.28,cont,1.7472339747440757,148000000071,1,0,1,0
44b6_eb2880fc,28045,60,36.160000000000004,52.64,204.00000000000003,cont,2.125708705349828,148000000037,1,1,0,0
44b6_eb2880fc,28482,61,35.84,53.280000000000015,204.00000000000003,cont,1.7562032911938212,149000000037,1,0,1,0
44b6_eb2880fc,31569,69,23.68,130.72,122.71999999999998,cont,1.8184033174738805,157000000069,1,1,0,0
44b6_eb2880fc,31951,70,22.84,128.64000000000004,121.28000000000003,cont,0.8842449109268514,158000000069,1,0,1,0
44b6_eb2880fc,31998,70,31.68,58.08000000000001,211.36,cont,0.9728309205612192,158000000037,1,1,0,0
44b6_eb2880fc,33672,75,6.160000000000002,156.64,151.36,cont,1.7420879175575423,163000000048,1,1,0,0
44b6_eb2880fc,34034,76,10.480000000000002,149.44,177.44000000000003,cont,1.6962384855909778,164000000049,1,1,0,0
44b6_eb2880fc,34037,76,11.32,141.28000000000003,154.56,cont,2.8131279854460893,164000000048,1,0,1,0
44b6_eb2880fc,34382,77,10.040000000000003,167.35999999999999,173.92000000000002,cont,6.889999999999994,165000000049,1,0,1,0
44b6_eb2880fc,36854,84,24.520000000000003,67.35999999999999,226.72,cont,0.8300235312929342,172000000035,1,1,0,0
44b6_eb2880fc,37147,85,23.68,68.64,228.0,cont,0.845156235556483,173000000035,2,1,1,0
44b6_eb2880fc,37458,86,23.040000000000006,69.28,227.83999999999997,cont,0.9128971738372199,174000000036,1,0,1,0
44b6_eb2880fc,38678,90,1.6800000000000002,128.64000000000001,156.64,cont,6.608497607247817,178000000045,2,1,1,0
44b6_eb2880fc,38713,90,9.319999999999999,131.36,150.72,cont,1.3404033907745836,178000000066,1,1,0,0
44b6_f28707c6,3962,14,4.32,249.27999999999997,230.72000000000008,cont,2.3991186532766586,15000000076,1,1,0,0
44b6_f28707c6,4265,15,4.6800000000000015,247.36,226.07999999999998,cont,2.6319984090040713,16000000076,1,0,1,0
44b6_f28707c6,4575,16,5.3999999999999995,245.43999999999997,220.00000000000003,cont,3.7452989283233524,17000000077,1,1,0,0
44b6_f28707c6,4858,17,4.960000000000001,241.91999999999996,214.56,cont,5.358583797282639,18000000077,5,0,1,4
44b6_f28707c6,5176,18,9.0,244.53333333333327,211.9999999999999,cont,1.63602165030839,19000000078,5,0,0,5
44b6_f28707c6,5468,19,10.079999999999997,239.04000000000002,208.00000000000006,cont,1.5450323621206137,20000000079,1,1,0,0
44b6_f28707c6,5766,20,11.159999999999998,232.0,204.64,cont,1.518918714908732,21000000079,1,0,1,0
44b6_f28707c6,6067,21,11.399999999999999,228.00000000000003,203.19999999999996,cont,2.779177598679147,22000000080,3,3,0,0
44b6_f28707c6,6369,22,11.24,223.68,202.56,cont,4.765047546982092,23000000080,4,1,3,0
44b6_f28707c6,6651,23,12.240000000000002,199.2,199.03999999999996,cont,2.440315041137092,24000000080,6,0,2,4
44b6_f28707c6,10772,37,23.36,176.00000000000003,170.72000000000003,cont,2.745480661468951,38000000080,1,1,0,0
44b6_f28707c6,11066,38,24.119999999999997,171.99999999999994,167.36,cont,1.5775896369144948,39000000080,1,0,1,0
44b6_f28707c6,12219,42,27.64,150.07999999999998,152.64,cont,1.3307152681922694,43000000080,2,2,0,0
44b6_f28707c6,12507,43,28.16,143.67999999999998,149.28000000000003,cont,3.039836035874296,44000000080,2,0,2,0
44b6_f28707c6,14627,51,30.999999999999996,115.67999999999999,126.08000000000001,cont,3.380156246388622,52000000080,3,3,0,0
44b6_f28707c6,14887,52,31.039999999999996,102.08,124.00000000000003,cont,2.121543144152401,53000000080,3,0,3,0
44b6_f28707c6,15152,53,36.768043212890625,92.80937622070313,110.90920166015624,cont,5.378080040836704,54000000081,1,0,1,0
44b6_f28707c6,15395,54,37.408043212890625,88.48937622070314,109.46920166015624,cont,2.7028909025319137,55000000081,1,1,0,0
44b6_f28707c6,15649,55,39.519999999999996,81.12,106.72,cont,2.778559935020296,56000000082,5,4,1,0
44b6_f28707c6,15871,56,37.60165466308594,51.76155639648438,109.44294921875,cont,2.3125783530731905,57000000082,11,2,5,4
44b6_f28707c6,17054,61,46.31999999999999,58.879999999999995,79.84,cont,2.3463672053836726,62000000083,2,0,0,2
6bba_05b6850b,105,1,9.16,85.28000000000003,98.56000000000003,cont,1.388495701289712,2000011,1,1,0,0
6bba_05b6850b,191,2,8.64,88.00000000000001,100.64,cont,2.2620210128334324,3000021,1,0,1,0
6bba_05b6850b,672,8,10.519999999999998,30.72,42.72,cont,2.410374063293916,9000071,1,1,0,0
6bba_05b6850b,690,8,31.8,137.28,194.71999999999997,cont,0.451796760169871,9000072,1,1,0,0
6bba_05b6850b,743,9,9.920000000000002,31.360000000000003,41.28000000000001,cont,1.6626696342027807,10000078,1,0,1,0
6bba_05b6850b,760,9,25.266666666666666,135.46666666666664,186.93333333333322,cont,3.0299460253718618,10000081,1,0,0,1
6bba_05b6850b,1778,22,27.519999999999996,139.35999999999999,170.72000000000003,cont,2.4216862023185493,23000192,1,1,0,0
6bba_05b6850b,1846,23,27.480000000000008,136.64,167.36,cont,2.8340298163569035,24000202,1,0,1,0
6bba_05b6850b,2078,26,57.36,9.28,94.72,cont,1.1989813384702934,27000234,1,1,0,0
6bba_05b6850b,2140,27,57.16,8.64,93.28000000000003,cont,1.629948714683996,28000244,1,0,1,0
6bba_05b6850b,3450,47,53.31999999999999,91.36,115.36000000000004,cont,1.1241899861678388,48000425,1,1,0,0
6bba_05b6850b,3475,48,1.6800000000000002,192.0,191.36,cont,0.5401750295043285,49000427,1,1,0,0
6bba_05b6850b,3526,48,53.0,92.64,114.71999999999997,cont,0.5640880361255719,49000435,1,0,1,0
6bba_05b6850b,3548,49,0.9999999999999998,195.99999999999997,195.99999999999997,cont,3.4946946397360508,50000437,2,0,0,2
6bba_05b6850b,3553,49,1.68,192.0,189.28,cont,0.6696104277861861,50000437,1,0,1,0
6bba_05b6850b,3609,49,56.36,125.28000000000002,196.64000000000001,cont,1.61489357234463,50000444,1,1,0,0
6bba_05b6850b,3685,50,53.999999999999986,131.99999999999997,195.99999999999997,cont,3.300390601883387,51000454,2,0,0,2
6bba_05b6850b,3688,50,56.84000000000001,126.72000000000001,196.00000000000003,cont,2.2475577022626223,51000454,1,0,1,0
6bba_05b6850b,3900,53,24.840000000000003,129.28,126.72000000000001,cont,3.6319386180110382,54000480,1,1,0,0
6bba_05b6850b,3933,53,57.48,177.92,250.72000000000003,cont,2.7508615127810345,54000484,1,1,0,0
6bba_05b6850b,3938,53,60.47999999999999,188.00000000000003,250.72000000000003,cont,5.5143076684657375,54000484,1,1,0,0
6bba_05b6850b,3986,54,24.680000000000003,130.72,124.00000000000003,cont,3.966231878307167,55000490,1,0,1,0
6bba_05b6850b,3992,54,30.73333333333334,131.99999999999997,116.0,cont,6.55064040815435,55000490,2,0,0,2
6bba_05b6850b,4031,54,58.2,179.35999999999999,249.27999999999997,cont,1.9905420901101334,55000492,1,0,1,0
6bba_05b6850b,4034,54,61.000000000000014,188.00000000000003,249.27999999999997,cont,3.8437004061841566,55000492,1,0,1,0
6bba_05b6850b,4334,58,12.0,94.08000000000001,68.0,cont,2.0315099833621297,59000525,1,1,0,0
6bba_05b6850b,4416,59,12.68,93.91999999999999,68.0,cont,1.528968708803416,60000535,1,0,1,0
6bba_05b6850b,4540,60,58.680000000000014,146.08,171.2,cont,6.965793318962321,61000553,1,1,0,0
6bba_05b6850b,4615,61,58.31999999999999,142.56,173.43999999999997,cont,1.927252740301605,62000564,1,0,1,0
6bba_05b6850b,4706,63,1.8399999999999999,200.0,177.28,cont,1.5863529320425553,64000577,1,1,0,0
6bba_05b6850b,4730,63,23.840000000000003,242.08,238.07999999999998,cont,1.672171585842792,64000580,1,1,0,0
6bba_05b6850b,4774,64,1.64,200.0,178.72,cont,1.2316813965064164,65000587,1,0,1,0
6bba_05b6850b,4800,64,23.560000000000006,240.0,235.35999999999999,cont,4.162570006918814,65000591,1,0,1,0
6bba_05b6850b,4880,65,18.999999999999996,183.46666666666667,144.0,cont,3.3041666666666734,66000602,1,0,0,1
6bba_05b6850b,5036,67,27.16,131.35999999999999,114.71999999999997,cont,2.9957349891136924,68000623,1,1,0,0
6bba_05b6850b,5110,68,27.84,130.56,113.28000000000002,cont,3.7761937268233456,69000634,1,0,1,0
6bba_05b6850b,5184,69,28.320000000000004,130.72,112.0,cont,3.070261084175087,70000644,1,1,0,0
6bba_05b6850b,5191,69,34.0,192.0,188.64000000000004,cont,3.3952779709472947,70000645,1,1,0,0
6bba_05b6850b,5260,70,29.159999999999997,129.44000000000003,110.08,cont,2.9955146156378634,71000654,1,0,1,0
6bba_05b6850b,5266,70,33.84000000000001,192.64,187.99999999999997,cont,3.830383874365577,71000655,1,0,1,0
6bba_05b6850b,5272,70,39.53333333333333,192.5333333333333,184.5333333333333,cont,6.002610109360753,71000655,1,0,0,1
6bba_05b6850b,5412,72,24.0,231.36,212.64,cont,1.722576649237999,73000681,1,1,0,0
6bba_05b6850b,5483,73,23.519999999999996,230.72000000000008,211.36,cont,1.424449455052709,74000694,1,0,1,0
6bba_05b6850b,5484,73,24.866666666666667,239.99999999999994,236.5333333333333,cont,2.0966418669042137,74000695,2,0,0,2
6bba_05b6850b,5780,77,33.32,120.63999999999999,102.72000000000001,cont,4.65411461101894,78000746,1,1,0,0
6bba_05b6850b,5814,77,61.00000000000001,132.64000000000001,156.64000000000004,cont,1.790230868491529,78000751,1,1,0,0
6bba_05b6850b,5842,78,18.84,100.64000000000003,68.64000000000001,cont,2.7477880308531915,79000754,1,1,0,0
6bba_05b6850b,5855,78,33.84,120.00000000000003,100.63999999999999,cont,3.638585104473984,79000758,1,0,1,0
6bba_05b6850b,5887,78,61.00000000000001,132.00000000000006,155.36,cont,1.756278469520122,79000763,3,1,1,1
6bba_05b6850b,5910,79,18.64,101.43999999999998,69.44000000000001,cont,3.179689450245099,80000766,1,0,1,0
6bba_05b6850b,5948,79,59.160000000000004,12.639999999999997,36.64,cont,2.155377359651898,80000772,1,1,0,0
6bba_05b6850b,5954,79,61.160000000000004,132.00000000000006,154.71999999999997,cont,1.3576289073233636,80000774,1,0,1,0
6bba_05b6850b,6022,80,59.31999999999999,12.0,35.99999999999999,cont,1.5941573008960082,81000783,1,0,1,0
6bba_05b6850b,6583,88,36.160000000000004,114.71999999999997,94.56,cont,1.6412499999999917,89000863,1,1,0,0
6bba_05b6850b,6658,89,36.31999999999999,113.28000000000002,95.36000000000001,cont,2.9588389826078836,90000873,1,0,1,0
6bba_05b6850b,6714,90,17.999999999999996,216.0,184.64,cont,2.2724449938337448,91000879,1,1,0,0
6bba_05b6850b,6784,91,18.48,215.36,185.28,cont,2.761735188246696,92000893,1,0,1,0
6bba_05b6850b,6842,92,5.0,188.64000000000004,165.92,cont,1.2597680838154266,93000902,1,1,0,0
6bba_05b6850b,6895,92,61.00000000000001,191.36,229.27999999999997,cont,0.5640880361255731,93000909,1,1,0,0
6bba_05b6850b,6908,93,5.159999999999999,189.28000000000003,169.28,cont,1.3329953347630266,94000911,1,0,1,0
6bba_05b6850b,6959,93,61.00000000000001,191.36,230.72000000000008,cont,0.7587201394453533,94000918,1,0,1,0
6bba_05db0fb1,4733,5,56.52,80.64,14.719999999999999,cont,1.984164260463333,6000083,2,1,1,0
6bba_05db0fb1,8522,10,38.0,241.28000000000003,177.44000000000003,cont,1.7174335867799941,11000146,3,3,0,0
6bba_05db0fb1,8692,10,49.84000000000001,136.00000000000003,156.00000000000003,cont,1.4809804607083996,11000153,1,1,0,0
6bba_05db0fb1,9571,11,54.519999999999996,190.56000000000003,125.44000000000001,cont,1.3197561327760474,12000169,2,0,2,0
6bba_05db0fb1,10194,12,40.800000000000004,79.36000000000001,88.64,cont,1.5167440456451582,13000175,1,1,0,0
6bba_05db0fb1,16236,19,61.86666666666667,228.0,152.53333333333325,cont,3.0823879462826804,20000297,10,1,0,9
6bba_05db0fb1,16535,20,23.959999999999997,176.64000000000001,229.44,cont,0.2781551949901385,21000300,2,0,2,0
6bba_05db0fb1,76469,22,35.51355026245117,96.28681732177735,5.79033193588257,cont,0.8469086672435804,23000342,1,0,1,0
6bba_05db0fb1,20107,24,39.16,87.2,115.36000000000001,parent,2.3045227271606525,25000381,2,2,0,0
6bba_05db0fb1,20909,25,36.86666666666667,68.53333333333333,122.93333333333335,daughter,1.0520750824758907,26000400,10,1,0,9
6bba_05db0fb1,20951,25,39.36000000000001,90.72,114.4,daughter,2.4601448128108228,26000403,6,4,2,0
6bba_05db0fb1,21064,25,47.68000000000001,131.36000000000004,216.16,cont,5.802319579271722,26000410,3,3,0,0
6bba_05db0fb1,21677,26,33.160000000000004,94.72,14.08,cont,1.321056040635675,27000417,4,0,4,0
6bba_05db0fb1,21847,26,43.16000000000001,210.08000000000004,177.28000000000003,cont,6.9824720282289565,27000425,9,0,4,5
6bba_05db0fb1,21848,26,42.999999999999986,220.0,194.56,cont,1.3736777733515266,27000425,10,0,4,6
6bba_05db0fb1,22081,26,61.84,230.72000000000003,166.71999999999997,cont,2.1804085053035474,27000431,1,1,0,0
6bba_05db0fb1,22905,27,61.68,231.36,170.71999999999997,cont,0.9198131603755278,28000449,2,1,1,0
6bba_05db0fb1,23113,28,16.36,116.80000000000001,27.839999999999996,cont,2.307385560542491,29000451,11,0,4,7
6bba_05db0fb1,23437,28,37.32,194.08000000000004,212.0,cont,0.7583720228225679,29000456,1,0,1,0
6bba_05db0fb1,25068,30,36.68000000000001,95.36000000000001,124.00000000000003,cont,1.1351761977772523,31000502,1,1,0,0
6bba_05db0fb1,25072,30,36.36,142.08,208.64000000000001,cont,1.2775642107150653,31000503,1,0,1,0
6bba_05db0fb1,25083,30,37.67999999999999,13.44,8.0,cont,0.9149197369168569,31000500,2,2,0,0
6bba_05db0fb1,25531,31,12.0,107.52,46.55999999999998,cont,2.2569372222106616,32000517,6,0,1,5
6bba_05db0fb1,25716,31,24.520000000000003,147.84,55.199999999999996,cont,2.4304481325467475,32000518,10,0,3,7
6bba_05db0fb1,25905,31,37.480000000000004,9.919999999999998,10.719999999999999,cont,0.930942063181155,32000524,3,1,2,0
6bba_05db0fb1,26123,31,53.000000000000014,160.64000000000001,108.8,cont,2.0861481401137603,32000533,1,0,1,0
6bba_05db0fb1,31487,38,25.84,69.12,35.36,cont,6.647044573530402,39000680,5,5,0,0
6bba_05db0fb1,31520,38,28.32,85.28,37.92,cont,1.3178539609911302,39000680,5,5,0,0
6bba_05db0fb1,31639,38,34.72,4.0,26.88,cont,6.335791436355839,39000686,4,3,1,0
6bba_05db0fb1,31656,38,35.36,210.08000000000004,217.28000000000003,cont,1.5962265072664332,39000684,4,4,0,0
6bba_05db0fb1,31679,38,37.52000000000001,126.08000000000001,98.08000000000003,cont,5.969237938170992,39000687,7,7,0,0
6bba_05db0fb1,31713,38,40.71999999999999,130.23999999999998,109.75999999999996,cont,5.981258359450775,39000687,5,5,0,0
6bba_05db0fb1,31749,38,42.0,184.0,193.27999999999997,cont,1.9140541071244475,39000689,8,8,0,0
6bba_05db0fb1,32304,39,25.08,72.8,36.64000000000001,cont,4.251324242221472,40000697,12,0,5,7
6bba_05db0fb1,32356,39,26.160000000000007,100.0,136.0,cont,1.5885154232804812,40000699,15,1,7,7
6bba_05db0fb1,32395,39,29.159999999999997,87.83999999999999,131.2,cont,5.758372708500205,40000699,14,4,4,6
6bba_05db0fb1,32428,39,33.56,208.64000000000004,221.44,cont,4.2604140790538265,40000702,6,2,4,0
6bba_05db0fb1,32448,39,32.64,216.0,234.56,cont,6.521131276473436,40000702,9,4,5,0
6bba_05db0fb1,32473,39,33.199999999999996,3.3600000000000003,21.439999999999998,cont,1.70059180875365,40000704,8,1,3,4
6bba_05db0fb1,32514,39,37.0,120.16000000000001,248.00000000000006,cont,0.8812535460921804,40000705,5,1,2,2
6bba_05db0fb1,32610,39,41.879999999999995,181.28000000000003,196.80000000000004,cont,6.306549423020477,40000706,19,2,8,9
6bba_05db0fb1,32728,39,55.320000000000014,215.83999999999997,206.56,cont,1.2117968321876527,40000714,9,0,7,2
6bba_05db0fb1,33006,40,16.840000000000003,130.4,89.27999999999999,cont,6.101977778761567,41000719,10,0,1,9
6bba_05db0fb1,33049,40,19.2,144.32,98.4,cont,3.7051781755402775,41000719,8,0,2,6
6bba_05db0fb1,33136,40,27.44,85.92,132.64000000000001,cont,6.429457840673655,41000725,10,0,4,6
6bba_05db0fb1,33198,40,30.480000000000004,127.99999999999997,237.92000000000002,cont,2.6556696424819233,41000726,6,0,3,3
6bba_05db0fb1,33355,40,41.16,179.35999999999999,199.36,cont,1.4798210407005374,41000733,10,0,2,8
6bba_05db0fb1,35404,43,18.68,136.64,104.0,cont,0.8451562355564723,44000797,1,0,1,0
6bba_05db0fb1,40021,49,13.480000000000002,125.43999999999998,126.08000000000001,cont,1.5883491823273577,50000927,1,0,1,0
6bba_05db0fb1,40804,50,12.200000000000003,120.96,128.0,cont,3.8394730269921133,51000948,4,4,0,0
6bba_05db0fb1,41686,51,22.000000000000004,89.27999999999999,9.919999999999998,cont,6.369771998470597,52000971,9,1,2,6
6bba_05db0fb1,41912,51,37.839999999999996,194.72000000000003,116.0,cont,3.6265180011272493,52000981,1,0,1,0
6bba_05db0fb1,42062,51,50.44,117.27999999999997,110.71999999999998,cont,0.9312256708231317,52000989,6,0,0,6
6bba_05db0fb1,42500,52,24.840000000000003,88.64,167.36000000000004,cont,0.9774344607184471,53000996,1,0,1,0
6bba_05db0fb1,42791,52,45.32,203.36,229.44,parent,6.4022937784047445,53001011,1,1,0,0
6bba_05db0fb1,43541,53,45.64000000000001,200.64,230.08000000000004,daughter,4.295689882021272,54001033,1,0,1,0
6bba_05db0fb1,44550,55,8.360000000000001,113.44,77.28,cont,1.3242503775721541,56001060,1,1,0,0
6bba_05db0fb1,44867,55,33.68000000000001,40.000000000000014,35.36,cont,2.2120303173781415,56001067,1,1,0,0
6bba_05db0fb1,45356,56,14.999999999999996,96.0,44.00000000000001,cont,1.8168052317185759,57001084,10,0,0,10
6bba_05db0fb1,45806,56,46.839999999999996,193.27999999999997,230.72000000000008,cont,1.9298542691094704,57001097,1,0,1,0
6bba_05db0fb1,46146,57,17.0,132.53333333333333,116.0,cont,6.692871998647349,58001106,6,0,0,6
6bba_05db0fb1,52073,65,14.04,140.00000000000003,147.2,cont,1.2630128908684848,66001284,2,2,0,0
6bba_05db0fb1,53014,66,32.28,49.440000000000005,199.2,cont,5.3963703079014165,67001312,3,2,1,0
6bba_05db0fb1,53027,66,32.0,41.28,25.28,cont,0.7575010313524337,67001309,6,0,2,4
6bba_05db0fb1,53091,66,36.06857788085937,51.96559204101563,37.67312316894531,cont,5.703227955735483,67001310,2,0,2,0
6bba_05db0fb1,53101,66,37.160000000000004,41.92000000000001,40.000000000000014,cont,1.8211603443958433,67001310,6,0,0,6
6bba_05db0fb1,53757,67,32.160000000000004,50.72000000000001,198.56,cont,6.554634453003455,68001331,2,0,2,0
6bba_05db0fb1,53814,67,36.0,55.99999999999999,215.99999999999994,cont,5.218375735082691,68001331,7,0,0,7
6bba_05db0fb1,54261,68,13.48,100.64000000000001,106.72,cont,6.385422974439514,69001343,1,0,1,0
6bba_05db0fb1,54538,68,35.84,58.08,113.92,cont,0.511036018789277,69001350,1,1,0,0
6bba_05db0fb1,57811,73,10.360000000000001,142.88,189.92000000000002,cont,3.2381565453973966,74001433,6,0,2,4
6bba_05db0fb1,58099,73,36.63999999999999,195.36000000000004,137.44000000000003,cont,2.6934344362727742,74001440,1,1,0,0
6bba_05db0fb1,62683,80,5.84,140.16,196.00000000000003,cont,2.842449702721226,81001577,4,4,0,0
6bba_05db0fb1,63349,81,3.999999999999999,155.46666666666664,198.93333333333334,cont,6.188111992742389,82001599,2,0,0,2
6bba_05db0fb1,67618,87,14.840000000000003,101.28000000000002,18.080000000000002,cont,4.997608177918718,88001706,3,3,0,0
6bba_05db0fb1,71360,92,36.52,100.0,131.35999999999999,cont,5.75580413582673,93001821,3,3,0,0
6bba_05db0fb1,71999,93,32.56,52.0,220.0,cont,2.783260228580867,94001839,1,1,0,0
6bba_05db0fb1,72020,93,31.159999999999997,102.56,141.92000000000002,cont,0.6857808049952941,94001835,16,6,4,6
6bba_05db0fb1,72043,93,33.800000000000004,56.16000000000001,53.120000000000005,cont,2.036119163998018,94001830,13,4,4,5
6bba_05db0fb1,72069,93,34.519999999999996,17.440000000000005,44.00000000000001,cont,3.397960095778059,94001843,10,3,2,5
6bba_05db0fb1,72133,93,41.239999999999995,58.72,131.36000000000004,cont,5.279524718191956,94001841,5,5,0,0
6bba_05db0fb1,72151,93,41.60000000000001,149.27999999999997,145.44,cont,3.5442140348742988,94001837,7,5,2,0
6bba_05db0fb1,72178,93,43.760000000000005,133.44,154.56,cont,6.106584816818654,94001837,3,2,1,0
6bba_05db0fb1,72766,94,33.36,52.0,220.0,cont,2.815567493863359,95001858,1,0,1,0
6bba_05db0fb1,72770,94,32.88000000000001,101.28,141.28000000000003,cont,1.8962434080307202,95001856,6,0,6,0
6bba_062c8d37,225,2,35.64,42.72,216.16000000000003,cont,1.8482187370546754,3000046,1,1,0,0
6bba_062c8d37,283,3,14.133333333333331,203.46666666666664,72.0,cont,1.4210364590873679,4000057,1,0,0,1
6bba_062c8d37,285,3,15.13333333333333,171.46666666666664,152.0,cont,0.5088496353431862,4000056,1,0,0,1
6bba_062c8d37,303,3,31.160000000000004,65.28,167.51999999999998,cont,2.2778419748744674,4000061,1,0,1,0
6bba_062c8d37,308,3,36.839999999999996,38.559999999999995,205.76000000000002,cont,2.9480653083844692,4000062,1,0,1,0
6bba_062c8d37,719,9,5.6800000000000015,189.92000000000002,85.28,cont,1.317853960991128,10000149,1,1,0,0
6bba_062c8d37,756,9,36.79999999999999,50.080000000000005,125.6,cont,3.734107155666527,10000158,1,0,1,0
6bba_062c8d37,793,10,7.000000000000001,187.35999999999999,76.64000000000001,cont,0.2983103459486465,11000162,1,0,1,0
6bba_062c8d37,2198,32,2.3600000000000003,176.00000000000003,96.63999999999999,cont,1.1260675490395726,33000474,1,1,0,0
6bba_062c8d37,2205,32,8.120000000000003,162.87999999999997,60.0,cont,2.5057689687998086,33000475,1,1,0,0
6bba_062c8d37,2263,33,2.8800000000000003,171.20000000000002,92.48,cont,4.164821424743208,34000488,1,0,1,0
6bba_062c8d37,2272,33,10.2,155.2,57.28000000000001,cont,1.8616004539105573,34000492,2,1,1,0
6bba_062c8d37,2332,34,6.466666666666666,159.99999999999997,80.53333333333333,cont,1.1274606577270043,35000508,2,0,0,2
6bba_062c8d37,2337,34,12.0,149.92000000000002,54.080000000000005,cont,1.1866951219668933,35000509,1,0,1,0
6bba_062c8d37,3178,46,23.16,117.28,33.43999999999999,cont,1.0780255388904332,47000681,1,1,0,0
6bba_062c8d37,3242,47,23.319999999999997,119.36000000000001,28.640000000000008,cont,0.8020325741514415,48000695,1,0,1,0
6bba_062c8d37,3353,49,12.159999999999997,216.0,20.639999999999997,cont,1.5275864337247882,50000718,1,1,0,0
6bba_062c8d37,3415,50,12.32,216.0,20.0,cont,1.1773122196341974,51000731,1,0,1,0
6bba_062c8d37,3761,55,35.36,104.0,234.08000000000004,cont,1.6290574345307678,56000787,1,1,0,0
6bba_062c8d37,3825,56,35.32000000000001,104.0,232.0,cont,1.2454329066633727,57000798,1,0,1,0
6bba_062c8d37,4332,64,28.000000000000004,32.64,226.72,cont,0.9654743005383414,65000908,1,1,0,0
6bba_062c8d37,4398,65,28.48,31.360000000000007,227.36000000000004,cont,1.0582481100857148,66000923,1,0,1,0
6bba_062c8d37,4581,68,6.800000000000001,144.64,93.44,cont,0.4743218843781029,69000969,1,1,0,0
6bba_062c8d37,4650,69,6.68,147.36000000000004,90.72,cont,1.1722547717966392,70000984,1,0,1,0
6bba_07477033,388,6,1.4799999999999998,116.0,226.72,cont,1.6109643850811877,7000053,1,1,0,0
6bba_07477033,446,7,1.5199999999999998,116.0,225.28000000000003,cont,1.2772541348533657,8000061,1,0,1,0
6bba_07477033,497,8,1.4799999999999998,116.63999999999999,222.72000000000003,cont,1.9791672807016774,9000070,1,1,0,0
6bba_07477033,552,9,1.64,116.63999999999999,220.79999999999998,cont,1.2216714983988137,10000078,1,0,1,0
6bba_07477033,1266,22,16.0,108.0,212.64,cont,1.6813842883172194,23000173,1,1,0,0
6bba_07477033,1318,23,17.0,108.0,210.07999999999998,cont,2.01191800590879,24000183,1,0,1,0
6bba_07477033,1468,26,3.999999999999999,127.99999999999997,193.06666666666666,cont,4.8011654039525675,27000209,1,0,0,1
6bba_07477033,2080,37,26.320000000000004,120.64000000000001,170.08000000000004,cont,1.6418934382291577,38000288,1,1,0,0
6bba_07477033,2134,38,26.839999999999996,122.08000000000003,166.55999999999997,cont,0.8134744157009512,39000294,1,0,1,0
6bba_07477033,2296,41,20.640000000000004,139.36,30.72,cont,4.396548539763885,42000314,1,1,0,0
6bba_07477033,2303,41,28.48,123.35999999999999,159.35999999999999,cont,1.0425359706024513,42000315,1,1,0,0
6bba_07477033,2355,42,21.16,140.00000000000003,28.640000000000008,cont,4.731011057374525,43000322,1,0,1,0
6bba_07477033,2364,42,29.68,124.63999999999999,155.36,cont,2.1656440294979253,43000323,1,0,1,0
6bba_07477033,2899,52,15.0,136.64000000000001,158.08000000000004,cont,1.3086043376437217,53000394,1,1,0,0
6bba_07477033,2904,52,26.16,143.36,16.8,cont,3.0330092832861584,53000395,1,1,0,0
6bba_07477033,2950,53,15.6,137.28,154.87999999999997,cont,2.686562918023707,54000401,1,0,1,0
6bba_07477033,2958,53,26.320000000000007,143.99999999999997,15.360000000000003,cont,2.961113863481094,54000402,1,0,1,0
6bba_07477033,3345,60,13.48,148.00000000000003,151.2,cont,6.806189852810462,61000445,1,1,0,0
6bba_07477033,3370,60,36.36000000000001,132.64000000000001,120.48000000000002,cont,3.027955426108512,61000447,1,1,0,0
6bba_07477033,3426,61,40.999999999999986,135.99999999999994,103.99999999999999,cont,0.9084026158592736,62000454,3,0,0,3
6bba_07477033,3653,65,34.839999999999996,148.00000000000003,2.7199999999999998,cont,3.6528178335361927,66000480,1,1,0,0
6bba_07477033,3716,66,35.68,148.00000000000003,1.2800000000000002,cont,3.9495523876130587,67000490,1,0,1,0
6bba_07477033,3812,68,3.133333333333333,163.99999999999994,75.99999999999999,cont,5.7022477445545565,69000503,1,0,0,1
6bba_07477033,3816,68,6.880000000000001,156.0,53.919999999999995,cont,6.49335441528491,69000503,1,0,1,0
6bba_07477033,3929,70,3.36,164.64,72.79999999999998,cont,4.920209122842228,71000513,1,1,0,0
6bba_07477033,3984,71,3.1200000000000006,165.28000000000003,68.64000000000001,cont,4.118148855978871,72000518,1,0,1,0
6bba_07477033,4849,84,59.68000000000001,104.0,168.0,cont,1.0467068894872056,85000606,1,1,0,0
6bba_07477033,4917,85,59.83999999999999,104.0,167.36000000000004,cont,1.3704059617500064,86000611,1,0,1,0
6bba_07477033,5132,89,17.28,162.72,41.92,cont,3.177571055460444,90000630,1,1,0,0
6bba_07477033,5204,90,18.36,161.28,40.640000000000015,cont,1.4003314339826802,91000638,1,0,1,0
6bba_07477033,5320,92,9.520000000000001,138.72000000000003,152.0,cont,1.4762478831483563,93000651,1,0,0,1
6bba_07e24132,1605,3,46.84,247.36,51.99999999999999,cont,5.875043616859367,4000041,2,1,1,0
6bba_07e24132,2019,4,47.68000000000001,244.64000000000001,51.19999999999999,cont,5.334852973606683,5000051,1,0,1,0
6bba_07e24132,3596,8,36.36,200.64,33.279999999999994,cont,1.269269598036598,9000087,1,1,0,0
6bba_07e24132,3999,9,35.99999999999999,199.36,34.72,cont,1.4425021663761926,10000095,1,0,1,0
6bba_07e24132,4039,9,41.519999999999996,230.08000000000004,37.44000000000001,cont,3.295546474031271,10000098,1,1,0,0
6bba_07e24132,4467,10,40.48,225.28000000000003,36.63999999999999,cont,0.9625982092752959,11000105,1,0,1,0
6bba_07e24132,4559,10,56.84,75.36,37.44000000000001,cont,6.953557093315617,11000108,1,1,0,0
6bba_07e24132,4579,10,60.16000000000001,84.0,28.639999999999997,cont,0.891827477710801,11000108,1,1,0,0
6bba_07e24132,4990,11,56.160000000000004,73.27999999999999,37.28000000000001,cont,6.755564951393777,12000119,1,0,1,0
6bba_07e24132,5013,11,59.84,84.0,28.640000000000008,cont,1.5275864337247895,12000119,1,0,1,0
6bba_07e24132,5026,11,61.99999999999999,83.99999999999997,39.999999999999986,cont,6.3588183110150105,12000119,6,0,0,6
6bba_07e24132,6418,15,18.04,18.720000000000002,225.28,cont,4.999721242229412,16000151,1,1,0,0
6bba_07e24132,8458,20,34.36,138.72000000000003,24.0,cont,2.6981851539507056,21000192,2,2,0,0
6bba_07e24132,8835,21,33.07999999999999,137.28000000000003,24.0,cont,0.8186517116576393,22000201,2,0,2,0
6bba_07e24132,8975,21,55.12,78.56,27.200000000000003,cont,1.8014794267490317,22000202,4,1,1,2
6bba_07e24132,9390,22,55.879999999999995,77.28,27.36,cont,3.816121435764335,23000209,1,0,1,0
6bba_07e24132,10608,26,17.0,16.0,232.0,cont,5.525597394173774,27000237,1,0,1,0
6bba_07e24132,10824,26,55.99999999999999,74.72,26.720000000000002,cont,2.326532425091051,27000242,1,1,0,0
6bba_07e24132,11201,27,55.84000000000001,72.64,25.28,cont,2.0657962296654317,28000250,1,0,1,0
6bba_07e24132,11701,29,9.062265625,13.294800262451172,19.05174865722656,cont,4.7978971101525705,30000261,1,1,0,0
6bba_07e24132,12104,30,10.261411437988281,12.492335968017578,17.291378479003903,cont,4.501372318545779,31000271,1,0,1,0
6bba_07e24132,19189,50,25.52,76.48,17.920000000000005,cont,1.3475747892046663,51000418,1,1,0,0
6bba_07e24132,19539,51,25.52,74.56,17.279999999999994,cont,2.0132300694406497,52000427,1,0,1,0
6bba_07e24132,19923,52,32.32,128.0,61.28,cont,1.356947977079447,53000439,1,1,0,0
6bba_07e24132,20257,53,32.68,128.0,62.56,cont,1.3370501579970753,54000448,1,0,1,0
6bba_07e24132,24362,65,33.120000000000005,122.07999999999997,72.64,parent,0.6949606553611535,66000550,1,1,0,0
6bba_07e24132,36619,65,31.9489444732666,136.67319396972655,66.40318145751954,parent,6.475946698047403,66000550,1,1,0,0
6bba_07e24132,24698,66,31.971585083007817,136.16517333984376,66.38615112304687,daughter,1.7124410499518397,67000557,1,0,1,0
6bba_07e24132,24714,66,34.040000000000006,120.0,74.72,daughter,1.806967850433427,67000558,1,0,1,0
6bba_07e24132,26523,72,1.0,42.08,10.720000000000002,cont,1.739053711792711,73000604,1,1,0,0
6bba_07e24132,27084,73,37.64000000000001,104.16,88.00000000000001,cont,3.528982236424556,74000619,2,0,2,0
6bba_07e24132,27088,73,37.36,106.56,77.28,cont,6.077282750127726,74000619,1,0,0,1
6bba_07e24132,27689,75,11.52,76.16,28.160000000000004,cont,6.489042326299008,76000639,5,5,0,0
6bba_07e24132,27705,75,16.0,77.43999999999998,29.760000000000005,cont,3.830211523206,76000639,1,1,0,0
6bba_07e24132,28039,76,10.919999999999996,73.28,22.560000000000002,cont,2.57109716366379,77000640,8,1,5,2
6bba_07e24132,28203,76,31.8,102.72000000000003,97.28000000000003,cont,1.9566241654952499,77000645,21,7,7,7
6bba_07e24132,28243,76,38.533333333333324,31.466666666666665,31.999999999999993,cont,2.490047027947254,77000647,5,1,0,4
6bba_07e24132,28496,77,11.519999999999996,70.56,19.200000000000003,cont,0.9475296829123683,78000658,1,0,1,0
6bba_07e24132,28659,77,33.71999999999999,103.36,96.63999999999999,cont,2.167776798934812,78000655,7,0,7,0
6bba_07e24132,28697,77,40.71999999999999,29.28000000000001,32.00000000000001,cont,5.338019881004579,78000657,1,0,1,0
6bba_07e24132,30958,83,38.31999999999999,25.28,32.64,cont,1.0314590697162875,84000716,1,1,0,0
6bba_07e24132,31317,84,37.8,25.44,34.080000000000005,cont,2.4563851743975342,85000722,2,1,1,0
6bba_07e24132,31648,85,37.68,24.64,34.559999999999995,cont,2.5487647130521904,86000728,1,0,1,0
6bba_07e24132,33162,90,7.3199999999999985,62.71999999999999,12.0,cont,0.7218000502216635,91000760,1,1,0,0
6bba_07e24132,33514,91,7.000000000000001,63.36,12.0,cont,0.8255575767322346,92000768,1,0,1,0
6bba_07e24132,33531,91,10.0,148.00000000000003,99.36000000000001,cont,3.9764716222425074,92000772,1,1,0,0
6bba_07e24132,33851,92,9.639999999999999,148.00000000000003,99.36000000000001,cont,2.6441601336719245,93000779,1,0,1,0
6bba_07e24132,34148,93,6.32,64.64,12.640000000000004,cont,0.5994906692351433,94000780,1,1,0,0
6bba_07e24132,34472,94,6.16,65.28,13.280000000000001,cont,0.4885821322971194,95000787,1,0,1,0
6bba_085bf656,933,6,35.84,235.99999999999994,215.36,cont,3.536457001717957,7000117,2,2,0,0
6bba_085bf656,1064,7,35.199999999999996,232.0,215.35999999999996,cont,4.669747885325295,8000132,2,0,2,0
6bba_085bf656,1265,9,4.680000000000001,236.64000000000001,240.00000000000006,cont,0.860636573996231,10000155,1,1,0,0
6bba_085bf656,1396,10,4.319999999999999,234.0800000000001,240.00000000000006,cont,1.1664964905647903,11000170,1,0,1,0
6bba_085bf656,1427,10,25.480000000000004,223.99999999999997,180.0,cont,1.0218136449470607,11000173,1,1,0,0
6bba_085bf656,1447,10,37.000000000000014,40.0,21.280000000000005,cont,1.7003588775608314,11000178,1,1,0,0
6bba_085bf656,1556,11,25.519999999999996,220.0,180.0,cont,0.7800000000000082,12000190,1,0,1,0
6bba_085bf656,1577,11,35.68,33.92,20.640000000000004,cont,2.1609445272843106,12000195,1,0,1,0
6bba_085bf656,1654,12,3.839999999999999,227.36000000000004,240.00000000000006,cont,0.7334134662657916,13000203,1,1,0,0
6bba_085bf656,1717,12,40.0,248.64000000000007,157.28,cont,1.7748987647187051,13000213,1,0,1,0
6bba_085bf656,1785,13,3.6799999999999997,222.71999999999997,240.00000000000006,cont,0.9610882503703617,14000220,1,0,1,0
6bba_085bf656,2232,16,28.32,184.64000000000001,209.28000000000003,cont,1.2695816240005924,17000283,1,1,0,0
6bba_085bf656,2361,17,27.679999999999996,178.08,208.64,cont,0.948782937504679,18000301,1,0,1,0
6bba_085bf656,3235,24,54.680000000000014,124.00000000000003,20.000000000000007,cont,1.2612345142755776,25000437,1,1,0,0
6bba_085bf656,3333,25,48.67999999999999,231.36,180.0,cont,5.462030872303827,26000453,1,1,0,0
6bba_085bf656,3338,25,53.67999999999999,117.92000000000002,20.640000000000008,cont,0.5411518386737766,26000454,1,0,1,0
6bba_085bf656,3442,26,48.0,225.28,179.36,cont,4.890683106683564,27000471,1,0,1,0
6bba_085bf656,3455,26,56.32000000000001,247.36,98.08,cont,1.3773253065271016,27000475,1,1,0,0
6bba_085bf656,3561,27,56.16000000000001,244.64000000000001,98.72000000000001,cont,1.7396609763111703,28000491,1,0,1,0
6bba_085bf656,3881,30,54.84,235.99999999999994,97.92000000000002,cont,1.3211559805715816,31000541,1,1,0,0
6bba_085bf656,3883,30,55.68000000000001,155.35999999999999,44.00000000000001,cont,1.360058018063927,31000543,1,1,0,0
6bba_085bf656,3986,31,54.160000000000004,232.64,98.72,cont,1.9079742202136822,32000559,1,0,1,0
6bba_085bf656,3988,31,55.00000000000001,152.0,43.36,cont,2.1050493610601992,32000560,2,1,1,0
6bba_085bf656,4095,32,54.32000000000001,148.00000000000003,42.72,cont,1.1827946250300552,33000577,1,0,1,0
6bba_085bf656,4757,39,33.999999999999986,252.0,187.99999999999994,cont,5.041429918683779,40000689,1,0,0,1
6bba_085bf656,5161,43,58.480000000000004,181.92000000000002,48.0,cont,0.8456247690317485,44000763,1,1,0,0
6bba_085bf656,5255,44,57.83999999999999,180.0,51.36000000000001,cont,1.8906649789161678,45000780,1,0,1,0
6bba_085bf656,5883,51,56.32,153.27999999999997,44.00000000000001,cont,1.4024041856754454,52000877,1,1,0,0
6bba_085bf656,5969,52,55.84,152.63999999999996,45.28,cont,0.38488837667561066,53000889,1,0,1,0
6bba_085bf656,7003,65,27.32,88.64,128.0,cont,1.186361296148856,66001029,1,1,0,0
6bba_085bf656,7078,66,26.839999999999996,89.92,131.35999999999999,cont,0.7519027945818503,67001040,1,0,1,0
6bba_085bf656,7821,76,60.16000000000001,168.0,55.35999999999999,cont,0.5479407472528492,77001170,1,1,0,0
6bba_085bf656,7881,77,60.000000000000014,171.99999999999994,60.640000000000015,cont,0.9825540697590339,78001181,1,0,1,0
6bba_085bf656,7996,79,52.800000000000004,121.92000000000002,65.92,cont,1.2854972287017976,80001204,1,1,0,0
6bba_085bf656,8061,80,52.68000000000001,127.36,72.00000000000001,cont,1.2592439447938477,81001216,1,0,1,0
6bba_085bf656,8191,82,58.999999999999986,193.27999999999997,88.64,cont,1.6714608131811233,83001242,1,1,0,0
6bba_085bf656,8256,83,58.680000000000014,197.28,93.91999999999999,cont,1.2771507595033507,84001255,1,0,1,0
6bba_085bf656,8741,91,56.99999999999999,244.63999999999996,128.0,cont,0.9587500000000233,92001357,1,1,0,0
6bba_085bf656,8796,92,56.00000000000001,248.32,132.0,cont,1.808720628648869,93001369,1,0,1,0
6bba_09961292,913,2,7.839999999999999,15.36,47.360000000000014,cont,1.1337796302633054,3000094,1,1,0,0
6bba_09961292,1615,4,4.840000000000001,27.36,121.28000000000003,cont,0.319261843946316,5000154,1,1,0,0
6bba_09961292,1688,4,9.999999999999998,7.999999999999998,47.466666666666654,cont,6.864917164366305,5000171,4,1,0,3
6bba_09961292,2080,5,11.32,178.72000000000003,120.00000000000003,cont,0.5966206918972977,6000222,1,0,1,0
6bba_09961292,3061,8,3.1599999999999993,147.36000000000004,48.640000000000015,cont,6.918531726457584,9000356,1,0,1,0
6bba_09961292,4455,12,1.0,42.720000000000006,142.08,cont,1.876013059656032,13000470,1,1,0,0
6bba_09961292,4758,12,30.480000000000004,170.72000000000003,10.720000000000002,cont,6.045393132377728,13000507,1,1,0,0
6bba_09961292,5196,14,2.1600000000000006,207.99999999999997,237.92000000000002,cont,5.4135918356023005,15000560,2,2,0,0
6bba_09961292,5607,15,5.0,232.64000000000004,203.35999999999999,cont,0.5715289253397217,16000606,1,1,0,0
6bba_09961292,6916,19,1.866666666666666,176.53333333333325,72.0,cont,5.523141027128972,20000746,4,1,0,3
6bba_09961292,7034,19,8.84,168.64,148.00000000000006,cont,4.810576401274595,20000767,1,0,1,0
6bba_09961292,7068,19,12.16,184.64,134.72,cont,6.793219160493801,20000773,1,0,1,0
6bba_09961292,7088,19,14.840000000000003,221.27999999999997,80.64,cont,0.9731023199026729,20000774,1,0,1,0
6bba_09961292,7110,19,17.32,183.20000000000002,53.280000000000015,cont,1.1574061787030632,20000749,1,1,0,0
6bba_09961292,8022,22,6.84,196.64000000000001,116.0,cont,1.0732383821872893,23000877,1,0,1,0
6bba_09961292,8453,23,16.866666666666667,167.99999999999994,32.533333333333324,cont,0.861999150070449,24000906,5,0,0,5
6bba_09961292,9364,26,8.68,177.28,44.96,cont,3.686780862080086,27000998,4,0,2,2
6bba_09961292,9383,26,10.999999999999996,195.46666666666664,103.46666666666663,cont,4.180670218065768,27001009,4,0,0,4
6bba_09961292,9693,27,8.84,220.64000000000004,56.0,cont,4.095257928384989,28001022,2,2,0,0
6bba_09961292,10435,29,23.48,128.64000000000001,21.279999999999998,cont,1.0666983230979643,30001109,1,1,0,0
6bba_09961292,11119,32,1.0,12.640000000000004,136.8,cont,2.1835550141913043,33001169,1,0,1,0
6bba_09961292,13124,38,3.4000000000000004,27.360000000000003,100.64000000000003,cont,2.971440359909647,39001338,2,2,0,0
6bba_09961292,13162,38,7.000000000000002,184.0,39.519999999999996,cont,2.7860102521347643,39001347,4,4,0,0
6bba_09961292,13267,38,17.84,175.2,6.72,cont,6.134348095967495,39001361,4,4,0,0
6bba_09961292,13340,39,2.12,26.880000000000003,100.80000000000001,cont,3.1771970744667355,40001363,3,1,2,0
6bba_09961292,13405,39,3.8,189.28,36.31999999999999,cont,4.350874912014825,40001366,9,2,4,3
6bba_09961292,13435,39,5.4799999999999995,226.72,171.2,cont,3.2399500864210817,40001375,7,5,2,0
6bba_09961292,13452,39,6.68,6.08,34.56,cont,0.6795920191556106,40001377,6,3,1,2
6bba_09961292,13464,39,8.959999999999999,154.87999999999997,37.92,cont,2.6002031170660587,40001379,9,4,5,0
6bba_09961292,13470,39,8.960000000000003,241.43999999999997,101.92,cont,2.0597155446808726,40001378,8,0,2,6
6bba_09961292,13472,39,9.280000000000003,249.44,84.64000000000001,cont,6.723377360002927,40001378,9,1,7,1
6bba_09961292,13475,39,9.16,115.83999999999999,55.99999999999999,cont,5.288045627876149,40001382,12,4,4,4
6bba_09961292,13534,39,18.64,132.64000000000001,30.560000000000002,cont,1.0745907651287523,40001383,8,1,2,5
6bba_09961292,13559,39,25.60000000000001,200.64,19.36,cont,4.285103739992782,40001385,6,3,3,0
6bba_09961292,13597,40,1.64,21.759999999999998,100.0,cont,6.802852457792981,41001388,7,0,1,6
6bba_09961292,13737,40,7.240000000000002,200.0,109.91999999999999,cont,3.1396581223120426,41001400,2,0,2,0
6bba_09961292,13739,40,9.16,234.56000000000006,171.84000000000003,cont,2.082157084612035,41001401,6,0,1,5
6bba_09961292,13791,40,12.560000000000004,188.64,108.64000000000001,cont,6.776425357443269,41001400,3,0,3,0
6bba_09961292,14597,43,3.6800000000000006,225.28,184.64,cont,6.761425630922823,44001463,2,0,2,0
6bba_09961292,14911,44,4.000000000000001,223.36000000000004,132.8,parent,0.9308002269552703,45001493,1,1,0,0
6bba_09961292,14972,44,9.52,204.00000000000003,110.71999999999998,cont,4.20708702667297,45001494,1,1,0,0
6bba_09961292,15159,45,0.9999999999999998,219.46666666666658,133.06666666666663,daughter,5.278471491324199,46001531,3,0,0,3
6bba_09961292,15199,45,3.6400000000000006,223.84,130.72,daughter,0.5994906692351428,46001531,7,0,1,6
6bba_09961292,15211,45,4.0,90.72,41.28000000000001,cont,0.7079481796007363,46001508,1,0,1,0
6bba_09961292,15245,45,5.866666666666664,219.99999999999997,141.06666666666666,daughter,0.5964485460251974,46001530,4,0,0,4
6bba_09961292,15252,45,8.120000000000003,203.20000000000002,111.36,cont,1.9020137354919453,46001518,6,0,1,5
6bba_09961292,15799,47,1.3199999999999998,236.64000000000004,136.96000000000004,cont,5.3218687977063315,48001580,6,0,0,6
6bba_09961292,15834,47,2.84,228.0,125.28000000000002,cont,1.6152205731725926,48001580,1,0,1,0
6bba_09961292,16330,48,21.160000000000004,138.72000000000003,32.0,cont,0.9990776996810564,49001597,1,0,1,0
6bba_09961292,17221,51,15.280000000000001,19.2,44.16,cont,4.092452084325486,52001669,5,0,0,5
6bba_09961292,17390,52,1.8400000000000005,42.72,103.36,parent,2.779700129960063,53001680,1,0,1,0
6bba_09961292,17392,52,1.5999999999999996,55.466666666666654,108.53333333333333,parent,2.895384407648755,53001680,4,0,0,4
6bba_09961292,19903,60,1.1600000000000001,188.64000000000001,7.360000000000001,cont,3.518903637853697,61001887,1,1,0,0
6bba_09961292,20273,61,3.839999999999999,200.0,40.000000000000014,cont,1.6456685571523815,62001917,1,1,0,0
6bba_09961292,21650,65,10.160000000000002,23.200000000000006,16.0,cont,2.2446773849486705,66002037,2,2,0,0
6bba_09961292,21834,66,1.3200000000000003,110.72000000000003,17.279999999999998,cont,3.3499050826105443,67002048,2,0,2,0
6bba_09961292,21910,66,5.000000000000002,4.000000000000001,66.72,cont,0.8204238614886815,67002061,2,0,2,0
6bba_09961292,22005,66,12.999999999999998,132.53333333333333,56.533333333333324,cont,6.361874423513529,67002070,5,0,0,5
6bba_09961292,23270,70,11.999999999999996,228.0,100.0,cont,6.953879874753367,71002179,6,0,0,6
6bba_09961292,23893,72,11.680000000000003,61.44000000000001,46.72,cont,2.3865380915669405,73002236,4,0,0,4
6bba_09961292,25791,78,12.64,78.56,48.0,cont,2.705368794175758,79002370,1,1,0,0
6bba_09961292,25833,78,16.0,248.00000000000006,80.00000000000003,cont,6.080193253507682,79002375,1,1,0,0
6bba_09961292,25837,78,17.0,225.44000000000003,70.56,cont,6.005732678033539,79002375,1,1,0,0
6bba_09961292,26138,79,15.999999999999996,239.46666666666658,67.46666666666667,cont,1.8484250991647675,80002400,6,0,0,6
6bba_09961292,26434,80,14.84,145.27999999999997,48.0,cont,6.960616890764789,81002422,1,1,0,0
6bba_09961292,26629,81,4.0,118.72000000000003,118.72000000000003,cont,6.729501513856725,82002428,2,0,2,0
6bba_09961292,26968,82,7.840000000000001,32.64,63.36,cont,0.9037396264964822,83002456,2,2,0,0
6bba_09961292,28906,88,19.999999999999993,164.5333333333333,23.999999999999993,cont,6.7362952437605585,89002604,6,0,0,6
6bba_09961292,29478,90,13.999999999999998,243.36000000000004,71.36,cont,0.2983103459486235,91002648,1,0,1,0
6bba_09961292,30042,92,8.360000000000001,48.640000000000015,71.99999999999999,cont,1.146402661589724,93002690,1,1,0,0
6bba_09961292,30272,93,2.640000000000001,248.0,106.72,cont,0.8818526308289845,94002708,10,4,2,4
6bba_09961292,30296,93,4.160000000000001,216.16000000000003,131.36,cont,1.5903429268557128,94002709,14,4,5,5
6bba_09961292,30341,93,11.24,151.83999999999997,46.56000000000001,cont,5.285947910498181,94002714,8,5,3,0
6bba_09961292,30352,93,10.32,242.88000000000002,66.24,cont,2.963966145639982,94002716,7,3,4,0
6bba_09961292,30360,93,10.68,157.44000000000003,53.280000000000015,cont,2.1600278499361933,94002714,4,3,1,0
6bba_09961292,30395,93,15.32,25.28,37.440000000000005,cont,2.8492696432594773,94002717,5,3,2,0
6bba_09961292,30426,93,18.880000000000003,139.04000000000005,22.880000000000003,cont,6.923510783013181,94002718,9,1,2,6
6bba_09961292,30586,94,3.880000000000001,250.08000000000004,107.36000000000001,cont,3.4749552605033602,95002727,4,0,4,0
6bba_09961292,30612,94,7.840000000000001,204.48000000000002,142.39999999999998,cont,6.2946467831801245,95002729,4,0,0,4
6bba_09961292,30626,94,9.68,66.56000000000002,32.16,cont,5.739747517312932,95002730,10,0,2,8
6bba_09961292,30635,94,9.999999999999998,215.99999999999994,131.46666666666664,cont,1.4917975966747121,95002729,5,0,0,5
6bba_09961292,30655,94,12.839999999999998,77.28,28.639999999999997,cont,3.3837869410617443,95002730,13,0,4,9
6bba_09961292,30661,94,12.200000000000001,158.24,49.440000000000005,cont,3.042571126941818,95002732,9,0,5,4
6bba_09961292,30674,94,12.719999999999999,243.19999999999993,80.64,cont,5.7490332067661765,95002733,4,0,4,0
6bba_09961292,30739,94,23.200000000000003,160.79999999999998,20.640000000000004,cont,6.4583477956827275,95002736,5,0,5,0
6bba_09961292,31003,95,15.160000000000004,244.64000000000001,64.47999999999999,cont,5.317525417193605,96002756,4,0,0,4
6bba_0c7fa718,247,1,39.0,172.96000000000004,232.32,cont,0.6630716873008579,2000038,1,0,1,0
6bba_0c7fa718,475,2,39.16,175.83999999999997,225.28,cont,2.7395109622887017,3000067,2,2,0,0
6bba_0c7fa718,577,2,53.519999999999996,232.0,114.72,cont,1.5965573314165735,3000081,2,2,0,0
6bba_0c7fa718,707,3,41.16,158.24,208.8,cont,2.518330610245606,4000098,5,0,2,3
6bba_0c7fa718,715,3,43.133333333333326,136.00000000000003,175.99999999999994,cont,3.1402654921377526,4000100,3,0,0,3
6bba_0c7fa718,747,3,47.84,140.00000000000003,165.92000000000002,cont,6.223134760713454,4000100,3,0,3,0
6bba_0c7fa718,796,3,53.639999999999986,229.28000000000003,105.28000000000002,cont,1.1989813384702663,4000109,2,0,2,0
6bba_0c7fa718,1143,5,37.32000000000001,236.64000000000004,138.72000000000003,cont,1.2406531797806966,6000157,1,1,0,0
6bba_0c7fa718,1639,7,42.67999999999999,224.0,243.36000000000004,cont,1.5248181244003975,8000233,1,1,0,0
6bba_0c7fa718,1905,8,45.84,220.0,162.72,cont,1.5925829060052177,9000269,1,1,0,0
6bba_0c7fa718,2121,9,42.96,217.44000000000003,231.36,cont,0.921820549239372,10000297,2,2,0,0
6bba_0c7fa718,2720,11,55.320000000000014,151.36,77.28,cont,1.1973284062862644,12000378,1,0,1,0
6bba_0c7fa718,2868,12,43.99999999999999,125.75999999999999,143.2,cont,2.747451660175294,13000411,1,1,0,0
6bba_0c7fa718,2869,12,44.32,124.16000000000003,127.2,cont,6.772371491951692,13000411,3,2,1,0
6bba_0c7fa718,2877,12,43.04,247.84000000000006,207.04000000000002,cont,2.5748944153110727,13000412,1,1,0,0
6bba_0c7fa718,3138,13,45.0,114.56000000000003,124.15999999999998,cont,2.2479688748957467,14000442,5,0,2,3
6bba_0c7fa718,3153,13,46.16,233.27999999999997,60.640000000000015,cont,0.7241740985426073,14000441,2,0,2,0
6bba_0c7fa718,3179,13,48.99999999999999,76.53333333333335,95.46666666666664,cont,3.2627268173603374,14000444,5,0,0,5
6bba_0c7fa718,3201,13,51.52,95.99999999999997,61.27999999999999,cont,2.4938853346735916,14000432,1,0,1,0
6bba_0c7fa718,3422,14,46.839999999999996,228.64000000000007,55.36,cont,0.6278895703067261,15000473,1,1,0,0
6bba_0c7fa718,3618,15,40.48,119.36000000000001,232.0,cont,5.833840528545494,16000503,2,0,2,0
6bba_0c7fa718,3802,15,59.83999999999999,118.07999999999998,61.91999999999999,cont,6.412349657106971,16000514,1,0,1,0
6bba_0c7fa718,4196,17,46.36,101.12,108.80000000000003,cont,2.7387397306973136,18000577,5,5,0,0
6bba_0c7fa718,4203,17,47.22444335937501,181.7432177734375,111.42317016601564,cont,1.7564276337376894,18000578,4,4,0,0
6bba_0c7fa718,4266,17,53.36000000000001,85.12,45.92,cont,3.401222436492496,18000582,3,3,0,0
6bba_0c7fa718,4295,17,54.75999999999999,71.20000000000002,74.56,cont,6.466846700672593,18000581,2,2,0,0
6bba_0c7fa718,4419,18,41.52,169.92000000000002,55.35999999999999,cont,0.8228342785761922,19000612,10,4,4,2
6bba_0c7fa718,4481,18,47.624443359375,178.70321777343747,105.02317016601563,cont,2.3519208586668534,19000618,6,2,4,0
6bba_0c7fa718,4563,18,56.73333333333333,171.46666666666664,18.93333333333333,cont,1.7216566331588927,19000624,5,0,2,3
6bba_0c7fa718,4704,19,42.37154052734375,163.44172607421876,30.49307861328125,cont,1.8481675082304865,20000649,6,4,2,0
6bba_0c7fa718,4719,19,44.96,149.6,225.76,cont,6.341186427830365,20000647,6,2,4,0
6bba_0c7fa718,4723,19,45.28,211.36,149.28,cont,2.281606478996762,20000654,2,2,0,0
6bba_0c7fa718,4736,19,45.160000000000004,168.48000000000002,160.16000000000003,cont,5.048391451739865,20000640,3,1,2,0
6bba_0c7fa718,14554,19,46.739478607177745,227.83050842285155,49.46509872436524,cont,6.196626128620828,20000656,4,2,2,0
6bba_0c7fa718,5008,20,44.00000000000001,157.92000000000002,12.16,cont,2.401758795653718,21000688,8,4,0,4
6bba_0c7fa718,5027,20,46.48,158.72,123.68000000000002,cont,4.201245722996921,21000678,12,3,5,4
6bba_0c7fa718,5032,20,45.84,207.20000000000005,133.28000000000003,cont,1.6254061992314546,21000691,2,0,2,0
6bba_0c7fa718,5042,20,46.67999999999999,143.04,199.52,cont,4.09709519812512,21000690,6,2,1,3
6bba_0c7fa718,5044,20,46.380771484374996,230.6677783203125,50.08348327636718,cont,2.1716821065130674,21000697,3,1,2,0
6bba_0c7fa718,5301,21,45.18296569824219,232.06618408203127,48.465292358398436,cont,6.370418997446127,22000741,5,0,1,4
6bba_0c7fa718,5313,21,46.79999999999999,130.08,188.64000000000001,cont,4.822856551101208,22000729,4,1,1,2
6bba_0c7fa718,5606,22,47.480000000000004,133.92000000000002,173.91999999999996,cont,0.9422197859310736,23000777,5,1,1,3
6bba_0c7fa718,5637,22,51.120000000000005,150.07999999999998,51.839999999999996,cont,5.091309384627883,23000785,4,1,1,2
6bba_0c7fa718,6213,24,51.280000000000015,150.56,83.68,cont,3.4774240645915095,25000851,3,3,0,0
6bba_0c7fa718,6259,24,54.88,140.16,11.52,cont,2.8808367382585227,25000858,5,1,1,3
6bba_0c7fa718,6500,25,51.24,133.92000000000002,54.72000000000001,cont,3.4329062487198714,26000885,5,0,3,2
6bba_0c7fa718,6502,25,51.79999999999999,187.99999999999994,46.93333333333331,cont,1.9754131072878083,26000884,5,0,0,5
6bba_0c7fa718,7032,27,51.994713134765625,174.63490966796874,15.172422485351563,cont,1.9224898829831651,28000956,3,0,3,0
6bba_0c7fa718,9187,37,34.4,236.47999999999996,177.92000000000002,cont,2.923780995987907,38001230,1,0,1,0
6bba_0c7fa718,9436,38,58.08000000000001,24.640000000000008,139.84,cont,2.9187189505843,39001262,2,2,0,0
6bba_0c7fa718,9506,39,34.0,195.99999999999991,233.5999999999999,cont,0.7665109669795828,40001273,3,0,0,3
6bba_0c7fa718,9677,40,39.32,98.08000000000001,107.04000000000002,cont,4.036547101174472,41001297,2,2,0,0
6bba_0c7fa718,9787,41,32.480000000000004,145.60000000000002,196.16000000000003,cont,3.130814590485988,42001313,3,0,2,1
6bba_0c7fa718,9832,41,42.16,87.52,80.16,cont,3.011340285072408,42001311,4,0,1,3
6bba_0c7fa718,9879,41,59.519999999999996,20.000000000000007,113.92,cont,0.8800541531633179,42001325,1,0,1,0
6bba_0c7fa718,10243,44,47.32,102.08000000000001,57.44,cont,2.347773546362593,45001379,1,0,1,0
6bba_0c7fa718,10366,45,44.84000000000001,77.44,51.52000000000001,cont,4.524421406378951,46001392,2,1,1,0
6bba_0c7fa718,10376,45,48.0,100.00000000000001,44.63999999999999,cont,1.898956950538898,46001397,1,1,0,0
6bba_0c7fa718,10412,46,26.0,194.56000000000006,169.60000000000002,cont,3.5011491417104823,47001404,2,0,2,0
6bba_0c7fa718,10447,46,34.32,125.44000000000003,151.2,cont,0.9537794884038923,47001408,7,0,3,4
6bba_0c7fa718,10495,46,49.959999999999994,36.160000000000004,35.36,cont,2.916546813013629,47001417,4,0,1,3
6bba_0c7fa718,10563,47,33.84,128.64000000000001,151.84000000000003,cont,1.0345266127558181,48001426,1,1,0,0
6bba_0c7fa718,10644,48,26.866666666666664,189.0666666666666,139.99999999999994,cont,0.9459801648143772,49001440,5,0,0,5
6bba_0c7fa718,10675,48,35.160000000000004,117.44,132.16,cont,1.8963130345488823,49001448,4,0,1,3
6bba_0c7fa718,11248,54,31.64,168.47999999999996,51.360000000000014,cont,3.634373441530175,55001513,5,0,1,4
6bba_0c7fa718,11460,56,39.52000000000001,88.64,53.760000000000005,cont,2.1570918646409094,57001542,2,2,0,0
6bba_0c7fa718,11550,57,41.60000000000001,80.15999999999998,23.36,cont,3.582268179868739,58001560,3,0,0,3
6bba_0c7fa718,11982,63,27.52,143.99999999999997,156.16000000000003,cont,1.7908207790005053,64001628,1,1,0,0
6bba_0c7fa718,12001,63,37.32,57.44,172.79999999999998,cont,1.9981553993621268,64001632,1,1,0,0
6bba_0c7fa718,12008,63,40.000000000000014,39.36,198.88000000000002,cont,4.004891540354089,64001633,1,1,0,0
6bba_0c7fa718,12082,64,40.000000000000014,34.56,187.20000000000002,cont,2.0934137789744143,65001643,1,0,1,0
6bba_0c7fa718,12454,70,23.479999999999997,234.72,232.0,cont,1.0721306182084374,71001704,1,0,1,0
6bba_0c7fa718,12466,70,28.16,128.0,101.91999999999999,cont,0.9313674422589586,71001706,1,0,1,0
6bba_0c7fa718,12470,70,27.999999999999996,242.08,19.360000000000003,cont,1.308604337643737,71001707,2,0,2,0
6bba_0c7fa718,12589,72,23.68,224.64000000000004,222.56,cont,0.8327233183957355,73001727,1,0,1,0
6bba_0c7fa718,12734,74,24.799999999999997,209.60000000000002,214.08,cont,2.6666344073569532,75001752,1,1,0,0
6bba_0c7fa718,12942,77,25.439999999999998,188.95999999999998,198.72,cont,1.81396880141309,78001790,1,1,0,0
6bba_0c7fa718,13045,78,32.52,140.64000000000001,103.2,cont,1.5381807436059016,79001801,2,0,2,0
6bba_0c7fa718,13663,88,17.000000000000004,178.72000000000003,218.08000000000004,cont,0.9988133584408834,89001884,2,0,1,1
6bba_0c7fa718,14155,95,19.359999999999996,139.20000000000002,175.20000000000005,cont,3.237626445407202,96001948,1,0,1,0
6bba_0e7c0d07,576,18,4.640000000000001,105.92000000000002,16.640000000000004,cont,2.257873031195507,19000095,2,2,0,0
6bba_0e7c0d07,610,19,5.0,106.56000000000003,19.360000000000003,cont,3.258195915687084,20000101,2,0,2,0
6bba_0e7c0d07,651,20,5.48,108.0,21.280000000000005,cont,2.5566265003906996,21000107,1,1,0,0
6bba_0e7c0d07,693,21,6.48,110.72000000000003,25.92,cont,4.270288522161,22000115,2,1,1,0
6bba_0e7c0d07,736,22,7.5200000000000005,112.64,30.560000000000002,cont,4.039064905705279,23000121,2,1,1,0
6bba_0e7c0d07,783,23,8.680000000000001,114.72,38.080000000000005,cont,5.432873462772713,24000125,1,0,1,0
6bba_0e7c0d07,1519,32,18.16,79.99999999999999,9.760000000000003,cont,3.2920591010794418,33000182,2,1,1,0
6bba_0e7c0d07,1665,33,19.520000000000003,83.36000000000003,13.280000000000001,cont,2.512661997265044,34000189,1,0,1,0
6bba_0e7c0d07,1960,35,21.68,89.27999999999999,24.64,cont,2.163692229153678,36000205,2,2,0,0
6bba_0e7c0d07,2131,36,22.68,92.63999999999999,32.0,cont,2.2983842830345007,37000211,2,0,2,0
6bba_0e7c0d07,2527,38,24.040000000000003,91.68000000000004,45.44,cont,5.366831583904964,39000224,2,2,0,0
6bba_0e7c0d07,2775,39,26.400000000000002,108.0,69.12,cont,4.562576369223858,40000230,5,0,3,2
6bba_0e7c0d07,2918,40,7.480000000000001,40.64000000000001,112.0,cont,4.117699979964541,41000233,1,1,0,0
6bba_0e7c0d07,3189,41,8.16,44.00000000000001,115.35999999999996,cont,3.090150178389397,42000239,1,0,1,0
6bba_0e7c0d07,3310,41,28.68,115.36000000000004,75.36,cont,2.689166343869415,42000242,1,1,0,0
6bba_0e7c0d07,3594,42,29.0,119.36000000000001,80.64,cont,3.3632789525253464,43000248,1,0,1,0
6bba_0e7c0d07,5610,48,32.32,140.00000000000003,116.0,cont,1.5941573008959988,49000287,1,1,0,0
6bba_0e7c0d07,5999,49,32.84,143.99999999999997,122.07999999999997,cont,2.0800634755939487,50000293,1,0,1,0
6bba_0e7c0d07,6646,51,15.680000000000003,90.07999999999998,184.8,cont,2.120484909165829,52000303,5,5,0,0
6bba_0e7c0d07,6664,51,17.52,104.8,196.64000000000001,cont,6.137683296651921,52000303,3,3,0,0
6bba_0e7c0d07,6794,51,33.52,148.16000000000003,126.24000000000002,cont,3.7191205697718237,52000305,2,2,0,0
6bba_0e7c0d07,6799,51,35.16,161.44000000000003,146.07999999999998,cont,6.738971198372936,52000305,2,2,0,0
6bba_0e7c0d07,7079,52,18.16,106.56,202.72000000000003,cont,1.5922512560836546,53000310,5,0,3,2
6bba_0e7c0d07,7227,52,35.68,162.56000000000003,148.64000000000001,cont,1.5438355239467771,53000312,5,1,2,2
6bba_0e7c0d07,7671,53,36.00000000000001,165.91999999999996,150.56,cont,2.641562333638921,54000318,1,0,1,0
6bba_0e7c0d07,9250,57,6.959999999999999,77.6,224.64000000000001,parent,1.6787123041188412,58000343,1,1,0,0
6bba_0e7c0d07,9780,58,8.2,80.48000000000002,233.44,daughter,2.404231296069503,59000348,3,0,1,2
6bba_0e7c0d07,10570,59,40.16,211.84000000000006,187.20000000000002,cont,2.4359828611876875,60000350,3,0,0,3
6bba_0e7c0d07,11745,62,2.7200000000000006,9.28,71.36,cont,3.7194045642414313,63000357,2,1,1,0
6bba_0e7c0d07,12270,63,3.16,15.36,80.64,cont,4.655220859422246,64000359,4,3,1,0
6bba_0e7c0d07,12805,64,3.4799999999999995,21.280000000000005,91.35999999999999,cont,5.820789921694478,65000362,4,1,3,0
6bba_0e7c0d07,13224,64,41.84000000000001,237.92000000000004,210.56000000000003,cont,2.723075295047846,65000365,1,1,0,0
6bba_0e7c0d07,13375,65,3.6799999999999997,27.359999999999996,99.36000000000001,cont,5.407515603306199,66000366,1,0,1,0
6bba_0e7c0d07,13764,65,41.68,243.35999999999999,216.80000000000007,cont,3.6365161899818075,66000369,1,0,1,0
6bba_0e7c0d07,14013,66,12.0,39.36,113.44,cont,6.926332565831647,67000371,1,0,1,0
6bba_0e7c0d07,15015,68,4.68,60.800000000000004,135.36000000000004,cont,3.061950205751236,69000379,6,5,1,0
6bba_0e7c0d07,15518,69,3.6,52.47999999999999,132.64000000000001,cont,2.555180093946412,70000382,8,0,5,3
6bba_0e7c0d07,15532,69,3.6400000000000006,34.4,134.71999999999997,cont,5.497329607409397,70000382,9,0,5,4
6bba_0e7c0d07,15579,69,4.680000000000001,82.56000000000002,151.04,cont,6.034878960053798,70000383,9,0,5,4
6bba_0e7c0d07,16070,70,3.28,48.800000000000004,132.8,cont,2.6816931451976345,71000384,1,1,0,0
6bba_0e7c0d07,16102,70,4.079999999999999,37.28,143.99999999999997,cont,6.99199564680928,71000384,2,2,0,0
6bba_0e7c0d07,16137,70,5.200000000000001,90.55999999999999,155.83999999999997,cont,4.285873962799175,71000385,1,1,0,0
6bba_0e7c0d07,16627,71,3.1261869430541998,72.86188598632813,154.91258300781251,cont,5.706631202967377,72000385,6,0,1,5
6bba_0e7c0d07,16665,71,5.68,99.36000000000001,162.07999999999998,cont,1.8228994761094222,72000386,1,0,1,0
6bba_0e7c0d07,17146,72,3.286186943054199,68.70188598632812,157.15258300781252,cont,3.2118334845277436,73000386,1,1,0,0
6bba_0e7c0d07,17184,72,5.32,109.44000000000001,167.52000000000004,cont,2.0547737862110433,73000387,1,1,0,0
6bba_0e7c0d07,17202,72,7.840000000000002,73.28,143.36,cont,6.627948518395411,73000386,1,0,1,0
6bba_0e7c0d07,17717,73,7.0,81.92,152.0,cont,5.5803787113868895,74000387,1,1,0,0
6bba_0e7c0d07,17750,73,10.000000000000004,121.28000000000003,168.64,cont,4.253187224012144,74000388,6,5,1,0
6bba_0e7c0d07,18173,74,2.2,96.16000000000001,187.52000000000004,cont,6.9440948384220835,75000389,4,1,0,3
6bba_0e7c0d07,18202,74,5.6400000000000015,139.68,189.92000000000002,cont,5.50558537419047,75000390,6,1,1,4
6bba_0e7c0d07,18248,74,10.639999999999999,129.44000000000003,177.59999999999997,cont,6.316987637513633,75000390,7,2,5,0
6bba_0e7c0d07,18582,75,0.08616325378417967,99.95185424804686,181.86697021484375,cont,6.360244397110949,76000391,1,1,0,0
6bba_0e7c0d07,18665,75,4.280000000000001,143.2,192.16000000000003,cont,3.2833445695662102,76000392,6,5,1,0
6bba_0e7c0d07,18676,75,2.8800000000000003,101.92000000000002,192.16,cont,4.577887749825239,76000391,4,3,1,0
6bba_0e7c0d07,19119,76,3.16,109.91999999999999,193.92000000000002,cont,3.2465043700879272,77000394,6,0,3,3
6bba_0e7c0d07,19126,76,3.133333333333333,156.53333333333333,207.99999999999997,cont,4.7173227238080235,77000395,4,0,0,4
6bba_0e7c0d07,19609,77,3.6,163.04,212.00000000000006,cont,5.556003087877113,78000397,1,1,0,0
6bba_0e7c0d07,19645,77,8.8,160.0,201.28000000000003,cont,4.980803932348665,78000397,1,1,0,0
6bba_0e7c0d07,20054,78,3.0,126.08000000000001,199.2,cont,3.727914338138683,79000398,1,1,0,0
6bba_0e7c0d07,20077,78,3.52,169.28000000000003,216.64000000000001,cont,4.048957427536131,79000399,1,0,1,0
6bba_0e7c0d07,20106,78,8.159999999999998,165.27999999999997,205.92,cont,5.724153321671254,79000399,1,0,1,0
6bba_0e7c0d07,20500,79,3.160000000000001,134.08,204.64000000000001,cont,3.390803048320555,80000400,1,0,1,0
6bba_0e7c0d07,20568,79,7.839999999999999,173.28000000000003,212.0,cont,6.599380889522892,80000401,1,1,0,0
6bba_0e7c0d07,20884,80,0.0,145.92000000000002,208.64000000000001,cont,6.534136921774749,81000402,1,1,0,0
6bba_0e7c0d07,20939,80,3.4800000000000013,141.28000000000003,210.08000000000004,cont,1.7317499278186692,81000402,1,1,0,0
6bba_0e7c0d07,20998,80,7.520000000000001,181.28000000000003,218.72000000000006,cont,4.7537013539451385,81000403,1,0,1,0
6bba_0e7c0d07,21311,81,0.0,154.56,214.72,cont,6.551491361705363,82000404,1,0,1,0
6bba_0e7c0d07,21385,81,3.8400000000000007,149.27999999999997,216.0,cont,1.533452497634026,82000404,1,0,1,0
6bba_12665c0e,259,1,14.920000000000002,127.04000000000002,159.04,cont,1.5536393765929142,2000038,4,4,0,0
6bba_12665c0e,269,1,16.68,212.31999999999994,216.8,cont,4.024557817326031,2000040,1,1,0,0
6bba_12665c0e,423,2,13.280000000000001,125.28,153.92000000000002,cont,3.8217567949831697,3000059,4,0,4,0
6bba_12665c0e,432,2,15.160000000000004,213.92000000000002,210.56,cont,1.0725000000000073,3000058,1,0,1,0
6bba_12665c0e,455,2,21.0,92.0,156.15999999999997,cont,2.0597155446808695,3000063,3,0,1,2
6bba_12665c0e,1101,6,11.040000000000003,129.28,138.72000000000003,cont,4.314336746534744,7000147,1,1,0,0
6bba_12665c0e,1150,6,27.679999999999996,152.0,214.08,cont,5.423168181515306,7000154,1,1,0,0
6bba_12665c0e,1284,7,14.720000000000002,105.44,131.52,cont,3.3598225864619615,8000172,8,0,4,4
6bba_12665c0e,1310,7,26.12,154.08,209.28000000000003,cont,3.353844109749292,8000174,1,0,1,0
6bba_12665c0e,1324,7,31.68,131.35999999999999,148.64000000000001,cont,2.0657962296654597,8000178,1,1,0,0
6bba_12665c0e,1473,8,30.480000000000004,134.72000000000003,146.72000000000003,cont,0.9986811615826005,9000203,1,0,1,0
6bba_12665c0e,1650,9,48.84000000000001,101.28000000000002,243.36000000000004,cont,1.9131571583379963,10000229,1,1,0,0
6bba_12665c0e,1800,10,48.16,100.64000000000001,240.64000000000004,cont,0.3957133590112972,11000254,1,0,1,0
6bba_12665c0e,1945,11,47.32000000000001,100.0,237.28000000000003,cont,1.4668269325315744,12000275,1,1,0,0
6bba_12665c0e,2088,12,46.68,99.36000000000001,235.35999999999999,cont,1.574322274821762,13000295,1,0,1,0
6bba_12665c0e,2271,14,3.5200000000000005,232.0,164.8,cont,1.437826593334535,15000322,1,1,0,0
6bba_12665c0e,2365,14,44.52,97.28000000000003,228.16000000000003,cont,1.28662652409313,15000336,1,1,0,0
6bba_12665c0e,2415,15,3.16,231.36,157.92000000000002,cont,1.3513904876089597,16000340,1,0,1,0
6bba_12665c0e,2474,15,23.0,57.279999999999994,140.64000000000001,cont,1.9855611612337707,16000352,1,0,1,0
6bba_12665c0e,2507,15,42.67999999999999,96.63999999999999,221.28000000000006,cont,0.5520218519225655,16000357,1,0,1,0
6bba_12665c0e,2621,16,25.36,75.36,166.72000000000006,cont,2.625318135483769,17000374,1,1,0,0
6bba_12665c0e,2698,17,3.4800000000000004,114.71999999999997,91.36,cont,1.3408958059819622,18000383,1,1,0,0
6bba_12665c0e,2763,17,23.64,75.36,159.99999999999997,cont,3.1205500817163685,18000394,1,0,1,0
6bba_12665c0e,2834,18,2.4799999999999995,113.28000000000002,82.56000000000002,cont,1.0716379110968395,19000404,1,0,1,0
6bba_12665c0e,3188,20,46.48,86.88,223.36,parent,3.069013736121098,21000472,1,1,0,0
6bba_12665c0e,3312,21,40.32,117.28,233.27999999999997,cont,1.4070097947420215,22000493,1,0,1,0
6bba_12665c0e,3315,21,43.13333333333334,103.99999999999999,224.53333333333327,daughter,2.1586962773988247,22000494,2,0,0,2
6bba_12665c0e,3319,21,45.84,79.19999999999999,216.64,daughter,2.3045227271606485,22000495,1,0,1,0
6bba_12665c0e,3615,24,10.44,26.080000000000002,66.55999999999999,cont,0.9559917952053744,25000536,1,1,0,0
6bba_12665c0e,3621,24,11.68,52.160000000000004,86.88,cont,3.635390496562919,25000538,1,1,0,0
6bba_12665c0e,3622,24,12.0,145.44,89.76000000000002,cont,3.909535218539418,25000537,1,1,0,0
6bba_12665c0e,3633,24,16.520000000000003,68.80000000000001,125.60000000000002,cont,3.880621124575283,25000550,1,1,0,0
6bba_12665c0e,3671,24,37.519999999999996,119.36000000000001,224.15999999999997,cont,3.2863992929192407,25000552,1,1,0,0
6bba_12665c0e,3673,24,41.16,103.36,217.44000000000003,cont,0.34776518658427946,25000554,1,1,0,0
6bba_12665c0e,3679,24,44.52,60.800000000000004,189.76,cont,6.1721128118740625,25000555,1,1,0,0
6bba_12665c0e,3694,24,61.68,130.56,238.08000000000004,cont,1.1049999999999893,25000549,1,1,0,0
6bba_12665c0e,3719,25,4.039999999999999,220.16,145.44,cont,3.0107702502848017,26000552,4,0,2,2
6bba_12665c0e,3725,25,7.040000000000002,19.2,40.8,cont,3.82275853442249,26000555,6,0,3,3
6bba_12665c0e,3727,25,9.866666666666667,144.0,64.53333333333333,cont,2.24856918240368,26000556,2,0,0,2
6bba_12665c0e,3729,25,10.84,47.199999999999996,76.47999999999999,cont,2.520060143131507,26000557,1,0,1,0
6bba_12665c0e,3741,25,15.64,64.64000000000001,116.47999999999999,cont,2.874734770374474,26000575,1,0,1,0
6bba_12665c0e,3747,25,17.4,184.0,184.8,cont,3.3500233208143486,26000559,3,0,0,3
6bba_12665c0e,3759,25,23.919999999999998,80.47999999999999,156.79999999999998,cont,6.716933126435012,26000562,1,0,0,1
6bba_12665c0e,3781,25,36.31999999999999,118.08000000000001,217.92000000000002,cont,1.6755633381045223,26000576,1,0,1,0
6bba_12665c0e,3786,25,42.11999999999999,57.279999999999994,190.24000000000004,cont,2.0954940497410215,26000578,4,0,1,3
6bba_12665c0e,3788,25,40.36,102.72000000000001,212.0,cont,2.850474193007196,26000577,1,0,1,0
6bba_12665c0e,3807,25,61.32,131.36,234.56,cont,1.5818521580729312,26000573,1,0,1,0
6bba_12665c0e,4078,28,8.32,141.28000000000003,57.919999999999995,cont,1.3077969309109068,29000624,1,1,0,0
6bba_12665c0e,4195,29,7.6800000000000015,140.64,55.35999999999999,cont,0.8020325741514546,30000644,1,0,1,0
6bba_12665c0e,4297,30,2.713895320892334,217.60728515625001,132.75233642578124,cont,1.1964486800996148,31000664,1,1,0,0
6bba_12665c0e,4323,30,12.880000000000003,178.72,164.8,cont,3.1506985832510175,31000668,1,1,0,0
6bba_12665c0e,4360,30,30.560000000000002,106.08,195.51999999999998,cont,4.3248818625483905,31000684,1,1,0,0
6bba_12665c0e,4363,30,35.04,95.36000000000001,193.44,cont,2.5907413273038276,31000683,1,1,0,0
6bba_12665c0e,4399,31,1.0,203.52000000000007,114.88000000000002,cont,3.9954186654842387,32000685,2,0,1,1
6bba_12665c0e,4409,31,2.3600000000000003,223.20000000000005,114.72,cont,6.509032786251745,32000685,1,0,1,0
6bba_12665c0e,4430,31,11.799999999999999,173.44000000000003,160.0,cont,1.9446436852287283,32000687,1,0,1,0
6bba_12665c0e,4449,31,20.52,5.440000000000001,92.16,cont,4.115486947191063,32000690,2,0,1,1
6bba_12665c0e,4463,31,28.6,101.44,186.72000000000003,cont,2.514552954801306,32000703,1,0,1,0
6bba_12665c0e,4470,31,33.32000000000001,96.63999999999999,190.56,cont,2.241439966851679,32000704,1,0,1,0
6bba_12665c0e,4525,32,6.800000000000001,27.360000000000003,41.76,cont,1.978032877001795,33000706,1,1,0,0
6bba_12665c0e,4535,32,11.319999999999999,47.19999999999999,83.2,cont,1.5024880407178,33000723,2,2,0,0
6bba_12665c0e,4552,32,19.799999999999997,62.71999999999999,132.64000000000001,cont,2.0234349662146327,33000711,2,2,0,0
6bba_12665c0e,4568,32,27.640000000000004,96.63999999999999,178.72,cont,2.4157908254855287,33000726,1,1,0,0
6bba_12665c0e,4576,32,32.0,97.12,188.48,cont,2.4991739260203514,33000725,1,1,0,0
6bba_12665c0e,4631,33,5.866666666666664,43.46666666666665,55.466666666666654,cont,3.522603721562602,34000744,2,0,0,2
6bba_12665c0e,4634,33,7.040000000000002,24.0,40.8,cont,1.593494587377061,34000745,5,0,1,4
6bba_12665c0e,4646,33,13.2,67.84000000000002,108.0,cont,3.247602240499907,34000726,3,0,1,2
6bba_12665c0e,4661,33,20.0,90.71999999999998,146.72000000000003,cont,3.8004855775282147,34000746,3,0,1,2
6bba_12665c0e,4673,33,26.519999999999996,91.84,168.47999999999996,cont,3.0189158857112908,34000730,3,0,1,2
6bba_12665c0e,4682,33,32.00000000000001,43.999999999999986,83.46666666666664,cont,1.6857063512894925,34000735,1,0,0,1
6bba_12665c0e,6156,52,28.84000000000001,124.63999999999999,119.36000000000001,cont,1.9084585566629255,53001072,1,1,0,0
6bba_12665c0e,6223,53,27.52,124.64000000000001,115.36000000000004,cont,2.5931863845470113,54001086,1,0,1,0
6bba_12665c0e,6226,53,29.159999999999997,106.87999999999998,110.71999999999998,cont,1.4410369530307052,54001085,1,0,0,1
6bba_12665c0e,6818,63,31.84,217.44000000000005,232.63999999999993,cont,1.2060083954931569,64001197,1,1,0,0
6bba_12665c0e,6869,64,30.680000000000003,219.36,231.36,cont,0.6368673331236157,65001207,1,0,1,0
6bba_12665c0e,7720,79,60.16000000000001,38.08,185.28000000000003,cont,2.1927868255943292,80001319,1,1,0,0
6bba_12665c0e,7774,80,59.68000000000001,37.28000000000001,177.92,cont,1.471679525236381,81001327,1,0,1,0
6bba_12665c0e,8035,85,58.839999999999996,240.64000000000004,247.35999999999999,cont,1.4618679189653137,86001371,1,1,0,0
6bba_12665c0e,8086,86,57.84,240.00000000000006,244.00000000000003,cont,1.8353150274543821,87001381,1,0,1,0
6bba_12665c0e,8266,90,5.159999999999999,232.0,99.84,cont,0.8896040200561159,91001418,1,1,0,0
6bba_12665c0e,8319,91,2.2000000000000006,232.63999999999993,78.23999999999998,cont,4.014835379564153,92001428,1,0,0,1
6bba_12665c0e,8479,93,59.68000000000001,100.63999999999999,91.35999999999997,cont,1.5638039559036945,94001457,1,1,0,0
6bba_12665c0e,8538,94,58.48000000000002,99.36000000000001,85.92,cont,0.884692743273022,95001466,1,0,1,0
6bba_13f19531,398,5,19.861110229492184,118.02195556640629,149.33708007812498,cont,6.6311000201653645,6000062,1,1,0,0
6bba_13f19531,432,5,55.0,74.72,63.36,cont,1.9570964622368505,6000070,1,1,0,0
6bba_13f19531,467,6,18.320000000000004,103.36,143.99999999999997,cont,1.8901760830673926,7000074,1,0,0,1
6bba_13f19531,500,6,54.160000000000004,76.0,66.72,cont,1.9023607866280217,7000082,1,0,1,0
6bba_13f19531,6593,6,18.81006408691406,120.84964447021486,152.11928710937505,cont,6.333776397169372,7000074,1,0,1,0
6bba_13f19531,2991,47,52.13333333333332,95.99999999999997,163.46666666666664,cont,4.866566779796864,48000539,1,0,0,1
6bba_13f19531,3705,60,4.4799999999999995,104.64,135.36,cont,0.9669774105427746,61000650,1,1,0,0
6bba_13f19531,3767,61,4.0,108.64000000000001,137.28000000000003,cont,1.4897802900092463,62000661,1,0,1,0
6bba_13f19531,4098,66,60.519999999999996,16.64,101.92000000000002,cont,0.8228342785761913,67000723,1,1,0,0
6bba_13f19531,4153,67,60.16,21.28,108.0,cont,1.3959875536694535,68000735,1,0,1,0
6bba_13f19531,4267,69,60.000000000000014,35.2,122.55999999999997,cont,2.3518191097531047,70000759,1,1,0,0
6bba_13f19531,4334,70,60.000000000000014,44.16,130.88,cont,1.789345638075526,71000769,1,0,1,0
6bba_13f19531,4398,71,60.000000000000014,50.080000000000005,135.36,cont,1.0258114166356231,72000778,1,1,0,0
6bba_13f19531,4455,72,59.84000000000002,56.64,140.64,cont,0.3957133590112705,73000786,1,0,1,0
6bba_13f19531,4823,78,37.160000000000004,222.72000000000003,243.36000000000004,cont,3.1409614869494855,79000842,1,1,0,0
6bba_13f19531,4889,79,36.68000000000001,229.28000000000003,244.64000000000001,cont,3.7841571469086626,80000850,1,0,1,0
6bba_13f19531,4892,79,42.266666666666666,222.9333333333333,239.46666666666658,cont,6.4399701975889245,80000850,1,0,0,1
6bba_13f19531,4896,79,48.16000000000001,13.280000000000001,120.64000000000001,cont,3.042744700512989,80000851,1,1,0,0
6bba_13f19531,4959,80,48.36,18.72,125.44000000000001,cont,2.8305335738160804,81000858,1,0,1,0
6bba_13f19531,5164,83,49.00000000000001,34.08,135.36000000000004,cont,3.352387209139168,84000882,1,1,0,0
6bba_13f19531,5231,84,50.0,38.720000000000006,136.8,cont,2.5421773394080884,85000889,1,0,1,0
6bba_13f19531,5313,85,60.640000000000015,153.92000000000002,180.79999999999995,cont,1.3884006086141227,86000898,2,2,0,0
6bba_13f19531,5376,86,48.99999999999998,64.53333333333335,144.0,cont,1.9659216897623062,87000902,1,0,0,1
6bba_13f19531,5383,86,51.88000000000001,49.28,134.56,cont,6.913205321158929,87000902,1,0,0,1
6bba_13f19531,5392,86,59.133333333333326,177.06666666666666,185.0666666666666,cont,1.4737415182113982,87000904,1,0,0,1
6bba_13f19531,5471,87,58.67999999999999,185.76000000000005,186.07999999999998,cont,2.3302749301316386,88000909,1,1,0,0
6bba_13f19531,5544,88,57.84,192.0,186.55999999999997,cont,1.941654014622575,89000914,1,0,1,0
6bba_13f19531,5755,91,43.839999999999996,94.56,153.28000000000003,cont,3.807531393501577,92000931,1,1,0,0
6bba_13f19531,5833,92,43.04,102.07999999999998,156.79999999999998,cont,1.8109821230481609,93000940,1,0,1,0
6bba_13f19531,5867,92,61.68,94.72000000000003,123.36,cont,0.767199656217334,93000939,1,1,0,0
6bba_13f19531,5928,93,51.440000000000005,230.07999999999993,202.08000000000004,cont,4.18319947079983,94000946,1,1,0,0
6bba_13f19531,5941,93,59.31999999999999,28.640000000000008,87.35999999999999,cont,1.4858756887775089,94000948,1,1,0,0
6bba_13f19531,5945,93,61.32,101.28000000000002,126.08000000000001,cont,0.8716121915737479,94000949,1,0,1,0
6bba_13f19531,6026,94,50.0,237.44,203.36,cont,5.0086390990966825,95000956,1,0,1,0
6bba_13f19531,6031,94,55.133333333333326,227.46666666666664,195.99999999999997,cont,6.172267304690852,95000956,2,0,0,2
6bba_13f19531,6041,94,58.999999999999986,34.72,89.28000000000003,cont,2.1045475315373836,95000958,1,0,1,0
6bba_15403b9a,479,3,39.07999999999999,168.96,244.00000000000003,cont,1.9288962193441097,4000044,2,2,0,0
6bba_15403b9a,613,4,39.32000000000001,171.36,236.64000000000001,cont,2.7462499999999888,5000059,2,0,2,0
6bba_15403b9a,1234,9,40.6,113.28000000000002,244.00000000000003,cont,4.044552856929911,10000129,1,0,0,1
6bba_15403b9a,1327,10,30.679999999999996,144.00000000000003,229.27999999999997,cont,1.3569479770794382,11000137,1,1,0,0
6bba_15403b9a,1357,10,38.16,149.92000000000002,108.64,cont,0.36912904247700923,11000142,1,1,0,0
6bba_15403b9a,1447,11,30.52,145.92,224.16000000000003,cont,2.1891711473751863,12000152,1,0,1,0
6bba_15403b9a,1478,11,37.52,152.0,102.08,cont,1.1030865786510107,12000155,1,0,1,0
6bba_15403b9a,1579,12,33.84,186.07999999999998,131.68,cont,3.1572290560711656,13000167,1,1,0,0
6bba_15403b9a,1587,12,36.63999999999999,153.44,94.08000000000001,cont,1.4255612929649844,13000169,1,1,0,0
6bba_15403b9a,1592,12,36.68000000000001,179.36,185.12000000000003,cont,3.6012727784770746,13000168,1,1,0,0
6bba_15403b9a,1686,13,32.0,175.04,102.07999999999998,cont,5.000143747933656,14000181,4,0,0,4
6bba_15403b9a,1700,13,36.480000000000004,154.08000000000004,85.92,cont,1.8500751673918507,14000183,1,0,1,0
6bba_15403b9a,1701,13,35.52,183.36,158.07999999999998,cont,1.894292826492246,14000182,4,0,1,3
6bba_15403b9a,1720,13,40.32,124.64000000000001,217.92000000000002,cont,1.461054841544288,14000184,1,1,0,0
6bba_15403b9a,1831,14,40.000000000000014,127.36000000000001,209.28,cont,3.2623670949940338,15000200,1,0,1,0
6bba_15403b9a,2007,16,32.76,180.64000000000001,79.84,cont,1.0080240572526025,17000221,2,2,0,0
6bba_15403b9a,2021,16,33.36,183.36,149.12,cont,4.886794066665789,17000222,1,1,0,0
6bba_15403b9a,2029,16,35.84,152.0,63.84000000000002,cont,2.8389638294631405,17000223,1,1,0,0
6bba_15403b9a,2055,16,42.0,107.2,95.03999999999999,cont,3.30542735512369,17000229,1,1,0,0
6bba_15403b9a,2114,17,32.160000000000004,190.56,56.0,cont,0.6850102645216425,18000233,2,0,1,1
6bba_15403b9a,2130,17,34.519999999999996,188.00000000000003,123.36000000000003,cont,2.6245133477465883,18000234,5,0,2,3
6bba_15403b9a,2132,17,36.86666666666666,151.99999999999994,38.93333333333332,cont,5.418359110596955,18000235,2,0,0,2
6bba_15403b9a,2150,17,42.32000000000001,100.32000000000001,69.28,cont,4.09525792838497,18000241,3,0,1,2
6bba_15403b9a,2294,19,31.640000000000004,186.71999999999997,52.480000000000004,cont,2.0289086506050538,20000260,1,1,0,0
6bba_15403b9a,2385,20,30.0,195.20000000000005,33.44,cont,2.5342186295779676,21000272,2,0,0,2
6bba_15403b9a,2572,22,33.36,196.00000000000006,105.28000000000002,cont,1.154207655710193,23000295,1,1,0,0
6bba_15403b9a,2639,23,30.479999999999997,205.44000000000005,75.36,cont,1.4965006055795522,24000305,1,0,1,0
6bba_15403b9a,2653,23,35.84,159.84,131.36000000000004,cont,3.7038951203428985,24000307,3,0,1,2
6bba_15403b9a,2661,23,37.56,121.76000000000002,17.440000000000005,cont,3.1465052538491034,24000314,1,0,1,0
6bba_15403b9a,3324,33,32.52,186.72000000000003,96.64000000000001,cont,2.4742726203876475,34000447,1,1,0,0
6bba_15403b9a,3382,34,31.84,190.08000000000004,90.72,cont,3.5119930684014724,35000461,1,0,1,0
6bba_15403b9a,4068,45,42.68000000000001,66.56,227.36000000000004,cont,2.172643838736554,46000600,1,1,0,0
6bba_15403b9a,4122,46,42.32,70.71999999999998,223.36000000000004,cont,1.2406531797807299,47000609,1,0,1,0
6bba_15403b9a,4522,54,26.48,156.16,238.07999999999998,cont,2.89090194619257,55000678,1,1,0,0
6bba_15403b9a,4572,55,26.360000000000003,160.64000000000004,232.64000000000004,cont,2.023630709887546,56000689,1,0,1,0
6bba_15403b9a,4889,61,33.84,141.92000000000002,149.28000000000003,cont,1.3210560406356824,62000759,1,1,0,0
6bba_15403b9a,4919,62,19.133333333333336,211.46666666666664,188.53333333333325,cont,4.599544248256164,63000761,1,0,0,1
6bba_15403b9a,4937,62,30.99999999999999,156.53333333333333,120.0,cont,0.46041666666666753,63000766,1,0,0,1
6bba_15403b9a,4944,62,35.13333333333333,111.99999999999996,155.46666666666667,cont,1.481683598047245,63000765,1,0,0,1
6bba_15403b9a,5269,69,21.44,173.28000000000003,170.08000000000004,cont,2.1428445609749547,70000816,1,1,0,0
6bba_15403b9a,5314,70,20.839999999999996,181.43999999999997,165.27999999999997,cont,2.1726438387365845,71000822,1,0,1,0
6bba_15403b9a,5412,72,24.480000000000004,128.0,233.27999999999997,cont,1.63939870836841,73000835,1,1,0,0
6bba_15403b9a,5462,73,23.840000000000003,134.07999999999998,232.64000000000004,cont,2.2319953236958137,74000841,1,0,1,0
6bba_15403b9a,5676,77,35.32000000000001,9.28,71.99999999999999,cont,1.1430578506794735,78000869,1,1,0,0
6bba_15403b9a,5723,78,35.0,13.280000000000001,68.0,cont,1.7061726172928695,79000876,1,0,1,0
6bba_15403b9a,5894,82,12.520000000000001,244.64000000000001,90.72,cont,1.911292919727378,83000899,1,1,0,0
6bba_15403b9a,5945,83,12.0,246.72000000000006,85.28,cont,1.062233054701268,84000907,1,0,1,0
6bba_15403b9a,6319,90,17.32,215.99999999999997,230.56,cont,6.013488094483934,91000961,1,0,1,0
6bba_15403b9a,6530,94,9.0,239.46666666666658,136.5333333333333,cont,1.753737328694234,95000994,1,0,0,1
6bba_1d0d8384,489,1,19.68,216.96,83.99999999999999,cont,1.4351611407782792,2000077,1,0,1,0
6bba_1d0d8384,686,2,4.266666666666667,184.5333333333333,167.99999999999994,cont,1.814785436506368,3000103,6,1,0,5
6bba_1d0d8384,724,2,8.0,200.00000000000003,128.64000000000001,cont,6.309437871157775,3000077,1,1,0,0
6bba_1d0d8384,1202,3,28.68,146.07999999999998,76.0,cont,1.6211767523931537,4000124,2,1,1,0
6bba_1d0d8384,1389,4,10.000000000000004,160.16,206.72,cont,2.2715151881508593,5000140,2,0,0,2
6bba_1d0d8384,1408,4,11.84,138.56,152.64,cont,6.302675151275047,5000171,1,1,0,0
6bba_1d0d8384,1463,4,19.68,197.92000000000002,54.720000000000006,cont,0.5975052301026341,5000149,1,1,0,0
6bba_1d0d8384,1526,4,28.519999999999996,148.00000000000003,76.0,cont,1.2406531797807006,5000153,1,0,1,0
6bba_1d0d8384,1654,5,1.04,209.92000000000002,182.08,cont,1.6788695973779517,6000166,1,1,0,0
6bba_1d0d8384,1802,5,20.16,210.07999999999998,70.72,cont,6.852203226882577,6000182,1,0,1,0
6bba_1d0d8384,2240,6,39.00000000000001,130.72,20.000000000000007,cont,0.9262500000000031,7000219,1,0,1,0
6bba_1d0d8384,2610,8,0.31999999999999995,213.28000000000003,184.0,cont,2.895419874560512,9000254,1,0,1,0
6bba_1d0d8384,2948,9,0.68,217.28000000000003,184.0,cont,2.4736855726829994,10000284,5,0,0,5
6bba_1d0d8384,3247,9,53.0,193.27999999999997,20.640000000000004,cont,0.2837940494443051,10000311,2,1,1,0
6bba_1d0d8384,3505,10,34.86666666666666,183.46666666666658,81.06666666666665,cont,6.688212578858413,11000329,2,0,0,2
6bba_1d0d8384,3728,11,19.52,212.0,66.08000000000001,cont,0.9555773778192939,12000352,1,0,1,0
6bba_1d0d8384,3870,11,47.16000000000001,203.2,61.44,cont,1.7839510223097332,12000363,1,0,1,0
6bba_1d0d8384,4467,13,36.24,143.52,34.720000000000006,cont,4.885659180192571,14000426,6,0,0,6
6bba_1d0d8384,4611,14,10.000000000000004,186.55999999999997,154.88000000000002,cont,2.3438899958189046,15000472,7,0,0,7
6bba_1d0d8384,4748,14,32.160000000000004,206.72,41.28,parent,4.940347438439931,15000455,1,1,0,0
6bba_1d0d8384,4965,15,17.24,252.0,75.99999999999999,cont,1.362095123330237,16000482,2,2,0,0
6bba_1d0d8384,5045,15,30.839999999999996,198.72,51.36000000000001,daughter,2.9725065706403373,16000515,1,0,1,0
6bba_1d0d8384,5052,15,32.16000000000001,209.44000000000005,42.720000000000006,daughter,4.10134676813606,16000515,1,0,1,0
6bba_1d0d8384,5056,15,32.86666666666666,170.93333333333328,59.46666666666667,cont,1.6538583360836374,16000498,6,0,0,6
6bba_1d0d8384,5281,16,18.48,220.64,72.64,cont,2.5906393998393527,17000514,3,0,0,3
6bba_1d0d8384,5593,17,18.199999999999996,245.44000000000003,98.72000000000001,cont,3.8572790361211897,18000550,3,0,1,2
6bba_1d0d8384,6276,19,25.360000000000003,196.64,93.91999999999999,cont,6.4322295172436,20000617,6,6,0,0
6bba_1d0d8384,6312,19,32.480000000000004,175.20000000000005,60.64,cont,1.072992311249257,20000622,8,8,0,0
6bba_1d0d8384,6340,19,38.440000000000005,165.28,14.560000000000002,cont,2.4269689171886912,20000624,3,3,0,0
6bba_1d0d8384,6367,19,44.96,125.28000000000003,7.359999999999999,cont,3.267058357682031,20000627,3,3,0,0
6bba_1d0d8384,6373,19,43.720000000000006,201.92000000000002,36.63999999999999,cont,6.8016296300960155,20000628,4,4,0,0
6bba_1d0d8384,6386,19,46.56,215.99999999999997,66.72,cont,2.549230888915327,20000630,2,2,0,0
6bba_1d0d8384,6481,20,11.36,209.28000000000006,169.92000000000002,cont,1.371561609990591,21000661,8,0,3,5
6bba_1d0d8384,6510,20,15.040000000000003,172.0,153.91999999999996,cont,3.6106461506217884,21000637,8,0,3,5
6bba_1d0d8384,6551,20,22.880000000000003,102.56000000000003,150.72,cont,2.0499489322907505,21000640,4,0,2,2
6bba_1d0d8384,6568,20,25.866666666666664,204.53333333333325,107.99999999999999,cont,1.308155839871146,21000644,5,0,0,5
6bba_1d0d8384,6594,20,31.48,190.56,80.15999999999998,cont,5.2421301133222515,21000645,12,1,5,6
6bba_1d0d8384,6596,20,31.320000000000004,206.55999999999997,89.12,cont,2.9229228735975865,21000645,11,1,6,4
6bba_1d0d8384,6627,20,36.0,212.53333333333325,115.46666666666664,cont,1.1892019966721197,21000648,3,1,0,2
6bba_1d0d8384,6651,20,42.64000000000001,145.27999999999997,24.0,cont,2.140131786245883,21000650,11,0,7,4
6bba_1d0d8384,6667,20,43.20000000000001,185.28,30.72,cont,0.36263359607186163,21000659,7,0,3,4
6bba_1d0d8384,6668,20,45.28,214.55999999999995,63.84,cont,3.106513821794439,21000651,5,0,2,3
6bba_1d0d8384,6878,21,30.64,209.44000000000005,88.64,cont,1.8166598814857997,22000667,1,0,1,0
6bba_1d0d8384,6928,21,40.92000000000001,203.04,80.31999999999998,cont,6.99607322896207,22000676,1,0,1,0
6bba_1d0d8384,6986,21,55.99999999999998,127.99999999999997,0.5333333333333338,cont,5.418291422989788,22000682,4,0,0,4
6bba_1d0d8384,7618,24,11.84,212.64,196.00000000000003,cont,1.1759656935897278,25000760,1,1,0,0
6bba_1d0d8384,7842,24,60.640000000000015,168.63999999999996,22.560000000000006,cont,5.187238246408229,25000759,2,0,0,2
6bba_1d0d8384,7900,25,11.999999999999996,196.5333333333333,207.99999999999997,cont,3.769256691569885,26000784,5,0,0,5
6bba_1d0d8384,7991,25,28.639999999999997,222.08000000000004,94.07999999999998,cont,2.3718320977674496,26000772,1,0,1,0
6bba_1d0d8384,8453,27,12.68,11.200000000000001,235.99999999999994,cont,1.676902371189191,28000816,1,1,0,0
6bba_1d0d8384,8704,28,10.000000000000004,240.0,179.35999999999999,cont,1.6950705184445807,29000879,2,0,0,2
6bba_1d0d8384,8930,28,60.000000000000014,145.28,13.920000000000005,cont,0.29430001698946273,29000869,1,0,1,0
6bba_1d0d8384,9106,29,36.68000000000001,239.36,124.00000000000003,cont,1.3527576002373907,30000887,1,1,0,0
6bba_1d0d8384,9561,31,23.68,14.719999999999999,234.56000000000006,cont,1.2422484654850667,32000938,1,1,0,0
6bba_1d0d8384,9613,31,33.52,29.279999999999994,108.64000000000001,cont,1.0666983230979594,32000947,1,1,0,0
6bba_1d0d8384,9630,31,39.480000000000004,177.92,32.64,cont,2.67454464582291,32000949,1,1,0,0
6bba_1d0d8384,9778,32,14.544279785156252,237.51105224609373,169.05641113281246,cont,4.859938993038631,33000962,1,0,1,0
6bba_1d0d8384,9822,32,23.840000000000003,16.639999999999997,235.35999999999999,cont,0.9037396264964833,33000964,1,0,1,0
6bba_1d0d8384,9882,32,34.999999999999986,43.466666666666654,103.99999999999999,cont,1.9518798808954332,33000969,3,0,0,3
6bba_1d0d8384,9896,32,39.16000000000001,184.8,35.36,cont,1.9028465518795623,33000971,1,0,1,0
6bba_1d0d8384,10484,34,56.88908508300782,97.07504760742188,3.005049476623535,cont,6.935219959667086,35001021,2,1,1,0
6bba_1d0d8384,10497,34,60.133333333333326,107.99999999999999,7.999999999999998,cont,3.913893078225019,35001021,2,0,0,2
6bba_1d0d8384,10661,35,38.16,200.80000000000004,42.72,cont,3.3344192241678323,36001034,1,1,0,0
6bba_1d0d8384,10735,35,58.0,184.0,16.000000000000004,cont,6.662995408410543,36001040,1,1,0,0
6bba_1d0d8384,10752,35,60.6,111.99999999999999,8.000000000000002,cont,4.75781019088615,36001042,1,1,0,0
6bba_1d0d8384,11103,37,27.866666666666667,127.99999999999997,119.46666666666667,cont,2.456683768597204,38001070,4,0,0,4
6bba_1d0d8384,11703,39,59.133333333333326,211.99999999999994,32.533333333333324,cont,4.200362190897606,40001125,5,0,0,5
6bba_1d0d8384,12510,43,33.68000000000001,91.84000000000002,110.71999999999998,cont,0.9176575137272093,44001216,1,1,0,0
6bba_1d0d8384,12737,44,41.0,14.08,188.64,cont,1.3086043376437397,45001222,2,1,1,0
6bba_1d0d8384,13802,50,14.68,219.2,189.28,cont,1.1574061787030472,51001305,6,0,3,3
6bba_1d0d8384,13917,50,51.00000000000001,57.44000000000001,92.80000000000001,cont,1.4823171261575536,51001314,2,0,2,0
6bba_1d0d8384,15294,58,42.32,17.44,179.36,cont,2.2028192163679643,59001422,1,0,1,0
6bba_1d0d8384,15948,62,49.64000000000001,96.0,105.28000000000002,cont,1.3926733509692855,63001459,2,2,0,0
6bba_1d0d8384,15983,62,60.000000000000014,32.480000000000004,137.28,cont,3.844353001481514,63001464,2,2,0,0
6bba_1d0d8384,16040,63,29.520000000000003,191.36,151.84,cont,2.82282659757911,64001466,4,0,3,1
6bba_1d0d8384,16069,63,38.040000000000006,42.56,201.28000000000006,cont,1.9217643260816366,64001468,2,1,0,1
6bba_1d0d8384,16075,63,40.68,65.44,224.0,cont,4.21780637594947,64001469,1,0,1,0
6bba_1d0d8384,16139,63,60.000000000000014,36.800000000000004,138.72000000000003,cont,1.1801125475987368,64001476,2,0,2,0
6bba_1d0d8384,17329,72,43.32000000000001,151.2,123.35999999999999,cont,2.447230577612163,73001576,2,2,0,0
6bba_1d0d8384,18673,82,56.16000000000001,80.64,175.36,cont,0.8234758648558088,83001725,1,1,0,0
6bba_1d0d8384,18877,84,35.16,166.08,206.08,cont,1.46376334323552,85001743,1,0,1,0
6bba_1d0d8384,19876,91,62.000000000000014,52.640000000000015,157.92000000000002,cont,1.0290241129341873,92001893,1,1,0,0
6bba_1d0d8384,19969,92,55.0,53.92000000000001,157.44000000000003,cont,1.2999999999999965,93001906,3,0,1,2
6bba_1ebfb80d,946,3,5.0,233.91999999999996,101.92000000000002,cont,5.472970086022775,4000063,3,0,1,2
6bba_1ebfb80d,2077,8,0.0,198.56000000000003,183.20000000000002,cont,5.551723972109915,9000143,4,2,0,2
6bba_1ebfb80d,2115,8,1.5200000000000002,213.6,192.00000000000006,cont,2.6961781399788904,9000143,3,0,1,2
6bba_1ebfb80d,2311,9,0.0,225.44000000000003,175.36,cont,5.700878907019509,10000155,2,2,0,0
6bba_1ebfb80d,2350,9,1.8400000000000005,218.56,192.64000000000004,cont,2.8855992120355336,10000155,1,1,0,0
6bba_1ebfb80d,3011,12,4.0,235.99999999999994,16.0,cont,5.929023554726346,13000197,1,0,1,0
6bba_1ebfb80d,3029,12,6.999999999999997,233.0666666666666,28.53333333333333,cont,2.5355844233737237,13000197,2,0,0,2
6bba_1ebfb80d,4262,18,28.866666666666667,28.000000000000004,199.4666666666667,cont,0.5088496353431786,19000272,1,0,0,1
6bba_1ebfb80d,4713,21,23.0,231.4666666666667,191.46666666666664,cont,5.976647988676913,22000298,3,0,0,3
6bba_1ebfb80d,4910,23,0.48,28.8,101.43999999999998,cont,1.5670932486613554,24000313,1,0,1,0
6bba_1ebfb80d,5051,23,42.52000000000001,122.08000000000003,236.0,cont,2.802029677305347,24000322,1,1,0,0
6bba_1ebfb80d,5787,29,0.68,52.160000000000004,101.92000000000002,cont,2.932618550459641,30000375,1,1,0,0
6bba_1ebfb80d,5933,30,0.9999999999999998,76.0,100.0,cont,1.1490485194281397,31000386,2,0,0,2
6bba_1ebfb80d,6042,30,43.31999999999999,148.15999999999997,221.28000000000006,cont,1.4318454001742262,31000391,1,0,1,0
6bba_1ebfb80d,6192,31,58.16000000000001,152.79999999999998,240.00000000000006,cont,1.5485320669911684,32000403,1,1,0,0
6bba_1ebfb80d,6310,32,47.84,64.0,244.63999999999996,cont,2.0690532255115994,33000413,1,0,1,0
6bba_1ebfb80d,6322,32,58.0,156.00000000000003,235.99999999999994,cont,0.5745242597141,33000414,2,1,1,0
6bba_1ebfb80d,6462,33,57.84,159.35999999999999,232.64,cont,1.9084585566629344,34000426,1,0,1,0
6bba_1ebfb80d,6547,34,28.640000000000004,79.36,169.28,cont,1.1719168325866856,35000434,1,1,0,0
6bba_1ebfb80d,6687,35,28.68,83.35999999999999,166.71999999999997,cont,2.948781792961283,36000446,1,0,1,0
6bba_1ebfb80d,6738,35,57.84,162.56000000000003,226.08,cont,1.22749363542138,36000451,1,1,0,0
6bba_1ebfb80d,6873,36,57.68000000000001,165.28,222.72000000000003,cont,2.674939543709337,37000462,1,0,1,0
6bba_1ebfb80d,7412,40,59.160000000000004,39.36,237.44,cont,6.9616221394514035,41000510,1,0,1,0
6bba_1ebfb80d,7422,40,61.99999999999999,23.46666666666666,235.99999999999994,cont,1.6494172476800415,41000510,1,0,0,1
6bba_1ebfb80d,7432,41,1.0800000000000003,6.079999999999999,44.959999999999994,cont,1.644705520754399,42000511,2,2,0,0
6bba_1ebfb80d,7433,41,1.0,24.159999999999997,42.72,cont,5.775955007615622,42000511,2,2,0,0
6bba_1ebfb80d,7501,41,30.839999999999996,104.0,153.44,cont,1.3838284756428398,42000515,2,2,0,0
6bba_1ebfb80d,7540,41,56.52,178.56000000000006,204.80000000000004,cont,2.4604131385805883,42000520,2,2,0,0
6bba_1ebfb80d,7545,41,59.36,41.28000000000001,234.72000000000006,cont,6.671787733621336,42000522,1,1,0,0
6bba_1ebfb80d,7549,41,62.0,24.640000000000008,232.0,cont,1.7712499999999967,42000522,2,2,0,0
6bba_1ebfb80d,7559,42,1.0,25.92,41.28000000000001,cont,1.1916912876244432,43000523,3,0,2,1
6bba_1ebfb80d,7624,42,30.480000000000004,107.36,147.84,cont,1.8313540926047072,43000528,2,0,2,0
6bba_1ebfb80d,7666,42,56.160000000000004,182.72,199.2,cont,1.7124292868612057,43000533,3,1,2,0
6bba_1ebfb80d,7679,42,62.000000000000014,25.44,228.64000000000004,cont,2.3758922666021856,43000536,2,0,2,0
6bba_1ebfb80d,7777,43,42.68000000000001,194.72,183.36,cont,2.2119706287606804,44000545,1,1,0,0
6bba_1ebfb80d,7797,43,56.16000000000001,185.28,196.00000000000003,cont,3.112203801167255,44000546,1,0,1,0
6bba_1ebfb80d,7810,43,60.000000000000014,134.72000000000003,224.00000000000003,cont,6.419243651903218,44000547,2,0,2,0
6bba_1ebfb80d,7820,43,61.99999999999999,183.46666666666658,191.99999999999994,cont,6.630938114458444,44000546,1,0,0,1
6bba_1ebfb80d,7902,44,42.32,198.72,179.99999999999997,cont,2.732368764003135,45000557,1,0,1,0
6bba_1ebfb80d,8024,45,39.0,97.28000000000003,174.72,cont,0.5966206918972806,46000569,1,1,0,0
6bba_1ebfb80d,8148,46,39.0,99.36000000000001,170.71999999999997,cont,0.5813776741499505,47000582,1,0,1,0
6bba_1ebfb80d,8189,46,62.000000000000014,27.36,219.36,cont,1.8784044093325802,47000588,1,0,1,0
6bba_1ebfb80d,8302,47,55.99999999999999,195.36000000000004,182.07999999999993,cont,3.2967882097581147,48000597,1,0,1,0
6bba_1ebfb80d,8312,47,60.16000000000001,27.36,232.0,cont,4.719834809079651,48000599,2,2,0,0
6bba_1ebfb80d,8314,47,62.000000000000014,28.640000000000008,216.64000000000001,cont,2.40818201658431,48000599,1,1,0,0
6bba_1ebfb80d,8321,47,62.000000000000014,195.84000000000003,178.08,cont,6.733855693434512,48000597,1,0,1,0
6bba_1ebfb80d,8449,48,62.000000000000014,31.360000000000007,212.64,cont,1.6521543700574997,49000611,1,0,1,0
6bba_1ebfb80d,8970,52,60.35999999999999,48.0,208.0,cont,5.8453940200811125,53000659,1,1,0,0
6bba_1ebfb80d,9068,53,42.84,209.28000000000003,152.0,cont,1.9971640049079535,54000668,1,1,0,0
6bba_1ebfb80d,9196,54,43.16,210.71999999999997,148.64000000000001,cont,1.4769632062106428,55000680,1,0,1,0
6bba_1ebfb80d,9451,56,43.68,213.28000000000003,144.00000000000003,cont,2.3123013428184485,57000706,1,1,0,0
6bba_1ebfb80d,9570,57,37.64000000000001,111.99999999999999,120.63999999999999,cont,2.751821271903405,58000714,1,0,1,0
6bba_1ebfb80d,9890,60,3.4800000000000013,87.84,29.439999999999998,cont,2.8722076113157295,61000749,2,2,0,0
6bba_1ebfb80d,9961,60,44.32,218.56,134.88000000000002,cont,3.081250253549683,61000757,1,1,0,0
6bba_1ebfb80d,9991,60,59.48,35.36,170.07999999999998,cont,5.214300648217361,61000761,1,1,0,0
6bba_1ebfb80d,10020,61,4.0,92.16,26.560000000000002,cont,2.005871459615497,62000762,2,0,2,0
6bba_1ebfb80d,10075,61,35.160000000000004,129.12,101.28,cont,2.73314179333967,62000767,3,0,3,0
6bba_1ebfb80d,10092,61,44.68,221.43999999999997,128.48,cont,3.368260864303711,62000770,2,1,1,0
6bba_1ebfb80d,10180,62,23.86666666666666,87.46666666666665,64.53333333333333,cont,1.22235577710965,63000778,2,0,0,2
6bba_1ebfb80d,10219,62,44.84000000000001,222.72,124.0,cont,2.1392061903893005,63000783,1,0,1,0
6bba_1ebfb80d,10371,63,59.84000000000002,79.19999999999999,148.15999999999997,cont,0.64939033908736,64000798,1,1,0,0
6bba_1ebfb80d,10432,64,25.16,84.8,52.64,cont,5.78502282298869,65000803,1,0,0,1
6bba_1ebfb80d,10496,64,57.519999999999996,90.07999999999998,126.08000000000001,cont,6.76694955278964,65000811,1,0,1,0
6bba_1ebfb80d,10499,64,59.68000000000001,82.72,141.28000000000003,cont,1.5219580316158563,65000811,1,0,1,0
6bba_1ebfb80d,10844,67,43.839999999999996,146.72000000000003,91.36000000000001,cont,3.557822367685045,68000850,1,1,0,0
6bba_1ebfb80d,10955,68,37.2,134.72000000000003,71.36,cont,2.8007100104259335,69000860,1,1,0,0
6bba_1ebfb80d,10965,68,43.84,147.35999999999999,87.36,cont,3.7301803324370177,69000863,2,1,1,0
6bba_1ebfb80d,11304,71,37.040000000000006,164.0,44.480000000000004,cont,2.191341270204165,72000905,1,0,1,0
6bba_1ebfb80d,11466,72,59.03999999999999,43.20000000000001,98.08000000000003,cont,5.023589665269256,73000915,1,1,0,0
6bba_1ebfb80d,11590,73,61.99999999999999,31.999999999999993,103.99999999999996,cont,1.9902104160113163,74000928,1,0,0,1
6bba_1ebfb80d,11795,75,56.160000000000004,201.28000000000006,49.28,cont,3.0370115347986406,76000951,1,1,0,0
6bba_1ebfb80d,12215,79,59.160000000000004,112.16000000000003,46.56000000000001,cont,4.014046041714031,80001001,3,1,0,2
6bba_1ebfb80d,12290,80,48.16000000000001,237.28000000000003,42.559999999999995,cont,1.7407990263094528,81001006,1,1,0,0
6bba_1ebfb80d,12319,80,59.999999999999986,99.46666666666667,35.466666666666654,cont,0.8809371982282456,81001008,2,0,0,2
6bba_1ebfb80d,12414,81,58.72,205.28,45.92,cont,5.978078835420285,82001014,4,0,0,4
6bba_1ebfb80d,12644,84,47.64,237.92000000000002,23.200000000000003,cont,1.1547794702885885,85001036,1,1,0,0
6bba_1ebfb80d,13009,89,47.00000000000001,236.64000000000001,9.440000000000001,cont,1.4144969291235627,90001065,1,1,0,0
6bba_1ebfb80d,13035,89,59.36,37.279999999999994,57.919999999999995,cont,3.11830683224086,90001068,1,1,0,0
6bba_1ebfb80d,13087,90,58.999999999999986,12.0,125.28000000000002,cont,2.3631880611792373,91001071,1,0,1,0
6bba_1ebfb80d,13094,90,58.72,35.36,34.88,cont,4.707566382962643,91001072,4,0,1,3
6bba_1f58c2f6,4079,9,58.84000000000001,114.88,201.92000000000002,cont,1.000926133638234,10000011,2,2,0,0
6bba_1f58c2f6,6592,15,58.519999999999996,106.72,201.28000000000006,cont,1.8296229973685978,16000029,1,1,0,0
6bba_1f58c2f6,6975,16,55.16000000000001,106.72,79.36000000000001,cont,1.7396609763111686,17000033,3,2,1,0
6bba_1f58c2f6,7386,17,54.0,107.36000000000001,80.64,cont,2.0009286906334247,18000037,2,0,2,0
6bba_1f58c2f6,8627,20,55.32000000000001,100.00000000000001,196.64000000000001,cont,1.741102380677247,21000043,1,0,1,0
6bba_1f58c2f6,9053,21,54.52000000000001,98.72000000000001,196.00000000000003,cont,0.9374433316206241,22000049,2,2,0,0
6bba_1f58c2f6,9479,22,54.2,99.19999999999999,196.00000000000003,cont,0.8788531233943613,23000054,2,0,2,0
6bba_1f58c2f6,9867,23,46.13333333333333,116.0,93.06666666666666,cont,0.9224250948210152,24000059,8,0,0,8
6bba_1f58c2f6,10309,24,48.26666666666665,94.93333333333331,195.46666666666664,cont,2.6294555440876466,25000065,4,0,0,4
6bba_1f58c2f6,11232,26,57.68000000000002,99.36000000000001,152.64,cont,5.583500945867176,27000083,2,0,2,0
6bba_1f58c2f6,11276,26,61.99999999999999,100.0,150.93333333333334,cont,2.455638530052361,27000083,4,0,0,4
6bba_1f58c2f6,11719,27,62.0,96.96000000000004,154.24,cont,2.471763350828717,28000091,2,2,0,0
6bba_1f58c2f6,12150,28,62.000000000000014,96.48,159.36,cont,1.491551729743214,29000100,3,1,2,0
6bba_1f58c2f6,12588,29,62.000000000000014,98.08000000000001,163.36,cont,1.5252510039334406,30000111,1,0,1,0
6bba_1f58c2f6,14128,33,32.000000000000014,114.72,112.16,cont,2.0338483381265213,34000135,1,1,0,0
6bba_1f58c2f6,14291,33,53.19999999999999,97.92000000000002,179.84,cont,5.036058261428276,34000140,3,3,0,0
6bba_1f58c2f6,14292,33,53.36,113.44,161.27999999999997,cont,5.080328022135988,34000140,6,5,1,0
6bba_1f58c2f6,14573,34,30.520000000000007,110.56,109.92000000000002,cont,6.598860697120382,35000144,1,0,1,0
6bba_1f58c2f6,14599,34,35.96000000000001,77.28,211.36,cont,1.774229145995533,35000142,1,0,1,0
6bba_1f58c2f6,14729,34,51.839999999999996,100.16000000000001,184.0,cont,4.407675833134739,35000150,8,0,3,5
6bba_1f58c2f6,15069,35,35.0,76.64000000000001,212.64,cont,2.3778364640992526,36000153,1,1,0,0
6bba_1f58c2f6,15469,36,26.68,104.64000000000003,109.28,cont,6.035053981738682,37000161,1,0,1,0
6bba_1f58c2f6,16919,39,29.119999999999997,78.72000000000001,220.0,cont,1.823551222889002,40000186,3,3,0,0
6bba_1f58c2f6,17032,39,44.120000000000005,76.0,175.83999999999997,cont,4.320330752673916,40000189,2,2,0,0
6bba_1f58c2f6,17050,39,46.99158142089844,199.5962915039063,174.89843505859372,cont,4.304933507171059,40000191,2,2,0,0
6bba_1f58c2f6,17353,40,22.16,70.72,223.20000000000002,cont,1.4538983157360135,41000194,4,0,1,3
6bba_1f58c2f6,17516,40,43.56,76.0,180.15999999999997,cont,2.8415205546502857,41000198,3,1,2,0
6bba_1f58c2f6,17973,41,42.16,76.0,182.08,cont,0.8221921916437769,42000208,1,0,1,0
6bba_1f58c2f6,18478,42,45.951581420898435,196.23629150390627,173.45843505859375,cont,1.8850577071032653,43000221,1,1,0,0
6bba_1f58c2f6,22074,50,28.16,120.8,120.80000000000001,cont,6.224110629238529,51000329,2,2,0,0
6bba_1f58c2f6,22104,50,32.0,132.64000000000001,122.72000000000001,cont,2.801888314155295,51000329,2,2,0,0
6bba_1f58c2f6,22445,51,18.200000000000003,34.720000000000006,201.92000000000007,cont,1.3839238824805415,52000332,2,0,1,1
6bba_1f58c2f6,22497,51,24.720000000000002,126.72000000000001,134.56000000000003,cont,2.177378727966269,52000337,7,0,3,4
6bba_1f58c2f6,23274,53,6.280000000000001,172.64,212.00000000000006,cont,6.415972500330724,54000358,3,3,0,0
6bba_1f58c2f6,23306,53,11.640000000000002,179.84,209.92,cont,2.9279776510417594,54000358,4,4,0,0
6bba_1f58c2f6,23387,53,23.959999999999997,129.28,132.8,cont,5.391719635236976,54000363,5,5,0,0
6bba_1f58c2f6,23750,54,6.4,173.44000000000005,213.92000000000002,cont,2.7118518810952725,55000372,7,0,3,4
6bba_1f58c2f6,23773,54,10.520000000000001,184.16000000000003,214.07999999999998,cont,5.545846700459733,55000372,6,0,4,2
6bba_1f58c2f6,24140,54,60.559999999999995,41.599999999999994,29.120000000000005,cont,4.574454373201242,55000380,3,0,1,2
6bba_1f58c2f6,24709,56,13.040000000000001,24.64,201.28000000000006,cont,1.5939088156164962,57000392,2,2,0,0
6bba_1f58c2f6,25142,57,12.120000000000003,24.0,203.35999999999999,cont,1.4915517297432284,58000405,2,0,2,0
6bba_1f58c2f6,25436,57,55.26666666666666,36.53333333333333,36.0,cont,5.32826297068023,58000412,5,1,0,4
6bba_1f58c2f6,25498,58,0.31999999999999995,169.27999999999997,225.28,cont,1.2265251984773815,59000416,1,1,0,0
6bba_1f58c2f6,25647,58,17.919999999999998,180.0,199.2,cont,3.51030091302726,59000419,1,1,0,0
6bba_1f58c2f6,25767,58,40.99999999999999,96.63999999999999,91.99999999999999,cont,2.0365081941892673,59000420,4,4,0,0
6bba_1f58c2f6,25827,58,54.360000000000014,35.04,36.32,cont,3.89190044354942,59000421,2,1,1,0
6bba_1f58c2f6,26161,59,39.52000000000001,95.36000000000001,94.72,cont,1.3310128896821252,60000433,4,0,4,0
6bba_1f58c2f6,26243,59,52.84,33.28,37.279999999999994,cont,2.4552561602610896,60000431,1,0,1,0
6bba_1f58c2f6,28991,66,34.133333333333326,26.933333333333337,48.53333333333332,cont,6.524160825666916,67000491,6,0,0,6
6bba_1f58c2f6,29296,67,24.240000000000002,84.0,104.0,cont,3.0007991123199163,68000496,1,1,0,0
6bba_1f58c2f6,29608,68,17.560000000000002,78.72000000000001,113.28000000000003,cont,2.533593359736319,69000502,7,0,3,4
6bba_1f58c2f6,30460,70,59.68000000000001,205.28,216.64000000000001,cont,2.5450322321141448,71000520,1,1,0,0
6bba_1f58c2f6,30773,71,58.15999999999998,204.00000000000003,218.08,cont,2.709903711389029,72000527,2,1,1,0
6bba_1f58c2f6,30800,71,61.99999999999999,215.99999999999994,223.99999999999997,cont,6.107276653713964,72000527,4,0,0,4
6bba_1f58c2f6,31078,72,57.0,202.72000000000003,217.92000000000002,cont,3.685133138707471,73000536,1,0,1,0
6bba_1f58c2f6,31472,74,12.439999999999998,77.28,129.28,cont,0.9312256708231352,75000547,1,1,0,0
6bba_1f58c2f6,31682,74,60.879999999999995,207.36000000000004,224.64000000000001,cont,6.6363098038367685,75000552,1,1,0,0
6bba_1f58c2f6,31741,75,7.639999999999999,74.88,139.2,cont,2.7531643725902017,76000555,8,2,1,5
6bba_1f58c2f6,31977,75,59.68000000000001,205.28,225.28,cont,1.226525198477385,76000560,1,0,1,0
6bba_1f58c2f6,32031,76,2.56,79.20000000000002,144.64,cont,3.1349025602879546,77000564,5,0,2,3
6bba_1f58c2f6,32818,78,58.520000000000024,88.00000000000001,109.91999999999999,cont,2.9723733152146097,79000602,3,3,0,0
6bba_1f58c2f6,33074,79,57.36,88.00000000000001,110.71999999999998,cont,3.9594684775863525,80000615,4,1,3,0
6bba_1f58c2f6,33081,79,60.32000000000001,106.08,119.36000000000001,cont,6.525017480819504,80000615,3,2,1,0
6bba_1f58c2f6,33333,80,55.67999999999999,87.36000000000001,112.0,cont,4.290276979694899,81000629,1,0,1,0
6bba_1f58c2f6,33343,80,59.84,104.00000000000003,117.28,cont,6.57080962001641,81000629,2,0,2,0
6bba_1f58c2f6,33856,82,55.68000000000001,8.0,44.00000000000001,cont,0.9646534351776161,83000655,1,1,0,0
6bba_1f58c2f6,34312,84,46.839999999999996,198.08,228.64,cont,0.6114838509723807,85000672,1,1,0,0
6bba_1f58c2f6,34791,86,55.16000000000001,95.36000000000001,116.63999999999999,cont,4.6552208594222275,87000697,2,1,1,0
6bba_1f58c2f6,34987,87,43.16,192.64,227.99999999999997,cont,1.7225766492379924,88000705,1,0,1,0
6bba_1f58c2f6,35021,87,59.32000000000001,19.360000000000003,56.00000000000001,cont,2.943224475384097,88000711,1,1,0,0
6bba_1f58c2f6,35224,88,48.16,8.0,51.35999999999999,cont,1.273008861909451,89000719,2,1,1,0
6bba_1f58c2f6,35443,89,47.00000000000001,8.0,52.0,cont,1.21875,90000734,1,0,1,0
6bba_1f58c2f6,35449,89,48.86666666666666,15.999999999999996,64.53333333333333,cont,6.265154197468362,90000734,4,0,0,4
6bba_1f58c2f6,35657,90,49.00000000000001,95.36000000000001,121.28000000000003,cont,1.4629513192515993,91000746,1,1,0,0
6bba_1f58c2f6,36080,92,46.0,94.72,124.0,cont,1.6289763848810088,93000773,2,1,1,0
6bba_1f58c2f6,36104,92,56.160000000000004,123.36,143.99999999999997,cont,3.4676525290893854,93000777,1,1,0,0
6bba_1f58c2f6,36293,93,44.480000000000004,93.28000000000003,125.44000000000001,cont,2.0066611728191566,94000787,1,0,1,0
6bba_1f58c2f6,36499,94,43.51999999999999,92.63999999999999,124.63999999999999,cont,1.1040437038451167,95000798,1,1,0,0
6bba_1f58c2f6,36522,94,54.360000000000014,121.28000000000003,143.99999999999997,cont,4.3191081617041975,95000803,1,1,0,0
6bba_1f58c2f6,36701,95,42.32000000000001,92.0,124.63999999999999,cont,0.9756768483980817,96000808,1,0,1,0
6bba_207c6aaf,649,7,60.32000000000001,138.72000000000003,196.64,cont,2.1636922291536838,8000016,1,1,0,0
6bba_207c6aaf,755,8,60.35999999999999,137.28000000000003,193.44,cont,2.2408508428719593,9000019,1,0,1,0
6bba_207c6aaf,1074,11,42.239999999999995,142.08,224.64000000000004,cont,1.318755331742789,12000024,2,0,0,2
6bba_207c6aaf,1339,13,41.47999999999999,136.0,215.36000000000007,cont,2.684202919397122,14000032,1,1,0,0
6bba_207c6aaf,1491,14,41.0,134.56000000000003,209.92,cont,0.41429533547940905,15000037,1,0,1,0
6bba_207c6aaf,1807,16,41.0,134.24,197.44000000000003,cont,2.103543513336476,17000047,4,0,0,4
6bba_207c6aaf,2388,19,41.480000000000004,129.28,175.36,cont,1.0666983230979543,20000064,2,0,2,0
6bba_207c6aaf,3141,23,2.1600000000000006,60.640000000000015,223.36000000000004,cont,3.0267342652601714,24000086,2,0,2,0
6bba_207c6aaf,3270,23,43.16,143.99999999999997,106.72,cont,2.139206190389319,24000090,5,0,0,5
6bba_207c6aaf,3279,23,45.266666666666666,139.99999999999997,105.06666666666662,cont,5.904042429885565,24000090,5,0,0,5
6bba_207c6aaf,3743,25,40.96,142.08,95.99999999999997,cont,3.4229304879006803,26000104,2,2,0,0
6bba_207c6aaf,4011,26,39.99999999999999,140.64000000000001,90.56,cont,5.371774177355173,27000108,2,0,2,0
6bba_207c6aaf,4289,27,39.56000000000001,138.56,85.28000000000002,cont,5.8218559111335,28000116,4,0,0,4
6bba_207c6aaf,4299,27,43.68,148.6400000000001,248.64000000000007,cont,2.413603698310919,28000118,2,1,1,0
6bba_207c6aaf,4571,28,43.68,146.56000000000003,245.44000000000003,cont,2.0962500000000075,29000127,2,1,1,0
6bba_207c6aaf,5098,30,40.480000000000004,134.07999999999998,152.63999999999996,cont,0.9632837717931283,31000145,2,2,0,0
6bba_207c6aaf,5348,31,35.279999999999994,132.64000000000001,63.84000000000002,cont,4.601390584649373,32000155,1,0,1,0
6bba_207c6aaf,6105,34,14.0,118.72,230.72000000000008,cont,1.9413819980879454,35000185,1,0,1,0
6bba_207c6aaf,6159,34,27.36,135.36000000000004,34.24000000000001,cont,5.197790546232123,35000186,6,0,2,4
6bba_207c6aaf,6377,35,13.160000000000002,117.44,222.07999999999998,cont,1.8226821753942726,36000196,2,2,0,0
6bba_207c6aaf,6793,36,39.480000000000004,131.99999999999997,129.28,cont,0.9821508603570064,37000211,1,0,1,0
6bba_207c6aaf,7153,37,48.20000000000001,148.64000000000001,228.64000000000001,cont,6.494655615388696,38000226,5,1,4,0
6bba_207c6aaf,8416,41,38.879999999999995,131.35999999999999,114.72,parent,4.789589296067051,42000280,1,0,1,0
6bba_207c6aaf,8418,41,40.73333333333333,129.06666666666666,132.53333333333333,parent,4.073859119434546,42000280,4,0,0,4
6bba_207c6aaf,8419,41,41.959999999999994,141.92000000000002,159.52,cont,2.8281069750806713,42000281,6,0,1,5
6bba_207c6aaf,8896,43,7.520000000000001,101.92000000000002,184.16000000000003,cont,2.6934834554902998,44000310,2,1,0,1
6bba_207c6aaf,8922,43,13.839999999999998,100.0,240.8,cont,2.5019723469694837,44000308,3,3,0,0
6bba_207c6aaf,8945,43,20.119999999999997,150.71999999999997,225.92,cont,1.2372430288750849,44000312,2,2,0,0
6bba_207c6aaf,9046,43,41.64,138.72000000000003,162.56000000000003,cont,0.9107251574981373,44000315,1,1,0,0
6bba_207c6aaf,9270,44,17.26666666666666,147.46666666666667,203.4666666666666,cont,1.361998187407037,45000328,4,1,0,3
6bba_207c6aaf,9361,44,36.96000000000001,128.8,97.28000000000003,cont,1.98123422706654,45000332,3,0,3,0
6bba_207c6aaf,9383,44,40.360000000000014,134.07999999999998,138.88000000000002,cont,2.805608501288076,45000333,5,0,1,4
6bba_207c6aaf,9483,44,61.35999999999999,124.63999999999999,14.559999999999999,cont,4.531040857242405,45000339,3,0,3,0
6bba_207c6aaf,10035,46,38.56,168.8,174.56000000000006,cont,2.472831436329619,47000372,2,0,2,0
6bba_207c6aaf,10337,47,36.31999999999999,124.00000000000003,84.64000000000001,cont,2.852094898929564,48000393,1,1,0,0
6bba_207c6aaf,10345,47,37.86666666666666,135.46666666666664,127.99999999999997,cont,2.2165354754411113,48000390,3,0,0,3
6bba_207c6aaf,10719,48,50.13333333333333,165.6,77.06666666666665,cont,3.470578847212989,49000415,2,0,0,2
6bba_207c6aaf,10762,48,59.68000000000001,114.07999999999998,12.0,cont,1.671302822650659,49000417,1,1,0,0
6bba_207c6aaf,10984,49,36.44000000000001,151.2,178.08,cont,6.037481884030786,50000429,1,1,0,0
6bba_207c6aaf,10985,49,34.040000000000006,163.36,167.51999999999998,cont,6.536783421530797,50000429,1,1,0,0
6bba_207c6aaf,11035,49,46.68,171.20000000000002,77.28,cont,3.01274298812561,50000432,3,3,0,0
6bba_207c6aaf,11046,49,49.68000000000001,160.8,73.76000000000002,cont,3.6943294807718585,50000432,3,3,0,0
6bba_207c6aaf,11105,49,61.99999999999999,124.53333333333335,13.06666666666666,cont,5.816930062087913,50000434,2,0,0,2
6bba_207c6aaf,11300,50,30.32,146.56000000000003,140.64000000000001,cont,3.817920122592929,51000443,1,0,1,0
6bba_207c6aaf,11374,50,42.73333333333332,158.9333333333333,67.46666666666667,cont,2.0698826204642797,51000452,4,0,0,4
6bba_207c6aaf,11538,51,3.6799999999999997,92.48,160.64000000000001,cont,2.3463672053836726,52000458,5,2,0,3
6bba_207c6aaf,11894,52,4.8,89.27999999999999,123.83999999999997,cont,3.316951019008264,53000478,4,0,2,2
6bba_207c6aaf,12015,52,27.93333333333333,109.86666666666663,99.46666666666667,cont,0.7960841664045205,53000481,3,0,0,3
6bba_207c6aaf,12241,53,3.5199999999999996,97.92000000000002,180.48,cont,3.0390975737050607,54000493,2,0,0,2
6bba_207c6aaf,13933,58,2.84,89.28000000000003,93.44,cont,5.61098169886518,59000575,1,1,0,0
6bba_207c6aaf,14266,59,2.6399999999999997,88.79999999999998,88.16,cont,6.965490044677404,60000596,2,1,1,0
6bba_207c6aaf,14383,59,22.36,155.35999999999999,166.72000000000003,cont,4.8198716269211825,60000600,1,1,0,0
6bba_207c6aaf,14715,60,21.079999999999995,152.95999999999998,157.92000000000002,cont,4.578752900626982,61000616,2,1,1,0
6bba_207c6aaf,15062,61,23.96,150.56000000000003,147.68,cont,5.961801452790923,62000628,6,0,3,3
6bba_207c6aaf,15841,63,58.680000000000014,184.64,231.20000000000002,cont,2.0924044183665704,64000662,2,2,0,0
6bba_207c6aaf,16147,64,57.48000000000001,153.44000000000003,223.36,cont,2.6243121122496196,65000674,2,0,2,0
6bba_207c6aaf,16150,64,58.31999999999999,185.28,227.35999999999999,cont,1.9771649526531676,65000675,2,0,2,0
6bba_207c6aaf,16173,64,61.99999999999999,189.59999999999994,227.46666666666664,cont,6.9123543283826905,65000675,1,0,0,1
6bba_207c6aaf,16257,65,15.799999999999997,147.36000000000004,129.92000000000002,cont,1.4035334873097802,66000679,1,0,1,0
6bba_207c6aaf,16432,65,58.16000000000001,186.72000000000006,222.07999999999996,cont,1.6398818555005936,66000685,1,1,0,0
6bba_207c6aaf,17843,70,54.0,148.64000000000001,195.2,cont,0.5524999999999999,71000734,2,2,0,0
6bba_207c6aaf,18108,71,53.31999999999999,184.0,192.0,cont,1.8348113833579842,72000748,1,0,1,0
6bba_207c6aaf,18111,71,53.68000000000001,148.64000000000001,192.00000000000006,cont,1.1351761977772625,72000747,2,0,2,0
6bba_207c6aaf,18398,72,58.44000000000001,158.71999999999997,232.64000000000004,cont,0.9286700840449017,73000762,1,1,0,0
6bba_207c6aaf,18475,73,4.866666666666665,135.99999999999994,90.93333333333332,cont,1.3115158347203737,74000766,3,1,0,2
6bba_207c6aaf,18694,73,58.64,160.64000000000001,228.16000000000003,cont,1.4436915572586833,74000774,6,1,1,4
6bba_207c6aaf,18950,74,54.26666666666665,159.99999999999994,223.99999999999991,cont,1.704529555902283,75000784,8,2,0,6
6bba_207c6aaf,19979,78,43.04,177.28,166.08,cont,1.8788963682172615,79000820,1,1,0,0
6bba_207c6aaf,20235,79,41.68000000000001,176.0,158.72,cont,1.2131035868795552,80000830,1,0,1,0
6bba_207c6aaf,20259,79,47.519999999999996,178.72000000000003,180.0,cont,2.460574119997198,80000831,1,1,0,0
6bba_207c6aaf,20384,80,9.32,149.44,240.0,cont,5.982670938844957,81000837,2,1,1,0
6bba_207c6aaf,20647,81,7.639999999999999,148.64000000000001,233.27999999999997,cont,3.909332583702747,82000847,1,0,1,0
6bba_207c6aaf,21282,83,37.64,145.27999999999997,104.0,cont,2.667426486803337,84000869,1,1,0,0
6bba_207c6aaf,21296,83,41.6,175.36,154.4,cont,2.855610978844985,84000872,1,1,0,0
6bba_207c6aaf,21518,84,33.86666666666666,168.53333333333333,107.99999999999999,cont,1.8513989174105405,85000879,4,0,0,4
6bba_207c6aaf,21524,84,37.32,145.28,96.0,cont,2.351931387179485,85000880,1,0,1,0
6bba_207c6aaf,21754,85,40.51213195800781,152.40980712890627,136.61617919921872,cont,5.351552312292421,86000894,1,0,1,0
6bba_207c6aaf,22852,90,30.32,163.36,80.64,cont,5.783288019154852,91000949,4,2,2,0
6bba_207c6aaf,22870,90,35.533333333333324,149.59999999999997,83.99999999999997,cont,4.650532470260191,91000949,1,0,1,0
6bba_207c6aaf,22927,90,51.52,183.36,142.23999999999998,cont,1.0142916247312732,91000950,3,0,0,3
6bba_207c6aaf,23081,91,30.360000000000003,145.44,66.24,cont,3.1269324025312755,92000957,5,0,1,4
6bba_207c6aaf,23320,92,48.519999999999996,180.64000000000001,129.28,cont,3.031790158058438,93000967,2,0,0,2
6bba_20852818,601,2,11.559999999999999,222.08000000000004,87.36,cont,5.183367932628354,3000051,2,2,0,0
6bba_20852818,626,2,18.040000000000003,20.799999999999997,50.080000000000005,cont,3.4402443590535787,3000036,4,4,0,0
6bba_20852818,774,2,61.84,82.08,153.44000000000003,cont,1.7335787586665883,3000049,2,2,0,0
6bba_20852818,775,2,61.52000000000001,85.44,171.36,cont,6.695729632571801,3000049,3,3,0,0
6bba_20852818,813,3,4.6400000000000015,203.36,72.64,cont,2.28270569993155,4000065,2,1,1,0
6bba_20852818,833,3,9.319999999999999,218.08000000000004,84.8,cont,4.310172741607931,4000066,2,0,2,0
6bba_20852818,866,3,16.32,14.559999999999999,46.56,cont,1.1422490151013485,4000051,4,0,4,0
6bba_20852818,876,3,17.040000000000003,156.79999999999998,74.72,cont,4.42241891531094,4000053,2,1,1,0
6bba_20852818,884,3,19.84,178.72000000000003,69.44000000000001,cont,6.8597522960016715,4000053,2,0,2,0
6bba_20852818,939,3,32.388789672851566,94.94163452148437,128.27710205078125,cont,5.0285262576110314,4000067,3,0,1,2
6bba_20852818,956,3,38.040000000000006,49.43999999999999,82.88000000000001,cont,2.9562497357293815,4000055,4,0,1,3
6bba_20852818,1020,3,60.84000000000001,80.0,166.72000000000003,cont,6.8096031685040375,4000064,3,0,3,0
6bba_20852818,1024,3,61.36000000000001,76.63999999999999,148.00000000000006,cont,1.3862116766569186,4000064,2,0,2,0
6bba_20852818,1138,4,20.159999999999997,153.43999999999997,79.36,cont,6.243743993991101,5000069,1,1,0,0
6bba_20852818,1190,4,32.4,37.279999999999994,70.08,cont,6.221394789353914,5000072,2,2,0,0
6bba_20852818,1204,4,37.36,45.44000000000001,80.79999999999998,cont,4.157015531002982,5000072,2,2,0,0
6bba_20852818,1236,4,48.52000000000001,57.28000000000001,110.08,cont,1.4495327911089078,5000076,1,1,0,0
6bba_20852818,1309,5,6.120000000000001,211.84,78.56,cont,2.2955677402333414,6000082,1,1,0,0
6bba_20852818,1354,5,14.8,150.07999999999998,67.2,cont,1.973823225367465,6000084,3,2,1,0
6bba_20852818,1393,5,23.599999999999994,88.53333333333333,101.86666666666663,cont,4.898565726879214,6000097,2,0,0,2
6bba_20852818,1460,5,46.160000000000004,51.99999999999999,104.00000000000001,cont,1.4809804607083725,6000091,2,1,1,0
6bba_20852818,1486,5,59.0,67.36000000000001,136.0,cont,1.6315679766715212,6000094,3,3,0,0
6bba_20852818,1493,5,61.133333333333354,55.99999999999998,144.0,cont,5.88456704781907,6000094,7,3,0,4
6bba_20852818,1527,6,1.6800000000000004,201.44,67.35999999999999,cont,5.4145916858984,7000096,4,0,2,2
6bba_20852818,1552,6,5.52,210.72000000000003,76.0,cont,4.116673793853478,7000096,4,0,1,3
6bba_20852818,1587,6,13.680000000000001,144.48000000000002,64.64,cont,5.159829212289883,7000097,9,0,2,7
6bba_20852818,1660,6,35.16,34.56000000000001,73.92,cont,6.727205609500867,7000105,1,0,1,0
6bba_20852818,1664,6,35.04,60.48,82.08,cont,4.664316167724912,7000105,6,0,1,5
6bba_20852818,1682,6,43.959999999999994,46.56000000000001,98.56,cont,1.9791005690717094,7000106,1,0,1,0
6bba_20852818,1697,6,50.12,7.840000000000001,104.64,cont,1.0201324975217658,7000109,1,0,1,0
6bba_20852818,1711,6,57.48,63.2,130.56,cont,1.5775896369144928,7000111,3,0,3,0
6bba_20852818,11395,6,9.339449691772462,145.30858642578124,57.52219467163087,cont,2.719909370595082,7000097,2,0,2,0
6bba_20852818,2358,9,48.31999999999999,156.0,166.08,cont,4.43398751830675,10000153,1,1,0,0
6bba_20852818,2923,12,21.066666666666666,69.06666666666666,90.93333333333331,cont,1.9261563085585784,13000181,6,0,0,6
6bba_20852818,3480,15,12.360000000000003,78.72000000000001,55.36,cont,5.695062828889601,16000213,1,1,0,0
6bba_20852818,3641,16,3.5600000000000005,68.8,38.08,cont,2.788189376638538,17000224,3,0,2,1
6bba_20852818,3697,16,20.080000000000002,44.63999999999999,96.95999999999998,cont,3.7169899196123803,17000230,1,0,1,0
6bba_20852818,3727,16,30.839999999999996,7.5200000000000005,66.08,cont,3.5361583144706663,17000228,4,0,2,2
6bba_20852818,3736,16,34.999999999999986,2.666666666666659,76.0,cont,6.746414299668944,17000228,4,0,0,4
6bba_20852818,4305,19,58.839999999999996,239.36,235.99999999999994,cont,0.8225132977648529,20000263,1,0,1,0
6bba_20852818,4426,20,28.16,10.559999999999999,69.76,cont,2.9017968571214663,21000268,1,1,0,0
6bba_20852818,4576,21,23.240000000000006,2.72,52.800000000000004,cont,3.3706511333123674,22000279,3,0,1,2
6bba_20852818,4601,21,35.56,47.360000000000014,85.44000000000001,cont,5.786186676041478,22000283,5,0,2,3
6bba_20852818,4628,21,56.48,242.71999999999997,233.27999999999997,cont,0.9820164204329901,22000287,1,1,0,0
6bba_20852818,4776,22,55.16000000000001,243.36000000000004,232.0,cont,0.5479407472528375,23000299,1,0,1,0
6bba_20852818,4920,23,59.68000000000001,192.64,187.99999999999997,cont,1.111671039471666,24000313,1,1,0,0
6bba_20852818,5062,24,58.839999999999996,193.27999999999997,185.28,cont,0.4075479266294958,25000323,1,0,1,0
6bba_20852818,5832,30,50.36000000000001,201.92000000000002,172.0,cont,2.7215718114721787,31000385,1,1,0,0
6bba_20852818,5896,31,10.799999999999999,124.16000000000003,114.07999999999998,cont,2.6161995336747514,32000389,1,0,1,0
6bba_20852818,5928,31,25.479999999999997,68.8,76.0,cont,1.2953196999196805,32000391,1,0,1,0
6bba_20852818,6210,33,61.0,79.36000000000001,156.64000000000001,cont,1.722576649237998,34000420,1,1,0,0
6bba_20852818,6250,34,10.000000000000004,10.080000000000002,39.36,cont,1.645989444073071,35000425,1,0,1,0
6bba_20852818,6810,39,13.999999999999995,75.99999999999999,72.53333333333332,cont,6.600169992023775,40000489,3,0,0,3
6bba_20852818,7099,42,2.5199999999999996,13.280000000000001,23.360000000000007,cont,1.3580178570254506,43000527,1,1,0,0
6bba_20852818,7117,42,10.04,238.07999999999998,115.36000000000001,daughter,1.696627629298779,43000542,1,1,0,0
6bba_20852818,7146,42,24.880000000000003,123.19999999999999,120.63999999999999,cont,3.564829282378045,43000534,2,2,0,0
6bba_20852818,7214,43,10.959999999999999,68.0,53.28,cont,1.9459332593899525,44000545,1,0,1,0
6bba_20852818,7265,43,50.83999999999999,114.88,134.88,cont,3.2519900157442048,44000552,2,0,1,1
6bba_20852818,7267,43,55.960000000000015,39.36,130.56,cont,1.4975589512603484,44000556,1,0,1,0
6bba_20852818,7562,46,51.8,117.44,137.44,cont,2.205275280435526,47000603,1,1,0,0
6bba_20852818,7563,46,52.72,241.28000000000003,193.43999999999997,cont,2.586201667020579,47000606,1,1,0,0
6bba_20852818,7705,47,50.96000000000001,242.71999999999997,189.92000000000002,cont,0.9290965033299914,48000624,1,0,1,0
6bba_20852818,7951,50,30.52,158.08000000000004,115.36000000000001,cont,2.308930014638801,51000663,1,1,0,0
6bba_20852818,8004,51,12.64,149.44,110.72000000000003,cont,0.6780359964633204,52000672,1,0,1,0
6bba_20852818,8188,53,38.36000000000001,118.72000000000003,113.28,cont,0.6638676920140074,54000701,2,2,0,0
6bba_20852818,8232,54,9.639999999999999,146.72000000000003,105.28000000000002,cont,0.8355724085918526,55000709,1,1,0,0
6bba_20852818,8256,54,37.319999999999986,116.0,111.36,cont,0.5401750295043026,55000714,2,0,2,0
6bba_20852818,8258,54,42.0,33.44,102.72000000000001,cont,1.6447857953241205,55000715,1,1,0,0
6bba_20852818,8259,54,44.48,244.00000000000003,173.27999999999997,cont,1.1636633963479257,55000716,1,0,1,0
6bba_20852818,8264,54,47.32000000000001,212.64,205.28,cont,1.2299650655607972,55000719,1,0,1,0
6bba_20852818,8304,55,8.839999999999998,145.92000000000002,104.64,cont,2.012705346666523,56000721,1,0,1,0
6bba_20852818,8331,55,41.0,32.00000000000001,100.63999999999999,cont,1.8989569505389015,56000725,1,0,1,0
6bba_20852818,8546,58,41.879999999999995,240.15999999999997,169.44,cont,2.612967661491449,59000758,1,1,0,0
6bba_20852818,8874,63,30.279999999999994,86.08,98.72,cont,2.567397149838718,64000810,1,0,0,1
6bba_20852818,9149,67,37.84,28.640000000000004,106.72,cont,2.031964874204268,68000866,1,1,0,0
6bba_20852818,9218,68,37.160000000000004,26.720000000000002,105.28000000000002,cont,1.6753269270503544,69000878,1,0,1,0
6bba_20852818,9286,69,37.000000000000014,201.28000000000006,192.64,cont,1.6575796549487203,70000890,1,0,1,0
6bba_20852818,9609,74,28.04,212.79999999999995,147.35999999999996,cont,2.093476847853845,75000947,2,2,0,0
6bba_20852818,9674,75,26.44,209.11999999999998,144.64000000000001,cont,1.4783035040207297,76000958,2,0,2,0
6bba_20852818,9680,75,33.2,232.64000000000004,170.07999999999998,cont,0.7420042536939943,76000962,2,0,0,2
6bba_20852818,10141,82,24.28,117.44,112.64,parent,2.330104946670856,83001034,1,1,0,0
6bba_20852818,10202,83,16.4,120.0,108.64,daughter,2.6889208453392617,84001041,1,0,0,1
6bba_20852818,10209,83,25.72,114.55999999999999,113.44,daughter,3.8118634714532966,84001044,1,0,1,0
6bba_20852818,10622,89,12.840000000000002,113.28,96.63999999999999,cont,1.8940837164708415,90001112,1,1,0,0
6bba_2312ac41,533,2,10.160000000000002,130.72,245.28000000000003,cont,6.028487191037243,3000042,1,1,0,0
6bba_2312ac41,755,3,5.7333333333333325,131.99999999999997,236.5333333333333,cont,2.298894850732029,4000061,2,0,0,2
6bba_2312ac41,944,3,58.839999999999996,100.0,54.08,cont,0.9170818188689625,4000075,1,1,0,0
6bba_2312ac41,1150,4,50.52,138.07999999999998,138.08000000000004,cont,2.434085107899885,5000093,1,1,0,0
6bba_2312ac41,1176,4,58.999999999999986,96.0,56.0,cont,1.7235727791422364,5000094,1,0,1,0
6bba_2312ac41,1195,4,61.99999999999999,110.39999999999995,60.533333333333346,cont,6.726815249639126,5000094,2,0,0,2
6bba_2312ac41,1370,5,50.00000000000001,138.72,138.56,cont,3.27105678642238,6000111,1,0,1,0
6bba_2312ac41,2267,9,48.32000000000001,140.00000000000003,144.64000000000001,cont,4.408664232224985,10000179,1,1,0,0
6bba_2312ac41,2462,10,41.133333333333326,211.99999999999994,139.46666666666664,parent,2.29458337116395,11000190,5,1,0,4
6bba_2312ac41,2477,10,47.959999999999994,140.00000000000003,145.27999999999997,cont,4.989173434798607,11000194,1,0,1,0
6bba_2312ac41,2649,11,33.00000000000001,198.56000000000003,145.44,daughter,5.088144615181437,12000213,3,0,1,2
6bba_2312ac41,2676,11,42.60000000000001,212.00000000000006,136.96000000000004,daughter,3.983769610871088,12000206,1,0,1,0
6bba_2312ac41,3469,15,36.115916137695315,216.1403198242187,159.2154248046875,cont,6.637227309359223,16000259,1,1,0,0
6bba_2312ac41,3883,17,50.0,205.28,40.000000000000014,cont,1.889756928813864,18000288,1,1,0,0
6bba_2312ac41,4075,18,50.16,202.72000000000003,40.63999999999999,cont,1.3941893926221143,19000300,1,0,1,0
6bba_2312ac41,4607,21,42.84,216.0,136.64,cont,2.1257087053498123,22000334,1,1,0,0
6bba_2312ac41,4783,22,42.480000000000004,215.36,138.08000000000004,cont,1.1127394405250424,23000345,1,0,1,0
6bba_2312ac41,6222,31,24.0,117.28,217.28000000000003,cont,3.4451532984759874,32000446,1,1,0,0
6bba_2312ac41,6385,32,24.0,117.28,218.72000000000003,cont,1.7099602517310108,33000458,1,0,1,0
6bba_2312ac41,6687,34,16.0,81.28000000000002,174.72,cont,2.121729836713433,35000483,1,1,0,0
6bba_2312ac41,6843,35,16.16,80.00000000000003,174.72,cont,1.5863529320425556,36000493,1,0,1,0
6bba_2312ac41,7370,38,49.68,128.64000000000001,160.00000000000006,cont,2.86018465312993,39000540,1,1,0,0
6bba_2312ac41,7513,39,45.13333333333333,127.99999999999997,163.99999999999994,cont,3.060416666666667,40000553,3,0,0,3
6bba_2312ac41,7520,39,50.00000000000001,129.28,160.00000000000006,cont,5.13996002537958,40000553,1,0,1,0
6bba_2312ac41,7724,41,8.999999999999998,3.9999999999999996,212.95999999999998,cont,2.021541800210913,42000570,1,1,0,0
6bba_2312ac41,7825,41,58.839999999999996,164.0,49.28,cont,0.999077699681062,42000580,1,1,0,0
6bba_2312ac41,7856,42,5.999999999999998,3.999999999999999,211.99999999999994,cont,5.025034981221534,43000583,1,0,0,1
6bba_2312ac41,7866,42,8.999999999999996,3.999999999999999,215.4666666666666,cont,0.18958333333331723,43000583,1,0,1,0
6bba_2312ac41,7963,42,58.48000000000002,164.0,51.36000000000001,cont,0.9489220858426421,43000593,1,0,1,0
6bba_2312ac41,7968,42,59.68000000000001,192.0,92.0,cont,5.71468858512867,43000595,1,1,0,0
6bba_2312ac41,7969,42,59.68000000000001,192.64,107.35999999999999,cont,2.3540636408134583,43000595,1,1,0,0
6bba_2312ac41,8106,43,59.0,191.36,95.36000000000001,cont,4.665674689956417,44000607,1,0,1,0
6bba_2312ac41,8107,43,59.160000000000004,193.92000000000002,108.64,cont,1.7757168446855436,44000607,1,0,1,0
6bba_2312ac41,8472,46,36.36000000000001,225.28000000000003,145.44,cont,1.1848021986812793,47000641,1,1,0,0
6bba_2312ac41,8610,47,32.99999999999999,228.0,144.0,cont,6.525341226709925,48000653,4,0,0,4
6bba_2312ac41,8616,47,36.48,228.0,144.64000000000001,cont,0.9489220858426727,48000653,1,0,1,0
6bba_2312ac41,8899,49,40.839999999999996,133.28,173.92000000000002,cont,3.549388371607147,50000680,1,1,0,0
6bba_2312ac41,9040,50,41.31999999999999,135.36,173.91999999999996,cont,2.8695862071037554,51000692,1,0,1,0
6bba_2312ac41,9983,58,9.159999999999998,180.64000000000001,215.36000000000004,cont,1.0268405730686951,59000778,1,1,0,0
6bba_2312ac41,10111,59,8.799999999999999,183.99999999999997,213.28000000000003,cont,0.6132087735836885,60000789,1,0,1,0
6bba_2312ac41,10821,65,8.000000000000002,204.00000000000003,208.0,cont,0.5745242597140799,66000851,1,1,0,0
6bba_2312ac41,10929,66,7.6800000000000015,207.36000000000004,208.0,cont,0.6758906161502796,67000862,1,0,1,0
6bba_2312ac41,11515,71,61.839999999999996,192.0,32.64,cont,0.5479407472528417,72000933,1,1,0,0
6bba_2312ac41,11609,72,61.84,194.08000000000004,31.360000000000007,cont,0.8350982352394091,73000943,1,0,1,0
6bba_2312ac41,12491,83,31.68,86.72,210.72000000000003,cont,1.0683061944030865,84001070,1,1,0,0
6bba_2312ac41,12542,84,10.000000000000002,180.0,170.72000000000003,cont,0.8204238614886794,85001075,1,1,0,0
6bba_2312ac41,12567,84,30.680000000000003,92.0,210.08000000000004,cont,0.6403819660952251,85001080,1,0,1,0
6bba_2312ac41,12593,84,55.00000000000001,76.63999999999999,118.72,cont,1.8761538116582983,85001084,1,1,0,0
6bba_2312ac41,12622,85,8.799999999999999,182.72,169.28,cont,0.6236698345278499,86001087,1,0,1,0
6bba_2312ac41,12670,85,54.32000000000001,82.08000000000001,117.28,cont,1.2026097922850834,86001096,1,0,1,0
6bba_2312ac41,13032,90,25.8,5.920000000000001,198.08,cont,0.4963570413522884,91001157,1,0,1,0
6bba_2312ac41,13065,90,60.16000000000001,17.919999999999998,114.07999999999998,cont,0.5890484912127487,91001161,1,1,0,0
6bba_2312ac41,13140,91,59.32000000000001,26.719999999999995,113.28000000000002,cont,1.7040819559223142,92001172,1,0,1,0
6bba_23af9eeb,4845,89,47.84,2.7199999999999998,134.72000000000003,cont,2.2708757259039927,90000678,1,1,0,0
6bba_23af9eeb,4873,90,46.120000000000005,1.2800000000000002,137.44,cont,2.1658269206009977,91000683,1,0,1,0
6bba_23af9eeb,4874,90,49.266666666666666,3.999999999999999,135.99999999999994,cont,5.8412163996703566,91000683,1,0,0,1
6bba_2646afc7,246,4,26.48,175.99999999999997,215.99999999999997,cont,1.2406531797807285,5000061,1,1,0,0
6bba_2646afc7,306,5,25.640000000000008,177.92,212.64,cont,0.7457316038629354,6000074,1,0,1,0
6bba_2646afc7,358,6,14.160000000000004,204.00000000000003,232.8,cont,1.6616369940513362,7000083,1,1,0,0
6bba_2646afc7,412,7,14.68,204.00000000000003,228.64,cont,2.282532173924384,8000097,1,0,1,0
6bba_2646afc7,711,12,42.360000000000014,139.35999999999999,173.44,cont,1.5111626318831366,13000154,1,1,0,0
6bba_2646afc7,740,13,3.16,134.72,71.36,cont,3.003437873920485,14000159,1,1,0,0
6bba_2646afc7,774,13,41.52,138.56,167.84,cont,2.6771104408484927,14000166,1,0,1,0
6bba_2646afc7,777,13,44.99999999999999,139.99999999999994,179.99999999999997,cont,6.953879874753352,14000166,2,0,0,2
6bba_2646afc7,804,14,3.1599999999999993,135.35999999999999,71.35999999999999,cont,1.4136566061105535,15000171,1,0,1,0
6bba_2646afc7,1566,27,3.6799999999999997,80.64,11.36,cont,1.5296593779335332,28000326,1,1,0,0
6bba_2646afc7,1620,28,3.3200000000000003,81.44,10.720000000000002,cont,2.3077860738162004,29000339,1,0,1,0
6bba_2646afc7,1742,30,12.840000000000002,178.72,171.99999999999994,cont,3.8900002008354573,31000368,1,1,0,0
6bba_2646afc7,1801,31,12.32,179.36000000000004,171.99999999999994,cont,2.729080946490967,32000380,1,0,1,0
6bba_2646afc7,2003,34,48.040000000000006,149.28,241.28000000000003,cont,0.7109258927623806,35000425,1,1,0,0
6bba_2646afc7,2062,35,47.68000000000001,152.0,242.71999999999997,cont,1.9122597888362523,36000437,1,0,1,0
6bba_2646afc7,2324,40,8.84,196.64000000000001,185.92000000000002,cont,2.336047008302705,41000483,1,1,0,0
6bba_2646afc7,2386,41,8.16,199.36000000000007,190.07999999999998,cont,1.4571638763365165,42000495,2,1,1,0
6bba_2646afc7,2447,42,1.0,157.28,91.35999999999997,cont,2.666683919215029,43000504,1,1,0,0
6bba_2646afc7,2455,42,7.866666666666666,192.0,179.99999999999994,cont,4.1882078589110945,43000505,1,0,0,1
6bba_2646afc7,2456,42,7.76,200.96000000000004,192.64,cont,2.219657208895108,43000505,1,0,1,0
6bba_2646afc7,2522,43,1.0,160.00000000000003,92.64,cont,1.7163569704464032,44000516,1,0,1,0
6bba_2646afc7,2871,48,5.320000000000001,210.08000000000004,194.08000000000004,cont,1.5119487631860908,49000561,1,1,0,0
6bba_2646afc7,2949,49,5.0,212.00000000000006,196.00000000000003,cont,1.8168052317185537,50000570,1,0,1,0
6bba_2646afc7,3061,50,41.19999999999999,177.28,182.07999999999993,cont,6.765310414164273,51000580,1,1,0,0
6bba_2646afc7,3066,50,44.879999999999995,79.19999999999999,74.72,cont,3.5968338942047415,51000583,1,1,0,0
6bba_2646afc7,3136,51,40.160000000000004,188.00000000000003,246.72,cont,0.860636573996236,52000594,1,1,0,0
6bba_2646afc7,3137,51,43.64,82.72,73.27999999999999,cont,3.0936517661171865,52000595,1,0,1,0
6bba_2646afc7,3207,52,39.12,190.08000000000004,245.28,cont,0.5130987356250158,53000602,1,0,1,0
6bba_2646afc7,3427,55,37.32000000000001,139.35999999999999,132.00000000000006,cont,2.7639334516228726,56000626,1,1,0,0
6bba_2646afc7,3497,56,36.68000000000001,141.28000000000003,132.00000000000006,cont,2.186093119013904,57000636,1,0,1,0
6bba_2646afc7,3840,61,28.400000000000006,192.0,156.16,cont,4.845552729049593,62000675,1,1,0,0
6bba_2646afc7,3860,61,53.56,110.56,146.72000000000003,cont,2.698723367168259,62000679,1,1,0,0
6bba_2646afc7,3896,62,21.733333333333327,207.46666666666664,172.0,cont,2.2988948507320397,63000686,2,0,0,2
6bba_2646afc7,3924,62,51.96000000000001,114.08000000000001,145.27999999999997,cont,2.289116914882248,63000685,1,0,1,0
6bba_2646afc7,4315,69,30.839999999999996,129.92000000000002,84.0,cont,2.752205081838921,70000719,1,1,0,0
6bba_2646afc7,4368,70,29.840000000000003,132.64000000000001,84.64000000000001,cont,2.8526966408996226,71000724,1,0,1,0
6bba_2646afc7,4653,75,25.159999999999997,175.36,145.92000000000002,cont,1.4227800690901014,76000760,1,1,0,0
6bba_2646afc7,4711,76,24.0,176.64000000000001,145.92000000000002,cont,1.7166646440117517,77000766,1,0,1,0
6bba_2646afc7,4946,80,21.880000000000006,180.64000000000001,148.00000000000003,cont,1.9449152455826915,81000803,1,1,0,0
6bba_2646afc7,5001,81,21.04,181.28000000000003,147.36000000000004,cont,1.0044813649341757,82000812,1,0,1,0
6bba_2646afc7,5193,84,57.359999999999985,9.28,29.439999999999998,cont,2.8756531866690933,85000844,1,1,0,0
6bba_2646afc7,5249,85,55.800000000000004,11.36,26.560000000000002,cont,1.7931784803805868,86000853,1,0,1,0
6bba_2646afc7,5572,92,6.2,245.92,238.72000000000003,cont,5.1854562239016,93000910,1,1,0,0
6bba_2646afc7,5628,93,6.800000000000001,244.64000000000001,235.99999999999994,cont,6.3245910193861805,94000916,2,1,1,0
6bba_2646afc7,5681,94,3.999999999999999,243.99999999999997,247.99999999999997,cont,1.8616713760758083,95000923,2,0,0,2
6bba_268e1230,145,6,58.15999999999998,177.28,218.08000000000004,cont,0.2856489322577416,7000042,1,1,0,0
6bba_268e1230,166,7,57.68000000000001,184.0,216.64000000000004,cont,0.5813776741499377,8000047,1,0,1,0
6bba_268e1230,204,9,55.84000000000001,194.72,216.0,cont,2.117494096804028,10000059,1,1,0,0
6bba_268e1230,225,10,54.84,199.36000000000004,215.36,cont,0.33223203487924635,11000066,1,0,1,0
6bba_268e1230,626,25,45.13333333333333,172.5333333333333,235.46666666666673,daughter,1.0686768980847068,26000180,1,0,0,1
6bba_268e1230,641,26,8.16,204.64000000000001,228.00000000000003,cont,1.848290172700169,27000180,1,1,0,0
6bba_268e1230,674,27,7.0,211.36,224.00000000000003,cont,1.1468632492585993,28000190,1,0,1,0
6bba_268e1230,711,28,23.040000000000006,172.48000000000002,222.88000000000002,cont,3.6478987958823477,29000204,1,1,0,0
6bba_268e1230,738,29,4.160000000000001,224.16000000000003,212.00000000000006,cont,0.5908389162876542,30000210,1,1,0,0
6bba_268e1230,748,29,21.200000000000003,179.99999999999997,216.0,cont,0.9647902945718377,30000215,1,0,1,0
6bba_268e1230,789,30,2.8000000000000007,228.64000000000001,205.28,cont,1.8620259463820543,31000224,1,0,1,0
6bba_268e1230,872,31,55.519999999999996,196.00000000000006,160.64,cont,1.1033259377899018,32000242,1,1,0,0
6bba_268e1230,884,32,4.32,28.639999999999997,227.99999999999997,cont,1.1638902976655556,33000244,1,1,0,0
6bba_268e1230,915,32,53.639999999999986,201.43999999999997,157.28000000000003,cont,1.198650934384169,33000252,1,0,1,0
6bba_268e1230,926,33,3.1599999999999993,32.64,223.99999999999997,cont,0.993378861512565,34000254,1,0,1,0
6bba_268e1230,1121,36,60.47999999999999,208.64000000000001,118.72,cont,1.2197246103116817,37000302,1,1,0,0
6bba_268e1230,1180,37,59.83999999999999,217.44000000000003,112.47999999999999,cont,2.37694788541946,38000313,1,0,1,0
6bba_268e1230,1721,49,2.0000000000000004,24.640000000000008,163.36000000000004,cont,1.1675147215345778,50000420,1,1,0,0
6bba_268e1230,1774,50,1.24,27.039999999999996,159.68,cont,1.2939939383552028,51000429,1,0,1,0
6bba_268e1230,2287,58,49.67999999999999,60.800000000000004,171.04,cont,6.662995408410551,59000499,1,1,0,0
6bba_268e1230,2979,70,16.719999999999995,155.35999999999999,86.72,cont,3.7503366515554406,71000617,1,1,0,0
6bba_268e1230,3038,71,13.600000000000001,156.64,84.64000000000001,cont,1.3362599391585368,72000628,1,0,1,0
6bba_268e1230,3231,73,35.533333333333324,154.4,136.5333333333333,cont,3.176264238675522,74000652,2,0,0,2
6bba_268e1230,3730,81,3.3200000000000003,176.0,79.36,cont,2.2983842830344976,82000718,1,1,0,0
6bba_268e1230,3805,82,2.4799999999999995,178.72000000000003,78.08000000000001,cont,0.7889203461059902,83000726,1,0,1,0
6bba_268e1230,4046,85,19.68,195.35999999999999,123.35999999999999,cont,4.797494300413502,86000757,1,1,0,0
6bba_268e1230,4128,86,18.479999999999993,198.72,121.92000000000002,cont,3.2594113732390406,87000765,1,0,1,0
6bba_2819ca14,98,1,29.68,148.32,237.60000000000002,cont,2.222926351119174,2000022,1,0,1,0
6bba_2819ca14,177,2,28.733333333333334,1.0666666666666675,207.99999999999997,cont,5.366123565298839,3000038,1,0,0,1
6bba_2819ca14,193,2,35.64,1.2799999999999998,200.64,cont,6.664501223835137,3000038,1,0,1,0
6bba_2819ca14,259,3,30.320000000000007,152.63999999999996,231.36,cont,2.797455316890684,4000053,1,1,0,0
6bba_2819ca14,341,4,30.679999999999996,156.00000000000003,228.64000000000001,cont,2.251959216437988,5000069,1,0,1,0
6bba_2819ca14,557,7,24.0,72.64,237.9200000000001,cont,1.6318916362614269,8000106,1,1,0,0
6bba_2819ca14,627,8,23.840000000000003,77.28,238.72000000000003,cont,2.076505854193517,9000121,1,0,1,0
6bba_2819ca14,754,9,42.67999999999999,20.000000000000004,72.0,cont,2.1831317098379666,10000147,1,1,0,0
6bba_2819ca14,830,10,42.68000000000001,22.560000000000006,68.0,cont,2.3796681097371386,11000164,1,0,1,0
6bba_2819ca14,1240,16,39.52,192.16000000000003,7.840000000000001,cont,1.837471938561232,17000256,1,0,0,1
6bba_2819ca14,1346,18,34.52,184.64,118.72,cont,2.41212626638822,19000279,1,1,0,0
6bba_2819ca14,1403,19,34.0,187.35999999999999,117.28,cont,1.759958273653102,20000289,1,0,1,0
6bba_2819ca14,1524,21,33.96,188.79999999999995,113.44,cont,1.999278385443117,22000308,1,1,0,0
6bba_2819ca14,1557,21,43.68,52.480000000000004,33.44,cont,2.0882356488193565,22000311,1,1,0,0
6bba_2819ca14,1583,22,31.680000000000003,218.40000000000003,114.72,cont,3.807427363062891,23000318,1,0,0,1
6bba_2819ca14,1613,22,43.31999999999999,59.519999999999996,29.28,cont,2.191762989011367,23000321,1,0,1,0
6bba_2819ca14,2007,30,38.0,16.000000000000004,165.28000000000003,cont,3.3163140777827436,31000390,1,1,0,0
6bba_2819ca14,2059,31,37.84,21.919999999999998,166.07999999999998,cont,3.5570058511337788,32000398,1,0,1,0
6bba_2819ca14,2068,31,41.480000000000004,127.36,7.360000000000001,cont,1.7055534292422492,32000399,1,1,0,0
6bba_2819ca14,2120,32,41.52,130.72,4.640000000000001,cont,1.2197246103116914,33000408,1,0,1,0
6bba_2819ca14,2155,33,37.160000000000004,12.64,174.72,cont,2.2257754968774304,34000416,1,1,0,0
6bba_2819ca14,2208,34,37.000000000000014,18.72,172.64,cont,1.793399355971765,35000426,1,0,1,0
6bba_2819ca14,2384,37,41.68000000000001,8.0,96.0,cont,2.2937264985171884,38000458,1,1,0,0
6bba_2819ca14,2437,38,41.84,9.28,95.36000000000001,cont,3.689000542152296,39000467,1,0,1,0
6bba_2819ca14,2544,40,42.0,13.920000000000002,94.56,cont,2.32806417705784,41000483,1,1,0,0
6bba_2819ca14,2592,41,42.0,18.080000000000002,94.72,cont,3.656358331728443,42000492,1,0,1,0
6bba_2819ca14,2867,47,29.519999999999996,10.559999999999999,170.72000000000003,cont,2.510822200993146,48000543,1,1,0,0
6bba_2819ca14,2881,47,34.68000000000001,34.56,192.64000000000004,cont,2.4212499999999824,48000545,1,1,0,0
6bba_2819ca14,2901,48,12.520000000000001,193.92000000000002,242.71999999999997,cont,1.4788392830189478,49000551,1,1,0,0
6bba_2819ca14,2911,48,29.32,16.0,166.72000000000003,cont,1.539296206875083,49000553,1,0,1,0
6bba_2819ca14,2927,48,34.16,41.92000000000001,190.72000000000003,cont,3.2736390301925478,49000555,1,0,1,0
6bba_2819ca14,2947,49,11.319999999999997,203.35999999999996,240.8,cont,2.6962271093140675,50000561,1,0,1,0
6bba_2819ca14,3020,50,33.160000000000004,59.359999999999985,184.0,cont,1.9166735507122743,51000576,1,1,0,0
6bba_2819ca14,3073,51,32.84,68.64000000000001,180.64000000000001,cont,1.3805807202043552,52000587,1,0,1,0
6bba_2819ca14,3097,52,12.48,233.27999999999997,244.64000000000001,cont,6.528557579013917,53000592,1,1,0,0
6bba_2819ca14,3105,52,24.199999999999992,4.000000000000002,221.27999999999997,cont,2.034432506867722,53000595,1,0,1,0
6bba_2819ca14,3146,53,11.08,243.04,241.75999999999996,cont,6.735443679706621,54000604,1,0,1,0
6bba_2819ca14,3496,60,29.0,166.24,127.2,cont,2.9858020802625256,61000695,2,2,0,0
6bba_2819ca14,3499,60,29.36,193.76000000000005,85.44,cont,5.226036380470392,61000700,2,2,0,0
6bba_2819ca14,3500,60,30.84,125.43999999999998,143.36,cont,2.8614768913971633,61000696,1,1,0,0
6bba_2819ca14,3513,60,34.72,137.28,53.120000000000005,cont,5.5230161879628765,61000698,1,1,0,0
6bba_2819ca14,3539,61,23.639999999999997,47.36,241.44000000000003,parent,1.4935862880998765,62000719,2,1,1,0
6bba_2819ca14,3546,61,26.999999999999993,193.06666666666666,107.46666666666665,cont,1.6826576614259678,62000716,2,0,0,2
6bba_2819ca14,3549,61,28.16,203.35999999999999,78.56,cont,1.6458290084331395,62000711,2,0,2,0
6bba_2819ca14,3552,61,28.519999999999996,162.24,127.52000000000004,cont,6.806286845630292,62000713,2,0,2,0
6bba_2819ca14,3564,61,33.519999999999996,164.31999999999996,39.84,cont,4.162316249697986,62000714,1,0,1,0
6bba_2819ca14,3589,62,22.48,47.839999999999996,244.8,daughter,3.1664575040571745,63000734,1,0,1,0
6bba_2819ca14,3593,62,24.133333333333326,66.93333333333334,232.0,daughter,1.9350847401054148,63000735,3,0,0,3
6bba_2819ca14,4234,74,1.0,197.28000000000003,234.0800000000001,cont,1.9332037043467305,75000913,1,1,0,0
6bba_2819ca14,4285,75,1.0,202.08000000000004,232.0,cont,1.592499999999987,76000928,1,0,1,0
6bba_2819ca14,4489,78,35.519999999999996,84.48,52.00000000000001,cont,4.154823439389459,79000983,1,1,0,0
6bba_2819ca14,4504,79,7.6,211.2,220.5333333333333,cont,1.2685874964349488,80001001,1,0,1,0
6bba_2819ca14,4542,79,34.32,94.88,52.15999999999999,cont,4.183357279147933,80000995,1,0,1,0
6bba_2819ca14,4556,80,5.120000000000001,232.0,216.00000000000003,cont,3.550690073774392,81001014,1,0,0,1
6bba_2819ca14,4612,81,4.720000000000001,234.24,216.0,cont,4.651021393199559,82001028,1,1,0,0
6bba_2819ca14,4639,81,24.36,141.12,132.00000000000006,cont,3.5936021708725545,82001025,1,1,0,0
6bba_2819ca14,4666,82,3.959999999999999,239.52,216.64000000000001,cont,3.1094874979648903,83001040,1,0,1,0
6bba_2819ca14,4693,82,23.159999999999997,152.8,132.00000000000006,cont,2.0526522477029525,83001035,1,0,1,0
6bba_2819ca14,5117,90,28.840000000000003,152.48,45.28,cont,2.1572754807858807,91001120,1,1,0,0
6bba_2819ca14,5163,91,28.320000000000004,158.23999999999998,42.07999999999999,cont,2.7379682818652267,92001130,1,0,1,0
6bba_2819ca14,5213,92,28.16,160.79999999999998,40.16,cont,2.8793697769650977,93001140,2,2,0,0
6bba_2819ca14,5265,93,26.266666666666655,179.46666666666667,51.99999999999999,cont,1.4584821352656114,94001148,1,0,0,1
6bba_283bf9f1,729,3,20.520000000000003,128.64000000000004,128.0,cont,1.0882648172664542,4000100,1,0,1,0
6bba_283bf9f1,858,3,51.68,196.0,239.35999999999996,cont,2.532081653501724,4000125,1,1,0,0
6bba_283bf9f1,864,3,54.99999999999999,84.53333333333333,167.46666666666664,cont,0.8619991500704859,4000121,2,0,0,2
6bba_283bf9f1,1054,4,43.32000000000001,19.360000000000003,200.64000000000001,cont,1.2008519111447535,5000149,3,0,3,0
6bba_283bf9f1,1509,6,38.040000000000006,153.27999999999997,223.36,cont,1.963225980370047,7000211,1,1,0,0
6bba_283bf9f1,1987,8,35.839999999999996,164.0,216.64000000000004,cont,1.9028465518795858,9000279,2,0,2,0
6bba_283bf9f1,2054,8,57.68000000000001,168.00000000000003,247.36,cont,2.188024937015101,9000292,1,1,0,0
6bba_283bf9f1,2290,9,56.32,167.83999999999997,244.79999999999995,cont,3.09561433725198,10000328,1,0,1,0
6bba_283bf9f1,3523,14,44.68000000000001,180.0,244.00000000000003,cont,2.144999999999982,15000520,1,1,0,0
6bba_283bf9f1,6790,27,0.8666666666666668,111.46666666666665,51.99999999999997,cont,6.937669542472854,28000913,1,0,1,0
6bba_283bf9f1,7969,31,34.839999999999996,23.36,246.72000000000006,cont,1.9726188177141575,32001058,1,1,0,0
6bba_283bf9f1,8042,31,59.52000000000001,38.71999999999999,239.36000000000007,cont,0.8300235312929196,32001064,2,2,0,0
6bba_283bf9f1,8296,32,58.48000000000002,38.559999999999995,235.99999999999994,cont,1.4002371450936164,33001092,3,1,2,0
6bba_283bf9f1,8302,32,59.519999999999996,132.64000000000001,245.28,cont,0.8457808891787534,33001093,1,0,1,0
6bba_283bf9f1,8766,34,38.76,10.560000000000002,176.64000000000004,cont,3.345211596446473,35001141,1,1,0,0
6bba_283bf9f1,9077,35,56.68,141.92000000000002,240.0,cont,0.521014635111141,36001174,1,1,0,0
6bba_283bf9f1,9842,38,52.96,149.28,233.27999999999997,cont,0.5362499999999989,39001250,1,0,1,0
6bba_283bf9f1,9849,38,56.48,47.20000000000001,226.72,cont,0.9268200000539506,39001251,1,1,0,0
6bba_283bf9f1,11406,44,50.839999999999996,164.64,234.07999999999998,cont,1.0582481100857173,45001404,1,0,1,0
6bba_283bf9f1,11414,44,52.879999999999995,61.92,220.64000000000004,cont,0.7319718659210752,45001405,1,1,0,0
6bba_283bf9f1,11633,45,42.67999999999999,4.000000000000002,204.32,cont,2.2110153917374795,46001427,1,0,1,0
6bba_283bf9f1,11876,46,37.03999999999999,108.64000000000001,240.00000000000006,cont,1.99663505991956,47001450,1,1,0,0
6bba_283bf9f1,12126,47,35.36,113.28000000000003,235.99999999999994,cont,0.7699482531313504,48001472,1,0,1,0
6bba_283bf9f1,12538,49,9.0,219.36,45.919999999999995,cont,0.4013453936449263,50001506,1,1,0,0
6bba_283bf9f1,12885,50,27.880000000000003,201.28,137.44,cont,1.0642199314521386,51001542,1,1,0,0
6bba_283bf9f1,13095,51,17.4,84.96000000000001,130.72,cont,3.5648292823780507,52001564,6,0,1,5
6bba_283bf9f1,14567,57,23.675689392089836,27.766574096679687,194.18194091796877,cont,0.5405458259539422,58001718,3,2,1,0
6bba_283bf9f1,14681,57,58.52000000000001,26.080000000000002,230.56000000000003,cont,2.515025472037996,58001731,1,1,0,0
6bba_283bf9f1,14799,58,20.0,41.28,200.64,cont,6.145100309392838,59001744,1,0,1,0
6bba_283bf9f1,23815,58,22.56030113220215,33.554791717529305,192.22936706542964,cont,1.3454162685821418,59001744,3,1,2,0
6bba_283bf9f1,15057,59,24.720000000000006,76.0,160.79999999999998,cont,2.4862499999999987,60001772,1,1,0,0
6bba_283bf9f1,15133,59,48.31999999999999,59.84000000000001,227.36,cont,0.6389371056684592,60001780,1,1,0,0
6bba_283bf9f1,15279,60,21.035689392089843,43.76657409667968,187.46194091796875,cont,0.5382020781633293,61001793,1,0,1,0
6bba_283bf9f1,16094,63,39.16,148.64000000000001,239.36,cont,3.0517136579305806,64001884,1,0,1,0
6bba_283bf9f1,17075,67,48.86666666666666,215.46666666666664,247.99999999999997,cont,0.3064129385141934,68001999,2,0,0,2
6bba_283bf9f1,17617,70,4.840000000000001,4.640000000000001,56.80000000000001,cont,1.7225766492379948,71002056,1,1,0,0
6bba_283bf9f1,18555,74,7.640000000000001,148.00000000000006,108.0,cont,0.819940927750296,75002165,1,1,0,0
6bba_283bf9f1,18695,74,43.68,206.55999999999997,243.36000000000004,cont,2.241439966851662,75002183,2,1,1,0
6bba_283bf9f1,18899,75,38.64,116.0,248.64000000000007,cont,0.6030041977465755,76002206,2,0,2,0
6bba_283bf9f1,19265,77,17.2,223.36,68.8,cont,6.303680596484885,78002245,1,1,0,0
6bba_283bf9f1,19328,77,35.32000000000001,121.28000000000003,242.71999999999997,cont,4.547736043901399,78002258,1,0,1,0
6bba_283bf9f1,19425,78,4.840000000000001,167.36000000000004,94.08000000000001,cont,2.2423233626085177,79002268,1,1,0,0
6bba_283bf9f1,19478,78,16.360000000000007,224.64000000000001,65.44000000000003,cont,4.675173292242744,79002273,1,0,1,0
6bba_283bf9f1,19488,78,18.866666666666667,215.46666666666664,74.93333333333331,cont,4.3661323822310685,79002273,2,0,0,2
6bba_283bf9f1,19492,78,19.839999999999996,84.64,119.36000000000001,cont,2.400274072892509,79002275,1,1,0,0
6bba_283bf9f1,19646,79,5.800000000000001,89.92,130.08,cont,6.713000679651393,80002289,2,0,0,2
6bba_283bf9f1,19716,79,20.68,121.60000000000001,144.64,cont,3.343316560692388,80002298,4,0,1,3
6bba_283bf9f1,20488,83,1.8400000000000005,185.28,74.72,cont,1.977164952653167,84002381,1,1,0,0
6bba_283bf9f1,20496,83,3.6399999999999997,104.64000000000001,98.88000000000001,cont,3.4685662113472753,84002379,2,2,0,0
6bba_283bf9f1,20547,83,13.480000000000002,230.08000000000004,51.2,cont,3.2584390434685124,84002388,1,1,0,0
6bba_283bf9f1,20551,83,13.360000000000001,99.20000000000002,88.00000000000001,cont,1.6408477229773641,84002384,1,1,0,0
6bba_283bf9f1,20585,83,21.679999999999996,129.92000000000002,145.28,cont,2.4685563215166937,84002403,1,1,0,0
6bba_283bf9f1,20691,84,1.1600000000000001,187.35999999999999,68.48,cont,1.4448799690631744,85002406,1,0,1,0
6bba_283bf9f1,20723,84,4.4,107.36,90.39999999999998,cont,4.95840731737521,85002404,6,0,2,4
6bba_283bf9f1,20759,84,13.680000000000001,228.64000000000001,50.72,cont,4.345287677473146,85002409,1,0,1,0
6bba_283bf9f1,20762,84,13.560000000000002,154.56,120.64000000000001,cont,1.5849374001833585,85002414,1,0,1,0
6bba_283bf9f1,20787,84,21.160000000000004,136.8,141.28,cont,2.5683226530169407,85002429,1,0,1,0
6bba_283bf9f1,20823,84,33.32,245.43999999999997,203.84000000000003,cont,1.4769632062106441,85002423,1,0,1,0
6bba_283bf9f1,21285,86,54.360000000000014,189.27999999999997,227.99999999999997,cont,1.0430424248323074,87002472,1,0,1,0
6bba_283bf9f1,21701,88,61.160000000000004,34.56,240.64000000000004,cont,0.6566689139132413,89002520,2,1,1,0
6bba_283bf9f1,21703,88,62.4,44.533333333333324,252.0,cont,6.09615693205791,89002520,1,0,0,1
6bba_283bf9f1,22214,91,15.2,189.28000000000003,152.16000000000003,cont,3.033139874783222,92002577,3,3,0,0
6bba_283bf9f1,22227,91,21.44,156.64,189.28000000000003,cont,1.0706518166518966,92002578,2,2,0,0
6bba_283bf9f1,22268,91,36.760000000000005,200.0,227.99999999999997,cont,1.7820257082881972,92002585,2,2,0,0
6bba_283bf9f1,22420,92,15.880000000000003,238.08000000000004,136.8,cont,6.69951455797359,93002595,3,0,1,2
6bba_283bf9f1,22422,92,17.04,163.20000000000002,170.88000000000002,cont,5.56844134947114,93002598,5,0,1,4
6bba_283bf9f1,22436,92,24.480000000000004,32.8,139.2,cont,1.5925000000000038,93002601,1,0,1,0
6bba_283bf9f1,22440,92,23.04,160.00000000000003,187.35999999999999,cont,6.489774769782072,93002598,5,0,2,3
6bba_283bf9f1,22462,92,32.16000000000001,200.00000000000003,210.23999999999998,cont,3.833140438987327,93002604,2,0,1,1
6bba_283bf9f1,22583,93,7.640000000000001,237.28000000000003,130.56,cont,5.604271389975683,94002613,1,1,0,0
6bba_283bf9f1,22603,93,12.32,193.92000000000002,141.92000000000002,cont,3.255358083222187,94002615,4,4,0,0
6bba_283bf9f1,22651,93,26.88,84.64000000000001,195.36,cont,5.358953372627902,94002621,3,2,1,0
6bba_283bf9f1,22660,93,31.320000000000004,205.12,205.60000000000002,cont,3.3699851631720823,94002622,1,1,0,0
6bba_283bf9f1,22778,94,7.533333333333332,243.4666666666667,109.6,cont,2.773100576170205,95002631,3,0,0,3
6bba_283bf9f1,22802,94,12.840000000000002,164.64,151.04000000000002,cont,5.358509879154832,95002636,6,0,2,4
6bba_283bf9f1,22823,94,18.240000000000002,164.0,169.44,cont,6.296156813882576,95002636,5,0,1,4
6bba_283bf9f1,22838,94,25.160000000000007,87.36,189.27999999999997,cont,1.7396609763111772,95002641,2,0,2,0
6bba_283bf9f1,22846,94,27.04,202.72000000000003,194.88,cont,4.614799731570156,95002642,5,0,3,2
6bba_283bf9f1,22856,94,30.679999999999996,213.44,198.56000000000006,cont,5.041194209956592,95002642,4,0,1,3
6bba_283bf9f1,22973,95,6.004103393554686,200.6202490234375,114.47105957031249,cont,5.582924730186044,96002653,1,0,1,0
6bba_312f0dc3,564,4,15.52,166.72,235.99999999999994,cont,2.4417213446665094,5000064,1,1,0,0
6bba_312f0dc3,692,5,14.159999999999998,161.92000000000002,235.36000000000004,cont,1.5810172753641794,6000079,1,0,1,0
6bba_312f0dc3,1123,8,59.68000000000001,242.71999999999997,201.92000000000004,cont,1.1260675490395822,9000136,1,1,0,0
6bba_312f0dc3,1226,9,59.160000000000004,236.64000000000001,197.92,cont,1.4227800690900845,10000151,2,1,1,0
6bba_312f0dc3,1327,10,48.0,249.44,223.99999999999997,cont,3.336042284654081,11000166,1,1,0,0
6bba_312f0dc3,1339,10,58.680000000000014,230.72000000000003,194.71999999999997,cont,2.1510237388276123,11000168,1,0,1,0
6bba_312f0dc3,1434,11,47.32000000000001,247.36,224.64000000000001,cont,1.4858756887774842,12000182,1,0,1,0
6bba_312f0dc3,1448,11,58.31999999999999,235.99999999999994,136.64000000000001,cont,1.9705428344748386,12000185,1,1,0,0
6bba_312f0dc3,1558,12,57.68000000000001,232.0,137.28,cont,2.4843374997169607,13000201,1,0,1,0
6bba_312f0dc3,1570,12,61.99999999999999,235.46666666666667,140.53333333333333,cont,4.986055642294359,13000201,3,0,0,3
6bba_312f0dc3,1765,14,56.160000000000004,225.92000000000002,141.92000000000002,cont,1.5724761524423825,15000231,1,1,0,0
6bba_312f0dc3,1868,15,55.320000000000014,224.00000000000003,146.08000000000007,cont,1.4400287540531775,16000244,1,0,1,0
6bba_312f0dc3,1974,16,58.839999999999996,220.64000000000004,92.0,cont,3.4177964849446605,17000263,1,1,0,0
6bba_312f0dc3,2073,17,58.15999999999998,218.72,95.36000000000001,cont,4.3704948232437255,18000277,1,0,1,0
6bba_312f0dc3,2361,20,56.52,210.56000000000003,106.56000000000003,cont,2.4855595446498544,21000315,1,1,0,0
6bba_312f0dc3,2447,21,55.480000000000004,207.51999999999998,111.68,cont,2.569967169050998,22000325,2,1,1,0
6bba_312f0dc3,2538,22,54.68000000000001,203.35999999999999,114.56000000000003,cont,3.000799112319897,23000336,3,2,1,0
6bba_312f0dc3,2625,23,53.52,199.2,119.04,cont,4.020192198452708,24000350,2,0,2,0
6bba_312f0dc3,2717,24,57.36,169.28,12.0,cont,1.0462022091832974,25000363,1,1,0,0
6bba_312f0dc3,2799,25,56.160000000000004,167.36000000000004,18.080000000000002,cont,1.9031240763544386,26000376,1,0,1,0
6bba_312f0dc3,2873,26,50.0,192.64,136.0,cont,5.081367464816931,27000386,1,1,0,0
6bba_312f0dc3,2891,26,60.16000000000001,201.28000000000006,61.44000000000001,cont,1.473831104638521,27000389,1,1,0,0
6bba_312f0dc3,2957,27,49.011098022460935,191.75679443359377,139.74680419921876,cont,6.2109510926363525,28000397,1,0,1,0
6bba_312f0dc3,2976,27,59.68000000000001,199.36000000000007,64.64000000000001,cont,0.636867333123604,28000401,1,0,1,0
6bba_312f0dc3,3209,30,56.99999999999999,191.36,74.72,cont,1.635528393211212,31000429,2,2,0,0
6bba_312f0dc3,3284,31,55.99999999999999,188.64000000000004,77.91999999999999,cont,0.7977409510612019,32000438,2,0,2,0
6bba_312f0dc3,4044,42,37.67999999999999,187.35999999999999,216.64,cont,1.2199410846430223,43000537,1,1,0,0
6bba_312f0dc3,4108,43,36.160000000000004,184.0,218.72000000000003,cont,2.050979186876345,44000549,1,0,1,0
6bba_312f0dc3,4110,43,37.0,191.99999999999994,224.00000000000003,cont,2.3688242072808863,44000549,2,0,0,2
6bba_32db13fc,802,1,0.3199999999999999,92.0,173.59999999999997,cont,2.8180517427825933,2000048,2,1,1,0
6bba_32db13fc,1526,1,58.0,167.68,204.64000000000004,cont,0.5959564266790024,2000083,1,0,1,0
6bba_32db13fc,1690,2,5.4799999999999995,89.27999999999999,153.27999999999997,cont,1.8296229973685738,3000089,3,2,1,0
6bba_32db13fc,2024,2,32.0,56.64,55.35999999999999,cont,1.841132888332616,3000110,1,0,1,0
6bba_32db13fc,2425,3,0.0,74.23999999999998,147.04,cont,1.7071783188934904,4000129,1,0,1,0
6bba_32db13fc,2850,3,33.120000000000005,180.64000000000001,178.72000000000003,cont,2.0473710276840373,4000149,1,0,1,0
6bba_32db13fc,4010,5,0.9999999999999998,15.46666666666667,183.99999999999991,cont,6.539881988507981,6000203,1,0,1,0
6bba_32db13fc,4056,5,4.0,35.36,69.28,parent,4.807638778548156,6000204,4,1,0,3
6bba_32db13fc,4057,5,4.133333333333331,37.066666666666656,83.46666666666664,parent,1.0255968790579044,6000204,7,1,0,6
6bba_32db13fc,4452,5,34.999999999999986,184.5333333333333,85.06666666666666,cont,1.6826576614259607,6000226,7,0,0,7
6bba_32db13fc,4771,6,1.44,40.8,59.84000000000001,daughter,5.415835144509106,7000281,3,1,0,2
6bba_32db13fc,4772,6,0.31999999999999995,46.08,95.99999999999997,daughter,6.6734498434093315,7000244,3,3,0,0
6bba_32db13fc,4835,6,2.76,32.00000000000001,93.28,daughter,2.4924025181539156,7000244,1,0,1,0
6bba_32db13fc,4862,6,3.9999999999999996,32.48,67.84,daughter,1.3086043376437382,7000281,3,2,1,0
6bba_32db13fc,5125,6,25.0,121.92000000000002,103.84,cont,2.040653234873584,7000260,2,2,0,0
6bba_32db13fc,5177,6,30.60000000000001,178.88,184.0,cont,2.8290405286068427,7000266,1,1,0,0
6bba_32db13fc,5251,6,34.839999999999996,183.35999999999999,85.12000000000002,cont,2.1768935722721956,7000268,2,2,0,0
6bba_32db13fc,5422,6,49.96000000000001,168.00000000000003,219.2,cont,5.433821169306164,7000278,3,2,1,0
6bba_32db13fc,5466,6,53.8,210.07999999999998,28.320000000000004,cont,3.3536866374931336,7000279,3,2,1,0
6bba_32db13fc,5727,7,10.640000000000004,26.560000000000002,114.72,cont,2.301885775836841,8000290,3,3,0,0
6bba_32db13fc,5792,7,14.68,144.16,144.00000000000003,cont,1.329126828034102,8000291,7,1,2,4
6bba_32db13fc,5915,7,25.160000000000007,174.08,192.80000000000004,cont,2.266860439021321,8000297,10,3,2,5
6bba_32db13fc,5939,7,27.32,58.56,74.88,cont,4.147858257583063,8000298,3,1,2,0
6bba_32db13fc,5958,7,28.479999999999997,180.0,90.56,cont,4.928869038633513,8000299,6,4,2,0
6bba_32db13fc,5992,7,29.720000000000006,187.99999999999997,238.40000000000003,cont,3.0502423059488226,8000300,12,3,3,6
6bba_32db13fc,6138,7,42.16000000000001,153.91999999999996,63.36000000000001,cont,1.6919519976937831,8000305,11,2,3,6
6bba_32db13fc,6239,7,50.680000000000014,173.12000000000003,221.28000000000006,cont,2.147030289143597,8000310,11,2,2,7
6bba_32db13fc,6659,8,18.400000000000002,186.24,161.11999999999998,cont,4.06964371904961,9000326,9,0,4,5
6bba_32db13fc,6671,8,21.0,166.71999999999997,209.12000000000003,cont,2.68656291802371,9000328,7,0,0,7
6bba_32db13fc,6773,8,28.839999999999996,185.92000000000004,240.0,cont,0.9170818188689746,9000332,3,0,3,0
6bba_32db13fc,6804,8,30.040000000000003,176.64,93.12000000000002,cont,5.77689214132478,9000333,7,0,3,4
6bba_32db13fc,6951,8,41.52,153.28000000000003,70.24,cont,2.490494742516036,9000337,2,0,2,0
6bba_32db13fc,6956,8,42.84,44.16000000000001,122.08000000000003,cont,1.6809916158327376,9000338,2,1,1,0
6bba_32db13fc,7042,8,49.36000000000001,212.15999999999997,93.92,cont,3.110548836218459,9000342,7,0,2,5
6bba_32db13fc,7466,9,19.84,166.72000000000003,209.92000000000002,cont,0.46928469504128906,10000358,1,1,0,0
6bba_32db13fc,8319,10,26.000000000000004,182.07999999999993,242.71999999999997,cont,1.0249100570293928,11000393,1,1,0,0
6bba_32db13fc,9051,11,24.159999999999997,179.36000000000004,243.36000000000004,cont,0.903739626496476,12000420,2,1,1,0
6bba_32db13fc,10716,13,37.160000000000004,36.960000000000015,131.04,cont,2.2928053231794463,14000482,1,0,1,0
6bba_32db13fc,12884,16,33.24,45.120000000000005,134.56,cont,2.633101783068783,17000570,3,0,0,3
6bba_32db13fc,13206,17,4.160000000000001,139.84000000000003,182.07999999999993,cont,2.0280624990369573,18000587,3,3,0,0
6bba_32db13fc,13215,17,4.360000000000001,7.360000000000001,149.92000000000002,cont,4.5834218930838135,18000588,2,2,0,0
6bba_32db13fc,14238,18,31.360000000000007,111.36,163.2,cont,2.8738620030370208,19000623,6,0,1,5
6bba_32db13fc,15116,19,59.19999999999999,201.43999999999997,234.08000000000004,cont,3.0248579512102856,20000659,2,1,1,0
6bba_32db13fc,18049,24,21.44,50.720000000000006,159.35999999999996,cont,2.3571463812203146,25000767,9,2,3,4
6bba_32db13fc,18052,24,19.733333333333327,95.46666666666664,173.0666666666666,cont,1.892154168093439,25000766,8,0,3,5
6bba_32db13fc,18189,24,31.4,131.51999999999998,95.84000000000002,cont,4.517529226524168,25000769,6,0,1,5
6bba_32db13fc,18612,25,15.480000000000004,93.44,172.16,cont,4.948678643082408,26000787,9,1,2,6
6bba_32db13fc,19839,27,17.800000000000004,46.72,171.35999999999996,cont,0.43146733653893765,28000837,1,0,1,0
6bba_32db13fc,24199,34,20.319999999999997,69.28,58.56,cont,3.3011506043348007,35000978,1,1,0,0
6bba_32db13fc,24244,34,23.68,151.2,6.72,cont,2.6749889018461337,35000980,2,2,0,0
6bba_32db13fc,24408,34,45.51999999999998,123.35999999999999,104.64,cont,0.9905837925183568,35000985,1,1,0,0
6bba_32db13fc,24750,35,19.999999999999996,144.53333333333333,16.000000000000004,cont,2.518021848153378,36000992,9,0,0,9
6bba_32db13fc,26293,38,6.32,190.56,121.76000000000002,cont,3.2573448255442763,39001036,6,0,3,3
6bba_32db13fc,26365,38,13.8,140.64,30.720000000000006,cont,1.5505770095354787,39001037,3,0,3,0
6bba_32db13fc,26663,38,43.88,198.56,2.5599999999999996,cont,6.96547108959616,39001046,6,0,2,4
6bba_32db13fc,28076,41,15.840000000000002,147.2,17.919999999999998,cont,2.542748574377733,42001091,8,0,2,6
6bba_32db13fc,30327,45,29.4,208.53333333333327,38.93333333333332,cont,4.804219969348705,46001154,6,0,0,6
6bba_32db13fc,30578,46,2.3600000000000003,132.64000000000001,70.56,cont,1.787352266622334,47001160,1,1,0,0
6bba_32db13fc,31024,47,1.0399999999999998,132.0,74.88,cont,1.4397536681668837,48001173,3,2,1,0
6bba_32db13fc,31155,47,9.399999999999997,15.466666666666672,13.066666666666663,cont,0.6776247835064894,48001177,4,0,0,4
6bba_32db13fc,31179,47,9.52,142.56,38.08,cont,1.5504918574439526,48001176,3,1,2,0
6bba_32db13fc,31715,48,8.16,145.44,38.56000000000001,cont,6.856806837005105,49001193,4,0,1,3
6bba_32db13fc,31921,48,28.519999999999996,199.52,33.440000000000005,cont,6.067019706989576,49001198,6,0,2,4
6bba_32db13fc,31953,48,33.00000000000001,208.0,34.720000000000006,cont,5.646120902885438,49001199,2,0,2,0
6bba_32db13fc,31976,48,37.160000000000004,202.72000000000003,21.439999999999998,cont,3.525763215248576,49001199,2,0,2,0
6bba_32db13fc,32356,49,18.68,104.64,10.719999999999999,cont,5.017909487525658,50001215,1,0,1,0
6bba_32db13fc,33252,51,5.52,61.92,87.20000000000002,cont,2.243324123928599,52001242,3,3,0,0
6bba_32db13fc,33827,52,14.920000000000003,102.56,20.640000000000004,cont,3.6604364920730386,53001262,8,0,2,6
6bba_32db13fc,34441,53,24.279999999999998,202.56,56.16000000000001,cont,2.2367226303232206,54001274,4,0,0,4
6bba_32db13fc,34767,54,7.133333333333333,10.933333333333334,36.53333333333334,cont,6.128498150716318,55001282,2,0,0,2
6bba_32db13fc,35422,55,21.84,200.64000000000004,65.28,cont,4.907015704580536,56001300,2,2,0,0
6bba_32db13fc,37357,59,17.160000000000004,196.0,78.88000000000001,cont,3.086002926521615,60001352,3,3,0,0
6bba_32db13fc,37587,59,55.99999999999998,203.99999999999994,21.06666666666667,cont,6.559871828227853,60001357,8,3,0,5
6bba_32db13fc,37835,60,16.16,193.92000000000002,79.84,cont,3.1771139616009973,61001361,7,0,3,4
6bba_32db13fc,37940,60,28.04,203.35999999999999,60.64,cont,3.3214459388946893,61001366,3,0,1,2
6bba_32db13fc,38041,60,48.640000000000015,204.00000000000003,29.439999999999998,cont,2.889440094983808,61001368,2,0,2,0
6bba_32db13fc,38257,61,12.0,196.16000000000003,105.12,cont,6.9291531995258975,62001373,5,0,0,5
6bba_32db13fc,38932,62,42.24,212.32,36.32,cont,5.9291126285895395,63001391,1,1,0,0
6bba_32db13fc,40276,65,41.68,207.84000000000003,47.360000000000014,cont,2.518540313852464,66001426,2,2,0,0
6bba_32db13fc,40457,66,7.32,193.27999999999997,111.84,cont,4.6579428130882015,67001429,5,0,2,3
6bba_32db13fc,40587,66,21.32,206.72,78.72,cont,6.415602076188955,67001434,1,0,1,0
6bba_32db13fc,44890,76,41.31999999999999,191.36,21.439999999999998,cont,1.2822063503976433,77001547,1,0,1,0
6bba_32db13fc,45720,78,38.67999999999999,189.28000000000003,28.640000000000008,cont,1.3816323000349997,79001572,2,2,0,0
6bba_32db13fc,46112,79,37.480000000000004,189.28000000000003,32.0,cont,1.163663396347933,80001580,2,0,2,0
6bba_32db13fc,49392,88,13.84,201.28000000000006,186.07999999999998,cont,6.955474574930153,89001647,7,0,0,7
6bba_32db13fc,49897,89,46.68,188.64000000000004,12.64,cont,1.5616917909754071,90001664,2,0,2,0
6bba_337b1b3a,5836,20,20.16,17.919999999999998,120.0,cont,1.5888478530054428,21000068,2,0,2,0
6bba_337b1b3a,7853,26,16.48,28.0,164.64,cont,0.7935925040598635,27000142,1,0,1,0
6bba_337b1b3a,8540,28,22.48,25.92,100.64000000000001,cont,0.8581784852232062,29000169,2,1,1,0
6bba_337b1b3a,10331,34,6.48,22.72,216.0,cont,1.1777607195436592,35000233,3,2,1,0
6bba_337b1b3a,10630,35,7.160000000000001,28.640000000000008,216.0,cont,1.0732383821872913,36000246,2,0,2,0
6bba_337b1b3a,10662,35,17.16,11.36,124.64000000000001,cont,0.8234758648557925,36000248,2,2,0,0
6bba_337b1b3a,11565,38,17.84,20.160000000000004,128.0,parent,2.4210318693689254,39000297,2,1,1,0
6bba_337b1b3a,11864,39,17.866666666666664,44.00000000000001,135.99999999999994,daughter,0.6140216359741948,40000324,6,0,0,6
6bba_337b1b3a,11866,39,18.04,18.56,128.0,daughter,1.032482596705629,40000311,1,0,1,0
6bba_337b1b3a,12426,41,5.840000000000002,22.08,252.0,cont,1.2805577398149612,42000339,1,1,0,0
6bba_337b1b3a,12727,42,6.32,25.28,251.36,cont,2.82179741033973,43000354,1,0,1,0
6bba_337b1b3a,16039,53,31.000000000000007,190.72000000000003,235.35999999999999,cont,1.1919128533579928,54000550,1,0,1,0
6bba_337b1b3a,17094,57,3.866666666666668,103.99999999999999,236.5333333333333,cont,1.2522930356838382,58000623,3,0,0,3
6bba_337b1b3a,18276,61,20.133333333333333,132.53333333333333,251.46666666666664,cont,6.475239297379931,62000708,4,0,0,4
6bba_337b1b3a,18282,61,22.000000000000004,72.16,233.44000000000003,cont,1.0945554177381724,62000709,1,1,0,0
6bba_337b1b3a,18531,62,12.159999999999997,148.64000000000001,232.63999999999993,cont,0.6278895703067525,63000726,2,1,1,0
6bba_337b1b3a,18546,62,17.32,116.0,241.43999999999997,cont,2.72738714248638,63000729,2,1,1,0
6bba_337b1b3a,18826,63,18.16,112.64,235.35999999999999,cont,2.321987968315087,64000751,3,2,1,0
6bba_337b1b3a,19072,64,3.6799999999999997,109.92,243.35999999999993,cont,0.6568699452707449,65000771,1,0,1,0
6bba_337b1b3a,19083,64,9.159999999999998,119.19999999999999,248.00000000000006,cont,1.5052974166257065,65000772,1,0,1,0
6bba_337b1b3a,19093,64,13.68,142.56,244.64000000000004,cont,0.7920937444519921,65000776,2,1,1,0
6bba_337b1b3a,19109,64,18.84,109.28,227.99999999999997,cont,1.8884289932375022,65000777,2,0,2,0
6bba_337b1b3a,19275,64,51.52,233.27999999999997,196.64000000000001,cont,0.9487829375046652,65000792,1,0,1,0
6bba_337b1b3a,19654,66,4.0,107.36,233.27999999999997,cont,1.0785153278929287,67000823,2,1,1,0
6bba_337b1b3a,19967,67,9.680000000000001,117.27999999999997,240.0,cont,0.9610882503703878,68000852,1,1,0,0
6bba_337b1b3a,19979,67,14.84,136.0,229.92,cont,0.8537051598766408,68000853,2,2,0,0
6bba_337b1b3a,20257,68,9.32,117.28,237.92000000000004,cont,1.3838284756428532,69000887,2,1,1,0
6bba_337b1b3a,20275,68,15.0,133.28,227.36000000000004,cont,1.7933993559717893,69000888,2,0,2,0
6bba_337b1b3a,20547,69,4.4799999999999995,104.64,222.72,cont,0.9060741208642945,70000910,1,0,1,0
6bba_337b1b3a,20627,69,27.32,50.56,192.0,cont,0.5675880988886244,70000920,1,1,0,0
6bba_337b1b3a,20906,70,21.200000000000003,106.56,199.2,cont,3.1424323779518275,71000948,1,0,1,0
6bba_337b1b3a,20936,70,28.000000000000004,47.360000000000014,188.8,cont,0.9056368615510284,71000950,1,0,1,0
6bba_337b1b3a,21164,71,10.639999999999999,114.07999999999998,228.15999999999997,cont,2.6172591221543353,72000971,3,3,0,0
6bba_337b1b3a,21190,71,17.040000000000003,92.63999999999999,232.64000000000004,cont,2.006924341498706,72000976,3,3,0,0
6bba_337b1b3a,21215,71,21.400000000000002,107.2,199.20000000000002,cont,4.607670235596292,72000978,3,3,0,0
6bba_337b1b3a,21316,71,42.480000000000004,128.0,98.08000000000001,cont,1.219616358737447,72000987,2,2,0,0
6bba_337b1b3a,21348,71,48.31999999999999,208.64000000000004,206.72,cont,2.7650796778754683,72000990,2,2,0,0
6bba_337b1b3a,21460,72,8.48,41.11999999999999,228.8,cont,3.2818563725428267,73000997,8,2,1,5
6bba_337b1b3a,21461,72,7.866666666666665,94.93333333333331,204.5333333333333,cont,1.9043262150517553,73001001,3,0,0,3
6bba_337b1b3a,21469,72,10.840000000000002,80.64,204.64000000000001,cont,6.924254472504606,73001001,2,0,2,0
6bba_337b1b3a,21471,72,11.840000000000003,110.56,220.64000000000004,cont,0.7504967105191218,73001004,3,0,3,0
6bba_337b1b3a,21483,72,14.520000000000003,72.00000000000001,233.27999999999997,cont,1.51159942197,73001007,3,1,2,0
6bba_337b1b3a,21508,72,21.000000000000004,83.99999999999997,215.99999999999994,cont,1.7235727791422062,73001011,4,0,0,4
6bba_337b1b3a,21532,72,25.040000000000003,91.67999999999999,183.20000000000002,cont,3.043569039795222,73001012,6,0,2,4
6bba_337b1b3a,21597,72,37.68,247.99999999999997,219.20000000000002,cont,6.824013321535929,73001020,2,0,2,0
6bba_337b1b3a,21609,72,41.76,236.8,226.72,cont,3.045520532273586,73001020,5,0,3,2
6bba_337b1b3a,21619,72,42.480000000000004,39.52000000000001,127.36,cont,2.373223348633665,73001021,6,0,3,3
6bba_337b1b3a,21636,72,43.519999999999996,128.0,96.0,cont,2.538539196073207,73001024,2,0,2,0
6bba_337b1b3a,21685,72,52.519999999999996,93.92000000000002,128.64000000000004,cont,1.2363890215057574,73001028,1,0,1,0
6bba_337b1b3a,21696,72,56.640000000000015,55.519999999999996,118.56000000000003,cont,2.463684359856197,73001029,2,0,1,1
6bba_337b1b3a,21777,73,9.000000000000002,38.559999999999995,224.0,cont,1.9293068703552536,74001038,2,0,2,0
6bba_337b1b3a,22383,75,7.840000000000002,51.19999999999999,245.43999999999997,cont,1.1776486105795774,76001103,1,0,1,0
6bba_337b1b3a,22742,76,23.866666666666667,115.46666666666664,189.0666666666666,cont,1.8898267943121194,77001145,3,0,0,3
6bba_337b1b3a,23327,78,20.640000000000004,49.440000000000005,194.72,cont,3.3476972029292043,79001233,4,0,1,3
6bba_337b1b3a,23607,79,11.200000000000003,221.27999999999997,240.64,cont,3.1396581223120354,80001271,2,2,0,0
6bba_337b1b3a,23665,79,25.32,113.6,173.91999999999996,cont,2.8589380130565987,80001276,3,3,0,0
6bba_337b1b3a,23788,79,47.52000000000001,125.28,84.64000000000001,cont,1.1151100001793666,80001296,1,1,0,0
6bba_337b1b3a,23939,80,16.12,20.160000000000004,198.72000000000003,cont,3.7834592719494147,81001306,6,0,2,4
6bba_337b1b3a,23949,80,17.999999999999996,36.31999999999999,178.08,cont,2.961693455609479,81001319,8,0,4,4
6bba_337b1b3a,24014,80,29.866666666666667,100.53333333333332,141.0666666666666,cont,0.7679450392660395,81001329,6,0,0,6
6bba_337b1b3a,24044,80,35.99999999999999,48.53333333333332,115.99999999999997,cont,1.6857063512895163,81001332,2,0,0,2
6bba_337b1b3a,24061,80,39.16,110.08,170.72000000000003,cont,1.3210560406356648,81001333,6,0,3,3
6bba_337b1b3a,24079,80,42.64,226.24,127.52,cont,3.3931385268656555,81001336,6,0,3,3
6bba_337b1b3a,24135,80,52.32000000000001,119.36000000000001,52.8,cont,1.5052097029982308,81001340,4,0,2,2
6bba_337b1b3a,24286,81,21.840000000000003,139.35999999999999,230.07999999999998,cont,6.257115199914423,82001354,1,1,0,0
6bba_337b1b3a,24324,81,28.36,166.72000000000003,235.99999999999994,cont,1.1281760722511485,82001365,1,1,0,0
6bba_337b1b3a,25174,84,4.0,185.28,243.36000000000004,cont,0.845156235556459,85001477,1,1,0,0
6bba_337b1b3a,26123,87,5.84,39.36,240.16000000000005,cont,2.460574119997184,88001603,2,2,0,0
6bba_337b1b3a,26179,87,21.160000000000004,17.44,152.8,cont,2.573304356756112,88001613,1,1,0,0
6bba_337b1b3a,26442,88,5.133333333333335,72.53333333333332,251.46666666666664,cont,5.3369974392598785,89001638,5,1,0,4
6bba_337b1b3a,26444,88,6.800000000000001,35.2,234.08000000000004,cont,1.5203957132930788,89001639,3,1,2,0
6bba_337b1b3a,26498,88,20.640000000000008,173.92000000000002,244.8,cont,1.4397536681668923,89001647,3,2,1,0
6bba_337b1b3a,26510,88,24.320000000000007,117.44,182.24,cont,2.727919594214613,89001651,2,0,0,2
6bba_337b1b3a,26774,89,8.16,32.0,226.56000000000003,cont,2.104798461254667,90001677,1,0,1,0
6bba_337b1b3a,26776,89,8.48,184.64,241.27999999999997,cont,6.370932648560944,90001680,1,1,0,0
6bba_337b1b3a,26848,89,25.52,240.00000000000006,235.99999999999994,cont,2.6653963119956314,90001692,4,2,2,0
6bba_337b1b3a,27097,90,9.36,184.0,235.99999999999994,cont,6.539408660192426,91001714,2,1,1,0
6bba_337b1b3a,27167,90,26.52,236.64000000000004,228.64000000000004,cont,1.334183387881905,91001732,3,1,2,0
6bba_337b1b3a,27173,90,27.99999999999999,251.46666666666664,232.0,cont,6.566744989486359,91001732,4,1,0,3
6bba_337b1b3a,27407,91,7.520000000000001,181.28000000000006,216.64,cont,1.057374401524817,92001754,1,0,1,0
6bba_337b1b3a,27413,91,9.68,183.36,230.72000000000008,cont,6.208138056213019,92001754,2,1,1,0
6bba_337b1b3a,27431,91,14.360000000000003,119.36000000000001,227.36000000000004,cont,2.993398204800042,92001756,1,0,1,0
6bba_337b1b3a,27738,92,10.159999999999998,183.36,225.92,cont,6.678651154986313,93001797,2,1,1,0
6bba_372c8cb8,300,4,25.0,243.36000000000004,181.28,cont,2.384379505238206,5000060,1,1,0,0
6bba_372c8cb8,375,5,24.160000000000004,242.72000000000003,178.72000000000003,cont,0.7541820900817149,6000073,1,0,1,0
6bba_372c8cb8,3115,43,34.0,48.0,8.0,cont,0.9084026158592896,44000582,1,1,0,0
6bba_372c8cb8,3596,49,41.36,73.92000000000002,102.72000000000001,cont,4.6440327975801425,50000670,1,1,0,0
6bba_372c8cb8,3615,49,57.52000000000001,205.92,200.0,cont,0.8949310378459218,50000672,1,1,0,0
6bba_372c8cb8,3668,50,39.96,77.44,100.63999999999999,cont,2.9794719540549472,51000680,1,0,1,0
6bba_372c8cb8,3696,50,57.00000000000001,208.0,200.0,cont,1.6750116604071745,51000682,1,0,1,0
6bba_372c8cb8,4330,59,14.159999999999998,235.35999999999999,107.35999999999999,cont,1.7420879175575539,60000787,1,1,0,0
6bba_372c8cb8,4405,60,13.480000000000002,235.99999999999994,104.64000000000003,cont,1.365386849944007,61000800,1,0,1,0
6bba_372c8cb8,4627,62,50.199999999999996,174.08,240.00000000000006,cont,1.3526599951577005,63000838,1,1,0,0
6bba_372c8cb8,4707,63,50.839999999999996,173.28000000000003,240.00000000000006,cont,1.907558976807798,64000852,1,0,1,0
6bba_372c8cb8,5213,70,23.36,235.35999999999999,152.8,cont,1.5714682624857579,71000934,1,1,0,0
6bba_372c8cb8,5221,70,29.479999999999997,228.80000000000007,176.80000000000004,cont,2.1033552065687564,71000936,1,1,0,0
6bba_372c8cb8,5294,71,23.159999999999997,232.8,149.44,cont,2.642911510058559,72000945,1,0,1,0
6bba_372c8cb8,5305,71,29.519999999999996,227.36000000000004,173.44000000000005,cont,2.117556448480158,72000946,1,0,1,0
6bba_372c8cb8,6319,83,55.519999999999996,155.2,226.72,cont,1.50503426040738,84001104,1,1,0,0
6bba_372c8cb8,6385,84,54.64,158.08,224.0,cont,1.1170028256454918,85001115,1,0,1,0
6bba_3a1849c2,680,10,35.199999999999996,36.64000000000001,27.999999999999996,cont,3.0043169460794283,11000104,1,1,0,0
6bba_3a1849c2,706,10,61.00000000000001,184.0,188.0,cont,2.4374999999999907,11000108,1,1,0,0
6bba_3a1849c2,750,11,34.160000000000004,34.72,27.999999999999996,cont,1.8388366736880108,12000114,1,0,1,0
6bba_3a1849c2,779,11,61.00000000000001,185.28,189.28000000000003,cont,1.9867577230251134,12000118,1,0,1,0
6bba_3a1849c2,795,11,61.99999999999999,183.46666666666667,207.99999999999997,cont,6.580971358984751,12000118,1,0,0,1
6bba_3a1849c2,1189,16,61.680000000000014,189.92000000000002,192.64,cont,1.4633122744308504,17000177,1,1,0,0
6bba_3a1849c2,1267,17,61.839999999999996,190.56,193.27999999999997,cont,1.065707658084524,18000190,1,0,1,0
6bba_3a1849c2,1342,18,62.000000000000014,192.80000000000004,194.71999999999997,cont,1.5199614509914559,19000201,1,1,0,0
6bba_3a1849c2,1414,19,62.000000000000014,192.64,195.36000000000004,cont,0.7151846352516,20000212,2,0,1,1
6bba_3a1849c2,1416,19,62.000000000000014,208.48,192.15999999999997,cont,6.9777621099103095,20000212,2,0,0,2
6bba_3a1849c2,4839,62,53.52,46.72000000000001,8.0,cont,2.4076885310396783,63000683,1,1,0,0
6bba_3a1849c2,4857,62,60.000000000000014,177.28,177.28000000000003,cont,4.158000007515628,63000686,2,2,0,0
6bba_3a1849c2,4918,63,52.84,45.919999999999995,8.640000000000002,cont,3.5802035993082764,64000697,1,0,1,0
6bba_3a1849c2,4937,63,59.84000000000002,176.64000000000004,178.56000000000003,cont,3.981515297283164,64000702,2,0,2,0
6bba_3a1849c2,5012,64,58.16000000000001,228.64000000000007,156.00000000000006,cont,0.5040120286263192,65000715,1,1,0,0
6bba_3a1849c2,5112,65,57.639999999999986,227.99999999999997,157.28000000000003,cont,2.5040295650211655,66000729,1,0,1,0
6bba_3a1849c2,5387,68,58.999999999999986,180.0,185.91999999999996,cont,3.521191648433819,69000780,1,1,0,0
6bba_3a1849c2,5472,69,58.999999999999986,136.0,158.72,cont,4.626943172873437,70000801,1,1,0,0
6bba_3a1849c2,5473,69,58.680000000000014,180.0,188.00000000000006,cont,3.4327139408928113,70000798,1,0,1,0
6bba_3a1849c2,5482,69,61.160000000000004,115.36000000000001,162.72,cont,6.442587067708124,70000801,1,1,0,0
6bba_3a1849c2,5551,70,57.13333333333333,15.466666666666672,39.46666666666666,cont,3.839438638980964,71000815,1,0,0,1
6bba_3a1849c2,5568,70,58.999999999999986,136.64000000000001,160.0,cont,5.562652947335484,71000819,1,0,1,0
6bba_3a1849c2,5570,70,60.000000000000014,7.359999999999999,46.72,cont,6.078129980717769,71000815,1,0,1,0
6bba_3a1849c2,5576,70,61.00000000000001,115.35999999999999,163.36,cont,5.69531784121836,71000819,1,0,1,0
6bba_3a1849c2,5829,73,55.36000000000001,29.279999999999994,80.64,cont,0.9336328574445139,74000862,1,1,0,0
6bba_3a1849c2,5830,73,56.32000000000001,178.56000000000003,199.36,cont,2.8806992423715356,74000865,1,1,0,0
6bba_3a1849c2,5919,74,55.16000000000001,20.000000000000007,53.92,cont,2.21680019961203,75000880,1,1,0,0
6bba_3a1849c2,5920,74,54.68000000000001,29.440000000000005,81.28,cont,0.8276340525860318,75000879,3,2,1,0
6bba_3a1849c2,5926,74,56.16000000000001,179.35999999999999,202.08000000000004,cont,2.102539015690298,75000881,1,0,1,0
6bba_3a1849c2,5936,74,59.16000000000001,69.44000000000001,99.36,cont,2.3057255501251515,75000885,2,1,1,0
6bba_3a1849c2,6008,75,53.84000000000002,28.0,83.36000000000001,cont,0.8655318090630699,76000894,2,0,2,0
6bba_3a1849c2,6010,75,54.680000000000014,20.000000000000007,56.00000000000001,cont,3.8565601577052906,76000897,1,0,1,0
6bba_3a1849c2,6028,75,59.160000000000004,68.64000000000001,100.00000000000001,cont,1.9726857485418143,76000900,1,0,1,0
6bba_3a1849c2,6303,78,58.680000000000014,68.0,103.36,cont,2.282532173924372,79000949,1,1,0,0
6bba_3a1849c2,6391,79,57.16000000000001,128.64000000000001,155.35999999999999,cont,4.5664523976496225,80000961,1,1,0,0
6bba_3a1849c2,6395,79,58.31999999999999,68.0,104.0,cont,1.245432906663398,80000964,1,0,1,0
6bba_3a1849c2,6408,79,60.12454711914063,119.72855346679685,172.79152099609377,cont,6.078110684563335,80000961,1,1,0,0
6bba_3a1849c2,6480,80,55.86666666666666,111.99999999999999,159.99999999999997,cont,5.607885228507384,81000977,3,0,0,3
6bba_3a1849c2,6485,80,57.52000000000001,129.28,154.56,cont,5.723922660641044,81000977,1,0,1,0
6bba_3a1849c2,6499,80,60.07092651367188,120.96451293945313,175.28663085937498,cont,6.0761324593013715,81000977,1,0,1,0
6bba_3a1849c2,6575,81,58.0,68.0,107.84000000000002,cont,1.7094196749189492,82000995,1,1,0,0
6bba_3a1849c2,6620,82,13.839999999999998,111.36,148.00000000000006,cont,2.2064723683971175,83000998,1,1,0,0
6bba_3a1849c2,6624,82,16.839999999999996,208.64000000000001,221.27999999999997,cont,1.9859600952687908,83001000,1,1,0,0
6bba_3a1849c2,6629,82,26.840000000000003,8.640000000000004,15.2,cont,3.513985006370963,83001001,1,1,0,0
6bba_3a1849c2,6650,82,53.8,179.36,216.0,cont,3.5118802834094445,83001005,1,1,0,0
6bba_3a1849c2,6663,82,58.12000000000001,68.0,112.47999999999999,cont,2.0514941201475376,83001009,2,1,1,0
6bba_3a1849c2,6717,83,12.68,109.28,150.07999999999998,cont,2.5358331964070517,84001012,1,0,1,0
6bba_3a1849c2,6719,83,16.16,206.72,222.72000000000003,cont,3.2865198158538376,84001013,1,0,1,0
6bba_3a1849c2,6764,83,53.52,178.72000000000003,216.0,cont,2.364807735313797,84001024,1,0,1,0
6bba_3a1849c2,6790,83,58.919999999999995,68.00000000000001,121.12000000000002,cont,1.3373463696813965,84001026,1,0,1,0
6bba_3a1849c2,6877,84,50.86666666666666,11.999999999999996,103.46666666666663,cont,1.9213902392249391,85001039,4,1,0,3
6bba_3a1849c2,6883,84,52.67999999999999,13.279999999999998,67.2,cont,3.5096990611731997,85001040,1,1,0,0
6bba_3a1849c2,6885,84,53.16,107.35999999999999,167.2,cont,5.924835572612975,85001045,1,1,0,0
6bba_3a1849c2,6932,84,60.16,76.0,141.28000000000003,cont,5.843338220786134,85001046,1,0,1,0
6bba_3a1849c2,7008,85,50.24000000000001,12.0,104.96000000000002,cont,1.723725978802897,86001059,1,0,1,0
6bba_3a1849c2,7011,85,50.0,18.080000000000002,90.56000000000002,cont,6.329161180598889,86001059,4,0,0,4
6bba_3a1849c2,7019,85,52.480000000000004,105.28000000000002,168.64000000000004,cont,6.283288972544543,86001063,2,1,1,0
6bba_3a1849c2,7022,85,53.16,10.559999999999999,65.28,cont,5.31491768515751,86001060,1,0,1,0
6bba_3a1849c2,7028,85,54.13333333333333,119.99999999999997,184.5333333333333,cont,3.9631533154028715,86001063,2,0,0,2
6bba_3a1849c2,7168,86,52.16,104.64,170.07999999999998,cont,5.573560492405196,87001081,1,0,1,0
6bba_3a1849c2,7170,86,51.84000000000001,176.00000000000003,216.0,cont,4.551624709925005,87001082,1,1,0,0
6bba_3a1849c2,7178,86,53.760000000000005,120.00000000000006,186.72000000000008,cont,4.445050794141768,87001081,1,1,0,0
6bba_3a1849c2,7313,87,50.84000000000001,175.36,217.28000000000003,cont,6.196046370468807,88001098,1,0,1,0
6bba_3a1849c2,7321,87,53.16,119.36000000000001,187.36,cont,4.844244652161998,88001097,1,0,1,0
6bba_3a1849c2,7444,88,45.84,208.0,193.27999999999997,cont,3.535262101315818,89001107,1,1,0,0
6bba_3a1849c2,7449,88,48.84,12.0,108.0,cont,2.4887446634799573,89001110,1,1,0,0
6bba_3a1849c2,7510,88,60.120000000000005,69.44000000000001,134.88000000000002,cont,2.253072901616804,89001114,1,1,0,0
6bba_3a1849c2,7563,89,7.840000000000001,107.36,161.44000000000005,cont,1.9940544438404797,90001116,1,1,0,0
6bba_3a1849c2,7581,89,45.160000000000004,208.0,195.36,cont,3.02865301124113,90001121,1,0,1,0
6bba_3a1849c2,7585,89,48.16000000000001,12.0,108.0,cont,2.8877946278258904,90001122,1,0,1,0
6bba_3a1849c2,7588,89,48.16,171.35999999999996,224.0,cont,3.2239770664506984,90001123,1,0,1,0
6bba_3a1849c2,7590,89,50.67999999999999,14.559999999999999,87.84,cont,4.103277843687904,90001126,2,0,0,2
6bba_3a1849c2,7597,89,52.32000000000001,118.56000000000003,189.28000000000003,cont,2.0194507297777857,90001127,1,1,0,0
6bba_3a1849c2,7610,89,55.16000000000001,73.92000000000002,137.92000000000002,cont,4.680564202102118,90001128,7,0,1,6
6bba_3a1849c2,7648,89,61.04,67.2,130.4,cont,6.100051741583834,90001128,3,0,1,2
6bba_3a1849c2,7662,90,6.999999999999999,106.72,162.71999999999997,cont,1.7099602517310193,91001132,1,0,1,0
6bba_3a1849c2,7688,90,50.13333333333333,100.53333333333332,195.99999999999997,cont,4.76489668842766,91001142,2,0,0,2
6bba_3a1849c2,7695,90,52.16,119.35999999999999,190.72000000000003,cont,3.9189461673388566,91001142,1,0,1,0
6bba_3a1849c2,8040,94,46.16,10.720000000000002,114.07999999999998,cont,3.6257533699908553,95001207,1,1,0,0
6bba_3a1849c2,8135,95,45.84,9.28,113.92,cont,4.740685110034205,96001222,1,0,1,0
6bba_3abfe10a,6265,20,3.999999999999999,11.999999999999996,75.46666666666667,cont,3.0387693106932465,21000003,5,0,0,5
6bba_3abfe10a,8324,25,2.3600000000000003,26.400000000000006,109.12000000000002,cont,4.320177947723916,26000010,5,1,1,3
6bba_3abfe10a,10297,29,5.520000000000001,40.000000000000014,143.36,cont,2.4528889217614354,30000023,1,1,0,0
6bba_3abfe10a,13734,35,12.13333333333333,80.53333333333333,169.06666666666666,cont,1.8927355643794104,36000057,7,1,0,6
6bba_3abfe10a,17107,40,21.800000000000004,102.56,229.27999999999997,cont,2.2129254534439196,41000094,1,1,0,0
6bba_3abfe10a,17777,41,16.759999999999998,116.00000000000003,176.95999999999998,cont,2.97765454048988,42000106,2,1,1,0
6bba_3abfe10a,21701,46,32.35999999999999,157.28,251.36000000000007,cont,6.232484080404874,47000199,3,3,0,0
6bba_3abfe10a,22458,47,31.64,163.52000000000004,252.0,cont,2.816458325628109,48000220,3,0,3,0
6bba_3abfe10a,26099,52,17.64,5.28,7.840000000000001,cont,0.6867427647962527,53000326,2,1,1,0
6bba_3abfe10a,26170,52,23.96,86.56,15.36,cont,6.385092134221713,53000329,4,3,1,0
6bba_3abfe10a,26349,52,35.96000000000001,18.720000000000002,208.64000000000004,cont,1.1428268132136299,53000335,2,0,2,0
6bba_3abfe10a,26365,52,35.84,33.92,216.0,cont,6.401963810035168,53000335,3,2,1,0
6bba_3abfe10a,26735,52,59.83999999999999,15.200000000000003,110.72000000000001,cont,1.3131367503044014,53000344,4,4,0,0
6bba_3abfe10a,26805,52,61.52000000000001,146.24,220.79999999999998,cont,3.5663104621583424,53000347,5,5,0,0
6bba_3abfe10a,26952,53,20.520000000000003,146.08,234.07999999999998,cont,1.8500751673918494,54000356,1,0,1,0
6bba_3abfe10a,27320,53,45.51999999999999,98.56,138.72,cont,0.8204238614886967,54000368,3,0,0,3
6bba_3abfe10a,27481,53,56.68,121.92000000000002,234.72,cont,1.9142610356218441,54000373,5,0,3,2
6bba_3abfe10a,27555,53,60.67999999999999,17.92,134.07999999999998,cont,6.635832299907817,54000374,6,0,3,3
6bba_3abfe10a,27963,54,37.000000000000014,28.0,208.64000000000004,cont,1.9351150644083044,55000392,3,3,0,0
6bba_3abfe10a,28269,54,58.0,13.280000000000001,225.28000000000006,cont,6.602741144782216,55000399,2,2,0,0
6bba_3abfe10a,28980,55,49.52000000000001,112.47999999999999,137.28000000000003,cont,1.6963163214742616,56000426,6,0,1,5
6bba_3abfe10a,29022,55,53.96000000000001,120.64000000000001,242.08,cont,2.326702669551927,56000429,4,0,4,0
6bba_3abfe10a,30122,57,18.479999999999993,208.32,223.20000000000002,cont,6.005205033968458,58000474,2,2,0,0
6bba_3abfe10a,30651,57,55.12,33.120000000000005,159.84000000000003,cont,5.991469978227393,58000493,3,2,1,0
6bba_3abfe10a,30969,58,21.32,15.36,9.920000000000002,cont,1.1978796371088343,59000509,1,0,1,0
6bba_3abfe10a,31530,58,58.0,26.080000000000002,240.64000000000004,cont,2.1224764427432508,59000535,2,1,1,0
6bba_3abfe10a,31531,58,59.599999999999994,28.159999999999997,147.52000000000004,cont,3.035098072962375,59000525,3,0,1,2
6bba_3abfe10a,31532,58,57.040000000000006,37.44,164.16000000000003,cont,5.833116260842057,59000525,6,0,2,4
6bba_3abfe10a,34750,62,59.31999999999999,174.56,252.0,cont,1.2738383188223097,63000671,1,1,0,0
6bba_3abfe10a,35003,63,22.36,94.72,52.0,cont,5.590802994651125,64000679,1,0,1,0
6bba_3abfe10a,35345,63,46.16,48.0,210.71999999999997,cont,1.5161345792837901,64000687,1,1,0,0
6bba_3abfe10a,35359,63,46.86666666666666,51.99999999999999,223.99999999999997,cont,5.0297038687127795,64000687,9,1,0,8
6bba_3abfe10a,35593,63,59.733333333333334,183.99999999999997,235.99999999999997,cont,6.805196135633754,64000702,7,0,0,7
6bba_3abfe10a,37952,66,54.0,69.28,95.36000000000001,cont,6.73356158080551,67000804,1,1,0,0
6bba_3abfe10a,38015,66,58.0,9.920000000000002,203.36,cont,0.4013453936449296,67000790,2,2,0,0
6bba_3abfe10a,38224,67,16.68,6.720000000000001,56.00000000000001,cont,2.8782690653411804,68000812,1,0,1,0
6bba_3abfe10a,38235,67,17.0,221.43999999999997,229.44,cont,5.175618622445057,68000811,5,0,1,4
6bba_3abfe10a,38574,67,41.266666666666666,108.53333333333333,63.999999999999986,cont,3.189055312270744,68000825,4,1,0,3
6bba_3abfe10a,38576,67,40.04,150.08000000000004,6.720000000000001,cont,3.3458430365156095,68000823,1,1,0,0
6bba_3abfe10a,38839,67,59.60000000000001,74.72,80.64,cont,6.885284353242068,68000841,1,0,1,0
6bba_3abfe10a,38874,67,60.31999999999999,39.199999999999996,118.56000000000003,cont,6.897545985711152,68000846,7,0,2,5
6bba_3abfe10a,39352,68,38.99999999999999,107.99999999999997,67.99999999999997,cont,4.596194077712579,69000864,4,0,0,4
6bba_3abfe10a,39364,68,39.04,119.52000000000001,76.0,cont,4.769866350328909,69000864,1,0,1,0
6bba_3abfe10a,40448,69,56.160000000000004,12.0,158.71999999999997,cont,1.6714608131810917,70000921,1,0,1,0
6bba_3abfe10a,40529,69,60.000000000000014,33.28,120.64000000000001,cont,6.0279615387376,70000918,1,1,0,0
6bba_3abfe10a,41024,70,39.68,155.83999999999997,150.72,cont,6.544313013028956,71000934,1,0,1,0
6bba_3abfe10a,42055,71,54.0,8.64,1.2799999999999998,cont,2.4173752553130843,72000975,1,1,0,0
6bba_3abfe10a,42252,71,62.20380554199218,71.47849487304687,103.81837890625,cont,5.306516024000642,72000987,1,1,0,0
6bba_3abfe10a,42863,72,53.84,8.0,2.72,cont,3.776333580670534,73001009,1,0,1,0
6bba_3abfe10a,42882,72,55.96000000000001,74.56,100.64,cont,5.011037348942815,73001018,2,0,2,0
6bba_3abfe10a,43434,73,39.80000000000001,166.08,129.28,cont,6.9139501191431805,74001032,1,0,1,0
6bba_3abfe10a,44913,75,33.86666666666666,7.999999999999998,60.53333333333332,cont,2.219016024740603,76001095,2,0,2,0
6bba_3abfe10a,44961,75,37.000000000000014,106.72,68.64,cont,1.0785153278929345,76001094,2,0,2,0
6bba_3abfe10a,45280,75,56.879999999999995,178.72,129.28,cont,1.8270955434787823,76001110,1,0,1,0
6bba_3abfe10a,45569,76,23.840000000000003,85.28,35.84,cont,6.2079679143017525,77001124,4,0,0,4
6bba_3abfe10a,45680,76,32.480000000000004,119.36000000000001,69.28,cont,5.3497803693235895,77001128,4,2,2,0
6bba_3abfe10a,47234,78,33.36,163.36,128.0,cont,6.656096007608363,79001189,1,1,0,0
6bba_3abfe10a,47538,78,56.800000000000004,16.64,95.36000000000001,cont,6.457959357451858,79001202,2,2,0,0
6bba_3abfe10a,48313,79,54.99999999999999,11.999999999999996,76.0,cont,5.5255973941737855,80001233,2,0,2,0
6bba_3abfe10a,49345,81,17.88778289794922,10.267101135253908,69.99031494140624,cont,3.629812201657223,82001284,1,1,0,0
6bba_3abfe10a,50125,82,17.679999999999996,10.719999999999999,71.36,cont,3.808502207233186,83001322,1,0,1,0
6bba_3abfe10a,50496,82,46.999999999999986,27.466666666666665,23.999999999999993,cont,5.70899707211736,83001338,6,0,0,6
6bba_3abfe10a,50733,82,60.84000000000001,52.64000000000001,235.99999999999994,cont,6.514446446168735,83001349,1,1,0,0
6bba_3abfe10a,63760,82,50.094369964599615,20.429431381225584,91.93155242919923,daughter,6.32821520587393,83001350,1,1,0,0
6bba_3abfe10a,51167,83,41.00000000000001,7.360000000000001,6.56,cont,0.6898120214957109,84001368,1,1,0,0
6bba_3abfe10a,53347,86,34.64,108.0,78.72000000000001,cont,6.900416654086913,87001479,2,0,2,0
6bba_3abfe10a,53387,86,37.32000000000001,120.64000000000001,66.72,cont,2.2223917296462488,87001479,6,0,0,6
6bba_3abfe10a,53821,87,4.160000000000001,13.280000000000001,27.36,cont,0.9998703040894853,88001503,1,1,0,0
6bba_3abfe10a,53880,87,17.733333333333327,7.4666666666666694,103.99999999999999,cont,2.223474039775507,88001507,4,0,0,4
6bba_3abfe10a,54094,87,35.32000000000001,108.64000000000001,78.72000000000001,cont,6.724221725411791,88001517,1,1,0,0
6bba_3abfe10a,55288,88,62.000000000000014,70.56,99.36000000000001,cont,6.238328226576434,89001578,1,0,1,0
6bba_3abfe10a,55396,89,17.16,107.99999999999999,31.360000000000007,cont,5.132582500554663,90001593,2,1,1,0
6bba_3abfe10a,55445,89,23.159999999999997,95.83999999999999,29.279999999999994,cont,5.902710447328408,90001593,6,0,0,6
6bba_3abfe10a,55657,89,37.2,112.0,76.64000000000001,cont,6.42760939230131,90001602,4,0,0,4
6bba_3abfe10a,55944,89,57.0,182.72,126.72000000000001,cont,1.742466821778821,90001620,1,1,0,0
6bba_3abfe10a,55986,89,60.640000000000015,22.88,4.000000000000001,cont,3.365947349929889,90001625,5,0,0,5
6bba_3abfe10a,56150,90,16.520000000000003,107.36000000000001,33.28,cont,4.354545218791507,91001633,4,3,1,0
6bba_3abfe10a,56704,90,57.733333333333334,175.4666666666667,132.53333333333333,cont,6.273987302080436,91001653,6,2,0,4
6bba_3abfe10a,56915,91,19.133333333333326,108.53333333333333,27.999999999999996,cont,1.4779660620445894,92001668,6,0,0,6
6bba_3abfe10a,56951,91,23.159999999999997,62.07999999999999,170.07999999999998,cont,1.6563047160773319,92001670,1,0,1,0
6bba_3abfe10a,57212,91,42.64000000000001,16.639999999999997,127.36,cont,1.3058773391861855,92001674,2,1,1,0
6bba_3c5691b6,1469,22,1.0,248.00000000000006,88.64,cont,1.7637801202247345,23000171,1,1,0,0
6bba_3c5691b6,1532,23,0.6799999999999999,248.00000000000006,88.00000000000001,cont,2.329424888786922,24000177,1,0,1,0
6bba_3c5691b6,2273,35,10.679999999999998,58.72,23.36,cont,4.154219616847431,36000280,1,1,0,0
6bba_3c5691b6,2332,36,10.52,57.279999999999994,24.0,cont,2.913829363998519,37000291,1,0,1,0
6bba_3c5691b6,2333,36,11.266666666666664,71.46666666666665,32.53333333333333,cont,5.307849680347647,37000291,1,0,0,1
6bba_3c5691b6,3021,45,1.0,145.27999999999997,35.36,cont,0.7138911856858968,46000379,1,1,0,0
6bba_3c5691b6,3104,46,1.0,144.64,34.720000000000006,cont,0.6251499820043278,47000389,1,0,1,0
6bba_3c5691b6,3220,47,27.999999999999996,171.99999999999994,143.36,cont,1.9351150644082946,48000403,1,1,0,0
6bba_3c5691b6,3224,47,32.160000000000004,133.92000000000002,136.0,cont,1.66393969617292,48000404,1,1,0,0
6bba_3c5691b6,3299,48,27.999999999999996,171.99999999999994,142.72,cont,1.8897569288138445,49000412,1,0,1,0
6bba_3c5691b6,3305,48,32.16000000000001,131.20000000000002,133.28,cont,0.6251499820043286,49000413,1,0,1,0
6bba_3c5691b6,3795,54,56.32,16.0,112.64,cont,1.5925829060052115,55000475,1,1,0,0
6bba_3c5691b6,3867,55,55.64,16.0,110.56,cont,2.299073522639059,56000487,1,0,1,0
6bba_3c5691b6,3989,57,26.68,129.28,116.63999999999999,cont,1.8953380305370244,58000508,1,1,0,0
6bba_3c5691b6,4061,58,25.84,174.72,134.72000000000003,cont,1.9854946644350353,59000521,1,1,0,0
6bba_3c5691b6,4136,59,25.480000000000004,173.27999999999997,133.27999999999997,cont,2.9698847831186943,60000535,1,0,1,0
6bba_3c5691b6,5517,77,51.68,19.360000000000003,100.64000000000001,cont,2.1762869755618133,78000750,1,1,0,0
6bba_3c5691b6,5598,78,51.52,18.720000000000002,100.0,cont,2.4605741199972018,79000759,1,0,1,0
6bba_3c5691b6,6070,83,50.68000000000001,23.840000000000003,96.63999999999999,cont,4.088288150314244,84000811,1,1,0,0
6bba_3c5691b6,6144,84,50.320000000000014,29.439999999999998,96.64000000000001,cont,2.691129525868263,85000821,1,0,1,0
6bba_3c5691b6,6188,85,22.16,137.28000000000003,90.72,cont,1.9587149007704014,86000828,1,1,0,0
6bba_3c5691b6,6267,86,22.000000000000004,137.28,89.27999999999999,cont,1.6550287950667282,87000838,1,0,1,0
6bba_3c5691b6,6874,92,21.12,179.2,105.28000000000002,cont,3.9158116974645245,93000898,1,1,0,0
6bba_3c5691b6,6985,93,18.0,188.53333333333325,92.5333333333333,cont,1.7537373286942088,94000908,2,0,0,2
6bba_3db54e20,2264,40,41.0,19.360000000000003,88.00000000000001,cont,0.6662499999999989,41000358,1,1,0,0
6bba_3db54e20,2501,42,40.71999999999999,29.759999999999998,94.72000000000003,cont,1.4865863790913734,43000375,1,0,1,0
6bba_3db54e20,2688,43,56.08,40.480000000000004,41.92,cont,6.5356926277939404,44000384,3,0,2,1
6bba_3db54e20,2794,44,36.48,68.0,111.84,cont,4.421075236862634,45000390,7,3,1,3
6bba_3db54e20,2841,44,56.120000000000005,49.120000000000005,44.00000000000001,cont,3.9597352342044245,45000392,1,1,0,0
6bba_3db54e20,2922,45,28.0,54.400000000000006,135.20000000000002,cont,3.0455205322735903,46000395,2,0,1,1
6bba_3db54e20,2959,45,37.2,79.36,113.27999999999999,cont,1.8673364218854573,46000397,3,0,3,0
6bba_3db54e20,8978,64,59.86666666666666,137.06666666666666,0.5333333333333338,cont,2.6214931932965335,65000510,4,0,0,4
6bba_3db54e20,10243,67,31.240000000000006,104.64000000000003,133.44,cont,4.85472659503086,68000529,4,0,3,1
6bba_3db54e20,10401,67,61.160000000000004,118.24,7.2,cont,5.865597262853973,68000535,2,0,2,0
6bba_3db54e20,10729,68,30.920000000000005,104.0,136.64000000000001,cont,5.051764079259827,69000540,4,4,0,0
6bba_3db54e20,10883,68,60.839999999999996,120.80000000000001,4.640000000000001,cont,5.38490772901449,69000546,2,2,0,0
6bba_3db54e20,11022,69,9.84,32.64,166.55999999999997,cont,1.9043031041564775,70000548,1,0,1,0
6bba_3db54e20,11234,69,29.36,127.2,140.16,cont,3.7974967083593394,70000552,5,0,3,2
6bba_3db54e20,11415,69,58.839999999999996,143.04000000000002,10.719999999999999,cont,6.968692712769018,70000558,5,1,2,2
6bba_3db54e20,11697,70,22.880000000000003,42.72,67.52,cont,5.092554000695526,71000564,3,2,1,0
6bba_3db54e20,11750,70,29.519999999999996,9.28,64.64,cont,4.043246877510705,71000566,3,2,1,0
6bba_3db54e20,12223,71,21.640000000000004,47.36,69.92,cont,4.304900732014619,72000577,2,0,2,0
6bba_3db54e20,12461,71,50.68,202.40000000000003,40.63999999999999,cont,5.8501128461338245,72000582,4,0,1,3
6bba_3db54e20,12580,72,3.8400000000000007,119.20000000000002,20.0,cont,5.312557205574355,73000585,3,3,0,0
6bba_3db54e20,12852,72,29.160000000000004,20.640000000000004,63.36,cont,3.3527022683202836,73000590,3,2,1,0
6bba_3db54e20,13178,73,8.48,50.720000000000006,174.08,cont,1.2620717095315923,74000597,2,2,0,0
6bba_3db54e20,13272,73,14.866666666666667,49.599999999999994,77.06666666666665,cont,3.673861903327645,74000599,7,1,0,6
6bba_3db54e20,13405,73,27.160000000000004,144.64000000000001,150.07999999999998,cont,2.102539015690318,74000600,1,0,1,0
6bba_3db54e20,13422,73,28.840000000000003,25.92,62.71999999999999,cont,1.955679229832942,74000602,3,1,2,0
6bba_3db54e20,13494,73,35.13333333333335,225.06666666666666,100.0,cont,3.3783676613943934,74000603,5,0,0,5
6bba_3db54e20,13577,73,49.48,207.36000000000004,50.080000000000005,cont,4.221529824897599,74000605,4,1,3,0
6bba_3db54e20,13760,74,8.519999999999998,56.80000000000001,175.99999999999997,cont,1.0055323527863305,75000610,3,1,2,0
6bba_3db54e20,13836,74,14.159999999999998,56.80000000000001,81.76000000000002,cont,1.7075649694813928,75000611,1,0,1,0
6bba_3db54e20,14328,75,8.519999999999998,61.920000000000016,177.92,cont,1.8122210992315566,76000623,2,1,1,0
6bba_3db54e20,14525,75,26.000000000000004,159.20000000000002,153.91999999999996,cont,2.385708097190435,76000625,2,2,0,0
6bba_3db54e20,14842,76,3.8400000000000007,152.0,27.36,cont,1.93381827610559,77000647,3,0,3,0
6bba_3db54e20,14887,76,8.16,68.00000000000001,180.64000000000001,cont,1.431660967198591,77000636,1,0,1,0
6bba_3db54e20,15182,76,34.2,137.44000000000003,81.27999999999999,cont,6.371160608554769,77000641,2,0,2,0
6bba_3db54e20,15429,77,5.000000000000001,212.79999999999998,47.36,cont,6.932125057476959,78000648,1,1,0,0
6bba_3db54e20,15480,77,9.84,75.36,92.16000000000003,cont,2.3083009173415787,78000650,1,0,1,0
6bba_3db54e20,15977,78,2.56,164.15999999999997,33.599999999999994,cont,2.990927166030633,79000673,2,2,0,0
6bba_3db54e20,15982,78,4.640000000000001,218.24,50.56,cont,5.395097890678166,79000660,6,5,1,0
6bba_3db54e20,15992,78,5.6400000000000015,201.60000000000002,73.76,cont,6.912116625896875,79000660,9,5,0,4
6bba_3db54e20,16010,78,7.3199999999999985,74.88,186.71999999999997,cont,2.008831283358564,79000661,5,4,1,0
6bba_3db54e20,16013,78,5.719999999999999,180.79999999999998,27.999999999999996,cont,6.316820427438148,79000673,3,3,0,0
6bba_3db54e20,16224,78,24.160000000000004,180.16,161.92000000000002,cont,2.1188030966798195,79000663,5,5,0,0
6bba_3db54e20,16303,78,32.96,143.36,66.08000000000001,cont,2.2866354240455498,79000667,4,4,0,0
6bba_3db54e20,16315,78,33.839999999999996,31.360000000000007,69.28,cont,5.142836188573388,79000668,2,2,0,0
6bba_3db54e20,16536,79,4.28,184.63999999999996,27.999999999999996,cont,1.2655192857084356,80000673,9,1,3,5
6bba_3db54e20,16576,79,7.4399999999999995,97.92000000000002,192.64000000000004,cont,2.5549733976110276,80000671,5,0,3,2
6bba_3db54e20,16769,79,21.733333333333327,75.99999999999999,68.53333333333333,cont,3.8172130476813804,80000678,4,0,0,4
6bba_3db54e20,16785,79,24.480000000000004,60.64,72.79999999999998,cont,6.330934098732981,80000678,8,0,3,5
6bba_3db54e20,16787,79,24.480000000000004,84.8,71.99999999999999,cont,3.681584450002468,80000678,2,0,2,0
6bba_3db54e20,16858,79,30.520000000000007,171.84,62.88000000000001,cont,3.264349571507311,80000680,5,0,4,1
6bba_3db54e20,16866,79,33.44,57.44000000000001,76.63999999999999,cont,6.8252514834619875,80000684,2,0,2,0
6bba_3db54e20,16966,79,48.99999999999998,196.5333333333333,68.53333333333333,cont,3.979775804161475,80000685,4,0,0,4
6bba_3db54e20,18232,82,3.4800000000000004,202.72000000000003,12.640000000000004,cont,6.790264285173297,83000715,2,2,0,0
6bba_3db54e20,18298,82,6.5200000000000005,203.35999999999999,40.64000000000001,cont,6.402603108306808,83000715,1,1,0,0
6bba_3db54e20,18511,82,25.000000000000004,18.56,76.0,parent,2.039099832033739,83000724,3,1,2,0
6bba_3db54e20,18872,83,6.866666666666666,198.93333333333334,31.999999999999993,cont,4.822050408165483,84000725,6,0,0,6
6bba_3db54e20,18992,83,17.400000000000002,20.320000000000004,83.04,daughter,6.018316443366864,84000737,1,0,1,0
6bba_3db54e20,19095,83,25.84,25.279999999999994,76.0,daughter,5.436590441168065,84000738,1,0,1,0
6bba_3db54e20,19106,83,27.133333333333333,7.4666666666666694,83.46666666666664,daughter,6.806597213365862,84000738,5,0,0,5
6bba_3db54e20,19693,84,27.039999999999996,189.27999999999997,76.64000000000001,cont,1.8327954229809715,85000741,2,2,0,0
6bba_3db54e20,19974,85,4.0,8.0,86.71999999999998,cont,6.5004874817201195,86000745,2,1,1,0
6bba_3db54e20,20173,85,18.640000000000004,109.91999999999999,82.55999999999999,cont,4.063994725021174,86000749,7,0,3,4
6bba_3db54e20,20184,85,18.84,95.83999999999999,84.16000000000001,cont,5.875627891978871,86000749,6,0,1,5
6bba_3db54e20,20266,85,25.999999999999996,195.36,78.72000000000001,cont,0.28379404944430103,86000750,4,2,2,0
6bba_3db54e20,20569,86,4.0,11.36,90.72,cont,6.99939394697855,87000756,1,0,1,0
6bba_3db54e20,21749,88,4.8,14.719999999999999,91.51999999999998,cont,5.476828604448011,89000774,3,3,0,0
6bba_3db54e20,21765,88,4.64,134.72000000000003,201.28000000000006,cont,1.4230584360805296,89000783,3,3,0,0
6bba_3db54e20,21782,88,4.68,27.200000000000003,83.35999999999999,cont,2.512451803020309,89000774,4,4,0,0
6bba_3db54e20,21917,88,15.84,123.20000000000002,81.92,cont,5.2971012119837795,89000778,3,3,0,0
6bba_3db54e20,22026,88,25.359999999999996,194.56000000000006,88.64,cont,5.70136524220294,89000780,2,2,0,0
6bba_3db54e20,22308,89,2.5199999999999996,153.76000000000002,205.28,cont,2.8640596187928837,90000791,5,0,0,5
6bba_3db54e20,22369,89,5.360000000000001,135.20000000000002,202.71999999999997,cont,6.5429207927346935,90000791,6,0,3,3
6bba_3db54e20,22463,89,14.84,121.76000000000002,94.56,cont,6.930353526336151,90000785,7,0,3,4
6bba_3db54e20,22465,89,15.000000000000004,130.08,82.72000000000001,cont,3.3824990761269897,90000785,3,0,3,0
6bba_3db54e20,22507,89,16.36,242.40000000000003,169.28,cont,5.730746297385713,90000786,5,0,2,3
6bba_3db54e20,22533,89,19.84,237.44,177.28,cont,1.9245104897090028,90000786,3,0,3,0
6bba_3db54e20,22551,89,21.039999999999996,230.71999999999997,161.28,cont,6.71410199691664,90000786,5,0,2,3
6bba_3db54e20,24511,93,4.3199999999999985,43.20000000000001,86.56,cont,1.1655906496279054,94000814,2,2,0,0
6bba_3db54e20,24526,93,4.16,167.2,204.64,cont,1.3752147559563208,94000820,2,2,0,0
6bba_3db54e20,24864,93,39.279999999999994,242.72000000000003,108.95999999999998,cont,3.2827011077007993,94000818,1,1,0,0
6bba_3db54e20,25048,94,3.6799999999999997,1.9200000000000004,164.0,cont,4.970959728764657,95000819,1,1,0,0
6bba_3db54e20,25050,94,2.96,15.839999999999996,165.75999999999996,cont,4.826496205841252,95000819,4,1,3,0
6bba_3db54e20,25471,95,0.3999999999999979,5.599999999999989,165.0666666666666,cont,2.1823520775250356,96000831,3,0,1,2
6bba_3db54e20,25472,95,0.31999999999999995,23.200000000000003,158.56,cont,6.46016701119251,96000831,4,0,0,4
6bba_3fda6b25,350,2,37.32000000000001,190.88000000000002,37.44,cont,3.2663712664055637,3000042,1,0,1,0
6bba_3fda6b25,391,2,58.2,190.56,34.72,cont,1.4300923265649716,3000048,1,0,1,0
6bba_3fda6b25,626,4,60.84000000000001,92.0,125.28000000000002,cont,0.2837940494442908,5000093,1,1,0,0
6bba_3fda6b25,647,5,9.360000000000003,221.27999999999997,138.56,cont,1.0706518166518872,6000096,1,1,0,0
6bba_3fda6b25,729,5,59.20000000000002,185.92000000000002,35.99999999999999,cont,1.835171143245204,6000113,1,1,0,0
6bba_3fda6b25,732,5,60.68000000000001,92.0,126.72000000000001,cont,1.0622330547012617,6000114,1,0,1,0
6bba_3fda6b25,752,6,9.0,224.00000000000003,138.07999999999998,cont,0.8794538433027764,7000117,1,0,1,0
6bba_3fda6b25,765,6,20.733333333333334,240.5333333333333,120.53333333333336,cont,1.4856387164561113,7000122,1,0,0,1
6bba_3fda6b25,832,6,59.36,186.56,35.36,cont,1.0653359212004498,7000132,1,0,1,0
6bba_3fda6b25,1455,12,26.16,80.64,235.35999999999999,cont,0.6636687803415223,13000230,1,1,0,0
6bba_3fda6b25,1573,13,27.160000000000004,82.72000000000001,228.64,cont,1.4407620596406527,14000248,1,0,1,0
6bba_3fda6b25,2404,21,3.0,207.36,244.64000000000001,cont,0.29831034594863415,22000391,1,1,0,0
6bba_3fda6b25,2491,21,62.000000000000014,22.72,246.72,cont,1.3099153121480762,22000410,1,1,0,0
6bba_3fda6b25,2504,22,3.0,208.0,244.00000000000003,cont,0.8125,23000411,1,0,1,0
6bba_3fda6b25,2597,22,62.000000000000014,21.280000000000005,245.28,cont,1.7310636181261252,23000430,1,0,1,0
6bba_3fda6b25,2821,25,2.6800000000000006,209.28000000000003,245.28000000000003,cont,0.6073675267084955,26000468,1,1,0,0
6bba_3fda6b25,2923,26,2.32,210.71999999999997,246.72000000000006,cont,1.1826829932403693,27000489,1,0,1,0
6bba_3fda6b25,3635,32,60.000000000000014,80.64,91.35999999999999,cont,1.6680607190687005,33000615,1,1,0,0
6bba_3fda6b25,3737,33,60.000000000000014,82.72000000000001,89.27999999999999,cont,1.9413819980879332,34000633,1,0,1,0
6bba_3fda6b25,3889,35,23.319999999999997,77.92,236.16000000000003,cont,1.1569497882362918,36000659,1,1,0,0
6bba_3fda6b25,4000,36,23.840000000000003,78.72000000000001,231.20000000000002,cont,2.6492982561425555,37000679,1,0,1,0
6bba_3fda6b25,4056,36,60.000000000000014,94.72,73.44,cont,1.9981553993620988,37000691,1,0,0,1
6bba_3fda6b25,4553,41,59.68000000000001,192.64,33.279999999999994,cont,1.3155476521206086,42000800,1,1,0,0
6bba_3fda6b25,4646,42,59.83999999999999,191.36,34.72,cont,0.6368673331236301,43000827,1,0,1,0
6bba_3fda6b25,4964,45,61.0,103.36,68.64000000000001,cont,0.6106195624118125,46000888,1,1,0,0
6bba_3fda6b25,5073,46,61.32000000000001,104.0,68.0,cont,1.2612345142756007,47000908,1,0,1,0
6bba_3fda6b25,5234,48,30.0,155.84000000000003,109.44000000000001,cont,2.487736483733754,49000947,1,1,0,0
6bba_3fda6b25,5292,48,62.000000000000014,20.000000000000007,238.72000000000003,cont,0.9646534351776221,49000956,1,1,0,0
6bba_3fda6b25,5348,49,29.68,160.79999999999998,106.56,cont,2.0647733683143144,50000970,1,0,1,0
6bba_3fda6b25,5406,49,62.000000000000014,20.000000000000007,237.27999999999997,cont,1.0716379110968552,50000977,1,0,1,0
6bba_3fda6b25,5463,50,32.160000000000004,127.36,128.0,cont,0.8225132977648446,51001002,1,1,0,0
6bba_3fda6b25,5568,51,31.160000000000004,131.99999999999997,124.64000000000001,cont,0.5040120286263143,52001028,1,0,1,0
6bba_3fda6b25,5734,52,62.000000000000014,20.000000000000007,234.72000000000006,cont,0.9646534351776221,53001049,1,1,0,0
6bba_3fda6b25,5849,53,62.000000000000014,20.000000000000007,233.91999999999996,cont,0.4387500000000131,54001072,1,0,1,0
6bba_3fda6b25,5850,53,61.680000000000014,25.28,139.35999999999999,cont,4.802115581959687,54001070,1,1,0,0
6bba_3fda6b25,5959,54,61.68,26.719999999999995,137.28000000000003,cont,4.482673762666642,55001093,1,0,1,0
6bba_3fda6b25,6397,58,62.000000000000014,16.0,149.28,cont,6.593256024150738,59001192,1,0,1,0
6bba_3fda6b25,6795,62,60.279999999999994,26.080000000000002,143.36,cont,6.382527541068658,63001289,1,1,0,0
6bba_3fda6b25,6911,63,61.99999999999999,35.46666666666666,159.46666666666664,cont,1.6503064214630476,64001315,2,0,0,2
6bba_3fda6b25,7019,64,61.99999999999999,15.999999999999996,157.6,cont,6.2878261595324085,65001343,1,0,0,1
6bba_3fda6b25,7435,68,58.0,235.35999999999999,21.28,cont,1.4171080807404954,69001438,1,1,0,0
6bba_3fda6b25,7547,69,58.16,234.72,21.28,cont,3.918710326752913,70001467,1,0,1,0
6bba_3fda6b25,8171,76,17.479999999999997,208.79999999999995,106.08,cont,2.5086652950324053,77001618,1,1,0,0
6bba_3fda6b25,8261,77,12.266666666666666,217.06666666666666,106.93333333333331,cont,1.268298359679884,78001639,2,0,0,2
6bba_3fda6b25,8266,77,17.959999999999997,207.83999999999997,101.91999999999999,cont,2.5878348213322977,78001640,1,0,1,0
6bba_3fda6b25,8334,77,59.44,232.64,30.72,cont,1.797884607670923,78001659,2,2,0,0
6bba_3fda6b25,8443,78,60.0,233.44,31.360000000000003,cont,3.484365302677094,79001687,2,0,2,0
6bba_3fda6b25,8932,83,53.68,178.72000000000003,37.92,cont,0.5332872701462053,84001803,1,1,0,0
6bba_3fda6b25,9044,84,53.67999999999999,180.00000000000003,37.92,cont,2.1773180894164375,85001830,1,0,1,0
6bba_3fda6b25,9161,85,53.52,181.43999999999997,40.0,cont,0.8974355826464647,86001854,1,1,0,0
6bba_3fda6b25,9278,86,53.000000000000014,180.64,41.92000000000001,cont,1.756579150650461,87001875,1,0,1,0
6bba_3fda6b25,9506,88,51.68000000000001,180.0,48.0,cont,2.5002831089698465,89001924,1,1,0,0
6bba_3fda6b25,10868,88,60.56802230834961,220.56085021972658,50.881357421875,cont,1.3334067760916288,89001927,1,1,0,0
6bba_3fda6b25,9615,89,50.960000000000015,180.0,51.36000000000001,cont,1.8931075642973632,90001952,1,0,1,0
6bba_3fda6b25,10869,89,60.69222229003907,215.80893005371098,53.13953186035157,cont,1.3291426477886967,90001957,1,0,1,0
6bba_3fda6b25,9791,91,12.64,225.92,122.08000000000003,cont,0.5868027777712047,92001981,1,1,0,0
6bba_3fda6b25,9799,91,20.640000000000004,236.0,86.08,cont,2.384102622057191,92001984,1,1,0,0
6bba_3fda6b25,9801,91,21.48,207.36,108.64000000000001,cont,1.085957239029239,92001983,1,1,0,0
6bba_3fda6b25,9904,92,12.68,224.64000000000004,124.00000000000003,cont,0.6758906161502802,93002000,1,0,1,0
6bba_3fda6b25,9914,92,21.32,236.64000000000001,84.00000000000001,cont,1.9040950934761547,93002003,1,0,1,0
6bba_3fda6b25,9918,92,22.320000000000004,208.0,107.36,cont,1.3005077133565934,93002004,1,0,1,0
6bba_43fea39d,182,2,23.840000000000003,121.91999999999999,215.36000000000004,cont,2.454556986504901,3000021,1,1,0,0
6bba_43fea39d,190,2,29.640000000000008,147.35999999999999,140.0,cont,1.1231324777157745,3000022,1,1,0,0
6bba_43fea39d,265,3,23.0,122.72000000000001,212.64,cont,2.395428589000299,4000031,1,0,1,0
6bba_43fea39d,273,3,29.32,146.72,136.64,cont,1.315547652120588,4000033,1,0,1,0
6bba_43fea39d,366,4,27.2,99.36000000000001,250.72000000000003,cont,1.4171080807404977,5000042,1,1,0,0
6bba_43fea39d,367,4,28.84000000000001,146.72000000000003,134.71999999999997,cont,2.2134623613244417,5000043,1,1,0,0
6bba_43fea39d,448,5,26.840000000000003,100.0,249.27999999999997,cont,2.07338792800575,6000052,1,0,1,0
6bba_43fea39d,452,5,28.679999999999993,145.27999999999997,131.2,cont,1.1687580213628428,6000053,1,0,1,0
6bba_43fea39d,528,6,28.000000000000004,144.64,130.72,cont,1.9688670600119134,7000064,1,1,0,0
6bba_43fea39d,595,7,16.36,160.00000000000006,57.28000000000001,cont,1.154207655710193,8000072,1,1,0,0
6bba_43fea39d,611,7,27.360000000000003,143.99999999999997,129.28,cont,2.7454806614689438,8000075,1,0,1,0
6bba_43fea39d,639,7,57.16000000000001,77.44000000000001,244.00000000000003,cont,1.504946531442211,8000078,1,1,0,0
6bba_43fea39d,640,7,56.68,154.71999999999997,8.64,cont,2.1697857757161136,8000079,2,2,0,0
6bba_43fea39d,643,7,58.04,164.64,60.0,cont,3.214133339642277,8000080,2,2,0,0
6bba_43fea39d,681,8,15.16,160.00000000000006,58.72,cont,0.2837940494443122,9000083,1,0,1,0
6bba_43fea39d,730,8,55.64000000000001,153.92000000000002,8.640000000000002,cont,1.2355344238425807,9000094,2,0,2,0
6bba_43fea39d,732,8,56.28,166.72000000000003,54.559999999999995,cont,2.6974021066574325,9000090,2,0,2,0
6bba_43fea39d,733,8,56.480000000000004,76.63999999999999,244.00000000000003,cont,1.3086043376437246,9000092,1,0,1,0
6bba_43fea39d,1059,12,45.84,126.72000000000001,147.35999999999996,cont,2.0156551354336303,13000142,1,1,0,0
6bba_43fea39d,1095,13,16.040000000000003,95.36000000000001,249.27999999999997,cont,1.5856036902391393,14000151,1,1,0,0
6bba_43fea39d,1122,13,45.160000000000004,125.28,145.92000000000002,cont,1.6559060963714103,14000159,1,0,1,0
6bba_43fea39d,1160,14,14.960000000000003,94.56,250.71999999999997,cont,0.26252678434780935,15000167,1,0,1,0
6bba_43fea39d,1428,17,57.68000000000001,163.36,29.920000000000005,cont,2.297177616119381,18000214,1,1,0,0
6bba_43fea39d,1468,18,18.319999999999997,138.72000000000003,136.0,cont,1.3690564725386694,19000219,1,1,0,0
6bba_43fea39d,1513,18,56.68,163.35999999999999,32.64,cont,0.8020325741514431,19000227,1,0,1,0
6bba_43fea39d,1549,19,18.000000000000004,137.28000000000003,136.0,cont,1.6788695973779437,20000232,1,0,1,0
6bba_43fea39d,1756,22,15.880000000000003,137.28,138.56,cont,3.6359352304462162,23000279,1,1,0,0
6bba_43fea39d,1771,22,34.4,148.00000000000003,74.55999999999999,cont,2.637610779664054,23000292,1,1,0,0
6bba_43fea39d,1777,22,44.0,156.0,101.92000000000002,cont,2.175862257244243,23000281,1,1,0,0
6bba_43fea39d,1778,22,43.04,167.36,57.92000000000001,cont,4.960936309054978,23000287,1,1,0,0
6bba_43fea39d,1789,22,56.55999999999999,134.56000000000003,43.36,cont,4.009767761354782,23000290,2,2,0,0
6bba_43fea39d,1821,23,13.959999999999999,138.07999999999998,142.08,cont,0.7833781015576006,24000295,1,0,1,0
6bba_43fea39d,1837,23,32.12,148.64000000000001,77.44,cont,0.679008928512722,24000309,1,0,1,0
6bba_43fea39d,1841,23,33.2,173.92000000000002,56.63999999999999,cont,4.784376428804486,24000311,1,0,0,1
6bba_43fea39d,1850,23,40.99999999999999,168.0,62.08000000000001,cont,3.3422896343674195,24000312,2,0,1,1
6bba_43fea39d,1861,23,54.8,137.44,43.36,cont,2.165095263493041,24000306,2,0,2,0
6bba_43fea39d,2062,26,29.519999999999996,175.36,51.36000000000001,cont,1.3513904876089635,27000351,2,2,0,0
6bba_43fea39d,2065,26,36.60000000000001,169.44,68.64000000000001,cont,2.812329994861929,27000344,1,1,0,0
6bba_43fea39d,2083,26,58.68,48.0,244.00000000000003,cont,2.2937264985172017,27000348,1,1,0,0
6bba_43fea39d,2091,26,60.52,124.00000000000003,55.99999999999999,cont,2.5592589259393077,27000349,1,1,0,0
6bba_43fea39d,2132,27,27.239999999999995,176.0,48.959999999999994,cont,1.5374939024269392,28000366,2,0,2,0
6bba_43fea39d,2136,27,31.133333333333326,172.0,68.53333333333333,cont,1.4779660620445894,28000355,2,0,0,2
6bba_43fea39d,2156,27,57.52000000000001,49.28,244.00000000000003,cont,2.50445134959734,28000363,1,0,1,0
6bba_43fea39d,2164,27,59.84000000000002,124.63999999999999,55.35999999999999,cont,1.9084585566629413,28000364,1,0,1,0
6bba_43fea39d,2240,28,58.84000000000001,125.28000000000002,54.720000000000006,cont,1.9771649526531432,29000375,1,1,0,0
6bba_43fea39d,2316,29,57.84,126.72000000000001,53.92000000000001,cont,0.5822853681829819,30000389,1,0,1,0
6bba_43fea39d,2490,32,21.479999999999997,174.07999999999996,67.84,cont,2.10931009159394,33000439,1,1,0,0
6bba_43fea39d,2507,32,40.56000000000002,148.00000000000006,37.279999999999994,cont,2.6891172459563624,33000427,1,1,0,0
6bba_43fea39d,2570,33,38.11999999999999,148.64000000000001,38.72,cont,0.9816129901850303,34000445,1,0,1,0
6bba_43fea39d,2582,33,49.53333333333332,131.99999999999997,60.53333333333332,cont,3.192848182092133,34000449,1,0,0,1
6bba_43fea39d,2625,34,7.733333333333329,172.0,100.0,cont,3.7056691038116853,35000470,2,0,0,2
6bba_43fea39d,2809,36,58.0,96.0,143.36,cont,1.6813842883172194,37000499,1,1,0,0
6bba_43fea39d,2870,37,56.68000000000001,98.72000000000001,141.28000000000003,cont,2.339209936068146,38000511,1,0,1,0
6bba_43fea39d,3090,40,54.64,129.28,44.8,cont,2.4017587956537123,41000546,1,1,0,0
6bba_43fea39d,3177,41,53.31999999999999,132.00000000000003,42.720000000000006,cont,1.402404185675449,42000560,1,0,1,0
6bba_43fea39d,3358,44,47.2,112.0,140.64000000000001,cont,3.0856178473686557,45000595,1,1,0,0
6bba_43fea39d,3426,45,45.959999999999994,114.07999999999998,140.64,cont,1.0278686990564516,46000606,1,0,1,0
6bba_43fea39d,3714,49,39.13333333333333,148.53333333333336,24.53333333333333,cont,3.0451592391312894,50000657,1,0,0,1
6bba_43fea39d,3896,52,37.16,127.36,125.43999999999998,cont,0.7326930206437087,53000681,1,1,0,0
6bba_43fea39d,3899,52,39.67999999999999,152.0,25.439999999999998,cont,0.5498650402598898,53000682,1,1,0,0
6bba_43fea39d,3912,52,56.00000000000001,128.0,46.08000000000001,cont,0.8794538433027572,53000683,1,1,0,0
6bba_43fea39d,3956,53,36.96,126.72000000000001,122.55999999999997,cont,0.6471500019315373,54000689,1,0,1,0
6bba_43fea39d,3958,53,39.480000000000004,152.0,21.92,cont,1.1030865786510158,54000690,1,0,1,0
6bba_43fea39d,3972,53,55.84,128.0,42.72,cont,1.395987553669443,54000691,1,0,1,0
6bba_43fea39d,4141,56,55.68000000000001,125.92000000000002,34.720000000000006,cont,0.5975052301026175,57000715,1,1,0,0
6bba_43fea39d,4197,57,55.64,126.08,33.28,cont,0.8673603994303597,58000723,1,0,1,0
6bba_43fea39d,4973,70,8.16,164.0,11.36,cont,1.3704059617500208,71000848,1,1,0,0
6bba_43fea39d,5037,71,8.32,164.0,10.720000000000002,cont,2.7624044101108733,72000856,1,0,1,0
6bba_43fea39d,5381,76,4.32,126.72,169.44,cont,0.8276340525860503,77000897,1,1,0,0
6bba_43fea39d,5455,77,4.84,126.56000000000002,166.55999999999995,cont,2.4589638315762077,78000907,1,0,1,0
6bba_474be664,381,1,14.399999999999999,51.68000000000001,240.95999999999998,cont,6.150147355958216,2000011,1,1,0,0
6bba_474be664,994,3,14.866666666666667,72.53333333333332,227.46666666666664,cont,1.4715000849473376,4000023,2,0,0,2
6bba_474be664,1526,5,1.0,14.719999999999999,24.64,cont,1.7804691270842075,6000032,1,1,0,0
6bba_474be664,1625,5,16.68,93.44,181.92000000000004,cont,3.7312774287099075,6000036,3,0,0,3
6bba_474be664,3566,11,14.319999999999999,88.00000000000001,218.07999999999998,cont,6.77359960157818,12000070,2,1,1,0
6bba_474be664,3569,11,15.400000000000002,77.06666666666666,207.46666666666664,cont,1.0836718221193904,12000070,4,1,0,3
6bba_474be664,3570,11,15.000000000000004,107.46666666666665,179.99999999999997,cont,5.323029189939157,12000071,3,1,0,2
6bba_474be664,3909,12,15.920000000000002,108.96000000000001,176.32,cont,5.100118871163686,13000078,1,0,1,0
6bba_474be664,3913,12,16.04,78.07999999999998,205.59999999999997,cont,0.8629346730778573,13000077,1,0,1,0
6bba_474be664,4241,13,16.52,108.8,172.64,cont,4.293507089198758,14000083,1,1,0,0
6bba_474be664,4244,13,18.519999999999996,96.0,168.64000000000004,cont,5.340542165829234,14000083,1,1,0,0
6bba_474be664,4575,14,16.839999999999996,108.16000000000001,169.44,cont,3.559009693720997,15000088,1,0,1,0
6bba_474be664,4581,14,19.64,96.0,167.36000000000004,cont,4.991475232834476,15000088,2,1,1,0
6bba_474be664,4922,15,17.0,104.48,165.28,cont,3.4783351800825644,16000093,4,0,0,4
6bba_474be664,4928,15,20.6,96.0,166.08,cont,5.183571705397737,16000093,1,0,1,0
6bba_474be664,5265,16,14.8,90.72,196.00000000000003,cont,6.037131976360953,17000097,1,0,1,0
6bba_474be664,6683,20,19.84,101.28,164.79999999999995,cont,2.6002538938534365,21000120,1,1,0,0
6bba_474be664,7031,21,21.0,102.72000000000003,164.64,cont,4.1843355580665245,22000125,1,0,1,0
6bba_474be664,8698,26,19.866666666666667,87.99999999999997,200.53333333333333,cont,1.8513989174105334,27000151,6,0,0,6
6bba_474be664,10063,30,21.720000000000002,92.79999999999998,195.35999999999999,cont,2.1351906384442643,31000172,4,1,0,3
6bba_474be664,10440,31,22.16,94.07999999999998,194.72000000000003,cont,0.7283553820079679,32000176,2,1,1,0
6bba_474be664,10792,32,22.48,94.56,193.27999999999997,cont,0.9226795218275999,33000181,1,0,1,0
6bba_474be664,11514,34,22.839999999999993,95.99999999999997,190.72000000000003,cont,1.4171080807404728,35000192,1,1,0,0
6bba_474be664,11539,34,27.160000000000004,124.64000000000001,160.79999999999998,cont,4.583594727394642,35000193,1,1,0,0
6bba_474be664,11886,35,27.160000000000004,126.72,156.79999999999998,cont,2.2045567156460297,36000199,2,1,1,0
6bba_474be664,12228,36,22.36,97.44,184.64000000000004,cont,2.7432194817403857,37000204,2,0,2,0
6bba_474be664,12262,36,27.519999999999996,127.2,152.0,cont,3.5501322598601828,37000205,1,0,1,0
6bba_474be664,12580,37,21.88,96.80000000000001,181.92000000000002,cont,3.8445247186745974,38000209,1,1,0,0
6bba_474be664,12965,38,27.0,113.28000000000003,170.72000000000003,cont,4.764687325785391,39000216,2,0,2,0
6bba_474be664,12972,38,27.999999999999996,111.2,191.84,cont,6.498801451998668,39000216,2,0,2,0
6bba_474be664,13316,39,27.52,114.72,169.28,cont,5.03689714134605,40000222,1,1,0,0
6bba_474be664,13318,39,27.84,132.00000000000006,140.64,cont,1.5014331695083774,40000223,2,1,1,0
6bba_474be664,13662,40,25.733333333333334,111.99999999999999,176.53333333333336,cont,1.36092065702677,41000228,4,0,0,4
6bba_474be664,13675,40,28.16,112.0,188.00000000000003,cont,5.133071235868453,41000228,1,1,0,0
6bba_474be664,13676,40,27.84,115.35999999999999,168.0,cont,5.800976884327327,41000228,1,0,1,0
6bba_474be664,13677,40,28.16,132.00000000000006,140.00000000000003,cont,1.4876517779709295,41000229,1,0,1,0
6bba_474be664,14039,41,28.320000000000007,113.28000000000003,188.00000000000003,cont,5.418174998327409,42000235,2,1,1,0
6bba_474be664,14401,42,25.519999999999996,111.36,173.92000000000002,cont,1.0517403731434931,43000241,1,1,0,0
6bba_474be664,14420,42,28.68,114.72,187.35999999999999,cont,6.917787421748367,43000241,1,0,1,0
6bba_474be664,14790,43,28.36,116.0,164.0,cont,6.4632523739987136,44000247,1,0,1,0
6bba_474be664,15521,45,28.133333333333333,119.46666666666667,133.06666666666663,cont,1.8927355643794201,46000260,3,0,0,3
6bba_474be664,15890,46,29.76,132.00000000000003,140.00000000000003,cont,4.624145867076441,47000266,1,0,1,0
6bba_474be664,16245,47,26.68,109.28,173.27999999999997,cont,0.7441364542205968,48000271,1,1,0,0
6bba_474be664,16267,47,30.639999999999993,132.64000000000001,140.00000000000003,cont,5.194334804621292,48000272,1,1,0,0
6bba_474be664,16658,48,31.72,135.36,139.35999999999999,cont,6.3358539529017,49000278,1,0,1,0
6bba_474be664,16672,48,34.84,117.28,140.64,cont,6.964049311463845,49000278,3,0,3,0
6bba_474be664,16994,49,27.84,108.0,170.72000000000003,cont,1.0443074858488592,50000283,1,1,0,0
6bba_474be664,17401,50,33.000000000000014,138.72000000000003,137.28000000000003,cont,6.376401439095585,51000290,1,1,0,0
6bba_474be664,17772,51,27.680000000000007,108.0,168.64000000000004,cont,1.1116710394716447,52000295,1,1,0,0
6bba_474be664,17788,51,31.000000000000007,126.08,128.00000000000003,cont,1.8170958973042481,52000296,1,0,1,0
6bba_474be664,17802,51,33.000000000000014,139.36,136.00000000000003,cont,6.167640351058113,52000296,1,0,1,0
6bba_474be664,17817,51,35.00000000000001,119.36000000000001,140.00000000000003,cont,6.895363478454215,52000296,1,1,0,0
6bba_474be664,18154,52,27.480000000000004,108.64000000000001,168.0,cont,1.038602576782865,53000301,2,1,1,0
6bba_474be664,18529,53,27.479999999999997,109.28,168.0,cont,1.1636633963479255,54000306,1,0,1,0
6bba_474be664,18564,53,33.000000000000014,140.00000000000003,133.27999999999997,cont,6.292003357437135,54000307,1,1,0,0
6bba_474be664,20109,57,29.720000000000006,119.51999999999998,166.72,cont,1.8582640588732386,58000328,6,1,0,5
6bba_474be664,20519,58,29.879999999999995,120.63999999999999,165.27999999999997,cont,1.5606769364605857,59000332,1,0,1,0
6bba_474be664,22410,63,29.64,122.72000000000001,165.44000000000005,parent,2.8756531866690658,64000351,2,1,1,0
6bba_474be664,22759,64,26.439999999999998,135.84,173.27999999999997,daughter,3.100344990884075,65000356,5,0,2,3
6bba_474be664,22792,64,30.520000000000007,119.84,164.64,daughter,1.7801724811096344,65000355,1,0,1,0
6bba_474be664,23826,67,25.360000000000007,134.56,172.0,cont,5.917700276500991,68000361,1,1,0,0
6bba_474be664,24155,68,24.26666666666665,138.9333333333333,172.0,cont,4.672326367084791,69000364,3,0,1,2
6bba_474be664,24166,68,26.000000000000007,141.92000000000002,177.28,cont,4.236796032971137,69000364,1,1,0,0
6bba_474be664,24193,68,31.32,124.00000000000003,161.28,cont,1.2870369312882894,69000363,1,1,0,0
6bba_474be664,25203,71,30.839999999999996,134.56000000000003,179.36000000000004,cont,5.842276151039102,72000377,1,1,0,0
6bba_474be664,25548,72,31.68,140.00000000000003,151.36,cont,5.631627345847731,73000383,1,0,1,0
6bba_474be664,26262,74,28.68,141.43999999999997,172.64,cont,6.397136077964882,75000392,1,0,1,0
6bba_474be664,26281,74,31.680000000000003,140.00000000000003,149.28,cont,4.493000111284217,75000392,1,0,1,0
6bba_474be664,26628,75,29.880000000000006,140.64000000000001,173.44000000000005,cont,5.834655222247516,76000395,1,1,0,0
6bba_474be664,26934,76,28.959999999999997,154.08,156.0,cont,6.141124184544723,77000400,1,1,0,0
6bba_474be664,26946,76,31.159999999999997,139.35999999999999,147.36000000000004,cont,5.252245353179899,77000400,1,1,0,0
6bba_474be664,26947,76,30.680000000000003,140.64,172.64,cont,5.190749584597576,77000400,1,0,1,0
6bba_474be664,27258,77,29.120000000000005,150.72000000000003,156.0,cont,4.834013116707487,78000405,1,0,1,0
6bba_474be664,27268,77,31.159999999999997,139.35999999999999,147.36,cont,4.855814330521702,78000405,2,1,1,0
6bba_474be664,27269,77,31.000000000000007,140.00000000000003,172.64000000000004,cont,5.600500675162909,78000405,2,0,0,2
6bba_474be664,27590,78,29.680000000000007,147.84,156.0,cont,4.251945326847457,79000409,3,0,0,3
6bba_474be664,27592,78,31.320000000000004,138.56,146.56000000000003,cont,4.379699083555844,79000409,1,0,1,0
6bba_474be664,27594,78,31.000000000000007,140.00000000000003,171.99999999999994,cont,6.809990937035052,79000409,1,1,0,0
6bba_474be664,29540,84,29.36,147.36000000000004,153.91999999999996,cont,5.952492781389995,85000439,3,3,0,0
6bba_474be664,29558,84,33.480000000000004,140.00000000000003,141.44,cont,5.348324065770504,85000439,3,3,0,0
6bba_474be664,29566,84,35.36,219.83999999999997,11.360000000000003,cont,4.6891317160003005,85000438,1,1,0,0
6bba_474be664,29868,85,33.84000000000001,140.00000000000003,138.56,cont,3.3064657302019453,86000443,3,0,3,0
6bba_474be664,29873,85,35.04,130.88,149.44,cont,6.305041881304198,86000443,6,0,1,5
6bba_48816121,2607,8,25.160000000000007,145.28,192.64000000000004,cont,1.1702256726802902,9000047,1,1,0,0
6bba_48816121,2922,9,26.160000000000007,140.64000000000004,190.72000000000003,cont,1.8940837164708522,10000051,1,0,1,0
6bba_48816121,2940,9,28.519999999999996,134.72000000000003,236.64000000000001,cont,5.8406035647011745,10000050,1,0,1,0
6bba_48816121,3524,11,20.52,171.99999999999994,204.64000000000001,cont,2.467646905454668,12000057,1,1,0,0
6bba_48816121,3574,11,28.519999999999996,157.92000000000002,140.64000000000004,parent,4.33570586092969,12000059,1,1,0,0
6bba_48816121,3848,12,20.520000000000003,171.36,203.35999999999999,cont,0.9905837925183328,13000061,1,0,1,0
6bba_48816121,3879,12,27.64,158.72,140.16,daughter,1.9456618411224498,13000062,1,0,1,0
6bba_48816121,3920,12,34.133333333333326,148.0,136.5333333333333,daughter,1.728459945940976,13000063,4,0,0,4
6bba_48816121,5032,16,4.4799999999999995,231.36,246.72,cont,2.5374987684726062,17000089,1,1,0,0
6bba_48816121,5152,16,30.439999999999998,128.8,229.27999999999997,parent,1.9370926991241262,17000095,1,1,0,0
6bba_48816121,5348,17,5.0,230.56000000000003,245.28,cont,3.2710567864223945,18000096,1,0,1,0
6bba_48816121,5442,17,23.866666666666667,136.53333333333336,223.99999999999997,daughter,3.540880427434525,18000098,2,0,0,2
6bba_48816121,5524,17,38.84,127.36000000000001,243.35999999999999,daughter,4.844244652161985,18000100,1,0,1,0
6bba_48816121,7785,25,7.32,233.27999999999997,231.36,cont,2.8253044636817464,26000168,1,1,0,0
6bba_48816121,8849,28,45.13333333333332,55.466666666666654,252.0,cont,5.227363982103096,29000201,1,0,0,1
6bba_48816121,9358,30,27.160000000000004,147.35999999999996,192.0,cont,1.2547273110122361,31000214,1,0,1,0
6bba_48816121,10049,32,47.999999999999986,55.99999999999999,235.99999999999994,cont,4.958913218891792,33000237,2,0,0,2
6bba_48816121,11197,36,46.999999999999986,53.59999999999999,225.0666666666666,cont,3.2878095809030015,37000273,3,0,0,3
6bba_48816121,13163,43,47.64,44.00000000000001,200.0,cont,1.080472263642149,44000341,1,1,0,0
6bba_48816121,13207,43,56.99999999999999,36.64,181.43999999999997,cont,1.76348066689149,44000342,1,1,0,0
6bba_48816121,13269,44,10.319999999999999,206.72,240.64000000000004,cont,2.9398131849660074,45000344,1,1,0,0
6bba_48816121,13347,44,27.733333333333327,155.99999999999997,142.9333333333333,cont,1.7367154503705033,45000345,2,0,0,2
6bba_48816121,13456,44,48.52,43.36,195.35999999999999,cont,1.389921310722305,45000352,1,0,1,0
6bba_48816121,13488,44,56.99999999999999,35.36,175.99999999999997,cont,1.8226821753942921,45000353,1,0,1,0
6bba_48816121,13724,45,44.68,49.280000000000015,247.99999999999997,cont,3.9286034414280038,46000361,2,1,1,0
6bba_48816121,13745,45,48.99999999999999,63.46666666666666,252.53333333333336,cont,5.971122637792286,46000361,4,0,0,4
6bba_48816121,13996,46,44.99999999999999,48.64,244.64000000000001,cont,3.4332139188375783,47000373,1,0,1,0
6bba_48816121,14268,47,45.32000000000001,48.0,241.28000000000003,cont,4.5759838423010075,48000385,2,2,0,0
6bba_48816121,14540,48,45.68,47.36,238.72000000000003,cont,5.525095587408415,49000398,2,0,2,0
6bba_48816121,15910,53,56.36000000000001,25.28,142.72,cont,5.2854733113506445,54000467,1,1,0,0
6bba_48816121,16173,54,59.13333333333331,20.0,147.46666666666667,cont,1.790984608547956,55000480,1,0,0,1
6bba_48816121,16745,57,9.679999999999998,227.2,158.08,parent,3.8805871012000264,58000508,1,1,0,0
6bba_48816121,16767,57,15.320000000000004,228.64000000000001,169.28,parent,6.668521669005812,58000508,2,2,0,0
6bba_48816121,16995,58,8.48,229.44,154.56,daughter,0.8437490740735686,59000533,2,0,1,1
6bba_48816121,17015,58,14.266666666666666,215.46666666666664,172.5333333333333,daughter,1.2259509489235951,59000522,3,0,0,3
6bba_48816121,17029,58,15.679999999999998,227.20000000000002,166.71999999999997,daughter,5.512391864018754,59000522,2,0,2,0
6bba_48816121,18198,62,50.68000000000001,38.559999999999995,42.720000000000006,cont,2.347492345567907,63000577,1,1,0,0
6bba_48816121,18308,63,18.16,196.00000000000003,218.72000000000006,cont,1.259243944793837,64000583,1,0,1,0
6bba_48816121,18450,63,51.32000000000001,35.35999999999999,38.720000000000006,cont,2.322613358374565,64000589,1,0,1,0
6bba_48816121,19230,66,54.79999999999999,33.92,202.56,cont,3.2631764145537576,67000622,1,1,0,0
6bba_48816121,19238,66,58.16,0.0,199.68000000000006,cont,3.144490469456064,67000624,1,1,0,0
6bba_48816121,19487,67,55.52,32.0,196.0,cont,3.493901159520686,68000632,1,0,1,0
6bba_48816121,19971,69,56.480000000000004,27.36,183.36,cont,3.346711072082565,70000651,1,1,0,0
6bba_48816121,20206,70,52.866666666666646,31.999999999999993,172.5333333333333,cont,3.56719476230587,71000663,3,0,0,3
6bba_48816121,20218,70,57.0,24.640000000000008,177.92000000000002,cont,5.550273078867379,71000663,1,0,1,0
6bba_48816121,20427,71,45.04,60.000000000000014,244.00000000000003,cont,3.666743275714836,72000673,1,1,0,0
6bba_48816121,20671,72,45.68,60.000000000000014,240.00000000000006,cont,4.044552856929911,73000684,1,0,1,0
6bba_48816121,20687,72,50.16,50.080000000000005,244.00000000000003,cont,6.366288778597774,73000684,1,0,1,0
6bba_48816121,20724,73,0.68,250.72000000000003,240.0,cont,3.9286034414280144,74000688,1,0,1,0
6bba_48816121,20794,73,19.0,208.0,128.0,cont,3.2752922102462856,74000690,1,1,0,0
6bba_48816121,20904,73,46.12,59.36,235.99999999999994,cont,3.153086285689002,74000695,1,1,0,0
6bba_48816121,20924,73,50.640000000000015,49.440000000000005,239.36,cont,6.432578458518807,74000695,1,1,0,0
6bba_48816121,20948,73,58.16,20.0,162.07999999999998,cont,6.77471055654778,74000697,1,0,1,0
6bba_48816121,20960,74,0.8000000000000004,250.56,238.08000000000004,cont,2.5500076593022216,75000699,1,1,0,0
6bba_48816121,21039,74,18.84,208.0,124.00000000000003,cont,2.0526522477029583,75000701,1,0,1,0
6bba_48816121,21138,74,42.15999999999999,86.08,244.64000000000001,cont,3.2708953284995457,75000705,2,1,1,0
6bba_48816121,21158,74,47.16,58.08000000000001,232.0,cont,3.6002094123120165,75000707,3,2,1,0
6bba_48816121,21177,74,51.48,48.0,234.72000000000006,cont,6.971477269022686,75000707,2,1,1,0
6bba_48816121,21214,75,1.1600000000000001,250.72000000000003,233.92000000000007,cont,3.1017500100749573,76000711,1,0,1,0
6bba_48816121,21379,75,41.63999999999999,83.35999999999999,243.36000000000004,cont,6.815959786596476,76000718,3,0,1,2
6bba_48816121,21397,75,44.99999999999999,83.99999999999997,236.53333333333336,cont,1.5951600516179703,76000718,4,0,0,4
6bba_48816121,21405,75,48.0,56.0,227.99999999999997,cont,3.494694639736072,76000720,2,0,2,0
6bba_48816121,21427,75,52.52000000000001,45.92,228.64000000000001,cont,6.160314774830916,76000720,1,0,1,0
6bba_48816121,22551,80,40.84,98.56,240.64000000000004,cont,0.6516229738122908,81000782,1,1,0,0
6bba_48816121,22596,80,55.44,4.640000000000001,234.08000000000004,cont,1.5175272855866424,81000787,2,2,0,0
6bba_48816121,22784,81,40.84,100.0,235.99999999999994,cont,1.588515423280514,82000795,1,0,1,0
6bba_48816121,22835,81,55.800000000000004,5.279999999999999,228.00000000000003,cont,0.922679521827579,82000799,3,1,2,0
6bba_48816121,23071,82,56.72,6.720000000000001,221.92000000000002,cont,1.7168184601174352,83000812,1,0,1,0
6bba_48816121,23187,83,27.160000000000004,159.36,242.71999999999997,cont,3.003437873920479,84000819,1,1,0,0
6bba_48816121,23372,84,15.000000000000004,206.08,244.63999999999996,cont,1.923412494760295,85000829,4,2,2,0
6bba_48816121,23423,84,27.519999999999996,160.64,237.27999999999997,cont,2.4271321168201894,85000832,1,0,1,0
6bba_48816121,23892,86,27.679999999999996,162.72,227.36,cont,0.6508119928212762,87000858,1,0,1,0
6bba_48816121,24071,87,15.160000000000004,210.71999999999997,230.08000000000004,cont,1.2197246103116777,88000867,1,1,0,0
6bba_48816121,24310,88,15.360000000000003,211.36,223.99999999999997,cont,1.202609792285102,89000879,1,0,1,0
6bba_48816121,24911,90,44.84,96.0,150.72000000000003,cont,1.997164004907948,91000908,1,1,0,0
6bba_48816121,24915,90,47.480000000000004,71.35999999999999,152.8,cont,1.6875763871599998,91000910,1,1,0,0
6bba_48816121,24934,90,55.64,18.72,54.080000000000005,cont,2.1572754807858883,91000913,1,1,0,0
6bba_48816121,25124,91,44.64000000000001,95.36000000000001,142.56,cont,1.0898408714119596,92000921,1,0,1,0
6bba_48816121,25135,91,47.480000000000004,71.36,143.83999999999997,cont,1.310217658444574,92000923,1,0,1,0
6bba_48816121,25159,91,55.64,17.440000000000005,176.64000000000004,cont,2.6786881700750516,92000925,1,0,1,0
6bba_48816121,25164,91,55.839999999999996,18.720000000000006,43.199999999999996,cont,2.282532173924392,92000926,1,0,1,0
6bba_48816121,26018,95,39.839999999999996,105.92000000000002,145.44,cont,2.31561074557016,96000963,2,2,0,0
6bba_48816121,26056,95,48.37051025390626,70.12621704101564,122.84011352539065,cont,2.9243905328008335,96000965,1,1,0,0
6bba_48816121,26066,95,54.319999999999986,13.439999999999998,12.159999999999998,cont,1.7186631796253864,96000967,1,1,0,0
6bba_48816121,26238,96,39.67999999999999,105.92000000000002,135.20000000000002,cont,1.1693227206378847,97000974,2,0,2,0
6bba_48816121,26270,96,48.31999999999999,64.64,96.32000000000002,cont,3.445958003589129,97000977,2,0,1,1
6bba_48816121,26455,97,39.32,107.35999999999999,127.36,cont,0.5994906692351487,98000985,1,1,0,0
6bba_48816121,26512,97,55.99999999999999,16.0,117.28,cont,1.6788695973779364,98000990,1,1,0,0
6bba_48816121,26674,98,39.08,108.95999999999998,117.91999999999999,cont,1.738370285641126,99000995,1,0,1,0
6bba_48816121,26687,98,41.68,99.04000000000002,109.59999999999998,cont,5.8341799980802795,99000995,2,0,2,0
6bba_48816121,26728,98,56.0,15.999999999999998,105.28000000000002,cont,1.7061726172928702,99001000,1,0,1,0
6bba_4aa14c89,89,1,44.68000000000001,139.04000000000002,174.4,cont,2.5239864921389565,2000011,1,1,0,0
6bba_4aa14c89,139,2,43.16,144.00000000000003,174.08,cont,1.490223347354353,3000019,1,0,1,0
6bba_4aa14c89,1148,21,13.520000000000001,184.0,154.08,cont,4.124844125842814,22000178,1,0,0,1
6bba_4aa14c89,1262,23,56.32,82.72000000000001,143.99999999999997,cont,1.1827946250300583,24000195,1,1,0,0
6bba_4aa14c89,1312,24,55.84,83.35999999999999,143.99999999999997,cont,3.542984485218631,25000202,1,0,1,0
6bba_4aa14c89,2457,45,54.31999999999999,13.920000000000005,107.36,cont,2.742545578472689,46000356,1,1,0,0
6bba_4aa14c89,2521,46,54.0,16.0,107.36,cont,2.1377244034252865,47000363,1,0,1,0
6bba_4aa14c89,3825,63,60.44000000000001,155.52,71.52000000000001,cont,2.9819524853692756,64000454,2,1,0,1
6bba_4aa14c89,3923,64,58.0,161.44000000000003,72.16,cont,6.837311811852669,65000459,1,1,0,0
6bba_4aa14c89,3931,64,60.16,187.52000000000004,106.88000000000001,cont,1.728086459787255,65000460,1,1,0,0
6bba_4aa14c89,3938,64,60.67999999999999,156.64000000000001,70.56,cont,3.2944245039915567,65000459,2,1,1,0
6bba_4aa14c89,4027,65,58.2,158.56,67.2,cont,3.445076650090676,66000465,2,0,1,1
6bba_4aa14c89,4035,65,60.48000000000002,189.28000000000003,103.36,cont,1.7508573792573885,66000466,2,1,1,0
6bba_4aa14c89,4042,65,60.52,157.28000000000003,72.00000000000001,cont,3.7543478961065997,66000465,1,0,1,0
6bba_4aa14c89,4143,66,57.60000000000001,192.5333333333333,108.5333333333333,cont,3.9941260038614397,67000473,3,1,0,2
6bba_4aa14c89,4180,66,61.2,191.2,99.84,cont,3.6820147762468314,67000473,5,1,1,3
6bba_4aa14c89,4244,67,56.88000000000001,194.71999999999997,107.04,cont,3.592279264269396,68000481,1,0,1,0
6bba_4aa14c89,4290,67,61.36000000000001,193.44,97.28,cont,5.4173951593455,68000481,1,0,1,0
6bba_4aa14c89,4384,68,61.48,194.08,97.92000000000002,cont,5.0912575190221885,69000490,1,1,0,0
6bba_4aa14c89,4473,69,57.48000000000001,197.28,102.08000000000001,cont,2.742449293059018,70000499,1,1,0,0
6bba_4aa14c89,4496,69,61.47999999999999,194.72000000000003,99.84000000000003,cont,4.63606547893791,70000499,2,1,1,0
6bba_4aa14c89,4577,70,52.86666666666666,195.99999999999997,103.99999999999999,cont,6.874526814524134,71000507,5,0,0,5
6bba_4aa14c89,4599,70,58.2,198.72,98.56,cont,2.206951021771896,71000507,7,1,1,5
6bba_4aa14c89,4637,70,61.68,227.99999999999997,136.0,cont,5.635002356920533,71000510,1,1,0,0
6bba_4aa14c89,4714,71,58.52000000000001,199.36,97.28,cont,4.249429373457111,72000519,1,0,1,0
6bba_4aa14c89,4732,71,61.32,227.99999999999997,136.0,cont,6.1326044630319965,72000521,1,0,1,0
6bba_4aa14c89,4749,71,61.99999999999999,211.99999999999994,121.06666666666663,cont,6.561548872873794,72000521,3,0,0,3
6bba_4aa14c89,4822,72,57.00000000000001,228.64000000000007,124.00000000000003,cont,6.678453460570039,73000532,1,1,0,0
6bba_4aa14c89,4828,72,58.31999999999999,200.0,95.36000000000001,cont,4.576907047887232,73000529,1,1,0,0
6bba_4aa14c89,4865,72,62.0,212.64,126.72000000000001,cont,5.489758931410386,73000532,1,1,0,0
6bba_4aa14c89,4952,73,58.2,200.0,95.36000000000001,cont,4.417669422331189,74000540,1,0,1,0
6bba_4aa14c89,4982,73,62.000000000000014,214.55999999999995,131.84,cont,5.790155708182665,74000543,1,0,1,0
6bba_4aa14c89,4985,73,61.99999999999999,221.60000000000002,112.5333333333333,cont,4.756962185692793,74000543,6,0,0,6
6bba_4aa14c89,5101,74,62.0,224.48000000000005,111.04000000000002,cont,6.417618571167961,75000554,1,1,0,0
6bba_4aa14c89,5226,75,62.000000000000014,224.0,109.28,cont,5.892143338802965,76000567,1,0,1,0
6bba_4aa14c89,5282,76,52.16,144.64,68.0,cont,3.6176602417861266,77000575,1,1,0,0
6bba_4aa14c89,5409,77,51.84,143.99999999999997,67.36000000000001,cont,4.0212430059000495,78000589,1,0,1,0
6bba_4aa14c89,5580,78,60.68000000000001,77.28,10.720000000000002,cont,0.7441364542205815,79000609,1,1,0,0
6bba_4aa14c89,5710,79,60.31999999999999,78.72000000000001,9.28,cont,1.2557791406135221,80000623,2,1,1,0
6bba_4aa14c89,5734,79,61.99999999999999,164.0,116.0,cont,5.585014128227426,80000624,2,0,0,2
6bba_4aa14c89,5784,80,46.56,199.68000000000006,96.32,cont,6.570488114478246,81000630,1,1,0,0
6bba_4aa14c89,5804,80,53.279999999999994,203.36,91.2,cont,5.41066438272601,81000630,1,1,0,0
6bba_4aa14c89,5817,80,56.16000000000001,245.28000000000003,106.08,cont,6.712528631223862,81000632,1,1,0,0
6bba_4aa14c89,5826,80,58.31999999999999,240.00000000000006,132.00000000000003,cont,6.799299826085045,81000632,1,1,0,0
6bba_4aa14c89,5841,80,60.16000000000001,79.36,8.64,cont,0.9037396264964854,81000636,2,1,1,0
6bba_4aa14c89,5847,80,61.160000000000004,152.0,108.64000000000001,cont,4.730313315468646,81000637,1,1,0,0
6bba_4aa14c89,5913,81,45.73333333333332,200.0,95.99999999999997,cont,6.432847563698201,82000641,1,0,1,0
6bba_4aa14c89,5929,81,49.99999999999999,215.46666666666664,79.99999999999997,cont,5.907892555255582,82000641,5,0,0,5
6bba_4aa14c89,5978,81,60.000000000000014,80.00000000000003,8.0,cont,1.7235727791421995,82000647,1,0,1,0
6bba_4aa14c89,5985,81,59.388626708984376,236.24688476562503,110.24266723632813,cont,6.409864952353395,82000646,3,2,1,0
6bba_4aa14c89,5987,81,61.00000000000001,154.56,109.28,cont,2.058368968989766,82000649,1,0,1,0
6bba_4aa14c89,6120,82,58.839999999999996,239.36,111.36000000000001,cont,5.631346003621154,83000659,3,1,2,0
6bba_4aa14c89,6131,82,60.91714416503906,158.54920166015626,110.74123535156251,cont,1.0988799665960716,83000664,1,1,0,0
6bba_4aa14c89,6147,82,61.84,172.79999999999998,116.0,cont,5.512583474424664,83000664,1,1,0,0
6bba_4aa14c89,6225,83,51.84000000000001,86.71999999999998,44.00000000000001,cont,2.308415311420352,84000673,1,0,1,0
6bba_4aa14c89,6254,83,58.16000000000001,4.640000000000001,29.279999999999994,cont,3.8749010068000613,84000671,1,1,0,0
6bba_4aa14c89,6259,83,58.480000000000004,242.08,111.36,cont,6.570990459968725,84000672,1,0,1,0
6bba_4aa14c89,6279,83,60.75714416503907,160.62920166015627,111.38123535156252,cont,1.8903022319825213,84000675,1,0,1,0
6bba_4aa14c89,6301,83,61.68,172.64,116.0,cont,4.84252726502391,84000675,1,0,1,0
6bba_4aa14c89,6413,84,58.32000000000001,4.640000000000001,28.640000000000008,cont,4.42005974224332,85000684,1,0,1,0
6bba_4aa14c89,6588,85,59.597144165039055,163.02920166015625,112.02123535156252,cont,2.3140887539710153,86000702,1,1,0,0
6bba_4aa14c89,6593,85,60.839999999999996,175.2,117.28,cont,5.258827400191033,86000702,1,1,0,0
6bba_4aa14c89,6673,86,46.39999999999999,207.99999999999997,89.06666666666665,cont,2.5810288780725354,87000713,4,1,0,3
6bba_4aa14c89,6678,86,47.480000000000004,221.92000000000002,80.64,cont,6.591052884403221,87000713,2,1,1,0
6bba_4aa14c89,6834,87,45.35999999999998,208.00000000000006,86.39999999999998,cont,1.6408477229773684,88000727,1,0,1,0
6bba_4aa14c89,6840,87,47.519999999999996,224.16,81.44,cont,6.346160748830742,88000727,4,0,1,3
6bba_4aa14c89,7631,92,44.0,210.71999999999997,83.36000000000003,cont,1.090688572645752,93000791,1,1,0,0
6bba_4aa14c89,7794,93,43.68,211.36,82.72,cont,2.2283247631348515,94000803,1,0,1,0
6bba_4aa14c89,8294,96,45.08,214.72,86.71999999999998,cont,5.313948773040624,97000836,1,0,1,0
6bba_4f99ce20,87,1,26.000000000000007,233.60000000000002,154.24000000000007,cont,1.118302284715565,2000014,1,1,0,0
6bba_4f99ce20,115,1,38.68,151.04000000000002,220.80000000000004,cont,1.5859367302953662,2000020,1,0,1,0
6bba_4f99ce20,123,1,43.32000000000002,210.71999999999997,222.08,cont,1.898678816967194,2000021,1,1,0,0
6bba_4f99ce20,157,2,24.52,221.44,126.88,cont,3.184709808208594,3000026,2,0,2,0
6bba_4f99ce20,193,2,42.999999999999986,204.64000000000001,210.56,cont,1.1776486105795765,3000033,1,0,1,0
6bba_4f99ce20,688,10,30.000000000000007,114.07999999999998,42.24000000000001,cont,4.071103390052869,11000131,1,1,0,0
6bba_4f99ce20,753,11,29.52000000000001,111.84,33.12,cont,3.382538109541404,12000145,1,0,1,0
6bba_4f99ce20,756,11,32.800000000000004,92.96000000000002,55.36,cont,2.9309522769912086,12000147,1,0,0,1
6bba_4f99ce20,757,11,31.52,124.80000000000001,117.60000000000001,cont,5.355823495271289,12000148,1,0,0,1
6bba_4f99ce20,882,13,33.31999999999999,80.96000000000001,17.440000000000005,cont,2.8850500970520456,14000175,2,0,2,0
6bba_4f99ce20,885,13,32.72,101.44,93.60000000000001,cont,3.3095789007062466,14000177,2,0,0,2
6bba_4f99ce20,1143,17,33.36,90.08,80.96000000000001,cont,3.4500552531807362,18000216,1,1,0,0
6bba_4f99ce20,1171,17,55.84,146.07999999999998,242.07999999999996,cont,3.9324999999999997,18000222,1,1,0,0
6bba_4f99ce20,1206,18,32.64,87.99999999999999,73.12,cont,1.4538983157360117,19000226,1,0,1,0
6bba_4f99ce20,1237,18,55.32000000000001,144.64000000000001,237.28000000000003,cont,1.621258191960785,19000233,1,0,1,0
6bba_4f99ce20,1575,24,31.199999999999996,79.36000000000001,46.88000000000001,cont,3.5365690029320773,25000297,1,1,0,0
6bba_4f99ce20,1632,25,30.320000000000007,76.64000000000001,38.4,cont,2.8288538425482455,26000306,1,0,1,0
6bba_4f99ce20,2139,33,51.48,97.92000000000002,175.20000000000005,cont,3.20636013456068,34000395,1,0,1,0
6bba_4f99ce20,2259,35,51.16,89.44000000000001,165.44000000000005,cont,4.154823439389433,36000417,1,1,0,0
6bba_4f99ce20,3010,48,55.99999999999999,6.72,95.36000000000001,cont,2.015262079730581,49000564,1,1,0,0
6bba_4f99ce20,3072,50,13.0,228.0,133.28000000000003,cont,1.840200396695967,51000579,1,0,1,0
6bba_4f99ce20,3137,51,21.52,230.08000000000004,231.36,cont,1.0917775070956444,52000590,2,2,0,0
6bba_4f99ce20,3179,52,15.679999999999998,244.00000000000003,192.0,cont,2.183131709837957,53000603,1,0,1,0
6bba_4f99ce20,3186,52,21.160000000000004,224.64000000000004,227.99999999999997,cont,1.5723082156498451,53000601,2,0,2,0
6bba_4f99ce20,3293,54,15.319999999999997,230.24,181.44,cont,2.100340002237737,55000628,1,1,0,0
6bba_4f99ce20,3313,54,27.519999999999996,179.84000000000003,252.0,cont,4.321400236034625,55000632,1,1,0,0
6bba_4f99ce20,3353,55,15.320000000000004,220.48,175.20000000000005,cont,1.9897459793903385,56000640,1,0,1,0
6bba_4f99ce20,3369,55,28.86666666666666,156.53333333333327,247.46666666666667,cont,2.3261729792014125,56000643,1,0,0,1
6bba_4f99ce20,3943,65,18.16,150.88000000000002,166.08,cont,2.5158128035487772,66000793,1,1,0,0
6bba_4f99ce20,4006,66,18.36,143.20000000000002,164.00000000000003,cont,1.7136624704124266,67000809,1,0,1,0
6bba_4f99ce20,4719,77,43.68,0.0,226.08,cont,2.7923533981213744,78000998,1,1,0,0
6bba_4f99ce20,4769,78,25.680000000000003,114.71999999999997,242.72000000000003,cont,2.393609003680422,79001008,1,1,0,0
6bba_4f99ce20,4838,79,25.159999999999997,113.28000000000002,240.64000000000004,cont,1.403627554766583,80001025,1,0,1,0
6bba_4f99ce20,5007,82,6.840000000000001,143.52,40.8,cont,1.9020137354919324,83001068,1,1,0,0
6bba_4f99ce20,5020,82,12.360000000000003,131.36,103.52000000000001,cont,1.1936838986934466,83001069,1,1,0,0
6bba_4f99ce20,5072,83,6.999999999999999,136.48,37.27999999999999,cont,1.9814341397079,84001085,1,0,1,0
6bba_4f99ce20,5079,83,11.64,129.92000000000002,97.12,cont,1.6972500736485436,84001086,1,0,1,0
6bba_4f99ce20,5557,91,3.3200000000000003,194.56000000000006,30.72,cont,1.4528990415717091,92001207,1,1,0,0
6bba_4f99ce20,5617,92,3.1599999999999993,192.64000000000004,32.0,cont,1.4477099372802595,93001222,1,0,1,0
6bba_4ffd3da3,438,1,53.600000000000016,70.72,176.31999999999996,cont,2.393112513025646,2000012,1,1,0,0
6bba_4ffd3da3,825,3,20.52,180.64000000000001,4.64,cont,1.1040437038450952,4000021,1,1,0,0
6bba_4ffd3da3,1060,4,19.840000000000003,180.64000000000001,4.640000000000001,cont,1.3972165410200357,5000027,1,0,1,0
6bba_4ffd3da3,1791,7,18.8,179.04,3.9999999999999996,cont,1.183129457202381,8000045,2,2,0,0
6bba_4ffd3da3,2014,8,15.160000000000004,179.36000000000004,3.3600000000000003,cont,6.355557607322899,9000051,3,0,1,2
6bba_4ffd3da3,2034,8,19.59999999999999,176.5333333333333,3.999999999999999,cont,1.402069403100679,9000051,2,0,2,0
6bba_4ffd3da3,2208,9,4.0,142.72,116.64000000000001,cont,0.5401750295043282,10000055,1,1,0,0
6bba_4ffd3da3,2390,9,57.36,70.56,181.44,cont,2.4530503969547737,10000060,1,0,1,0
6bba_4ffd3da3,2445,10,3.6799999999999997,143.36,115.36000000000001,cont,1.0097253648888882,11000061,1,0,1,0
6bba_4ffd3da3,3355,13,57.266666666666666,72.0,175.99999999999994,cont,2.098040794076971,14000077,1,0,0,1
6bba_4ffd3da3,3371,13,59.866666666666646,76.0,180.53333333333325,cont,6.855508037438827,14000077,1,0,1,0
6bba_4ffd3da3,3547,14,39.52000000000001,75.2,200.16,cont,1.959793117270287,15000080,1,1,0,0
6bba_4ffd3da3,3792,15,39.199999999999996,76.64000000000001,196.8,cont,5.06053804575166,16000084,4,0,1,3
6bba_4ffd3da3,3797,15,42.13333333333333,75.46666666666667,204.5333333333333,cont,1.0686768980847186,16000084,2,0,0,2
6bba_4ffd3da3,4105,16,59.36,75.36,174.08,cont,5.7079151458741855,17000089,1,0,1,0
6bba_4ffd3da3,5523,22,53.8,21.280000000000005,237.27999999999997,cont,3.588749129223166,23000123,1,0,1,0
6bba_4ffd3da3,5774,23,53.88000000000001,24.0,235.99999999999994,cont,3.4688707186200896,24000130,1,1,0,0
6bba_4ffd3da3,6026,24,53.2,26.080000000000002,234.72000000000006,cont,4.57468526922891,25000135,1,0,1,0
6bba_4ffd3da3,6099,25,3.4799999999999995,157.92000000000002,27.36,cont,0.9598510626654543,26000137,1,1,0,0
6bba_4ffd3da3,6273,25,52.239999999999995,27.679999999999996,232.95999999999998,cont,2.889394400216085,26000140,1,1,0,0
6bba_4ffd3da3,6351,26,3.1599999999999993,157.92000000000002,25.28,cont,0.28564893225776156,27000142,1,0,1,0
6bba_4ffd3da3,6526,26,50.866666666666646,28.0,231.46666666666658,cont,5.160636602095638,27000145,1,0,1,0
6bba_4ffd3da3,6534,26,54.73333333333331,31.999999999999993,232.0,cont,1.4422987535335257,27000145,7,0,0,7
6bba_4ffd3da3,6862,28,6.999999999999997,148.53333333333336,91.99999999999999,cont,5.048263590362311,29000152,4,0,0,4
6bba_4ffd3da3,7008,28,52.68000000000001,33.92,231.36,cont,4.007989287036563,29000157,1,1,0,0
6bba_4ffd3da3,7249,29,51.64,36.64,230.08000000000004,cont,2.357986429138207,30000165,2,1,1,0
6bba_4ffd3da3,7490,30,51.32,39.36,228.00000000000003,cont,1.1863612961488672,31000173,1,0,1,0
6bba_4ffd3da3,7927,32,37.160000000000004,100.63999999999999,161.28,cont,3.1827192088684138,33000184,1,1,0,0
6bba_4ffd3da3,8173,33,36.84000000000001,102.72000000000003,159.20000000000005,cont,2.1499800144419825,34000190,1,0,1,0
6bba_4ffd3da3,10986,45,2.6399999999999997,26.720000000000002,249.27999999999997,cont,2.091710200410176,46000252,1,1,0,0
6bba_4ffd3da3,11099,45,32.0,119.36000000000001,141.92000000000002,cont,0.7639228527803014,46000255,1,1,0,0
6bba_4ffd3da3,11229,46,2.64,28.639999999999997,248.0,cont,1.1435197855743477,47000258,1,0,1,0
6bba_4ffd3da3,11718,48,4.960000000000001,35.839999999999996,243.36000000000004,cont,1.848361605584784,49000269,1,0,1,0
6bba_4ffd3da3,12201,50,2.2,78.72,232.64000000000004,cont,3.4473370610516136,51000281,1,1,0,0
6bba_4ffd3da3,12220,50,6.880000000000001,5.28,250.72000000000003,cont,6.732326167455048,51000283,1,1,0,0
6bba_4ffd3da3,12223,50,8.240000000000002,45.44,234.72000000000006,cont,1.3927681519190425,51000282,1,1,0,0
6bba_4ffd3da3,12425,51,1.32,82.08000000000001,228.64000000000004,cont,3.6799702784125783,52000289,1,0,1,0
6bba_4ffd3da3,12462,51,8.000000000000002,6.080000000000001,250.72000000000003,cont,4.945662872659232,52000291,1,0,1,0
6bba_4ffd3da3,12466,51,8.520000000000001,48.16,229.92,cont,3.0237228708993853,52000290,1,0,1,0
6bba_4ffd3da3,13621,56,0.0,95.68,209.92,cont,2.257112715949292,57000330,1,1,0,0
6bba_4ffd3da3,13866,57,0.0,100.0,207.46666666666658,cont,1.7403017478618898,58000338,1,0,1,0
6bba_4ffd3da3,14148,58,7.800000000000002,21.279999999999998,235.36000000000004,cont,1.3576289073233505,59000349,1,1,0,0
6bba_4ffd3da3,14293,58,51.04000000000001,67.36,195.35999999999999,cont,0.7181323433044992,59000355,1,0,1,0
6bba_4ffd3da3,14375,59,2.84,48.800000000000004,206.55999999999997,cont,6.462864230548248,60000357,1,1,0,0
6bba_4ffd3da3,14389,59,7.480000000000001,26.560000000000002,230.72000000000008,cont,2.5833922708330523,60000356,1,0,1,0
6bba_4ffd3da3,14625,60,7.480000000000001,33.440000000000005,225.92,cont,2.6912767067880656,61000366,2,0,0,2
6bba_4ffd3da3,14853,61,5.4,64.53333333333333,204.5333333333333,cont,2.2175280281330365,62000372,3,0,0,3
6bba_4ffd3da3,15338,63,8.16,68.0,198.72000000000003,cont,2.447230577612161,64000387,1,1,0,0
6bba_4ffd3da3,15838,65,14.133333333333331,163.99999999999994,59.99999999999997,cont,1.169297628683296,66000406,2,0,0,2
6bba_4ffd3da3,16017,66,0.32,125.92,159.35999999999999,cont,1.1756288210995849,67000413,1,0,1,0
6bba_4ffd3da3,16019,66,1.2666666666666662,111.46666666666667,155.99999999999994,cont,5.648027710085085,67000413,5,0,0,5
6bba_4ffd3da3,16064,66,13.800000000000002,53.280000000000015,209.28000000000003,cont,1.4303692705032562,67000414,2,1,1,0
6bba_4ffd3da3,16532,68,15.000000000000004,165.28000000000003,45.28,cont,0.8710060634117223,69000437,1,1,0,0
6bba_4ffd3da3,16983,70,18.04,57.44000000000001,202.72000000000006,cont,1.9217643260816266,71000462,1,1,0,0
6bba_4ffd3da3,17192,71,18.36,56.0,201.44,cont,3.0592324404987625,72000474,1,0,1,0
6bba_4ffd3da3,17377,72,4.319999999999999,85.28,235.99999999999994,cont,1.1430578506794837,73000482,2,2,0,0
6bba_4ffd3da3,17601,73,4.160000000000001,88.64,232.0,cont,2.993574629519029,74000495,2,0,2,0
6bba_4ffd3da3,17636,73,15.399999999999999,174.56000000000006,17.279999999999998,cont,1.1684190707961029,74000498,1,1,0,0
6bba_4ffd3da3,17637,73,15.84,79.19999999999999,168.0,cont,1.9554766842128308,74000499,1,1,0,0
6bba_4ffd3da3,17853,74,16.16,82.08,170.07999999999998,cont,4.779655126157115,75000512,2,0,1,1
6bba_4ffd3da3,17856,74,15.866666666666664,178.93333333333328,11.466666666666661,cont,1.3498263777241715,75000511,1,0,1,0
6bba_4ffd3da3,17936,74,44.533333333333324,120.53333333333336,116.5333333333333,cont,0.8111447029764214,75000517,4,0,0,4
6bba_4ffd3da3,18282,76,13.747882843017578,85.26352172851561,164.0799072265625,cont,6.918936326168805,77000536,1,0,1,0
6bba_4ffd3da3,18495,77,14.707760620117192,84.47650634765625,163.6243408203125,cont,3.268638694344506,78000549,3,0,0,3
6bba_4ffd3da3,18960,79,21.84,59.52000000000001,182.56000000000003,cont,4.004265108918237,80000574,2,0,0,2
6bba_4ffd3da3,19008,79,37.519999999999996,59.36,237.92,cont,2.419231541213044,80000576,1,1,0,0
6bba_4ffd3da3,19261,80,52.480000000000004,35.99999999999999,238.08000000000004,cont,5.855820021568282,81000590,1,1,0,0
6bba_4ffd3da3,19498,81,58.0,32.0,244.0,cont,5.658407737606049,82000601,1,0,1,0
6bba_4ffd3da3,19601,82,18.133333333333333,91.99999999999999,163.99999999999997,cont,1.465756405504604,83000606,6,0,0,6
6bba_4ffd3da3,20268,85,21.879999999999995,54.080000000000005,186.07999999999998,cont,2.33242698170811,86000644,3,0,0,3
6bba_4ffd3da3,20570,86,52.36,54.720000000000006,217.28000000000003,cont,2.6834157919524935,87000659,1,0,1,0
6bba_4ffd3da3,20808,87,58.640000000000015,28.000000000000004,240.64000000000004,cont,0.7270853629388843,88000672,1,1,0,0
6bba_4ffd3da3,21161,89,34.80000000000001,88.00000000000001,201.44,cont,2.2055147500073367,90000690,1,1,0,0
6bba_4ffd3da3,21366,90,34.8,91.36000000000001,196.64000000000001,cont,2.133458533110028,91000702,1,0,1,0
6bba_4ffd3da3,21443,90,58.839999999999996,38.080000000000005,227.36,cont,0.8069560861657769,91000705,1,1,0,0
6bba_4ffd3da3,21664,91,58.640000000000015,41.28000000000001,221.28000000000003,cont,2.248379972335615,92000718,1,0,1,0
6bba_4ffd3da3,21759,92,20.28,117.28,106.72,cont,1.5206562111470192,93000722,1,1,0,0
6bba_4ffd3da3,21971,93,20.48,120.0,97.28000000000003,cont,1.2081959692036686,94000734,1,0,1,0
6bba_4ffd3da3,22204,94,29.266666666666666,119.99999999999994,160.53333333333327,daughter,2.8521206169951685,95000758,2,0,0,2
6bba_4ffd3da3,22634,96,36.31999999999999,102.72000000000003,165.27999999999997,cont,1.0683061944030618,97000784,1,0,1,0
6bba_4ffd3da3,22928,98,1.6799999999999997,154.07999999999998,98.72,cont,2.818192295160142,99000807,1,1,0,0
6bba_55b7eebe,2639,48,53.32000000000001,70.39999999999999,33.279999999999994,parent,2.2145954060504986,49000351,1,1,0,0
6bba_55b7eebe,2691,49,52.999999999999986,47.999999999999986,35.466666666666654,daughter,1.4917975966747146,50000359,2,0,0,2
6bba_55b7eebe,2692,49,53.68,74.72,34.72,daughter,2.8551948269776584,50000360,1,0,1,0
6bba_55b7eebe,2998,55,56.160000000000004,85.92,41.28000000000001,cont,3.0241594906684353,56000418,1,1,0,0
6bba_55b7eebe,3042,56,56.160000000000004,85.91999999999999,45.28,cont,1.4451540791555695,57000430,1,0,1,0
6bba_55b7eebe,4372,85,49.32,156.0,133.44,cont,1.946272478996712,86000645,1,1,0,0
6bba_55b7eebe,4384,85,58.31999999999999,87.84,44.16,cont,1.6335897970420994,86000647,1,1,0,0
6bba_55b7eebe,4419,86,49.68,155.36,130.56,cont,2.3038924264166485,87000653,1,0,1,0
6bba_55b7eebe,4432,86,58.0,71.46666666666665,31.999999999999993,cont,2.7737617670372874,87000655,1,0,0,1
6bba_55b7eebe,4434,86,58.680000000000014,92.80000000000001,39.199999999999996,cont,6.641639598961997,87000655,1,0,1,0
6bba_55c70843,1460,14,32.24,18.72,246.55999999999997,cont,3.8845998024249497,15000004,3,0,0,3
6bba_55c70843,2417,18,32.8,12.32,248.80000000000007,cont,5.225050986593334,19000016,6,1,5,0
6bba_55c70843,9003,34,23.840000000000003,29.279999999999994,237.27999999999997,cont,5.870929573968666,35000103,2,1,1,0
6bba_55c70843,9528,35,21.679999999999996,24.64,227.99999999999997,cont,0.6758906161502701,36000112,1,0,1,0
6bba_55c70843,9550,35,23.68,32.64,236.64000000000001,cont,5.694135415056444,36000112,1,0,1,0
6bba_55c70843,10062,36,21.000000000000004,27.36,227.99999999999997,cont,0.1462500000000002,37000125,1,1,0,0
6bba_55c70843,10121,36,25.84,35.99999999999999,181.43999999999997,cont,4.407076695497824,37000128,1,1,0,0
6bba_55c70843,11167,38,22.48,10.719999999999999,81.92,cont,5.932518699928385,39000149,1,0,1,0
6bba_55c70843,12438,40,36.84000000000001,32.64,243.2,cont,2.811625697439105,41000183,1,1,0,0
6bba_55c70843,12825,41,19.159999999999997,40.0,225.28000000000003,cont,0.28379404944430997,42000190,1,1,0,0
6bba_55c70843,12827,41,18.2,45.919999999999995,212.0,cont,5.946900584548223,42000190,2,2,0,0
6bba_55c70843,12836,41,19.840000000000003,27.200000000000003,214.72,cont,6.807663994535863,42000190,2,2,0,0
6bba_55c70843,12863,41,22.039999999999996,19.200000000000003,82.56000000000002,cont,6.504812281111573,42000200,4,4,0,0
6bba_55c70843,12877,41,24.0,21.279999999999998,100.0,cont,1.820362687076397,42000200,3,3,0,0
6bba_55c70843,13394,42,17.119999999999997,32.79999999999999,107.36,cont,6.136392466262245,43000200,3,0,3,0
6bba_55c70843,13403,42,17.64,50.080000000000005,212.0,cont,6.235110388958644,43000201,2,0,2,0
6bba_55c70843,13432,42,19.0,42.72,224.64000000000004,cont,0.5401750295043208,43000201,1,0,1,0
6bba_55c70843,13440,42,20.800000000000004,24.8,83.36,cont,6.444021454030082,43000200,4,0,4,0
6bba_55c70843,13472,42,23.159999999999997,25.44,100.0,cont,5.5190703474407625,43000200,3,0,3,0
6bba_55c70843,13587,42,32.68,39.36,235.99999999999994,cont,0.581377674149948,43000205,9,2,2,5
6bba_55c70843,13743,42,47.84,92.80000000000001,227.99999999999997,cont,3.6000627164675905,43000207,1,0,1,0
6bba_55c70843,13983,43,16.839999999999996,36.63999999999999,107.36000000000001,cont,5.659410995191999,44000211,1,1,0,0
6bba_55c70843,14172,43,32.04,60.640000000000015,240.00000000000006,cont,6.359794120095411,44000228,1,0,1,0
6bba_55c70843,14186,43,33.519999999999996,42.72,238.56,cont,1.8897569288138614,44000228,2,0,2,0
6bba_55c70843,14883,44,46.47999999999999,135.20000000000002,236.64000000000001,cont,5.360283632887344,45000233,1,1,0,0
6bba_55c70843,15191,45,19.48,35.36,84.0,cont,6.556326262664176,46000237,1,0,1,0
6bba_55c70843,15473,45,45.88729553222657,137.76099365234376,237.22479980468748,cont,5.357559539711627,46000245,1,0,1,0
6bba_55c70843,16469,47,27.680000000000007,56.00000000000001,232.0,cont,5.634416562520017,48000268,1,1,0,0
6bba_55c70843,17087,48,28.193540039062498,75.75467285156252,243.00274658203125,cont,6.631105894331006,49000282,1,0,1,0
6bba_55c70843,17130,48,31.360000000000007,65.28,237.27999999999997,cont,2.5971546931209244,49000282,5,0,0,5
6bba_55c70843,17817,49,45.48000000000001,123.35999999999999,233.92000000000002,cont,3.2657648893482905,50000301,5,5,0,0
6bba_55c70843,18389,50,40.86666666666665,115.46666666666664,243.99999999999997,cont,6.155309431814858,51000317,5,1,0,4
6bba_55c70843,18420,50,45.160000000000004,128.64000000000001,237.44000000000005,cont,4.418476299868555,51000317,6,1,5,0
6bba_55c70843,18436,50,46.46666666666666,116.53333333333333,232.0,cont,6.268665530615119,51000317,6,1,0,5
6bba_55c70843,18981,51,40.32000000000001,116.96000000000004,244.0,cont,6.365044309547249,52000333,1,0,1,0
6bba_55c70843,20144,53,39.67999999999999,120.0,244.64000000000001,cont,6.134886154200427,54000358,1,0,1,0
6bba_55c70843,20915,55,3.0,88.64,211.36,cont,1.1675147215345902,56000374,2,1,1,0
6bba_55c70843,21677,56,18.733333333333327,28.53333333333333,159.46666666666664,cont,6.491191948838557,57000390,5,0,0,5
6bba_55c70843,21931,56,42.32000000000001,126.08000000000001,238.07999999999998,cont,5.017330590064804,57000397,2,2,0,0
6bba_55c70843,46361,56,38.54734069824218,125.32603027343751,249.3806323242188,cont,5.7019545549588075,57000397,1,1,0,0
6bba_55c70843,22410,57,30.839999999999996,32.0,42.720000000000006,cont,1.4538983157360037,58000408,1,0,1,0
6bba_55c70843,23732,60,2.3600000000000003,97.28000000000003,221.92000000000002,cont,1.6610806911465839,61000437,2,2,0,0
6bba_55c70843,23872,60,15.840000000000002,69.92,220.16,cont,6.997431894988335,61000438,7,7,0,0
6bba_55c70843,24303,61,3.5200000000000005,111.36,241.11999999999998,cont,3.4344443236861326,62000450,2,0,1,1
6bba_55c70843,24379,61,11.64,106.56,221.92000000000004,cont,6.790011505881259,62000451,7,0,2,5
6bba_55c70843,24422,61,14.320000000000002,82.56000000000002,229.92,cont,5.377841690678518,62000451,9,0,4,5
6bba_55c70843,24724,61,39.0,154.71999999999997,243.84000000000003,cont,4.418685466289713,62000460,10,0,1,9
6bba_55c70843,24732,61,39.199999999999996,132.00000000000003,246.55999999999997,cont,5.225227865844688,62000460,7,0,0,7
6bba_55c70843,26087,64,2.84,119.36000000000001,244.64000000000001,cont,1.7814328853482055,65000492,2,0,2,0
6bba_55c70843,26298,64,19.92,30.080000000000002,156.0,cont,1.2813823102415602,65000496,4,0,0,4
6bba_55c70843,27161,65,55.73333333333333,83.99999999999997,232.0,cont,2.3550044600689124,66000519,5,0,0,5
6bba_55c70843,27423,66,17.64,46.08000000000001,158.08000000000004,cont,6.607358705451074,67000525,1,1,0,0
6bba_55c70843,28000,67,18.32,44.63999999999999,157.28,cont,3.8729583331092003,68000536,1,0,1,0
6bba_55c70843,28601,68,22.960000000000004,83.35999999999999,92.0,cont,0.48668682178583944,69000548,3,2,1,0
6bba_55c70843,29219,69,26.52,91.36,247.36,cont,5.171126871872316,70000562,1,0,1,0
6bba_55c70843,29639,70,12.520000000000001,119.36000000000001,246.88,cont,6.336854133755966,71000569,1,1,0,0
6bba_55c70843,30155,71,11.64,120.00000000000003,241.76000000000005,cont,2.232941585666776,72000583,5,0,1,4
6bba_55c70843,30156,71,10.680000000000001,122.72000000000001,223.99999999999997,cont,5.352593121562683,72000583,6,0,1,5
6bba_55c70843,30332,71,26.2,108.0,78.56,cont,5.800612710093649,72000586,8,0,2,6
6bba_55c70843,30527,71,53.639999999999986,85.28,232.0,cont,2.140131786245875,72000590,1,0,1,0
6bba_55c70843,30775,72,16.0,50.080000000000005,143.36,cont,6.747371015810521,73000593,1,0,1,0
6bba_55c70843,31554,73,36.160000000000004,156.0,248.63999999999996,cont,3.101877707534579,74000612,1,0,1,0
6bba_55c70843,32780,75,52.96,88.00000000000001,235.35999999999999,cont,1.9850291244714744,76000640,1,1,0,0
6bba_55c70843,33175,76,27.520000000000003,168.64000000000004,127.36,cont,4.279060388975588,77000649,2,0,2,0
6bba_55c70843,33209,76,32.64,105.28000000000002,116.0,cont,1.0430424248322785,77000651,2,1,1,0
6bba_55c70843,33777,77,32.84,106.72,116.0,cont,2.6851865042302028,78000667,2,1,1,0
6bba_55c70843,34247,78,24.866666666666667,159.99999999999994,39.466666666666654,cont,4.80055425764103,79000683,4,1,0,3
6bba_55c70843,34352,78,36.68000000000001,156.0,252.0,cont,1.9651081395180328,79000686,1,0,1,0
6bba_55c70843,34855,79,29.775895690917963,179.52489257812496,128.73948730468751,cont,5.681414039536302,80000702,1,0,1,0
6bba_55c70843,35431,80,34.519999999999996,107.36000000000001,107.84000000000002,cont,1.990276755252912,81000719,7,1,0,6
6bba_55c70843,35458,80,37.68,92.64000000000001,108.64000000000001,cont,6.908849510772394,81000719,2,1,1,0
6bba_55c70843,35477,80,40.64000000000001,140.64000000000004,17.28,cont,2.9193974121383466,81000723,1,1,0,0
6bba_55c70843,35967,81,35.0,108.0,105.92,cont,0.8943407138781063,82000739,2,1,1,0
6bba_55c70843,36013,81,40.32,141.28000000000003,20.0,cont,2.968595259125093,82000740,2,1,1,0
6bba_55c70843,37033,83,35.52,108.64000000000001,104.0,cont,1.7509327871166345,84000772,2,2,0,0
6bba_55c70843,37557,84,31.999999999999993,167.99999999999994,79.46666666666665,cont,4.9769263614147015,85000794,4,0,0,4
6bba_55c70843,40103,89,19.0,60.000000000000014,157.28000000000003,cont,2.0344325068677063,90000856,1,1,0,0
6bba_55c70843,40646,90,19.0,60.000000000000014,158.72,cont,0.4218745370367888,91000870,1,0,1,0
6bba_55c70843,40692,90,22.040000000000003,134.72,207.36,cont,4.8231576793631765,91000872,2,1,1,0
6bba_55c70843,41388,91,34.839999999999996,109.28,108.0,cont,0.9620494075150239,92000885,2,0,2,0
6bba_57b7cc1e,1169,1,16.240000000000002,110.4,131.99999999999997,cont,1.258824476446176,2000040,1,0,1,0
6bba_57b7cc1e,2632,2,48.16000000000001,116.63999999999999,119.36000000000001,cont,1.6882021687286035,3000085,1,0,1,0
6bba_57b7cc1e,3966,4,4.999999999999998,72.53333333333333,43.999999999999986,cont,6.925553089124208,5000128,3,1,0,2
6bba_57b7cc1e,4193,4,21.32,125.28,227.36000000000004,cont,3.1068538145525797,5000135,4,0,0,4
6bba_57b7cc1e,5239,5,27.999999999999996,47.360000000000014,125.28000000000002,cont,4.582672869079797,6000167,1,0,1,0
6bba_57b7cc1e,8573,8,54.84,79.36,16.8,cont,5.592385028992541,9000263,2,2,0,0
6bba_57b7cc1e,9630,9,59.92,38.56000000000001,161.28,cont,5.554410691063459,10000297,2,1,1,0
6bba_57b7cc1e,11297,11,38.52,62.239999999999995,100.64,cont,6.858212344518061,12000350,1,0,1,0
6bba_57b7cc1e,13175,13,29.839999999999996,108.0,232.0,daughter,6.618379829119813,14000412,1,1,0,0
6bba_57b7cc1e,13403,13,44.68,91.36000000000001,85.28,cont,6.522892499689068,14000415,1,1,0,0
6bba_57b7cc1e,14155,14,28.36,62.72000000000001,4.0,cont,1.168419070796088,15000445,1,0,1,0
6bba_57b7cc1e,14502,14,52.16000000000001,80.79999999999998,33.92,cont,4.676218089974004,15000455,2,2,0,0
6bba_57b7cc1e,14987,15,19.266666666666666,144.53333333333333,235.99999999999994,cont,2.085592525052693,16000471,3,0,0,3
6bba_57b7cc1e,18400,18,47.91147888183595,74.97019287109376,29.535253906249995,cont,2.1658985326457936,19000572,1,0,1,0
6bba_57b7cc1e,18969,19,18.39997344970703,151.60305664062497,239.939521484375,cont,4.892727205349494,20000586,1,0,1,0
6bba_57b7cc1e,19545,19,58.47999999999999,156.64000000000004,60.64,cont,2.59063939983936,20000600,2,2,0,0
6bba_57b7cc1e,20546,20,59.160000000000004,160.00000000000006,60.000000000000014,cont,1.6396402997303865,21000628,2,0,2,0
6bba_57b7cc1e,21506,21,60.48000000000002,2.0799999999999996,134.72000000000003,cont,5.160033914617247,22000656,1,1,0,0
6bba_57b7cc1e,24509,24,61.99999999999999,72.53333333333332,179.99999999999994,cont,4.419565184903468,25000749,6,0,0,6
6bba_57b7cc1e,25723,26,16.0,33.279999999999994,116.63999999999999,cont,5.416517706285108,27000789,3,1,2,0
6bba_57b7cc1e,25745,26,16.68,149.92000000000002,60.640000000000015,cont,5.56391077615377,27000791,2,2,0,0
6bba_57b7cc1e,27702,28,14.840000000000003,166.71999999999997,240.0,cont,6.774613111647039,29000853,1,1,0,0
6bba_57b7cc1e,28378,28,60.0,12.64,149.28000000000003,cont,2.6666839192150182,29000875,3,3,0,0
6bba_57b7cc1e,28729,29,17.679999999999996,142.08,89.28000000000003,cont,6.353147349936101,30000893,1,0,1,0
6bba_57b7cc1e,29467,29,61.99999999999999,57.06666666666665,87.99999999999997,cont,6.087487522802616,30000914,4,0,0,4
6bba_57b7cc1e,31048,31,37.160000000000004,9.28,248.0,cont,1.6988829050290621,32000968,1,1,0,0
6bba_57b7cc1e,31192,31,46.32,99.20000000000002,44.63999999999999,cont,6.91303343330553,32000975,2,1,1,0
6bba_57b7cc1e,33305,33,51.68,50.56,190.72000000000003,cont,5.9837522195525334,34001047,1,0,1,0
6bba_57b7cc1e,34416,34,58.0,52.0,122.71999999999998,cont,6.441316344505989,35001085,1,0,1,0
6bba_57b7cc1e,34838,35,21.0,6.08,62.71999999999999,cont,5.102034214408994,36001100,1,1,0,0
6bba_57b7cc1e,35821,36,22.533333333333335,17.599999999999998,111.46666666666667,cont,2.786952643071322,37001135,2,0,0,2
6bba_57b7cc1e,36722,37,14.16,60.000000000000014,144.64000000000001,cont,3.021014419859658,38001159,1,0,1,0
6bba_57b7cc1e,37711,38,15.44,180.64000000000001,153.27999999999997,cont,4.054073761354126,39001185,1,1,0,0
6bba_57b7cc1e,37868,38,24.840000000000003,14.719999999999999,245.28000000000003,cont,2.1033552065687737,39001192,1,0,1,0
6bba_57b7cc1e,38287,38,52.84,74.07999999999998,16.0,cont,6.996186461208699,39001203,2,0,2,0
6bba_57b7cc1e,38469,38,61.99999999999999,64.53333333333333,88.53333333333333,cont,4.532960405157113,39001206,4,1,0,3
6bba_57b7cc1e,39916,40,24.040000000000003,57.919999999999995,104.64,cont,5.61568589978108,41001250,4,1,3,0
6bba_57b7cc1e,40068,40,36.24,170.88,133.28000000000003,cont,6.528557579013903,41001255,11,0,3,8
6bba_57b7cc1e,41819,42,21.4,76.5333333333333,107.46666666666665,cont,2.739168219500623,43001312,7,0,0,7
6bba_57b7cc1e,42479,42,61.99999999999999,67.99999999999997,130.9333333333333,cont,4.893097196419777,43001330,3,0,0,3
6bba_57b7cc1e,43006,43,35.84,125.28000000000002,127.36,cont,3.689000542152314,44001348,3,3,0,0
6bba_57b7cc1e,43349,43,59.64,48.640000000000015,85.28,cont,4.8594293003397,44001358,2,2,0,0
6bba_57b7cc1e,46349,46,59.13333333333331,4.533333333333333,143.99999999999997,cont,5.787727977271264,47001440,3,0,0,3
6bba_57b7cc1e,46355,46,59.160000000000004,74.07999999999998,120.47999999999999,cont,5.129160049413555,47001447,6,0,1,5
6bba_57b7cc1e,46452,46,62.000000000000014,178.72000000000003,71.99999999999999,cont,6.912441342427456,47001446,2,1,1,0
6bba_57b7cc1e,46938,47,31.866666666666667,172.5333333333333,159.99999999999994,cont,1.872083379701739,48001464,5,0,0,5
6bba_57b7cc1e,49343,49,55.16000000000001,3.3600000000000003,139.35999999999999,cont,6.037000755756133,50001539,5,1,1,3
6bba_57b7cc1e,49390,49,57.39999999999999,53.06666666666665,95.46666666666664,cont,4.782413620204991,50001540,2,0,2,0
6bba_57b7cc1e,53772,54,12.280000000000003,79.36,173.28000000000003,cont,5.5675877013748085,55001659,7,0,1,6
6bba_57b7cc1e,54516,54,58.999999999999986,68.0,132.64000000000001,cont,2.160761222925865,55001682,1,1,0,0
6bba_57b7cc1e,57556,57,59.63999999999999,60.640000000000015,153.27999999999997,cont,5.919574124250819,58001763,1,0,1,0
6bba_57b7cc1e,57765,58,9.479999999999999,74.07999999999998,176.64000000000001,cont,2.5116108501318473,59001773,2,2,0,0
6bba_57b7cc1e,59525,59,58.0,67.46666666666667,160.53333333333333,cont,6.152627600374406,60001822,5,0,0,5
6bba_57b7cc1e,61336,61,45.466666666666654,72.0,227.46666666666664,cont,4.2627652450870945,62001868,3,0,0,3
6bba_57b7cc1e,62297,62,46.040000000000006,76.0,225.28,cont,3.2859573376719196,63001894,3,3,0,0
6bba_57b7cc1e,63525,63,58.95976806640624,66.73207153320314,165.973154296875,cont,6.623830099694895,64001922,2,2,0,0
6bba_57b7cc1e,63840,64,15.533333333333328,57.06666666666666,123.46666666666665,cont,6.986712649439172,65001933,5,0,0,5
6bba_57b7cc1e,66190,66,31.04,182.56000000000003,182.07999999999998,cont,2.5404628638301268,67001988,1,0,1,0
6bba_57b7cc1e,71552,71,53.480000000000004,14.560000000000002,135.35999999999999,cont,5.914665169094523,72002125,1,1,0,0
6bba_57b7cc1e,73966,74,20.599999999999998,31.999999999999993,19.999999999999993,cont,2.415736171439264,75002194,3,0,0,3
6bba_57b7cc1e,75887,76,21.28,51.36000000000001,14.719999999999999,cont,6.70130770820144,77002248,1,0,1,0
6bba_57b7cc1e,76236,76,42.88000000000001,61.44,242.72000000000003,cont,2.065987959790677,77002256,3,0,1,2
6bba_57b7cc1e,76485,76,60.0,65.28,161.91999999999996,cont,4.270164846056883,77002265,1,0,1,0
6bba_57b7cc1e,77231,77,45.800000000000004,177.28,240.00000000000006,cont,6.933896136011828,78002283,1,0,1,0
6bba_57b7cc1e,80303,80,58.16,49.28,201.43999999999997,cont,3.2984297874140154,81002365,2,0,2,0
6bba_57b7cc1e,83108,83,53.760000000000005,33.120000000000005,122.72000000000001,cont,2.9422822774166226,84002441,2,2,0,0
6bba_57b7cc1e,83290,83,61.52000000000001,61.43999999999998,204.63999999999996,cont,4.935721831708096,84002444,3,3,0,0
6bba_57b7cc1e,83773,84,33.2,143.04000000000002,62.07999999999999,cont,6.482304036760383,85002457,3,1,0,2
6bba_57b7cc1e,83870,84,40.96,177.12,66.72,cont,5.775589255651758,85002459,6,0,1,5
6bba_57b7cc1e,84067,84,51.32,167.2,145.92000000000002,cont,6.009271092445413,85002466,5,0,1,4
6bba_57b7cc1e,84147,84,57.00000000000001,29.279999999999994,118.72,cont,4.689272498213345,85002467,3,1,2,0
6bba_57b7cc1e,84236,84,60.52,32.0,126.72000000000001,cont,6.750168632152829,85002467,2,1,1,0
6bba_57b7cc1e,84504,85,18.16,90.72,142.08,cont,3.716563643004112,86002479,2,1,1,0
6bba_57b7cc1e,84768,85,36.00000000000001,194.71999999999997,61.43999999999998,cont,3.9141929244736877,86002489,2,0,2,0
6bba_57b7cc1e,86068,86,58.31999999999999,65.44,203.36,cont,2.93293368489642,87002530,2,1,1,0
6bba_57b7cc1e,88794,89,58.31999999999999,115.36000000000001,119.36000000000001,cont,6.0023021625039945,90002626,1,0,1,0
6bba_57b7cc1e,88932,90,3.0674136734008783,135.56538330078126,218.975498046875,cont,6.0643798938890665,91002631,2,2,0,0
6bba_57b7cc1e,89807,91,2.6800000000000006,160.15999999999997,138.72000000000003,cont,1.5682723376378276,92002659,3,0,3,0
6bba_57b7cc1e,90564,91,51.360000000000014,174.55999999999995,150.56000000000003,cont,2.3653659864807177,92002676,6,1,0,5
6bba_57b7cc1e,91527,92,51.839999999999996,172.64,148.63999999999996,cont,1.9084585566629457,93002700,1,0,1,0
6bba_57b7cc1e,92147,93,31.2,229.92,221.92000000000002,cont,5.764926549835655,94002716,1,1,0,0
6bba_5a4d9360,303,3,37.52,142.56,246.72000000000006,cont,0.9543330459016769,4000030,1,1,0,0
6bba_5a4d9360,381,4,36.32,147.35999999999999,243.36000000000004,cont,1.864222777057505,5000038,1,0,1,0
6bba_5a4d9360,608,7,33.36,157.92000000000002,230.72000000000008,cont,1.163209460931248,8000062,1,1,0,0
6bba_5a4d9360,682,8,31.84,161.92,225.44000000000003,cont,2.07345160601833,9000071,2,1,1,0
6bba_5a4d9360,756,9,30.320000000000007,167.36000000000004,217.92000000000002,cont,1.7261753242356146,10000080,1,0,1,0
6bba_5a4d9360,787,9,56.48,79.36,136.0,cont,1.4245421413563004,10000084,1,1,0,0
6bba_5a4d9360,856,10,55.839999999999996,84.00000000000001,128.64000000000001,cont,1.254727311012231,11000093,1,0,1,0
6bba_5a4d9360,1188,15,56.99999999999999,55.35999999999999,223.36,cont,2.0578557559994426,16000146,1,1,0,0
6bba_5a4d9360,1199,15,61.84,61.44000000000001,64.64,cont,1.7222700318184763,16000148,1,1,0,0
6bba_5a4d9360,1250,16,55.99999999999999,60.000000000000014,217.92000000000002,cont,1.7162031129793587,17000156,1,0,1,0
6bba_5a4d9360,1262,16,61.68,64.0,58.56,cont,1.3831604435133362,17000158,1,0,1,0
6bba_5a4d9360,1431,19,55.28,71.20000000000002,212.48,cont,1.8777717013790507,20000186,1,1,0,0
6bba_5a4d9360,1613,22,61.84,23.36,147.2,cont,0.8655318090630709,23000224,1,0,1,0
6bba_5a4d9360,2005,29,56.52,6.7200000000000015,239.36,cont,1.3821100263727157,30000316,1,1,0,0
6bba_5a4d9360,2058,30,56.160000000000004,9.280000000000001,236.64000000000001,cont,1.4036275547665715,31000330,1,0,1,0
6bba_5a4d9360,2223,33,53.480000000000004,18.720000000000002,229.27999999999997,cont,1.0343989800845743,34000370,1,1,0,0
6bba_5a4d9360,2277,34,53.0,21.280000000000005,226.72,cont,1.772516748863039,35000383,1,0,1,0
6bba_5a4d9360,2279,34,56.68,68.64000000000001,89.28000000000003,cont,2.5708403903198684,35000384,1,1,0,0
6bba_5a4d9360,2334,35,55.99999999999999,73.44,86.71999999999998,cont,2.050335338426385,36000394,1,0,1,0
6bba_5a4d9360,2443,37,48.36,114.56000000000002,31.52000000000001,cont,3.8876575530774353,38000417,1,1,0,0
6bba_5a4d9360,2501,38,47.68000000000001,119.36000000000001,21.119999999999997,cont,2.015851634545546,39000430,1,0,1,0
6bba_5a4d9360,2505,38,50.86666666666666,96.5333333333333,58.93333333333331,cont,3.7183158009821335,39000431,1,0,0,1
6bba_5a4d9360,2599,40,23.04,195.84000000000003,72.79999999999998,cont,3.628410679967205,41000449,1,1,0,0
6bba_5a4d9360,2601,40,24.04,127.20000000000003,251.04000000000002,cont,3.462127236252307,41000448,1,1,0,0
6bba_5a4d9360,2623,40,49.68000000000001,6.560000000000001,251.04000000000002,cont,2.2572881960440743,41000455,1,1,0,0
6bba_5a4d9360,2657,41,22.12,134.07999999999998,249.27999999999997,cont,1.9293753036928818,42000458,1,0,1,0
6bba_5a4d9360,2658,41,21.799999999999997,202.07999999999998,62.56,cont,2.3038351177981453,42000457,1,0,1,0
6bba_5a4d9360,2683,41,48.84000000000001,12.640000000000004,249.27999999999997,cont,2.3456355748495796,42000463,1,0,1,0
6bba_5a4d9360,2850,44,45.84,128.64000000000004,32.8,cont,2.3971917163631327,45000492,1,1,0,0
6bba_5a4d9360,2905,45,45.31999999999999,130.07999999999998,26.080000000000002,cont,1.458612889871753,46000500,1,0,1,0
6bba_5a4d9360,2908,45,46.999999999999986,139.46666666666667,34.93333333333332,cont,6.373796523113631,46000500,1,0,0,1
6bba_5a4d9360,3237,51,35.52000000000001,127.20000000000003,119.84000000000002,cont,3.9918481387071716,52000545,1,1,0,0
6bba_5a4d9360,3245,51,39.839999999999996,61.92,223.52,cont,3.3304175789230976,52000546,1,1,0,0
6bba_5a4d9360,3266,51,62.000000000000014,12.16,126.08,cont,3.560938254168415,52000548,1,1,0,0
6bba_5a4d9360,3296,52,34.480000000000004,133.44,112.47999999999998,cont,1.4596987060691569,53000551,1,0,1,0
6bba_5a4d9360,3301,52,38.0,70.71999999999998,217.12,cont,1.8390520656033675,53000552,1,0,1,0
6bba_5a4d9360,3327,52,60.86666666666666,31.466666666666672,121.6,cont,2.017481111366888,53000554,2,0,0,2
6bba_5a4d9360,3680,58,49.84,4.000000000000001,202.08000000000004,cont,2.0400061274417647,59000612,1,1,0,0
6bba_5a4d9360,3738,59,49.36000000000001,5.920000000000001,198.72000000000003,cont,3.4211556892371733,60000623,1,0,1,0
6bba_5a4d9360,3784,60,31.2,97.92000000000002,190.07999999999998,cont,3.0273885198302577,61000630,1,1,0,0
6bba_5a4d9360,3814,60,58.36000000000001,29.919999999999998,155.36,cont,1.5570349426072507,61000636,1,1,0,0
6bba_5a4d9360,3817,60,59.68000000000001,12.0,40.000000000000014,cont,0.9646534351776131,61000638,1,1,0,0
6bba_5a4d9360,3845,61,30.0,100.00000000000001,186.07999999999998,cont,1.8690325873296023,62000641,2,1,1,0
6bba_5a4d9360,3872,61,57.160000000000004,37.279999999999994,147.36,cont,3.1420121936427874,62000647,2,1,1,0
6bba_5a4d9360,3877,61,58.839999999999996,13.920000000000005,35.99999999999999,cont,1.885280151595518,62000649,2,1,1,0
6bba_5a4d9360,3887,62,4.32,204.79999999999995,160.00000000000006,cont,2.1694815048762215,63000650,1,1,0,0
6bba_5a4d9360,3906,62,28.959999999999997,102.08000000000001,180.48,cont,2.7283067665678558,63000651,1,0,1,0
6bba_5a4d9360,3936,62,55.800000000000004,46.239999999999995,139.84,cont,3.4222361110829245,63000658,1,0,1,0
6bba_5a4d9360,3940,62,57.800000000000004,16.0,32.64,cont,0.4162030754331261,63000660,1,0,1,0
6bba_5a4d9360,3955,63,3.84,206.08,156.64,cont,1.3899213107222999,64000661,1,0,1,0
6bba_5a4d9360,4829,76,20.16,171.99999999999994,204.00000000000003,cont,2.733962897517056,77000813,2,2,0,0
6bba_5a4d9360,4856,76,46.040000000000006,80.0,97.76,cont,5.410347146902859,77000818,1,1,0,0
6bba_5a4d9360,4893,77,19.159999999999997,173.91999999999996,204.00000000000003,cont,3.125116247837821,78000823,2,0,2,0
6bba_5a4d9360,4912,77,37.160000000000004,135.20000000000002,18.880000000000006,cont,4.100187686252907,78000833,1,0,1,0
6bba_5a4d9360,4915,77,42.480000000000004,101.28,85.76000000000002,cont,1.8214503149962762,78000835,3,0,1,2
6bba_5a4d9360,5501,87,21.999999999999993,164.5333333333333,83.99999999999997,cont,1.7307924084393171,88000955,2,0,0,2
6bba_5b28472a,707,8,9.48,23.840000000000003,26.560000000000002,cont,2.357146381220312,9000064,1,1,0,0
6bba_5b28472a,781,9,4.533333333333333,3.999999999999999,19.999999999999993,cont,1.1114070786370067,10000073,3,0,0,3
6bba_5b28472a,1173,14,9.0,11.36,22.08,cont,0.8221921916437797,15000122,1,1,0,0
6bba_5b28472a,1249,15,7.666666666666665,33.59999999999999,27.999999999999996,cont,1.9681590823942836,16000136,2,0,0,2
6bba_5b28472a,1252,15,9.52,10.079999999999998,22.08,cont,0.9422197859310765,16000137,1,0,1,0
6bba_5b28472a,2314,29,1.0,149.44,98.72,cont,1.7479139352382311,30000282,2,0,0,2
6bba_5b28472a,3544,48,10.000000000000004,11.360000000000003,28.640000000000008,cont,1.762357263723786,49000471,1,1,0,0
6bba_5b28472a,3847,53,10.160000000000002,26.560000000000002,36.160000000000004,cont,1.996370534870718,54000520,1,1,0,0
6bba_5b28472a,3910,54,10.84,28.160000000000007,31.840000000000003,cont,4.152916332229677,55000529,2,0,1,1
6bba_5b28472a,4056,56,19.680000000000003,227.99999999999997,156.0,cont,2.329424888786931,57000546,1,1,0,0
6bba_5b28472a,4126,57,19.84,227.99999999999997,156.0,cont,2.0526522477029605,58000554,1,0,1,0
6bba_5b28472a,5052,70,7.800000000000002,148.8,87.36,cont,1.968934118374708,71000651,1,1,0,0
6bba_5b28472a,5060,70,22.320000000000007,227.36,143.36,cont,2.7462499999999896,71000652,1,1,0,0
6bba_5b28472a,5115,71,8.2,147.84,86.56,cont,1.7473095389769975,72000658,1,0,1,0
6bba_5b28472a,5123,71,22.84,225.28000000000003,141.28000000000003,cont,2.1033552065687844,72000659,1,0,1,0
6bba_5b28472a,5788,82,11.160000000000002,140.64,84.0,cont,1.57230821564984,83000729,1,1,0,0
6bba_5b28472a,5852,83,11.68,139.35999999999999,84.0,cont,2.384379505238205,84000736,1,0,1,0
6bba_5c039895,1310,11,26.320000000000004,249.27999999999997,238.72000000000003,cont,1.4712308834781747,12000176,1,1,0,0
6bba_5c039895,1420,12,26.320000000000004,248.0,237.27999999999997,cont,1.4024041856754434,13000190,1,0,1,0
6bba_5c039895,1625,14,5.68,24.640000000000008,247.35999999999999,cont,1.076186844836904,15000214,1,1,0,0
6bba_5c039895,1739,15,5.48,22.720000000000006,248.00000000000006,cont,1.2537799099124192,16000230,1,0,1,0
6bba_5c039895,1949,16,47.00000000000001,246.72,9.919999999999998,cont,2.5044513495973404,17000262,1,1,0,0
6bba_5c039895,2057,17,46.839999999999996,242.08,10.56,cont,2.4781116802920775,18000280,1,0,1,0
6bba_5c039895,2111,18,29.840000000000003,231.36,223.36000000000004,cont,2.107368768156139,19000289,1,1,0,0
6bba_5c039895,2224,19,30.679999999999996,227.36,220.64000000000001,cont,3.8680803136051876,20000306,1,0,1,0
6bba_5c039895,2575,22,41.68,84.0,30.72,cont,1.3299212852646565,23000354,1,1,0,0
6bba_5c039895,2679,23,41.31999999999999,80.0,31.360000000000003,cont,1.3527576002373958,24000367,1,0,1,0
6bba_5c039895,3571,32,39.24000000000001,247.99999999999997,167.04000000000002,cont,1.5245583376833938,33000486,1,0,1,0
6bba_5c039895,3583,32,43.36,238.07999999999998,120.0,cont,2.893230244899295,33000487,1,1,0,0
6bba_5c039895,3598,32,47.68000000000001,242.71999999999997,89.27999999999999,cont,3.8073926856183187,33000489,1,1,0,0
6bba_5c039895,3677,33,43.480000000000004,231.84000000000003,118.72,cont,1.1338960765872785,34000502,1,0,1,0
6bba_5c039895,3689,33,47.480000000000004,237.27999999999997,88.00000000000001,cont,2.4872587822741736,34000504,1,0,1,0
6bba_5c039895,3741,34,31.680000000000003,199.36,134.08,cont,2.2769723592964333,35000514,1,1,0,0
6bba_5c039895,3834,35,31.32,195.36,134.72000000000003,cont,3.1068538145525855,36000526,1,0,1,0
6bba_5c039895,3848,35,37.0,200.5333333333333,132.0,cont,6.6425429419847255,36000526,1,0,0,1
6bba_5c039895,3859,35,39.839999999999996,235.36000000000004,162.72,cont,2.6495972571317274,36000530,1,1,0,0
6bba_5c039895,3871,35,43.84,223.36000000000004,114.07999999999998,cont,2.324715708425431,36000531,1,1,0,0
6bba_5c039895,3960,36,39.67999999999999,230.72000000000008,161.28,cont,3.9167893819428192,37000543,1,0,1,0
6bba_5c039895,3972,36,44.160000000000004,220.64000000000001,109.92000000000003,cont,0.9939103644192427,37000544,1,0,1,0
6bba_5c039895,3986,36,48.16,223.99999999999997,80.64,cont,1.472576398697204,37000546,1,1,0,0
6bba_5c039895,4080,37,48.31999999999999,220.64000000000001,77.28,cont,1.1523759423469577,38000559,1,0,1,0
6bba_5c039895,4146,38,39.64,220.0,159.36000000000004,cont,2.5371345180143603,39000568,1,1,0,0
6bba_5c039895,4242,39,39.839999999999996,214.72,158.55999999999997,cont,1.96356221253619,40000580,1,0,1,0
6bba_5c039895,4257,39,45.160000000000004,136.64000000000001,26.719999999999995,cont,1.4679966152890087,40000583,1,1,0,0
6bba_5c039895,4347,40,45.31999999999999,133.28,27.679999999999996,cont,2.8903538386329277,41000595,1,0,1,0
6bba_5c039895,4427,41,39.84,243.36000000000004,44.00000000000001,cont,2.206472368397101,42000605,1,1,0,0
6bba_5c039895,4521,42,40.000000000000014,239.36,44.00000000000001,cont,4.093452088396759,43000618,1,0,1,0
6bba_5c039895,4803,45,47.32000000000001,235.99999999999994,28.639999999999997,cont,6.825096725505067,46000655,2,1,1,0
6bba_5c039895,4990,48,12.16,73.44,230.72000000000008,cont,1.3813455849279674,49000682,1,1,0,0
6bba_5c039895,5030,48,38.86666666666666,163.4666666666666,152.0,cont,1.9861521460407496,49000692,1,0,0,1
6bba_5c039895,5070,49,11.84,68.0,229.28000000000003,cont,1.9971640049079638,50000698,1,0,1,0
6bba_5c039895,5071,49,11.840000000000003,226.72,248.00000000000006,cont,1.931632243725486,50000699,1,1,0,0
6bba_5c039895,5137,49,45.32000000000001,155.35999999999999,74.56,cont,1.3102176584445746,50000709,1,1,0,0
6bba_5c039895,5159,50,12.0,222.71999999999997,248.00000000000006,cont,3.288327129788028,51000712,1,0,1,0
6bba_5c039895,5226,50,45.68,148.64000000000001,70.72,cont,1.1973284062862575,51000723,1,0,1,0
6bba_5c039895,5932,59,21.160000000000004,45.28,171.99999999999994,cont,3.1762411452060633,60000862,1,1,0,0
6bba_5c039895,5938,59,25.520000000000003,13.280000000000001,190.56000000000003,cont,2.583392270833057,60000864,1,1,0,0
6bba_5c039895,5986,59,46.16000000000001,242.71999999999997,93.28000000000003,cont,1.9298542691094498,60000874,2,2,0,0
6bba_5c039895,6013,60,20.320000000000004,44.00000000000001,176.64000000000001,cont,4.770419923864143,61000880,1,0,1,0
6bba_5c039895,6020,60,25.000000000000004,8.64,191.36,cont,4.913092839546178,61000882,1,0,1,0
6bba_5c039895,6073,60,46.16,238.72000000000003,92.63999999999999,cont,4.7093049513702185,61000891,2,0,2,0
6bba_5c039895,6233,62,45.68,229.28000000000003,93.27999999999999,cont,2.274825886194366,63000927,1,1,0,0
6bba_5c039895,6299,63,45.320000000000014,224.64000000000004,94.72,cont,1.4492595091976959,64000942,1,0,1,0
6bba_5c039895,7128,75,24.0,69.92000000000002,171.36,cont,1.687576387160001,76001109,1,1,0,0
6bba_5c039895,7192,76,23.840000000000003,70.72,172.0,cont,2.304121646636732,77001124,1,0,1,0
6bba_5c039895,7882,87,13.48,35.35999999999999,235.99999999999994,cont,1.8870301567542724,88001273,1,1,0,0
6bba_5c039895,7909,87,38.31999999999999,150.07999999999998,164.64,cont,6.483912905221659,88001285,1,1,0,0
6bba_5c039895,7939,88,13.319999999999999,39.36,238.72000000000003,cont,1.849504156524122,89001285,1,0,1,0
6bba_5c039895,7965,88,35.13333333333335,171.46666666666664,164.5333333333333,cont,3.222461452445937,89001297,3,0,0,3
6bba_5c824876,322,1,17.92,95.68,119.03999999999999,cont,3.7455809379187133,2000028,1,1,0,0
6bba_5c824876,691,2,55.52000000000001,121.28000000000003,240.0,cont,1.8260835824244117,3000054,1,1,0,0
6bba_5c824876,923,3,52.86666666666666,131.99999999999997,43.999999999999986,cont,1.4806931677239878,4000069,4,0,0,4
6bba_5c824876,940,3,58.733333333333334,119.46666666666667,243.46666666666664,cont,6.362796735058143,4000070,5,1,0,4
6bba_5c824876,1167,4,53.16,116.64000000000001,234.56,cont,1.3915352358097248,5000085,2,1,1,0
6bba_5c824876,1283,5,14.733333333333327,116.0,211.46666666666658,cont,2.106588952284931,6000092,2,0,0,2
6bba_5c824876,1427,5,56.68,114.72,239.36,cont,6.535025941991362,6000100,1,1,0,0
6bba_5c824876,1928,7,56.52,137.28,164.00000000000003,cont,1.4491684037750685,8000129,2,0,2,0
6bba_5c824876,2736,11,6.880000000000001,113.28000000000002,213.92000000000002,cont,1.8238408133661212,12000178,1,1,0,0
6bba_5c824876,3155,12,61.32,136.64,236.64000000000001,cont,1.2624900989710854,13000214,1,1,0,0
6bba_5c824876,3397,13,61.000000000000014,135.36,234.72000000000006,cont,1.0785153278929216,14000234,3,2,1,0
6bba_5c824876,3634,14,58.520000000000024,237.27999999999997,86.72,cont,0.7964157990647422,15000253,1,1,0,0
6bba_5c824876,3648,14,60.52,132.64000000000001,232.0,cont,1.34099426732555,15000254,3,1,2,0
6bba_5c824876,3678,15,1.0799999999999996,101.6,231.03999999999996,cont,2.1582545030649283,16000257,1,1,0,0
6bba_5c824876,3874,15,57.48,235.99999999999994,85.28,cont,0.944459170636837,16000274,1,0,1,0
6bba_5c824876,3885,15,59.84000000000002,131.36,229.27999999999997,cont,0.7241740985425892,16000275,1,0,1,0
6bba_5c824876,4136,16,58.999999999999986,128.64000000000004,226.72,cont,1.6355283932112232,17000295,1,1,0,0
6bba_5c824876,4137,16,59.160000000000004,164.00000000000003,20.640000000000008,cont,1.4477099372802469,17000296,1,0,1,0
6bba_5c824876,4207,17,7.68,72.64,234.72000000000006,cont,0.9198131603755074,18000300,1,0,1,0
6bba_5c824876,4386,17,57.84,127.36000000000001,223.36000000000004,cont,1.920527271350227,18000314,2,1,1,0
6bba_5c824876,4632,18,56.84,160.64000000000001,15.36,cont,2.1307338084565974,19000335,2,2,0,0
6bba_5c824876,5132,20,54.36,120.64000000000001,211.36,cont,4.704311851907786,21000375,1,0,1,0
6bba_5c824876,5502,22,19.84,84.0,228.64000000000004,cont,1.855419848039782,23000407,2,2,0,0
6bba_5c824876,6186,25,3.3200000000000003,66.72,224.64000000000001,cont,1.2406531797807125,26000460,1,0,1,0
6bba_5c824876,6431,26,2.3599999999999994,64.64,220.0,cont,1.1260675490395775,27000480,1,1,0,0
6bba_5c824876,6619,26,53.0,111.36000000000001,241.27999999999997,cont,1.8901760830673986,27000497,2,0,2,0
6bba_5c824876,6673,27,0.7199999999999999,63.04,214.24,cont,3.7391246468926385,28000500,1,0,1,0
6bba_5c824876,6898,27,58.520000000000024,151.35999999999999,191.36,cont,2.82703300741251,28000518,2,1,1,0
6bba_5c824876,6915,27,61.52000000000001,160.64,176.00000000000003,cont,6.963082336508728,28000518,1,0,1,0
6bba_5c824876,7152,28,57.16,148.64000000000001,188.64000000000004,cont,1.39721654102005,29000539,1,0,1,0
6bba_5c824876,7646,30,55.64000000000001,208.0,208.8,cont,2.518540313852444,31000581,1,0,1,0
6bba_5c824876,7909,31,59.31999999999999,150.56000000000003,246.72,cont,1.5728959159779168,32000604,2,1,1,0
6bba_5c824876,8648,34,55.51999999999999,143.36,236.0,cont,4.793336846602801,35000667,2,1,1,0
6bba_5c824876,8894,35,54.0,140.64000000000004,232.0,cont,5.73099972190716,36000688,1,0,1,0
6bba_5c824876,8922,35,60.67999999999999,169.28,187.35999999999999,cont,1.1973284062862708,36000689,1,1,0,0
6bba_5c824876,9378,37,47.36,198.08,197.28000000000003,cont,1.4001428498549802,38000728,1,1,0,0
6bba_5c824876,9391,37,50.839999999999996,134.72,224.64000000000001,cont,4.298915924393036,38000729,4,1,3,0
6bba_5c824876,9404,37,55.16000000000001,143.99999999999997,231.36,cont,4.167166678331945,38000729,3,2,1,0
6bba_5c824876,9623,38,45.51999999999998,196.0,196.00000000000003,cont,1.126302024325635,39000750,1,0,1,0
6bba_5c824876,9635,38,49.36000000000001,132.00000000000003,220.0,cont,4.866759702307058,39000751,1,0,1,0
6bba_5c824876,9650,38,53.519999999999996,140.64000000000004,226.72,cont,3.297749232431125,39000751,2,0,2,0
6bba_5c824876,9662,38,56.84000000000001,160.64,175.36,cont,2.1736766807646197,39000753,1,0,1,0
6bba_5c824876,9894,39,51.519999999999996,137.28,221.28000000000003,cont,3.3014705530263377,40000770,1,1,0,0
6bba_5c824876,10056,40,28.640000000000008,79.36000000000001,184.64,cont,0.8456247690317436,41000786,2,2,0,0
6bba_5c824876,10132,40,49.64000000000001,135.36,216.63999999999996,cont,5.176026770120107,41000790,1,0,1,0
6bba_5c824876,10172,40,60.84000000000001,168.63999999999996,216.0,cont,0.7334134662657986,41000793,1,1,0,0
6bba_5c824876,10294,41,26.640000000000004,76.64000000000001,180.0,cont,0.7270853629388995,42000804,2,0,2,0
6bba_5c824876,10412,41,60.16000000000001,165.28,212.0,cont,0.5640880361255789,42000812,1,0,1,0
6bba_5c824876,10909,43,58.039999999999985,160.64000000000001,204.00000000000003,cont,1.875590740273601,44000848,2,2,0,0
6bba_5c824876,11157,44,56.160000000000004,158.72,199.36,cont,3.0459973736035884,45000866,2,0,2,0
6bba_5c824876,11899,47,44.67999999999999,136.0,242.56000000000006,cont,2.190437313643108,48000923,2,2,0,0
6bba_5c824876,12042,48,12.32,74.07999999999998,224.64000000000004,cont,1.2907247237501829,49000931,2,2,0,0
6bba_5c824876,12159,48,42.68000000000001,132.00000000000006,239.35999999999996,cont,2.480720976349392,49000940,2,0,2,0
6bba_5c824876,13209,52,36.68,119.36000000000001,221.92000000000004,cont,1.192355861309862,53001007,1,1,0,0
6bba_5c824876,14002,55,34.84,132.64000000000001,154.71999999999997,cont,2.1178681793256064,56001057,2,2,0,0
6bba_5c824876,14269,56,33.000000000000014,130.72,153.27999999999997,cont,1.6329430868832902,57001073,2,0,2,0
6bba_5c824876,15250,60,12.48,91.36,145.44000000000003,cont,5.77060357545032,61001120,1,1,0,0
6bba_5c824876,15310,60,25.0,118.71999999999997,148.00000000000003,cont,1.8897569288138587,61001126,1,1,0,0
6bba_5c824876,15541,61,16.2,98.72000000000001,116.80000000000001,cont,6.223431780376481,62001133,2,2,0,0
6bba_5c824876,15569,61,22.999999999999996,114.56000000000003,144.64000000000001,cont,3.393177437889158,62001138,1,0,1,0
6bba_5c824876,15816,62,17.160000000000004,90.71999999999998,196.00000000000003,cont,1.649595120779639,63001145,1,0,1,0
6bba_5c824876,18469,71,60.84000000000001,170.07999999999998,31.360000000000007,cont,0.8623224454923974,72001253,3,3,0,0
6bba_5c824876,18732,72,60.000000000000014,165.28,29.279999999999994,cont,0.7575010313524377,73001265,3,0,3,0
6bba_5c824876,18740,72,61.99999999999999,155.46666666666664,189.0666666666666,cont,3.6642659043873596,73001266,3,0,0,3
6bba_5c824876,19004,73,58.999999999999986,148.00000000000003,182.07999999999996,cont,6.653159611229857,74001278,1,0,1,0
6bba_5c824876,19288,74,58.0,141.92000000000002,186.08,cont,6.645793077203046,75001287,1,1,0,0
6bba_5c824876,19567,75,56.99999999999999,152.63999999999996,27.999999999999996,cont,1.8226821753942881,76001297,1,0,1,0
6bba_5c824876,19872,76,61.64,145.27999999999997,188.00000000000003,cont,2.681003776573252,77001309,1,1,0,0
6bba_5c824876,21231,81,23.0,226.72,110.08,cont,1.7466293431921944,82001353,1,1,0,0
6bba_5c824876,21545,82,23.479999999999997,224.64000000000001,116.0,cont,2.5310385714168846,83001362,1,0,1,0
6bba_5c824876,22327,84,52.16000000000002,120.00000000000003,40.64000000000001,cont,1.5275864337247484,85001379,1,1,0,0
6bba_5c824876,22664,85,52.16,116.63999999999999,42.720000000000006,cont,1.4679966152890216,86001388,1,0,1,0
6bba_5c824876,23036,86,59.31999999999999,112.64,209.28000000000006,cont,2.2606781338792765,87001399,1,1,0,0
6bba_5c824876,23355,87,52.0,110.71999999999998,44.64000000000001,cont,1.7933993559717818,88001408,1,0,1,0
6bba_5c824876,23390,87,59.160000000000004,109.28,214.07999999999998,cont,2.246323732790984,88001409,1,0,1,0
6bba_5c824876,24255,90,14.479999999999999,42.56,140.79999999999998,cont,2.3393792579442976,91001433,1,1,0,0
6bba_5c824876,24621,91,10.866666666666664,35.46666666666666,159.46666666666664,cont,1.7794801148556585,92001443,7,0,0,7
6bba_5c824876,24641,91,15.079999999999998,42.07999999999999,140.64000000000001,cont,6.537449928106533,92001445,1,0,1,0
6bba_5c824876,25798,94,7.440000000000001,30.080000000000005,184.0,cont,2.66222416411541,95001474,1,1,0,0
6bba_5c824876,26231,95,9.16,41.28000000000001,203.35999999999999,cont,6.800989013555307,96001483,1,0,1,0
6bba_5dfe9ad1,692,2,50.32000000000001,113.92,233.27999999999997,cont,1.4214803023608824,3000049,1,0,1,0
6bba_5dfe9ad1,939,3,49.84,115.36000000000001,230.72000000000003,cont,1.1094121472653757,4000065,1,1,0,0
6bba_5dfe9ad1,983,3,60.67999999999999,153.44000000000003,130.72,cont,1.1251291592524066,4000072,2,1,1,0
6bba_5dfe9ad1,1218,4,60.32000000000001,64.64,232.0,cont,6.379134076228528,5000084,2,0,2,0
6bba_5dfe9ad1,1233,4,61.99999999999999,79.99999999999997,228.53333333333333,cont,3.2555248179483574,5000084,4,0,0,4
6bba_5dfe9ad1,1715,6,58.36,153.27999999999997,120.00000000000003,cont,1.6062037931096962,7000137,2,2,0,0
6bba_5dfe9ad1,1960,7,57.16,154.72000000000003,116.0,cont,0.28379404944429815,8000159,2,0,2,0
6bba_5dfe9ad1,3010,11,61.99999999999999,80.53333333333332,188.5333333333333,cont,6.314135170503467,12000246,5,0,0,5
6bba_5dfe9ad1,3502,13,57.68000000000002,186.08,74.72,cont,1.6413304435731284,14000297,4,0,0,4
6bba_5dfe9ad1,3731,14,54.84,89.28000000000003,184.64000000000001,cont,0.6211242327425263,15000315,1,1,0,0
6bba_5dfe9ad1,3968,15,53.68,90.72,182.07999999999993,cont,1.3573371219045236,16000340,1,0,1,0
6bba_5dfe9ad1,4620,18,30.840000000000007,84.00000000000001,240.0,cont,1.9282800788526473,19000412,1,0,1,0
6bba_5dfe9ad1,4640,18,35.68,83.84000000000002,232.64,cont,6.694546404537056,19000412,2,0,2,0
6bba_5dfe9ad1,4718,18,57.84,116.0,221.28000000000006,cont,2.263654746753566,19000425,2,2,0,0
6bba_5dfe9ad1,4828,19,19.0,135.36,210.08000000000004,cont,1.8084286169213297,20000431,1,0,1,0
6bba_5dfe9ad1,5475,21,58.84000000000001,170.72000000000003,48.000000000000014,cont,0.5813776741499187,22000500,1,0,1,0
6bba_5dfe9ad1,5710,22,53.83999999999999,120.64000000000001,208.0,cont,2.125708705349834,23000525,1,0,1,0
6bba_5dfe9ad1,6217,24,55.519999999999996,171.35999999999996,41.28,cont,0.8017032649303582,25000579,3,2,1,0
6bba_5dfe9ad1,6226,24,58.2,92.63999999999999,231.36,cont,0.7555813159415685,25000580,2,1,1,0
6bba_5dfe9ad1,6381,25,29.32,155.36,248.00000000000006,cont,1.1116710394716447,26000592,1,1,0,0
6bba_5dfe9ad1,6745,26,55.480000000000004,98.72000000000001,225.28,cont,0.9820164204329783,27000629,1,1,0,0
6bba_5dfe9ad1,7005,27,54.0,100.63999999999999,223.36,cont,0.6106195624118222,28000654,1,0,1,0
6bba_5dfe9ad1,7189,28,35.32,198.08000000000004,50.72,cont,6.047030908842794,29000669,1,1,0,0
6bba_5dfe9ad1,7444,29,34.0,200.64000000000004,46.08,cont,6.6463294616276825,30000691,1,0,1,0
6bba_5dfe9ad1,8149,32,11.680000000000003,125.28,177.92,cont,0.8249176095223085,33000748,1,1,0,0
6bba_5dfe9ad1,8210,32,25.999999999999996,198.93333333333328,27.466666666666658,cont,6.455271451603651,33000756,1,0,0,1
6bba_5dfe9ad1,8839,34,48.640000000000015,103.36,243.36000000000004,cont,1.0819376426116472,35000814,3,2,1,0
6bba_5dfe9ad1,9150,35,55.68000000000001,153.91999999999996,126.72000000000001,cont,6.928714933160391,36000842,1,1,0,0
6bba_5dfe9ad1,9217,36,4.2,190.08000000000004,104.00000000000003,cont,2.9854925263681364,37000849,3,0,0,3
6bba_5dfe9ad1,9733,37,61.36000000000001,131.36000000000004,201.28000000000003,cont,0.7038332544004972,38000899,2,2,0,0
6bba_5dfe9ad1,10015,38,60.67999999999999,132.64000000000001,197.28,cont,1.1204254214359717,39000926,3,1,2,0
6bba_5dfe9ad1,10305,39,59.64,133.28,194.71999999999997,cont,0.6638676920139985,40000953,1,0,1,0
6bba_5dfe9ad1,10324,39,61.99999999999999,124.53333333333335,188.53333333333325,cont,5.67930461131076,40000953,5,0,0,5
6bba_5dfe9ad1,10365,40,1.5199999999999998,160.64,231.36,cont,2.7162787619277906,41000956,1,1,0,0
6bba_5dfe9ad1,10610,40,58.680000000000014,135.36,191.36,cont,2.4122357368217346,41000983,1,1,0,0
6bba_5dfe9ad1,10631,40,61.52000000000001,136.0,204.00000000000003,cont,4.754451203872028,41000983,1,1,0,0
6bba_5dfe9ad1,10825,41,38.64,119.36000000000001,220.64000000000004,cont,1.0819376426116298,42001000,2,2,0,0
6bba_5dfe9ad1,10910,41,57.52000000000001,136.0,188.00000000000003,cont,2.7266124816335564,42001009,1,0,1,0
6bba_5dfe9ad1,11025,42,14.68,159.35999999999999,66.72,cont,0.7497926796788547,43001016,2,2,0,0
6bba_5dfe9ad1,11130,42,37.36,122.08000000000003,218.08,cont,1.3004061865432666,43001025,3,1,2,0
6bba_5dfe9ad1,11216,42,60.000000000000014,136.64000000000001,198.72,cont,6.881448112497856,43001034,1,1,0,0
6bba_5dfe9ad1,12935,48,32.68,81.44,238.08000000000004,cont,1.9718154705752666,49001161,2,1,1,0
6bba_5dfe9ad1,13445,50,8.239999999999998,203.2,179.2,cont,6.281019149986726,51001192,1,0,0,1
6bba_5dfe9ad1,14525,53,35.36,231.36,161.28,cont,3.2419055449688807,54001279,1,1,0,0
6bba_5dfe9ad1,14835,54,33.32,232.64000000000004,160.64000000000001,cont,2.159538796247921,55001298,1,0,1,0
6bba_5dfe9ad1,15636,56,61.68000000000001,144.64000000000001,198.08,cont,0.8457808891787446,57001355,1,1,0,0
6bba_5dfe9ad1,16619,59,58.52000000000001,144.64000000000001,188.64000000000004,cont,0.9905837925183231,60001420,2,1,1,0
6bba_5dfe9ad1,16949,60,57.00000000000001,146.72000000000003,187.35999999999999,cont,0.1852785065785991,61001440,1,0,1,0
6bba_5dfe9ad1,18490,65,24.0,95.36000000000001,246.08000000000004,cont,0.45529008609894583,66001535,1,0,1,0
6bba_5dfe9ad1,18644,65,56.00000000000001,120.00000000000003,237.44,cont,1.8965218987662744,66001543,3,2,1,0
6bba_5dfe9ad1,18978,66,54.20000000000002,119.84,232.64000000000004,cont,1.823840813366096,67001563,3,1,2,0
6bba_5dfe9ad1,19209,67,29.72,120.63999999999999,140.00000000000003,cont,3.795862262055878,68001578,5,0,0,5
6bba_5dfe9ad1,19307,67,51.79999999999999,121.44000000000001,228.0,cont,0.8199409277502868,68001583,2,1,1,0
6bba_5dfe9ad1,19339,67,57.84,146.72000000000003,74.72,cont,1.362385889716994,68001584,4,2,2,0
6bba_5dfe9ad1,19673,68,56.08909179687501,147.3809326171875,71.11034790039062,cont,1.8235037386560264,69001607,3,1,2,0
6bba_5dfe9ad1,19716,68,62.26666666666667,151.99999999999997,171.46666666666664,cont,5.966329861304086,69001608,5,0,0,5
6bba_5dfe9ad1,20900,72,19.32,60.000000000000014,235.99999999999994,cont,5.808073717894768,73001677,1,0,1,0
6bba_5dfe9ad1,21422,73,51.32,140.00000000000003,65.92,cont,6.527930615823046,74001707,1,0,1,0
6bba_5dfe9ad1,21569,74,8.64,197.28,86.72,cont,1.3973110337358663,75001713,1,1,0,0
6bba_5dfe9ad1,21608,74,16.479999999999997,60.000000000000014,234.72000000000006,cont,6.547620870400171,75001714,1,1,0,0
6bba_5dfe9ad1,21645,74,22.86666666666666,111.46666666666667,243.46666666666664,cont,1.8669828601248701,75001716,3,1,0,2
6bba_5dfe9ad1,21678,74,29.840000000000003,104.00000000000003,249.28000000000003,cont,6.590872594922455,75001717,1,0,1,0
6bba_5dfe9ad1,21827,74,60.0,87.35999999999999,236.0,cont,3.130224133348926,75001728,3,3,0,0
6bba_5dfe9ad1,21836,74,61.48,102.72000000000001,231.36,cont,4.568360277495635,75001728,3,2,1,0
6bba_5dfe9ad1,21924,75,7.839999999999999,199.36000000000007,84.64000000000001,cont,0.7299850169695364,76001732,1,0,1,0
6bba_5dfe9ad1,22084,75,40.160000000000004,117.28,206.55999999999997,cont,1.0657076580845315,76001744,4,0,0,4
6bba_5dfe9ad1,22175,75,59.0,87.20000000000002,233.27999999999997,cont,1.3349748359051625,76001747,3,0,3,0
6bba_5dfe9ad1,23047,78,16.32,130.56,119.20000000000002,cont,0.847496312676343,79001794,1,0,1,0
6bba_5dfe9ad1,23729,80,4.840000000000001,228.0,20.640000000000008,cont,6.836829034720701,81001833,1,1,0,0
6bba_5dfe9ad1,23809,80,20.16,188.64000000000001,204.00000000000003,cont,0.7334134662657769,81001840,1,0,1,0
6bba_5dfe9ad1,24092,81,4.319999999999999,226.72,21.28,cont,6.372756100189301,82001860,3,2,1,0
6bba_5dfe9ad1,24309,81,43.35999999999999,134.72000000000003,164.64,cont,1.0902042526517781,82001870,1,0,1,0
6bba_5dfe9ad1,24778,82,61.0,140.00000000000003,75.36,cont,2.3027459830602144,83001893,1,0,1,0
6bba_5dfe9ad1,25040,83,38.99999999999999,147.35999999999999,70.71999999999998,cont,1.7933993559717947,84001910,1,1,0,0
6bba_5dfe9ad1,25206,84,2.4800000000000004,198.72,84.64000000000001,cont,1.2278162779911326,85001918,2,2,0,0
6bba_5dfe9ad1,25475,84,49.839999999999996,82.72000000000001,230.08000000000004,cont,5.788376823428135,85001930,1,1,0,0
6bba_5dfe9ad1,26072,86,16.200000000000006,112.64,235.35999999999999,cont,3.050372159671661,87001958,1,1,0,0
6bba_5dfe9ad1,27564,90,0.13333333333333344,237.5999999999999,39.99999999999999,cont,6.6759180600456824,91002050,3,0,0,3
6bba_5dfe9ad1,28178,91,37.64,203.35999999999999,11.360000000000003,cont,0.8178449195905056,92002088,2,0,2,0
6bba_5dfe9ad1,29003,93,37.839999999999996,195.35999999999996,20.0,cont,0.5040120286262988,94002140,3,1,2,0
6bba_5f89039d,216,4,51.36000000000001,28.640000000000004,213.44,cont,2.794480329596178,5000052,1,1,0,0
6bba_5f89039d,262,5,50.64000000000001,29.440000000000005,210.56000000000003,cont,1.5765850159442683,6000064,1,0,1,0
6bba_5f89039d,1427,30,4.839999999999999,248.64000000000001,13.919999999999998,cont,1.903124076354458,31000328,1,1,0,0
6bba_5f89039d,1478,31,5.000000000000001,247.36,12.0,cont,1.6813842883172176,32000339,1,0,1,0
6bba_5f89039d,2114,43,31.32,188.64000000000001,157.92000000000002,cont,2.8924999999999965,44000470,1,1,0,0
6bba_5f89039d,2164,44,31.159999999999997,191.36,155.36,cont,3.012523858826689,45000481,1,0,1,0
6bba_5f89039d,2166,44,32.86666666666666,183.99999999999997,169.06666666666666,cont,6.227989019115428,45000481,1,0,0,1
6bba_5f89039d,2883,58,32.32,192.8,141.92000000000002,cont,2.8322589261930147,59000622,1,0,1,0
6bba_5f89039d,4414,91,44.20000000000001,188.64000000000001,103.84,cont,3.8246921853268034,92000865,1,1,0,0
6bba_5f89039d,4462,92,48.133333333333326,179.99999999999997,92.0,cont,1.862459218420047,93000872,1,0,0,1
6bba_61dd1e0d,3284,51,11.160000000000002,75.36,237.27999999999997,cont,1.6054638153817045,52000283,1,1,0,0
6bba_61dd1e0d,3376,52,11.32,74.72,235.99999999999994,cont,2.8086187107722504,53000290,1,0,1,0
6bba_61dd1e0d,3548,54,1.0,240.0,236.64000000000004,cont,4.8007956697301895,55000302,1,1,0,0
6bba_61dd1e0d,3634,55,1.0,241.43999999999997,233.92000000000002,cont,5.047972984277944,56000309,1,0,1,0
6bba_61dd1e0d,3644,55,4.866666666666665,228.53333333333333,235.46666666666673,cont,4.830561345916933,56000309,1,0,0,1
6bba_61dd1e0d,4884,66,17.240000000000002,230.24,169.28000000000003,cont,5.714388227317078,67000391,1,0,1,0
6bba_61dd1e0d,7902,86,3.36,198.72,203.35999999999999,cont,4.094097122687737,87000533,1,1,0,0
6bba_61dd1e0d,8092,87,3.6799999999999997,199.36,199.36,cont,3.7957231277452137,88000545,1,0,1,0
6bba_61dd1e0d,9323,93,5.160000000000002,201.28000000000006,192.0,cont,5.0245094598876205,94000600,1,1,0,0
6bba_61dd1e0d,9537,94,5.160000000000001,203.35999999999999,192.0,cont,5.051554989555192,95000607,1,0,1,0
6bba_61ecbe65,745,7,61.0,172.64000000000004,14.080000000000002,cont,1.9234124947602798,8000056,1,1,0,0
6bba_61ecbe65,825,8,60.47999999999999,175.36,9.919999999999998,cont,1.377804526230031,9000063,1,0,1,0
6bba_61ecbe65,1117,12,57.84,144.00000000000003,99.36000000000001,cont,0.8655318090630608,13000095,1,1,0,0
6bba_61ecbe65,1126,12,61.160000000000004,157.28000000000003,100.0,cont,6.894386869947185,13000095,1,1,0,0
6bba_61ecbe65,1183,13,56.68,150.71999999999997,91.36000000000001,cont,0.5520218519225493,14000101,1,0,1,0
6bba_61ecbe65,1230,14,27.68,127.36000000000001,234.72000000000006,cont,2.217931539182402,15000104,1,1,0,0
6bba_61ecbe65,1297,15,26.68,133.28,228.00000000000003,cont,2.202639396837354,16000110,1,0,1,0
6bba_61ecbe65,1321,15,53.84000000000001,164.0,74.56,cont,2.1528643710182744,16000114,1,1,0,0
6bba_61ecbe65,1379,16,52.32000000000001,170.07999999999998,67.35999999999999,cont,1.3408958059819573,17000122,1,0,1,0
6bba_61ecbe65,2425,32,56.160000000000004,19.36,204.64,cont,3.503863269663922,33000301,1,1,0,0
6bba_61ecbe65,2485,33,55.04,24.000000000000007,200.64,cont,3.4133510843451202,34000313,1,0,1,0
6bba_61ecbe65,2544,34,53.68000000000001,28.639999999999997,197.28,cont,4.11510195043815,35000325,1,1,0,0
6bba_61ecbe65,2597,35,48.26666666666665,176.53333333333325,52.0,daughter,1.2775182782993477,36000339,1,0,0,1
6bba_61ecbe65,2598,35,48.84,168.0,68.64000000000001,daughter,2.0401355653485314,36000340,1,0,0,1
6bba_61ecbe65,2605,35,52.16000000000001,33.279999999999994,194.71999999999997,cont,3.201043775083353,36000335,1,0,1,0
6bba_61ecbe65,3005,42,37.59999999999999,85.6,166.9333333333333,cont,4.126810419211339,43000406,1,0,0,1
6bba_61ecbe65,3111,44,16.200000000000003,165.92000000000002,118.07999999999998,cont,3.126087810666872,45000428,1,1,0,0
6bba_61ecbe65,3168,45,14.640000000000002,170.71999999999997,113.28,cont,3.2639450822585925,46000443,1,0,1,0
6bba_61ecbe65,3719,53,61.00000000000001,7.200000000000002,132.8,cont,1.3025365877778607,54000538,1,1,0,0
6bba_61ecbe65,3787,54,61.0,13.120000000000001,123.83999999999997,cont,2.6527845040636024,55000548,1,0,1,0
6bba_61ecbe65,3798,55,4.840000000000001,201.92000000000002,69.44,cont,1.6636222715809015,56000550,2,2,0,0
6bba_61ecbe65,3824,55,39.519999999999996,67.2,149.44,cont,2.768467818577634,56000556,1,1,0,0
6bba_61ecbe65,3858,56,4.159999999999999,206.08,61.11999999999999,cont,3.3379414502054963,57000559,2,0,2,0
6bba_61ecbe65,3893,56,38.16,74.07999999999998,143.20000000000002,cont,2.703806633711084,57000565,1,0,1,0
6bba_61ecbe65,4160,60,58.31999999999999,46.719999999999985,88.00000000000001,cont,1.1827946250300752,61000601,1,1,0,0
6bba_61ecbe65,4220,61,58.16000000000001,50.08000000000002,82.56,cont,2.166863011936836,62000609,1,0,1,0
6bba_61ecbe65,4435,65,17.560000000000002,168.0,57.919999999999995,cont,2.746442300959548,66000642,1,1,0,0
6bba_61ecbe65,4499,66,15.799999999999997,168.63999999999996,56.00000000000001,cont,1.4697943095889374,67000654,1,0,1,0
6bba_61ecbe65,5272,77,48.16,10.079999999999998,192.64,cont,4.745055650358597,78000792,1,1,0,0
6bba_61ecbe65,5337,78,47.68000000000001,11.360000000000003,190.72000000000003,cont,5.6736474159045125,79000805,1,0,1,0
6bba_61ecbe65,5420,80,5.84,181.28000000000006,27.999999999999996,cont,3.754418230565165,81000820,1,1,0,0
6bba_61ecbe65,5433,80,22.64,27.360000000000003,240.64000000000004,cont,1.7241854961111507,81000826,1,1,0,0
6bba_61ecbe65,5481,81,4.84,184.0,27.999999999999996,cont,1.480980460708379,82000834,1,0,1,0
6bba_61ecbe65,5493,81,16.133333333333333,39.99999999999999,240.5333333333333,cont,3.0662830942182926,82000837,1,0,0,1
6bba_6321a359,95,1,37.19999999999999,196.96000000000004,245.76,cont,2.968105982777587,2000018,1,1,0,0
6bba_6321a359,146,2,34.839999999999996,200.00000000000003,240.64,cont,1.9992783854431095,3000028,1,0,1,0
6bba_6321a359,531,10,19.68,33.28000000000001,207.36000000000004,cont,1.5716362890312539,11000109,1,1,0,0
6bba_6321a359,582,11,17.84,38.080000000000005,204.00000000000003,cont,0.8537051598766273,12000120,1,0,1,0
6bba_6321a359,621,12,1.8400000000000003,109.92000000000003,176.16000000000003,cont,0.6136392466262063,13000137,1,1,0,0
6bba_6321a359,622,12,4.12,81.92,185.28,cont,2.0760607228595203,13000136,1,1,0,0
6bba_6321a359,668,13,1.24,115.35999999999999,169.60000000000005,cont,1.55499849276456,14000148,1,0,1,0
6bba_6321a359,671,13,1.36,71.52000000000001,167.51999999999998,cont,2.934418862824464,14000147,1,0,0,1
6bba_6321a359,1420,26,22.4,49.44000000000001,202.72,cont,2.949498103491506,27000262,1,1,0,0
6bba_6321a359,1482,27,19.96,46.56,198.56000000000003,cont,1.7991326035064825,28000272,1,0,1,0
6bba_6321a359,2300,41,60.86666666666666,108.53333333333333,239.99999999999994,cont,2.3326282416812187,42000379,2,0,0,2
6bba_6321a359,2613,47,54.36000000000001,120.64000000000001,240.00000000000006,cont,2.9820853127467735,48000436,1,1,0,0
6bba_6321a359,2674,48,52.48000000000001,121.44000000000001,240.00000000000006,cont,4.397269216513805,49000446,1,0,1,0
6bba_6321a359,2938,52,56.32000000000001,80.64,240.64,cont,1.2440539829525035,53000485,1,1,0,0
6bba_6321a359,3000,53,55.0,80.64,240.64000000000004,cont,3.2636619099563595,54000492,1,0,1,0
6bba_6321a359,3006,53,60.133333333333326,71.46666666666665,243.99999999999997,cont,6.279187748803351,54000492,2,0,0,2
6bba_6321a359,4132,73,5.32,216.16000000000005,113.28000000000003,cont,2.0992710788509203,74000678,1,1,0,0
6bba_6321a359,4193,74,5.840000000000002,210.56,114.72,cont,0.33539622090893845,75000688,1,0,1,0
6bba_6321a359,4263,75,22.319999999999997,102.72000000000001,232.0,cont,2.850613146763345,76000698,1,1,0,0
6bba_6321a359,4321,76,21.840000000000003,100.63999999999999,232.0,cont,2.1257087053498123,77000706,1,0,1,0
6bba_6321a359,5111,88,35.32000000000001,125.28,138.72000000000003,cont,1.17988876594363,89000811,1,1,0,0
6bba_6321a359,5180,89,35.16,124.63999999999999,139.36,cont,1.6744598009507559,90000819,1,0,1,0
6bba_6479435d,761,1,56.84,152.0,192.64000000000004,cont,1.447709937280267,2000025,1,0,1,0
6bba_6479435d,2299,5,39.8,58.72,100.64,cont,1.379336910439216,6000075,2,0,2,0
6bba_6479435d,2565,6,9.84,105.28000000000002,194.24,cont,3.801979130992176,7000081,2,2,0,0
6bba_6479435d,2731,6,35.88,140.00000000000003,181.92000000000002,cont,0.6289400806118058,7000091,3,3,0,0
6bba_6479435d,2765,6,41.04,76.0,92.80000000000001,cont,6.804482552883796,7000094,1,1,0,0
6bba_6479435d,2798,6,47.96000000000001,148.00000000000003,172.64,cont,5.048548368095511,7000096,1,1,0,0
6bba_6479435d,2940,7,2.3200000000000003,104.64000000000003,205.92000000000002,cont,6.088526094220175,8000100,1,0,1,0
6bba_6479435d,3106,7,28.119999999999997,40.00000000000001,197.92000000000002,cont,6.457161963664843,8000111,2,0,2,0
6bba_6479435d,3265,7,48.04,148.64000000000001,170.56000000000003,cont,1.0982884411665332,8000118,4,0,1,3
6bba_6479435d,3542,8,23.840000000000003,55.35999999999999,240.64000000000004,cont,2.0742155896868453,9000125,2,1,1,0
6bba_6479435d,3606,8,33.00000000000001,64.64,83.2,cont,2.198619595337029,9000136,3,3,0,0
6bba_6479435d,4881,11,26.866666666666664,47.999999999999986,60.53333333333332,cont,0.7529946002831417,12000182,5,0,0,5
6bba_6479435d,5387,12,36.0,151.99999999999997,169.06666666666658,cont,6.314599829517654,13000201,3,0,0,3
6bba_6479435d,5737,13,24.680000000000003,136.48000000000002,191.2,cont,4.360120590362156,14000211,2,2,0,0
6bba_6479435d,5816,13,35.84000000000001,152.0,165.75999999999993,cont,6.829196626434182,14000217,2,2,0,0
6bba_6479435d,5864,13,43.04,53.440000000000005,242.72000000000003,cont,3.3247436461327333,14000220,1,1,0,0
6bba_6479435d,5930,13,53.56,140.00000000000003,132.8,cont,0.7196016693282503,14000223,2,2,0,0
6bba_6479435d,5939,13,56.84000000000001,132.00000000000006,141.28000000000003,cont,6.570809620016406,14000223,4,4,0,0
6bba_6479435d,6099,14,15.84,52.0,225.76000000000002,cont,1.9511507021498915,15000227,7,0,2,5
6bba_6479435d,6153,14,24.840000000000003,81.28,229.60000000000002,cont,3.434867173559984,15000229,3,0,3,0
6bba_6479435d,6157,14,24.04,141.12000000000003,194.08000000000007,cont,6.877590534664002,15000228,4,0,2,2
6bba_6479435d,6172,14,27.133333333333326,79.46666666666667,251.46666666666664,cont,6.557970659180049,15000229,4,0,0,4
6bba_6479435d,6215,14,30.44,24.640000000000008,199.83999999999997,cont,2.987879682985904,15000233,2,0,1,1
6bba_6479435d,6244,14,36.26666666666666,139.99999999999994,175.4666666666667,cont,3.776644348455674,15000234,8,0,0,8
6bba_6479435d,6323,14,47.676264648437495,103.46726928710937,211.82804443359373,cont,6.400511453638069,15000241,2,1,1,0
6bba_6479435d,6346,14,51.79999999999999,140.00000000000003,130.56,cont,1.5022243923595247,15000242,2,0,2,0
6bba_6479435d,6726,15,43.76917663574219,49.49916320800781,187.73468261718747,cont,0.7239205382501546,16000263,2,2,0,0
6bba_6479435d,6824,15,62.26666666666667,119.99999999999997,95.46666666666667,cont,6.595222650188221,16000265,5,0,0,5
6bba_6479435d,7084,16,34.16,140.00000000000003,180.0,cont,3.7377119505119607,17000275,1,1,0,0
6bba_6479435d,7869,18,22.839999999999996,86.08,223.20000000000005,cont,0.4174700588066109,19000317,1,1,0,0
6bba_6479435d,8100,18,55.99999999999999,124.63999999999999,76.0,cont,6.061838108197882,19000335,5,0,0,5
6bba_6479435d,8108,18,58.08000000000001,111.04000000000002,75.04000000000002,cont,2.54435772838646,19000335,1,0,1,0
6bba_6479435d,8260,19,12.879999999999999,49.28,220.64000000000004,cont,0.5968419493467082,20000340,2,2,0,0
6bba_6479435d,8333,19,23.519999999999996,56.00000000000001,145.28,cont,4.483586733854945,20000345,2,0,2,0
6bba_6479435d,8576,19,60.06337585449219,123.3343896484375,99.46572753906251,cont,5.904808945629403,20000356,1,0,1,0
6bba_6479435d,8770,20,21.84,52.00000000000001,52.00000000000001,cont,1.4876517779709113,21000367,1,1,0,0
6bba_6479435d,8911,20,40.47999999999999,141.43999999999997,134.72,cont,2.5667285233347292,21000377,2,2,0,0
6bba_6479435d,9114,21,5.039999999999999,125.6,223.36,cont,5.722146259271954,22000389,5,1,1,3
6bba_6479435d,9214,21,21.48,51.84,51.2,cont,3.2891702031059467,22000394,1,0,1,0
6bba_6479435d,9230,21,23.0,137.44,190.72000000000003,cont,1.9326572542745373,22000393,3,0,2,1
6bba_6479435d,9350,21,40.00000000000001,140.64000000000001,135.35999999999999,cont,2.023891672002247,22000398,2,0,2,0
6bba_6479435d,9441,21,55.16000000000001,135.35999999999999,100.00000000000001,cont,2.8384987229167367,22000403,2,0,2,0
6bba_6479435d,9469,21,57.760000000000005,130.08,76.0,cont,6.69768150183331,22000404,7,2,1,4
6bba_6479435d,9792,22,41.67999999999999,34.56,179.84,cont,2.6094282180201946,23000420,1,1,0,0
6bba_6479435d,9856,22,53.84,108.0,76.0,cont,4.2345517029551,23000423,4,2,2,0
6bba_6479435d,10078,23,21.52,53.440000000000005,56.16000000000001,cont,6.565502765401905,24000432,1,0,1,0
6bba_6479435d,10190,23,39.96,34.07999999999999,180.16000000000003,cont,3.510188073664993,24000439,2,1,1,0
6bba_6479435d,10275,23,55.133333333333326,112.53333333333333,77.06666666666665,cont,1.500377556650327,24000442,9,2,0,7
6bba_6479435d,10980,25,31.64,12.0,182.72000000000003,cont,2.936038200109113,26000472,1,0,0,1
6bba_6479435d,11095,25,50.0,105.28000000000002,80.0,cont,3.795305694209623,26000478,1,0,1,0
6bba_6479435d,11291,26,17.68,4.640000000000001,21.92,cont,1.1529486653793395,27000485,2,2,0,0
6bba_6479435d,11296,26,17.200000000000003,78.08000000000001,203.36,cont,2.486303104008035,27000484,1,1,0,0
6bba_6479435d,11493,26,52.2,137.28,110.56,cont,1.2918494494328647,27000497,2,2,0,0
6bba_6479435d,11678,27,15.880000000000003,129.44000000000003,205.12,cont,2.9091131058279562,28000503,7,1,3,3
6bba_6479435d,11754,27,28.839999999999996,14.240000000000002,177.92,cont,3.378319894932978,28000508,2,0,0,2
6bba_6479435d,11786,27,35.120000000000005,29.279999999999994,177.28,cont,5.056910185577752,28000509,1,1,0,0
6bba_6479435d,12506,29,11.920000000000002,36.64,49.120000000000005,cont,6.080280112790854,30000544,2,0,0,2
6bba_6479435d,13032,30,26.48,63.84,117.28,cont,6.853359235623064,31000563,2,0,2,0
6bba_6479435d,13917,32,44.99999999999999,123.46666666666665,100.53333333333332,cont,6.294064112761759,33000600,6,0,0,6
6bba_6479435d,14161,33,12.879999999999999,70.24000000000001,207.99999999999997,cont,2.0749792920653443,34000609,2,2,0,0
6bba_6479435d,14543,34,9.26666666666666,12.533333333333324,39.99999999999999,cont,3.5529782596983135,35000628,2,0,2,0
6bba_6479435d,14964,35,2.6399999999999997,48.0,218.08,cont,2.2589838008494,36000645,2,1,1,0
6bba_6479435d,15067,35,15.719999999999999,137.28,173.76000000000002,cont,1.843426242761022,36000648,6,0,1,5
6bba_6479435d,16352,38,3.2,71.35999999999999,215.99999999999997,cont,1.3865926086994662,39000692,1,0,1,0
6bba_6479435d,25649,61,31.64,99.04,83.99999999999999,cont,4.100413088946041,62000933,2,2,0,0
6bba_6479435d,25887,62,28.240000000000002,108.0,95.36000000000001,cont,6.271152705045547,63000947,3,0,1,2
6bba_6479435d,26363,64,25.480000000000004,95.36000000000001,79.52,cont,3.207224754363176,65000978,1,1,0,0
6bba_6479435d,26581,65,19.52,97.92000000000002,85.28000000000002,cont,4.1192067273323385,66000991,3,0,1,2
6bba_6479435d,26597,65,24.520000000000003,94.72,75.04,cont,6.143467185759198,66000991,6,0,1,5
6bba_6479435d,26964,67,8.440000000000001,132.00000000000006,199.84000000000003,cont,3.332319950124851,68001017,4,4,0,0
6bba_6479435d,27154,68,2.6800000000000006,132.00000000000006,193.92000000000002,cont,1.0597442203192393,69001023,2,0,1,1
6bba_6479435d,27423,69,20.24,98.4,86.39999999999998,cont,1.647272366671649,70001031,2,2,0,0
6bba_6479435d,27615,70,20.52,95.84000000000002,84.47999999999999,cont,5.513373796959536,71001038,7,1,2,4
6bba_6479435d,27807,71,20.520000000000003,96.0,85.28,cont,6.33235207584828,72001047,1,0,1,0
6bba_6479435d,28379,74,33.13333333333333,91.99999999999999,87.99999999999997,cont,1.8176125213526142,75001066,5,0,0,5
6bba_6479435d,28398,74,44.480000000000004,102.72000000000001,97.28000000000003,cont,1.6712238217844935,75001067,2,0,2,0
6bba_6479435d,28729,76,41.0,99.36000000000001,96.0,cont,2.1377244034252842,77001077,1,0,1,0
6bba_6479435d,29260,80,3.04,95.36000000000001,124.00000000000003,cont,0.486686821785845,81001095,2,2,0,0
6bba_6479435d,29789,83,54.480000000000004,98.72000000000001,154.71999999999997,cont,1.6393181700024038,84001114,1,0,1,0
6bba_6479435d,31096,94,9.866666666666662,67.99999999999997,87.46666666666665,cont,1.32901644919175,95001181,2,0,0,2
6bba_67ebd073,752,1,57.13333333333333,191.99999999999994,148.0,cont,2.298735310508328,2000009,3,0,0,3
6bba_67ebd073,785,1,60.6,194.71999999999997,154.4,cont,4.345895333530257,2000009,4,0,1,3
6bba_67ebd073,1531,3,59.84000000000002,195.35999999999999,153.27999999999997,cont,6.263336895178182,4000017,1,0,1,0
6bba_67ebd073,2253,5,57.84,193.27999999999997,152.0,cont,6.241036697737012,6000024,1,1,0,0
6bba_67ebd073,2581,6,50.51999999999999,188.64,148.00000000000003,cont,3.0954010745782337,7000027,2,1,1,0
6bba_67ebd073,4074,10,47.2,204.00000000000003,164.0,cont,6.788630775973909,11000036,1,1,0,0
6bba_67ebd073,4437,11,45.480000000000004,204.00000000000003,163.36,cont,6.824226146604479,12000038,1,0,1,0
6bba_67ebd073,4452,11,46.72,191.36,147.36000000000004,cont,3.180021619423358,12000038,5,0,1,4
6bba_67ebd073,6183,16,36.36,190.56000000000003,149.44,cont,4.4298167639869686,17000055,2,1,1,0
6bba_67ebd073,7469,19,60.92,88.32000000000001,89.27999999999999,cont,0.7625389334322563,20000078,2,2,0,0
6bba_67ebd073,7476,19,61.60000000000001,156.64000000000001,129.28000000000003,cont,6.694230841739786,20000077,1,0,1,0
6bba_67ebd073,7824,20,59.519999999999996,89.28000000000003,88.00000000000001,cont,0.8330403651684568,21000086,2,0,2,0
6bba_67ebd073,8555,22,55.51999999999999,90.72,84.64000000000001,cont,2.911834956260411,23000101,1,1,0,0
6bba_67ebd073,8578,22,60.67999999999999,46.72,64.64,cont,2.2342420414986472,23000102,1,1,0,0
6bba_67ebd073,9434,25,15.32,199.36,152.0,cont,5.725214242716159,26000119,1,1,0,0
6bba_67ebd073,9791,26,13.360000000000001,198.72,152.0,cont,3.299190241559281,27000124,1,0,1,0
6bba_67ebd073,10013,26,57.20000000000002,52.00000000000001,65.44,cont,2.8758827449150477,27000137,1,1,0,0
6bba_67ebd073,10368,27,56.20000000000001,53.280000000000015,66.72,cont,2.7032205907953673,28000151,2,1,1,0
6bba_67ebd073,10970,29,44.00000000000001,54.720000000000006,58.72,cont,6.859213352309418,30000168,1,0,1,0
6bba_67ebd073,11305,30,50.68,57.28000000000001,66.72,cont,6.286713171443409,31000172,1,0,1,0
6bba_67ebd073,11489,31,23.520000000000003,155.35999999999999,119.36000000000001,cont,4.087319185603194,32000174,2,2,0,0
6bba_67ebd073,11595,31,48.2,58.72,65.28,cont,5.4841763852925105,32000182,1,1,0,0
6bba_67ebd073,11781,32,21.44,157.92000000000002,117.91999999999999,cont,0.8074467861723188,33000184,3,1,2,0
6bba_67ebd073,12069,33,19.080000000000005,159.84000000000003,118.07999999999998,cont,1.7881646491584728,34000195,4,3,1,0
6bba_67ebd073,12252,33,61.52000000000001,77.44,103.52,cont,2.5984253043911076,34000206,2,1,1,0
6bba_67ebd073,12355,34,12.0,157.92000000000002,114.07999999999998,cont,3.378788845503667,35000207,9,0,3,6
6bba_67ebd073,12544,34,60.16000000000001,168.0,195.36,cont,6.846478109217912,35000218,1,1,0,0
6bba_67ebd073,12811,35,58.15999999999998,178.72000000000003,181.28,cont,3.95476393890964,36000229,2,0,2,0
6bba_67ebd073,12882,36,4.7333333333333325,154.9333333333333,116.0,cont,4.505693254249445,37000233,6,0,0,6
6bba_67ebd073,12894,36,6.466666666666666,166.93333333333328,131.99999999999997,cont,6.237697962233768,37000233,4,0,0,4
6bba_67ebd073,12914,36,10.879999999999999,162.07999999999998,116.0,cont,6.372258846947446,37000233,6,0,2,4
6bba_67ebd073,13118,37,4.5600000000000005,157.92000000000002,116.0,cont,1.6543904655491688,38000243,1,1,0,0
6bba_67ebd073,13280,37,56.120000000000005,171.52,172.79999999999998,cont,4.500780661451965,38000248,2,2,0,0
6bba_67ebd073,13284,37,57.68000000000001,82.56,99.36,cont,0.8247575401292073,38000249,1,1,0,0
6bba_67ebd073,13353,38,4.839999999999999,161.44000000000005,116.0,cont,1.936547346439009,39000255,1,0,1,0
6bba_67ebd073,13400,38,15.639999999999997,57.279999999999994,54.720000000000006,cont,2.2905584062625453,39000256,3,0,3,0
6bba_67ebd073,13511,38,48.83999999999999,177.28,171.35999999999996,cont,0.38488837667565284,39000260,3,0,2,1
6bba_67ebd073,13515,38,51.03999999999999,80.00000000000003,93.44,cont,4.104918543649782,39000263,2,0,1,1
6bba_67ebd073,14699,43,34.8,232.0,229.75999999999993,cont,0.9279589497924929,44000331,1,1,0,0
6bba_67ebd073,14846,44,25.0,174.9333333333333,167.99999999999994,cont,5.8881184801353275,45000338,2,0,0,2
6bba_67ebd073,14852,44,26.36,90.07999999999998,89.44000000000001,cont,2.2631297466119746,45000344,3,0,1,2
6bba_67ebd073,14880,44,39.519999999999996,124.64000000000001,125.28,cont,5.931450339082339,45000347,2,0,0,2
6bba_67ebd073,15020,45,24.92,177.92,168.32,cont,5.420830482961071,46000354,2,2,0,0
6bba_67ebd073,15026,45,25.72,91.99999999999999,91.36,cont,2.3402256835613073,46000360,2,2,0,0
6bba_67ebd073,15176,46,20.999999999999993,172.53333333333336,164.5333333333333,cont,4.557168212613571,47000369,1,0,0,1
6bba_67ebd073,15178,46,21.52,219.52,207.52000000000004,cont,4.244237453595151,47000371,3,0,2,1
6bba_67ebd073,15183,46,24.680000000000003,94.72,94.08000000000001,cont,2.8426819422861946,47000370,2,0,2,0
6bba_67ebd073,15185,46,24.52,181.44,169.28,cont,2.7020481236462053,47000369,3,0,2,1
6bba_67ebd073,15210,46,40.04,34.56,61.920000000000016,cont,3.268876431589304,47000375,4,0,1,3
6bba_67ebd073,15211,46,40.04,127.52000000000004,130.23999999999995,cont,3.5730051252412087,47000374,3,0,2,1
6bba_67ebd073,15468,48,17.16,87.19999999999999,89.28000000000003,cont,5.364469335358341,49000400,2,2,0,0
6bba_67ebd073,15473,48,19.479999999999997,173.12000000000003,163.84,cont,2.9577232058629046,49000403,2,2,0,0
6bba_67ebd073,15476,48,20.96,221.75999999999996,212.48000000000002,cont,3.215119068557175,49000395,1,1,0,0
6bba_67ebd073,15603,49,11.929575042724611,80.16272338867189,89.00611206054687,cont,6.59531958178153,50000417,1,0,1,0
6bba_67ebd073,15613,49,15.84,88.79999999999998,90.72,cont,1.7201221795267934,50000417,4,0,2,2
6bba_67ebd073,15614,49,15.519999999999998,212.16,200.79999999999998,cont,3.289170203105956,50000422,3,0,0,3
6bba_67ebd073,15625,49,19.0,176.16000000000003,165.44,cont,3.3198157422513654,50000421,4,0,2,2
6bba_67ebd073,15736,50,10.769575042724611,83.36272338867187,89.00611206054687,cont,4.4565269562647485,51000437,1,1,0,0
6bba_67ebd073,15780,50,31.559999999999995,117.28,120.64000000000001,cont,4.860515983411626,51000447,1,1,0,0
6bba_67ebd073,15817,50,60.16000000000001,139.35999999999999,185.28,cont,0.4698470495810472,51000452,1,1,0,0
6bba_67ebd073,15957,51,59.48000000000001,138.72000000000003,183.36,cont,0.8300235312929488,52000472,1,0,1,0
6bba_67ebd073,15960,51,61.160000000000004,96.0,119.36000000000001,cont,0.7151846352516185,52000473,1,1,0,0
6bba_67ebd073,16244,54,5.866666666666667,207.99999999999997,203.99999999999994,cont,4.271446094253243,55000494,2,0,0,2
6bba_67ebd073,16249,54,7.7200000000000015,167.52000000000004,157.44000000000003,cont,1.3275364872574997,55000493,4,0,3,1
6bba_67ebd073,16611,57,4.0,92.0,95.83999999999999,cont,0.8812535460921542,58000531,1,1,0,0
6bba_67ebd073,16721,58,3.3200000000000003,165.27999999999997,158.55999999999997,cont,1.242248465485056,59000545,1,0,1,0
6bba_67ebd073,16727,58,4.133333333333334,75.46666666666667,91.99999999999999,cont,6.303671287542593,59000544,3,0,0,3
6bba_67ebd073,16802,58,48.96000000000001,134.72000000000003,180.0,cont,1.5871850081197396,59000554,1,0,1,0
6bba_67ebd073,17003,60,26.68,220.00000000000003,227.36,cont,3.7526594736266556,61000579,1,1,0,0
6bba_67ebd073,17074,61,3.3999999999999995,212.95999999999998,212.0,cont,1.1052389447535667,62000588,1,1,0,0
6bba_67ebd073,17114,61,25.48,217.28000000000003,224.64000000000001,cont,1.0159824371021355,62000593,1,0,1,0
6bba_67ebd073,17159,61,58.999999999999986,188.00000000000003,243.36000000000004,cont,1.9889495500137833,62000599,1,0,1,0
6bba_67ebd073,17227,62,24.52,215.36000000000004,223.36,cont,2.7162787619277875,63000606,2,2,0,0
6bba_67ebd073,17230,62,26.16,219.52000000000007,241.76000000000005,cont,6.4890423262990256,63000606,1,1,0,0
6bba_67ebd073,17329,63,23.159999999999997,213.28000000000003,220.64000000000004,cont,5.915892789765529,64000617,2,0,2,0
6bba_67ebd073,17757,67,43.16,71.20000000000002,101.92000000000002,cont,2.580119304412101,68000651,1,1,0,0
6bba_67ebd073,18038,70,45.60000000000001,171.99999999999994,232.0,cont,2.631546895364026,71000679,1,1,0,0
6bba_67ebd073,18936,81,34.84,168.15999999999997,226.08,cont,5.496272742732111,82000745,1,1,0,0
6bba_67ebd073,19012,82,33.28000000000001,163.2,221.28000000000003,cont,0.6764763946362141,83000751,1,0,1,0
6bba_67ebd073,19678,90,26.519999999999996,216.0,230.08000000000004,cont,5.78949439178413,91000805,1,0,1,0
6bba_6ca87370,545,10,4.0,24.64,189.28000000000003,cont,1.789714537293579,11000037,1,1,0,0
6bba_6ca87370,760,15,3.4800000000000004,138.08000000000004,232.64,cont,1.0765548348783665,16000059,1,1,0,0
6bba_6ca87370,870,17,14.36,3.3600000000000008,179.36,cont,1.2621763198935374,18000071,1,1,0,0
6bba_6ca87370,1019,20,15.84,8.0,177.28,cont,2.263654746753579,21000089,1,1,0,0
6bba_6ca87370,1074,21,16.52,10.079999999999998,178.08,cont,2.4340851078998873,22000096,1,0,1,0
6bba_6ca87370,1282,25,4.7333333333333325,214.66666666666663,237.86666666666667,cont,0.5138701197773519,26000122,2,0,1,1
6bba_6ca87370,1339,26,5.319999999999999,195.35999999999999,240.0,cont,1.619302511885909,27000128,2,0,1,1
6bba_6ca87370,1389,27,2.5199999999999996,15.52,14.080000000000002,cont,3.1152568112436585,28000133,1,1,0,0
6bba_6ca87370,1396,27,2.7199999999999998,165.76000000000002,135.36,cont,4.6191464647161835,28000134,1,1,0,0
6bba_6ca87370,1417,27,16.268043212890625,9.869622497558595,175.38703613281248,cont,3.0984327430787197,28000138,1,1,0,0
6bba_6ca87370,1746,32,26.84,97.28000000000003,240.00000000000006,cont,1.9971640049079544,33000183,1,1,0,0
6bba_6ca87370,2061,37,25.320000000000004,16.640000000000004,207.36000000000004,cont,2.897152157723164,38000227,1,1,0,0
6bba_6ca87370,2188,39,25.680000000000007,23.200000000000003,206.72000000000008,cont,2.166375501269337,40000246,1,1,0,0
6bba_6ca87370,2253,40,26.000000000000004,29.439999999999998,206.08000000000004,cont,1.9651753261477636,41000256,1,0,1,0
6bba_6ca87370,2288,41,8.16,76.0,138.72000000000003,cont,1.6988829050290606,42000263,1,1,0,0
6bba_6ca87370,2371,42,16.68,97.44,158.71999999999997,cont,2.255708383745557,43000272,1,0,1,0
6bba_6ca87370,2562,45,8.0,42.720000000000006,20.000000000000007,cont,1.6289763848810084,46000301,1,0,1,0
6bba_6ca87370,2620,46,2.8400000000000007,121.28000000000003,10.720000000000002,cont,2.163692229153676,47000309,1,1,0,0
6bba_6ca87370,2953,50,21.84,16.64,98.72000000000001,cont,1.90624344261167,51000355,1,1,0,0
6bba_6ca87370,3026,51,22.16,19.840000000000003,97.28000000000003,cont,1.8110550274632753,52000365,2,1,1,0
6bba_6ca87370,3198,53,32.84,11.36,187.35999999999999,cont,1.1132139562995038,54000388,1,1,0,0
6bba_6ca87370,3275,54,31.000000000000007,78.72,187.35999999999999,cont,1.7124292868611983,55000397,2,1,1,0
6bba_6ca87370,3277,54,33.160000000000004,13.920000000000005,185.28,cont,1.7508573792573707,55000399,1,0,1,0
6bba_6ca87370,3357,55,31.320000000000004,81.28000000000002,186.71999999999997,cont,1.31233213212204,56000408,1,0,1,0
6bba_6ca87370,3440,56,31.84,184.64,198.72000000000003,cont,2.031964874204272,57000421,1,0,1,0
6bba_6ca87370,3575,58,16.040000000000003,25.28000000000001,38.08,cont,6.159350234602673,59000439,2,0,0,2
6bba_6ca87370,3839,61,20.0,14.079999999999998,49.92,cont,4.64286700891809,62000476,1,1,0,0
6bba_6ca87370,3993,63,12.8,186.55999999999997,101.28000000000002,cont,2.9483787918108484,64000499,2,0,2,0
6bba_6ca87370,4103,64,21.000000000000004,25.28,46.08000000000001,cont,3.8197179624286477,65000515,1,1,0,0
6bba_6ca87370,4125,64,28.48,62.56,84.64000000000001,cont,0.9128971738372292,65000516,1,1,0,0
6bba_6ca87370,4137,64,36.160000000000004,43.20000000000001,170.07999999999998,cont,0.46248310779963003,65000520,1,1,0,0
6bba_6ca87370,4189,65,19.68,122.71999999999998,21.28,cont,2.5277500494510936,66000528,1,0,1,0
6bba_6ca87370,4190,65,21.32,30.720000000000006,44.00000000000001,cont,4.825265051010153,66000527,1,0,1,0
6bba_6ca87370,4193,65,20.52,235.36000000000004,104.64000000000003,cont,2.637360482091887,66000529,1,0,1,0
6bba_6ca87370,4213,65,29.0,67.36000000000001,84.0,cont,1.935115064408314,66000531,1,0,1,0
6bba_6ca87370,4263,66,14.84,198.23999999999995,97.28000000000003,parent,1.576836231192082,67000540,1,1,0,0
6bba_6ca87370,4356,67,14.999999999999996,215.99999999999994,95.99999999999997,daughter,3.4946946397360685,68000570,3,0,0,3
6bba_6ca87370,4360,67,15.679999999999998,197.92,96.63999999999999,daughter,2.2463237327909846,68000569,1,0,1,0
6bba_6ca87370,4633,70,14.440000000000001,203.84000000000003,8.0,cont,2.4858782401799284,71000599,1,1,0,0
6bba_6ca87370,4651,70,20.68,137.12000000000003,26.560000000000002,cont,3.880621124575303,71000602,2,2,0,0
6bba_6ca87370,4675,70,29.200000000000003,74.08000000000001,86.56,cont,3.2923799332701544,71000605,3,3,0,0
6bba_6ca87370,4677,70,30.480000000000004,193.12000000000006,114.88,cont,3.6335014534880874,71000606,1,1,0,0
6bba_6ca87370,4686,70,37.160000000000004,57.12,158.88,cont,3.4403978694331228,71000607,2,2,0,0
6bba_6ca87370,4723,71,13.0,246.88000000000005,75.83999999999999,cont,6.197409993295596,72000632,1,0,1,0
6bba_6ca87370,4728,71,15.200000000000005,239.35999999999996,82.56,cont,1.4536258545444136,72000632,4,0,2,2
6bba_6ca87370,4732,71,15.879999999999999,208.79999999999995,8.0,cont,2.7883787852442232,72000614,1,0,1,0
6bba_6ca87370,4733,71,18.04,204.16000000000003,85.92000000000002,cont,4.1448969001049925,72000619,1,0,1,0
6bba_6ca87370,4748,71,23.999999999999993,143.46666666666664,11.999999999999996,cont,0.7436828867945825,72000618,1,0,0,1
6bba_6ca87370,4750,71,24.13333333333334,246.93333333333337,83.99999999999997,cont,1.528469698114926,72000616,1,0,0,1
6bba_6ca87370,4771,71,32.266666666666666,87.99999999999997,67.46666666666667,cont,0.7367463192231692,72000621,1,0,0,1
6bba_6ca87370,4801,71,44.16000000000001,72.16,216.48000000000002,cont,3.0069526205944754,72000629,2,0,2,0
6bba_6ca87370,4960,73,32.8,90.56,62.720000000000006,cont,1.2073214153654344,74000659,2,2,0,0
6bba_6ca87370,5061,74,33.67999999999999,94.07999999999998,60.0,cont,0.6606779188833316,75000679,2,0,2,0
6bba_6ca87370,5079,74,40.16,142.08,142.56,cont,1.8850000000000036,75000687,1,1,0,0
6bba_6ca87370,5176,75,40.88000000000001,143.36,139.35999999999999,cont,2.015327594337929,76000707,1,0,1,0
6bba_6ca87370,5640,80,35.64,105.28000000000002,44.80000000000001,cont,1.188029487218225,81000804,4,4,0,0
6bba_6ca87370,5646,80,38.31999999999999,71.35999999999999,187.20000000000005,cont,1.7897145372935948,81000808,1,1,0,0
6bba_6ca87370,5653,80,40.96,104.64000000000003,190.56,cont,0.9875134492754917,81000809,3,3,0,0
6bba_6ca87370,5665,80,44.48,81.92,177.44000000000003,cont,2.70595437184369,81000810,3,3,0,0
6bba_6ca87370,5667,80,43.84,239.35999999999996,127.2,cont,2.3456918622231724,81000814,1,1,0,0
6bba_6ca87370,5677,80,52.480000000000004,190.56,180.15999999999997,cont,1.7532688220863482,81000816,2,2,0,0
6bba_6ca87370,5680,80,57.48,229.92,204.16000000000003,cont,3.1791080706701287,81000817,2,2,0,0
6bba_6ca87370,5706,81,29.519999999999996,18.720000000000002,25.28,cont,2.4778985552278043,82000819,1,0,1,0
6bba_6ca87370,5740,81,38.86666666666666,83.99999999999997,171.46666666666664,cont,0.3064129385141883,82000827,5,0,0,5
6bba_6ca87370,5741,81,39.480000000000004,107.35999999999999,164.8,cont,2.865027541316138,82000828,3,0,2,1
6bba_6ca87370,5758,81,44.68,83.99999999999999,171.84000000000003,cont,2.7739468722381915,82000829,3,0,3,0
6bba_6ca87370,5771,81,53.0,193.44,172.48000000000005,cont,2.840962919944589,82000836,2,0,2,0
6bba_6ca87370,5774,81,58.0,232.0,196.48000000000002,cont,3.6142644912208644,82000837,2,0,2,0
6bba_6ca87370,6041,84,41.86666666666666,119.46666666666667,148.00000000000003,cont,6.783712750445388,85000887,2,0,0,2
6bba_6ca87370,6065,84,54.32000000000001,197.28,162.72,cont,1.2557791406134982,85000896,1,1,0,0
6bba_6ca87370,6069,84,58.999999999999986,235.99999999999994,186.07999999999998,cont,1.667427378478623,85000897,1,1,0,0
6bba_6ca87370,6162,85,59.160000000000004,235.99999999999994,182.71999999999997,cont,1.3697313833376195,86000917,1,0,1,0
6bba_6ca87370,6311,87,40.52,20.000000000000004,221.44000000000003,cont,1.292360364797681,88000946,1,0,1,0
6bba_6ca87370,6430,88,49.20000000000001,246.72000000000006,92.63999999999999,cont,1.3131367503043825,89000977,2,2,0,0
6bba_6ca87370,6434,88,51.68,171.2,234.88,cont,3.01537129761825,89000978,1,1,0,0
6bba_6ca87370,6516,89,48.48000000000002,248.00000000000006,90.07999999999998,cont,5.805436164708012,90000996,2,0,2,0
6bba_6ca87370,6546,90,32.68,26.080000000000002,116.63999999999999,cont,2.455525020234983,91001006,1,0,1,0
6bba_6ca87370,6843,93,42.84,132.00000000000003,12.159999999999998,cont,2.743171351283039,94001069,1,1,0,0
6bba_6ca87370,6859,93,47.36,165.28000000000003,80.8,cont,2.82605207179557,94001073,1,1,0,0
6bba_6ca87370,6862,93,49.32,191.84,163.52,cont,2.508717924558275,94001074,1,1,0,0
6bba_6ca87370,6939,94,46.47999999999999,166.72000000000003,76.63999999999999,cont,6.304267032137842,95001091,2,0,1,1
6bba_6ca87370,6946,94,50.0,173.06666666666666,71.46666666666665,cont,1.2601755967809525,95001091,2,0,0,2
6bba_6ca87370,6952,94,52.733333333333306,146.4,248.5333333333334,cont,1.9305307002796572,95001094,1,0,0,1
6bba_6db0e9b4,4555,13,48.16,100.96000000000002,81.92000000000002,cont,4.688934613800886,14000071,2,2,0,0
6bba_6db0e9b4,4560,13,50.92,38.72,98.08,cont,0.6534440010437002,14000072,4,4,0,0
6bba_6db0e9b4,4572,13,49.84,103.84,69.92,cont,5.268133712473513,14000071,1,1,0,0
6bba_6db0e9b4,4615,13,56.36,71.20000000000002,23.200000000000003,cont,4.382863290703927,14000073,1,1,0,0
6bba_6db0e9b4,4961,14,46.800000000000004,57.92000000000001,148.15999999999997,cont,6.7348359761021666,15000078,5,0,2,3
6bba_6db0e9b4,4962,14,45.199999999999996,70.72,157.92000000000002,cont,3.289491317058014,15000078,11,0,4,7
6bba_6db0e9b4,4965,14,46.480000000000004,104.0,85.44000000000001,cont,2.8909476171317894,15000079,2,0,2,0
6bba_6db0e9b4,4971,14,46.68000000000001,49.28,115.2,cont,4.139701453607479,15000080,9,0,5,4
6bba_6db0e9b4,4978,14,48.23999999999998,36.31999999999999,120.63999999999999,cont,6.58768666718447,15000080,9,0,4,5
6bba_6db0e9b4,4982,14,48.26666666666665,91.46666666666663,90.93333333333331,cont,5.570614399836593,15000079,7,0,0,7
6bba_6db0e9b4,4983,14,48.480000000000004,106.08,72.16,cont,6.899536442761937,15000079,1,0,1,0
6bba_6db0e9b4,5019,14,54.96000000000001,74.07999999999998,22.72,cont,4.093871372246559,15000081,1,0,1,0
6bba_6db0e9b4,5369,15,44.72,69.44,158.56,cont,4.232056607017444,16000086,2,2,0,0
6bba_6db0e9b4,5372,15,45.519999999999996,107.2,87.35999999999999,cont,4.181336882505406,16000087,1,1,0,0
6bba_6db0e9b4,5378,15,46.35999999999999,50.56000000000002,114.56000000000002,cont,4.571365017694395,16000088,2,2,0,0
6bba_6db0e9b4,5387,15,47.519999999999996,93.28000000000002,93.91999999999999,cont,5.222346515217077,16000087,3,3,0,0
6bba_6db0e9b4,5388,15,47.68000000000001,108.63999999999999,73.44,cont,6.51270321180076,16000087,1,1,0,0
6bba_6db0e9b4,5397,15,49.68000000000001,69.28,166.56,cont,6.580628057032263,16000086,3,3,0,0
6bba_6db0e9b4,5761,16,41.86666666666666,75.99999999999999,8.533333333333333,cont,2.099264090633261,17000093,1,0,0,1
6bba_6db0e9b4,5762,16,41.84000000000001,84.47999999999999,165.12000000000003,cont,5.474803221349584,17000094,7,0,2,5
6bba_6db0e9b4,5770,16,43.8,66.4,160.79999999999998,cont,6.26257796857652,17000094,6,0,2,4
6bba_6db0e9b4,5772,16,44.11999999999999,112.00000000000001,89.28000000000003,cont,3.362532992403212,17000095,1,0,1,0
6bba_6db0e9b4,5783,16,45.48000000000001,52.0,114.07999999999998,cont,1.149967390842013,17000096,2,0,2,0
6bba_6db0e9b4,5795,16,46.519999999999996,45.92,130.56,cont,6.465458220420265,17000096,6,0,2,4
6bba_6db0e9b4,6589,18,44.00000000000001,58.55999999999998,115.2,cont,3.2942241119875115,19000111,1,1,0,0
6bba_6db0e9b4,7362,20,42.32000000000001,92.80000000000001,176.64000000000001,cont,2.4173752553130936,21000125,3,0,0,3
6bba_6db0e9b4,7720,21,40.64000000000001,118.07999999999998,94.72,cont,4.422687602013055,22000136,6,0,0,6
6bba_6db0e9b4,8446,23,42.040000000000006,69.92,123.19999999999999,cont,3.207883355890598,24000155,1,1,0,0
6bba_6db0e9b4,8818,24,41.32,72.00000000000001,125.43999999999998,cont,4.400870830017627,25000163,1,0,1,0
6bba_6db0e9b4,9154,25,38.47999999999999,100.64000000000001,8.64,cont,2.4719770048485343,26000170,1,0,1,0
6bba_6db0e9b4,9168,25,40.64000000000001,73.44,126.72,cont,2.6122096130479227,26000171,1,1,0,0
6bba_6db0e9b4,9837,27,36.36000000000001,104.00000000000001,192.0,cont,3.226392383529945,28000185,1,1,0,0
6bba_6db0e9b4,9856,27,38.88000000000001,75.2,129.92000000000002,cont,3.5068765037423115,28000187,1,1,0,0
6bba_6db0e9b4,9861,27,40.32,111.19999999999997,94.08000000000004,cont,1.9653768722563043,28000188,2,2,0,0
6bba_6db0e9b4,10164,28,35.32000000000001,106.07999999999998,194.72000000000003,cont,2.718028065068509,29000193,1,0,1,0
6bba_6db0e9b4,10176,28,37.64,76.0,132.00000000000003,cont,1.8201450835029516,29000195,1,0,1,0
6bba_6db0e9b4,10187,28,37.2,118.56000000000003,109.75999999999999,cont,4.0153286369237575,29000196,5,0,0,5
6bba_6db0e9b4,10203,28,40.000000000000014,113.44,91.84,cont,5.238880038949171,29000196,6,0,2,4
6bba_6db0e9b4,10445,29,25.799999999999997,244.0,27.36,cont,1.9972301131817563,30000199,1,0,1,0
6bba_6db0e9b4,10507,29,37.12,76.8,134.07999999999998,cont,1.92087097757763,30000203,1,1,0,0
6bba_6db0e9b4,10825,30,34.64,109.28,198.72,cont,2.047048561343871,31000209,2,2,0,0
6bba_6db0e9b4,10838,30,36.80000000000001,76.63999999999999,135.35999999999999,cont,2.2930356517071306,31000211,1,0,1,0
6bba_6db0e9b4,11138,31,29.133333333333333,103.46666666666663,202.9333333333333,cont,6.415320813944797,32000217,4,0,0,4
6bba_6db0e9b4,11165,31,34.68,110.71999999999998,199.36,cont,3.4019987231772966,32000217,2,0,2,0
6bba_6db0e9b4,12149,34,28.519999999999996,108.80000000000001,212.0,cont,5.733832705965539,35000241,1,1,0,0
6bba_6db0e9b4,12468,35,23.68,249.28000000000003,23.36,cont,2.1697857757161207,36000247,1,1,0,0
6bba_6db0e9b4,12498,35,28.680000000000007,108.00000000000001,213.92000000000002,cont,5.466211124947506,36000249,1,0,1,0
6bba_6db0e9b4,12816,36,22.68,249.92000000000004,24.0,cont,2.1833736080890995,37000256,1,0,1,0
6bba_6db0e9b4,13178,37,29.360000000000007,104.00000000000001,216.0,cont,1.3808675941233388,38000264,3,0,0,3
6bba_6db0e9b4,13506,38,30.0,113.28,215.36000000000004,cont,5.214705768305623,39000270,1,0,1,0
6bba_6db0e9b4,13555,38,36.83999999999999,120.00000000000003,4.0,cont,5.43600755380269,39000272,1,0,1,0
6bba_6db0e9b4,14143,40,28.04,82.72,218.72000000000003,cont,3.411106858484492,41000283,6,0,2,4
6bba_6db0e9b4,16192,47,31.159999999999997,98.72,152.0,cont,1.4606933285258845,48000324,1,1,0,0
6bba_6db0e9b4,16200,47,32.32,128.64000000000004,122.07999999999997,cont,4.399790655531253,48000323,1,1,0,0
6bba_6db0e9b4,16472,48,29.133333333333333,127.99999999999997,127.46666666666667,cont,1.5926289630321542,49000328,5,0,0,5
6bba_6db0e9b4,16488,48,31.000000000000007,99.36000000000001,150.72,cont,2.384379505238201,49000329,1,0,1,0
6bba_6db0e9b4,16501,48,32.84,127.36,119.36000000000001,cont,6.054471746155895,49000328,1,0,1,0
6bba_6db0e9b4,16512,48,34.999999999999986,103.99999999999999,144.0,cont,6.918188211157584,49000329,3,0,0,3
6bba_6db0e9b4,17066,50,29.200000000000003,128.64000000000001,126.56,cont,1.7256398269917128,51000339,1,1,0,0
6bba_6db0e9b4,17092,50,34.36,110.08,138.56,cont,6.372735382078877,51000340,1,1,0,0
6bba_6db0e9b4,17143,50,44.160000000000004,88.79999999999998,40.64000000000001,cont,2.2843824384283824,51000341,1,1,0,0
6bba_6db0e9b4,17347,51,25.86666666666666,95.46666666666667,233.06666666666666,cont,1.9918682342213399,52000343,2,0,0,2
6bba_6db0e9b4,17361,51,28.48,129.44000000000003,127.99999999999997,cont,2.5135025984669284,52000344,1,0,1,0
6bba_6db0e9b4,17460,51,44.52,85.28,40.64000000000001,cont,2.5087179245582796,52000346,1,0,1,0
6bba_6db0e9b4,23244,72,29.640000000000008,90.72,179.36,cont,1.4783035040207366,73000442,1,1,0,0
6bba_6db0e9b4,23495,73,27.480000000000004,116.63999999999999,151.36,cont,1.0859572390292247,74000445,1,0,1,0
6bba_6db0e9b4,23516,73,29.839999999999996,89.27999999999999,179.20000000000005,cont,1.9503385122844776,74000446,1,0,1,0
6bba_6db0e9b4,24778,78,29.040000000000006,86.72,185.91999999999996,cont,1.7019111500016613,79000464,1,1,0,0
6bba_6db0e9b4,25036,79,27.879999999999995,87.2,187.35999999999999,cont,3.449060107478565,80000468,2,1,1,0
6bba_6db0e9b4,25282,80,26.08,89.92,188.0,cont,2.264412865402424,81000471,1,0,1,0
6bba_6db0e9b4,25883,83,2.0000000000000004,236.00000000000006,36.64000000000001,cont,3.3423686399019514,84000481,1,0,1,0
6bba_6db0e9b4,26129,84,2.16,236.64000000000001,35.36,cont,3.0668188934138265,85000485,1,1,0,0
6bba_6db0e9b4,26388,85,2.84,235.99999999999994,32.64,cont,0.5040120286263245,86000489,1,0,1,0
6bba_6db0e9b4,27458,89,3.6799999999999997,238.56000000000006,26.720000000000002,cont,1.5728959159779003,90000505,3,2,1,0
6bba_6db0e9b4,27714,90,3.3200000000000003,241.43999999999997,25.28,cont,1.5791789797233253,91000509,2,0,2,0
6bba_6db0e9b4,27738,90,6.959999999999999,235.84000000000003,29.439999999999998,cont,5.453515494614452,91000509,3,0,0,3
6bba_6db0e9b4,27986,91,3.0,243.35999999999993,24.64,cont,1.9045804163909745,92000513,1,1,0,0
6bba_6db0e9b4,28261,92,2.8400000000000007,244.64000000000001,24.0,cont,2.2596850582548,93000517,2,1,1,0
6bba_6db0e9b4,28273,92,5.320000000000001,228.64000000000007,32.0,cont,6.903955501196368,93000517,2,1,1,0
6bba_6db0e9b4,28514,93,2.6399999999999997,246.08,23.360000000000007,cont,1.2221037190026043,94000521,1,0,1,0
6bba_6feb10f0,293,1,22.599999999999994,157.06666666666666,218.93333333333328,cont,6.150839063132394,2000036,2,1,0,1
6bba_6feb10f0,588,2,41.999999999999986,23.46666666666666,175.99999999999994,cont,5.36728532277829,3000064,1,0,0,1
6bba_6feb10f0,603,2,44.99999999999999,87.99999999999997,232.0,cont,6.809990937035102,3000067,3,0,0,3
6bba_6feb10f0,985,4,41.84000000000001,26.719999999999995,176.00000000000003,cont,6.945900881995064,5000110,1,0,1,0
6bba_6feb10f0,1419,6,40.68,111.52000000000001,227.20000000000002,cont,6.428369374304809,7000158,3,0,0,3
6bba_6feb10f0,1444,6,49.16,211.36,244.64000000000001,cont,0.9037396264964883,7000162,1,1,0,0
6bba_6feb10f0,1550,7,20.520000000000003,237.27999999999997,237.9200000000001,cont,4.539308868098744,8000175,3,3,0,0
6bba_6feb10f0,1614,7,41.16,115.35999999999999,227.20000000000002,cont,6.83731181185267,8000182,2,2,0,0
6bba_6feb10f0,1747,8,25.0,232.5333333333333,221.59999999999997,cont,5.009318573363495,9000200,3,0,0,3
6bba_6feb10f0,1807,8,39.999999999999986,12.533333333333331,239.99999999999994,cont,6.7289412371941095,9000206,1,0,0,1
6bba_6feb10f0,1812,8,40.86666666666666,99.46666666666667,223.46666666666664,cont,3.478600877148759,9000208,3,1,0,2
6bba_6feb10f0,1900,8,59.83999999999999,181.92000000000004,34.559999999999995,cont,1.9351150644083372,9000216,1,0,1,0
6bba_6feb10f0,2040,9,41.239999999999995,100.96000000000002,224.96,cont,3.126932402531289,10000232,2,1,1,0
6bba_6feb10f0,2062,9,49.16,225.28,245.28,cont,6.342789459299426,10000235,1,1,0,0
6bba_6feb10f0,2072,9,50.12,210.23999999999998,246.08,cont,4.027706000318289,10000235,1,1,0,0
6bba_6feb10f0,2253,10,41.68000000000001,102.72000000000003,226.72,cont,3.349905082610548,11000256,1,0,1,0
6bba_6feb10f0,2271,10,47.26666666666665,211.99999999999994,243.99999999999997,cont,1.7522728296256966,11000261,2,0,0,2
6bba_6feb10f0,2284,10,50.39999999999998,210.93333333333328,245.0666666666666,cont,4.235577038282016,11000261,1,0,1,0
6bba_6feb10f0,2531,11,59.84000000000002,25.92,205.44000000000003,cont,6.964902413171361,12000289,1,0,1,0
6bba_6feb10f0,2669,12,42.0,104.0,227.99999999999997,cont,4.737648414561799,13000313,1,0,1,0
6bba_6feb10f0,2912,13,53.31999999999999,61.91999999999999,146.72,cont,2.8600461644001673,14000344,1,1,0,0
6bba_6feb10f0,3085,14,42.480000000000004,101.44,226.72,cont,2.8180517427825933,15000359,1,1,0,0
6bba_6feb10f0,3102,14,48.0,227.99999999999997,244.00000000000003,cont,6.499999999999986,15000362,1,1,0,0
6bba_6feb10f0,3104,14,48.68,225.44000000000003,76.0,cont,1.1908046124364868,15000363,1,1,0,0
6bba_6feb10f0,3114,14,53.0,60.0,147.36,cont,5.027162780833738,15000366,1,0,1,0
6bba_6feb10f0,3267,15,35.86666666666665,139.46666666666664,211.99999999999994,cont,6.653521219478835,16000376,3,0,0,3
6bba_6feb10f0,3288,15,42.760000000000005,7.999999999999999,178.39999999999998,cont,1.6087499999999937,16000378,1,0,1,0
6bba_6feb10f0,3291,15,42.999999999999986,98.08000000000001,225.28000000000003,cont,2.945376938950252,16000379,1,0,1,0
6bba_6feb10f0,3297,15,45.160000000000004,206.08,243.36000000000004,cont,3.839988810205048,16000382,2,1,1,0
6bba_6feb10f0,3303,15,47.480000000000004,227.99999999999997,244.00000000000003,cont,6.559225492579123,16000382,2,1,1,0
6bba_6feb10f0,3365,15,60.2,186.07999999999998,88.64,cont,4.655702988002994,16000389,1,0,1,0
6bba_6feb10f0,3516,16,44.839999999999996,204.64000000000001,241.92000000000007,cont,5.765109766951537,17000403,1,0,1,0
6bba_6feb10f0,3531,16,49.36000000000001,206.24,242.71999999999997,cont,2.9387800402548083,17000403,3,0,0,3
6bba_6feb10f0,3553,16,58.99999999999999,11.999999999999996,211.9999999999999,cont,5.218375735082705,17000407,2,0,0,2
6bba_6feb10f0,3920,18,42.866666666666674,79.46666666666665,232.53333333333336,cont,5.133131252786498,19000445,3,0,0,3
6bba_6feb10f0,4123,19,38.199999999999996,87.03999999999998,227.99999999999997,cont,6.200626102459335,20000478,1,1,0,0
6bba_6feb10f0,4338,20,38.4,88.5333333333333,227.99999999999997,cont,5.854010970646063,21000490,1,0,1,0
6bba_6feb10f0,5550,26,39.32,78.24000000000001,221.6,cont,4.258771279665069,27000616,3,0,0,3
6bba_6feb10f0,6138,29,40.199999999999996,65.92,236.64000000000004,cont,5.929802905662215,30000678,2,1,1,0
6bba_6feb10f0,6675,32,38.839999999999996,68.0,235.99999999999994,cont,5.437828869365028,33000741,2,2,0,0
6bba_6feb10f0,6676,32,38.839999999999996,79.36,226.08,cont,3.951824934444842,33000741,1,1,0,0
6bba_6feb10f0,6863,33,38.76,64.0,215.2,cont,6.189437070525882,34000763,3,0,0,3
6bba_6feb10f0,6874,33,39.00000000000001,78.08,224.64000000000001,cont,5.5692711596761155,34000763,1,0,1,0
6bba_6feb10f0,7597,37,14.320000000000002,188.64000000000004,168.00000000000003,cont,0.9990776996810791,38000844,1,1,0,0
6bba_6feb10f0,7644,37,27.999999999999996,204.53333333333325,131.46666666666664,cont,0.6252673039500258,38000850,3,0,0,3
6bba_6feb10f0,7663,37,33.96000000000001,11.36,187.35999999999999,cont,2.109560454810421,38000851,1,1,0,0
6bba_6feb10f0,7801,38,13.84,189.28000000000003,166.55999999999997,cont,1.7198151245119289,39000860,1,0,1,0
6bba_6feb10f0,8236,40,19.84,17.44,235.35999999999999,cont,5.106043753484687,41000904,2,2,0,0
6bba_6feb10f0,8349,40,57.68000000000001,220.64,44.63999999999999,cont,1.2985773465604735,41000923,1,0,1,0
6bba_6feb10f0,9831,48,49.64,25.92,227.99999999999997,cont,1.6290574345307791,49001099,1,0,0,1
6bba_6feb10f0,10197,50,58.36,38.72,130.56,cont,1.4111326346236954,51001144,1,0,1,0
6bba_6feb10f0,10409,52,8.680000000000001,17.28,208.0,cont,0.596620691897287,53001167,1,1,0,0
6bba_6feb10f0,10701,53,54.866666666666646,200.0,243.99999999999997,cont,3.4650793892219722,54001200,1,0,0,1
6bba_6feb10f0,11169,56,30.160000000000004,19.359999999999996,200.64,cont,2.1198621712271675,57001255,1,1,0,0
6bba_6feb10f0,11454,58,12.479999999999997,23.04,245.44000000000003,cont,4.66960651447208,59001293,1,1,0,0
6bba_6feb10f0,11625,59,12.733333333333333,24.53333333333333,245.0666666666666,cont,4.042680822074207,60001311,1,0,1,0
6bba_6feb10f0,11663,59,27.26666666666666,87.99999999999997,239.46666666666664,cont,3.9378086298797697,60001314,1,0,1,0
6bba_6feb10f0,11681,59,31.2,98.39999999999999,233.27999999999997,cont,4.430144608249256,60001314,3,0,0,3
6bba_6feb10f0,11753,59,58.86666666666668,207.99999999999997,107.46666666666665,cont,6.590382894899904,60001323,1,0,0,1
6bba_6feb10f0,12377,63,57.160000000000004,27.999999999999996,136.0,cont,2.160761222925843,64001389,1,0,1,0
6bba_6feb10f0,12421,64,6.0,244.00000000000003,164.64,cont,2.047822395253072,65001394,1,0,1,0
6bba_6feb10f0,13223,69,34.133333333333326,137.06666666666666,175.4666666666667,cont,3.852026996162754,70001484,1,0,0,1
6bba_6feb10f0,13263,69,57.32000000000001,22.72,140.64000000000004,cont,3.5751477242206393,70001489,1,0,1,0
6bba_6feb10f0,13270,69,58.0,210.71999999999997,115.35999999999999,cont,6.811541790409859,70001486,1,0,1,0
6bba_6feb10f0,14201,76,34.32000000000001,129.44000000000003,147.35999999999996,cont,3.640906695110428,77001587,1,1,0,0
6bba_6feb10f0,14240,76,55.16000000000001,12.639999999999997,150.07999999999998,cont,5.5426794343981145,77001592,1,1,0,0
6bba_6feb10f0,14286,77,4.0,245.28,173.27999999999997,cont,2.1045475315373494,78001608,1,1,0,0
6bba_6feb10f0,14379,77,54.84,12.0,149.28,cont,4.611393837550632,78001603,1,0,1,0
6bba_6feb10f0,14431,78,4.0,244.64000000000001,172.64000000000004,cont,1.952503601149058,79001622,1,0,1,0
6bba_6feb10f0,14492,78,33.31999999999999,140.00000000000003,176.00000000000003,cont,5.531543918744559,79001614,1,0,1,0
6bba_6feb10f0,14539,78,57.16000000000002,239.35999999999996,89.92,cont,2.642361928748592,79001621,1,0,1,0
6bba_6feb10f0,14677,79,54.31999999999999,12.0,148.00000000000003,cont,3.7489281801603136,80001630,1,1,0,0
6bba_6feb10f0,14711,79,58.99999999999999,212.5333333333333,116.5333333333333,cont,6.492830247130698,80001631,3,0,0,3
6bba_6feb10f0,14746,80,15.0,29.28,247.36,cont,3.4768924702958546,81001650,1,0,1,0
6bba_6feb10f0,14816,80,54.08,11.999999999999993,148.0,cont,4.2990080614602295,81001645,1,0,1,0
6bba_6feb10f0,14833,80,58.266666666666666,7.999999999999998,143.46666666666664,cont,5.881824124769651,81001645,2,0,0,2
6bba_6feb10f0,15493,85,35.52,132.00000000000006,196.64000000000001,cont,6.159693199543293,86001713,1,0,1,0
6bba_6feb10f0,15943,88,58.47999999999999,204.64,105.44000000000001,cont,5.952492781389981,89001761,1,0,1,0
6bba_6feb10f0,15947,88,58.31999999999999,244.0,82.56000000000002,cont,2.0455000152774283,89001762,1,0,0,1
6bba_6feb10f0,16201,90,52.599999999999994,16.533333333333335,152.53333333333325,cont,6.549688552815982,91001789,1,0,0,1
6bba_6feeb0b1,727,9,2.4799999999999995,199.36,85.92,cont,1.6159560830975652,10000160,1,1,0,0
6bba_6feeb0b1,823,10,6.5200000000000005,138.56,180.0,cont,0.9544713851132476,11000173,2,2,0,0
6bba_6feeb0b1,868,10,32.16000000000001,14.719999999999999,187.36,cont,1.6560655557676305,11000186,1,0,1,0
6bba_6feeb0b1,870,10,38.0,20.000000000000004,63.36000000000001,cont,1.695070518444586,11000187,1,1,0,0
6bba_6feeb0b1,889,11,5.800000000000001,146.08000000000007,183.36,cont,1.5381807436058854,12000188,2,0,2,0
6bba_6feeb0b1,904,11,8.680000000000001,169.28000000000003,240.64000000000004,cont,1.3404033907745811,12000192,1,1,0,0
6bba_6feeb0b1,1083,13,24.160000000000004,28.000000000000004,99.36000000000001,cont,4.943232905953348,14000234,1,1,0,0
6bba_6feeb0b1,1538,19,26.359999999999996,34.56,86.08,cont,1.7816552170664266,20000334,1,1,0,0
6bba_6feeb0b1,1616,20,26.16,38.080000000000005,83.35999999999999,cont,1.3731971134909964,21000348,1,0,1,0
6bba_6feeb0b1,1841,23,26.480000000000004,8.0,169.92000000000002,cont,0.8456247690317414,24000396,1,1,0,0
6bba_6feeb0b1,1848,23,26.36,25.28,121.28000000000003,cont,1.1683060654212185,24000397,1,1,0,0
6bba_6feeb0b1,1935,24,34.839999999999996,36.64,78.56,cont,2.1578262354740287,25000420,1,1,0,0
6bba_6feeb0b1,2017,25,34.52,41.280000000000015,81.92,cont,2.4887977142588302,26000440,1,0,1,0
6bba_6feeb0b1,2099,26,34.0,45.92000000000001,86.71999999999998,cont,2.015131044125918,27000457,1,1,0,0
6bba_6feeb0b1,2154,27,20.640000000000004,216.16000000000003,9.440000000000001,cont,0.8649214198411285,28000464,1,1,0,0
6bba_6feeb0b1,2237,28,20.84,219.36,10.719999999999999,cont,2.345635574849601,29000481,1,0,1,0
6bba_6feeb0b1,2264,28,43.16000000000001,8.64,62.720000000000006,cont,1.358212290107847,29000488,1,1,0,0
6bba_6feeb0b1,2347,29,42.999999999999986,10.720000000000002,64.0,cont,1.6788695973779775,30000507,1,0,1,0
6bba_6feeb0b1,2730,34,17.0,192.95999999999995,137.28,cont,4.741130699790913,35000581,1,1,0,0
6bba_6feeb0b1,2737,34,21.799999999999997,229.11999999999998,13.280000000000001,cont,3.3552610274760886,35000585,1,1,0,0
6bba_6feeb0b1,2753,34,27.52,75.68,220.0,cont,2.7429788210082884,35000588,2,2,0,0
6bba_6feeb0b1,2758,34,32.04,64.64,99.36,cont,2.402198537277884,35000593,1,1,0,0
6bba_6feeb0b1,2766,34,42.68000000000001,14.560000000000002,72.32000000000001,cont,3.7210016292659613,35000594,2,2,0,0
6bba_6feeb0b1,2807,35,17.16,202.23999999999998,138.72000000000003,cont,2.323295410080251,36000603,1,0,1,0
6bba_6feeb0b1,2823,35,22.519999999999996,233.43999999999994,14.719999999999999,cont,2.0761879129789986,36000606,1,0,1,0
6bba_6feeb0b1,2831,35,27.480000000000004,82.88,220.0,cont,3.872412846017325,36000609,2,0,2,0
6bba_6feeb0b1,2852,35,42.32,18.080000000000002,78.88,cont,2.968105982777569,36000614,2,0,2,0
6bba_6feeb0b1,2932,36,42.16000000000001,20.000000000000004,84.16000000000003,cont,1.9766974509266524,37000633,1,1,0,0
6bba_6feeb0b1,3012,37,42.0,23.360000000000007,88.64,cont,1.9643017207140032,38000653,1,0,1,0
6bba_6feeb0b1,3249,40,40.64000000000001,31.360000000000007,94.72,cont,1.0278686990564472,41000710,1,1,0,0
6bba_6feeb0b1,3324,41,40.32,34.720000000000006,95.36000000000001,cont,1.8170958973042708,42000730,1,0,1,0
6bba_6feeb0b1,3356,42,16.68,229.28000000000003,145.28,cont,0.8784023138630712,43000733,1,1,0,0
6bba_6feeb0b1,3375,42,24.520000000000003,81.92,165.28000000000003,cont,0.9380065298280185,43000744,1,1,0,0
6bba_6feeb0b1,3428,43,16.32,232.0,145.44,cont,1.8200725431971108,44000752,1,0,1,0
6bba_6feeb0b1,3661,46,11.32,232.0,69.12,cont,2.417539102579314,47000810,1,1,0,0
6bba_6feeb0b1,3698,46,26.000000000000007,116.63999999999999,225.92000000000002,cont,1.673828881636339,47000820,1,1,0,0
6bba_6feeb0b1,3738,47,11.13333333333333,227.99999999999994,73.06666666666662,cont,3.8287251513685803,48000829,2,0,0,2
6bba_6feeb0b1,3739,47,11.159999999999998,236.79999999999995,75.52000000000001,cont,2.6515399417131165,48000829,2,0,1,1
6bba_6feeb0b1,3766,47,21.0,232.0,68.64,cont,1.2274936354213815,48000835,1,0,1,0
6bba_6feeb0b1,3774,47,25.8,120.80000000000005,228.8,cont,1.068676898084715,48000839,1,0,1,0
6bba_6feeb0b1,3801,47,38.0,27.360000000000003,165.44000000000005,cont,1.4939398414929397,48000844,2,1,1,0
6bba_6feeb0b1,3886,48,38.0,13.280000000000001,236.64000000000004,cont,0.6758906161502953,49000864,1,1,0,0
6bba_6feeb0b1,4340,54,16.999999999999993,207.99999999999997,107.46666666666665,cont,4.895570028380543,55000957,2,0,0,2
6bba_6feeb0b1,4678,58,20.320000000000004,243.36000000000004,202.72000000000003,cont,1.2083052439263968,59001027,1,1,0,0
6bba_6feeb0b1,4762,59,20.999999999999996,246.08000000000004,201.28000000000006,cont,0.47460016066155725,60001044,1,0,1,0
6bba_6feeb0b1,4812,59,39.160000000000004,20.799999999999997,144.0,cont,2.542540867419838,60001053,1,0,1,0
6bba_6feeb0b1,4830,59,56.99999999999999,26.72,4.0,cont,1.9140541071244441,60001057,1,0,1,0
6bba_6feeb0b1,5055,62,37.96,30.080000000000005,144.16,cont,1.9608707408954833,63001101,1,1,0,0
6bba_6feeb0b1,5068,62,44.16,2.72,162.72,cont,1.4651157164197035,63001105,1,1,0,0
6bba_6feeb0b1,5142,63,38.0,31.360000000000003,143.36,cont,1.103565245012723,64001117,1,0,1,0
6bba_6feeb0b1,5152,63,44.16000000000001,3.36,162.71999999999994,cont,1.6054638153817022,64001120,1,0,1,0
6bba_6feeb0b1,5228,64,42.0,21.920000000000005,104.64,cont,0.26202337681970056,65001134,1,1,0,0
6bba_6feeb0b1,5695,70,42.999999999999986,9.920000000000002,160.00000000000006,cont,0.5979470085216767,71001244,1,0,1,0
6bba_6feeb0b1,5838,72,37.160000000000004,99.36000000000001,169.28,cont,1.5468258830585866,73001277,1,1,0,0
6bba_6feeb0b1,5919,73,37.52,102.08000000000001,170.72000000000003,cont,2.436524804921953,74001294,1,0,1,0
6bba_6feeb0b1,6214,77,29.520000000000003,96.80000000000001,245.60000000000008,cont,3.160572782344974,78001352,1,1,0,0
6bba_6feeb0b1,6229,77,35.00000000000001,89.28000000000003,230.72000000000008,cont,2.436037060986522,78001353,1,1,0,0
6bba_6feeb0b1,6247,77,40.16,58.879999999999995,96.63999999999999,cont,2.915777126256395,78001361,1,1,0,0
6bba_6feeb0b1,6249,77,39.67999999999999,126.55999999999999,133.28,cont,0.7568035164426873,78001360,1,1,0,0
6bba_6feeb0b1,6285,78,27.999999999999996,120.64000000000001,244.8,cont,1.8226821753942681,79001369,3,0,1,2
6bba_6feeb0b1,6313,78,35.00000000000001,90.72,229.92,cont,1.7390537117927125,79001370,1,0,1,0
6bba_6feeb0b1,6315,78,36.160000000000004,81.92,128.80000000000004,cont,1.8852801515954976,79001376,1,0,1,0
6bba_6feeb0b1,6325,78,38.99999999999999,87.46666666666664,92.53333333333332,cont,0.6511274943426049,79001378,3,0,0,3
6bba_6feeb0b1,6326,78,39.32,129.44000000000003,134.72,cont,2.7176394283458505,79001379,1,0,1,0
6bba_6feeb0b1,6336,78,44.04,104.0,29.120000000000005,cont,0.6134240478657496,79001380,2,0,1,1
6bba_6feeb0b1,6337,78,45.0,214.88000000000008,13.280000000000001,cont,2.351089175786384,79001381,1,1,0,0
6bba_6feeb0b1,6345,78,53.480000000000004,77.44,8.0,cont,1.8616713760758088,79001383,1,0,1,0
6bba_6feeb0b1,6349,79,25.16,116.0,171.36,cont,3.653070840060458,80001384,1,0,0,1
6bba_6feeb0b1,6455,80,35.16000000000001,92.63999999999999,229.44,cont,1.4009912785238672,81001409,1,1,0,0
6bba_6feeb0b1,6492,81,25.0,118.71999999999997,174.08,cont,5.545346726084861,82001420,1,1,0,0
6bba_6feeb0b1,6527,81,35.160000000000004,94.08000000000001,228.64,cont,1.8270232791346603,82001424,1,0,1,0
6bba_6feeb0b1,6966,87,42.84,133.76,48.480000000000004,cont,1.8659925307996323,88001541,1,1,0,0
6bba_6feeb0b1,6967,87,46.16,242.56000000000006,21.919999999999998,cont,1.2355344238425747,88001542,2,2,0,0
6bba_6feeb0b1,6988,88,28.320000000000007,146.23999999999998,241.43999999999997,cont,3.137596861054659,89001548,2,0,0,2
6bba_6feeb0b1,7015,88,38.15999999999999,42.07999999999999,157.92000000000002,cont,2.2612620922838773,89001556,1,1,0,0
6bba_6feeb0b1,7019,88,39.16,101.44,92.0,cont,2.0394882936658383,89001557,1,0,1,0
6bba_6feeb0b1,7027,88,42.32,142.23999999999998,53.44,cont,2.7995783722732326,89001560,1,0,1,0
6bba_6feeb0b1,7035,88,46.32,245.43999999999997,24.0,cont,1.33705015799708,89001562,2,0,2,0
6bba_6feeb0b1,7090,89,39.839999999999996,220.0,148.64000000000001,cont,1.6660807303369203,90001579,1,1,0,0
6bba_6feeb0b1,7158,90,40.519999999999996,223.36,147.36000000000004,cont,1.2231836023263098,91001599,1,0,1,0
6bba_705ec2c9,349,3,22.36,84.00000000000001,203.35999999999999,cont,2.934103887901724,4000019,1,1,0,0
6bba_705ec2c9,390,3,58.999999999999986,114.71999999999997,152.0,cont,1.8149147707261843,4000022,1,1,0,0
6bba_705ec2c9,438,4,20.440000000000005,81.28000000000002,208.8,cont,1.214408935449679,5000024,1,0,1,0
6bba_705ec2c9,484,4,58.0,113.28000000000002,155.36,cont,2.3593298836745973,5000028,1,0,1,0
6bba_705ec2c9,1066,10,61.19999999999999,98.08000000000001,213.28000000000003,cont,1.4005199927170053,11000069,1,1,0,0
6bba_705ec2c9,1147,11,60.16,97.92000000000002,216.0,cont,1.2144089354496759,12000077,1,0,1,0
6bba_705ec2c9,1256,13,6.0,70.08,235.36000000000004,cont,2.0561227164252633,14000089,1,1,0,0
6bba_705ec2c9,1306,13,42.88000000000001,104.64,180.64000000000001,cont,5.120194240700228,14000093,1,1,0,0
6bba_705ec2c9,1328,13,60.52,143.36,105.28000000000002,cont,2.4849220309699884,14000096,1,1,0,0
6bba_705ec2c9,1343,14,4.640000000000001,68.0,238.07999999999998,cont,1.6451871094194734,15000098,1,0,1,0
6bba_705ec2c9,1388,14,40.8,104.0,183.36,cont,2.2150723041246305,15000101,1,0,1,0
6bba_705ec2c9,1409,14,59.84000000000002,143.99999999999997,106.72,cont,1.9316322437254718,15000107,1,0,1,0
6bba_705ec2c9,1578,16,60.2,136.0,154.56,cont,3.556337652698346,17000129,1,1,0,0
6bba_705ec2c9,1769,18,61.29270202636719,150.79603027343745,26.43068206787109,cont,2.3435779061307636,19000150,1,1,0,0
6bba_705ec2c9,1854,19,57.64000000000001,149.28000000000003,11.520000000000001,cont,2.6596936501221373,20000164,2,0,0,2
6bba_705ec2c9,1858,19,60.000000000000014,120.80000000000001,149.92000000000002,cont,2.701070678916027,20000165,1,1,0,0
6bba_705ec2c9,1916,20,28.44,114.72,208.8,cont,5.724176387263061,21000169,1,0,1,0
6bba_705ec2c9,1935,20,52.0,133.27999999999997,154.56,cont,0.7348522385486862,21000172,1,0,1,0
6bba_705ec2c9,1939,20,56.68,150.72,14.080000000000002,cont,2.1802873950468062,21000175,1,1,0,0
6bba_705ec2c9,1940,20,59.16000000000001,119.36000000000001,152.63999999999996,cont,1.75718035855742,21000173,1,0,1,0
6bba_705ec2c9,2017,21,55.67999999999999,151.35999999999999,17.440000000000005,cont,0.5689820955003855,22000189,3,1,1,1
6bba_705ec2c9,2018,21,57.266666666666666,148.0,87.99999999999997,cont,2.9595280744759123,22000187,2,0,0,2
6bba_705ec2c9,2029,21,61.04,150.07999999999998,81.92,cont,4.691411867711036,22000187,3,0,1,2
6bba_705ec2c9,2088,22,47.68000000000001,132.64000000000001,155.35999999999999,cont,0.7726870728179512,23000196,1,1,0,0
6bba_705ec2c9,2091,22,54.84,152.0,16.640000000000004,cont,0.29831034594863587,23000199,1,0,1,0
6bba_705ec2c9,2093,22,57.52000000000001,116.63999999999999,159.35999999999999,cont,4.087319185603184,23000197,1,1,0,0
6bba_705ec2c9,2174,23,45.800000000000004,133.28,156.0,cont,3.702861224782793,24000209,1,0,1,0
6bba_705ec2c9,2187,23,56.64000000000001,116.0,162.07999999999998,cont,0.8043339635996938,24000210,1,0,1,0
6bba_705ec2c9,2453,26,54.88000000000001,121.12,167.83999999999997,cont,4.305299423385068,27000252,1,1,0,0
6bba_705ec2c9,2527,27,31.266666666666662,81.06666666666665,247.99999999999997,cont,6.182597665131465,28000266,1,0,0,1
6bba_705ec2c9,2565,27,56.72,143.99999999999997,112.64,cont,2.4839122996796665,28000271,2,0,0,2
6bba_705ec2c9,3072,35,45.00000000000001,115.36000000000001,176.00000000000003,cont,1.929991094539035,36000371,1,1,0,0
6bba_705ec2c9,3082,35,55.36,132.64000000000001,76.8,cont,1.5245583376834162,36000374,1,1,0,0
6bba_705ec2c9,3140,36,45.35999999999999,113.28000000000002,176.00000000000003,cont,1.1542076557101844,37000385,1,0,1,0
6bba_705ec2c9,3149,36,55.0,132.00000000000006,76.63999999999999,cont,1.695070518444591,37000388,1,0,1,0
6bba_705ec2c9,3255,38,28.200000000000003,125.28,86.08,cont,2.9942802244445974,39000414,1,1,0,0
6bba_705ec2c9,3317,39,25.439999999999998,125.44000000000001,86.56,cont,4.323171940254059,40000425,1,0,1,0
6bba_705ec2c9,3541,42,46.63999999999999,140.00000000000003,92.16000000000003,parent,5.558640233906118,43000460,1,1,0,0
6bba_705ec2c9,3543,42,47.96,111.99999999999999,174.72,cont,2.051622833271261,43000461,1,1,0,0
6bba_705ec2c9,3575,43,6.84,98.08,189.28000000000003,cont,6.5728990226915895,44000464,1,1,0,0
6bba_705ec2c9,3619,43,46.999999999999986,139.99999999999994,115.46666666666664,daughter,0.4604166666666521,44000474,1,0,0,1
6bba_705ec2c9,3621,43,47.36,140.00000000000003,85.92000000000002,daughter,1.4025924657576176,44000477,1,0,1,0
6bba_705ec2c9,3622,43,49.040000000000006,111.99999999999999,173.27999999999997,cont,1.1069101137852349,44000475,1,0,1,0
6bba_705ec2c9,3655,44,4.866666666666665,104.5333333333333,187.99999999999994,cont,4.000456788154181,45000477,2,0,0,2
6bba_705ec2c9,3895,47,4.840000000000001,108.0,90.88,cont,2.096187014557621,48000517,1,1,0,0
6bba_705ec2c9,3962,48,1.0,112.64,99.36000000000001,cont,6.356990861445379,49000530,1,0,1,0
6bba_705ec2c9,3970,48,4.319999999999999,108.0,85.12000000000002,cont,2.0894366016943366,49000530,1,0,1,0
6bba_705ec2c9,4116,50,4.0,109.28,78.08000000000001,cont,2.858938013056591,51000561,1,1,0,0
6bba_705ec2c9,4197,51,4.0,111.36,72.79999999999998,cont,1.4125353977865556,52000579,1,0,1,0
6bba_705ec2c9,4202,51,7.0,103.46666666666663,84.53333333333329,cont,6.5443107713663915,52000579,1,0,0,1
6bba_705ec2c9,4260,51,61.839999999999996,88.00000000000001,233.44000000000003,cont,3.102516115993587,52000597,1,1,0,0
6bba_705ec2c9,4340,52,62.000000000000014,88.00000000000001,230.56,cont,3.482242677140701,53000612,1,0,1,0
6bba_705ec2c9,4358,53,7.800000000000002,114.56000000000002,13.12,cont,2.410538386439842,54000620,1,0,0,1
6bba_705ec2c9,4493,55,4.640000000000001,116.16000000000003,14.240000000000002,cont,3.4027360314899506,56000645,1,1,0,0
6bba_705ec2c9,4565,56,3.8,115.84000000000002,13.280000000000001,cont,5.022801135820522,57000660,1,0,1,0
6bba_705ec2c9,4579,56,9.266666666666664,119.99999999999997,23.999999999999993,cont,4.67695526921926,57000661,1,0,0,1
6bba_705ec2c9,4813,58,62.000000000000014,92.63999999999999,219.83999999999997,cont,2.5067698962010976,59000702,1,1,0,0
6bba_705ec2c9,4893,59,62.000000000000014,92.0,215.2,cont,3.5611236376318107,60000717,1,0,1,0
6bba_705ec2c9,5171,63,13.680000000000001,97.28000000000003,235.99999999999994,cont,1.2131035868795292,64000758,1,1,0,0
6bba_705ec2c9,5226,64,1.0,119.36000000000001,135.35999999999999,cont,1.7772775880542646,65000767,1,1,0,0
6bba_705ec2c9,5246,64,13.839999999999998,96.63999999999999,238.07999999999998,cont,0.5305569008127239,65000769,1,0,1,0
6bba_705ec2c9,5298,65,1.1600000000000001,118.72,130.72,cont,1.5376656414838665,66000776,1,0,1,0
6bba_705ec2c9,5410,66,13.680000000000001,97.28000000000003,240.00000000000006,cont,2.635206941589205,67000789,1,1,0,0
6bba_705ec2c9,5452,66,58.0,116.0,145.44,cont,2.258749999999999,67000796,1,1,0,0
6bba_705ec2c9,5470,67,1.36,116.63999999999999,122.56,cont,2.0392293274666358,68000798,1,0,0,1
6bba_705ec2c9,5493,67,13.32,98.72,240.00000000000006,cont,5.173858116773196,68000801,1,0,1,0
6bba_705ec2c9,5496,67,15.866666666666667,95.99999999999997,243.99999999999997,cont,0.9338842310181878,68000801,1,0,0,1
6bba_705ec2c9,5535,67,48.830737304687496,150.87803955078124,45.424114379882816,cont,6.978234217049268,68000807,1,1,0,0
6bba_705ec2c9,5541,67,58.16,116.0,140.48000000000002,cont,2.288020774053417,68000808,1,0,1,0
6bba_705ec2c9,5650,69,13.320000000000004,100.0,241.28000000000003,cont,6.632170294292803,70000822,1,1,0,0
6bba_705ec2c9,5655,69,17.0,96.0,244.64000000000001,cont,0.8255575767322341,70000822,1,1,0,0
6bba_705ec2c9,5702,69,60.000000000000014,136.0,26.720000000000002,cont,1.7003588775608247,70000829,1,0,1,0
6bba_705ec2c9,5730,70,13.520000000000001,100.64,242.71999999999997,cont,6.344454517529462,71000831,1,0,1,0
6bba_705ec2c9,5733,70,17.48,96.0,245.28000000000003,cont,1.123250027821049,71000831,1,0,1,0
6bba_705ec2c9,6198,76,16.68,111.36,125.43999999999998,cont,1.2822063503976298,77000879,1,0,1,0
6bba_74686d6a,948,2,31.640000000000004,129.28,71.99999999999999,cont,2.4708551151372604,3000011,1,1,0,0
6bba_74686d6a,1292,3,32.0,128.64000000000004,71.36,cont,2.138094946441796,4000014,1,0,1,0
6bba_74686d6a,2655,7,30.639999999999993,108.0,123.99999999999999,cont,1.351878900826553,8000032,1,1,0,0
6bba_74686d6a,2667,7,32.8,120.79999999999997,68.8,cont,2.3450163245700533,8000033,1,1,0,0
6bba_74686d6a,2914,8,21.133333333333333,170.93333333333328,228.53333333333333,parent,4.001923365708057,9000037,3,0,0,3
6bba_74686d6a,2915,8,20.680000000000003,186.55999999999997,234.0800000000001,parent,3.7717506711075126,9000037,5,0,1,4
6bba_74686d6a,3007,8,30.680000000000003,108.16000000000001,123.52000000000001,cont,3.4969229777763107,9000038,1,0,1,0
6bba_74686d6a,3014,8,33.00000000000001,116.16,67.36,cont,2.2675592605266046,9000040,1,0,1,0
6bba_74686d6a,4262,12,22.000000000000004,171.99999999999994,212.64,cont,1.6950705184445853,13000061,1,1,0,0
6bba_74686d6a,4590,13,22.16,172.64,212.64,cont,1.67445980095075,14000067,1,0,1,0
6bba_74686d6a,6006,17,32.16000000000001,108.64,74.56,cont,2.9989065715690253,18000093,1,1,0,0
6bba_74686d6a,6333,18,31.320000000000004,116.63999999999999,105.28000000000002,cont,6.1990928015718465,19000096,1,0,1,0
6bba_74686d6a,6339,18,32.160000000000004,110.71999999999998,75.35999999999999,cont,3.2837868764583336,19000098,1,0,1,0
6bba_74686d6a,6658,19,31.48,69.28,108.64000000000001,cont,1.5423809719067485,20000101,1,1,0,0
6bba_74686d6a,6974,20,31.48,68.64000000000001,109.92,cont,0.8771990224572763,21000106,1,0,1,0
6bba_74686d6a,6976,20,30.999999999999996,100.53333333333332,115.46666666666664,cont,0.6511274943426226,21000107,4,0,0,4
6bba_74686d6a,7204,21,19.84,232.0,247.35999999999999,cont,3.3603334097973048,22000109,1,1,0,0
6bba_74686d6a,7515,22,18.866666666666667,228.0,243.99999999999997,cont,0.4604166666666659,23000114,2,0,0,2
6bba_74686d6a,7528,22,20.000000000000007,234.72000000000006,248.00000000000006,cont,3.268189722537574,23000114,1,0,1,0
6bba_74686d6a,8192,24,22.36,186.71999999999997,216.0,cont,2.9795605800184735,25000126,3,3,0,0
6bba_74686d6a,8258,24,31.159999999999997,92.63999999999999,127.2,cont,2.3555214390873176,25000130,2,2,0,0
6bba_74686d6a,8271,24,33.32000000000001,104.96,80.64,parent,3.451203141659438,25000128,1,1,0,0
6bba_74686d6a,8490,25,19.0,234.07999999999998,241.27999999999997,cont,3.4615932928349626,26000131,1,1,0,0
6bba_74686d6a,8528,25,21.999999999999993,199.46666666666667,228.0,cont,1.787089599351739,26000132,2,0,0,2
6bba_74686d6a,8592,25,31.320000000000004,91.36000000000001,132.15999999999997,cont,3.513383785611829,26000138,3,1,2,0
6bba_74686d6a,8611,25,31.080000000000002,130.72,88.64,daughter,4.836770939893679,26000134,6,1,1,4
6bba_74686d6a,8619,25,33.71999999999999,98.39999999999999,81.44,daughter,4.570411804476711,26000135,4,1,1,2
6bba_74686d6a,8831,26,19.36,235.99999999999994,240.64000000000004,cont,2.699750567182089,27000138,1,0,1,0
6bba_74686d6a,8922,26,31.520000000000003,90.08,134.07999999999998,cont,2.617057328852397,27000146,1,0,1,0
6bba_74686d6a,9279,27,34.999999999999986,87.99999999999997,83.99999999999997,cont,3.3500233208143735,28000149,4,0,0,4
6bba_74686d6a,10631,31,32.32,83.83999999999999,136.8,cont,3.6576941592347496,32000186,4,4,0,0
6bba_74686d6a,10644,31,32.67999999999999,73.92,137.92000000000002,cont,0.641206138850851,32000185,3,3,0,0
6bba_74686d6a,10655,31,33.040000000000006,127.36000000000001,91.35999999999997,cont,2.4021985372778785,32000181,2,2,0,0
6bba_74686d6a,10673,31,40.0,74.72,8.64,cont,2.11749409680405,32000183,1,1,0,0
6bba_74686d6a,10943,32,23.68,225.28000000000003,222.72000000000003,cont,2.354680511767992,33000187,2,0,2,0
6bba_74686d6a,11001,32,30.040000000000003,101.12,158.55999999999997,cont,3.1139426877352703,33000188,3,0,0,3
6bba_74686d6a,11007,32,32.480000000000004,74.72,141.44,cont,2.677504960312862,33000189,5,0,3,2
6bba_74686d6a,11014,32,33.00000000000001,86.72,134.56,cont,5.606720994708047,33000189,7,0,4,3
6bba_74686d6a,11030,32,33.68000000000001,150.56000000000003,106.55999999999999,cont,2.394656812572519,33000191,3,0,2,1
6bba_74686d6a,11051,32,40.16,73.27999999999999,10.720000000000002,cont,3.4711536716918845,33000193,4,0,1,3
6bba_74686d6a,11055,32,41.68000000000001,58.55999999999999,14.719999999999999,cont,6.295045298685311,33000193,2,0,2,0
6bba_74686d6a,11245,33,17.266666666666666,42.93333333333332,248.5333333333334,cont,3.0226747261324776,34000195,2,0,0,2
6bba_74686d6a,11267,33,18.88,45.28000000000001,243.36000000000004,cont,1.7215799429593615,34000195,3,0,1,2
6bba_74686d6a,11306,33,23.36,225.28,223.99999999999997,cont,2.7116084364266184,34000196,1,1,0,0
6bba_74686d6a,11664,34,23.68,225.28,225.28,cont,2.167837704372727,35000204,1,0,1,0
6bba_74686d6a,13030,38,16.359999999999996,56.48000000000002,240.0,cont,4.500105901531659,39000236,1,1,0,0
6bba_74686d6a,13168,38,35.52,140.00000000000003,116.0,cont,1.3887809402493856,39000240,1,1,0,0
6bba_74686d6a,13382,39,15.84,64.0,237.28000000000003,cont,3.5455176367492527,40000244,1,0,1,0
6bba_74686d6a,13475,39,28.0,118.08000000000001,168.63999999999996,cont,5.062598764468706,40000246,1,1,0,0
6bba_74686d6a,13520,39,35.32000000000001,138.72,116.63999999999999,cont,1.8170958973042661,40000248,1,0,1,0
6bba_74686d6a,13831,40,28.519999999999996,115.35999999999999,169.28,cont,4.095161207144363,41000254,2,1,1,0
6bba_74686d6a,14189,41,29.0,112.00000000000001,170.72000000000003,cont,1.7003588775608567,42000263,1,0,1,0
6bba_74686d6a,14602,42,42.120000000000005,64.0,20.000000000000004,cont,3.0549999999999926,43000277,1,1,0,0
6bba_74686d6a,14916,43,35.68,133.28,116.63999999999999,cont,0.7497926796788623,44000287,1,1,0,0
6bba_74686d6a,14948,43,42.68000000000001,64.0,21.28,cont,3.803090231969242,44000288,1,0,1,0
6bba_74686d6a,15247,44,35.84,134.72,116.63999999999999,cont,0.6211242327425376,45000298,1,0,1,0
6bba_74686d6a,16447,48,12.16,13.280000000000001,233.27999999999997,cont,4.804864234554394,49000333,2,2,0,0
6bba_74686d6a,16564,48,29.32,93.27999999999999,183.36,cont,2.996528199433471,49000336,1,1,0,0
6bba_74686d6a,16790,49,11.440000000000001,14.719999999999999,234.72000000000006,cont,2.953479409950233,50000343,2,0,2,0
6bba_74686d6a,16808,49,11.999999999999996,3.999999999999999,228.0,cont,4.337563039887723,50000343,4,0,0,4
6bba_74686d6a,16932,49,29.36,93.91999999999999,184.0,cont,0.7129658564756108,50000354,1,0,1,0
6bba_74686d6a,17811,52,11.160000000000002,3.3600000000000003,228.64000000000004,cont,6.8306658030165615,53000375,2,2,0,0
6bba_74686d6a,18146,53,10.320000000000002,18.72,236.64000000000001,cont,6.082993840412791,54000385,4,0,0,4
6bba_74686d6a,20290,59,28.48,120.63999999999999,192.0,cont,2.122476442743235,60000462,1,1,0,0
6bba_74686d6a,20616,60,28.48,123.36,194.08000000000004,cont,2.6942186367293974,61000470,1,0,1,0
6bba_74686d6a,20781,61,5.320000000000001,20.000000000000007,248.64000000000007,cont,4.291046273929014,62000474,1,1,0,0
6bba_74686d6a,21099,62,5.079999999999999,19.999999999999993,251.04000000000002,cont,3.4429298025083233,63000481,1,0,1,0
6bba_74686d6a,21874,64,30.0,131.20000000000002,198.56,cont,1.1485888080597064,65000496,1,1,0,0
6bba_74686d6a,21898,64,33.84000000000001,72.64,155.35999999999999,cont,2.0046204441240105,65000497,1,1,0,0
6bba_74686d6a,21902,64,34.84,115.36000000000004,142.08,cont,4.0937746182465755,65000498,1,1,0,0
6bba_74686d6a,22160,65,30.480000000000004,134.56,201.28,cont,2.411578839681589,66000502,1,0,1,0
6bba_74686d6a,22179,65,33.64000000000001,72.64,156.79999999999995,cont,1.0993698479128933,66000503,1,0,1,0
6bba_74686d6a,22180,65,34.120000000000005,115.35999999999999,141.28000000000003,cont,5.7480915311083765,66000504,3,0,1,2
6bba_74686d6a,22187,65,34.999999999999986,107.99999999999997,151.46666666666664,cont,1.8296791233559344,66000504,3,0,0,3
6bba_74686d6a,22443,66,31.000000000000007,140.16,204.79999999999995,cont,6.765564116354226,67000509,1,0,0,1
6bba_74686d6a,25837,78,34.64,74.08000000000001,172.0,cont,2.7239963748140315,79000585,1,1,0,0
6bba_74686d6a,26077,79,30.13404052734375,137.37692871093753,215.61327392578124,cont,2.5907395968582554,80000589,2,2,0,0
6bba_74686d6a,26107,79,34.839999999999996,72.16,173.28000000000003,cont,2.212985116307841,80000592,1,0,1,0
6bba_74686d6a,26345,80,29.614040527343747,131.2969287109375,218.4932739257813,cont,3.025987438451853,81000595,2,0,2,0
6bba_767a1e17,22782,48,3.8400000000000007,54.080000000000005,151.36,cont,2.3247157084254364,49000378,3,3,0,0
6bba_767a1e17,30051,60,6.16,48.64,133.28,cont,1.9337499999999992,61000531,4,0,4,0
6bba_767a1e17,36752,70,2.5999999999999996,141.92000000000002,183.20000000000002,cont,3.2061954186387407,71000731,6,0,4,2
6bba_767a1e17,36828,70,10.120000000000001,83.20000000000002,35.2,cont,6.56807632130596,71000735,7,1,3,3
6bba_767a1e17,37002,70,26.960000000000004,105.76000000000003,199.36000000000007,cont,2.2521350980791732,71000757,5,0,1,4
6bba_767a1e17,37602,71,15.080000000000002,71.68000000000002,44.32000000000001,cont,3.5263623253999277,72000770,2,0,2,0
6bba_767a1e17,37659,71,21.160000000000004,48.0,24.64,cont,1.8357466124168624,72000772,2,1,1,0
6bba_767a1e17,37823,71,34.86666666666667,92.0,163.46666666666664,cont,0.8683577251852469,72000783,6,0,0,6
6bba_767a1e17,38167,72,0.6000000000000009,39.99999999999999,131.46666666666667,cont,5.357436582556169,73000796,2,0,0,2
6bba_767a1e17,38923,73,4.800000000000001,99.36000000000001,68.48,cont,6.010303652894749,74000828,8,0,1,7
6bba_767a1e17,39645,74,4.840000000000001,63.36,150.07999999999998,cont,5.493028820696282,75000866,2,1,1,0
6bba_767a1e17,39660,74,8.266666666666667,51.46666666666666,156.53333333333333,cont,3.1109308293499582,75000866,6,0,0,6
6bba_767a1e17,42521,78,8.2,73.44,164.64,cont,6.703533699661692,79001019,2,2,0,0
6bba_767a1e17,42693,78,26.68,119.99999999999997,166.08,cont,3.323432901383748,79001032,4,4,0,0
6bba_767a1e17,42713,78,27.240000000000002,81.28,32.0,cont,4.739041635710747,79001034,4,4,0,0
6bba_767a1e17,42747,78,31.720000000000002,67.35999999999999,58.72000000000001,cont,3.428480439640857,79001028,5,5,0,0
6bba_767a1e17,42817,78,37.519999999999996,155.51999999999998,138.07999999999998,cont,4.087674499333345,79001041,6,6,0,0
6bba_767a1e17,42868,78,43.480000000000004,136.16000000000003,192.8,cont,6.645038092441615,79001043,4,4,0,0
6bba_767a1e17,42881,78,43.68,116.16000000000003,202.07999999999998,cont,4.1950186233197995,79001043,3,3,0,0
6bba_767a1e17,43233,79,10.68,125.92,74.56,cont,6.275193722109304,80001056,5,0,1,4
6bba_767a1e17,43267,79,15.48,27.840000000000003,216.64,cont,1.7875738621103139,80001057,6,0,3,3
6bba_767a1e17,43390,79,27.239999999999995,92.95999999999998,89.91999999999999,cont,2.0153275943379523,80001066,6,1,2,3
6bba_767a1e17,43397,79,26.36,149.44,178.88,cont,6.483240894799453,80001067,6,0,6,0
6bba_767a1e17,43464,79,33.64000000000001,84.64,47.36,cont,0.6909594778277408,80001074,9,0,5,4
6bba_767a1e17,43476,79,33.36,120.80000000000001,64.16,cont,2.622097502477738,80001073,2,0,2,0
6bba_767a1e17,43504,79,37.66666666666667,57.06666666666666,74.39999999999998,cont,6.595055821198354,80001076,5,0,0,5
6bba_767a1e17,43520,79,37.519999999999996,108.16000000000001,23.360000000000007,cont,4.4450804970214035,80001078,3,0,3,0
6bba_767a1e17,43534,79,39.160000000000004,139.2,215.36000000000004,cont,6.245942808135528,80001080,8,0,4,4
6bba_767a1e17,43586,79,42.84,173.44,168.16000000000003,cont,5.8039577014309955,80001079,6,0,6,0
6bba_767a1e17,43859,80,2.6799999999999997,104.64,211.36,cont,2.313043517208444,81001090,5,5,0,0
6bba_767a1e17,43900,80,5.834692001342773,94.79745361328125,140.35122314453125,cont,6.958748572951875,81001089,2,2,0,0
6bba_767a1e17,44840,81,28.86814331054687,153.88627685546874,185.17195556640624,cont,2.020712615096083,82001140,1,0,1,0
6bba_767a1e17,44923,81,37.32000000000001,51.19999999999999,186.55999999999997,cont,4.385844595115051,82001149,1,0,1,0
6bba_767a1e17,44925,81,37.0,135.46666666666664,79.99999999999999,cont,3.355383449067746,82001151,3,0,0,3
6bba_767a1e17,45961,83,0.16000000000000006,13.280000000000001,230.07999999999998,cont,1.5991188980185311,84001198,2,2,0,0
6bba_767a1e17,46898,84,18.48,121.28000000000003,101.91999999999999,cont,1.1119085506461326,85001247,1,1,0,0
6bba_767a1e17,46965,84,24.799999999999997,117.28,3.8400000000000007,cont,1.334084423865293,85001252,3,3,0,0
6bba_767a1e17,47056,84,32.480000000000004,74.72,61.44,cont,6.443365774189766,85001254,3,3,0,0
6bba_767a1e17,47058,84,32.32,152.0,40.8,cont,1.7538711647381615,85001259,1,1,0,0
6bba_767a1e17,47154,84,40.8,144.64000000000001,91.84,cont,2.4155722018809533,85001265,2,2,0,0
6bba_767a1e17,47157,84,40.00000000000001,190.56,183.84,cont,5.74379461353728,85001267,1,1,0,0
6bba_767a1e17,47221,84,44.08,116.16000000000003,67.36000000000001,cont,6.917921020798096,85001268,4,4,0,0
6bba_767a1e17,47235,84,46.519999999999996,133.28,46.72,cont,2.629137513044917,85001271,4,4,0,0
6bba_767a1e17,47329,84,56.160000000000004,132.64000000000001,201.92,cont,2.1134997634255823,85001273,1,1,0,0
6bba_767a1e17,47492,85,3.6400000000000006,127.99999999999997,53.280000000000015,cont,5.126997324214244,86001282,7,0,3,4
6bba_767a1e17,47763,85,27.4,120.0,7.360000000000001,cont,2.7363764817729317,86001298,6,0,3,3
6bba_767a1e17,47778,85,30.480000000000004,176.64,18.560000000000002,cont,1.3082006965676243,86001299,10,2,3,5
6bba_767a1e17,47846,85,35.32000000000001,152.8,57.92000000000001,cont,2.12048490916583,86001304,6,0,2,4
6bba_767a1e17,47869,85,37.480000000000004,77.92000000000002,130.39999999999998,cont,2.369270061643455,86001321,5,1,1,3
6bba_767a1e17,47958,85,44.84,103.36,63.84,cont,2.225300892576096,86001312,3,0,3,0
6bba_767a1e17,47972,85,46.480000000000004,138.07999999999998,47.2,cont,6.982925828404594,86001315,11,0,4,7
6bba_767a1e17,48021,85,50.0,136.53333333333336,57.06666666666666,cont,0.19150808657134724,86001315,7,0,0,7
6bba_767a1e17,48941,87,0.4800000000000001,36.63999999999999,112.00000000000001,cont,0.8575628621273197,88001371,2,1,1,0
6bba_767a1e17,48987,87,4.120000000000001,25.28,234.56,cont,0.39437648573412537,88001370,1,1,0,0
6bba_767a1e17,49087,87,12.64,120.64000000000001,178.56000000000003,cont,1.226632840339767,88001381,1,1,0,0
6bba_767a1e17,49602,87,58.0,145.28,213.28000000000003,cont,1.8739005109663691,88001411,2,2,0,0
6bba_767a1e17,50044,88,30.133333333333333,182.93333333333325,33.06666666666666,cont,1.5358900785321046,89001437,7,0,0,7
6bba_767a1e17,51333,90,8.4,153.6,59.46666666666667,cont,3.907985170171171,91001510,5,2,0,3
6bba_767a1e17,51400,90,14.959999999999999,135.36000000000004,191.36,cont,3.494807979345939,91001524,3,2,1,0
6bba_767a1e17,51432,90,17.266666666666666,123.99999999999999,75.99999999999999,cont,2.2498736461435125,91001525,9,2,0,7
6bba_767a1e17,51639,90,32.36000000000001,186.24000000000007,39.36000000000001,cont,4.357697392258872,91001535,4,3,1,0
6bba_767a1e17,51779,90,45.44000000000001,168.0,124.00000000000003,cont,0.822352760377222,91001542,6,6,0,0
6bba_767a1e17,51824,90,48.64,221.44000000000005,202.07999999999996,cont,2.535520779839911,91001546,3,2,1,0
6bba_767a1e17,52037,91,3.8,32.480000000000004,116.63999999999999,cont,5.53097103816319,92001561,11,1,4,6
6bba_767a1e17,52087,91,4.879999999999999,25.28,20.000000000000007,cont,6.018930682646214,92001557,3,2,1,0
6bba_767a1e17,52105,91,8.0,19.2,217.12,cont,2.252193722240606,92001555,10,1,5,4
6bba_767a1e17,52172,91,12.680000000000003,37.44,240.64,cont,1.5438355239467711,92001567,5,0,5,0
6bba_767a1e17,52249,91,18.439999999999994,127.67999999999999,76.0,cont,1.3325990816821174,92001580,2,0,2,0
6bba_767a1e17,52416,91,32.0,143.20000000000002,123.19999999999999,cont,1.917568854695966,92001587,8,0,3,5
6bba_767a1e17,52449,91,33.999999999999986,203.99999999999991,48.53333333333332,cont,1.6857063512895334,92001592,7,0,0,7
6bba_767a1e17,52458,91,34.68,219.36,40.00000000000001,cont,6.904433584661387,92001592,3,0,3,0
6bba_767a1e17,52604,91,46.52,174.08,128.0,cont,2.5283245440409754,92001601,6,0,6,0
6bba_767a1e17,52628,91,50.80000000000001,90.56,79.36,cont,1.8005263987234468,92001604,12,1,5,6
6bba_767a1e17,52629,91,50.0,95.36000000000001,48.480000000000004,cont,1.8563446945543245,92001605,8,0,4,4
6bba_767a1e17,52708,91,56.36000000000001,158.88000000000002,92.16000000000003,cont,4.665221893168626,92001608,11,0,5,6
6bba_767a1e17,54575,94,14.16,197.92000000000002,88.00000000000001,cont,2.589059011011523,95001767,1,0,1,0
6bba_767a1e17,56110,96,8.84,53.919999999999995,35.36,cont,1.8574823350438643,97001876,7,0,0,7
6bba_767a1e17,56248,96,20.96,28.640000000000008,161.28,cont,0.5440717438904492,97001880,5,0,0,5
6bba_767a1e17,56666,96,54.480000000000004,102.56,91.99999999999999,cont,0.8002196963959348,97001915,2,2,0,0
6bba_767a1e17,57504,97,56.36,107.36,92.63999999999999,cont,1.5010813810383536,98001976,2,0,2,0
6bba_76db78c1,60,1,29.480000000000004,62.720000000000006,232.96000000000004,cont,2.59766301220924,2000009,1,0,1,0
6bba_76db78c1,624,12,29.800000000000004,69.28,214.56000000000006,cont,1.6062037931097153,13000082,1,1,0,0
6bba_76db78c1,670,13,25.99999999999999,76.5333333333333,215.99999999999994,cont,2.0748378869069657,14000089,1,0,0,1
6bba_76db78c1,3977,76,48.84000000000001,80.00000000000003,26.719999999999995,cont,2.4282742071273336,77000456,1,1,0,0
6bba_76db78c1,4030,77,49.00000000000001,79.36,27.360000000000003,cont,1.8784044093325452,78000463,1,0,1,0
6bba_784a78c9,240,1,29.759999999999998,238.71999999999997,140.0,cont,3.8974601104437276,2000037,1,1,0,0
6bba_784a78c9,421,2,30.320000000000007,235.35999999999999,140.64000000000004,cont,2.822124919719175,3000061,2,1,1,0
6bba_784a78c9,423,2,30.999999999999996,252.0,144.0,cont,6.512682938889319,3000061,1,0,0,1
6bba_784a78c9,607,3,31.159999999999997,232.64,140.64000000000001,cont,3.3329538325635526,4000087,1,0,1,0
6bba_784a78c9,770,4,25.32,247.36,141.43999999999997,cont,2.814629471617889,5000114,1,0,1,0
6bba_784a78c9,1092,5,61.00000000000001,86.72,55.199999999999996,cont,2.5551284214301173,6000156,2,0,0,2
6bba_784a78c9,1381,7,38.26666666666665,252.53333333333336,83.46666666666664,cont,6.36366128072773,8000202,2,0,0,2
6bba_784a78c9,1535,8,28.0,239.36,140.00000000000003,cont,0.2599999999999909,9000233,1,1,0,0
6bba_784a78c9,1724,9,28.679999999999993,236.64000000000001,140.00000000000003,cont,0.860636573996258,10000254,2,1,1,0
6bba_784a78c9,1870,10,16.159999999999997,226.55999999999997,97.44,cont,3.0648809622071824,11000256,1,1,0,0
6bba_784a78c9,1909,10,29.159999999999997,235.35999999999999,140.00000000000003,cont,4.680507784952408,11000263,1,0,1,0
6bba_784a78c9,1938,10,37.000000000000014,234.72000000000006,79.36,cont,3.2734776996338115,11000267,1,1,0,0
6bba_784a78c9,1957,10,41.36000000000001,244.64000000000001,86.08,cont,6.284276509670791,11000267,1,1,0,0
6bba_784a78c9,2070,11,17.2,224.0,96.63999999999999,cont,2.956696319374042,12000277,1,0,1,0
6bba_784a78c9,2134,11,37.67999999999999,232.63999999999993,78.72,cont,2.5385391960732124,12000285,1,0,1,0
6bba_784a78c9,2149,11,41.67999999999999,241.91999999999996,85.28,cont,6.081322327421875,12000285,1,0,1,0
6bba_784a78c9,2437,13,4.16,214.08,168.64000000000001,cont,0.835098235239424,14000312,1,1,0,0
6bba_784a78c9,2619,14,4.5200000000000005,209.92000000000004,169.44,cont,2.415954780205954,15000332,1,0,1,0
6bba_784a78c9,3267,17,41.0,247.36,95.36000000000001,cont,1.206446227562584,18000410,1,1,0,0
6bba_784a78c9,3433,18,40.96,246.72,92.63999999999999,cont,1.4439658929489987,19000430,1,0,1,0
6bba_784a78c9,3476,18,55.0,246.72,32.64,cont,1.9503385122844683,19000432,1,0,1,0
6bba_784a78c9,3511,19,1.0,234.72000000000006,173.28000000000003,cont,1.7310636181261436,20000435,1,1,0,0
6bba_784a78c9,3607,19,41.8,218.08000000000004,61.43999999999998,cont,2.3688242072809005,20000449,1,1,0,0
6bba_784a78c9,3683,20,1.1600000000000001,232.64000000000004,174.72,cont,1.6054638153817173,21000454,1,0,1,0
6bba_784a78c9,4041,22,2.84,183.2,230.56000000000003,cont,6.705502987285893,23000494,1,1,0,0
6bba_784a78c9,4139,22,43.68,244.00000000000003,86.72,cont,2.164851322839511,23000509,1,1,0,0
6bba_784a78c9,4212,23,3.3200000000000003,180.64000000000004,231.36,cont,6.9360284475339515,24000513,1,0,1,0
6bba_784a78c9,4309,23,44.480000000000004,244.00000000000003,84.64000000000001,cont,2.7232692324667456,24000526,1,0,1,0
6bba_784a78c9,4528,24,58.31999999999999,231.36,21.280000000000005,cont,0.6142843905716496,25000546,1,1,0,0
6bba_784a78c9,4582,25,21.360000000000003,197.92000000000002,218.07999999999998,parent,0.6949606553611579,26000566,1,1,0,0
6bba_784a78c9,4699,25,58.680000000000014,229.28000000000003,20.640000000000004,cont,0.6142843905716429,26000565,1,0,1,0
6bba_784a78c9,4748,26,20.679999999999996,200.0,215.35999999999996,daughter,2.901432835686527,27000586,2,0,1,1
6bba_784a78c9,4755,26,23.999999999999993,182.9333333333333,228.0,daughter,1.681785592094823,27000585,2,0,0,2
6bba_784a78c9,4866,26,58.839999999999996,228.64000000000007,20.000000000000007,cont,0.8655318090630679,27000584,1,1,0,0
6bba_784a78c9,5035,27,58.999999999999986,227.36,20.000000000000007,cont,0.7803384682302871,28000601,1,0,1,0
6bba_784a78c9,5314,29,47.120000000000005,242.08,81.43999999999998,cont,2.340451345360549,30000640,1,1,0,0
6bba_784a78c9,5477,30,47.480000000000004,241.43999999999997,80.0,cont,2.7636945946504285,31000661,1,0,1,0
6bba_784a78c9,5651,31,53.36,182.07999999999996,54.720000000000006,cont,0.9816129901850401,32000685,3,0,0,3
6bba_784a78c9,5730,32,27.52,170.72000000000003,196.00000000000003,cont,4.052021486246093,33000696,1,1,0,0
6bba_784a78c9,5792,32,48.20000000000001,235.99999999999994,78.56,cont,3.020315070485184,33000705,2,2,0,0
6bba_784a78c9,5885,33,27.480000000000004,169.28,196.64000000000001,cont,2.5839032973391145,34000717,1,0,1,0
6bba_784a78c9,5948,33,48.0,232.64,80.00000000000001,cont,5.158728798114899,34000726,2,0,2,0
6bba_784a78c9,5957,33,50.999999999999986,227.9999999999999,70.93333333333332,cont,2.8994349605186582,34000726,3,0,0,3
6bba_784a78c9,6627,38,2.0,119.36000000000001,244.00000000000003,cont,0.43177323330654843,39000825,1,1,0,0
6bba_784a78c9,6775,39,2.1600000000000006,115.99999999999997,244.64000000000001,cont,1.447709937280259,40000845,1,0,1,0
6bba_784a78c9,7020,40,48.640000000000015,218.56,77.44000000000001,cont,2.4530503969547595,41000884,1,0,1,0
6bba_784a78c9,8879,53,54.0,185.92000000000004,66.72,cont,1.8260835824244257,54001167,1,1,0,0
6bba_784a78c9,8984,54,43.36,166.71999999999997,128.64000000000001,cont,1.2403338764219987,55001185,1,1,0,0
6bba_784a78c9,9013,54,54.680000000000014,184.64,65.28,cont,2.736279979278455,55001189,1,0,1,0
6bba_784a78c9,9050,55,10.160000000000002,240.0,237.27999999999997,cont,3.3035894304528846,56001192,2,2,0,0
6bba_784a78c9,9112,55,43.480000000000004,164.64,128.00000000000006,cont,0.9170818188689679,56001206,1,0,1,0
6bba_784a78c9,9179,56,10.680000000000001,238.56,238.72000000000003,cont,2.7309670906109336,57001212,2,0,2,0
6bba_784a78c9,10335,65,14.84,232.8,148.64000000000001,cont,3.6249521548566688,66001403,1,1,0,0
6bba_784a78c9,10456,66,14.479999999999999,231.36,150.08000000000004,cont,2.5116108501318424,67001424,1,0,1,0
6bba_784a78c9,11626,76,17.839999999999996,166.72000000000003,179.36,cont,2.1053629437700256,77001616,1,1,0,0
6bba_784a78c9,11651,76,34.480000000000004,67.36000000000001,197.28000000000003,cont,4.372699573775896,77001624,1,1,0,0
6bba_784a78c9,11735,77,17.999999999999996,167.36,178.72000000000003,cont,1.1116710394716556,78001635,1,0,1,0
6bba_784a78c9,11766,77,34.839999999999996,67.2,197.43999999999997,cont,2.2073697583323084,78001641,1,0,1,0
6bba_784a78c9,12631,86,21.160000000000004,87.36,247.36,cont,3.0668188934138185,87001818,1,1,0,0
6bba_784a78c9,12724,87,21.32,89.92,246.72,cont,1.221671498398808,88001840,1,0,1,0
6bba_784a78c9,12832,88,34.0,5.28,204.64,cont,1.77489876471871,89001865,1,1,0,0
6bba_784a78c9,12922,89,34.0,8.16,204.64000000000001,cont,2.0680958180171425,90001887,1,0,1,0
6bba_784a78c9,13045,90,50.320000000000014,179.35999999999999,9.28,cont,0.9089838076115705,91001918,1,1,0,0
6bba_784a78c9,13135,91,50.16,181.28000000000006,8.64,cont,1.5553380862371984,92001940,1,0,1,0
6bba_784a78c9,13167,92,7.88,156.79999999999998,223.36000000000004,cont,0.5859020822628915,93001945,1,1,0,0
6bba_784a78c9,13254,93,7.32,159.36,223.36,cont,1.1445584574411212,94001966,1,0,1,0
6bba_784a78c9,13496,95,58.999999999999986,175.36,35.99999999999999,cont,1.1468632492585955,96002023,1,1,0,0
6bba_784a78c9,13579,96,58.999999999999986,180.79999999999998,31.359999999999996,cont,0.825557576732243,97002041,1,0,1,0
6bba_786893ac,1321,1,43.92,153.27999999999997,36.00000000000001,cont,1.8304166192427265,2000084,1,1,0,0
6bba_786893ac,2649,3,23.320000000000007,29.279999999999994,220.0,cont,1.4234295073870038,4000166,1,1,0,0
6bba_786893ac,2684,3,25.480000000000004,121.44000000000001,137.28000000000003,cont,1.736850598065345,4000168,6,0,1,5
6bba_786893ac,5057,6,26.000000000000004,28.640000000000008,113.28000000000002,cont,1.63552839321119,7000299,2,2,0,0
6bba_786893ac,5142,6,31.359999999999996,109.91999999999999,40.64000000000001,cont,1.2567250345640515,7000295,3,3,0,0
6bba_786893ac,5831,7,23.880000000000003,119.36000000000001,139.36,cont,3.5520656402296327,8000344,1,0,1,0
6bba_786893ac,6646,8,26.76,40.00000000000001,122.24000000000001,cont,6.569443112814058,9000390,1,1,0,0
6bba_786893ac,6918,8,45.68000000000001,30.080000000000005,237.27999999999997,cont,3.8343458128473347,9000412,3,2,1,0
6bba_786893ac,7104,8,58.519999999999996,54.07999999999999,182.71999999999997,cont,4.528388406762396,9000418,2,2,0,0
6bba_786893ac,7796,9,49.99999999999999,123.46666666666668,107.46666666666665,cont,4.789080824008798,10000458,6,1,0,5
6bba_786893ac,7904,9,57.16000000000001,168.79999999999995,79.35999999999999,cont,6.784486900274773,10000460,1,0,1,0
6bba_786893ac,8961,11,22.64,111.84,142.72,cont,3.313804931645797,12000513,8,2,0,6
6bba_786893ac,9503,11,58.999999999999986,71.36,4.640000000000001,cont,1.892410004465181,12000541,2,2,0,0
6bba_786893ac,9910,12,35.049735717773444,114.91474121093754,34.7580517578125,cont,2.111161973628748,13000561,1,0,1,0
6bba_786893ac,10112,12,45.73333333333331,62.93333333333332,64.53333333333333,cont,6.320115043784141,13000570,7,0,0,7
6bba_786893ac,11634,14,39.160000000000004,27.36,226.07999999999998,cont,6.689021906265515,15000650,1,0,1,0
6bba_786893ac,13434,16,56.32,159.35999999999999,74.72,cont,4.476985593007873,17000752,1,0,1,0
6bba_786893ac,14327,17,61.36000000000001,97.92000000000002,234.56000000000006,cont,6.321772150671041,18000796,5,0,1,4
6bba_786893ac,14542,18,26.88,14.719999999999999,115.35999999999996,cont,2.1155602833528624,19000805,4,4,0,0
6bba_786893ac,14634,18,36.441636352539064,17.62828094482422,224.28471191406248,cont,3.934643915351271,19000819,2,2,0,0
6bba_786893ac,14891,18,49.2,138.88,48.64,cont,6.988728089752814,19000834,5,5,0,0
6bba_786893ac,15333,19,25.88,12.64,113.92,cont,3.058671331231912,20000850,9,5,4,0
6bba_786893ac,15481,19,35.667971801757815,104.91782836914062,28.697789611816408,cont,4.470727350506084,20000857,9,5,4,0
6bba_786893ac,15600,19,41.66666666666666,112.0,198.93333333333328,cont,1.0475334786482462,20000870,2,0,1,1
6bba_786893ac,15671,19,45.32000000000001,152.47999999999996,33.760000000000005,cont,3.009717594725453,20000875,13,2,5,6
6bba_786893ac,15692,19,45.36,113.44,205.60000000000002,cont,6.426294616651182,20000870,6,1,2,3
6bba_786893ac,15817,19,52.333333333333314,156.53333333333333,70.93333333333332,cont,3.0832207186642027,20000878,9,0,1,8
6bba_786893ac,16450,20,40.13333333333333,187.99999999999994,95.46666666666667,cont,5.319169424642894,21000906,7,0,0,7
6bba_786893ac,16524,20,46.31999999999999,85.92000000000002,37.599999999999994,cont,2.8589380130566036,21000917,8,0,2,6
6bba_786893ac,16712,20,57.48,152.0,77.76,cont,2.4477700310690973,21000920,8,0,1,7
6bba_786893ac,73389,20,36.04293655395508,102.54511383056641,28.130026626586915,cont,1.605551301281308,21000902,5,0,5,0
6bba_786893ac,17527,21,59.48000000000001,56.64,93.92000000000002,cont,0.9564060330215518,22000963,1,1,0,0
6bba_786893ac,17797,22,24.519999999999996,76.0,36.63999999999999,cont,1.801112937186341,23000973,2,0,2,0
6bba_786893ac,18302,22,59.68000000000001,57.280000000000015,177.92000000000002,cont,4.129450818510856,23001003,6,0,1,5
6bba_786893ac,18839,23,47.16,14.560000000000002,222.07999999999996,cont,4.006242979713023,24001030,1,1,0,0
6bba_786893ac,19052,23,59.83999999999999,54.08000000000001,178.56000000000003,cont,4.022687417945862,24001042,1,1,0,0
6bba_786893ac,19494,24,37.519999999999996,206.08,95.36000000000001,cont,5.04310575191519,25001064,1,0,1,0
6bba_786893ac,19780,24,56.94804321289064,140.1108740234375,75.928466796875,cont,1.7682363133406098,25001078,1,0,1,0
6bba_786893ac,19807,24,59.31999999999999,55.99999999999999,94.72,cont,1.9683975875061552,25001082,2,1,1,0
6bba_786893ac,20459,25,52.16000000000001,92.64,62.08,cont,5.42956734579286,26001110,1,0,1,0
6bba_786893ac,21078,26,42.040000000000006,25.44,74.56,cont,2.134757743866962,27001146,1,0,1,0
6bba_786893ac,21956,27,49.68000000000001,144.0,32.64,cont,2.3084153114203523,28001194,1,1,0,0
6bba_786893ac,22105,27,58.52000000000001,113.44,217.27999999999997,cont,6.476578113672385,28001199,2,1,1,0
6bba_786893ac,22395,28,26.320000000000004,71.36,40.000000000000014,cont,0.9377249730064839,29001208,1,1,0,0
6bba_786893ac,25747,32,42.52000000000001,8.8,61.27999999999999,cont,6.384781955948066,33001375,1,1,0,0
6bba_786893ac,26725,33,58.13333333333334,47.999999999999986,31.46666666666666,cont,5.684145836349462,34001425,8,1,0,7
6bba_786893ac,26797,33,60.84000000000001,50.72,29.279999999999994,cont,6.333415350346146,34001425,7,1,1,5
6bba_786893ac,28562,36,24.720000000000002,54.720000000000006,20.000000000000007,cont,6.522467420386242,37001505,9,1,0,8
6bba_786893ac,29312,37,25.360000000000007,54.72,20.000000000000007,cont,6.128749490108073,38001538,1,0,1,0
6bba_786893ac,30757,39,24.32,53.76000000000002,42.56000000000001,cont,2.612967661491439,40001603,8,0,2,6
6bba_786893ac,31088,39,49.733333333333334,56.533333333333324,55.99999999999999,cont,6.408228014414147,40001620,7,0,0,7
6bba_786893ac,31188,39,58.44,51.84,28.159999999999997,cont,2.217276623473039,40001625,8,0,2,6
6bba_786893ac,32002,40,59.480000000000004,135.84,162.71999999999994,cont,6.6717877336213185,41001653,2,1,1,0
6bba_786893ac,32773,41,60.000000000000014,168.0,248.00000000000006,cont,6.9893892794721575,42001683,1,1,0,0
6bba_786893ac,33043,42,27.64,86.72000000000003,142.08,cont,4.552146814965436,43001690,3,2,1,0
6bba_786893ac,35647,45,56.84,31.360000000000003,105.28000000000002,cont,6.556346400625883,46001789,1,0,1,0
6bba_786893ac,36902,47,41.96000000000001,99.52,38.08,cont,6.252851349584436,48001828,2,2,0,0
6bba_786893ac,37112,47,58.0,35.839999999999996,27.520000000000003,cont,3.881675701291903,48001836,9,2,1,6
6bba_786893ac,37114,47,58.48000000000002,46.72000000000001,46.56000000000001,cont,6.7691344535324465,48001836,3,2,1,0
6bba_786893ac,37159,47,60.47999999999999,28.160000000000004,37.92,cont,3.467157517333173,48001836,3,2,1,0
6bba_786893ac,37693,48,47.28,107.36,47.360000000000014,cont,3.084119860268079,49001851,6,0,1,5
6bba_786893ac,37770,48,51.13333333333331,61.06666666666666,103.46666666666665,cont,6.055351240500705,49001856,6,0,0,6
6bba_786893ac,40089,51,59.31999999999999,40.0,188.64,cont,6.155833742272459,52001946,1,0,1,0
6bba_786893ac,41500,53,56.88000000000001,21.920000000000005,44.16,cont,4.253466600903307,54002001,2,1,1,0
6bba_786893ac,44832,58,51.16,34.720000000000006,94.72,cont,2.9710848713054263,59002094,1,0,1,0
6bba_786893ac,45153,59,15.000000000000004,8.64,55.20000000000001,cont,6.968219038068484,60002102,7,0,0,7
6bba_786893ac,55637,74,38.120000000000005,121.28000000000003,117.60000000000001,cont,6.53025614447855,75002427,5,2,3,0
6bba_786893ac,55670,74,41.28000000000001,65.12,62.71999999999999,cont,4.1720747611350495,75002426,8,0,2,6
6bba_786893ac,56262,75,24.119999999999997,40.16000000000001,55.99999999999999,cont,5.379339091607824,76002443,8,0,3,5
6bba_786893ac,56393,75,34.86666666666667,96.53333333333336,151.99999999999997,cont,2.2165354754411055,76002448,5,1,0,4
6bba_786893ac,56471,75,40.839999999999996,122.88000000000002,138.56,cont,4.362179250386212,76002450,8,0,4,4
6bba_786893ac,56734,75,57.28,24.0,102.72,cont,2.4702138065357824,76002457,3,0,3,0
6bba_786893ac,57156,76,39.839999999999996,60.640000000000015,66.72,cont,4.566972807560827,77002473,2,0,2,0
6bba_786893ac,57861,77,43.133333333333326,126.93333333333332,134.39999999999998,cont,1.6207061325408696,78002495,5,1,0,4
6bba_786893ac,58474,78,37.13333333333334,96.53333333333336,166.93333333333328,cont,5.668704060820822,79002513,5,1,0,4
6bba_786893ac,60125,80,60.0,68.16000000000001,198.87999999999997,cont,4.376200728085944,81002560,9,0,1,8
6bba_786893ac,60613,81,44.38613830566406,124.22401367187503,134.06824218749998,cont,1.0708039210526488,82002572,1,0,1,0
6bba_786893ac,67052,90,59.68000000000001,21.28,16.0,cont,0.8710060634117203,91002769,2,1,1,0
6bba_786893ac,67722,91,59.31999999999999,23.360000000000007,16.0,cont,1.8460028609132852,92002783,1,0,1,0
6bba_786893ac,70803,96,36.96,56.8,90.72,cont,6.144240824747998,97002865,9,0,0,9
6bba_789f8168,433,1,30.240000000000002,173.76000000000002,242.4,cont,2.1194884406384404,2000060,2,1,1,0
6bba_789f8168,747,2,29.360000000000014,141.92000000000002,180.64000000000004,cont,2.8556109788449686,3000106,1,1,0,0
6bba_789f8168,756,2,30.679999999999996,168.64,234.72000000000006,cont,2.2752321310143286,3000104,2,1,1,0
6bba_789f8168,1042,3,29.840000000000003,136.64,170.56,cont,2.16850755128959,4000136,1,0,1,0
6bba_789f8168,1050,3,30.839999999999996,164.64,226.56000000000003,cont,3.576218542329319,4000138,1,0,1,0
6bba_789f8168,1069,3,34.00000000000001,160.00000000000006,212.64000000000004,cont,5.522155517775623,4000138,1,1,0,0
6bba_789f8168,1360,4,34.68000000000001,156.0,206.08,cont,6.331830797644555,5000169,1,0,1,0
6bba_789f8168,1945,6,35.76,163.99999999999997,246.08,cont,2.0892470234512754,7000247,1,0,1,0
6bba_789f8168,2015,6,51.32,98.72,247.99999999999997,cont,2.8506131467633495,7000256,1,0,1,0
6bba_789f8168,2028,6,54.13333333333333,111.99999999999996,252.00000000000006,cont,5.6518575639882345,7000256,4,0,0,4
6bba_789f8168,2478,8,35.32000000000001,157.28000000000003,233.92000000000002,cont,1.647352516160392,9000323,1,1,0,0
6bba_789f8168,2751,9,35.32000000000001,154.72000000000003,228.64,cont,1.229965065560794,10000358,1,0,1,0
6bba_789f8168,3104,10,53.31999999999999,100.63999999999999,240.64000000000004,cont,5.876144702311198,11000410,1,1,0,0
6bba_789f8168,3256,11,30.199999999999992,189.28000000000003,244.00000000000003,cont,1.332500000000009,12000432,1,1,0,0
6bba_789f8168,3515,12,29.840000000000003,187.99999999999997,240.00000000000006,cont,0.853086308646459,13000469,1,0,1,0
6bba_789f8168,3710,13,16.0,247.35999999999999,198.08000000000004,cont,0.14981759743100728,14000499,1,1,0,0
6bba_789f8168,3769,13,29.519999999999996,186.55999999999997,235.99999999999994,cont,1.1490485194281237,14000508,1,1,0,0
6bba_789f8168,3962,14,15.679999999999996,243.99999999999997,192.95999999999998,cont,0.5202538442914261,15000538,1,0,1,0
6bba_789f8168,4004,14,25.040000000000003,217.28000000000003,242.08,cont,1.6467112110203106,15000543,1,0,1,0
6bba_789f8168,4012,14,28.8,186.08,230.72000000000008,cont,0.7181323433044666,15000545,1,0,1,0
6bba_789f8168,4269,15,33.64000000000001,92.16,97.44,cont,2.778369857038466,16000591,4,4,0,0
6bba_789f8168,4348,15,52.16000000000001,74.07999999999998,214.72,cont,1.6559060963713963,16000607,2,2,0,0
6bba_789f8168,4349,15,52.80000000000001,90.71999999999998,223.36,cont,6.693737745086822,16000607,3,2,1,0
6bba_789f8168,4375,15,58.16000000000001,85.44,55.35999999999999,cont,2.109873367053092,16000611,1,1,0,0
6bba_789f8168,4431,16,13.840000000000003,231.51999999999998,166.88,cont,3.5827104352012604,17000614,2,0,0,2
6bba_789f8168,4459,16,20.840000000000007,221.28000000000006,216.80000000000007,cont,6.70359278670772,17000616,2,0,0,2
6bba_789f8168,4510,16,34.36,87.19999999999999,89.92,cont,2.7779896620577977,17000627,4,0,4,0
6bba_789f8168,4574,16,52.16000000000001,71.35999999999999,208.64000000000001,cont,3.0517136579305535,17000640,2,0,2,0
6bba_789f8168,5310,20,13.16,223.36,147.36000000000004,cont,3.831452278510057,21000753,2,2,0,0
6bba_789f8168,5442,20,53.4686395263672,47.12074890136719,142.87040039062498,cont,2.4895010195762333,21000779,1,1,0,0
6bba_789f8168,5509,21,13.0,220.64000000000004,143.36,cont,3.3916206487459597,22000785,2,0,2,0
6bba_789f8168,12051,21,53.97443893432618,46.18168502807617,137.61840087890624,cont,1.8221238105536164,22000811,1,0,1,0
6bba_789f8168,6565,26,38.31999999999999,196.0,214.55999999999995,cont,1.7155105836164335,27000922,1,0,1,0
6bba_789f8168,6629,26,57.36,35.35999999999999,121.28000000000003,cont,1.078025538890437,27000931,1,1,0,0
6bba_789f8168,6682,27,13.84,232.0,179.35999999999999,cont,1.3704059617500253,28000936,1,1,0,0
6bba_789f8168,6802,27,58.16000000000001,32.0,120.8,cont,1.928280078852644,28000955,1,0,1,0
6bba_789f8168,6850,28,13.32,230.72000000000008,176.00000000000003,cont,2.3742801540677267,29000959,1,0,1,0
6bba_789f8168,7220,30,23.840000000000003,208.00000000000006,236.32,cont,3.2191820098434816,31001003,2,2,0,0
6bba_789f8168,7253,30,37.68,188.64000000000001,200.16000000000003,cont,3.1602385689216437,31001005,1,1,0,0
6bba_789f8168,7261,30,42.64,40.31999999999999,38.87999999999999,cont,2.3404513453605427,31001010,2,2,0,0
6bba_789f8168,7277,30,47.16,37.440000000000005,46.72,cont,6.507633018233273,31001010,1,1,0,0
6bba_789f8168,7292,30,53.96000000000001,62.879999999999995,68.8,cont,2.631546895364029,31001013,1,1,0,0
6bba_789f8168,7383,31,23.040000000000006,179.36000000000004,154.88000000000002,cont,3.2896117267087797,32001023,1,1,0,0
6bba_789f8168,7384,31,23.68,205.28,225.44000000000005,cont,2.0470485613438503,32001022,2,0,2,0
6bba_789f8168,7414,31,37.519999999999996,186.72000000000003,192.96,cont,3.984697486949283,32001025,2,1,1,0
6bba_789f8168,7439,31,47.84,33.92,41.76000000000001,cont,6.745629256229555,32001029,3,2,1,0
6bba_789f8168,7464,31,57.68000000000002,36.47999999999999,153.6,cont,3.0048442659312586,32001034,3,3,0,0
6bba_789f8168,7465,31,57.19999999999999,49.760000000000005,161.27999999999997,cont,6.951221228136826,32001034,3,3,0,0
6bba_789f8168,7482,31,61.334901733398446,18.21622543334961,109.72642211914062,cont,5.55327100067049,32001033,4,4,0,0
6bba_789f8168,7515,32,11.68,215.36,129.6,cont,5.147173071211031,33001038,4,2,1,1
6bba_789f8168,7535,32,22.480000000000004,176.00000000000003,140.64000000000001,cont,0.8575628621273105,33001043,1,0,1,0
6bba_789f8168,7564,32,37.8,86.72,94.88000000000004,cont,3.8285220712436705,33001049,1,0,1,0
6bba_789f8168,7611,32,57.84,36.800000000000004,140.64000000000001,cont,4.2549563011857146,33001056,4,0,3,1
6bba_789f8168,7613,32,58.33333333333333,27.466666666666654,132.79999999999998,cont,1.8001760137707528,33001056,3,0,1,2
6bba_789f8168,7648,33,12.0,210.71999999999997,118.72000000000003,cont,3.628083171593502,34001059,2,0,2,0
6bba_789f8168,7746,33,59.800000000000004,14.240000000000002,114.88000000000002,cont,5.419929225783304,34001075,2,0,0,2
6bba_789f8168,7986,35,53.8,24.64,29.279999999999994,cont,0.7045832101320634,36001110,3,0,0,3
6bba_789f8168,8256,38,11.599999999999998,191.51999999999998,53.12,cont,2.631697408517933,39001154,2,0,1,1
6bba_789f8168,8366,39,19.520000000000003,244.64000000000001,224.15999999999994,cont,1.5331080563678723,40001172,2,2,0,0
6bba_789f8168,8373,39,24.840000000000003,173.28000000000003,154.08,cont,2.5646701176759423,40001176,1,1,0,0
6bba_789f8168,8434,39,61.839999999999996,3.3599999999999994,59.519999999999996,cont,1.8567713914211421,40001186,2,2,0,0
6bba_789f8168,8460,40,18.640000000000004,242.08,211.20000000000002,cont,3.066043867918407,41001190,2,0,2,0
6bba_789f8168,8463,40,20.68,139.35999999999999,26.24,cont,3.350653854100722,41001191,2,0,1,1
6bba_789f8168,8547,41,18.16,238.72000000000003,201.28000000000006,cont,1.093589988295443,42001207,1,1,0,0
6bba_789f8168,8636,42,17.68,235.35999999999999,191.36,cont,1.1645707363659792,43001222,1,0,1,0
6bba_789f8168,8968,46,15.48,240.64000000000004,192.0,cont,2.40944268711667,47001278,1,1,0,0
6bba_789f8168,9047,47,15.16,235.99999999999994,184.0,cont,0.48232671758883927,48001294,1,0,1,0
6bba_789f8168,9461,53,11.639999999999999,217.92000000000004,126.4,cont,2.223638979803161,54001378,1,1,0,0
6bba_789f8168,9462,53,12.32,198.08,95.36000000000001,cont,2.3790577178790753,54001377,1,1,0,0
6bba_789f8168,9522,54,11.159999999999998,220.16,124.79999999999998,cont,3.030178418839397,55001391,1,0,1,0
6bba_789f8168,9523,54,12.0,195.36000000000004,86.56,cont,2.207369758332289,55001389,1,0,1,0
6bba_789f8168,9524,54,11.866666666666664,208.53333333333327,107.46666666666665,cont,0.6862298873798401,55001390,2,0,0,2
6bba_789f8168,9993,62,10.68,193.27999999999997,71.84,cont,1.7246448909848044,63001476,1,1,0,0
6bba_789f8168,9994,62,11.000000000000002,207.20000000000002,99.67999999999999,cont,2.818754434586314,63001479,1,1,0,0
6bba_789f8168,9995,62,11.680000000000003,177.92,47.2,cont,2.7662254291723922,63001478,1,1,0,0
6bba_789f8168,10046,63,10.480000000000002,194.71999999999997,74.07999999999998,cont,1.476605587995665,64001489,2,0,1,1
6bba_789f8168,10047,63,10.999999999999996,167.99999999999994,23.466666666666654,cont,2.0880529873012943,64001487,2,0,0,2
6bba_789f8168,10048,63,11.160000000000002,179.35999999999999,47.52,cont,3.0392713456517835,64001488,2,0,1,1
6bba_789f8168,10531,71,60.48000000000002,135.36000000000004,172.64000000000004,cont,2.3047518846938875,72001561,1,1,0,0
6bba_789f8168,10589,72,60.32000000000001,132.00000000000003,167.36,cont,2.168324886288944,73001566,1,0,1,0
6bba_78a7bd97,2499,10,45.0,14.560000000000002,114.56000000000002,cont,3.371316971896889,11000041,1,1,0,0
6bba_78a7bd97,2830,11,45.36,19.360000000000003,120.63999999999999,cont,5.489831082328484,12000047,1,0,1,0
6bba_78a7bd97,3722,14,35.64,36.800000000000004,146.56000000000003,cont,2.082093672844722,15000064,2,0,0,2
6bba_78a7bd97,4088,15,35.32000000000001,41.440000000000005,148.00000000000003,cont,3.0894237427876323,16000072,3,3,0,0
6bba_78a7bd97,4832,17,34.24000000000001,60.96000000000001,152.95999999999998,cont,3.0822784794693514,18000088,3,3,0,0
6bba_78a7bd97,4833,17,33.52,74.07999999999998,145.92,cont,6.384451082512881,18000088,2,2,0,0
6bba_78a7bd97,8315,25,33.32000000000001,83.52,144.00000000000003,cont,3.380156246388608,26000175,2,0,2,0
6bba_78a7bd97,9030,27,6.0,28.0,131.99999999999997,cont,2.437500000000005,28000193,1,1,0,0
6bba_78a7bd97,10147,29,28.999999999999993,14.080000000000002,75.52000000000001,cont,2.1860931190139263,30000221,4,3,1,0
6bba_78a7bd97,10628,30,28.840000000000003,30.560000000000002,90.56000000000002,cont,2.0016543938702243,31000231,8,1,2,5
6bba_78a7bd97,10662,30,33.44,121.92000000000002,164.0,cont,1.0581233387464846,31000232,8,0,2,6
6bba_78a7bd97,13099,35,34.680000000000014,216.0,206.56,cont,1.34443587147174,36000287,2,2,0,0
6bba_78a7bd97,14004,37,22.92,11.680000000000003,164.95999999999998,cont,0.9515620381772264,38000304,2,0,2,0
6bba_78a7bd97,15040,39,17.48,171.99999999999994,125.92,cont,3.153421257142793,40000328,1,1,0,0
6bba_78a7bd97,16337,41,44.160000000000004,33.92000000000001,141.43999999999997,cont,1.3842100635380425,42000365,1,1,0,0
6bba_78a7bd97,17116,43,23.199999999999996,28.16,171.36,cont,2.37144240126131,44000383,1,0,1,0
6bba_78a7bd97,17662,44,21.84,189.28,140.16,cont,6.430997808466427,45000392,3,3,0,0
6bba_78a7bd97,17667,44,22.36,28.800000000000004,170.72000000000003,cont,3.206854231564006,45000394,2,2,0,0
6bba_78a7bd97,17759,44,30.439999999999998,90.07999999999998,126.72,cont,6.3663717345517865,45000395,5,5,0,0
6bba_78a7bd97,18323,45,30.880000000000003,105.92,120.0,cont,3.5554836608399643,46000409,6,0,0,6
6bba_78a7bd97,18403,45,39.00000000000001,41.44000000000001,243.83999999999997,cont,4.870583416029329,46000412,5,1,4,0
6bba_78a7bd97,18459,45,44.839999999999996,60.160000000000004,150.07999999999998,cont,4.326926774571076,46000414,5,0,2,3
6bba_78a7bd97,18970,46,39.67999999999999,46.08,243.99999999999994,cont,3.3402743618600916,47000429,1,0,1,0
6bba_78a7bd97,19294,47,23.399999999999995,19.466666666666665,156.53333333333333,cont,4.272304624620724,48000435,9,2,0,7
6bba_78a7bd97,20412,49,24.040000000000003,70.4,185.92000000000002,cont,3.8194414267664816,50000468,6,1,0,5
6bba_78a7bd97,20515,49,32.68,120.16000000000001,128.64,cont,2.4494415817691992,50000472,1,1,0,0
6bba_78a7bd97,21668,51,26.680000000000003,43.36,169.44,cont,2.408949459826833,52000505,2,1,1,0
6bba_78a7bd97,21770,51,33.480000000000004,122.39999999999998,134.07999999999998,cont,6.041701048752748,52000506,7,0,1,6
6bba_78a7bd97,22093,51,60.12000000000002,33.92,71.2,cont,6.361890565115079,52000515,9,1,2,6
6bba_78a7bd97,23541,54,29.64,66.56000000000002,192.0,cont,6.6920017558276275,55000562,6,6,0,0
6bba_78a7bd97,24116,55,30.839999999999996,73.44,193.27999999999997,cont,1.4167353537269998,56000584,7,1,6,0
6bba_78a7bd97,24117,55,30.199999999999992,86.56,202.72000000000006,cont,6.539388470071204,56000584,7,0,3,4
6bba_78a7bd97,24174,55,33.2,156.48000000000002,152.64,cont,6.971420452461596,56000586,3,0,3,0
6bba_78a7bd97,24203,55,35.52,174.55999999999995,143.36,cont,2.234360227223873,56000586,6,1,3,2
6bba_78a7bd97,24825,56,37.400000000000006,177.92000000000002,144.64000000000001,cont,4.148844907019305,57000607,5,0,1,4
6bba_78a7bd97,24972,56,52.320000000000014,26.560000000000002,158.56,parent,3.5477884836895073,57000619,2,0,2,0
6bba_78a7bd97,25668,57,59.68000000000001,29.279999999999994,146.72000000000003,daughter,4.963490612714021,58000647,2,1,1,0
6bba_78a7bd97,28280,62,34.0,216.0,158.71999999999997,cont,0.5200000000000102,63000736,1,1,0,0
6bba_78a7bd97,28324,62,37.64000000000001,31.360000000000007,76.64000000000001,cont,3.1186878751327463,63000738,4,3,1,0
6bba_78a7bd97,28330,62,36.519999999999996,119.52000000000002,212.0,cont,3.588749129223162,63000739,5,4,1,0
6bba_78a7bd97,28833,63,34.32,224.80000000000004,160.64000000000001,cont,1.725869346155714,64000759,2,1,1,0
6bba_78a7bd97,30161,65,49.32000000000001,114.72,78.56,daughter,2.850427873583183,66000836,1,1,0,0
6bba_78a7bd97,30542,66,38.47999999999999,153.43999999999997,227.35999999999999,cont,4.6435210239644755,67000848,5,5,0,0
6bba_78a7bd97,30579,66,41.32,42.72,77.44000000000001,cont,6.155061560415135,67000849,2,2,0,0
6bba_78a7bd97,30612,66,42.2,144.16000000000003,153.28000000000003,cont,5.157679347342167,67000850,8,8,0,0
6bba_78a7bd97,31119,67,43.639999999999986,168.64,156.64000000000001,cont,0.8456247690317725,68000873,9,0,4,5
6bba_78a7bd97,33173,71,40.71999999999999,194.07999999999998,159.36,cont,3.4219274539358664,72000969,2,1,1,0
6bba_78a7bd97,33239,71,45.87999999999999,62.4,239.36,cont,1.0050069962443082,72000972,4,0,2,2
6bba_78a7bd97,33658,72,38.11231140136718,205.47718505859373,167.35544677734376,cont,3.3623322562717175,73001001,1,1,0,0
6bba_78a7bd97,33899,72,55.99999999999999,37.06666666666666,96.53333333333336,cont,1.6924374783298781,73001015,9,2,0,7
6bba_78a7bd97,47897,72,43.785392456054694,158.11749023437503,212.38579345703124,cont,4.250730877290966,73001004,4,4,0,0
6bba_78a7bd97,34171,73,39.32000000000001,227.36,172.16,cont,5.452740709954958,74001032,3,0,3,0
6bba_78a7bd97,34257,73,46.16,148.8,225.44,cont,6.482752116192622,74001034,2,0,2,0
6bba_78a7bd97,34381,73,53.64,66.55999999999999,219.36,cont,2.6675749755161418,74001040,8,0,5,3
6bba_78a7bd97,34429,73,55.84000000000001,133.44,40.8,cont,3.175077016782425,74001043,2,0,2,0
6bba_78a7bd97,34492,73,59.03999999999999,34.080000000000005,92.64,cont,6.4761092196626775,74001044,2,0,2,0
6bba_78a7bd97,34907,74,53.31999999999999,66.56,221.28000000000003,cont,2.9359932305780427,75001068,1,1,0,0
6bba_78a7bd97,35539,75,58.400000000000006,50.72,108.16,cont,2.673359602167275,76001105,5,0,1,4
6bba_78a7bd97,35666,76,32.0,18.720000000000002,241.91999999999996,cont,3.521154152050153,77001116,4,0,1,3
6bba_78a7bd97,36420,77,48.120000000000005,190.08000000000004,241.44000000000003,cont,5.5308278087461895,78001150,2,1,0,1
6bba_78a7bd97,36956,78,51.360000000000014,90.88,83.84000000000002,cont,6.416610402891836,79001189,5,0,0,5
6bba_78a7bd97,37003,78,54.720000000000006,80.64,249.44,cont,6.294353124031088,79001187,1,0,0,1
6bba_78a7bd97,37043,78,57.800000000000004,193.27999999999997,187.35999999999999,cont,5.258827400191037,79001188,5,0,0,5
6bba_78a7bd97,37088,78,59.733333333333334,51.99999999999999,103.99999999999999,cont,6.677071632855071,79001196,9,0,0,9
6bba_78a7bd97,37148,78,62.133333333333326,63.999999999999986,107.46666666666663,cont,5.260933142962592,79001196,1,0,1,0
6bba_78a7bd97,37156,78,61.99999999999999,91.99999999999999,171.4666666666667,cont,3.3290473747922285,79001198,4,0,0,4
6bba_78a7bd97,37990,80,53.999999999999986,100.0,239.99999999999994,cont,6.512682938889317,81001244,4,0,0,4
6bba_78a7bd97,38020,80,55.73333333333333,100.0,87.99999999999997,cont,1.2590109240766834,81001245,13,3,0,10
6bba_78a7bd97,38099,80,59.999999999999986,127.99999999999997,0.0,cont,5.873088119762905,81001254,5,0,0,5
6bba_78a7bd97,38482,81,54.720000000000006,100.00000000000001,88.32000000000001,cont,1.2870369312883139,82001275,3,0,3,0
6bba_78a7bd97,39305,83,41.84,184.0,242.71999999999997,cont,2.85787563147873,84001328,3,0,3,0
6bba_78a7bd97,39500,83,55.11999999999999,62.71999999999999,125.28000000000002,cont,5.286672216527162,84001341,3,1,2,0
6bba_78a7bd97,39576,83,58.359999999999985,50.72,248.00000000000006,cont,2.745480661468977,84001344,2,0,1,1
6bba_78a7bd97,39672,83,60.72,80.16,121.11999999999998,cont,6.936942094684082,84001341,8,1,0,7
6bba_78a7bd97,39923,84,50.48000000000001,87.36,235.35999999999999,cont,2.5714566032698127,85001364,1,0,1,0
6bba_78a7bd97,40126,84,61.32,39.36,246.72000000000006,cont,6.371347115210407,85001371,1,1,0,0
6bba_78a7bd97,40616,85,61.32,42.07999999999999,247.35999999999999,cont,4.958487199993565,86001399,1,0,1,0
6bba_78a7bd97,40991,86,55.24,20.319999999999997,161.28,cont,6.1183785638026675,87001425,2,2,0,0
6bba_78a7bd97,41217,87,37.00000000000001,195.83999999999997,246.56,cont,3.493258686384377,88001442,1,0,0,1
6bba_78a7bd97,43396,91,50.80000000000001,97.28000000000003,173.91999999999996,cont,3.603618407240136,92001571,1,1,0,0
6bba_78a7bd97,43411,91,51.00000000000001,29.920000000000005,169.28,cont,3.263297795175903,92001568,2,2,0,0
6bba_78a7bd97,45540,95,58.839999999999996,94.56,98.07999999999998,cont,5.7050230060885845,96001698,5,0,1,4
6bba_78a7bd97,45623,95,61.99999999999999,85.06666666666665,92.53333333333332,cont,2.0908613987753797,96001698,4,0,2,2
6bba_7af54fde,205,1,47.59999999999998,207.99999999999997,243.99999999999997,cont,5.539915528462538,2000028,1,0,0,1
6bba_7af54fde,328,2,41.16,18.56,122.71999999999998,cont,3.499451742773432,3000042,1,1,0,0
6bba_7af54fde,469,3,41.16,20.640000000000008,123.35999999999999,cont,4.4448428684262815,4000058,1,0,1,0
6bba_7af54fde,511,3,55.133333333333326,12.533333333333331,111.99999999999999,cont,0.7529946002831341,4000065,2,0,0,2
6bba_7af54fde,650,4,52.16000000000002,138.72000000000003,179.36000000000004,cont,3.003437873920449,5000082,1,1,0,0
6bba_7af54fde,750,5,31.000000000000007,52.64,155.36,cont,0.6106195624118268,6000089,1,1,0,0
6bba_7af54fde,1561,10,58.96000000000001,14.080000000000002,43.36,cont,0.7666831972855539,11000187,1,1,0,0
6bba_7af54fde,1849,12,51.84000000000001,93.92,143.99999999999997,cont,3.618499559555017,13000215,1,1,0,0
6bba_7af54fde,1879,12,59.519999999999996,17.919999999999998,44.00000000000001,cont,1.00933300996252,13000216,1,1,0,0
6bba_7af54fde,1951,13,34.0,55.36000000000001,65.44,cont,2.0684150030881114,14000222,1,0,1,0
6bba_7af54fde,1983,13,45.84,226.72,243.36000000000004,cont,5.190749584597578,14000226,1,1,0,0
6bba_7af54fde,1989,13,49.160000000000004,41.28,31.360000000000007,cont,1.6560655557676445,14000227,1,0,1,0
6bba_7af54fde,2140,14,45.63999999999999,225.92,242.72000000000003,cont,3.8616920676045696,15000245,1,0,1,0
6bba_7af54fde,2243,15,13.133333333333333,43.999999999999986,144.0,cont,5.344688432869072,16000251,1,0,0,1
6bba_7af54fde,2581,17,33.96,71.19999999999999,150.71999999999997,cont,0.7157382552302256,18000288,1,1,0,0
6bba_7af54fde,2747,18,35.480000000000004,75.35999999999999,149.28,cont,4.200994301948044,19000305,1,0,1,0
6bba_7af54fde,2955,19,49.367156982421875,166.02425048828127,182.23875244140623,cont,5.646676572629084,20000324,2,2,0,0
6bba_7af54fde,2963,19,49.52000000000001,180.64000000000004,190.07999999999998,cont,1.6503153107815625,20000324,2,2,0,0
6bba_7af54fde,3129,20,48.640000000000015,182.56000000000003,186.56000000000006,cont,1.6838952238782683,21000337,4,0,2,2
6bba_7af54fde,12495,20,49.26627487182618,164.92766784667973,184.57280212402344,cont,6.574827166536884,21000337,2,0,2,0
6bba_7af54fde,3628,23,54.84,143.99999999999997,144.64,cont,5.200939368277613,24000379,1,1,0,0
6bba_7af54fde,3902,25,43.31999999999999,103.36,150.07999999999998,cont,4.2002085141692485,26000408,1,1,0,0
6bba_7af54fde,4062,26,41.63999999999999,33.92,90.72,cont,6.353604537976854,27000425,2,0,2,0
6bba_7af54fde,4073,26,44.20000000000002,104.64000000000003,148.64000000000001,cont,5.351088236284313,27000423,2,1,1,0
6bba_7af54fde,4079,26,47.397493896484384,47.193970947265626,93.57273681640625,cont,5.351621425640934,27000425,1,1,0,0
6bba_7af54fde,4211,27,40.48000000000001,105.92000000000002,142.72,cont,3.0436557993636435,28000448,1,0,1,0
6bba_7af54fde,4215,27,42.88,41.599999999999994,88.47999999999999,cont,3.585620587848072,28000444,3,0,0,3
6bba_7af54fde,4223,27,45.0,107.35999999999999,148.64000000000001,cont,5.0128550373015965,28000448,1,0,1,0
6bba_7af54fde,4371,28,45.68,109.28,148.00000000000003,cont,4.383676574805678,29000470,2,2,0,0
6bba_7af54fde,4541,29,45.84,112.0,148.00000000000003,cont,3.0174722968902388,30000496,2,0,2,0
6bba_7af54fde,4706,30,44.79999999999999,49.28,88.47999999999999,cont,3.28507324880284,31000515,1,1,0,0
6bba_7af54fde,4760,30,57.040000000000006,93.27999999999999,86.08,cont,2.6834157919524952,31000523,2,1,1,0
6bba_7af54fde,4862,31,45.84,52.64,89.44,cont,3.7342485857264505,32000533,5,0,1,4
6bba_7af54fde,4893,31,56.28,96.0,84.00000000000001,cont,1.8746753885673155,32000539,1,0,1,0
6bba_7af54fde,5290,34,41.73333333333333,11.466666666666665,28.53333333333333,cont,2.579180970611928,35000585,1,0,0,1
6bba_7af54fde,5460,35,43.0,234.56,189.28000000000003,cont,3.09369444394561,36000607,1,0,1,0
6bba_7af54fde,5783,37,45.86666666666667,80.53333333333333,75.46666666666667,cont,3.5272857541836156,38000648,2,0,0,2
6bba_7af54fde,5837,37,61.00000000000001,120.00000000000003,83.36,cont,6.586884933335337,38000657,2,0,2,0
6bba_7af54fde,6008,39,21.88,98.72,132.64000000000001,cont,3.5106770180408238,40000681,1,1,0,0
6bba_7af54fde,6040,39,46.72,38.08,27.999999999999996,cont,0.9901838528778389,40000684,1,1,0,0
6bba_7af54fde,6098,39,62.000000000000014,220.0,155.04,cont,5.710112192417963,40000689,2,1,0,1
6bba_7af54fde,6206,40,57.16,244.00000000000003,154.56,cont,6.259984275140008,41000702,1,0,1,0
6bba_7af54fde,6233,40,61.680000000000014,135.52000000000004,84.0,cont,6.48714979786963,41000703,1,0,1,0
6bba_7af54fde,6868,45,53.16,125.28000000000002,73.28,cont,2.9943243186067963,46000773,1,0,1,0
6bba_7af54fde,6877,45,57.08,197.60000000000002,72.96,cont,1.8449297073330402,46000772,1,1,0,0
6bba_7af54fde,6972,46,56.32,204.79999999999998,74.72,cont,1.6787909525905913,47000785,1,0,1,0
6bba_7af54fde,6990,46,60.99999999999999,178.4,73.06666666666666,cont,3.3948285834650314,47000790,2,0,0,2
6bba_7af54fde,7094,47,62.000000000000014,80.64,18.720000000000006,cont,1.8531413363529514,48000802,1,1,0,0
6bba_7af54fde,7253,49,54.52,218.72000000000003,80.00000000000001,cont,1.1777607195436526,50000826,1,1,0,0
6bba_7af54fde,7337,50,51.00000000000001,98.08,55.99999999999999,cont,1.9771649526531543,51000838,1,1,0,0
6bba_7af54fde,7344,50,53.67999999999999,222.72000000000003,82.08000000000001,cont,0.9820164204329844,51000841,1,0,1,0
6bba_7af54fde,7532,52,50.16,99.36000000000001,55.35999999999999,cont,3.0267342652601625,53000863,1,1,0,0
6bba_7af54fde,7542,52,55.16,142.72,72.64,parent,3.364220981520086,53000865,1,0,1,0
6bba_7af54fde,8049,57,61.680000000000014,189.28,69.92000000000002,cont,1.9228632621692174,58000938,1,1,0,0
6bba_7af54fde,8801,65,47.16000000000001,232.0,156.0,cont,1.7842470400703898,66001045,1,1,0,0
6bba_7af54fde,8831,65,59.68000000000001,109.28,24.0,cont,3.603435209976719,66001047,2,1,1,0
6bba_7af54fde,9105,68,48.0,137.28000000000003,59.36,cont,1.7258693461557313,69001080,1,1,0,0
6bba_7af54fde,9657,73,47.32,96.0,32.00000000000001,cont,4.594728807285587,74001136,1,1,0,0
6bba_7af54fde,9764,74,47.160000000000004,96.0,29.920000000000005,cont,4.847432471164498,75001150,1,0,1,0
6bba_7af54fde,9766,74,48.0,135.35999999999999,52.0,cont,1.0725000000000051,75001149,1,0,1,0
6bba_7af54fde,9969,76,47.32,136.0,52.0,cont,1.3715616099906007,77001178,1,0,1,0
6bba_7af54fde,10090,77,56.86666666666665,186.93333333333328,50.93333333333333,cont,2.265306730518719,78001195,1,0,1,0
6bba_7af54fde,10098,77,59.999999999999986,191.99999999999994,63.46666666666666,cont,5.290893972456591,78001195,5,0,0,5
6bba_7af54fde,10486,81,51.03895080566407,133.4109692382813,46.75668212890625,cont,6.1991317002947675,82001239,1,1,0,0
6bba_7af54fde,10602,82,56.73333333333333,150.13333333333335,23.999999999999993,cont,2.453995112849438,83001253,2,0,0,2
6bba_7af54fde,10614,82,61.839999999999996,141.92000000000002,22.72,cont,6.637364169984942,83001253,1,0,1,0
6bba_7af54fde,10681,83,47.480000000000004,104.64,28.640000000000004,cont,2.6622737579933418,84001263,1,1,0,0
6bba_7af54fde,10779,84,43.47999999999999,143.36,52.0,cont,4.152026049111459,85001275,1,1,0,0
6bba_7af54fde,10810,84,55.86666666666666,192.5333333333333,47.466666666666654,cont,2.025282902049341,85001280,2,0,0,2
6bba_7af54fde,10820,84,59.52000000000001,192.0,58.879999999999995,cont,6.6938955399677464,85001280,1,0,1,0
6bba_7af54fde,10891,85,43.16000000000001,143.99999999999997,52.0,cont,4.63284621615048,86001291,1,0,1,0
6bba_7af54fde,11017,86,48.519999999999996,107.36000000000001,30.080000000000002,cont,1.326541556077299,87001307,1,1,0,0
6bba_7af54fde,11126,87,48.480000000000004,108.0,29.279999999999994,cont,0.9821508603570022,88001324,2,0,1,1
6bba_7af54fde,11127,87,48.99999999999999,132.53333333333333,48.53333333333332,cont,5.021238313542689,88001325,3,0,0,3
6bba_7af54fde,11264,88,58.266666666666666,243.99999999999994,127.46666666666667,cont,2.913189892063712,89001346,1,0,1,0
6bba_7af54fde,11274,88,59.86666666666666,239.46666666666658,116.0,cont,2.85828626054534,89001346,4,0,0,4
6bba_7af54fde,11300,89,6.32,252.0,204.00000000000003,cont,1.2454329066633776,90001348,1,1,0,0
6bba_7af54fde,11408,90,5.800000000000001,252.0,204.00000000000003,cont,1.9918682342213323,91001365,1,0,1,0
6bba_7af54fde,11770,93,40.84000000000001,218.72000000000003,56.00000000000001,cont,0.5813776741499187,94001419,1,1,0,0
6bba_7af54fde,11875,94,40.84,219.36,55.36000000000001,cont,0.7299850169695155,95001435,1,0,1,0
6bba_7af54fde,12074,96,41.16,221.27999999999997,54.08000000000001,cont,0.5411518386737695,97001468,1,1,0,0
6bba_7b5d3b2c,612,10,13.320000000000002,68.0,180.64000000000001,cont,5.4837430191430405,11000078,1,1,0,0
6bba_7b5d3b2c,676,11,11.133333333333331,67.46666666666667,179.99999999999997,cont,2.1484783455713203,12000083,1,0,0,1
6bba_7b5d3b2c,1625,25,32.839999999999996,229.27999999999997,24.0,cont,1.9503385122844776,26000165,1,1,0,0
6bba_7b5d3b2c,1686,26,32.64,230.72000000000008,24.0,cont,0.8818526308289653,27000172,1,0,1,0
6bba_7b5d3b2c,2523,39,1.8399999999999999,119.36000000000001,240.64000000000004,cont,1.5259433557639115,40000263,1,1,0,0
6bba_7b5d3b2c,2592,40,0.8666666666666671,128.53333333333333,235.99999999999997,cont,0.7746022724957429,41000269,2,1,0,1
6bba_7b5d3b2c,2594,40,1.68,117.44000000000003,241.91999999999996,cont,4.505501151370387,41000269,1,0,1,0
6bba_7b5d3b2c,2663,41,1.24,130.72,235.99999999999994,cont,1.8276735553703203,42000275,1,0,1,0
6bba_7d3058ae,1100,14,15.76,164.0,21.279999999999998,cont,0.9475296829123617,15000205,1,1,0,0
6bba_7d3058ae,1203,15,35.56,26.560000000000002,114.72,cont,0.9019847906145633,16000224,1,1,0,0
6bba_7d3058ae,1273,16,34.04,28.640000000000004,113.44,cont,5.176919480975146,17000238,1,0,1,0
6bba_7d3058ae,1579,20,33.32,40.000000000000014,110.71999999999998,cont,0.7218000502216633,21000293,1,1,0,0
6bba_7d3058ae,1655,21,32.99999999999999,43.36000000000001,110.72,cont,2.0152620797305625,22000307,1,0,1,0
6bba_7d3058ae,2978,38,35.160000000000004,95.36000000000001,17.919999999999998,cont,4.6430945028612065,39000585,1,1,0,0
6bba_7d3058ae,3055,39,34.733333333333334,100.53333333333332,7.999999999999998,cont,1.1648982132453587,40000601,1,0,0,1
6bba_7d3058ae,3058,39,35.35999999999999,94.56,19.359999999999996,cont,6.108314262134846,40000601,1,0,1,0
6bba_7d3058ae,4692,59,49.00000000000001,9.28,110.72000000000003,cont,0.7079481796007374,60000943,1,1,0,0
6bba_7d3058ae,4772,60,49.00000000000001,10.720000000000002,109.28,cont,1.3373463696813916,61000957,1,0,1,0
6bba_7d3058ae,5283,66,49.160000000000004,13.44,109.91999999999999,cont,2.315610745570156,67001085,1,1,0,0
6bba_7d3058ae,5372,67,53.64,14.719999999999999,91.52000000000001,cont,2.7952361818815925,68001104,2,0,1,1
6bba_7d3058ae,5550,69,51.72,15.359999999999996,100.64,cont,2.1995201891548963,70001140,1,1,0,0
6bba_7d3058ae,5640,70,50.279999999999994,14.719999999999999,103.20000000000002,cont,2.2667439478247076,71001157,3,2,1,0
6bba_7d3058ae,5736,71,53.86666666666666,11.999999999999996,97.06666666666666,cont,0.9224250948210201,72001179,1,0,0,1
6bba_7d3058ae,5989,74,32.99999999999999,167.46666666666664,7.466666666666664,cont,0.28789978242824155,75001230,1,0,0,1
6bba_7d3058ae,6536,80,33.68,171.36,12.0,cont,0.6758906161502806,81001343,1,1,0,0
6bba_7d3058ae,6623,81,33.839999999999996,172.64,12.640000000000004,cont,1.9814341397079112,82001363,1,0,1,0
6bba_7d3058ae,6773,83,17.64,113.44,202.88,parent,3.439515091186545,84001401,1,1,0,0
6bba_7d3058ae,6856,84,13.999999999999998,123.99999999999999,219.99999999999997,daughter,1.6750116604071814,85001403,2,0,0,2
6bba_7d3058ae,6864,84,18.000000000000004,112.64000000000001,200.0,daughter,1.9351150644083115,85001421,3,0,1,2
6bba_7d3058ae,7350,89,54.28,22.08,94.88000000000001,cont,3.6116333302953096,90001524,1,1,0,0
6bba_7d3058ae,7435,90,48.0,33.92,105.12000000000002,cont,3.523403228769031,91001538,3,1,0,2
6bba_7d3058ae,7527,91,51.99999999999999,31.46666666666666,100.0,cont,1.9234887646611893,92001563,3,0,0,3
6bba_7f87b3d8,361,1,39.839999999999996,48.0,189.12,cont,2.009357017804454,2000060,1,1,0,0
6bba_7f87b3d8,580,2,39.63999999999999,49.27999999999999,186.88000000000002,cont,3.7902580716885352,3000091,2,1,1,0
6bba_7f87b3d8,767,3,33.959999999999994,241.60000000000002,87.83999999999997,cont,4.204073173720951,4000131,3,0,1,2
6bba_7f87b3d8,799,3,37.760000000000005,128.16,60.79999999999998,cont,5.405219988585119,4000124,5,0,2,3
6bba_7f87b3d8,983,4,35.32,248.47999999999996,46.72,cont,1.7683405002996293,5000168,4,0,1,3
6bba_7f87b3d8,984,4,35.6,42.56,140.64,cont,1.2351069032678907,5000166,4,2,1,1
6bba_7f87b3d8,998,4,37.480000000000004,216.80000000000007,104.32000000000001,cont,4.007462180108487,5000169,4,2,2,0
6bba_7f87b3d8,1173,5,34.44,42.4,122.72000000000001,cont,2.990176625217977,6000202,5,1,2,2
6bba_7f87b3d8,1197,5,35.84,217.28,95.52000000000001,cont,2.3798900315981,6000204,2,0,2,0
6bba_7f87b3d8,1198,5,34.720000000000006,250.08000000000004,45.75999999999999,cont,5.760137937801491,6000203,4,1,0,3
6bba_7f87b3d8,1394,6,33.68,32.959999999999994,128.64000000000001,cont,6.596399230830412,7000230,3,1,1,1
6bba_7f87b3d8,1563,7,31.439999999999998,29.919999999999998,101.27999999999999,cont,2.4260439119067856,8000264,6,3,1,2
6bba_7f87b3d8,1589,7,33.96,115.2,19.839999999999996,cont,4.846669765158342,8000288,1,1,0,0
6bba_7f87b3d8,1952,9,27.600000000000005,24.480000000000008,72.48000000000002,cont,3.7485407577482834,10000325,3,0,1,2
6bba_7f87b3d8,1955,9,25.04,125.11999999999998,74.23999999999998,cont,3.7852734792218206,10000343,5,0,1,4
6bba_7f87b3d8,3976,22,33.040000000000006,218.72000000000003,70.88,cont,5.493221106509354,23000638,2,2,0,0
6bba_7f87b3d8,4020,22,48.84,191.36,211.35999999999996,cont,2.016113603569994,23000644,1,1,0,0
6bba_7f87b3d8,4132,23,46.733333333333334,182.39999999999998,195.99999999999997,cont,2.5250034378414505,24000658,3,0,0,3
6bba_7f87b3d8,4136,23,47.84,224.64000000000001,176.80000000000004,cont,3.66389755758808,24000659,5,0,1,4
6bba_7f87b3d8,4193,24,30.159999999999997,26.720000000000002,91.35999999999999,cont,1.8121482417285941,25000667,1,1,0,0
6bba_7f87b3d8,4226,24,39.00000000000001,26.080000000000002,225.44000000000003,cont,3.8222058830601866,25000672,1,1,0,0
6bba_7f87b3d8,4262,24,54.24,241.76000000000005,197.60000000000002,cont,1.8625222314109378,25000680,1,1,0,0
6bba_7f87b3d8,4300,25,29.0,21.28,81.92,cont,1.3328962825366417,26000684,1,0,1,0
6bba_7f87b3d8,4328,25,37.160000000000004,21.92,213.91999999999996,cont,2.142844560974954,26000689,1,0,1,0
6bba_7f87b3d8,4345,25,41.52,170.23999999999998,174.55999999999995,cont,3.150405231870972,26000693,2,0,1,1
6bba_7f87b3d8,4352,25,42.72,145.28000000000003,137.76000000000002,cont,2.33197408326079,26000692,2,0,2,0
6bba_7f87b3d8,4367,25,48.67999999999999,182.55999999999997,216.96000000000004,cont,4.542623484837809,26000696,4,0,1,3
6bba_7f87b3d8,4368,25,50.52,235.52000000000004,181.28,cont,3.6840223007603914,26000697,4,0,2,2
6bba_7f87b3d8,4504,27,24.999999999999996,117.06666666666666,84.53333333333329,parent,2.0643825583506294,28000723,1,0,0,1
6bba_7f87b3d8,4769,29,48.480000000000004,218.40000000000003,186.55999999999997,cont,2.4130018907990998,30000774,2,2,0,0
6bba_7f87b3d8,4794,30,21.480000000000004,97.75999999999999,52.32,cont,5.857984134922187,31000780,2,0,2,0
6bba_7f87b3d8,4839,30,37.4,220.5333333333333,105.59999999999997,cont,2.6702848175699283,31000791,4,0,0,4
6bba_7f87b3d8,4854,30,43.639999999999986,235.36000000000004,136.48,cont,2.636359054169228,31000794,2,0,2,0
6bba_7f87b3d8,4859,30,44.04,209.92000000000004,160.96,cont,4.875568716826789,31000793,5,0,2,3
6bba_7f87b3d8,4889,31,21.32,95.84000000000002,49.760000000000005,cont,4.8047817978551395,32000800,3,3,0,0
6bba_7f87b3d8,4935,31,36.28,219.03999999999996,101.12,cont,5.039308143237525,32000811,1,1,0,0
6bba_7f87b3d8,4949,31,43.0,234.72000000000006,130.72,cont,3.419186900785037,32000814,2,2,0,0
6bba_7f87b3d8,4980,32,18.84,83.35999999999999,26.080000000000002,cont,4.989199898280284,33000819,4,0,2,2
6bba_7f87b3d8,5011,32,32.47999999999999,161.92,107.99999999999999,cont,0.8456247690317628,33000824,1,0,1,0
6bba_7f87b3d8,5074,33,19.160000000000004,86.56000000000002,27.360000000000007,cont,5.181125903218339,34000835,2,2,0,0
6bba_7f87b3d8,5105,33,31.840000000000003,159.36,103.52,cont,2.8312331677203746,34000840,2,2,0,0
6bba_7f87b3d8,5217,34,35.68,146.4,145.92,cont,5.140062773206181,35000863,3,0,2,1
6bba_7f87b3d8,5222,34,37.52000000000001,232.47999999999996,91.67999999999999,cont,5.707475635077198,35000860,4,0,2,2
6bba_7f87b3d8,5224,34,39.239999999999995,196.00000000000003,139.2,cont,1.3884957012897092,35000864,2,0,2,0
6bba_7f87b3d8,5234,34,42.88,207.84000000000003,138.08,cont,6.426232979942465,35000864,5,0,2,3
6bba_7f87b3d8,5266,35,19.360000000000003,94.72,31.839999999999996,cont,2.8162708055156873,36000874,1,1,0,0
6bba_7f87b3d8,5308,35,35.68,145.44,145.11999999999998,cont,3.1047282489776764,36000883,1,1,0,0
6bba_7f87b3d8,5314,35,37.36,230.4,90.88000000000002,cont,4.168528854704015,36000884,3,3,0,0
6bba_7f87b3d8,5379,36,30.800000000000004,87.19999999999999,81.92,cont,0.9515620381772237,37000895,1,0,1,0
6bba_7f87b3d8,5391,36,32.04,132.16000000000003,124.79999999999998,cont,4.113593699552738,37000898,2,0,1,1
6bba_7f87b3d8,5392,36,34.32,170.56,33.279999999999994,cont,1.6224792949372324,37000897,1,0,1,0
6bba_7f87b3d8,5393,36,33.84000000000001,206.88,89.92,cont,4.103535251767661,37000899,5,0,1,4
6bba_7f87b3d8,5418,36,48.39999999999999,193.0666666666666,148.53333333333336,cont,1.203914333423374,37000904,2,0,0,2
6bba_7f87b3d8,5480,37,35.400000000000006,248.32000000000002,60.31999999999999,cont,1.2217795678844814,38000916,1,0,1,0
6bba_7f87b3d8,5484,37,36.00000000000001,230.07999999999998,83.36000000000003,cont,2.279927698963272,38000920,1,0,1,0
6bba_7f87b3d8,5499,37,42.999999999999986,224.00000000000003,140.0,cont,1.723572779142233,38000921,1,1,0,0
6bba_7f87b3d8,5565,38,35.32000000000001,188.16000000000003,117.27999999999997,daughter,1.8466464537100569,39000938,1,0,0,1
6bba_7f87b3d8,5704,40,24.480000000000004,176.64000000000001,46.56000000000001,cont,1.9024301892579423,41000960,3,3,0,0
6bba_7f87b3d8,5727,40,30.720000000000006,198.08,74.24,cont,6.046419524189496,41000967,2,2,0,0
6bba_7f87b3d8,5732,40,34.0,245.91999999999996,50.56,cont,1.4937630752565776,41000968,3,3,0,0
6bba_7f87b3d8,5780,41,21.32,166.08,22.239999999999995,cont,3.0385327462444773,42000976,5,0,3,2
6bba_7f87b3d8,5798,41,28.32,148.64000000000001,96.16000000000001,cont,3.1550118858730105,42000995,2,0,0,2
6bba_7f87b3d8,6300,48,14.479999999999999,87.2,9.919999999999998,cont,1.2303943727520912,49001129,3,0,3,0
6bba_7f87b3d8,6316,48,21.56,32.0,55.99999999999999,cont,2.5101385046447167,49001137,2,0,1,1
6bba_7f87b3d8,6323,48,24.680000000000007,164.8,62.88000000000001,cont,4.324851334150099,49001145,4,0,2,2
6bba_7f87b3d8,6334,48,31.266666666666662,200.53333333333333,87.99999999999997,cont,2.8789978242823104,49001147,3,0,0,3
6bba_7f87b3d8,6353,48,38.133333333333326,132.53333333333333,127.99999999999997,cont,1.7316397976741422,49001149,2,0,0,2
6bba_7f87b3d8,6692,53,21.36,152.16000000000003,49.44000000000001,cont,4.788458944430864,54001245,2,2,0,0
6bba_7f87b3d8,6734,53,44.48,114.72,234.56,cont,1.1773122196342012,54001253,2,2,0,0
6bba_7f87b3d8,6763,54,20.0,147.35999999999999,44.00000000000001,cont,3.4807257235381237,55001263,2,0,2,0
6bba_7f87b3d8,7170,59,34.480000000000004,77.44000000000001,208.64000000000004,cont,1.5052974166257052,60001362,1,0,1,0
6bba_7f87b3d8,7227,60,21.52,204.8,41.28000000000001,cont,2.8217506201824416,61001375,1,0,1,0
6bba_7f87b3d8,7291,61,15.96,134.71999999999997,42.39999999999999,cont,1.792294705817097,62001385,1,1,0,0
6bba_7f87b3d8,7356,62,10.0,98.56,17.28,cont,1.7479139352382311,63001399,1,0,1,0
6bba_7f87b3d8,7362,62,12.733333333333333,120.0,32.0,cont,2.2128923067133455,63001404,5,0,0,5
6bba_7f87b3d8,7463,63,23.480000000000004,220.64000000000001,45.92000000000001,cont,1.1789932145691118,64001423,1,0,1,0
6bba_7f87b3d8,8185,72,8.520000000000001,144.32,17.919999999999998,cont,4.0264929296473415,73001546,2,0,2,0
6bba_7f87b3d8,8342,73,45.67999999999999,247.52000000000004,209.44,daughter,2.587987876710402,74001569,1,1,0,0
6bba_7f87b3d8,8630,77,3.2,137.92000000000002,6.72,cont,1.9887503928346613,78001610,4,4,0,0
6bba_7f87b3d8,9765,88,11.4,54.56000000000001,185.43999999999997,cont,3.2456095344326323,89001765,1,1,0,0
6bba_7f87b3d8,9862,89,9.6,53.440000000000005,183.20000000000002,cont,4.823841991867064,90001778,2,0,1,1
6bba_7f87b3d8,9869,89,11.8,37.44000000000001,181.44,cont,2.80598495407584,90001778,3,0,1,2
6bba_7f87b3d8,10100,91,9.84,236.0,103.36,cont,2.0401355653485376,92001805,1,1,0,0
6bba_80d12824,998,4,13.839999999999998,199.36000000000004,60.0,cont,1.9338182761055944,5000026,5,1,0,4
6bba_80d12824,1216,5,13.800000000000002,196.16000000000003,58.55999999999998,cont,1.321355837766634,6000031,4,3,1,0
6bba_80d12824,1340,5,58.999999999999986,127.36000000000001,138.07999999999998,cont,3.510150459738183,6000036,1,1,0,0
6bba_80d12824,1531,6,58.16,125.28,133.92000000000004,cont,5.937568552235848,7000043,1,0,1,0
6bba_80d12824,1549,6,61.99999999999999,135.99999999999994,127.99999999999997,cont,2.1877232028983817,7000043,2,0,0,2
6bba_80d12824,1928,8,61.68,134.72000000000003,126.08000000000001,cont,1.1720294898167032,9000062,1,1,0,0
6bba_80d12824,2102,9,57.68000000000002,185.92000000000004,169.28,cont,5.842140553769645,10000072,1,1,0,0
6bba_80d12824,2114,9,61.160000000000004,133.27999999999997,123.35999999999999,cont,1.8940837164708468,10000071,1,0,1,0
6bba_80d12824,2115,9,60.84000000000001,176.64000000000004,172.0,cont,1.6096525246151958,10000072,1,1,0,0
6bba_80d12824,2282,10,50.199999999999996,243.35999999999999,236.64000000000004,cont,4.652213518853157,11000078,1,1,0,0
6bba_80d12824,2297,10,60.000000000000014,176.00000000000003,169.28,cont,1.9600625787203905,11000081,1,0,1,0
6bba_80d12824,2304,10,61.99999999999999,119.46666666666667,117.06666666666666,cont,6.364295206244142,11000080,2,0,0,2
6bba_80d12824,2451,11,48.2,241.92000000000007,233.27999999999997,cont,2.9632533324877937,12000088,1,0,1,0
6bba_80d12824,2457,11,52.99999999999997,251.46666666666664,239.99999999999994,cont,6.912831830994475,12000088,1,0,0,1
6bba_80d12824,2466,11,58.52000000000001,13.280000000000001,13.440000000000001,cont,2.843703571049545,12000090,1,1,0,0
6bba_80d12824,2471,11,59.68000000000001,130.72,116.0,cont,1.3299212852646511,12000091,1,1,0,0
6bba_80d12824,2644,12,57.00000000000001,11.36,8.639999999999997,cont,1.952503601149036,13000100,1,0,1,0
6bba_80d12824,2650,12,57.84,176.00000000000003,164.64,cont,0.7334134662657851,13000099,1,1,0,0
6bba_80d12824,2651,12,58.52000000000001,129.92000000000002,112.00000000000001,cont,1.175516508816438,13000101,1,0,1,0
6bba_80d12824,2831,13,56.84,176.00000000000003,163.36,cont,2.0401355653485242,14000109,1,0,1,0
6bba_80d12824,3904,20,50.64,57.44000000000001,37.28,cont,3.028827382420457,21000163,1,1,0,0
6bba_80d12824,4034,21,43.08,66.56,26.079999999999995,cont,6.481978141354695,22000171,4,1,2,1
6bba_80d12824,4056,21,60.99999999999999,52.53333333333333,60.53333333333332,cont,6.8779403434498825,22000172,2,0,0,2
6bba_80d12824,4165,22,42.2,68.00000000000001,26.080000000000002,cont,4.61637303518682,23000178,1,0,1,0
6bba_80d12824,4417,24,55.0,47.36,68.0,cont,3.2532889608056643,25000193,2,2,0,0
6bba_80d12824,4425,24,59.68000000000001,197.92,220.00000000000003,cont,1.021684424125177,25000194,1,1,0,0
6bba_80d12824,4528,25,53.68,50.080000000000005,67.36000000000001,cont,2.297177616119393,26000202,2,0,2,0
6bba_80d12824,4534,25,58.0,200.00000000000003,217.28000000000003,cont,1.6289763848810093,26000204,1,0,1,0
6bba_80d12824,4547,25,61.99999999999999,192.5333333333333,223.99999999999997,cont,6.407312242556232,26000204,4,1,0,3
6bba_80d12824,4643,26,56.52,202.08000000000004,215.36000000000004,cont,2.5416579234822243,27000210,2,2,0,0
6bba_80d12824,4648,26,61.00000000000001,194.71999999999997,223.68000000000004,cont,6.907454306761669,27000210,3,2,1,0
6bba_80d12824,4744,27,54.64,204.00000000000003,213.28000000000003,cont,0.7699482531313225,28000219,2,0,2,0
6bba_80d12824,4753,27,61.52000000000001,44.00000000000001,55.99999999999999,cont,0.9687508064512668,28000220,1,0,1,0
6bba_80d12824,5027,30,58.52000000000001,47.360000000000014,51.35999999999999,cont,2.646106810013522,31000248,1,1,0,0
6bba_80d12824,5071,31,19.68,77.28,0.0,cont,2.164851322839515,32000255,1,0,1,0
6bba_80d12824,5115,31,56.68,48.0,48.0,cont,0.6598780663880228,32000260,1,0,1,0
6bba_80d12824,6370,47,22.66666666666666,163.99999999999994,132.5333333333333,cont,2.2907441324939737,48000415,1,0,0,1
6bba_80d12824,6486,48,48.11999999999999,124.0,138.4,cont,3.2273334581353885,49000429,1,0,1,0
6bba_80d12824,6595,50,16.2,164.0,119.35999999999999,cont,3.1154263351907416,51000446,1,1,0,0
6bba_80d12824,6596,50,16.36,236.64000000000001,226.08,cont,1.4125353977865456,51000445,1,1,0,0
6bba_80d12824,6634,50,54.68,143.36,178.72,cont,1.0935899882954232,51000449,1,1,0,0
6bba_80d12824,6665,51,13.440000000000001,164.0,116.0,cont,2.5673457232129806,52000454,1,0,1,0
6bba_80d12824,6666,51,14.640000000000002,235.99999999999994,221.28000000000006,cont,0.7827036476215133,52000455,1,0,1,0
6bba_80d12824,6709,51,53.0,143.36,174.72,cont,1.6355283932111977,52000459,1,0,1,0
6bba_80d12824,6795,52,57.36,164.64,175.99999999999997,cont,3.982974116473771,53000473,1,1,0,0
6bba_80d12824,6873,53,54.84,30.080000000000002,60.000000000000014,cont,1.9216956217101513,54000485,1,1,0,0
6bba_80d12824,6875,53,56.52,167.36000000000004,175.36,cont,2.8768007947197187,54000483,1,0,1,0
6bba_80d12824,6948,54,53.52,28.639999999999997,55.99999999999999,cont,1.2959311324294955,55000492,1,0,1,0
6bba_80d12824,7090,56,46.16,141.43999999999997,154.08,cont,1.4334121223500331,57000507,1,1,0,0
6bba_80d12824,7161,57,45.00000000000001,140.64000000000001,152.0,cont,3.3532141763537724,58000515,1,0,1,0
6bba_80d12824,7192,58,5.200000000000001,164.0,102.72000000000001,cont,1.7855785126955344,59000518,2,2,0,0
6bba_80d12824,7259,59,4.319999999999999,164.0,101.28000000000002,cont,2.5212648735307437,60000526,2,0,2,0
6bba_80d12824,7560,63,25.279999999999994,111.99999999999999,85.28,cont,2.08310802948382,64000558,1,1,0,0
6bba_80d12824,7631,64,23.959999999999997,111.36,83.99999999999999,cont,0.48668682178584344,65000565,1,0,1,0
6bba_80d12824,7837,67,21.84,107.35999999999999,80.64,cont,0.39571335901129523,68000590,1,1,0,0
6bba_80d12824,7846,67,29.840000000000003,101.43999999999998,53.440000000000005,cont,3.8488151585910098,68000591,1,1,0,0
6bba_80d12824,7905,68,21.160000000000004,104.64000000000003,79.36,cont,1.4136566061105473,69000599,1,0,1,0
6bba_80d12824,7916,68,29.520000000000003,96.63999999999999,48.639999999999986,cont,2.6020811982910867,69000600,1,0,1,0
6bba_80d12824,8415,75,28.999999999999993,127.99999999999997,84.53333333333329,cont,0.7211493767667851,76000668,2,0,0,2
6bba_80d12824,8845,81,2.6800000000000006,244.00000000000003,244.00000000000003,cont,0.6598780663880162,82000735,1,1,0,0
6bba_80d12824,8871,81,25.000000000000004,116.0,69.28,cont,1.6788695973779433,82000740,1,1,0,0
6bba_80d12824,8879,81,31.000000000000007,149.92000000000002,113.28,cont,3.2814540412749804,82000741,2,2,0,0
6bba_80d12824,8925,82,2.32,244.00000000000003,244.00000000000003,cont,1.1773122196341925,83000749,1,0,1,0
6bba_80d12824,8950,82,24.320000000000007,113.92,67.36,cont,1.1151100001793441,83000753,1,0,1,0
6bba_80d12824,8958,82,30.480000000000004,148.00000000000006,110.71999999999998,cont,0.8526218754524111,83000754,2,0,2,0
6bba_80d12824,9047,83,48.0,203.35999999999999,225.28,cont,1.7411023806772528,84000769,1,1,0,0
6bba_80d12824,9119,84,47.52000000000001,202.56,222.72000000000003,cont,1.0467068894872138,85000782,1,0,1,0
6bba_80d12824,9225,86,11.840000000000003,115.52000000000001,94.72,cont,2.1403785442066003,87000803,1,1,0,0
6bba_80d12824,9300,87,10.64,110.56,89.27999999999999,cont,1.265623611110346,88000813,1,0,1,0
6bba_825bd1c6,383,13,4.840000000000001,4.640000000000001,213.92000000000002,cont,1.4729349951712067,14000022,1,0,1,0
6bba_825bd1c6,481,16,5.319999999999999,11.36,224.64000000000001,cont,0.7726870728179642,17000032,1,1,0,0
6bba_825bd1c6,524,17,5.4799999999999995,14.080000000000002,228.8,cont,1.5614381391845118,18000035,1,0,1,0
6bba_825bd1c6,1197,29,3.999999999999999,28.000000000000004,148.53333333333336,cont,6.189830509481758,30000098,3,0,0,3
6bba_825bd1c6,1757,37,5.6400000000000015,22.08,198.08,cont,1.6212581919608022,38000186,2,0,2,0
6bba_825bd1c6,1791,37,22.839999999999993,18.720000000000006,31.359999999999996,cont,0.9965636269200292,38000195,1,0,1,0
6bba_825bd1c6,2156,43,6.32,147.35999999999999,120.00000000000003,cont,1.1146362915767634,44000265,1,1,0,0
6bba_825bd1c6,2192,43,23.840000000000003,29.28,35.36,cont,2.10536294377002,44000271,1,1,0,0
6bba_825bd1c6,2219,44,4.360000000000001,70.56,191.36,cont,1.614648278263716,45000272,4,0,2,2
6bba_825bd1c6,2255,44,17.839999999999996,29.440000000000005,31.360000000000007,cont,6.353272040846348,45000283,1,0,1,0
6bba_825bd1c6,2259,44,24.000000000000007,34.080000000000005,36.64000000000001,cont,2.0561227164252522,45000284,1,0,1,0
6bba_825bd1c6,2292,45,6.0,180.64,13.280000000000001,cont,1.4629513192516068,46000293,1,1,0,0
6bba_825bd1c6,2297,45,7.32,110.72000000000001,245.28000000000003,cont,1.1017691738744493,46000291,1,1,0,0
6bba_825bd1c6,2367,46,7.640000000000001,114.07999999999998,246.72000000000006,cont,1.6610806911465807,47000305,1,0,1,0
6bba_825bd1c6,2466,47,19.479999999999997,17.28,164.00000000000003,cont,1.748518122582656,48000328,1,0,1,0
6bba_825bd1c6,2470,47,24.680000000000003,9.280000000000003,92.64,cont,0.650811992821271,48000333,1,1,0,0
6bba_825bd1c6,2509,48,9.32,120.47999999999999,248.00000000000006,cont,0.8072832526443292,49000341,3,3,0,0
6bba_825bd1c6,2510,48,9.000000000000002,166.56000000000003,127.99999999999997,cont,1.1165299201096335,49000342,1,1,0,0
6bba_825bd1c6,2531,48,13.839999999999998,201.28,32.0,cont,0.5640880361255707,49000346,2,2,0,0
6bba_825bd1c6,2538,48,20.319999999999997,20.479999999999997,165.28000000000003,cont,1.4996734019445728,49000344,2,2,0,0
6bba_825bd1c6,2576,49,5.360000000000001,101.28,196.64000000000001,cont,1.2873446508219903,50000352,2,0,1,1
6bba_825bd1c6,2596,49,10.360000000000001,126.08,248.00000000000006,cont,3.2702494170934435,50000361,3,0,3,0
6bba_825bd1c6,2613,49,14.16,204.79999999999995,34.080000000000005,cont,1.8940140079999506,50000358,2,0,2,0
6bba_825bd1c6,2722,50,25.84,20.0,96.64000000000001,cont,2.493038118942426,51000387,1,0,1,0
6bba_825bd1c6,2896,52,22.999999999999996,35.36,172.0,cont,3.278555798671109,53000426,1,1,0,0
6bba_825bd1c6,3044,54,12.480000000000004,81.44000000000001,162.72,cont,6.0201589731916645,55000455,1,1,0,0
6bba_825bd1c6,3081,54,24.320000000000007,7.360000000000001,216.0,cont,0.9377249730064833,55000462,2,2,0,0
6bba_825bd1c6,3119,55,7.68,126.08,212.8,cont,2.4991739260203554,56000471,1,0,1,0
6bba_825bd1c6,3139,55,13.200000000000001,82.56,162.56,cont,1.8107633921912636,56000474,2,0,1,1
6bba_825bd1c6,3181,55,29.999999999999993,58.93333333333335,50.933333333333316,cont,6.166560387597675,56000485,1,0,0,1
6bba_825bd1c6,3346,57,23.320000000000007,19.359999999999996,126.08000000000001,cont,1.989148687252915,58000529,2,1,1,0
6bba_825bd1c6,3351,57,26.320000000000004,11.36,221.43999999999997,cont,2.4321315882369565,58000530,1,1,0,0
6bba_825bd1c6,3810,62,23.0,27.52,150.56,cont,2.4571912954631756,63000615,1,0,1,0
6bba_825bd1c6,3907,63,25.999999999999996,51.99999999999999,191.99999999999994,cont,5.759587794495364,64000633,1,0,0,1
6bba_825bd1c6,4215,67,10.96,188.64000000000004,174.07999999999998,cont,3.6707019934884566,68000698,1,1,0,0
6bba_825bd1c6,4216,67,11.32,165.44,176.15999999999997,cont,6.683986693957438,68000698,2,2,0,0
6bba_825bd1c6,4226,67,13.480000000000002,233.76,160.64,cont,1.721349851860458,68000695,1,1,0,0
6bba_825bd1c6,4246,67,19.119999999999997,175.83999999999997,250.72000000000003,cont,2.5698130403786026,68000701,1,1,0,0
6bba_825bd1c6,4276,67,28.640000000000004,34.56,220.0,cont,1.9579058487322663,68000709,3,3,0,0
6bba_825bd1c6,4280,67,29.0,109.76,74.55999999999999,cont,2.707320227180374,68000711,1,1,0,0
6bba_825bd1c6,4310,68,11.719999999999999,190.56,171.83999999999997,cont,5.633760400478521,69000715,2,0,1,1
6bba_825bd1c6,4377,68,28.266666666666662,78.39999999999999,188.53333333333325,cont,1.231026135881064,69000725,3,0,0,3
6bba_825bd1c6,4380,68,28.733333333333327,67.99999999999999,191.46666666666664,cont,4.600182099354575,69000725,7,0,5,2
6bba_825bd1c6,4382,68,29.519999999999996,37.44,220.64000000000001,cont,2.394105391372739,69000726,3,0,3,0
6bba_825bd1c6,4383,68,29.999999999999993,80.53333333333333,124.53333333333335,cont,1.7411445090482673,69000720,1,0,0,1
6bba_825bd1c6,4470,69,26.68,9.920000000000003,196.64000000000001,cont,0.6568699452707534,70000737,1,0,1,0
6bba_825bd1c6,4553,70,22.16,187.19999999999996,248.00000000000006,cont,1.3381359469425793,71000747,1,1,0,0
6bba_825bd1c6,4582,70,28.48,85.28,190.72000000000003,cont,1.581935622109826,71000756,1,1,0,0
6bba_825bd1c6,4585,70,29.0,127.83999999999999,80.00000000000003,cont,1.7400404090135424,71000752,2,2,0,0
6bba_825bd1c6,4591,70,31.000000000000007,41.92,221.44,cont,1.9651753261477867,71000757,1,1,0,0
6bba_825bd1c6,4656,71,22.68,191.36,248.63999999999996,cont,2.2302200003587167,72000764,2,1,1,0
6bba_825bd1c6,4710,72,11.840000000000003,221.60000000000002,172.64,cont,0.40200279849771015,73000777,1,1,0,0
6bba_825bd1c6,4718,72,14.866666666666667,182.39999999999998,170.93333333333328,cont,1.9276789620047068,73000778,1,0,0,1
6bba_825bd1c6,4754,72,22.999999999999996,194.72,248.64000000000001,cont,1.7411023806772594,73000783,1,0,1,0
6bba_825bd1c6,4814,73,12.159999999999997,216.0,170.72000000000003,cont,3.2875641628719556,74000794,1,0,0,1
6bba_825bd1c6,4817,73,11.999999999999996,226.9333333333333,171.4666666666667,cont,2.3626230861528077,74000794,1,0,1,0
6bba_825bd1c6,4990,74,29.840000000000003,100.64000000000001,191.36,cont,1.2890870073427916,75000820,1,1,0,0
6bba_825bd1c6,5291,77,33.480000000000004,27.839999999999996,129.44000000000003,cont,4.444783459292482,78000870,1,1,0,0
6bba_825bd1c6,5302,77,37.32000000000001,156.00000000000003,20.000000000000007,cont,0.7749052361418298,78000872,1,1,0,0
6bba_825bd1c6,5388,78,32.36000000000001,152.0,80.64,cont,3.187527695958106,79000881,1,0,1,0
6bba_825bd1c6,5392,78,34.519999999999996,33.440000000000005,123.83999999999997,cont,2.208206993241341,79000886,1,0,1,0
6bba_825bd1c6,5807,82,36.68000000000001,1.2799999999999998,236.64000000000001,cont,1.0314590697162844,83000948,1,1,0,0
6bba_825bd1c6,5828,82,48.67999999999999,6.720000000000001,40.63999999999999,cont,2.211970628760709,83000951,1,1,0,0
6bba_825bd1c6,6050,84,52.0,26.720000000000002,8.640000000000002,cont,0.7587201394453678,85000984,1,0,1,0
6bba_825bd1c6,6347,87,35.99999999999999,176.00000000000003,244.64000000000001,cont,0.8255575767322201,88001023,1,1,0,0
6bba_825bd1c6,6349,87,37.480000000000004,52.00000000000001,101.28,cont,6.866523979605687,88001027,1,1,0,0
6bba_825bd1c6,6352,87,37.839999999999996,64.8,113.92,cont,0.5593872205369042,88001027,1,1,0,0
6bba_825bd1c6,6373,87,49.32,15.359999999999996,48.0,cont,1.2903154895218498,88001031,1,1,0,0
6bba_825bd1c6,6461,88,38.0,69.44,114.72,cont,1.1223092822390826,89001038,1,0,1,0
6bba_825bd1c6,6638,90,30.84,74.55999999999999,116.16,cont,2.400274072892514,91001072,2,2,0,0
6bba_825bd1c6,6673,90,37.160000000000004,61.280000000000015,157.44000000000003,cont,1.941586014061695,91001080,2,2,0,0
6bba_825bd1c6,6689,90,40.48,97.92000000000002,206.08,cont,1.6198731624729195,91001084,2,2,0,0
6bba_825bd1c6,6786,91,37.480000000000004,62.71999999999999,152.64,cont,1.6499152440352722,92001100,2,0,2,0
6bba_825bd1c6,6799,91,38.52,188.00000000000003,240.0,cont,2.902524763029581,92001102,2,0,2,0
6bba_825bd1c6,6805,91,41.0,101.44,202.56000000000003,cont,1.0645920580203485,92001101,2,0,2,0
6bba_825bd1c6,6889,92,35.0,243.36000000000004,221.92000000000002,cont,2.122476442743228,93001121,1,1,0,0
6bba_825bd1c6,7001,93,35.68,244.00000000000003,219.36,cont,2.6900990339576643,94001140,1,0,1,0
6bba_825bd1c6,7332,96,38.0,199.03999999999996,87.36000000000001,cont,1.8299837772504919,97001203,1,1,0,0
6bba_825bd1c6,7338,96,39.480000000000004,193.92000000000002,232.0,cont,1.5465697931228313,97001215,1,1,0,0
6bba_825bd1c6,7436,97,36.04,165.44,112.64,cont,4.385031713112235,98001230,1,0,1,0
6bba_87289e13,1052,3,49.52,207.36,243.84000000000003,cont,2.671432785042522,4000019,2,1,1,0
6bba_87289e13,1281,4,40.68,202.56,199.84000000000003,cont,2.6483511969714355,5000022,3,3,0,0
6bba_87289e13,1315,4,48.04,205.28000000000003,243.68000000000004,cont,4.841136553021426,5000024,1,0,1,0
6bba_87289e13,1531,5,38.0,191.99999999999994,183.4666666666667,cont,1.6857063512895,6000026,3,0,0,3
6bba_87289e13,1567,5,45.51999999999998,193.44000000000005,218.23999999999995,cont,5.221183414705933,6000028,7,0,3,4
6bba_87289e13,3362,12,39.480000000000004,250.72000000000003,220.64000000000001,cont,1.0159824371021322,13000067,2,1,1,0
6bba_87289e13,3800,14,23.16,167.36,117.28,cont,1.4036275547665793,15000083,1,0,1,0
6bba_87289e13,3913,14,48.52,207.52000000000004,235.52000000000004,cont,3.748681643258045,15000090,9,2,3,4
6bba_87289e13,3920,14,52.00000000000001,211.20000000000002,250.08000000000004,cont,5.2788494544266165,15000090,5,2,1,2
6bba_87289e13,4416,16,45.68,207.36,226.72,cont,1.0935899882954232,17000117,1,1,0,0
6bba_87289e13,4544,17,18.16,152.64,104.0,cont,3.232157271080724,18000121,1,0,1,0
6bba_87289e13,4876,18,38.0,239.20000000000002,215.36000000000007,cont,0.5089649423094069,19000145,1,1,0,0
6bba_87289e13,5265,20,14.680000000000003,141.92000000000002,92.16,cont,1.6447055207544001,21000168,4,4,0,0
6bba_87289e13,5300,20,23.960000000000004,142.72,127.52,cont,1.135641338627651,21000171,1,1,0,0
6bba_87289e13,5311,20,27.440000000000005,228.64000000000007,170.71999999999997,cont,0.7386156815286333,21000172,5,5,0,0
6bba_87289e13,5364,20,40.12,198.72,196.80000000000004,cont,2.124528256460696,21000176,1,1,0,0
6bba_87289e13,5489,21,10.519999999999998,126.72000000000001,66.72,cont,1.8296229973685791,22000195,1,0,1,0
6bba_87289e13,5523,21,18.479999999999993,130.23999999999995,100.16000000000001,cont,3.269482231562071,22000183,1,0,1,0
6bba_87289e13,5529,21,19.04,223.36000000000004,132.16000000000003,cont,2.514185379998851,22000194,8,0,5,3
6bba_87289e13,5569,21,27.880000000000003,227.36,166.88,cont,4.079008955922987,22000188,9,1,5,3
6bba_87289e13,5584,21,32.52,188.8,164.16000000000003,cont,2.439936282467217,22000196,4,0,2,2
6bba_87289e13,5607,21,38.07999999999999,87.36000000000001,140.79999999999998,cont,5.049149835615905,22000190,6,0,3,3
6bba_87289e13,5823,22,28.040000000000006,226.08,160.00000000000003,cont,1.6547894503531113,23000203,1,0,1,0
6bba_87289e13,7022,27,7.040000000000001,103.36,26.719999999999995,cont,2.056828945853301,28000288,2,2,0,0
6bba_87289e13,7055,27,13.719999999999999,206.72,92.64,cont,2.5584849642903893,28000287,1,1,0,0
6bba_87289e13,7059,27,15.359999999999996,114.07999999999998,62.71999999999999,cont,2.6934834554903135,28000277,2,2,0,0
6bba_87289e13,7137,27,35.720000000000006,212.0,173.28000000000003,cont,2.1821638486832224,28000282,1,1,0,0
6bba_87289e13,7260,28,5.920000000000001,97.28,19.36,cont,3.4545680808170514,29000303,5,3,2,0
6bba_87289e13,7291,28,11.280000000000003,201.44000000000003,87.84,cont,2.175376763344683,29000302,4,3,1,0
6bba_87289e13,7295,28,14.240000000000002,108.64000000000001,54.720000000000006,cont,3.6861361884770383,29000291,6,4,2,0
6bba_87289e13,7336,28,24.080000000000002,207.36000000000007,115.36000000000004,cont,3.555595062714534,29000293,2,1,1,0
6bba_87289e13,7353,28,28.240000000000006,172.64,114.07999999999998,cont,2.4026381984185763,29000294,3,2,1,0
6bba_87289e13,7367,28,30.640000000000008,59.52000000000001,84.96000000000001,cont,4.683440575581158,29000295,10,4,3,3
6bba_87289e13,7376,28,33.64000000000001,211.84,168.48000000000002,cont,2.612967661491443,29000296,3,2,1,0
6bba_87289e13,7506,29,5.84,91.2,11.200000000000001,cont,2.269538086263372,30000319,3,0,3,0
6bba_87289e13,7542,29,12.520000000000001,194.07999999999998,68.64,cont,1.779282246440963,30000318,2,0,2,0
6bba_87289e13,7547,29,14.52,103.2,45.919999999999995,cont,2.8429606047217773,30000305,4,0,4,0
6bba_87289e13,7584,29,23.840000000000003,204.00000000000003,106.55999999999999,cont,1.8986788169671989,30000306,1,0,1,0
6bba_87289e13,7604,29,27.84,169.28,103.84,cont,2.2293318085246954,30000308,2,0,2,0
6bba_87289e13,7619,29,31.72,56.80000000000001,80.0,cont,3.832864871737586,30000309,4,0,4,0
6bba_87289e13,7680,29,50.839999999999996,135.20000000000002,240.64,cont,2.7366659688387363,30000315,3,0,0,3
6bba_87289e13,7944,30,59.67999999999999,168.32,243.68,cont,1.111314677532878,31000331,2,2,0,0
6bba_87289e13,8173,31,59.0,169.28,238.72000000000003,cont,0.5322960290101663,32000346,2,0,2,0
6bba_87289e13,8322,32,31.680000000000003,48.0,68.64000000000001,cont,2.1499800144419905,33000358,2,0,2,0
6bba_87289e13,9077,35,59.48000000000001,161.92,241.91999999999996,cont,0.9245378913273246,36000417,1,1,0,0
6bba_87289e13,9161,36,16.32,187.35999999999999,68.0,cont,0.5813776741499472,37000436,1,0,1,0
6bba_87289e13,9181,36,20.640000000000004,155.36,64.64,cont,1.2437355526799097,37000424,1,0,1,0
6bba_87289e13,9577,38,16.919999999999998,185.92,66.4,cont,3.357621228265631,39000455,3,3,0,0
6bba_87289e13,9599,38,18.4,154.56,54.72,cont,3.2961874283025843,39000456,3,3,0,0
6bba_87289e13,9617,38,26.239999999999995,195.2,102.24000000000002,cont,1.8625222314109486,39000457,2,2,0,0
6bba_87289e13,9622,38,28.68,41.12,51.36,cont,1.7718462299251587,39000459,4,4,0,0
6bba_87289e13,9701,38,55.47999999999999,213.12000000000006,240.48000000000002,cont,3.1773632936760934,39000465,2,2,0,0
6bba_87289e13,9771,39,12.360000000000003,167.51999999999998,40.0,cont,2.096187014557627,40000470,6,1,3,2
6bba_87289e13,9788,39,15.719999999999999,130.24,39.519999999999996,cont,4.384158449748365,40000471,7,0,3,4
6bba_87289e13,9808,39,23.04,193.59999999999997,88.64,cont,5.39739781051943,40000473,4,1,1,2
6bba_87289e13,9815,39,26.52,14.240000000000002,40.64000000000001,cont,3.0894664790219037,40000474,9,0,6,3
6bba_87289e13,9882,39,51.99999999999999,199.4666666666667,214.9333333333333,cont,3.5359052402539923,40000480,6,1,0,5
6bba_87289e13,9984,40,21.680000000000003,78.56,36.63999999999999,cont,2.2517246862127664,41000488,1,0,1,0
6bba_87289e13,9985,40,20.39999999999999,181.12000000000003,72.79999999999998,cont,5.338044615072098,41000489,5,0,1,4
6bba_87289e13,10135,41,9.56,169.92000000000002,30.080000000000002,cont,4.305299423385091,42000502,2,2,0,0
6bba_87289e13,10165,41,21.68,75.35999999999999,33.44,cont,2.475179602069317,42000504,2,2,0,0
6bba_87289e13,10303,42,10.960000000000003,152.8,26.560000000000002,cont,2.7269998395672874,43000517,5,0,1,4
6bba_87289e13,10329,42,19.133333333333326,163.99999999999997,60.53333333333332,cont,1.5822323474726907,43000519,5,0,0,5
6bba_87289e13,11344,49,15.159999999999997,28.480000000000004,2.5600000000000005,cont,2.2863467038050076,50000637,1,0,1,0
6bba_87289e13,11425,49,57.24,212.32,229.92,cont,1.1073871274310572,50000649,1,0,1,0
6bba_87289e13,11520,50,33.36,90.71999999999998,82.08000000000001,cont,2.347773546362595,51000655,1,1,0,0
6bba_87289e13,11529,50,38.64,235.99999999999994,162.56000000000003,cont,2.2990735226390675,51000656,3,3,0,0
6bba_87289e13,11599,51,12.44,121.91999999999999,21.119999999999997,cont,3.348012703455585,52000665,1,0,1,0
6bba_87289e13,11637,51,32.16,85.91999999999999,74.88,cont,1.686324127355111,52000668,1,0,1,0
6bba_87289e13,11645,51,34.73333333333332,232.0,143.46666666666664,cont,2.4405074039815946,52000669,3,0,0,3
6bba_87289e13,11671,51,52.2,213.28000000000003,208.79999999999995,cont,1.7254102772963882,52000680,1,0,1,0
6bba_87289e13,11760,52,35.39999999999999,232.0,144.64000000000001,cont,1.3842100635380545,53000685,2,2,0,0
6bba_87289e13,11850,53,28.959999999999997,62.879999999999995,52.00000000000001,cont,2.6164518555670013,54000699,2,0,1,1
6bba_87289e13,11861,53,32.84000000000001,224.8,125.6,cont,2.563228250175936,54000700,3,0,1,2
6bba_87289e13,11869,53,36.56,104.00000000000003,92.16000000000003,cont,4.638428646912648,54000702,1,0,1,0
6bba_87289e13,12224,57,25.680000000000003,49.280000000000015,38.720000000000006,cont,0.7441364542205968,58000759,1,0,1,0
6bba_87289e13,12231,57,29.0,218.56,108.64000000000001,cont,1.6473525161604001,58000760,1,0,1,0
6bba_87289e13,12649,62,36.36,234.56000000000006,190.72000000000003,cont,2.8305335738160697,63000825,1,1,0,0
6bba_87289e13,13596,75,53.480000000000004,163.36,245.28,cont,1.1151100001793492,76001010,1,0,1,0
6bba_87289e13,13873,79,57.48000000000001,44.64,221.28000000000006,cont,1.051112505871735,80001059,2,1,1,0
6bba_8b7818bf,196,3,7.000000000000001,86.56,144.8,parent,2.120049011815526,4000023,1,1,0,0
6bba_8b7818bf,261,4,7.3199999999999985,87.36,139.2,daughter,1.2354275575686342,5000029,1,0,1,0
6bba_8b7818bf,264,4,8.8,88.00000000000001,172.48000000000002,daughter,1.4888050997360283,5000030,1,0,0,1
6bba_8b7818bf,1439,20,60.68000000000001,104.0,169.44,cont,1.2316813965064155,21000134,1,1,0,0
6bba_8b7818bf,1511,21,61.00000000000001,104.0,170.08000000000004,cont,0.5520218519225396,22000141,1,0,1,0
6bba_8b7818bf,1959,27,58.31999999999999,79.36000000000001,8.640000000000002,cont,1.8642227770575097,28000188,1,1,0,0
6bba_8b7818bf,2028,28,58.16000000000001,80.64,9.28,cont,1.9337499999999879,29000195,1,0,1,0
6bba_8b7818bf,3757,51,43.519999999999996,98.72,37.28,cont,0.982016420432981,52000422,1,1,0,0
6bba_8b7818bf,3826,52,42.36,100.00000000000001,40.63999999999999,cont,1.4307384456985852,53000433,1,0,1,0
6bba_8b7818bf,6417,93,59.160000000000004,44.64000000000001,99.36000000000001,cont,1.4618679189653148,94000795,1,1,0,0
6bba_8b7818bf,6470,94,59.31999999999999,45.28,98.08,cont,1.6473525161604128,95000803,1,0,1,0
6bba_907271db,1932,17,30.0,221.28000000000003,205.28,cont,1.9683975875061361,18000214,1,1,0,0
6bba_907271db,2028,18,29.520000000000003,216.0,205.28,cont,2.5410864433348133,19000227,1,0,1,0
6bba_907271db,2356,21,38.16,248.00000000000006,153.44000000000003,cont,3.083306362494655,22000274,2,2,0,0
6bba_907271db,2452,22,37.84,245.28000000000003,153.28000000000003,cont,2.0135579517361686,23000288,2,0,2,0
6bba_907271db,2717,25,28.519999999999996,235.99999999999994,216.64000000000001,cont,2.5518192040189853,26000327,1,1,0,0
6bba_907271db,2803,26,28.0,232.0,217.28000000000003,cont,3.348998451029798,27000340,1,0,1,0
6bba_907271db,2814,26,33.86666666666666,232.5333333333333,219.99999999999997,cont,6.299306206286167,27000340,2,0,0,2
6bba_907271db,3096,29,35.84,224.64000000000004,154.71999999999997,cont,2.1163090659211368,30000382,1,1,0,0
6bba_907271db,3189,30,35.160000000000004,220.0,153.27999999999997,cont,1.9350468340585483,31000403,1,0,1,0
6bba_907271db,3289,31,37.84,235.35999999999999,145.2800000000001,cont,1.9062434426116663,32000419,1,0,1,0
6bba_907271db,3293,31,38.52,181.28000000000003,80.64,cont,2.5646701176759477,32000415,1,1,0,0
6bba_907271db,3375,32,37.480000000000004,178.72,80.00000000000003,cont,2.505768968799787,33000428,1,0,1,0
6bba_907271db,4047,40,33.84,148.00000000000003,84.00000000000001,cont,0.9448784644069438,41000518,1,1,0,0
6bba_907271db,4075,40,46.52,237.44,9.440000000000001,cont,2.2631297466119844,41000521,1,0,1,0
6bba_907271db,4077,40,49.39999999999999,251.46666666666664,19.46666666666666,cont,6.261113729911691,41000521,2,0,0,2
6bba_907271db,4122,41,33.160000000000004,143.99999999999997,85.28,cont,1.4287068016216535,42000531,1,0,1,0
6bba_907271db,4249,43,16.520000000000003,122.71999999999998,220.64000000000001,cont,1.4244494550527251,44000549,1,1,0,0
6bba_907271db,4306,43,49.35999999999999,243.36000000000004,15.200000000000003,cont,1.1260675490395937,44000557,1,1,0,0
6bba_907271db,4322,44,15.479999999999997,120.0,223.99999999999997,cont,1.24065317978071,45000560,1,0,1,0
6bba_907271db,4379,44,48.31999999999999,240.00000000000006,16.64,cont,1.3503153566852635,45000569,2,1,1,0
6bba_907271db,4454,45,47.480000000000004,236.64000000000004,19.36,cont,0.921534047119263,46000581,1,0,1,0
6bba_907271db,4577,47,33.32000000000001,100.80000000000003,13.920000000000005,cont,1.2109248789664806,48000607,1,1,0,0
6bba_907271db,4582,47,35.480000000000004,124.63999999999999,32.64,cont,2.4879487660520585,48000606,1,1,0,0
6bba_907271db,4650,48,32.36000000000001,97.92000000000002,20.640000000000004,cont,2.920753862447155,49000618,1,0,1,0
6bba_907271db,4657,48,34.68000000000001,120.64000000000001,40.800000000000004,cont,4.025935450302191,49000617,1,0,1,0
6bba_907271db,5635,62,27.680000000000007,90.72,88.00000000000001,cont,2.2442067780175585,63000765,1,1,0,0
6bba_907271db,5697,63,27.32,88.64,92.0,cont,1.3959875536694417,64000777,1,0,1,0
6bba_907271db,5725,63,43.68,207.36000000000004,61.27999999999999,cont,2.23424204149864,64000780,1,1,0,0
6bba_907271db,5784,64,43.31999999999999,204.64000000000001,64.0,cont,2.8601846531299517,65000791,1,0,1,0
6bba_907271db,5902,66,40.480000000000004,232.64,210.71999999999997,cont,2.5839032973391274,67000812,1,1,0,0
6bba_907271db,5957,67,40.48,229.92,208.64000000000001,cont,2.649746744973939,68000825,1,0,1,0
6bba_907271db,5990,68,23.32,52.0,89.28000000000003,cont,2.7624044101108702,69000832,1,1,0,0
6bba_907271db,6041,69,22.8,52.64,95.36000000000001,cont,1.9726857485418143,70000845,1,0,1,0
6bba_907271db,6080,69,51.32,245.28,100.00000000000001,cont,1.21310358687954,70000853,1,1,0,0
6bba_907271db,6135,70,51.68,244.00000000000006,98.72000000000001,cont,1.0622330547012686,71000866,1,0,1,0
6bba_907271db,6248,73,5.320000000000001,20.640000000000004,212.00000000000006,cont,1.3793369104391973,74000893,1,1,0,0
6bba_907271db,6261,73,23.519999999999996,60.0,116.63999999999999,cont,0.8575628621273129,74000898,1,1,0,0
6bba_907271db,6301,74,5.32,24.000000000000007,217.28000000000003,cont,1.2131035868795343,75000906,1,0,1,0
6bba_907271db,6313,74,23.520000000000003,62.56,121.92000000000002,cont,0.9755415162872317,75000910,1,0,1,0
6bba_907271db,6342,74,42.999999999999986,206.08,92.0,cont,1.8548504825187644,75000914,1,1,0,0
6bba_907271db,6398,75,42.84,205.92000000000002,96.0,cont,2.0529095206559798,76000925,1,0,1,0
6bba_907271db,6901,85,44.67999999999999,228.0,120.00000000000003,cont,2.1831317098379643,86001042,1,1,0,0
6bba_907271db,6911,85,56.63999999999999,226.72,28.16,cont,1.66995929965374,86001044,1,1,0,0
6bba_907271db,6944,86,44.36000000000001,230.71999999999997,124.63999999999999,cont,3.4308672373031275,87001052,1,0,1,0
6bba_907271db,6957,86,56.84000000000001,225.28000000000003,24.0,cont,1.9554091643438523,87001055,1,0,1,0
6bba_907271db,7101,90,30.680000000000003,166.08,208.0,cont,2.4511658578113367,91001092,1,1,0,0
6bba_907271db,7142,91,30.839999999999996,172.0,212.64,cont,0.8918274777107947,92001103,1,0,1,0
6bba_91951b3a,1470,7,11.680000000000003,185.92000000000002,230.07999999999993,cont,3.8679437770603635,8000098,1,1,0,0
6bba_91951b3a,1678,8,11.680000000000003,188.00000000000003,229.44,cont,2.4181943806278237,9000110,1,0,1,0
6bba_91951b3a,1692,8,18.479999999999993,248.00000000000006,111.99999999999999,cont,5.734408344589712,9000115,1,1,0,0
6bba_91951b3a,1890,9,18.839999999999996,248.00000000000006,111.36,cont,6.76158184617328,10000129,1,0,1,0
6bba_91951b3a,3137,15,50.52,89.27999999999999,121.92000000000002,cont,2.9410704365077764,16000211,1,1,0,0
6bba_91951b3a,3219,16,17.679999999999996,244.00000000000003,140.64,cont,2.298384283034499,17000216,1,1,0,0
6bba_91951b3a,3321,16,50.48000000000002,88.64,118.72000000000003,cont,1.2830298613048534,17000224,1,0,1,0
6bba_91951b3a,3331,16,52.86666666666666,84.53333333333329,120.53333333333333,cont,4.060874674869935,17000224,2,0,0,2
6bba_91951b3a,3404,17,18.68,244.00000000000003,136.0,cont,2.3294248887869244,18000228,1,0,1,0
6bba_91951b3a,4477,23,27.32,251.36000000000007,64.0,cont,2.772280480488937,24000307,1,1,0,0
6bba_91951b3a,5614,30,14.16,174.55999999999995,242.71999999999997,cont,3.0703900912587567,31000380,1,1,0,0
6bba_91951b3a,5640,30,25.0,222.07999999999998,19.200000000000003,cont,3.2804882449111137,31000383,1,1,0,0
6bba_91951b3a,5776,31,14.16,179.35999999999999,239.36000000000007,cont,2.9971449956583673,32000391,1,0,1,0
6bba_91951b3a,5796,31,23.96,227.36,13.280000000000001,cont,0.8476520881234164,32000393,1,0,1,0
6bba_91951b3a,6839,38,10.960000000000003,172.0,158.72,cont,1.741860535462004,39000484,1,1,0,0
6bba_91951b3a,6876,38,28.16,102.72000000000003,238.72000000000003,cont,0.968750806451258,39000475,1,1,0,0
6bba_91951b3a,6983,39,11.679999999999996,173.28000000000003,154.72,cont,2.354680511768004,40000484,1,0,1,0
6bba_91951b3a,7023,39,28.84000000000001,103.36,233.92000000000007,cont,2.2007204076165507,40000489,1,0,1,0
6bba_91951b3a,7164,40,29.68,104.64,230.07999999999998,cont,2.5126619972650555,41000501,1,1,0,0
6bba_91951b3a,7305,41,30.160000000000004,105.92,225.28,cont,1.3963658188311503,42000513,1,0,1,0
6bba_91951b3a,7867,45,41.16,165.27999999999997,94.72,cont,2.1033552065687653,46000561,1,1,0,0
6bba_91951b3a,7949,46,18.0,221.06666666666663,163.4666666666666,cont,3.325850977151894,47000566,1,0,0,1
6bba_91951b3a,8005,46,41.360000000000014,168.64,88.64,cont,2.3734458730925603,47000572,1,0,1,0
6bba_91951b3a,8231,48,29.679999999999993,52.0,203.35999999999999,cont,0.9377249730064874,49000590,1,1,0,0
6bba_91951b3a,8347,49,28.640000000000008,56.64,202.08000000000004,cont,0.8872261901567001,50000600,1,0,1,0
6bba_91951b3a,8503,50,40.00000000000001,182.55999999999997,68.0,cont,1.030434526061695,51000614,1,1,0,0
6bba_91951b3a,8532,50,54.680000000000014,4.640000000000001,88.00000000000001,cont,2.25195921643797,51000616,1,1,0,0
6bba_91951b3a,8628,51,39.52000000000001,186.72000000000003,62.56,cont,1.0114238107736908,52000625,1,0,1,0
6bba_91951b3a,8656,51,54.31999999999999,7.360000000000001,86.72,cont,1.2486092262994228,52000628,1,0,1,0
6bba_91951b3a,8972,54,26.0,25.92,234.07999999999998,cont,2.033393868634408,55000657,1,1,0,0
6bba_91951b3a,8999,54,37.84,196.64000000000001,48.800000000000004,cont,1.3509996299037208,55000662,1,1,0,0
6bba_91951b3a,9009,54,43.84,4.640000000000001,219.36,cont,3.7115512255659246,55000664,1,1,0,0
6bba_91951b3a,9091,55,25.16,34.07999999999999,229.92,cont,0.8228342785761927,56000670,1,0,1,0
6bba_91951b3a,9118,55,37.32,199.36,41.919999999999995,cont,1.1500821981493268,56000674,1,0,1,0
6bba_91951b3a,9128,55,43.480000000000004,8.0,216.64000000000001,cont,0.9729666296949762,56000676,1,0,1,0
6bba_91951b3a,9243,56,43.52,12.0,213.27999999999997,cont,2.5044513495973586,57000689,1,1,0,0
6bba_91951b3a,9355,57,43.0,17.92,210.08000000000004,cont,2.4052195845701823,58000702,1,0,1,0
6bba_91951b3a,9467,58,42.68000000000001,23.200000000000003,206.72,cont,3.9126072269651417,59000715,1,1,0,0
6bba_91951b3a,9484,58,51.00000000000001,19.360000000000003,164.00000000000003,cont,3.2603834130359433,59000718,1,1,0,0
6bba_91951b3a,9573,59,42.160000000000004,31.52,203.20000000000002,cont,3.0051518264473756,60000732,1,0,1,0
6bba_91951b3a,9589,59,50.48000000000001,27.52,161.27999999999997,cont,3.327165178496541,60000735,1,0,1,0
6bba_91951b3a,9705,60,59.32000000000001,19.360000000000003,103.36000000000003,cont,1.316249999999989,61000754,1,1,0,0
6bba_91951b3a,9819,61,59.20000000000002,25.28,100.00000000000001,cont,1.5307810506404609,62000770,1,0,1,0
6bba_91951b3a,10100,64,48.120000000000005,54.72,143.52,cont,3.2787974167368046,65000816,1,1,0,0
6bba_91951b3a,10166,65,31.84,101.75999999999999,194.71999999999997,cont,4.204355814212209,66000828,2,0,1,1
6bba_91951b3a,10198,65,53.86666666666666,72.0,83.99999999999997,cont,3.6970486070347928,66000835,1,0,0,1
6bba_91951b3a,10533,70,3.1599999999999993,190.72000000000003,228.64000000000007,cont,1.4836525873667332,71000906,1,1,0,0
6bba_91951b3a,10610,71,3.0,193.27999999999997,227.36,cont,1.6714608131810944,72000924,1,0,1,0
6bba_91951b3a,11044,76,49.84,24.0,160.64000000000001,cont,2.1257087053498087,77001024,1,1,0,0
6bba_91951b3a,11116,77,49.52,30.720000000000006,156.0,cont,2.49388533467359,78001040,1,0,1,0
6bba_91951b3a,11936,90,36.160000000000004,16.64,178.72,cont,1.9131571583380116,91001233,1,1,0,0
6bba_91951b3a,11992,91,35.199999999999996,22.560000000000006,175.36,cont,3.0052396950992133,92001247,1,0,1,0
6bba_96833384,847,14,52.000000000000014,4.320000000000001,60.96000000000001,cont,1.9476287666030967,15000112,1,1,0,0
6bba_96833384,856,14,60.67999999999999,228.0,247.35999999999999,cont,1.3959875536694373,15000115,1,1,0,0
6bba_96833384,905,15,51.00000000000001,5.279999999999999,63.36000000000001,cont,1.649595120779627,16000119,1,0,1,0
6bba_96833384,915,15,59.84000000000002,231.36,247.36,cont,1.1132139562994936,16000122,1,0,1,0
6bba_96833384,3568,64,61.99999999999999,101.86666666666663,171.4666666666667,cont,6.617040882050066,65000739,1,0,0,1
6bba_96833384,3617,65,57.52000000000001,95.2,177.44000000000003,cont,2.575765966173934,66000749,1,1,0,0
6bba_96833384,3681,66,56.72,100.64,176.00000000000003,cont,1.1363386929080568,67000758,1,0,1,0
6bba_96833384,4761,86,28.36,9.920000000000003,60.64,cont,1.2057894198407932,87000895,1,1,0,0
6bba_96833384,4810,87,26.48,14.08,60.000000000000014,cont,1.4766950006687218,88000903,1,0,1,0
6bba_96833384,4936,89,53.480000000000004,25.28,72.64,cont,2.6257204368515685,90000921,1,0,1,0
6bba_969618f6,307,1,22.599999999999998,25.28,200.00000000000003,cont,2.379890031598104,2000012,1,1,0,0
6bba_969618f6,527,2,21.68,28.000000000000004,199.36,cont,2.1607001180173078,3000020,1,0,1,0
6bba_969618f6,1434,6,27.360000000000007,138.72000000000003,123.36,cont,1.1831294572023736,7000057,1,1,0,0
6bba_969618f6,1661,7,26.68,140.0,122.71999999999998,cont,0.7218000502216673,8000065,1,0,1,0
6bba_969618f6,2553,11,23.16,146.72000000000003,116.64000000000001,cont,0.3848883766756267,12000099,1,1,0,0
6bba_969618f6,2756,12,19.68,16.0,205.28,cont,2.2071304900254525,13000105,1,1,0,0
6bba_969618f6,2766,12,22.48,147.35999999999999,115.36000000000001,cont,0.966977410542769,13000106,1,0,1,0
6bba_969618f6,2966,13,17.32,30.72,208.64,cont,6.993713806876856,14000112,1,0,1,0
6bba_969618f6,2970,13,19.32,18.080000000000002,204.00000000000003,cont,1.4215731822526843,14000112,1,0,1,0
6bba_969618f6,3461,15,35.36,14.719999999999999,196.00000000000003,cont,2.7116084364266175,16000133,1,1,0,0
6bba_969618f6,3684,16,35.36,16.000000000000004,196.00000000000003,cont,2.7861050321192127,17000141,1,0,1,0
6bba_969618f6,3845,17,18.0,26.720000000000002,198.72,cont,1.0622330547012717,18000145,1,1,0,0
6bba_969618f6,4064,18,17.32,29.279999999999994,197.28000000000003,cont,1.339713705423658,19000153,1,0,1,0
6bba_969618f6,4284,19,16.160000000000004,217.27999999999997,60.000000000000014,cont,0.3913518238107666,20000160,1,1,0,0
6bba_969618f6,4494,20,16.0,220.64000000000001,56.64,cont,0.969840515239488,21000167,1,0,1,0
6bba_969618f6,4544,20,34.16,23.360000000000007,193.92000000000002,cont,1.5934945873770723,21000171,1,1,0,0
6bba_969618f6,4763,21,33.84000000000001,25.28,194.07999999999998,cont,1.9078358157870763,22000177,1,0,1,0
6bba_969618f6,5868,26,59.160000000000004,67.36000000000001,140.64,cont,1.413656606110543,27000226,1,1,0,0
6bba_969618f6,6074,27,58.999999999999986,70.71999999999998,140.00000000000003,cont,0.11375000000000668,28000235,1,0,1,0
6bba_969618f6,8143,38,11.159999999999998,8.64,219.36,cont,1.5410107275745986,39000329,1,1,0,0
6bba_969618f6,8340,39,10.84,12.0,216.64,cont,2.125708705349822,40000337,1,0,1,0
6bba_969618f6,8535,40,9.159999999999998,124.63999999999999,135.20000000000002,daughter,1.553299411736189,41000358,2,0,0,2
6bba_969618f6,8536,40,8.73333333333333,136.0,126.93333333333332,daughter,2.2402353701986093,41000359,4,0,0,4
6bba_969618f6,8675,40,59.31999999999999,101.28,119.36000000000001,cont,1.248609226299421,41000352,1,1,0,0
6bba_969618f6,8876,41,59.31999999999999,104.64000000000001,116.64000000000001,cont,1.2985773465604755,42000360,1,0,1,0
6bba_969618f6,9147,43,11.68,28.639999999999997,210.08000000000004,cont,2.1502256422292056,44000370,1,1,0,0
6bba_969618f6,9335,44,12.0,33.279999999999994,208.00000000000006,cont,1.700358877560859,45000377,1,0,1,0
6bba_969618f6,9467,44,59.68000000000001,112.64,110.07999999999998,cont,0.6568699452707422,45000381,1,1,0,0
6bba_969618f6,9661,45,59.800000000000004,115.36000000000001,105.92,cont,0.3578691276151035,46000388,1,0,1,0
6bba_969618f6,11109,53,25.16,8.0,210.71999999999997,cont,1.5161345792837837,54000440,1,1,0,0
6bba_969618f6,11314,54,24.320000000000007,12.0,209.28000000000003,cont,0.7218000502216695,55000447,1,0,1,0
6bba_969618f6,11317,54,25.48,140.00000000000003,120.0,cont,1.9195645665098018,55000448,1,1,0,0
6bba_969618f6,11518,55,25.32,144.64000000000001,117.28,cont,1.4673669019710045,56000457,1,0,1,0
6bba_969618f6,11807,56,58.16000000000001,150.56,79.36000000000001,cont,1.4900461402251801,57000468,2,2,0,0
6bba_969618f6,12002,57,57.68000000000002,155.36,76.64000000000001,cont,2.2610869205981117,58000479,2,0,2,0
6bba_969618f6,12047,58,6.0,93.27999999999999,171.35999999999996,cont,2.2343011351427147,59000481,2,2,0,0
6bba_969618f6,12088,58,22.000000000000004,27.999999999999996,206.08,cont,1.6253249675064914,59000485,1,1,0,0
6bba_969618f6,12232,59,5.48,98.08000000000001,168.0,cont,2.170029161670408,60000493,2,0,2,0
6bba_969618f6,12288,59,21.68,31.360000000000007,204.00000000000003,cont,3.794644663865121,60000496,1,0,1,0
6bba_969618f6,13339,65,3.5200000000000005,119.36000000000001,144.64000000000001,cont,4.483233347707875,66000548,1,1,0,0
6bba_969618f6,13504,66,3.0,124.00000000000003,141.43999999999997,cont,3.6884636452729165,67000557,1,0,1,0
6bba_969618f6,13511,66,4.0,219.36,55.35999999999999,cont,3.398426336188563,67000568,1,1,0,0
6bba_969618f6,13666,67,3.16,220.64000000000004,49.92,cont,1.8414197138349435,68000578,1,0,1,0
6bba_969618f6,14174,70,1.32,228.0,35.36,cont,1.7411023806772534,71000610,1,1,0,0
6bba_969618f6,14369,71,0.9999999999999999,230.71999999999997,30.239999999999995,cont,2.1472147656906713,72000621,1,0,1,0
6bba_969618f6,14626,72,17.16,211.36,44.63999999999999,cont,1.8451443879815936,73000623,1,1,0,0
6bba_969618f6,14672,72,35.480000000000004,40.64000000000001,124.00000000000003,cont,4.103245666542516,73000626,1,1,0,0
6bba_969618f6,14789,73,16.12,214.72,39.84,cont,3.432136952759897,74000633,1,0,1,0
6bba_969618f6,14831,73,34.00000000000001,45.44,121.28000000000003,cont,2.8850043327523784,74000636,1,0,1,0
6bba_969618f6,15208,76,10.639999999999999,225.92000000000002,26.719999999999995,cont,0.8972884499981048,77000658,1,1,0,0
6bba_969618f6,15339,77,8.28,232.16,20.48,cont,1.5978799430808324,78000666,1,0,1,0
6bba_969618f6,15411,77,47.76,50.08000000000002,152.63999999999996,cont,1.100450191966917,78000671,1,1,0,0
6bba_969618f6,15535,78,46.0,52.64000000000001,152.64,cont,1.6521543700574723,79000680,1,0,1,0
6bba_969618f6,15653,79,44.20000000000001,55.36000000000001,153.27999999999997,cont,3.0576351666770054,80000688,1,1,0,0
6bba_969618f6,15766,80,42.800000000000004,56.640000000000015,154.72000000000003,cont,0.6304078540912855,81000696,1,0,1,0
6bba_969618f6,16024,82,39.72,60.640000000000015,154.08,cont,2.3192000910874397,83000711,2,2,0,0
6bba_969618f6,16319,84,58.239999999999995,8.000000000000002,120.00000000000006,cont,1.533108056367864,85000726,1,1,0,0
6bba_969618f6,16435,85,58.0,11.36,120.00000000000003,cont,1.645668557152381,86000732,1,0,1,0
6bba_969618f6,16663,87,47.2,193.92000000000002,10.080000000000002,cont,6.9216798223046965,88000746,1,0,0,1
6bba_969618f6,16993,90,53.64,30.720000000000006,112.64,cont,0.6136392466262183,91000769,1,1,0,0
6bba_969618f6,17096,91,52.48,36.64000000000001,109.91999999999999,cont,2.585435770813113,92000775,1,0,1,0
6bba_971fa5e0,110,1,19.28,196.0,179.84000000000003,cont,2.18319218691345,2000012,1,1,0,0
6bba_971fa5e0,119,1,31.52000000000001,176.00000000000003,239.83999999999995,cont,1.5561019005836534,2000015,1,1,0,0
6bba_971fa5e0,180,2,17.8,196.64000000000001,171.2,cont,1.9197021253309037,3000021,1,0,1,0
6bba_971fa5e0,194,2,30.479999999999997,176.64000000000001,231.20000000000002,cont,1.9908737089529402,3000023,1,0,1,0
6bba_971fa5e0,490,6,44.84000000000001,169.28,231.36,cont,1.89408371647082,7000067,1,1,0,0
6bba_971fa5e0,558,7,44.00000000000001,170.72000000000003,223.84,cont,1.1593438338129034,8000076,1,0,1,0
6bba_971fa5e0,574,7,60.16000000000001,126.08000000000003,212.48,cont,1.6354476642497522,8000079,1,0,1,0
6bba_971fa5e0,746,10,30.04,196.0,238.72000000000003,cont,4.855052941266449,11000109,1,1,0,0
6bba_971fa5e0,811,11,27.360000000000007,197.92000000000002,232.64000000000004,cont,2.677850117164881,12000121,1,0,1,0
6bba_971fa5e0,821,11,39.36,148.64000000000001,153.92000000000002,cont,1.5570349426072627,12000123,1,1,0,0
6bba_971fa5e0,885,12,37.8,148.00000000000003,151.2,cont,1.352269504388838,13000135,1,0,1,0
6bba_971fa5e0,1071,15,25.16,180.64000000000001,64.0,cont,1.2547273110122363,16000168,1,1,0,0
6bba_971fa5e0,1132,16,24.0,180.0,58.08000000000001,cont,1.998749999999994,17000178,1,0,1,0
6bba_971fa5e0,1257,18,23.319999999999997,173.27999999999997,184.64,cont,1.1973284062862504,19000201,1,1,0,0
6bba_971fa5e0,1258,18,24.66123443603516,189.17388916015628,172.11522949218747,cont,1.1264489335497867,19000202,1,1,0,0
6bba_971fa5e0,1276,18,45.51999999999999,124.63999999999999,236.64000000000001,cont,2.556006712139085,19000199,1,1,0,0
6bba_971fa5e0,1315,19,22.839999999999996,172.64000000000004,183.36,cont,0.7609790158079417,20000204,1,0,1,0
6bba_971fa5e0,1316,19,23.48,190.72000000000003,167.84,cont,1.8199999999999974,20000205,1,0,1,0
6bba_971fa5e0,1334,19,44.68000000000001,125.28000000000002,235.36000000000004,cont,0.6508119928212504,20000210,1,0,1,0
6bba_971fa5e0,1585,23,62.000000000000014,36.64,36.64,cont,0.2983103459486445,24000264,1,1,0,0
6bba_971fa5e0,1639,24,62.000000000000014,35.99999999999999,33.92,cont,0.03249999999999886,25000277,1,0,1,0
6bba_971fa5e0,1734,26,37.64,152.64,138.56,cont,1.4935862880998847,27000297,1,1,0,0
6bba_971fa5e0,1735,26,39.36000000000001,129.28,233.27999999999997,cont,3.120211581367509,27000299,1,1,0,0
6bba_971fa5e0,1781,27,23.799999999999997,165.76000000000005,137.44,cont,2.9545074145278596,28000321,2,0,1,1
6bba_971fa5e0,1786,27,29.200000000000003,160.79999999999998,140.00000000000003,cont,1.7745267855121234,28000320,2,0,0,2
6bba_971fa5e0,1791,27,38.32000000000001,130.72,234.72000000000006,cont,1.226525198477373,28000310,1,0,1,0
6bba_971fa5e0,1972,30,58.36000000000001,45.92,24.640000000000008,cont,1.3257450735341192,31000346,1,1,0,0
6bba_971fa5e0,2024,31,57.00000000000001,48.0,23.360000000000007,cont,1.8989569505388888,32000355,1,0,1,0
6bba_971fa5e0,2182,34,56.00000000000001,73.28000000000002,125.44000000000001,cont,1.4507164178432594,35000382,1,1,0,0
6bba_971fa5e0,2233,35,53.999999999999986,76.32,124.63999999999997,cont,1.1768635488025039,36000390,1,0,1,0
6bba_971fa5e0,2254,36,10.2,177.92,133.28000000000003,cont,4.413842217954788,37000401,1,1,0,0
6bba_971fa5e0,2266,36,25.56,151.2,222.07999999999998,cont,2.677061121827443,37000394,1,1,0,0
6bba_971fa5e0,2307,37,7.799999999999998,180.0,133.28,cont,1.6471120559937622,38000409,1,0,1,0
6bba_971fa5e0,2308,37,6.680000000000001,202.56,142.40000000000003,cont,3.2508123984628963,38000410,1,0,0,1
6bba_971fa5e0,2319,37,23.28,155.36,220.8,cont,1.0360569784041747,38000402,1,0,1,0
6bba_971fa5e0,2709,44,9.840000000000003,176.64000000000001,206.72,cont,1.925196418550587,45000457,1,1,0,0
6bba_971fa5e0,2761,45,8.000000000000002,179.35999999999999,204.64000000000001,cont,1.8411328883326192,46000464,1,0,1,0
6bba_971fa5e0,2922,48,6.04,157.92000000000002,23.360000000000007,cont,1.567177498721827,49000490,1,1,0,0
6bba_971fa5e0,2936,48,27.04,112.00000000000001,5.4399999999999995,cont,1.8748866632412764,49000492,1,1,0,0
6bba_971fa5e0,2976,49,4.28,161.44000000000005,21.280000000000005,cont,2.5502147581135155,50000498,1,0,1,0
6bba_971fa5e0,2990,49,21.800000000000004,152.48,89.44,cont,2.7798901169830397,50000510,1,0,1,0
6bba_971fa5e0,2992,49,25.28,114.72,2.5600000000000005,cont,2.476672730196705,50000501,1,0,1,0
6bba_971fa5e0,3014,49,50.0,107.99999999999997,252.0,cont,0.9084026158593023,50000512,1,0,0,1
6bba_971fa5e0,3212,53,13.960000000000003,164.48000000000002,74.88000000000002,cont,3.4180668878768268,54000550,2,0,1,1
6bba_971fa5e0,3427,56,57.16000000000002,18.720000000000002,7.360000000000001,cont,0.6368673331236386,57000592,1,1,0,0
6bba_971fa5e0,3432,56,62.000000000000014,23.36,240.16000000000005,cont,2.3867040558058044,57000594,1,1,0,0
6bba_971fa5e0,3478,57,56.99999999999999,17.279999999999998,4.640000000000001,cont,1.7933993559717938,58000604,1,0,1,0
6bba_971fa5e0,3479,57,57.48,43.36,109.28,cont,1.2197246103116912,58000603,1,1,0,0
6bba_971fa5e0,3485,57,62.000000000000014,24.0,234.56,cont,0.831930172851062,58000605,1,0,1,0
6bba_971fa5e0,3532,58,56.84,44.00000000000001,103.19999999999999,cont,0.9825540697590124,59000614,1,0,1,0
6bba_971fa5e0,3643,60,62.000000000000014,21.280000000000005,215.2,cont,0.9403963592549655,61000635,1,1,0,0
6bba_971fa5e0,3692,61,62.000000000000014,20.640000000000004,209.92000000000004,cont,0.5534550568926055,62000644,1,0,1,0
6bba_971fa5e0,4282,73,59.519999999999996,9.280000000000001,155.99999999999997,cont,1.5778406922119939,74000747,1,1,0,0
6bba_971fa5e0,4329,74,58.8,9.440000000000001,154.08,cont,0.7129658564756112,75000755,1,0,1,0
6bba_971fa5e0,4601,80,5.359999999999999,196.00000000000003,223.52,cont,2.859492142409204,81000800,1,1,0,0
6bba_971fa5e0,4656,81,4.319999999999999,197.28,217.12,cont,2.3580424216921956,82000810,1,0,1,0
6bba_971fa5e0,5280,92,33.32,77.43999999999998,223.36,cont,2.814629471617887,93000938,1,1,0,0
6bba_971fa5e0,5335,93,33.68,74.56000000000002,215.84000000000003,cont,3.2605858848526137,94000949,1,0,1,0
6bba_9a41d029,343,3,5.84,249.27999999999997,14.719999999999999,cont,2.1205471729013743,4000059,1,0,1,0
6bba_9a41d029,467,4,11.160000000000002,155.99999999999997,14.719999999999999,cont,2.3537270976049864,5000085,1,0,1,0
6bba_9a41d029,549,5,3.84,34.72,2.72,cont,1.09358998829543,6000100,1,1,0,0
6bba_9a41d029,560,5,6.0,246.72,17.919999999999998,cont,2.1017225078492143,6000101,1,1,0,0
6bba_9a41d029,574,5,10.928013916015624,244.52036132812503,16.390476837158204,cont,6.791616712994134,6000101,1,1,0,0
6bba_9a41d029,1026,9,15.679999999999998,118.72,87.36,cont,3.2399908371784005,10000186,1,1,0,0
6bba_9a41d029,1116,10,7.000000000000001,233.92000000000002,38.72,cont,1.706482127067258,11000199,3,3,0,0
6bba_9a41d029,1232,11,7.000000000000001,231.35999999999996,43.36,cont,1.878404409332565,12000218,3,0,3,0
6bba_9a41d029,1419,12,31.160000000000004,132.00000000000006,170.56,cont,1.2589293566360062,13000242,1,1,0,0
6bba_9a41d029,1439,12,48.16,162.72,205.28,cont,1.1890292784452339,13000243,2,2,0,0
6bba_9a41d029,1641,14,16.733333333333327,143.4666666666666,56.53333333333333,cont,1.2656931566010483,15000272,1,0,1,0
6bba_9a41d029,1796,15,21.2,112.00000000000001,90.56,cont,3.101196593655425,16000288,2,0,0,2
6bba_9a41d029,2297,19,7.32,205.92,61.280000000000015,cont,1.111314677532877,20000349,1,1,0,0
6bba_9a41d029,2694,22,3.36,143.36,29.279999999999994,cont,3.6272824710794103,23000407,3,0,0,3
6bba_9a41d029,3088,24,29.519999999999996,107.36,170.07999999999998,cont,1.0917775070956586,25000447,1,0,1,0
6bba_9a41d029,3440,26,49.160000000000004,241.44000000000003,235.99999999999994,cont,0.6850102645216294,27000487,1,1,0,0
6bba_9a41d029,3605,27,49.32,238.55999999999997,235.99999999999994,cont,1.128176072251138,28000504,1,0,1,0
6bba_9a41d029,3623,28,2.1600000000000006,123.83999999999997,28.640000000000004,cont,2.0281926991782524,29000507,3,0,3,0
6bba_9a41d029,3669,28,11.000000000000002,241.91999999999996,78.72,cont,0.8941930719928568,29000511,2,0,2,0
6bba_9a41d029,3853,29,12.866666666666665,234.93333333333334,84.53333333333329,cont,5.4600967253337105,30000527,4,0,0,4
6bba_9a41d029,3856,29,14.160000000000004,140.64000000000001,20.000000000000004,cont,2.1380949464418073,30000529,2,2,0,0
6bba_9a41d029,4033,30,14.84,137.28000000000003,26.080000000000002,cont,1.079004894567205,31000545,2,0,2,0
6bba_9a41d029,4448,32,30.639999999999993,224.00000000000003,105.28000000000002,cont,2.5263393724913663,33000581,1,1,0,0
6bba_9a41d029,4552,33,9.52,40.00000000000001,60.64,cont,0.9170818188689599,34000590,1,1,0,0
6bba_9a41d029,4594,33,16.84,244.00000000000003,110.72000000000003,cont,2.3041216466367307,34000596,1,0,1,0
6bba_9a41d029,6328,42,14.360000000000003,220.0,110.72000000000001,cont,4.4693408700277,43000736,1,0,1,0
6bba_9a41d029,6333,42,15.200000000000001,172.0,16.0,cont,4.257686064636516,43000735,1,0,1,0
6bba_9a41d029,6359,42,22.600000000000005,242.72000000000003,132.0,cont,1.104999999999999,43000739,4,0,1,3
6bba_9a41d029,6572,43,25.13333333333333,239.46666666666664,144.53333333333333,cont,6.7301947099124355,44000761,4,0,0,4
6bba_9a41d029,6931,45,14.133333333333331,180.53333333333327,33.06666666666666,cont,6.946439435662931,46000797,2,0,0,2
6bba_9a41d029,7030,45,44.32,67.36,184.64,cont,1.5889309495695512,46000805,1,1,0,0
6bba_9a41d029,7261,47,0.0,124.0,8.959999999999999,cont,2.074724755841121,48000829,2,2,0,0
6bba_9a41d029,7292,47,3.6799999999999997,139.36,10.719999999999999,cont,6.653159611229846,48000829,1,1,0,0
6bba_9a41d029,7392,47,22.68,117.28,82.72,cont,1.3397137054236576,48000836,2,2,0,0
6bba_9a41d029,7600,48,23.319999999999997,116.0,85.28,cont,1.7836549554215912,49000853,2,0,2,0
6bba_9a41d029,7720,49,4.000000000000001,124.96000000000001,0.3200000000000001,cont,6.1469263152408145,50000862,1,1,0,0
6bba_9a41d029,8266,51,19.0,128.0,125.92,cont,1.6753269270503592,52000902,1,0,1,0
6bba_9a41d029,8382,52,0.32000000000000006,214.55999999999995,38.56000000000001,cont,2.811813527334271,53000915,3,3,0,0
6bba_9a41d029,8395,52,2.4799999999999995,220.0,25.28000000000001,cont,5.405488674717575,53000915,3,3,0,0
6bba_9a41d029,8400,52,3.0,111.84,33.92,parent,3.3987371275960747,53000912,1,1,0,0
6bba_9a41d029,8405,52,4.640000000000001,129.28,7.200000000000002,cont,6.908543736743655,53000913,1,1,0,0
6bba_9a41d029,8605,53,0.0,138.72,26.72,cont,3.3043087401149434,54000929,3,2,1,0
6bba_9a41d029,8611,53,0.6399999999999999,218.72000000000003,42.56,cont,2.9815982501000935,54000931,3,0,3,0
6bba_9a41d029,8625,53,3.133333333333333,87.99999999999997,36.0,daughter,0.6140216359742184,54000928,6,2,0,4
6bba_9a41d029,8626,53,3.0,118.07999999999998,36.0,daughter,2.175862257244243,54000950,2,1,1,0
6bba_9a41d029,8636,53,5.0,83.36000000000001,40.64000000000001,daughter,4.589352146545301,54000928,2,1,1,0
6bba_9a41d029,8637,53,4.68,119.20000000000005,44.64,daughter,6.436825814211535,54000950,4,0,0,4
6bba_9a41d029,8818,53,48.8,122.55999999999997,200.64,cont,4.189759278586308,54000945,1,0,1,0
6bba_9a41d029,9101,55,2.0,224.64000000000001,52.0,cont,2.105049361060209,56000969,1,1,0,0
6bba_9a41d029,10080,59,3.160000000000001,95.99999999999997,26.720000000000002,cont,6.44451317013162,60001051,1,0,1,0
6bba_9a41d029,10495,60,32.160000000000004,212.64,185.28,cont,1.3941893926221,61001088,1,0,1,0
6bba_9a41d029,10568,61,0.0,82.72000000000001,4.640000000000001,cont,3.424280261967468,62001097,1,0,1,0
6bba_9a41d029,10833,62,0.6399999999999999,212.64,45.28000000000001,cont,2.382773134920739,63001120,1,0,1,0
6bba_9a41d029,11377,64,3.84,156.64,52.64,cont,1.5998618104386302,65001160,1,0,1,0
6bba_9a41d029,12013,66,21.68,143.36,140.0,cont,2.6368097438002587,67001208,1,0,1,0
6bba_9a41d029,12188,67,6.5200000000000005,245.43999999999997,87.51999999999998,cont,3.1903017098700928,68001223,1,1,0,0
6bba_9a41d029,12442,68,6.160000000000002,176.0,73.12,cont,2.8652579508658556,69001241,6,0,1,5
6bba_9a41d029,12446,68,6.16,243.20000000000005,84.48,cont,2.2516660498395473,69001242,3,0,1,2
6bba_9a41d029,14087,74,37.64,207.36000000000004,206.55999999999997,cont,1.0898408714119507,75001363,1,1,0,0
6bba_9a41d029,14162,75,3.6799999999999997,92.0,30.72,cont,1.423429507387001,76001369,1,0,1,0
6bba_9a41d029,14335,75,36.266666666666666,215.99999999999994,196.5333333333333,cont,5.632421801252302,76001380,5,0,0,5
6bba_9a41d029,14724,77,15.000000000000004,89.28000000000003,105.92,cont,1.6928881571149264,78001408,1,1,0,0
6bba_9a41d029,14913,78,5.960000000000001,129.92000000000002,63.2,cont,2.312130037973645,79001421,5,0,1,4
6bba_9a41d029,14941,78,8.999999999999998,201.43999999999997,94.07999999999998,cont,1.7139706276654902,79001423,2,0,2,0
6bba_9a41d029,15825,81,48.52,216.0,171.36,cont,4.170270524798121,82001488,1,0,1,0
6bba_9a41d029,16026,82,34.480000000000004,225.92,201.92000000000002,cont,2.5259212428339803,83001505,2,2,0,0
6bba_9a41d029,16045,82,40.16,227.20000000000002,206.55999999999997,cont,6.848213498789317,83001504,1,1,0,0
6bba_9a41d029,16102,83,1.8,133.92000000000004,21.120000000000008,cont,1.5233454138835463,84001511,8,0,3,5
6bba_9a41d029,16119,83,5.320000000000001,137.92000000000002,53.120000000000005,cont,1.722576649237998,84001513,3,0,1,2
6bba_9a41d029,16132,83,7.3199999999999985,161.12,71.99999999999999,cont,2.7509095086898063,84001514,4,0,1,3
6bba_9a41d029,16155,83,12.440000000000001,224.64000000000001,120.47999999999999,cont,2.8080545422943577,84001515,7,0,2,5
6bba_9a41d029,16234,83,26.0,142.08,88.64,cont,3.388621816092791,84001518,1,0,1,0
6bba_9a41d029,16235,83,25.84,165.44000000000003,162.07999999999998,cont,3.1048132995399214,84001519,1,0,1,0
6bba_9a41d029,17692,89,17.32,113.44,111.83999999999999,cont,1.3823010706788845,90001613,3,3,0,0
6bba_9a41d029,17714,89,20.36,154.56,133.44,cont,2.869770243416708,90001614,3,3,0,0
6bba_9a41d029,17799,89,40.72,245.92,212.80000000000004,cont,2.91994006787811,90001618,3,3,0,0
6bba_9a41d029,17850,89,59.519999999999996,32.160000000000004,230.72000000000008,cont,2.514500447405014,90001621,2,2,0,0
6bba_9a41d029,17934,90,17.32,134.56000000000003,115.83999999999999,cont,1.5553380862372097,91001631,4,0,2,2
6bba_9a41d029,17958,90,21.2,173.12,143.99999999999997,cont,3.5891537999924186,91001632,5,0,2,3
6bba_9a41d029,18025,90,38.6,249.44,211.2,cont,5.149506802840446,91001636,5,0,3,2
6bba_9a41d029,18095,90,61.99999999999999,109.33333333333333,243.99999999999997,cont,5.557035393185014,91001640,2,0,0,2
6bba_9a41d029,18906,94,21.119999999999997,181.59999999999997,145.27999999999997,cont,3.1254542169899127,95001704,1,1,0,0
6bba_9a41d029,19083,95,11.120000000000001,217.44000000000003,110.88,cont,5.818861572507119,96001717,3,1,2,0
6bba_9a41d029,19129,95,20.4,187.46666666666664,147.46666666666664,cont,1.4341385819446637,96001720,1,0,1,0
6bba_9a41d029,19296,96,11.320000000000002,221.28000000000006,108.48,cont,2.488001833801575,97001734,5,0,1,4
6bba_9e23430b,523,12,58.0,36.64000000000001,179.36,cont,1.735938146939571,13000145,1,1,0,0
6bba_9e23430b,564,13,57.48,39.36,178.72000000000003,cont,2.58390329733912,14000157,1,0,1,0
6bba_9e23430b,1831,38,6.359999999999999,108.64,196.0,cont,1.4939398414929557,39000429,1,1,0,0
6bba_9e23430b,1903,39,7.360000000000002,110.71999999999998,193.27999999999997,cont,1.0863219193682823,40000441,1,0,1,0
6bba_9e23430b,2510,47,36.36,70.72,234.56000000000006,cont,3.2149137332905027,48000545,1,1,0,0
6bba_9e23430b,2582,48,35.84,69.44000000000001,235.35999999999999,cont,2.549127301646592,49000558,2,1,1,0
6bba_9e23430b,2661,49,35.00000000000001,67.2,235.35999999999999,cont,2.4364706159730343,50000571,1,0,1,0
6bba_9e23430b,3365,57,39.999999999999986,76.5333333333333,241.0666666666666,cont,5.836756807913266,58000661,2,0,0,2
6bba_9e23430b,3448,58,37.480000000000004,51.360000000000014,250.72000000000003,cont,6.856537254693522,59000673,1,1,0,0
6bba_9e23430b,3866,62,40.32,76.0,221.28000000000003,cont,6.5156826436299005,63000716,1,1,0,0
6bba_9e23430b,4236,66,7.2,101.92000000000002,195.36000000000004,cont,4.552465849679709,67000754,1,1,0,0
6bba_9e23430b,4341,67,8.520000000000001,102.72000000000001,192.0,cont,2.4605741199971978,68000764,1,0,1,0
6bba_9e23430b,4390,67,46.47999999999999,68.64000000000001,211.36,cont,4.49866803204014,68000772,1,0,1,0
6bba_9e23430b,4613,69,47.16,71.35999999999999,206.08,cont,3.426631454504549,70000800,1,1,0,0
6bba_9e23430b,4731,70,47.84,71.99999999999999,202.72,cont,3.2623670949940546,71000813,1,0,1,0
6bba_9e23430b,4850,71,49.96000000000001,62.720000000000006,214.55999999999995,cont,5.98743593389522,72000826,1,1,0,0
6bba_9e23430b,4970,72,48.36,74.55999999999999,197.27999999999997,cont,5.356735538086605,73000839,1,1,0,0
6bba_9e23430b,5145,74,5.000000000000002,232.0,76.64000000000001,cont,1.6813842883172148,75000855,1,1,0,0
6bba_9e23430b,5267,75,5.319999999999999,232.0,74.72,cont,1.1108393504463194,76000866,1,0,1,0
6bba_9e23430b,5882,79,35.84,137.6,178.08,cont,6.695315526545401,80000917,3,0,1,2
6bba_9e23430b,6556,83,59.64,44.00000000000001,90.72,cont,0.5959564266789925,84000970,1,1,0,0
6bba_9e23430b,6706,84,59.83999999999999,46.08000000000001,89.27999999999999,cont,0.7462625627083379,85000982,1,0,1,0
6bba_9e23430b,6854,85,60.35999999999999,77.28,139.35999999999999,cont,6.497988751336836,86000994,1,0,1,0
6bba_9e23430b,6863,85,61.99999999999999,80.53333333333332,147.46666666666667,cont,5.7102175266261765,86000994,2,0,0,2
6bba_9e23430b,7443,89,48.0,233.28000000000003,42.07999999999999,parent,1.675248115951791,90001032,1,1,0,0
6bba_9e23430b,7488,89,60.84000000000001,57.44,77.28,cont,1.2796294629696467,90001035,1,1,0,0
6bba_9e23430b,7602,90,47.999999999999986,223.46666666666664,53.066666666666656,daughter,3.51718561040719,91001046,1,0,0,1
6bba_9e23430b,7604,90,48.52000000000001,233.27999999999997,40.00000000000001,daughter,0.7882506343162532,91001045,1,0,1,0
6bba_9e23430b,7647,90,61.00000000000001,56.63999999999999,75.36,cont,0.8655318090630778,91001044,1,0,1,0
6bba_a5e926bb,335,1,23.840000000000003,175.68,124.0,cont,1.9326572542745297,2000036,2,2,0,0
6bba_a5e926bb,343,1,24.12,148.32,81.11999999999999,cont,1.868184697105724,2000035,2,2,0,0
6bba_a5e926bb,348,1,26.479999999999993,151.52000000000004,149.12000000000006,cont,2.909476166339902,2000037,1,0,1,0
6bba_a5e926bb,424,1,53.19999999999999,140.48000000000002,168.15999999999997,cont,2.3665936755176396,2000046,1,1,0,0
6bba_a5e926bb,441,1,61.75999999999999,31.36,218.08000000000004,cont,1.2572502237024978,2000048,1,0,1,0
6bba_a5e926bb,531,2,18.4,175.84000000000003,100.16,cont,6.278517191582746,3000057,3,0,1,2
6bba_a5e926bb,542,2,22.320000000000004,149.28000000000003,71.2,cont,1.0745907651287527,3000058,2,0,2,0
6bba_a5e926bb,549,2,22.400000000000006,145.28,130.88,cont,3.456058312948433,3000059,4,0,2,2
6bba_a5e926bb,558,2,25.880000000000003,157.6,139.2,cont,6.797260592510775,3000059,3,0,0,3
6bba_a5e926bb,566,2,28.16,248.64000000000007,108.16000000000001,cont,2.4846032127887203,3000062,3,0,1,2
6bba_a5e926bb,592,2,35.68,179.35999999999999,174.72,cont,4.036547101174469,3000065,3,0,1,2
6bba_a5e926bb,629,2,51.64000000000001,137.92000000000002,160.48,cont,1.8576955650751708,3000068,1,0,1,0
6bba_a5e926bb,1511,7,15.0,165.28000000000003,84.64000000000001,cont,1.6355283932111973,8000160,4,4,0,0
6bba_a5e926bb,1571,7,32.32,171.36,152.32000000000002,cont,4.511211900642656,8000168,1,1,0,0
6bba_a5e926bb,1604,7,44.84000000000001,239.51999999999998,173.60000000000002,cont,4.782720362147464,8000171,1,1,0,0
6bba_a5e926bb,1696,8,10.999999999999996,165.0666666666666,73.59999999999997,cont,1.845247743830374,9000181,4,0,0,4
6bba_a5e926bb,1714,8,15.0,164.16,84.15999999999998,cont,6.674023570905328,9000181,7,0,4,3
6bba_a5e926bb,1735,8,18.880000000000003,242.07999999999998,87.36,cont,3.6527816883164506,9000187,3,0,1,2
6bba_a5e926bb,1759,8,27.679999999999993,168.00000000000003,144.79999999999998,cont,3.3331915073994893,9000190,4,0,2,2
6bba_a5e926bb,1800,8,43.199999999999996,235.84000000000003,165.92000000000002,cont,5.712285280428473,9000192,1,0,1,0
6bba_a5e926bb,2123,10,36.02860534667969,140.813291015625,102.76822998046877,cont,2.4145301224557456,11000226,1,0,1,0
6bba_a5e926bb,2195,11,8.040000000000001,234.72,29.440000000000005,cont,5.10741403500832,12000233,1,1,0,0
6bba_a5e926bb,2340,12,7.3199999999999985,234.72000000000006,23.36,cont,2.978851498480579,13000247,1,0,1,0
6bba_a5e926bb,2456,12,61.92000000000001,43.36,223.68000000000004,cont,0.3184336665617991,13000257,1,1,0,0
6bba_a5e926bb,2606,13,61.52000000000001,42.56,218.07999999999998,cont,0.8008793994728419,14000270,1,0,1,0
6bba_a5e926bb,2706,14,38.31999999999999,240.64000000000004,153.27999999999997,cont,1.3329953347630479,15000280,1,1,0,0
6bba_a5e926bb,2841,15,37.160000000000004,240.00000000000006,148.64000000000001,cont,0.9933788615125615,16000296,1,0,1,0
6bba_a5e926bb,3000,16,57.68000000000002,150.72,238.72000000000003,cont,1.704081955922286,17000315,1,0,1,0
6bba_a5e926bb,3237,18,54.16,153.27999999999997,226.56000000000003,cont,0.8247575401292045,19000339,1,1,0,0
6bba_a5e926bb,3352,19,52.28,154.72000000000003,221.28,cont,1.1555795353414577,20000351,1,0,1,0
6bba_a5e926bb,3699,22,60.16000000000001,239.36,225.28000000000003,cont,0.3192618439463367,23000390,1,1,0,0
6bba_a5e926bb,3754,23,23.8,235.35999999999999,96.64000000000001,cont,1.0228468238206472,24000396,1,1,0,0
6bba_a5e926bb,3808,23,59.32000000000001,237.27999999999997,220.64000000000001,cont,0.650811992821294,24000402,1,0,1,0
6bba_a5e926bb,3868,24,23.0,233.27999999999997,91.35999999999999,cont,0.1852785065785947,25000406,1,0,1,0
6bba_a5e926bb,4073,26,17.000000000000004,112.16,17.28,cont,0.2996351948620179,27000424,1,1,0,0
6bba_a5e926bb,4134,26,56.99999999999999,232.63999999999993,208.64000000000004,cont,0.6106195624118472,27000433,1,1,0,0
6bba_a5e926bb,4183,27,16.520000000000003,110.08,14.719999999999999,cont,0.9960335398469315,28000434,1,0,1,0
6bba_a5e926bb,4241,27,56.160000000000004,231.36,205.28,cont,1.5231720602085548,28000443,1,0,1,0
6bba_a5e926bb,4611,31,17.64,225.28,62.56,cont,1.5599999999999976,32000485,1,1,0,0
6bba_a5e926bb,4623,31,23.68,59.36,66.08,cont,2.862676304614271,32000487,1,1,0,0
6bba_a5e926bb,4702,32,12.999999999999998,224.53333333333327,43.466666666666654,cont,1.7505976162125085,33000497,4,0,0,4
6bba_a5e926bb,4716,32,22.16,58.72,60.0,cont,1.4606933285258767,33000499,1,0,1,0
6bba_a5e926bb,4856,33,58.680000000000014,111.36,235.36000000000004,cont,1.0761868448368748,34000521,1,1,0,0
6bba_a5e926bb,4945,34,57.68000000000002,112.64,230.72000000000008,cont,2.2752321310142904,35000533,1,0,1,0
6bba_a5e926bb,5378,39,60.32000000000001,212.64,225.92,cont,1.175628821099577,40000601,1,1,0,0
6bba_a5e926bb,5460,40,59.52000000000001,213.92000000000002,221.28000000000006,cont,0.7889203461059837,41000614,1,0,1,0
6bba_a5e926bb,5539,41,58.519999999999996,216.0,218.08,cont,0.9555773778192966,42000626,1,1,0,0
6bba_a5e926bb,5618,42,57.32000000000001,218.72000000000006,213.92000000000002,cont,1.317853960991118,43000640,1,0,1,0
6bba_a5e926bb,5749,44,32.16000000000001,249.27999999999997,120.64000000000001,cont,1.0355471078130685,45000664,1,1,0,0
6bba_a5e926bb,5827,45,31.320000000000004,250.71999999999997,116.63999999999999,cont,1.2406531797807177,46000683,1,0,1,0
6bba_a5e926bb,5966,47,17.84,52.640000000000015,39.52,cont,2.6778501171648874,48000712,1,1,0,0
6bba_a5e926bb,6019,47,60.199999999999996,190.72000000000003,187.51999999999998,cont,2.2394952695864423,48000724,1,1,0,0
6bba_a5e926bb,6042,48,15.466666666666663,49.06666666666665,78.93333333333331,cont,2.8192696305958465,49000728,3,0,2,1
6bba_a5e926bb,6043,48,16.48,53.440000000000005,27.200000000000003,cont,1.059370390845429,49000727,2,1,1,0
6bba_a5e926bb,6088,48,58.519999999999996,190.72000000000003,175.20000000000005,cont,1.2952177664392976,49000740,1,0,1,0
6bba_a5e926bb,6107,49,11.919999999999996,48.64,56.960000000000015,cont,4.707482242398795,50000745,2,0,0,2
6bba_a5e926bb,6110,49,15.0,51.99999999999999,15.839999999999996,cont,1.411132634623692,50000744,1,0,1,0
6bba_a5e926bb,7071,65,22.519999999999996,144.8,18.080000000000002,cont,1.9129501104315298,66000987,1,1,0,0
6bba_a5e926bb,7124,66,20.799999999999997,138.88000000000002,8.48,cont,3.2683513141184792,67000997,1,0,1,0
6bba_a5e926bb,7348,70,23.680000000000007,6.080000000000001,66.24,cont,4.084831124110265,71001042,1,0,0,1
6bba_a5e926bb,7800,78,55.72000000000001,226.08,92.80000000000001,cont,2.4490103230692983,79001131,1,1,0,0
6bba_a5e926bb,7802,78,58.96000000000001,88.64,164.0,cont,1.704081955922324,79001134,1,1,0,0
6bba_a5e926bb,7845,79,51.56,94.72,144.16000000000003,cont,4.139797134220467,80001143,1,0,1,0
6bba_a5e926bb,7846,79,53.64,221.92000000000004,85.59999999999998,cont,1.8474327626465799,80001141,1,0,1,0
6bba_a5e926bb,8057,83,59.0,213.28000000000003,140.64000000000004,cont,0.5813776741499631,84001183,1,0,1,0
6bba_a5e926bb,8556,93,33.2,69.44,88.79999999999998,cont,2.315040496406055,94001281,2,0,1,1
6bba_a90a0b9c,578,1,11.399999999999993,4.5333333333333306,123.99999999999999,cont,5.804938331091704,2000026,3,0,0,3
6bba_a90a0b9c,632,1,18.080000000000002,8.64,84.32000000000002,cont,3.2665733670774926,2000031,2,2,0,0
6bba_a90a0b9c,633,1,18.800000000000008,24.320000000000007,78.72,cont,6.138823304388226,2000031,2,1,1,0
6bba_a90a0b9c,868,1,54.28000000000001,8.000000000000002,90.56000000000002,cont,3.8636062331712853,2000039,1,0,1,0
6bba_a90a0b9c,972,2,4.8,54.56,10.720000000000002,cont,1.3517812323005522,3000042,1,0,1,0
6bba_a90a0b9c,1059,2,14.999999999999996,27.999999999999996,63.999999999999986,cont,1.149048519428146,3000046,4,0,0,4
6bba_a90a0b9c,1073,2,17.64,7.840000000000001,86.88,cont,2.4045607707022096,3000049,2,0,2,0
6bba_a90a0b9c,1656,3,43.0,11.999999999999996,55.99999999999999,cont,2.872621298570352,4000077,3,0,0,3
6bba_a90a0b9c,1865,4,10.68,21.28,240.64000000000004,cont,0.6142843905716586,5000083,1,1,0,0
6bba_a90a0b9c,2139,4,56.86666666666666,103.99999999999999,100.0,cont,1.1692976286833319,5000101,4,0,0,4
6bba_a90a0b9c,2301,5,14.319999999999999,27.36,64.64,cont,1.8374719385612375,6000110,1,1,0,0
6bba_a90a0b9c,2738,6,16.84,11.36,84.64000000000001,cont,5.88752547765188,7000135,1,0,1,0
6bba_a90a0b9c,3121,7,11.866666666666664,100.0,23.46666666666666,cont,3.329488016405663,8000157,3,0,0,3
6bba_a90a0b9c,3329,7,43.86666666666667,15.466666666666665,47.999999999999986,cont,5.8623374918248805,8000168,6,0,0,6
6bba_a90a0b9c,3555,8,11.48,29.92000000000001,216.64000000000004,cont,6.674656592289991,9000178,1,1,0,0
6bba_a90a0b9c,3991,9,11.680000000000003,16.0,210.71999999999997,cont,3.577695006914362,10000200,1,0,1,0
6bba_a90a0b9c,4008,9,15.32,44.00000000000001,71.35999999999999,cont,6.559245621639734,10000202,1,0,1,0
6bba_a90a0b9c,4357,10,4.7333333333333325,0.0,127.99999999999997,cont,5.16546693059894,11000219,5,0,0,5
6bba_a90a0b9c,4867,11,11.159999999999998,16.640000000000004,212.64,cont,3.5830420908775182,12000241,1,0,1,0
6bba_a90a0b9c,4934,11,18.519999999999996,12.0,81.44,cont,6.426972581433649,12000262,2,1,1,0
6bba_a90a0b9c,4996,11,27.733333333333327,193.06666666666672,75.99999999999999,cont,4.450410600595081,12000251,5,0,0,5
6bba_a90a0b9c,5063,11,38.31999999999999,15.36,111.36,cont,6.671629415854861,12000253,1,1,0,0
6bba_a90a0b9c,5304,12,9.160000000000002,101.28,28.640000000000004,cont,3.1535468622647698,13000265,7,0,2,5
6bba_a90a0b9c,5357,12,17.48,12.0,81.28,cont,5.826888374166782,13000285,1,0,1,0
6bba_a90a0b9c,6273,14,18.48,59.199999999999996,17.44,cont,3.1697914955088127,15000311,1,1,0,0
6bba_a90a0b9c,6564,15,2.8000000000000003,2.72,131.2,cont,4.284703169415592,16000322,1,1,0,0
6bba_a90a0b9c,6694,15,15.119999999999997,12.640000000000004,84.64000000000001,cont,5.519764063118999,16000342,1,1,0,0
6bba_a90a0b9c,6771,15,24.999999999999993,40.96,32.959999999999994,cont,5.90843385551534,16000331,2,2,0,0
6bba_a90a0b9c,7074,16,4.999999999999998,15.999999999999996,215.46666666666664,cont,6.225279584306053,17000345,3,0,0,3
6bba_a90a0b9c,7242,16,23.0,20.000000000000004,43.46666666666665,cont,6.76156448910145,17000351,4,0,0,4
6bba_a90a0b9c,8009,18,2.84,58.56,49.28,cont,4.209377366369045,19000381,4,0,1,3
6bba_a90a0b9c,8514,19,8.68,15.36,97.28000000000003,cont,1.3230534050067648,20000405,1,1,0,0
6bba_a90a0b9c,8566,19,16.000000000000004,28.0,47.040000000000006,cont,4.072270750146656,20000408,2,1,1,0
6bba_a90a0b9c,8936,20,7.133333333333333,3.999999999999999,91.99999999999999,cont,5.46580317430822,21000424,3,0,0,3
6bba_a90a0b9c,9722,22,2.4799999999999995,55.2,74.72,cont,6.984721831433233,23000457,5,0,2,3
6bba_a90a0b9c,10569,24,3.0,27.360000000000003,116.63999999999999,cont,6.961698001385863,25000493,1,1,0,0
6bba_a90a0b9c,10583,24,2.999999999999999,103.99999999999999,36.0,cont,6.319766856854773,25000491,3,1,0,2
6bba_a90a0b9c,10979,25,2.999999999999999,28.53333333333333,95.46666666666667,cont,6.130233379303127,26000509,6,2,0,4
6bba_a90a0b9c,11970,27,25.680000000000003,26.719999999999995,125.28000000000002,cont,4.607584270797446,28000559,1,1,0,0
6bba_a90a0b9c,13777,32,1.8400000000000003,25.92,110.71999999999998,cont,2.086337999941523,33000637,3,2,1,0
6bba_a90a0b9c,13805,32,4.640000000000001,40.63999999999999,111.36,cont,5.811687039922227,33000637,7,2,0,5
6bba_a90a0b9c,13867,32,11.479999999999999,28.640000000000004,37.28,cont,6.586945066758944,33000641,6,0,0,6
6bba_a90a0b9c,14152,33,3.8,41.28,112.00000000000001,cont,5.910042565413553,34000657,2,0,2,0
6bba_a90a0b9c,14241,33,11.36,72.64,57.28000000000001,cont,5.919574124250834,34000664,1,1,0,0
6bba_a90a0b9c,14336,33,24.680000000000003,4.6400000000000015,218.08,cont,3.99065725319777,34000670,1,1,0,0
6bba_a90a0b9c,15059,35,19.480000000000004,37.279999999999994,127.36,cont,4.688174284836269,36000705,1,0,1,0
6bba_a90a0b9c,15698,37,7.6800000000000015,40.000000000000014,48.800000000000004,cont,6.799299826085039,38000738,9,0,1,8
6bba_a90a0b9c,15832,37,25.999999999999996,29.066666666666656,61.06666666666666,cont,2.0152984769976183,38000748,8,1,0,7
6bba_a90a0b9c,16492,39,10.160000000000002,9.280000000000001,150.72,cont,5.685270991782185,40000782,1,0,1,0
6bba_a90a0b9c,18046,43,22.13333333333333,48.53333333333333,61.59999999999998,cont,3.680145679483662,44000856,4,0,0,4
6bba_a90a0b9c,18390,44,16.133333333333333,79.99999999999999,52.53333333333333,cont,2.566208360056174,45000866,7,0,0,7
6bba_a90a0b9c,19185,46,20.040000000000003,25.28,81.92,cont,4.135585093127698,47000897,1,1,0,0
6bba_a90a0b9c,21622,53,2.3600000000000003,11.68,40.15999999999999,cont,4.139797134220474,54000999,3,3,0,0
6bba_a90a0b9c,22093,54,18.439999999999998,60.0,66.23999999999998,cont,3.052968074268058,55001025,2,0,2,0
6bba_a90a0b9c,24649,61,8.88,20.48,43.36,cont,6.958036720225038,62001122,6,0,2,4
6bba_a90a0b9c,24669,61,12.999999999999998,7.999999999999998,43.999999999999986,cont,6.894291116568837,62001122,3,1,0,2
6bba_a90a0b9c,24755,61,25.999999999999996,11.999999999999996,67.99999999999997,cont,1.723572779142216,62001134,5,0,0,5
6bba_a90a0b9c,25792,64,22.000000000000004,5.4399999999999995,60.15999999999998,cont,3.613095322919116,65001175,3,3,0,0
6bba_a90a0b9c,25813,64,25.040000000000003,16.479999999999997,71.83999999999999,cont,3.432790865316437,65001176,5,5,0,0
6bba_a90a0b9c,25957,65,1.0,6.56,40.16,cont,6.519511340967204,66001175,1,0,1,0
6bba_a90a0b9c,26046,65,5.960000000000001,23.200000000000003,41.28000000000001,cont,6.826740849409769,66001175,11,0,6,5
6bba_a90a0b9c,26083,65,9.52,60.000000000000014,47.199999999999996,cont,6.342789459299433,66001179,6,0,1,5
6bba_a90a0b9c,27319,68,15.84,16.0,44.00000000000001,cont,6.924502351252397,69001212,1,1,0,0
6bba_a90a0b9c,27721,69,16.866666666666667,12.533333333333331,63.999999999999986,cont,5.050733066980361,70001220,4,0,0,4
6bba_a90a0b9c,29420,74,2.16,63.36,52.0,cont,5.793575105235109,75001276,2,2,0,0
6bba_a90a0b9c,29857,75,12.96,16.640000000000004,62.71999999999999,cont,4.768786697106505,76001297,2,0,2,0
6bba_a90a0b9c,30543,77,12.200000000000001,17.919999999999998,63.36000000000001,cont,5.6912130736777025,78001314,2,2,0,0
6bba_a90a0b9c,30814,78,6.760000000000002,44.64000000000001,178.71999999999997,cont,2.761591762462356,79001324,5,0,0,5
6bba_a90a0b9c,30862,78,11.320000000000002,20.799999999999997,48.0,cont,1.4212016262656029,79001326,2,0,2,0
6bba_a90a0b9c,31171,79,5.800000000000002,45.28,177.44000000000003,cont,2.949498103491512,80001334,1,1,0,0
6bba_a90a0b9c,32199,82,4.680000000000001,48.0,173.91999999999996,cont,2.318289037738818,83001362,1,1,0,0
6bba_a90a0b9c,32576,83,9.52,24.8,61.44000000000001,cont,6.168582191233254,84001376,1,1,0,0
6bba_a90a0b9c,32822,84,3.3200000000000003,82.71999999999998,73.92,cont,6.928714933160407,85001384,1,1,0,0
6bba_a90a0b9c,32879,84,7.840000000000002,20.000000000000004,41.92000000000001,cont,2.2986714913184048,85001388,3,0,3,0
6bba_a90a0b9c,33502,86,7.160000000000002,23.360000000000007,41.44,cont,3.5204041405071633,87001410,1,0,1,0
6bba_a90a0b9c,33507,86,8.133333333333331,10.933333333333337,37.066666666666656,cont,6.259766327840577,87001410,7,0,0,7
6bba_a90a0b9c,33811,87,6.999999999999999,20.799999999999997,41.43999999999999,cont,1.3712727892363383,88001431,3,3,0,0
6bba_a90a0b9c,34089,88,4.319999999999999,14.560000000000002,47.84,cont,6.567915503605388,89001443,3,1,2,0
6bba_a90a0b9c,34420,89,3.1599999999999993,35.36,28.640000000000008,cont,6.876899396712154,90001454,1,0,1,0
6bba_a90a0b9c,34460,89,6.48,17.279999999999998,28.000000000000004,cont,5.718822677133817,90001454,1,0,1,0
6bba_ab78413d,478,2,18.880000000000003,233.92000000000007,169.28,cont,2.297924675114472,3000056,1,1,0,0
6bba_ab78413d,689,3,18.16,234.0800000000001,170.72000000000003,cont,1.9216269148822753,4000077,1,0,1,0
6bba_ab78413d,845,3,58.999999999999986,27.999999999999996,108.64000000000001,cont,6.065104337519697,4000095,1,1,0,0
6bba_ab78413d,1023,4,50.79999999999999,188.00000000000003,252.0,cont,0.6600781203766715,5000111,1,1,0,0
6bba_ab78413d,1081,4,61.99999999999999,31.999999999999993,124.53333333333335,cont,1.491797596674712,5000115,4,0,0,4
6bba_ab78413d,1147,5,28.733333333333327,193.60000000000002,206.9333333333333,cont,5.65600904950262,6000125,6,0,0,6
6bba_ab78413d,1157,5,30.200000000000003,188.00000000000003,204.00000000000003,cont,6.571211479818611,6000125,2,0,2,0
6bba_ab78413d,1158,5,30.866666666666664,215.99999999999994,196.5333333333333,cont,5.4981834910621705,6000125,4,0,0,4
6bba_ab78413d,1222,5,51.39999999999999,187.35999999999999,252.0,cont,1.1808954494365804,6000133,1,0,1,0
6bba_ab78413d,1728,8,27.64,196.64000000000001,196.64000000000001,cont,2.373445873092539,9000193,1,1,0,0
6bba_ab78413d,1948,9,28.36,196.00000000000003,195.36,cont,1.0118153549437665,10000213,1,0,1,0
6bba_ab78413d,2290,10,61.48,27.999999999999996,126.72000000000001,cont,1.2081959692036708,11000242,2,0,0,2
6bba_ab78413d,2333,11,12.680000000000003,202.72,184.64,cont,3.9304850209611475,12000249,1,1,0,0
6bba_ab78413d,2340,11,19.080000000000002,244.64000000000001,168.64,cont,3.8788174744372843,12000252,1,1,0,0
6bba_ab78413d,2543,12,14.68,244.0,171.36,cont,1.135176197777242,13000275,2,0,0,2
6bba_ab78413d,4109,19,50.86666666666666,72.0,175.4666666666667,cont,1.6536365649346565,20000430,3,0,0,3
6bba_ab78413d,4543,21,58.359999999999985,22.88,142.72,cont,3.38253810954143,22000471,1,0,1,0
6bba_ab78413d,4605,22,14.159999999999998,6.08,200.0,cont,0.9313674422589608,23000477,1,0,1,0
6bba_ab78413d,5237,25,21.68,212.16,203.99999999999997,cont,2.8490842827126106,26000535,1,0,1,0
6bba_ab78413d,5543,26,55.16000000000001,111.36,171.36,cont,3.54413952885607,27000562,2,2,0,0
6bba_ab78413d,5738,27,55.0,113.28000000000002,171.84,cont,4.346381397208489,28000580,2,0,2,0
6bba_ab78413d,6092,29,45.99999999999999,72.53333333333332,191.99999999999994,cont,5.565542629155247,30000613,2,0,0,2
6bba_ab78413d,6143,29,59.68000000000001,130.72,66.08000000000001,cont,1.1435197855743637,30000618,1,1,0,0
6bba_ab78413d,6363,30,59.160000000000004,27.360000000000003,150.72,cont,1.7301481150468034,31000638,2,1,1,0
6bba_ab78413d,6581,31,58.999999999999986,27.999999999999996,152.64,cont,1.5885154232805003,32000661,1,0,1,0
6bba_ab78413d,6667,32,20.0,203.84000000000003,206.72,cont,2.696716753758157,33000674,1,0,1,0
6bba_ab78413d,6985,33,47.84,72.64,194.07999999999998,cont,3.8571421172287663,34000697,1,1,0,0
6bba_ab78413d,7675,36,58.680000000000014,27.360000000000003,160.64000000000001,cont,2.1349432808625055,37000758,1,1,0,0
6bba_ab78413d,7777,37,23.0,223.99999999999997,206.55999999999997,cont,6.802153726026776,38000766,2,1,1,0
6bba_ab78413d,7882,37,58.519999999999996,26.560000000000002,161.12000000000003,cont,2.372110413218564,38000775,1,0,1,0
6bba_ab78413d,7965,38,18.04,210.87999999999994,213.28000000000003,cont,4.042103799384691,39000784,3,0,0,3
6bba_ab78413d,8258,39,57.64,26.719999999999995,165.44,cont,2.612209613047925,40000812,2,2,0,0
6bba_ab78413d,8440,40,57.120000000000005,27.360000000000003,168.00000000000003,cont,1.896800348613407,41000829,2,0,2,0
6bba_ab78413d,8609,41,47.999999999999986,79.99999999999997,191.46666666666664,cont,3.89416519544877,42000844,2,0,0,2
6bba_ab78413d,9020,43,47.84,80.64,194.71999999999997,cont,5.961469250528777,44000881,2,0,2,0
6bba_ab78413d,9057,43,58.0,98.56,110.56,cont,1.9118454762349393,44000885,2,1,1,0
6bba_ab78413d,9666,46,47.00000000000001,130.56,133.92000000000002,cont,3.2550741639016603,47000938,1,1,0,0
6bba_ab78413d,9708,46,58.839999999999996,92.63999999999999,111.36000000000001,cont,1.397216541020049,47000945,1,0,1,0
6bba_ab78413d,9921,47,60.133333333333326,16.533333333333335,179.99999999999997,cont,6.226398841603549,48000961,2,0,0,2
6bba_ab78413d,10829,52,56.84,12.8,179.36,cont,3.603032143209387,53001046,1,1,0,0
6bba_ab78413d,19359,52,58.46224456787109,92.39074462890625,115.47155212402346,cont,2.581261672394486,53001049,1,1,0,0
6bba_ab78413d,11021,53,58.23844299316406,92.25473999023438,113.45597778320311,cont,1.9699729577386145,54001064,1,0,1,0
6bba_ab78413d,11194,54,58.051041259765626,92.10880371093748,112.58845947265624,cont,0.5810892232303021,55001082,1,1,0,0
6bba_ab78413d,11359,55,58.0,92.0,111.99999999999999,cont,1.218750000000007,56001097,1,0,1,0
6bba_ab78413d,11601,57,17.160000000000004,190.72000000000003,207.36,cont,5.624143545909882,58001131,2,0,0,2
6bba_ab78413d,11688,57,54.68000000000001,12.16,173.91999999999996,cont,6.515601588686978,58001126,1,0,1,0
6bba_ab78413d,11931,59,4.319999999999999,34.720000000000006,243.36000000000004,cont,3.6337558049489163,60001151,1,1,0,0
6bba_ab78413d,12055,59,55.99999999999999,87.99999999999997,128.53333333333333,cont,6.501918119763679,60001165,2,0,0,2
6bba_ab78413d,12090,59,59.08,109.44000000000001,238.56,cont,3.858511087518093,60001167,1,0,1,0
6bba_ab78413d,12103,60,4.160000000000001,33.28,244.00000000000003,cont,4.408664232224991,61001170,2,1,1,0
6bba_ab78413d,12143,60,27.32,116.0,134.72,cont,1.287036931288297,61001174,1,1,0,0
6bba_ab78413d,12235,60,58.480000000000004,108.63999999999999,239.35999999999996,cont,4.7360038864532115,61001185,1,1,0,0
6bba_ab78413d,12276,61,4.0,32.63999999999999,244.00000000000003,cont,4.770447600854661,62001188,1,0,1,0
6bba_ab78413d,12743,63,58.31999999999999,108.64000000000001,238.07999999999998,cont,4.7573661633828594,64001239,1,1,0,0
6bba_ab78413d,12807,64,15.159999999999997,197.44,236.64000000000001,cont,2.380389280138859,65001242,4,2,1,1
6bba_ab78413d,12894,64,47.120000000000005,107.36,193.92000000000002,cont,4.639197128814429,65001248,2,2,0,0
6bba_ab78413d,12956,64,58.680000000000014,90.56,241.43999999999997,cont,4.977277117059084,65001253,2,0,2,0
6bba_ab78413d,12957,64,58.520000000000024,109.44000000000001,236.0,cont,6.782306945280505,65001253,4,0,1,3
6bba_ab78413d,13019,65,19.64,200.64,222.72000000000003,cont,5.947943972500078,66001274,5,1,0,4
6bba_ab78413d,13091,65,47.0,107.36,193.27999999999997,cont,4.894056503811547,66001264,5,0,2,3
6bba_ab78413d,13207,66,18.120000000000005,197.92,223.36000000000004,cont,4.1375001888217415,67001291,5,0,1,4
6bba_ab78413d,13378,67,11.600000000000001,198.72,239.35999999999996,parent,0.8720665112249204,68001291,1,1,0,0
6bba_ab78413d,13492,67,51.16,79.36,124.00000000000003,cont,2.788047311739884,68001301,1,1,0,0
6bba_ab78413d,13545,67,59.68000000000001,83.99999999999999,74.72,cont,1.981767282629335,68001303,1,1,0,0
6bba_ab78413d,13585,68,11.479999999999999,197.92000000000002,240.00000000000006,daughter,1.4744580572196666,69001319,1,0,1,0
6bba_ab78413d,13586,68,10.466666666666663,215.46666666666664,243.4666666666667,daughter,5.376024611643814,69001320,2,0,0,2
6bba_ab78413d,13962,69,59.68000000000001,100.64,225.28000000000003,cont,6.028487191037233,70001333,1,0,0,1
6bba_ab78413d,14015,70,17.373055419921876,189.90315917968752,221.2386572265625,cont,5.906208191341445,71001339,1,1,0,0
6bba_ab78413d,14073,70,40.32,106.55999999999999,146.72,cont,6.083861977601075,71001341,3,0,1,2
6bba_ab78413d,14194,71,4.0548658752441415,216.28572265625002,229.06017333984374,cont,4.292367005413673,72001354,1,0,1,0
6bba_ab78413d,14614,73,16.999999999999993,199.4666666666667,225.06666666666666,cont,6.68387914697661,74001390,4,0,0,4
6bba_ab78413d,14886,74,44.16000000000001,97.92000000000002,201.92,cont,6.300181793012007,75001418,1,1,0,0
6bba_ab78413d,15742,79,43.466666666666654,107.99999999999997,144.0,cont,4.604883522625115,80001502,2,0,0,2
6bba_ab78413d,15970,80,59.31999999999999,79.36,85.44,cont,6.21816818886398,81001529,1,1,0,0
6bba_ab78413d,15979,80,60.400000000000006,52.95999999999999,91.03999999999998,cont,6.7197630631593,81001529,1,1,0,0
6bba_ab78413d,16741,85,53.39999999999999,233.06666666666666,51.99999999999999,cont,4.009979954577211,86001606,1,0,0,1
6bba_ab78413d,16784,86,3.6800000000000006,204.80000000000004,240.64000000000004,cont,4.022654596159102,87001612,2,2,0,0
6bba_ab78413d,17583,90,45.0,93.44,193.27999999999997,cont,1.392673350969266,91001688,1,1,0,0
6bba_ab78413d,17640,90,61.680000000000014,88.64,232.0,cont,3.05223278928723,91001694,1,0,1,0
6bba_ab78413d,17751,91,44.84,92.80000000000001,193.27999999999997,cont,1.6172628303711039,92001705,1,0,1,0
6bba_ab78413d,17912,92,44.52,91.52000000000001,193.92000000000002,cont,2.30824371817622,93001722,1,1,0,0
6bba_ab78413d,18118,93,58.16,51.2,107.36,cont,3.7458629262827996,94001746,2,1,1,0
6bba_ab78413d,18288,94,55.86666666666666,52.53333333333331,112.5333333333333,cont,2.3450163245700315,95001759,3,0,0,3
6bba_acd782a8,7151,30,60.839999999999996,92.0,242.71999999999997,cont,1.8884289932375062,31000033,1,1,0,0
6bba_acd782a8,7339,31,47.00000000000001,122.72,246.24000000000004,cont,0.5825120706903927,32000036,1,0,1,0
6bba_acd782a8,7390,31,60.16000000000001,92.0,237.27999999999997,cont,0.49555839716425537,32000037,1,0,1,0
6bba_acd782a8,7576,32,46.0,123.35999999999999,242.71999999999997,cont,1.712429286861215,33000040,1,1,0,0
6bba_acd782a8,7756,33,25.64,153.92000000000002,237.44,cont,2.470908550007471,34000044,1,0,1,0
6bba_acd782a8,8115,34,58.999999999999986,94.72,222.07999999999996,cont,1.8060908234361075,35000052,2,1,1,0
6bba_acd782a8,8347,35,58.52000000000001,95.99999999999997,215.35999999999996,cont,0.89153133708243,36000062,2,1,1,0
6bba_acd782a8,8348,35,59.64,103.36,240.00000000000006,cont,1.359766753895654,36000063,2,1,1,0
6bba_acd782a8,8825,37,58.2,104.64,230.08000000000004,cont,2.9602219616947596,38000079,1,1,0,0
6bba_acd782a8,9303,39,54.480000000000004,105.28000000000002,194.40000000000003,cont,2.8865141715397913,40000094,1,1,0,0
6bba_acd782a8,9481,40,34.52000000000001,129.28,181.60000000000002,cont,1.9720832893161377,41000100,3,0,0,3
6bba_acd782a8,9534,40,50.52,108.0,167.51999999999998,cont,1.8075522986901438,41000102,2,0,0,2
6bba_acd782a8,9540,40,53.31999999999999,108.8,195.51999999999998,cont,2.494679338111429,41000104,2,0,1,1
6bba_acd782a8,10185,43,30.26666666666666,135.46666666666664,176.53333333333336,cont,1.3579530398115258,44000126,8,2,0,6
6bba_acd782a8,10235,43,46.0,109.28,153.28000000000003,cont,1.7310636181261498,44000127,2,2,0,0
6bba_acd782a8,10244,43,50.6,110.88,186.55999999999997,cont,3.877251362112084,44000129,1,1,0,0
6bba_acd782a8,10361,44,13.319999999999999,149.27999999999994,216.48000000000002,cont,1.1394715057867857,45000133,1,0,1,0
6bba_acd782a8,10410,44,25.519999999999996,143.20000000000002,162.23999999999998,cont,3.52628744219754,45000135,5,0,1,4
6bba_acd782a8,10428,44,30.360000000000003,133.28000000000006,175.04,cont,6.773463155949102,45000135,6,0,2,4
6bba_acd782a8,10465,44,43.8,110.71999999999998,147.84,cont,2.0961870145576236,45000136,2,0,2,0
6bba_acd782a8,10471,44,43.199999999999996,121.28000000000003,174.08,cont,4.6455677801534785,45000138,2,0,0,2
6bba_acd782a8,10521,44,59.31999999999999,79.36,183.2,cont,2.0152620797305816,45000139,1,0,1,0
6bba_acd782a8,11047,47,3.1599999999999993,164.0,181.28,cont,0.5813776741499412,48000156,1,1,0,0
6bba_acd782a8,11171,47,38.480000000000004,121.28000000000003,165.92,cont,5.72756601882509,48000161,1,0,1,0
6bba_acd782a8,11248,47,60.32000000000001,74.07999999999998,178.56000000000003,cont,4.713872644917353,48000162,1,1,0,0
6bba_acd782a8,11300,48,4.0,153.27999999999997,201.92000000000004,cont,3.5323852157571918,49000165,2,0,0,2
6bba_acd782a8,11420,48,38.16,121.43999999999998,163.20000000000002,cont,3.322399829189749,49000170,3,0,0,3
6bba_acd782a8,11664,49,37.84,112.64,128.48000000000002,cont,1.9955767493383894,50000178,3,3,0,0
6bba_acd782a8,11739,49,59.160000000000004,77.28,171.99999999999994,cont,3.8215149564668507,50000180,2,2,0,0
6bba_acd782a8,11882,50,28.8,117.92000000000002,95.19999999999999,cont,0.5019119693731192,51000185,3,0,1,2
6bba_acd782a8,12526,53,4.866666666666665,148.0,132.5333333333333,cont,3.722653180379407,54000212,4,0,0,4
6bba_acd782a8,13977,59,18.68,121.28000000000003,61.44000000000001,cont,2.849269643259475,60000249,1,1,0,0
6bba_acd782a8,14221,60,17.32,120.63999999999999,55.2,cont,1.428706801621664,61000256,1,0,1,0
6bba_acd782a8,14313,60,39.88,103.36,97.44,cont,6.954278584080456,61000258,3,0,0,3
6bba_acd782a8,14526,61,34.760000000000005,100.00000000000001,78.72,cont,3.960468722260023,62000263,1,1,0,0
6bba_acd782a8,14761,62,33.32,100.0,72.0,cont,2.84834271989871,63000269,2,1,1,0
6bba_acd782a8,14992,63,32.16000000000001,100.0,65.92,cont,1.4723970677096407,64000275,1,0,1,0
6bba_acd782a8,17014,72,21.000000000000004,109.28,26.720000000000002,cont,1.8931773061443489,73000324,3,3,0,0
6bba_acd782a8,17026,72,22.72,121.28000000000003,41.440000000000005,cont,6.92160352176431,73000324,2,2,0,0
6bba_acd782a8,17240,73,19.32,110.71999999999998,21.28,cont,1.1487037531496107,74000331,3,0,3,0
6bba_acd782a8,17243,73,20.320000000000004,122.71999999999998,34.559999999999995,cont,6.9982431947239405,74000331,2,0,2,0
6bba_acd782a8,17550,74,45.13333333333333,111.99999999999999,224.53333333333325,cont,1.4249025307796317,75000340,5,1,0,4
6bba_acd782a8,17558,74,46.36,107.36000000000001,231.04000000000002,cont,3.4749172652021567,75000340,4,1,0,3
6bba_acd782a8,17982,76,44.68000000000001,108.64000000000001,224.00000000000003,cont,6.77266391920493,77000350,1,1,0,0
6bba_acd782a8,18180,77,40.0,113.92,204.00000000000003,cont,0.5979470085216477,78000356,1,1,0,0
6bba_acd782a8,18393,78,41.84000000000001,110.71999999999998,206.08,cont,6.842079819031654,79000360,2,0,2,0
6bba_acd782a8,18403,78,45.866666666666674,103.99999999999999,215.99999999999994,cont,1.9292004136717094,79000361,5,0,0,5
6bba_acd782a8,18414,78,48.88,97.92000000000002,222.08000000000004,cont,4.675992207542699,79000361,5,1,1,3
6bba_acd782a8,18619,79,48.2,97.92000000000002,214.08,cont,6.788786365212855,80000366,1,0,1,0
6bba_acd782a8,18816,80,47.00000000000001,99.04000000000002,244.64000000000004,cont,2.0478868682131606,81000374,1,1,0,0
6bba_acd782a8,18849,80,59.16000000000001,69.44,216.64,cont,1.5120360858788884,81000376,2,0,2,0
6bba_acd782a8,19012,81,46.0,101.28,238.08000000000004,cont,3.284470324496781,82000381,1,0,1,0
6bba_acd782a8,19022,81,49.733333333333334,95.46666666666667,252.00000000000006,cont,6.547504361820299,82000381,3,0,0,3
6bba_acd782a8,20281,88,32.87999999999999,105.92,105.12000000000002,cont,3.8482662492348636,89000427,2,2,0,0
6bba_acd782a8,20288,88,38.8,103.36,150.88,cont,5.54086875521339,89000428,3,3,0,0
6bba_acd782a8,20326,88,52.2,78.56,157.44000000000003,cont,4.953372115286711,89000431,2,2,0,0
6bba_acd782a8,20447,89,30.599999999999994,108.0,107.36000000000001,cont,1.2355344238425716,90000438,3,1,2,0
6bba_acd782a8,20457,89,32.84,114.72,151.04,cont,3.7789898203752776,90000439,4,2,2,0
6bba_acd782a8,20489,89,47.160000000000004,83.99999999999999,127.52,cont,1.8828274117666786,90000443,1,0,1,0
6bba_acd782a8,20587,90,15.040000000000003,134.72,91.52000000000001,cont,4.164694616655576,91000446,4,0,2,2
6bba_acd782a8,20594,90,18.599999999999994,131.99999999999997,109.91999999999999,cont,5.731529573333801,91000446,5,0,1,4
6bba_acd782a8,20615,90,24.760000000000005,109.44,88.32,cont,6.973995671421081,91000449,4,0,1,3
6bba_acd782a8,20620,90,29.68,110.72000000000003,109.6,cont,5.182145127743911,91000449,5,0,1,4
6bba_acd782a8,20623,90,31.320000000000004,118.07999999999998,148.8,cont,2.177318089416426,91000450,2,0,2,0
6bba_acd782a8,20672,90,53.68,73.27999999999999,168.00000000000003,cont,2.351931387179472,91000456,1,0,1,0
6bba_acd782a8,20822,91,51.56,83.99999999999999,241.28000000000003,cont,1.2882673684061092,92000470,1,1,0,0
6bba_acd782a8,20972,92,51.15999999999999,84.0,235.99999999999994,cont,1.639640299730435,93000484,1,0,1,0
6bba_acd782a8,21262,94,49.23999999999999,84.64000000000001,223.36000000000004,cont,2.948065308384475,95000511,1,0,1,0
6bba_acd782a8,21354,95,25.640000000000008,118.87999999999998,120.64000000000001,cont,2.046532509514579,96000518,1,1,0,0
6bba_acd782a8,21377,95,37.20000000000001,86.40000000000002,84.48000000000002,cont,4.4870013650543745,96000521,2,2,0,0
6bba_acd782a8,21378,95,39.96000000000001,92.64,93.11999999999999,cont,2.6580549749770124,96000521,3,3,0,0
6bba_acd782a8,21416,95,60.36000000000001,48.0,174.08,cont,1.881073496703415,96000525,1,1,0,0
6bba_acd782a8,21467,96,13.24,107.36,22.72,cont,6.142972865396687,97000529,4,0,4,0
6bba_acd782a8,21481,96,17.44,119.84,56.00000000000001,cont,1.357337121904507,97000530,8,1,1,6
6bba_acd782a8,21487,96,19.04,127.84000000000002,102.88000000000001,cont,5.467732619651407,97000531,5,1,1,3
6bba_acd782a8,21516,96,34.84,94.07999999999998,80.0,cont,0.6101869590543553,97000534,4,0,2,2
6bba_acd782a8,21553,96,59.48,49.28,166.07999999999998,cont,2.6840553505842624,97000538,1,0,1,0
6bba_acd782a8,21626,97,18.240000000000002,120.0,56.8,cont,6.586784709742681,98000544,3,0,1,2
6bba_acd782a8,21628,97,19.72,126.56,96.48000000000002,cont,4.176250000000001,98000545,1,0,1,0
6bba_acd782a8,21652,97,34.4,91.52000000000001,62.71999999999999,cont,4.864643165228051,98000548,3,0,1,2
6bba_ae82a791,166,2,22.839999999999996,131.35999999999999,141.28000000000003,cont,0.5994906692351532,3000031,1,1,0,0
6bba_ae82a791,233,3,22.16,134.72000000000003,136.64,cont,1.306180620167056,4000044,1,0,1,0
6bba_ae82a791,296,4,20.000000000000007,171.99999999999994,52.0,cont,1.4647552056572615,5000064,1,1,0,0
6bba_ae82a791,364,5,20.000000000000007,171.99999999999994,52.64,cont,2.2461473960984883,6000076,1,0,1,0
6bba_ae82a791,901,13,25.320000000000007,100.00000000000001,135.35999999999999,cont,0.6758906161502787,14000183,1,1,0,0
6bba_ae82a791,967,14,25.480000000000004,98.72000000000001,130.72,cont,0.9986811615826087,15000200,1,0,1,0
6bba_ae82a791,2388,35,20.640000000000004,114.71999999999997,47.20000000000001,cont,0.6014695545079533,36000576,1,1,0,0
6bba_ae82a791,2466,36,20.520000000000003,111.36000000000001,49.28,cont,1.1500821981493228,37000594,1,0,1,0
6bba_ae82a791,3027,43,23.0,124.00000000000003,233.27999999999997,cont,0.6598780663880204,44000737,1,1,0,0
6bba_ae82a791,3105,44,23.0,124.00000000000003,229.92,cont,0.37375000000000114,45000757,1,0,1,0
6bba_ae82a791,3471,49,18.84,128.0,147.2,cont,1.386592608699467,50000856,1,1,0,0
6bba_ae82a791,3496,49,39.68,4.640000000000001,90.88,cont,2.544253942710905,50000865,1,1,0,0
6bba_ae82a791,3547,50,18.0,127.46666666666667,128.00000000000003,cont,0.21666666666666856,51000875,3,0,0,3
6bba_ae82a791,3571,50,37.84,14.719999999999999,120.79999999999997,cont,3.1886872788813934,51000884,1,0,0,1
6bba_ae82a791,3574,50,39.67999999999999,4.0,82.56000000000002,cont,1.7384462351479348,51000885,1,0,1,0
6bba_ae82a791,3839,54,16.32,220.64000000000001,244.00000000000003,cont,2.8601846531299304,55000960,1,1,0,0
6bba_ae82a791,3916,55,16.68,220.64,244.00000000000003,cont,2.2983842830344923,56000980,1,0,1,0
6bba_ae82a791,4165,58,35.99999999999999,16.0,87.36000000000001,cont,1.6950705184445989,59001052,1,1,0,0
6bba_ae82a791,4244,59,35.99999999999999,16.0,86.72,cont,1.8203626870764098,60001073,1,0,1,0
6bba_ae82a791,4379,61,21.160000000000004,96.63999999999999,36.63999999999999,cont,1.1366872096139826,62001109,2,2,0,0
6bba_ae82a791,4454,62,20.960000000000004,97.44,37.279999999999994,cont,1.8428531717149825,63001129,2,0,2,0
6bba_ae82a791,5055,70,3.6799999999999997,223.99999999999997,35.36,cont,1.290315489521845,71001271,1,1,0,0
6bba_ae82a791,5136,71,4.000000000000001,221.92000000000002,29.920000000000005,cont,1.6835031370627151,72001290,1,0,1,0
6bba_ae82a791,5519,75,38.480000000000004,79.36000000000001,184.64000000000001,cont,2.4288722305835586,76001383,1,1,0,0
6bba_ae82a791,5597,76,38.15999999999999,80.64,186.08,cont,2.30618360338028,77001401,1,0,1,0
6bba_ae82a791,5794,79,13.320000000000004,151.35999999999999,52.16000000000001,cont,3.825520593657815,80001448,1,1,0,0
6bba_ae82a791,5799,79,15.320000000000004,139.36,104.32000000000001,cont,3.5738549302538827,80001450,1,1,0,0
6bba_ae82a791,5807,79,20.8,106.72,36.160000000000004,cont,1.5148277542347843,80001451,1,1,0,0
6bba_ae82a791,5837,79,43.32000000000001,12.0,241.76000000000005,cont,2.2639463608928647,80001461,1,1,0,0
6bba_ae82a791,5877,80,13.840000000000002,152.0,44.480000000000004,cont,2.3562499999999984,81001469,1,0,1,0
6bba_ae82a791,5884,80,15.840000000000002,139.35999999999999,94.39999999999999,cont,1.5102012490062302,81001467,2,0,1,1
6bba_ae82a791,5894,80,21.52,108.0,32.48000000000001,cont,1.9801010075246201,81001470,1,0,1,0
6bba_ae82a791,5923,80,41.999999999999986,16.533333333333335,217.59999999999997,cont,0.6689089894082066,81001480,2,0,0,2
6bba_ae82a791,5938,81,2.8400000000000007,243.35999999999999,1.28,cont,0.9998703040894885,82001484,1,1,0,0
6bba_ae82a791,5968,81,16.32,138.08,89.27999999999999,cont,0.8716121915737557,82001486,1,1,0,0
6bba_ae82a791,6019,82,2.6399999999999997,246.72000000000006,1.2799999999999998,cont,1.2554636842617142,83001503,1,0,1,0
6bba_ae82a791,6056,82,17.0,136.0,82.56000000000002,cont,1.6845223729591714,83001506,1,0,1,0
6bba_ae82a791,6408,86,18.320000000000004,130.56,74.07999999999998,cont,1.6906248844732035,87001581,1,1,0,0
6bba_ae82a791,6490,87,18.84,131.36,69.92,cont,2.0565079625423293,88001599,1,0,1,0
6bba_aeee7805,263,4,56.36000000000001,68.64000000000001,36.64,cont,1.7356338899664208,5000054,1,1,0,0
6bba_aeee7805,317,5,55.64,68.0,35.36,cont,1.3862116766569212,6000065,1,0,1,0
6bba_aeee7805,744,12,60.31999999999999,196.00000000000003,113.28000000000002,cont,2.172461521523454,13000152,1,1,0,0
6bba_aeee7805,761,13,7.160000000000002,114.56000000000002,123.19999999999999,parent,1.838621256539802,14000155,1,1,0,0
6bba_aeee7805,803,13,60.16000000000001,196.00000000000003,112.0,cont,1.3107214139549164,14000164,1,0,1,0
6bba_aeee7805,821,14,4.7333333333333325,103.46666666666665,110.93333333333331,daughter,1.3449268102763066,15000176,1,0,0,1
6bba_aeee7805,823,14,6.999999999999999,116.0,124.00000000000003,daughter,1.7235727791422113,15000177,1,0,1,0
6bba_aeee7805,1903,33,10.000000000000004,85.44,98.72000000000001,cont,1.1223092822390768,34000386,1,1,0,0
6bba_aeee7805,1973,34,10.159999999999998,80.64000000000001,96.0,cont,1.722576649237999,35000395,1,0,1,0
6bba_aeee7805,2102,36,23.319999999999997,113.28000000000002,16.64,cont,1.3230534050067695,37000414,1,1,0,0
6bba_aeee7805,2162,37,24.0,112.64,13.919999999999998,cont,0.55345505689261,38000425,1,0,1,0
6bba_aeee7805,2202,38,6.32,32.0,47.360000000000014,cont,0.9377249730064738,39000430,1,1,0,0
6bba_aeee7805,2258,39,6.84,30.72,45.28,cont,0.6508119928212752,40000438,1,0,1,0
6bba_aeee7805,2726,47,11.360000000000003,63.36,67.36000000000001,cont,2.6816439104586527,48000513,1,1,0,0
6bba_aeee7805,3342,58,45.160000000000004,22.72,31.360000000000007,cont,1.4036275547665724,59000618,1,1,0,0
6bba_aeee7805,3385,59,44.32,21.280000000000005,29.279999999999994,cont,0.9006664199358134,60000625,1,0,1,0
6bba_aeee7805,4477,80,31.680000000000003,142.72000000000003,143.36,cont,2.2179315391823957,81000754,1,1,0,0
6bba_aeee7805,4532,81,31.32,142.08,143.36,cont,1.2358549672190517,82000763,1,0,1,0
6bba_af149c94,215,2,35.32000000000001,227.99999999999997,221.27999999999997,cont,1.3073930405581953,3000036,1,1,0,0
6bba_af149c94,294,3,35.68,227.99999999999997,222.72000000000003,cont,2.3364426084327388,4000048,1,0,1,0
6bba_af149c94,586,7,30.04,184.0,225.44,cont,2.4556863220900245,8000098,1,1,0,0
6bba_af149c94,658,8,28.6,182.72,220.64,cont,3.14805743673778,9000110,1,0,1,0
6bba_af149c94,663,8,31.760000000000005,17.919999999999998,116.0,cont,4.668305701750911,9000111,1,1,0,0
6bba_af149c94,737,9,31.680000000000003,20.0,116.0,cont,1.694525191609735,10000124,1,0,1,0
6bba_af149c94,1006,13,10.040000000000003,232.0,119.52000000000001,cont,1.0882648172664533,14000174,1,1,0,0
6bba_af149c94,1042,13,31.000000000000007,24.639999999999993,113.28000000000002,cont,2.5593105150606443,14000180,1,1,0,0
6bba_af149c94,1082,14,9.999999999999998,227.99999999999997,116.0,cont,1.6750116604071745,15000188,1,0,1,0
6bba_af149c94,1118,14,29.866666666666667,40.533333333333324,119.99999999999997,cont,6.685689656314856,15000194,1,0,0,1
6bba_af149c94,1119,14,31.000000000000007,24.64,112.0,cont,2.0365081941892598,15000194,1,0,1,0
6bba_af149c94,2102,26,27.84,156.64,188.64000000000004,cont,2.562558535916774,27000381,1,1,0,0
6bba_af149c94,2181,27,26.866666666666664,151.99999999999997,200.0,cont,4.478915206119796,28000396,1,0,0,1
6bba_af149c94,2185,27,27.84,153.91999999999996,185.92000000000004,cont,1.7188936303622566,28000396,1,0,1,0
6bba_af149c94,2440,30,27.160000000000004,153.44000000000003,184.0,cont,2.5280111995400607,31000442,1,1,0,0
6bba_af149c94,2472,30,46.68000000000001,12.64,132.64000000000001,cont,0.5994906692351298,31000448,1,1,0,0
6bba_af149c94,2523,31,26.32,153.28000000000003,198.56,cont,5.228966198016588,32000457,1,0,0,1
6bba_af149c94,2525,31,27.0,152.64,184.0,cont,0.9825540697590184,32000457,1,0,1,0
6bba_af149c94,2556,31,46.839999999999996,15.36,135.36,cont,1.9697386406576944,32000462,1,0,1,0
6bba_af149c94,3522,43,24.64,90.24000000000001,129.92000000000002,cont,3.4866002423564475,44000639,2,0,0,2
6bba_af149c94,4056,49,27.0,192.0,146.72000000000003,cont,1.870444883577161,50000746,2,2,0,0
6bba_af149c94,4082,49,44.36000000000001,173.91999999999996,226.72,cont,3.0029103071020846,50000750,1,1,0,0
6bba_af149c94,4140,50,27.160000000000004,192.0,144.64000000000001,cont,3.040617741512399,51000762,2,0,2,0
6bba_af149c94,4168,50,43.68,176.0,223.36000000000004,cont,1.1116710394716691,51000764,1,0,1,0
6bba_af149c94,4312,52,25.13333333333333,107.99999999999997,124.53333333333335,cont,4.723150082454375,53000790,1,0,0,1
6bba_af149c94,4717,57,6.679999999999999,209.11999999999998,60.000000000000014,cont,1.7991326035064659,58000870,2,0,0,2
6bba_af149c94,5030,60,26.13333333333333,204.53333333333325,189.59999999999994,cont,3.973412069270212,61000922,3,0,0,3
6bba_af149c94,5124,61,28.04,70.56,108.0,cont,6.474294483764235,62000936,2,2,0,0
6bba_af149c94,5132,61,32.32,51.36000000000001,123.19999999999999,cont,5.733855732620066,62000936,1,1,0,0
6bba_af149c94,5216,62,26.44000000000001,207.99999999999997,186.55999999999997,cont,2.8866971286402814,63000952,2,0,0,2
6bba_af149c94,5217,62,27.44,74.08000000000001,108.0,cont,4.708015107239996,63000950,2,0,2,0
6bba_af149c94,5224,62,29.39999999999999,55.466666666666654,115.99999999999997,cont,4.846327719120023,63000950,2,0,0,2
6bba_af149c94,5247,62,44.599999999999994,71.36,146.72000000000003,cont,2.6864154788490873,63000955,1,1,0,0
6bba_af149c94,5340,63,45.36,74.07999999999998,146.72,cont,1.3049670733394105,64000972,1,0,1,0
6bba_af149c94,5486,65,25.160000000000004,128.0,126.72000000000001,cont,3.63015689227063,66000998,1,1,0,0
6bba_af149c94,5575,66,25.0,128.0,124.00000000000003,cont,5.281249999999987,67001016,1,0,1,0
6bba_af149c94,5576,66,25.266666666666666,137.06666666666666,136.5333333333333,cont,1.721656633158883,67001016,1,0,0,1
6bba_af149c94,5756,68,25.799999999999997,128.0,125.12000000000003,cont,5.0265849490484,69001051,1,1,0,0
6bba_af149c94,5761,68,26.839999999999996,138.56,76.0,cont,1.988684002676146,69001052,1,1,0,0
6bba_af149c94,5850,69,26.480000000000004,142.72000000000003,76.0,cont,2.0970056777462354,70001069,1,0,1,0
6bba_af149c94,5858,69,26.72,128.0,128.96000000000004,cont,5.654556376277449,70001068,1,0,1,0
6bba_af149c94,6373,74,56.99999999999999,37.279999999999994,175.36,cont,2.0686064904906467,75001162,1,1,0,0
6bba_af149c94,6467,75,57.16000000000001,38.72,175.36,cont,1.8121482417285644,76001177,1,0,1,0
6bba_af149c94,7379,85,59.83999999999999,29.919999999999998,245.92,cont,0.4564485869186265,86001334,1,1,0,0
6bba_af149c94,7411,86,25.13333333333333,119.46666666666667,116.53333333333333,cont,1.1996418736439622,87001339,3,0,0,3
6bba_af149c94,7473,86,59.64,34.080000000000005,245.91999999999996,cont,2.0828544866360645,87001348,1,0,1,0
6bba_af149c94,7699,89,40.519999999999996,48.640000000000015,80.00000000000003,cont,0.8915313370824319,90001384,1,1,0,0
6bba_af149c94,7786,90,40.16000000000001,51.36000000000001,80.64,cont,1.4798210407005132,91001397,1,0,1,0
6bba_afb141ff,210,1,35.28,126.72,85.44,cont,0.9375841628888579,2000011,1,0,1,0
6bba_afb141ff,1116,10,19.04,137.28,96.64000000000001,cont,3.3281174295688554,11000088,1,1,0,0
6bba_afb141ff,1199,11,16.28,138.72000000000003,97.28000000000003,cont,2.103543513336484,12000095,1,0,1,0
6bba_afb141ff,1231,11,39.13333333333332,132.53333333333333,135.99999999999994,cont,1.4779660620446082,12000100,1,0,0,1
6bba_afb141ff,1250,11,57.120000000000005,97.44,70.08,cont,1.3589897442585814,12000103,1,1,0,0
6bba_afb141ff,1334,12,55.48000000000001,100.0,71.35999999999999,cont,1.5888478530054366,13000109,1,0,1,0
6bba_afb141ff,1810,18,41.319999999999986,115.36000000000001,76.0,cont,0.999077699681049,19000148,1,1,0,0
6bba_afb141ff,1882,19,39.32000000000001,116.63999999999999,76.63999999999999,cont,0.9385693900826015,20000154,1,0,1,0
6bba_afb141ff,1936,20,20.000000000000004,145.27999999999997,147.36000000000004,cont,1.9688670600119116,21000157,1,1,0,0
6bba_afb141ff,2001,21,18.479999999999997,147.35999999999996,150.71999999999997,cont,1.2622809215860087,22000165,1,0,1,0
6bba_afb141ff,2072,22,27.360000000000003,150.72,220.0,cont,2.711608436426608,23000172,1,1,0,0
6bba_afb141ff,2079,22,36.28000000000001,122.71999999999998,80.64,cont,2.2451478932355555,23000174,1,1,0,0
6bba_afb141ff,2092,22,54.88,134.07999999999998,233.44,cont,1.993127253840053,23000175,1,1,0,0
6bba_afb141ff,2132,23,25.439999999999998,151.35999999999999,220.0,cont,2.4330000128442233,24000180,1,0,1,0
6bba_afb141ff,2157,23,52.76,135.35999999999999,229.28000000000003,cont,3.017472296890233,24000182,1,0,1,0
6bba_afb141ff,2494,28,27.200000000000006,161.92,251.36000000000007,cont,3.39640549883254,29000232,1,1,0,0
6bba_afb141ff,2552,29,26.320000000000004,157.28,250.72000000000003,cont,0.7441364542205852,30000242,1,0,1,0
6bba_afb141ff,2686,31,29.160000000000004,165.27999999999997,153.60000000000002,cont,3.677134796618142,32000261,1,0,1,0
6bba_afb141ff,2873,34,9.479999999999999,160.00000000000006,209.92000000000002,cont,1.1267708285184004,35000290,1,1,0,0
6bba_afb141ff,2940,35,8.68,160.00000000000006,209.28,cont,1.1911371929799066,36000302,1,0,1,0
6bba_afb141ff,3018,36,21.84,159.36,239.35999999999996,cont,0.9037396264965009,37000315,1,1,0,0
6bba_afb141ff,3077,37,21.160000000000004,160.00000000000006,238.72000000000003,cont,1.2592439447938653,38000327,1,0,1,0
6bba_afb141ff,3136,38,24.680000000000003,164.64,137.92000000000002,cont,0.691150535339442,39000339,1,1,0,0
6bba_afb141ff,3146,38,36.68000000000001,139.35999999999999,168.15999999999997,cont,1.43018464716974,39000341,1,1,0,0
6bba_afb141ff,3154,38,47.2,147.35999999999999,149.28,cont,3.021800994440233,39000344,2,2,0,0
6bba_afb141ff,3161,38,53.88000000000002,96.0,62.07999999999999,cont,3.532198323990297,39000345,1,1,0,0
6bba_afb141ff,3191,39,20.13333333333333,160.53333333333333,120.5333333333333,cont,2.9867894901270513,40000350,2,0,0,2
6bba_afb141ff,3205,39,36.31999999999999,137.92000000000002,163.2,cont,1.4215731822526827,40000352,1,0,1,0
6bba_afb141ff,3210,39,41.480000000000004,142.72,106.72,cont,0.8407706658179749,40000355,1,1,0,0
6bba_afb141ff,3212,39,45.959999999999994,148.00000000000003,152.16000000000005,cont,1.1555795353414482,40000353,2,0,2,0
6bba_afb141ff,3217,39,52.32,95.36000000000001,58.08000000000001,cont,0.728355382007984,40000356,1,0,1,0
6bba_afb141ff,3271,40,41.0,141.28000000000003,103.36,cont,0.5813776741499536,41000364,1,0,1,0
6bba_afb141ff,3393,42,35.36,135.36,151.52,cont,1.4711411387083089,43000381,2,2,0,0
6bba_afb141ff,3411,42,56.160000000000004,126.08000000000001,237.44,cont,2.036508194189262,43000385,1,1,0,0
6bba_afb141ff,3450,43,34.480000000000004,132.64000000000001,143.83999999999997,cont,1.7931048491373702,44000396,2,0,2,0
6bba_afb141ff,3470,43,56.160000000000004,124.64000000000001,233.92000000000007,cont,2.1025390156903314,44000401,1,0,1,0
6bba_afb141ff,3488,44,3.3200000000000003,160.00000000000006,204.64,cont,1.79339935597181,45000403,1,1,0,0
6bba_afb141ff,3518,44,35.52,145.28,170.71999999999997,cont,1.072007462660594,45000411,1,0,1,0
6bba_afb141ff,3553,45,3.1599999999999993,160.00000000000006,204.00000000000003,cont,1.4876517779709222,46000420,1,0,1,0
6bba_afb141ff,3698,47,19.52,154.71999999999997,226.72,cont,1.4490772926245084,48000456,1,1,0,0
6bba_afb141ff,3759,48,19.32,153.27999999999997,225.28000000000003,cont,1.963293231409916,49000471,1,0,1,0
6bba_afb141ff,3916,51,2.6800000000000006,120.00000000000003,30.72,cont,0.9610882503703865,52000497,1,1,0,0
6bba_afb141ff,3970,52,2.160000000000001,120.00000000000003,30.080000000000002,cont,1.6479935717411094,53000507,1,0,1,0
6bba_afb141ff,3981,52,9.800000000000002,140.64,40.8,cont,3.6183536052326377,53000510,1,1,0,0
6bba_afb141ff,4035,53,8.48,139.99999999999997,40.32,cont,1.8717855779976582,54000519,1,0,1,0
6bba_afb141ff,4089,54,5.44,163.36,132.16000000000003,cont,6.146238941417747,55000532,1,1,0,0
6bba_afb141ff,4095,54,10.160000000000002,154.08,86.24,cont,5.606155796309982,55000533,1,1,0,0
6bba_afb141ff,4100,54,17.880000000000003,146.72000000000003,204.80000000000004,cont,3.567346922083675,55000534,2,2,0,0
6bba_afb141ff,4111,54,29.36,124.00000000000003,115.36000000000001,cont,2.6690099405022827,55000535,1,1,0,0
6bba_afb141ff,4119,54,43.199999999999996,78.72,18.080000000000002,cont,3.071550911184782,55000538,1,1,0,0
6bba_afb141ff,4148,55,7.240000000000001,162.72,125.43999999999998,cont,5.02295885161326,56000544,2,0,1,1
6bba_afb141ff,4152,55,10.079999999999998,151.35999999999999,77.44000000000001,cont,4.888955026383453,56000545,1,0,1,0
6bba_afb141ff,4154,55,16.8,145.27999999999997,199.83999999999997,cont,5.5217729546315235,56000548,2,0,2,0
6bba_afb141ff,4164,55,28.32,122.55999999999997,113.28000000000002,cont,1.5749930555402423,56000549,1,0,1,0
6bba_afb141ff,4175,55,41.8,77.28,13.919999999999998,cont,0.345861695624135,56000555,1,0,1,0
6bba_afb141ff,4228,56,37.000000000000014,116.0,25.92,cont,0.5520218519225493,57000568,1,1,0,0
6bba_afb141ff,4288,57,36.68000000000001,114.71999999999997,20.640000000000004,cont,0.8131497402077804,58000580,1,0,1,0
6bba_afb141ff,4402,59,27.0,135.36000000000004,61.27999999999999,cont,2.1422899546980205,60000600,1,1,0,0
6bba_afb141ff,4463,60,26.48,134.72000000000003,56.64,cont,2.7401374281229045,61000614,1,0,1,0
6bba_afb141ff,4522,61,23.319999999999997,120.63999999999999,101.28,cont,1.1204254214359883,62000626,1,1,0,0
6bba_afb141ff,4799,66,12.120000000000001,143.99999999999997,179.36,cont,1.0017172754824522,67000675,1,1,0,0
6bba_afb141ff,4854,67,12.32,143.99999999999997,175.84000000000003,cont,1.8341636377379276,68000684,1,0,1,0
6bba_afb141ff,5518,77,57.24,38.72,0.0,cont,1.5069630055180514,78000777,1,1,0,0
6bba_afb141ff,5574,78,59.133333333333326,52.53333333333331,21.06666666666666,cont,0.28917086932584424,79000783,1,0,0,1
6bba_afb141ff,5822,83,11.84,139.35999999999999,149.44,cont,1.9724180116040313,84000809,1,1,0,0
6bba_afb141ff,5882,84,11.520000000000003,139.35999999999999,146.72000000000003,cont,1.3310128896821265,85000817,1,0,1,0
6bba_afb141ff,5950,85,21.160000000000004,146.08,221.92000000000002,cont,1.4156165838601875,86000830,1,1,0,0
6bba_afb141ff,6011,86,21.0,145.92,217.92,cont,0.7806767897151818,87000839,1,0,1,0
6bba_b1ae37b9,815,2,0.0,139.99999999999994,224.53333333333333,cont,6.060497235055138,3000044,2,0,0,2
6bba_b1ae37b9,853,2,0.9999999999999998,25.066666666666663,107.99999999999997,cont,5.483689514774201,3000037,4,0,0,4
6bba_b1ae37b9,1731,4,3.6799999999999997,74.72,161.28,cont,1.44634128925368,5000067,1,0,1,0
6bba_b1ae37b9,1787,4,8.48,61.280000000000015,39.36,cont,1.0790048945672084,5000073,1,0,1,0
6bba_b1ae37b9,2400,6,0.0,102.08000000000001,96.0,cont,6.404932913387927,7000098,2,1,1,0
6bba_b1ae37b9,3522,8,15.919999999999998,221.60000000000008,68.00000000000001,cont,6.116759890865451,9000146,1,0,1,0
6bba_b1ae37b9,3530,8,15.560000000000002,196.64000000000001,57.279999999999994,cont,6.3070309229065895,9000146,7,0,0,7
6bba_b1ae37b9,3546,8,18.12,210.71999999999997,66.72,cont,6.447319331512905,9000146,1,0,1,0
6bba_b1ae37b9,4045,10,0.9600000000000002,72.64,172.64,cont,6.548124970172762,11000169,1,1,0,0
6bba_b1ae37b9,4347,10,16.52,222.72000000000003,75.36,cont,6.202925343134488,11000182,1,0,1,0
6bba_b1ae37b9,4907,12,0.0,131.35999999999999,137.92000000000002,cont,3.2746874850892267,13000211,1,0,1,0
6bba_b1ae37b9,4952,12,0.9999999999999998,19.999999999999993,131.99999999999997,cont,3.8325548348979925,13000202,4,0,0,4
6bba_b1ae37b9,5012,12,3.4800000000000004,24.640000000000008,132.00000000000003,cont,1.470153584663874,13000202,2,0,2,0
6bba_b1ae37b9,5196,12,15.519999999999998,172.64,69.28,cont,0.8726719028363426,13000220,1,0,1,0
6bba_b1ae37b9,5201,12,17.0,223.99999999999997,80.0,cont,6.906249999999993,13000221,1,1,0,0
6bba_b1ae37b9,5393,13,1.0000000000000002,20.000000000000004,132.32000000000002,cont,3.8698889680454656,14000223,1,1,0,0
6bba_b1ae37b9,5856,14,1.8399999999999999,18.72,244.00000000000006,cont,2.0733879280057446,15000244,1,0,1,0
6bba_b1ae37b9,6704,16,1.0,23.200000000000006,136.64,cont,3.848231939800404,17000286,3,3,0,0
6bba_b1ae37b9,6706,16,1.6400000000000006,22.720000000000006,119.84,cont,6.885878779248148,17000286,3,3,0,0
6bba_b1ae37b9,6806,16,4.360000000000001,32.64,137.28,cont,3.048813549973172,17000286,3,3,0,0
6bba_b1ae37b9,6846,16,8.32,71.51999999999998,45.44,cont,6.662777433248087,17000289,3,3,0,0
6bba_b1ae37b9,6870,16,9.84,182.56000000000003,181.28000000000006,cont,0.6943904701247082,17000303,1,1,0,0
6bba_b1ae37b9,6887,16,11.32,205.92,98.56,cont,6.176047608503353,17000304,3,3,0,0
6bba_b1ae37b9,6888,16,11.160000000000002,226.72,107.36,cont,3.7957231277452297,17000304,2,2,0,0
6bba_b1ae37b9,6963,16,17.16,219.83999999999997,74.56,cont,2.0821570846120023,17000308,3,3,0,0
6bba_b1ae37b9,7149,17,1.0,42.08,149.28000000000003,cont,5.5566446710222595,18000309,3,0,3,0
6bba_b1ae37b9,7200,17,3.4799999999999995,35.84,249.27999999999997,cont,1.397311033735868,18000310,4,0,1,3
6bba_b1ae37b9,7228,17,4.000000000000001,35.36,138.72000000000003,cont,1.7258693461557264,18000309,3,0,3,0
6bba_b1ae37b9,7397,17,17.48,188.48000000000002,71.84,cont,6.081582853172357,18000331,6,0,1,5
6bba_b1ae37b9,7398,17,17.160000000000004,222.72,78.72,cont,3.173080374415367,18000332,3,0,3,0
6bba_b1ae37b9,8033,19,3.1599999999999993,46.08000000000001,235.35999999999999,cont,6.79817346792505,20000355,1,1,0,0
6bba_b1ae37b9,9023,21,17.32,227.99999999999997,88.64,cont,1.6515948731453385,22000423,1,1,0,0
6bba_b1ae37b9,9188,22,1.6400000000000006,36.64,143.36,cont,2.6816439104586545,23000424,6,0,1,5
6bba_b1ae37b9,9356,22,11.160000000000002,237.43999999999994,128.16,cont,1.4229656531343067,23000433,1,0,1,0
6bba_b1ae37b9,9407,22,15.679999999999998,117.28,114.72,cont,2.7479802241828417,23000448,1,1,0,0
6bba_b1ae37b9,9436,22,17.68,227.36,92.64,cont,0.9385693900825937,23000446,1,0,1,0
6bba_b1ae37b9,9547,23,0.0,147.35999999999996,153.92000000000002,cont,3.3385742653114714,24000451,1,0,1,0
6bba_b1ae37b9,9639,23,3.0,31.360000000000007,131.36000000000004,cont,6.68746238307176,24000447,2,2,0,0
6bba_b1ae37b9,9665,23,4.720000000000001,45.919999999999995,144.00000000000003,cont,3.063372834801538,24000447,4,4,0,0
6bba_b1ae37b9,9718,23,7.4799999999999995,147.35999999999999,228.0,cont,1.3869734361190953,24000457,1,1,0,0
6bba_b1ae37b9,9764,23,11.32,235.04000000000002,126.40000000000003,cont,1.7445114824786716,24000460,3,0,0,3
6bba_b1ae37b9,9814,23,16.64,108.0,5.920000000000001,cont,4.5091920007912725,24000461,1,1,0,0
6bba_b1ae37b9,9949,24,0.0,120.64000000000001,204.64000000000001,cont,1.8030910958684225,25000470,1,1,0,0
6bba_b1ae37b9,10113,24,5.199999999999999,49.440000000000005,143.99999999999997,cont,2.0759971248775875,25000465,4,0,4,0
6bba_b1ae37b9,10502,25,3.999999999999999,126.93333333333334,195.99999999999997,cont,6.996521953402603,26000490,6,0,0,6
6bba_b1ae37b9,10911,26,3.5200000000000005,128.64000000000001,140.00000000000003,cont,1.009594596855587,27000514,2,2,0,0
6bba_b1ae37b9,11660,28,1.6800000000000002,72.79999999999998,148.16000000000003,cont,6.443693622449777,29000555,9,0,0,9
6bba_b1ae37b9,12279,29,17.360000000000003,119.36000000000001,27.999999999999996,cont,6.166484259689629,30000584,1,1,0,0
6bba_b1ae37b9,12670,30,15.73333333333333,126.93333333333334,15.999999999999996,cont,0.5757995648564652,31000604,7,0,0,7
6bba_b1ae37b9,12673,30,15.84,140.0,20.640000000000004,cont,5.997372862345646,31000604,1,0,1,0
6bba_b1ae37b9,13255,32,1.64,38.72,191.36,cont,3.680580159771008,33000628,1,0,1,0
6bba_b1ae37b9,13306,32,2.6800000000000006,64.64,169.44000000000005,cont,6.767222703960635,33000629,6,0,0,6
6bba_b1ae37b9,13307,32,3.0,60.800000000000004,192.0,cont,6.530903100835292,33000628,2,0,2,0
6bba_b1ae37b9,15552,38,5.000000000000002,140.64000000000001,220.64000000000004,cont,1.6660807303369258,39000718,2,2,0,0
6bba_b1ae37b9,15832,39,2.480000000000001,152.8,235.35999999999999,cont,6.766422730106355,40000732,3,0,3,0
6bba_b1ae37b9,15894,39,5.160000000000002,144.8,222.72000000000003,cont,1.330616046235724,40000732,2,0,2,0
6bba_b1ae37b9,17170,43,3.519999999999999,173.28000000000003,158.56,cont,1.062357343364279,44000801,3,3,0,0
6bba_b1ae37b9,17431,44,2.9600000000000004,174.55999999999995,162.07999999999998,cont,1.5768362311920636,45000818,3,0,3,0
6bba_b1ae37b9,17462,44,4.279999999999999,157.44000000000003,223.99999999999997,cont,3.8456235271409507,45000817,4,3,1,0
6bba_b1ae37b9,17473,44,3.0400000000000005,108.64,69.12,cont,2.3027459830602255,45000813,7,2,2,3
6bba_b1ae37b9,17480,44,4.48,222.72000000000003,198.07999999999998,cont,6.610415314108485,45000819,5,3,2,0
6bba_b1ae37b9,17497,44,5.999999999999998,223.46666666666664,213.06666666666666,cont,0.6235051567995055,45000819,3,1,0,2
6bba_b1ae37b9,17848,45,9.200000000000001,100.64000000000001,53.279999999999994,cont,6.578842152119171,46000828,3,0,3,0
6bba_b1ae37b9,17856,45,9.999999999999996,228.0,211.99999999999994,cont,0.8125,46000835,4,0,0,4
6bba_b1ae37b9,18009,46,0.0,157.28,215.68,cont,6.83684834646052,47000846,1,0,1,0
6bba_b1ae37b9,18812,48,5.199999999999999,177.28,178.72000000000003,cont,6.6555207346983885,49000880,2,2,0,0
6bba_b1ae37b9,19066,49,2.32,184.64000000000001,168.0,cont,1.6734344362717026,50000898,3,0,3,0
6bba_b1ae37b9,19085,49,2.999999999999999,162.9333333333333,159.99999999999994,cont,4.774738667223321,50000896,4,0,0,4
6bba_b1ae37b9,19396,50,3.080000000000001,165.6,160.00000000000003,cont,4.756811070717452,51000913,2,2,0,0
6bba_b1ae37b9,19433,50,5.160000000000001,169.92000000000002,220.64000000000004,cont,0.7159227000870837,51000914,1,1,0,0
6bba_b1ae37b9,19707,51,4.64,101.28000000000002,121.28000000000003,cont,4.658084537929729,52000925,5,0,0,5
6bba_b1ae37b9,19718,51,5.279999999999999,172.16,145.92000000000002,cont,1.4560761698826061,52000928,4,1,1,2
6bba_b1ae37b9,19945,52,1.9999999999999996,166.93333333333328,155.99999999999997,cont,6.919036366150356,53000957,6,0,0,6
6bba_b1ae37b9,19950,52,2.1600000000000006,192.0,168.0,cont,1.4876517779709069,53000949,1,0,1,0
6bba_b1ae37b9,19991,52,3.84,169.92000000000002,161.27999999999997,cont,6.439471300114612,53000957,1,0,1,0
6bba_b1ae37b9,20198,53,0.6,99.04000000000002,120.0,cont,0.8600227104559477,54000961,1,0,1,0
6bba_b1ae37b9,20558,54,4.5200000000000005,183.99999999999997,221.27999999999997,cont,2.407688531039676,55000988,2,1,1,0
6bba_b1ae37b9,21455,57,13.0,175.36,50.72,cont,1.4425021663761926,58001043,1,1,0,0
6bba_b1ae37b9,22720,62,12.16,181.92000000000002,46.56000000000001,cont,1.929991094539041,63001115,1,1,0,0
6bba_b1ae37b9,25411,76,3.0,213.92000000000002,235.36000000000004,cont,1.195231410648168,77001297,1,1,0,0
6bba_b204cac7,2309,35,39.839999999999996,224.00000000000003,198.72,cont,4.563907317200911,36000413,1,1,0,0
6bba_b204cac7,2374,36,40.68,222.72000000000003,197.28000000000003,cont,3.552660314820443,37000426,1,0,1,0
6bba_b204cac7,2912,44,52.0,122.71999999999998,140.0,cont,2.039294072099462,45000519,1,1,0,0
6bba_b204cac7,2980,45,52.16000000000001,120.64000000000001,136.0,cont,0.9933788615125653,46000530,1,0,1,0
6bba_b204cac7,3384,51,46.133333333333326,235.46666666666673,155.99999999999997,cont,4.6633693708400505,52000591,1,0,0,1
6bba_b204cac7,4699,73,56.52,73.92,88.79999999999998,parent,3.252355396478069,74000810,1,0,1,0
6bba_b204cac7,4701,73,58.99999999999999,59.999999999999986,91.46666666666664,parent,4.0055874603830315,74000810,2,0,0,2
6bba_b204cac7,5096,80,57.0,98.08,93.44,cont,2.081015377165676,81000886,1,1,0,0
6bba_b204cac7,5154,81,57.67999999999999,100.0,94.72000000000003,cont,1.337346369681389,82000897,1,0,1,0
6bba_b204cac7,5864,94,60.16000000000001,50.56,76.0,parent,2.9250902763846467,95001023,1,1,0,0
6bba_b204cac7,5911,95,61.48,20.799999999999997,83.2,daughter,2.2824743289903644,96001033,2,0,0,2
6bba_b204cac7,5912,95,60.51999999999999,53.43999999999999,78.08000000000001,daughter,3.115341574370299,96001038,1,0,1,0
6bba_b329af44,2057,2,20.840000000000007,139.35999999999999,164.48000000000002,cont,6.853590413972802,3000069,5,5,0,0
6bba_b329af44,2397,2,43.16,110.56,205.76,cont,6.363239397115282,3000081,3,3,0,0
6bba_b329af44,2924,3,19.133333333333326,187.99999999999994,96.5333333333333,cont,1.2566783404630428,4000096,5,0,0,5
6bba_b329af44,2943,3,20.680000000000003,135.36000000000004,191.36,cont,0.8842449109268175,4000098,6,1,5,0
6bba_b329af44,3072,3,29.0,203.52000000000007,119.84,cont,6.993355105026483,4000100,8,0,3,5
6bba_b329af44,3120,3,33.279999999999994,99.36000000000001,71.36,cont,6.714239648873144,4000104,1,0,1,0
6bba_b329af44,4116,4,39.0,88.64,178.08000000000004,cont,2.0172265923787465,5000130,1,0,1,0
6bba_b329af44,5963,6,48.16,110.08000000000001,225.28,cont,0.5411518386737618,7000187,1,0,1,0
6bba_b329af44,6078,6,57.160000000000004,167.36000000000004,200.0,cont,5.30835544213083,7000189,2,2,0,0
6bba_b329af44,6417,7,20.840000000000003,51.36000000000001,65.92,cont,6.7288736520312264,8000201,7,0,0,7
6bba_b329af44,7194,8,14.360000000000003,54.56,190.72000000000003,cont,1.5300908878233201,9000221,1,1,0,0
6bba_b329af44,7383,8,27.679999999999996,197.28000000000003,148.8,cont,6.086118554752281,9000230,5,0,0,5
6bba_b329af44,7546,8,39.16,77.28,187.35999999999999,cont,2.2257754968774424,9000234,1,0,1,0
6bba_b329af44,7688,8,48.519999999999996,107.20000000000002,225.92,cont,1.5614381391845191,9000238,1,0,1,0
6bba_b329af44,7768,8,55.84000000000001,165.28000000000003,202.71999999999997,cont,1.8315703644687122,9000240,1,0,1,0
6bba_b329af44,9710,11,10.999999999999996,123.99999999999999,195.99999999999997,cont,1.8616713760758223,12000295,7,0,0,7
6bba_b329af44,10289,11,53.480000000000004,164.64,203.35999999999999,cont,0.9215340471192565,12000313,1,1,0,0
6bba_b329af44,10656,12,19.0,42.72,66.72,cont,0.5322960290101735,13000323,1,1,0,0
6bba_b329af44,12410,14,26.16,83.36000000000001,12.0,cont,0.9933788615125734,15000374,1,0,1,0
6bba_b329af44,14655,17,8.959999999999999,131.84,204.00000000000003,cont,6.593416223779591,18000441,6,6,0,0
6bba_b329af44,14712,17,14.479999999999999,169.44,162.07999999999998,cont,2.667871928147981,18000442,3,3,0,0
6bba_b329af44,15092,17,41.360000000000014,59.360000000000014,198.88,cont,2.261086920598147,18000453,4,4,0,0
6bba_b329af44,15129,17,44.160000000000004,160.64,168.15999999999997,cont,5.542369754897256,18000455,1,1,0,0
6bba_b329af44,15195,17,48.52000000000001,194.56000000000006,217.92000000000002,cont,1.1263020243256205,18000457,2,2,0,0
6bba_b329af44,15643,18,21.84,189.44,197.92000000000002,cont,6.253210300517649,19000469,3,0,3,0
6bba_b329af44,15892,18,38.2,73.27999999999999,207.83999999999997,cont,1.7418605354619996,19000473,6,0,3,3
6bba_b329af44,16035,18,47.87999999999999,202.07999999999998,57.76000000000001,cont,3.8483348671860873,19000478,3,0,0,3
6bba_b329af44,16084,18,53.2,160.00000000000006,216.0,cont,1.1941262496067808,19000480,3,0,3,0
6bba_b329af44,20305,24,19.64,188.00000000000003,222.07999999999998,cont,6.436723254109963,25000617,1,1,0,0
6bba_b329af44,21098,25,20.000000000000004,188.00000000000003,222.56,cont,1.8309935827304138,26000639,5,0,1,4
6bba_b329af44,21364,25,39.64,61.11999999999999,233.92000000000002,cont,2.253658830657386,26000645,5,0,1,4
6bba_b329af44,21470,25,46.0,162.07999999999998,174.72,cont,1.8422799224059292,26000650,2,0,2,0
6bba_b329af44,25322,30,47.32000000000001,80.00000000000003,84.64000000000001,cont,1.1863612961488437,31000767,1,0,1,0
6bba_b329af44,26825,32,45.160000000000004,180.0,189.92000000000004,cont,0.9170818188689746,33000808,4,4,0,0
6bba_b329af44,27434,33,35.64,100.16000000000001,206.08,cont,1.032993979895333,34000825,1,0,1,0
6bba_b329af44,27484,33,37.52,108.8,217.92000000000002,cont,5.690586662418912,34000825,3,0,3,0
6bba_b329af44,27600,33,45.160000000000004,182.07999999999996,194.72,cont,3.4817876604554727,34000831,4,0,4,0
6bba_b329af44,27632,33,48.519999999999996,89.92,101.28000000000002,cont,1.7769061068328866,34000832,7,0,2,5
6bba_b329af44,31735,39,16.68,48.0,100.63999999999999,cont,1.395987553669443,40000935,2,1,1,0
6bba_b329af44,32023,39,39.16,114.72,225.92,cont,6.850276112135912,40000943,2,2,0,0
6bba_b329af44,32627,40,29.0,8.0,191.84,cont,5.263495155312673,41000958,4,0,1,3
6bba_b329af44,32961,40,55.93893188476563,149.14052490234377,152.9415869140625,cont,6.405670951668612,41000966,1,0,1,0
6bba_b329af44,34312,42,42.32,73.28000000000002,154.71999999999997,cont,1.179888765943637,43000993,1,0,1,0
6bba_b329af44,35561,44,25.32,68.32,62.40000000000001,cont,4.742884802522624,45001023,4,4,0,0
6bba_b329af44,35635,44,31.000000000000007,12.320000000000004,204.96,cont,0.13101168840985303,45001025,2,2,0,0
6bba_b329af44,35755,44,39.59999999999999,105.92000000000002,228.00000000000003,cont,1.248609226299395,45001027,4,4,0,0
6bba_b329af44,35820,44,40.760000000000005,79.19999999999999,152.63999999999996,cont,3.7509702911380014,45001028,2,2,0,0
6bba_b329af44,36289,45,23.56,83.84000000000002,59.84000000000001,cont,4.468543177032085,46001044,12,1,5,6
6bba_b329af44,36290,45,21.159999999999997,94.56,66.72,cont,6.782813068705343,46001044,14,3,3,8
6bba_b329af44,36369,45,27.64,21.12,211.20000000000002,cont,2.35512904370441,46001045,11,2,3,6
6bba_b329af44,36404,45,29.480000000000004,125.92,131.2,cont,1.5614381391845167,46001046,8,3,5,0
6bba_b329af44,36509,45,36.0,109.92000000000002,227.36,cont,2.279927698963281,46001047,16,2,7,7
6bba_b329af44,36534,45,38.64,83.84,152.0,cont,2.2064723683971232,46001048,4,2,2,0
6bba_b329af44,36575,45,40.16,115.83999999999999,200.48,cont,1.095037813274034,46001049,13,2,3,8
6bba_b329af44,36611,45,45.400000000000006,200.00000000000003,104.64000000000001,cont,2.315610745570164,46001053,3,0,3,0
6bba_b329af44,75396,45,41.220741271972656,77.51643127441406,110.73555572509767,cont,6.881246872485488,46001052,4,4,0,0
6bba_b329af44,37166,46,31.32,16.640000000000004,207.36,cont,1.2226437798884884,47001064,7,0,1,6
6bba_b329af44,37330,46,42.96,80.64,152.0,cont,1.247868808208619,47001067,8,0,1,7
6bba_b329af44,37359,46,42.73333333333332,219.4666666666667,224.00000000000003,cont,2.1505258158697567,47001070,6,0,0,6
6bba_b329af44,37362,46,45.20436828613282,91.98558349609374,102.82775512695315,cont,3.3744810202304127,47001072,5,0,0,5
6bba_b329af44,37368,46,43.440000000000005,191.68000000000006,209.28,cont,4.25222478533296,47001069,12,0,4,8
6bba_b329af44,37391,46,46.86666666666666,91.99999999999999,105.06666666666662,cont,0.8147538184766345,47001072,5,0,0,5
6bba_b329af44,38899,48,46.12436828613282,93.26558349609375,105.22775512695311,cont,3.424499747607859,49001111,1,0,1,0
6bba_b329af44,42798,54,4.44,27.999999999999996,217.28000000000003,cont,3.462813613956135,55001205,2,0,0,2
6bba_b329af44,46318,58,45.91999999999999,208.00000000000006,218.4,cont,0.7774567914038516,59001283,1,0,1,0
6bba_b329af44,46987,59,38.99999999999999,136.53333333333336,239.99999999999994,cont,0.46041666666669595,60001296,5,0,0,5
6bba_b329af44,50809,64,44.00000000000001,122.08000000000003,105.28000000000002,cont,4.478430423987848,65001381,1,1,0,0
6bba_b329af44,51947,66,15.84,43.36000000000001,175.36,cont,3.582120749290835,67001406,1,1,0,0
6bba_b329af44,55311,70,41.0,127.36,160.00000000000006,cont,0.7803384682303062,71001490,2,1,1,0
6bba_b329af44,58018,74,15.999999999999996,148.0,8.533333333333333,cont,6.774569802352358,75001561,5,0,0,5
6bba_b329af44,58193,74,29.32,101.92000000000002,206.4,cont,6.108962677902034,75001573,5,0,0,5
6bba_b329af44,59773,76,32.36000000000001,190.72000000000003,116.63999999999999,cont,2.29671776672713,77001614,2,2,0,0
6bba_b329af44,63597,81,31.999999999999993,127.99999999999997,215.99999999999994,cont,6.039340428598112,82001733,5,1,0,4
6bba_b329af44,64291,82,27.0,162.08000000000004,35.36,parent,0.7639228527802987,83001755,1,1,0,0
6bba_b329af44,65021,83,26.84,168.0,32.64,daughter,0.5040120286263008,84001784,1,0,1,0
6bba_b329af44,68498,88,14.880000000000003,207.04,6.08,cont,2.7648409199264945,89001875,3,3,0,0
6bba_b329af44,68810,88,43.8,163.84000000000003,87.99999999999999,cont,6.723966463331003,89001890,3,3,0,0
6bba_b329af44,68932,89,1.0,27.360000000000003,135.36000000000004,cont,4.038443775144093,90001894,3,1,2,0
6bba_b329af44,69209,89,14.866666666666667,228.53333333333333,11.999999999999996,cont,6.170484460090815,90001899,2,0,0,2
6bba_b329af44,69414,89,34.480000000000004,160.32,201.28000000000006,cont,5.297898757998325,90001908,6,1,3,2
6bba_b329af44,69605,90,1.0,32.8,134.56000000000003,cont,4.171410133276283,91001914,1,0,1,0
6bba_b329af44,69707,90,3.5200000000000005,30.72,188.00000000000003,cont,1.629300559289173,91001913,1,0,1,0
6bba_b693381b,220,1,38.98344177246094,110.9532470703125,190.51146484375,cont,3.543243182347217,2000026,1,1,0,0
6bba_b693381b,365,2,41.63999999999999,98.72000000000001,176.00000000000003,cont,1.898191985153231,3000043,1,0,0,1
6bba_b693381b,568,4,23.0,175.99999999999997,237.44,cont,6.9808078418618535,5000073,1,1,0,0
6bba_b693381b,571,4,23.68,177.44000000000003,215.5200000000001,cont,2.372388696019233,5000073,2,2,0,0
6bba_b693381b,678,5,23.64,174.72,211.36,cont,0.6502030932716295,6000087,2,0,2,0
6bba_b693381b,693,5,33.64000000000001,124.15999999999998,135.04,cont,5.350496034247674,6000089,1,0,1,0
6bba_b693381b,811,6,45.48000000000001,79.35999999999999,65.44000000000001,cont,1.2488206936946453,7000113,1,1,0,0
6bba_b693381b,820,6,49.00000000000001,69.11999999999999,65.12,cont,6.902291512606533,7000113,1,1,0,0
6bba_b693381b,823,6,52.800000000000004,54.08,130.72,cont,0.9921819389607924,7000116,1,1,0,0
6bba_b693381b,912,7,46.279999999999994,74.88000000000002,109.44000000000001,cont,2.5377589153621205,8000129,1,0,0,1
6bba_b693381b,913,7,46.839999999999996,73.91999999999999,53.760000000000005,cont,2.7006796019705916,8000130,1,0,1,0
6bba_b693381b,917,7,49.48,71.52000000000001,62.08,cont,5.919038799712331,8000130,1,0,1,0
6bba_b693381b,923,7,53.84,49.92,120.48000000000002,cont,2.1376008748126907,8000133,1,0,1,0
6bba_b693381b,1103,9,43.839999999999996,96.0,239.36,cont,0.8655318090630785,10000165,1,1,0,0
6bba_b693381b,1197,10,44.68000000000001,91.36000000000001,234.08000000000004,cont,2.6319984090040567,11000181,1,0,1,0
6bba_b693381b,1277,11,37.000000000000014,106.72,98.08,cont,1.8760130596560185,12000195,1,1,0,0
6bba_b693381b,1291,11,45.52,72.64,127.36,cont,1.436632346844521,12000198,1,1,0,0
6bba_b693381b,1363,12,37.67999999999999,102.56,89.12,cont,1.1890292784452394,13000211,1,0,1,0
6bba_b693381b,1371,12,46.0,69.28,119.19999999999999,cont,2.0730058188533853,13000214,1,0,1,0
6bba_b693381b,1611,15,47.32,62.07999999999999,100.16000000000001,cont,2.6437606382764662,16000261,1,1,0,0
6bba_b693381b,1612,15,46.519999999999996,75.19999999999999,209.76,cont,3.6513717456594343,16000262,1,1,0,0
6bba_b693381b,1686,16,41.239999999999995,83.03999999999999,52.15999999999999,cont,2.1569082328648137,17000276,1,0,1,0
6bba_b693381b,1692,16,48.079999999999984,57.92,90.56,cont,2.221500407945067,17000279,1,0,1,0
6bba_b693381b,1693,16,48.26666666666665,66.4,178.4,cont,2.0254639814812117,17000278,2,0,0,2
6bba_b693381b,1699,16,53.64,42.07999999999999,44.0,cont,1.9020137354919464,17000281,2,0,2,0
6bba_b693381b,1773,17,56.640000000000015,34.72,168.8,cont,1.4138433877201428,18000300,1,1,0,0
6bba_b693381b,1818,18,33.32,112.00000000000001,68.32000000000002,cont,3.5874612367243675,19000306,2,0,2,0
6bba_b693381b,1846,18,57.36,29.919999999999998,155.2,cont,1.6651294994684365,19000316,1,0,1,0
6bba_b693381b,1884,19,27.840000000000007,151.36,240.8,cont,2.2302792005710788,20000321,1,1,0,0
6bba_b693381b,1954,20,28.68,145.92000000000002,237.28000000000003,cont,1.4307384456985945,21000336,1,0,1,0
6bba_b693381b,2112,22,43.84,74.07999999999998,194.24000000000007,cont,2.7932043650617353,23000371,1,1,0,0
6bba_b693381b,2181,23,44.00000000000001,70.56,185.92,cont,2.019842970381598,24000388,1,0,1,0
6bba_b693381b,2192,23,56.96,19.519999999999996,20.0,cont,3.6267000323986016,24000393,1,0,1,0
6bba_b693381b,2197,23,59.960000000000015,10.720000000000002,110.08,cont,3.438286500933306,24000394,1,0,1,0
6bba_b693381b,2471,28,24.32,144.64000000000001,195.04000000000002,cont,1.460783715167988,29000475,1,1,0,0
6bba_b693381b,2479,28,28.839999999999996,135.67999999999998,232.31999999999994,cont,1.0159824371021573,29000479,1,1,0,0
6bba_b693381b,2519,29,4.2,243.36000000000004,167.51999999999998,cont,1.8040427690329215,30000492,2,2,0,0
6bba_b693381b,2536,29,23.86666666666666,143.4666666666666,188.53333333333327,cont,1.852191129221457,30000496,1,0,1,0
6bba_b693381b,2541,29,27.999999999999996,117.44000000000003,151.52,cont,3.171457176914097,30000499,1,1,0,0
6bba_b693381b,2545,29,29.133333333333326,131.99999999999997,227.99999999999991,cont,1.237859445552876,30000503,1,0,1,0
6bba_b693381b,2566,29,53.0,43.36000000000001,214.88000000000002,cont,3.486751711837237,30000510,1,1,0,0
6bba_b693381b,2584,30,3.6399999999999997,240.64000000000004,158.39999999999998,cont,2.432185873859144,31000515,2,0,2,0
6bba_b693381b,2604,30,23.733333333333327,137.06666666666666,163.4666666666666,cont,0.615215070795045,31000519,2,1,0,1
6bba_b693381b,2611,30,27.999999999999996,112.47999999999999,140.48,cont,2.6397123422827695,31000523,2,1,1,0
6bba_b693381b,2637,30,53.0,39.199999999999996,205.76,cont,1.8800905895727473,31000533,1,0,1,0
6bba_b693381b,2669,31,22.640000000000004,137.76000000000005,157.28000000000006,cont,1.1206610772218095,32000544,1,0,1,0
6bba_b693381b,2676,31,27.999999999999996,109.28,131.84,cont,1.9961721180800087,32000545,1,0,1,0
6bba_b693381b,2894,35,4.32,228.79999999999995,144.48,cont,3.531076765662283,36000614,2,2,0,0
6bba_b693381b,2958,36,4.133333333333334,223.46666666666664,120.53333333333333,cont,2.1922448540252146,37000634,2,0,0,2
6bba_b693381b,3123,38,48.31999999999999,29.28,104.64000000000001,cont,1.0935899882954228,39000688,1,1,0,0
6bba_b693381b,3183,39,48.12,35.360000000000014,135.36,cont,3.7579684089278773,40000709,1,0,0,1
6bba_b693381b,3186,39,49.35999999999999,25.28,97.92000000000002,cont,2.1846431344729993,40000710,1,0,1,0
6bba_b693381b,3966,53,8.48,241.44000000000003,108.8,cont,2.739703736264192,54000940,1,1,0,0
6bba_b693381b,4031,54,9.84,233.92000000000007,101.92000000000002,cont,1.9577035334799873,55000957,1,0,1,0
6bba_b693381b,4040,54,18.599999999999998,121.60000000000001,93.44,cont,4.663098822135335,55000959,1,0,0,1
6bba_b693381b,4350,59,14.520000000000003,139.2,13.280000000000001,cont,2.4712825617480427,60001011,1,1,0,0
6bba_b693381b,4419,60,19.133333333333326,123.46666666666668,7.466666666666664,cont,1.956196751138958,61001021,2,0,0,2
6bba_b693381b,5272,73,25.599999999999994,119.99999999999997,241.86666666666667,cont,6.890034067001734,74001173,1,0,0,1
6bba_b693381b,5275,73,27.36,119.36000000000001,250.72000000000003,cont,2.727677583586442,74001173,1,0,1,0
6bba_b693381b,6326,89,28.840000000000003,173.28000000000003,196.64000000000001,cont,2.679328857195386,90001335,1,1,0,0
6bba_b693381b,6370,90,11.68,235.99999999999994,235.35999999999999,cont,0.9377249730064937,91001340,1,1,0,0
6bba_b693381b,6387,90,28.839999999999996,170.72000000000003,196.0,cont,3.217541035091858,91001343,1,0,1,0
6bba_b693381b,6434,91,11.000000000000002,235.99999999999994,235.35999999999999,cont,2.137724403425302,92001347,1,0,1,0
6bba_b693381b,6438,91,13.733333333333329,228.53333333333333,233.06666666666666,cont,5.3448256714633136,92001347,1,0,0,1
6bba_bb9f20c3,1620,6,12.0,230.72000000000008,178.56000000000006,cont,2.2366636006560947,7000159,1,1,0,0
6bba_bb9f20c3,1684,6,19.680000000000003,147.84,41.28000000000001,cont,3.273840682211031,7000164,2,2,0,0
6bba_bb9f20c3,1712,6,22.839999999999996,141.92000000000002,28.640000000000008,cont,6.555440131104847,7000164,2,2,0,0
6bba_bb9f20c3,1972,7,18.84,154.07999999999998,41.28,cont,1.9438287765387188,8000190,2,0,2,0
6bba_bb9f20c3,1983,7,20.52,161.28,100.00000000000001,cont,1.2405467544594948,8000192,1,0,1,0
6bba_bb9f20c3,2959,10,36.31999999999999,118.56,27.36,cont,0.8638522009001242,11000295,1,0,1,0
6bba_bb9f20c3,3785,13,27.64,160.0,28.640000000000004,cont,1.1435197855743464,14000374,1,0,1,0
6bba_bb9f20c3,4718,16,34.519999999999996,189.44,19.36,cont,1.0155924994799936,17000460,1,0,1,0
6bba_bb9f20c3,5422,19,12.13333333333333,87.99999999999997,163.46666666666664,cont,0.6340045784972961,20000526,3,0,0,3
6bba_bb9f20c3,5759,20,16.16,172.64000000000004,122.08000000000003,cont,0.478203016510755,21000559,1,0,1,0
6bba_bb9f20c3,5868,20,30.133333333333326,87.46666666666665,67.99999999999997,cont,1.2566783404630282,21000576,4,0,0,4
6bba_bb9f20c3,6817,24,3.0,242.71999999999997,96.0,cont,0.9262500000000102,25000654,2,0,2,0
6bba_bb9f20c3,6846,24,8.133333333333331,216.53333333333325,72.53333333333333,cont,1.4412812586954213,25000657,6,0,0,6
6bba_bb9f20c3,6894,24,14.159999999999998,172.64,139.52,cont,1.9197021253309026,25000659,2,0,2,0
6bba_bb9f20c3,6962,24,22.000000000000004,170.71999999999997,44.00000000000001,cont,2.2324685048170303,25000667,1,0,1,0
6bba_bb9f20c3,6968,24,22.86666666666666,116.53333333333333,147.46666666666667,cont,0.6862298873798335,25000666,3,0,0,3
6bba_bb9f20c3,7038,24,33.00000000000001,150.56,14.720000000000002,cont,1.3444358714717475,25000674,2,0,2,0
6bba_bb9f20c3,7044,24,34.16,104.00000000000001,13.280000000000001,cont,0.49555839716423894,25000675,3,1,2,0
6bba_bb9f20c3,7052,24,36.160000000000004,116.0,14.719999999999999,cont,5.725214242716166,25000675,4,1,3,0
6bba_bb9f20c3,7063,24,40.160000000000004,20.159999999999997,13.280000000000001,cont,5.979381761520167,25000676,2,2,0,0
6bba_bb9f20c3,7578,26,28.16,74.72,114.71999999999997,cont,5.859674292356189,27000722,4,0,0,4
6bba_bb9f20c3,8837,30,39.52000000000001,136.64000000000001,21.919999999999998,cont,6.861926888637043,31000823,1,1,0,0
6bba_bb9f20c3,8893,31,4.68,240.0,91.84,cont,1.3291268280341066,32000828,1,1,0,0
6bba_bb9f20c3,8993,31,19.160000000000004,112.16,76.0,parent,1.6328622301039393,32000835,1,1,0,0
6bba_bb9f20c3,9003,31,19.52,184.0,59.2,cont,1.1862500000000014,32000836,1,1,0,0
6bba_bb9f20c3,9091,31,31.84,149.92000000000002,28.000000000000004,cont,0.6101869590543572,32000843,3,1,2,0
6bba_bb9f20c3,9264,32,16.73333333333333,131.99999999999997,76.5333333333333,daughter,0.4729900824071933,33000858,5,0,0,5
6bba_bb9f20c3,9286,32,18.84,109.92000000000002,80.16000000000001,daughter,2.3492914490330947,33000860,1,0,1,0
6bba_bb9f20c3,9287,32,19.64,138.72000000000003,83.52,daughter,5.7334872950500255,33000858,1,1,0,0
6bba_bb9f20c3,9427,32,42.83999999999999,13.920000000000005,6.080000000000001,cont,1.2912360841457335,33000873,1,0,1,0
6bba_bb9f20c3,9640,33,26.866666666666664,136.53333333333336,72.0,cont,2.934263879389333,34000894,4,0,0,4
6bba_bb9f20c3,9982,34,36.16,132.64000000000001,13.280000000000001,cont,0.7597635322914578,35000927,1,1,0,0
6bba_bb9f20c3,10291,35,41.16,22.080000000000005,10.720000000000002,cont,1.5273271178761894,36000955,1,0,1,0
6bba_bb9f20c3,11172,39,3.16,222.72,204.00000000000003,cont,1.6495951207796393,40001037,1,0,1,0
6bba_bb9f20c3,11545,40,17.84,108.0,106.72,cont,5.152300760339988,41001074,1,1,0,0
6bba_bb9f20c3,11839,41,20.866666666666664,107.46666666666665,107.46666666666665,cont,1.0686768980847319,42001102,6,0,0,6
6bba_bb9f20c3,11919,41,32.13333333333334,211.99999999999994,4.533333333333333,cont,2.161243212186523,42001112,4,0,0,4
6bba_bb9f20c3,12175,42,28.48,133.28,48.640000000000015,cont,6.436907861116239,43001138,5,0,0,5
6bba_bb9f20c3,12567,44,8.0,100.64,192.0,cont,4.525442657906516,45001183,5,0,0,5
6bba_bb9f20c3,13911,49,12.079999999999998,138.4,110.71999999999997,cont,0.23716094218905778,50001327,1,1,0,0
6bba_bb9f20c3,14301,50,30.0,148.00000000000003,34.72,cont,0.6598780663880168,51001372,2,2,0,0
6bba_bb9f20c3,14390,51,4.0,187.35999999999999,138.72000000000003,cont,2.0152620797305745,52001381,2,0,2,0
6bba_bb9f20c3,14427,51,8.04,153.92000000000002,125.75999999999999,cont,2.604110032717513,52001385,4,0,1,3
6bba_bb9f20c3,14438,51,10.266666666666664,251.9999999999999,159.4666666666666,cont,3.264075423156655,52001388,1,0,1,0
6bba_bb9f20c3,14578,51,30.0,151.99999999999997,0.0,cont,6.1072766537139955,52001402,3,0,0,3
6bba_bb9f20c3,14590,51,32.000000000000014,162.72,13.920000000000002,cont,4.714684838353457,52001402,2,0,2,0
6bba_bb9f20c3,14603,51,35.480000000000004,146.07999999999998,16.640000000000004,cont,5.8118233477111225,52001402,5,0,0,5
6bba_bb9f20c3,14704,52,11.160000000000002,204.64000000000001,96.0,cont,1.0732383821872897,53001414,1,0,1,0
6bba_bb9f20c3,16101,57,33.160000000000004,155.35999999999999,17.28,cont,3.9733160741375766,58001558,1,1,0,0
6bba_bb9f20c3,16597,59,32.52,159.20000000000005,20.639999999999997,cont,4.749060762403448,60001614,2,2,0,0
6bba_bb9f20c3,16646,60,3.3200000000000003,207.36000000000004,230.72000000000008,cont,0.8527767146211029,61001621,1,1,0,0
6bba_bb9f20c3,16884,61,1.1333333333333326,217.0666666666666,224.00000000000003,cont,5.8263797841274165,62001643,6,0,0,6
6bba_bb9f20c3,16893,61,3.0,209.28000000000003,231.36,cont,1.111671039471648,62001643,1,0,1,0
6bba_bb9f20c3,17621,64,2.0799999999999996,216.95999999999998,232.0,cont,0.13101168840985258,65001714,1,1,0,0
6bba_bb9f20c3,17682,64,11.2,233.44,90.72000000000001,cont,4.401620795229868,65001737,1,1,0,0
6bba_bb9f20c3,17928,65,12.839999999999998,131.2,180.64000000000001,cont,1.7378385461543842,66001743,2,2,0,0
6bba_bb9f20c3,17954,65,14.68,250.71999999999997,108.64000000000001,cont,0.5520218519225487,66001746,3,2,1,0
6bba_bb9f20c3,18110,66,4.159999999999999,227.51999999999998,165.27999999999997,cont,6.18672735074853,67001765,2,0,2,0
6bba_bb9f20c3,18126,66,6.999999999999999,228.00000000000003,122.08000000000003,cont,2.169785775716108,67001767,2,0,2,0
6bba_bb9f20c3,18310,67,0.0,224.0,237.92000000000002,cont,3.350180965261429,68001787,1,1,0,0
6bba_bb9f20c3,19120,70,22.68,243.36000000000004,50.72,cont,6.579323792191731,71001867,1,1,0,0
6bba_bb9f20c3,19287,71,13.839999999999998,206.08000000000004,111.99999999999999,cont,0.9170818188689525,72001878,1,0,1,0
6bba_bb9f20c3,19715,73,14.16,204.64,113.28000000000003,cont,0.9279589497924984,74001919,1,0,1,0
6bba_bb9f20c3,19984,74,22.13333333333333,211.46666666666658,87.46666666666665,cont,4.6281605192199775,75001946,2,0,0,2
6bba_bb9f20c3,20781,78,13.600000000000001,177.92,76.0,cont,2.7495172871796982,79002027,3,3,0,0
6bba_bb9f20c3,20808,78,15.0,252.0,56.64,cont,6.972480951928087,79002033,2,2,0,0
6bba_bb9f20c3,20824,78,17.2,198.72,55.36,cont,1.3131367503044,79002034,4,4,0,0
6bba_bb9f20c3,20835,78,19.52,175.2,18.72,cont,1.6027476407719323,79002040,3,3,0,0
6bba_bb9f20c3,20924,79,0.31999999999999995,238.39999999999992,147.84,cont,4.827781743461475,80002047,5,1,1,3
6bba_bb9f20c3,20974,79,7.533333333333334,180.53333333333327,79.99999999999999,cont,1.207564410520431,80002051,2,0,0,2
6bba_bb9f20c3,20982,79,6.239999999999999,237.44,141.92000000000002,cont,5.4237281055100794,80002047,7,0,3,4
6bba_bb9f20c3,20990,79,10.480000000000002,150.56,135.35999999999999,cont,4.986950001002618,80002049,10,1,6,3
6bba_bb9f20c3,20992,79,10.319999999999999,191.20000000000002,200.64000000000004,cont,2.7853467019385407,80002053,5,2,1,2
6bba_bb9f20c3,20998,79,10.04,173.27999999999997,155.35999999999999,cont,6.5997009875448125,80002048,6,1,1,4
6bba_bb9f20c3,21035,79,18.4,179.35999999999999,19.36,cont,3.9172949850630334,80002058,3,0,3,0
6bba_bb9f20c3,21063,79,21.84,180.15999999999997,37.28000000000001,cont,2.442694465134778,80002060,9,2,3,4
6bba_bb9f20c3,21068,79,22.86666666666666,149.06666666666666,4.533333333333333,cont,0.8430707142148135,80002067,3,0,2,1
6bba_bb9f20c3,21080,79,25.999999999999996,161.86666666666662,3.999999999999999,cont,6.57679032712342,80002067,7,1,0,6
6bba_bb9f20c3,21174,80,9.64,172.64000000000004,155.36,cont,0.6566689139132357,81002071,7,0,1,6
6bba_bb9f20c3,21409,81,13.875028228759763,178.75128662109375,155.97717529296875,cont,6.76427623921756,82002093,1,1,0,0
6bba_bb9f20c3,21880,83,26.0,157.92000000000002,3.3600000000000003,cont,1.0729923112492463,84002156,1,0,1,0
6bba_bb9f20c3,22027,84,15.36,143.83999999999997,136.16000000000003,cont,3.0337057063103514,85002163,5,0,0,5
6bba_bb9f20c3,22199,85,10.000000000000004,180.0,149.92000000000002,parent,2.735073410806365,86002188,1,0,1,0
6bba_bb9f20c3,22205,85,11.266666666666664,167.99999999999994,151.99999999999997,parent,2.8765764096018613,86002188,6,0,0,6
6bba_bb9f20c3,22219,85,14.16,184.0,157.28000000000003,parent,6.658634549402757,86002188,1,0,1,0
6bba_bb9f20c3,22502,86,29.0,171.35999999999996,16.0,cont,6.96253242811119,87002229,1,0,1,0
6bba_bb9f20c3,23321,90,37.8,191.36,4.0,cont,1.6715398028464594,91002320,1,1,0,0
6bba_bbb708ca,236,2,10.520000000000001,29.279999999999994,232.64000000000004,cont,2.4981699551871994,3000020,1,1,0,0
6bba_bbb708ca,259,2,27.680000000000007,116.63999999999999,167.36000000000004,cont,2.2610869205981263,3000021,1,1,0,0
6bba_bbb708ca,293,2,49.00000000000001,139.36,168.64000000000004,cont,2.116371452510153,3000023,1,1,0,0
6bba_bbb708ca,335,3,7.8,28.640000000000008,227.36,cont,0.9245378913273459,4000027,1,0,1,0
6bba_bbb708ca,359,3,25.479999999999997,118.72000000000003,161.92000000000002,cont,1.1555795353414706,4000028,1,0,1,0
6bba_bbb708ca,389,3,46.99999999999999,140.64000000000004,164.0,cont,0.4823267175888249,4000030,1,0,1,0
6bba_bbb708ca,724,7,19.839999999999996,120.00000000000003,146.07999999999998,cont,1.928553943373125,8000058,1,1,0,0
6bba_bbb708ca,759,7,44.52,110.72000000000003,194.07999999999998,cont,0.8723692595455208,8000061,1,1,0,0
6bba_bbb708ca,782,7,59.31999999999999,206.72,67.36000000000001,cont,1.5841874889040268,8000064,1,1,0,0
6bba_bbb708ca,859,8,42.8,109.28,189.92000000000002,cont,1.2352137972836952,9000070,1,0,1,0
6bba_bbb708ca,885,8,58.999999999999986,207.36,62.56,cont,2.1543970299134996,9000072,1,0,1,0
6bba_bbb708ca,924,9,17.52,120.00000000000003,136.64,cont,2.7522050818389334,10000074,1,1,0,0
6bba_bbb708ca,939,9,30.52,52.64000000000001,234.72,cont,2.97130705666715,10000075,1,1,0,0
6bba_bbb708ca,959,9,41.84000000000001,108.64000000000001,188.00000000000006,cont,1.933818276105574,10000078,1,1,0,0
6bba_bbb708ca,1018,10,16.0,119.36000000000001,132.64000000000001,cont,0.5715289253397492,11000082,1,0,1,0
6bba_bbb708ca,1034,10,28.639999999999997,54.080000000000005,229.92,cont,0.9755415162872421,11000084,1,0,1,0
6bba_bbb708ca,1049,10,40.160000000000004,108.64000000000001,184.64,cont,2.086907475788027,11000087,1,0,1,0
6bba_bbb708ca,1111,11,14.84,118.72,131.36,cont,2.020561880393667,12000091,1,1,0,0
6bba_bbb708ca,1129,11,27.679999999999996,54.72,227.36000000000004,cont,2.1804085053035362,12000093,1,1,0,0
6bba_bbb708ca,1138,11,35.519999999999996,145.92,126.72000000000001,cont,2.749661342783876,12000094,2,2,0,0
6bba_bbb708ca,1168,11,57.36,206.72,53.280000000000015,cont,2.842310349873854,12000097,1,1,0,0
6bba_bbb708ca,1220,12,25.68,53.92000000000001,222.71999999999997,cont,0.7040208182291208,13000102,1,0,1,0
6bba_bbb708ca,1232,12,33.839999999999996,146.72,121.91999999999999,cont,0.5879267067415853,13000104,2,0,2,0
6bba_bbb708ca,1237,12,37.480000000000004,110.71999999999998,178.72000000000003,cont,2.619074562226119,13000107,1,0,1,0
6bba_bbb708ca,1267,12,56.0,204.64000000000001,51.36000000000001,cont,0.29831034594864997,13000108,2,1,1,0
6bba_bbb708ca,1322,13,24.040000000000003,54.08000000000001,218.72000000000006,cont,1.7478383971351503,14000114,1,1,0,0
6bba_bbb708ca,1369,13,54.84,203.35999999999999,49.28,cont,2.0025117040856397,14000119,3,2,1,0
6bba_bbb708ca,1403,14,10.320000000000002,115.36000000000004,121.92000000000002,cont,1.6277601558583554,15000124,1,0,1,0
6bba_bbb708ca,1413,14,21.640000000000004,52.64,213.28000000000003,cont,1.6966276292987923,15000125,1,0,1,0
6bba_bbb708ca,1430,14,34.32,111.99999999999999,170.56000000000003,cont,1.864860081882822,15000128,1,0,1,0
6bba_bbb708ca,1453,14,53.32000000000001,201.28000000000006,48.64,cont,1.2953196999196703,15000129,2,0,2,0
6bba_bbb708ca,1748,18,6.36,111.36,116.16000000000003,cont,2.690737004056693,19000159,1,1,0,0
6bba_bbb708ca,1773,18,26.359999999999996,142.72,110.72000000000003,cont,2.717639428345862,19000162,1,1,0,0
6bba_bbb708ca,1804,18,50.20000000000001,200.0,45.44,cont,2.9584819798335595,19000167,1,1,0,0
6bba_bbb708ca,1861,19,25.32,141.28000000000003,108.64,cont,2.0653487901562793,20000171,1,0,1,0
6bba_bbb708ca,1885,19,49.160000000000004,200.0,42.56000000000001,cont,2.336668635579293,20000175,1,0,1,0
6bba_bbb708ca,2059,21,62.000000000000014,195.36,62.56,cont,2.2252415599210793,22000196,1,0,1,0
6bba_bbb708ca,2368,25,8.520000000000001,35.36,194.07999999999998,cont,3.301950417934828,26000240,1,1,0,0
6bba_bbb708ca,2383,25,19.2,131.52000000000004,105.92000000000003,cont,3.313286934827712,26000243,1,1,0,0
6bba_bbb708ca,2386,25,22.36,99.52000000000001,153.27999999999997,cont,4.973907072161675,26000244,1,1,0,0
6bba_bbb708ca,2414,25,42.88000000000001,194.07999999999998,42.56,cont,3.617915706936779,26000247,1,1,0,0
6bba_bbb708ca,2457,26,7.000000000000001,30.560000000000006,193.92000000000002,cont,2.081015377165679,27000254,1,0,1,0
6bba_bbb708ca,2471,26,20.640000000000004,94.56,154.72000000000003,cont,1.3827785659678147,27000259,1,0,1,0
6bba_bbb708ca,2493,26,40.96000000000001,190.56,44.00000000000001,cont,0.7555813159415755,27000262,1,0,1,0
6bba_bbb708ca,2714,29,28.000000000000004,73.27999999999999,219.36,cont,0.7587201394453729,30000297,1,1,0,0
6bba_bbb708ca,2864,31,13.519999999999998,115.36000000000001,116.63999999999999,cont,2.9007956882552044,32000325,1,1,0,0
6bba_bbb708ca,2917,31,59.519999999999996,182.07999999999993,80.0,cont,1.8710800691044924,32000332,1,1,0,0
6bba_bbb708ca,2947,32,12.839999999999998,112.64,118.72000000000003,cont,0.31926184394631135,33000340,1,0,1,0
6bba_bbb708ca,2999,32,58.839999999999996,179.99999999999997,81.28,cont,0.28379404944431685,33000347,1,0,1,0
6bba_bbb708ca,3043,33,24.680000000000003,77.92000000000002,210.72000000000003,cont,3.2642686776673204,34000358,1,1,0,0
6bba_bbb708ca,3153,34,59.84,208.00000000000006,35.36,cont,0.8918274777108189,35000380,1,1,0,0
6bba_bbb708ca,3222,35,59.0,205.28,37.28,cont,2.121729836713428,36000394,1,0,1,0
6bba_bbb708ca,3294,36,60.839999999999996,109.44000000000001,172.64,cont,1.6134212484654984,37000409,1,1,0,0
6bba_bbb708ca,3388,37,60.0,105.92,175.36,cont,1.885280151595516,38000425,1,0,1,0
6bba_bbb708ca,3445,38,28.04,51.36000000000001,243.36000000000004,cont,3.198939521544604,39000431,1,1,0,0
6bba_bbb708ca,3468,38,50.31999999999999,168.8,93.27999999999999,cont,1.2637444361895478,39000434,1,1,0,0
6bba_bbb708ca,3477,38,56.04,200.16,46.72000000000001,cont,3.294624883806959,39000437,1,1,0,0
6bba_bbb708ca,3509,39,13.280000000000005,60.48,213.44,cont,3.5657180640791077,40000444,1,1,0,0
6bba_bbb708ca,3529,39,25.68,50.72,243.36000000000004,cont,2.217931539182407,40000445,3,2,1,0
6bba_bbb708ca,3559,39,48.519999999999996,164.64,97.44,cont,2.7084904121115114,40000448,1,0,1,0
6bba_bbb708ca,3569,39,54.0,196.64000000000001,50.72,cont,1.950338512284473,40000450,1,0,1,0
6bba_bbb708ca,3618,40,23.12,48.640000000000015,242.56000000000006,cont,0.7305274207721543,41000456,2,0,2,0
6bba_bbb708ca,3706,41,21.32,152.63999999999996,80.0,cont,1.2354275575686382,42000467,1,1,0,0
6bba_bbb708ca,3746,41,50.48000000000001,192.0,56.00000000000001,cont,1.0218136449470505,42000471,1,1,0,0
6bba_bbb708ca,3794,42,19.68,150.07999999999998,83.36000000000001,cont,2.160944527284306,43000478,1,0,1,0
6bba_bbb708ca,3830,42,49.160000000000004,190.08000000000004,58.08000000000001,cont,1.3657735903142905,43000482,1,0,1,0
6bba_bbb708ca,4403,48,60.67999999999999,193.44,54.72000000000001,cont,0.6228224566439546,49000549,1,1,0,0
6bba_bbb708ca,4452,49,31.72,138.72,121.28000000000003,cont,4.044814002213206,50000554,1,0,1,0
6bba_bbb708ca,4462,49,37.88,177.44000000000005,72.64,cont,3.5133837856118086,50000556,1,1,0,0
6bba_bbb708ca,4515,50,8.866666666666665,165.0666666666666,51.46666666666666,cont,1.8545657342084174,51000562,1,0,0,1
6bba_bbb708ca,4909,55,3.8400000000000007,157.44000000000003,60.64,cont,2.00725325382723,56000613,1,1,0,0
6bba_bbb708ca,4934,55,25.2,162.08,86.56,cont,3.3233137171353575,56000615,1,1,0,0
6bba_bbb708ca,5012,56,22.96,157.92000000000002,89.28000000000002,cont,1.5875177164365775,57000625,1,0,1,0
6bba_bbb708ca,5625,63,55.6,20.533333333333328,247.99999999999997,cont,0.9002917158405369,64000712,1,0,0,1
6bba_bbb708ca,5815,66,24.040000000000003,238.07999999999998,13.280000000000001,cont,0.39604687222600504,67000739,1,1,0,0
6bba_bbb708ca,5883,67,21.96,236.0,14.719999999999999,cont,1.8591874905452672,68000748,1,0,1,0
6bba_bbb708ca,7209,86,9.48,221.28000000000006,55.35999999999999,cont,1.0029028180736272,87000929,1,1,0,0
6bba_bbb708ca,7367,88,28.68,218.72000000000006,17.919999999999998,cont,0.8563302882065921,89000949,1,1,0,0
6bba_bbb708ca,7498,90,8.999999999999998,57.44,223.20000000000005,cont,4.3099583089630995,91000965,1,1,0,0
6bba_bbb708ca,7536,90,43.96000000000001,191.36,76.0,cont,0.4866868217858309,91000970,1,1,0,0
6bba_c27cba08,3168,48,40.28000000000001,27.999999999999996,9.28,cont,1.3432568862656062,49000406,1,0,1,0
6bba_c27cba08,3314,50,40.00000000000001,28.640000000000004,12.0,cont,3.3215651901625995,51000425,1,1,0,0
6bba_c27cba08,3335,50,61.00000000000001,49.28,44.64000000000001,cont,0.7138911856858859,51000428,1,0,1,0
6bba_c27cba08,3388,51,40.519999999999996,30.080000000000002,13.280000000000001,cont,2.451381307038953,52000435,1,0,1,0
6bba_c27cba08,3692,55,61.84000000000002,62.56,66.56,cont,1.5220447800574273,56000486,1,1,0,0
6bba_c27cba08,3733,56,29.800000000000004,37.28,21.28000000000001,cont,2.112374996301557,57000490,1,0,1,0
6bba_c27cba08,3773,56,61.839999999999996,68.16000000000001,74.24,cont,0.5285623189369457,57000497,1,0,1,0
6bba_c27cba08,4118,61,34.160000000000004,46.72,36.64,cont,1.9726188177141621,62000559,1,1,0,0
6bba_c27cba08,4198,62,34.480000000000004,48.0,39.36,cont,0.8915313370824454,63000571,1,0,1,0
6bba_c27cba08,4226,62,59.64,13.280000000000001,9.920000000000003,cont,1.3074940248429485,63000576,1,1,0,0
6bba_c27cba08,4307,63,59.83999999999999,15.36,14.720000000000002,cont,1.5404108624000121,64000589,1,0,1,0
6bba_c27cba08,4312,63,61.99999999999999,7.4666666666666694,3.4666666666666686,cont,6.937669542472847,64000589,1,0,0,1
6bba_c27cba08,4471,65,59.84000000000002,19.360000000000003,23.36,cont,1.7961947625466423,66000615,1,1,0,0
6bba_c27cba08,4474,65,62.000000000000014,10.720000000000002,7.360000000000001,cont,6.65119467652242,66000615,1,1,0,0
6bba_c27cba08,4551,66,59.68000000000001,20.639999999999997,26.719999999999995,cont,1.5841874889040237,67000629,1,0,1,0
6bba_c27cba08,5481,78,57.00000000000001,60.640000000000015,64.63999999999999,cont,6.6399098732588095,79000748,1,1,0,0
6bba_c27cba08,5543,79,40.519999999999996,78.56,76.0,cont,1.1858047109452712,80000754,1,1,0,0
6bba_c27cba08,5625,80,41.00000000000001,82.72,79.36,cont,1.950338512284484,81000766,1,0,1,0
6bba_c328f2fd,2136,5,56.68,237.76000000000005,103.99999999999999,cont,2.2033585869077155,6000002,2,2,0,0
6bba_c328f2fd,2475,6,53.68,230.72000000000008,112.00000000000001,cont,6.489795114254381,7000003,3,1,2,0
6bba_c328f2fd,2487,6,55.67999999999999,231.36,100.0,cont,0.675890616150286,7000003,3,1,2,0
6bba_c328f2fd,2821,7,54.519999999999996,225.44000000000003,96.0,cont,3.142600435944725,8000005,2,1,1,0
6bba_c328f2fd,3238,8,61.0,241.28000000000003,113.92,cont,1.9039564070639876,9000008,1,1,0,0
6bba_c328f2fd,3605,9,60.32000000000001,236.64000000000001,112.0,cont,0.9377249730064874,10000010,1,0,1,0
6bba_c328f2fd,3959,10,59.52000000000001,231.51999999999998,110.08,cont,2.0035663515341846,11000012,1,1,0,0
6bba_c328f2fd,4325,11,58.519999999999996,224.0,107.36000000000001,cont,1.0258114166356258,12000014,1,0,1,0
6bba_c328f2fd,4678,12,57.24,216.32,103.99999999999999,cont,1.3091087139729918,13000016,2,2,0,0
6bba_c328f2fd,4999,13,46.2,179.67999999999995,79.19999999999999,cont,3.4149752927363677,14000017,3,3,0,0
6bba_c328f2fd,5046,13,55.86666666666665,207.9999999999999,99.99999999999997,cont,1.4806931677240152,14000019,2,0,2,0
6bba_c328f2fd,5348,14,43.26666666666665,152.0,71.46666666666665,cont,1.2775182782993557,15000021,4,0,0,4
6bba_c328f2fd,5404,14,52.360000000000014,199.36000000000007,92.80000000000001,cont,3.167791521707178,15000022,7,0,3,4
6bba_c328f2fd,5781,15,55.320000000000014,237.9200000000001,113.28,cont,6.116500863443109,16000026,1,1,0,0
6bba_c328f2fd,5805,15,58.839999999999996,227.36000000000004,104.00000000000001,cont,3.2757759099944517,16000026,1,1,0,0
6bba_c328f2fd,6147,16,54.680000000000014,234.72000000000006,111.99999999999999,cont,6.614728121774929,17000029,3,2,1,0
6bba_c328f2fd,6175,16,58.36000000000001,223.36000000000004,102.72000000000001,cont,4.060094287698267,17000029,1,0,1,0
6bba_c328f2fd,6516,17,54.0,232.0,111.36,cont,6.485887805458865,18000032,2,0,2,0
6bba_c328f2fd,8617,23,37.959999999999994,106.88000000000001,65.92,cont,3.390335759552436,24000050,1,1,0,0
6bba_c328f2fd,8962,24,32.88,98.72000000000001,57.92000000000001,cont,1.943013526072314,25000053,6,0,2,4
6bba_c328f2fd,10252,28,31.000000000000007,89.27999999999999,59.2,parent,1.7061726172928724,29000074,2,2,0,0
6bba_c328f2fd,10293,28,41.0,134.4,70.72,cont,2.1093100915939322,29000072,2,2,0,0
6bba_c328f2fd,10380,28,59.519999999999996,228.0,102.72000000000001,cont,1.7447385190910387,29000073,1,1,0,0
6bba_c328f2fd,10582,29,30.999999999999996,60.53333333333332,55.99999999999999,daughter,1.9120142810949752,30000075,4,1,0,3
6bba_c328f2fd,10618,29,36.879999999999995,111.2,66.08,cont,1.9208709775776278,30000082,6,0,1,5
6bba_c328f2fd,15080,43,23.680000000000007,65.44,46.72000000000001,cont,5.389368295542631,44000160,1,1,0,0
6bba_c328f2fd,15368,44,17.16,62.56,45.919999999999995,cont,6.215619689540212,45000165,1,1,0,0
6bba_c328f2fd,15402,44,22.880000000000003,63.52000000000001,44.80000000000001,cont,6.27969470794561,45000165,1,0,1,0
6bba_c328f2fd,15412,44,26.64,61.280000000000015,55.99999999999999,cont,4.034616811111062,45000166,6,0,0,6
6bba_c328f2fd,15711,45,17.84,63.36,46.08000000000001,cont,6.795259606887734,46000170,1,0,1,0
6bba_c328f2fd,15726,45,21.480000000000004,58.56000000000001,41.28000000000001,cont,1.8862603545110168,46000170,3,0,0,3
6bba_c328f2fd,16065,46,20.320000000000004,53.92000000000001,38.56,cont,0.679592019155616,47000176,2,2,0,0
6bba_c328f2fd,16094,46,25.520000000000003,62.08000000000001,54.720000000000006,cont,5.91636145046091,47000177,1,1,0,0
6bba_c328f2fd,16243,46,58.999999999999986,152.0,102.72000000000001,cont,1.224046823042321,47000180,1,1,0,0
6bba_c328f2fd,16401,47,19.0,50.080000000000005,36.00000000000001,cont,2.039617764435284,48000181,2,0,2,0
6bba_c328f2fd,16429,47,25.000000000000004,61.27999999999999,53.28,cont,5.67350778839687,48000182,3,0,1,2
6bba_c328f2fd,16437,47,27.133333333333333,64.53333333333333,48.53333333333332,cont,1.7388259329214066,48000182,2,0,0,2
6bba_c328f2fd,16458,47,32.0,120.0,55.99999999999999,cont,3.3745659443104685,48000183,1,1,0,0
6bba_c328f2fd,16583,47,58.999999999999986,154.08,101.28000000000002,cont,0.937443331620634,48000185,1,0,1,0
6bba_c328f2fd,19307,55,42.12,239.36000000000007,75.2,cont,1.5691981431609996,56000227,1,1,0,0
6bba_c328f2fd,19511,56,8.84,36.800000000000004,31.360000000000007,cont,2.9210250769207726,57000232,2,2,0,0
6bba_c328f2fd,19598,56,22.84,62.07999999999999,37.28,cont,2.4702672552782636,57000233,2,2,0,0
6bba_c328f2fd,19873,57,8.0,32.64,28.000000000000004,cont,0.8255575767322367,58000237,2,0,2,0
6bba_c328f2fd,19951,57,21.479999999999997,57.280000000000015,35.36,cont,2.491554798213362,58000238,2,0,2,0
6bba_c328f2fd,20313,58,24.52,61.44000000000001,50.080000000000005,cont,6.751655005441261,59000243,2,2,0,0
6bba_c328f2fd,20761,59,39.000000000000014,228.64,72.64,cont,1.7359381469395567,60000250,1,1,0,0
6bba_c328f2fd,20940,60,5.800000000000001,21.919999999999998,20.0,cont,0.9515620381772273,61000253,1,1,0,0
6bba_c328f2fd,21123,60,38.16,224.64000000000004,71.36,cont,1.5728119722331724,61000260,1,0,1,0
6bba_c328f2fd,21315,61,5.520000000000001,19.360000000000003,17.279999999999998,cont,0.8726719028363389,62000263,1,0,1,0
6bba_c328f2fd,21616,61,61.44,175.67999999999998,163.04000000000008,cont,0.9193824353880148,62000270,1,1,0,0
6bba_c328f2fd,21737,62,14.133333333333331,83.46666666666665,43.999999999999986,cont,6.77153746338714,63000272,5,0,0,5
6bba_c328f2fd,21983,62,60.32000000000001,172.48000000000002,160.64000000000001,cont,2.351987523882735,63000277,1,0,1,0
6bba_c328f2fd,23053,65,56.160000000000004,178.56000000000003,167.36,cont,6.514872048820303,66000301,1,0,1,0
6bba_c328f2fd,23069,65,59.160000000000004,170.72000000000003,156.64000000000004,cont,1.3654835453054528,66000301,1,0,1,0
6bba_c328f2fd,23144,66,3.2,12.639999999999997,12.639999999999997,cont,3.0275629638374166,67000302,1,1,0,0
6bba_c328f2fd,23363,66,45.51999999999999,124.00000000000003,103.36,cont,6.113974770965301,67000307,4,0,0,4
6bba_c328f2fd,23483,67,2.3200000000000003,12.0,12.0,cont,1.2454329066633814,68000309,1,0,1,0
6bba_c328f2fd,23807,67,59.36,167.84000000000003,154.71999999999997,cont,3.5698627862286423,68000318,1,1,0,0
6bba_c328f2fd,24163,68,59.16000000000001,168.0,153.27999999999997,cont,1.5925829060051928,69000328,1,0,1,0
6bba_c328f2fd,24244,69,5.32,239.36000000000007,35.35999999999999,cont,1.2624900989710712,70000333,1,1,0,0
6bba_c328f2fd,24583,70,4.840000000000001,238.72000000000003,35.36,cont,2.0319648742042706,71000343,1,0,1,0
6bba_c328f2fd,24882,71,1.0,240.00000000000006,44.00000000000001,cont,5.296851954227166,72000350,2,0,2,0
6bba_c328f2fd,25409,72,26.8,189.92000000000002,68.0,cont,0.9515620381772268,73000368,1,1,0,0
6bba_c328f2fd,25761,73,26.16,188.64000000000001,68.0,cont,1.6096525246151754,74000380,1,0,1,0
6bba_c328f2fd,26278,74,60.26666666666665,167.46666666666664,147.46666666666667,cont,2.7684678185776153,75000393,3,0,0,3
6bba_c328f2fd,27006,76,62.000000000000014,165.27999999999997,143.99999999999997,cont,5.345830150687578,77000411,1,1,0,0
6bba_c328f2fd,27188,77,23.520000000000003,180.0,66.08,cont,2.4782182359509766,78000419,1,1,0,0
6bba_c328f2fd,27348,77,57.84,169.44000000000005,152.0,cont,2.673952189643615,78000421,1,0,1,0
6bba_c328f2fd,27448,78,6.200000000000001,78.72000000000001,27.36,cont,2.186515850045452,79000426,1,1,0,0
6bba_c328f2fd,27542,78,22.200000000000003,176.64000000000001,64.0,cont,2.956696319374038,79000428,1,0,1,0
6bba_c328f2fd,27549,78,23.86666666666666,179.46666666666664,75.99999999999999,cont,5.379843586377655,79000428,5,0,0,5
6bba_c328f2fd,27796,79,4.96,75.36,27.360000000000003,cont,3.3005106139050664,80000435,1,0,1,0
6bba_c328f2fd,28230,80,19.480000000000004,170.07999999999998,60.64,cont,1.1789932145691038,81000444,1,1,0,0
6bba_c328f2fd,28574,81,18.000000000000004,165.28,59.36,cont,1.6495951207796347,82000452,1,0,1,0
6bba_c328f2fd,29089,82,54.64,157.59999999999997,149.28000000000003,cont,2.6810037765732506,83000462,1,1,0,0
6bba_c328f2fd,29256,83,14.999999999999996,199.99999999999997,91.99999999999999,cont,3.4946946397360614,84000470,4,1,0,3
6bba_c328f2fd,29473,83,61.68,143.83999999999997,145.44,cont,6.519997363879841,84000474,1,0,1,0
6bba_c328f2fd,29589,84,15.000000000000007,196.32000000000002,92.0,cont,3.4800808180270684,85000481,1,0,1,0
6bba_c328f2fd,32335,92,51.2,150.08000000000004,152.0,cont,3.1824288091016193,93000547,1,1,0,0
6bba_c328f2fd,32500,93,17.160000000000004,216.0,76.63999999999999,cont,6.338270786460617,94000551,1,0,1,0
6bba_c328f2fd,32641,93,49.92000000000001,147.99999999999997,151.35999999999999,cont,5.0281869558022425,94000554,1,0,1,0
6bba_c328f2fd,33230,95,45.76,144.8,147.20000000000002,cont,2.9441663718275146,96000570,3,0,0,3
6bba_c73a1d11,1150,6,59.83999999999999,205.28,244.00000000000003,cont,0.8490527810448696,7000008,3,3,0,0
6bba_c73a1d11,1349,7,59.84000000000002,204.64,240.00000000000006,cont,1.7168184601174652,8000010,3,0,3,0
6bba_c73a1d11,2397,12,58.53333333333333,186.66666666666666,232.0,cont,1.7983414909453563,13000024,11,6,0,5
6bba_c73a1d11,2627,13,56.160000000000004,168.8,227.20000000000002,cont,1.6636222715808942,14000029,9,0,3,6
6bba_c73a1d11,3934,18,51.879999999999995,193.44,238.08000000000004,cont,2.0194507297777795,19000074,1,1,0,0
6bba_c73a1d11,3968,18,58.36,190.72000000000003,134.07999999999998,cont,6.182457667667452,19000079,3,3,0,0
6bba_c73a1d11,4286,19,56.599999999999994,147.46666666666667,139.99999999999997,cont,3.064727789480008,20000091,6,0,0,6
6bba_c73a1d11,4632,20,57.519999999999996,176.64000000000001,225.28,cont,2.498169955187199,21000103,2,2,0,0
6bba_c73a1d11,32454,20,50.21707824707032,165.08674438476567,197.04091308593752,cont,4.965101854339959,21000100,1,1,0,0
6bba_c73a1d11,5004,21,56.480000000000004,145.28,128.48,cont,2.1185538257264054,22000116,2,0,2,0
6bba_c73a1d11,5005,21,55.04,166.07999999999998,199.36000000000007,cont,3.6117795561883264,22000117,4,0,0,4
6bba_c73a1d11,5039,21,59.83999999999999,200.79999999999998,97.92000000000002,cont,1.5381807436059172,22000121,6,1,3,2
6bba_c73a1d11,5756,23,55.320000000000014,171.35999999999996,194.07999999999998,cont,4.538639836173366,24000139,1,0,1,0
6bba_c73a1d11,6064,24,43.59999999999998,160.53333333333333,199.99999999999997,cont,4.77312535917968,25000152,6,1,0,5
6bba_c73a1d11,6078,24,45.51999999999999,149.28,170.08000000000004,cont,2.5927281119507892,25000146,3,0,3,0
6bba_c73a1d11,6955,26,57.68000000000002,145.60000000000002,125.43999999999998,cont,1.1990914529342545,27000173,2,0,0,2
6bba_c73a1d11,7271,27,46.800000000000004,126.08000000000003,246.56,cont,4.470965359964221,28000177,1,1,0,0
6bba_c73a1d11,7722,28,51.16,205.44000000000003,97.44,cont,0.7079481796007286,29000188,1,0,1,0
6bba_c73a1d11,7863,29,8.639999999999997,136.16,162.07999999999998,cont,6.89938335106841,30000193,2,2,0,0
6bba_c73a1d11,7904,29,14.04,180.0,115.35999999999999,cont,1.9362064230086673,30000194,1,1,0,0
6bba_c73a1d11,8124,29,50.28,202.56000000000003,97.28000000000003,cont,3.943898768984825,30000200,2,2,0,0
6bba_c73a1d11,8157,29,55.080000000000005,130.72,113.11999999999998,cont,3.658849076212348,30000205,4,4,0,0
6bba_c73a1d11,8338,30,16.48,123.35999999999999,194.56,cont,2.6840061591769877,31000212,1,0,1,0
6bba_c73a1d11,8627,30,56.400000000000006,142.72,228.0,cont,5.909484035218301,31000224,1,0,1,0
6bba_c73a1d11,8653,30,60.2,204.00000000000003,81.92,cont,0.875692583044974,31000225,4,1,1,2
6bba_c73a1d11,9618,33,5.160000000000001,134.56,167.20000000000002,cont,4.273997141143179,34000263,4,0,0,4
6bba_c73a1d11,10250,34,35.4,134.93333333333334,115.46666666666667,cont,0.9768789764972042,35000292,7,2,0,5
6bba_c73a1d11,10648,35,30.99999999999999,122.93333333333335,116.53333333333333,cont,5.6616475934327015,36000312,5,0,0,5
6bba_c73a1d11,11103,36,33.26666666666666,139.46666666666664,156.53333333333333,cont,0.9647902945718307,37000331,3,0,0,3
6bba_c73a1d11,11110,36,33.68,135.04,128.00000000000003,cont,6.292024341378548,37000330,1,0,1,0
6bba_c73a1d11,11822,38,10.68,109.28,182.07999999999998,cont,5.025744347855352,39000368,4,0,0,4
6bba_c73a1d11,11964,38,33.16,135.35999999999999,152.64,cont,2.970596005013134,39000371,1,1,0,0
6bba_c73a1d11,11999,38,36.68,184.64,221.28000000000006,cont,6.843430471992837,39000378,1,1,0,0
6bba_c73a1d11,12012,38,39.32,196.64000000000001,223.20000000000002,cont,1.569618783335629,39000378,2,2,0,0
6bba_c73a1d11,13298,41,28.76,126.56000000000002,86.71999999999998,cont,4.399400527344611,42000437,3,0,1,2
6bba_c73a1d11,14759,44,28.266666666666662,163.4666666666666,243.46666666666667,cont,2.224133726495165,45000509,5,0,0,5
6bba_c73a1d11,14877,44,45.32000000000001,105.92,27.999999999999996,cont,1.3767500226983895,45000517,2,2,0,0
6bba_c73a1d11,14980,44,59.36,216.8,58.72,cont,1.4665568732579066,45000523,1,1,0,0
6bba_c73a1d11,15208,45,23.880000000000003,128.64000000000004,83.99999999999999,cont,1.9768978223469131,46000529,2,0,2,0
6bba_c73a1d11,15347,45,43.84,104.00000000000001,23.360000000000007,cont,1.0824256614197596,46000537,2,0,2,0
6bba_c73a1d11,16393,47,54.16,140.00000000000003,167.84000000000003,cont,3.180063138131059,48000583,2,0,0,2
6bba_c73a1d11,16711,48,28.000000000000004,105.28000000000002,210.56000000000003,cont,3.438324901023748,49000597,1,1,0,0
6bba_c73a1d11,18553,52,18.199999999999996,126.56,103.84,cont,1.3337874877580764,53000693,1,1,0,0
6bba_c73a1d11,18720,52,42.040000000000006,191.36,224.0,cont,1.6963163214742558,53000707,1,1,0,0
6bba_c73a1d11,19440,54,16.0,125.28000000000002,105.44000000000001,cont,5.027530457391587,55000742,1,0,1,0
6bba_c73a1d11,19763,55,2.0000000000000004,163.36,20.000000000000007,cont,1.6456685571523788,56000757,2,0,2,0
6bba_c73a1d11,19851,55,10.399999999999999,120.48000000000002,43.84000000000002,cont,3.017647314133982,56000759,4,1,1,2
6bba_c73a1d11,21099,58,11.840000000000003,142.72,178.72000000000003,cont,1.5268948269281497,59000827,3,0,0,3
6bba_c73a1d11,21124,58,17.919999999999998,158.08000000000004,119.36000000000001,cont,3.4804601886819464,59000829,5,2,3,0
6bba_c73a1d11,21198,58,24.86666666666666,113.06666666666665,135.99999999999994,cont,2.507220475723845,59000834,4,0,0,4
6bba_c73a1d11,21508,59,3.2800000000000002,117.28,66.72,cont,6.089566897571617,60000845,5,0,2,3
6bba_c73a1d11,21509,59,7.440000000000001,116.63999999999999,79.51999999999998,cont,4.7414370184575825,60000845,6,0,2,4
6bba_c73a1d11,21954,60,9.640000000000002,156.79999999999998,98.08000000000001,cont,3.3233137171353566,61000866,7,1,2,4
6bba_c73a1d11,22021,60,17.36,109.28,133.28000000000003,cont,1.7154336186515642,61000875,1,0,1,0
6bba_c73a1d11,24618,67,27.64,130.88000000000002,12.639999999999997,cont,4.178999581239508,68000972,1,0,1,0
6bba_c73a1d11,25559,70,23.36,169.12,226.07999999999998,cont,1.0384754450635816,71001010,4,0,1,3
6bba_c73a1d11,25583,70,28.999999999999993,204.5333333333333,252.53333333333336,cont,6.2087027994089645,71001014,5,1,0,4
6bba_c73a1d11,26132,72,11.68,189.44,200.00000000000006,cont,2.2732582618347577,73001033,2,2,0,0
6bba_c73a1d11,26151,72,15.679999999999998,191.36,189.28,cont,6.454216887043695,73001033,1,1,0,0
6bba_c73a1d11,26712,74,13.319999999999999,192.0,236.64000000000004,cont,1.114636291576765,75001067,1,1,0,0
6bba_c73a1d11,27239,76,6.48,186.71999999999997,180.80000000000004,cont,5.545346726084837,77001092,1,1,0,0
6bba_c73a1d11,27932,78,44.32,227.36,234.24,cont,2.0192545809035582,79001119,2,2,0,0
6bba_c73a1d11,28091,79,15.680000000000001,171.36,140.79999999999998,cont,2.754363016198125,80001122,8,0,3,5
6bba_c73a1d11,28100,79,16.6,179.99999999999997,187.04000000000002,cont,6.636707697533474,80001127,5,0,3,2
6bba_c73a1d11,28347,80,14.520000000000003,170.72000000000003,136.8,cont,2.508402130839475,81001129,3,3,0,0
6bba_c73a1d11,28585,81,10.880000000000003,174.72,148.8,cont,6.005754662196915,82001142,6,0,3,3
6bba_c73a1d11,28588,81,13.160000000000004,168.64000000000004,129.91999999999996,cont,2.3571463812203195,82001141,3,0,3,0
6bba_c73a1d11,30088,88,6.44,173.91999999999996,96.80000000000001,cont,0.7860701304591102,89001187,3,0,0,3
6bba_c73a1d11,30296,89,6.28,177.76,97.44000000000003,cont,1.6977945252591728,90001193,3,3,0,0
6bba_c73a1d11,30298,89,6.640000000000001,184.16000000000003,77.91999999999999,cont,6.475905342112408,90001194,4,4,0,0
6bba_c73a1d11,30494,90,6.98752670288086,180.84020996093753,77.72497558593751,cont,1.9177118795624302,91001198,6,0,4,2
6bba_c73a1d11,30496,90,7.3199999999999985,188.8,54.08,cont,5.446853592901138,91001199,6,0,2,4
6bba_c73a1d11,30536,90,22.119999999999997,204.64,175.83999999999997,cont,5.52179686560452,91001196,3,0,3,0
6bba_c73a1d11,30566,90,32.64,221.28000000000006,139.68,cont,3.4694415977214543,91001197,3,1,2,0
6bba_c73a1d11,30681,91,7.787526702880859,180.84020996093753,77.5649755859375,cont,1.490273305807955,92001206,4,4,0,0
6bba_c73a1d11,30682,91,7.000000000000001,189.28000000000003,55.839999999999996,cont,6.223240840791886,92001205,4,4,0,0
6bba_c73a1d11,30866,92,4.315044021606445,168.86421875000002,32.31311401367188,cont,2.7441055023471956,93001212,5,1,1,3
6bba_c73a1d11,30872,92,6.36,190.72000000000003,60.160000000000004,cont,4.5864455532034984,93001208,8,0,4,4
6bba_c73a1d11,30902,92,16.68,197.44,154.08,cont,2.293726498517209,93001206,6,0,2,4
6bba_c73a1d11,30939,92,30.199999999999992,219.36,120.16000000000001,cont,2.009816922632499,93001207,5,0,4,1
6bba_cdcfe533,7089,22,2.84,24.64,241.28000000000003,cont,0.41778620429592206,23000354,1,1,0,0
6bba_cdcfe533,7374,23,2.8400000000000007,33.28000000000001,238.72000000000003,cont,1.0055323527863058,24000370,2,1,1,0
6bba_cdcfe533,7658,24,2.6399999999999997,42.720000000000006,235.35999999999999,cont,1.0278686990564523,25000386,2,1,1,0
6bba_cdcfe533,9121,29,10.866666666666662,88.53333333333333,168.53333333333333,daughter,1.4336270290188204,30000480,4,0,0,4
6bba_cdcfe533,9621,31,1.0,32.64,235.99999999999994,cont,0.6662499999999998,32000514,3,1,2,0
6bba_cdcfe533,9649,31,5.999999999999998,12.000000000000004,224.00000000000003,cont,0.8124999999999858,32000521,5,1,0,4
6bba_cdcfe533,9917,32,5.159999999999999,20.000000000000004,227.36000000000004,cont,1.1759656935897185,33000539,2,1,1,0
6bba_cdcfe533,13820,46,4.64,7.360000000000001,180.0,cont,6.801299623049994,47000863,1,0,1,0
6bba_cdcfe533,13826,46,4.32,146.08,243.36000000000004,cont,1.8028714076439147,47000866,1,1,0,0
6bba_cdcfe533,14396,48,5.480000000000002,36.63999999999999,222.72000000000003,cont,0.9625982092752939,49000924,2,1,1,0
6bba_cdcfe533,14525,48,31.840000000000003,135.35999999999999,123.20000000000002,parent,1.5210902750987636,49000935,1,1,0,0
6bba_cdcfe533,14679,49,5.160000000000001,68.64000000000001,228.0,cont,0.7151846352516303,50000944,1,1,0,0
6bba_cdcfe533,14807,49,32.160000000000004,132.64000000000001,124.64000000000003,daughter,1.0268405730686638,50000962,1,0,1,0
6bba_cdcfe533,14826,49,35.000000000000014,155.46666666666664,119.99999999999994,daughter,1.7778305384965825,50000963,3,0,0,3
6bba_cdcfe533,14838,49,37.36,101.43999999999998,89.92,cont,0.8630876621178227,50000964,1,0,1,0
6bba_cdcfe533,14969,50,5.840000000000002,68.64000000000001,226.72,cont,1.5231720602085672,51000970,1,0,1,0
6bba_cdcfe533,15658,52,30.160000000000004,33.28,140.64000000000001,parent,1.6054638153817085,53001039,1,1,0,0
6bba_cdcfe533,15672,52,34.8,124.64000000000001,129.28,cont,1.4652058302504805,53001040,1,1,0,0
6bba_cdcfe533,15941,53,30.679999999999996,36.0,141.76,daughter,1.6258122969764943,54001078,1,0,1,0
6bba_cdcfe533,15946,53,32.4,22.933333333333323,129.06666666666666,daughter,0.8426355809139678,54001079,4,0,0,4
6bba_cdcfe533,15969,53,36.36,123.19999999999999,130.72,cont,1.7441330224498341,54001066,4,0,1,3
6bba_cdcfe533,16747,56,18.000000000000004,50.72,244.00000000000003,cont,4.289322863226778,57001148,1,0,1,0
6bba_cdcfe533,18153,61,1.8399999999999999,202.72000000000006,217.28000000000003,cont,0.5924010993406271,62001296,1,1,0,0
6bba_cdcfe533,18456,62,2.6799999999999997,202.72000000000003,215.36000000000004,cont,0.5520218519225514,63001327,2,1,1,0
6bba_cdcfe533,19341,65,0.8,86.71999999999998,223.36000000000004,cont,2.5938990921005307,66001413,1,1,0,0
6bba_cdcfe533,19361,65,4.5200000000000005,134.72000000000003,239.36000000000007,cont,1.8799501323173229,66001414,4,2,2,0
6bba_cdcfe533,19953,67,3.0,212.0,223.36,cont,0.7803384682302852,68001474,4,3,1,0
6bba_cdcfe533,21518,72,15.360000000000003,71.99999999999999,177.92,cont,1.0931069538247462,73001624,1,1,0,0
6bba_cdcfe533,21551,72,22.000000000000004,61.27999999999999,169.28000000000003,cont,0.5322960290101831,73001626,1,1,0,0
6bba_cdcfe533,21832,73,20.133333333333333,91.99999999999999,179.99999999999997,cont,6.296452718815393,74001652,6,0,0,6
6bba_cdcfe533,22356,75,2.999999999999999,158.08,206.08,cont,1.4197144299118765,76001704,3,3,0,0
6bba_cdcfe533,22379,75,9.639999999999997,190.72000000000003,181.44,cont,2.5077704241018486,76001706,2,2,0,0
6bba_cdcfe533,22468,75,28.880000000000003,119.36000000000001,157.28000000000003,cont,3.576809201299388,76001718,3,3,0,0
6bba_cdcfe533,22482,75,30.080000000000002,32.48,151.36,cont,2.16277671639954,76001719,2,2,0,0
6bba_cdcfe533,22495,75,34.2,125.92,109.12,cont,3.67293139059253,76001720,3,3,0,0
6bba_cdcfe533,22574,75,53.32000000000001,55.040000000000006,81.92,cont,2.1127499852088563,76001726,4,4,0,0
6bba_cdcfe533,22655,76,4.200000000000001,154.71999999999997,199.84,cont,1.348847888014063,77001732,3,0,3,0
6bba_cdcfe533,22678,76,12.0,30.400000000000006,171.35999999999996,cont,3.4968097067183948,77001730,5,0,4,1
6bba_cdcfe533,22681,76,11.04,185.28,175.84000000000003,cont,1.192023620781072,77001735,2,0,2,0
6bba_cdcfe533,22692,76,15.96,83.52,189.12,cont,2.1918834651960846,77001736,4,0,2,2
6bba_cdcfe533,22697,76,15.200000000000005,157.12,146.55999999999997,cont,2.3607285024966256,77001737,1,0,1,0
6bba_cdcfe533,22749,76,29.44,32.16,145.27999999999997,cont,3.058541829859455,77001740,5,0,2,3
6bba_cdcfe533,22758,76,29.64,117.28,89.91999999999999,cont,2.5408786005041706,77001746,4,0,1,3
6bba_cdcfe533,22763,76,31.60000000000001,167.36,125.6,cont,4.341548722806192,77001749,6,0,2,4
6bba_cdcfe533,22774,76,32.68,49.91999999999999,127.84000000000002,cont,2.910020672177442,77001750,2,0,2,0
6bba_cdcfe533,22779,76,34.480000000000004,106.24000000000001,140.16,cont,3.079192771652986,77001751,4,0,1,3
6bba_cdcfe533,22797,76,37.86666666666666,115.99999999999997,95.99999999999997,cont,1.9292004136717162,77001747,6,0,0,6
6bba_cdcfe533,22804,76,41.76,80.64,95.36000000000001,cont,2.883539491666447,77001753,6,0,3,3
6bba_cdcfe533,22882,76,57.64,37.919999999999995,66.72,cont,2.347773546362597,77001757,5,0,3,2
6bba_cdcfe533,22895,76,60.96000000000001,81.12,68.64000000000001,cont,2.7711372327079067,77001755,3,0,1,2
6bba_cdcfe533,24152,81,13.32,245.43999999999997,231.52000000000004,cont,1.9029853191761488,82001879,1,1,0,0
6bba_cdcfe533,24441,82,14.319999999999999,241.28000000000003,224.64000000000001,cont,1.1204254214359872,83001904,2,1,1,0
6bba_cdcfe533,25289,85,4.840000000000001,244.00000000000003,234.71999999999997,cont,2.263654746753567,86001988,3,3,0,0
6bba_cdcfe533,25344,85,17.36,230.72000000000008,205.28000000000003,cont,0.6067150278343169,86001997,1,1,0,0
6bba_cdcfe533,25464,85,44.99999999999999,228.0,88.00000000000001,cont,0.4062500000000071,86002010,1,1,0,0
6bba_cdcfe533,25592,86,6.16,240.00000000000006,227.36000000000004,cont,1.8482901727001415,87002022,3,0,3,0
6bba_cdcfe533,25768,86,45.99999999999999,224.64000000000001,84.0,cont,0.431773233306557,87002044,1,0,1,0
6bba_cdcfe533,26160,88,0.32000000000000006,148.64000000000001,244.00000000000003,cont,1.395987553669437,89002086,1,0,1,0
6bba_cdcfe533,26175,88,3.7333333333333347,151.46666666666664,252.53333333333336,cont,5.362500000000008,89002086,4,0,0,4
6bba_cdcfe533,26187,88,7.360000000000001,124.63999999999999,237.92000000000002,cont,0.7094385984706546,89002087,2,1,1,0
6bba_cdcfe533,26190,88,7.68,233.27999999999997,217.92,cont,0.5975052301026411,89002088,2,0,2,0
6bba_cdcfe533,26484,89,8.16,120.00000000000003,234.56,cont,1.9886840026761434,90002123,1,0,1,0
6bba_cdcfe533,27031,91,0.0,38.08,223.20000000000002,cont,3.089808347616409,92002198,3,2,1,0
6bba_cdcfe533,27047,91,1.08,66.56,247.36,cont,4.864588882937591,92002199,3,3,0,0
6bba_cdcfe533,27054,91,3.8400000000000007,133.44,230.08000000000004,cont,1.0724999999999993,92002201,3,2,1,0
6bba_cdcfe533,27075,91,10.64,108.15999999999998,226.88,cont,2.3616231838504684,92002203,1,1,0,0
6bba_cdcfe533,27112,91,17.479999999999997,170.72000000000003,196.16000000000003,cont,1.9748263373015662,92002211,1,1,0,0
6bba_cdcfe533,27184,91,35.160000000000004,152.64000000000004,123.52000000000001,cont,1.982300194849402,92002222,1,1,0,0
6bba_cdcfe533,27217,91,40.72,245.92,119.36000000000001,cont,2.1843409332107484,92002225,1,1,0,0
6bba_cdcfe533,27322,92,0.0,33.28,222.39999999999998,cont,6.022351726070137,93002237,4,2,2,0
6bba_cdcfe533,27343,92,4.32,47.999999999999986,241.43999999999997,cont,1.536892664599587,93002244,10,4,1,5
6bba_cdcfe533,27353,92,5.520000000000001,125.92,223.99999999999997,cont,0.9381472765509649,93002239,3,1,2,0
6bba_cdcfe533,27379,92,12.266666666666666,157.86666666666662,219.46666666666667,cont,2.6193938150521383,93002246,3,0,1,2
6bba_cdcfe533,27424,92,20.733333333333327,228.53333333333333,157.59999999999997,cont,1.1620612224500972,93002254,7,2,0,5
6bba_cdcfe533,27517,92,44.13333333333331,231.99999999999994,113.86666666666667,cont,0.2233348880542488,93002263,6,0,0,6
6bba_cdcfe533,27559,92,53.0,235.35999999999999,70.72,cont,1.3576289073233532,93002268,2,1,1,0
6bba_cdcfe533,27625,93,0.8,215.99999999999997,194.56000000000006,cont,2.247028941179898,94002276,1,0,1,0
6bba_cdcfe533,27635,93,3.959999999999999,50.24000000000001,242.23999999999998,cont,3.413002939128531,94002275,12,1,4,7
6bba_cdcfe533,27640,93,4.319999999999999,19.520000000000003,206.08000000000004,cont,1.9293753036928776,94002274,6,0,1,5
6bba_cdcfe533,27716,93,21.16,70.56,175.36,cont,5.4705571299914295,94002296,3,0,0,3
6bba_cdcfe533,27724,93,23.959999999999997,76.64000000000001,166.07999999999998,cont,1.6005218875416902,94002296,3,0,2,1
6bba_cdcfe533,27728,93,23.0,219.99999999999997,140.53333333333333,cont,0.4483089367587551,94002290,9,1,0,8
6bba_cdcfe533,27780,93,36.86666666666667,59.46666666666664,72.53333333333332,cont,1.1858047109452723,94002304,4,0,0,4
6bba_cdcfe533,27786,93,39.800000000000004,124.16000000000003,111.36,cont,2.0359246210260324,94002305,5,0,3,2
6bba_cdcfe533,27846,93,54.120000000000005,60.32000000000001,59.36,cont,3.9767372383022757,94002308,7,0,4,3
6bba_cdcfe533,28050,94,29.68,245.28000000000003,127.36,cont,0.9089838076115482,95002341,2,0,2,0
6bba_cdcfe533,28156,94,54.84,222.55999999999997,63.36,cont,2.2536002446086023,95002351,1,0,1,0
6bba_cf35214c,2029,6,54.0,66.08,247.35999999999999,cont,1.8211603443958482,7000073,1,1,0,0
6bba_cf35214c,2318,7,53.0,68.0,244.64000000000001,cont,0.6857808049952947,8000085,1,0,1,0
6bba_cf35214c,3200,10,60.000000000000014,85.44,203.35999999999999,cont,1.6613191294871403,11000124,2,2,0,0
6bba_cf35214c,3489,11,58.999999999999986,84.64000000000001,202.08000000000004,cont,1.8852801515955304,12000137,3,1,2,0
6bba_cf35214c,3782,12,58.0,84.64000000000001,200.0,cont,1.763780120224739,13000149,1,0,1,0
6bba_cf35214c,7979,26,61.680000000000014,129.92000000000002,132.64000000000001,cont,1.321056040635674,27000276,1,0,1,0
6bba_cf35214c,8287,27,60.84000000000001,130.72,130.72,cont,0.6508119928212673,28000285,1,1,0,0
6bba_cf35214c,8294,27,61.839999999999996,144.64000000000001,172.32000000000005,cont,0.3254059964106491,28000286,2,2,0,0
6bba_cf35214c,8596,28,60.000000000000014,131.36000000000004,127.36,cont,1.501433169508382,29000294,1,0,1,0
6bba_cf35214c,8615,28,61.35999999999999,145.44,171.52000000000004,cont,2.1762869755617973,29000295,2,0,2,0
6bba_cf35214c,8924,29,60.32000000000001,145.28,168.0,cont,0.9713368687021056,30000304,1,1,0,0
6bba_cf35214c,8941,29,61.839999999999996,134.08,163.99999999999997,cont,5.878930201363168,30000304,2,1,1,0
6bba_cf35214c,9235,30,57.68000000000002,132.00000000000006,120.64000000000001,cont,2.2150126974805033,31000313,1,1,0,0
6bba_cf35214c,9246,30,59.360000000000014,144.64000000000001,165.28,cont,1.1831294572023583,31000314,1,0,1,0
6bba_cf35214c,9261,30,61.19999999999999,133.28000000000003,160.00000000000006,cont,5.8848562270033185,31000314,1,0,1,0
6bba_cf35214c,9556,31,56.36000000000001,132.00000000000006,117.92000000000002,cont,1.177424360627895,32000322,2,1,1,0
6bba_cf35214c,9868,32,54.64,131.35999999999999,117.28,cont,1.0563749926044312,33000331,1,0,1,0
6bba_cf35214c,10521,34,54.52000000000001,140.00000000000003,158.07999999999998,cont,2.4675398932945236,35000346,1,1,0,0
6bba_cf35214c,12926,42,8.48,72.64,166.72000000000003,cont,1.9387278367527336,43000389,1,1,0,0
6bba_cf35214c,13875,44,59.68000000000001,95.36000000000001,240.00000000000006,cont,0.9377249730064492,45000404,2,2,0,0
6bba_cf35214c,14216,45,58.15999999999998,94.72,235.99999999999994,cont,1.428706801621693,46000411,3,1,2,0
6bba_cf35214c,14242,45,61.99999999999999,87.99999999999997,232.0,cont,5.998209368219814,46000411,5,1,0,4
6bba_cf35214c,14554,46,56.84,93.91999999999999,232.0,cont,2.287155028632727,47000420,2,1,1,0
6bba_cf35214c,14572,46,60.000000000000014,196.64000000000001,60.0,cont,4.939064467335887,47000421,3,2,1,0
6bba_cf35214c,14593,46,62.26666666666667,191.46666666666664,51.99999999999999,cont,2.2209984694701825,47000421,6,2,0,4
6bba_cf35214c,14894,47,55.16000000000001,93.28,228.64000000000004,cont,1.808501624273505,48000429,1,0,1,0
6bba_cf35214c,14914,47,58.999999999999986,195.35999999999999,58.72,cont,5.661440298192692,48000430,2,0,2,0
6bba_cf35214c,15220,48,48.0,237.27999999999997,247.36,cont,2.2387287129082925,49000438,1,1,0,0
6bba_cf35214c,15577,49,47.0,236.64000000000001,246.72,cont,3.6446761258718143,50000447,1,0,1,0
6bba_cf35214c,15979,50,55.32000000000001,192.64,51.99999999999999,cont,6.53807597845114,51000458,1,1,0,0
6bba_cf35214c,16709,52,57.84,96.0,216.0,cont,1.9282800788526457,53000480,3,1,2,0
6bba_cf35214c,16906,53,28.16,32.64,250.08000000000004,cont,0.6114838509723703,54000487,2,2,0,0
6bba_cf35214c,17067,53,56.2,94.72,214.08,cont,1.3839238824805329,54000491,1,0,1,0
6bba_cf35214c,17270,54,26.48,32.63999999999999,247.35999999999999,cont,0.862322445492406,55000499,2,0,2,0
6bba_cf35214c,17443,54,58.0,224.64000000000001,37.28,cont,5.23492179979797,55000504,1,0,1,0
6bba_cf35214c,17462,54,60.84000000000001,213.44,53.28,cont,6.760390613714567,55000504,2,2,0,0
6bba_cf35214c,17471,54,61.92000000000001,218.72000000000003,36.96,cont,1.6382708109467239,55000504,2,1,1,0
6bba_cf35214c,17629,55,25.000000000000004,32.0,244.64000000000001,cont,1.8867502650059353,56000511,1,1,0,0
6bba_cf35214c,17799,55,57.16000000000001,225.28,37.44000000000001,cont,5.486053909687708,56000516,1,1,0,0
6bba_cf35214c,17819,55,60.000000000000014,212.64,52.64,cont,6.333582122306464,56000516,3,1,2,0
6bba_cf35214c,17835,55,61.68,216.0,35.36,cont,3.0435690397952238,56000516,2,1,1,0
6bba_cf35214c,17985,56,23.68,32.0,241.28000000000003,cont,1.0080240572526022,57000522,1,0,1,0
6bba_cf35214c,18184,56,61.160000000000004,213.92000000000002,33.28,cont,1.8887086362115304,57000527,1,0,1,0
6bba_cf35214c,19033,59,17.160000000000004,247.36,140.64000000000004,cont,3.187154883277554,60000551,1,1,0,0
6bba_cf35214c,19414,60,17.2,246.72,141.92000000000002,cont,2.9710404322391843,61000562,1,0,1,0
6bba_cf35214c,19643,60,58.680000000000014,202.56,33.28000000000001,cont,0.8355724085918419,61000569,1,1,0,0
6bba_cf35214c,20011,61,57.48000000000001,200.0,34.72,cont,0.9444591706368115,62000579,2,1,1,0
6bba_cf35214c,21317,65,10.680000000000001,27.999999999999996,223.99999999999997,cont,1.3859259089143363,66000612,1,1,0,0
6bba_cf35214c,22280,67,36.31999999999999,154.71999999999997,32.63999999999999,cont,0.7799999999999988,68000640,1,1,0,0
6bba_cf35214c,22398,67,57.68000000000002,182.07999999999993,97.92000000000002,cont,2.6975489453390757,68000644,2,2,0,0
6bba_cf35214c,22669,68,34.68000000000001,152.64,32.0,cont,0.8606365739962476,69000653,1,0,1,0
6bba_cf35214c,22792,68,56.0,179.36000000000004,96.63999999999999,cont,1.7623572637237808,69000656,3,1,2,0
6bba_cf35214c,23193,69,54.52,176.64000000000001,96.0,cont,0.9558536760404263,70000668,1,0,1,0
6bba_cf35214c,23893,71,36.00000000000001,235.99999999999994,208.0,cont,2.7252078475778942,72000690,1,0,1,0
6bba_cf35214c,24097,72,5.160000000000002,135.35999999999999,60.000000000000014,cont,1.0824256614197563,73000695,1,1,0,0
6bba_cf35214c,24168,72,17.32,69.28,178.72,cont,0.6073675267084987,73000698,1,1,0,0
6bba_cf35214c,24493,73,4.32,132.64000000000001,60.640000000000015,cont,1.4702433897147735,74000705,1,0,1,0
6bba_cf35214c,24566,73,16.48,68.64000000000001,176.00000000000003,cont,0.9558536760404263,74000708,1,0,1,0
6bba_cf35214c,25313,75,3.16,128.00000000000006,62.07999999999999,cont,1.76968394070805,76000725,1,1,0,0
6bba_cf35214c,27518,80,23.480000000000004,242.71999999999997,149.92000000000002,cont,2.551663979151639,81000787,1,0,1,0
6bba_cf35214c,29948,86,0.31999999999999995,87.36,93.27999999999999,cont,3.185538857477647,87000843,1,1,0,0
6bba_cf35214c,29959,86,0.84,97.28000000000003,152.0,cont,2.222451138383019,87000844,1,1,0,0
6bba_cf35214c,30394,87,0.0,83.99999999999997,111.99999999999999,cont,6.201137118504954,88000853,3,0,0,3
6bba_cf35214c,30402,87,0.8400000000000002,84.0,98.71999999999998,cont,1.8884289932375005,88000853,1,0,1,0
6bba_cf35214c,30404,87,1.1600000000000001,94.72,155.36,cont,1.8121482417285797,88000854,1,0,1,0
6bba_cf35214c,31302,89,2.0,76.0,108.0,cont,1.86167137607581,90000873,1,1,0,0
6bba_cf35214c,31765,90,2.8400000000000007,71.99999999999999,112.0,cont,0.9448784644069246,91000883,1,0,1,0
6bba_cf35214c,32220,91,3.4800000000000004,68.0,116.0,cont,1.2406531797807145,92000893,1,1,0,0
6bba_cf35214c,32707,92,4.360000000000001,64.0,121.28000000000003,cont,1.5708800280415987,93000903,1,0,1,0
6bba_cf35214c,33175,93,4.000000000000001,70.71999999999998,181.28000000000003,cont,0.9332085109984878,94000913,1,1,0,0
6bba_cf35214c,33183,93,4.840000000000001,59.36,126.55999999999999,cont,0.43238437760862336,94000912,2,2,0,0
6bba_cf35214c,33332,93,28.320000000000007,62.71999999999999,105.92,cont,1.1692098026017435,94000918,4,4,0,0
6bba_cf35214c,33458,93,45.320000000000014,95.67999999999999,147.36,cont,3.1622433196229434,94000921,2,1,1,0
6bba_cf35214c,33641,94,4.32,64.64,186.72,cont,1.430922997578832,95000922,1,0,1,0
6bba_cf35214c,33651,94,5.160000000000001,54.07999999999999,133.44,cont,1.3842100635380437,95000923,2,0,2,0
6bba_cf35214c,33816,94,28.68,56.63999999999999,112.0,cont,1.6193025118859037,95000928,4,0,4,0
6bba_cf35214c,33872,94,36.64000000000001,76.16000000000001,100.47999999999999,cont,2.4674328764932887,95000930,7,0,4,3
6bba_cf35214c,35612,98,6.08,30.71999999999999,158.23999999999998,cont,1.6827971505799504,99000963,2,2,0,0
6bba_cf35214c,35613,98,6.239999999999998,50.879999999999995,205.11999999999998,cont,3.2324432032906754,99000964,1,1,0,0
6bba_cf35214c,35823,98,37.160000000000004,54.399999999999984,120.0,cont,1.4854313346634436,99000968,2,2,0,0
6bba_cff5865f,1032,4,2.8400000000000007,48.000000000000014,155.35999999999999,cont,0.3676955262170076,5000004,3,3,0,0
6bba_cff5865f,2971,10,12.0,61.920000000000016,40.000000000000014,cont,0.8131497402077847,11000036,1,1,0,0
6bba_cff5865f,5955,18,0.6400000000000001,51.35999999999999,239.36,cont,0.8456247690317559,19000106,3,1,2,0
6bba_cff5865f,7720,22,0.0,140.53333333333333,244.53333333333327,cont,6.307161177428056,23000153,4,1,0,3
6bba_cff5865f,8171,23,1.4400000000000006,32.8,190.72000000000003,cont,1.3869734361190789,24000167,6,0,0,6
6bba_cff5865f,8686,24,6.999999999999999,86.72000000000003,238.72000000000003,cont,1.7099602517310115,25000188,2,1,1,0
6bba_cff5865f,10565,28,12.880000000000003,96.63999999999999,226.72,cont,2.148075446766237,29000267,1,1,0,0
6bba_cff5865f,11885,31,3.133333333333333,19.999999999999993,192.5333333333333,cont,5.197813125221233,32000321,4,0,0,4
6bba_cff5865f,11960,31,14.959999999999999,102.08000000000001,211.84,cont,0.5102603502134937,32000330,1,0,1,0
6bba_cff5865f,13280,34,0.31999999999999995,44.64000000000001,185.44,cont,2.806408403101731,35000402,1,1,0,0
6bba_cff5865f,13949,35,24.200000000000003,103.2,246.71999999999997,cont,1.9106019993708827,36000448,4,1,1,2
6bba_cff5865f,14932,37,25.84,104.64000000000003,242.71999999999997,cont,1.605463815381721,38000508,1,0,1,0
6bba_cff5865f,16195,40,4.36,37.28000000000001,223.84000000000006,cont,1.8085016242735517,41000579,4,0,4,0
6bba_cff5865f,16288,40,16.88,179.36,213.12000000000003,cont,3.7094518210242335,41000585,2,0,2,0
6bba_cff5865f,16360,40,25.733333333333334,96.5333333333333,228.0,cont,2.2988948507320193,41000593,4,0,0,4
6bba_cff5865f,16479,40,37.03999999999999,198.08,211.84,cont,3.2928210264908278,41000601,2,0,2,0
6bba_cff5865f,19764,47,20.48,182.55999999999997,70.08,cont,1.5547437489824472,48000782,1,1,0,0
6bba_cff5865f,19804,47,24.320000000000007,133.92000000000002,78.88,cont,2.1007171394550084,48000786,2,2,0,0
6bba_cff5865f,20260,48,19.32,37.12,154.56,cont,2.156051077780856,49000815,4,0,2,2
6bba_cff5865f,20271,48,18.56,89.12,207.36000000000007,cont,3.8720036802668405,49000818,5,0,1,4
6bba_cff5865f,20295,48,21.879999999999995,63.36000000000001,152.8,cont,2.8425890487370866,49000821,5,0,3,2
6bba_cff5865f,20450,48,37.68,187.2,189.12,cont,1.635770556190566,49000832,2,0,2,0
6bba_cff5865f,22280,52,11.840000000000003,164.64,229.28000000000003,cont,1.9062434426116661,53000963,2,1,1,0
6bba_cff5865f,22733,53,3.4800000000000004,4.0,139.35999999999999,cont,0.8915313370824384,54000985,1,0,1,0
6bba_cff5865f,22844,53,16.520000000000003,69.92,203.36,cont,1.233716438246651,54001007,1,0,1,0
6bba_cff5865f,22885,53,20.839999999999996,153.92000000000002,252.0,cont,1.885280151595518,54001013,1,0,1,0
6bba_cff5865f,23810,55,6.32,135.36000000000004,226.72,cont,0.7497926796788563,56001063,1,1,0,0
6bba_cff5865f,24868,57,7.0,6.880000000000001,112.00000000000001,cont,3.079321404546785,58001136,4,0,2,2
6bba_cff5865f,24928,57,14.16,37.92,60.0,cont,0.9170818188689615,58001147,2,2,0,0
6bba_cff5865f,26158,59,31.68,111.36,191.36,cont,2.4610033522935306,60001242,1,0,1,0
6bba_cff5865f,27090,61,16.36,113.28000000000002,216.0,cont,1.2316813965064186,62001313,1,0,1,0
6bba_cff5865f,28223,63,23.68,246.08000000000004,203.20000000000002,cont,0.6455157821463355,64001407,1,0,1,0
6bba_cff5865f,28575,64,4.360000000000001,121.44,225.28000000000003,cont,1.3827785659678073,65001426,6,0,1,5
6bba_cff5865f,30411,67,27.84,102.72,192.0,cont,1.671460813181092,68001597,1,1,0,0
6bba_cff5865f,31849,70,5.160000000000001,5.280000000000002,97.92000000000002,cont,1.3701168973850353,71001699,2,0,2,0
6bba_cff5865f,32124,70,33.13333333333333,219.99999999999997,252.0,cont,6.162217215170568,71001727,4,0,0,4
6bba_cff5865f,32478,71,11.84,139.20000000000002,247.35999999999999,cont,1.95250360114905,72001748,2,0,2,0
6bba_cff5865f,34379,74,31.68,124.00000000000003,239.36000000000007,cont,0.9756768483980919,75001890,1,0,1,0
6bba_cff5865f,35297,76,8.520000000000001,131.2,238.72000000000003,cont,1.0566249334555637,77001959,3,2,1,0
6bba_cff5865f,35842,77,7.000000000000001,131.36,234.56000000000006,cont,1.6613191294871672,78001996,2,0,2,0
6bba_cff5865f,35997,77,22.48,220.0,225.92000000000002,cont,1.2087422440288917,78002013,1,0,1,0
6bba_cff5865f,36507,78,15.680000000000001,132.64000000000001,237.28000000000003,cont,0.5520218519225455,79002039,1,0,1,0
6bba_cff5865f,39829,84,0.3989104413986206,47.352802124023434,164.37876953125004,cont,6.829902083782915,85002217,1,1,0,0
6bba_cff5865f,39919,84,6.36,91.67999999999999,206.88000000000002,cont,3.05931875586707,85002227,1,1,0,0
6bba_cff5865f,40471,85,2.3600000000000003,186.72000000000003,192.16,cont,1.4338725971996142,86002265,3,3,0,0
6bba_cff5865f,40513,85,6.5200000000000005,24.320000000000004,52.8,cont,3.5160884694501084,86002269,4,2,2,0
6bba_cff5865f,40620,85,15.520000000000003,232.63999999999993,200.79999999999998,cont,6.572416911798896,86002279,2,2,0,0
6bba_cff5865f,40632,85,17.0,198.08000000000004,144.8,cont,2.4819979980854074,86002283,2,2,0,0
6bba_cff5865f,40714,85,27.44,104.00000000000003,219.20000000000002,cont,1.8940837164708575,86002287,3,3,0,0
6bba_cff5865f,40717,85,25.36,154.56,232.8,cont,1.962956953170392,86002286,1,1,0,0
6bba_cff5865f,40814,85,37.4,141.86666666666662,223.99999999999997,cont,1.304506079369181,86002290,6,3,0,3
6bba_cff5865f,41100,86,6.280000000000001,13.440000000000001,30.880000000000003,cont,3.5699737393992126,87002293,5,0,2,3
6bba_cff5865f,41102,86,5.48,27.520000000000003,45.44000000000001,cont,6.3175519388446695,87002293,5,0,2,3
6bba_cff5865f,41108,86,4.040000000000001,126.56,175.2,cont,3.2838270832216465,87002307,6,0,2,4
6bba_cff5865f,41173,86,11.48,194.56,201.43999999999997,cont,1.6667937972346785,87002311,4,2,2,0
6bba_cff5865f,41204,86,13.680000000000001,69.28,80.0,cont,1.53929620687508,87002298,1,0,1,0
6bba_cff5865f,41222,86,15.319999999999997,193.92000000000002,140.16,cont,1.319556033293017,87002318,4,2,2,0
6bba_cff5865f,41315,86,23.680000000000007,228.96,133.60000000000002,cont,4.351997565773665,87002322,7,0,2,5
6bba_cff5865f,41354,86,27.840000000000003,100.00000000000001,215.52,cont,6.989861519551024,87002321,4,0,3,1
6bba_cff5865f,41421,86,36.36,136.48,212.0000000000001,cont,3.044002812909346,87002324,3,0,3,0
6bba_cff5865f,41704,87,4.640000000000001,154.72,249.44,cont,6.32375593002291,88002340,5,1,1,3
6bba_cff5865f,41709,87,4.065243377685547,143.0611083984375,230.34527343750003,cont,3.185142977951822,88002340,2,0,2,0
6bba_cff5865f,41803,87,13.320000000000004,190.56,133.28000000000003,cont,0.6385236878926293,88002348,2,0,2,0
6bba_cff5865f,41890,87,20.839999999999996,220.0,239.52,cont,2.35625,88002349,4,3,1,0
6bba_cff5865f,42303,88,4.840000000000001,8.64,15.200000000000003,cont,1.4568013977546848,89002359,1,1,0,0
6bba_cff5865f,43065,89,19.799999999999997,221.92000000000002,225.28,cont,1.7635555349917378,90002407,3,1,2,0
6bba_cff5865f,44405,91,36.0,127.99999999999997,166.9333333333333,cont,1.6520833333333371,92002474,3,2,0,1
6bba_cff5865f,44769,92,9.999999999999996,151.46666666666664,225.33333333333334,cont,1.995547343810435,93002489,4,0,0,4
6bba_cff5865f,44850,92,16.2,221.43999999999997,195.51999999999998,cont,3.5561148919853593,93002497,7,0,1,6
6bba_cff5865f,45297,93,3.1599999999999993,193.27999999999997,232.0,cont,0.9018384001582601,94002521,4,2,2,0
6bba_cff5865f,45344,93,9.6,72.16,58.55999999999998,cont,5.113253000781406,94002530,5,1,0,4
6bba_cff5865f,45348,93,7.68,181.28000000000003,147.36,cont,0.8527767146211191,94002527,3,3,0,0
6bba_cff5865f,45373,93,9.760000000000003,152.95999999999998,225.12,cont,2.737823611009293,94002531,2,2,0,0
6bba_cff5865f,45375,93,9.84,171.84,159.35999999999999,cont,6.364069305483713,94002527,3,3,0,0
6bba_cff5865f,45513,93,22.839999999999996,146.72000000000003,162.07999999999998,cont,1.923343849263561,94002537,4,4,0,0
6bba_cff5865f,45552,93,27.26666666666666,204.53333333333325,222.4,cont,1.3705879339620102,94002542,8,3,0,5
6bba_cff5865f,45911,94,2.999999999999999,158.9333333333333,183.99999999999997,cont,1.2458333333333371,95002553,4,0,0,4
6bba_cff5865f,45971,94,7.160000000000002,182.72,139.2,cont,1.9554091643438667,95002562,3,0,3,0
6bba_cff5865f,46012,94,10.119999999999997,64.16,51.359999999999985,cont,1.730529633522642,95002564,7,0,1,6
6bba_cff5865f,46131,94,22.16,143.36,153.28000000000003,cont,0.5994906692351535,95002568,4,0,4,0
6bba_d0fc38b5,2005,24,58.84000000000001,181.92,161.92,cont,1.935660820624293,25000256,1,1,0,0
6bba_d0fc38b5,2081,25,58.680000000000014,186.08000000000004,162.71999999999997,cont,0.9820164204329552,26000270,1,0,1,0
6bba_d1acb6ff,1843,6,44.32,77.28,237.27999999999997,cont,1.3123321321220607,7000094,2,2,0,0
6bba_d1acb6ff,4879,19,15.0,13.280000000000001,247.35999999999999,cont,1.8901760830673968,20000194,1,1,0,0
6bba_d1acb6ff,5109,20,14.28,17.28,244.00000000000003,cont,2.38869472725169,21000205,2,1,1,0
6bba_d1acb6ff,5226,20,45.36,190.56,42.24000000000001,cont,5.155451756393411,21000210,1,0,1,0
6bba_d1acb6ff,5250,20,51.48,163.36,87.84,cont,1.793104849137384,21000214,1,0,1,0
6bba_d1acb6ff,5728,22,44.04000000000001,73.28000000000002,248.0,cont,1.5641416376082826,23000233,1,0,1,0
6bba_d1acb6ff,5872,23,14.84,31.200000000000006,232.64,cont,2.8983368075501583,24000239,1,1,0,0
6bba_d1acb6ff,6115,24,15.000000000000004,36.64000000000001,229.44,cont,2.0411060880071847,25000251,2,1,1,0
6bba_d1acb6ff,7105,28,16.48,33.28,231.36,cont,2.6420121356647845,29000302,1,1,0,0
6bba_d1acb6ff,7360,29,16.0,37.28,226.72,cont,1.655028795066721,30000315,2,1,1,0
6bba_d1acb6ff,7477,29,48.16,182.07999999999993,49.920000000000016,cont,1.6159560830975814,30000327,1,1,0,0
6bba_d1acb6ff,7736,30,54.4,178.71999999999997,55.84,cont,4.406717173815443,31000341,4,0,0,4
6bba_d1acb6ff,8199,32,50.519999999999996,82.72,242.07999999999998,cont,6.1633788673502785,33000370,1,1,0,0
6bba_d1acb6ff,8389,33,34.4,184.64,60.0,cont,4.280294418903917,34000377,2,0,0,2
6bba_d1acb6ff,8403,33,40.84,113.28000000000002,192.64000000000004,cont,2.2302200003587114,34000378,1,1,0,0
6bba_d1acb6ff,8432,33,48.58035888671875,185.95773193359378,38.71869384765624,cont,2.348235701284852,34000379,2,0,2,0
6bba_d1acb6ff,8644,34,39.84000000000001,116.63999999999999,184.0,cont,1.945729699238799,35000389,1,0,1,0
6bba_d1acb6ff,8680,34,48.94035888671876,189.3177319335938,31.518693847656255,cont,5.254209426470385,35000390,2,2,0,0
6bba_d1acb6ff,8886,35,38.32000000000001,120.64000000000001,174.72,cont,3.7110531692499307,36000398,1,1,0,0
6bba_d1acb6ff,9476,37,59.160000000000004,69.28,247.35999999999999,cont,3.9197883393111974,38000419,1,1,0,0
6bba_d1acb6ff,9913,39,44.00000000000001,103.46666666666663,196.5333333333333,cont,3.3623715640485208,40000446,3,0,0,3
6bba_d1acb6ff,11395,46,24.73333333333333,76.5333333333333,183.46666666666667,cont,1.1081022628801132,47000537,4,0,0,4
6bba_d1acb6ff,11864,48,36.2,84.00000000000001,200.79999999999998,cont,3.2265560683955297,49000567,3,3,0,0
6bba_d1acb6ff,12033,49,18.919999999999998,145.44,79.52000000000001,cont,5.431099106074203,50000573,4,0,1,3
6bba_d1acb6ff,12077,49,35.0,84.64000000000001,195.36,cont,0.29831034594863415,50000586,3,0,3,0
6bba_d1acb6ff,12422,50,62.0,186.56000000000003,101.76,cont,6.587907126887876,51000598,1,1,0,0
6bba_d1acb6ff,12499,51,23.200000000000003,95.83999999999999,158.07999999999998,cont,3.0420503468055844,52000620,3,0,1,2
6bba_d1acb6ff,12620,51,59.52000000000001,177.28,105.44000000000001,cont,2.6227016724171888,52000617,5,1,0,4
6bba_d1acb6ff,13013,53,54.86666666666665,87.99999999999997,219.99999999999997,cont,3.583709403715357,54000652,3,1,0,2
6bba_d1acb6ff,13030,53,58.839999999999996,180.64000000000004,98.08,cont,1.9272527403015969,54000654,2,2,0,0
6bba_d1acb6ff,13031,53,59.68000000000001,90.72,216.64000000000001,cont,4.445169604469569,54000652,2,1,1,0
6bba_d1acb6ff,13218,54,53.31999999999999,88.00000000000001,216.32000000000002,cont,2.7631212536731167,55000670,1,0,1,0
6bba_d1acb6ff,13239,54,58.2,183.35999999999999,91.99999999999999,cont,1.3698277720209928,55000672,2,0,2,0
6bba_d1acb6ff,13354,55,32.00000000000001,25.28,244.00000000000003,cont,1.9651081395180385,56000684,1,0,1,0
6bba_d1acb6ff,13902,58,12.360000000000003,21.919999999999998,204.79999999999998,cont,2.7348803104523585,59000729,2,2,0,0
6bba_d1acb6ff,13908,58,13.200000000000001,99.19999999999999,124.8,cont,3.2377895623403306,59000745,1,0,1,0
6bba_d1acb6ff,13994,58,48.16000000000001,93.27999999999999,201.43999999999997,cont,4.729308386275926,59000740,1,0,1,0
6bba_d1acb6ff,14076,59,10.48,24.64,198.08000000000004,cont,0.8846927432730524,60000746,2,0,2,0
6bba_d1acb6ff,14090,59,15.319999999999997,130.56,67.03999999999999,cont,5.024929881351575,60000748,2,2,0,0
6bba_d1acb6ff,14109,59,24.733333333333324,25.06666666666665,211.99999999999994,cont,6.628338061036814,60000753,2,0,1,1
6bba_d1acb6ff,14275,60,9.959999999999997,109.44000000000003,105.92,cont,4.144259772564943,61000776,1,0,1,0
6bba_d1acb6ff,14282,60,10.040000000000003,141.92000000000002,47.36,cont,1.6966276292987765,61000761,2,0,1,1
6bba_d1acb6ff,14349,60,39.96,160.16,91.36,cont,1.5828534360451716,61000769,4,0,1,3
6bba_d1acb6ff,14371,60,49.44,194.72000000000003,56.48000000000002,cont,1.0691709697237364,61000771,1,0,1,0
6bba_d1acb6ff,14445,61,8.839999999999998,113.44,96.63999999999999,cont,5.181125903218336,62000794,1,1,0,0
6bba_d1acb6ff,14748,62,61.68,183.35999999999999,100.64,cont,1.2008519111447464,63000804,1,1,0,0
6bba_d1acb6ff,14911,63,61.160000000000004,185.28,95.36000000000001,cont,1.3941893926221167,64000818,1,0,1,0
6bba_d1acb6ff,14966,64,15.733333333333333,75.46666666666667,152.5333333333333,cont,0.7891434570680582,65000825,3,0,0,3
6bba_d1acb6ff,15411,67,4.16,118.72,56.64,cont,1.3941893926221078,68000865,1,1,0,0
6bba_d1acb6ff,15427,67,11.840000000000003,127.36,72.47999999999999,parent,2.2387876881249746,68000881,1,1,0,0
6bba_d1acb6ff,15430,67,12.520000000000005,79.36,136.64,cont,1.3513904876089575,68000868,1,0,1,0
6bba_d1acb6ff,15466,67,30.199999999999992,17.28,240.0,cont,1.3049670733394116,68000871,1,0,1,0
6bba_d1acb6ff,15581,68,8.4,131.99999999999997,59.466666666666654,daughter,4.926264026312887,69000897,3,0,0,3
6bba_d1acb6ff,15587,68,11.32,126.72000000000001,74.07999999999998,daughter,0.9478083271421555,69000898,1,0,1,0
6bba_d1acb6ff,15697,68,59.68000000000001,194.71999999999997,70.56,cont,0.8276340525860342,69000896,1,1,0,0
6bba_d1acb6ff,15903,70,8.999999999999998,83.36000000000001,121.28000000000003,cont,1.6495951207796373,71000916,1,1,0,0
6bba_d1acb6ff,16019,70,58.839999999999996,196.00000000000003,62.720000000000006,cont,0.5640880361255621,71000927,1,1,0,0
6bba_d1acb6ff,16174,71,58.0,196.00000000000003,58.08000000000001,cont,0.552021851922534,72000942,1,0,1,0
6bba_d1acb6ff,16277,72,31.720000000000006,120.0,132.8,cont,2.263888041070043,73000952,1,1,0,0
6bba_d1acb6ff,16341,72,57.040000000000006,196.64000000000001,51.99999999999999,cont,2.003236833851644,73000958,1,1,0,0
6bba_d1acb6ff,16456,73,43.866666666666646,95.46666666666664,172.79999999999993,cont,3.113641161842908,74000974,4,0,1,3
6bba_d1acb6ff,16610,74,44.32,98.72,165.44000000000003,cont,2.6735571459200136,75000991,2,0,0,2
6bba_d1acb6ff,16851,76,18.84,35.99999999999999,200.64000000000001,cont,1.8906649789161496,77001013,1,1,0,0
6bba_d1acb6ff,17287,79,18.13333333333333,131.99999999999997,84.53333333333333,cont,3.147731215582997,80001055,4,0,0,4
6bba_d1acb6ff,17320,79,34.00000000000001,116.0,136.0,cont,2.0312499999999885,80001058,5,0,0,5
6bba_d1acb6ff,17539,81,14.84,132.64000000000001,72.00000000000001,cont,1.389541291218072,82001078,2,0,2,0
6bba_d1acb6ff,17803,83,27.13333333333332,131.99999999999997,103.46666666666664,cont,2.4789758329914564,84001102,4,0,0,4
6bba_d1acb6ff,17806,83,27.239999999999995,55.84000000000001,156.64000000000004,cont,5.031546885650586,84001103,2,0,1,1
6bba_d1acb6ff,18273,87,49.2,100.96,133.76000000000002,cont,2.9266696623466033,88001146,1,1,0,0
6bba_d1acb6ff,18345,88,22.2,44.64000000000001,187.52000000000004,cont,3.5088337182744853,89001154,1,1,0,0
6bba_d1acb6ff,18370,88,37.88,65.28,150.24,cont,3.9846974869492837,89001155,1,1,0,0
6bba_d1acb6ff,18399,88,59.280000000000015,14.719999999999999,240.48000000000002,cont,5.1739601914297815,89001157,1,1,0,0
6bba_d1acb6ff,18451,89,19.679999999999996,45.28,177.28000000000003,cont,0.7914267180225961,90001164,1,0,1,0
6bba_d1acb6ff,18505,89,55.99999999999998,20.533333333333328,219.46666666666667,cont,1.7411445090482809,90001168,2,0,0,2
6bba_d1acb6ff,18719,91,61.52000000000001,87.36,142.08,cont,1.7792822464409521,92001183,2,2,0,0
6bba_d1acb6ff,18803,92,60.84000000000001,88.00000000000001,140.00000000000003,cont,0.6306172571377933,93001190,2,0,2,0
6bba_d1acb6ff,18917,94,8.133333333333331,52.53333333333333,83.99999999999997,cont,2.2607689817601657,95001198,1,0,0,1
6bba_d1acb6ff,18941,94,31.68,102.56,76.8,cont,1.9848960929983213,95001202,1,1,0,0
6bba_d1acb6ff,19026,95,29.479999999999997,103.36000000000003,68.64000000000001,cont,1.5934945873770698,96001209,1,0,1,0
6bba_d1acb6ff,19099,96,19.159999999999997,67.35999999999999,87.36,cont,0.7609790158079281,97001217,1,1,0,0
6bba_d1acb6ff,19185,97,17.160000000000004,68.0,79.84,cont,1.2505111454921174,98001226,1,0,1,0
6bba_d1acb6ff,19208,97,38.99999999999999,24.64,165.27999999999997,cont,2.1728869045580805,98001228,1,0,1,0
6bba_d1acb6ff,19220,97,53.0,91.35999999999997,111.20000000000002,cont,0.9056368615510325,98001229,1,0,1,0
6bba_d2b9fc0c,882,5,58.52000000000001,244.0,142.72,cont,6.358880601175034,6000056,1,1,0,0
6bba_d2b9fc0c,2060,13,46.519999999999996,156.0,170.72000000000003,cont,1.2772541348533455,14000129,1,1,0,0
6bba_d2b9fc0c,2209,14,46.0,152.64,171.35999999999996,cont,1.7623572637237968,15000139,1,0,1,0
6bba_d2b9fc0c,5377,33,55.480000000000004,245.28000000000003,237.27999999999997,cont,1.1023681893995216,34000285,1,1,0,0
6bba_d2b9fc0c,5551,34,55.35999999999999,242.71999999999997,237.27999999999997,cont,1.3973110337358778,35000293,1,0,1,0
6bba_d2b9fc0c,7443,45,26.84,244.64000000000001,38.71999999999999,cont,2.308758459648819,46000377,1,1,0,0
6bba_d2b9fc0c,7637,46,26.519999999999996,242.71999999999997,36.64000000000001,cont,2.412126266388234,47000385,1,0,1,0
6bba_d2b9fc0c,9950,57,61.080000000000005,247.03999999999996,203.68000000000006,cont,1.0282539812711409,58000503,1,0,1,0
6bba_d2b9fc0c,10373,59,59.68000000000001,234.72000000000006,200.00000000000003,cont,1.35694797707943,60000525,2,2,0,0
6bba_d2b9fc0c,10564,60,52.67999999999999,140.64000000000004,244.15999999999994,cont,1.6508752413492973,61000536,1,0,1,0
6bba_d2b9fc0c,10587,60,58.839999999999996,227.36000000000004,196.80000000000004,cont,2.3045227271606508,61000537,3,1,2,0
6bba_d2b9fc0c,10772,61,51.32000000000001,133.28,239.36,cont,1.1973284062862586,62000548,1,1,0,0
6bba_d2b9fc0c,10800,61,58.0,220.0,190.72000000000003,cont,2.0967538154251635,62000549,1,0,1,0
6bba_d2b9fc0c,10980,62,50.000000000000014,126.72000000000001,233.27999999999997,cont,2.1217298367134085,63000561,1,0,1,0
6bba_d2b9fc0c,11192,63,48.84,119.36000000000001,227.36,cont,1.5388672782277086,64000573,2,2,0,0
6bba_d2b9fc0c,11395,64,47.64000000000001,111.99999999999999,221.28000000000003,cont,0.9977552931455609,65000582,2,0,2,0
6bba_d2b9fc0c,11422,64,51.00000000000001,241.43999999999997,232.0,cont,2.0934137789744214,65000583,1,1,0,0
6bba_d2b9fc0c,11616,65,47.000000000000014,104.64,216.63999999999996,cont,1.7225766492379762,66000593,1,1,0,0
6bba_d2b9fc0c,11637,65,50.320000000000014,235.99999999999994,231.20000000000002,cont,2.802783493689782,66000595,1,0,1,0
6bba_d2b9fc0c,11833,66,46.0,97.28000000000003,211.36,cont,1.8316424494152779,67000603,1,0,1,0
6bba_d2b9fc0c,12331,68,56.64000000000001,237.27999999999997,156.0,cont,0.7699482531313409,69000626,2,1,1,0
6bba_d2b9fc0c,12547,69,56.32,232.0,152.0,cont,0.6598780663880228,70000637,1,0,1,0
6bba_d2b9fc0c,12998,71,55.51999999999999,221.28000000000003,143.99999999999997,cont,2.4076885310396934,72000657,1,1,0,0
6bba_d2b9fc0c,13221,72,54.84,216.0,140.00000000000003,cont,0.9448784644069224,73000671,1,0,1,0
6bba_d2b9fc0c,13403,73,47.86666666666666,131.46666666666658,124.53333333333335,daughter,2.107807287799354,74000680,6,0,0,6
6bba_d2b9fc0c,13423,73,49.68000000000001,246.72000000000003,89.27999999999997,cont,1.1017691738744344,74000685,2,1,1,0
6bba_d2b9fc0c,13461,73,59.68000000000001,184.8,247.35999999999999,cont,1.0662031056510664,74000683,1,1,0,0
6bba_d2b9fc0c,13642,74,49.0,240.0,85.28000000000002,cont,1.6511151534644692,75000700,1,0,1,0
6bba_d2b9fc0c,13685,74,59.160000000000004,179.36000000000004,244.00000000000003,cont,0.36769552621699186,75000698,2,1,1,0
6bba_d2b9fc0c,13911,75,58.839999999999996,173.27999999999997,240.00000000000006,cont,0.4955583971642713,76000713,2,1,1,0
6bba_d2b9fc0c,14601,78,57.0,151.46666666666664,231.4666666666667,parent,3.520624993835933,79000759,2,0,0,2
6bba_d2b9fc0c,15197,81,38.84,230.55999999999995,130.08,cont,1.9299910945390315,82000805,1,1,0,0
6bba_d2b9fc0c,15436,82,38.67999999999999,229.44,129.28,cont,2.2963153229685282,83000822,1,0,1,0
6bba_d2b9fc0c,15792,83,57.36,158.08000000000004,220.79999999999998,cont,2.917089999040145,84000844,1,1,0,0
6bba_d2b9fc0c,16028,84,57.16,153.92000000000002,216.64000000000004,cont,1.4729349951712067,85000861,2,1,1,0
6bba_d2b9fc0c,16038,84,58.999999999999986,145.92000000000002,204.64,cont,6.547762022248514,85000861,1,0,1,0
6bba_d2b9fc0c,16189,85,42.84,180.64000000000001,68.0,cont,2.057855755999425,86000871,1,1,0,0
6bba_d2b9fc0c,16210,85,46.16,145.27999999999997,245.28000000000003,cont,1.4896916627275616,86000872,1,1,0,0
6bba_d2b9fc0c,16268,85,55.199999999999996,112.64,223.36,cont,2.9480205223166376,86000875,1,1,0,0
6bba_d2b9fc0c,16272,85,57.16000000000002,149.92000000000002,213.92000000000002,cont,0.26403124815070056,86000876,1,0,1,0
6bba_d2b9fc0c,16432,86,42.68000000000001,176.00000000000003,68.0,cont,2.220608728479628,87000886,1,0,1,0
6bba_d2b9fc0c,16453,86,46.160000000000004,143.36,246.72000000000006,cont,1.48365258736672,87000887,1,0,1,0
6bba_d2b9fc0c,16504,86,54.32000000000001,110.07999999999998,221.28000000000003,cont,1.3838284756428394,87000890,1,0,1,0
6bba_d2b9fc0c,17228,89,50.999999999999986,239.36,188.00000000000006,cont,2.109748207725279,90000928,1,1,0,0
6bba_d2b9fc0c,17466,90,51.16,236.64000000000001,188.64,cont,4.511914262538692,91000941,2,1,1,0
6bba_d2b9fc0c,17704,91,51.00000000000001,234.72000000000006,191.36,cont,3.8111360265017873,92000954,1,0,1,0
6bba_d2b9fc0c,18105,93,39.160000000000004,136.8,60.64,cont,2.200480416295491,94000976,2,2,0,0
6bba_d2b9fc0c,18106,93,39.0,182.72,140.0,cont,1.7538711647381617,94000975,1,1,0,0
6bba_d2b9fc0c,18149,93,45.32,110.71999999999998,111.99999999999999,cont,1.814842021361641,94000977,2,2,0,0
6bba_d2b9fc0c,18197,93,52.0,76.16000000000001,212.0,cont,1.964570563761962,94000978,1,1,0,0
6bba_d2b9fc0c,18344,94,38.839999999999996,128.64000000000001,60.640000000000015,cont,0.729985016969532,95000987,2,0,2,0
6bba_d2b9fc0c,18346,94,39.16,177.28,143.36,cont,1.7604083333136116,95000986,1,0,1,0
6bba_d2b9fc0c,18397,94,45.0,101.92000000000002,112.64,cont,1.689531184974102,95000988,2,0,2,0
6bba_d2b9fc0c,18440,94,51.84,66.55999999999999,211.36,cont,2.0640058896475977,95000989,1,0,1,0
6bba_d3da753b,455,1,55.080000000000005,91.68,233.43999999999997,cont,2.0505929020651505,2000007,1,1,0,0
6bba_d3da753b,1168,4,53.999999999999986,87.46666666666664,216.5333333333333,cont,2.3885596107045415,5000017,4,0,0,4
6bba_d3da753b,1175,4,56.36000000000001,88.64,212.64,cont,3.5325720976365202,5000017,1,0,1,0
6bba_d3da753b,2322,9,35.52000000000001,215.04000000000002,242.72000000000003,cont,4.135106180619775,10000038,2,2,0,0
6bba_d3da753b,2570,10,35.32000000000001,212.64,236.0,cont,2.763933451622876,11000045,3,1,2,0
6bba_d3da753b,2593,10,40.48,180.32000000000002,240.31999999999994,cont,2.5774057329997606,11000046,2,1,1,0
6bba_d3da753b,2824,11,35.52,210.71999999999997,228.64000000000004,cont,4.152789160612408,12000051,2,1,1,0
6bba_d3da753b,2850,11,41.2,186.08,221.92000000000002,cont,6.147849852794063,12000052,1,1,0,0
6bba_d3da753b,3046,12,32.480000000000004,150.72000000000003,106.72,cont,1.8005263987234401,13000064,2,2,0,0
6bba_d3da753b,3084,12,40.0,179.2,229.28000000000003,cont,2.4129471735825465,13000059,4,3,1,0
6bba_d3da753b,3092,12,40.68,180.79999999999995,215.36000000000004,cont,5.67367068682169,13000059,3,2,1,0
6bba_d3da753b,3299,13,31.84,146.56000000000003,96.64000000000001,cont,0.4323843776086362,14000074,2,0,2,0
6bba_d3da753b,3332,13,39.63999999999999,175.20000000000005,208.8,cont,3.0442630471100958,14000065,4,0,2,2
6bba_d3da753b,3359,13,43.559999999999995,69.44,81.6,cont,4.23614155954449,14000073,4,0,1,3
6bba_d3da753b,3384,13,50.2,66.72,138.56,cont,5.582531347874368,14000069,6,1,1,4
6bba_d3da753b,3399,13,54.84,85.76,204.16000000000003,cont,2.93662274134761,14000071,5,0,1,4
6bba_d3da753b,3634,14,50.35999999999999,65.43999999999998,132.00000000000003,cont,6.402108173289483,15000081,1,0,1,0
6bba_d3da753b,3657,14,58.480000000000004,52.64000000000001,244.96,cont,2.5660597226097432,15000084,1,0,1,0
6bba_d3da753b,3885,15,48.84,132.00000000000006,225.92,cont,1.663939696172916,16000092,2,0,2,0
6bba_d3da753b,3914,15,58.519999999999996,53.440000000000005,239.36,cont,2.6243121122496094,16000094,1,1,0,0
6bba_d3da753b,4162,16,58.0,52.64,232.63999999999993,cont,1.8924100044652066,17000103,1,0,1,0
6bba_d3da753b,4662,18,57.48,51.36000000000001,220.64000000000004,cont,2.7488929589927595,19000122,2,1,1,0
6bba_d3da753b,4911,19,57.32000000000001,50.080000000000005,215.36000000000004,cont,1.195120940532783,20000131,1,0,1,0
6bba_d3da753b,5796,23,31.520000000000003,170.72000000000003,146.07999999999998,cont,2.6840553505842584,24000169,1,1,0,0
6bba_d3da753b,5850,23,43.80000000000001,41.92,70.56,cont,1.383637642231518,24000171,2,2,0,0
6bba_d3da753b,5896,23,55.68000000000001,44.00000000000001,186.08,cont,1.8618841263623198,24000175,3,2,1,0
6bba_d3da753b,6032,24,29.96,170.56000000000003,137.76,cont,1.1372678334499702,25000183,1,0,1,0
6bba_d3da753b,6089,24,41.480000000000004,35.99999999999999,49.44,cont,2.0762515051168515,25000185,4,0,2,2
6bba_d3da753b,6122,24,48.31999999999999,70.56,124.79999999999998,cont,1.8929680728950489,25000187,2,0,0,2
6bba_d3da753b,6139,24,53.0,101.28,240.48000000000002,cont,1.7363944324375178,25000188,2,2,0,0
6bba_d3da753b,6775,27,27.0,175.83999999999997,122.55999999999997,cont,2.48109350891899,28000221,4,4,0,0
6bba_d3da753b,6841,27,39.839999999999996,32.64,38.720000000000006,cont,0.3192618439463166,28000223,2,2,0,0
6bba_d3da753b,6879,27,50.800000000000004,99.52000000000001,222.56,cont,1.7849129005360418,28000226,4,4,0,0
6bba_d3da753b,6886,27,52.96,37.60000000000001,157.76000000000005,cont,3.810165883855985,28000227,2,2,0,0
6bba_d3da753b,6994,28,19.239999999999995,204.00000000000003,94.24,cont,4.9512392703746455,29000229,6,0,2,4
6bba_d3da753b,7008,28,26.120000000000005,170.07999999999998,100.0,cont,0.9008130008497905,29000230,7,0,3,4
6bba_d3da753b,7068,28,37.000000000000014,132.64000000000001,128.8,cont,1.114754737599263,29000232,5,0,2,3
6bba_d3da753b,7074,28,39.32,30.720000000000006,32.48000000000001,cont,2.1322204535413327,29000233,2,0,2,0
6bba_d3da753b,7102,28,45.0,70.72,90.72,cont,1.3642259710180014,29000235,6,0,2,4
6bba_d3da753b,7125,28,51.480000000000004,42.56000000000001,130.08,cont,1.883107886580051,29000237,5,0,1,4
6bba_d3da753b,8276,33,32.36000000000001,131.84,106.56,cont,2.3128722673334106,34000285,4,4,0,0
6bba_d3da753b,8295,33,36.160000000000004,186.07999999999998,226.08,cont,2.2670934034573897,34000287,2,2,0,0
6bba_d3da753b,8317,33,42.96,62.71999999999999,73.92,cont,1.6863241273551213,34000288,3,3,0,0
6bba_d3da753b,8334,33,47.12,40.64000000000001,106.56,cont,1.089598549925612,34000290,4,4,0,0
6bba_d3da753b,8417,34,8.839999999999998,198.72,36.63999999999999,cont,0.8020325741514492,35000309,3,0,3,0
6bba_d3da753b,8448,34,14.4,154.88,38.08,cont,4.202691102139197,35000306,7,0,2,5
6bba_d3da753b,8458,34,19.48,165.59999999999997,48.800000000000004,cont,6.461372725667512,35000306,4,0,2,2
6bba_d3da753b,8502,34,28.16,127.04000000000002,64.8,cont,2.249143239658165,35000308,6,0,2,4
6bba_d3da753b,8520,34,31.32,189.44,191.36,cont,3.3054273551236917,35000310,3,0,1,2
6bba_d3da753b,8550,34,38.16,57.279999999999994,55.04,cont,6.803318239102149,35000305,9,0,4,5
6bba_d3da753b,8571,34,41.56,98.56,141.92000000000002,cont,2.3763923550205304,35000300,5,0,3,2
6bba_d3da753b,8572,34,44.48,36.00000000000001,72.64,cont,1.3010152285811245,35000304,8,0,5,3
6bba_d3da753b,9260,37,42.32,124.63999999999999,222.72,cont,1.4492595091977156,38000342,1,1,0,0
6bba_d3da753b,9754,39,52.86666666666666,76.5333333333333,252.53333333333336,cont,6.9852951947883675,40000374,2,0,0,2
6bba_d3da753b,10085,41,22.319999999999997,238.72000000000003,238.72000000000003,cont,1.532763537699131,42000395,2,2,0,0
6bba_d3da753b,10305,42,21.520000000000003,240.64000000000004,236.00000000000006,cont,2.419013228570688,43000411,2,0,2,0
6bba_d3da753b,10582,43,37.36,19.520000000000003,27.52,cont,2.528376764349015,44000440,3,2,1,0
6bba_d3da753b,10716,44,17.560000000000002,243.84000000000003,208.00000000000006,cont,4.337623917538261,45000440,4,0,2,2
6bba_d3da753b,10717,44,19.919999999999995,107.35999999999999,3.8399999999999994,cont,2.189894760599248,45000454,3,0,1,2
6bba_d3da753b,10728,44,20.56,193.27999999999997,124.16,cont,3.372648254710237,45000443,4,0,2,2
6bba_d3da753b,10781,44,33.4,92.63999999999999,72.64,cont,1.1301638432103567,45000447,3,0,1,2
6bba_d3da753b,10785,44,36.36,16.640000000000004,17.919999999999998,cont,1.463041566395157,45000453,2,0,2,0
6bba_d3da753b,10792,44,37.67999999999999,140.79999999999998,177.92,cont,5.309996763181682,45000449,5,0,1,4
6bba_d3da753b,10825,44,47.040000000000006,95.83999999999999,213.44,cont,3.860974010855284,45000451,4,0,1,3
6bba_d3da753b,12136,51,35.160000000000004,167.36,228.00000000000003,cont,4.635154056231567,52000537,3,2,1,0
6bba_d3da753b,12996,56,52.32000000000001,67.36,220.0,cont,1.186361296148849,57000613,1,1,0,0
6bba_d3da753b,13134,57,44.36,36.480000000000004,105.44000000000001,cont,3.939108599988074,58000633,2,0,1,1
6bba_d3da753b,13143,57,51.16,68.64000000000001,212.64,cont,0.6636687803415081,58000629,1,0,1,0
6bba_d3da753b,13910,63,20.999999999999996,186.56,135.20000000000002,parent,1.7059404481106675,64000722,1,0,1,0
6bba_d3da753b,14084,64,40.73333333333333,84.53333333333333,149.33333333333331,cont,2.8668702992113286,65000746,3,0,0,3
6bba_d3da753b,14201,65,40.48,86.72,149.12,cont,1.8695976338506621,66000762,1,1,0,0
6bba_d3da753b,14202,65,39.68,102.55999999999999,145.75999999999996,cont,6.75462677077719,66000762,1,1,0,0
6bba_d3da753b,14611,69,9.680000000000003,177.92000000000002,56.8,cont,2.8322589261930133,70000821,3,0,0,3
6bba_d3da753b,14915,72,7.8,175.36,41.28000000000001,cont,0.7843886950995654,73000861,2,2,0,0
6bba_d3da753b,15007,73,5.319999999999999,175.84000000000003,86.24,cont,2.5082968574313633,74000866,3,0,2,1
6bba_d3da753b,15168,74,51.08,50.72000000000002,228.0,cont,2.1611889089110172,75000889,1,0,1,0
6bba_d3da753b,15544,79,23.36,128.0,144.0,cont,2.786105032119212,80000956,1,1,0,0
6bba_d3da753b,15585,80,1.5333333333333328,167.99999999999994,52.53333333333332,cont,1.3413771038459654,81000962,1,0,1,0
6bba_d3da753b,16307,90,17.04,195.36,200.16000000000003,cont,2.596849653907586,91001096,2,2,0,0
6bba_d3da753b,16370,91,15.96,196.80000000000004,189.12,cont,2.6365593701830314,92001108,2,0,2,0
6bba_d3da753b,16605,94,52.160000000000004,30.080000000000002,166.08,cont,1.846646453710072,95001147,1,0,1,0
6bba_d5eae175,133,3,56.839999999999996,19.36,92.64,cont,1.234144440493091,4000022,1,0,1,0
6bba_d6ecebbb,646,11,12.0,136.64,84.64000000000001,cont,1.8411328883326108,12000062,1,1,0,0
6bba_d6ecebbb,708,12,12.16,135.36,84.0,cont,1.2730088619094513,13000067,1,0,1,0
6bba_d6ecebbb,962,16,33.84,141.28000000000003,118.71999999999997,cont,2.0233697141155362,17000092,1,1,0,0
6bba_d6ecebbb,1016,17,34.16,140.64,118.56000000000003,cont,1.7507819681502352,18000097,1,0,1,0
6bba_d6ecebbb,1229,21,58.999999999999986,86.72,18.720000000000002,cont,1.9683975875061637,22000119,1,1,0,0
6bba_d6ecebbb,1278,22,58.999999999999986,85.92,17.920000000000005,cont,1.7241854961111613,23000124,1,0,1,0
6bba_d6ecebbb,1611,30,13.479999999999999,136.0,103.2,cont,1.5182231596507811,31000160,1,1,0,0
6bba_d6ecebbb,1631,30,35.68,142.08,120.64000000000001,cont,1.756503985193323,31000163,1,1,0,0
6bba_d6ecebbb,1664,31,13.84,135.36000000000004,106.08000000000001,cont,1.4389280949721013,32000166,1,0,1,0
6bba_d6ecebbb,1688,31,35.68,141.28000000000003,120.63999999999999,cont,2.4539114185316526,32000168,1,0,1,0
6bba_d6ecebbb,1750,32,35.32000000000001,140.0,120.00000000000003,cont,2.0066611728191655,33000173,1,1,0,0
6bba_d6ecebbb,1760,32,57.84,150.71999999999997,182.72,cont,2.7047342753032004,33000174,1,1,0,0
6bba_d6ecebbb,1813,33,35.16,138.72000000000003,120.00000000000003,cont,2.076569436594902,34000181,1,0,1,0
6bba_d6ecebbb,1827,33,58.0,149.28,181.92,cont,2.8017940680928026,34000182,1,0,1,0
6bba_d6ecebbb,2333,41,58.15999999999998,88.00000000000001,21.279999999999998,cont,1.5863529320425733,42000247,1,1,0,0
6bba_d6ecebbb,2355,42,12.64,125.28,116.63999999999999,cont,1.1259502930858014,43000249,1,1,0,0
6bba_d6ecebbb,2377,42,33.68000000000001,134.72000000000003,120.00000000000003,cont,1.6490347555464107,43000253,1,1,0,0
6bba_d6ecebbb,2388,42,57.84,88.00000000000001,22.560000000000002,cont,1.8934562478441284,43000254,1,0,1,0
6bba_d6ecebbb,2412,43,12.68,124.63999999999999,117.28,cont,1.2272784932524394,44000259,1,0,1,0
6bba_d6ecebbb,2434,43,33.31999999999999,133.27999999999997,120.00000000000003,cont,2.164851322839495,44000261,1,0,1,0
6bba_d6ecebbb,2508,44,56.84,88.00000000000001,26.720000000000002,cont,1.9554091643438631,45000270,1,1,0,0
6bba_d6ecebbb,2572,45,56.160000000000004,88.00000000000001,27.36,cont,1.3895412912180718,46000278,1,0,1,0
6bba_d6ecebbb,2831,49,30.479999999999997,129.92000000000002,126.08,cont,1.5036299827417712,50000309,1,1,0,0
6bba_d6ecebbb,2893,50,30.0,130.08,125.92000000000004,cont,2.097320463114773,51000317,1,0,1,0
6bba_d6ecebbb,2911,50,55.16000000000001,148.64000000000001,184.0,cont,1.4477099372802378,51000319,1,1,0,0
6bba_d6ecebbb,2972,51,54.52,149.28000000000003,185.28,cont,0.8407706658179782,52000327,1,0,1,0
6bba_d6ecebbb,3037,52,53.52,151.36,187.35999999999999,cont,2.4813063595614344,53000335,1,1,0,0
6bba_d6ecebbb,3105,53,52.48000000000001,152.64,188.64,cont,1.4412201861617049,54000343,1,0,1,0
6bba_d6ecebbb,3556,60,6.679999999999999,120.63999999999999,141.28000000000003,cont,2.621694645167506,61000402,1,1,0,0
6bba_d6ecebbb,3627,61,5.84,120.00000000000003,143.36,cont,2.444909571436126,62000413,1,0,1,0
6bba_d6ecebbb,3697,62,5.319999999999999,120.00000000000003,144.64000000000001,cont,2.240202627554036,63000422,1,1,0,0
6bba_d6ecebbb,3776,63,4.64,120.00000000000003,147.35999999999996,cont,1.467186934238465,64000434,1,0,1,0
6bba_d6ecebbb,4006,65,60.16000000000001,127.36,54.080000000000005,cont,1.0582481100857226,66000466,1,1,0,0
6bba_d6ecebbb,4095,66,60.000000000000014,130.08,57.44000000000001,cont,0.7357500424736583,67000478,1,0,1,0
6bba_d6ecebbb,4320,69,19.32,128.0,153.27999999999997,cont,2.745624928864103,70000508,1,1,0,0
6bba_d6ecebbb,4380,70,3.4799999999999995,80.00000000000003,76.64000000000001,cont,0.8915313370824433,71000516,1,1,0,0
6bba_d6ecebbb,4404,70,18.48,128.0,154.72000000000003,cont,2.602690014677121,71000520,1,0,1,0
6bba_d6ecebbb,4458,70,59.84000000000002,130.72,55.36,cont,3.802673607148022,71000527,1,1,0,0
6bba_d6ecebbb,4472,71,3.0,80.00000000000003,78.72000000000001,cont,0.8635464666131322,72000530,1,0,1,0
6bba_d6ecebbb,4548,71,59.68000000000001,129.28,53.28,cont,3.0269087469727327,72000541,1,0,1,0
6bba_d6ecebbb,4665,73,16.48,129.28,158.56,cont,2.4830616811710495,74000561,1,1,0,0
6bba_d6ecebbb,4719,73,59.999999999999986,159.46666666666664,131.99999999999997,cont,2.04277287208945,74000570,1,0,0,1
6bba_d6ecebbb,4760,74,16.16,131.36,162.07999999999998,cont,1.8270232791346677,75000575,1,0,1,0
6bba_d6ecebbb,5237,79,14.880000000000003,132.8,168.0,cont,1.9274582518176588,80000639,1,1,0,0
6bba_d6ecebbb,5292,79,60.32000000000001,136.8,69.44000000000001,cont,5.7347767229858295,80000650,1,1,0,0
6bba_d6ecebbb,5293,79,59.84000000000002,165.28000000000003,135.20000000000002,cont,2.610389398633842,80000649,1,1,0,0
6bba_d6ecebbb,5324,80,13.639999999999999,129.28,168.0,cont,1.4507164178432683,81000652,1,0,1,0
6bba_d6ecebbb,5362,80,35.96,170.72000000000003,223.99999999999997,cont,2.2729678423814326,81000657,2,1,0,1
6bba_d6ecebbb,5382,80,52.52000000000001,140.64000000000004,75.2,cont,4.270381276888502,81000665,1,0,0,1
6bba_d6ecebbb,5388,80,59.68000000000001,166.72000000000003,138.72,cont,1.5219580316158428,81000663,1,0,1,0
6bba_d6ecebbb,5446,81,34.8,169.28,223.99999999999997,cont,3.3232739881027022,82000671,1,0,1,0
6bba_d6ecebbb,5687,84,21.86666666666666,155.99999999999997,235.99999999999997,cont,0.9338842310181781,85000720,2,0,0,2
6bba_d6ecebbb,6048,88,29.840000000000003,116.64000000000001,73.28,cont,0.4698470495810293,89000788,1,1,0,0
6bba_d6ecebbb,6123,89,28.879999999999995,115.36000000000001,73.92,cont,1.8387648707760387,90000804,1,0,1,0
6bba_d6ecebbb,6124,89,30.640000000000008,171.84,227.36,cont,3.5179654986938154,90000805,1,1,0,0
6bba_d6ecebbb,6130,89,34.559999999999995,143.99999999999997,102.56000000000003,cont,2.3858741191647277,90000806,1,1,0,0
6bba_d6ecebbb,6148,89,52.160000000000004,153.27999999999997,99.19999999999999,cont,4.27254498220908,90000810,1,1,0,0
6bba_d6ecebbb,6154,89,57.68000000000001,171.99999999999994,148.64000000000001,cont,2.1607001180172882,90000812,1,1,0,0
6bba_d6ecebbb,6208,90,22.720000000000006,161.28,226.88,cont,6.27504643907118,91000818,3,0,1,2
6bba_d6ecebbb,6215,90,32.96,143.99999999999997,106.08,cont,1.2555688451853264,91000824,1,0,1,0
6bba_d6ecebbb,6217,90,32.64,211.36,82.08,cont,1.84607438230425,91000823,1,1,0,0
6bba_d6ecebbb,6230,90,41.999999999999986,148.53333333333336,100.53333333333332,cont,3.2627268173603503,91000826,1,0,0,1
6bba_d6ecebbb,6239,90,57.00000000000001,171.99999999999994,150.72,cont,0.8204238614887106,91000829,1,0,1,0
6bba_d6ecebbb,6300,91,31.32,209.92,84.00000000000001,cont,1.1777607195436548,92000838,1,0,1,0
6bba_d6ecebbb,6327,91,59.03999999999999,132.00000000000006,71.99999999999999,cont,1.6120294856174489,92000845,1,1,0,0
6bba_d6ecebbb,6420,92,58.680000000000014,132.00000000000006,71.99999999999999,cont,2.2206087284796276,93000859,1,0,1,0
6bba_d6ecebbb,6422,92,60.68000000000001,188.00000000000003,199.36000000000004,cont,1.4356210546310755,93000860,1,1,0,0
6bba_d6ecebbb,6433,93,2.84,113.28000000000002,184.64,cont,0.41778620429593016,94000861,1,1,0,0
6bba_d6ecebbb,6521,93,60.31999999999999,188.00000000000003,198.56000000000003,cont,2.088614971697758,94000878,1,0,1,0
6bba_d6ecebbb,6535,94,2.6800000000000006,112.64,186.72000000000006,cont,0.552021851922544,95000879,1,0,1,0
6bba_d82a4fc6,720,1,58.999999999999986,121.76,24.0,cont,1.677846927612913,2000016,1,1,0,0
6bba_d82a4fc6,1094,2,58.999999999999986,120.64000000000001,24.0,cont,1.6813842883172458,3000023,1,0,1,0
6bba_d82a4fc6,1613,4,17.84,19.360000000000003,105.92000000000002,cont,0.5724522359987714,5000036,1,1,0,0
6bba_d82a4fc6,3549,9,56.84,166.72,65.28,cont,0.7541820900816957,10000093,1,1,0,0
6bba_d82a4fc6,3894,10,56.160000000000004,165.27999999999997,67.36,cont,1.739660976311185,11000102,1,0,1,0
6bba_d82a4fc6,4008,11,11.000000000000002,3.36,164.0,cont,1.6456685571523777,12000105,1,1,0,0
6bba_d82a4fc6,4039,11,15.679999999999998,11.36,106.72,cont,1.3004061865432686,12000106,1,1,0,0
6bba_d82a4fc6,6364,18,5.800000000000001,4.0,40.8,cont,2.205754193580963,19000174,1,1,0,0
6bba_d82a4fc6,6690,19,3.0,4.0,58.55999999999999,cont,4.168243784857116,20000182,5,0,0,5
6bba_d82a4fc6,9147,26,15.039999999999996,3.3599999999999994,111.99999999999999,cont,5.384539935082665,27000251,1,1,0,0
6bba_d82a4fc6,9316,26,45.720000000000006,121.28000000000003,64.47999999999999,cont,2.3139566358296193,27000258,1,1,0,0
6bba_d82a4fc6,9500,27,17.68,19.039999999999996,113.44,cont,2.3546244392683926,28000264,5,0,0,5
6bba_d82a4fc6,9640,27,44.120000000000005,122.71999999999998,71.52000000000001,cont,2.7354595706206335,28000268,1,0,1,0
6bba_d82a4fc6,10167,29,17.999999999999996,13.440000000000001,116.0,cont,0.46561283541156767,30000292,1,0,1,0
6bba_d82a4fc6,10727,31,4.0,4.6400000000000015,180.0,cont,1.8989569505389003,32000310,1,1,0,0
6bba_d82a4fc6,12099,35,11.840000000000003,7.359999999999999,163.36,cont,2.0161136035699894,36000359,1,1,0,0
6bba_d82a4fc6,12443,36,16.32,13.919999999999998,134.72000000000003,cont,6.671451303876849,37000373,2,0,2,0
6bba_d82a4fc6,13054,38,10.68,6.56,164.64,cont,2.3605047791733056,39000398,1,1,0,0
6bba_d82a4fc6,13204,38,36.36000000000001,120.00000000000003,96.0,cont,1.11652992010961,39000405,3,3,0,0
6bba_d82a4fc6,13349,39,5.040000000000001,9.280000000000001,156.95999999999998,cont,4.948678643082418,40000410,6,2,0,4
6bba_d82a4fc6,13590,39,58.120000000000026,42.720000000000006,4.64,cont,1.4736519263381058,40000420,1,0,1,0
6bba_d82a4fc6,13686,40,10.160000000000002,6.720000000000001,165.27999999999997,cont,1.5376656414838763,41000422,7,0,3,4
6bba_d82a4fc6,14972,44,10.0,6.720000000000001,160.00000000000003,cont,2.2324685048170254,45000461,1,0,1,0
6bba_d82a4fc6,14992,44,12.999999999999998,-0.5333333333333313,163.99999999999994,cont,5.072181243749517,45000461,4,0,0,4
6bba_d82a4fc6,15237,45,2.5199999999999996,8.0,193.27999999999997,cont,1.0472113265716845,46000468,1,1,0,0
6bba_d82a4fc6,15311,45,14.000000000000002,16.0,168.64000000000004,cont,6.091171120975024,46000469,1,1,0,0
6bba_d82a4fc6,15539,46,2.0,8.0,194.07999999999998,cont,0.37375000000000114,47000476,1,0,1,0
6bba_d82a4fc6,15586,46,9.320000000000002,8.639999999999999,156.64,cont,1.7622823447166513,47000477,1,0,1,0
6bba_d82a4fc6,16890,50,13.839999999999998,19.36,168.63999999999996,cont,6.719586227030637,51000507,1,1,0,0
6bba_d82a4fc6,17171,51,9.266666666666664,23.999999999999993,152.5333333333333,cont,6.395337928696364,52000515,5,0,0,5
6bba_d82a4fc6,17208,51,14.000000000000002,20.000000000000007,168.64000000000001,cont,6.8299505854727895,52000515,1,0,1,0
6bba_d82a4fc6,17747,53,2.999999999999999,-0.5333333333333313,187.46666666666664,cont,6.919407401267999,54000533,3,0,0,3
6bba_d82a4fc6,17935,53,32.68,179.36000000000004,169.92000000000002,cont,1.2170154323179225,54000537,1,1,0,0
6bba_d82a4fc6,18241,54,32.0,180.0,172.0,cont,0.5745242597140698,55000546,1,0,1,0
6bba_d82a4fc6,18693,56,2.0,17.279999999999998,196.00000000000003,cont,1.7003588775608487,57000556,1,1,0,0
6bba_d82a4fc6,19315,58,2.1599999999999997,18.72,195.35999999999999,cont,1.483652587366734,59000576,2,1,1,0
6bba_d82a4fc6,19501,58,31.0,185.60000000000002,175.20000000000002,cont,3.58330002406998,59000582,1,1,0,0
6bba_d82a4fc6,19575,58,50.88,44.64000000000001,36.64,cont,0.35639032324124376,59000583,2,2,0,0
6bba_d82a4fc6,19630,59,2.5199999999999996,19.36,194.72000000000003,cont,0.8300235312929373,60000586,2,1,1,0
6bba_d82a4fc6,19697,59,13.439999999999998,11.36,157.92000000000002,cont,6.247718208474192,60000588,2,2,0,0
6bba_d82a4fc6,19781,59,26.640000000000008,10.720000000000002,130.72000000000003,cont,5.689101560220213,60000591,3,2,1,0
6bba_d82a4fc6,19785,59,27.4,184.5333333333333,185.06666666666663,cont,2.123236604939642,60000592,1,0,0,1
6bba_d82a4fc6,19853,59,49.559999999999995,45.28,39.36,cont,4.22315584752682,60000593,3,1,2,0
6bba_d82a4fc6,19921,60,2.48,18.72,193.92000000000002,cont,2.6618273798276246,61000595,1,0,1,0
6bba_d82a4fc6,19966,60,7.160000000000001,26.079999999999995,204.00000000000003,cont,6.559849464736222,61000595,3,1,2,0
6bba_d82a4fc6,19978,60,9.320000000000002,18.72,152.0,cont,2.863321934397175,61000597,5,0,1,4
6bba_d82a4fc6,20016,60,13.84,24.0,164.0,cont,6.6138897405384665,61000597,2,0,2,0
6bba_d82a4fc6,20110,60,31.48000000000001,6.72,135.84000000000003,cont,5.7174603737149035,61000600,3,0,2,1
6bba_d82a4fc6,20551,62,2.8400000000000007,16.48,193.44,cont,0.4049479133172588,63000614,4,0,0,4
6bba_d82a4fc6,20933,63,8.84,18.560000000000002,150.56,cont,2.3909046405074377,64000623,1,0,1,0
6bba_d82a4fc6,21562,65,8.520000000000001,20.639999999999997,152.0,cont,1.7205059212336324,66000640,1,1,0,0
6bba_d82a4fc6,21868,66,8.319999999999999,22.08,152.64,cont,3.20434178108391,67000650,1,0,1,0
6bba_d82a4fc6,22454,68,3.6399999999999997,24.639999999999993,194.72000000000003,cont,1.226955709265811,69000665,1,1,0,0
6bba_d82a4fc6,22785,69,4.16,24.64,193.27999999999997,cont,1.0355471078130685,70000673,2,1,1,0
6bba_d82a4fc6,22814,69,8.319999999999999,23.36,152.63999999999996,cont,1.8410611749749173,70000674,1,1,0,0
6bba_d82a4fc6,23120,70,8.32,23.36,152.0,cont,2.893458408634899,71000688,2,1,1,0
6bba_d82a4fc6,23428,71,8.36,24.639999999999993,152.0,cont,2.8727132209811685,72000698,1,0,1,0
6bba_d82a4fc6,24364,74,8.519999999999998,25.28,152.0,cont,2.5912509044860967,75000730,1,1,0,0
6bba_d82a4fc6,24757,75,22.733333333333334,15.999999999999996,131.46666666666664,cont,5.854637436587252,76000743,2,0,0,2
6bba_d82a4fc6,24768,75,24.840000000000003,8.0,141.44,cont,1.5588146538315655,76000743,1,0,1,0
6bba_d82a4fc6,25151,76,45.160000000000004,79.36,92.64,cont,3.733718195378433,77000757,1,1,0,0
6bba_d82a4fc6,25271,77,7.999999999999998,36.0,151.99999999999997,cont,4.542013079296453,78000762,5,0,0,5
6bba_d82a4fc6,25278,77,8.520000000000001,25.28,152.64,cont,1.270101496928499,78000762,1,0,1,0
6bba_d82a4fc6,25469,77,45.000000000000014,80.00000000000003,95.36000000000001,cont,1.0507356054212738,78000769,1,0,1,0
6bba_d82a4fc6,27203,83,2.3600000000000003,49.760000000000005,194.72000000000003,cont,2.612967661491431,84000830,8,0,1,7
6bba_d82a4fc6,27302,83,17.8,14.4,158.72,cont,6.8491195830194105,84000837,3,2,1,0
6bba_d82a4fc6,27336,83,22.64,13.280000000000001,156.80000000000004,cont,1.8055059020950204,84000837,3,2,1,0
6bba_d82a4fc6,27919,85,13.733333333333329,7.4666666666666694,129.06666666666666,cont,0.9168418393230893,86000861,8,0,0,8
6bba_d82a4fc6,27956,85,18.200000000000003,26.240000000000002,162.72,cont,6.47731196658768,86000862,6,0,1,5
6bba_d82a4fc6,28610,87,22.360000000000003,22.560000000000006,164.0,cont,1.4436001004433259,88000889,3,2,1,0
6bba_d82a4fc6,28924,88,22.159999999999993,24.639999999999993,164.0,cont,1.9726857485418274,89000902,2,0,2,0
6bba_d82a4fc6,30146,92,12.48,201.28,207.35999999999999,cont,2.8255381124663703,93000952,2,1,0,1
6bba_d82a4fc6,30442,93,11.439999999999998,11.68,143.35999999999999,cont,5.367569573373034,94000960,2,2,0,0
6bba_d82a4fc6,30451,93,11.840000000000003,208.16000000000003,212.8,cont,2.869034027333952,94000966,2,0,1,1
6bba_d82a4fc6,30470,93,14.959999999999999,8.0,124.00000000000003,cont,6.072936120609198,94000960,1,1,0,0
6bba_d82a4fc6,30492,93,18.0,47.466666666666654,167.46666666666664,cont,6.5300045330297385,94000968,7,0,0,7
6bba_d82a4fc6,30506,93,20.800000000000004,35.99999999999999,172.0,cont,1.5003775566503272,94000968,1,0,1,0
6bba_d82a4fc6,30751,94,11.266666666666664,11.999999999999996,144.53333333333333,cont,5.569560905543237,95000975,2,0,2,0
6bba_d82a4fc6,30770,94,12.999999999999998,22.93333333333333,133.06666666666666,cont,1.7805103240051396,95000975,4,0,0,4
6bba_d82a4fc6,31014,95,3.6799999999999997,76.80000000000001,189.28000000000003,cont,1.7836549554215835,96000993,4,0,0,4
6bba_debd7bfa,1620,6,19.439999999999998,249.27999999999997,9.119999999999997,cont,3.6821223499498177,7000185,2,2,0,0
6bba_debd7bfa,2952,11,53.68,240.64000000000004,147.35999999999996,cont,1.6075184680121088,12000324,3,2,1,0
6bba_debd7bfa,3172,12,52.00000000000001,240.64000000000004,141.28000000000003,cont,1.7201221795267996,13000350,2,0,2,0
6bba_debd7bfa,4202,17,33.00000000000001,133.92000000000002,167.36000000000007,cont,1.3261433746016917,18000468,1,1,0,0
6bba_debd7bfa,4245,17,45.6,241.44000000000003,109.28,cont,1.7685644779029097,18000475,2,2,0,0
6bba_debd7bfa,4379,18,23.639999999999997,128.00000000000006,58.55999999999998,cont,1.1394715057867824,19000487,1,0,1,0
6bba_debd7bfa,4404,18,31.48,134.07999999999998,161.91999999999996,cont,1.6198731624729144,19000491,1,0,1,0
6bba_debd7bfa,4447,18,40.360000000000014,248.64000000000001,96.80000000000001,cont,2.8146763801545416,19000499,4,0,0,4
6bba_debd7bfa,4646,19,40.68,42.08,244.8,cont,2.6266756556529733,20000521,1,1,0,0
6bba_debd7bfa,4845,20,40.00000000000001,41.28000000000001,242.07999999999998,cont,1.8260835824244064,21000544,1,0,1,0
6bba_debd7bfa,4875,20,50.0,19.36,235.99999999999994,cont,2.0542596750411195,21000548,1,1,0,0
6bba_debd7bfa,5068,21,49.160000000000004,18.560000000000002,232.0,cont,1.558814653831554,22000573,1,0,1,0
6bba_debd7bfa,5987,26,27.360000000000003,169.44,109.44000000000001,parent,4.889549122874209,27000673,1,0,1,0
6bba_debd7bfa,6001,26,31.866666666666667,176.00000000000003,113.06666666666665,parent,3.17349182246027,27000673,3,0,0,3
6bba_debd7bfa,6262,27,57.52000000000001,137.28000000000003,238.72000000000003,cont,2.4632019938486454,28000690,2,1,1,0
6bba_debd7bfa,6690,30,19.0,134.56000000000003,97.76,cont,2.6581046466420486,31000747,2,2,0,0
6bba_debd7bfa,6707,30,22.839999999999996,122.72000000000001,85.11999999999999,cont,6.754783143262257,31000747,1,1,0,0
6bba_debd7bfa,6758,30,37.68,83.99999999999999,141.44,cont,2.618721658557855,31000738,1,1,0,0
6bba_debd7bfa,6869,31,13.400000000000002,128.8,76.16000000000001,cont,4.745055650358588,32000764,4,0,2,2
6bba_debd7bfa,6882,31,18.36,137.44,96.16,cont,4.926403987697314,32000750,4,0,2,2
6bba_debd7bfa,6909,31,27.16,44.64000000000001,180.96,cont,3.9567331853436873,32000753,4,0,2,2
6bba_debd7bfa,6943,31,35.99999999999999,86.08,134.56,cont,2.0247395665862924,32000756,1,0,1,0
6bba_debd7bfa,6969,31,44.879999999999995,193.44,130.24,cont,3.909332583702754,32000759,3,0,1,2
6bba_debd7bfa,6980,31,50.35999999999999,131.52,211.52000000000004,cont,3.0044487847190764,32000761,4,0,1,3
6bba_debd7bfa,7487,34,46.52,82.56000000000002,229.43999999999994,cont,0.8931588954379871,35000810,1,0,1,0
6bba_debd7bfa,7505,34,57.68000000000002,120.63999999999999,243.36000000000004,cont,2.627077749134919,35000816,1,1,0,0
6bba_debd7bfa,7944,37,40.84,86.71999999999998,206.07999999999998,parent,1.2491378276635434,38000863,1,1,0,0
6bba_debd7bfa,8088,38,38.84,87.36,197.91999999999996,daughter,0.8069560861657732,39000897,3,0,1,2
6bba_debd7bfa,8095,38,42.13333333333333,84.53333333333333,212.5333333333333,daughter,2.1051957054709525,39000898,3,0,0,3
6bba_debd7bfa,8298,40,4.0,141.28000000000003,19.360000000000003,cont,0.5401750295043366,41000927,1,1,0,0
6bba_debd7bfa,8303,40,6.680000000000001,144.00000000000003,38.720000000000006,cont,2.244206778017566,41000911,1,1,0,0
6bba_debd7bfa,8375,40,36.800000000000004,146.08,156.64,cont,3.1670828947945155,41000919,1,1,0,0
6bba_debd7bfa,8434,41,3.3200000000000003,143.04,12.0,cont,0.9647902945718311,42000944,1,0,1,0
6bba_debd7bfa,8509,41,33.13333333333333,151.99999999999997,152.0,cont,0.8408927960474184,42000936,3,0,0,3
6bba_debd7bfa,8515,41,36.96,146.72000000000003,152.64,cont,6.867350744646727,42000936,4,0,1,3
6bba_debd7bfa,8534,41,46.48000000000001,43.36,220.0,cont,1.6258122969764917,42000940,1,0,1,0
6bba_debd7bfa,9001,45,19.439999999999998,97.44,73.75999999999999,cont,2.8872002008866695,46001000,1,1,0,0
6bba_debd7bfa,9124,46,17.359999999999996,20.000000000000004,110.88,cont,3.1005153301507806,47001017,1,1,0,0
6bba_debd7bfa,9125,46,18.266666666666655,93.06666666666663,69.06666666666668,cont,0.7206406293476934,47001018,1,0,1,0
6bba_debd7bfa,9146,46,26.920000000000005,93.92000000000002,155.2,cont,2.107368768156169,47001031,3,3,0,0
6bba_debd7bfa,9177,46,40.56000000000002,48.64,190.88,cont,3.5462995678594145,47001026,1,1,0,0
6bba_debd7bfa,9241,47,13.866666666666664,87.46666666666664,47.466666666666654,cont,2.5517329691473183,48001035,1,1,0,0
6bba_debd7bfa,9243,47,15.160000000000004,17.28,99.36000000000001,cont,0.3192618439463216,48001034,1,0,1,0
6bba_debd7bfa,9259,47,21.733333333333324,93.06666666666663,131.46666666666664,cont,2.06722312737,48001048,3,0,1,2
6bba_debd7bfa,9274,47,27.6,96.16000000000001,152.95999999999998,cont,6.605120291296746,48001039,6,1,3,2
6bba_debd7bfa,9287,47,35.86666666666665,151.99999999999997,95.46666666666664,cont,4.08886583109821,48001043,2,1,0,1
6bba_debd7bfa,9291,47,38.16,49.44000000000001,180.00000000000003,cont,2.0297544494347153,48001044,1,0,1,0
6bba_debd7bfa,9773,52,5.160000000000001,8.0,46.72000000000001,cont,0.2837940494443108,53001113,2,2,0,0
6bba_debd7bfa,9778,52,6.880000000000001,70.08,10.719999999999999,cont,1.6865589968927868,53001114,1,1,0,0
6bba_debd7bfa,9787,52,15.279999999999998,134.23999999999998,60.64,cont,2.8379870176764466,53001116,2,2,0,0
6bba_debd7bfa,9866,53,3.6400000000000006,5.919999999999999,35.36,cont,1.256725034564046,54001128,2,0,2,0
6bba_debd7bfa,9879,53,9.680000000000001,132.8,39.52,cont,3.7759140035890617,54001131,2,0,1,1
6bba_debd7bfa,9890,53,17.000000000000004,89.92,100.00000000000001,cont,1.7162031129793385,54001132,2,0,1,1
6bba_debd7bfa,9997,54,20.560000000000006,145.28000000000003,33.120000000000005,cont,4.298393777040433,55001150,2,2,0,0
6bba_debd7bfa,10090,55,19.160000000000004,141.28000000000003,23.840000000000003,cont,2.0934137789744263,56001163,2,0,2,0
6bba_debd7bfa,10898,64,61.35999999999999,142.56,175.83999999999997,cont,3.4417024312104583,65001274,1,1,0,0
6bba_debd7bfa,10966,65,59.133333333333326,131.46666666666664,157.6,cont,1.5591157795009742,66001285,1,0,0,1
6bba_debd7bfa,11042,66,58.6,132.96,152.8,cont,2.8437964281924173,67001298,1,1,0,0
6bba_debd7bfa,11048,66,60.44000000000001,68.00000000000001,192.8,cont,1.338629196230217,67001299,2,2,0,0
6bba_debd7bfa,11076,67,23.0,82.08,108.0,cont,1.6753269270503592,68001304,1,0,1,0
6bba_debd7bfa,11118,67,57.36,134.72,145.28,cont,1.4865863790913807,68001310,1,0,1,0
6bba_debd7bfa,11240,69,37.99999999999999,51.99999999999999,163.99999999999997,cont,0.908402615859299,70001332,1,0,0,1
6bba_debd7bfa,11762,77,32.36000000000001,152.16000000000003,47.360000000000014,cont,2.9794719540549335,78001429,1,1,0,0
6bba_debd7bfa,11792,77,61.84,123.36000000000003,163.36,cont,2.678441711144733,78001435,1,1,0,0
6bba_debd7bfa,11809,78,23.33333333333333,36.53333333333333,113.86666666666665,cont,1.7081834283823305,79001438,2,0,0,2
6bba_debd7bfa,11832,78,42.160000000000004,135.67999999999998,89.91999999999999,cont,0.8673603994303567,79001443,2,0,2,0
6bba_debd7bfa,11929,80,16.88,37.120000000000005,93.60000000000001,cont,3.61824413562711,81001462,2,0,1,1
6bba_debd7bfa,11939,80,30.120000000000005,155.35999999999999,71.2,cont,0.926392532893051,81001464,2,0,2,0
6bba_debd7bfa,11943,80,32.99999999999999,192.5333333333333,52.53333333333332,cont,1.7537373286942346,81001465,1,0,0,1
6bba_debd7bfa,12045,82,15.32,28.000000000000004,92.0,cont,2.3294248887869298,83001489,1,0,1,0
6bba_debd7bfa,12177,84,22.720000000000006,119.36000000000001,28.160000000000004,cont,2.4701069055812033,85001519,1,1,0,0
6bba_debd7bfa,12178,84,24.199999999999996,141.6,57.92000000000001,cont,4.194546496642997,85001520,2,2,0,0
6bba_debd7bfa,12186,84,32.2,110.71999999999998,67.36,cont,1.9772317295906423,85001523,1,1,0,0
6bba_debd7bfa,12207,84,57.2,220.79999999999998,96.0,cont,2.863414155339749,85001528,1,1,0,0
6bba_debd7bfa,12209,84,58.040000000000006,107.35999999999999,140.16,cont,1.9236184165525072,85001527,1,1,0,0
6bba_debd7bfa,12211,84,58.36000000000001,244.00000000000003,50.239999999999995,cont,2.7889942743756126,85001531,1,1,0,0
6bba_debd7bfa,12214,84,59.88,202.08000000000004,90.08000000000001,cont,2.0150655231282073,85001529,2,2,0,0
6bba_debd7bfa,12237,85,21.68,154.56,40.16000000000001,cont,3.510188073664998,86001534,2,0,1,1
6bba_debd7bfa,12246,85,30.799999999999997,107.36000000000001,65.28,cont,1.0154624869979176,86001538,1,0,1,0
6bba_debd7bfa,12268,85,53.36,239.04000000000002,70.24000000000001,cont,4.3400887447724,86001543,3,0,1,2
6bba_debd7bfa,12270,85,56.96,104.64000000000003,135.84,cont,1.50152110374779,86001541,1,0,1,0
6bba_debd7bfa,12274,85,57.64,244.00000000000003,44.480000000000004,cont,2.816458325628123,86001544,1,0,1,0
6bba_debd7bfa,12275,85,59.160000000000004,200.0,86.56,cont,2.3010826256351558,86001545,2,0,2,0
6bba_debd7bfa,12683,92,23.44,77.12,53.92,cont,1.858050894082294,93001639,1,0,1,0
6bba_debd7bfa,12711,92,50.800000000000004,224.48,35.99999999999999,cont,1.1338960765872577,93001643,1,0,1,0
6bba_df673a83,2055,8,7.999999999999998,175.99999999999994,111.46666666666665,cont,6.706010532122966,9000059,1,0,1,0
6bba_df673a83,2062,8,10.160000000000002,160.00000000000006,116.0,cont,0.944878464406907,9000059,1,1,0,0
6bba_df673a83,2296,9,9.639999999999999,160.64,116.0,cont,1.6312442528634428,10000066,1,0,1,0
6bba_df673a83,4455,18,7.840000000000002,171.35999999999996,108.0,cont,4.617316760034989,19000129,1,1,0,0
6bba_df673a83,4691,19,7.160000000000002,171.99999999999994,106.72,cont,1.9971640049079578,20000136,1,0,1,0
6bba_df673a83,4814,19,43.0,80.00000000000003,231.36,cont,0.6857808049953146,20000139,1,1,0,0
6bba_df673a83,5057,20,43.360000000000014,80.00000000000003,230.72000000000008,cont,0.6540498834187227,21000145,1,0,1,0
6bba_df673a83,5298,21,43.0,80.64,228.64000000000001,cont,1.7754194046759784,22000154,1,1,0,0
6bba_df673a83,5545,22,43.0,81.28,226.72,cont,1.6550287950667213,23000161,1,0,1,0
6bba_df673a83,5554,22,45.99999999999999,83.99999999999997,235.99999999999997,cont,5.3433844904236265,23000161,2,0,0,2
6bba_df673a83,5790,23,46.0,83.99999999999999,235.68,cont,6.276561185274946,24000168,1,1,0,0
6bba_df673a83,6320,25,57.68000000000002,43.36,200.0,cont,2.1880249370150837,26000184,1,1,0,0
6bba_df673a83,6563,26,57.84,44.64000000000001,198.72,cont,2.1163090659211283,27000190,1,0,1,0
6bba_df673a83,7205,29,37.84,110.71999999999998,152.64,cont,0.5994906692351443,30000207,1,1,0,0
6bba_df673a83,7448,30,36.79999999999999,112.0,152.63999999999996,cont,0.5816047304656339,31000212,1,0,1,0
6bba_df673a83,12011,49,19.64,150.56000000000003,34.080000000000005,cont,3.2741229741718647,50000347,3,1,1,1
6bba_df673a83,12251,50,19.160000000000004,149.92000000000002,32.00000000000001,cont,1.3010152285811263,51000356,1,0,1,0
6bba_df673a83,12261,50,20.84,156.79999999999995,24.0,cont,6.241057853040612,51000356,1,0,1,0
6bba_df673a83,12594,51,47.96000000000001,74.08000000000001,140.16,parent,1.7453438056726938,52000372,1,1,0,0
6bba_df673a83,12823,52,41.88000000000001,84.48,139.36,daughter,2.9772997938568246,53000382,2,0,0,2
6bba_df673a83,12848,52,49.720000000000006,72.64,134.71999999999997,daughter,2.199099957937319,53000381,1,0,1,0
6bba_df673a83,13795,56,36.96,104.00000000000001,161.92000000000002,daughter,0.4435386820785744,57000415,1,1,0,0
6bba_df673a83,16098,66,1.3200000000000005,177.28,26.080000000000002,cont,2.745817273599975,67000516,1,1,0,0
6bba_df673a83,16328,67,1.1600000000000001,178.72,24.0,cont,3.0619502057512293,68000528,1,0,1,0
6bba_df673a83,16800,68,61.000000000000014,52.0,146.72000000000003,cont,2.8817532098533443,69000548,1,1,0,0
6bba_df673a83,17037,69,60.640000000000015,38.879999999999995,148.64000000000001,cont,1.7790596181690992,70000561,3,0,0,3
6bba_df673a83,17057,69,61.52000000000001,52.0,140.64000000000001,cont,5.061294609583598,70000561,3,0,1,2
6bba_df673a83,20950,86,30.360000000000003,136.64,46.72000000000001,cont,1.1719168325866738,87000709,1,1,0,0
6bba_df673a83,21179,87,29.160000000000004,138.72000000000003,41.28000000000001,cont,0.30574192548618656,88000717,1,0,1,0
6bba_df673a83,21521,88,62.000000000000014,47.360000000000014,59.36,cont,0.6821129855089934,89000733,1,1,0,0
6bba_df673a83,21746,89,62.000000000000014,48.0,54.720000000000006,cont,0.8635464666131188,90000743,1,0,1,0
6bba_df673a83,22385,92,53.67999999999999,91.36,20.640000000000004,cont,0.5994906692351509,93000764,1,1,0,0
6bba_df673a83,22603,93,52.52000000000001,92.64,15.359999999999996,cont,2.413877197580676,94000769,1,0,1,0
6bba_e16ffc58,117,1,24.0,168.96,75.36000000000001,cont,1.6853843218981246,2000026,1,1,0,0
6bba_e16ffc58,191,2,23.840000000000003,170.72000000000003,75.84,cont,1.88954731681955,3000041,1,0,1,0
6bba_e16ffc58,1029,13,25.16,115.35999999999999,11.360000000000003,cont,0.3957133590112949,14000203,1,1,0,0
6bba_e16ffc58,1113,14,25.56,115.2,10.559999999999999,cont,0.8836474480809693,15000220,1,0,1,0
6bba_e16ffc58,1513,20,6.999999999999999,179.36,214.55999999999995,cont,2.4103740632939137,21000301,1,1,0,0
6bba_e16ffc58,1583,21,4.866666666666665,192.5333333333333,211.46666666666658,cont,1.8432591154438214,22000317,1,0,0,1
6bba_e16ffc58,1587,21,6.999999999999999,178.56000000000003,216.80000000000007,cont,2.53421862957795,22000318,2,0,1,1
6bba_e16ffc58,1958,26,17.880000000000003,147.84000000000003,31.52,parent,1.6110463408915412,27000395,1,1,0,0
6bba_e16ffc58,2032,27,16.866666666666667,149.06666666666666,54.933333333333316,daughter,0.43754464057968095,28000420,3,0,0,3
6bba_e16ffc58,2033,27,17.479999999999997,153.27999999999997,27.360000000000003,daughter,1.0159824371021429,28000419,2,0,1,1
6bba_e16ffc58,2600,35,9.200000000000003,237.76,135.84,cont,2.140563594009765,36000518,1,1,0,0
6bba_e16ffc58,2601,35,10.000000000000004,247.2,67.67999999999999,cont,1.9029853191761594,36000520,1,1,0,0
6bba_e16ffc58,2613,35,15.0,168.0,57.919999999999995,cont,1.1862499999999976,36000528,1,1,0,0
6bba_e16ffc58,2646,35,39.84,99.36000000000001,90.07999999999998,cont,1.707487646368203,36000526,1,1,0,0
6bba_e16ffc58,2671,36,9.159999999999998,241.44000000000003,138.72,cont,4.782057774901928,37000530,1,0,1,0
6bba_e16ffc58,2674,36,10.000000000000004,249.44,74.88000000000001,cont,4.177229943395503,37000531,1,0,1,0
6bba_e16ffc58,2680,36,12.040000000000003,197.92000000000002,54.56,cont,3.731878925219846,37000541,2,0,1,1
6bba_e16ffc58,2712,36,36.86666666666667,108.53333333333333,107.46666666666665,cont,1.2087786535728267,37000536,1,0,0,1
6bba_e16ffc58,3834,54,13.999999999999998,233.27999999999997,81.28000000000002,cont,1.9018748973052966,55000733,1,1,0,0
6bba_e16ffc58,3872,54,46.32,8.0,7.360000000000001,cont,2.160700118017302,55000739,1,1,0,0
6bba_e16ffc58,3890,55,13.999999999999998,234.72000000000006,83.99999999999999,cont,1.3324999999999818,56000744,1,0,1,0
6bba_e16ffc58,3926,55,45.84,10.079999999999998,8.0,cont,1.623780792625663,56000751,1,0,1,0
6bba_e5e44988,856,15,30.680000000000003,203.35999999999996,190.56000000000003,cont,0.6243046131497098,16000228,1,0,1,0
6bba_e5e44988,1334,25,53.86666666666666,127.99999999999997,60.53333333333332,cont,2.3537582610346695,26000299,1,0,0,1
6bba_e5e44988,1458,28,32.84,24.0,200.0,cont,1.884999999999991,29000320,2,2,0,0
6bba_e5e44988,1504,29,32.32,30.72,200.0,cont,1.4418613187474003,30000327,2,0,2,0
6bba_e5e44988,1960,38,59.160000000000004,63.36,30.72,cont,1.6560655557676476,39000413,1,1,0,0
6bba_e5e44988,2006,39,58.839999999999996,71.36,28.640000000000008,cont,1.2890870073427978,40000422,1,0,1,0
6bba_e5e44988,2300,46,10.320000000000002,156.64000000000001,227.36,cont,3.048163894051629,47000476,1,1,0,0
6bba_e5e44988,2347,47,10.2,167.36000000000004,222.56,cont,3.066129991536559,48000485,1,0,1,0
6bba_e5e44988,3033,61,34.28,33.92,168.63999999999996,cont,2.2262500000000003,62000647,1,1,0,0
6bba_e5e44988,3091,62,33.0,45.44000000000001,168.64000000000001,cont,1.050232861083669,63000659,1,0,1,0
6bba_e5e44988,3099,62,41.680000000000014,18.72,136.00000000000003,cont,1.4869415968692228,63000660,1,1,0,0
6bba_e5e44988,3152,63,39.99999999999999,27.360000000000003,138.72000000000003,cont,0.6758906161502723,64000671,1,0,1,0
6bba_e5e44988,3577,71,31.799999999999994,88.15999999999998,142.08,cont,3.438324901023758,72000771,1,1,0,0
6bba_e5e44988,3807,75,11.999999999999996,200.5333333333333,159.99999999999994,cont,0.7211493767668034,76000813,1,0,0,1
6bba_e5e44988,4396,84,27.52,32.64,175.84000000000003,cont,1.4003314339826982,85000883,1,0,1,0
6bba_e5e44988,4456,85,19.52,195.20000000000005,10.720000000000002,cont,4.386085420109833,86000888,1,1,0,0
6bba_e5e44988,4538,86,17.8,205.44000000000003,9.28,cont,2.300680945394203,87000894,1,0,1,0
6bba_e5e44988,4819,89,14.68,223.20000000000002,8.0,cont,2.468395860675512,90000911,1,1,0,0
6bba_e5e44988,4852,89,33.00000000000001,45.919999999999995,138.08,cont,2.046338956526982,90000916,2,2,0,0
6bba_e5e44988,4918,90,13.480000000000002,229.44,8.0,cont,1.8616713760758128,91000919,1,0,1,0
6bba_e5e44988,4955,90,32.68,52.64000000000001,132.64000000000001,cont,1.3137398905415076,91000924,2,0,2,0
6bba_e5e44988,5265,93,31.000000000000007,69.92000000000002,123.36,cont,1.6318916362614135,94000940,2,2,0,0
6bba_e5e44988,5375,94,30.480000000000004,74.72,120.8,cont,1.50503426040739,95000947,2,0,2,0
6bba_ebdf3b34,1341,2,7.120000000000001,20.0,55.36000000000001,cont,2.1512078729169812,3000092,4,4,0,0
6bba_ebdf3b34,1433,2,15.0,60.160000000000004,137.12000000000003,cont,3.243900526372539,3000102,2,2,0,0
6bba_ebdf3b34,1504,2,20.839999999999996,136.8,166.88,cont,2.7028298249242377,3000107,2,2,0,0
6bba_ebdf3b34,1513,2,22.000000000000004,48.8,128.48,cont,2.431751554949636,3000103,2,2,0,0
6bba_ebdf3b34,1941,3,2.6800000000000006,62.720000000000006,187.35999999999999,cont,6.242073223296892,4000122,3,0,3,0
6bba_ebdf3b34,1957,3,3.999999999999999,91.46666666666664,92.53333333333332,cont,1.7505976162125056,4000126,3,0,0,3
6bba_ebdf3b34,1969,3,6.159999999999998,108.48,205.44000000000003,cont,2.378336143714756,4000149,7,3,2,2
6bba_ebdf3b34,2060,3,14.320000000000002,135.20000000000002,192.0,cont,6.128835661241048,4000150,6,2,4,0
6bba_ebdf3b34,2079,3,15.733333333333333,128.53333333333336,179.99999999999994,cont,0.4729900824071815,4000150,6,3,0,3
6bba_ebdf3b34,2107,3,20.759999999999998,41.12,133.28000000000003,cont,1.867619223101961,4000133,2,0,2,0
6bba_ebdf3b34,2254,3,31.039999999999996,22.08,21.92,cont,1.861600453910551,4000144,2,0,0,2
6bba_ebdf3b34,2622,4,5.32,100.64000000000001,209.44,cont,1.8385494452149014,5000180,3,0,3,0
6bba_ebdf3b34,2722,4,14.480000000000002,126.72,184.0,cont,1.282412277701674,5000162,4,1,3,0
6bba_ebdf3b34,2771,4,18.0,108.5333333333333,201.06666666666663,cont,1.2732047546338583,5000163,4,0,0,4
6bba_ebdf3b34,2778,4,19.0,112.80000000000001,192.32,cont,4.690426752972489,5000163,5,0,1,4
6bba_ebdf3b34,3359,5,13.68,124.00000000000003,187.35999999999999,cont,0.9990776996810504,6000196,1,0,1,0
6bba_ebdf3b34,3905,6,5.360000000000001,90.72,98.72,cont,6.958985423536969,7000222,4,0,0,4
6bba_ebdf3b34,8198,13,12.68,11.36,186.55999999999997,cont,2.2575806325577816,14000408,1,1,0,0
6bba_ebdf3b34,8542,13,45.199999999999996,199.36000000000007,155.2,parent,5.513182214021208,14000412,2,2,0,0
6bba_ebdf3b34,8545,13,46.79999999999999,220.0,139.35999999999999,parent,5.513182214021227,14000412,2,2,0,0
6bba_ebdf3b34,8619,13,53.72000000000001,222.07999999999998,222.55999999999997,cont,3.8388883889089396,14000413,2,2,0,0
6bba_ebdf3b34,8856,14,16.8,56.64,74.08000000000001,cont,1.3483583768790788,15000420,4,1,3,0
6bba_ebdf3b34,9147,14,40.56,210.88,145.92,daughter,5.216452488281668,15000432,10,0,5,5
6bba_ebdf3b34,9164,14,44.16,197.28000000000003,158.72,daughter,2.2223917296462363,15000433,2,0,2,0
6bba_ebdf3b34,9239,14,51.79999999999999,218.56,225.92,cont,2.0506572879201617,15000429,2,0,2,0
6bba_ebdf3b34,9297,14,58.0,227.36000000000004,208.0,cont,1.6813842883172208,15000430,2,0,2,0
6bba_ebdf3b34,9413,15,3.7199999999999998,7.5200000000000005,54.56000000000001,cont,5.929001286051472,16000436,1,1,0,0
6bba_ebdf3b34,10450,16,40.52,209.92000000000004,152.0,cont,2.802029677305371,17000459,1,1,0,0
6bba_ebdf3b34,11297,17,61.00000000000001,164.32,68.64000000000001,cont,0.7699482531313319,18000466,2,0,2,0
6bba_ebdf3b34,12086,19,11.22790023803711,55.33775085449219,87.38055419921874,cont,1.8625122500076592,20000481,7,5,2,0
6bba_ebdf3b34,12401,19,39.33333333333333,187.99999999999994,13.066666666666663,cont,6.452202735414405,20000487,7,2,0,5
6bba_ebdf3b34,13104,20,43.16,197.44,12.0,cont,1.9090810944011816,21000498,3,0,3,0
6bba_ebdf3b34,13656,21,32.68,217.44000000000003,153.28000000000003,cont,6.368030626692992,22000511,2,2,0,0
6bba_ebdf3b34,13662,21,34.480000000000004,206.72,168.0,cont,2.3182890377388228,22000511,1,1,0,0
6bba_ebdf3b34,14318,22,33.84000000000001,205.28,171.36,cont,2.2004204143753854,23000520,1,0,1,0
6bba_ebdf3b34,14539,22,55.99999999999999,152.0,75.36,cont,1.533538595862525,23000518,2,0,2,0
6bba_ebdf3b34,14959,23,30.160000000000004,240.00000000000006,190.72000000000003,cont,5.3348529736066626,24000523,1,0,1,0
6bba_ebdf3b34,14995,23,33.31999999999999,218.08000000000004,186.72000000000006,cont,6.941565587999607,24000529,1,1,0,0
6bba_ebdf3b34,15625,24,29.800000000000004,240.64000000000004,189.28,cont,5.4529586292672265,25000532,5,0,0,5
6bba_ebdf3b34,17884,27,51.48,145.92,83.36000000000001,cont,1.5682723376378145,28000563,2,0,2,0
6bba_ebdf3b34,19363,29,61.99999999999999,141.06666666666666,38.93333333333332,cont,3.7176252967750996,30000583,4,0,0,4
6bba_ebdf3b34,19737,30,31.159999999999997,200.0,191.20000000000002,cont,3.437787260724559,31000586,3,3,0,0
6bba_ebdf3b34,19927,30,48.2,142.72,90.56000000000002,cont,1.4866751914927439,31000588,4,4,0,0
6bba_ebdf3b34,20426,31,30.84,199.36,194.71999999999997,cont,2.1762869755618257,32000595,3,0,3,0
6bba_ebdf3b34,20665,31,51.199999999999996,240.0,21.919999999999998,cont,2.0812691440561046,32000599,7,0,5,2
6bba_ebdf3b34,25155,38,19.84,160.00000000000006,213.28000000000003,cont,0.5640880361255413,39000676,1,1,0,0
6bba_ebdf3b34,25816,39,13.999999999999998,215.36000000000004,226.88,cont,6.269215461283791,40000689,2,0,2,0
6bba_ebdf3b34,25834,39,14.879999999999999,228.00000000000003,225.12,cont,2.0764422698452205,40000689,6,0,1,5
6bba_ebdf3b34,25998,39,28.133333333333326,183.46666666666667,219.4666666666667,cont,6.972682933539037,40000693,6,1,0,5
6bba_ebdf3b34,26127,39,38.0,237.27999999999997,62.56,cont,3.324108198073584,40000698,3,0,3,0
6bba_ebdf3b34,27003,40,55.64,132.00000000000006,64.64,cont,1.631244252863453,41000710,1,0,1,0
6bba_ebdf3b34,27306,41,19.0,164.0,237.27999999999997,cont,5.262441507988457,42000713,1,1,0,0
6bba_ebdf3b34,27994,42,16.720000000000002,161.44,226.72,cont,1.41934238733999,43000725,5,0,0,5
6bba_ebdf3b34,29246,44,6.68,178.88,116.16000000000003,cont,2.43473593280667,45000747,2,2,0,0
6bba_ebdf3b34,29292,44,10.520000000000001,211.20000000000005,240.0,cont,5.397569041939147,45000748,2,2,0,0
6bba_ebdf3b34,29294,44,11.48,226.56,237.28000000000003,cont,1.304967073339407,45000748,3,3,0,0
6bba_ebdf3b34,29345,44,16.16,158.72,233.27999999999997,cont,2.1283157954119583,45000749,2,2,0,0
6bba_ebdf3b34,29555,44,31.68,234.56000000000006,78.72,cont,2.239554224617042,45000753,3,3,0,0
6bba_ebdf3b34,29950,45,9.640000000000002,213.44,242.08,cont,6.7417135479416554,46000758,5,0,2,3
6bba_ebdf3b34,29977,45,11.000000000000002,228.0,241.43999999999997,cont,1.1165299201096284,46000758,3,0,3,0
6bba_ebdf3b34,30041,45,16.32,167.36000000000004,246.72000000000006,cont,6.442955940405021,46000760,3,0,3,0
6bba_ebdf3b34,30221,45,30.119999999999997,234.72000000000006,82.08000000000001,cont,2.0744701883854475,46000763,3,0,3,0
6bba_ebdf3b34,36423,54,20.68,227.99999999999997,116.64000000000001,cont,0.9377249730064896,55000860,1,0,1,0
6bba_ebdf3b34,38485,57,15.16,225.28,145.44,cont,6.857827302615604,58000886,4,4,0,0
6bba_ebdf3b34,38509,57,17.96,222.88,133.92,cont,3.4939011595206844,58000886,4,4,0,0
6bba_ebdf3b34,39490,58,39.13333333333333,100.53333333333332,98.93333333333334,cont,6.856363946181683,59000902,7,0,0,7
6bba_ebdf3b34,39509,58,38.879999999999995,204.00000000000003,44.00000000000001,cont,3.6542272182364384,59000901,9,0,3,6
6bba_ebdf3b34,45204,67,4.319999999999999,113.28000000000003,113.44,cont,4.859918337791275,68000985,1,0,1,0
6bba_ebdf3b34,49594,73,36.8,132.64000000000001,10.079999999999998,cont,1.6156292311356535,74001058,2,0,2,0
6bba_ebdf3b34,50649,75,14.879999999999999,169.28,85.92,cont,3.807392685618337,76001072,4,0,1,3
6bba_ebdf3b34,51411,76,25.159999999999997,97.28000000000003,139.36,cont,6.847905017229711,77001083,1,0,1,0
6bba_ebdf3b34,51852,77,10.000000000000004,28.640000000000008,30.560000000000002,cont,0.8407706658179729,78001088,2,0,2,0
6bba_ebdf3b34,51906,77,14.479999999999999,92.0,169.28,cont,1.1320314980158437,78001089,1,1,0,0
6bba_ebdf3b34,52545,78,15.000000000000004,92.0,184.0,cont,6.21443003219121,79001097,1,0,1,0
6bba_ebdf3b34,56679,85,11.959999999999997,201.44000000000003,149.28000000000003,cont,2.592931798659574,86001157,3,3,0,0
6bba_ebdf3b34,56726,85,15.160000000000004,97.75999999999999,187.36,cont,6.112095717714188,86001158,3,3,0,0
6bba_ebdf3b34,56820,85,24.119999999999994,102.08000000000003,159.2,cont,2.3491228416155545,86001159,3,3,0,0
6bba_ebdf3b34,56889,85,28.96000000000001,114.72,17.919999999999998,daughter,2.8271731132882647,86001165,3,3,0,0
6bba_ebdf3b34,57229,86,6.32,200.16,155.2,cont,2.0961870145576227,87001165,9,0,4,5
6bba_ebdf3b34,57269,86,8.4,91.84,188.00000000000003,cont,4.598089616895274,87001166,22,5,8,9
6bba_ebdf3b34,57514,86,32.76,118.56000000000003,107.36,cont,2.971040432239182,87001171,4,1,3,0
6bba_ebdf3b34,57975,87,21.999999999999993,99.46666666666667,155.99999999999997,cont,1.368713533253452,88001176,6,0,0,6
6bba_ebdf3b34,58132,87,35.96000000000001,106.72,104.79999999999998,cont,6.398456845208862,88001181,4,0,1,3
6bba_ebdf3b34,59021,89,13.133333333333333,91.46666666666664,176.5333333333333,cont,1.5551116988285203,90001193,5,0,0,5
6bba_ebdf3b34,59128,89,22.320000000000007,96.63999999999999,162.07999999999998,cont,0.8457808891787645,90001194,1,0,1,0
6bba_ebff6e76,1180,1,18.32,148.0,204.96,cont,2.87950733633377,2000051,1,1,0,0
6bba_ebff6e76,2200,2,26.266666666666666,216.53333333333325,135.99999999999994,cont,4.996856650824525,3000088,5,1,0,4
6bba_ebff6e76,3194,3,31.0,78.72,208.00000000000006,cont,1.8897569288138496,4000127,1,1,0,0
6bba_ebff6e76,5467,6,3.999999999999999,100.53333333333332,83.99999999999997,cont,4.3530929696915255,7000212,6,0,0,6
6bba_ebff6e76,5681,6,19.159999999999997,68.0,188.00000000000003,cont,3.2031878652367567,7000221,1,1,0,0
6bba_ebff6e76,5986,6,37.00000000000001,154.08,80.00000000000003,cont,0.407547926629481,7000228,7,0,2,5
6bba_ebff6e76,6082,6,43.480000000000004,93.44,229.28000000000003,cont,4.699847238208927,7000230,1,0,1,0
6bba_ebff6e76,6154,6,48.64,121.28000000000003,223.35999999999999,cont,1.9769646083073957,7000232,1,0,1,0
6bba_ebff6e76,6632,7,22.6,223.35999999999999,151.35999999999999,cont,4.382592162465042,8000251,5,0,1,4
6bba_ebff6e76,8158,9,7.133333333333333,119.99999999999997,83.99999999999997,cont,4.623958685326995,10000304,4,0,0,4
6bba_ebff6e76,8387,9,24.64,221.28000000000003,164.64,cont,3.7936355007433047,10000342,6,0,0,6
6bba_ebff6e76,9534,10,40.199999999999996,80.47999999999999,224.0,cont,3.475525140176666,11000353,1,0,1,0
6bba_ebff6e76,10471,11,47.00000000000001,11.36,164.64,cont,5.137724751774868,12000385,1,0,1,0
6bba_ebff6e76,10904,12,21.64,10.720000000000002,140.64000000000001,cont,5.657847702527881,13000406,1,0,1,0
6bba_ebff6e76,11163,12,36.839999999999996,85.28,226.72,cont,5.7413115063110824,13000414,1,0,1,0
6bba_ebff6e76,12179,13,46.12000000000002,18.4,196.64000000000001,cont,3.927830388713313,14000456,5,0,0,5
6bba_ebff6e76,12862,14,33.32000000000001,150.72,90.72,cont,6.027786311117861,15000473,3,2,1,0
6bba_ebff6e76,13122,14,50.320000000000014,41.28000000000001,76.0,cont,6.425904240844234,15000483,1,0,1,0
6bba_ebff6e76,13736,15,31.880000000000003,103.84000000000002,37.92,cont,5.090012585937298,16000499,9,1,1,7
6bba_ebff6e76,14537,16,28.879999999999995,93.92,220.0,cont,6.799280407697575,17000528,1,1,0,0
6bba_ebff6e76,14705,16,40.679999999999986,13.280000000000001,172.0,cont,2.2026393968373728,17000532,1,1,0,0
6bba_ebff6e76,15226,17,21.0,163.36,111.36,cont,2.2655204286212043,18000548,4,0,4,0
6bba_ebff6e76,15240,17,21.52,205.44000000000003,160.00000000000003,cont,6.612771733547133,18000569,1,1,0,0
6bba_ebff6e76,15241,17,21.52,216.64000000000001,183.83999999999997,cont,5.303104780456433,18000570,6,0,1,5
6bba_ebff6e76,15423,17,32.35999999999999,90.88,227.84,cont,3.2058659614057596,18000554,8,1,0,7
6bba_ebff6e76,15615,17,44.52,132.64000000000001,240.64000000000004,cont,6.835457761737677,18000560,1,1,0,0
6bba_ebff6e76,16450,18,44.839999999999996,9.280000000000001,173.28000000000003,cont,4.688033469643743,19000590,1,1,0,0
6bba_ebff6e76,17272,19,45.31999999999999,19.36,191.36,cont,5.186296396514561,20000619,1,0,1,0
6bba_ebff6e76,17977,20,37.266666666666666,83.99999999999997,231.46666666666658,cont,6.603558725961508,21000641,2,0,2,0
6bba_ebff6e76,19538,22,34.68000000000001,102.72000000000001,248.0,cont,5.654859912057952,23000690,1,1,0,0
6bba_ebff6e76,22290,26,8.0,38.71999999999999,69.28,cont,4.403959865848007,27000786,3,0,0,3
6bba_ebff6e76,22345,26,12.840000000000002,11.360000000000003,197.28,cont,1.4836525873667328,27000792,1,1,0,0
6bba_ebff6e76,23542,27,38.88000000000001,34.72,113.44,cont,5.280349923300525,28000834,8,0,1,7
6bba_ebff6e76,24852,29,24.64,105.92,68.0,cont,5.861273858983218,30000884,2,1,1,0
6bba_ebff6e76,24917,29,28.48,97.28000000000003,62.08000000000001,cont,5.870997040537497,30000884,6,1,0,5
6bba_ebff6e76,25316,30,2.5199999999999996,28.640000000000004,65.92,cont,3.4772342188152923,31000897,1,1,0,0
6bba_ebff6e76,25406,30,10.866666666666664,42.93333333333332,244.53333333333342,cont,3.1408493908230155,31000900,3,1,0,2
6bba_ebff6e76,25624,30,26.800000000000004,95.36000000000001,61.27999999999999,cont,1.235427557568623,31000911,7,1,1,5
6bba_ebff6e76,25874,30,42.559999999999995,32.8,128.64000000000001,cont,5.1211224660712835,31000917,8,0,0,8
6bba_ebff6e76,26225,31,16.000000000000004,156.53333333333333,251.46666666666664,cont,6.166025093314888,32000930,3,0,0,3
6bba_ebff6e76,27056,32,22.8,104.0,72.00000000000001,cont,6.791061450539233,33000962,5,0,0,5
6bba_ebff6e76,27752,33,16.24,158.72,249.92000000000004,cont,6.742359796465928,34000979,2,2,0,0
6bba_ebff6e76,28418,34,11.000000000000002,143.99999999999997,241.28000000000003,cont,3.607133990441725,35001003,1,0,1,0
6bba_ebff6e76,29316,35,24.520000000000007,90.08000000000001,64.0,cont,2.530934239564525,36001034,1,1,0,0
6bba_ebff6e76,29400,35,30.799999999999997,140.64000000000001,123.99999999999999,cont,5.369414103279802,36001037,2,1,1,0
6bba_ebff6e76,29438,35,35.080000000000005,32.48000000000001,106.08000000000001,cont,6.651333630370667,36001038,3,1,0,2
6bba_ebff6e76,29819,36,6.640000000000001,28.480000000000004,245.11999999999995,cont,3.894884666135838,37001046,2,0,0,2
6bba_ebff6e76,30937,37,34.39999999999999,28.53333333333333,119.99999999999994,cont,1.1426470851103343,38001077,6,0,0,6
6bba_ebff6e76,32032,39,7.039999999999999,22.560000000000006,218.56,cont,2.8208146540494354,40001105,6,1,0,5
6bba_ebff6e76,33842,41,34.86666666666666,75.99999999999999,135.99999999999994,cont,5.469827671594815,42001153,5,0,0,5
6bba_ebff6e76,39010,49,19.64,143.20000000000002,32.64,cont,6.073175266077872,50001295,4,0,1,3
6bba_ebff6e76,39508,50,7.0,148.64,155.35999999999999,cont,3.306865018412463,51001312,1,1,0,0
6bba_ebff6e76,40320,51,22.999999999999996,127.99999999999997,55.99999999999999,cont,6.550584420492576,52001337,7,0,0,7
6bba_ebff6e76,43993,57,5.9999999999999964,149.06666666666666,160.53333333333327,cont,3.4904942840838915,58001435,1,0,1,0
6bba_ebff6e76,44019,57,8.639999999999999,154.08,170.72000000000003,cont,3.5331700532241754,58001435,1,1,0,0
6bba_ebff6e76,44653,58,8.320000000000002,156.48,156.79999999999998,cont,3.503863269663928,59001451,2,0,0,2
6bba_ebff6e76,47013,62,3.839999999999999,4.640000000000001,231.36,cont,3.2971085817728274,63001504,1,0,1,0
6bba_ebff6e76,50095,67,13.839999999999998,4.0,199.36,cont,2.069053225511611,68001587,1,1,0,0
6bba_ebff6e76,51401,69,25.200000000000006,35.36,214.72,cont,3.01414503839811,70001622,1,1,0,0
6bba_ebff6e76,51767,70,9.0,135.46666666666664,164.53333333333333,cont,6.768287021203297,71001623,4,0,0,4
6bba_ebff6e76,52389,71,13.319999999999999,4.0,196.00000000000003,cont,1.0467068894872016,72001640,1,1,0,0
6bba_ebff6e76,52649,71,33.86666666666666,14.933333333333334,144.0,cont,0.8147538184766154,72001648,5,0,0,5
6bba_ebff6e76,53109,72,23.999999999999993,39.999999999999986,207.99999999999997,cont,5.745242597140708,73001660,11,1,0,10
6bba_ebff6e76,53127,72,25.0,31.360000000000007,212.0,cont,1.898956950538901,73001660,1,0,1,0
6bba_ebff6e76,53467,73,5.999999999999998,195.99999999999997,212.5333333333333,cont,6.703549018575507,74001669,8,1,0,7
6bba_ebff6e76,54851,75,24.680000000000003,29.920000000000005,208.64000000000001,cont,3.810408442673825,76001713,1,0,1,0
6bba_ebff6e76,54862,75,25.520000000000003,25.28,225.28000000000003,cont,6.927762084540729,76001713,11,0,1,10
6bba_ebff6e76,55185,76,7.120000000000001,117.28,137.28,cont,5.629563843007733,77001715,1,1,0,0
6bba_ebff6e76,56051,77,28.199999999999996,36.64000000000001,207.36000000000007,cont,1.6541510284735115,78001746,8,0,1,7
6bba_ebff6e76,57728,80,28.8,49.91999999999999,152.64,cont,4.264967211186968,81001783,2,2,0,0
6bba_ebff6e76,58020,81,6.999999999999999,210.71999999999997,194.71999999999997,cont,6.705424226885285,82001792,3,1,2,0
6bba_ebff6e76,58275,81,30.000000000000007,14.719999999999999,214.72,cont,6.457161963664847,82001810,2,1,1,0
6bba_ebff6e76,58673,82,15.84,148.00000000000003,168.0,cont,4.9095711689006025,83001813,1,0,1,0
6bba_ebff6e76,60125,85,7.640000000000001,206.08,213.92000000000002,cont,3.135407918038737,86001848,2,2,0,0
6bba_ebff6e76,61729,88,14.0,2.72,232.0,cont,1.8057983864484983,89001888,1,0,1,0
6bba_ebff6e76,62166,89,9.160000000000002,203.35999999999999,216.63999999999996,cont,6.029888706477085,90001897,6,1,1,4
6bba_ebff6e76,62660,90,9.0,202.72000000000003,216.0,cont,4.285473464216068,91001908,2,1,1,0
6bba_ebff6e76,62831,90,25.319999999999997,35.36,152.0,cont,4.252411080786995,91001915,1,1,0,0
6bba_ebff6e76,63370,91,28.16,17.279999999999998,188.00000000000003,cont,2.0733879280057503,92001929,1,0,1,0
6bba_ebff6e76,64851,94,27.0,34.080000000000005,148.64000000000001,cont,6.646329461627677,95001951,1,0,1,0
6bba_ed9377fd,1774,34,19.0,155.35999999999999,100.0,cont,2.5360414527369244,35000270,1,1,0,0
6bba_ed9377fd,2286,45,30.86666666666667,120.53333333333336,235.46666666666664,cont,0.694200304307065,46000340,1,0,0,1
6bba_ed9377fd,2288,45,34.839999999999996,119.20000000000002,230.72000000000008,cont,6.749229701417768,46000340,2,0,2,0
6bba_ed9377fd,2787,56,27.16,134.56000000000003,240.15999999999997,cont,4.6445729418860475,57000424,2,2,0,0
6bba_ed9377fd,2831,57,27.84,138.72,233.92000000000002,cont,3.8190611575359714,58000430,2,0,2,0
6bba_ed9377fd,3193,65,3.0,208.64000000000004,24.639999999999993,cont,1.2625946746680115,66000476,1,1,0,0
6bba_ed9377fd,3247,66,2.6800000000000006,209.28000000000003,25.28,cont,1.4093537969580303,67000482,1,0,1,0
6bba_ed9377fd,3791,77,8.0,225.28000000000003,4.640000000000001,cont,1.7124292868612154,78000547,1,1,0,0
6bba_ed9377fd,3838,78,7.32,228.0,5.279999999999999,cont,1.2131035868795381,79000553,1,0,1,0
6bba_ed9377fd,4126,84,18.84,179.36,242.72000000000003,cont,3.211626583602157,85000602,1,1,0,0
6bba_ed9377fd,4169,85,18.68,179.36,244.00000000000003,cont,3.3825771425054088,86000610,1,0,1,0
6bba_edf14583,2775,4,40.999999999999986,15.466666666666672,51.46666666666666,cont,1.051726424609663,5000070,5,0,0,5
6bba_edf14583,4229,7,0.9999999999999998,24.53333333333333,51.99999999999999,cont,6.177375274157929,8000117,5,0,0,5
6bba_edf14583,6675,11,0.4800000000000001,23.360000000000007,60.000000000000014,cont,2.658253656068207,12000187,1,1,0,0
6bba_edf14583,7736,12,41.32000000000001,26.720000000000002,52.0,cont,1.1372678334499793,13000213,1,0,1,0
6bba_edf14583,9302,15,3.7252756500244146,22.536034545898435,80.25996459960939,cont,6.169690803393534,16000256,1,0,1,0
6bba_edf14583,10632,17,2.3600000000000003,47.19999999999999,73.44,cont,2.5340102234403066,18000287,9,1,0,8
6bba_edf14583,10661,17,4.84,43.199999999999996,89.92,cont,5.4693985329741714,18000287,5,2,3,0
6bba_edf14583,11104,17,45.26666666666666,103.99999999999999,127.46666666666667,cont,4.979210250855272,18000301,7,0,0,7
6bba_edf14583,11310,18,2.6799999999999997,48.0,75.35999999999999,cont,2.6368097438002644,19000303,1,0,1,0
6bba_edf14583,11983,19,3.6799999999999997,3.36,80.00000000000001,cont,1.5925829060052117,20000322,1,1,0,0
6bba_edf14583,11992,19,4.52,13.920000000000002,91.36,cont,6.4549328278844245,20000322,1,1,0,0
6bba_edf14583,12444,19,45.64000000000001,104.64000000000001,125.92,cont,5.6538791992754875,20000341,1,1,0,0
6bba_edf14583,13334,21,5.68,67.2,92.0,cont,6.677820799108045,22000361,3,2,1,0
6bba_edf14583,14014,22,4.040000000000001,54.720000000000006,90.56000000000002,cont,5.106767721612175,23000379,8,0,1,7
6bba_edf14583,14016,22,5.320000000000001,70.71999999999998,95.36000000000001,cont,5.739011375010513,23000379,3,1,2,0
6bba_edf14583,14017,22,4.680000000000001,76.0,83.99999999999999,cont,4.919457700041742,23000379,3,1,2,0
6bba_edf14583,16044,25,4.0,37.28,104.64,cont,6.221437233469448,26000430,1,1,0,0
6bba_edf14583,16743,26,5.520000000000001,28.000000000000004,101.92000000000002,cont,6.2358091395583966,27000454,2,1,1,0
6bba_edf14583,17167,26,42.67999999999999,16.640000000000004,49.28,cont,0.614284390571672,27000464,1,0,1,0
6bba_edf14583,18102,28,2.84,86.24,94.07999999999998,cont,3.804756275045748,29000486,2,1,1,0
6bba_edf14583,18598,28,44.68,147.35999999999999,139.36,cont,0.8577168093257852,29000500,1,1,0,0
6bba_edf14583,18771,29,2.160000000000001,87.36000000000001,112.64,cont,6.696459185644306,30000510,1,0,1,0
6bba_edf14583,18803,29,2.6800000000000006,83.2,95.36000000000001,cont,2.903707220950485,30000510,9,0,1,8
6bba_edf14583,19291,29,44.52,150.72,136.64000000000001,cont,1.5991188980185336,30000521,1,0,1,0
6bba_edf14583,23876,36,15.880000000000003,80.0,63.36,cont,2.0682873349706483,37000650,4,4,0,0
6bba_edf14583,23946,36,22.200000000000003,43.84000000000001,41.440000000000005,cont,2.0433688482503616,37000651,4,4,0,0
6bba_edf14583,24600,37,14.4,96.63999999999999,59.36,cont,2.8245099132415854,38000669,5,0,2,3
6bba_edf14583,24733,37,25.680000000000007,199.83999999999997,243.20000000000005,cont,2.76082669910662,38000673,5,0,2,3
6bba_edf14583,24951,37,43.28,126.72000000000004,110.56,cont,1.0473373978809146,38000682,2,0,2,0
6bba_edf14583,27143,41,2.84,5.119999999999999,192.0,cont,1.7093424357044438,42000745,2,2,0,0
6bba_edf14583,27745,42,2.2800000000000002,6.719999999999999,192.0,cont,2.2477339283153603,43000768,2,0,2,0
6bba_edf14583,28491,43,2.84,30.080000000000002,237.27999999999997,cont,1.831930760836776,44000794,1,0,1,0
6bba_edf14583,29177,44,3.9999999999999996,20.533333333333328,191.99999999999994,cont,3.850122314976212,45000814,5,1,0,4
6bba_edf14583,29792,45,3.08,23.040000000000006,192.0,cont,2.051622833271265,46000836,1,0,1,0
6bba_edf14583,31751,48,2.0,32.0,196.64000000000001,cont,2.297005183385534,49000898,1,1,0,0
6bba_edf14583,31765,48,2.5199999999999996,10.559999999999999,133.27999999999997,cont,1.232110105672371,49000895,1,1,0,0
6bba_edf14583,32434,49,3.999999999999999,4.0000000000000036,135.99999999999994,cont,5.984436194204089,50000918,4,0,0,4
6bba_edf14583,33025,50,1.4800000000000002,17.919999999999998,131.35999999999999,cont,0.858178485223207,51000940,1,1,0,0
6bba_edf14583,33099,50,4.000000000000001,8.000000000000002,136.00000000000003,cont,5.585014128227436,51000940,2,2,0,0
6bba_edf14583,33102,50,3.36,42.720000000000006,218.72000000000003,cont,6.036475844812121,51000942,1,1,0,0
6bba_edf14583,33646,51,1.1600000000000001,22.72,130.72,cont,1.7296138477995606,52000960,1,0,1,0
6bba_edf14583,33704,51,3.3200000000000003,48.64,220.64000000000001,cont,6.412802624438091,52000964,1,0,1,0
6bba_edf14583,33955,51,22.000000000000004,238.07999999999998,239.36000000000007,cont,1.8211603443958377,52000973,1,1,0,0
6bba_edf14583,35181,53,21.360000000000003,242.71999999999997,237.27999999999997,cont,1.0523678658149829,54001010,2,2,0,0
6bba_edf14583,35683,54,14.16,140.00000000000003,60.0,cont,1.24617477205246,55001027,1,1,0,0
6bba_edf14583,35760,54,20.84,244.0,236.64000000000001,cont,0.3676955262170069,55001031,3,1,2,0
6bba_edf14583,36035,54,51.16,207.36,4.640000000000001,cont,1.9697386406576807,55001041,1,1,0,0
6bba_edf14583,36289,55,14.16,143.99999999999997,60.800000000000004,cont,1.3257450735341216,56001051,1,0,1,0
6bba_edf14583,36729,56,2.4799999999999995,27.36,139.36,cont,6.609596361541305,57001069,1,0,1,0
6bba_edf14583,36812,56,6.840000000000001,29.440000000000005,242.07999999999996,cont,3.514698822445526,57001071,2,0,2,0
6bba_edf14583,37314,57,1.0,157.44,136.8,cont,1.554064228080683,58001098,1,1,0,0
6bba_edf14583,39595,61,2.6799999999999997,162.07999999999998,126.72000000000001,cont,6.8240713653947065,62001191,2,1,1,0
6bba_edf14583,41190,64,4.8,56.64,244.64000000000001,cont,3.1931146628331404,65001257,1,0,1,0
6bba_edf14583,43788,69,7.133333333333333,2.933333333333334,116.0,cont,3.853074169485393,70001362,4,0,0,4
6bba_edf14583,46251,74,7.839999999999999,10.079999999999998,119.36000000000001,cont,6.265549906033794,75001482,3,2,1,0
6bba_edf14583,46523,74,39.68,208.64,116.63999999999999,cont,2.0801269492509453,75001493,2,0,2,0
6bba_edf14583,48662,79,13.800000000000002,148.00000000000003,41.28,cont,1.7463269489130537,80001584,1,1,0,0
6bba_edf14583,48916,79,44.00000000000001,230.72000000000008,27.52,cont,0.7949223625612546,80001595,3,3,0,0
6bba_edf14583,49384,80,39.32,217.44000000000005,123.36000000000003,cont,3.435674287894567,81001617,2,0,2,0
6bba_edf14583,49411,80,41.519999999999996,245.28,49.120000000000005,cont,3.221928770162371,81001618,2,0,0,2
6bba_edf14583,49870,81,40.000000000000014,235.35999999999999,132.00000000000003,cont,5.042241719959487,82001635,1,1,0,0
6bba_edf14583,50366,82,44.32,231.36,24.000000000000007,cont,0.6758906161502772,83001655,1,0,1,0
6bba_edf14583,50601,83,12.999999999999998,191.99999999999994,33.066666666666656,cont,0.896617873517514,84001664,5,0,0,5
6bba_edf14583,50996,84,3.5200000000000005,248.00000000000006,148.00000000000003,cont,0.9375841628888522,85001681,1,1,0,0
6bba_edf14583,51117,84,21.84,4.000000000000001,90.72,cont,0.564088036125565,85001688,1,0,1,0
6bba_edf14583,51143,84,24.866666666666667,0.0,91.99999999999999,cont,5.146474740727328,85001688,2,0,0,2
6bba_edf14583,51516,85,12.0,182.72,25.92,cont,6.798892028301372,86001702,1,1,0,0
6bba_edf14583,51908,86,5.159999999999999,23.360000000000007,100.0,cont,6.170978953942395,87001719,1,1,0,0
6bba_edf14583,52408,87,12.0,186.72000000000006,26.080000000000002,cont,5.616320665035057,88001747,1,1,0,0
6bba_edf14583,52762,88,1.8666666666666674,3.999999999999999,27.999999999999996,cont,6.864917164366303,89001758,4,0,0,4
6bba_edf14583,53137,88,42.84,243.36000000000004,39.36,cont,3.0343584577303817,89001775,2,1,1,0
6bba_edf14583,53381,89,18.16,9.919999999999998,92.0,cont,1.8302002110424958,90001789,2,2,0,0
6bba_edf14583,53686,90,2.0,12.0,210.71999999999997,cont,3.2772668681387547,91001798,1,0,1,0
6bba_edf14583,54300,91,17.0,174.72,8.64,cont,0.1852785065786073,92001840,1,0,1,0
6bba_edf14583,54363,91,26.640000000000008,12.64,92.64,cont,4.9002035990864625,92001843,1,1,0,0
6bba_edf14583,54624,92,2.0,16.000000000000004,212.0,cont,2.0312499999999982,93001853,1,0,1,0
6bba_edf14583,54822,92,25.866666666666656,7.999999999999998,80.53333333333333,cont,2.28737951261313,93001868,3,0,0,3
6bba_edf14583,55281,93,26.240000000000006,8.000000000000002,82.72000000000001,cont,1.5670932486613474,94001896,1,1,0,0
6bba_edf14583,55768,94,26.68,8.640000000000002,83.35999999999999,cont,1.1212500000000016,95001918,1,0,1,0
6bba_edf14583,56109,95,9.84,14.720000000000002,29.439999999999998,cont,5.724545424092641,96001943,1,0,0,1
6bba_eebc57a5,265,3,4.6800000000000015,109.44000000000001,153.92000000000002,cont,1.9243732746013618,4000041,1,1,0,0
6bba_eebc57a5,318,3,32.64,87.36000000000001,14.079999999999998,cont,6.540115275169083,4000051,1,1,0,0
6bba_eebc57a5,332,3,53.12,1.9200000000000004,10.239999999999998,cont,3.127185735609579,4000054,1,1,0,0
6bba_eebc57a5,337,4,1.4000000000000001,123.99999999999999,148.53333333333336,cont,1.0731290242919438,5000055,2,0,0,2
6bba_eebc57a5,354,4,5.68,128.64000000000001,214.72,cont,0.8131497402077899,5000056,1,1,0,0
6bba_eebc57a5,362,4,13.200000000000001,145.92,61.44,cont,6.821129855089991,5000058,1,0,1,0
6bba_eebc57a5,402,4,33.120000000000005,90.72,9.759999999999998,cont,2.9443009038649626,5000065,3,0,1,2
6bba_eebc57a5,411,4,48.80000000000001,7.2,17.919999999999998,cont,1.021813644947058,5000068,2,0,1,1
6bba_eebc57a5,443,5,10.133333333333331,126.93333333333332,215.99999999999994,cont,1.3281883154842438,6000070,1,0,0,1
6bba_eebc57a5,1154,13,13.16,168.00000000000003,70.08,cont,3.987380484102324,14000185,1,0,1,0
6bba_eebc57a5,1276,14,32.16000000000001,128.0,13.920000000000005,cont,1.472397067709641,15000209,1,1,0,0
6bba_eebc57a5,1367,15,32.0,133.44,16.0,cont,1.7908945041514865,16000227,1,0,1,0
6bba_eebc57a5,1580,17,56.32000000000001,10.720000000000002,22.72,cont,3.762041707969218,18000278,1,1,0,0
6bba_eebc57a5,1679,18,49.480000000000004,30.72,31.360000000000003,cont,1.0159824371021382,19000296,1,0,1,0
6bba_eebc57a5,1683,18,57.0,9.28,21.439999999999998,cont,4.409113431292056,19000298,2,1,1,0
6bba_eebc57a5,1783,19,56.86666666666666,3.999999999999999,31.999999999999993,cont,2.35313491402664,20000316,2,0,0,2
6bba_eebc57a5,1784,19,57.52000000000001,8.64,20.000000000000004,cont,3.8893213177236934,20000316,1,0,1,0
6bba_eebc57a5,2174,23,47.20000000000002,20.640000000000008,53.279999999999994,cont,3.318542837225372,24000393,1,1,0,0
6bba_eebc57a5,2269,24,46.24,24.64,53.280000000000015,cont,2.0235002162095275,25000410,3,2,1,0
6bba_eebc57a5,2276,24,55.239999999999995,10.559999999999999,33.44,cont,3.947713341290113,25000415,1,1,0,0
6bba_eebc57a5,2375,25,46.67999999999999,29.440000000000005,54.080000000000005,cont,1.131564873305991,26000429,2,0,2,0
6bba_eebc57a5,2382,25,55.839999999999996,13.440000000000001,32.64,cont,5.301411516049291,26000431,1,0,1,0
6bba_eebc57a5,2385,25,60.133333333333326,0.0,36.0,cont,4.769205053284844,26000431,2,0,0,2
6bba_eebc57a5,2583,27,58.84000000000001,114.07999999999998,18.56,cont,2.818379687426807,28000474,1,1,0,0
6bba_eebc57a5,2601,28,7.8,162.88000000000002,235.99999999999994,cont,3.947412324865481,29000476,1,0,1,0
6bba_eebc57a5,2682,28,58.13333333333334,136.5333333333333,12.000000000000004,cont,6.540947412688421,29000493,1,0,0,1
6bba_eebc57a5,2683,28,58.680000000000014,113.92,21.440000000000005,cont,3.605303385292281,29000493,1,0,1,0
6bba_eebc57a5,3109,33,3.0,189.28000000000003,168.0,cont,1.715818408952407,34000575,1,1,0,0
6bba_eebc57a5,3157,33,25.200000000000003,199.84,35.99999999999999,cont,3.852860949995469,34000585,2,0,1,1
6bba_eebc57a5,3213,34,3.0,191.36,168.0,cont,2.4310456212296714,35000598,1,0,1,0
6bba_eebc57a5,3461,36,26.000000000000004,204.00000000000003,36.63999999999999,cont,1.9470185540975136,37000649,1,0,1,0
6bba_eebc57a5,3605,37,56.99999999999999,143.36,24.640000000000008,cont,5.0795742931864005,38000681,1,1,0,0
6bba_eebc57a5,3710,38,56.99999999999999,144.64000000000001,27.36,cont,1.9643017207140194,39000699,1,0,1,0
6bba_eebc57a5,3821,39,56.99999999999999,146.56000000000003,28.640000000000008,cont,4.686568740827361,40000717,1,1,0,0
6bba_eebc57a5,3926,40,57.16000000000001,150.07999999999998,30.72,cont,4.181305306061713,41000733,1,0,1,0
6bba_eebc57a5,4667,47,37.2,84.64000000000001,143.36,cont,3.050372159671663,48000841,1,1,0,0
6bba_eebc57a5,4781,48,36.86666666666667,96.5333333333333,160.53333333333333,cont,1.0426203967248537,49000862,1,0,0,1
6bba_eebc57a5,4944,49,60.99999999999999,49.06666666666665,87.46666666666665,cont,3.991921638516414,50000883,1,0,0,1
6bba_eebc57a5,4947,49,61.99999999999999,26.933333333333323,98.13333333333333,cont,6.466340802034453,50000883,1,0,1,0
6bba_eebc57a5,5386,53,51.16,34.07999999999999,101.92000000000002,cont,1.214843739951769,54000971,2,0,0,2
6bba_eebc57a5,5643,55,58.839999999999996,37.92,75.36,cont,2.113499763425589,56001022,1,1,0,0
6bba_eebc57a5,5766,56,58.48000000000002,41.92,75.36,cont,0.7942577116402746,57001038,1,0,1,0
6bba_eebc57a5,6090,59,38.480000000000004,231.84000000000003,51.35999999999999,cont,1.2561996258557033,60001104,1,1,0,0
6bba_eebc57a5,6212,60,38.68000000000001,237.44,52.64,cont,2.4116883349637073,61001125,1,0,1,0
6bba_eebc57a5,6220,60,42.519999999999996,2.72,237.44000000000005,cont,2.657707247328027,61001126,1,0,1,0
6bba_eebc57a5,6248,60,52.88,241.92000000000007,12.0,cont,0.9437599337225391,61001133,1,0,0,1
6bba_eebc57a5,6365,61,57.800000000000004,69.28,104.00000000000001,cont,4.54488998766746,62001151,1,1,0,0
6bba_eebc57a5,6487,62,57.68000000000002,75.52000000000001,105.28000000000002,cont,3.9351179302531816,63001165,1,0,1,0
6bba_eebc57a5,6680,64,41.32,8.64,244.0,cont,1.3005077133565985,65001191,1,1,0,0
6bba_eebc57a5,6703,64,51.2,208.0,37.92,cont,1.8061639252847443,65001193,1,0,1,0
6bba_eebc57a5,6705,64,53.000000000000014,204.64,77.28,cont,0.7587201394453752,65001195,1,1,0,0
6bba_eebc57a5,6812,65,52.32000000000001,212.0,79.36000000000001,cont,1.1723673965528034,66001216,1,0,1,0
6bba_eebc57a5,6893,66,40.160000000000004,13.280000000000001,246.72000000000006,cont,1.5504918574439428,67001229,1,1,0,0
6bba_eebc57a5,6986,67,33.31999999999999,149.44,189.28,cont,2.7550340288279678,68001246,1,1,0,0
6bba_eebc57a5,7001,67,39.839999999999996,16.0,248.00000000000006,cont,1.1780969824254524,68001248,2,1,1,0
6bba_eebc57a5,7033,67,55.00000000000001,89.27999999999999,118.07999999999998,cont,5.538199306859586,68001253,1,1,0,0
6bba_eebc57a5,7093,68,33.160000000000004,152.0,192.64,cont,1.3728124644320465,69001270,1,0,1,0
6bba_eebc57a5,7105,68,39.32,18.080000000000002,249.27999999999997,cont,1.383828475642846,69001271,1,0,1,0
6bba_eebc57a5,7133,68,54.800000000000004,72.80000000000001,127.99999999999997,cont,4.886496859970338,69001275,3,0,0,3
6bba_eebc57a5,7134,68,54.480000000000004,92.63999999999999,119.99999999999997,cont,4.129706595207461,69001275,1,0,1,0
6bba_eebc57a5,7479,72,26.360000000000003,241.28000000000003,149.92000000000002,cont,1.596226507266439,73001352,1,0,1,0
6bba_eebc57a5,7606,73,38.64,30.080000000000005,252.0,cont,2.289462955585874,74001377,1,1,0,0
6bba_eebc57a5,7695,74,38.839999999999996,32.0,252.0,cont,1.9706098358122592,75001396,1,0,1,0
6bba_eebc57a5,8193,79,50.68000000000001,243.36000000000004,104.64000000000003,cont,0.5994906692351145,80001502,1,1,0,0
6bba_eebc57a5,8257,80,42.64,122.71999999999998,154.08,cont,1.1575202482030256,81001515,1,1,0,0
6bba_eebc57a5,8276,80,50.160000000000004,244.64000000000001,107.35999999999999,cont,1.1961148042725662,81001522,1,0,1,0
6bba_eebc57a5,8346,81,42.64,124.00000000000003,154.08000000000004,cont,1.4025924657575872,82001533,1,0,1,0
6bba_eebc57a5,8735,85,48.16000000000001,131.20000000000005,154.07999999999998,cont,2.758195175291236,86001607,1,0,0,1
6bba_eebc57a5,8859,86,58.999999999999986,216.0,44.00000000000001,cont,5.65840773760603,87001616,1,1,0,0
6bba_eebc57a5,8865,86,60.67999999999999,238.08000000000004,33.279999999999994,cont,1.471679525236382,87001617,1,1,0,0
6bba_eebc57a5,8938,87,47.64000000000001,142.72,154.71999999999997,cont,1.6941355649711274,88001625,1,1,0,0
6bba_eebc57a5,8963,87,58.999999999999986,218.72000000000003,44.00000000000001,cont,5.517204058669544,88001628,1,0,1,0
6bba_eebc57a5,9197,90,46.2,233.91999999999996,96.48,cont,2.257405175195619,91001663,1,1,0,0
6bba_eebc57a5,9215,90,58.2,225.76000000000002,45.92,cont,3.4886445254568437,91001664,1,1,0,0
6bba_eebc57a5,9275,91,41.81526977539062,157.513388671875,157.58841552734376,cont,6.8968906570106325,92001674,3,0,3,0
6bba_eebc57a5,9278,91,42.559999999999995,198.08000000000004,132.48000000000002,cont,3.0653547918634185,92001675,1,0,0,1
6bba_eebc57a5,9279,91,43.866666666666646,250.93333333333328,101.06666666666663,cont,0.867512607977542,92001673,2,0,2,0
6bba_eebc57a5,9282,91,45.32,235.99999999999994,101.44000000000001,cont,6.535854238353266,92001673,2,0,1,1
6bba_eebc57a5,9305,91,57.32000000000001,232.16,48.0,cont,5.984767121618026,92001676,1,0,1,0
6bba_eebc57a5,9494,93,56.16000000000001,240.64000000000004,52.64000000000001,cont,4.748588112534104,94001694,1,1,0,0
6bba_ef7b4f7e,486,7,20.840000000000003,88.64,229.28000000000003,cont,2.002511704085638,8000115,1,1,0,0
6bba_ef7b4f7e,540,8,20.320000000000004,90.72,228.0,cont,1.1827946250300574,9000130,1,0,1,0
6bba_ef7b4f7e,917,14,50.00000000000001,71.99999999999999,146.08,parent,1.6435009317003708,15000245,1,1,0,0
6bba_ef7b4f7e,973,15,46.999999999999986,67.99999999999997,159.99999999999994,daughter,0.9084026158592864,16000265,2,0,0,2
6bba_ef7b4f7e,977,15,50.52000000000001,71.99999999999999,141.28000000000003,daughter,1.2772541348533597,16000264,1,0,1,0
6bba_ef7b4f7e,1041,16,55.99999999999999,48.64,240.00000000000006,cont,0.26000000000000156,17000277,1,1,0,0
6bba_ef7b4f7e,1111,17,55.84000000000001,49.28,239.36000000000007,cont,5.0626248441396475,18000293,1,0,1,0
6bba_ef7b4f7e,1112,17,58.133333333333326,39.999999999999986,220.5333333333333,cont,4.905972214358301,18000293,2,0,0,2
6bba_ef7b4f7e,1183,18,55.51999999999999,50.080000000000005,238.07999999999998,cont,5.346151214892834,19000309,1,1,0,0
6bba_ef7b4f7e,1185,18,58.76,39.68,219.03999999999996,cont,5.4948312360344715,19000309,1,1,0,0
6bba_ef7b4f7e,1255,19,55.00000000000001,50.08000000000002,236.0,cont,0.552021851922542,20000326,1,0,1,0
6bba_ef7b4f7e,1458,22,50.79999999999999,67.36000000000001,110.56,cont,6.421361810395034,23000373,1,1,0,0
6bba_ef7b4f7e,1480,23,5.159999999999999,204.00000000000003,142.55999999999997,cont,1.6047234964317139,24000379,1,1,0,0
6bba_ef7b4f7e,1513,23,45.20000000000001,70.56,114.56000000000002,cont,3.0592324404987448,24000386,1,0,0,1
6bba_ef7b4f7e,1544,24,5.68,202.72000000000003,140.0,cont,0.9610882503703939,25000393,1,0,1,0
6bba_ef7b4f7e,1599,25,1.0,144.64000000000001,134.07999999999998,cont,0.6670422119326486,26000407,1,1,0,0
6bba_ef7b4f7e,1865,28,58.680000000000014,40.000000000000014,210.71999999999997,cont,1.3299212852646454,29000462,1,1,0,0
6bba_ef7b4f7e,1937,29,58.36000000000001,40.000000000000014,208.8,cont,1.1428268132136201,30000476,1,0,1,0
6bba_ef7b4f7e,2876,44,10.2,162.88,163.36,cont,3.650467649836112,45000658,1,1,0,0
6bba_ef7b4f7e,2943,45,12.13333333333333,186.93333333333334,165.0666666666666,cont,1.4587335760286617,46000669,1,0,0,1
6bba_ef7b4f7e,3364,51,6.32,89.27999999999999,227.99999999999997,cont,1.9683975875061304,52000743,1,1,0,0
6bba_ef7b4f7e,3434,52,6.16,88.64,226.72000000000003,cont,2.3680437601531077,53000756,1,0,1,0
6bba_ef7b4f7e,3615,54,47.68000000000001,32.64,206.72000000000008,cont,0.5520218519225246,55000789,1,1,0,0
6bba_ef7b4f7e,3691,55,47.84,32.63999999999999,206.08000000000004,cont,0.6114838509723689,56000802,1,0,1,0
6bba_ef7b4f7e,4648,68,33.68,76.0,68.64000000000001,cont,1.1723673965528085,69001010,1,1,0,0
6bba_ef7b4f7e,4720,69,32.76,76.63999999999999,68.0,cont,1.8850700418021644,70001028,1,0,1,0
6bba_ef7b4f7e,5214,76,51.64,143.99999999999997,35.99999999999999,cont,1.6528734449436973,77001158,1,1,0,0
6bba_ef7b4f7e,5277,77,44.08,27.360000000000003,186.71999999999997,cont,2.4841249087958555,78001170,1,1,0,0
6bba_ef7b4f7e,5283,77,47.533333333333324,144.0,36.0,cont,2.012931285740046,78001179,1,0,0,1
6bba_ef7b4f7e,5352,78,44.36000000000001,27.999999999999996,185.28,cont,1.6706707111217398,79001189,1,0,1,0
6bba_ef7b4f7e,5423,79,48.839999999999996,143.99999999999997,42.07999999999999,cont,1.3010152285811418,80001210,1,0,0,1
6bba_ef7b4f7e,5544,81,31.68,81.28,71.99999999999999,cont,1.30739304055819,82001234,1,1,0,0
6bba_ef7b4f7e,5613,82,31.000000000000007,82.72000000000001,71.99999999999999,cont,0.519999999999996,83001251,1,0,1,0
6bba_ef7b4f7e,5696,83,48.16,143.99999999999997,47.360000000000014,cont,0.5479407472528429,84001271,1,1,0,0
6bba_ef7b4f7e,5764,84,47.68000000000001,144.64000000000001,48.64,cont,0.599490669235127,85001287,1,0,1,0
6bba_f17befbc,902,5,10.880000000000003,48.800000000000004,108.32,cont,4.067631758775616,6000081,4,0,2,2
6bba_f17befbc,965,5,20.320000000000004,217.92000000000002,51.99999999999999,cont,2.7301934455272545,6000098,2,0,2,0
6bba_f17befbc,1027,5,29.480000000000004,209.28,204.64000000000001,cont,0.8300235312929475,6000092,1,0,1,0
6bba_f17befbc,1120,6,16.68,20.640000000000004,231.52000000000004,cont,1.8984701900477454,7000106,1,1,0,0
6bba_f17befbc,1179,6,25.64,246.39999999999998,104.48000000000003,cont,4.107426581814949,7000111,1,1,0,0
6bba_f17befbc,1287,7,14.866666666666667,78.93333333333334,78.93333333333334,cont,1.8658038370453942,8000125,5,0,0,5
6bba_f17befbc,4020,21,8.16,131.36,112.64,cont,2.0046204441240243,22000390,2,2,0,0
6bba_f17befbc,4284,22,9.080000000000002,125.43999999999998,111.99999999999999,cont,5.024640845373928,23000408,10,5,2,3
6bba_f17befbc,4475,23,4.319999999999999,32.800000000000004,177.44,cont,1.998155399362125,24000428,5,0,3,2
6bba_f17befbc,4510,23,6.68,140.79999999999998,138.72,cont,2.603704392303398,24000432,6,0,2,4
6bba_f17befbc,4534,23,9.200000000000001,119.52000000000001,177.44,cont,2.6547745902430164,24000430,7,0,3,4
6bba_f17befbc,4938,24,32.68,238.08000000000004,88.00000000000001,cont,6.357323164076208,25000467,3,2,1,0
6bba_f17befbc,6327,29,18.319999999999997,224.48000000000002,114.55999999999999,cont,6.263927107853338,30000582,10,3,1,6
6bba_f17befbc,6364,29,23.72,202.88000000000002,104.16,cont,4.066852668218987,30000586,7,3,1,3
6bba_f17befbc,6591,30,12.32,223.36000000000004,97.28000000000003,cont,6.164385614154901,31000593,6,2,4,0
6bba_f17befbc,6672,30,24.800000000000004,241.60000000000002,88.0,cont,4.741130699790932,31000594,7,1,3,3
6bba_f17befbc,6680,30,23.400000000000002,226.08,97.12000000000002,cont,4.491971483936648,31000594,10,1,5,4
6bba_f17befbc,6961,31,17.84,220.8,109.91999999999999,cont,2.290039573893866,32000613,2,0,2,0
6bba_f17befbc,9949,40,6.679999999999999,158.88,100.64000000000001,cont,6.7859852131064935,41000776,6,0,4,2
6bba_f17befbc,10002,40,11.36,151.36,117.11999999999998,cont,3.929040316985806,41000776,5,0,3,2
6bba_f17befbc,10755,42,6.359999999999999,227.99999999999997,19.36,cont,1.4783928140044595,43000822,3,3,0,0
6bba_f17befbc,10990,42,46.15999999999999,244.00000000000003,56.640000000000015,cont,2.2596850582547736,43000831,2,2,0,0
6bba_f17befbc,10998,43,0.8000000000000004,68.96000000000001,140.64000000000001,cont,3.1376389411466667,44000832,7,1,1,5
6bba_f17befbc,11123,43,4.800000000000001,227.20000000000002,23.360000000000007,cont,1.7125063868494095,44000836,3,0,3,0
6bba_f17befbc,11964,45,8.491968154907227,250.9351000976563,114.83864135742189,cont,1.4953747248980869,46000883,5,5,0,0
6bba_f17befbc,12350,46,8.651968154907225,249.49510009765626,122.19864135742188,cont,1.263855409638309,47000910,5,0,5,0
6bba_f17befbc,12370,46,11.8,104.00000000000001,60.640000000000015,cont,0.7588941378743073,47000912,2,0,2,0
6bba_f17befbc,13052,48,4.866666666666665,136.00000000000003,123.46666666666665,cont,1.5822323474726754,49000948,5,0,0,5
6bba_f17befbc,13113,48,10.84,190.56,9.280000000000001,cont,1.4144035845542806,49000955,2,1,1,0
6bba_f17befbc,13344,49,0.0,219.99999999999997,47.999999999999986,cont,4.942244555867302,50000976,6,0,0,6
6bba_f17befbc,13404,49,2.8400000000000007,61.92,150.72,cont,1.655906096371412,50000966,1,1,0,0
6bba_f17befbc,13485,49,8.159999999999998,245.28,142.07999999999998,cont,0.46928469504129816,50000977,3,2,1,0
6bba_f17befbc,13618,49,22.559999999999995,238.40000000000003,7.680000000000002,cont,5.7978579018461565,50000982,3,1,2,0
6bba_f17befbc,14407,51,20.839999999999996,186.71999999999997,119.84,cont,6.03050176705885,52001026,2,2,0,0
6bba_f17befbc,14524,52,0.9600000000000002,197.11999999999995,52.96000000000001,cont,6.487821403406554,53001044,2,1,1,0
6bba_f17befbc,14536,52,0.9600000000000002,214.08,59.36,cont,2.11880309667982,53001044,1,1,0,0
6bba_f17befbc,14669,52,7.480000000000001,248.0,164.64,cont,4.254894240753822,53001045,2,2,0,0
6bba_f17befbc,14694,52,9.040000000000001,183.36,39.040000000000006,cont,2.064965193411261,53001041,8,1,3,4
6bba_f17befbc,15006,53,1.9200000000000006,250.72000000000003,125.44000000000001,cont,2.2404383639145142,54001066,2,0,2,0
6bba_f17befbc,15065,53,3.5600000000000005,243.36000000000004,174.72000000000003,cont,2.7488929589927604,54001065,7,0,0,7
6bba_f17befbc,15143,53,10.880000000000003,133.92000000000002,34.07999999999999,cont,5.548298303308861,54001067,1,0,1,0
6bba_f17befbc,15186,53,16.0,234.72,37.440000000000005,cont,1.0462022091832885,54001071,1,0,1,0
6bba_f17befbc,15454,54,3.0400000000000005,156.64,232.64000000000004,cont,1.9485098472935587,55001081,1,0,1,0
6bba_f17befbc,16563,57,0.0,187.99999999999994,203.99999999999991,cont,1.8168052317185919,58001144,5,0,0,5
6bba_f17befbc,17393,59,0.0,140.64000000000001,246.55999999999997,cont,3.3054672986583924,60001185,2,1,1,0
6bba_f17befbc,18374,61,4.160000000000001,180.0,221.44000000000003,cont,5.726298024247425,62001229,1,0,1,0
6bba_f17befbc,19760,64,10.04,170.07999999999998,134.56000000000003,cont,1.7247979954186012,65001306,9,1,1,7
6bba_f17befbc,20041,65,2.6800000000000006,197.27999999999997,242.08,cont,5.519668383381361,66001315,1,0,1,0
6bba_f17befbc,21266,68,3.000000000000001,241.76,25.92,cont,5.778240432865358,69001379,8,0,2,6
6bba_f17befbc,21273,68,4.160000000000001,46.88000000000001,98.72,cont,1.624106004083482,69001374,3,0,0,3
6bba_f17befbc,22193,70,9.48,233.92000000000007,16.16,cont,4.221060663506284,71001431,3,3,0,0
6bba_f17befbc,22319,70,25.64,232.8,120.8,cont,5.05113678418037,71001437,5,2,0,3
6bba_f17befbc,22427,71,0.6400000000000001,248.64000000000007,50.56,cont,2.3137283899585137,72001450,7,0,3,4
6bba_f17befbc,22469,71,2.1000468063354494,229.73216064453123,107.7718103027344,cont,6.0302611868072535,72001447,2,0,2,0
6bba_f17befbc,22488,71,3.960000000000001,157.28000000000003,148.64000000000001,cont,1.6508752413492709,72001444,3,0,0,3
6bba_f17befbc,22910,72,6.120000000000001,200.0,8.64,cont,1.6351247085467215,73001471,8,0,2,6
6bba_f17befbc,23649,74,3.133333333333333,211.9999999999999,92.0,cont,6.570095141899467,75001524,8,0,0,8
6bba_f17befbc,24066,75,5.8,231.36,153.27999999999997,cont,6.827030947820568,76001550,1,0,1,0
6bba_f17befbc,25033,78,0.6400000000000001,119.36000000000001,176.64000000000001,cont,1.3915352358097106,79001624,1,1,0,0
6bba_f17befbc,25141,78,1.8400000000000003,249.44,167.36,cont,5.967490312099384,79001627,3,0,0,3
6bba_f17befbc,25257,78,8.519999999999998,209.28000000000003,20.640000000000004,cont,6.709577576122064,79001630,2,2,0,0
6bba_f17befbc,25414,79,0.9600000000000002,116.0,181.28000000000003,cont,1.8786152646297756,80001648,2,1,1,0
6bba_f17befbc,26168,81,0.0,243.46666666666664,92.53333333333332,cont,3.31624329898389,82001704,6,0,0,6
6bba_f17befbc,26202,81,0.8400000000000001,240.64000000000004,77.11999999999999,cont,6.100159962041979,82001704,8,1,4,3
6bba_f17befbc,26938,83,3.6799999999999997,17.919999999999998,50.56,cont,0.9646534351776286,84001740,2,1,1,0
6bba_f17befbc,27041,83,11.64,209.28000000000006,152.48,cont,2.8847297317079863,84001753,4,4,0,0
6bba_f17befbc,27043,83,9.560000000000002,232.0,77.6,cont,6.5980002889133,84001755,4,4,0,0
6bba_f17befbc,27269,84,3.3200000000000003,21.440000000000005,54.720000000000006,cont,2.4208682584973498,85001765,1,0,1,0
6bba_f17befbc,27287,84,2.5199999999999996,212.8,35.36,cont,1.5230853759720742,85001773,3,0,3,0
6bba_f17befbc,27357,84,8.479999999999999,235.99999999999994,77.91999999999999,cont,2.6816439104586527,85001776,10,0,4,6
6bba_f17befbc,27401,84,13.600000000000001,195.51999999999998,151.36,cont,1.4259317129512243,85001779,7,1,1,5
6bba_f17befbc,27477,84,32.0,213.92000000000002,241.43999999999997,cont,2.096816783841643,85001781,2,0,1,1
6bba_f17befbc,27624,85,3.1999999999999997,96.80000000000003,218.24,cont,1.7993527482125322,86001788,1,0,1,0
6bba_f17befbc,28109,86,24.266666666666666,206.39999999999998,192.0,cont,4.383569484291433,87001817,4,0,0,4
6bba_f17befbc,28520,88,0.31999999999999995,231.20000000000002,128.8,cont,4.446506001907568,89001849,1,1,0,0
6bba_f17befbc,28677,88,6.147360305786132,161.2771118164063,192.60591796875002,cont,6.77733008036234,89001847,1,0,1,0
6bba_f17befbc,28911,89,0.68,232.0,129.12,cont,5.121534956680469,90001867,5,1,1,3
6bba_f17befbc,32759,91,3.2565759468078617,163.37219055175783,203.21606811523438,cont,2.169848243373963,92001900,1,1,0,0
6bba_f17befbc,30265,93,3.6799999999999997,136.0,18.72,cont,4.1651384280717485,94001943,1,0,1,0
6bba_f17befbc,30279,93,3.3600000000000003,216.0,153.27999999999997,cont,2.411359834201439,94001948,3,1,2,0
6bba_f1fde7e0,398,1,30.999999999999996,207.99999999999997,210.93333333333328,cont,1.8559495173929552,2000048,4,0,0,4
6bba_f1fde7e0,1098,4,27.479999999999997,199.2,200.0,cont,5.823511209957449,5000148,7,3,0,4
6bba_f1fde7e0,1190,4,35.39999999999999,88.63999999999997,157.28,cont,2.640562499260326,5000151,2,2,0,0
6bba_f1fde7e0,1411,5,32.86666666666666,78.93333333333332,135.73333333333332,cont,1.4088540703943255,6000177,3,0,0,3
6bba_f1fde7e0,1418,5,31.88,183.35999999999999,184.48,cont,4.720030620928217,6000165,7,1,4,2
6bba_f1fde7e0,1631,6,31.680000000000003,179.2,175.52,cont,5.883599690028214,7000191,1,0,1,0
6bba_f1fde7e0,1642,6,33.16,68.48,112.16,cont,2.1841595923833035,7000204,4,0,2,2
6bba_f1fde7e0,1698,6,43.32000000000001,15.2,213.92000000000002,cont,2.7745655628404142,7000209,1,1,0,0
6bba_f1fde7e0,1801,7,26.2,176.64000000000001,146.56,cont,3.0865162906584533,8000230,3,0,3,0
6bba_f1fde7e0,1843,7,31.156506652832032,59.00757690429688,85.30504272460938,cont,6.704899412563385,8000232,3,2,1,0
6bba_f1fde7e0,1934,7,42.68000000000001,9.919999999999998,200.64,cont,2.399778961175364,8000239,1,0,1,0
6bba_f1fde7e0,2025,8,26.0,174.08,133.28000000000003,cont,2.278073816297445,9000248,3,3,0,0
6bba_f1fde7e0,2106,8,30.876506652832038,65.72757690429688,81.46504272460938,cont,2.1031669066358427,9000263,3,0,2,1
6bba_f1fde7e0,2267,9,26.2,172.64,127.20000000000003,cont,1.822682175394286,10000282,3,0,3,0
6bba_f1fde7e0,2305,9,30.360000000000003,56.16,62.40000000000002,cont,1.8784044093325512,10000289,4,0,0,4
6bba_f1fde7e0,2457,10,25.480000000000004,171.20000000000005,120.16,cont,5.566544175922799,11000308,3,3,0,0
6bba_f1fde7e0,2504,10,29.840000000000003,53.599999999999994,59.04000000000001,cont,4.2415924191275165,11000315,1,1,0,0
6bba_f1fde7e0,2663,11,24.320000000000007,166.55999999999995,81.44,cont,2.3946568125725087,12000338,4,0,3,1
6bba_f1fde7e0,3452,15,27.0,180.15999999999997,160.0,cont,1.7886814277562186,16000432,2,2,0,0
6bba_f1fde7e0,3618,16,26.16,176.0,148.64000000000001,cont,1.7071783188934935,17000446,2,0,2,0
6bba_f1fde7e0,3781,17,25.32,172.64,137.28000000000003,cont,0.7497926796788673,18000463,1,1,0,0
6bba_f1fde7e0,3945,18,24.840000000000003,168.0,126.72000000000001,cont,1.6495951207796469,19000476,1,0,1,0
6bba_f1fde7e0,4091,19,24.160000000000004,164.0,115.35999999999999,cont,0.865531809063076,20000489,1,1,0,0
6bba_f1fde7e0,4230,20,23.840000000000003,160.00000000000006,104.64,cont,1.5723082156498596,21000505,1,0,1,0
6bba_f1fde7e0,4516,22,22.199999999999996,155.2,93.60000000000001,cont,3.5425372704461466,23000531,1,1,0,0
6bba_f1fde7e0,4633,23,22.64,140.48000000000002,53.120000000000005,cont,4.06002924866312,24000541,2,0,1,1
6bba_f1fde7e0,4727,24,17.68,132.00000000000003,77.6,cont,3.7813648742881165,25000552,3,3,0,0
6bba_f1fde7e0,4840,25,17.360000000000003,129.44,64.64,cont,1.3149453410693546,26000563,3,0,3,0
6bba_f1fde7e0,4857,25,20.86666666666666,128.26666666666668,22.4,cont,2.1693733094042402,26000564,2,0,0,2
6bba_f1fde7e0,4950,26,16.48,125.43999999999998,54.080000000000005,cont,1.8189115048566866,27000574,4,4,0,0
6bba_f1fde7e0,5059,27,16.0,120.63999999999999,39.84,cont,1.9716815475628948,28000585,4,0,4,0
6bba_f1fde7e0,5098,27,21.679999999999996,193.44,110.71999999999998,cont,5.2673316596261515,28000586,1,0,0,1
6bba_f1fde7e0,5265,29,15.16,111.36000000000001,20.0,cont,1.1035652450127185,30000608,1,1,0,0
6bba_f1fde7e0,5357,30,15.000000000000004,106.08000000000001,14.559999999999999,cont,2.2865776829139124,31000621,1,0,1,0
6bba_f1fde7e0,5463,31,17.84,227.36,82.72,cont,1.3775170144139777,32000633,1,1,0,0
6bba_f1fde7e0,5547,32,17.000000000000004,222.56,71.84,cont,2.560960818228197,33000642,1,0,1,0
6bba_f1fde7e0,5558,32,19.32,174.40000000000003,41.44,cont,6.767593391487105,33000641,4,0,1,3
6bba_f1fde7e0,5629,33,16.16,179.99999999999997,50.72,cont,1.8388366736880188,34000649,2,2,0,0
6bba_f1fde7e0,5631,33,16.520000000000003,220.00000000000003,65.44000000000001,cont,1.3325990816821018,34000650,1,1,0,0
6bba_f1fde7e0,5642,33,19.160000000000004,173.75999999999996,41.440000000000005,cont,6.75134211194338,34000649,1,1,0,0
6bba_f1fde7e0,5660,33,20.577259216308594,178.43590087890624,58.93063903808594,cont,6.456996057035901,34000649,1,1,0,0
6bba_f1fde7e0,5709,34,15.64,217.27999999999997,58.400000000000006,cont,3.490725434991985,35000657,1,0,1,0
6bba_f1fde7e0,5711,34,15.84,176.64000000000001,40.64000000000001,cont,0.7299850169695276,35000658,2,0,2,0
6bba_f1fde7e0,5729,34,19.0,176.80000000000004,47.20000000000001,cont,5.905036250947155,35000658,3,0,1,2
6bba_f1fde7e0,5787,35,15.679999999999998,174.72,34.24,cont,2.206292846609442,36000668,2,2,0,0
6bba_f1fde7e0,5793,35,18.319999999999997,215.83999999999992,114.55999999999999,cont,1.7090334439383956,36000669,1,1,0,0
6bba_f1fde7e0,5805,35,18.999999999999996,180.32,50.39999999999999,cont,6.858077582311234,36000668,1,1,0,0
6bba_f1fde7e0,5838,35,26.84,10.079999999999998,35.519999999999996,cont,1.6354476642497633,36000672,1,1,0,0
6bba_f1fde7e0,5865,36,15.0,174.55999999999995,22.56,cont,3.7061758002690524,37000677,3,0,2,1
6bba_f1fde7e0,5873,36,17.279999999999998,198.88000000000002,90.07999999999998,cont,3.7550160202720715,37000679,3,0,1,2
6bba_f1fde7e0,5919,36,26.560000000000006,5.919999999999999,24.799999999999997,cont,1.8449297073330513,37000683,1,0,1,0
6bba_f1fde7e0,5943,37,14.360000000000003,174.40000000000003,14.560000000000002,cont,2.3572023937286426,38000689,2,2,0,0
6bba_f1fde7e0,5952,37,16.640000000000004,200.79999999999998,89.28000000000002,cont,1.6686938192790322,38000690,1,1,0,0
6bba_f1fde7e0,6017,38,13.319999999999999,175.68,7.840000000000001,cont,4.017695429285295,39000699,2,0,2,0
6bba_f1fde7e0,6027,38,15.0,188.15999999999997,69.28,cont,2.5468991563271755,39000700,5,0,2,3
6bba_f1fde7e0,6060,38,20.866666666666664,189.59999999999997,44.533333333333324,cont,1.942462283346124,39000701,1,0,0,1
6bba_f1fde7e0,6078,38,25.999999999999996,43.466666666666654,73.06666666666666,cont,2.8617485974002377,39000706,1,0,0,1
6bba_f1fde7e0,6368,42,26.04,245.28,230.24000000000004,cont,2.8146294716178817,43000766,2,2,0,0
6bba_f1fde7e0,6433,43,24.800000000000004,244.64000000000001,221.92000000000004,cont,1.2779775281670689,44000776,2,0,2,0
6bba_f1fde7e0,6889,50,18.680000000000003,189.76,105.60000000000001,cont,2.5582785349722945,51000832,2,0,1,1
6bba_f1fde7e0,6943,51,17.080000000000002,228.0,158.08,cont,1.5495548756013748,52000838,1,1,0,0
6bba_f1fde7e0,6998,52,16.32,227.36,149.92000000000002,cont,0.9241093685273369,53000845,1,0,1,0
6bba_f1fde7e0,7592,62,11.159999999999998,80.80000000000001,53.440000000000005,cont,1.740874869857104,63000919,1,1,0,0
6bba_f1fde7e0,7649,63,11.000000000000002,72.00000000000001,48.800000000000004,cont,1.2078680857196218,64000927,2,1,1,0
6bba_f1fde7e0,7710,64,11.000000000000002,63.84,41.92,cont,1.7889766627879728,65000935,1,0,1,0
6bba_f1fde7e0,8273,73,4.680000000000001,103.84000000000003,37.440000000000005,cont,2.494308834627345,74001016,1,1,0,0
6bba_f1fde7e0,8301,73,14.479999999999999,47.52,59.36,cont,1.746704933582086,74001019,1,1,0,0
6bba_f1fde7e0,8311,73,16.360000000000003,235.51999999999998,166.72000000000003,cont,2.8778561899441764,74001022,1,1,0,0
6bba_f1fde7e0,8341,74,4.960000000000001,71.67999999999999,28.639999999999997,cont,3.368143266326422,75001026,2,0,2,0
6bba_f1fde7e0,8374,74,14.16,36.48000000000001,57.279999999999994,cont,2.6959332799607667,75001030,1,0,1,0
6bba_f1fde7e0,8379,74,15.64,225.92,167.36,cont,1.1808954494365667,75001033,1,0,1,0
6bba_f1fde7e0,8701,79,4.360000000000001,45.28000000000001,12.160000000000002,cont,2.1333966461959224,80001080,1,1,0,0
6bba_f1fde7e0,8783,80,5.800000000000002,10.24,8.16,cont,4.574656407862782,81001092,1,0,1,0
6bba_f1fde7e0,8829,81,1.0,212.48,79.20000000000002,cont,6.810843950091648,82001103,2,0,1,1
6bba_f1fde7e0,8831,81,1.4800000000000002,226.4,95.2,cont,2.14530774307558,82001103,2,0,2,0
6bba_f20478e9,244,3,59.19999999999999,28.000000000000004,210.08000000000004,cont,0.5460096725333756,4000040,1,1,0,0
6bba_f20478e9,263,4,21.84,184.0,71.36,cont,2.20647236839712,5000044,1,1,0,0
6bba_f20478e9,291,4,44.68000000000001,172.79999999999998,214.07999999999998,cont,1.2952177664392976,5000050,1,1,0,0
6bba_f20478e9,302,4,58.0,25.44,202.08,cont,0.6345827861674148,5000051,1,0,1,0
6bba_f20478e9,321,5,20.48,180.64000000000001,61.91999999999999,cont,2.6032479712850995,6000055,1,0,1,0
6bba_f20478e9,349,5,42.480000000000004,167.20000000000002,202.56,cont,1.3025365877778698,6000059,1,0,1,0
6bba_f20478e9,366,5,58.839999999999996,214.07999999999998,228.00000000000003,cont,0.8537051598766553,6000064,1,1,0,0
6bba_f20478e9,424,6,58.0,209.92000000000004,221.92000000000002,cont,0.780676789715181,7000076,1,0,1,0
6bba_f20478e9,561,9,16.840000000000003,171.36,32.0,cont,1.8906649789161436,10000100,1,1,0,0
6bba_f20478e9,622,10,15.64,168.64000000000004,23.840000000000003,cont,1.0982884411665192,11000111,1,0,1,0
6bba_f20478e9,847,13,37.96,69.6,105.28000000000002,cont,2.693042250411234,14000159,1,1,0,0
6bba_f20478e9,861,13,43.84,241.44000000000003,149.44,cont,4.452411039257261,14000162,2,2,0,0
6bba_f20478e9,919,14,38.519999999999996,59.51999999999999,102.72,cont,3.0702180806092634,15000175,1,0,1,0
6bba_f20478e9,922,14,42.8,237.91999999999993,142.88000000000002,cont,0.9175136238770331,15000172,2,0,2,0
6bba_f20478e9,998,15,56.800000000000004,92.96000000000002,249.92000000000004,cont,3.1417600560832115,16000201,1,1,0,0
6bba_f20478e9,999,15,57.2,55.040000000000006,249.12,cont,2.3763367948378,16000203,1,1,0,0
6bba_f20478e9,1001,15,61.68,78.88,229.44,cont,1.7200454209119027,16000204,1,1,0,0
6bba_f20478e9,1002,15,62.0,229.76,244.32000000000002,cont,2.6012692214570863,16000205,1,1,0,0
6bba_f20478e9,1053,16,52.07999999999999,88.64,236.32000000000005,cont,4.143240164714084,17000219,3,0,1,2
6bba_f20478e9,1054,16,55.63999999999999,49.91999999999999,243.2,cont,1.3716578700608906,17000220,1,0,1,0
6bba_f20478e9,1060,16,57.56,84.48,250.72000000000003,cont,6.734032154846015,17000219,3,0,1,2
6bba_f20478e9,1062,16,61.32,71.20000000000002,220.48,cont,1.7515359331169857,17000223,1,0,1,0
6bba_f20478e9,1063,16,62.000000000000014,224.64000000000001,241.28000000000003,cont,0.6251499820043094,17000224,1,0,1,0
6bba_f20478e9,1086,17,28.640000000000004,229.92000000000004,114.55999999999999,cont,0.939975066158669,18000226,1,1,0,0
6bba_f20478e9,1142,18,27.84,225.92000000000002,109.92,cont,1.5724761524423851,19000246,1,0,1,0
6bba_f20478e9,1256,20,26.359999999999996,220.79999999999998,100.32000000000001,cont,3.040530895254975,21000281,1,1,0,0
6bba_f20478e9,1281,20,50.839999999999996,78.08000000000001,229.28000000000003,cont,1.5762499999999935,21000292,1,1,0,0
6bba_f20478e9,1282,20,52.360000000000014,42.72,228.16000000000003,cont,5.08019807684698,21000295,1,1,0,0
6bba_f20478e9,1337,21,51.31999999999999,41.28000000000001,223.20000000000005,cont,1.4509894253577655,22000311,1,0,1,0
6bba_f20478e9,1338,21,50.52000000000001,76.63999999999999,228.80000000000007,cont,1.9908737089528974,22000310,1,0,1,0
6bba_f20478e9,1675,27,41.31999999999999,66.72,185.12000000000003,cont,2.76999351307904,28000420,1,1,0,0
6bba_f20478e9,1679,27,45.64000000000001,74.55999999999999,211.20000000000005,cont,2.0820936728447212,28000421,1,1,0,0
6bba_f20478e9,1681,27,46.2,32.0,200.8,cont,3.3023902328465025,28000422,1,1,0,0
6bba_f20478e9,1691,27,55.68000000000001,24.16,165.44,cont,4.63797319014458,28000425,1,1,0,0
6bba_f20478e9,1731,28,35.86666666666666,60.53333333333332,162.9333333333333,cont,3.4719523506235026,29000438,2,0,0,2
6bba_f20478e9,1735,28,41.0,65.28,186.88,cont,2.390076162489387,29000439,2,0,1,1
6bba_f20478e9,1738,28,44.63999999999999,29.920000000000005,194.56000000000006,cont,2.965168629268832,29000440,1,0,1,0
6bba_f20478e9,1751,28,54.64,16.480000000000004,158.56,cont,1.5616917909754184,29000446,1,0,1,0
6bba_f20478e9,1995,33,13.32,168.64000000000001,27.84000000000001,cont,2.7346871964632475,34000517,2,2,0,0
6bba_f20478e9,2053,34,10.24,157.28,5.76,cont,3.4647576123734836,35000535,2,0,1,1
6bba_f20478e9,2067,34,16.36,178.88,52.16,cont,4.593205579984418,35000539,1,0,0,1
6bba_f20478e9,2302,38,12.520000000000001,160.79999999999998,37.44,cont,3.6984371699408443,39000601,1,1,0,0
6bba_f20478e9,2360,39,11.480000000000004,152.64000000000004,32.64,cont,2.875653186669058,40000616,1,0,1,0
6bba_f20478e9,3489,57,32.32,239.52,203.20000000000002,cont,3.317070431872071,58000809,1,1,0,0
6bba_f20478e9,3547,58,30.96,230.56000000000003,203.35999999999996,cont,1.9239615705621633,59000819,1,0,1,0
6bba_f20478e9,4052,66,5.999999999999998,147.46666666666667,100.0,cont,2.0880529873012788,67000885,1,0,0,1
6bba_f20478e9,4207,68,34.839999999999996,116.0,246.56,cont,1.8986788169672149,69000913,2,2,0,0
6bba_f20478e9,4265,69,30.0,100.0,231.4666666666667,cont,1.7778305384966042,70000915,2,0,0,2
6bba_f20478e9,4746,77,5.000000000000002,232.64000000000004,125.28000000000002,cont,1.9092885618732367,78001000,1,1,0,0
6bba_f20478e9,4816,78,5.000000000000002,229.27999999999997,126.72000000000001,cont,1.9413819980879563,79001013,1,0,1,0
6bba_f20478e9,4946,80,1.68,75.36,77.43999999999998,cont,2.6000507807541027,81001037,1,1,0,0
6bba_f20478e9,5003,81,1.3200000000000005,72.79999999999998,74.56000000000002,cont,1.1655906496279003,82001050,1,0,1,0
6bba_f4ae811c,390,1,59.519999999999996,103.67999999999999,196.63999999999996,cont,2.4683958606755145,2000040,2,0,2,0
6bba_f4ae811c,590,2,59.03999999999999,106.72,197.43999999999997,cont,1.6540712084127613,3000059,1,1,0,0
6bba_f4ae811c,775,3,58.0,109.28,196.00000000000006,cont,3.263135953343041,4000079,1,0,1,0
6bba_f4ae811c,789,3,61.99999999999999,111.99999999999999,207.99999999999997,cont,5.915089285040401,4000079,1,0,0,1
6bba_f4ae811c,1155,5,55.480000000000004,116.0,193.92000000000002,cont,2.62692697129174,6000112,1,1,0,0
6bba_f4ae811c,1305,6,39.19999999999999,58.72,180.0,cont,6.747644959539573,7000131,1,1,0,0
6bba_f4ae811c,1339,6,54.36000000000001,120.00000000000003,194.08,cont,2.695982253743512,7000133,1,0,1,0
6bba_f4ae811c,1482,7,37.160000000000004,61.91999999999999,178.72000000000003,cont,6.385939877966275,8000148,1,0,1,0
6bba_f4ae811c,1879,9,58.32000000000001,71.36,203.35999999999999,cont,1.2008519111447558,10000178,2,2,0,0
6bba_f4ae811c,2040,10,57.16,73.92,201.28000000000006,cont,1.8887086362115177,11000189,2,0,2,0
6bba_f4ae811c,2047,10,59.999999999999986,75.46666666666667,203.46666666666664,cont,6.603725340143634,11000189,4,0,0,4
6bba_f4ae811c,2379,12,61.36000000000001,199.36000000000007,185.92000000000004,cont,0.6410001950077677,13000216,1,1,0,0
6bba_f4ae811c,2526,13,61.15999999999999,201.92000000000002,184.64,cont,0.5305569008127117,14000230,1,0,1,0
6bba_f4ae811c,2535,13,61.99999999999999,192.0,194.93333333333325,cont,6.236580732750182,14000230,3,0,0,3
6bba_f4ae811c,3038,17,12.440000000000001,25.92,125.28000000000002,cont,2.634755977789978,18000277,1,1,0,0
6bba_f4ae811c,3185,18,12.16,32.48000000000001,122.71999999999998,cont,1.1632094609312684,19000291,1,0,1,0
6bba_f4ae811c,3328,19,7.2,202.72000000000006,125.92000000000004,cont,0.4384489708050532,20000304,1,1,0,0
6bba_f4ae811c,3470,20,6.32,206.72,124.0,cont,1.3762704358155775,21000320,1,0,1,0
6bba_f4ae811c,3582,20,61.32,81.28000000000002,198.72,cont,0.5443143622944334,21000330,1,1,0,0
6bba_f4ae811c,3624,21,8.999999999999998,137.92000000000002,148.64000000000001,cont,1.645989444073082,22000334,1,1,0,0
6bba_f4ae811c,3735,21,61.160000000000004,84.64000000000001,196.0,cont,0.2983103459486465,22000343,1,0,1,0
6bba_f4ae811c,3768,22,8.16,140.64,147.35999999999996,cont,1.688202168728603,23000348,1,0,1,0
6bba_f4ae811c,3970,23,34.0,213.28000000000003,100.64000000000001,cont,0.32702494170933283,24000368,1,1,0,0
6bba_f4ae811c,4019,23,60.52,93.27999999999999,193.27999999999997,cont,0.9986811615826093,24000372,1,1,0,0
6bba_f4ae811c,4113,24,34.0,215.99999999999997,99.36000000000001,cont,0.1462500000000091,25000382,1,0,1,0
6bba_f4ae811c,4162,24,60.0,99.36000000000001,193.27999999999997,cont,0.3270249417093646,25000387,1,0,1,0
6bba_f4ae811c,4275,25,42.68000000000001,12.64,188.00000000000003,cont,1.1638902976655359,26000399,1,0,1,0
6bba_f4ae811c,4279,25,45.13333333333333,4.0000000000000036,178.93333333333334,cont,6.979412309109957,26000399,3,1,0,2
6bba_f4ae811c,4442,26,60.0,219.36,180.0,cont,0.2599999999999909,27000417,1,1,0,0
6bba_f4ae811c,4578,27,59.480000000000004,222.72000000000003,180.0,cont,0.9444591706368222,28000430,1,0,1,0
6bba_f4ae811c,5036,31,11.32,155.52,146.72000000000003,cont,3.246626374022728,32000474,1,1,0,0
6bba_f4ae811c,5038,31,12.0,170.07999999999998,139.83999999999997,cont,3.8125215163327346,32000474,1,1,0,0
6bba_f4ae811c,5089,31,37.519999999999996,156.64000000000004,176.00000000000003,cont,0.9170818188689788,32000478,1,1,0,0
6bba_f4ae811c,5091,31,39.000000000000014,25.28,174.72,cont,6.729011001811482,32000479,1,0,1,0
6bba_f4ae811c,5155,32,9.96,172.64,141.43999999999997,cont,0.987513449275514,33000486,3,0,1,2
6bba_f4ae811c,5156,32,9.133333333333333,183.4666666666666,131.46666666666664,cont,5.667927632374402,33000486,1,0,0,1
6bba_f4ae811c,5210,32,36.160000000000004,159.35999999999999,173.28000000000003,cont,1.5889309495695616,33000490,1,0,1,0
6bba_f4ae811c,5241,32,56.0,241.43999999999997,173.28000000000003,cont,3.535822261157938,33000494,1,1,0,0
6bba_f4ae811c,5370,33,55.0,244.63999999999996,171.36,cont,3.4332139188375743,34000506,1,0,1,0
6bba_f4ae811c,5592,35,30.599999999999994,232.64000000000004,156.00000000000006,cont,1.009071355256901,36000530,1,1,0,0
6bba_f4ae811c,5709,36,24.866666666666667,235.46666666666664,152.5333333333333,cont,3.572229039759346,37000541,2,0,0,2
6bba_f4ae811c,5720,36,31.199999999999996,235.99999999999994,153.91999999999996,cont,6.974809697224711,37000541,2,0,1,1
6bba_f4ae811c,6455,42,49.52,51.99999999999999,191.36,cont,1.5488730782410869,43000619,1,0,1,0
6bba_f4ae811c,6784,45,60.000000000000014,24.64,200.0,cont,1.6950705184445594,46000658,1,1,0,0
6bba_f4ae811c,6885,46,59.84000000000002,27.999999999999996,200.0,cont,2.4472845283088906,47000672,1,0,1,0
6bba_f4ae811c,7078,48,58.519999999999996,113.92,193.43999999999997,cont,2.4873118647447514,49000696,1,1,0,0
6bba_f4ae811c,7173,49,56.68,120.0,189.92000000000002,cont,3.7701400836043173,50000708,1,0,1,0
6bba_f4ae811c,7175,49,57.68000000000002,40.63999999999999,193.92000000000002,cont,2.8430070458055816,50000711,1,1,0,0
6bba_f4ae811c,7184,49,62.26666666666667,123.4666666666667,196.5333333333333,cont,6.099705423215125,50000708,2,0,0,2
6bba_f4ae811c,7222,50,13.840000000000002,228.64000000000004,127.36000000000001,cont,2.458695347638674,51000713,1,1,0,0
6bba_f4ae811c,7266,50,55.04,125.28000000000002,187.35999999999999,cont,4.88368601186645,51000722,1,1,0,0
6bba_f4ae811c,7267,50,56.20000000000001,45.28,192.00000000000006,cont,1.5962265072664201,51000724,1,0,1,0
6bba_f4ae811c,7280,50,61.44,128.64000000000001,194.71999999999997,cont,6.340436819533809,51000722,1,1,0,0
6bba_f4ae811c,7308,51,11.640000000000002,233.44,121.28000000000003,cont,1.5872681917369957,52000727,1,0,1,0
6bba_f4ae811c,7329,51,28.519999999999996,209.76,145.44,cont,5.00821731008151,52000728,2,0,0,2
6bba_f4ae811c,7355,51,52.440000000000005,132.16,183.36,cont,5.883936288744121,52000734,1,0,1,0
6bba_f4ae811c,8000,58,41.04,117.92000000000002,131.36000000000004,cont,1.7370026266531642,59000827,1,1,0,0
6bba_f4ae811c,8074,59,38.959999999999994,124.8,128.64000000000004,cont,0.8281124999056779,60000839,1,0,1,0
6bba_f4ae811c,8464,64,37.720000000000006,52.0,130.72,cont,2.4702138065357766,65000901,1,1,0,0
6bba_f4ae811c,8546,65,35.64,51.99999999999999,127.67999999999999,cont,1.1501969939971126,66000914,1,0,1,0
6bba_f4ae811c,8624,66,29.240000000000002,157.92000000000002,149.28000000000003,cont,2.2489671240371707,67000924,1,1,0,0
6bba_f4ae811c,8680,67,2.7199999999999998,181.92000000000002,69.12,cont,3.2630550294164533,68000942,2,2,0,0
6bba_f4ae811c,8687,67,10.719999999999999,73.12,129.44000000000005,cont,2.5168621957707544,68000933,3,0,2,1
6bba_f4ae811c,8692,67,19.2,34.88000000000001,146.72000000000003,cont,3.6010527974468793,68000936,1,0,0,1
6bba_f4ae811c,8783,68,54.92000000000001,66.72,147.68,cont,4.012697230355148,69000952,1,1,0,0
6bba_f4ae811c,8847,69,52.84,67.35999999999999,146.72000000000003,cont,4.201842787694462,70000963,1,0,1,0
6bba_f4ae811c,8850,69,56.99999999999997,87.46666666666665,147.46666666666667,cont,6.7422270703422065,70000963,1,0,0,1
6bba_f4ae811c,9873,86,58.2,16.640000000000004,176.64,cont,6.177928581854925,87001120,1,1,0,0
6bba_f4ae811c,9927,87,56.879999999999995,16.64,176.0,cont,6.500507792665119,88001129,1,0,1,0
6bba_f4ae811c,10150,91,55.68000000000001,84.64000000000001,168.64000000000004,cont,1.24405398295252,92001168,1,1,0,0
6bba_f4ae811c,10205,92,54.31999999999999,85.92,166.71999999999997,cont,0.6898120214956988,93001177,1,0,1,0
6bba_f8ffd5e7,486,1,55.080000000000005,128.95999999999998,124.0,cont,2.2946473040535116,2000021,1,1,0,0
6bba_f8ffd5e7,744,2,54.480000000000004,131.36,124.00000000000003,cont,1.1033259377899072,3000033,1,0,1,0
6bba_f8ffd5e7,1285,4,60.84000000000001,124.63999999999999,136.0,cont,1.0163722251222718,5000054,1,1,0,0
6bba_f8ffd5e7,1543,5,60.000000000000014,127.36,133.28,cont,0.28379404944431036,6000065,1,0,1,0
6bba_f8ffd5e7,1562,5,61.99999999999999,139.99999999999994,136.00000000000003,cont,5.984436194204074,6000065,2,0,0,2
6bba_f8ffd5e7,1974,7,34.0,130.72,165.27999999999997,cont,0.5966206918973008,8000083,1,1,0,0
6bba_f8ffd5e7,2064,7,58.0,132.64000000000001,128.0,cont,0.9825540697590104,8000086,2,2,0,0
6bba_f8ffd5e7,2085,7,62.000000000000014,139.35999999999999,128.64000000000004,cont,6.938426514707811,8000086,1,1,0,0
6bba_f8ffd5e7,2313,8,54.0,120.64000000000001,171.99999999999994,cont,0.2600000000000051,9000098,2,1,1,0
6bba_f8ffd5e7,2325,8,56.99999999999999,134.72,125.92,cont,0.8330403651684594,9000096,2,0,2,0
6bba_f8ffd5e7,2571,9,52.84,122.71999999999998,170.07999999999998,cont,2.060869052123391,10000108,1,0,1,0
6bba_f8ffd5e7,2750,10,29.720000000000006,136.64,160.0,parent,3.9123372569986796,11000111,1,1,0,0
6bba_f8ffd5e7,3000,11,27.12,139.35999999999999,157.28000000000003,daughter,0.7034579767121908,12000123,1,0,1,0
6bba_f8ffd5e7,3035,11,33.86666666666666,143.46666666666664,155.99999999999997,daughter,1.2566783404630364,12000124,4,0,0,4
6bba_f8ffd5e7,6579,23,61.35999999999999,55.35999999999999,223.20000000000002,cont,1.2377764842652395,24000271,1,0,1,0
6bba_f8ffd5e7,7154,25,59.83999999999999,54.56,222.07999999999996,cont,0.8413985752899955,26000289,1,1,0,0
6bba_f8ffd5e7,7448,26,59.0,55.36,221.28000000000003,cont,1.7748987647187051,27000299,1,0,1,0
6bba_f8ffd5e7,8052,28,57.36,57.28000000000001,218.72000000000006,cont,1.7684898218536536,29000324,1,1,0,0
6bba_f8ffd5e7,8250,29,32.160000000000004,174.72,88.00000000000001,cont,1.997164004907962,30000332,1,1,0,0
6bba_f8ffd5e7,8278,29,38.839999999999996,158.72,100.64000000000003,cont,1.3823010706788872,30000333,1,1,0,0
6bba_f8ffd5e7,8361,29,56.0,58.08,217.27999999999997,cont,1.6928881571149381,30000337,1,0,1,0
6bba_f8ffd5e7,8573,30,31.000000000000007,175.36,86.72,cont,0.28379404944430386,31000344,1,0,1,0
6bba_f8ffd5e7,8606,30,37.84,159.36,99.19999999999999,cont,1.503629982741754,31000345,1,0,1,0
6bba_f8ffd5e7,8688,30,54.68000000000001,58.08000000000001,216.63999999999996,cont,0.972830920561204,31000348,2,2,0,0
6bba_f8ffd5e7,9009,31,52.84,57.279999999999994,216.0,cont,0.28379404944430103,32000360,2,0,2,0
6bba_f8ffd5e7,9127,32,7.68,188.64,129.92000000000002,cont,2.287097300619277,33000362,1,1,0,0
6bba_f8ffd5e7,9460,33,6.32,190.72000000000003,130.08,cont,3.273356696496746,34000374,1,0,1,0
6bba_f8ffd5e7,11135,38,8.84,184.0,128.64000000000001,cont,0.5040120286263001,39000434,1,1,0,0
6bba_f8ffd5e7,11466,39,8.16,183.36,130.08,cont,1.5365489945003432,40000435,1,0,1,0
6bba_f8ffd5e7,12956,43,21.52,172.64,97.92000000000002,cont,1.0517403731434887,44000468,1,1,0,0
6bba_f8ffd5e7,13310,44,21.160000000000004,170.72000000000003,102.72000000000001,cont,1.5376656414838732,45000475,1,0,1,0
6bba_f8ffd5e7,15171,49,30.480000000000004,136.8,126.72000000000001,cont,1.1436352401443362,50000529,2,2,0,0
6bba_f8ffd5e7,15561,50,31.52,130.56,129.28,cont,1.5962265072664361,51000541,2,0,2,0
6bba_f8ffd5e7,17487,55,34.68000000000001,107.35999999999999,150.72,cont,2.211970628760688,56000584,1,1,0,0
6bba_f8ffd5e7,17863,56,35.32000000000001,100.0,155.2,cont,1.4781248678646783,57000590,1,0,1,0
6bba_f8ffd5e7,18978,59,28.16,92.16,164.64,cont,2.6715316299830665,60000613,5,0,0,5
6bba_f8ffd5e7,19374,60,28.320000000000007,89.44,168.64000000000004,cont,1.1719168325866829,61000620,2,2,0,0
6bba_f8ffd5e7,20595,63,40.31999999999999,47.360000000000014,191.20000000000002,cont,1.2354275575686533,64000651,1,1,0,0
6bba_f8ffd5e7,20920,64,28.639999999999997,76.0,189.27999999999997,daughter,1.684443992093523,65000662,3,0,0,3
6bba_f8ffd5e7,20940,64,32.64,46.56000000000001,205.28,daughter,6.890766466801784,65000663,2,0,2,0
6bba_f8ffd5e7,20946,64,32.99999999999999,64.0,195.99999999999997,daughter,3.040096626753831,65000663,4,0,0,4
6bba_f8ffd5e7,21009,64,41.67999999999999,40.000000000000014,198.72,cont,2.207130490025463,65000657,1,0,1,0
6bba_f8ffd5e7,21800,66,44.32,24.64,213.28000000000003,cont,2.7447111072934476,67000681,1,1,0,0
6bba_f8ffd5e7,22110,67,30.840000000000007,61.27999999999999,203.35999999999999,cont,1.1094121472653806,68000691,1,1,0,0
6bba_f8ffd5e7,22202,67,45.68,17.919999999999998,220.64000000000004,cont,2.2971776161193964,68000696,1,0,1,0
6bba_f8ffd5e7,22515,68,32.52,54.080000000000005,207.36000000000004,cont,0.9031550600533537,69000700,6,5,1,0
6bba_f8ffd5e7,22915,69,33.8,46.39999999999999,210.23999999999998,cont,2.9141918369421087,70000714,5,0,5,0
6bba_f8ffd5e7,22944,69,37.959999999999994,37.599999999999994,221.44000000000003,cont,6.0614242602295985,70000714,7,0,4,3
6bba_f8ffd5e7,28794,84,11.96,96.0,196.15999999999997,cont,4.816610744340864,85000942,2,2,0,0
6bba_f8ffd5e7,28870,84,23.64,161.44,86.72,cont,3.140120668461646,85000948,1,1,0,0
6bba_f8ffd5e7,29247,85,14.599999999999998,95.46666666666664,189.59999999999988,cont,0.9385850202713181,86000982,7,0,2,5
6bba_f8ffd5e7,29258,85,17.32,100.8,201.28,cont,6.990768131078297,86000982,4,0,4,0
6bba_f8ffd5e7,29287,85,20.2,155.99999999999997,111.2,cont,6.9215272203828055,86000981,3,0,0,3
6bba_f8ffd5e7,29321,85,25.680000000000007,157.28,89.27999999999999,cont,5.981611535573012,86000981,4,0,1,3
6bba_f8ffd5e7,29709,86,12.0,110.72000000000003,190.56,cont,0.5675880988886142,87000998,1,0,1,0
6bba_f8ffd5e7,29737,86,18.280000000000005,97.28,201.12000000000003,cont,5.704698995126751,87000999,3,0,0,3
6bba_f8ffd5e7,30240,87,22.519999999999996,163.36,96.0,cont,2.633302346864108,88001022,1,1,0,0
6bba_f8ffd5e7,35917,87,25.77302253723145,166.922109375,81.87665679931641,cont,6.431296154397219,88001022,1,1,0,0
6bba_f8ffd5e7,30691,88,18.839999999999996,70.71999999999998,211.36,cont,6.221034002679952,89001046,1,1,0,0
6bba_f8ffd5e7,31159,89,20.640000000000004,65.91999999999999,213.28000000000003,cont,5.91171783938477,90001069,1,0,1,0
6bba_f8ffd5e7,31614,90,18.8,89.28000000000002,200.64,cont,1.7933993559717827,91001085,1,1,0,0
6bba_f8ffd5e7,31998,91,3.9199999999999995,248.64,104.64000000000003,cont,1.9685988037180029,92001092,1,0,1,0
6bba_f8ffd5e7,32078,91,20.520000000000003,84.8,202.08,cont,2.2433241239286015,92001105,2,1,1,0
6bba_f8ffd5e7,32101,91,24.760000000000005,57.75999999999999,221.27999999999997,cont,5.611217002576163,92001104,1,1,0,0
6bba_f8ffd5e7,32103,91,25.840000000000003,73.92,216.64000000000001,cont,4.85643966811079,92001104,1,1,0,0
6bba_f8ffd5e7,32172,91,33.72,118.56000000000003,139.52,cont,6.366060644150353,92001097,3,3,0,0
6bba_f8ffd5e7,32477,92,9.24,233.76,106.55999999999999,cont,6.294814582257369,93001114,1,1,0,0
6bba_f8ffd5e7,32540,92,22.879999999999995,179.68,91.36,cont,4.28285389664416,93001116,1,1,0,0
6bba_f8ffd5e7,32555,92,26.640000000000004,159.20000000000005,97.60000000000001,cont,6.737305655824137,93001116,4,0,2,2
6bba_f8ffd5e7,32558,92,27.133333333333326,53.06666666666665,227.46666666666664,cont,4.663448015774733,93001128,1,0,1,0
6bba_f8ffd5e7,32566,92,27.08,69.28,217.60000000000002,cont,4.095290168290887,93001128,3,2,1,0
6bba_f8ffd5e7,32602,92,34.16,117.28,136.64000000000001,cont,0.41778620429591967,93001120,8,1,3,4
6bba_f8ffd5e7,32885,93,5.640000000000001,243.52000000000004,113.60000000000002,cont,5.067290942900343,94001138,2,2,0,0
6bba_f8ffd5e7,32998,93,26.840000000000007,160.0,98.72,cont,4.422866717413035,94001141,1,1,0,0
6bba_f8ffd5e7,33029,93,33.83522705078126,58.90890380859375,234.24036376953123,cont,5.179890864655202,94001150,4,3,1,0
6bba_f8ffd5e7,33064,93,37.800000000000004,103.99999999999999,151.83999999999997,cont,1.3635483352268882,94001144,2,2,0,0
6bba_f8ffd5e7,33324,94,6.48,238.72000000000003,113.92,cont,5.711984796242025,95001154,6,0,2,4
6bba_f8ffd5e7,33337,94,9.44,234.24,123.52000000000001,cont,4.2106318186823195,95001154,6,0,2,4
6bba_f8ffd5e7,33456,94,32.13333333333333,36.533333333333324,224.53333333333327,cont,4.797420910847154,95001174,4,0,0,4
6bba_f8ffd5e7,33457,94,33.31522705078125,54.908903808593756,234.72036376953122,cont,4.515660600847161,95001174,3,0,3,0
6bba_f8ffd5e7,33461,94,32.69639038085937,146.01915771484374,111.10767333984377,cont,2.314989628418801,95001170,1,0,1,0
6bba_f8ffd5e7,33474,94,35.72503051757813,82.95596069335939,157.6441772460937,cont,6.957709935302878,95001172,1,0,1,0
6bba_f8ffd5e7,33938,95,37.32,102.72000000000001,153.92000000000002,cont,1.3573371219044965,96001182,1,0,1,0
6bba_f8ffd5e7,34002,95,47.43999999999999,54.08000000000001,227.83999999999997,cont,6.355827665418577,96001184,2,0,0,2
6bba_fbc898dc,703,12,16.36,79.36,143.99999999999997,cont,1.1260675490395815,13000101,1,1,0,0
6bba_fbc898dc,758,13,16.160000000000004,78.72,143.99999999999997,cont,1.592582906005213,14000111,1,0,1,0
6bba_fbc898dc,2329,39,57.68000000000001,28.000000000000004,136.64000000000001,cont,0.9377249730064708,40000320,1,1,0,0
6bba_fbc898dc,2381,40,56.84000000000001,26.720000000000002,137.28,cont,2.1636922291536553,41000329,1,0,1,0
6bba_fbc898dc,2975,50,52.24,24.64,141.6,cont,3.372883132425434,51000423,1,1,0,0
6bba_fbc898dc,3028,51,51.13333333333331,20.533333333333328,155.46666666666664,cont,2.638645091077377,52000430,2,0,0,2
6bba_fbc898dc,3029,51,51.866666666666646,23.466666666666654,139.4666666666666,cont,4.510167910769346,52000430,1,0,1,0
6bba_fbc898dc,3433,58,57.48,28.640000000000008,228.64000000000004,cont,1.3538307916796597,59000494,1,0,0,1
6bba_fc516dc6,1084,5,53.16,93.12,128.48000000000002,cont,3.082535482358645,6000079,1,1,0,0
6bba_fc516dc6,1143,6,10.360000000000001,226.56000000000003,174.07999999999996,cont,3.055086434947467,7000082,2,0,0,2
6bba_fc516dc6,1355,7,37.040000000000006,232.63999999999993,219.36,cont,3.3639070011669334,8000104,2,0,1,1
6bba_fc516dc6,1436,8,10.2,221.44000000000005,161.59999999999997,cont,2.21745525603112,9000112,2,2,0,0
6bba_fc516dc6,1497,8,38.16000000000001,230.72000000000003,220.64000000000004,cont,1.7396609763111581,9000120,2,2,0,0
6bba_fc516dc6,1531,8,57.32000000000001,25.28,82.56000000000002,cont,1.6882803751154694,9000126,1,1,0,0
6bba_fc516dc6,1575,9,9.319999999999997,217.92000000000002,151.2,cont,1.8760130596560374,10000128,2,0,2,0
6bba_fc516dc6,1629,9,34.73333333333333,232.0,199.46666666666658,cont,2.4718642453060213,10000134,3,0,1,2
6bba_fc516dc6,1632,9,36.48,138.23999999999998,92.80000000000001,cont,2.788189376638544,10000135,3,0,2,1
6bba_fc516dc6,1635,9,37.8,180.64000000000001,143.20000000000002,cont,0.6038793857220203,10000137,1,1,0,0
6bba_fc516dc6,1663,9,54.0,14.240000000000002,56.96,cont,4.31274510173045,10000142,2,0,0,2
6bba_fc516dc6,1760,10,37.00000000000001,179.35999999999999,135.20000000000002,cont,0.6711859839120655,11000153,1,0,1,0
6bba_fc516dc6,2338,15,31.840000000000003,118.72,97.44,cont,2.2634214283910894,16000229,1,1,0,0
6bba_fc516dc6,2384,15,60.68000000000001,91.99999999999999,206.08,cont,1.5375797743531878,16000237,1,1,0,0
6bba_fc516dc6,2445,16,31.160000000000004,116.64000000000001,89.92000000000002,cont,1.9031240763544583,17000244,1,0,1,0
6bba_fc516dc6,2491,16,60.0,89.92,197.92000000000002,cont,0.5285623189369458,17000250,1,0,1,0
6bba_fc516dc6,2549,17,29.639999999999993,128.0,51.2,cont,1.0681825979204167,18000252,2,2,0,0
6bba_fc516dc6,2644,18,23.880000000000006,120.80000000000001,29.279999999999994,cont,5.357400984852629,19000265,3,0,0,3
6bba_fc516dc6,2646,18,25.200000000000003,221.28000000000003,138.07999999999998,cont,1.7840250313546628,19000280,2,0,1,1
6bba_fc516dc6,2714,19,6.44,209.12000000000003,96.64000000000001,cont,2.6714327850425263,20000280,1,1,0,0
6bba_fc516dc6,2738,19,24.040000000000003,122.72000000000001,28.000000000000004,cont,4.96529911611576,20000281,2,2,0,0
6bba_fc516dc6,2762,19,39.720000000000006,141.92000000000002,148.8,cont,5.9070035233610465,20000292,1,1,0,0
6bba_fc516dc6,2805,20,2.04,205.27999999999997,74.24,cont,3.112034101676903,21000296,3,0,1,2
6bba_fc516dc6,2829,20,24.53333333333333,164.53333333333333,65.06666666666665,cont,1.0362693504425766,21000303,1,0,0,1
6bba_fc516dc6,2832,20,25.0,108.5333333333333,52.53333333333332,cont,1.7505976162124932,21000302,2,0,0,2
6bba_fc516dc6,2834,20,26.200000000000003,242.56000000000006,215.67999999999998,cont,4.475687796305732,21000304,2,0,1,1
6bba_fc516dc6,2848,20,37.8,143.36,139.83999999999997,cont,2.2514901315573166,21000306,1,0,1,0
6bba_fc516dc6,2872,20,55.133333333333326,79.99999999999997,157.06666666666658,cont,1.458482135265617,21000308,2,0,0,2
6bba_fc516dc6,3119,23,58.0,126.08,218.08,cont,1.1030865786510158,24000348,1,1,0,0
6bba_fc516dc6,3163,24,32.84,228.64,232.0,cont,0.5479407472528307,25000354,1,1,0,0
6bba_fc516dc6,3191,24,57.52000000000001,124.63999999999999,213.44,cont,1.0155924994799785,25000359,1,0,1,0
6bba_fc516dc6,3234,25,32.16000000000001,228.0,226.56,cont,0.6850102645216478,26000367,1,0,1,0
6bba_fc516dc6,3338,26,60.67999999999999,120.00000000000003,178.71999999999997,cont,1.4234295073869963,27000384,1,1,0,0
6bba_fc516dc6,3409,27,59.52000000000001,120.64000000000001,173.44,cont,2.6654953807875814,28000399,1,0,1,0
6bba_fc516dc6,3478,28,55.36000000000002,117.28,195.35999999999999,cont,1.2796294629696219,29000413,1,1,0,0
6bba_fc516dc6,3481,28,58.040000000000006,121.28000000000003,166.72000000000003,cont,1.7131230618376323,29000414,1,1,0,0
6bba_fc516dc6,3491,29,2.16,191.84000000000003,43.99999999999999,cont,6.896627119469335,30000415,1,1,0,0
6bba_fc516dc6,3495,29,5.199999999999999,199.68000000000006,48.48,cont,1.9403615964556764,30000415,1,1,0,0
6bba_fc516dc6,3506,29,15.84,88.48000000000002,9.919999999999998,cont,0.49529189625916875,30000418,1,1,0,0
6bba_fc516dc6,3521,29,28.519999999999996,217.92000000000002,202.72,cont,2.4607887455041797,30000422,1,1,0,0
6bba_fc516dc6,3548,29,54.31999999999999,115.36000000000001,190.72000000000003,cont,0.9923150016501565,30000428,1,0,1,0
6bba_fc516dc6,3552,29,56.0,122.71999999999998,159.36000000000004,cont,2.2578145544973207,30000429,1,0,1,0
6bba_fc516dc6,3567,30,2.3600000000000003,194.07999999999998,39.199999999999996,cont,4.539832389527172,31000430,3,0,1,2
6bba_fc516dc6,3578,30,14.12,88.00000000000003,6.56,cont,1.503893384685227,31000433,1,0,1,0
6bba_fc516dc6,3591,30,26.960000000000004,216.63999999999996,195.84000000000003,cont,0.3769159389041764,31000437,1,0,1,0
6bba_fc516dc6,3599,30,30.56,97.12,27.359999999999996,cont,3.2009200337559216,31000438,2,0,2,0
6bba_fc516dc6,4549,44,13.880000000000003,219.20000000000002,171.36,cont,3.4890229649573796,45000600,1,1,0,0
6bba_fc516dc6,4612,45,12.16,222.72,170.72000000000003,cont,3.0370115347986384,46000613,1,0,1,0
6bba_fc516dc6,4615,45,15.200000000000001,236.31999999999996,165.91999999999996,cont,5.754267030864997,46000613,2,0,1,1
6bba_fc516dc6,4627,45,30.720000000000006,120.00000000000003,72.79999999999998,cont,4.998928947784712,46000615,1,0,1,0
6bba_fc516dc6,4694,46,30.359999999999996,120.64000000000001,72.16,cont,3.3150796559057283,47000626,1,1,0,0
6bba_fc516dc6,4765,47,29.880000000000003,121.92000000000002,70.08,cont,1.9228632621691972,48000639,1,0,1,0
6bba_fc516dc6,4933,49,54.32000000000001,1.2799999999999998,113.92,cont,1.221671498398817,50000665,1,0,1,0
6bba_fc516dc6,4934,49,55.0,59.36,178.72000000000003,cont,2.0152620797305705,50000666,1,1,0,0
6bba_fc516dc6,5001,50,54.160000000000004,62.08,177.28,cont,1.576249999999996,51000677,1,0,1,0
6bba_fc516dc6,5244,54,26.88,206.55999999999997,232.64000000000004,cont,1.8399851392062903,55000727,1,1,0,0
6bba_fc516dc6,5310,55,24.119999999999997,213.44,232.0,cont,2.032224766112258,56000737,1,0,1,0
6bba_fc516dc6,5335,55,54.0,98.72,235.35999999999999,cont,0.8451562355564858,56000741,1,0,1,0
6bba_fc516dc6,6010,66,7.200000000000001,182.56000000000003,107.35999999999999,cont,3.0085329169713275,67000869,1,1,0,0
6bba_fc516dc6,6034,66,34.04,64.47999999999999,87.36,cont,3.414085940995043,67000877,1,1,0,0
6bba_fc516dc6,6071,67,5.280000000000002,188.16000000000003,105.28000000000002,cont,2.0214111716817924,68000886,1,0,1,0
6bba_fc516dc6,6074,67,6.999999999999999,211.99999999999994,103.99999999999999,cont,1.8616713760758208,68000887,1,0,0,1
6bba_fc516dc6,6094,67,32.12,70.88,85.28,cont,2.91491664760761,68000893,1,0,1,0
6bba_fc516dc6,6262,70,3.0399999999999996,194.56,100.16,cont,3.7963144133488234,71000937,1,1,0,0
6bba_fc516dc6,6319,71,1.399999999999999,198.93333333333328,93.59999999999997,cont,4.190308966664777,72000952,1,0,1,0
6bba_fc516dc6,6331,71,6.880000000000001,207.19999999999996,96.0,cont,6.422225324994446,72000952,1,0,0,1
6bba_fc5f39dc,416,3,47.480000000000004,100.0,213.28000000000003,cont,0.9821508603569984,4000008,1,0,1,0
6bba_fc5f39dc,778,6,49.20000000000001,101.43999999999998,215.36,cont,1.6947589245966324,7000014,1,1,0,0
6bba_fc5f39dc,888,7,46.84,136.0,188.64000000000001,cont,1.0163722251222678,8000015,1,1,0,0
6bba_fc5f39dc,892,7,49.160000000000004,100.64,214.72,cont,1.7396609763111839,8000016,1,0,1,0
6bba_fc5f39dc,1006,8,47.16000000000001,136.0,187.35999999999999,cont,1.4130961131501343,9000017,1,0,1,0
6bba_fc5f39dc,1414,11,50.680000000000014,102.08,208.00000000000003,cont,2.145246197992184,12000025,1,0,1,0
6bba_fc5f39dc,1716,13,61.68,156.0,134.71999999999997,cont,1.0958814945056818,14000031,1,1,0,0
6bba_fc5f39dc,1832,14,61.32,156.0,133.27999999999997,cont,0.8401422870561774,15000034,1,0,1,0
6bba_fc5f39dc,2472,19,52.31999999999999,103.36,197.92,cont,1.1951209405327883,20000048,1,1,0,0
6bba_fc5f39dc,2604,20,51.639999999999986,100.63999999999999,199.36,cont,1.6276790416111029,21000051,1,0,1,0
6bba_fc5f39dc,2773,21,60.0,160.00000000000006,127.36,cont,1.072499999999998,22000055,1,1,0,0
6bba_fc5f39dc,2912,22,59.519999999999996,160.00000000000006,126.72000000000001,cont,1.163663396347952,23000058,1,0,1,0
6bba_fc5f39dc,3166,24,49.84,99.84,202.08,parent,3.2662499999999963,25000064,1,1,0,0
6bba_fc5f39dc,3303,25,49.00000000000001,104.80000000000001,199.83999999999997,daughter,2.6659906695260647,26000067,4,0,1,3
6bba_fc5f39dc,3313,25,52.26666666666666,82.93333333333331,212.5333333333333,daughter,1.211506250843698,26000068,2,0,0,2
6bba_fc5f39dc,3590,27,47.64,112.64,196.0,cont,2.941968144372068,28000077,1,0,1,0
6bba_fc5f39dc,3899,29,47.519999999999996,114.07999999999998,194.08,cont,1.6521543700574755,30000087,1,1,0,0
6bba_fc5f39dc,4056,30,47.800000000000004,113.92,192.64000000000004,cont,0.3578691276150918,31000092,1,0,1,0
6bba_fc5f39dc,4535,33,50.84,116.63999999999999,190.08000000000004,cont,1.4227800690900958,34000107,1,1,0,0
6bba_fc5f39dc,4693,34,51.68,118.72000000000003,187.99999999999997,cont,1.4668269325315852,35000112,1,0,1,0
6bba_fc5f39dc,5381,38,53.2,120.00000000000003,184.0,cont,0.6600781203766819,39000133,1,1,0,0
6bba_fc5f39dc,5553,39,53.31999999999999,121.28000000000003,184.0,cont,0.7353910524340038,40000138,1,0,1,0
6bba_fc5f39dc,7050,47,53.16,128.0,178.72,cont,0.49555839716424377,48000191,1,1,0,0
6bba_fc5f39dc,7251,48,53.52000000000001,128.64000000000001,178.72000000000003,cont,1.5657446311579464,49000197,1,0,1,0
6bba_fc5f39dc,7417,49,42.2,169.92000000000002,80.00000000000003,cont,1.5333663945710985,50000205,1,1,0,0
6bba_fc5f39dc,7617,50,41.16,171.36,80.00000000000003,cont,1.016372225122264,51000214,1,0,1,0
6bba_fc5f39dc,7695,50,61.32,148.64000000000001,68.64000000000001,cont,1.222643779888476,51000217,1,1,0,0
6bba_fc5f39dc,7716,51,1.6800000000000002,100.00000000000001,244.8,cont,2.7348320332334803,52000218,1,0,1,0
6bba_fc5f39dc,7899,51,61.160000000000004,149.92000000000002,68.64000000000001,cont,2.2773782212228153,52000224,1,0,1,0
6bba_fc5f39dc,7918,52,1.4400000000000004,100.8,239.36,cont,2.5957817343721348,53000225,1,1,0,0
6bba_fc5f39dc,8064,52,48.040000000000006,85.92,142.56,cont,5.083575574829209,53000231,1,0,0,1
6bba_fc5f39dc,8542,54,60.31999999999999,155.35999999999999,63.36,cont,0.884244910926825,55000250,1,0,1,0
6bba_fc5f39dc,8763,55,60.16000000000001,156.0,60.000000000000014,cont,0.6306172571378017,56000261,1,1,0,0
6bba_fc5f39dc,8986,56,60.000000000000014,155.36,55.99999999999999,cont,0.7803384682302871,57000270,1,0,1,0
6bba_fc5f39dc,9741,60,24.040000000000003,108.16,95.2,cont,2.706100746461599,61000299,1,1,0,0
6bba_fc5f39dc,9972,61,23.680000000000007,110.08,91.2,cont,1.9986839419978417,62000309,1,0,1,0
6bba_fc5f39dc,10064,61,48.0,136.0,112.0,cont,1.8168052317185792,62000314,1,1,0,0
6bba_fc5f39dc,10583,63,55.84000000000001,156.64,30.72,cont,2.2785374283737525,64000334,2,2,0,0
6bba_fc5f39dc,10827,64,54.680000000000014,157.44000000000003,26.720000000000002,cont,1.375886645948724,65000343,2,0,2,0
6bba_fc5f39dc,10919,65,10.680000000000001,103.36,191.36,cont,1.7753450368871988,66000345,1,1,0,0
6bba_fc5f39dc,11466,67,24.0,171.36,47.360000000000014,cont,1.7754194046759735,68000372,1,1,0,0
6bba_fc5f39dc,11526,67,39.76837097167969,86.58294677734374,95.678369140625,cont,2.086640444977272,68000376,1,1,0,0
6bba_fc5f39dc,11569,67,50.160000000000004,156.0,12.000000000000004,cont,2.3872018792929968,68000377,1,1,0,0
6bba_fc5f39dc,11715,68,23.0,170.72000000000003,43.36,cont,0.8451562355564698,69000383,1,0,1,0
6bba_fc5f39dc,11816,68,49.0,156.0,8.64,cont,2.0542596750411084,69000388,1,0,1,0
6bba_fc5f39dc,19976,68,40.15883468627929,90.41163024902345,89.99633056640626,cont,1.4463000190031796,69000387,1,0,1,0
6bba_fc5f39dc,11937,69,14.320000000000002,122.07999999999997,69.44000000000001,cont,6.975661482970616,70000394,1,1,0,0
6bba_fc5f39dc,11954,69,18.48,106.72,75.36,cont,3.041572886682155,70000394,1,1,0,0
6bba_fc5f39dc,12193,70,14.36,92.16,179.35999999999999,cont,2.7681339608118685,71000403,1,1,0,0
6bba_fc5f39dc,12208,70,17.68,106.72,70.71999999999998,cont,0.9006664199358216,71000409,1,0,1,0
6bba_fc5f39dc,12217,70,20.36,128.64000000000004,153.44,cont,3.216227652468033,71000405,1,1,0,0
6bba_fc5f39dc,12444,71,14.64,87.83999999999999,179.36000000000004,cont,2.7717089155428996,72000417,2,1,1,0
6bba_fc5f39dc,12454,71,16.133333333333333,135.99999999999994,159.99999999999997,cont,4.627526524625327,72000422,5,0,0,5
6bba_fc5f39dc,12468,71,18.84,128.0,146.56,cont,3.1927011487140478,72000422,1,0,1,0
6bba_fc5f39dc,12709,72,14.800000000000004,85.92,179.36,cont,1.8102529174123765,73000431,1,0,1,0
6bba_fc5f39dc,12793,72,34.839999999999996,118.07999999999998,46.72,cont,2.2301015952193834,73000434,1,1,0,0
6bba_fc5f39dc,13005,73,23.32,107.36,56.0,cont,0.9377249730064784,74000446,1,1,0,0
6bba_fc5f39dc,13050,73,32.52,114.07999999999998,40.00000000000001,cont,1.9505415914560784,74000445,1,0,1,0
6bba_fc5f39dc,13270,74,21.52,104.8,52.64000000000001,cont,2.746201922656092,75000463,1,0,1,0
6bba_fc5f39dc,13492,75,10.68,96.64000000000001,38.720000000000006,cont,2.052523599133511,76000471,1,1,0,0
6bba_fc5f39dc,13748,76,9.160000000000002,93.92,32.64,cont,1.8432113619984012,77000485,1,0,1,0
6bba_fc5f39dc,13783,76,17.64,97.28,44.64,cont,1.0563749926044248,77000486,2,0,0,2
6bba_fc5f39dc,14265,78,4.640000000000001,108.64000000000001,146.08,cont,2.521683775277936,79000508,2,2,0,0
6bba_fc5f39dc,14313,78,14.480000000000002,92.64,161.27999999999997,cont,1.025682699473854,79000509,1,1,0,0
6bba_fc5f39dc,14543,79,3.4799999999999986,110.71999999999997,142.4,cont,4.813073342470483,80000519,2,0,2,0
6bba_fc5f39dc,14555,79,5.533333333333334,108.5333333333333,144.53333333333333,cont,1.6141303127897249,80000519,5,1,0,4
6bba_fc5f39dc,14647,79,26.36,100.64000000000003,15.359999999999996,cont,1.243735552679908,80000528,1,1,0,0
6bba_fc5f39dc,14835,80,6.8000000000000025,107.04,142.72000000000003,cont,1.8113466157806348,81000535,1,0,1,0
6bba_fc5f39dc,14911,80,25.480000000000004,100.80000000000001,11.360000000000003,cont,1.0095945968555808,81000540,1,0,1,0
6bba_fc5f39dc,20037,81,32.76852249145507,88.03787597656252,33.194756774902345,cont,6.251171802328651,82000554,1,1,0,0
6bba_fc5f39dc,15358,82,7.959999999999998,104.64,134.07999999999998,cont,1.1969975459456967,83000559,2,2,0,0
6bba_fc5f39dc,15463,82,33.99260925292968,91.10625,32.69863037109375,cont,2.116703176568936,83000564,1,0,1,0
6bba_fc5f39dc,15631,83,8.72,104.0,128.48,cont,0.7540070042778052,84000570,2,0,2,0
6bba_fc5f39dc,16724,87,23.92000000000001,140.32,246.40000000000003,cont,0.30530978120589525,88000614,1,0,1,0
6bba_fc5f39dc,17397,90,5.359999999999999,110.71999999999998,52.160000000000004,cont,1.5910069531589113,91000645,1,1,0,0
6bba_fc5f39dc,17446,90,20.68,140.64000000000001,226.72,cont,1.453807501012426,91000648,2,1,1,0
6bba_fc5f39dc,17622,91,4.4799999999999995,108.64000000000001,44.64000000000001,cont,1.799719596214926,92000653,1,0,1,0
6bba_fc5f39dc,17682,91,19.479999999999997,140.64,219.36000000000004,cont,1.7381424172086717,92000656,2,1,1,0
6bba_fc5f39dc,17930,92,18.000000000000004,140.00000000000003,210.71999999999997,cont,2.039294072099454,93000665,3,2,1,0
6bba_fc5f39dc,18180,93,16.52,140.00000000000003,201.28,cont,1.2405467544594901,94000674,2,0,2,0
6bba_fc5f39dc,18395,94,3.28,85.44,79.19999999999999,cont,6.536056245933017,95000677,3,0,0,3
6bba_fc5f39dc,19110,96,32.64,155.35999999999999,68.0,cont,1.3002031091333404,97000695,1,1,0,0
6bba_fc5f39dc,19363,97,31.160000000000007,156.0,62.56,cont,0.7964157990647883,98000701,1,0,1,0
6bba_fc83837d,307,1,3.0,191.03999999999996,170.07999999999998,cont,6.935914233177902,2000012,3,3,0,0
6bba_fc83837d,472,1,34.2,204.32,205.76,cont,4.494527922930281,2000017,3,1,2,0
6bba_fc83837d,558,2,3.160000000000001,189.92000000000002,166.55999999999995,cont,3.930552203571388,3000024,3,0,3,0
6bba_fc83837d,718,2,31.480000000000004,224.64000000000004,239.20000000000002,cont,5.712631973311419,3000032,2,1,1,0
6bba_fc83837d,1017,3,38.6,239.46666666666658,223.99999999999997,cont,3.396772620607839,4000050,2,1,0,1
6bba_fc83837d,1102,4,4.32,192.64,164.64,cont,4.519019528614585,5000056,2,1,1,0
6bba_fc83837d,1256,4,38.8,240.95999999999998,224.0,cont,2.107869926015348,5000066,1,0,1,0
6bba_fc83837d,1356,5,5.160000000000001,196.64000000000001,164.64,cont,4.665136988878245,6000071,1,0,1,0
6bba_fc83837d,1502,5,39.32000000000001,8.640000000000002,235.99999999999994,cont,0.540175029504336,6000077,1,1,0,0
6bba_fc83837d,1758,6,39.52,9.28,235.99999999999994,cont,0.9268200000539615,7000091,1,0,1,0
6bba_fc83837d,1995,7,33.13333333333334,235.46666666666664,232.0,cont,6.960627428715657,8000103,1,0,1,0
6bba_fc83837d,2005,7,36.68000000000001,226.88,236.80000000000007,cont,4.707818795631351,8000103,4,0,0,4
6bba_fc83837d,2015,7,39.32,246.72000000000006,221.28000000000003,cont,2.9473486496341006,8000105,1,0,1,0
6bba_fc83837d,18479,7,28.631152954101562,209.0978552246094,221.82135742187506,cont,6.908074678930758,8000100,3,0,3,0
6bba_fc83837d,2467,9,28.04,194.07999999999998,219.36,cont,3.210187397177931,10000121,7,0,1,6
6bba_fc83837d,2477,9,31.000000000000007,201.28000000000006,195.36,cont,5.457799035554506,10000122,1,0,1,0
6bba_fc83837d,2724,10,36.64,234.56000000000006,238.07999999999998,cont,1.1628688930829774,11000133,1,0,1,0
6bba_fc83837d,2730,10,38.67999999999999,250.71999999999997,216.64,cont,1.5945713569796673,11000134,1,1,0,0
6bba_fc83837d,3488,13,40.16,22.560000000000002,227.99999999999997,cont,1.504946531442239,14000164,2,1,1,0
6bba_fc83837d,4190,16,39.720000000000006,1.2799999999999998,249.27999999999997,cont,2.8312798011676503,17000193,1,0,1,0
6bba_fc83837d,5524,22,33.68000000000001,14.080000000000002,227.99999999999997,cont,4.172422857285667,23000253,3,1,1,1
6bba_fc83837d,5534,22,36.733333333333334,48.53333333333333,199.99999999999997,cont,5.545804376333129,23000260,1,0,0,1
6bba_fc83837d,5776,23,54.2,20.799999999999997,243.84000000000003,cont,5.036608792133855,24000264,1,0,1,0
6bba_fc83837d,6305,26,34.519999999999996,32.64,215.35999999999996,cont,1.4840974740562205,27000292,1,0,1,0
6bba_fc83837d,6490,27,33.86666666666666,60.53333333333332,195.46666666666664,cont,3.4801819875786286,28000301,3,0,0,3
6bba_fc83837d,7003,30,32.480000000000004,43.36,203.36,cont,1.0859572390292311,31000326,1,1,0,0
6bba_fc83837d,7221,31,55.160000000000004,45.28,232.0,cont,2.137662640011278,32000338,1,1,0,0
6bba_fc83837d,8306,38,56.84,164.0,141.28000000000003,cont,1.6988829050290792,39000433,1,0,1,0
6bba_fc83837d,8589,40,54.32000000000001,241.92000000000007,86.08,cont,3.222748244899093,41000457,1,0,1,0
6bba_fc83837d,9003,43,44.84,26.719999999999995,227.36,cont,0.7597635322914598,44000498,1,1,0,0
6bba_fc83837d,9025,43,56.13333333333332,219.99999999999997,127.46666666666667,cont,6.963788093994211,44000504,3,0,0,3
6bba_fc83837d,9241,45,23.320000000000007,59.2,161.44,cont,2.2528384873310396,46000524,1,1,0,0
6bba_fc83837d,9600,47,49.905158081054694,237.6006103515625,60.05162109375,cont,2.193188740358974,48000562,1,0,1,0
6bba_fc83837d,9709,48,31.48,128.64000000000001,133.28000000000003,cont,3.7755992901789854,49000576,1,0,1,0
6bba_fc83837d,9798,49,2.1600000000000006,8.0,72.0,cont,1.4241713599493557,50000585,1,0,1,0
6bba_fc83837d,9820,49,9.52,154.71999999999997,71.99999999999999,cont,4.30480872106996,50000589,1,1,0,0
6bba_fc83837d,10639,54,59.480000000000025,133.28000000000003,195.20000000000005,cont,6.2694892385663685,55000686,1,1,0,0
6bba_fc83837d,10650,54,62.000000000000014,138.56,191.36,cont,1.6473525161603724,55000686,1,1,0,0
6bba_fc83837d,10794,55,61.99999999999999,148.0,200.53333333333333,cont,6.399580199990557,56000702,3,0,0,3
6bba_fc83837d,11007,57,30.84,5.279999999999999,196.64000000000001,cont,1.906243442611671,58000722,1,0,1,0
6bba_fc83837d,11371,59,62.000000000000014,146.72000000000003,176.64000000000001,cont,1.6495951207796136,60000759,2,1,1,0
6bba_fc83837d,11439,60,26.48,116.0,107.99999999999999,cont,1.2406531797807099,61000764,1,1,0,0
6bba_fc83837d,11483,60,45.32,240.00000000000006,48.64,cont,2.331181299255802,61000774,1,0,1,0
6bba_fc83837d,11490,60,50.0,183.36,84.0,cont,2.1377244034252842,61000768,1,1,0,0
6bba_fc83837d,11521,60,62.000000000000014,148.64000000000001,173.91999999999996,cont,1.7565791506504602,61000773,1,0,1,0
6bba_fc83837d,11581,61,26.16,116.0,105.28000000000002,cont,0.28379404944431036,62000778,1,0,1,0
6bba_fc83837d,11936,63,55.84000000000001,201.28,80.8,cont,3.7868078988113263,64000811,1,1,0,0
6bba_fc83837d,12051,64,40.328154296875,62.996604003906256,203.98005615234374,cont,6.432053287074278,65000819,1,1,0,0
6bba_fc83837d,12074,64,52.03999999999999,53.92,226.23999999999995,cont,3.2855956937061093,65000822,1,1,0,0
6bba_fc83837d,12080,64,55.68000000000001,202.56,76.96000000000001,cont,4.536079150819552,65000824,1,0,1,0
6bba_fc83837d,12088,64,60.640000000000015,160.64,181.28000000000003,cont,6.8995747151545395,65000825,2,2,0,0
6bba_fc83837d,12105,65,2.32,7.359999999999999,18.4,cont,2.545084109612097,66000826,1,0,1,0
6bba_fc83837d,12233,65,61.84,159.36,159.2,cont,1.8500751673918476,66000835,2,0,2,0
6bba_fc83837d,13345,74,28.840000000000003,120.00000000000003,36.160000000000004,cont,6.473478706422061,75000923,1,1,0,0
6bba_fc83837d,13385,74,44.32,193.43999999999997,14.720000000000002,cont,1.45969870606917,75000926,1,1,0,0
6bba_fc83837d,13533,75,44.16000000000001,189.92000000000002,10.560000000000004,cont,1.235534423842579,76000938,1,0,1,0
6bba_fc83837d,13662,76,37.64,247.36,142.72,cont,2.8472763849861873,77000950,1,1,0,0
6bba_fc83837d,14189,79,46.519999999999996,68.0,186.71999999999997,cont,2.4565464197730926,80000992,1,0,1,0
6bba_fc83837d,15176,84,51.133333333333326,215.99999999999994,252.0,cont,6.387533974686787,85001054,2,0,0,2
6bba_fc83837d,15748,87,42.8,222.72000000000003,134.72000000000003,cont,4.795760171755882,88001093,2,2,0,0
6bba_fc83837d,15972,88,48.0,252.53333333333336,116.53333333333333,cont,4.6810311846204335,89001111,4,1,0,3
6bba_fc83837d,15991,88,52.84,238.56000000000006,130.72,cont,6.863600663099504,89001111,4,1,3,0
6bba_fc83837d,18462,88,35.67149322509766,240.05944152832032,138.33301208496093,cont,6.435530164295742,89001104,1,0,1,0
6bba_fc83837d,16108,89,35.800000000000004,228.00000000000003,245.28,cont,3.7028612247827954,90001117,2,0,2,0
6bba_fc83837d,16146,89,44.879999999999995,218.72000000000006,135.36000000000004,cont,5.227198401151428,90001119,2,1,1,0
6bba_fc83837d,16186,89,52.84,235.99999999999994,128.64000000000004,cont,6.8546691431826545,90001122,3,2,1,0
6bba_fc83837d,16223,89,61.0,200.8,242.56000000000006,cont,1.6667937972346771,90001124,1,1,0,0
6bba_fc83837d,16320,90,37.040000000000006,225.92000000000002,244.64000000000001,cont,3.3421711225638857,91001137,4,0,0,4
6bba_fc83837d,16444,90,61.99999999999999,203.99999999999994,252.0,cont,6.953879874753348,91001141,1,0,0,1
6bba_fc83837d,16945,93,46.28,212.0,136.0,cont,3.879089777254451,94001191,3,3,0,0
6bba_fc83837d,16949,93,46.959999999999994,235.84000000000003,122.56,cont,3.965566045661075,94001192,4,4,0,0
6bba_fc83837d,17015,93,61.160000000000004,190.72000000000003,242.56,cont,3.0238101998968245,94001194,1,1,0,0
6bba_fc83837d,17106,94,36.00000000000001,227.20000000000005,143.99999999999997,cont,3.266209576864274,95001204,3,0,3,0
6bba_fc83837d,17125,94,40.16,226.56000000000003,132.8,cont,5.776229306390104,95001204,3,0,3,0
6bba_fc83837d,17131,94,40.99999999999999,229.59999999999997,155.99999999999994,cont,6.924864619615292,95001204,2,0,2,0
6bba_fc83837d,17155,94,47.84,211.36,136.64000000000001,cont,4.94005345365818,95001207,9,0,3,6
6bba_fc83837d,17160,94,48.0,239.51999999999992,124.00000000000003,cont,1.7796532316718487,95001206,9,0,4,5
6bba_fc83837d,17203,94,55.68000000000001,168.0,95.36000000000001,cont,3.2216419047591174,95001212,1,0,1,0
6bba_fc83837d,17232,94,61.32000000000001,188.64000000000004,245.60000000000008,cont,3.319537335985273,95001214,2,0,1,1
6bba_fc83837d,17336,95,36.84,224.64000000000004,143.99999999999997,cont,3.7879234499260797,96001222,1,1,0,0
6bba_fe670320,147,3,59.999999999999986,248.5333333333333,67.99999999999997,cont,1.6857063512895367,4000057,1,0,0,1
6bba_fe670320,395,9,58.999999999999986,28.640000000000004,80.64000000000001,cont,1.7623572637238167,10000141,1,1,0,0
6bba_fe670320,401,9,60.67999999999999,38.71999999999999,27.36,cont,0.5924010993406504,10000143,1,1,0,0
6bba_fe670320,438,10,58.999999999999986,34.080000000000005,84.0,cont,1.7162031129793731,11000156,1,0,1,0
6bba_fe670320,446,10,60.84000000000001,41.92000000000001,29.279999999999994,cont,0.8650740575233931,11000158,1,0,1,0
6bba_fe670320,633,15,51.16,114.56000000000002,211.36,cont,1.848004413549928,16000219,1,1,0,0
6bba_fe670320,674,16,51.00000000000001,120.80000000000001,209.92000000000002,cont,0.6558641722948405,17000232,1,0,1,0
6bba_fe670320,2840,54,54.31999999999999,10.560000000000002,104.64000000000003,cont,2.251724686212755,55000598,1,1,0,0
6bba_fe670320,2938,55,54.0,16.639999999999997,107.35999999999999,cont,1.6521543700574723,56000607,1,0,1,0
6bba_fe670320,3142,57,52.64000000000001,27.360000000000003,110.71999999999998,cont,1.4003314339826909,58000623,1,1,0,0
6bba_fe670320,3250,58,52.00000000000001,33.92,112.64,cont,1.1952314106481654,59000633,1,0,1,0
6bba_fe670320,3363,59,51.2,40.480000000000004,115.83999999999999,cont,2.060933116939991,60000644,2,2,0,0
6bba_fe670320,3504,60,50.32000000000001,48.800000000000004,120.0,cont,2.3223291110434796,61000654,2,0,2,0
6bba_fe670320,3671,61,51.99999999999999,15.466666666666672,81.06666666666665,cont,1.6956848829373512,62000665,4,0,0,4
6bba_fe670320,3674,61,52.84,4.480000000000001,74.07999999999998,cont,5.268208898667557,62000665,3,0,1,2
6bba_fe670320,3810,62,49.32000000000001,58.72000000000001,123.52000000000001,cont,1.9629569531703903,63000674,3,3,0,0
6bba_fe670320,3823,62,52.000000000000014,17.28,78.72,cont,1.7836549554215657,63000675,3,3,0,0
6bba_fe670320,3827,62,53.64,1.9199999999999997,76.0,cont,6.036607076827182,63000675,3,3,0,0
6bba_fe670320,3963,63,47.86666666666666,68.0,139.46666666666667,cont,1.127460657727015,64000683,4,0,0,4
6bba_fe670320,3972,63,49.52,26.72,101.12,cont,3.4609829709202553,64000684,5,0,3,2
6bba_fe670320,5330,71,41.68,87.19999999999997,1.2799999999999998,cont,2.8285271122971314,72000751,1,1,0,0
6bba_fe670320,5514,72,41.31999999999999,94.07999999999998,2.72,cont,3.13637629279398,73000760,1,0,1,0
6bba_fe670320,5684,73,38.88000000000001,144.64000000000004,221.28000000000006,cont,0.43724278152988455,74000765,2,2,0,0
6bba_fe670320,5693,73,41.16,98.56,4.000000000000001,cont,3.2683109169722595,74000768,1,1,0,0
6bba_fe670320,5696,73,41.64000000000001,102.72000000000001,178.23999999999998,cont,2.5683740600815947,74000766,1,1,0,0
6bba_fe670320,5869,74,37.480000000000004,149.44,225.28,cont,2.504240466189308,75000771,2,0,2,0
6bba_fe670320,5881,74,38.68000000000001,116.47999999999999,201.76,cont,3.132248094420362,75000773,4,0,1,3
6bba_fe670320,5886,74,40.0,106.72,207.36000000000007,cont,5.861949601028676,75000773,2,0,2,0
6bba_fe670320,5893,74,41.0,107.36,5.279999999999999,cont,3.162702562682743,75000774,1,0,1,0
6bba_fe670320,7189,80,49.8,17.28,2.5599999999999996,cont,3.6243329127578803,81000808,1,1,0,0
6bba_fe670320,7403,81,49.52,18.72,6.7200000000000015,cont,4.06500172970443,82000815,2,1,1,0
6bba_fe670320,7423,81,53.84,4.0,6.720000000000001,cont,6.807663994535869,82000815,1,1,0,0
6bba_fe670320,7609,82,49.519999999999996,17.439999999999998,9.92,cont,4.515161265115568,83000822,3,2,1,0
6bba_fe670320,7619,82,52.533333333333324,20.00000000000001,-1.0666666666666627,cont,5.439638787538088,83000822,5,0,0,5
6bba_fe670320,7625,82,53.67999999999999,25.92,26.720000000000002,cont,6.763827041697618,83000822,2,1,1,0
6bba_fe670320,7707,83,31.840000000000003,149.92000000000002,222.56000000000003,cont,1.0908096190444916,84000826,2,2,0,0
6bba_fe670320,7818,83,49.480000000000004,16.160000000000004,15.36,cont,4.591192995834524,84000829,3,1,2,0
6bba_fe670320,7834,83,52.040000000000006,27.360000000000003,1.9200000000000004,cont,6.649129877284697,84000829,5,2,0,3
6bba_fe670320,7917,84,30.320000000000007,155.52,227.51999999999998,cont,3.2423535047863035,85000832,2,0,2,0
6bba_fe670320,8032,84,46.67999999999999,24.159999999999997,29.91999999999999,cont,5.7948739470759945,85000835,5,0,2,3
6bba_fe670320,8049,84,51.48,30.72,38.72,cont,5.023773637665218,85000835,2,0,2,0
6bba_fe670320,8238,85,46.32,28.16,31.200000000000006,cont,6.313809888846833,86000841,1,1,0,0
6bba_fe670320,8259,85,50.48000000000001,33.440000000000005,43.36,cont,5.486174241901182,86000841,1,1,0,0
6bba_fe670320,8456,86,46.0,31.360000000000003,32.0,cont,4.944407984025995,87000846,1,0,1,0
6bba_fe670320,8474,86,49.84000000000002,35.36,47.36,cont,5.870119914661032,87000846,1,0,1,0
6bba_fe670320,8958,89,25.84,165.27999999999997,239.36,cont,2.413603698310884,90000858,1,1,0,0
6bba_fe670320,9078,89,45.01934448242188,34.80686645507813,41.46705871582031,cont,3.2424502243634534,90000861,1,1,0,0
6bba_fe670320,9096,89,48.68000000000001,36.63999999999999,53.92,cont,5.661533581989963,90000861,1,1,0,0
6bba_fe670320,9169,90,25.16,168.64,242.72000000000003,cont,2.4094974839165033,91000863,1,0,1,0
6bba_fe670320,9204,90,31.680000000000003,45.28,41.92000000000001,cont,2.148259833562966,91000866,1,1,0,0
6bba_fe670320,9296,90,44.69934448242188,35.446866455078116,44.9870587158203,cont,2.522590570346514,91000867,1,0,1,0
6bba_fe670320,9307,90,48.16000000000001,37.92,56.64,cont,4.977940242961957,91000867,1,0,1,0
6bba_fe670320,9413,91,31.32,50.080000000000005,45.44,cont,1.4936746842937347,92000873,1,0,1,0


In [ ]:
import hashlib
EXPECTED_SHA256 = {
 "tools/divnet_infer.py": "9ad85ab0418870e06b40a2ab2108d18dd9e5e6507b01a4c6429e7fc716783d72",
 "tools/daughternet.py": "c609eb7ca3044b87a59917c4b7e925e6cf7940728f4f44a71a21a1fd11272e26",
 "kdn_core.py": "62d413bf62e3c10c48de554e425b24004409b2e1d534b71ace85abdb6fec63e3",
 "splits.json": "0fd3cff10858b4387a3d03bb2dc80b0ad8206a76b4bd8cf580bec75a4c8ff213",
 "pool_samples.csv": "e626c3b0332041fdcd22d43c9ab597f0ab63a8864bb4c8d2f2c77f1f501bbc32"
}
_bad = {}
for _rel, _want in EXPECTED_SHA256.items():
    _got = hashlib.sha256((KDN_SRC / _rel).read_text(encoding="utf-8").rstrip().encode("utf-8")).hexdigest()
    if _got != _want:
        _bad[_rel] = _got
if _bad:
    raise RuntimeError(f"embedded source drift: {_bad}")
print("embedded sources OK:", ", ".join(EXPECTED_SHA256))


In [ ]:
# ---- configuration (every knob is an env var read by kdn_core.config_from_env) ----
os.environ.setdefault("KDN_SMOKE", "0")        # 1 = 5 division-rich videos, 1 epoch, 2 folds
os.environ.setdefault("KDN_MAX_VIDEOS", "0")   # 0 = all videos
os.environ.setdefault("KDN_EPOCHS", "1" if os.environ["KDN_SMOKE"] != "0" else "30")
os.environ.setdefault("KDN_PATIENCE", "6")
os.environ.setdefault("KDN_BATCH", "64")
os.environ.setdefault("KDN_LR", "1e-3")
os.environ.setdefault("KDN_EPOCH_SAMPLES", "0")  # 0 = one pass worth of training samples per epoch
os.environ.setdefault("KDN_CONT_PER_VIDEO", "60")
os.environ.setdefault("KDN_HARDNEG_PER_VIDEO", "40")
os.environ.setdefault("KDN_INNER_VAL", "0")    # 1 = early-stop on fold k+1 and train on the other K-2 folds
os.environ.setdefault("KDN_CACHE_DTYPE", "float32")
os.environ.setdefault("KDN_DIVNET", "1")        # in-sample DivNet comparison (giorgosi/biohub-divnet-v2)
os.environ.setdefault("KDN_EXTRA_SAMPLES", str(KDN_SRC / "pool_samples.csv"))  # v2 pool rows ("" = v1)
os.environ.setdefault("KDN_POOL_WEIGHT", "1.0")   # loss weight of pool rows
os.environ.setdefault("KDN_ES_SOURCE", "all")     # early-stop loss over all / gt / pool rows

if str(KDN_SRC) not in sys.path:
    sys.path.insert(0, str(KDN_SRC))
import kdn_core
cfg = kdn_core.config_from_env()
if Path("/kaggle/working").exists() and not torch.cuda.is_available() and not cfg["smoke"]:
    raise RuntimeError("K-DN full run needs the GPU accelerator (2x T4)")
for _k, _v in cfg.items():
    print(f"  {_k:20s} {_v}")


In [ ]:
state = kdn_core.step_prepare(cfg)


In [ ]:
state = kdn_core.step_build_cache(cfg, state)


In [ ]:
state = kdn_core.step_train(cfg, state)


In [ ]:
state = kdn_core.step_oof(cfg, state)


In [ ]:
state = kdn_core.step_verify(cfg, state)
import json
print(json.dumps(state["timing"], indent=1))
for _p in sorted(Path(state["work_dir"]).rglob("*")):
    if _p.is_file() and "kdn_src" not in _p.parts:
        print(f"{_p.stat().st_size / 2**20:9.2f} MiB  {_p}")
if not state["verify"]["pass"]:
    raise RuntimeError("contract check failed: see contract_check.json")
